<a href="https://colab.research.google.com/github/Vishesh-Goyal7/NARAD-QC/blob/main/NARAD_QC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <b>NARAD-QC : Network Architecture for Routing, Authentication and Delivery in Quantum Communications</b>

## <i>The Big Question</i>

How can a Quantum Secure Communication Network Dynamically Select Reliable, Authenticated, Low QBER Paths Between Multiple Users and Repeaters?

<br><hr><br>

## <b>Sec 1 : The Phase Plan</b>

    Phase 0 :
        Goal : Rebuild point to point CrypTon key generation and AES baseline.
        Objective : Prove the original point-to-point system is working as the baseline.

    Phase 1 :
        Goal : Add distance-dependent photon loss, QBER, keygen latency, and accepted/rejected rounds.
        Objective : Evaluate how distance affects QKD in terms of photon loss, QBER, Key generation latency.

        Variables :
            * Distance: 10 km, 25 km, 50 km, 75 km, 100 km
            * Fiber attenuation model: approximate 0.2 dB/km
            * Noise factor: optional small baseline noise
        
        Outputs :
            * Distance vs QBER
            * Distance vs key generation time
            * Distance vs successful key rate
            * Distance vs accepted/rejected rounds

        Question : How does increasing physical separation affect secure key establishment in CrypTon?

    Phase 2 :
        Goal : Add a single repeater so that Alice <--> Bob chain becomes ===> Alice <--> R1 <--> Bob
        Objective : Measure whether splitting distance improves stability and key availability.

        Metrics :
            * end-to-end QBER
            * key generation time
            * accepted key rate
            * maximum supported distance
            * improvement over direct link
        
        Question : Can a single repeater improve the effective QKD range and reliability compared to direct transmission?

    Phase 3 :
        Goal : Add multiple repeaters, quantum memory, and reimagine the entanglement swapping once again.
        Objectives :
        Phase 3A :
            * Introduce ML-DSA based authentication
            * Introduce a control plane.
            * Create a visual graphic to start imagining the way network is being laid out.

        Phase 3B :
            * Entanglement creation engine
            * Parallel Entanglement Pairs are generated and stored

        Phase 3C :
            * Bell State Measurement based entanglement swapping begins
            * First baseline assumes an ideal swap and ideal fidelity.
            * Changed to real probability based swapping and imperfect Bell Pairs after the first experiment succeeded
            * Giving the control plane the authority, the intelligence to control the parallel entanglement swaps so that it can do it more safely, more securely  
            * Modelling Werner-State Physical Fidelity for non ideal fidelity (Refer Sec 2)
            * Modelling entanglement swap policies (Refer Sec 3)

        Phase 3D :
            * There is a problem till now : Although the repeater swapping is happening successfully, we still are not being able to generate a good quality Bell Pair.
            * The pair which starts at 0.995 at the time of generation loses its fidelity to 0.5-0.6 which causes a huge spike in QBER.
            * So, we can't send the information reliably. To counteract this, we tried to implement a hardware based Fidelity restoration from RRI in this Phase 3D.
            * But it failed terribly. The linear algebra and probability maths was not adding up giving huge errors with minimal improvement.

<br><hr><br>

## <b>Sec 2 : Werner State Fidelity Model</b>

### <u><i>2.1 : The Fidelity</i></u>
We need Bell Pairs to transfer data from one point to other if we wish to use <u>entanglement</u>. The Bell Pair is represented as :
$$
|\Phi^+\rangle = \frac{|00\rangle+|11\rangle}{\sqrt{2}}
$$

<br>

But this is a perfect world. In a real, noise affected, non ideal environment, the Bell Pair is not just this pure state but a mixture of multiple quantum states. The Fidelity answers <i>"how close a quantum state is to the ideal state we want?"</i> which is represented as

$$F = \langle\Phi^+|\rho|\Phi^+\rangle$$

In an ideal state, $F = 1$.

<br>

### <u><i>2.2 : Werner State Physical Fidelity Model</i></u>
Werner State Physical Fidelity Model, which we implemented, uses Bell Depolarised Form :
$$\rho_W = w|\Phi^+\rangle\langle\Phi^+| + (1-w)\frac{I_4}{4}$$

where
*  $|Φ^+⟩$ is our desired Bell State
*  $w \epsilon [0,1]$ is the Werner Visibility constant
*  $\frac{I_4}{4}$ is completely mixed two qubit system

When the werner visibility is 1, we have a perfect, ideal scenario.

When the werner visibility is 0, we have the highest mixed state. The Bell Pair Fidelity with relation to werner visibility is represented as

$$F = \frac{1+3w}{4}$$

<br>

At $w = 0$, the $F_{floor} = 0.25$

This means that in the least perfect state, there are four possibilities of Bell Pairs with equal probabilities of each Bell Pair.

<br>

For our elementary Bell Pair getting generated, we model the starting pair as $F_{elementary} = 0.995$

<br>

### <u><i>2.3 : Time Decoherence Fidelity</i></u>
Even when a Bell Pair is generated, it can lose coherence as the time passes on. Suppose the Bell Pair is sitting in the memory because the neighbouring Bell Pair is not generated yet (quite a possible scenario). The Bell Pair would decay which is represented and we model it as Time Exponential Decay :

$$F(t) = F_{floor} + (F_0-F_{floor})e^{-\lambda t}$$

with $F_{floor} = 0.25$. To calculate the $\lambda$ instead of arbitrarily assuming it, we use chosen benchmark values as $F(0) = 1$ and $F(1) = 0.90$

<br>

$$0.9 = 0.25 + 0.75*e^{-\lambda} \Rightarrow e^{-\lambda} = \frac{0.65}{0.75} \rightarrow \lambda = -ln(\frac{0.65}{0.75}) \Rightarrow \lambda = 0.1431/second$$

But the problem is that until now, we were modelling simulation slots which represent the ordering mechanism. Hence, we made an explicit conversion assumption of $20 slots/second$

<br>

### <u><i>2.4 : Outer Entanglement Swapping</u></i>
Once a Bell Pair Swapping succeeds, the outer qubit Bell Pair that is left has a lower Fidelity which is represented as

$$w_{out} = w_1w_2 \implies F_{out} = \frac{1+3w_{out}}{4} = \frac{1+3w_1w_2}{4}$$

Suppose we have two ideal Bell Pairs created at the same slot. According to the implementation model, they both will have a fidelity of $F_{elementary}=0.995$. Then, they get swapped at the next slot immediately.

At the time of generation, both the pairs will have werner visibility of

$$w = \frac{4F-1}{3} = \frac{4*0.995-1}{3} = 0.9933$$

After the swap, the visibility of the outer states is going to be $w = 0.9867$. Hence the fidelity becomes

$$F_{outer} = \frac{1+3w_{out}}{4} = 0.990025$$

Hence, we can see that even two Bell Pairs which generated perfectly will lose Decohrence when they interact by measurement.

<br>

### <u><i>2.5 : Noisy Bell Pair Swapping</u></i>
When the Bell Pairs swap, we assume that an ideal swap happens. But in a real world, the swap will lose data due to gate imperfections etc. So, to adjust that, the werner visibility will get affected according to

$$w_{out} = w_1w_2q_{swap}$$

To prepare for that, our code models this but we currently assume it as $q_{swap} = 1$ for now.

<br><hr><br>

## <b>Sec 3 : Entanglement Swapping Policies</b>
To model the age swapping, we use three different policies.
<i><br>Check <a href="https://github.com/Vishesh-Goyal7/NARAD-QC/blob/main/References/1.pdf">Paper 1</a> in the References of the git.</i>

### <i><u>3.1 : Sequential Left to Right</u></i>
This is the most basic policy. The rule is "

> <i>Take exactly one leftmost BSM". </i>

Advantages :
1. Simplest
2. Most deterministic
3. Excellent baseline.

Disadvantages :
1. Parallelism
2. Terribly long wait time
3. Risks the expiry of pairs that were generated at a prior time.

<br>

### <i><u>3.2 : Static Balanced Swap</u></i>
This swapping policy creates the swapping tree before the execution begins and only allows those swaps which are present as adjacent neighbours until all the nodes in the tree are resolved.

> <i>Only merge resources that are siblings in the predefined tree.</i>

Advantages :
1. Fewer logical swap levels generally mean less accumulated waiting
2. More parallel execution.

Terrible Disadvantage :
The plan says merge E2 and E3.

But the E3 ain't available yet. However, another Bell Pair is sitting somewhere with both neighbours ready, waiting for the execution.

And until E3 ain't ready, the Static Balanced Tree says
> <i>Not on my watch, wait for my plan.</i>

<br>

### <i><u>3.3 : Max Parallel Oldest First</u></i>
> <i>I am workaholic. Let's start working with what we have.</i>

This policy discovers all the currently valid candidates, checks all the repeaters where the neighbouring Bell Pairs exist, and then all the unconflicted resources.

Then, it initiates parallel swaps so that oldest Bell Pair gets consumed first. The priority order :

1. Maximum Parallel BSM Count
2. Absolutely no conflicts
3. Which is the oldest?

In [ ]:
# @title Sys Conf

import os
import psutil

print("Logical CPUs :", os.cpu_count())
print("Physical CPUs:", psutil.cpu_count(logical=False))
print("Logical CPUs :", psutil.cpu_count(logical=True))

vm = psutil.virtual_memory()

print(
    "RAM total     :",
    f"{vm.total / (1024**3):.2f} GiB"
)

print(
    "RAM available :",
    f"{vm.available / (1024**3):.2f} GiB"
)

try:
    print(
        "CPU affinity  :",
        psutil.Process().cpu_affinity()
    )
except Exception:
    pass

## Phase 0 : 06-07-2026T16:15:32 -- 06-07-2026T16:48:53

Goal : Rebuild point to point CrypTon key generation and AES baseline.

Objective : Prove the original point-to-point system is working as the baseline.

In [ ]:
# @title Dependency Installation

!pip install qiskit qiskit-aer pycryptodome pandas matplotlib "pqcrypto==0.4.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# @title Imports and Global Config

import time
import secrets
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

RNG = np.random.default_rng()
BACKEND = AerSimulator()

QBER_CUTOFF = 0.11
NONCE_LEN = 12

In [ ]:
# @title AES-GCM Engine

def aes_gcm_encrypt(data: bytes, key: bytes) -> bytes:
    nonce = get_random_bytes(NONCE_LEN)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return nonce + ciphertext + tag


def aes_gcm_decrypt(blob: bytes, key: bytes) -> bytes:
    nonce = blob[:NONCE_LEN]
    rest = blob[NONCE_LEN:]
    ciphertext = rest[:-16]
    tag = rest[-16:]

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    return cipher.decrypt_and_verify(ciphertext, tag)

In [ ]:
# @title BB84 Step 1 : QRNG

def random_bitstring(n: int) -> list[int]:
    return RNG.integers(0, 2, size=n).tolist()


def random_bases(n: int) -> list[int]:
    return RNG.integers(0, 2, size=n).tolist()

In [ ]:
# @title BB84 Step 2 : Qubit Encoding

def encode_qubits(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    circuits = []

    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)

        if bit == 1:
            qc.x(0)

        if basis == 1:
            qc.h(0)

        circuits.append(qc)

    return circuits

In [ ]:
# @title BB84 Step 3 : Measure Qubits With Photon Loss Support

def measure_qubits(
    qubits: list,
    bob_bases: list[int],
    eve_present: bool = False
) -> list:

    measured_bits = []

    for i, prepared_qubit in enumerate(qubits):

        # Photon lost in QuantumLink
        if prepared_qubit is None:
            measured_bits.append(None)
            continue

        circuit = prepared_qubit.copy()

        if eve_present:
            eve_basis = int(RNG.integers(0, 2))

            eve_circuit = circuit.copy()

            if eve_basis == 1:
                eve_circuit.h(0)

            eve_circuit.measure(0, 0)

            transpiled_eve = transpile(eve_circuit, BACKEND)
            eve_result = BACKEND.run(
                transpiled_eve,
                shots=1,
                memory=True
            ).result()

            eve_bit = int(eve_result.get_memory()[0])

            circuit = QuantumCircuit(1, 1)

            if eve_bit == 1:
                circuit.x(0)

            if eve_basis == 1:
                circuit.h(0)

        if bob_bases[i] == 1:
            circuit.h(0)

        circuit.measure(0, 0)

        transpiled_bob = transpile(circuit, BACKEND)
        bob_result = BACKEND.run(
            transpiled_bob,
            shots=1,
            memory=True
        ).result()

        measured_bits.append(int(bob_result.get_memory()[0]))

    return measured_bits

In [ ]:
# @title QBER, Sifting, and Key Reconciliation With Photon Loss

def sift_keys(
    alice_bits: list[int],
    alice_bases: list[int],
    bob_bits: list,
    bob_bases: list[int]
) -> tuple[list[int], list[int]]:

    alice_sifted = []
    bob_sifted = []

    for i in range(len(alice_bits)):
        if bob_bits[i] is None:
            continue

        if alice_bases[i] == bob_bases[i]:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])

    return alice_sifted, bob_sifted

def calculate_qber(alice_key: list[int], bob_key: list[int]) -> float:
    if len(alice_key) == 0:
        return 1.0

    errors = sum(a != b for a, b in zip(alice_key, bob_key))
    return errors / len(alice_key)

In [ ]:
# @title One BB84 Round

def bb84_round(
    n_qubits: int = 256,
    eve_present: bool = False
) -> dict:

    alice_bits = random_bitstring(n_qubits)
    alice_bases = random_bases(n_qubits)
    bob_bases = random_bases(n_qubits)

    qubits = encode_qubits(alice_bits, alice_bases)
    bob_bits = measure_qubits(qubits, bob_bases, eve_present=eve_present)

    alice_sifted, bob_sifted = sift_keys(
        alice_bits,
        alice_bases,
        bob_bits,
        bob_bases
    )

    qber = calculate_qber(alice_sifted, bob_sifted)

    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,
        "eve_present": eve_present
    }

In [ ]:
# @title Accumulate Session Key

def bits_to_bytes(bits: list[int]) -> bytes:
    usable_len = len(bits) - (len(bits) % 8)
    bits = bits[:usable_len]

    return bytes(
        int("".join(map(str, bits[i:i+8])), 2)
        for i in range(0, usable_len, 8)
    )


def generate_bb84_session_key(
    target_bits: int = 128,
    n_qubits_per_round: int = 256,
    max_rounds: int = 30,
    eve_present: bool = False,
    verbose: bool = True
) -> tuple[bytes, dict]:

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):
        result = bb84_round(
            n_qubits=n_qubits_per_round,
            eve_present=eve_present
        )

        round_logs.append({
            "round": round_id,
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"]
        })

        if verbose:
            print(
                f"Round {round_id}: "
                f"sifted={result['sifted_length']}, "
                f"QBER={result['qber']:.2%}, "
                f"accepted={result['accepted']}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough accepted BB84 key bits."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    metadata = {
        "target_bits": target_bits,
        "rounds_used": len(round_logs),
        "accepted_rounds": sum(r["accepted"] for r in round_logs),
        "rejected_rounds": sum(not r["accepted"] for r in round_logs),
        "total_sifted_bits": sum(r["sifted_length"] for r in round_logs),
        "keygen_time_s": elapsed,
        "round_logs": round_logs
    }

    return key, metadata

In [ ]:
# @title Phase 0 End to End System COMMENTED OUT

# message = b"NARAD-QC Phase 0: CrypTon baseline reconstruction."

# session_key, meta = generate_bb84_session_key(
#     target_bits=128,
#     n_qubits_per_round=256,
#     max_rounds=30,
#     eve_present=False,
#     verbose=True
# )

# ciphertext = aes_gcm_encrypt(message, session_key)
# decrypted = aes_gcm_decrypt(ciphertext, session_key)

# print("\nSession Key:", session_key.hex())
# print("Key Metadata:", meta)
# print("Ciphertext:", ciphertext.hex())
# print("Decrypted:", decrypted.decode())
# print("Roundtrip OK:", decrypted == message)

In [ ]:
# @title Eve Presence Validation

# try:
#     eve_key, eve_meta = generate_bb84_session_key(
#         target_bits=128,
#         n_qubits_per_round=256,
#         max_rounds=5,
#         eve_present=True,
#         verbose=True
#     )

#     print("Unexpectedly generated key with Eve present.")
#     print(eve_meta)

# except RuntimeError as e:
#     print("Eve disrupted key generation as expected.")
#     print(str(e))

## Phase 1 : 06-07-2026T17:08:12 -- 07-07-2026T16:37:51

Goal : Add distance-dependent photon loss, QBER, keygen latency, and accepted/rejected rounds.

Objective : Evaluate how distance affects QKD in terms of photon loss, QBER, Key generation latency.

Question : How does increasing physical separation affect secure key establishment in CrypTon?

### Phase 1A

QuantumLink and Node classes

In [ ]:
# @title Node Class Definition

ROLE_USER = "USER"
ROLE_REPEATER = "REPEATER"

class Node:
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role

    def __str__(self):
        return f"{self.name} ({self.role})"

In [ ]:
# @title Create 2 User Nodes

alice = Node(name="Alice", role=ROLE_USER)
bob = Node(name="Bob", role=ROLE_USER)

print(alice)
print(bob)

In [ ]:
# @title Quantum Link Class

import random
import math

STANDARD_FIBER_ATTENUATION = 0.2  # dB/km


class QuantumLink:
    """
    NARAD-QC Phase 1.1 / 1.2

    Models a physical quantum channel with distance-dependent photon loss.
    Lost photons are represented as None.
    """

    def __init__(
        self,
        source: Node,
        destination: Node,
        distance_km: float,
        attenuation_db_per_km: float = STANDARD_FIBER_ATTENUATION
    ):
        self.source = source
        self.destination = destination
        self.distance_km = distance_km
        self.attenuation = attenuation_db_per_km
        self.status = "ACTIVE"

        # η(d) = 10^(-(αd)/10)
        self.survival_probability = 10 ** (
            -(self.attenuation * self.distance_km) / 10
        )

        self.loss_probability = 1 - self.survival_probability

        self.statistics = {
            "lifetime_sent": 0,
            "lifetime_received": 0,
            "lifetime_lost": 0
        }

    def transmit(self, qubit):
        """
        Transmit one qubit through the quantum link.
        Returns the qubit if it survives, otherwise None.
        """

        self.statistics["lifetime_sent"] += 1

        if random.random() < self.survival_probability:
            self.statistics["lifetime_received"] += 1
            return qubit

        self.statistics["lifetime_lost"] += 1
        return None

    def transmit_batch(self, qubits: list):
        """
        Transmit a batch of qubits.

        Returns:
        - transmitted_qubits: list containing QuantumCircuit objects or None
        - session-level transmission statistics
        """

        transmitted_qubits = []
        survived = 0

        for qubit in qubits:
            received = self.transmit(qubit)
            transmitted_qubits.append(received)

            if received is not None:
                survived += 1

        sent = len(qubits)
        lost = sent - survived

        return {
            "transmitted_qubits": transmitted_qubits,
            "sent": sent,
            "survived": survived,
            "lost": lost,
            "expected_survival_probability": self.survival_probability,
            "observed_survival_probability": survived / sent if sent else 0,
            "survival_percentage": (survived / sent) * 100 if sent else 0
        }

    def reset_statistics(self):
        self.statistics = {
            "lifetime_sent": 0,
            "lifetime_received": 0,
            "lifetime_lost": 0
        }

    def __str__(self):
        return (
            f"\n{'='*60}\n"
            f"Quantum Link\n"
            f"{'='*60}\n"
            f"Source                          : {self.source.name}\n"
            f"Destination                     : {self.destination.name}\n"
            f"Distance                        : {self.distance_km} km\n"
            f"Attenuation                     : {self.attenuation} dB/km\n"
            f"Status                          : {self.status}\n"
            f"Photon Survival Probability     : {self.survival_probability*100:.2f}%\n"
            f"Photon Loss Probability         : {self.loss_probability*100:.2f}%\n"
            f"{'='*60}"
        )

In [ ]:
# @title Testing one Quantum Link

link = QuantumLink(
    source=alice,
    destination=bob,
    distance_km=50
)

print(link)

In [ ]:
# @title Testing Batch Transmission

dummy_qubits = [f"Photon_{i}" for i in range(1000)]

result = link.transmit_batch(dummy_qubits)

print("Transmission Statistics\n")

print(f"Sent        : {result['sent']}")
print(f"Survived    : {result['survived']}")
print(f"Lost        : {result['lost']}")
print(f"Survival %  : {result['survival_percentage']:.2f}%")

In [ ]:
# @title Distance Comparison

distances = [10, 20, 50, 80, 100, 150, 200]

print(
    f"{'Distance':<12}"
    f"{'Survival %':<15}"
    f"{'Loss %':<15}"
)

print("-" * 42)

for d in distances:
    test_link = QuantumLink(alice, bob, d)

    print(
        f"{d:<12}"
        f"{test_link.survival_probability*100:<15.2f}"
        f"{test_link.loss_probability*100:<15.2f}"
    )

### Phase 1B

Distance --> Attenuation Model --> Testing Over Multiple Distances

In [ ]:
# @title BB84 / Biased-Basis Generator

def random_bases(n: int, z_probability: float = 0.5) -> list[int]:
    """
    Generate random bases.

    0 = Z / computational basis
    1 = X / diagonal basis

    z_probability = 0.5      -> Standard BB84
    z_probability = 0.8      -> Biased BB84
    z_probability = 15/16    -> T12-style biased basis
    """

    return [
        0 if random.random() < z_probability else 1
        for _ in range(n)
    ]


def basis_match_probability(z_probability: float) -> float:
    """
    P(match) = P(Z)^2 + P(X)^2
    """

    return z_probability**2 + (1 - z_probability)**2

In [ ]:
# @title One Distance-Aware BB84 Round With Estimated Photon Budget

import math

def bb84_round_over_link(
    link: QuantumLink,
    target_bits: int = 128,
    transmission_margin: float = 1.3,
    eve_present: bool = False,
    z_probability: float = 0.5
) -> dict:

    p_basis = basis_match_probability(z_probability)

    estimated_photons_required = math.ceil(
        (target_bits * transmission_margin) /
        (p_basis * link.survival_probability)
    )

    n_qubits = max(1, estimated_photons_required)

    alice_bits = random_bitstring(n_qubits)
    alice_bases = random_bases(n_qubits, z_probability=z_probability)
    bob_bases = random_bases(n_qubits, z_probability=z_probability)

    encoded_qubits = encode_qubits(alice_bits, alice_bases)

    transmission = link.transmit_batch(encoded_qubits)
    received_qubits = transmission["transmitted_qubits"]

    bob_bits = measure_qubits(received_qubits, bob_bases, eve_present=eve_present)

    alice_sifted, bob_sifted = sift_keys(alice_bits, alice_bases, bob_bits, bob_bases)

    qber = calculate_qber(alice_sifted, bob_sifted)
    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,
        "eve_present": eve_present,
        "z_probability": z_probability,
        "basis_match_probability": p_basis,
        "target_bits_for_round": target_bits,
        "transmission_margin": transmission_margin,
        "estimated_photons_required": estimated_photons_required,
        "photons_sent": transmission["sent"],
        "photons_received": transmission["survived"],
        "photons_lost": transmission["lost"],
        "expected_survival_probability": transmission["expected_survival_probability"],
        "observed_survival_probability": transmission["observed_survival_probability"]
    }

In [ ]:
# @title Generate Session Key Over Quantum Link

def generate_bb84_session_key_over_link(
    link: QuantumLink,
    target_bits: int = 128,
    max_rounds: int = 100,
    transmission_margin: float = 1.3,
    eve_present: bool = False,
    z_probability: float = 0.5,
    verbose: bool = True
) -> tuple[bytes, dict]:

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):

        remaining_bits = target_bits - len(collected_bits)

        result = bb84_round_over_link(
            link=link,
            target_bits=remaining_bits,
            transmission_margin=transmission_margin,
            eve_present=eve_present,
            z_probability=z_probability
        )

        round_log = {
            "round": round_id,
            "distance_km": link.distance_km,
            "photons_sent": result["photons_sent"],
            "photons_received": result["photons_received"],
            "photons_lost": result["photons_lost"],
            "observed_survival_probability": result["observed_survival_probability"],
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"],
            "z_probability": z_probability,
            "basis_match_probability": result["basis_match_probability"],
            "target_bits_for_round": result["target_bits_for_round"],
            "transmission_margin": result["transmission_margin"],
            "estimated_photons_required": result["estimated_photons_required"],
        }

        round_logs.append(round_log)

        if verbose:
            print(
                f"Round {round_id}: "
                f"distance={link.distance_km} km | "
                f"sent={result['photons_sent']} | "
                f"received={result['photons_received']} | "
                f"sifted={result['sifted_length']} | "
                f"QBER={result['qber']:.2%} | "
                f"accepted={result['accepted']}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough accepted BB84 key bits over the quantum link."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    total_sent = sum(r["photons_sent"] for r in round_logs)
    total_received = sum(r["photons_received"] for r in round_logs)
    total_lost = sum(r["photons_lost"] for r in round_logs)
    total_sifted = sum(r["sifted_length"] for r in round_logs)

    metadata = {
        "source": link.source.name,
        "destination": link.destination.name,
        "distance_km": link.distance_km,
        "target_bits": target_bits,
        "z_probability": z_probability,
        "basis_match_probability": basis_match_probability(z_probability),
        "rounds_used": len(round_logs),
        "accepted_rounds": sum(r["accepted"] for r in round_logs),
        "rejected_rounds": sum(not r["accepted"] for r in round_logs),
        "total_photons_sent": total_sent,
        "total_photons_received": total_received,
        "total_photons_lost": total_lost,
        "observed_survival_probability": total_received / total_sent if total_sent else 0,
        "total_sifted_bits": total_sifted,
        "effective_key_yield": target_bits / total_sent if total_sent else 0,
        "keygen_time_s": elapsed,
        "round_logs": round_logs
    }

    return key, metadata

In [ ]:
# @title End-to-End Distance-Aware Test

# distance_link = QuantumLink(
#     source=alice,
#     destination=bob,
#     distance_km=50
# )

# session_key, meta = generate_bb84_session_key_over_link(
#     link=distance_link,
#     target_bits=128,
#     max_rounds=100,
#     eve_present=False,
#     z_probability=0.5,   # Standard BB84
#     verbose=True,
#     transmission_margin = 1.3,
# )

# message = b"NARAD-QC Phase 1.2: Distance-aware BB84 session key."

# ciphertext = aes_gcm_encrypt(message, session_key)
# decrypted = aes_gcm_decrypt(ciphertext, session_key)

# print("\n" + "="*60)
# print("NARAD-QC Phase 1.2 Session Report")
# print("="*60)
# print(f"Source                     : {meta['source']}")
# print(f"Destination                : {meta['destination']}")
# print(f"Distance                   : {meta['distance_km']} km")
# print(f"Target Key Bits            : {meta['target_bits']}")
# print(f"Z-Basis Probability        : {meta['z_probability']}")
# print(f"Basis Match Probability    : {meta['basis_match_probability']*100:.2f}%")
# print(f"Rounds Used                : {meta['rounds_used']}")
# print(f"Total Photons Sent         : {meta['total_photons_sent']}")
# print(f"Total Photons Received     : {meta['total_photons_received']}")
# print(f"Total Photons Lost         : {meta['total_photons_lost']}")
# print(f"Observed Survival          : {meta['observed_survival_probability']*100:.2f}%")
# print(f"Total Sifted Bits          : {meta['total_sifted_bits']}")
# print(f"Effective Key Yield        : {meta['effective_key_yield']*100:.4f}%")
# print(f"Keygen Time                : {meta['keygen_time_s']:.4f} s")
# print(f"AES Roundtrip OK           : {decrypted == message}")
# print("="*60)

In [ ]:
# @title Standard BB84 vs T12-Style Biased Basis

# test_distance = 50
# target_bits = 128

# protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15/16
# }

# comparison_results = []

# for protocol_name, z_prob in protocols.items():

#     test_link = QuantumLink(alice, bob, test_distance)

#     key, meta = generate_bb84_session_key_over_link(
#         link=test_link,
#         target_bits=target_bits,
#         max_rounds=100,
#         eve_present=False,
#         z_probability=z_prob,
#         verbose=False,
#         transmission_margin = 1.3
#     )

#     comparison_results.append({
#         "Protocol": protocol_name,
#         "Distance (km)": test_distance,
#         "Z Probability": z_prob,
#         "Basis Match Probability (%)": meta["basis_match_probability"] * 100,
#         "Rounds Used": meta["rounds_used"],
#         "Photons Sent": meta["total_photons_sent"],
#         "Photons Received": meta["total_photons_received"],
#         "Sifted Bits": meta["total_sifted_bits"],
#         "Effective Key Yield (%)": meta["effective_key_yield"] * 100,
#         "Keygen Time (s)": meta["keygen_time_s"]
#     })

# df_protocol_comparison = pd.DataFrame(comparison_results)
# display(df_protocol_comparison)

In [ ]:
# @title Distance Performance : BB84 vs Biased BB84 vs T12

# distances = [10, 50, 80, 100, 150, 200]
# target_bits = 128
# transmission_margin = 1.3

# protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15/16
# }

# distance_sweep_results = []

# for d in distances:
#     for protocol_name, z_prob in protocols.items():

#         test_link = QuantumLink(
#             source=alice,
#             destination=bob,
#             distance_km=d
#         )

#         try:
#             key, meta = generate_bb84_session_key_over_link(
#                 link=test_link,
#                 target_bits=target_bits,
#                 transmission_margin=transmission_margin,
#                 max_rounds=10,
#                 eve_present=False,
#                 z_probability=z_prob,
#                 verbose=False
#             )

#             distance_sweep_results.append({
#                 "Protocol": protocol_name,
#                 "Distance (km)": d,
#                 "Z Probability": z_prob,
#                 "Basis Match Probability (%)": meta["basis_match_probability"] * 100,
#                 "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                 "Observed Photon Survival (%)": meta["observed_survival_probability"] * 100,
#                 "Rounds Used": meta["rounds_used"],
#                 "Photons Sent": meta["total_photons_sent"],
#                 "Photons Received": meta["total_photons_received"],
#                 "Photons Lost": meta["total_photons_lost"],
#                 "Sifted Bits": meta["total_sifted_bits"],
#                 "Effective Key Yield (%)": meta["effective_key_yield"] * 100,
#                 "Keygen Time (s)": meta["keygen_time_s"],
#                 "Status": "SUCCESS"
#             })

#         except RuntimeError as e:
#             distance_sweep_results.append({
#                 "Protocol": protocol_name,
#                 "Distance (km)": d,
#                 "Z Probability": z_prob,
#                 "Basis Match Probability (%)": basis_match_probability(z_prob) * 100,
#                 "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                 "Observed Photon Survival (%)": None,
#                 "Rounds Used": None,
#                 "Photons Sent": None,
#                 "Photons Received": None,
#                 "Photons Lost": None,
#                 "Sifted Bits": None,
#                 "Effective Key Yield (%)": None,
#                 "Keygen Time (s)": None,
#                 "Status": "FAILED"
#             })

# df_distance_sweep = pd.DataFrame(distance_sweep_results)

# display(df_distance_sweep)

# df_distance_sweep.to_csv("distance_sweep_results.csv", index=False)

In [ ]:
# @title Plot: Distance vs Key Generation Time

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Keygen Time (s)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Key Generation Time (s)")
# plt.title("Distance vs Key Generation Time")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Key Generation Time", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Effective Key Yield

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Effective Key Yield (%)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Effective Key Yield (%)")
# plt.title("Distance vs Effective Key Yield")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Effective Key Yield", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Photon Survival

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Observed Photon Survival (%)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Observed Photon Survival (%)")
# plt.title("Distance vs Observed Photon Survival")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Observed Photon Survival", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Photons Sent

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Photons Sent"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Photons Sent")
# plt.title("Distance vs Photons Required for 128-bit Key")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Photons Required for 128-bit Key", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# @title Plot: Distance vs Sifted Bits

# plt.figure(figsize=(10, 6))

# for protocol_name in protocols.keys():
#     subset = df_distance_sweep[df_distance_sweep["Protocol"] == protocol_name]
#     plt.plot(
#         subset["Distance (km)"],
#         subset["Sifted Bits"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Sifted Bits")
# plt.title("Distance vs Sifted Bits Generated")
# plt.legend()
# plt.grid(True)
# plt.savefig("Phase 1 Distance vs Sifted Bits Generated", dpi=300, bbox_inches="tight")
# plt.show()

### Phase 1C

Monte Carlo - 100 Trials

In [ ]:
# @title Phase 1 Monte Carlo Experiment — 100 Trials

# import gc
# import os
# import time
# import pandas as pd

# PHASE1_MONTE_CARLO_TRIALS = 100
# PHASE1_CHECKPOINT_INTERVAL = 1

# phase1_mc_distances = [10, 50, 80, 100, 150, 200]

# phase1_mc_protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15 / 16
# }

# phase1_mc_target_bits = 128
# phase1_mc_transmission_margin = 1.3
# phase1_mc_max_rounds = 10

# PHASE1_RAW_FILE = "phase1_monte_carlo_raw_100_trials.csv"
# PHASE1_LATEST_CHECKPOINT_FILE = "phase1_monte_carlo_latest_checkpoint.csv"

# # Set this to True only when resuming from a saved latest checkpoint.
# RESUME_FROM_CHECKPOINT = True

# phase1_raw_results = []
# starting_trial = 98

# if RESUME_FROM_CHECKPOINT and os.path.exists(PHASE1_LATEST_CHECKPOINT_FILE):

#     existing_checkpoint = pd.read_csv(PHASE1_LATEST_CHECKPOINT_FILE)

#     phase1_raw_results = existing_checkpoint.to_dict("records")

#     if not existing_checkpoint.empty:
#         last_completed_trial = int(existing_checkpoint["Trial"].max())
#         starting_trial = last_completed_trial + 1

#         print(
#             f"Resuming Phase 1 Monte Carlo experiment "
#             f"from Trial {starting_trial}."
#         )

# else:
#     print("Starting a new Phase 1 Monte Carlo experiment.")


# # ---------------------------------------------------------
# # Monte Carlo execution
# # ---------------------------------------------------------

# experiment_start_time = time.perf_counter()

# for trial in range(
#     starting_trial,
#     PHASE1_MONTE_CARLO_TRIALS + 1
# ):

#     trial_start_time = time.perf_counter()

#     print(
#         f"\n{'=' * 72}\n"
#         f"Phase 1 Monte Carlo — Trial "
#         f"{trial}/{PHASE1_MONTE_CARLO_TRIALS}\n"
#         f"{'=' * 72}"
#     )

#     for distance_km in phase1_mc_distances:

#         for protocol_name, z_probability in phase1_mc_protocols.items():

#             configuration_start_time = time.perf_counter()

#             test_link = QuantumLink(
#                 source=alice,
#                 destination=bob,
#                 distance_km=distance_km
#             )

#             try:
#                 session_key, metadata = (
#                     generate_bb84_session_key_over_link(
#                         link=test_link,
#                         target_bits=phase1_mc_target_bits,
#                         transmission_margin=(
#                             phase1_mc_transmission_margin
#                         ),
#                         max_rounds=phase1_mc_max_rounds,
#                         eve_present=False,
#                         z_probability=z_probability,
#                         verbose=False
#                     )
#                 )

#                 configuration_elapsed = (
#                     time.perf_counter()
#                     - configuration_start_time
#                 )

#                 phase1_raw_results.append({
#                     "Trial": trial,
#                     "Protocol": protocol_name,
#                     "Distance (km)": distance_km,
#                     "Target Key Bits": phase1_mc_target_bits,
#                     "Transmission Margin": phase1_mc_transmission_margin,
#                     "Z Probability": z_probability,
#                     "Basis Match Probability (%)": metadata["basis_match_probability"] * 100,
#                     "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                     "Observed Photon Survival (%)": metadata["observed_survival_probability"] * 100,
#                     "Rounds Used": metadata["rounds_used"],
#                     "Accepted Rounds": metadata["accepted_rounds"],
#                     "Rejected Rounds": metadata["rejected_rounds"],
#                     "Photons Sent": metadata["total_photons_sent"],
#                     "Photons Received": metadata["total_photons_received"],
#                     "Photons Lost": metadata["total_photons_lost"],
#                     "Sifted Bits": metadata["total_sifted_bits"],
#                     "Effective Key Yield (%)": metadata["effective_key_yield"] * 100,
#                     "Keygen Time (s)": metadata["keygen_time_s"],
#                     "Configuration Runtime (s)": configuration_elapsed,
#                     "Status": "SUCCESS",
#                     "Error": None
#                 })

#                 print(
#                     f"Trial {trial:3d} | "
#                     f"{distance_km:3d} km | "
#                     f"{protocol_name:<31} | "
#                     f"sent={metadata['total_photons_sent']:,} | "
#                     f"received="
#                     f"{metadata['total_photons_received']:,} | "
#                     f"time="
#                     f"{metadata['keygen_time_s']:.3f}s"
#                 )

#             except Exception as error:

#                 configuration_elapsed = (time.perf_counter() - configuration_start_time)
#                 phase1_raw_results.append({
#                     "Trial": trial,
#                     "Protocol": protocol_name,
#                     "Distance (km)": distance_km,
#                     "Target Key Bits": phase1_mc_target_bits,
#                     "Transmission Margin": phase1_mc_transmission_margin,
#                     "Z Probability": z_probability,
#                     "Basis Match Probability (%)": basis_match_probability(z_probability) * 100,
#                     "Expected Photon Survival (%)": test_link.survival_probability * 100,
#                     "Observed Photon Survival (%)": None,
#                     "Rounds Used": None,
#                     "Accepted Rounds": None,
#                     "Rejected Rounds": None,
#                     "Photons Sent": None,
#                     "Photons Received": None,
#                     "Photons Lost": None,
#                     "Sifted Bits": None,
#                     "Effective Key Yield (%)": None,
#                     "Keygen Time (s)": None,
#                     "Configuration Runtime (s)": configuration_elapsed,
#                     "Status": "FAILED",
#                     "Error": str(error)
#                 })

#                 print(
#                     f"Trial {trial:3d} | "
#                     f"{distance_km:3d} km | "
#                     f"{protocol_name:<31} | "
#                     f"FAILED: {error}"
#                 )

#             # Qiskit circuit batches can consume substantial memory.
#             del test_link

#             if "session_key" in locals():
#                 del session_key

#             if "metadata" in locals():
#                 del metadata

#             gc.collect()

#     trial_elapsed = time.perf_counter() - trial_start_time

#     print(
#         f"\nTrial {trial} completed in "
#         f"{trial_elapsed / 60:.2f} minutes."
#     )

#     if (trial % PHASE1_CHECKPOINT_INTERVAL == 0 or trial == PHASE1_MONTE_CARLO_TRIALS):

#         checkpoint_df = pd.DataFrame(
#             phase1_raw_results
#         )

#         numbered_checkpoint_file = (
#             f"phase1_monte_carlo_checkpoint_"
#             f"trial_{trial}.csv"
#         )

#         checkpoint_df.to_csv(
#             numbered_checkpoint_file,
#             index=False
#         )

#         checkpoint_df.to_csv(
#             PHASE1_LATEST_CHECKPOINT_FILE,
#             index=False
#         )

#         print(
#             f"\nCheckpoint saved after Trial {trial}:"
#             f"\n  {numbered_checkpoint_file}"
#             f"\n  {PHASE1_LATEST_CHECKPOINT_FILE}"
#             f"\n  Rows saved: {len(checkpoint_df):,}"
#         )

# df_phase1_monte_carlo_raw = pd.DataFrame(phase1_raw_results)

# df_phase1_monte_carlo_raw.to_csv(PHASE1_RAW_FILE, index=False)

# experiment_elapsed = time.perf_counter() - experiment_start_time

# print(
#     f"\n{'=' * 72}\n"
#     f"PHASE 1 MONTE CARLO COMPLETE\n"
#     f"{'=' * 72}\n"
#     f"Trials completed       : "
#     f"{PHASE1_MONTE_CARLO_TRIALS}\n"
#     f"Configurations/trial   : "
#     f"{len(phase1_mc_distances) * len(phase1_mc_protocols)}\n"
#     f"Raw rows generated     : "
#     f"{len(df_phase1_monte_carlo_raw):,}\n"
#     f"Expected raw rows      : "
#     f"{PHASE1_MONTE_CARLO_TRIALS * len(phase1_mc_distances) * len(phase1_mc_protocols):,}\n"
#     f"Total runtime          : "
#     f"{experiment_elapsed / 3600:.2f} hours\n"
#     f"Final raw CSV          : "
#     f"{PHASE1_RAW_FILE}\n"
#     f"{'=' * 72}"
# )

# display(
#     df_phase1_monte_carlo_raw.head(18)
# )

In [ ]:
# @title Phase 1 Monte Carlo Aggregate Statistics

# import numpy as np
# import pandas as pd

# # Load from disk so this cell also works after a runtime restart.
# df_phase1_monte_carlo_raw = pd.read_csv(
#     "phase1_monte_carlo_raw_100_trials.csv"
# )

# successful_phase1_runs = (
#     df_phase1_monte_carlo_raw[df_phase1_monte_carlo_raw["Status"]== "SUCCESS"].copy()
# )

# phase1_group_columns = [
#     "Protocol",
#     "Distance (km)"
# ]

# phase1_metrics_to_aggregate = [
#     "Expected Photon Survival (%)",
#     "Observed Photon Survival (%)",
#     "Rounds Used",
#     "Accepted Rounds",
#     "Rejected Rounds",
#     "Photons Sent",
#     "Photons Received",
#     "Photons Lost",
#     "Sifted Bits",
#     "Effective Key Yield (%)",
#     "Keygen Time (s)",
#     "Configuration Runtime (s)"
# ]

# phase1_summary = (
#     successful_phase1_runs
#     .groupby(
#         phase1_group_columns,
#         as_index=False
#     )[phase1_metrics_to_aggregate]
#     .agg([
#         "count",
#         "mean",
#         "std",
#         "median",
#         "min",
#         "max"
#     ])
# )

# # Flatten Pandas multi-level column names.
# phase1_summary.columns = [
#     " ".join(
#         str(part)
#         for part in column
#         if str(part) != ""
#     ).strip()
#     if isinstance(column, tuple)
#     else column
#     for column in phase1_summary.columns
# ]

# # Restore readable group-column names after flattening.
# phase1_summary = phase1_summary.rename(
#     columns={
#         "Protocol": "Protocol",
#         "Distance (km)": "Distance (km)"
#     }
# )

# phase1_run_counts = (
#     df_phase1_monte_carlo_raw
#     .groupby(
#         phase1_group_columns
#     )
#     .agg(
#         Total_Trials=("Trial", "count"),
#         Successful_Trials=(
#             "Status",
#             lambda values: (
#                 values == "SUCCESS"
#             ).sum()
#         ),
#         Failed_Trials=(
#             "Status",
#             lambda values: (
#                 values == "FAILED"
#             ).sum()
#         )
#     )
#     .reset_index()
# )

# phase1_run_counts[
#     "Success Rate (%)"
# ] = (
#     phase1_run_counts["Successful_Trials"]
#     / phase1_run_counts["Total_Trials"]
#     * 100
# )

# df_phase1_monte_carlo_summary = (
#     phase1_run_counts.merge(
#         phase1_summary,
#         on=phase1_group_columns,
#         how="left"
#     )
# )

# # Sort consistently for the table and graphs.
# protocol_order = {
#     "Standard BB84 (50-50)": 0,
#     "Biased BB84 (80-20)": 1,
#     "T12-style (15/16 - 1/16)": 2
# }

# df_phase1_monte_carlo_summary[
#     "_Protocol Order"
# ] = (
#     df_phase1_monte_carlo_summary[
#         "Protocol"
#     ].map(protocol_order)
# )

# df_phase1_monte_carlo_summary = (
#     df_phase1_monte_carlo_summary
#     .sort_values(
#         by=[
#             "_Protocol Order",
#             "Distance (km)"
#         ]
#     )
#     .drop(
#         columns=["_Protocol Order"]
#     )
#     .reset_index(drop=True)
# )

# PHASE1_SUMMARY_FILE = (
#     "phase1_monte_carlo_summary_100_trials.csv"
# )

# df_phase1_monte_carlo_summary.to_csv(
#     PHASE1_SUMMARY_FILE,
#     index=False
# )

# print(
#     f"Phase 1 aggregate summary saved to "
#     f"{PHASE1_SUMMARY_FILE}"
# )

# print(
#     f"Raw rows       : "
#     f"{len(df_phase1_monte_carlo_raw):,}"
# )

# print(
#     f"Summary rows   : "
#     f"{len(df_phase1_monte_carlo_summary):,}"
# )

# print(
#     f"Failed runs    : "
#     f"{(df_phase1_monte_carlo_raw['Status'] == 'FAILED').sum():,}"
# )

# display(
#     df_phase1_monte_carlo_summary
# )

In [ ]:
# @title Phase 1 Compact Paper-Ready Summary

# phase1_paper_columns = [
#     "Protocol",
#     "Distance (km)",
#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",

#     "Observed Photon Survival (%) mean",
#     "Observed Photon Survival (%) std",

#     "Photons Sent mean",
#     "Photons Sent std",

#     "Effective Key Yield (%) mean",
#     "Effective Key Yield (%) std",

#     "Keygen Time (s) mean",
#     "Keygen Time (s) std",

#     "Sifted Bits mean",
#     "Sifted Bits std"
# ]

# df_phase1_paper_summary = (
#     df_phase1_monte_carlo_summary[phase1_paper_columns].copy()
# )

# df_phase1_paper_summary.to_csv(
#     "phase1_monte_carlo_paper_summary_100_trials.csv",
#     index=False
# )

# display(df_phase1_paper_summary)

In [ ]:
# @title Phase 1 Monte Carlo Paper-Ready Graphs

# import matplotlib.pyplot as plt
# import pandas as pd

# summary_df = pd.read_csv(
#     "phase1_monte_carlo_summary_100_trials.csv"
# )

# phase1_plot_protocols = [
#     "Standard BB84 (50-50)",
#     "Biased BB84 (80-20)",
#     "T12-style (15/16 - 1/16)"
# ]


# # =========================================================
# # Graph 1: Distance vs mean key-generation time

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Keygen Time (s) mean"],
#         yerr=subset["Keygen Time (s) std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Key Generation Time (s)")
# plt.title(
#     "Distance vs Key Generation Time "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Key Generation Time.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 2: Distance vs mean effective key yield

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Effective Key Yield (%) mean"],
#         yerr=subset["Effective Key Yield (%) std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Effective Key Yield (%)")
# plt.title(
#     "Distance vs Effective Key Yield "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Effective Key Yield.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 3: Distance vs observed photon survival

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset[
#             "Observed Photon Survival (%) mean"
#         ],
#         yerr=subset[
#             "Observed Photon Survival (%) std"
#         ],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Observed Photon Survival (%)")
# plt.title(
#     "Distance vs Observed Photon Survival "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Observed Photon Survival.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 4: Distance vs photons required

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Photons Sent mean"],
#         yerr=subset["Photons Sent std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Photons Sent")
# plt.title(
#     "Distance vs Photon Requirement for a "
#     "128-bit Key (100 Trials)"
# )
# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Photons Required.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 5: Distance vs sifted bits

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Sifted Bits mean"],
#         yerr=subset["Sifted Bits std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Mean Sifted Bits")
# plt.title(
#     "Distance vs Sifted Key Bits "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Sifted Bits.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 6: Success rate

# plt.figure(figsize=(10, 6))

# for protocol_name in phase1_plot_protocols:

#     subset = summary_df[
#         summary_df["Protocol"] == protocol_name
#     ].sort_values("Distance (km)")

#     plt.plot(
#         subset["Distance (km)"],
#         subset["Success Rate (%)"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Distance (km)")
# plt.ylabel("Successful Sessions (%)")
# plt.title(
#     "Distance vs Session Success Rate "
#     "(100 Trials)"
# )
# plt.ylim(0, 105)
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 1 MC Distance vs Session Success Rate.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

## Phase 2 : 07-07-2026T16:50:12 -- 09-07-2026T02:17:32

Goal : Add a single repeater so that Alice <--> Bob chain becomes ===> Alice <--> R1 <--> Bob


Objective : Measure whether splitting distance improves stability and key availability.

Question : Can a single repeater improve the effective QKD range and reliability compared to direct transmission?

### Phase 2A

Single Quantum Repeater Abstraction

In [ ]:
# @title Single Repeater Node

r1 = Node(name="R1", role=ROLE_REPEATER)

print(r1)

In [ ]:
# @title Bell Pair Abstraction

class BellPair:
    """
    Represents an entangled Bell pair between two nodes.

    In Phase 2A, this is a network-level abstraction.
    It does not simulate the full photonic hardware circuit.
    """

    def __init__(
        self,
        node_a: Node,
        node_b: Node,
        fidelity: float = 1.0,
        source_link_distance_km: float = None
    ):
        self.node_a = node_a
        self.node_b = node_b
        self.fidelity = fidelity
        self.source_link_distance_km = source_link_distance_km

    def involves(self, node: Node) -> bool:
        return self.node_a == node or self.node_b == node

    def other_node(self, node: Node) -> Node:
        if self.node_a == node:
            return self.node_b
        if self.node_b == node:
            return self.node_a

        raise ValueError("Node is not part of this Bell pair.")

    def __str__(self):
        return (
            f"BellPair({self.node_a.name} ↔ {self.node_b.name}, "
            f"F={self.fidelity:.4f})"
        )

In [ ]:
# @title Bell Pair Generation Attempt

def attempt_bell_pair_generation(
    link: QuantumLink,
    source_fidelity: float = 1.0
):
    """
    Attempt to generate a Bell pair across one elementary quantum link.

    Phase 2A abstraction:
    - Success probability is approximated using link photon survival probability.
    - Successful attempt returns a BellPair object.
    - Failed attempt returns None.
    """

    if random.random() < link.survival_probability:
        return BellPair(
            node_a=link.source,
            node_b=link.destination,
            fidelity=source_fidelity,
            source_link_distance_km=link.distance_km
        )

    return None

In [ ]:
# @title Entanglement Swapping

def entanglement_swap(
    left_pair: BellPair,
    right_pair: BellPair,
    repeater: Node,
    swapping_success_probability: float = 1.0
):
    """
    Perform entanglement swapping at the repeater.

    Example:
        left_pair  = BellPair(Alice ↔ R1)
        right_pair = BellPair(R1 ↔ Bob)

    Output:
        BellPair(Alice ↔ Bob)

    The repeater does not learn the key/message.
    It only helps convert two elementary entangled links
    into one end-to-end entangled link.
    """

    if left_pair is None or right_pair is None:
        return None

    if not left_pair.involves(repeater):
        raise ValueError("Left Bell pair does not involve the repeater.")

    if not right_pair.involves(repeater):
        raise ValueError("Right Bell pair does not involve the repeater.")

    if random.random() > swapping_success_probability:
        return None

    end_a = left_pair.other_node(repeater)
    end_b = right_pair.other_node(repeater)

    combined_fidelity = left_pair.fidelity * right_pair.fidelity

    return BellPair(
        node_a=end_a,
        node_b=end_b,
        fidelity=combined_fidelity,
        source_link_distance_km=(left_pair.source_link_distance_km or 0) + (right_pair.source_link_distance_km or 0)
    )

In [ ]:
# @title Single Repeater Swapping Attempt Without Memory

def attempt_single_repeater_connection(
    link_left: QuantumLink,
    link_right: QuantumLink,
    repeater: Node,
    swapping_success_probability: float = 1.0
) -> dict:

    left_pair = attempt_bell_pair_generation(link_left)
    right_pair = attempt_bell_pair_generation(link_right)

    end_to_end_pair = None

    if left_pair is not None and right_pair is not None:
        end_to_end_pair = entanglement_swap(
            left_pair=left_pair,
            right_pair=right_pair,
            repeater=repeater,
            swapping_success_probability=swapping_success_probability
        )

    return {
        "left_pair": left_pair,
        "right_pair": right_pair,
        "end_to_end_pair": end_to_end_pair,
        "left_success": left_pair is not None,
        "right_success": right_pair is not None,
        "swap_success": end_to_end_pair is not None,
        "end_to_end_success": end_to_end_pair is not None,
        "left_survival_probability": link_left.survival_probability,
        "right_survival_probability": link_right.survival_probability,
        "swapping_success_probability": swapping_success_probability
    }

In [ ]:
# @title Direct vs Single Repeater Theoretical Comparison

def compare_direct_vs_single_repeater_theory(
    source: Node,
    repeater: Node,
    destination: Node,
    total_distance_km: float,
    swapping_success_probability: float = 1.0
) -> dict:

    direct_link = QuantumLink(source, destination, total_distance_km)

    segment_distance = total_distance_km / 2

    left_link = QuantumLink(source, repeater, segment_distance)
    right_link = QuantumLink(repeater, destination, segment_distance)

    direct_success_probability = direct_link.survival_probability

    no_memory_repeater_success_probability = left_link.survival_probability * right_link.survival_probability * swapping_success_probability

    direct_expected_attempts = 1 / direct_success_probability if direct_success_probability > 0 else float("inf")

    no_memory_repeater_expected_attempts = 1 / no_memory_repeater_success_probability if no_memory_repeater_success_probability > 0 else float("inf")

    return {
        "total_distance_km": total_distance_km,
        "direct_distance_km": total_distance_km,
        "segment_distance_km": segment_distance,
        "direct_success_probability": direct_success_probability,
        "left_segment_success_probability": left_link.survival_probability,
        "right_segment_success_probability": right_link.survival_probability,
        "swapping_success_probability": swapping_success_probability,
        "no_memory_repeater_success_probability": no_memory_repeater_success_probability,
        "direct_expected_attempts": direct_expected_attempts,
        "no_memory_repeater_expected_attempts": no_memory_repeater_expected_attempts
    }

In [ ]:
# @title Memory-Assisted Single Repeater Model With Bell Pairs

def generate_entanglement_via_single_repeater_with_memory(
    source: Node,
    repeater: Node,
    destination: Node,
    total_distance_km: float,
    swapping_success_probability: float = 1.0,
    memory_lifetime_attempts: int = 10000,
    max_attempts: int = 1000000,
    verbose: bool = False
) -> dict:
    """
    Memory-assisted single quantum repeater model.

    Topology:
        source ---- repeater ---- destination

    Process:
    1. Try generating BellPair(source ↔ repeater).
    2. Try generating BellPair(repeater ↔ destination).
    3. Store successful Bell pairs in quantum memory.
    4. Retry only failed side.
    5. Once both Bell pairs exist, perform entanglement swapping.
    6. Output BellPair(source ↔ destination).
    """

    segment_distance = total_distance_km / 2

    link_left = QuantumLink(source, repeater, segment_distance)
    link_right = QuantumLink(repeater, destination, segment_distance)
    direct_link = QuantumLink(source, destination, total_distance_km)

    left_pair = None
    right_pair = None

    left_memory_age = 0
    right_memory_age = 0

    left_attempts = 0
    right_attempts = 0

    swap_attempts = 0

    left_memory_expirations = 0
    right_memory_expirations = 0

    start_time = time.perf_counter()

    for global_attempt in range(1, max_attempts + 1):

        if left_pair is not None:
            left_memory_age += 1

            if left_memory_age > memory_lifetime_attempts:
                left_pair = None
                left_memory_age = 0
                left_memory_expirations += 1

        if right_pair is not None:
            right_memory_age += 1

            if right_memory_age > memory_lifetime_attempts:
                right_pair = None
                right_memory_age = 0
                right_memory_expirations += 1

        if left_pair is None:
            left_attempts += 1
            generated_left = attempt_bell_pair_generation(link_left)

            if generated_left is not None:
                left_pair = generated_left
                left_memory_age = 0

        if right_pair is None:
            right_attempts += 1
            generated_right = attempt_bell_pair_generation(link_right)

            if generated_right is not None:
                right_pair = generated_right
                right_memory_age = 0

        if left_pair is not None and right_pair is not None:
            swap_attempts += 1
            end_to_end_pair = entanglement_swap(
                left_pair=left_pair,
                right_pair=right_pair,
                repeater=repeater,
                swapping_success_probability=swapping_success_probability
            )

            if end_to_end_pair is not None:
                elapsed = time.perf_counter() - start_time

                return {
                    "source": source.name,
                    "repeater": repeater.name,
                    "destination": destination.name,
                    "total_distance_km": total_distance_km,
                    "segment_distance_km": segment_distance,
                    "direct_survival_probability": direct_link.survival_probability,
                    "segment_survival_probability": link_left.survival_probability,
                    "swapping_success_probability": swapping_success_probability,
                    "memory_lifetime_attempts": memory_lifetime_attempts,
                    "global_attempts_required": global_attempt,
                    "left_attempts": left_attempts,
                    "right_attempts": right_attempts,
                    "swap_attempts": swap_attempts,
                    "left_memory_expirations": left_memory_expirations,
                    "right_memory_expirations": right_memory_expirations,
                    "final_bell_pair": str(end_to_end_pair),
                    "final_fidelity": end_to_end_pair.fidelity,
                    "success": True,
                    "time_s": elapsed
                }

            left_pair = None
            right_pair = None
            left_memory_age = 0
            right_memory_age = 0

        if verbose and global_attempt <= 20:
            print(
                f"Attempt {global_attempt}: "
                f"L_pair={left_pair is not None}, "
                f"R_pair={right_pair is not None}, "
                f"L_age={left_memory_age}, "
                f"R_age={right_memory_age}"
            )

    elapsed = time.perf_counter() - start_time

    return {
        "source": source.name,
        "repeater": repeater.name,
        "destination": destination.name,
        "total_distance_km": total_distance_km,
        "segment_distance_km": segment_distance,
        "direct_survival_probability": direct_link.survival_probability,
        "segment_survival_probability": link_left.survival_probability,
        "swapping_success_probability": swapping_success_probability,
        "memory_lifetime_attempts": memory_lifetime_attempts,
        "global_attempts_required": max_attempts,
        "left_attempts": left_attempts,
        "right_attempts": right_attempts,
        "swap_attempts": swap_attempts,
        "left_memory_expirations": left_memory_expirations,
        "right_memory_expirations": right_memory_expirations,
        "final_bell_pair": None,
        "final_fidelity": None,
        "success": False,
        "time_s": elapsed
    }

In [ ]:
# @title Single Repeater Test

result = generate_entanglement_via_single_repeater_with_memory(
    source=alice,
    repeater=r1,
    destination=bob,
    total_distance_km=100,
    swapping_success_probability=1.0,
    memory_lifetime_attempts=10000,
    max_attempts=1000000,
    verbose=True
)

print("\n")
print("=" * 70)
print("Phase 2A Single Repeater Result")
print("=" * 70)

for k, v in result.items():
    if isinstance(v, float):
        print(f"{k:<35}: {v:.6f}")
    else:
        print(f"{k:<35}: {v}")

print("=" * 70)

In [ ]:
# @title Distance Sweep Direct vs Single Repeater

# phase2_distances = [50, 80, 100, 150, 200, 250, 300]

# phase2_results = []

# for d in phase2_distances:

#     direct_link = QuantumLink(alice, bob, d)
#     repeater_result = generate_entanglement_via_single_repeater_with_memory(
#         source=alice,
#         repeater=r1,
#         destination=bob,
#         total_distance_km=d,
#         swapping_success_probability=1.0,
#         memory_lifetime_attempts=10000,
#         max_attempts=1000000,
#         verbose=False
#     )

#     phase2_results.append({
#         "Distance (km)": d,
#         "Direct Survival Probability (%)": direct_link.survival_probability * 100,
#         "Direct Expected Attempts": 1 / direct_link.survival_probability,
#         "Segment Distance (km)": d / 2,
#         "Segment Survival Probability (%)": repeater_result["segment_survival_probability"] * 100,
#         "Repeater Global Attempts": repeater_result["global_attempts_required"],
#         "Left Link Attempts": repeater_result["left_attempts"],
#         "Right Link Attempts": repeater_result["right_attempts"],
#         "Swap Attempts": repeater_result["swap_attempts"],
#         "Final Bell Pair": repeater_result["final_bell_pair"],
#         "Final Fidelity": repeater_result["final_fidelity"],
#         "Memory Lifetime": repeater_result["memory_lifetime_attempts"],
#         "Left Memory Expirations": repeater_result["left_memory_expirations"],
#         "Right Memory Expirations": repeater_result["right_memory_expirations"],
#         "Repeater Success": repeater_result["success"],
#         "Improvement Factor": (
#             (1 / direct_link.survival_probability)
#             / repeater_result["global_attempts_required"]
#             if repeater_result["global_attempts_required"] > 0
#             else None
#         )
#     })

# df_phase2_single_repeater = pd.DataFrame(phase2_results)

# display(df_phase2_single_repeater)

# df_phase2_single_repeater.to_csv(
#     "phase2_single_repeater_results.csv",
#     index=False
# )

In [ ]:
# @title Plot : Repeater Improvement Factor

# plt.figure(figsize=(10, 6))

# plt.plot(
#     df_phase2_single_repeater["Distance (km)"],
#     df_phase2_single_repeater["Improvement Factor"],
#     marker="o"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")
# plt.title("Single Repeater Improvement Over Direct Transmission")
# plt.grid(True)

# plt.savefig(
#     "Phase 2 Single Repeater Improvement Over Direct Transmission.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Direct Expected Attempts vs Repeater Attempts

# plt.figure(figsize=(10, 6))

# plt.plot(
#     df_phase2_single_repeater["Distance (km)"],
#     df_phase2_single_repeater["Direct Expected Attempts"],
#     marker="o",
#     label="Direct Expected Attempts"
# )

# plt.plot(
#     df_phase2_single_repeater["Distance (km)"],
#     df_phase2_single_repeater["Repeater Global Attempts"],
#     marker="s",
#     label="Single Repeater Attempts"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Attempts")
# plt.title("Direct Transmission vs Single Repeater Attempts")
# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2 Direct vs Repeater Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

### Phase 2A --- Monte Carlo Trials

100 Trial runs for Phase 2A

In [ ]:
# @title Monte Carlo Experiment

# import gc
# import math
# import os
# import time
# import pandas as pd

# PHASE2A_MONTE_CARLO_TRIALS = 100
# PHASE2A_CHECKPOINT_INTERVAL = 50

# phase2a_mc_distances = [50, 80, 100, 150, 200, 250, 300]

# PHASE2A_SWAP_SUCCESS_PROBABILITY = 1.0
# PHASE2A_MEMORY_LIFETIME_ATTEMPTS = 10000
# PHASE2A_MAX_ATTEMPTS = 1000000

# PHASE2A_RAW_FILE = ("phase2a_monte_carlo_raw_100_trials.csv")

# PHASE2A_LATEST_CHECKPOINT_FILE = ("phase2a_monte_carlo_latest_checkpoint.csv")

# # Set to True only when resuming after a runtime crash.
# RESUME_PHASE2A_FROM_CHECKPOINT = False

# # =========================================================
# # Theoretical Ideal-Memory Expectation
# # =========================================================

# def expected_attempts_with_ideal_memory(
#     segment_success_probability: float
# ) -> float:
#     """
#     Expected number of parallel global attempts required
#     for two independent geometric elementary-link processes.

#     Assumptions:
#     - Equal segment success probability p
#     - Both elementary links attempted in parallel
#     - A successful Bell pair is stored indefinitely
#     - Perfect entanglement swapping
#     """

#     p = segment_success_probability

#     if p <= 0:
#         return float("inf")

#     return ((2 / p) - (1 / (p * (2 - p))))


# # =========================================================
# # Resume Support
# # =========================================================

# phase2a_raw_results = []
# phase2a_starting_trial = 1

# if (RESUME_PHASE2A_FROM_CHECKPOINT and os.path.exists(PHASE2A_LATEST_CHECKPOINT_FILE)):
#     existing_checkpoint = pd.read_csv(PHASE2A_LATEST_CHECKPOINT_FILE)

#     phase2a_raw_results = existing_checkpoint.to_dict("records")

#     if not existing_checkpoint.empty:
#         last_completed_trial = int(existing_checkpoint["Trial"].max())

#         phase2a_starting_trial = last_completed_trial + 1

#         print("Resuming Phase 2A Monte Carlo "f"from Trial {phase2a_starting_trial}.")

# else:
#     print("Starting a new Phase 2A Monte Carlo experiment.")


# # =========================================================
# # Monte Carlo Execution
# # =========================================================

# phase2a_experiment_start = time.perf_counter()

# for trial in range(phase2a_starting_trial, PHASE2A_MONTE_CARLO_TRIALS + 1):

#     trial_start = time.perf_counter()

#     print(
#         f"\n{'=' * 78}\n"
#         f"Phase 2A Monte Carlo — Trial {trial}/{PHASE2A_MONTE_CARLO_TRIALS}\n"
#         f"{'=' * 78}"
#     )

#     for distance_km in phase2a_mc_distances:

#         configuration_start = time.perf_counter()

#         direct_link = None
#         repeater_result = None

#         try:
#             direct_link = QuantumLink(source=alice, destination=bob, distance_km=distance_km)

#             segment_distance_km = (distance_km / 2)

#             segment_link = QuantumLink(source=alice, destination=r1, distance_km=segment_distance_km)

#             direct_success_probability = (direct_link.survival_probability)

#             segment_success_probability = (segment_link.survival_probability)

#             direct_expected_attempts = 1 / direct_success_probability if direct_success_probability > 0 else float("inf")

#             no_memory_success_probability = segment_success_probability * segment_success_probability * PHASE2A_SWAP_SUCCESS_PROBABILITY

#             no_memory_expected_attempts = 1 / no_memory_success_probability if no_memory_success_probability > 0 else float("inf")

#             ideal_memory_expected_attempts = expected_attempts_with_ideal_memory(segment_success_probability)

#             repeater_result = (
#                 generate_entanglement_via_single_repeater_with_memory(
#                     source=alice,
#                     repeater=r1,
#                     destination=bob,
#                     total_distance_km=distance_km,
#                     swapping_success_probability=PHASE2A_SWAP_SUCCESS_PROBABILITY,
#                     memory_lifetime_attempts=PHASE2A_MEMORY_LIFETIME_ATTEMPTS,
#                     max_attempts=PHASE2A_MAX_ATTEMPTS,
#                     verbose=False
#                 )
#             )

#             configuration_runtime = time.perf_counter() - configuration_start

#             observed_global_attempts = repeater_result["global_attempts_required"]

#             observed_improvement_factor = direct_expected_attempts / observed_global_attempts if observed_global_attempts > 0 else None

#             theoretical_improvement_factor = direct_expected_attempts / ideal_memory_expected_attempts if ideal_memory_expected_attempts > 0 else None

#             phase2a_raw_results.append({
#                 "Trial":trial,
#                 "Distance (km)":distance_km,
#                 "Segment Distance (km)":segment_distance_km,
#                 "Direct Survival Probability (%)":direct_success_probability * 100,
#                 "Segment Survival Probability (%)":segment_success_probability * 100,
#                 "Swap Success Probability":PHASE2A_SWAP_SUCCESS_PROBABILITY,
#                 "Memory Lifetime (Attempts)":PHASE2A_MEMORY_LIFETIME_ATTEMPTS,
#                 "Maximum Allowed Attempts":PHASE2A_MAX_ATTEMPTS,

#                 # Theoretical baseline metrics
#                 "Direct Expected Attempts":direct_expected_attempts,
#                 "No-Memory Repeater Expected Attempts":no_memory_expected_attempts,
#                 "Ideal-Memory Repeater Expected Attempts":ideal_memory_expected_attempts,
#                 "Theoretical Improvement Factor":theoretical_improvement_factor,

#                 # Observed repeater metrics
#                 "Repeater Global Attempts":observed_global_attempts,
#                 "Left Link Attempts": repeater_result["left_attempts"],
#                 "Right Link Attempts": repeater_result["right_attempts"],
#                 "Total Elementary-Link Attempts": repeater_result["left_attempts"] + repeater_result["right_attempts"],

#                 "Swap Attempts": repeater_result["swap_attempts"],
#                 "Left Memory Expirations": repeater_result["left_memory_expirations"],
#                 "Right Memory Expirations": repeater_result["right_memory_expirations"],
#                 "Total Memory Expirations": repeater_result["left_memory_expirations"] + repeater_result["right_memory_expirations"],

#                 "Observed Improvement Factor":observed_improvement_factor,
#                 "Final Bell Pair": repeater_result["final_bell_pair"],
#                 "Final Fidelity": repeater_result["final_fidelity"],
#                 "Simulation Function Time (s)": repeater_result["time_s"],
#                 "Configuration Runtime (s)":configuration_runtime,
#                 "Status": "SUCCESS" if repeater_result["success"] else "FAILED",

#                 "Error": None
#             })

#             print(
#                 f"Trial {trial:3d} | "
#                 f"{distance_km:3d} km | "
#                 f"direct E[T]="
#                 f"{direct_expected_attempts:,.2f} | "
#                 f"ideal-memory E[T]="
#                 f"{ideal_memory_expected_attempts:,.2f} | "
#                 f"observed={observed_global_attempts:,} | "
#                 f"improvement="
#                 f"{observed_improvement_factor:,.2f}×"
#             )

#         except Exception as error:

#             configuration_runtime = time.perf_counter() - configuration_start
#             segment_distance_km = distance_km / 2
#             phase2a_raw_results.append({
#                 "Trial": trial,
#                 "Distance (km)": distance_km,
#                 "Segment Distance (km)": segment_distance_km,
#                 "Direct Survival Probability (%)": direct_link.survival_probability * 100 if direct_link is not None else None,
#                 "Segment Survival Probability (%)": None,
#                 "Swap Success Probability": PHASE2A_SWAP_SUCCESS_PROBABILITY,
#                 "Memory Lifetime (Attempts)": PHASE2A_MEMORY_LIFETIME_ATTEMPTS,
#                 "Maximum Allowed Attempts": PHASE2A_MAX_ATTEMPTS,
#                 "Direct Expected Attempts": None,
#                 "No-Memory Repeater Expected Attempts": None,
#                 "Ideal-Memory Repeater Expected Attempts": None,
#                 "Theoretical Improvement Factor": None,
#                 "Repeater Global Attempts": None,
#                 "Left Link Attempts": None,
#                 "Right Link Attempts": None,
#                 "Total Elementary-Link Attempts": None,
#                 "Swap Attempts": None,
#                 "Left Memory Expirations": None,
#                 "Right Memory Expirations": None,
#                 "Total Memory Expirations": None,
#                 "Observed Improvement Factor": None,
#                 "Final Bell Pair": None,
#                 "Final Fidelity": None,
#                 "Simulation Function Time (s)": None,
#                 "Configuration Runtime (s)": configuration_runtime,
#                 "Status": "FAILED",
#                 "Error": str(error)
#             })

#             print(
#                 f"Trial {trial:3d} | "
#                 f"{distance_km:3d} km | "
#                 f"FAILED: {error}"
#             )

#         del direct_link
#         del repeater_result

#         if "segment_link" in locals():
#             del segment_link

#         gc.collect()

#     trial_runtime = (
#         time.perf_counter()
#         - trial_start
#     )

#     print(
#         f"\nTrial {trial} completed in "
#         f"{trial_runtime:.3f} seconds."
#     )

#     # =====================================================
#     # Checkpoint Every 50 Fully Completed Trials
#     # =====================================================

#     if (trial % PHASE2A_CHECKPOINT_INTERVAL == 0 or trial == PHASE2A_MONTE_CARLO_TRIALS):
#         checkpoint_df = pd.DataFrame(phase2a_raw_results)

#         numbered_checkpoint_file = (f"phase2a_monte_carlo_checkpoint_trial_{trial}.csv")

#         checkpoint_df.to_csv(numbered_checkpoint_file, index=False)

#         checkpoint_df.to_csv(PHASE2A_LATEST_CHECKPOINT_FILE, index=False)

#         print(
#             f"\nCheckpoint saved after Trial {trial}:"
#             f"\n  {numbered_checkpoint_file}"
#             f"\n  {PHASE2A_LATEST_CHECKPOINT_FILE}"
#             f"\n  Rows saved: {len(checkpoint_df):,}"
#         )


# # =========================================================
# # Save Final Raw Dataset
# # =========================================================

# df_phase2a_monte_carlo_raw = pd.DataFrame(phase2a_raw_results)

# df_phase2a_monte_carlo_raw.to_csv(PHASE2A_RAW_FILE, index=False)

# phase2a_total_runtime = time.perf_counter() - phase2a_experiment_start

# expected_phase2a_rows = (PHASE2A_MONTE_CARLO_TRIALS * len(phase2a_mc_distances))

# print(
#     f"\n{'=' * 78}\n"
#     f"PHASE 2A MONTE CARLO COMPLETE\n"
#     f"{'=' * 78}\n"
#     f"Trials completed       : "
#     f"{PHASE2A_MONTE_CARLO_TRIALS}\n"
#     f"Distances per trial    : "
#     f"{len(phase2a_mc_distances)}\n"
#     f"Raw rows generated     : "
#     f"{len(df_phase2a_monte_carlo_raw):,}\n"
#     f"Expected raw rows      : "
#     f"{expected_phase2a_rows:,}\n"
#     f"Total runtime          : "
#     f"{phase2a_total_runtime / 60:.2f} minutes\n"
#     f"Final raw CSV          : "
#     f"{PHASE2A_RAW_FILE}\n"
#     f"{'=' * 78}"
# )

# display(
#     df_phase2a_monte_carlo_raw.head(14)
# )

In [ ]:
# @title Aggregate Statistics

# import pandas as pd

# df_phase2a_monte_carlo_raw = pd.read_csv("phase2a_monte_carlo_raw_100_trials.csv")

# successful_phase2a_runs = df_phase2a_monte_carlo_raw[df_phase2a_monte_carlo_raw["Status"] == "SUCCESS"].copy()

# phase2a_group_columns = "Distance (km)"

# phase2a_metrics_to_aggregate = [
#     "Segment Distance (km)",
#     "Direct Survival Probability (%)",
#     "Segment Survival Probability (%)",

#     "Direct Expected Attempts",
#     "No-Memory Repeater Expected Attempts",
#     "Ideal-Memory Repeater Expected Attempts",
#     "Theoretical Improvement Factor",

#     "Repeater Global Attempts",
#     "Left Link Attempts",
#     "Right Link Attempts",
#     "Total Elementary-Link Attempts",
#     "Swap Attempts",

#     "Left Memory Expirations",
#     "Right Memory Expirations",
#     "Total Memory Expirations",

#     "Observed Improvement Factor",
#     "Final Fidelity",

#     "Simulation Function Time (s)",
#     "Configuration Runtime (s)"
# ]

# phase2a_summary_statistics = (
#     successful_phase2a_runs
#     .groupby(
#         phase2a_group_columns,
#         as_index=False
#     )[phase2a_metrics_to_aggregate]
#     .agg([
#         "count",
#         "mean",
#         "std",
#         "median",
#         "min",
#         "max"
#     ])
# )

# # Flatten Pandas MultiIndex columns.
# phase2a_summary_statistics.columns = [
#     " ".join(
#         str(part)
#         for part in column
#         if str(part) != ""
#     ).strip()
#     if isinstance(column, tuple)
#     else column
#     for column in phase2a_summary_statistics.columns
# ]


# # =========================================================
# # Trial Counts and Success Rate
# # =========================================================

# phase2a_run_counts = (
#     df_phase2a_monte_carlo_raw
#     .groupby(
#         phase2a_group_columns
#     )
#     .agg(
#         Total_Trials=(
#             "Trial",
#             "count"
#         ),

#         Successful_Trials=(
#             "Status",
#             lambda values: (
#                 values == "SUCCESS"
#             ).sum()
#         ),

#         Failed_Trials=(
#             "Status",
#             lambda values: (
#                 values == "FAILED"
#             ).sum()
#         )
#     )
#     .reset_index()
# )

# phase2a_run_counts["Success Rate (%)"] = phase2a_run_counts["Successful_Trials"] / phase2a_run_counts["Total_Trials"] * 100

# df_phase2a_monte_carlo_summary = (
#     phase2a_run_counts.merge(
#         phase2a_summary_statistics,
#         on=phase2a_group_columns,
#         how="left"
#     )
# )

# df_phase2a_monte_carlo_summary = df_phase2a_monte_carlo_summary.sort_values("Distance (km)").reset_index(drop=True)

# PHASE2A_SUMMARY_FILE = "phase2a_monte_carlo_summary_100_trials.csv"

# df_phase2a_monte_carlo_summary.to_csv(PHASE2A_SUMMARY_FILE, index=False)

# print(f"Phase 2A aggregate summary saved to {PHASE2A_SUMMARY_FILE}")

# print(f"Raw rows       : {len(df_phase2a_monte_carlo_raw):,}")

# print(f"Summary rows   : {len(df_phase2a_monte_carlo_summary):,}")

# print(f"Failed runs    : {(df_phase2a_monte_carlo_raw['Status']== 'FAILED').sum():,}")

# display(df_phase2a_monte_carlo_summary)

In [ ]:
# @title Add Aggregate Improvement and Variability Metrics

# import numpy as np
# import pandas as pd

# PHASE2A_RAW_FILE = "phase2a_monte_carlo_raw_100_trials.csv"

# PHASE2A_SUMMARY_FILE = "phase2a_monte_carlo_summary_100_trials.csv"

# PHASE2A_AUGMENTED_SUMMARY_FILE = "phase2a_monte_carlo_summary_100_trials.csv"

# df_phase2a_raw = pd.read_csv(PHASE2A_RAW_FILE)

# df_phase2a_summary = pd.read_csv(PHASE2A_SUMMARY_FILE)

# df_phase2a_successful = df_phase2a_raw[df_phase2a_raw["Status"] == "SUCCESS"].copy()

# phase2a_additional_metrics = (
#     df_phase2a_successful
#     .groupby(
#         "Distance (km)",
#         as_index=False
#     )
#     .agg(
#         Direct_Expected_Attempts_Aggregate=("Direct Expected Attempts", "mean"),
#         Repeater_Global_Attempts_Aggregate=("Repeater Global Attempts", "mean"),
#         Repeater_Global_Attempts_SD=("Repeater Global Attempts", "std"),
#         Repeater_Global_Attempts_Median=("Repeater Global Attempts", "median")
#     )
# )

# phase2a_additional_metrics["Aggregate Improvement Factor"] = phase2a_additional_metrics["Direct_Expected_Attempts_Aggregate"] / phase2a_additional_metrics["Repeater_Global_Attempts_Aggregate"]

# phase2a_additional_metrics["Repeater Attempts Coefficient of Variation"] = phase2a_additional_metrics["Repeater_Global_Attempts_SD"] / phase2a_additional_metrics["Repeater_Global_Attempts_Aggregate"]

# phase2a_additional_metrics["Repeater Attempts Coefficient of Variation (%)"] = phase2a_additional_metrics["Repeater Attempts Coefficient of Variation"] * 100

# if ("Theoretical Improvement Factor mean" in df_phase2a_summary.columns):

#     theoretical_lookup = df_phase2a_summary[["Distance (km)", "Theoretical Improvement Factor mean"]].copy()

#     phase2a_additional_metrics = phase2a_additional_metrics.merge(theoretical_lookup, on="Distance (km)", how="left")

#     phase2a_additional_metrics["Aggregate vs Theory Difference"] = phase2a_additional_metrics["Aggregate Improvement Factor"] - phase2a_additional_metrics["Theoretical Improvement Factor mean"]

#     phase2a_additional_metrics["Aggregate vs Theory Error (%)"] = phase2a_additional_metrics["Aggregate vs Theory Difference"] / phase2a_additional_metrics["Theoretical Improvement Factor mean"] * 100

# columns_to_merge = [
#     "Distance (km)",
#     "Aggregate Improvement Factor",
#     "Repeater Attempts Coefficient of Variation",
#     "Repeater Attempts Coefficient of Variation (%)"
# ]

# if ("Aggregate vs Theory Difference" in phase2a_additional_metrics.columns):
#     columns_to_merge.extend([
#         "Aggregate vs Theory Difference",
#         "Aggregate vs Theory Error (%)"
#     ])

# df_phase2a_summary_augmented = (
#     df_phase2a_summary
#     .merge(
#         phase2a_additional_metrics[
#             columns_to_merge
#         ],
#         on="Distance (km)",
#         how="left"
#     )
#     .sort_values("Distance (km)")
#     .reset_index(drop=True)
# )


# # =========================================================
# # Save Augmented Full Summary
# # =========================================================

# df_phase2a_summary_augmented.to_csv(PHASE2A_AUGMENTED_SUMMARY_FILE, index=False)

# print(f"Augmented Phase 2A summary saved to : {PHASE2A_AUGMENTED_SUMMARY_FILE}")

# display(df_phase2a_summary_augmented)

In [ ]:
# @title Compact Paper-Ready Summary

# phase2a_paper_columns = [
#     "Distance (km)",

#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",

#     "Direct Survival Probability (%) mean",
#     "Segment Survival Probability (%) mean",

#     "Direct Expected Attempts mean",

#     "Ideal-Memory Repeater Expected Attempts mean",

#     "Repeater Global Attempts mean",
#     "Repeater Global Attempts std",
#     "Repeater Global Attempts median",
#     "Repeater Global Attempts min",
#     "Repeater Global Attempts max",

#     "Left Link Attempts mean",
#     "Left Link Attempts std",

#     "Right Link Attempts mean",
#     "Right Link Attempts std",

#     "Total Elementary-Link Attempts mean",
#     "Total Elementary-Link Attempts std",

#     "Observed Improvement Factor mean",
#     "Observed Improvement Factor std",

#     "Theoretical Improvement Factor mean",

#     "Total Memory Expirations mean",
#     "Total Memory Expirations std",

#     "Configuration Runtime (s) mean",
#     "Configuration Runtime (s) std"
# ]

# df_phase2a_paper_summary = df_phase2a_monte_carlo_summary[phase2a_paper_columns].copy()

# PHASE2A_PAPER_SUMMARY_FILE = "phase2a_monte_carlo_paper_summary_100_trials.csv"

# df_phase2a_paper_summary.to_csv(PHASE2A_PAPER_SUMMARY_FILE, index=False)

# print(f"Phase 2A paper-ready summary saved to {PHASE2A_PAPER_SUMMARY_FILE}")

# display(df_phase2a_paper_summary)

In [ ]:
# @title Create Augmented Paper-Ready Summary

# import pandas as pd

# PHASE2A_AUGMENTED_SUMMARY_FILE = ("phase2a_monte_carlo_summary_100_trials.csv")

# PHASE2A_AUGMENTED_PAPER_FILE = ("phase2a_monte_carlo_paper_summary_100_trials.csv")

# df_phase2a_summary_augmented = pd.read_csv(PHASE2A_AUGMENTED_SUMMARY_FILE)

# phase2a_augmented_paper_columns = [
#     "Distance (km)",
#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",
#     "Direct Survival Probability (%) mean",
#     "Segment Survival Probability (%) mean",
#     "Direct Expected Attempts mean",
#     "Ideal-Memory Repeater Expected Attempts mean",
#     "Repeater Global Attempts mean",
#     "Repeater Global Attempts std",
#     "Repeater Global Attempts median",
#     "Repeater Global Attempts min",
#     "Repeater Global Attempts max",
#     "Observed Improvement Factor mean",
#     "Observed Improvement Factor std",
#     "Aggregate Improvement Factor",
#     "Theoretical Improvement Factor mean",
#     "Repeater Attempts Coefficient of Variation",
#     "Repeater Attempts Coefficient of Variation (%)",
#     "Left Link Attempts mean",
#     "Right Link Attempts mean",
#     "Total Elementary-Link Attempts mean",
#     "Total Memory Expirations mean",
#     "Configuration Runtime (s) mean"
# ]

# optional_columns = [
#     "Aggregate vs Theory Difference",
#     "Aggregate vs Theory Error (%)"
# ]

# for column in optional_columns:
#     if column in df_phase2a_summary_augmented.columns:
#         phase2a_augmented_paper_columns.append(
#             column
#         )

# available_paper_columns = [
#     column
#     for column
#     in phase2a_augmented_paper_columns
#     if column
#     in df_phase2a_summary_augmented.columns
# ]


# df_phase2a_paper_summary_augmented = df_phase2a_summary_augmented[available_paper_columns].copy()

# df_phase2a_paper_summary_augmented.to_csv(
#     PHASE2A_AUGMENTED_PAPER_FILE,
#     index=False
# )


# print(
#     f"Augmented Phase 2A paper summary saved to : {PHASE2A_AUGMENTED_PAPER_FILE}"
# )

# display(df_phase2a_paper_summary_augmented)

In [ ]:
# @title Monte Carlo Paper-Ready Graphs

# import matplotlib.pyplot as plt
# import pandas as pd


# phase2a_summary_df = pd.read_csv("phase2a_monte_carlo_summary_100_trials.csv")

# phase2a_summary_df = phase2a_summary_df.sort_values("Distance (km)")


# # =========================================================
# # Graph 1:
# # Direct Expected Attempts vs Observed Repeater Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Direct Expected Attempts mean"
#     ],
#     marker="o",
#     label="Direct Expected Attempts"
# )

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Repeater Global Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Repeater Global Attempts std"
#     ],
#     marker="s",
#     capsize=4,
#     label=(
#         "Single Repeater Observed Attempts "
#         "(Mean ± SD)"
#     )
# )

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Ideal-Memory Repeater "
#         "Expected Attempts mean"
#     ],
#     marker="^",
#     linestyle="--",
#     label=(
#         "Ideal-Memory Theoretical Expectation"
#     )
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Attempts")
# plt.title(
#     "Direct Transmission vs "
#     "Memory-Assisted Single Repeater "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Direct vs Repeater Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 2:
# # Distance vs Observed Improvement Factor
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Observed Improvement Factor mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Observed Improvement Factor std"
#     ],
#     marker="o",
#     capsize=4,
#     label=(
#         "Observed Improvement "
#         "(Mean ± SD)"
#     )
# )

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Theoretical Improvement Factor mean"
#     ],
#     marker="s",
#     linestyle="--",
#     label="Ideal-Memory Theoretical Improvement"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")
# plt.title(
#     "Single-Repeater Improvement "
#     "Over Direct Transmission "
#     "(100 Trials)"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Repeater Improvement Factor.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 3:
# # Left vs Right Elementary-Link Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Left Link Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Left Link Attempts std"
#     ],
#     marker="o",
#     capsize=4,
#     label="Left Elementary Link"
# )

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Right Link Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Right Link Attempts std"
#     ],
#     marker="s",
#     capsize=4,
#     label="Right Elementary Link"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Attempts")
# plt.title(
#     "Elementary-Link Attempt Balance "
#     "(Mean ± Standard Deviation, 100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Left vs Right Link Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 4:
# # Global Attempts vs Total Elementary-Link Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Repeater Global Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Repeater Global Attempts std"
#     ],
#     marker="o",
#     capsize=4,
#     label="Global Time Slots"
# )

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Total Elementary-Link Attempts mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Total Elementary-Link Attempts std"
#     ],
#     marker="s",
#     capsize=4,
#     label="Total Elementary-Link Attempts"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Attempts")
# plt.title(
#     "Global Latency Cost vs "
#     "Physical Link-Attempt Cost "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Global vs Elementary Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 5:
# # Memory Expirations
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Total Memory Expirations mean"
#     ],
#     yerr=phase2a_summary_df[
#         "Total Memory Expirations std"
#     ],
#     marker="o",
#     capsize=4
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Memory Expirations")
# plt.title(
#     "Distance vs Quantum-Memory Expirations "
#     "(100 Trials)"
# )

# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Distance vs Memory Expirations.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Graph 6:
# # Session Success Rate
# # =========================================================

# plt.figure(figsize=(10, 6))

# plt.plot(
#     phase2a_summary_df["Distance (km)"],
#     phase2a_summary_df[
#         "Success Rate (%)"
#     ],
#     marker="o"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Successful Repeater Sessions (%)")
# plt.title(
#     "Distance vs Single-Repeater "
#     "Session Success Rate "
#     "(100 Trials)"
# )

# plt.ylim(0, 105)
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Distance vs Success Rate.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Aggregate Improvement Factor

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv("phase2a_monte_carlo_summary_100_trials.csv")

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Aggregate Improvement Factor"
#     ],
#     marker="o",
#     label="Aggregate Improvement Factor"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")
# plt.title(
#     "Single-Repeater Aggregate Improvement "
#     "Over Direct Transmission"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Aggregate Improvement Factor.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Improvement Metric Comparison

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv(
#     "phase2a_monte_carlo_summary_100_trials.csv"
# )

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.errorbar(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Observed Improvement Factor mean"
#     ],
#     yerr=summary_df[
#         "Observed Improvement Factor std"
#     ],
#     marker="o",
#     capsize=4,
#     label=(
#         "Mean of Trial-Wise Ratios "
#         "(Mean ± SD)"
#     )
# )


# # New, stable aggregate metric

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Aggregate Improvement Factor"
#     ],
#     marker="s",
#     linestyle="--",
#     label=(
#         "Ratio of Aggregate Means"
#     )
# )


# # Theoretical ideal-memory result

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Theoretical Improvement Factor mean"
#     ],
#     marker="^",
#     linestyle=":",
#     label=(
#         "Ideal-Memory Theoretical Improvement"
#     )
# )


# plt.xlabel("Total Distance (km)")
# plt.ylabel("Improvement Factor")

# plt.title(
#     "Comparison of Single-Repeater "
#     "Improvement Metrics"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Improvement Metric Comparison.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Repeater Attempt Variability

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv(
#     "phase2a_monte_carlo_summary_100_trials.csv"
# )

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Repeater Attempts Coefficient of Variation (%)"
#     ],
#     marker="o"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Coefficient of Variation (%)")

# plt.title(
#     "Relative Variability of "
#     "Repeater Establishment Attempts"
# )

# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Repeater Attempts Coefficient of Variation.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Mean vs Median Repeater Attempts

# import matplotlib.pyplot as plt
# import pandas as pd


# summary_df = pd.read_csv(
#     "phase2a_monte_carlo_summary_100_trials.csv"
# )

# summary_df = (
#     summary_df
#     .sort_values("Distance (km)")
# )


# plt.figure(figsize=(10, 6))

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Repeater Global Attempts mean"
#     ],
#     marker="o",
#     label="Mean Global Attempts"
# )

# plt.plot(
#     summary_df["Distance (km)"],
#     summary_df[
#         "Repeater Global Attempts median"
#     ],
#     marker="s",
#     label="Median Global Attempts"
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Repeater Global Attempts")

# plt.title(
#     "Mean vs Median Repeater "
#     "Establishment Attempts"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Mean vs Median Repeater Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot — Trial Distribution of Repeater Attempts

# import matplotlib.pyplot as plt
# import pandas as pd


# raw_df = pd.read_csv(
#     "phase2a_monte_carlo_raw_100_trials.csv"
# )

# successful_raw_df = (
#     raw_df[
#         raw_df["Status"] == "SUCCESS"
#     ]
#     .copy()
# )


# ordered_distances = sorted(
#     successful_raw_df[
#         "Distance (km)"
#     ].unique()
# )


# boxplot_data = [
#     successful_raw_df[
#         successful_raw_df[
#             "Distance (km)"
#         ] == distance
#     ][
#         "Repeater Global Attempts"
#     ].dropna().values

#     for distance in ordered_distances
# ]


# plt.figure(figsize=(11, 6))

# plt.boxplot(
#     boxplot_data,
#     labels=[
#         str(distance)
#         for distance
#         in ordered_distances
#     ],
#     showfliers=True
# )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Repeater Global Attempts")

# plt.title(
#     "Distribution of Single-Repeater "
#     "Establishment Attempts Across 100 Trials"
# )

# plt.yscale("log")
# plt.grid(True)

# plt.savefig(
#     "Phase 2A MC Repeater Attempt Distribution Boxplot.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

### Phase 2B

Repeater Assisted QKD Protocols --> AES Key Generation

In [ ]:
# @title Physical Bell Pair Generation

def attempt_bell_pair_generation_physical(
    link: QuantumLink,
    source_fidelity: float = 1.0
):
    """
    Attempts Bell-pair generation over a physical QuantumLink.

    This uses QuantumLink.transmit(), so fiber attenuation and photon loss
    are recorded in the physical layer statistics.
    """

    received = link.transmit("photon")

    if received is not None:
        return BellPair(
            node_a=link.source,
            node_b=link.destination,
            fidelity=source_fidelity,
            source_link_distance_km=link.distance_km
        )

    return None

In [ ]:
# @title Repeater-Assisted Quantum Channel With Layered Statistics

class RepeaterAssistedQuantumChannel:
    """
    Repeater-assisted quantum channel.

    Physical Layer:
    - photons emitted
    - photons survived/lost
    - Bell-pair generation attempts
    - memory expirations
    - entanglement swapping attempts

    Quantum Protocol Layer:
    - qubits requested
    - qubits delivered
    - qubits lost at protocol level
    """

    def __init__(
        self,
        source: Node,
        repeater: Node,
        destination: Node,
        total_distance_km: float,
        swapping_success_probability: float = 1.0,
        memory_lifetime_attempts: int = 10000,
        max_entanglement_attempts: int = 1000000
    ):
        self.source = source
        self.repeater = repeater
        self.destination = destination

        self.total_distance_km = total_distance_km
        self.segment_distance_km = total_distance_km / 2

        self.swapping_success_probability = swapping_success_probability
        self.memory_lifetime_attempts = memory_lifetime_attempts
        self.max_entanglement_attempts = max_entanglement_attempts

        self.left_link = QuantumLink(source, repeater, self.segment_distance_km)
        self.right_link = QuantumLink(repeater, destination, self.segment_distance_km)
        self.direct_link = QuantumLink(source, destination, total_distance_km)

        self.physical_stats = {
            "left_photons_sent": 0,
            "left_photons_survived": 0,
            "left_photons_lost": 0,

            "right_photons_sent": 0,
            "right_photons_survived": 0,
            "right_photons_lost": 0,

            "left_bell_pair_attempts": 0,
            "right_bell_pair_attempts": 0,

            "successful_left_bell_pairs": 0,
            "successful_right_bell_pairs": 0,
            "successful_end_to_end_bell_pairs": 0,

            "swap_attempts": 0,

            "memory_expirations_left": 0,
            "memory_expirations_right": 0,

            "total_entanglement_attempts": 0
        }

        self.protocol_stats = {
            "qubits_requested": 0,
            "qubits_delivered": 0,
            "qubits_lost": 0
        }

    def _sync_physical_stats_from_links(self):
        self.physical_stats["left_photons_sent"] = self.left_link.statistics["lifetime_sent"]
        self.physical_stats["left_photons_survived"] = self.left_link.statistics["lifetime_received"]
        self.physical_stats["left_photons_lost"] = self.left_link.statistics["lifetime_lost"]

        self.physical_stats["right_photons_sent"] = self.right_link.statistics["lifetime_sent"]
        self.physical_stats["right_photons_survived"] = self.right_link.statistics["lifetime_received"]
        self.physical_stats["right_photons_lost"] = self.right_link.statistics["lifetime_lost"]

        self.physical_stats["total_entanglement_attempts"] = (
            self.physical_stats["left_bell_pair_attempts"]
            + self.physical_stats["right_bell_pair_attempts"]
        )

    def generate_end_to_end_bell_pair(self):
        left_pair = None
        right_pair = None

        left_memory_age = 0
        right_memory_age = 0

        for global_attempt in range(1, self.max_entanglement_attempts + 1):

            if left_pair is not None:
                left_memory_age += 1
                if left_memory_age > self.memory_lifetime_attempts:
                    left_pair = None
                    left_memory_age = 0
                    self.physical_stats["memory_expirations_left"] += 1

            if right_pair is not None:
                right_memory_age += 1
                if right_memory_age > self.memory_lifetime_attempts:
                    right_pair = None
                    right_memory_age = 0
                    self.physical_stats["memory_expirations_right"] += 1

            if left_pair is None:
                self.physical_stats["left_bell_pair_attempts"] += 1
                generated_left = attempt_bell_pair_generation_physical(self.left_link)

                if generated_left is not None:
                    left_pair = generated_left
                    left_memory_age = 0
                    self.physical_stats["successful_left_bell_pairs"] += 1

            if right_pair is None:
                self.physical_stats["right_bell_pair_attempts"] += 1
                generated_right = attempt_bell_pair_generation_physical(self.right_link)

                if generated_right is not None:
                    right_pair = generated_right
                    right_memory_age = 0
                    self.physical_stats["successful_right_bell_pairs"] += 1

            if left_pair is not None and right_pair is not None:
                self.physical_stats["swap_attempts"] += 1

                end_to_end_pair = entanglement_swap(
                    left_pair=left_pair,
                    right_pair=right_pair,
                    repeater=self.repeater,
                    swapping_success_probability=self.swapping_success_probability
                )

                if end_to_end_pair is not None:
                    self.physical_stats["successful_end_to_end_bell_pairs"] += 1
                    self._sync_physical_stats_from_links()

                    return {
                        "success": True,
                        "end_to_end_pair": end_to_end_pair,
                        "global_attempts_required": global_attempt
                    }

                left_pair = None
                right_pair = None
                left_memory_age = 0
                right_memory_age = 0

        self._sync_physical_stats_from_links()

        return {
            "success": False,
            "end_to_end_pair": None,
            "global_attempts_required": self.max_entanglement_attempts
        }

    def transmit(self, qubit):
        self.protocol_stats["qubits_requested"] += 1

        result = self.generate_end_to_end_bell_pair()

        if result["success"]:
            self.protocol_stats["qubits_delivered"] += 1
            return qubit

        self.protocol_stats["qubits_lost"] += 1
        return None

    def transmit_batch(self, qubits: list):
        transmitted_qubits = [self.transmit(q) for q in qubits]

        sent = len(qubits)
        delivered = sum(q is not None for q in transmitted_qubits)
        lost = sent - delivered

        self._sync_physical_stats_from_links()

        avg_attempts_per_delivered_qubit = (
            self.physical_stats["total_entanglement_attempts"] / delivered
            if delivered else None
        )

        return {
            "transmitted_qubits": transmitted_qubits,
            "sent": sent,
            "survived": delivered,
            "lost": lost,
            "observed_survival_probability": delivered / sent if sent else 0,
            "physical_stats": self.physical_stats.copy(),
            "protocol_stats": self.protocol_stats.copy(),
            "average_entanglement_attempts_per_delivered_qubit": avg_attempts_per_delivered_qubit
        }

In [ ]:
# @title BB84 Round Over Repeater-Assisted Channel

def bb84_round_over_repeater_channel(
    channel: RepeaterAssistedQuantumChannel,
    target_bits: int = 128,
    transmission_margin: float = 1.3,
    eve_present: bool = False,
    z_probability: float = 0.5
) -> dict:

    p_basis = basis_match_probability(z_probability)

    # Important:
    # In repeater-assisted mode, fiber attenuation is paid inside Bell-pair generation.
    # Therefore protocol qubit budget depends only on basis matching.
    estimated_qubits_required = math.ceil(
        (target_bits * transmission_margin) / p_basis
    )

    alice_bits = random_bitstring(estimated_qubits_required)
    alice_bases = random_bases(estimated_qubits_required, z_probability=z_probability)
    bob_bases = random_bases(estimated_qubits_required, z_probability=z_probability)

    encoded_qubits = encode_qubits(alice_bits, alice_bases)

    transmission = channel.transmit_batch(encoded_qubits)
    received_qubits = transmission["transmitted_qubits"]

    bob_bits = measure_qubits(
        received_qubits,
        bob_bases,
        eve_present=eve_present
    )

    alice_sifted, bob_sifted = sift_keys(
        alice_bits,
        alice_bases,
        bob_bits,
        bob_bases
    )

    qber = calculate_qber(alice_sifted, bob_sifted)
    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,

        "z_probability": z_probability,
        "basis_match_probability": p_basis,
        "estimated_qubits_required": estimated_qubits_required,

        "qubits_sent": transmission["sent"],
        "qubits_received": transmission["survived"],
        "qubits_lost": transmission["lost"],

        "physical_stats": transmission["physical_stats"],
        "protocol_stats": transmission["protocol_stats"],
        "average_entanglement_attempts_per_delivered_qubit":
            transmission["average_entanglement_attempts_per_delivered_qubit"]
    }

In [ ]:
# @title Generate Session Key Over Repeater-Assisted Channel

def generate_bb84_session_key_over_repeater_channel(
    channel: RepeaterAssistedQuantumChannel,
    target_bits: int = 128,
    transmission_margin: float = 1.3,
    max_rounds: int = 100,
    eve_present: bool = False,
    z_probability: float = 0.5,
    verbose: bool = True
):

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):

        remaining_bits = target_bits - len(collected_bits)

        result = bb84_round_over_repeater_channel(
            channel=channel,
            target_bits=remaining_bits,
            transmission_margin=transmission_margin,
            eve_present=eve_present,
            z_probability=z_probability
        )

        round_logs.append({
            "round": round_id,
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"],
            "qubits_sent": result["qubits_sent"],
            "qubits_received": result["qubits_received"],
            "qubits_lost": result["qubits_lost"],
            "average_entanglement_attempts_per_delivered_qubit":
                result["average_entanglement_attempts_per_delivered_qubit"]
        })

        if verbose:
            ps = result["physical_stats"]
            avg_attempts = result["average_entanglement_attempts_per_delivered_qubit"]

            print(
                f"Round {round_id}: "
                f"sent={result['qubits_sent']} | "
                f"received={result['qubits_received']} | "
                f"sifted={result['sifted_length']} | "
                f"QBER={result['qber']:.2%} | "
                f"accepted={result['accepted']} | "
                f"entanglement_attempts={ps['total_entanglement_attempts']} | "
                f"avg_attempts/qubit={avg_attempts:.2f}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough key bits over repeater-assisted channel."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    total_qubits_sent = sum(r["qubits_sent"] for r in round_logs)
    total_qubits_received = sum(r["qubits_received"] for r in round_logs)
    total_qubits_lost = sum(r["qubits_lost"] for r in round_logs)

    ps = channel.physical_stats
    qs = channel.protocol_stats

    total_photons_sent_physical = ps["left_photons_sent"] + ps["right_photons_sent"]
    total_photons_survived_physical = ps["left_photons_survived"] + ps["right_photons_survived"]
    total_photons_lost_physical = ps["left_photons_lost"] + ps["right_photons_lost"]

    metadata = {
        "source": channel.source.name,
        "repeater": channel.repeater.name,
        "destination": channel.destination.name,

        "total_distance_km": channel.total_distance_km,
        "segment_distance_km": channel.segment_distance_km,

        "target_bits": target_bits,
        "z_probability": z_probability,
        "basis_match_probability": basis_match_probability(z_probability),

        "rounds_used": len(round_logs),

        # Protocol layer
        "total_qubits_sent": total_qubits_sent,
        "total_qubits_received": total_qubits_received,
        "total_qubits_lost": total_qubits_lost,
        "effective_key_yield": target_bits / total_qubits_sent if total_qubits_sent else 0,
        "keygen_time_s": elapsed,

        # Physical layer
        "left_photons_sent": ps["left_photons_sent"],
        "left_photons_survived": ps["left_photons_survived"],
        "left_photons_lost": ps["left_photons_lost"],

        "right_photons_sent": ps["right_photons_sent"],
        "right_photons_survived": ps["right_photons_survived"],
        "right_photons_lost": ps["right_photons_lost"],

        "total_photons_sent_physical": total_photons_sent_physical,
        "total_photons_survived_physical": total_photons_survived_physical,
        "total_photons_lost_physical": total_photons_lost_physical,

        "total_entanglement_attempts": ps["total_entanglement_attempts"],
        "average_entanglement_attempts_per_delivered_qubit":
            ps["total_entanglement_attempts"] / total_qubits_received
            if total_qubits_received else None,

        "successful_end_to_end_bell_pairs": ps["successful_end_to_end_bell_pairs"],
        "swap_attempts": ps["swap_attempts"],
        "memory_expirations_left": ps["memory_expirations_left"],
        "memory_expirations_right": ps["memory_expirations_right"],

        "protocol_qubits_requested": qs["qubits_requested"],
        "protocol_qubits_delivered": qs["qubits_delivered"],
        "protocol_qubits_lost": qs["qubits_lost"],

        "round_logs": round_logs
    }

    return key, metadata

In [ ]:
# @title Fixed Distance Protocol Comparison

# test_distance = 100
# target_bits = 128
# transmission_margin = 1.3

# protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15/16
# }

# phase2b_fixed_results = []

# for protocol_name, z_prob in protocols.items():

#     channel = RepeaterAssistedQuantumChannel(
#         source=alice,
#         repeater=r1,
#         destination=bob,
#         total_distance_km=test_distance,
#         swapping_success_probability=1.0,
#         memory_lifetime_attempts=10000,
#         max_entanglement_attempts=1000000
#     )

#     key, meta = generate_bb84_session_key_over_repeater_channel(
#         channel=channel,
#         target_bits=target_bits,
#         transmission_margin=transmission_margin,
#         max_rounds=100,
#         eve_present=False,
#         z_probability=z_prob,
#         verbose=False
#     )

#     phase2b_fixed_results.append({
#         "Protocol": protocol_name,
#         "Distance (km)": test_distance,
#         "Segment Distance (km)": meta["segment_distance_km"],

#         "Z Probability": z_prob,
#         "Basis Match Probability (%)": meta["basis_match_probability"] * 100,

#         "Rounds Used": meta["rounds_used"],

#         # Protocol layer
#         "Qubits Sent": meta["total_qubits_sent"],
#         "Qubits Received": meta["total_qubits_received"],
#         "Qubits Lost Protocol": meta["total_qubits_lost"],
#         "Effective Key Yield (%)": meta["effective_key_yield"] * 100,

#         # Physical layer
#         "Left Photons Sent": meta["left_photons_sent"],
#         "Left Photons Survived": meta["left_photons_survived"],
#         "Left Photons Lost": meta["left_photons_lost"],

#         "Right Photons Sent": meta["right_photons_sent"],
#         "Right Photons Survived": meta["right_photons_survived"],
#         "Right Photons Lost": meta["right_photons_lost"],

#         "Total Physical Photons Sent": meta["total_photons_sent_physical"],
#         "Total Physical Photons Survived": meta["total_photons_survived_physical"],
#         "Total Physical Photons Lost": meta["total_photons_lost_physical"],

#         "Entanglement Attempts": meta["total_entanglement_attempts"],
#         "Avg Entanglement Attempts / Delivered Qubit":
#             meta["average_entanglement_attempts_per_delivered_qubit"],

#         "Successful End-to-End Bell Pairs": meta["successful_end_to_end_bell_pairs"],
#         "Swap Attempts": meta["swap_attempts"],

#         "Keygen Time (s)": meta["keygen_time_s"]
#     })

# df_phase2b_fixed = pd.DataFrame(phase2b_fixed_results)

# df_phase2b_fixed.to_csv(
#     "phase2b_key_generation_at_100_km_with_repeater.csv",
#     index=False
# )

# display(df_phase2b_fixed)

### Phase 2C

Single repeater assisted distances and non ideal fidelity and non ideal entanglement swapping

In [ ]:
# @title Ideal Single-Repeater Distance Sweep

# import time
# import pandas as pd

# # =========================================================
# # Experiment Configuration
# # =========================================================

# phase2c_distances = [10, 50, 80, 100, 150, 200]

# phase2c_target_bits = 128
# phase2c_transmission_margin = 1.3
# phase2c_max_rounds = 100

# # Ideal repeater baseline
# phase2c_swapping_success_probability = 1.0
# phase2c_max_entanglement_attempts = 1_000_000

# # Setting memory lifetime equal to the maximum permitted
# # entanglement attempts effectively disables expiration
# # during one end-to-end Bell-pair establishment session.
# phase2c_memory_lifetime_attempts = (
#     phase2c_max_entanglement_attempts
# )

# phase2c_protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15 / 16
# }

# phase2c_results = []

# phase2c_start_time = time.perf_counter()


# # =========================================================
# # Execute One Session Per Distance and Protocol
# # =========================================================

# for distance_km in phase2c_distances:

#     for protocol_name, z_probability in phase2c_protocols.items():

#         configuration_start = time.perf_counter()

#         channel = RepeaterAssistedQuantumChannel(
#             source=alice,
#             repeater=r1,
#             destination=bob,
#             total_distance_km=distance_km,
#             swapping_success_probability=(
#                 phase2c_swapping_success_probability
#             ),
#             memory_lifetime_attempts=(
#                 phase2c_memory_lifetime_attempts
#             ),
#             max_entanglement_attempts=(
#                 phase2c_max_entanglement_attempts
#             )
#         )

#         try:
#             session_key, metadata = (
#                 generate_bb84_session_key_over_repeater_channel(
#                     channel=channel,
#                     target_bits=phase2c_target_bits,
#                     transmission_margin=(
#                         phase2c_transmission_margin
#                     ),
#                     max_rounds=phase2c_max_rounds,
#                     eve_present=False,
#                     z_probability=z_probability,
#                     verbose=False
#                 )
#             )

#             configuration_runtime = (
#                 time.perf_counter() - configuration_start
#             )

#             # Aggregate QBER across accepted round logs.
#             accepted_round_qbers = [
#                 round_log["qber"]
#                 for round_log in metadata["round_logs"]
#                 if round_log["accepted"]
#             ]

#             mean_accepted_qber = (
#                 sum(accepted_round_qbers)
#                 / len(accepted_round_qbers)
#                 if accepted_round_qbers
#                 else None
#             )

#             accepted_rounds = sum(
#                 bool(round_log["accepted"])
#                 for round_log in metadata["round_logs"]
#             )

#             rejected_rounds = (
#                 len(metadata["round_logs"])
#                 - accepted_rounds
#             )

#             phase2c_results.append({
#                 # -------------------------------------------------
#                 # Experiment identity
#                 # -------------------------------------------------
#                 "Protocol":
#                     protocol_name,

#                 "Distance (km)":
#                     distance_km,

#                 "Segment Distance (km)":
#                     metadata["segment_distance_km"],

#                 "Target Key Bits":
#                     phase2c_target_bits,

#                 "Transmission Margin":
#                     phase2c_transmission_margin,

#                 "Z Probability":
#                     z_probability,

#                 "Basis Match Probability (%)":
#                     metadata[
#                         "basis_match_probability"
#                     ] * 100,

#                 "Swap Success Probability":
#                     phase2c_swapping_success_probability,

#                 "Memory Lifetime (Attempts)":
#                     phase2c_memory_lifetime_attempts,

#                 # -------------------------------------------------
#                 # Quantum protocol layer
#                 # -------------------------------------------------
#                 "Rounds Used":
#                     metadata["rounds_used"],

#                 "Accepted Rounds":
#                     accepted_rounds,

#                 "Rejected Rounds":
#                     rejected_rounds,

#                 "Mean Accepted QBER (%)":
#                     (
#                         mean_accepted_qber * 100
#                         if mean_accepted_qber is not None
#                         else None
#                     ),

#                 "Protocol Qubits Requested":
#                     metadata["total_qubits_sent"],

#                 "Protocol Qubits Delivered":
#                     metadata["total_qubits_received"],

#                 "Protocol Qubits Lost":
#                     metadata["total_qubits_lost"],

#                 "Protocol Delivery Rate (%)":
#                     (
#                         metadata["total_qubits_received"]
#                         / metadata["total_qubits_sent"]
#                         * 100
#                         if metadata["total_qubits_sent"]
#                         else 0
#                     ),

#                 "Effective Key Yield (%)":
#                     metadata["effective_key_yield"] * 100,

#                 "Keygen Time (s)":
#                     metadata["keygen_time_s"],

#                 # -------------------------------------------------
#                 # Physical layer — left elementary link
#                 # -------------------------------------------------
#                 "Left Link Photon Attempts":
#                     metadata["left_photons_sent"],

#                 "Left Link Successful Transmissions":
#                     metadata["left_photons_survived"],

#                 "Left Link Photon Losses":
#                     metadata["left_photons_lost"],

#                 "Left Link Observed Survival (%)":
#                     (
#                         metadata["left_photons_survived"]
#                         / metadata["left_photons_sent"]
#                         * 100
#                         if metadata["left_photons_sent"]
#                         else 0
#                     ),

#                 # -------------------------------------------------
#                 # Physical layer — right elementary link
#                 # -------------------------------------------------
#                 "Right Link Photon Attempts":
#                     metadata["right_photons_sent"],

#                 "Right Link Successful Transmissions":
#                     metadata["right_photons_survived"],

#                 "Right Link Photon Losses":
#                     metadata["right_photons_lost"],

#                 "Right Link Observed Survival (%)":
#                     (
#                         metadata["right_photons_survived"]
#                         / metadata["right_photons_sent"]
#                         * 100
#                         if metadata["right_photons_sent"]
#                         else 0
#                     ),

#                 # -------------------------------------------------
#                 # Physical layer — path totals
#                 # -------------------------------------------------
#                 "Total Physical Photon Attempts":
#                     metadata["total_photons_sent_physical"],

#                 "Total Successful Physical Transmissions":
#                     metadata[
#                         "total_photons_survived_physical"
#                     ],

#                 "Total Physical Photon Losses":
#                     metadata["total_photons_lost_physical"],

#                 "Total Entanglement Attempts":
#                     metadata["total_entanglement_attempts"],

#                 "Average Entanglement Attempts per Delivered Qubit":
#                     metadata[
#                         "average_entanglement_attempts_per_delivered_qubit"
#                     ],

#                 "Successful End-to-End Bell Pairs":
#                     metadata[
#                         "successful_end_to_end_bell_pairs"
#                     ],

#                 "Swap Attempts":
#                     metadata["swap_attempts"],

#                 "Left Memory Expirations":
#                     metadata["memory_expirations_left"],

#                 "Right Memory Expirations":
#                     metadata["memory_expirations_right"],

#                 "Total Memory Expirations":
#                     (
#                         metadata["memory_expirations_left"]
#                         + metadata["memory_expirations_right"]
#                     ),

#                 "Configuration Runtime (s)":
#                     configuration_runtime,

#                 "Status":
#                     "SUCCESS",

#                 "Error":
#                     None
#             })

#             print(
#                 f"{distance_km:3d} km | "
#                 f"{protocol_name:<31} | "
#                 f"protocol qubits="
#                 f"{metadata['total_qubits_sent']:,} | "
#                 f"physical attempts="
#                 f"{metadata['total_entanglement_attempts']:,} | "
#                 f"avg attempts/qubit="
#                 f"{metadata['average_entanglement_attempts_per_delivered_qubit']:.2f}"
#             )

#         except Exception as error:

#             configuration_runtime = (
#                 time.perf_counter() - configuration_start
#             )

#             phase2c_results.append({
#                 "Protocol":
#                     protocol_name,

#                 "Distance (km)":
#                     distance_km,

#                 "Segment Distance (km)":
#                     distance_km / 2,

#                 "Target Key Bits":
#                     phase2c_target_bits,

#                 "Transmission Margin":
#                     phase2c_transmission_margin,

#                 "Z Probability":
#                     z_probability,

#                 "Basis Match Probability (%)":
#                     basis_match_probability(
#                         z_probability
#                     ) * 100,

#                 "Swap Success Probability":
#                     phase2c_swapping_success_probability,

#                 "Memory Lifetime (Attempts)":
#                     phase2c_memory_lifetime_attempts,

#                 "Configuration Runtime (s)":
#                     configuration_runtime,

#                 "Status":
#                     "FAILED",

#                 "Error":
#                     str(error)
#             })

#             print(
#                 f"{distance_km:3d} km | "
#                 f"{protocol_name:<31} | "
#                 f"FAILED: {error}"
#             )


# # =========================================================
# # Save Single-Run Dataset
# # =========================================================

# df_phase2c_distance_sweep = pd.DataFrame(
#     phase2c_results
# )

# df_phase2c_distance_sweep.to_csv(
#     "phase2c_single_repeater_distance_sweep_ideal.csv",
#     index=False
# )

# phase2c_elapsed = (
#     time.perf_counter() - phase2c_start_time
# )

# print(
#     f"\nPhase 2C ideal distance sweep completed "
#     f"in {phase2c_elapsed / 60:.2f} minutes."
# )

# display(
#     df_phase2c_distance_sweep
# )

In [ ]:
# @title Plot : Distance vs Total Entanglement Attempts

# import matplotlib.pyplot as plt

# successful_phase2c = (
#     df_phase2c_distance_sweep[
#         df_phase2c_distance_sweep["Status"]
#         == "SUCCESS"
#     ]
#     .copy()
# )

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_protocols.keys():

#     subset = successful_phase2c[
#         successful_phase2c["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.plot(
#         subset["Distance (km)"],
#         subset["Total Entanglement Attempts"],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Total Elementary-Link Entanglement Attempts")

# plt.title(
#     "Distance vs Repeater Entanglement Resource Cost"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C Distance vs Total Entanglement Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Distance vs Average Entanglement Attempts per Delivered Qubit

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_protocols.keys():

#     subset = successful_phase2c[
#         successful_phase2c["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.plot(
#         subset["Distance (km)"],
#         subset[
#             "Average Entanglement Attempts per Delivered Qubit"
#         ],
#         marker="o",
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel(
#     "Average Entanglement Attempts per Delivered Qubit"
# )

# plt.title(
#     "Distance vs Per-Qubit Repeater Resource Cost"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C Distance vs Average Entanglement Attempts per Delivered Qubit.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

### Phase 2C Monte Carlo Trials

100 Trials

In [ ]:
# @title Monte Carlo Experiment — 100 Trials

# import gc
# import os
# import time
# import pandas as pd


# # =========================================================
# # Experiment Configuration
# # =========================================================

# PHASE2C_MONTE_CARLO_TRIALS = 100
# PHASE2C_CHECKPOINT_INTERVAL = 1

# phase2c_mc_distances = [10, 50, 80, 100, 150, 200]

# phase2c_mc_protocols = {
#     "Standard BB84 (50-50)": 0.5,
#     "Biased BB84 (80-20)": 0.8,
#     "T12-style (15/16 - 1/16)": 15 / 16
# }

# PHASE2C_TARGET_BITS = 128
# PHASE2C_TRANSMISSION_MARGIN = 1.3
# PHASE2C_MAX_ROUNDS = 100

# # Ideal repeater assumptions
# PHASE2C_SWAP_SUCCESS_PROBABILITY = 1.0
# PHASE2C_MAX_ENTANGLEMENT_ATTEMPTS = 1_000_000

# # Effectively ideal memory during one Bell-pair establishment
# PHASE2C_MEMORY_LIFETIME_ATTEMPTS = (
#     PHASE2C_MAX_ENTANGLEMENT_ATTEMPTS
# )

# PHASE2C_RAW_FILE = (
#     "phase2c_monte_carlo_raw_100_trials.csv"
# )

# PHASE2C_LATEST_CHECKPOINT_FILE = (
#     "phase2c_monte_carlo_latest_checkpoint.csv"
# )

# # Change to True only when resuming after a crash.
# RESUME_PHASE2C_FROM_CHECKPOINT = True


# # =========================================================
# # Resume Support
# # =========================================================

# phase2c_raw_results = []
# phase2c_starting_trial = 1

# if (
#     RESUME_PHASE2C_FROM_CHECKPOINT
#     and os.path.exists(
#         PHASE2C_LATEST_CHECKPOINT_FILE
#     )
# ):
#     existing_checkpoint = pd.read_csv(
#         PHASE2C_LATEST_CHECKPOINT_FILE
#     )

#     phase2c_raw_results = (
#         existing_checkpoint.to_dict("records")
#     )

#     if not existing_checkpoint.empty:
#         last_completed_trial = int(
#             existing_checkpoint["Trial"].max()
#         )

#         phase2c_starting_trial = (
#             last_completed_trial + 1
#         )

#         print(
#             "Resuming Phase 2C Monte Carlo "
#             f"from Trial {phase2c_starting_trial}."
#         )

# else:
#     print(
#         "Starting a new Phase 2C Monte Carlo experiment."
#     )


# # =========================================================
# # Monte Carlo Execution
# # =========================================================

# phase2c_experiment_start = time.perf_counter()

# for trial in range(
#     phase2c_starting_trial,
#     PHASE2C_MONTE_CARLO_TRIALS + 1
# ):

#     trial_start = time.perf_counter()

#     print(
#         f"\n{'=' * 82}\n"
#         f"Phase 2C Monte Carlo — "
#         f"Trial {trial}/{PHASE2C_MONTE_CARLO_TRIALS}\n"
#         f"{'=' * 82}"
#     )

#     for distance_km in phase2c_mc_distances:

#         for protocol_name, z_probability in (
#             phase2c_mc_protocols.items()
#         ):

#             configuration_start = time.perf_counter()

#             channel = None
#             session_key = None
#             metadata = None

#             try:
#                 channel = RepeaterAssistedQuantumChannel(
#                     source=alice,
#                     repeater=r1,
#                     destination=bob,
#                     total_distance_km=distance_km,
#                     swapping_success_probability=(
#                         PHASE2C_SWAP_SUCCESS_PROBABILITY
#                     ),
#                     memory_lifetime_attempts=(
#                         PHASE2C_MEMORY_LIFETIME_ATTEMPTS
#                     ),
#                     max_entanglement_attempts=(
#                         PHASE2C_MAX_ENTANGLEMENT_ATTEMPTS
#                     )
#                 )

#                 session_key, metadata = (
#                     generate_bb84_session_key_over_repeater_channel(
#                         channel=channel,
#                         target_bits=PHASE2C_TARGET_BITS,
#                         transmission_margin=(
#                             PHASE2C_TRANSMISSION_MARGIN
#                         ),
#                         max_rounds=PHASE2C_MAX_ROUNDS,
#                         eve_present=False,
#                         z_probability=z_probability,
#                         verbose=False
#                     )
#                 )

#                 configuration_runtime = (
#                     time.perf_counter()
#                     - configuration_start
#                 )

#                 round_logs = metadata["round_logs"]

#                 accepted_rounds = sum(
#                     bool(round_log["accepted"])
#                     for round_log in round_logs
#                 )

#                 rejected_rounds = (
#                     len(round_logs)
#                     - accepted_rounds
#                 )

#                 accepted_qbers = [
#                     round_log["qber"]
#                     for round_log in round_logs
#                     if round_log["accepted"]
#                 ]

#                 mean_accepted_qber = (
#                     sum(accepted_qbers)
#                     / len(accepted_qbers)
#                     if accepted_qbers
#                     else None
#                 )

#                 left_survival = (
#                     metadata["left_photons_survived"]
#                     / metadata["left_photons_sent"]
#                     if metadata["left_photons_sent"]
#                     else 0
#                 )

#                 right_survival = (
#                     metadata["right_photons_survived"]
#                     / metadata["right_photons_sent"]
#                     if metadata["right_photons_sent"]
#                     else 0
#                 )

#                 protocol_delivery_rate = (
#                     metadata["total_qubits_received"]
#                     / metadata["total_qubits_sent"]
#                     if metadata["total_qubits_sent"]
#                     else 0
#                 )

#                 phase2c_raw_results.append({
#                     # =====================================
#                     # Experiment Identity
#                     # =====================================
#                     "Trial":
#                         trial,

#                     "Protocol":
#                         protocol_name,

#                     "Distance (km)":
#                         distance_km,

#                     "Segment Distance (km)":
#                         metadata["segment_distance_km"],

#                     "Target Key Bits":
#                         PHASE2C_TARGET_BITS,

#                     "Transmission Margin":
#                         PHASE2C_TRANSMISSION_MARGIN,

#                     "Z Probability":
#                         z_probability,

#                     "Basis Match Probability (%)":
#                         metadata[
#                             "basis_match_probability"
#                         ] * 100,

#                     "Swap Success Probability":
#                         PHASE2C_SWAP_SUCCESS_PROBABILITY,

#                     "Memory Lifetime (Attempts)":
#                         PHASE2C_MEMORY_LIFETIME_ATTEMPTS,

#                     # =====================================
#                     # Quantum Protocol Layer
#                     # =====================================
#                     "Rounds Used":
#                         metadata["rounds_used"],

#                     "Accepted Rounds":
#                         accepted_rounds,

#                     "Rejected Rounds":
#                         rejected_rounds,

#                     "Mean Accepted QBER (%)":
#                         (
#                             mean_accepted_qber * 100
#                             if mean_accepted_qber is not None
#                             else None
#                         ),

#                     "Protocol Qubits Requested":
#                         metadata["total_qubits_sent"],

#                     "Protocol Qubits Delivered":
#                         metadata["total_qubits_received"],

#                     "Protocol Qubits Lost":
#                         metadata["total_qubits_lost"],

#                     "Protocol Delivery Rate (%)":
#                         protocol_delivery_rate * 100,

#                     "Effective Key Yield (%)":
#                         metadata[
#                             "effective_key_yield"
#                         ] * 100,

#                     "Keygen Time (s)":
#                         metadata["keygen_time_s"],

#                     # =====================================
#                     # Left Elementary Link
#                     # =====================================
#                     "Left Link Photon Attempts":
#                         metadata["left_photons_sent"],

#                     "Left Link Successful Transmissions":
#                         metadata["left_photons_survived"],

#                     "Left Link Photon Losses":
#                         metadata["left_photons_lost"],

#                     "Left Link Observed Survival (%)":
#                         left_survival * 100,

#                     # =====================================
#                     # Right Elementary Link
#                     # =====================================
#                     "Right Link Photon Attempts":
#                         metadata["right_photons_sent"],

#                     "Right Link Successful Transmissions":
#                         metadata["right_photons_survived"],

#                     "Right Link Photon Losses":
#                         metadata["right_photons_lost"],

#                     "Right Link Observed Survival (%)":
#                         right_survival * 100,

#                     # =====================================
#                     # Complete Physical Layer
#                     # =====================================
#                     "Total Physical Photon Attempts":
#                         metadata[
#                             "total_photons_sent_physical"
#                         ],

#                     "Total Successful Physical Transmissions":
#                         metadata[
#                             "total_photons_survived_physical"
#                         ],

#                     "Total Physical Photon Losses":
#                         metadata[
#                             "total_photons_lost_physical"
#                         ],

#                     "Total Entanglement Attempts":
#                         metadata[
#                             "total_entanglement_attempts"
#                         ],

#                     "Average Entanglement Attempts per Delivered Qubit":
#                         metadata[
#                             "average_entanglement_attempts_per_delivered_qubit"
#                         ],

#                     "Successful End-to-End Bell Pairs":
#                         metadata[
#                             "successful_end_to_end_bell_pairs"
#                         ],

#                     "Swap Attempts":
#                         metadata["swap_attempts"],

#                     "Left Memory Expirations":
#                         metadata[
#                             "memory_expirations_left"
#                         ],

#                     "Right Memory Expirations":
#                         metadata[
#                             "memory_expirations_right"
#                         ],

#                     "Total Memory Expirations":
#                         (
#                             metadata[
#                                 "memory_expirations_left"
#                             ]
#                             +
#                             metadata[
#                                 "memory_expirations_right"
#                             ]
#                         ),

#                     "Configuration Runtime (s)":
#                         configuration_runtime,

#                     "Status":
#                         "SUCCESS",

#                     "Error":
#                         None
#                 })

#                 print(
#                     f"Trial {trial:3d} | "
#                     f"{distance_km:3d} km | "
#                     f"{protocol_name:<31} | "
#                     f"protocol qubits="
#                     f"{metadata['total_qubits_sent']:,} | "
#                     f"physical attempts="
#                     f"{metadata['total_entanglement_attempts']:,} | "
#                     f"avg attempts/qubit="
#                     f"{metadata['average_entanglement_attempts_per_delivered_qubit']:.2f} | "
#                     f"time="
#                     f"{metadata['keygen_time_s']:.2f}s"
#                 )

#             except Exception as error:

#                 configuration_runtime = (
#                     time.perf_counter()
#                     - configuration_start
#                 )

#                 phase2c_raw_results.append({
#                     "Trial":
#                         trial,

#                     "Protocol":
#                         protocol_name,

#                     "Distance (km)":
#                         distance_km,

#                     "Segment Distance (km)":
#                         distance_km / 2,

#                     "Target Key Bits":
#                         PHASE2C_TARGET_BITS,

#                     "Transmission Margin":
#                         PHASE2C_TRANSMISSION_MARGIN,

#                     "Z Probability":
#                         z_probability,

#                     "Basis Match Probability (%)":
#                         basis_match_probability(
#                             z_probability
#                         ) * 100,

#                     "Swap Success Probability":
#                         PHASE2C_SWAP_SUCCESS_PROBABILITY,

#                     "Memory Lifetime (Attempts)":
#                         PHASE2C_MEMORY_LIFETIME_ATTEMPTS,

#                     "Configuration Runtime (s)":
#                         configuration_runtime,

#                     "Status":
#                         "FAILED",

#                     "Error":
#                         str(error)
#                 })

#                 print(
#                     f"Trial {trial:3d} | "
#                     f"{distance_km:3d} km | "
#                     f"{protocol_name:<31} | "
#                     f"FAILED: {error}"
#                 )

#             # Explicit cleanup for long Colab runs
#             del channel
#             del session_key
#             del metadata

#             gc.collect()

#     trial_runtime = (
#         time.perf_counter() - trial_start
#     )

#     completed_rows = len(
#         phase2c_raw_results
#     )

#     print(
#         f"\nTrial {trial} completed in "
#         f"{trial_runtime / 60:.2f} minutes."
#     )

#     print(
#         f"Rows collected so far: "
#         f"{completed_rows:,}"
#     )

#     # =====================================================
#     # Checkpoint Every 50 Completed Trials
#     # =====================================================

#     if (
#         trial % PHASE2C_CHECKPOINT_INTERVAL == 0
#         or trial == PHASE2C_MONTE_CARLO_TRIALS
#     ):

#         checkpoint_df = pd.DataFrame(
#             phase2c_raw_results
#         )

#         numbered_checkpoint_file = (
#             "phase2c_monte_carlo_checkpoint_"
#             f"trial_{trial}.csv"
#         )

#         checkpoint_df.to_csv(
#             numbered_checkpoint_file,
#             index=False
#         )

#         checkpoint_df.to_csv(
#             PHASE2C_LATEST_CHECKPOINT_FILE,
#             index=False
#         )

#         print(
#             f"\nCheckpoint saved after Trial {trial}:"
#             f"\n  {numbered_checkpoint_file}"
#             f"\n  {PHASE2C_LATEST_CHECKPOINT_FILE}"
#             f"\n  Rows saved: {len(checkpoint_df):,}"
#         )


# # =========================================================
# # Save Final Raw Dataset
# # =========================================================

# df_phase2c_monte_carlo_raw = pd.DataFrame(
#     phase2c_raw_results
# )

# df_phase2c_monte_carlo_raw.to_csv(
#     PHASE2C_RAW_FILE,
#     index=False
# )

# phase2c_total_runtime = (
#     time.perf_counter()
#     - phase2c_experiment_start
# )

# expected_phase2c_rows = (
#     PHASE2C_MONTE_CARLO_TRIALS
#     * len(phase2c_mc_distances)
#     * len(phase2c_mc_protocols)
# )

# print(
#     f"\n{'=' * 82}\n"
#     f"PHASE 2C MONTE CARLO COMPLETE\n"
#     f"{'=' * 82}\n"
#     f"Trials completed       : "
#     f"{PHASE2C_MONTE_CARLO_TRIALS}\n"
#     f"Distances per trial    : "
#     f"{len(phase2c_mc_distances)}\n"
#     f"Protocols per distance : "
#     f"{len(phase2c_mc_protocols)}\n"
#     f"Raw rows generated     : "
#     f"{len(df_phase2c_monte_carlo_raw):,}\n"
#     f"Expected raw rows      : "
#     f"{expected_phase2c_rows:,}\n"
#     f"Total runtime          : "
#     f"{phase2c_total_runtime / 3600:.2f} hours\n"
#     f"Final raw CSV          : "
#     f"{PHASE2C_RAW_FILE}\n"
#     f"{'=' * 82}"
# )

# display(
#     df_phase2c_monte_carlo_raw.head(18)
# )

In [ ]:
# @title Monte Carlo Aggregate Statistics

# import pandas as pd


# df_phase2c_monte_carlo_raw = pd.read_csv(
#     "phase2c_monte_carlo_raw_100_trials.csv"
# )

# successful_phase2c_runs = (
#     df_phase2c_monte_carlo_raw[
#         df_phase2c_monte_carlo_raw[
#             "Status"
#         ] == "SUCCESS"
#     ]
#     .copy()
# )

# phase2c_group_columns = [
#     "Protocol",
#     "Distance (km)"
# ]

# phase2c_metrics_to_aggregate = [
#     "Segment Distance (km)",
#     "Basis Match Probability (%)",

#     "Rounds Used",
#     "Accepted Rounds",
#     "Rejected Rounds",
#     "Mean Accepted QBER (%)",

#     "Protocol Qubits Requested",
#     "Protocol Qubits Delivered",
#     "Protocol Qubits Lost",
#     "Protocol Delivery Rate (%)",
#     "Effective Key Yield (%)",
#     "Keygen Time (s)",

#     "Left Link Photon Attempts",
#     "Left Link Successful Transmissions",
#     "Left Link Photon Losses",
#     "Left Link Observed Survival (%)",

#     "Right Link Photon Attempts",
#     "Right Link Successful Transmissions",
#     "Right Link Photon Losses",
#     "Right Link Observed Survival (%)",

#     "Total Physical Photon Attempts",
#     "Total Successful Physical Transmissions",
#     "Total Physical Photon Losses",

#     "Total Entanglement Attempts",
#     "Average Entanglement Attempts per Delivered Qubit",

#     "Successful End-to-End Bell Pairs",
#     "Swap Attempts",

#     "Left Memory Expirations",
#     "Right Memory Expirations",
#     "Total Memory Expirations",

#     "Configuration Runtime (s)"
# ]


# phase2c_summary_statistics = (
#     successful_phase2c_runs
#     .groupby(
#         phase2c_group_columns,
#         as_index=False
#     )[phase2c_metrics_to_aggregate]
#     .agg([
#         "count",
#         "mean",
#         "std",
#         "median",
#         "min",
#         "max"
#     ])
# )


# # Flatten MultiIndex columns
# phase2c_summary_statistics.columns = [
#     " ".join(
#         str(part)
#         for part in column
#         if str(part) != ""
#     ).strip()
#     if isinstance(column, tuple)
#     else column
#     for column in phase2c_summary_statistics.columns
# ]


# # =========================================================
# # Success and Failure Counts
# # =========================================================

# phase2c_run_counts = (
#     df_phase2c_monte_carlo_raw
#     .groupby(
#         phase2c_group_columns
#     )
#     .agg(
#         Total_Trials=(
#             "Trial",
#             "count"
#         ),

#         Successful_Trials=(
#             "Status",
#             lambda values: (
#                 values == "SUCCESS"
#             ).sum()
#         ),

#         Failed_Trials=(
#             "Status",
#             lambda values: (
#                 values == "FAILED"
#             ).sum()
#         )
#     )
#     .reset_index()
# )

# phase2c_run_counts[
#     "Success Rate (%)"
# ] = (
#     phase2c_run_counts[
#         "Successful_Trials"
#     ]
#     /
#     phase2c_run_counts[
#         "Total_Trials"
#     ]
#     * 100
# )


# df_phase2c_monte_carlo_summary = (
#     phase2c_run_counts
#     .merge(
#         phase2c_summary_statistics,
#         on=phase2c_group_columns,
#         how="left"
#     )
# )


# # =========================================================
# # Coefficients of Variation
# # =========================================================

# df_phase2c_monte_carlo_summary[
#     "Entanglement Attempts CV"
# ] = (
#     df_phase2c_monte_carlo_summary[
#         "Total Entanglement Attempts std"
#     ]
#     /
#     df_phase2c_monte_carlo_summary[
#         "Total Entanglement Attempts mean"
#     ]
# )

# df_phase2c_monte_carlo_summary[
#     "Entanglement Attempts CV (%)"
# ] = (
#     df_phase2c_monte_carlo_summary[
#         "Entanglement Attempts CV"
#     ]
#     * 100
# )

# df_phase2c_monte_carlo_summary[
#     "Keygen Time CV"
# ] = (
#     df_phase2c_monte_carlo_summary[
#         "Keygen Time (s) std"
#     ]
#     /
#     df_phase2c_monte_carlo_summary[
#         "Keygen Time (s) mean"
#     ]
# )

# df_phase2c_monte_carlo_summary[
#     "Keygen Time CV (%)"
# ] = (
#     df_phase2c_monte_carlo_summary[
#         "Keygen Time CV"
#     ]
#     * 100
# )


# # =========================================================
# # Sorting
# # =========================================================

# phase2c_protocol_order = {
#     "Standard BB84 (50-50)": 0,
#     "Biased BB84 (80-20)": 1,
#     "T12-style (15/16 - 1/16)": 2
# }

# df_phase2c_monte_carlo_summary[
#     "_Protocol Order"
# ] = (
#     df_phase2c_monte_carlo_summary[
#         "Protocol"
#     ].map(phase2c_protocol_order)
# )

# df_phase2c_monte_carlo_summary = (
#     df_phase2c_monte_carlo_summary
#     .sort_values([
#         "_Protocol Order",
#         "Distance (km)"
#     ])
#     .drop(
#         columns=["_Protocol Order"]
#     )
#     .reset_index(drop=True)
# )


# PHASE2C_SUMMARY_FILE = (
#     "phase2c_monte_carlo_summary_100_trials.csv"
# )

# df_phase2c_monte_carlo_summary.to_csv(
#     PHASE2C_SUMMARY_FILE,
#     index=False
# )

# print(
#     f"Phase 2C summary saved to "
#     f"{PHASE2C_SUMMARY_FILE}"
# )

# print(
#     f"Raw rows     : "
#     f"{len(df_phase2c_monte_carlo_raw):,}"
# )

# print(
#     f"Summary rows : "
#     f"{len(df_phase2c_monte_carlo_summary):,}"
# )

# print(
#     f"Failed runs  : "
#     f"{(df_phase2c_monte_carlo_raw['Status']== 'FAILED').sum():,}"
# )

# display(
#     df_phase2c_monte_carlo_summary
# )

In [ ]:
# @title Compact Paper-Ready Summary

# phase2c_paper_columns = [
#     "Protocol",
#     "Distance (km)",

#     "Total_Trials",
#     "Successful_Trials",
#     "Failed_Trials",
#     "Success Rate (%)",

#     "Protocol Qubits Requested mean",
#     "Protocol Qubits Requested std",

#     "Protocol Qubits Delivered mean",
#     "Protocol Qubits Delivered std",

#     "Protocol Delivery Rate (%) mean",
#     "Protocol Delivery Rate (%) std",

#     "Effective Key Yield (%) mean",
#     "Effective Key Yield (%) std",

#     "Keygen Time (s) mean",
#     "Keygen Time (s) std",

#     "Total Physical Photon Attempts mean",
#     "Total Physical Photon Attempts std",

#     "Total Physical Photon Losses mean",
#     "Total Physical Photon Losses std",

#     "Total Entanglement Attempts mean",
#     "Total Entanglement Attempts std",
#     "Total Entanglement Attempts median",

#     "Average Entanglement Attempts per Delivered Qubit mean",
#     "Average Entanglement Attempts per Delivered Qubit std",

#     "Successful End-to-End Bell Pairs mean",
#     "Swap Attempts mean",

#     "Total Memory Expirations mean",

#     "Entanglement Attempts CV (%)",
#     "Keygen Time CV (%)"
# ]

# available_phase2c_paper_columns = [
#     column
#     for column in phase2c_paper_columns
#     if column
#     in df_phase2c_monte_carlo_summary.columns
# ]

# df_phase2c_paper_summary = (
#     df_phase2c_monte_carlo_summary[
#         available_phase2c_paper_columns
#     ]
#     .copy()
# )

# PHASE2C_PAPER_SUMMARY_FILE = (
#     "phase2c_monte_carlo_"
#     "paper_summary_100_trials.csv"
# )

# df_phase2c_paper_summary.to_csv(
#     PHASE2C_PAPER_SUMMARY_FILE,
#     index=False
# )

# print(
#     "Phase 2C paper-ready summary saved to:"
# )

# print(
#     PHASE2C_PAPER_SUMMARY_FILE
# )

# display(
#     df_phase2c_paper_summary
# )

In [ ]:
# @title Plot : Distance vs Key Generation Time

# import matplotlib.pyplot as plt
# import pandas as pd

# phase2c_summary_df = pd.read_csv(
#     "phase2c_monte_carlo_summary_100_trials.csv"
# )

# phase2c_plot_protocols = [
#     "Standard BB84 (50-50)",
#     "Biased BB84 (80-20)",
#     "T12-style (15/16 - 1/16)"
# ]

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_plot_protocols:

#     subset = phase2c_summary_df[
#         phase2c_summary_df["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset["Keygen Time (s) mean"],
#         yerr=subset["Keygen Time (s) std"],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Simulation Key Generation Time (s)")

# plt.title(
#     "Repeater-Assisted QKD Key Generation Time "
#     "(Mean ± SD, 100 Trials)"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C MC Distance vs Key Generation Time.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Distance vs Effective Key Yield

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_plot_protocols:

#     subset = phase2c_summary_df[
#         phase2c_summary_df["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset[
#             "Effective Key Yield (%) mean"
#         ],
#         yerr=subset[
#             "Effective Key Yield (%) std"
#         ],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Effective Key Yield (%)")

# plt.title(
#     "Repeater-Assisted QKD Effective Key Yield "
#     "(Mean ± SD, 100 Trials)"
# )

# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C MC Distance vs Effective Key Yield.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Repeater Physical Resource Cost

# # =========================================================
# # Total Entanglement Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_plot_protocols:

#     subset = phase2c_summary_df[
#         phase2c_summary_df["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset[
#             "Total Entanglement Attempts mean"
#         ],
#         yerr=subset[
#             "Total Entanglement Attempts std"
#         ],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Total Entanglement Attempts")

# plt.title(
#     "Distance vs Total Repeater Entanglement Attempts "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C MC Distance vs Total Entanglement Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Average Attempts per Delivered Qubit
# # =========================================================

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_plot_protocols:

#     subset = phase2c_summary_df[
#         phase2c_summary_df["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset[
#             "Average Entanglement Attempts per Delivered Qubit mean"
#         ],
#         yerr=subset[
#             "Average Entanglement Attempts per Delivered Qubit std"
#         ],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel(
#     "Mean Entanglement Attempts per Delivered Qubit"
# )

# plt.title(
#     "Distance vs Per-Qubit Repeater Resource Cost "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C MC Distance vs Attempts per Delivered Qubit.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()


# # =========================================================
# # Total Physical Photon Attempts
# # =========================================================

# plt.figure(figsize=(10, 6))

# for protocol_name in phase2c_plot_protocols:

#     subset = phase2c_summary_df[
#         phase2c_summary_df["Protocol"]
#         == protocol_name
#     ].sort_values("Distance (km)")

#     plt.errorbar(
#         subset["Distance (km)"],
#         subset[
#             "Total Physical Photon Attempts mean"
#         ],
#         yerr=subset[
#             "Total Physical Photon Attempts std"
#         ],
#         marker="o",
#         capsize=4,
#         label=protocol_name
#     )

# plt.xlabel("Total Distance (km)")
# plt.ylabel("Mean Physical Photon Attempts")

# plt.title(
#     "Distance vs Physical Photon Resource Requirement "
#     "(100 Trials)"
# )

# plt.yscale("log")
# plt.legend()
# plt.grid(True)

# plt.savefig(
#     "Phase 2C MC Distance vs Physical Photon Attempts.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# @title Plot : Distribution Plots

# import matplotlib.pyplot as plt
# import pandas as pd


# phase2c_raw_df = pd.read_csv(
#     "phase2c_monte_carlo_raw_100_trials.csv"
# )

# phase2c_successful_raw = (
#     phase2c_raw_df[
#         phase2c_raw_df["Status"] == "SUCCESS"
#     ]
#     .copy()
# )

# ordered_distances = sorted(
#     phase2c_successful_raw[
#         "Distance (km)"
#     ].unique()
# )


# # =========================================================
# # Entanglement Attempt Distribution by Protocol
# # =========================================================

# for protocol_name in phase2c_plot_protocols:

#     protocol_data = phase2c_successful_raw[
#         phase2c_successful_raw[
#             "Protocol"
#         ] == protocol_name
#     ]

#     boxplot_values = [
#         protocol_data[
#             protocol_data[
#                 "Distance (km)"
#             ] == distance
#         ][
#             "Total Entanglement Attempts"
#         ].dropna().values

#         for distance in ordered_distances
#     ]

#     plt.figure(figsize=(11, 6))

#     plt.boxplot(
#         boxplot_values,
#         tick_labels=[
#             str(distance)
#             for distance in ordered_distances
#         ],
#         showfliers=True
#     )

#     plt.xlabel("Total Distance (km)")
#     plt.ylabel("Total Entanglement Attempts")

#     plt.title(
#         f"{protocol_name}: Entanglement Attempt "
#         f"Distribution Across 100 Trials"
#     )

#     plt.yscale("log")
#     plt.grid(True)

#     safe_protocol_name = (
#         protocol_name
#         .replace("/", "-")
#         .replace(" ", "_")
#         .replace("(", "")
#         .replace(")", "")
#     )

#     plt.savefig(
#         f"Phase 2C MC {safe_protocol_name} "
#         f"Entanglement Attempt Boxplot.png",
#         dpi=300,
#         bbox_inches="tight"
#     )

#     plt.show()


# # =========================================================
# # Key Generation Time Distribution by Protocol
# # =========================================================

# for protocol_name in phase2c_plot_protocols:

#     protocol_data = phase2c_successful_raw[
#         phase2c_successful_raw[
#             "Protocol"
#         ] == protocol_name
#     ]

#     boxplot_values = [
#         protocol_data[
#             protocol_data[
#                 "Distance (km)"
#             ] == distance
#         ][
#             "Keygen Time (s)"
#         ].dropna().values

#         for distance in ordered_distances
#     ]

#     plt.figure(figsize=(11, 6))

#     plt.boxplot(
#         boxplot_values,
#         tick_labels=[
#             str(distance)
#             for distance in ordered_distances
#         ],
#         showfliers=True
#     )

#     plt.xlabel("Total Distance (km)")
#     plt.ylabel("Simulation Key Generation Time (s)")

#     plt.title(
#         f"{protocol_name}: Key Generation Time "
#         f"Distribution Across 100 Trials"
#     )

#     plt.grid(True)

#     safe_protocol_name = (
#         protocol_name
#         .replace("/", "-")
#         .replace(" ", "_")
#         .replace("(", "")
#         .replace(")", "")
#     )

#     plt.savefig(
#         f"Phase 2C MC {safe_protocol_name} "
#         f"Key Generation Time Boxplot.png",
#         dpi=300,
#         bbox_inches="tight"
#     )

#     plt.show()

## Phase 3 : 09-09-2026T13:21:26 -- 13-10-2026T15:07:32

Phase 3 rebuilds NARAD-QC as a discrete-event, resource-aware quantum-network simulation architecture supporting arbitrary repeater chains, finite quantum memory, entanglement swapping trees, and future adaptive control-plane scheduling.

Phase 0–2 remain unchanged and serve as the validated cryptographic,
distance-dependent, and single-repeater baseline.

### Phase 3A : 09-09-2026T13:21:26 -- 10-09-2026T02:27:53

In [ ]:
# @title Simulation Event Record

from dataclasses import dataclass, field
from typing import Any, Optional
import time


@dataclass(frozen=True)
class SimulationEvent:
    """
    Immutable record describing one meaningful event in a NARAD-QC
    Phase 3 simulation.

    Important:
    - simulation_slot belongs to simulated network time/order.
    - wall_elapsed_s measures Python execution time only.
    - wall_elapsed_s must never be interpreted as physical network latency.
    """

    event_index: int
    event_type: str
    simulation_slot: int

    message: str = ""

    component: Optional[str] = None
    session_id: Optional[str] = None

    node_names: tuple[str, ...] = field(
        default_factory=tuple
    )

    link_ids: tuple[str, ...] = field(
        default_factory=tuple
    )

    resource_ids: tuple[str, ...] = field(
        default_factory=tuple
    )

    metadata: dict[str, Any] = field(
        default_factory=dict
    )

    wall_elapsed_s: Optional[float] = None

In [ ]:
# @title Simulation Kernel

import secrets
import numpy as np
import pandas as pd


class SimulationKernel:
    """
    Core execution context for reconstructed NARAD-QC Phase 3.

    Responsibilities
    ----------------
    1. Own the discrete simulation clock.
    2. Own the seeded random-number generator.
    3. Own the authoritative simulation event log.

    The kernel does NOT:
    - model network topology;
    - generate entanglement;
    - perform swapping;
    - manage quantum memory;
    - perform routing;
    - perform QKD.

    Those responsibilities belong to higher-level Phase 3 components.
    """

    def __init__(
        self,
        seed: Optional[int] = None
    ):
        if seed is None:
            seed = secrets.randbits(64)

        if (
            not isinstance(seed, (int, np.integer))
            or isinstance(seed, bool)
            or seed < 0
        ):
            raise ValueError(
                "Simulation seed must be a non-negative integer."
            )

        self.seed = int(seed)

        # Phase-3 stochastic processes use this RNG.
        self.rng = np.random.default_rng(
            self.seed
        )

        # Discrete simulation time.
        self._current_slot = 0

        # Python execution timer.
        self._wall_start = time.perf_counter()

        # Authoritative event history for this simulation.
        self._events: list[SimulationEvent] = []

        self.emit(
            event_type="SIMULATION_INITIALIZED",
            message=(
                f"Simulation initialized with seed "
                f"{self.seed}."
            ),
            component="SimulationKernel",
            metadata={
                "seed": self.seed
            }
        )

    # =========================================================
    # Simulation Clock
    # =========================================================

    @property
    def current_slot(self) -> int:
        return self._current_slot

    def advance_slot(
        self,
        steps: int = 1
    ) -> int:
        """
        Advance simulated time by an integer number of slots.

        Advancing the clock itself does not emit an event because
        future simulations may contain millions of empty/ordinary
        scheduling slots. Meaningful physical/control events are
        recorded separately.
        """

        if (
            not isinstance(steps, int)
            or isinstance(steps, bool)
            or steps <= 0
        ):
            raise ValueError(
                "steps must be a positive integer."
            )

        self._current_slot += steps

        return self._current_slot

    # =========================================================
    # Event System
    # =========================================================

    def emit(
        self,
        event_type: str,
        message: str = "",
        component: Optional[str] = None,
        session_id: Optional[str] = None,
        node_names=(),
        link_ids=(),
        resource_ids=(),
        metadata: Optional[dict] = None
    ) -> SimulationEvent:
        """
        Record one meaningful simulation event.
        """

        if not isinstance(event_type, str) or not event_type.strip():
            raise ValueError(
                "event_type must be a non-empty string."
            )

        event = SimulationEvent(
            event_index=len(self._events),
            event_type=event_type.strip(),
            simulation_slot=self.current_slot,

            message=str(message),

            component=component,
            session_id=session_id,

            node_names=tuple(node_names),
            link_ids=tuple(link_ids),
            resource_ids=tuple(resource_ids),

            metadata=dict(metadata or {}),

            wall_elapsed_s=(
                time.perf_counter()
                - self._wall_start
            )
        )

        self._events.append(event)

        return event

    # =========================================================
    # Event Inspection
    # =========================================================

    @property
    def events(self) -> tuple[SimulationEvent, ...]:
        """
        Return an immutable view of the event history.
        """
        return tuple(self._events)

    def events_dataframe(self) -> pd.DataFrame:
        """
        Return the event log as a dataframe for inspection,
        visualization, and later experiment export.
        """

        rows = []

        for event in self._events:
            rows.append({
                "Event Index":
                    event.event_index,

                "Event Type":
                    event.event_type,

                "Simulation Slot":
                    event.simulation_slot,

                "Component":
                    event.component,

                "Session ID":
                    event.session_id,

                "Node Names":
                    list(event.node_names),

                "Link IDs":
                    list(event.link_ids),

                "Resource IDs":
                    list(event.resource_ids),

                "Message":
                    event.message,

                "Metadata":
                    event.metadata,

                "Wall Elapsed (s)":
                    event.wall_elapsed_s
            })

        return pd.DataFrame(rows)

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:
        return (
            "SimulationKernel("
            f"seed={self.seed}, "
            f"slot={self.current_slot}, "
            f"events={len(self._events)}"
            ")"
        )

In [ ]:
# @title Simulation Kernel Verification

# print("=" * 76)
# print("NARAD-QC — Simulation Kernel Verification")
# print("=" * 76)


# # =============================================================
# # Test 1 — Same seed must reproduce the same random sequence
# # =============================================================

# test_kernel_a = SimulationKernel(
#     seed=123456
# )

# test_kernel_b = SimulationKernel(
#     seed=123456
# )

# sequence_a = test_kernel_a.rng.random(8)
# sequence_b = test_kernel_b.rng.random(8)

# assert np.array_equal(
#     sequence_a,
#     sequence_b
# )

# print(
#     "[PASS] Identical seeds reproduce "
#     "identical random sequences."
# )


# # =============================================================
# # Test 2 — Simulation slot advances independently
# # =============================================================

# assert test_kernel_a.current_slot == 0

# test_kernel_a.advance_slot()

# assert test_kernel_a.current_slot == 1

# test_kernel_a.advance_slot(
#     steps=4
# )

# assert test_kernel_a.current_slot == 5

# print(
#     "[PASS] Discrete simulation clock advances correctly."
# )


# # =============================================================
# # Test 3 — Meaningful events retain their simulation slot
# # =============================================================

# test_event = test_kernel_a.emit(
#     event_type="TEST_EVENT",
#     component="Phase3A.1",
#     message="Kernel event-system verification."
# )

# assert test_event.simulation_slot == 5
# assert test_event.event_index == 1

# print(
#     "[PASS] Events preserve simulation-slot ordering."
# )


# # =============================================================
# # Test 4 — Event history cannot be modified through property
# # =============================================================

# assert isinstance(
#     test_kernel_a.events,
#     tuple
# )

# print(
#     "[PASS] Event-history view is immutable."
# )


# # =============================================================
# # Display test event history
# # =============================================================

# print("\nTEST EVENT LOG")

# display(
#     test_kernel_a.events_dataframe()
# )


# print("\n" + "=" * 76)
# print("Verification completed successfully.")
# print("=" * 76)

In [ ]:
# @title Quantum Network Node

from dataclasses import dataclass
import math
import numbers


@dataclass(frozen=True)
class NetworkNode:
    """
    Immutable physical node descriptor for reconstructed NARAD-QC Phase 3.

    A NetworkNode represents physical identity, spatial position,
    and intrinsic quantum-network capabilities.

    It deliberately does NOT contain:
    - authentication state;
    - routing state;
    - session reservations;
    - busy/available flags;
    - quantum-memory contents;
    - swapping statistics;
    - physical-operation probabilities.

    Those belong to later runtime, control-plane, memory,
    and physical-model components.
    """

    name: str

    x_km: float
    y_km: float

    endpoint_capable: bool
    swapping_capable: bool

    memory_capacity: int

    def __post_init__(self):

        # =====================================================
        # Identity validation
        # =====================================================

        if (
            not isinstance(self.name, str)
            or not self.name.strip()
        ):
            raise ValueError(
                "Node name must be a non-empty string."
            )

        # Store normalized surrounding whitespace away.
        object.__setattr__(
            self,
            "name",
            self.name.strip()
        )

        # =====================================================
        # Coordinate validation
        # =====================================================

        for coordinate_name, coordinate_value in (
            ("x_km", self.x_km),
            ("y_km", self.y_km)
        ):

            if (
                isinstance(coordinate_value, bool)
                or not isinstance(
                    coordinate_value,
                    numbers.Real
                )
                or not math.isfinite(
                    float(coordinate_value)
                )
            ):
                raise ValueError(
                    f"{coordinate_name} must be a finite "
                    f"real number."
                )

        object.__setattr__(
            self,
            "x_km",
            float(self.x_km)
        )

        object.__setattr__(
            self,
            "y_km",
            float(self.y_km)
        )

        # =====================================================
        # Capability validation
        # =====================================================

        if not isinstance(
            self.endpoint_capable,
            bool
        ):
            raise TypeError(
                "endpoint_capable must be bool."
            )

        if not isinstance(
            self.swapping_capable,
            bool
        ):
            raise TypeError(
                "swapping_capable must be bool."
            )

        if (
            not self.endpoint_capable
            and not self.swapping_capable
        ):
            raise ValueError(
                "A NetworkNode must support at least one "
                "network role: endpoint and/or swapping."
            )

        # =====================================================
        # Memory hardware validation
        # =====================================================

        if (
            isinstance(self.memory_capacity, bool)
            or not isinstance(
                self.memory_capacity,
                numbers.Integral
            )
        ):
            raise TypeError(
                "memory_capacity must be an integer."
            )

        object.__setattr__(
            self,
            "memory_capacity",
            int(self.memory_capacity)
        )

        if self.memory_capacity < 1:
            raise ValueError(
                "memory_capacity must be at least 1."
            )

        # A swapping operation requires two local qubits.
        if (
            self.swapping_capable
            and self.memory_capacity < 2
        ):
            raise ValueError(
                "A swapping-capable node requires at "
                "least 2 quantum-memory slots."
            )

    # =========================================================
    # Spatial Properties
    # =========================================================

    @property
    def position(self) -> tuple[float, float]:
        return (
            self.x_km,
            self.y_km
        )

    def distance_to(
        self,
        other: "NetworkNode"
    ) -> float:
        """
        Euclidean physical separation in kilometres.
        """

        if not isinstance(
            other,
            NetworkNode
        ):
            raise TypeError(
                "distance_to() requires another NetworkNode."
            )

        return math.hypot(
            self.x_km - other.x_km,
            self.y_km - other.y_km
        )

    # =========================================================
    # Capability Helpers
    # =========================================================

    @property
    def functional_role(self) -> str:
        """
        Human-readable role derived from capabilities.

        This is descriptive only; it is not used to define
        the node's behaviour.
        """

        if (
            self.endpoint_capable
            and self.swapping_capable
        ):
            return "HYBRID"

        if self.endpoint_capable:
            return "ENDPOINT"

        return "REPEATER"

    @property
    def can_be_endpoint(self) -> bool:
        return self.endpoint_capable

    @property
    def can_swap(self) -> bool:
        return self.swapping_capable

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:
        return (
            "NetworkNode("
            f"name='{self.name}', "
            f"role='{self.functional_role}', "
            f"position=({self.x_km:.2f}, "
            f"{self.y_km:.2f}) km, "
            f"memory_capacity={self.memory_capacity}"
            ")"
        )

In [ ]:
# @title Network Node Verification

# from dataclasses import FrozenInstanceError


# print("=" * 76)
# print("NARAD-QC - Network Node Verification")
# print("=" * 76)


# # =============================================================
# # Test 1 — Endpoint node
# # =============================================================

# test_alice = NetworkNode(
#     name="Alice",
#     x_km=0,
#     y_km=0,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=1
# )

# assert test_alice.functional_role == "ENDPOINT"
# assert test_alice.can_be_endpoint
# assert not test_alice.can_swap
# assert test_alice.position == (0.0, 0.0)

# print(
#     "[PASS] Endpoint node created correctly."
# )


# # =============================================================
# # Test 2 — Repeater node
# # =============================================================

# test_r1 = NetworkNode(
#     name="R1",
#     x_km=3,
#     y_km=4,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )

# assert test_r1.functional_role == "REPEATER"
# assert not test_r1.can_be_endpoint
# assert test_r1.can_swap

# print(
#     "[PASS] Swapping-capable repeater created correctly."
# )


# # =============================================================
# # Test 3 — Hybrid node
# # =============================================================

# test_hybrid = NetworkNode(
#     name="Hybrid-1",
#     x_km=10,
#     y_km=10,
#     endpoint_capable=True,
#     swapping_capable=True,
#     memory_capacity=4
# )

# assert test_hybrid.functional_role == "HYBRID"
# assert test_hybrid.can_be_endpoint
# assert test_hybrid.can_swap

# print(
#     "[PASS] Hybrid endpoint/repeater capability is supported."
# )


# # =============================================================
# # Test 4 — Spatial distance
# # =============================================================

# distance = test_alice.distance_to(
#     test_r1
# )

# assert math.isclose(
#     distance,
#     5.0
# )

# assert math.isclose(
#     test_r1.distance_to(test_alice),
#     distance
# )

# print(
#     "[PASS] Euclidean node distance is correct and symmetric."
# )


# # =============================================================
# # Test 5 — Swapping requires at least two memory slots
# # =============================================================

# swap_capacity_error_raised = False

# try:
#     NetworkNode(
#         name="Bad-Repeater",
#         x_km=0,
#         y_km=0,
#         endpoint_capable=False,
#         swapping_capable=True,
#         memory_capacity=1
#     )

# except ValueError:
#     swap_capacity_error_raised = True

# assert swap_capacity_error_raised

# print(
#     "[PASS] Swapping node with insufficient memory is rejected."
# )


# # =============================================================
# # Test 6 — Node with no usable quantum-network role rejected
# # =============================================================

# role_error_raised = False

# try:
#     NetworkNode(
#         name="Useless-Node",
#         x_km=0,
#         y_km=0,
#         endpoint_capable=False,
#         swapping_capable=False,
#         memory_capacity=1
#     )

# except ValueError:
#     role_error_raised = True

# assert role_error_raised

# print(
#     "[PASS] Node with no supported network role is rejected."
# )


# # =============================================================
# # Test 7 — Invalid coordinates rejected
# # =============================================================

# coordinate_error_raised = False

# try:
#     NetworkNode(
#         name="Invalid-Position",
#         x_km=float("inf"),
#         y_km=0,
#         endpoint_capable=True,
#         swapping_capable=False,
#         memory_capacity=1
#     )

# except ValueError:
#     coordinate_error_raised = True

# assert coordinate_error_raised

# print(
#     "[PASS] Non-finite node coordinates are rejected."
# )


# # =============================================================
# # Test 8 — Physical node descriptor is immutable
# # =============================================================

# immutability_error_raised = False

# try:
#     test_alice.x_km = 999

# except FrozenInstanceError:
#     immutability_error_raised = True

# assert immutability_error_raised

# print(
#     "[PASS] NetworkNode physical identity is immutable."
# )


# # =============================================================
# # Summary
# # =============================================================

# print("\nTEST NODES")
# print(test_alice)
# print(test_r1)
# print(test_hybrid)


# print("\n" + "=" * 76)
# print("Verification completed successfully.")
# print("=" * 76)

In [ ]:
# @title Quantum Link Redefined

from dataclasses import dataclass, field
import math
import numbers


# Preserve the already validated Phase-1 fibre attenuation baseline
# without redefining or modifying the Phase 0–2 constant.
PHASE3_DEFAULT_FIBER_ATTENUATION_DB_PER_KM = float(
    STANDARD_FIBER_ATTENUATION
)


@dataclass(frozen=True)
class PhysicalQuantumLink:
    """
    Immutable physical fibre-link descriptor for reconstructed
    NARAD-QC Phase 3.

    The link represents only physical topology and deterministic
    attenuation properties.

    It deliberately does NOT:
    - generate Bell pairs;
    - perform stochastic transmission attempts;
    - own an RNG;
    - maintain runtime attempt/success counters;
    - store session reservations;
    - determine control-plane availability;
    - perform QKD or entanglement swapping.

    Those responsibilities belong to later simulation components.
    """

    node_a: NetworkNode
    node_b: NetworkNode

    attenuation_db_per_km: float = (
        PHASE3_DEFAULT_FIBER_ATTENUATION_DB_PER_KM
    )

    # Derived immutable fields
    link_id: str = field(init=False)
    distance_km: float = field(init=False)

    def __post_init__(self):

        # =====================================================
        # Endpoint validation
        # =====================================================

        if not isinstance(self.node_a, NetworkNode):
            raise TypeError(
                "node_a must be a NetworkNode."
            )

        if not isinstance(self.node_b, NetworkNode):
            raise TypeError(
                "node_b must be a NetworkNode."
            )

        if self.node_a.name == self.node_b.name:
            raise ValueError(
                "A physical quantum link cannot connect "
                "a node to itself."
            )

        # =====================================================
        # Attenuation validation
        # =====================================================

        if (
            isinstance(self.attenuation_db_per_km, bool)
            or not isinstance(
                self.attenuation_db_per_km,
                numbers.Real
            )
            or not math.isfinite(
                float(self.attenuation_db_per_km)
            )
        ):
            raise ValueError(
                "attenuation_db_per_km must be a finite "
                "real number."
            )

        attenuation = float(
            self.attenuation_db_per_km
        )

        if attenuation < 0:
            raise ValueError(
                "attenuation_db_per_km cannot be negative."
            )

        object.__setattr__(
            self,
            "attenuation_db_per_km",
            attenuation
        )

        # =====================================================
        # Canonical undirected identity
        # =====================================================

        endpoint_names = sorted(
            (
                self.node_a.name,
                self.node_b.name
            )
        )

        object.__setattr__(
            self,
            "link_id",
            f"{endpoint_names[0]}--{endpoint_names[1]}"
        )

        # =====================================================
        # Physical distance
        # =====================================================

        distance = self.node_a.distance_to(
            self.node_b
        )

        object.__setattr__(
            self,
            "distance_km",
            float(distance)
        )

    # =========================================================
    # Physical Attenuation Model
    # =========================================================

    def attenuation_db_for_distance(
        self,
        distance_km: float
    ) -> float:
        """
        Total attenuation in dB over a supplied fibre distance.
        """

        distance_km = self._validate_distance(
            distance_km
        )

        return (
            self.attenuation_db_per_km
            * distance_km
        )

    def transmission_probability_for_distance(
        self,
        distance_km: float
    ) -> float:
        """
        Photon survival/transmission probability over a supplied
        fibre distance using the Phase-1 attenuation relation:

            eta(d) = 10^(-alpha*d / 10)

        This method is deterministic. It does NOT perform a
        stochastic transmission attempt.
        """

        total_attenuation_db = (
            self.attenuation_db_for_distance(
                distance_km
            )
        )

        return 10.0 ** (
            -total_attenuation_db / 10.0
        )

    @property
    def total_attenuation_db(self) -> float:
        """
        Total attenuation across the complete physical link.
        """

        return self.attenuation_db_for_distance(
            self.distance_km
        )

    @property
    def survival_probability(self) -> float:
        """
        Photon survival probability across the complete link.
        """

        return (
            self.transmission_probability_for_distance(
                self.distance_km
            )
        )

    @property
    def loss_probability(self) -> float:
        """
        Photon loss probability across the complete link.
        """

        return 1.0 - self.survival_probability

    def transmission_probability_for_fraction(
        self,
        fraction: float
    ) -> float:
        """
        Photon transmission probability across a fraction of the
        physical link.

        Example:
            fraction = 0.5

        gives the probability across one half of the fibre.

        This is useful for future physical models such as
        midpoint-heralded entanglement generation, without
        coupling that protocol to the link itself.
        """

        if (
            isinstance(fraction, bool)
            or not isinstance(fraction, numbers.Real)
            or not math.isfinite(float(fraction))
        ):
            raise ValueError(
                "fraction must be a finite real number."
            )

        fraction = float(fraction)

        if not 0.0 <= fraction <= 1.0:
            raise ValueError(
                "fraction must lie within [0, 1]."
            )

        return (
            self.transmission_probability_for_distance(
                self.distance_km * fraction
            )
        )

    # =========================================================
    # Endpoint Helpers
    # =========================================================

    def connects(
        self,
        node: NetworkNode
    ) -> bool:
        """
        Return True when the supplied node is one of this
        physical link's endpoints.
        """

        if not isinstance(node, NetworkNode):
            raise TypeError(
                "connects() requires a NetworkNode."
            )

        return (
            node.name == self.node_a.name
            or node.name == self.node_b.name
        )

    def other_endpoint(
        self,
        node: NetworkNode
    ) -> NetworkNode:
        """
        Return the opposite endpoint of this undirected link.
        """

        if not isinstance(node, NetworkNode):
            raise TypeError(
                "other_endpoint() requires a NetworkNode."
            )

        if node.name == self.node_a.name:
            return self.node_b

        if node.name == self.node_b.name:
            return self.node_a

        raise ValueError(
            f"Node '{node.name}' is not an endpoint "
            f"of link '{self.link_id}'."
        )

    # =========================================================
    # Internal Validation
    # =========================================================

    @staticmethod
    def _validate_distance(
        distance_km: float
    ) -> float:

        if (
            isinstance(distance_km, bool)
            or not isinstance(
                distance_km,
                numbers.Real
            )
            or not math.isfinite(
                float(distance_km)
            )
        ):
            raise ValueError(
                "distance_km must be a finite real number."
            )

        distance_km = float(distance_km)

        if distance_km < 0:
            raise ValueError(
                "distance_km cannot be negative."
            )

        return distance_km

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:
        return (
            "PhysicalQuantumLink("
            f"id='{self.link_id}', "
            f"distance={self.distance_km:.2f} km, "
            f"attenuation="
            f"{self.attenuation_db_per_km:.3f} dB/km, "
            f"survival="
            f"{self.survival_probability:.6f}"
            ")"
        )

In [ ]:
# @title Physical Quantum Link Verification

# from dataclasses import FrozenInstanceError


# print("=" * 76)
# print("NARAD-QC Phase 3A.3 — Physical Quantum Link Verification")
# print("=" * 76)


# # =============================================================
# # Test nodes
# # =============================================================

# test_link_node_a = NetworkNode(
#     name="Link-A",
#     x_km=0,
#     y_km=0,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=1
# )

# test_link_node_b = NetworkNode(
#     name="Link-B",
#     x_km=30,
#     y_km=40,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )


# # =============================================================
# # Test 1 — Link construction and Euclidean distance
# # =============================================================

# test_link = PhysicalQuantumLink(
#     node_a=test_link_node_a,
#     node_b=test_link_node_b,
#     attenuation_db_per_km=0.2
# )

# assert test_link.link_id == "Link-A--Link-B"

# assert math.isclose(
#     test_link.distance_km,
#     50.0
# )

# print(
#     "[PASS] Physical link identity and Euclidean distance "
#     "are correct."
# )


# # =============================================================
# # Test 2 — Phase-1 attenuation relation
# #
# # alpha = 0.2 dB/km
# # d     = 50 km
# #
# # attenuation = 10 dB
# # eta = 10^(-10/10) = 0.1
# # =============================================================

# assert math.isclose(
#     test_link.total_attenuation_db,
#     10.0,
#     rel_tol=1e-12
# )

# assert math.isclose(
#     test_link.survival_probability,
#     0.1,
#     rel_tol=1e-12
# )

# assert math.isclose(
#     test_link.loss_probability,
#     0.9,
#     rel_tol=1e-12
# )

# print(
#     "[PASS] Phase-1 fibre attenuation equation is "
#     "preserved exactly."
# )


# # =============================================================
# # Test 3 — Half-link transmission probability
# #
# # 25 km at 0.2 dB/km -> 5 dB
# # eta = 10^(-5/10) ~= 0.316227766
# # =============================================================

# half_link_probability = (
#     test_link.transmission_probability_for_fraction(
#         0.5
#     )
# )

# expected_half_probability = 10 ** (-0.5)

# assert math.isclose(
#     half_link_probability,
#     expected_half_probability,
#     rel_tol=1e-12
# )

# print(
#     "[PASS] Fractional/half-link transmission probability "
#     "is correct."
# )


# # =============================================================
# # Test 4 — Undirected physical identity
# # =============================================================

# reversed_test_link = PhysicalQuantumLink(
#     node_a=test_link_node_b,
#     node_b=test_link_node_a,
#     attenuation_db_per_km=0.2
# )

# assert (
#     reversed_test_link.link_id
#     == test_link.link_id
# )

# assert math.isclose(
#     reversed_test_link.distance_km,
#     test_link.distance_km
# )

# print(
#     "[PASS] Link identity is canonical and undirected."
# )


# # =============================================================
# # Test 5 — Endpoint helpers
# # =============================================================

# assert test_link.connects(
#     test_link_node_a
# )

# assert test_link.connects(
#     test_link_node_b
# )

# assert (
#     test_link.other_endpoint(test_link_node_a)
#     is test_link_node_b
# )

# assert (
#     test_link.other_endpoint(test_link_node_b)
#     is test_link_node_a
# )

# print(
#     "[PASS] Endpoint helper operations behave correctly."
# )


# # =============================================================
# # Test 6 — Negative attenuation rejected
# # =============================================================

# negative_attenuation_error = False

# try:
#     PhysicalQuantumLink(
#         node_a=test_link_node_a,
#         node_b=test_link_node_b,
#         attenuation_db_per_km=-0.1
#     )

# except ValueError:
#     negative_attenuation_error = True

# assert negative_attenuation_error

# print(
#     "[PASS] Negative fibre attenuation is rejected."
# )


# # =============================================================
# # Test 7 — Self-links rejected
# # =============================================================

# self_link_error = False

# try:
#     PhysicalQuantumLink(
#         node_a=test_link_node_a,
#         node_b=test_link_node_a
#     )

# except ValueError:
#     self_link_error = True

# assert self_link_error

# print(
#     "[PASS] Physical self-links are rejected."
# )


# # =============================================================
# # Test 8 — Invalid fractional distance rejected
# # =============================================================

# fraction_error = False

# try:
#     test_link.transmission_probability_for_fraction(
#         1.5
#     )

# except ValueError:
#     fraction_error = True

# assert fraction_error

# print(
#     "[PASS] Invalid link-distance fractions are rejected."
# )


# # =============================================================
# # Test 9 — Physical link descriptor is immutable
# # =============================================================

# immutability_error = False

# try:
#     test_link.distance_km = 999

# except FrozenInstanceError:
#     immutability_error = True

# assert immutability_error

# print(
#     "[PASS] PhysicalQuantumLink descriptor is immutable."
# )


# # =============================================================
# # Summary
# # =============================================================

# print("\nTEST LINK")
# print(test_link)

# print(
#     "\nFull-link survival probability : "
#     f"{test_link.survival_probability:.6f}"
# )

# print(
#     "Half-link survival probability : "
#     f"{half_link_probability:.6f}"
# )


# print("\n" + "=" * 76)
# print("Verification completed successfully.")
# print("=" * 76)

In [ ]:
# @title Quantum Memory Errors and Local Entanglement Half

from dataclasses import dataclass
from typing import Optional
import numbers


# =============================================================
# Dedicated Memory Errors
# =============================================================

class QuantumMemoryError(Exception):
    """Base exception for reconstructed Phase-3 quantum memory."""
    pass


class QuantumMemoryFullError(QuantumMemoryError):
    """Raised when no free quantum-memory slot exists."""
    pass


class QuantumMemoryDuplicateResourceError(QuantumMemoryError):
    """
    Raised when one memory attempts to store two local halves
    associated with the same global entanglement resource.
    """
    pass


class QuantumMemoryResourceNotFoundError(QuantumMemoryError):
    """
    Raised when a requested resource is not present in the memory.
    """
    pass


# =============================================================
# Local Stored Quantum Half
# =============================================================

@dataclass(frozen=True)
class LocalEntanglementHalf:
    """
    Immutable local reference to one half of a global
    entanglement resource.

    Important:
    This object is NOT the Bell pair / entanglement resource itself.

    The global entanglement resource will be introduced later and
    will refer to local halves stored at its two endpoint memories.
    """

    resource_id: str
    remote_node_name: str
    stored_at_slot: int

    def __post_init__(self):

        if (
            not isinstance(self.resource_id, str)
            or not self.resource_id.strip()
        ):
            raise ValueError(
                "resource_id must be a non-empty string."
            )

        if (
            not isinstance(self.remote_node_name, str)
            or not self.remote_node_name.strip()
        ):
            raise ValueError(
                "remote_node_name must be a non-empty string."
            )

        if (
            isinstance(self.stored_at_slot, bool)
            or not isinstance(
                self.stored_at_slot,
                numbers.Integral
            )
            or self.stored_at_slot < 0
        ):
            raise ValueError(
                "stored_at_slot must be a non-negative integer."
            )

        object.__setattr__(
            self,
            "resource_id",
            self.resource_id.strip()
        )

        object.__setattr__(
            self,
            "remote_node_name",
            self.remote_node_name.strip()
        )

        object.__setattr__(
            self,
            "stored_at_slot",
            int(self.stored_at_slot)
        )

In [ ]:
# @title Quantum Memory Runtime State

class QuantumMemory:
    """
    Runtime quantum-memory state belonging to one NetworkNode.

    The memory stores local halves of global entanglement resources.

    Responsibilities
    ----------------
    - provide fixed-capacity local quantum storage;
    - track which resource occupies each local slot;
    - record storage time using SimulationKernel slots;
    - compute local qubit age;
    - identify locally expired resources;
    - emit memory events.

    The memory deliberately does NOT:
    - create or destroy global entanglement resources;
    - perform swapping;
    - independently invalidate the remote half of a pair;
    - manage sessions or routing;
    - model continuous fidelity decay.

    Global resource lifecycle will later belong to the
    EntanglementManager.
    """

    def __init__(
        self,
        owner: NetworkNode,
        kernel: SimulationKernel,
        max_storage_slots: Optional[int] = None
    ):

        # =====================================================
        # Owner / Kernel Validation
        # =====================================================

        if not isinstance(owner, NetworkNode):
            raise TypeError(
                "owner must be a NetworkNode."
            )

        if not isinstance(kernel, SimulationKernel):
            raise TypeError(
                "kernel must be a SimulationKernel."
            )

        # =====================================================
        # Storage-Lifetime Validation
        # =====================================================

        if max_storage_slots is not None:

            if (
                isinstance(max_storage_slots, bool)
                or not isinstance(
                    max_storage_slots,
                    numbers.Integral
                )
                or max_storage_slots <= 0
            ):
                raise ValueError(
                    "max_storage_slots must be a positive "
                    "integer or None."
                )

            max_storage_slots = int(
                max_storage_slots
            )

        self.owner = owner
        self.kernel = kernel

        self.max_storage_slots = (
            max_storage_slots
        )

        # Capacity comes from physical node hardware.
        self._slots: list[
            Optional[LocalEntanglementHalf]
        ] = [
            None
            for _ in range(
                self.owner.memory_capacity
            )
        ]

        self.kernel.emit(
            event_type="MEMORY_INITIALIZED",
            component="QuantumMemory",
            node_names=(self.owner.name,),
            message=(
                f"Quantum memory initialized at "
                f"{self.owner.name} with "
                f"{self.capacity} slot(s)."
            ),
            metadata={
                "capacity": self.capacity,
                "max_storage_slots":
                    self.max_storage_slots
            }
        )

    # =========================================================
    # Basic Properties
    # =========================================================

    @property
    def capacity(self) -> int:
        return self.owner.memory_capacity

    @property
    def occupancy(self) -> int:
        return sum(
            half is not None
            for half in self._slots
        )

    @property
    def free_slots(self) -> int:
        return (
            self.capacity
            - self.occupancy
        )

    @property
    def is_full(self) -> bool:
        return self.occupancy >= self.capacity

    @property
    def is_empty(self) -> bool:
        return self.occupancy == 0

    # =========================================================
    # Slot Identity
    # =========================================================

    def slot_id(
        self,
        slot_index: int
    ) -> str:

        self._validate_slot_index(
            slot_index
        )

        return (
            f"{self.owner.name}:M"
            f"{slot_index}"
        )

    # =========================================================
    # Lookup Helpers
    # =========================================================

    def contains(
        self,
        resource_id: str
    ) -> bool:

        return (
            self.find_slot(resource_id)
            is not None
        )

    def find_slot(
        self,
        resource_id: str
    ) -> Optional[int]:

        resource_id = (
            self._validate_resource_id(
                resource_id
            )
        )

        for index, half in enumerate(
            self._slots
        ):
            if (
                half is not None
                and half.resource_id
                == resource_id
            ):
                return index

        return None

    def get_local_half(
        self,
        resource_id: str
    ) -> LocalEntanglementHalf:

        slot_index = self.find_slot(
            resource_id
        )

        if slot_index is None:
            raise QuantumMemoryResourceNotFoundError(
                f"Resource '{resource_id}' is not stored "
                f"in memory at '{self.owner.name}'."
            )

        return self._slots[slot_index]

    # =========================================================
    # Storage
    # =========================================================

    def store_local_half(
        self,
        resource_id: str,
        remote_node: NetworkNode
    ) -> int:
        """
        Store one local half of a global entanglement resource.

        Returns
        -------
        int
            Slot index used for storage.

        Note:
        This method only stores the local reference.
        It does not create the corresponding global resource.
        """

        resource_id = (
            self._validate_resource_id(
                resource_id
            )
        )

        if not isinstance(
            remote_node,
            NetworkNode
        ):
            raise TypeError(
                "remote_node must be a NetworkNode."
            )

        if (
            remote_node.name
            == self.owner.name
        ):
            raise ValueError(
                "A local entanglement half cannot have "
                "its own node as the remote endpoint."
            )

        if self.contains(resource_id):
            raise QuantumMemoryDuplicateResourceError(
                f"Resource '{resource_id}' is already "
                f"stored at '{self.owner.name}'."
            )

        if self.is_full:
            raise QuantumMemoryFullError(
                f"Quantum memory at '{self.owner.name}' "
                f"is full."
            )

        slot_index = next(
            index
            for index, half
            in enumerate(self._slots)
            if half is None
        )

        stored_half = LocalEntanglementHalf(
            resource_id=resource_id,
            remote_node_name=remote_node.name,
            stored_at_slot=(
                self.kernel.current_slot
            )
        )

        self._slots[
            slot_index
        ] = stored_half

        self.kernel.emit(
            event_type="MEMORY_HALF_STORED",
            component="QuantumMemory",
            node_names=(
                self.owner.name,
                remote_node.name
            ),
            resource_ids=(resource_id,),
            message=(
                f"Local half of resource "
                f"'{resource_id}' stored in "
                f"{self.slot_id(slot_index)}."
            ),
            metadata={
                "slot_index": slot_index,
                "slot_id":
                    self.slot_id(slot_index),
                "stored_at_slot":
                    self.kernel.current_slot,
                "remote_node":
                    remote_node.name
            }
        )

        return slot_index

    # =========================================================
    # Removal
    # =========================================================

    def remove_local_half(
        self,
        resource_id: str,
        reason: str = "released"
    ) -> LocalEntanglementHalf:
        """
        Remove one local half from this memory.

        Important:
        This removes only the LOCAL memory reference.

        Later, the EntanglementManager will coordinate atomic
        removal of both endpoint halves when a global resource
        is consumed, expired, discarded, or destroyed.
        """

        slot_index = self.find_slot(
            resource_id
        )

        if slot_index is None:
            raise QuantumMemoryResourceNotFoundError(
                f"Resource '{resource_id}' is not stored "
                f"in memory at '{self.owner.name}'."
            )

        half = self._slots[
            slot_index
        ]

        self._slots[
            slot_index
        ] = None

        self.kernel.emit(
            event_type="MEMORY_HALF_REMOVED",
            component="QuantumMemory",
            node_names=(self.owner.name,),
            resource_ids=(
                half.resource_id,
            ),
            message=(
                f"Local half of resource "
                f"'{half.resource_id}' removed from "
                f"{self.slot_id(slot_index)}."
            ),
            metadata={
                "slot_index": slot_index,
                "slot_id":
                    self.slot_id(slot_index),
                "reason": str(reason)
            }
        )

        return half

    # =========================================================
    # Age / Expiration Inspection
    # =========================================================

    def age_slots(
        self,
        resource_id: str
    ) -> int:
        """
        Return current local storage age in simulation slots.

        Age is derived from the simulation clock rather than
        incremented manually.
        """

        half = self.get_local_half(
            resource_id
        )

        age = (
            self.kernel.current_slot
            - half.stored_at_slot
        )

        if age < 0:
            raise RuntimeError(
                "Simulation clock precedes resource "
                "storage slot."
            )

        return age

    def is_locally_expired(
        self,
        resource_id: str
    ) -> bool:
        """
        Determine whether the local qubit has exceeded the
        configured hard storage limit.

        This method does NOT remove the resource.
        """

        if self.max_storage_slots is None:
            return False

        return (
            self.age_slots(resource_id)
            >= self.max_storage_slots
        )

    def expired_resource_ids(
        self
    ) -> tuple[str, ...]:
        """
        Return locally expired resource IDs without mutating memory.
        """

        expired = []

        for half in self._slots:

            if (
                half is not None
                and self.is_locally_expired(
                    half.resource_id
                )
            ):
                expired.append(
                    half.resource_id
                )

        return tuple(expired)

    # =========================================================
    # Snapshot / Inspection
    # =========================================================

    def slots_dataframe(
        self
    ) -> pd.DataFrame:
        """
        Return a read-only dataframe snapshot of the current
        memory-slot state.
        """

        rows = []

        for index, half in enumerate(
            self._slots
        ):

            if half is None:

                rows.append({
                    "Owner":
                        self.owner.name,

                    "Slot Index":
                        index,

                    "Slot ID":
                        self.slot_id(index),

                    "Occupied":
                        False,

                    "Resource ID":
                        None,

                    "Remote Node":
                        None,

                    "Stored At Slot":
                        None,

                    "Age (slots)":
                        None,

                    "Locally Expired":
                        False
                })

                continue

            rows.append({
                "Owner":
                    self.owner.name,

                "Slot Index":
                    index,

                "Slot ID":
                    self.slot_id(index),

                "Occupied":
                    True,

                "Resource ID":
                    half.resource_id,

                "Remote Node":
                    half.remote_node_name,

                "Stored At Slot":
                    half.stored_at_slot,

                "Age (slots)":
                    self.age_slots(
                        half.resource_id
                    ),

                "Locally Expired":
                    self.is_locally_expired(
                        half.resource_id
                    )
            })

        return pd.DataFrame(
            rows
        )

    def summary(self) -> dict:
        """
        Compact runtime memory summary.
        """

        return {
            "owner":
                self.owner.name,

            "capacity":
                self.capacity,

            "occupancy":
                self.occupancy,

            "free_slots":
                self.free_slots,

            "max_storage_slots":
                self.max_storage_slots,

            "expired_local_resources":
                len(
                    self.expired_resource_ids()
                )
        }

    # =========================================================
    # Internal Validation
    # =========================================================

    @staticmethod
    def _validate_resource_id(
        resource_id: str
    ) -> str:

        if (
            not isinstance(resource_id, str)
            or not resource_id.strip()
        ):
            raise ValueError(
                "resource_id must be a non-empty string."
            )

        return resource_id.strip()

    def _validate_slot_index(
        self,
        slot_index: int
    ) -> int:

        if (
            isinstance(slot_index, bool)
            or not isinstance(
                slot_index,
                numbers.Integral
            )
        ):
            raise TypeError(
                "slot_index must be an integer."
            )

        slot_index = int(
            slot_index
        )

        if not (
            0 <= slot_index < self.capacity
        ):
            raise IndexError(
                f"slot_index must lie within "
                f"[0, {self.capacity - 1}]."
            )

        return slot_index

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:

        lifetime_text = (
            "unbounded"
            if self.max_storage_slots is None
            else (
                f"{self.max_storage_slots} slots"
            )
        )

        return (
            "QuantumMemory("
            f"owner='{self.owner.name}', "
            f"occupancy={self.occupancy}/"
            f"{self.capacity}, "
            f"max_storage={lifetime_text}"
            ")"
        )

In [ ]:
# @title Quantum Memory Verification

# from dataclasses import FrozenInstanceError


# print("=" * 76)
# print("NARAD-QC - Quantum Memory Verification")
# print("=" * 76)


# # =============================================================
# # Independent test simulation
# # =============================================================

# memory_test_kernel = SimulationKernel(
#     seed=20260909
# )


# # =============================================================
# # Test nodes
# # =============================================================

# memory_test_alice = NetworkNode(
#     name="Memory-Alice",
#     x_km=0,
#     y_km=0,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=1
# )

# memory_test_r1 = NetworkNode(
#     name="Memory-R1",
#     x_km=20,
#     y_km=0,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )

# memory_test_r2 = NetworkNode(
#     name="Memory-R2",
#     x_km=40,
#     y_km=0,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )


# # =============================================================
# # Test 1 — Memory capacity comes from node hardware
# # =============================================================

# alice_memory = QuantumMemory(
#     owner=memory_test_alice,
#     kernel=memory_test_kernel,
#     max_storage_slots=3
# )

# r1_memory = QuantumMemory(
#     owner=memory_test_r1,
#     kernel=memory_test_kernel,
#     max_storage_slots=3
# )

# assert alice_memory.capacity == 1
# assert r1_memory.capacity == 2

# assert alice_memory.is_empty
# assert r1_memory.is_empty

# print(
#     "[PASS] Memory capacity is derived from node hardware."
# )


# # =============================================================
# # Test 2 — Store one local half
# # =============================================================

# slot_0 = r1_memory.store_local_half(
#     resource_id="TEST-E-001",
#     remote_node=memory_test_alice
# )

# assert slot_0 == 0
# assert r1_memory.occupancy == 1
# assert r1_memory.free_slots == 1
# assert r1_memory.contains(
#     "TEST-E-001"
# )

# print(
#     "[PASS] Local entanglement half is stored correctly."
# )


# # =============================================================
# # Test 3 — Same global resource cannot occupy two local slots
# # =============================================================

# duplicate_error = False

# try:
#     r1_memory.store_local_half(
#         resource_id="TEST-E-001",
#         remote_node=memory_test_alice
#     )

# except QuantumMemoryDuplicateResourceError:
#     duplicate_error = True

# assert duplicate_error

# print(
#     "[PASS] Duplicate local storage of one resource is rejected."
# )


# # =============================================================
# # Test 4 — Fill second repeater memory slot
# # =============================================================

# slot_1 = r1_memory.store_local_half(
#     resource_id="TEST-E-002",
#     remote_node=memory_test_r2
# )

# assert slot_1 == 1
# assert r1_memory.occupancy == 2
# assert r1_memory.is_full

# print(
#     "[PASS] Multiple independent local halves can occupy "
#     "different memory slots."
# )


# # =============================================================
# # Test 5 — Full memory rejects another local half
# # =============================================================

# full_error = False

# try:
#     r1_memory.store_local_half(
#         resource_id="TEST-E-003",
#         remote_node=memory_test_r2
#     )

# except QuantumMemoryFullError:
#     full_error = True

# assert full_error

# print(
#     "[PASS] Full quantum memory rejects additional storage."
# )


# # =============================================================
# # Test 6 — Age is derived from simulation time
# # =============================================================

# assert (
#     r1_memory.age_slots(
#         "TEST-E-001"
#     )
#     == 0
# )

# memory_test_kernel.advance_slot()

# assert (
#     r1_memory.age_slots(
#         "TEST-E-001"
#     )
#     == 1
# )

# memory_test_kernel.advance_slot()

# assert (
#     r1_memory.age_slots(
#         "TEST-E-001"
#     )
#     == 2
# )

# print(
#     "[PASS] Local qubit age is derived correctly "
#     "from the simulation clock."
# )


# # =============================================================
# # Test 7 — Expiration boundary is age >= lifetime
# # =============================================================

# assert not r1_memory.is_locally_expired(
#     "TEST-E-001"
# )

# memory_test_kernel.advance_slot()

# assert (
#     memory_test_kernel.current_slot
#     == 3
# )

# assert r1_memory.is_locally_expired(
#     "TEST-E-001"
# )

# assert r1_memory.is_locally_expired(
#     "TEST-E-002"
# )

# print(
#     "[PASS] Local expiration occurs exactly at "
#     "age >= max_storage_slots."
# )


# # =============================================================
# # Test 8 — Expiration detection does NOT mutate memory
# # =============================================================

# expired_ids = (
#     r1_memory.expired_resource_ids()
# )

# assert set(expired_ids) == {
#     "TEST-E-001",
#     "TEST-E-002"
# }

# assert r1_memory.occupancy == 2
# assert r1_memory.is_full

# print(
#     "[PASS] Expiration detection is non-destructive; "
#     "global cleanup remains a later responsibility."
# )


# # =============================================================
# # Test 9 — Remove one local half explicitly
# # =============================================================

# removed_half = (
#     r1_memory.remove_local_half(
#         resource_id="TEST-E-001",
#         reason="verification"
#     )
# )

# assert (
#     removed_half.resource_id
#     == "TEST-E-001"
# )

# assert r1_memory.occupancy == 1
# assert r1_memory.free_slots == 1
# assert not r1_memory.contains(
#     "TEST-E-001"
# )

# print(
#     "[PASS] Local half removal updates memory correctly."
# )


# # =============================================================
# # Test 10 — Local half record is immutable
# # =============================================================

# immutability_error = False

# try:
#     removed_half.resource_id = "CHANGED"

# except FrozenInstanceError:
#     immutability_error = True

# assert immutability_error

# print(
#     "[PASS] Stored local-half records are immutable."
# )


# # =============================================================
# # Test 11 — Endpoint quantum memory is explicitly represented
# # =============================================================

# alice_memory.store_local_half(
#     resource_id="TEST-E-ENDPOINT",
#     remote_node=memory_test_r1
# )

# assert alice_memory.occupancy == 1
# assert alice_memory.is_full

# print(
#     "[PASS] Endpoint quantum memory is explicitly represented."
# )


# # =============================================================
# # Memory snapshots
# # =============================================================

# print("\nR1 MEMORY STATE")

# display(
#     r1_memory.slots_dataframe()
# )


# print("\nALICE MEMORY STATE")

# display(
#     alice_memory.slots_dataframe()
# )


# print("\nMEMORY TEST EVENT LOG")

# display(
#     memory_test_kernel.events_dataframe()
# )


# print("\n" + "=" * 76)
# print("Verification completed successfully.")
# print("=" * 76)

In [ ]:
# @title Global Entanglement Resource Model

from dataclasses import dataclass, replace
from enum import Enum
from typing import Optional
import numbers


# =============================================================
# Entanglement Resource Types
# =============================================================

class EntanglementKind(str, Enum):
    """
    Physical origin/type of an entanglement resource.
    """

    ELEMENTARY = "ELEMENTARY"
    SWAPPED = "SWAPPED"


class EntanglementStatus(str, Enum):
    """
    Lifecycle state of a global entanglement resource.
    """

    ACTIVE = "ACTIVE"

    # Included now for the future swapping / teleportation layer.
    CONSUMED = "CONSUMED"

    EXPIRED = "EXPIRED"
    DISCARDED = "DISCARDED"


# =============================================================
# Entanglement Manager Errors
# =============================================================

class EntanglementManagerError(Exception):
    """Base exception for the global entanglement manager."""
    pass


class EntanglementResourceNotFoundError(
    EntanglementManagerError
):
    """Raised when a requested global resource does not exist."""
    pass


class EntanglementResourceStateError(
    EntanglementManagerError
):
    """Raised when an operation is invalid for a resource state."""
    pass


class EntanglementMemoryNotRegisteredError(
    EntanglementManagerError
):
    """Raised when a resource endpoint has no registered memory."""
    pass


class EntanglementConsistencyError(
    EntanglementManagerError
):
    """
    Raised when global resource state and endpoint-memory state
    disagree.
    """
    pass


# =============================================================
# Immutable Global Entanglement Resource
# =============================================================

@dataclass(frozen=True)
class EntanglementResource:
    """
    Immutable global description of one entanglement relationship.

    Important:
    This object represents the GLOBAL quantum relationship.

    The physical local qubit halves are represented separately by
    LocalEntanglementHalf records in endpoint QuantumMemory objects.

    A resource may later represent either:
    - elementary entanglement over one physical link; or
    - longer-span entanglement created by swapping.
    """

    resource_id: str

    endpoint_a_name: str
    endpoint_b_name: str

    kind: EntanglementKind

    created_slot: int

    source_link_ids: tuple[str, ...]

    parent_resource_ids: tuple[str, ...] = ()
    swap_node_name: Optional[str] = None

    status: EntanglementStatus = (
        EntanglementStatus.ACTIVE
    )

    ended_slot: Optional[int] = None
    termination_reason: Optional[str] = None

    def __post_init__(self):

        # =====================================================
        # Basic identity validation
        # =====================================================

        if (
            not isinstance(self.resource_id, str)
            or not self.resource_id.strip()
        ):
            raise ValueError(
                "resource_id must be a non-empty string."
            )

        if (
            not isinstance(self.endpoint_a_name, str)
            or not self.endpoint_a_name.strip()
        ):
            raise ValueError(
                "endpoint_a_name must be a non-empty string."
            )

        if (
            not isinstance(self.endpoint_b_name, str)
            or not self.endpoint_b_name.strip()
        ):
            raise ValueError(
                "endpoint_b_name must be a non-empty string."
            )

        resource_id = self.resource_id.strip()

        endpoint_names = sorted(
            (
                self.endpoint_a_name.strip(),
                self.endpoint_b_name.strip()
            )
        )

        if endpoint_names[0] == endpoint_names[1]:
            raise ValueError(
                "An entanglement resource requires "
                "two distinct endpoint nodes."
            )

        object.__setattr__(
            self,
            "resource_id",
            resource_id
        )

        # Entanglement itself is an undirected relationship.
        object.__setattr__(
            self,
            "endpoint_a_name",
            endpoint_names[0]
        )

        object.__setattr__(
            self,
            "endpoint_b_name",
            endpoint_names[1]
        )

        # =====================================================
        # Slot validation
        # =====================================================

        if (
            isinstance(self.created_slot, bool)
            or not isinstance(
                self.created_slot,
                numbers.Integral
            )
            or self.created_slot < 0
        ):
            raise ValueError(
                "created_slot must be a non-negative integer."
            )

        object.__setattr__(
            self,
            "created_slot",
            int(self.created_slot)
        )

        # =====================================================
        # Enum validation
        # =====================================================

        if not isinstance(
            self.kind,
            EntanglementKind
        ):
            raise TypeError(
                "kind must be an EntanglementKind."
            )

        if not isinstance(
            self.status,
            EntanglementStatus
        ):
            raise TypeError(
                "status must be an EntanglementStatus."
            )

        # =====================================================
        # Provenance validation
        # =====================================================

        source_links = tuple(
            str(link_id).strip()
            for link_id in self.source_link_ids
        )

        if (
            not source_links
            or any(
                not link_id
                for link_id in source_links
            )
        ):
            raise ValueError(
                "source_link_ids must contain at least "
                "one non-empty physical link ID."
            )

        if len(set(source_links)) != len(source_links):
            raise ValueError(
                "source_link_ids cannot contain duplicates."
            )

        object.__setattr__(
            self,
            "source_link_ids",
            source_links
        )

        parents = tuple(
            str(parent_id).strip()
            for parent_id
            in self.parent_resource_ids
        )

        if any(
            not parent_id
            for parent_id in parents
        ):
            raise ValueError(
                "parent_resource_ids cannot contain "
                "empty identifiers."
            )

        if len(set(parents)) != len(parents):
            raise ValueError(
                "parent_resource_ids cannot contain duplicates."
            )

        object.__setattr__(
            self,
            "parent_resource_ids",
            parents
        )

        # =====================================================
        # Type-specific invariants
        # =====================================================

        if self.kind == EntanglementKind.ELEMENTARY:

            if len(self.source_link_ids) != 1:
                raise ValueError(
                    "An elementary entanglement resource "
                    "must originate from exactly one "
                    "physical link."
                )

            if self.parent_resource_ids:
                raise ValueError(
                    "An elementary resource cannot have "
                    "parent entanglement resources."
                )

            if self.swap_node_name is not None:
                raise ValueError(
                    "An elementary resource cannot have "
                    "a swap node."
                )

        elif self.kind == EntanglementKind.SWAPPED:

            if len(self.parent_resource_ids) != 2:
                raise ValueError(
                    "A swapped resource must have exactly "
                    "two parent resources."
                )

            if (
                not isinstance(self.swap_node_name, str)
                or not self.swap_node_name.strip()
            ):
                raise ValueError(
                    "A swapped resource requires the node "
                    "at which entanglement swapping occurred."
                )

            object.__setattr__(
                self,
                "swap_node_name",
                self.swap_node_name.strip()
            )

        # =====================================================
        # Terminal-state invariants
        # =====================================================

        if self.status == EntanglementStatus.ACTIVE:

            if self.ended_slot is not None:
                raise ValueError(
                    "An ACTIVE resource cannot have ended_slot."
                )

            if self.termination_reason is not None:
                raise ValueError(
                    "An ACTIVE resource cannot have a "
                    "termination reason."
                )

        else:

            if (
                isinstance(self.ended_slot, bool)
                or not isinstance(
                    self.ended_slot,
                    numbers.Integral
                )
                or self.ended_slot < self.created_slot
            ):
                raise ValueError(
                    "A terminal resource requires a valid "
                    "ended_slot >= created_slot."
                )

            object.__setattr__(
                self,
                "ended_slot",
                int(self.ended_slot)
            )

    # =========================================================
    # Convenience Helpers
    # =========================================================

    @property
    def endpoint_names(self) -> tuple[str, str]:
        return (
            self.endpoint_a_name,
            self.endpoint_b_name
        )

    @property
    def is_active(self) -> bool:
        return (
            self.status
            == EntanglementStatus.ACTIVE
        )

    def involves_node(
        self,
        node_or_name
    ) -> bool:

        node_name = (
            node_or_name.name
            if isinstance(
                node_or_name,
                NetworkNode
            )
            else str(node_or_name)
        )

        return node_name in self.endpoint_names

    def other_endpoint_name(
        self,
        node_or_name
    ) -> str:

        node_name = (
            node_or_name.name
            if isinstance(
                node_or_name,
                NetworkNode
            )
            else str(node_or_name)
        )

        if node_name == self.endpoint_a_name:
            return self.endpoint_b_name

        if node_name == self.endpoint_b_name:
            return self.endpoint_a_name

        raise ValueError(
            f"Node '{node_name}' is not an endpoint "
            f"of resource '{self.resource_id}'."
        )

    def __repr__(self) -> str:

        return (
            "EntanglementResource("
            f"id='{self.resource_id}', "
            f"endpoints="
            f"'{self.endpoint_a_name}'<->"
            f"'{self.endpoint_b_name}', "
            f"kind='{self.kind.value}', "
            f"status='{self.status.value}', "
            f"created_slot={self.created_slot}"
            ")"
        )

In [ ]:
# @title Global Entanglement Manager


class EntanglementManager:
    """
    Authoritative global registry and lifecycle manager for
    reconstructed Phase-3 entanglement resources.

    Responsibilities
    ----------------
    - register one QuantumMemory per physical node;
    - create globally consistent elementary entanglement;
    - ensure both local halves exist for every ACTIVE resource;
    - synchronize hard memory-lifetime expiration;
    - discard resources atomically across both endpoint memories;
    - retain immutable historical resource state.

    The manager deliberately does NOT:
    - attempt physical entanglement generation;
    - decide when entanglement should be generated;
    - perform entanglement swapping;
    - select swapping trees;
    - perform routing or authentication;
    - model fidelity.

    Those belong to later Phase-3 components.
    """

    RESOURCE_PREFIX = "ENT"

    def __init__(
        self,
        kernel: SimulationKernel
    ):

        if not isinstance(
            kernel,
            SimulationKernel
        ):
            raise TypeError(
                "kernel must be a SimulationKernel."
            )

        self.kernel = kernel

        self._memories: dict[
            str,
            QuantumMemory
        ] = {}

        self._resources: dict[
            str,
            EntanglementResource
        ] = {}

        self._next_resource_number = 1

        self.kernel.emit(
            event_type="ENTANGLEMENT_MANAGER_INITIALIZED",
            component="EntanglementManager",
            message=(
                "Global entanglement manager initialized."
            )
        )

    # =========================================================
    # Memory Registration
    # =========================================================

    def register_memory(
        self,
        memory: QuantumMemory
    ) -> None:

        if not isinstance(
            memory,
            QuantumMemory
        ):
            raise TypeError(
                "memory must be a QuantumMemory."
            )

        # Every runtime component in one simulation must use
        # the exact same kernel instance.
        if memory.kernel is not self.kernel:
            raise ValueError(
                "Registered memory must belong to the same "
                "SimulationKernel as the EntanglementManager."
            )

        node_name = memory.owner.name

        if node_name in self._memories:
            raise ValueError(
                f"A quantum memory for node '{node_name}' "
                f"is already registered."
            )

        self._memories[
            node_name
        ] = memory

        self.kernel.emit(
            event_type="ENTANGLEMENT_MEMORY_REGISTERED",
            component="EntanglementManager",
            node_names=(node_name,),
            message=(
                f"Quantum memory at '{node_name}' registered "
                f"with the global entanglement manager."
            )
        )

    def memory_for_node(
        self,
        node_or_name
    ) -> QuantumMemory:

        node_name = (
            node_or_name.name
            if isinstance(
                node_or_name,
                NetworkNode
            )
            else str(node_or_name)
        )

        if node_name not in self._memories:
            raise EntanglementMemoryNotRegisteredError(
                f"No registered quantum memory exists for "
                f"node '{node_name}'."
            )

        return self._memories[
            node_name
        ]

    # =========================================================
    # Resource Identity
    # =========================================================

    def _allocate_resource_id(
        self
    ) -> str:

        resource_id = (
            f"{self.RESOURCE_PREFIX}-"
            f"{self._next_resource_number:06d}"
        )

        self._next_resource_number += 1

        return resource_id

    # =========================================================
    # Lookup
    # =========================================================

    def get_resource(
        self,
        resource_id: str
    ) -> EntanglementResource:

        resource_id = (
            self._validate_resource_id(
                resource_id
            )
        )

        if resource_id not in self._resources:
            raise EntanglementResourceNotFoundError(
                f"Global entanglement resource "
                f"'{resource_id}' does not exist."
            )

        return self._resources[
            resource_id
        ]

    @property
    def resources(
        self
    ) -> tuple[EntanglementResource, ...]:

        return tuple(
            self._resources.values()
        )

    @property
    def active_resources(
        self
    ) -> tuple[EntanglementResource, ...]:

        return tuple(
            resource
            for resource
            in self._resources.values()
            if resource.is_active
        )

    def active_resources_for_node(
        self,
        node_or_name
    ) -> tuple[EntanglementResource, ...]:

        node_name = (
            node_or_name.name
            if isinstance(
                node_or_name,
                NetworkNode
            )
            else str(node_or_name)
        )

        return tuple(
            resource
            for resource
            in self.active_resources
            if resource.involves_node(
                node_name
            )
        )

    # =========================================================
    # Elementary Resource Creation
    # =========================================================

    def create_elementary_resource(
        self,
        link: PhysicalQuantumLink
    ) -> EntanglementResource:
        """
        Create one ACTIVE elementary entanglement resource
        across a physical quantum link.

        Important:
        This method does NOT determine whether physical
        generation succeeded.

        Phase 3B's physical generation model will call this
        method only AFTER a successful elementary-entanglement
        generation event.
        """

        if not isinstance(
            link,
            PhysicalQuantumLink
        ):
            raise TypeError(
                "link must be a PhysicalQuantumLink."
            )

        node_a = link.node_a
        node_b = link.node_b

        memory_a = self.memory_for_node(
            node_a
        )

        memory_b = self.memory_for_node(
            node_b
        )

        # =====================================================
        # Preflight capacity checks
        #
        # We validate both endpoints before mutating either one.
        # =====================================================

        if memory_a.is_full:
            raise QuantumMemoryFullError(
                f"Cannot create elementary entanglement: "
                f"memory at '{node_a.name}' is full."
            )

        if memory_b.is_full:
            raise QuantumMemoryFullError(
                f"Cannot create elementary entanglement: "
                f"memory at '{node_b.name}' is full."
            )

        resource_id = (
            self._allocate_resource_id()
        )

        resource = EntanglementResource(
            resource_id=resource_id,

            endpoint_a_name=node_a.name,
            endpoint_b_name=node_b.name,

            kind=(
                EntanglementKind.ELEMENTARY
            ),

            created_slot=(
                self.kernel.current_slot
            ),

            source_link_ids=(
                link.link_id,
            )
        )

        # =====================================================
        # Store both local halves
        # =====================================================

        stored_at_a = False

        try:

            memory_a.store_local_half(
                resource_id=resource_id,
                remote_node=node_b
            )

            stored_at_a = True

            memory_b.store_local_half(
                resource_id=resource_id,
                remote_node=node_a
            )

        except Exception:

            # Defensive rollback.
            #
            # Under the current single-threaded simulator and
            # successful preflight validation this should not
            # normally be required, but it prevents a partially
            # committed global resource if an unexpected failure
            # occurs.
            if (
                stored_at_a
                and memory_a.contains(
                    resource_id
                )
            ):
                memory_a.remove_local_half(
                    resource_id,
                    reason=(
                        "elementary_creation_rollback"
                    )
                )

            raise

        # Only after both local halves exist does the global
        # resource become authoritative.
        self._resources[
            resource_id
        ] = resource

        self._assert_active_storage_consistency(
            resource
        )

        self.kernel.emit(
            event_type="ENTANGLEMENT_RESOURCE_CREATED",
            component="EntanglementManager",
            node_names=(
                resource.endpoint_a_name,
                resource.endpoint_b_name
            ),
            link_ids=(
                link.link_id,
            ),
            resource_ids=(
                resource.resource_id,
            ),
            message=(
                f"Elementary entanglement resource "
                f"'{resource.resource_id}' established "
                f"between '{resource.endpoint_a_name}' "
                f"and '{resource.endpoint_b_name}'."
            ),
            metadata={
                "kind":
                    resource.kind.value,

                "created_slot":
                    resource.created_slot
            }
        )

        return resource

    # =========================================================
    # Expiration Synchronization
    # =========================================================

    def synchronize_expirations(
        self
    ) -> tuple[str, ...]:
        """
        Detect locally expired halves and atomically expire
        their corresponding global resources.

        A resource expires if EITHER endpoint local half has
        exceeded its configured memory storage lifetime.

        Returns
        -------
        tuple[str, ...]
            Resource IDs globally expired in this call.
        """

        locally_expired_ids = set()

        for memory in self._memories.values():

            locally_expired_ids.update(
                memory.expired_resource_ids()
            )

        expired_now = []

        for resource_id in sorted(
            locally_expired_ids
        ):

            # A registered memory containing an unknown resource
            # indicates state corruption.
            if (
                resource_id
                not in self._resources
            ):
                raise EntanglementConsistencyError(
                    f"Memory contains unknown global resource "
                    f"'{resource_id}'."
                )

            resource = self._resources[
                resource_id
            ]

            # Already-terminal resources should never still occupy
            # endpoint memory.
            if not resource.is_active:
                raise EntanglementConsistencyError(
                    f"Terminal resource '{resource_id}' "
                    f"still exists in quantum memory."
                )

            self._terminate_resource(
                resource_id=resource_id,
                terminal_status=(
                    EntanglementStatus.EXPIRED
                ),
                reason=(
                    "local_memory_lifetime_exceeded"
                )
            )

            expired_now.append(
                resource_id
            )

        return tuple(
            expired_now
        )

    # =========================================================
    # Explicit Discard
    # =========================================================

    def discard_resource(
        self,
        resource_id: str,
        reason: str = "explicit_discard"
    ) -> EntanglementResource:
        """
        Globally discard an ACTIVE entanglement resource.

        Both endpoint local halves are removed.
        """

        return self._terminate_resource(
            resource_id=resource_id,
            terminal_status=(
                EntanglementStatus.DISCARDED
            ),
            reason=reason
        )

    # =========================================================
    # Internal Terminal Transition
    # =========================================================

    def _terminate_resource(
        self,
        resource_id: str,
        terminal_status: EntanglementStatus,
        reason: str
    ) -> EntanglementResource:

        if terminal_status not in (
            EntanglementStatus.EXPIRED,
            EntanglementStatus.DISCARDED
        ):
            raise ValueError(
                "_terminate_resource() currently supports "
                "only EXPIRED and DISCARDED transitions."
            )

        resource = self.get_resource(
            resource_id
        )

        if not resource.is_active:
            raise EntanglementResourceStateError(
                f"Resource '{resource_id}' is already "
                f"{resource.status.value}."
            )

        # Verify the complete global/local state BEFORE mutating.
        self._assert_active_storage_consistency(
            resource
        )

        memory_a = self.memory_for_node(
            resource.endpoint_a_name
        )

        memory_b = self.memory_for_node(
            resource.endpoint_b_name
        )

        # Current simulation is single-threaded, so after the
        # preflight consistency assertion these two removals form
        # one logical atomic transition.
        memory_a.remove_local_half(
            resource.resource_id,
            reason=terminal_status.value.lower()
        )

        memory_b.remove_local_half(
            resource.resource_id,
            reason=terminal_status.value.lower()
        )

        terminal_resource = replace(
            resource,

            status=terminal_status,

            ended_slot=(
                self.kernel.current_slot
            ),

            termination_reason=str(
                reason
            )
        )

        self._resources[
            resource.resource_id
        ] = terminal_resource

        self.kernel.emit(
            event_type=(
                "ENTANGLEMENT_RESOURCE_"
                f"{terminal_status.value}"
            ),
            component="EntanglementManager",
            node_names=(
                resource.endpoint_a_name,
                resource.endpoint_b_name
            ),
            resource_ids=(
                resource.resource_id,
            ),
            message=(
                f"Entanglement resource "
                f"'{resource.resource_id}' became "
                f"{terminal_status.value}."
            ),
            metadata={
                "ended_slot":
                    self.kernel.current_slot,

                "reason":
                    str(reason)
            }
        )

        return terminal_resource

    # =========================================================
    # Global / Local Consistency
    # =========================================================

    def _assert_active_storage_consistency(
        self,
        resource: EntanglementResource
    ) -> None:
        """
        Verify that an ACTIVE global resource has exactly one
        correct local half stored at each endpoint memory.
        """

        if not resource.is_active:
            raise EntanglementResourceStateError(
                f"Consistency validation requires an ACTIVE "
                f"resource; '{resource.resource_id}' is "
                f"{resource.status.value}."
            )

        memory_a = self.memory_for_node(
            resource.endpoint_a_name
        )

        memory_b = self.memory_for_node(
            resource.endpoint_b_name
        )

        if not memory_a.contains(
            resource.resource_id
        ):
            raise EntanglementConsistencyError(
                f"ACTIVE resource '{resource.resource_id}' "
                f"is missing from endpoint memory "
                f"'{resource.endpoint_a_name}'."
            )

        if not memory_b.contains(
            resource.resource_id
        ):
            raise EntanglementConsistencyError(
                f"ACTIVE resource '{resource.resource_id}' "
                f"is missing from endpoint memory "
                f"'{resource.endpoint_b_name}'."
            )

        half_a = memory_a.get_local_half(
            resource.resource_id
        )

        half_b = memory_b.get_local_half(
            resource.resource_id
        )

        if (
            half_a.remote_node_name
            != resource.endpoint_b_name
        ):
            raise EntanglementConsistencyError(
                f"Incorrect remote endpoint recorded for "
                f"resource '{resource.resource_id}' at "
                f"'{resource.endpoint_a_name}'."
            )

        if (
            half_b.remote_node_name
            != resource.endpoint_a_name
        ):
            raise EntanglementConsistencyError(
                f"Incorrect remote endpoint recorded for "
                f"resource '{resource.resource_id}' at "
                f"'{resource.endpoint_b_name}'."
            )

    def validate_all_active_resources(
        self
    ) -> bool:
        """
        Validate the complete ACTIVE global entanglement state.
        """

        for resource in self.active_resources:

            self._assert_active_storage_consistency(
                resource
            )

        return True

    # =========================================================
    # Dataframe Inspection
    # =========================================================

    def resources_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for resource in self._resources.values():

            rows.append({
                "Resource ID":
                    resource.resource_id,

                "Kind":
                    resource.kind.value,

                "Status":
                    resource.status.value,

                "Endpoint A":
                    resource.endpoint_a_name,

                "Endpoint B":
                    resource.endpoint_b_name,

                "Created Slot":
                    resource.created_slot,

                "Ended Slot":
                    resource.ended_slot,

                "Source Links":
                    list(
                        resource.source_link_ids
                    ),

                "Parent Resources":
                    list(
                        resource.parent_resource_ids
                    ),

                "Swap Node":
                    resource.swap_node_name,

                "Termination Reason":
                    resource.termination_reason
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Internal Validation
    # =========================================================

    @staticmethod
    def _validate_resource_id(
        resource_id: str
    ) -> str:

        if (
            not isinstance(resource_id, str)
            or not resource_id.strip()
        ):
            raise ValueError(
                "resource_id must be a non-empty string."
            )

        return resource_id.strip()

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:

        return (
            "EntanglementManager("
            f"memories={len(self._memories)}, "
            f"resources={len(self._resources)}, "
            f"active={len(self.active_resources)}"
            ")"
        )

In [ ]:
# @title Entanglement Manager Verification

# from dataclasses import FrozenInstanceError


# print("=" * 76)
# print("NARAD-QC — Entanglement Manager Verification")
# print("=" * 76)


# # =============================================================
# # Independent simulation
# # =============================================================

# ent_test_kernel = SimulationKernel(
#     seed=350015
# )


# # =============================================================
# # Test nodes
# # =============================================================

# ent_node_a = NetworkNode(
#     name="ENT-A",
#     x_km=0,
#     y_km=0,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=2
# )

# ent_node_b = NetworkNode(
#     name="ENT-B",
#     x_km=30,
#     y_km=0,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )


# # =============================================================
# # Physical link
# # =============================================================

# ent_test_link = PhysicalQuantumLink(
#     node_a=ent_node_a,
#     node_b=ent_node_b,
#     attenuation_db_per_km=0.2
# )


# # =============================================================
# # Runtime memories
# # =============================================================

# ent_memory_a = QuantumMemory(
#     owner=ent_node_a,
#     kernel=ent_test_kernel,
#     max_storage_slots=3
# )

# ent_memory_b = QuantumMemory(
#     owner=ent_node_b,
#     kernel=ent_test_kernel,
#     max_storage_slots=3
# )


# # =============================================================
# # Global manager
# # =============================================================

# ent_manager = EntanglementManager(
#     kernel=ent_test_kernel
# )

# ent_manager.register_memory(
#     ent_memory_a
# )

# ent_manager.register_memory(
#     ent_memory_b
# )

# print(
#     "[PASS] Endpoint quantum memories registered successfully."
# )


# # =============================================================
# # Test 1 — Create elementary global resource
# # =============================================================

# resource_1 = (
#     ent_manager.create_elementary_resource(
#         ent_test_link
#     )
# )

# assert resource_1.kind == (
#     EntanglementKind.ELEMENTARY
# )

# assert resource_1.status == (
#     EntanglementStatus.ACTIVE
# )

# assert resource_1.source_link_ids == (
#     ent_test_link.link_id,
# )

# print(
#     "[PASS] Elementary global entanglement resource created."
# )


# # =============================================================
# # Test 2 — Both endpoint memories contain local halves
# # =============================================================

# assert ent_memory_a.contains(
#     resource_1.resource_id
# )

# assert ent_memory_b.contains(
#     resource_1.resource_id
# )

# half_a = ent_memory_a.get_local_half(
#     resource_1.resource_id
# )

# half_b = ent_memory_b.get_local_half(
#     resource_1.resource_id
# )

# assert (
#     half_a.remote_node_name
#     == ent_node_b.name
# )

# assert (
#     half_b.remote_node_name
#     == ent_node_a.name
# )

# # They represent two different physical local qubits.
# assert half_a is not half_b

# print(
#     "[PASS] One global resource maps to two distinct "
#     "local quantum halves."
# )


# # =============================================================
# # Test 3 — Global/local consistency validates
# # =============================================================

# assert (
#     ent_manager.validate_all_active_resources()
#     is True
# )

# print(
#     "[PASS] Global resource and endpoint-memory state "
#     "are mutually consistent."
# )


# # =============================================================
# # Test 4 — Resource identity is immutable
# # =============================================================

# immutability_error = False

# try:
#     resource_1.endpoint_a_name = "CHANGED"

# except FrozenInstanceError:
#     immutability_error = True

# assert immutability_error

# print(
#     "[PASS] Global entanglement resource records are immutable."
# )


# # =============================================================
# # Test 5 — Local qubit ages increase through simulation time
# # =============================================================

# ent_test_kernel.advance_slot(
#     steps=2
# )

# assert ent_memory_a.age_slots(
#     resource_1.resource_id
# ) == 2

# assert ent_memory_b.age_slots(
#     resource_1.resource_id
# ) == 2

# assert not ent_memory_a.is_locally_expired(
#     resource_1.resource_id
# )

# assert not ent_memory_b.is_locally_expired(
#     resource_1.resource_id
# )

# print(
#     "[PASS] Both local qubit halves age according to "
#     "simulation time."
# )


# # =============================================================
# # Test 6 — At lifetime boundary global resource expires
# # =============================================================

# ent_test_kernel.advance_slot()

# assert ent_test_kernel.current_slot == 3

# assert ent_memory_a.is_locally_expired(
#     resource_1.resource_id
# )

# assert ent_memory_b.is_locally_expired(
#     resource_1.resource_id
# )

# expired_ids = (
#     ent_manager.synchronize_expirations()
# )

# assert expired_ids == (
#     resource_1.resource_id,
# )

# expired_resource = (
#     ent_manager.get_resource(
#         resource_1.resource_id
#     )
# )

# assert expired_resource.status == (
#     EntanglementStatus.EXPIRED
# )

# assert expired_resource.ended_slot == 3

# assert not ent_memory_a.contains(
#     resource_1.resource_id
# )

# assert not ent_memory_b.contains(
#     resource_1.resource_id
# )

# print(
#     "[PASS] Local lifetime violation atomically expires "
#     "the global resource at both endpoints."
# )


# # =============================================================
# # Test 7 — Freed memory can hold another resource
# # =============================================================

# resource_2 = (
#     ent_manager.create_elementary_resource(
#         ent_test_link
#     )
# )

# assert resource_2.resource_id != (
#     resource_1.resource_id
# )

# assert ent_memory_a.contains(
#     resource_2.resource_id
# )

# assert ent_memory_b.contains(
#     resource_2.resource_id
# )

# print(
#     "[PASS] Endpoint memory can be reused after "
#     "global expiration."
# )


# # =============================================================
# # Test 8 — Explicit discard removes both local halves
# # =============================================================

# discarded_resource = (
#     ent_manager.discard_resource(
#         resource_2.resource_id,
#         reason="verification_discard"
#     )
# )

# assert discarded_resource.status == (
#     EntanglementStatus.DISCARDED
# )

# assert not ent_memory_a.contains(
#     resource_2.resource_id
# )

# assert not ent_memory_b.contains(
#     resource_2.resource_id
# )

# print(
#     "[PASS] Explicit discard removes both local halves "
#     "and terminates the global resource."
# )


# # =============================================================
# # Test 9 — Terminal resource cannot be discarded again
# # =============================================================

# state_error = False

# try:
#     ent_manager.discard_resource(
#         resource_2.resource_id
#     )

# except EntanglementResourceStateError:
#     state_error = True

# assert state_error

# print(
#     "[PASS] Invalid lifecycle transition from terminal "
#     "resource is rejected."
# )


# # =============================================================
# # Test 10 — Unknown resource lookup rejected
# # =============================================================

# not_found_error = False

# try:
#     ent_manager.get_resource(
#         "ENT-999999"
#     )

# except EntanglementResourceNotFoundError:
#     not_found_error = True

# assert not_found_error

# print(
#     "[PASS] Unknown global resource lookup is rejected."
# )


# # =============================================================
# # Final invariants
# # =============================================================

# assert len(
#     ent_manager.resources
# ) == 2

# assert len(
#     ent_manager.active_resources
# ) == 0

# assert ent_memory_a.is_empty
# assert ent_memory_b.is_empty

# print(
#     "[PASS] Final global/local state is internally clean."
# )


# # =============================================================
# # Inspection
# # =============================================================

# print("\nGLOBAL RESOURCE HISTORY")

# display(
#     ent_manager.resources_dataframe()
# )


# print("\nENDPOINT A MEMORY")

# display(
#     ent_memory_a.slots_dataframe()
# )


# print("\nENDPOINT B MEMORY")

# display(
#     ent_memory_b.slots_dataframe()
# )


# print("\nENTANGLEMENT TEST EVENT LOG")

# display(
#     ent_test_kernel.events_dataframe()
# )


# print("\n" + "=" * 76)
# print("Verification completed successfully.")
# print("=" * 76)

In [ ]:
# @title Physical Network Topology

import math
import pandas as pd


# =============================================================
# Dedicated Topology Errors
# =============================================================

class NetworkTopologyError(Exception):
    """Base exception for reconstructed Phase-3 topology."""
    pass


class TopologyNodeAlreadyExistsError(NetworkTopologyError):
    """Raised when a duplicate physical node is added."""
    pass


class TopologyNodeNotFoundError(NetworkTopologyError):
    """Raised when a requested physical node does not exist."""
    pass


class TopologyLinkAlreadyExistsError(NetworkTopologyError):
    """Raised when a duplicate undirected physical link is added."""
    pass


class TopologyLinkNotFoundError(NetworkTopologyError):
    """Raised when a requested physical link does not exist."""
    pass


class TopologyConsistencyError(NetworkTopologyError):
    """
    Raised when an object conflicts with the topology's
    authoritative physical state.
    """
    pass


class InvalidPhysicalPathError(NetworkTopologyError):
    """Raised when a supplied ordered physical path is invalid."""
    pass


# =============================================================
# Authoritative Physical Topology
# =============================================================

class NetworkTopology:
    """
    Authoritative physical graph for reconstructed NARAD-QC Phase 3.

    The topology owns the canonical physical NetworkNode and
    PhysicalQuantumLink objects that describe one network.

    Responsibilities
    ----------------
    - physical node ownership;
    - physical link ownership;
    - undirected adjacency;
    - spatial/network inspection;
    - validation of supplied physical paths.

    The topology deliberately does NOT:
    - own a SimulationKernel;
    - own quantum-memory runtime contents;
    - own entanglement resources;
    - generate elementary entanglement;
    - perform swapping;
    - select routes;
    - authenticate nodes;
    - reserve resources;
    - manage sessions;
    - perform QKD.

    Static physical topology and runtime simulation state remain
    separate.
    """

    def __init__(
        self,
        name: str = "NARAD-QC"
    ):

        if (
            not isinstance(name, str)
            or not name.strip()
        ):
            raise ValueError(
                "Topology name must be a non-empty string."
            )

        self.name = name.strip()

        # Canonical physical objects.
        self._nodes: dict[
            str,
            NetworkNode
        ] = {}

        self._links: dict[
            str,
            PhysicalQuantumLink
        ] = {}

        # Adjacency stores only canonical node names.
        self._adjacency: dict[
            str,
            set[str]
        ] = {}

    # =========================================================
    # Canonical Identity Helpers
    # =========================================================

    @staticmethod
    def canonical_link_id(
        node_a_name: str,
        node_b_name: str
    ) -> str:
        """
        Return the canonical undirected link ID between two names.
        """

        if (
            not isinstance(node_a_name, str)
            or not node_a_name.strip()
        ):
            raise ValueError(
                "node_a_name must be a non-empty string."
            )

        if (
            not isinstance(node_b_name, str)
            or not node_b_name.strip()
        ):
            raise ValueError(
                "node_b_name must be a non-empty string."
            )

        node_a_name = node_a_name.strip()
        node_b_name = node_b_name.strip()

        if node_a_name == node_b_name:
            raise ValueError(
                "A physical link requires two distinct nodes."
            )

        names = sorted(
            (
                node_a_name,
                node_b_name
            )
        )

        return (
            f"{names[0]}--{names[1]}"
        )

    @staticmethod
    def _node_name(
        node_or_name
    ) -> str:
        """
        Extract a normalized node name from a NetworkNode or string.
        """

        if isinstance(
            node_or_name,
            NetworkNode
        ):
            return node_or_name.name

        if (
            isinstance(node_or_name, str)
            and node_or_name.strip()
        ):
            return node_or_name.strip()

        raise TypeError(
            "Expected a NetworkNode or non-empty node-name string."
        )

    # =========================================================
    # Node Ownership
    # =========================================================

    def add_node(
        self,
        node: NetworkNode
    ) -> NetworkNode:

        if not isinstance(
            node,
            NetworkNode
        ):
            raise TypeError(
                "node must be a NetworkNode."
            )

        if node.name in self._nodes:
            raise TopologyNodeAlreadyExistsError(
                f"Node '{node.name}' already exists "
                f"in topology '{self.name}'."
            )

        self._nodes[
            node.name
        ] = node

        self._adjacency[
            node.name
        ] = set()

        return node

    def contains_node(
        self,
        node_or_name
    ) -> bool:

        node_name = self._node_name(
            node_or_name
        )

        if node_name not in self._nodes:
            return False

        # If a concrete NetworkNode object was supplied, the
        # topology recognizes only its exact canonical instance.
        if isinstance(
            node_or_name,
            NetworkNode
        ):
            return (
                self._nodes[node_name]
                is node_or_name
            )

        return True

    def get_node(
        self,
        node_or_name
    ) -> NetworkNode:

        node_name = self._node_name(
            node_or_name
        )

        if node_name not in self._nodes:
            raise TopologyNodeNotFoundError(
                f"Node '{node_name}' does not exist "
                f"in topology '{self.name}'."
            )

        canonical_node = self._nodes[
            node_name
        ]

        # Do not silently accept a different physical object
        # that merely reuses the same name.
        if (
            isinstance(
                node_or_name,
                NetworkNode
            )
            and canonical_node
            is not node_or_name
        ):
            raise TopologyConsistencyError(
                f"A different NetworkNode object named "
                f"'{node_name}' was supplied. "
                f"The topology owns its own canonical instance."
            )

        return canonical_node

    def remove_node(
        self,
        node_or_name,
        remove_incident_links: bool = False
    ) -> NetworkNode:
        """
        Remove a physical node.

        By default, removal is rejected when incident physical
        links still exist. This prevents silent graph corruption.

        Set remove_incident_links=True only when explicit cascading
        removal is intended.
        """

        node = self.get_node(
            node_or_name
        )

        neighbours = tuple(
            self._adjacency[node.name]
        )

        if (
            neighbours
            and not remove_incident_links
        ):
            raise TopologyConsistencyError(
                f"Cannot remove node '{node.name}' while "
                f"{len(neighbours)} incident link(s) remain. "
                f"Remove them first or explicitly set "
                f"remove_incident_links=True."
            )

        if remove_incident_links:

            for neighbour_name in neighbours:

                self.remove_link(
                    node.name,
                    neighbour_name
                )

        del self._adjacency[
            node.name
        ]

        del self._nodes[
            node.name
        ]

        return node

    # =========================================================
    # Physical Link Ownership
    # =========================================================

    def add_link(
        self,
        link: PhysicalQuantumLink
    ) -> PhysicalQuantumLink:

        if not isinstance(
            link,
            PhysicalQuantumLink
        ):
            raise TypeError(
                "link must be a PhysicalQuantumLink."
            )

        # Both endpoints must already physically exist here.
        for endpoint in (
            link.node_a,
            link.node_b
        ):

            if endpoint.name not in self._nodes:
                raise TopologyNodeNotFoundError(
                    f"Cannot add link '{link.link_id}': "
                    f"endpoint '{endpoint.name}' is not "
                    f"registered in topology '{self.name}'."
                )

            # Strong canonical-object invariant.
            if (
                self._nodes[endpoint.name]
                is not endpoint
            ):
                raise TopologyConsistencyError(
                    f"Link '{link.link_id}' references a "
                    f"non-canonical node object named "
                    f"'{endpoint.name}'."
                )

        expected_link_id = (
            self.canonical_link_id(
                link.node_a.name,
                link.node_b.name
            )
        )

        if link.link_id != expected_link_id:
            raise TopologyConsistencyError(
                f"Link ID '{link.link_id}' does not match "
                f"canonical topology identity "
                f"'{expected_link_id}'."
            )

        if link.link_id in self._links:
            raise TopologyLinkAlreadyExistsError(
                f"Physical link '{link.link_id}' already exists "
                f"in topology '{self.name}'."
            )

        self._links[
            link.link_id
        ] = link

        self._adjacency[
            link.node_a.name
        ].add(
            link.node_b.name
        )

        self._adjacency[
            link.node_b.name
        ].add(
            link.node_a.name
        )

        return link

    def contains_link(
        self,
        node_a,
        node_b
    ) -> bool:

        node_a_name = self._node_name(
            node_a
        )

        node_b_name = self._node_name(
            node_b
        )

        if node_a_name == node_b_name:
            return False

        link_id = (
            self.canonical_link_id(
                node_a_name,
                node_b_name
            )
        )

        return (
            link_id in self._links
        )

    def get_link(
        self,
        node_a,
        node_b
    ) -> PhysicalQuantumLink:

        node_a_name = self._node_name(
            node_a
        )

        node_b_name = self._node_name(
            node_b
        )

        link_id = (
            self.canonical_link_id(
                node_a_name,
                node_b_name
            )
        )

        if link_id not in self._links:
            raise TopologyLinkNotFoundError(
                f"Physical link '{link_id}' does not exist "
                f"in topology '{self.name}'."
            )

        return self._links[
            link_id
        ]

    def get_link_by_id(
        self,
        link_id: str
    ) -> PhysicalQuantumLink:

        if (
            not isinstance(link_id, str)
            or not link_id.strip()
        ):
            raise ValueError(
                "link_id must be a non-empty string."
            )

        link_id = link_id.strip()

        if link_id not in self._links:
            raise TopologyLinkNotFoundError(
                f"Physical link '{link_id}' does not exist "
                f"in topology '{self.name}'."
            )

        return self._links[
            link_id
        ]

    def remove_link(
        self,
        node_a,
        node_b
    ) -> PhysicalQuantumLink:

        link = self.get_link(
            node_a,
            node_b
        )

        node_a_name = link.node_a.name
        node_b_name = link.node_b.name

        self._adjacency[
            node_a_name
        ].remove(
            node_b_name
        )

        self._adjacency[
            node_b_name
        ].remove(
            node_a_name
        )

        del self._links[
            link.link_id
        ]

        return link

    # =========================================================
    # Read-Only Physical Views
    # =========================================================

    @property
    def nodes(
        self
    ) -> tuple[NetworkNode, ...]:
        """
        Immutable view of canonical physical nodes.
        """

        return tuple(
            self._nodes.values()
        )

    @property
    def links(
        self
    ) -> tuple[PhysicalQuantumLink, ...]:
        """
        Immutable view of canonical physical links.
        """

        return tuple(
            self._links.values()
        )

    @property
    def node_count(self) -> int:
        return len(
            self._nodes
        )

    @property
    def link_count(self) -> int:
        return len(
            self._links
        )

    def neighbours(
        self,
        node_or_name
    ) -> tuple[NetworkNode, ...]:
        """
        Return physical neighbours in deterministic name order.
        """

        node = self.get_node(
            node_or_name
        )

        neighbour_names = sorted(
            self._adjacency[
                node.name
            ]
        )

        return tuple(
            self._nodes[name]
            for name in neighbour_names
        )

    # American-spelling alias for convenience.
    neighbors = neighbours

    def degree(
        self,
        node_or_name
    ) -> int:

        node = self.get_node(
            node_or_name
        )

        return len(
            self._adjacency[
                node.name
            ]
        )

    def adjacency_snapshot(
        self
    ) -> dict[str, tuple[str, ...]]:
        """
        Return a copy of adjacency without exposing mutable sets.
        """

        return {
            node_name: tuple(
                sorted(neighbour_names)
            )
            for node_name, neighbour_names
            in self._adjacency.items()
        }

    # =========================================================
    # Physical Path Validation
    # =========================================================

    def _resolve_path(
        self,
        path
    ) -> tuple[NetworkNode, ...]:

        if (
            isinstance(path, (str, bytes))
            or path is None
        ):
            raise TypeError(
                "path must be an ordered collection of "
                "NetworkNode objects and/or node-name strings."
            )

        try:
            path_items = list(
                path
            )

        except TypeError as exc:
            raise TypeError(
                "path must be an ordered iterable."
            ) from exc

        if len(path_items) < 2:
            raise InvalidPhysicalPathError(
                "A physical path must contain at least "
                "two nodes."
            )

        resolved_nodes = tuple(
            self.get_node(item)
            for item in path_items
        )

        node_names = tuple(
            node.name
            for node in resolved_nodes
        )

        if len(set(node_names)) != len(node_names):
            raise InvalidPhysicalPathError(
                "A physical path must be simple; "
                "repeated nodes are not allowed."
            )

        return resolved_nodes

    def validate_path(
        self,
        path
    ) -> tuple[NetworkNode, ...]:
        """
        Validate an explicitly supplied physical path.

        This performs NO route selection.

        A valid path:
        - contains at least two registered nodes;
        - contains no repeated nodes;
        - has a physical link between every consecutive pair.

        Returns the canonical ordered NetworkNode tuple.
        """

        resolved_nodes = self._resolve_path(
            path
        )

        for left, right in zip(
            resolved_nodes[:-1],
            resolved_nodes[1:]
        ):

            if not self.contains_link(
                left,
                right
            ):
                raise InvalidPhysicalPathError(
                    f"Invalid physical path: no link exists "
                    f"between '{left.name}' and '{right.name}'."
                )

        return resolved_nodes

    def path_links(
        self,
        path
    ) -> tuple[PhysicalQuantumLink, ...]:
        """
        Return ordered physical links corresponding to a
        validated ordered node path.
        """

        nodes = self.validate_path(
            path
        )

        return tuple(
            self.get_link(
                left,
                right
            )
            for left, right in zip(
                nodes[:-1],
                nodes[1:]
            )
        )

    def path_distance_km(
        self,
        path
    ) -> float:
        """
        Total fibre length of a validated physical path.
        """

        return sum(
            link.distance_km
            for link in self.path_links(
                path
            )
        )

    # =========================================================
    # Network-Level Physical Metrics
    # =========================================================

    @property
    def total_link_length_km(
        self
    ) -> float:

        return sum(
            link.distance_km
            for link in self._links.values()
        )

    @property
    def average_degree(
        self
    ) -> float:

        if self.node_count == 0:
            return 0.0

        return (
            sum(
                len(neighbours)
                for neighbours
                in self._adjacency.values()
            )
            / self.node_count
        )

    # =========================================================
    # Inspection Tables
    # =========================================================

    def nodes_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for node in self._nodes.values():

            rows.append({
                "Node":
                    node.name,

                "Role":
                    node.functional_role,

                "X (km)":
                    node.x_km,

                "Y (km)":
                    node.y_km,

                "Endpoint Capable":
                    node.endpoint_capable,

                "Swapping Capable":
                    node.swapping_capable,

                "Memory Capacity":
                    node.memory_capacity,

                "Degree":
                    self.degree(
                        node
                    )
            })

        return pd.DataFrame(
            rows
        )

    def links_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for link in self._links.values():

            rows.append({
                "Link ID":
                    link.link_id,

                "Node A":
                    link.node_a.name,

                "Node B":
                    link.node_b.name,

                "Distance (km)":
                    link.distance_km,

                "Attenuation (dB/km)":
                    link.attenuation_db_per_km,

                "Total Attenuation (dB)":
                    link.total_attenuation_db,

                "Survival Probability":
                    link.survival_probability,

                "Loss Probability":
                    link.loss_probability
            })

        return pd.DataFrame(
            rows
        )

    def summary(
        self
    ) -> dict:

        endpoint_count = sum(
            node.endpoint_capable
            for node in self._nodes.values()
        )

        swapping_count = sum(
            node.swapping_capable
            for node in self._nodes.values()
        )

        hybrid_count = sum(
            (
                node.endpoint_capable
                and node.swapping_capable
            )
            for node in self._nodes.values()
        )

        isolated_count = sum(
            self.degree(node) == 0
            for node in self._nodes.values()
        )

        return {
            "name":
                self.name,

            "nodes":
                self.node_count,

            "links":
                self.link_count,

            "endpoint_capable_nodes":
                endpoint_count,

            "swapping_capable_nodes":
                swapping_count,

            "hybrid_nodes":
                hybrid_count,

            "isolated_nodes":
                isolated_count,

            "total_link_length_km":
                self.total_link_length_km,

            "average_degree":
                self.average_degree
        }

    # =========================================================
    # Full Internal Consistency Check
    # =========================================================

    def validate_internal_consistency(
        self
    ) -> bool:
        """
        Verify that node, link, and adjacency state agree.

        Intended for tests and defensive validation.
        """

        # Every node must have exactly one adjacency entry.
        if set(self._nodes) != set(
            self._adjacency
        ):
            raise TopologyConsistencyError(
                "Node registry and adjacency registry disagree."
            )

        # Every link must agree with canonical node ownership
        # and bidirectional adjacency.
        for link_id, link in self._links.items():

            if link.link_id != link_id:
                raise TopologyConsistencyError(
                    f"Stored link key '{link_id}' disagrees "
                    f"with link identity '{link.link_id}'."
                )

            for endpoint in (
                link.node_a,
                link.node_b
            ):

                if endpoint.name not in self._nodes:
                    raise TopologyConsistencyError(
                        f"Link '{link_id}' references absent "
                        f"node '{endpoint.name}'."
                    )

                if (
                    self._nodes[endpoint.name]
                    is not endpoint
                ):
                    raise TopologyConsistencyError(
                        f"Link '{link_id}' references a "
                        f"non-canonical node object."
                    )

            if (
                link.node_b.name
                not in self._adjacency[
                    link.node_a.name
                ]
            ):
                raise TopologyConsistencyError(
                    f"Adjacency missing for link '{link_id}'."
                )

            if (
                link.node_a.name
                not in self._adjacency[
                    link.node_b.name
                ]
            ):
                raise TopologyConsistencyError(
                    f"Reverse adjacency missing for "
                    f"link '{link_id}'."
                )

        # Every adjacency relation must correspond to a link
        # and must be symmetric.
        for node_name, neighbour_names in (
            self._adjacency.items()
        ):

            for neighbour_name in neighbour_names:

                if (
                    node_name
                    not in self._adjacency[
                        neighbour_name
                    ]
                ):
                    raise TopologyConsistencyError(
                        f"Asymmetric adjacency between "
                        f"'{node_name}' and '{neighbour_name}'."
                    )

                link_id = (
                    self.canonical_link_id(
                        node_name,
                        neighbour_name
                    )
                )

                if link_id not in self._links:
                    raise TopologyConsistencyError(
                        f"Adjacency '{node_name}'--"
                        f"'{neighbour_name}' has no "
                        f"corresponding physical link."
                    )

        return True

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:

        return (
            "NetworkTopology("
            f"name='{self.name}', "
            f"nodes={self.node_count}, "
            f"links={self.link_count}, "
            f"total_length="
            f"{self.total_link_length_km:.2f} km"
            ")"
        )

In [ ]:
# @title Network Topology Verification

# print("=" * 76)
# print("NARAD-QC — Network Topology Verification")
# print("=" * 76)


# # =============================================================
# # Independent physical topology
# # =============================================================

# topology_test = NetworkTopology(
#     name="Phase3A-Test-Network"
# )

# assert topology_test.node_count == 0
# assert topology_test.link_count == 0

# print(
#     "[PASS] Empty physical topology initializes correctly."
# )


# # =============================================================
# # Test nodes
# #
# # Geometry:
# #
# # A ----50 km---- R1 ----30 km---- R2
# #                                    |
# #                                   40 km
# #                                    |
# #                                    B
# #
# # Charlie is intentionally disconnected.
# # =============================================================

# topo_a = NetworkNode(
#     name="Topo-A",
#     x_km=0,
#     y_km=0,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=2
# )

# topo_r1 = NetworkNode(
#     name="Topo-R1",
#     x_km=30,
#     y_km=40,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )

# topo_r2 = NetworkNode(
#     name="Topo-R2",
#     x_km=60,
#     y_km=40,
#     endpoint_capable=False,
#     swapping_capable=True,
#     memory_capacity=2
# )

# topo_b = NetworkNode(
#     name="Topo-B",
#     x_km=60,
#     y_km=0,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=2
# )

# topo_charlie = NetworkNode(
#     name="Topo-Charlie",
#     x_km=100,
#     y_km=100,
#     endpoint_capable=True,
#     swapping_capable=False,
#     memory_capacity=1
# )


# # =============================================================
# # Test 1 — Node ownership
# # =============================================================

# for node in (
#     topo_a,
#     topo_r1,
#     topo_r2,
#     topo_b,
#     topo_charlie
# ):
#     topology_test.add_node(
#         node
#     )

# assert topology_test.node_count == 5

# assert topology_test.get_node(
#     "Topo-R1"
# ) is topo_r1

# print(
#     "[PASS] Canonical physical nodes are registered correctly."
# )


# # =============================================================
# # Test 2 — Duplicate node names rejected
# # =============================================================

# duplicate_node_error = False

# try:
#     topology_test.add_node(
#         NetworkNode(
#             name="Topo-R1",
#             x_km=999,
#             y_km=999,
#             endpoint_capable=False,
#             swapping_capable=True,
#             memory_capacity=4
#         )
#     )

# except TopologyNodeAlreadyExistsError:
#     duplicate_node_error = True

# assert duplicate_node_error

# print(
#     "[PASS] Duplicate physical node identity is rejected."
# )


# # =============================================================
# # Physical links
# # =============================================================

# topo_link_a_r1 = PhysicalQuantumLink(
#     node_a=topo_a,
#     node_b=topo_r1,
#     attenuation_db_per_km=0.2
# )

# topo_link_r1_r2 = PhysicalQuantumLink(
#     node_a=topo_r1,
#     node_b=topo_r2,
#     attenuation_db_per_km=0.2
# )

# topo_link_r2_b = PhysicalQuantumLink(
#     node_a=topo_r2,
#     node_b=topo_b,
#     attenuation_db_per_km=0.2
# )

# for link in (
#     topo_link_a_r1,
#     topo_link_r1_r2,
#     topo_link_r2_b
# ):
#     topology_test.add_link(
#         link
#     )

# assert topology_test.link_count == 3

# print(
#     "[PASS] Physical links are registered correctly."
# )


# # =============================================================
# # Test 3 — Canonical undirected link identity
# # =============================================================

# assert topology_test.contains_link(
#     topo_a,
#     topo_r1
# )

# assert topology_test.contains_link(
#     topo_r1,
#     topo_a
# )

# assert (
#     topology_test.get_link(
#         topo_a,
#         topo_r1
#     )
#     is topology_test.get_link(
#         topo_r1,
#         topo_a
#     )
# )

# print(
#     "[PASS] Physical links behave as canonical undirected edges."
# )


# # =============================================================
# # Test 4 — Reverse duplicate link rejected
# # =============================================================

# duplicate_link_error = False

# try:

#     reversed_duplicate = PhysicalQuantumLink(
#         node_a=topo_r1,
#         node_b=topo_a,
#         attenuation_db_per_km=0.2
#     )

#     topology_test.add_link(
#         reversed_duplicate
#     )

# except TopologyLinkAlreadyExistsError:
#     duplicate_link_error = True

# assert duplicate_link_error

# print(
#     "[PASS] Reverse-direction duplicate physical link is rejected."
# )


# # =============================================================
# # Test 5 — Non-canonical endpoint object rejected
# # =============================================================

# noncanonical_error = False

# try:

#     fake_r1 = NetworkNode(
#         name="Topo-R1",
#         x_km=30,
#         y_km=40,
#         endpoint_capable=False,
#         swapping_capable=True,
#         memory_capacity=2
#     )

#     bad_link = PhysicalQuantumLink(
#         node_a=fake_r1,
#         node_b=topo_r2,
#         attenuation_db_per_km=0.2
#     )

#     topology_test.add_link(
#         bad_link
#     )

# except TopologyConsistencyError:
#     noncanonical_error = True

# assert noncanonical_error

# print(
#     "[PASS] Links referencing non-canonical node objects "
#     "are rejected."
# )


# # =============================================================
# # Test 6 — Neighbour and degree queries
# # =============================================================

# assert topology_test.degree(
#     topo_a
# ) == 1

# assert topology_test.degree(
#     topo_r1
# ) == 2

# assert topology_test.degree(
#     topo_r2
# ) == 2

# assert topology_test.degree(
#     topo_b
# ) == 1

# assert topology_test.degree(
#     topo_charlie
# ) == 0

# assert tuple(
#     node.name
#     for node in topology_test.neighbours(
#         topo_r1
#     )
# ) == (
#     "Topo-A",
#     "Topo-R2"
# )

# print(
#     "[PASS] Physical adjacency and degree queries are correct."
# )


# # =============================================================
# # Test 7 — Valid ordered physical path
# # =============================================================

# test_path = (
#     topo_a,
#     topo_r1,
#     topo_r2,
#     topo_b
# )

# validated_path = (
#     topology_test.validate_path(
#         test_path
#     )
# )

# assert validated_path == (
#     test_path
# )

# path_links = (
#     topology_test.path_links(
#         test_path
#     )
# )

# assert path_links == (
#     topo_link_a_r1,
#     topo_link_r1_r2,
#     topo_link_r2_b
# )

# print(
#     "[PASS] Explicit ordered physical path validates correctly."
# )


# # =============================================================
# # Test 8 — Physical path distance
# #
# # A -> R1 = 50 km
# # R1 -> R2 = 30 km
# # R2 -> B = 40 km
# # Total = 120 km
# # =============================================================

# assert math.isclose(
#     topology_test.path_distance_km(
#         test_path
#     ),
#     120.0,
#     rel_tol=1e-12
# )

# assert math.isclose(
#     topology_test.total_link_length_km,
#     120.0,
#     rel_tol=1e-12
# )

# print(
#     "[PASS] Physical path and total fibre lengths are correct."
# )


# # =============================================================
# # Test 9 — Missing physical edge invalidates path
# # =============================================================

# missing_edge_error = False

# try:
#     topology_test.validate_path(
#         (
#             topo_a,
#             topo_r1,
#             topo_b
#         )
#     )

# except InvalidPhysicalPathError:
#     missing_edge_error = True

# assert missing_edge_error

# print(
#     "[PASS] Path containing a missing physical link is rejected."
# )


# # =============================================================
# # Test 10 — Repeated nodes invalid in a simple path
# # =============================================================

# repeated_node_error = False

# try:
#     topology_test.validate_path(
#         (
#             topo_a,
#             topo_r1,
#             topo_a
#         )
#     )

# except InvalidPhysicalPathError:
#     repeated_node_error = True

# assert repeated_node_error

# print(
#     "[PASS] Repeated-node physical paths are rejected."
# )


# # =============================================================
# # Test 11 — Removing connected node is not silently allowed
# # =============================================================

# connected_removal_error = False

# try:
#     topology_test.remove_node(
#         topo_r1
#     )

# except TopologyConsistencyError:
#     connected_removal_error = True

# assert connected_removal_error

# assert topology_test.contains_node(
#     topo_r1
# )

# print(
#     "[PASS] Connected node cannot be removed "
#     "without explicit link handling."
# )


# # =============================================================
# # Test 12 — Disconnected node can be safely removed
# # =============================================================

# removed_charlie = (
#     topology_test.remove_node(
#         topo_charlie
#     )
# )

# assert removed_charlie is topo_charlie

# assert not topology_test.contains_node(
#     "Topo-Charlie"
# )

# assert topology_test.node_count == 4

# print(
#     "[PASS] Disconnected physical node can be removed safely."
# )


# # =============================================================
# # Test 13 — Read-only views do not expose graph internals
# # =============================================================

# assert isinstance(
#     topology_test.nodes,
#     tuple
# )

# assert isinstance(
#     topology_test.links,
#     tuple
# )

# assert isinstance(
#     topology_test.neighbours(topo_r1),
#     tuple
# )

# adjacency_copy = (
#     topology_test.adjacency_snapshot()
# )

# assert isinstance(
#     adjacency_copy["Topo-R1"],
#     tuple
# )

# print(
#     "[PASS] Public topology views do not expose mutable "
#     "internal adjacency state."
# )


# # =============================================================
# # Test 14 — Internal graph consistency
# # =============================================================

# assert (
#     topology_test.validate_internal_consistency()
#     is True
# )

# print(
#     "[PASS] Node, link, and adjacency registries are "
#     "internally consistent."
# )


# # =============================================================
# # Expected final topology:
# #
# # Topo-A -- Topo-R1 -- Topo-R2 -- Topo-B
# # =============================================================

# assert topology_test.node_count == 4
# assert topology_test.link_count == 3

# assert math.isclose(
#     topology_test.average_degree,
#     1.5
# )

# print(
#     "[PASS] Final topology statistics are correct."
# )


# # =============================================================
# # Inspection
# # =============================================================

# print("\nTOPOLOGY")
# print(topology_test)


# print("\nNODE TABLE")

# display(
#     topology_test.nodes_dataframe()
# )


# print("\nLINK TABLE")

# display(
#     topology_test.links_dataframe()
# )


# print("\nADJACENCY")

# print(
#     topology_test.adjacency_snapshot()
# )


# print("\nSUMMARY")

# print(
#     topology_test.summary()
# )


# print("\n" + "=" * 76)
# print("Phase 3A verification completed successfully.")
# print("=" * 76)

In [ ]:
# @title ML-DSA Cryptographic Provider

from dataclasses import dataclass, field
from enum import Enum
from typing import Final
import hashlib
import importlib
import secrets


# =============================================================
# ML-DSA Parameter Sets
# =============================================================

class MLDSAParameterSet(str, Enum):
    """
    FIPS 204 ML-DSA parameter sets.
    """

    ML_DSA_44 = "ML-DSA-44"
    ML_DSA_65 = "ML-DSA-65"
    ML_DSA_87 = "ML-DSA-87"


@dataclass(frozen=True)
class MLDSAParameterSpec:
    """
    Standard metadata for one ML-DSA parameter set.
    """

    parameter_set: MLDSAParameterSet

    python_module_name: str

    security_category: int

    public_key_bytes: int
    private_key_bytes: int
    signature_bytes: int


MLDSA_PARAMETER_SPECS: Final[
    dict[MLDSAParameterSet, MLDSAParameterSpec]
] = {

    MLDSAParameterSet.ML_DSA_44:
        MLDSAParameterSpec(
            parameter_set=(
                MLDSAParameterSet.ML_DSA_44
            ),
            python_module_name="ml_dsa_44",
            security_category=2,
            public_key_bytes=1312,
            private_key_bytes=2560,
            signature_bytes=2420
        ),

    MLDSAParameterSet.ML_DSA_65:
        MLDSAParameterSpec(
            parameter_set=(
                MLDSAParameterSet.ML_DSA_65
            ),
            python_module_name="ml_dsa_65",
            security_category=3,
            public_key_bytes=1952,
            private_key_bytes=4032,
            signature_bytes=3309
        ),

    MLDSAParameterSet.ML_DSA_87:
        MLDSAParameterSpec(
            parameter_set=(
                MLDSAParameterSet.ML_DSA_87
            ),
            python_module_name="ml_dsa_87",
            security_category=5,
            public_key_bytes=2592,
            private_key_bytes=4896,
            signature_bytes=4627
        )
}


# Default for NARAD-QC.
PHASE3_DEFAULT_MLDSA_PARAMETER_SET = (
    MLDSAParameterSet.ML_DSA_65
)


# =============================================================
# ML-DSA Provider
# =============================================================

class MLDSAProvider:
    """
    Thin algorithm adapter around the installed ML-DSA backend.

    The provider separates NARAD-QC architecture from a specific
    Python cryptographic library.

    Public responsibility:
        verify(public_key, message, signature)

    Private-key operations are intentionally marked internal:
        _generate_keypair()
        _sign_with_private_key()

    In the NARAD-QC architecture, these internal methods should
    only be used by SimulatedHSM.
    """

    ALGORITHM_NAME = "ML-DSA"

    def __init__(
        self,
        parameter_set: MLDSAParameterSet = (
            PHASE3_DEFAULT_MLDSA_PARAMETER_SET
        )
    ):

        if not isinstance(
            parameter_set,
            MLDSAParameterSet
        ):
            raise TypeError(
                "parameter_set must be an MLDSAParameterSet."
            )

        self.parameter_set = parameter_set

        self.spec = MLDSA_PARAMETER_SPECS[
            parameter_set
        ]

        module_path = (
            "pqcrypto.sign."
            f"{self.spec.python_module_name}"
        )

        try:
            self._backend = (
                importlib.import_module(
                    module_path
                )
            )

        except ImportError as exc:
            raise ImportError(
                "ML-DSA backend unavailable. "
                "Run the Phase 3A.7 pqcrypto "
                "installation cell first."
            ) from exc

    # =========================================================
    # Internal Secret-Key Operations
    # =========================================================

    def _generate_keypair(
        self
    ) -> tuple[bytes, bytes]:
        """
        Generate a fresh ML-DSA key pair.

        This function intentionally accepts NO SimulationKernel
        RNG and NO deterministic seed.

        The cryptographic backend obtains its randomness
        independently.
        """

        public_key, private_key = (
            self._backend.generate_keypair()
        )

        public_key = bytes(
            public_key
        )

        private_key = bytes(
            private_key
        )

        # Verify that the backend matches the selected
        # FIPS-204 parameter-set dimensions.
        if len(public_key) != (
            self.spec.public_key_bytes
        ):
            raise RuntimeError(
                "ML-DSA backend returned an unexpected "
                "public-key size."
            )

        if len(private_key) != (
            self.spec.private_key_bytes
        ):
            raise RuntimeError(
                "ML-DSA backend returned an unexpected "
                "private-key size."
            )

        return (
            public_key,
            private_key
        )

    def _sign_with_private_key(
        self,
        private_key: bytes,
        message: bytes
    ) -> bytes:
        """
        Internal signing operation.

        Private-key access should remain confined to SimulatedHSM.
        """

        message = self._validate_message(
            message
        )

        if not isinstance(
            private_key,
            bytes
        ):
            raise TypeError(
                "private_key must be bytes."
            )

        if len(private_key) != (
            self.spec.private_key_bytes
        ):
            raise ValueError(
                "Invalid private-key size for "
                f"{self.parameter_set.value}."
            )

        signature = (
            self._backend.sign(
                private_key,
                message
            )
        )

        signature = bytes(
            signature
        )

        if len(signature) != (
            self.spec.signature_bytes
        ):
            raise RuntimeError(
                "ML-DSA backend returned an unexpected "
                "signature size."
            )

        return signature

    # =========================================================
    # Public Verification Operation
    # =========================================================

    def verify(
        self,
        public_key: bytes,
        message: bytes,
        signature: bytes
    ) -> bool:
        """
        Verify an ML-DSA signature.

        Verification handles untrusted inputs defensively and
        returns False when signature/key validation fails.
        """

        try:

            message = self._validate_message(
                message
            )

            if not isinstance(
                public_key,
                bytes
            ):
                return False

            if not isinstance(
                signature,
                bytes
            ):
                return False

            if len(public_key) != (
                self.spec.public_key_bytes
            ):
                return False

            if len(signature) != (
                self.spec.signature_bytes
            ):
                return False

            return bool(
                self._backend.verify(
                    public_key,
                    message,
                    signature
                )
            )

        except Exception:
            return False

    # =========================================================
    # Message Validation
    # =========================================================

    @staticmethod
    def _validate_message(
        message
    ) -> bytes:

        if isinstance(
            message,
            bytearray
        ):
            message = bytes(
                message
            )

        if isinstance(
            message,
            memoryview
        ):
            message = message.tobytes()

        if not isinstance(
            message,
            bytes
        ):
            raise TypeError(
                "ML-DSA messages must be bytes-like."
            )

        return message

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:

        return (
            "MLDSAProvider("
            f"parameter_set="
            f"'{self.parameter_set.value}', "
            f"security_category="
            f"{self.spec.security_category}"
            ")"
        )

In [ ]:
# @title Public ML-DSA Device Identity


@dataclass(frozen=True)
class MLDSAPublicIdentity:
    """
    Exportable public cryptographic identity of one NARAD-QC
    physical device or controller.

    Contains NO private-key material.
    """

    owner_id: str

    parameter_set: MLDSAParameterSet

    public_key: bytes

    fingerprint_sha256: str = field(
        init=False
    )

    key_id: str = field(
        init=False
    )

    def __post_init__(self):

        # =====================================================
        # Owner Identity
        # =====================================================

        if (
            not isinstance(self.owner_id, str)
            or not self.owner_id.strip()
        ):
            raise ValueError(
                "owner_id must be a non-empty string."
            )

        object.__setattr__(
            self,
            "owner_id",
            self.owner_id.strip()
        )

        # =====================================================
        # Parameter Set
        # =====================================================

        if not isinstance(
            self.parameter_set,
            MLDSAParameterSet
        ):
            raise TypeError(
                "parameter_set must be an MLDSAParameterSet."
            )

        spec = MLDSA_PARAMETER_SPECS[
            self.parameter_set
        ]

        # =====================================================
        # Public Key
        # =====================================================

        if not isinstance(
            self.public_key,
            bytes
        ):
            raise TypeError(
                "public_key must be bytes."
            )

        if len(self.public_key) != (
            spec.public_key_bytes
        ):
            raise ValueError(
                "public_key size does not match "
                f"{self.parameter_set.value}."
            )

        # =====================================================
        # Stable Public Fingerprint
        # =====================================================

        fingerprint = (
            hashlib.sha256(
                self.public_key
            ).hexdigest()
        )

        object.__setattr__(
            self,
            "fingerprint_sha256",
            fingerprint
        )

        # Short human-readable identifier.
        #
        # Full fingerprint remains available separately.
        key_id = (
            f"{self.parameter_set.value}:"
            f"{fingerprint[:24]}"
        )

        object.__setattr__(
            self,
            "key_id",
            key_id
        )

    # =========================================================
    # Convenience Properties
    # =========================================================

    @property
    def algorithm(self) -> str:
        return "ML-DSA"

    @property
    def public_key_size_bytes(
        self
    ) -> int:

        return len(
            self.public_key
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(self) -> str:

        return (
            "MLDSAPublicIdentity("
            f"owner='{self.owner_id}', "
            f"parameter_set="
            f"'{self.parameter_set.value}', "
            f"key_id='{self.key_id}'"
            ")"
        )

In [ ]:
# @title Simulated Hardware Security Module


class SimulatedHSM:
    """
    Simulation abstraction of a device Hardware Security Module.

    Security-model contract
    -----------------------
    1. The ML-DSA key pair is generated internally.
    2. The private key is retained internally.
    3. No public method exports the private key.
    4. The HSM exposes only:
         - public identity;
         - signing.
    5. Key material cannot be replaced through the normal API
       after provisioning.
    6. Cryptographic key generation is independent of the seeded
       SimulationKernel RNG.

    Important:
    This is a SOFTWARE SIMULATION of an HSM boundary. The class models the
    architectural interface of such hardware and in no context does it claim
    real HSM security.
    """

    __slots__ = (
        "__owner_id",
        "__provider",
        "__public_key",
        "__private_key",
        "__public_identity",
        "__sealed"
    )

    def __init__(
        self,
        owner_id: str,
        parameter_set: MLDSAParameterSet = (
            PHASE3_DEFAULT_MLDSA_PARAMETER_SET
        )
    ):

        object.__setattr__(
            self,
            "_SimulatedHSM__sealed",
            False
        )

        # =====================================================
        # Device Identity
        # =====================================================

        if (
            not isinstance(owner_id, str)
            or not owner_id.strip()
        ):
            raise ValueError(
                "owner_id must be a non-empty string."
            )

        owner_id = owner_id.strip()

        object.__setattr__(
            self,
            "_SimulatedHSM__owner_id",
            owner_id
        )

        # =====================================================
        # ML-DSA Provider
        # =====================================================

        provider = MLDSAProvider(
            parameter_set=parameter_set
        )

        object.__setattr__(
            self,
            "_SimulatedHSM__provider",
            provider
        )

        # =====================================================
        # INTERNAL KEY GENERATION
        #
        # The key pair is generated here inside the HSM abstraction.
        #
        # No caller supplies or receives the private key.
        # =====================================================

        public_key, private_key = (
            provider._generate_keypair()
        )

        object.__setattr__(
            self,
            "_SimulatedHSM__public_key",
            public_key
        )

        object.__setattr__(
            self,
            "_SimulatedHSM__private_key",
            private_key
        )

        # =====================================================
        # Exportable Public Identity
        # =====================================================

        public_identity = (
            MLDSAPublicIdentity(
                owner_id=owner_id,
                parameter_set=parameter_set,
                public_key=public_key
            )
        )

        object.__setattr__(
            self,
            "_SimulatedHSM__public_identity",
            public_identity
        )

        # Remove local constructor references.
        #
        # This does NOT claim secure memory zeroization in Python.
        # It merely ensures no caller-side reference is returned.
        del private_key
        del public_key

        # =====================================================
        # Seal HSM Configuration
        # =====================================================

        object.__setattr__(
            self,
            "_SimulatedHSM__sealed",
            True
        )

    # =========================================================
    # Prevent Normal Post-Provisioning Mutation
    # =========================================================

    def __setattr__(
        self,
        name,
        value
    ):

        try:
            sealed = object.__getattribute__(
                self,
                "_SimulatedHSM__sealed"
            )

        except AttributeError:
            sealed = False

        if sealed:
            raise AttributeError(
                "SimulatedHSM is sealed after provisioning. "
                "Key material and HSM identity cannot be "
                "modified through the normal API."
            )

        object.__setattr__(
            self,
            name,
            value
        )

    # =========================================================
    # Public Metadata
    # =========================================================

    @property
    def owner_id(
        self
    ) -> str:

        return object.__getattribute__(
            self,
            "_SimulatedHSM__owner_id"
        )

    @property
    def parameter_set(
        self
    ) -> MLDSAParameterSet:

        provider = object.__getattribute__(
            self,
            "_SimulatedHSM__provider"
        )

        return provider.parameter_set

    @property
    def public_identity(
        self
    ) -> MLDSAPublicIdentity:
        """
        Export the device's PUBLIC cryptographic identity.

        No private-key material is returned.
        """

        return object.__getattribute__(
            self,
            "_SimulatedHSM__public_identity"
        )

    @property
    def public_key(
        self
    ) -> bytes:
        """
        Exportable public key.
        """

        identity = self.public_identity

        return identity.public_key

    @property
    def key_id(
        self
    ) -> str:

        return self.public_identity.key_id

    @property
    def public_key_size_bytes(
        self
    ) -> int:

        return len(
            self.public_key
        )

    @property
    def private_key_size_bytes(
        self
    ) -> int:
        """
        Exposes only private-key SIZE metadata,
        never private-key contents.
        """

        provider = object.__getattribute__(
            self,
            "_SimulatedHSM__provider"
        )

        return (
            provider.spec.private_key_bytes
        )

    # =========================================================
    # Signing Interface
    # =========================================================

    def sign(
        self,
        message: bytes
    ) -> bytes:
        """
        Ask the HSM to sign a message.

        The caller receives a signature, not the private key.
        """

        provider = object.__getattribute__(
            self,
            "_SimulatedHSM__provider"
        )

        private_key = object.__getattribute__(
            self,
            "_SimulatedHSM__private_key"
        )

        return (
            provider._sign_with_private_key(
                private_key=private_key,
                message=message
            )
        )

    # =========================================================
    # Local Convenience Verification
    # =========================================================

    def verify_own_signature(
        self,
        message: bytes,
        signature: bytes
    ) -> bool:
        """
        Convenience self-test only.

        Runtime authentication will later be performed by
        external parties using the exported public identity.
        """

        provider = object.__getattribute__(
            self,
            "_SimulatedHSM__provider"
        )

        return provider.verify(
            public_key=self.public_key,
            message=message,
            signature=signature
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "SimulatedHSM("
            f"owner='{self.owner_id}', "
            f"parameter_set="
            f"'{self.parameter_set.value}', "
            f"key_id='{self.key_id}', "
            f"sealed=True"
            ")"
        )

In [ ]:
# @title Cryptographic Randomness


def secure_random_bytes(
    length: int = 32
) -> bytes:
    """
    Generate cryptographically secure random bytes for
    security-protocol material such as authentication nonces.
    """

    if (
        not isinstance(length, int)
        or isinstance(length, bool)
        or length <= 0
    ):
        raise ValueError(
            "length must be a positive integer."
        )

    return secrets.token_bytes(
        length
    )

In [ ]:
# @title ML-DSA Cryptographic Foundation Verification

print("=" * 76)
print("NARAD-QC — Cryptographic Foundation Verification")
print("=" * 76)


# =============================================================
# Test 1 — Provider uses ML-DSA-65 by default
# =============================================================

crypto_test_provider = MLDSAProvider()

assert (
    crypto_test_provider.parameter_set
    == MLDSAParameterSet.ML_DSA_65
)

assert (
    crypto_test_provider.spec.security_category
    == 3
)

print(
    "[PASS] Default cryptographic profile is ML-DSA-65."
)


# =============================================================
# Test 2 — Device HSM generates its identity internally
# =============================================================

crypto_test_alice_hsm = (
    SimulatedHSM(
        owner_id="Crypto-Alice"
    )
)

assert (
    crypto_test_alice_hsm.owner_id
    == "Crypto-Alice"
)

assert (
    crypto_test_alice_hsm.parameter_set
    == MLDSAParameterSet.ML_DSA_65
)

print(
    "[PASS] Device HSM generates an internal "
    "ML-DSA identity."
)


# =============================================================
# Test 3 — Standard key dimensions are correct
# =============================================================

assert (
    crypto_test_alice_hsm.public_key_size_bytes
    == 1952
)

assert (
    crypto_test_alice_hsm.private_key_size_bytes
    == 4032
)

print(
    "[PASS] ML-DSA-65 key dimensions match "
    "the configured standard profile."
)


# =============================================================
# Test 4 — Public identity contains only public material
# =============================================================

alice_public_identity = (
    crypto_test_alice_hsm.public_identity
)

assert isinstance(
    alice_public_identity,
    MLDSAPublicIdentity
)

assert (
    alice_public_identity.owner_id
    == "Crypto-Alice"
)

assert (
    alice_public_identity.public_key
    == crypto_test_alice_hsm.public_key
)

assert not hasattr(
    alice_public_identity,
    "private_key"
)

assert not hasattr(
    alice_public_identity,
    "secret_key"
)

print(
    "[PASS] Exported device identity contains "
    "no private-key field."
)


# =============================================================
# Test 5 — HSM exposes no normal private-key export API
# =============================================================

assert not hasattr(
    crypto_test_alice_hsm,
    "private_key"
)

assert not hasattr(
    crypto_test_alice_hsm,
    "secret_key"
)

assert not hasattr(
    crypto_test_alice_hsm,
    "export_private_key"
)

print(
    "[PASS] HSM exposes no normal private-key export API."
)


# =============================================================
# Test 6 — HSM signs without exposing private key
# =============================================================

test_message = (
    b"NARAD-QC Phase 3A.7 authentication foundation"
)

alice_signature = (
    crypto_test_alice_hsm.sign(
        test_message
    )
)

assert len(
    alice_signature
) == 3309

assert (
    crypto_test_alice_hsm.verify_own_signature(
        message=test_message,
        signature=alice_signature
    )
    is True
)

print(
    "[PASS] HSM signs internally and produces a valid "
    "ML-DSA-65 signature."
)


# =============================================================
# Test 7 — External verification needs only public identity
# =============================================================

external_verifier = MLDSAProvider(
    parameter_set=(
        alice_public_identity.parameter_set
    )
)

assert external_verifier.verify(
    public_key=(
        alice_public_identity.public_key
    ),
    message=test_message,
    signature=alice_signature
)

print(
    "[PASS] Signature verifies externally using only "
    "the exported public key."
)


# =============================================================
# Test 8 — Message tampering is detected
# =============================================================

tampered_message = (
    b"NARAD-QC Phase 3A.7 authentication foundation!"
)

assert not external_verifier.verify(
    public_key=(
        alice_public_identity.public_key
    ),
    message=tampered_message,
    signature=alice_signature
)

print(
    "[PASS] Modified message fails ML-DSA verification."
)


# =============================================================
# Test 9 — Different physical device gets independent key pair
# =============================================================

crypto_test_bob_hsm = (
    SimulatedHSM(
        owner_id="Crypto-Bob"
    )
)

assert (
    crypto_test_bob_hsm.public_key
    != crypto_test_alice_hsm.public_key
)

assert (
    crypto_test_bob_hsm.key_id
    != crypto_test_alice_hsm.key_id
)

print(
    "[PASS] Independent device HSMs receive "
    "independent cryptographic identities."
)


# =============================================================
# Test 10 — Wrong device public key cannot verify signature
# =============================================================

assert not external_verifier.verify(
    public_key=(
        crypto_test_bob_hsm.public_key
    ),
    message=test_message,
    signature=alice_signature
)

print(
    "[PASS] Signature cannot be verified with another "
    "device's public key."
)


# =============================================================
# Test 11 — HSM is sealed after provisioning
# =============================================================

sealed_error = False

try:
    crypto_test_alice_hsm.owner_id = "Changed-Alice"

except AttributeError:
    sealed_error = True

assert sealed_error

print(
    "[PASS] HSM configuration is sealed after provisioning."
)


# =============================================================
# Test 12 — Public identity itself is immutable
# =============================================================

public_identity_immutable = False

try:
    alice_public_identity.owner_id = "Changed"

except Exception:
    public_identity_immutable = True

assert public_identity_immutable

print(
    "[PASS] Exported public identity record is immutable."
)


# =============================================================
# Test 13 — Cryptographic nonces are generated independently
# =============================================================

nonce_a = secure_random_bytes(
    32
)

nonce_b = secure_random_bytes(
    32
)

assert len(nonce_a) == 32
assert len(nonce_b) == 32
assert nonce_a != nonce_b

print(
    "[PASS] Cryptographic randomness is available "
    "outside the seeded simulation RNG."
)


# =============================================================
# Inspection
# =============================================================

print("\nALICE HSM")
print(
    crypto_test_alice_hsm
)

print("\nALICE PUBLIC IDENTITY")
print(
    alice_public_identity
)

print("\nBOB HSM")
print(
    crypto_test_bob_hsm
)


print("\nPUBLIC CRYPTOGRAPHIC METADATA")

crypto_metadata = pd.DataFrame([
    {
        "Owner":
            crypto_test_alice_hsm.owner_id,

        "Algorithm":
            alice_public_identity.algorithm,

        "Parameter Set":
            alice_public_identity.parameter_set.value,

        "Key ID":
            alice_public_identity.key_id,

        "Public Key Bytes":
            alice_public_identity.public_key_size_bytes,

        "Private Key Bytes":
            crypto_test_alice_hsm.private_key_size_bytes
    },

    {
        "Owner":
            crypto_test_bob_hsm.owner_id,

        "Algorithm":
            crypto_test_bob_hsm.public_identity.algorithm,

        "Parameter Set":
            crypto_test_bob_hsm.parameter_set.value,

        "Key ID":
            crypto_test_bob_hsm.key_id,

        "Public Key Bytes":
            crypto_test_bob_hsm.public_key_size_bytes,

        "Private Key Bytes":
            crypto_test_bob_hsm.private_key_size_bytes
    }
])

display(
    crypto_metadata
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

In [ ]:
# @title Control-Plane Trust Records and Errors

from dataclasses import dataclass
from types import MappingProxyType
from typing import Optional


# =============================================================
# Control-Plane / Provisioning Errors
# =============================================================

class ClassicalControlPlaneError(Exception):
    """
    Base exception for reconstructed Phase-3 classical
    control-plane operations.
    """
    pass


class PeerAlreadyEnrolledError(
    ClassicalControlPlaneError
):
    """
    Raised when a physical device identity is already enrolled.
    """
    pass


class PeerNotEnrolledError(
    ClassicalControlPlaneError
):
    """
    Raised when the control plane has no trusted public identity
    for a requested physical device.
    """
    pass


class TrustedControllerNotFoundError(
    ClassicalControlPlaneError
):
    """
    Raised when a node runtime does not recognize a requested
    controller identity.
    """
    pass


class ProvisioningConsistencyError(
    ClassicalControlPlaneError
):
    """
    Raised when trusted provisioning would create an inconsistent
    identity relationship.
    """
    pass


# =============================================================
# Trusted Peer Record
# =============================================================

@dataclass(frozen=True)
class TrustedPeerRecord:
    """
    Immutable trusted-public-identity record maintained by the
    NARAD-QC classical control plane.

    This record means:

        "During trusted provisioning, the network established
         that this ML-DSA public identity belongs to this
         physical node."

    It does NOT mean that the node is currently authenticated.
    """

    node_name: str

    public_identity: MLDSAPublicIdentity

    enrollment_index: int

    def __post_init__(self):

        if (
            not isinstance(self.node_name, str)
            or not self.node_name.strip()
        ):
            raise ValueError(
                "node_name must be a non-empty string."
            )

        node_name = self.node_name.strip()

        object.__setattr__(
            self,
            "node_name",
            node_name
        )

        if not isinstance(
            self.public_identity,
            MLDSAPublicIdentity
        ):
            raise TypeError(
                "public_identity must be an "
                "MLDSAPublicIdentity."
            )

        # The cryptographic identity must explicitly belong
        # to the same physical device.
        if (
            self.public_identity.owner_id
            != node_name
        ):
            raise ProvisioningConsistencyError(
                f"Public identity owner "
                f"'{self.public_identity.owner_id}' does not "
                f"match physical node '{node_name}'."
            )

        if (
            isinstance(self.enrollment_index, bool)
            or not isinstance(
                self.enrollment_index,
                int
            )
            or self.enrollment_index < 0
        ):
            raise ValueError(
                "enrollment_index must be a non-negative integer."
            )

    @property
    def algorithm(self) -> str:
        return self.public_identity.algorithm

    @property
    def parameter_set(
        self
    ) -> MLDSAParameterSet:

        return (
            self.public_identity.parameter_set
        )

    @property
    def key_id(self) -> str:
        return self.public_identity.key_id

    @property
    def fingerprint_sha256(self) -> str:
        return (
            self.public_identity.fingerprint_sha256
        )

In [ ]:
# @title Physical Node Device Runtime


class NodeDeviceRuntime:
    """
    Runtime representation of one physical NARAD-QC network node.

    The immutable NetworkNode describes the physical device.
    NodeDeviceRuntime represents its operating cryptographic
    context.

    Current Phase-3A responsibilities
    ---------------------------------
    - own one simulated device HSM;
    - generate the device ML-DSA identity internally;
    - expose only the public identity;
    - pin trusted controller public identity during provisioning;
    - request signatures from the local HSM;
    - verify signatures using a pinned controller identity.

    It deliberately does NOT yet:
    - maintain authentication state;
    - execute challenge-response;
    - manage quantum memory;
    - generate entanglement;
    - perform swapping;
    - make routing decisions.

    Trusted controller provisioning is assumed to occur through
    a secure out-of-band bootstrap process.
    """

    __slots__ = (
        "__physical_node",
        "__device_hsm",
        "__trusted_controllers"
    )

    def __init__(
        self,
        physical_node: NetworkNode,
        trusted_controller_identity: MLDSAPublicIdentity,
        parameter_set: MLDSAParameterSet = (
            PHASE3_DEFAULT_MLDSA_PARAMETER_SET
        )
    ):

        # =====================================================
        # Physical Device
        # =====================================================

        if not isinstance(
            physical_node,
            NetworkNode
        ):
            raise TypeError(
                "physical_node must be a NetworkNode."
            )

        if not isinstance(
            trusted_controller_identity,
            MLDSAPublicIdentity
        ):
            raise TypeError(
                "trusted_controller_identity must be an "
                "MLDSAPublicIdentity."
            )

        if (
            physical_node.name
            == trusted_controller_identity.owner_id
        ):
            raise ProvisioningConsistencyError(
                "A physical node cannot use its own identity "
                "as the trusted controller identity."
            )

        object.__setattr__(
            self,
            "_NodeDeviceRuntime__physical_node",
            physical_node
        )

        # =====================================================
        # Device HSM
        #
        # Key pair is generated INSIDE the simulated HSM.
        # No controller or caller receives the private key.
        # =====================================================

        device_hsm = SimulatedHSM(
            owner_id=physical_node.name,
            parameter_set=parameter_set
        )

        object.__setattr__(
            self,
            "_NodeDeviceRuntime__device_hsm",
            device_hsm
        )

        # =====================================================
        # Trusted Controller Registry
        #
        # MappingProxyType gives us a read-only mapping.
        # For the current architecture only one controller is
        # provisioned, but the representation does not prevent
        # future multi-controller designs.
        # =====================================================

        trusted_controllers = MappingProxyType({
            trusted_controller_identity.owner_id:
                trusted_controller_identity
        })

        object.__setattr__(
            self,
            "_NodeDeviceRuntime__trusted_controllers",
            trusted_controllers
        )

    # =========================================================
    # Physical Identity
    # =========================================================

    @property
    def physical_node(
        self
    ) -> NetworkNode:

        return object.__getattribute__(
            self,
            "_NodeDeviceRuntime__physical_node"
        )

    @property
    def node_name(
        self
    ) -> str:

        return self.physical_node.name

    # =========================================================
    # Exportable Cryptographic Identity
    # =========================================================

    @property
    def public_identity(
        self
    ) -> MLDSAPublicIdentity:

        hsm = object.__getattribute__(
            self,
            "_NodeDeviceRuntime__device_hsm"
        )

        return hsm.public_identity

    @property
    def key_id(
        self
    ) -> str:

        return self.public_identity.key_id

    # =========================================================
    # Trusted Controller Inspection
    # =========================================================

    @property
    def trusted_controller_ids(
        self
    ) -> tuple[str, ...]:

        registry = object.__getattribute__(
            self,
            "_NodeDeviceRuntime__trusted_controllers"
        )

        return tuple(
            sorted(
                registry.keys()
            )
        )

    def trusts_controller(
        self,
        controller_id: str
    ) -> bool:

        if (
            not isinstance(controller_id, str)
            or not controller_id.strip()
        ):
            raise ValueError(
                "controller_id must be a non-empty string."
            )

        registry = object.__getattribute__(
            self,
            "_NodeDeviceRuntime__trusted_controllers"
        )

        return (
            controller_id.strip()
            in registry
        )

    def trusted_controller_identity(
        self,
        controller_id: str
    ) -> MLDSAPublicIdentity:

        if (
            not isinstance(controller_id, str)
            or not controller_id.strip()
        ):
            raise ValueError(
                "controller_id must be a non-empty string."
            )

        controller_id = controller_id.strip()

        registry = object.__getattribute__(
            self,
            "_NodeDeviceRuntime__trusted_controllers"
        )

        if controller_id not in registry:
            raise TrustedControllerNotFoundError(
                f"Node '{self.node_name}' does not trust "
                f"controller '{controller_id}'."
            )

        return registry[
            controller_id
        ]

    # =========================================================
    # Local HSM Signing
    # =========================================================

    def sign(
        self,
        message: bytes
    ) -> bytes:
        """
        Ask the local device HSM to sign a message.

        Private-key material never leaves the HSM abstraction.
        """

        hsm = object.__getattribute__(
            self,
            "_NodeDeviceRuntime__device_hsm"
        )

        return hsm.sign(
            message
        )

    # =========================================================
    # Trusted Controller Verification Primitive
    # =========================================================

    def verify_trusted_controller_signature(
        self,
        controller_id: str,
        message: bytes,
        signature: bytes
    ) -> bool:
        """
        Verify a signature against the PUBLIC identity pinned
        for a trusted controller.

        Important:
        Successful signature verification alone does NOT yet
        constitute runtime authentication.

        Phase 3A.9 will add:
        - fresh challenges;
        - authentication-session IDs;
        - replay prevention;
        - context binding;
        - mutual-authentication state.
        """

        controller_identity = (
            self.trusted_controller_identity(
                controller_id
            )
        )

        provider = MLDSAProvider(
            parameter_set=(
                controller_identity.parameter_set
            )
        )

        return provider.verify(
            public_key=(
                controller_identity.public_key
            ),
            message=message,
            signature=signature
        )

    # =========================================================
    # HSM Security-Surface Metadata
    # =========================================================

    @property
    def public_key_size_bytes(
        self
    ) -> int:

        return (
            self.public_identity.public_key_size_bytes
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "NodeDeviceRuntime("
            f"node='{self.node_name}', "
            f"parameter_set="
            f"'{self.public_identity.parameter_set.value}', "
            f"key_id='{self.key_id}', "
            f"trusted_controllers="
            f"{len(self.trusted_controller_ids)}"
            ")"
        )

In [ ]:
# @title The big player is born — Classical Control Plane Foundation


class ClassicalControlPlane:
    """
    Central classical trust/control authority for NARAD-QC.

    Phase 3A responsibilities
    ---------------------------
    - create the controller's own simulated HSM first;
    - expose the controller's public ML-DSA identity;
    - trusted-provision physical network devices;
    - maintain the authoritative trusted-peer public-key registry;
    - verify signatures using enrolled peer public identities;
    - sign control-plane material using the controller HSM.

    The control plane deliberately does NOT yet:
    - mark peers AUTHENTICATED;
    - execute mutual challenge-response;
    - route traffic;
    - reserve resources;
    - generate entanglement;
    - perform swapping;
    - own quantum memories.

    Trusted provisioning is modeled as a secure out-of-band
    bootstrap process.
    """

    DEFAULT_CONTROLLER_ID = (
        "NARAD-QC-CONTROLLER"
    )

    def __init__(
        self,
        controller_id: str = DEFAULT_CONTROLLER_ID,
        parameter_set: MLDSAParameterSet = (
            PHASE3_DEFAULT_MLDSA_PARAMETER_SET
        )
    ):

        # =====================================================
        # Controller Identity
        # =====================================================

        if (
            not isinstance(controller_id, str)
            or not controller_id.strip()
        ):
            raise ValueError(
                "controller_id must be a non-empty string."
            )

        self._controller_id = (
            controller_id.strip()
        )

        # Controller HSM is created FIRST.
        #
        # Its private key is generated internally and never
        # exported.
        self.__controller_hsm = SimulatedHSM(
            owner_id=self._controller_id,
            parameter_set=parameter_set
        )

        # =====================================================
        # Trusted Peer Registry
        # =====================================================

        self._trusted_peers: dict[
            str,
            TrustedPeerRecord
        ] = {}

        self._next_enrollment_index = 0

    # =========================================================
    # Controller Public Identity
    # =========================================================

    @property
    def controller_id(
        self
    ) -> str:

        return self._controller_id

    @property
    def public_identity(
        self
    ) -> MLDSAPublicIdentity:
        """
        Export controller PUBLIC identity only.
        """

        return (
            self.__controller_hsm.public_identity
        )

    @property
    def parameter_set(
        self
    ) -> MLDSAParameterSet:

        return (
            self.public_identity.parameter_set
        )

    @property
    def key_id(
        self
    ) -> str:

        return self.public_identity.key_id

    # =========================================================
    # Controller Signing Primitive
    # =========================================================

    def sign_control_message(
        self,
        message: bytes
    ) -> bytes:
        """
        Sign classical control-plane material inside the
        controller HSM.
        """

        return (
            self.__controller_hsm.sign(
                message
            )
        )

    # =========================================================
    # Trusted Device Provisioning
    # =========================================================

    def provision_node(
        self,
        node: NetworkNode,
        parameter_set: MLDSAParameterSet = (
            PHASE3_DEFAULT_MLDSA_PARAMETER_SET
        )
    ) -> NodeDeviceRuntime:
        """
        Trusted out-of-band provisioning of one physical node.

        Process
        -------
        1. Confirm node is not already enrolled.
        2. Create the node runtime.
        3. Node HSM generates its ML-DSA key pair internally.
        4. Controller public identity is pinned into the node.
        5. Node exports only its public identity.
        6. Control plane stores that public identity in its
           Trusted Peer Registry.
        7. Node runtime is returned to the simulation owner.

        The control plane never receives node private-key material.
        """

        if not isinstance(
            node,
            NetworkNode
        ):
            raise TypeError(
                "node must be a NetworkNode."
            )

        if (
            node.name
            == self.controller_id
        ):
            raise ProvisioningConsistencyError(
                "A physical node cannot reuse the "
                "controller's identity."
            )

        if node.name in self._trusted_peers:
            raise PeerAlreadyEnrolledError(
                f"Node '{node.name}' is already enrolled "
                f"in the trusted peer registry."
            )

        # =====================================================
        # Create node runtime
        #
        # Its HSM generates SK_i internally.
        # The controller never sees SK_i.
        # =====================================================

        runtime = NodeDeviceRuntime(
            physical_node=node,
            trusted_controller_identity=(
                self.public_identity
            ),
            parameter_set=parameter_set
        )

        node_public_identity = (
            runtime.public_identity
        )

        # =====================================================
        # Defensive key-identity uniqueness check
        # =====================================================

        for existing_record in (
            self._trusted_peers.values()
        ):

            if (
                existing_record.fingerprint_sha256
                == node_public_identity.fingerprint_sha256
            ):
                raise ProvisioningConsistencyError(
                    "The newly provisioned node produced a "
                    "public key already associated with another "
                    "trusted peer."
                )

        # =====================================================
        # Enroll PUBLIC identity only
        # =====================================================

        record = TrustedPeerRecord(
            node_name=node.name,

            public_identity=(
                node_public_identity
            ),

            enrollment_index=(
                self._next_enrollment_index
            )
        )

        self._next_enrollment_index += 1

        self._trusted_peers[
            node.name
        ] = record

        return runtime

    # =========================================================
    # Registry Queries
    # =========================================================

    @property
    def peer_count(
        self
    ) -> int:

        return len(
            self._trusted_peers
        )

    @property
    def trusted_peer_records(
        self
    ) -> tuple[TrustedPeerRecord, ...]:
        """
        Immutable deterministic view of trusted peer records.
        """

        return tuple(
            self._trusted_peers[
                node_name
            ]
            for node_name
            in sorted(
                self._trusted_peers
            )
        )

    def is_enrolled(
        self,
        node_or_name
    ) -> bool:

        node_name = self._resolve_node_name(
            node_or_name
        )

        return (
            node_name
            in self._trusted_peers
        )

    def get_peer_record(
        self,
        node_or_name
    ) -> TrustedPeerRecord:

        node_name = self._resolve_node_name(
            node_or_name
        )

        if node_name not in self._trusted_peers:
            raise PeerNotEnrolledError(
                f"Node '{node_name}' is not enrolled "
                f"in the trusted peer registry."
            )

        return self._trusted_peers[
            node_name
        ]

    def peer_public_identity(
        self,
        node_or_name
    ) -> MLDSAPublicIdentity:

        return (
            self.get_peer_record(
                node_or_name
            ).public_identity
        )

    # =========================================================
    # Registered-Peer Signature Verification Primitive
    # =========================================================

    def verify_registered_peer_signature(
        self,
        node_or_name,
        message: bytes,
        signature: bytes
    ) -> bool:
        """
        Verify a signature using the public identity enrolled
        for a physical peer.

        Important:
        This proves only that the signature is valid for the
        enrolled public key.

        It is NOT yet a complete authentication protocol because
        this method contains no freshness or replay protection.
        """

        identity = (
            self.peer_public_identity(
                node_or_name
            )
        )

        provider = MLDSAProvider(
            parameter_set=(
                identity.parameter_set
            )
        )

        return provider.verify(
            public_key=identity.public_key,
            message=message,
            signature=signature
        )

    # =========================================================
    # Registry Inspection
    # =========================================================

    def trusted_peer_registry_dataframe(
        self
    ) -> pd.DataFrame:
        """
        Public metadata view of the trusted-peer registry.

        Raw private keys never appear here.
        """

        rows = []

        for record in (
            self.trusted_peer_records
        ):

            rows.append({
                "Node":
                    record.node_name,

                "Algorithm":
                    record.algorithm,

                "Parameter Set":
                    record.parameter_set.value,

                "Security Category":
                    MLDSA_PARAMETER_SPECS[
                        record.parameter_set
                    ].security_category,

                "Enrollment Index":
                    record.enrollment_index,

                "Key ID":
                    record.key_id,

                "SHA-256 Fingerprint":
                    record.fingerprint_sha256,

                "Public Key Bytes":
                    record.public_identity.public_key_size_bytes
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Internal Helpers
    # =========================================================

    @staticmethod
    def _resolve_node_name(
        node_or_name
    ) -> str:

        if isinstance(
            node_or_name,
            NetworkNode
        ):
            return node_or_name.name

        if (
            isinstance(node_or_name, str)
            and node_or_name.strip()
        ):
            return node_or_name.strip()

        raise TypeError(
            "Expected a NetworkNode or non-empty "
            "node-name string."
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "ClassicalControlPlane("
            f"controller_id='{self.controller_id}', "
            f"parameter_set="
            f"'{self.parameter_set.value}', "
            f"trusted_peers={self.peer_count}"
            ")"
        )

In [ ]:
# @title Classical Control Plane Verification


print("=" * 76)
print("NARAD-QC - Classical Control Plane Verification")
print("=" * 76)


# =============================================================
# Test 1 — Controller must exist before any network device
# =============================================================

cp_test = ClassicalControlPlane(
    controller_id="TEST-NARAD-CONTROLLER"
)

assert (
    cp_test.controller_id
    == "TEST-NARAD-CONTROLLER"
)

assert (
    cp_test.parameter_set
    == MLDSAParameterSet.ML_DSA_65
)

assert cp_test.peer_count == 0

print(
    "[PASS] Classical control plane initializes first "
    "with its own ML-DSA identity."
)


# =============================================================
# Test physical devices
# =============================================================

cp_test_alice_node = NetworkNode(
    name="CP-Alice",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)

cp_test_r1_node = NetworkNode(
    name="CP-R1",
    x_km=20,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


# =============================================================
# Test 2 — Trusted provisioning creates node runtime
# =============================================================

cp_test_alice_runtime = (
    cp_test.provision_node(
        cp_test_alice_node
    )
)

assert isinstance(
    cp_test_alice_runtime,
    NodeDeviceRuntime
)

assert (
    cp_test_alice_runtime.physical_node
    is cp_test_alice_node
)

assert (
    cp_test_alice_runtime.node_name
    == "CP-Alice"
)

print(
    "[PASS] Trusted provisioning creates a physical "
    "node runtime."
)


# =============================================================
# Test 3 — Controller receives only node public identity
# =============================================================

assert cp_test.is_enrolled(
    cp_test_alice_node
)

alice_record = cp_test.get_peer_record(
    cp_test_alice_node
)

assert (
    alice_record.public_identity
    == cp_test_alice_runtime.public_identity
)

assert not hasattr(
    alice_record.public_identity,
    "private_key"
)

assert not hasattr(
    alice_record.public_identity,
    "secret_key"
)

print(
    "[PASS] Trusted peer registry stores only the "
    "device's public cryptographic identity."
)


# =============================================================
# Test 4 — Device pins controller public identity
# =============================================================

assert (
    cp_test_alice_runtime.trusts_controller(
        cp_test.controller_id
    )
)

alice_controller_identity = (
    cp_test_alice_runtime.trusted_controller_identity(
        cp_test.controller_id
    )
)

assert (
    alice_controller_identity
    == cp_test.public_identity
)

print(
    "[PASS] Provisioned node pins the controller's "
    "public identity."
)


# =============================================================
# Test 5 — Device signs locally, controller verifies
# =============================================================

peer_test_message = (
    b"NARAD-QC trusted-peer signature verification"
)

alice_peer_signature = (
    cp_test_alice_runtime.sign(
        peer_test_message
    )
)

assert cp_test.verify_registered_peer_signature(
    node_or_name=cp_test_alice_node,
    message=peer_test_message,
    signature=alice_peer_signature
)

print(
    "[PASS] Controller verifies a device signature using "
    "only its trusted peer registry."
)


# =============================================================
# Test 6 — Controller signs locally, device verifies
# =============================================================

controller_test_message = (
    b"NARAD-QC trusted-controller signature verification"
)

controller_signature = (
    cp_test.sign_control_message(
        controller_test_message
    )
)

assert (
    cp_test_alice_runtime
    .verify_trusted_controller_signature(
        controller_id=(
            cp_test.controller_id
        ),
        message=(
            controller_test_message
        ),
        signature=(
            controller_signature
        )
    )
)

print(
    "[PASS] Device verifies controller signature using "
    "its pinned controller identity."
)


# =============================================================
# Test 7 — Tampered device message fails verification
# =============================================================

assert not cp_test.verify_registered_peer_signature(
    node_or_name=cp_test_alice_node,
    message=(
        peer_test_message
        + b"!"
    ),
    signature=alice_peer_signature
)

print(
    "[PASS] Controller rejects a signature after "
    "message tampering."
)


# =============================================================
# Test 8 — Provision another independent device
# =============================================================

cp_test_r1_runtime = (
    cp_test.provision_node(
        cp_test_r1_node
    )
)

assert cp_test.peer_count == 2

assert (
    cp_test_r1_runtime.public_identity.public_key
    !=
    cp_test_alice_runtime.public_identity.public_key
)

print(
    "[PASS] Additional physical devices receive "
    "independent HSM identities."
)


# =============================================================
# Test 9 — Alice signature does not authenticate as R1
# =============================================================

assert not cp_test.verify_registered_peer_signature(
    node_or_name=cp_test_r1_node,
    message=peer_test_message,
    signature=alice_peer_signature
)

print(
    "[PASS] Signature from one enrolled device cannot "
    "verify as another device."
)


# =============================================================
# Test 10 — Duplicate physical identity cannot be enrolled
# =============================================================

duplicate_enrollment_error = False

try:
    cp_test.provision_node(
        cp_test_alice_node
    )

except PeerAlreadyEnrolledError:
    duplicate_enrollment_error = True

assert duplicate_enrollment_error

print(
    "[PASS] Duplicate peer enrollment is rejected."
)


# =============================================================
# Test 11 — Unknown peers are not silently trusted
# =============================================================

unknown_peer_error = False

try:
    cp_test.get_peer_record(
        "Unknown-Device"
    )

except PeerNotEnrolledError:
    unknown_peer_error = True

assert unknown_peer_error

print(
    "[PASS] Unknown physical devices are not trusted "
    "implicitly."
)


# =============================================================
# Test 12 — Registry views are immutable records
# =============================================================

registry_snapshot = (
    cp_test.trusted_peer_records
)

assert isinstance(
    registry_snapshot,
    tuple
)

assert len(
    registry_snapshot
) == 2

assert all(
    isinstance(
        record,
        TrustedPeerRecord
    )
    for record in registry_snapshot
)

print(
    "[PASS] Trusted peer registry exposes immutable "
    "public records."
)


# =============================================================
# Test 13 — Node runtimes do not expose private-key APIs
# =============================================================

assert not hasattr(
    cp_test_alice_runtime,
    "private_key"
)

assert not hasattr(
    cp_test_alice_runtime,
    "secret_key"
)

assert not hasattr(
    cp_test_alice_runtime,
    "device_hsm"
)

print(
    "[PASS] Node runtime exposes no normal private-key "
    "or HSM export interface."
)


# =============================================================
# Test 14 — Control plane exposes no private-key API
# =============================================================

assert not hasattr(
    cp_test,
    "private_key"
)

assert not hasattr(
    cp_test,
    "secret_key"
)

assert not hasattr(
    cp_test,
    "controller_hsm"
)

print(
    "[PASS] Control plane exposes no normal controller "
    "private-key interface."
)


# =============================================================
# Test 15 — Enrollment has NOT authenticated anybody
# =============================================================

# There intentionally is no authentication-state field yet.

assert not hasattr(
    alice_record,
    "authenticated"
)

assert not hasattr(
    cp_test_alice_runtime,
    "authenticated"
)

print(
    "[PASS] Trusted enrollment is kept distinct from "
    "runtime authentication."
)


# =============================================================
# Inspection
# =============================================================

print("\nCONTROL PLANE")
print(
    cp_test
)


print("\nCONTROLLER PUBLIC IDENTITY")
print(
    cp_test.public_identity
)


print("\nALICE DEVICE RUNTIME")
print(
    cp_test_alice_runtime
)


print("\nR1 DEVICE RUNTIME")
print(
    cp_test_r1_runtime
)


print("\nTRUSTED PEER REGISTRY")

display(
    cp_test.trusted_peer_registry_dataframe()
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

In [ ]:
# @title Mutual Authentication Protocol Records

from dataclasses import dataclass, replace
from enum import Enum
from typing import Optional
import hashlib
import secrets
import struct


# =============================================================
# Authentication Errors
# =============================================================

class MutualAuthenticationError(Exception):
    """Base exception for NARAD-QC mutual authentication."""
    pass


class AuthenticationSessionNotFoundError(
    MutualAuthenticationError
):
    pass


class AuthenticationProtocolStateError(
    MutualAuthenticationError
):
    pass


class AuthenticationReplayError(
    MutualAuthenticationError
):
    pass


class AuthenticationIdentityError(
    MutualAuthenticationError
):
    pass


# =============================================================
# Authentication State
# =============================================================

class AuthenticationStatus(str, Enum):

    NOT_AUTHENTICATED = "NOT_AUTHENTICATED"

    AUTHENTICATED = "AUTHENTICATED"

    FAILED = "FAILED"

    INVALIDATED = "INVALIDATED"


class AuthenticationStage(str, Enum):

    CHALLENGE_ISSUED = "CHALLENGE_ISSUED"

    CONTROLLER_PROOF_ISSUED = "CONTROLLER_PROOF_ISSUED"

    COMPLETED = "COMPLETED"

    FAILED = "FAILED"


# =============================================================
# Protocol Constants
# =============================================================

NARAD_AUTH_PROTOCOL_VERSION = "NARAD-QC-AUTH-v1"

NODE_PROOF_DOMAIN = (
    b"NARAD-QC|AUTH|NODE-PROOF|v1"
)

CONTROLLER_PROOF_DOMAIN = (
    b"NARAD-QC|AUTH|CONTROLLER-PROOF|v1"
)

AUTH_CHALLENGE_BYTES = 32

AUTH_SESSION_ID_BYTES = 16


# =============================================================
# Safe Transcript Encoding
# =============================================================

def _auth_field_bytes(
    value
) -> bytes:
    """
    Convert one authentication transcript field into bytes.
    """

    if isinstance(value, str):
        return value.encode("utf-8")

    if isinstance(value, bytes):
        return value

    if isinstance(value, bytearray):
        return bytes(value)

    if isinstance(value, memoryview):
        return value.tobytes()

    raise TypeError(
        "Authentication transcript fields must be "
        "str or bytes-like."
    )


def encode_auth_transcript(
    *fields
) -> bytes:
    """
    Length-prefix every field before concatenation.

    This avoids ambiguity such as:

        ("AB", "C")  vs  ("A", "BC")
    """

    encoded = bytearray()

    for field in fields:

        field_bytes = _auth_field_bytes(
            field
        )

        encoded.extend(
            struct.pack(
                ">I",
                len(field_bytes)
            )
        )

        encoded.extend(
            field_bytes
        )

    return bytes(
        encoded
    )


# =============================================================
# Authentication Messages
# =============================================================

@dataclass(frozen=True)
class AuthenticationRequest:
    """
    Controller -> device authentication challenge.
    """

    session_id: str

    controller_id: str
    controller_key_id: str

    node_name: str

    controller_challenge: bytes


@dataclass(frozen=True)
class NodeAuthenticationResponse:
    """
    Device -> controller proof of private-key possession.

    node_challenge is generated by the physical device and will
    subsequently challenge the controller.
    """

    session_id: str

    controller_id: str
    node_name: str

    controller_key_id: str
    node_key_id: str

    controller_challenge: bytes
    node_challenge: bytes

    signature: bytes


@dataclass(frozen=True)
class ControllerAuthenticationProof:
    """
    Controller -> device proof of controller private-key
    possession.
    """

    session_id: str

    controller_id: str
    node_name: str

    controller_key_id: str
    node_key_id: str

    controller_challenge: bytes
    node_challenge: bytes

    node_signature_sha256: bytes

    signature: bytes


@dataclass(frozen=True)
class AuthenticationRecord:
    """
    Immutable audit record of one completed/failed mutual
    authentication attempt.

    No private keys and no HSM internals are stored.
    """

    attempt_index: int

    session_id: str

    node_name: str

    status: AuthenticationStatus

    controller_key_id: str
    node_key_id: Optional[str]

    node_proof_verified: bool

    controller_proof_verified: bool

    reason: str

In [ ]:
# @title Node-Side Mutual Authentication Client

@dataclass
class _PendingDeviceAuthentication:
    """
    Internal node-side state for one unfinished authentication
    session.
    """

    request: AuthenticationRequest

    node_challenge: bytes

    node_signature: bytes


class NodeAuthenticationClient:
    """
    Device-side participant in NARAD-QC mutual authentication.

    Responsibilities
    ----------------
    - accept controller authentication challenges;
    - generate fresh device-side challenges;
    - ask the local device HSM to sign node proofs;
    - verify controller proofs using the pinned controller key;
    - prevent authentication-message replay locally.

    The client never receives or exports private-key material.
    """

    def __init__(
        self,
        runtime: NodeDeviceRuntime
    ):

        if not isinstance(
            runtime,
            NodeDeviceRuntime
        ):
            raise TypeError(
                "runtime must be a NodeDeviceRuntime."
            )

        self.runtime = runtime

        self._pending_sessions: dict[
            str,
            _PendingDeviceAuthentication
        ] = {}

        self._used_session_ids: set[str] = set()

    # =========================================================
    # Node Proof Generation
    # =========================================================

    def respond_to_authentication_request(
        self,
        request: AuthenticationRequest
    ) -> NodeAuthenticationResponse:

        if not isinstance(
            request,
            AuthenticationRequest
        ):
            raise TypeError(
                "request must be an AuthenticationRequest."
            )

        # -----------------------------------------------------
        # Identity binding
        # -----------------------------------------------------

        if (
            request.node_name
            != self.runtime.node_name
        ):
            raise AuthenticationIdentityError(
                f"Authentication request targets "
                f"'{request.node_name}', not "
                f"'{self.runtime.node_name}'."
            )

        if not self.runtime.trusts_controller(
            request.controller_id
        ):
            raise AuthenticationIdentityError(
                f"Node '{self.runtime.node_name}' does not "
                f"trust controller '{request.controller_id}'."
            )

        trusted_controller = (
            self.runtime.trusted_controller_identity(
                request.controller_id
            )
        )

        # The request must identify exactly the pinned
        # controller key.
        if (
            request.controller_key_id
            != trusted_controller.key_id
        ):
            raise AuthenticationIdentityError(
                "Controller key ID does not match the "
                "device's pinned trust anchor."
            )

        # -----------------------------------------------------
        # Replay prevention
        # -----------------------------------------------------

        if (
            request.session_id
            in self._used_session_ids
            or request.session_id
            in self._pending_sessions
        ):
            raise AuthenticationReplayError(
                f"Authentication session "
                f"'{request.session_id}' has already "
                f"been observed by this device."
            )

        # -----------------------------------------------------
        # Device-generated fresh challenge
        # -----------------------------------------------------

        node_challenge = secure_random_bytes(
            AUTH_CHALLENGE_BYTES
        )

        node_key_id = (
            self.runtime.key_id
        )

        # -----------------------------------------------------
        # Node proof transcript
        # -----------------------------------------------------

        transcript = encode_auth_transcript(

            NODE_PROOF_DOMAIN,

            NARAD_AUTH_PROTOCOL_VERSION,

            request.session_id,

            request.controller_id,

            request.node_name,

            request.controller_key_id,

            node_key_id,

            request.controller_challenge,

            node_challenge
        )

        signature = self.runtime.sign(
            transcript
        )

        # -----------------------------------------------------
        # Retain only session protocol state.
        #
        # No secret keys are present here.
        # -----------------------------------------------------

        self._pending_sessions[
            request.session_id
        ] = _PendingDeviceAuthentication(
            request=request,
            node_challenge=node_challenge,
            node_signature=signature
        )

        return NodeAuthenticationResponse(

            session_id=(
                request.session_id
            ),

            controller_id=(
                request.controller_id
            ),

            node_name=(
                request.node_name
            ),

            controller_key_id=(
                request.controller_key_id
            ),

            node_key_id=(
                node_key_id
            ),

            controller_challenge=(
                request.controller_challenge
            ),

            node_challenge=(
                node_challenge
            ),

            signature=(
                signature
            )
        )

    # =========================================================
    # Controller Proof Verification
    # =========================================================

    def verify_controller_proof(
        self,
        proof: ControllerAuthenticationProof
    ) -> bool:

        if not isinstance(
            proof,
            ControllerAuthenticationProof
        ):
            raise TypeError(
                "proof must be a ControllerAuthenticationProof."
            )

        session_id = proof.session_id

        # Already completed / consumed.
        if session_id in self._used_session_ids:
            raise AuthenticationReplayError(
                f"Controller proof for session "
                f"'{session_id}' has already been consumed."
            )

        if session_id not in self._pending_sessions:
            raise AuthenticationSessionNotFoundError(
                f"No pending node-side authentication "
                f"session '{session_id}'."
            )

        context = self._pending_sessions[
            session_id
        ]

        request = context.request

        # The proof is one-use regardless of whether verification
        # succeeds.
        del self._pending_sessions[
            session_id
        ]

        self._used_session_ids.add(
            session_id
        )

        # -----------------------------------------------------
        # Structural/context verification
        # -----------------------------------------------------

        if (
            proof.controller_id
            != request.controller_id
            or proof.node_name
            != request.node_name
            or proof.controller_key_id
            != request.controller_key_id
            or proof.node_key_id
            != self.runtime.key_id
        ):
            return False

        if not secrets.compare_digest(
            proof.controller_challenge,
            request.controller_challenge
        ):
            return False

        if not secrets.compare_digest(
            proof.node_challenge,
            context.node_challenge
        ):
            return False

        expected_node_signature_hash = (
            hashlib.sha256(
                context.node_signature
            ).digest()
        )

        if not secrets.compare_digest(
            proof.node_signature_sha256,
            expected_node_signature_hash
        ):
            return False

        # -----------------------------------------------------
        # Controller proof transcript
        # -----------------------------------------------------

        transcript = encode_auth_transcript(

            CONTROLLER_PROOF_DOMAIN,

            NARAD_AUTH_PROTOCOL_VERSION,

            proof.session_id,

            proof.controller_id,

            proof.node_name,

            proof.controller_key_id,

            proof.node_key_id,

            proof.controller_challenge,

            proof.node_challenge,

            proof.node_signature_sha256
        )

        # Verification uses only the controller public identity
        # pinned during trusted provisioning.
        return (
            self.runtime
            .verify_trusted_controller_signature(
                controller_id=(
                    proof.controller_id
                ),
                message=transcript,
                signature=proof.signature
            )
        )

    # =========================================================
    # Failed/Aborted Session Cleanup
    # =========================================================

    def abort_authentication_session(
        self,
        session_id: str
    ) -> None:
        """
        Consume a pending authentication context after the
        controller rejects the node proof or the protocol aborts.
        """

        if session_id in self._pending_sessions:

            del self._pending_sessions[
                session_id
            ]

        self._used_session_ids.add(
            session_id
        )

    # =========================================================
    # Inspection
    # =========================================================

    @property
    def pending_session_count(
        self
    ) -> int:

        return len(
            self._pending_sessions
        )

    def __repr__(
        self
    ) -> str:

        return (
            "NodeAuthenticationClient("
            f"node='{self.runtime.node_name}', "
            f"pending_sessions="
            f"{self.pending_session_count}"
            ")"
        )

In [ ]:
# @title Control-Plane Mutual Authentication Service


@dataclass
class _ControllerAuthenticationSession:
    """
    Internal controller-side authentication-session state.
    """

    request: AuthenticationRequest

    stage: AuthenticationStage

    node_key_id: Optional[str] = None

    node_challenge: Optional[bytes] = None

    node_signature_sha256: Optional[bytes] = None


class MutualAuthenticationService:
    """
    Runtime mutual-authentication service for the NARAD-QC
    ClassicalControlPlane.

    The service uses:
    - trusted peer public identities owned by ClassicalControlPlane;
    - controller HSM signing through ClassicalControlPlane;
    - node-side HSM signing through NodeDeviceRuntime.

    A device becomes quantum-operation eligible only after BOTH:

        controller verifies node
        AND
        node verifies controller

    have succeeded for the same fresh authentication session.
    """

    def __init__(
        self,
        control_plane: ClassicalControlPlane
    ):

        if not isinstance(
            control_plane,
            ClassicalControlPlane
        ):
            raise TypeError(
                "control_plane must be a ClassicalControlPlane."
            )

        self.control_plane = (
            control_plane
        )

        self._sessions: dict[
            str,
            _ControllerAuthenticationSession
        ] = {}

        self._authentication_history: list[
            AuthenticationRecord
        ] = []

        self._latest_record_by_node: dict[
            str,
            AuthenticationRecord
        ] = {}

    # =========================================================
    # Begin Authentication
    # =========================================================

    def begin_authentication(
        self,
        node_or_name
    ) -> AuthenticationRequest:
        """
        Begin a fresh authentication exchange.

        Enrollment is required, but enrollment alone does not
        grant authentication.
        """

        peer_record = (
            self.control_plane.get_peer_record(
                node_or_name
            )
        )

        node_name = (
            peer_record.node_name
        )

        # Fresh cryptographic session identifier.
        while True:

            session_id = secure_random_bytes(
                AUTH_SESSION_ID_BYTES
            ).hex()

            if session_id not in self._sessions:
                break

        controller_challenge = (
            secure_random_bytes(
                AUTH_CHALLENGE_BYTES
            )
        )

        request = AuthenticationRequest(

            session_id=session_id,

            controller_id=(
                self.control_plane.controller_id
            ),

            controller_key_id=(
                self.control_plane.key_id
            ),

            node_name=node_name,

            controller_challenge=(
                controller_challenge
            )
        )

        self._sessions[
            session_id
        ] = _ControllerAuthenticationSession(

            request=request,

            stage=(
                AuthenticationStage.CHALLENGE_ISSUED
            )
        )

        return request

    # =========================================================
    # Verify Device / Respond as Controller
    # =========================================================

    def verify_node_response_and_issue_proof(
        self,
        response: NodeAuthenticationResponse
    ) -> Optional[ControllerAuthenticationProof]:
        """
        Verify the device proof.

        On success:
            return a controller-signed mutual-authentication proof.

        On failure:
            record FAILED and return None.

        A node response is consumable exactly once.
        """

        if not isinstance(
            response,
            NodeAuthenticationResponse
        ):
            raise TypeError(
                "response must be a NodeAuthenticationResponse."
            )

        session_id = response.session_id

        if session_id not in self._sessions:
            raise AuthenticationSessionNotFoundError(
                f"Unknown authentication session "
                f"'{session_id}'."
            )

        session = self._sessions[
            session_id
        ]

        if (
            session.stage
            != AuthenticationStage.CHALLENGE_ISSUED
        ):
            raise AuthenticationReplayError(
                f"Node proof for authentication session "
                f"'{session_id}' has already been consumed."
            )

        request = session.request

        # -----------------------------------------------------
        # Structural binding
        # -----------------------------------------------------

        structure_valid = (
            response.controller_id
            == request.controller_id

            and response.node_name
            == request.node_name

            and response.controller_key_id
            == request.controller_key_id

            and secrets.compare_digest(
                response.controller_challenge,
                request.controller_challenge
            )
        )

        peer_record = (
            self.control_plane.get_peer_record(
                request.node_name
            )
        )

        # The responding device must claim the same enrolled key.
        key_identity_valid = (
            response.node_key_id
            == peer_record.key_id
        )

        if not structure_valid or not key_identity_valid:

            session.stage = (
                AuthenticationStage.FAILED
            )

            self._record_result(
                session_id=session_id,
                node_name=request.node_name,
                status=AuthenticationStatus.FAILED,
                node_key_id=response.node_key_id,
                node_proof_verified=False,
                controller_proof_verified=False,
                reason=(
                    "authentication_context_or_identity_mismatch"
                )
            )

            return None

        # -----------------------------------------------------
        # Verify node proof
        # -----------------------------------------------------

        node_transcript = encode_auth_transcript(

            NODE_PROOF_DOMAIN,

            NARAD_AUTH_PROTOCOL_VERSION,

            response.session_id,

            response.controller_id,

            response.node_name,

            response.controller_key_id,

            response.node_key_id,

            response.controller_challenge,

            response.node_challenge
        )

        node_verified = (
            self.control_plane
            .verify_registered_peer_signature(
                node_or_name=(
                    response.node_name
                ),
                message=node_transcript,
                signature=response.signature
            )
        )

        if not node_verified:

            session.stage = (
                AuthenticationStage.FAILED
            )

            self._record_result(
                session_id=session_id,
                node_name=request.node_name,
                status=AuthenticationStatus.FAILED,
                node_key_id=response.node_key_id,
                node_proof_verified=False,
                controller_proof_verified=False,
                reason="node_signature_verification_failed"
            )

            return None

        # -----------------------------------------------------
        # Node successfully authenticated TO controller.
        #
        # Do NOT mark mutually authenticated yet.
        # -----------------------------------------------------

        node_signature_hash = (
            hashlib.sha256(
                response.signature
            ).digest()
        )

        session.node_key_id = (
            response.node_key_id
        )

        session.node_challenge = (
            response.node_challenge
        )

        session.node_signature_sha256 = (
            node_signature_hash
        )

        # -----------------------------------------------------
        # Controller proof
        # -----------------------------------------------------

        controller_transcript = encode_auth_transcript(

            CONTROLLER_PROOF_DOMAIN,

            NARAD_AUTH_PROTOCOL_VERSION,

            response.session_id,

            response.controller_id,

            response.node_name,

            response.controller_key_id,

            response.node_key_id,

            response.controller_challenge,

            response.node_challenge,

            node_signature_hash
        )

        controller_signature = (
            self.control_plane.sign_control_message(
                controller_transcript
            )
        )

        proof = ControllerAuthenticationProof(

            session_id=(
                response.session_id
            ),

            controller_id=(
                response.controller_id
            ),

            node_name=(
                response.node_name
            ),

            controller_key_id=(
                response.controller_key_id
            ),

            node_key_id=(
                response.node_key_id
            ),

            controller_challenge=(
                response.controller_challenge
            ),

            node_challenge=(
                response.node_challenge
            ),

            node_signature_sha256=(
                node_signature_hash
            ),

            signature=(
                controller_signature
            )
        )

        session.stage = (
            AuthenticationStage.CONTROLLER_PROOF_ISSUED
        )

        return proof

    # =========================================================
    # Finalize Mutual Authentication
    # =========================================================

    def finalize_authentication(
        self,
        session_id: str,
        controller_verified_by_node: bool
    ) -> AuthenticationRecord:
        """
        Complete the authentication transaction after the node
        reports whether it successfully verified the controller.
        """

        if (
            not isinstance(session_id, str)
            or not session_id.strip()
        ):
            raise ValueError(
                "session_id must be a non-empty string."
            )

        if not isinstance(
            controller_verified_by_node,
            bool
        ):
            raise TypeError(
                "controller_verified_by_node must be bool."
            )

        if session_id not in self._sessions:
            raise AuthenticationSessionNotFoundError(
                f"Unknown authentication session "
                f"'{session_id}'."
            )

        session = self._sessions[
            session_id
        ]

        if (
            session.stage
            != AuthenticationStage.CONTROLLER_PROOF_ISSUED
        ):
            raise AuthenticationProtocolStateError(
                f"Authentication session '{session_id}' "
                f"cannot be finalized from stage "
                f"'{session.stage.value}'."
            )

        if controller_verified_by_node:

            session.stage = (
                AuthenticationStage.COMPLETED
            )

            return self._record_result(
                session_id=session_id,
                node_name=(
                    session.request.node_name
                ),
                status=(
                    AuthenticationStatus.AUTHENTICATED
                ),
                node_key_id=(
                    session.node_key_id
                ),
                node_proof_verified=True,
                controller_proof_verified=True,
                reason="mutual_authentication_success"
            )

        session.stage = (
            AuthenticationStage.FAILED
        )

        return self._record_result(
            session_id=session_id,
            node_name=(
                session.request.node_name
            ),
            status=AuthenticationStatus.FAILED,
            node_key_id=session.node_key_id,
            node_proof_verified=True,
            controller_proof_verified=False,
            reason="controller_signature_verification_failed"
        )

    # =========================================================
    # High-Level Complete Handshake
    # =========================================================

    def mutually_authenticate(
        self,
        client: NodeAuthenticationClient
    ) -> AuthenticationRecord:
        """
        Execute one complete fresh mutual-authentication exchange.

        This is the normal high-level API used by future NARAD-QC
        service establishment.
        """

        if not isinstance(
            client,
            NodeAuthenticationClient
        ):
            raise TypeError(
                "client must be a NodeAuthenticationClient."
            )

        node_name = (
            client.runtime.node_name
        )

        request = self.begin_authentication(
            node_name
        )

        try:

            response = (
                client.respond_to_authentication_request(
                    request
                )
            )

            controller_proof = (
                self.verify_node_response_and_issue_proof(
                    response
                )
            )

            if controller_proof is None:

                client.abort_authentication_session(
                    request.session_id
                )

                return (
                    self.latest_record(
                        node_name
                    )
                )

            controller_verified = (
                client.verify_controller_proof(
                    controller_proof
                )
            )

            return self.finalize_authentication(
                session_id=request.session_id,
                controller_verified_by_node=(
                    controller_verified
                )
            )

        except Exception:

            client.abort_authentication_session(
                request.session_id
            )

            # Do not overwrite a failure already recorded by
            # controller-side signature verification.
            session = self._sessions[
                request.session_id
            ]

            if (
                session.stage
                not in (
                    AuthenticationStage.FAILED,
                    AuthenticationStage.COMPLETED
                )
            ):

                session.stage = (
                    AuthenticationStage.FAILED
                )

                self._record_result(
                    session_id=request.session_id,
                    node_name=node_name,
                    status=AuthenticationStatus.FAILED,
                    node_key_id=None,
                    node_proof_verified=False,
                    controller_proof_verified=False,
                    reason="authentication_protocol_exception"
                )

            raise

    # =========================================================
    # Authentication / Authorization State
    # =========================================================

    def latest_record(
        self,
        node_or_name
    ) -> AuthenticationRecord:

        node_name = (
            self._resolve_node_name(
                node_or_name
            )
        )

        if node_name not in self._latest_record_by_node:
            raise AuthenticationSessionNotFoundError(
                f"Node '{node_name}' has no completed "
                f"authentication attempt."
            )

        return self._latest_record_by_node[
            node_name
        ]

    def authentication_status(
        self,
        node_or_name
    ) -> AuthenticationStatus:

        node_name = (
            self._resolve_node_name(
                node_or_name
            )
        )

        if node_name not in self._latest_record_by_node:
            return (
                AuthenticationStatus.NOT_AUTHENTICATED
            )

        return (
            self._latest_record_by_node[
                node_name
            ].status
        )

    def is_authenticated(
        self,
        node_or_name,
        session_id: Optional[str] = None
    ) -> bool:
        """
        True only for a successfully MUTUALLY authenticated node.

        Optional session_id allows future communication sessions
        to bind themselves to a specific authentication exchange.
        """

        node_name = (
            self._resolve_node_name(
                node_or_name
            )
        )

        if node_name not in self._latest_record_by_node:
            return False

        record = (
            self._latest_record_by_node[
                node_name
            ]
        )

        if (
            record.status
            != AuthenticationStatus.AUTHENTICATED
        ):
            return False

        if session_id is not None:

            return (
                record.session_id
                == session_id
            )

        return True

    def is_quantum_eligible(
        self,
        node_or_name
    ) -> bool:
        """
        Phase-3 authorization rule:

            mutually authenticated
                =>
            eligible for authorized quantum operations

        Later phases may add additional authorization conditions.
        """

        return self.is_authenticated(
            node_or_name
        )

    def invalidate_authentication(
        self,
        node_or_name,
        reason: str = "authentication_invalidated"
    ) -> AuthenticationRecord:
        """
        Remove current quantum-operation eligibility without
        removing the node's trusted enrollment.
        """

        node_name = (
            self._resolve_node_name(
                node_or_name
            )
        )

        # Enrollment remains mandatory.
        self.control_plane.get_peer_record(
            node_name
        )

        latest = (
            self._latest_record_by_node.get(
                node_name
            )
        )

        session_id = (
            latest.session_id
            if latest is not None
            else "NO-AUTH-SESSION"
        )

        node_key_id = (
            latest.node_key_id
            if latest is not None
            else (
                self.control_plane
                .get_peer_record(
                    node_name
                )
                .key_id
            )
        )

        return self._record_result(
            session_id=session_id,
            node_name=node_name,
            status=(
                AuthenticationStatus.INVALIDATED
            ),
            node_key_id=node_key_id,
            node_proof_verified=False,
            controller_proof_verified=False,
            reason=str(reason)
        )

    # =========================================================
    # Record Keeping
    # =========================================================

    def _record_result(
        self,
        session_id: str,
        node_name: str,
        status: AuthenticationStatus,
        node_key_id: Optional[str],
        node_proof_verified: bool,
        controller_proof_verified: bool,
        reason: str
    ) -> AuthenticationRecord:

        record = AuthenticationRecord(

            attempt_index=(
                len(
                    self._authentication_history
                )
            ),

            session_id=session_id,

            node_name=node_name,

            status=status,

            controller_key_id=(
                self.control_plane.key_id
            ),

            node_key_id=node_key_id,

            node_proof_verified=(
                node_proof_verified
            ),

            controller_proof_verified=(
                controller_proof_verified
            ),

            reason=str(
                reason
            )
        )

        self._authentication_history.append(
            record
        )

        self._latest_record_by_node[
            node_name
        ] = record

        return record

    @property
    def authentication_history(
        self
    ) -> tuple[AuthenticationRecord, ...]:

        return tuple(
            self._authentication_history
        )

    def authentication_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for record in (
            self._authentication_history
        ):

            rows.append({

                "Attempt":
                    record.attempt_index,

                "Session ID":
                    record.session_id,

                "Node":
                    record.node_name,

                "Status":
                    record.status.value,

                "Node Proof Verified":
                    record.node_proof_verified,

                "Controller Proof Verified":
                    record.controller_proof_verified,

                "Node Key ID":
                    record.node_key_id,

                "Controller Key ID":
                    record.controller_key_id,

                "Reason":
                    record.reason
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Internal Helpers
    # =========================================================

    @staticmethod
    def _resolve_node_name(
        node_or_name
    ) -> str:

        if isinstance(
            node_or_name,
            NetworkNode
        ):
            return node_or_name.name

        if isinstance(
            node_or_name,
            NodeDeviceRuntime
        ):
            return node_or_name.node_name

        if isinstance(
            node_or_name,
            NodeAuthenticationClient
        ):
            return (
                node_or_name.runtime.node_name
            )

        if (
            isinstance(node_or_name, str)
            and node_or_name.strip()
        ):
            return node_or_name.strip()

        raise TypeError(
            "Expected NetworkNode, NodeDeviceRuntime, "
            "NodeAuthenticationClient, or node-name string."
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        authenticated_count = sum(
            self.is_authenticated(
                record.node_name
            )
            for record
            in self.control_plane.trusted_peer_records
        )

        return (
            "MutualAuthenticationService("
            f"controller="
            f"'{self.control_plane.controller_id}', "
            f"authenticated_peers="
            f"{authenticated_count}/"
            f"{self.control_plane.peer_count}"
            ")"
        )

In [ ]:
# @title Mutual Authentication Verification


print("=" * 76)
print("NARAD-QC Phase 3A — Mutual Authentication Verification")
print("=" * 76)


# =============================================================
# Independent control-plane environment
# =============================================================

auth_test_cp = ClassicalControlPlane(
    controller_id="AUTH-TEST-CONTROLLER"
)

auth_service = MutualAuthenticationService(
    auth_test_cp
)


# =============================================================
# Physical devices
# =============================================================

auth_alice_node = NetworkNode(
    name="Auth-Alice",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)

auth_r1_node = NetworkNode(
    name="Auth-R1",
    x_km=25,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


# =============================================================
# Trusted provisioning
# =============================================================

auth_alice_runtime = (
    auth_test_cp.provision_node(
        auth_alice_node
    )
)

auth_r1_runtime = (
    auth_test_cp.provision_node(
        auth_r1_node
    )
)

alice_auth_client = (
    NodeAuthenticationClient(
        auth_alice_runtime
    )
)

r1_auth_client = (
    NodeAuthenticationClient(
        auth_r1_runtime
    )
)

print(
    "[PASS] Test devices are trusted-enrolled but "
    "not yet authenticated."
)


# =============================================================
# Test 1 — Enrollment is not authentication
# =============================================================

assert auth_test_cp.is_enrolled(
    auth_alice_node
)

assert not auth_service.is_authenticated(
    auth_alice_node
)

assert not auth_service.is_quantum_eligible(
    auth_alice_node
)

print(
    "[PASS] Trusted enrollment alone grants no "
    "quantum-operation eligibility."
)


# =============================================================
# Test 2 — Complete successful mutual authentication
# =============================================================

alice_auth_record = (
    auth_service.mutually_authenticate(
        alice_auth_client
    )
)

assert (
    alice_auth_record.status
    == AuthenticationStatus.AUTHENTICATED
)

assert alice_auth_record.node_proof_verified
assert alice_auth_record.controller_proof_verified

assert auth_service.is_authenticated(
    auth_alice_node
)

assert auth_service.is_quantum_eligible(
    auth_alice_node
)

print(
    "[PASS] Alice completes mutual ML-DSA authentication."
)


# =============================================================
# Test 3 — Authentication is bound to its session ID
# =============================================================

assert auth_service.is_authenticated(
    auth_alice_node,
    session_id=(
        alice_auth_record.session_id
    )
)

assert not auth_service.is_authenticated(
    auth_alice_node,
    session_id="wrong-session-id"
)

print(
    "[PASS] Authentication can be bound to its exact "
    "authentication session."
)


# =============================================================
# Test 4 — R1 can independently authenticate
# =============================================================

r1_auth_record = (
    auth_service.mutually_authenticate(
        r1_auth_client
    )
)

assert (
    r1_auth_record.status
    == AuthenticationStatus.AUTHENTICATED
)

assert auth_service.is_quantum_eligible(
    auth_r1_node
)

print(
    "[PASS] Repeater R1 independently completes "
    "mutual authentication."
)


# =============================================================
# Test 5 — Node-proof replay is rejected
# =============================================================

replay_request = (
    auth_service.begin_authentication(
        auth_alice_node
    )
)

replay_response = (
    alice_auth_client
    .respond_to_authentication_request(
        replay_request
    )
)

replay_controller_proof = (
    auth_service
    .verify_node_response_and_issue_proof(
        replay_response
    )
)

assert replay_controller_proof is not None


controller_replay_detected = False

try:
    auth_service.verify_node_response_and_issue_proof(
        replay_response
    )

except AuthenticationReplayError:
    controller_replay_detected = True

assert controller_replay_detected

print(
    "[PASS] Controller rejects replay of an already "
    "consumed node proof."
)


# =============================================================
# Test 6 — Controller-proof replay is rejected by device
# =============================================================

controller_verified = (
    alice_auth_client
    .verify_controller_proof(
        replay_controller_proof
    )
)

assert controller_verified


device_replay_detected = False

try:
    alice_auth_client.verify_controller_proof(
        replay_controller_proof
    )

except AuthenticationReplayError:
    device_replay_detected = True

assert device_replay_detected


replay_final_record = (
    auth_service.finalize_authentication(
        session_id=(
            replay_request.session_id
        ),
        controller_verified_by_node=True
    )
)

assert (
    replay_final_record.status
    == AuthenticationStatus.AUTHENTICATED
)

print(
    "[PASS] Device rejects replay of an already "
    "consumed controller proof."
)


# =============================================================
# Test 7 — Tampered node signature fails authentication
# =============================================================

tamper_request = (
    auth_service.begin_authentication(
        auth_r1_node
    )
)

tamper_response = (
    r1_auth_client
    .respond_to_authentication_request(
        tamper_request
    )
)

tampered_signature = bytearray(
    tamper_response.signature
)

tampered_signature[0] ^= 0x01

tampered_response = replace(
    tamper_response,
    signature=bytes(
        tampered_signature
    )
)

tamper_result = (
    auth_service
    .verify_node_response_and_issue_proof(
        tampered_response
    )
)

assert tamper_result is None

r1_auth_client.abort_authentication_session(
    tamper_request.session_id
)

assert (
    auth_service.authentication_status(
        auth_r1_node
    )
    == AuthenticationStatus.FAILED
)

assert not auth_service.is_quantum_eligible(
    auth_r1_node
)

print(
    "[PASS] Tampered node proof causes authentication "
    "failure and removes quantum eligibility."
)


# =============================================================
# Test 8 — Fresh authentication restores eligibility
# =============================================================

r1_reauth_record = (
    auth_service.mutually_authenticate(
        r1_auth_client
    )
)

assert (
    r1_reauth_record.status
    == AuthenticationStatus.AUTHENTICATED
)

assert auth_service.is_quantum_eligible(
    auth_r1_node
)

print(
    "[PASS] A later fresh successful authentication "
    "restores quantum eligibility."
)


# =============================================================
# Test 9 — Impostor with same physical node name but wrong
#          HSM key cannot authenticate as Alice
# =============================================================

rogue_alice_runtime = (
    NodeDeviceRuntime(
        physical_node=auth_alice_node,
        trusted_controller_identity=(
            auth_test_cp.public_identity
        )
    )
)

rogue_alice_client = (
    NodeAuthenticationClient(
        rogue_alice_runtime
    )
)

assert (
    rogue_alice_runtime.key_id
    != auth_alice_runtime.key_id
)


rogue_request = (
    auth_service.begin_authentication(
        auth_alice_node
    )
)

rogue_response = (
    rogue_alice_client
    .respond_to_authentication_request(
        rogue_request
    )
)

rogue_result = (
    auth_service
    .verify_node_response_and_issue_proof(
        rogue_response
    )
)

assert rogue_result is None

rogue_alice_client.abort_authentication_session(
    rogue_request.session_id
)

assert (
    auth_service.authentication_status(
        auth_alice_node
    )
    == AuthenticationStatus.FAILED
)

assert not auth_service.is_quantum_eligible(
    auth_alice_node
)

print(
    "[PASS] Device possessing the wrong HSM key cannot "
    "impersonate an enrolled physical node."
)


# =============================================================
# Test 10 — Genuine Alice can authenticate again
# =============================================================

alice_reauth_record = (
    auth_service.mutually_authenticate(
        alice_auth_client
    )
)

assert (
    alice_reauth_record.status
    == AuthenticationStatus.AUTHENTICATED
)

assert auth_service.is_quantum_eligible(
    auth_alice_node
)

print(
    "[PASS] Genuine enrolled Alice re-authenticates "
    "successfully after impersonation attempt."
)


# =============================================================
# Test 11 — Explicit invalidation removes eligibility
# =============================================================

invalidated_record = (
    auth_service.invalidate_authentication(
        auth_alice_node,
        reason="verification_invalidation"
    )
)

assert (
    invalidated_record.status
    == AuthenticationStatus.INVALIDATED
)

assert not auth_service.is_authenticated(
    auth_alice_node
)

assert not auth_service.is_quantum_eligible(
    auth_alice_node
)

# Enrollment remains intact.
assert auth_test_cp.is_enrolled(
    auth_alice_node
)

print(
    "[PASS] Authentication can be invalidated without "
    "removing trusted device enrollment."
)


# =============================================================
# Test 12 — Fresh mutual authentication after invalidation
# =============================================================

alice_final_auth = (
    auth_service.mutually_authenticate(
        alice_auth_client
    )
)

assert (
    alice_final_auth.status
    == AuthenticationStatus.AUTHENTICATED
)

assert auth_service.is_quantum_eligible(
    auth_alice_node
)

print(
    "[PASS] Invalidated device can regain eligibility "
    "only through a fresh mutual authentication."
)


# =============================================================
# Final inspection
# =============================================================

print("\nAUTHENTICATION SERVICE")
print(
    auth_service
)


print("\nCURRENT DEVICE STATES")

current_auth_state = pd.DataFrame([
    {
        "Node":
            auth_alice_node.name,

        "Enrolled":
            auth_test_cp.is_enrolled(
                auth_alice_node
            ),

        "Authentication Status":
            auth_service
            .authentication_status(
                auth_alice_node
            ).value,

        "Quantum Eligible":
            auth_service
            .is_quantum_eligible(
                auth_alice_node
            )
    },

    {
        "Node":
            auth_r1_node.name,

        "Enrolled":
            auth_test_cp.is_enrolled(
                auth_r1_node
            ),

        "Authentication Status":
            auth_service
            .authentication_status(
                auth_r1_node
            ).value,

        "Quantum Eligible":
            auth_service
            .is_quantum_eligible(
                auth_r1_node
            )
    }
])

display(
    current_auth_state
)


print("\nAUTHENTICATION AUDIT HISTORY")

display(
    auth_service.authentication_dataframe()
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

In [ ]:
# @title Quantum Network Simulator

from types import MappingProxyType

# =============================================================
# Simulator Errors
# =============================================================

class QuantumNetworkSimulatorError(Exception):
    """
    Base exception for reconstructed NARAD-QC simulation assembly.
    """
    pass


class QuantumNetworkRuntimeConsistencyError(
    QuantumNetworkSimulatorError
):
    """
    Raised when static topology and assembled runtime state
    no longer agree.
    """
    pass


class QuantumOperationAuthorizationError(
    QuantumNetworkSimulatorError
):
    """
    Raised when an unauthenticated device is requested to
    participate in an authorized quantum operation.
    """
    pass


# =============================================================
# Quantum Network Simulator
# =============================================================

class QuantumNetworkSimulator:
    """
    Composition/runtime owner for one reconstructed NARAD-QC
    simulation.

    Responsibilities
    ----------------
    - own one fixed physical NetworkTopology;
    - create the network controller before any node devices;
    - create one SimulationKernel;
    - trusted-provision every physical node;
    - create one NodeDeviceRuntime per physical node;
    - create one NodeAuthenticationClient per physical node;
    - create one QuantumMemory per physical node;
    - create and configure one EntanglementManager;
    - own one MutualAuthenticationService;
    - provide the current quantum-operation authorization gate;
    - validate cross-component runtime consistency.

    The simulator deliberately does NOT itself:
    - implement ML-DSA;
    - perform authentication protocol steps internally;
    - model fibre loss;
    - generate entanglement;
    - execute swapping;
    - choose routes;
    - perform QKD.

    Those responsibilities remain delegated to their respective
    components.
    """

    def __init__(
        self,
        topology: NetworkTopology,
        simulation_seed: Optional[int] = None,
        controller_id: str = (
            ClassicalControlPlane.DEFAULT_CONTROLLER_ID
        ),
        parameter_set: MLDSAParameterSet = (
            PHASE3_DEFAULT_MLDSA_PARAMETER_SET
        ),
        max_storage_slots: Optional[int] = None
    ):

        # =====================================================
        # Static Physical Topology
        # =====================================================

        if not isinstance(
            topology,
            NetworkTopology
        ):
            raise TypeError(
                "topology must be a NetworkTopology."
            )

        if topology.node_count < 2:
            raise ValueError(
                "A quantum-network simulation requires "
                "at least two physical nodes."
            )

        topology.validate_internal_consistency()

        self.topology = topology

        # =====================================================
        # STEP 1 — Create network trust authority FIRST
        #
        # Controller receives the first cryptographic identity.
        # =====================================================

        self.control_plane = (
            ClassicalControlPlane(
                controller_id=controller_id,
                parameter_set=parameter_set
            )
        )

        # =====================================================
        # STEP 2 — Create physical simulation context
        # =====================================================

        self.kernel = SimulationKernel(
            seed=simulation_seed
        )

        # =====================================================
        # STEP 3 — Authentication service
        # =====================================================

        self.authentication_service = (
            MutualAuthenticationService(
                self.control_plane
            )
        )

        # =====================================================
        # STEP 4 — Global entanglement authority
        # =====================================================

        self.entanglement_manager = (
            EntanglementManager(
                kernel=self.kernel
            )
        )

        # =====================================================
        # STEP 5 — Runtime Registries
        # =====================================================

        self._device_runtimes: dict[
            str,
            NodeDeviceRuntime
        ] = {}

        self._authentication_clients: dict[
            str,
            NodeAuthenticationClient
        ] = {}

        self._quantum_memories: dict[
            str,
            QuantumMemory
        ] = {}

        # =====================================================
        # STEP 6 — Provision every physical device
        # =====================================================

        for node in self.topology.nodes:

            # -------------------------------------------------
            # Cryptographic device runtime
            # -------------------------------------------------

            runtime = (
                self.control_plane.provision_node(
                    node=node,
                    parameter_set=parameter_set
                )
            )

            self._device_runtimes[
                node.name
            ] = runtime

            # -------------------------------------------------
            # Device authentication agent
            # -------------------------------------------------

            auth_client = (
                NodeAuthenticationClient(
                    runtime
                )
            )

            self._authentication_clients[
                node.name
            ] = auth_client

            # -------------------------------------------------
            # Runtime quantum memory
            # -------------------------------------------------

            memory = QuantumMemory(
                owner=node,
                kernel=self.kernel,
                max_storage_slots=(
                    max_storage_slots
                )
            )

            self._quantum_memories[
                node.name
            ] = memory

            self.entanglement_manager.register_memory(
                memory
            )

        # =====================================================
        # STEP 7 — Capture physical assembly snapshot
        #
        # Phase 3A does not yet support live topology mutation.
        # =====================================================

        self._assembled_node_objects = {
            node.name: node
            for node in self.topology.nodes
        }

        self._assembled_link_objects = {
            link.link_id: link
            for link in self.topology.links
        }

        # =====================================================
        # STEP 8 — Validate complete world
        # =====================================================

        self.validate_runtime_consistency()

        self.kernel.emit(
            event_type="QUANTUM_NETWORK_SIMULATOR_ASSEMBLED",
            component="QuantumNetworkSimulator",
            node_names=tuple(
                node.name
                for node in self.topology.nodes
            ),
            link_ids=tuple(
                link.link_id
                for link in self.topology.links
            ),
            message=(
                f"QuantumNetworkSimulator assembled for "
                f"topology '{self.topology.name}'."
            ),
            metadata={
                "node_count":
                    self.topology.node_count,

                "link_count":
                    self.topology.link_count,

                "controller_id":
                    self.control_plane.controller_id,

                "ml_dsa_parameter_set":
                    self.control_plane.parameter_set.value,

                "simulation_seed":
                    self.kernel.seed,

                "max_storage_slots":
                    max_storage_slots
            }
        )

    # =========================================================
    # Name Resolution
    # =========================================================

    @staticmethod
    def _resolve_node_name(
        node_or_name
    ) -> str:

        if isinstance(
            node_or_name,
            NetworkNode
        ):
            return node_or_name.name

        if isinstance(
            node_or_name,
            NodeDeviceRuntime
        ):
            return node_or_name.node_name

        if isinstance(
            node_or_name,
            NodeAuthenticationClient
        ):
            return (
                node_or_name.runtime.node_name
            )

        if (
            isinstance(node_or_name, str)
            and node_or_name.strip()
        ):
            return node_or_name.strip()

        raise TypeError(
            "Expected NetworkNode, NodeDeviceRuntime, "
            "NodeAuthenticationClient, or node-name string."
        )

    # =========================================================
    # Runtime Component Access
    # =========================================================

    def device_runtime(
        self,
        node_or_name
    ) -> NodeDeviceRuntime:

        node_name = self._resolve_node_name(
            node_or_name
        )

        if node_name not in self._device_runtimes:
            raise TopologyNodeNotFoundError(
                f"No runtime exists for node '{node_name}'."
            )

        return self._device_runtimes[
            node_name
        ]

    def authentication_client(
        self,
        node_or_name
    ) -> NodeAuthenticationClient:

        node_name = self._resolve_node_name(
            node_or_name
        )

        if node_name not in self._authentication_clients:
            raise TopologyNodeNotFoundError(
                f"No authentication client exists for "
                f"node '{node_name}'."
            )

        return self._authentication_clients[
            node_name
        ]

    def quantum_memory(
        self,
        node_or_name
    ) -> QuantumMemory:

        node_name = self._resolve_node_name(
            node_or_name
        )

        if node_name not in self._quantum_memories:
            raise TopologyNodeNotFoundError(
                f"No quantum memory exists for node "
                f"'{node_name}'."
            )

        return self._quantum_memories[
            node_name
        ]

    # =========================================================
    # Read-Only Runtime Registry Views
    # =========================================================

    @property
    def device_runtimes(self):
        return MappingProxyType(
            self._device_runtimes
        )

    @property
    def authentication_clients(self):
        return MappingProxyType(
            self._authentication_clients
        )

    @property
    def quantum_memories(self):
        return MappingProxyType(
            self._quantum_memories
        )

    # =========================================================
    # Authentication Orchestration
    # =========================================================

    def authenticate_node(
        self,
        node_or_name
    ) -> AuthenticationRecord:
        """
        Execute one fresh mutual ML-DSA authentication exchange
        for a provisioned physical device.
        """

        self.validate_runtime_consistency()

        node_name = self._resolve_node_name(
            node_or_name
        )

        client = self.authentication_client(
            node_name
        )

        record = (
            self.authentication_service
            .mutually_authenticate(
                client
            )
        )

        self.kernel.emit(
            event_type=(
                "DEVICE_MUTUAL_AUTHENTICATION_"
                f"{record.status.value}"
            ),
            component="QuantumNetworkSimulator",
            node_names=(node_name,),
            message=(
                f"Mutual authentication for '{node_name}' "
                f"completed with status "
                f"{record.status.value}."
            ),
            metadata={
                "authentication_session_id":
                    record.session_id,

                "authentication_attempt":
                    record.attempt_index,

                "reason":
                    record.reason
            }
        )

        return record

    def authenticate_all_nodes(
        self
    ) -> dict[str, AuthenticationRecord]:
        """
        Mutually authenticate every physical device currently
        present in the assembled network.
        """

        self.validate_runtime_consistency()

        results = {}

        for node in self.topology.nodes:

            results[
                node.name
            ] = self.authenticate_node(
                node
            )

        return results

    # =========================================================
    # Current Authorization Gate
    # =========================================================

    def is_quantum_eligible(
        self,
        node_or_name
    ) -> bool:

        return (
            self.authentication_service
            .is_quantum_eligible(
                node_or_name
            )
        )

    def require_quantum_eligibility(
        self,
        *nodes
    ) -> bool:
        """
        Hard authorization gate for future quantum operations.

        All supplied devices must be mutually authenticated.

        Phase 3B and later quantum-resource operations should
        call this gate BEFORE attempting authorized physical
        quantum work.
        """

        if not nodes:
            raise ValueError(
                "At least one node must be supplied."
            )

        ineligible = []

        for node in nodes:

            node_name = self._resolve_node_name(
                node
            )

            if not self.is_quantum_eligible(
                node_name
            ):
                ineligible.append(
                    node_name
                )

        if ineligible:
            raise QuantumOperationAuthorizationError(
                "Quantum operation denied. "
                "The following node(s) are not currently "
                "mutually authenticated: "
                + ", ".join(ineligible)
            )

        return True

    @property
    def eligible_node_names(
        self
    ) -> tuple[str, ...]:

        return tuple(
            node.name
            for node in self.topology.nodes
            if self.is_quantum_eligible(
                node
            )
        )

    @property
    def ineligible_node_names(
        self
    ) -> tuple[str, ...]:

        return tuple(
            node.name
            for node in self.topology.nodes
            if not self.is_quantum_eligible(
                node
            )
        )

    # =========================================================
    # Cross-Layer Runtime Consistency
    # =========================================================

    def validate_runtime_consistency(
        self
    ) -> bool:
        """
        Verify that the assembled runtime still corresponds
        exactly to the physical topology used at construction.

        Live topology mutation is deliberately unsupported in
        Phase 3A.
        """

        self.topology.validate_internal_consistency()

        current_nodes = {
            node.name: node
            for node in self.topology.nodes
        }

        current_links = {
            link.link_id: link
            for link in self.topology.links
        }

        # -----------------------------------------------------
        # Detect post-assembly topology changes
        # -----------------------------------------------------

        if (
            set(current_nodes)
            != set(self._assembled_node_objects)
        ):
            raise QuantumNetworkRuntimeConsistencyError(
                "Physical topology node set changed after "
                "simulator assembly."
            )

        if (
            set(current_links)
            != set(self._assembled_link_objects)
        ):
            raise QuantumNetworkRuntimeConsistencyError(
                "Physical topology link set changed after "
                "simulator assembly."
            )

        for node_name, original_node in (
            self._assembled_node_objects.items()
        ):

            if (
                current_nodes[node_name]
                is not original_node
            ):
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Canonical physical node '{node_name}' "
                    f"changed after simulator assembly."
                )

        for link_id, original_link in (
            self._assembled_link_objects.items()
        ):

            if (
                current_links[link_id]
                is not original_link
            ):
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Canonical physical link '{link_id}' "
                    f"changed after simulator assembly."
                )

        expected_names = set(
            current_nodes
        )

        # -----------------------------------------------------
        # Runtime registries must cover exactly the topology
        # -----------------------------------------------------

        if (
            set(self._device_runtimes)
            != expected_names
        ):
            raise QuantumNetworkRuntimeConsistencyError(
                "Device-runtime registry disagrees with "
                "physical topology."
            )

        if (
            set(self._authentication_clients)
            != expected_names
        ):
            raise QuantumNetworkRuntimeConsistencyError(
                "Authentication-client registry disagrees "
                "with physical topology."
            )

        if (
            set(self._quantum_memories)
            != expected_names
        ):
            raise QuantumNetworkRuntimeConsistencyError(
                "Quantum-memory registry disagrees with "
                "physical topology."
            )

        # -----------------------------------------------------
        # Validate every node's cross-layer identity
        # -----------------------------------------------------

        for node_name, node in (
            current_nodes.items()
        ):

            runtime = self._device_runtimes[
                node_name
            ]

            client = self._authentication_clients[
                node_name
            ]

            memory = self._quantum_memories[
                node_name
            ]

            if runtime.physical_node is not node:
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Runtime for '{node_name}' references "
                    f"the wrong physical node object."
                )

            if client.runtime is not runtime:
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Authentication client for '{node_name}' "
                    f"references the wrong runtime."
                )

            if memory.owner is not node:
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Quantum memory for '{node_name}' "
                    f"references the wrong physical node."
                )

            if memory.kernel is not self.kernel:
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Quantum memory for '{node_name}' "
                    f"uses the wrong SimulationKernel."
                )

            # Trusted peer registry must contain exactly the
            # runtime's exported public identity.
            peer_identity = (
                self.control_plane.peer_public_identity(
                    node_name
                )
            )

            if (
                peer_identity
                != runtime.public_identity
            ):
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Trusted peer identity for '{node_name}' "
                    f"does not match its device runtime."
                )

            # Node must pin this controller.
            if not runtime.trusts_controller(
                self.control_plane.controller_id
            ):
                raise QuantumNetworkRuntimeConsistencyError(
                    f"Node '{node_name}' does not trust the "
                    f"assembled network controller."
                )

            # Global resource manager must own this same memory.
            if (
                self.entanglement_manager
                .memory_for_node(node_name)
                is not memory
            ):
                raise QuantumNetworkRuntimeConsistencyError(
                    f"EntanglementManager memory for "
                    f"'{node_name}' disagrees with simulator."
                )

        # -----------------------------------------------------
        # Entanglement consistency
        # -----------------------------------------------------

        self.entanglement_manager.validate_all_active_resources()

        return True

    # =========================================================
    # Runtime Inspection
    # =========================================================

    def nodes_runtime_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for node in self.topology.nodes:

            runtime = self.device_runtime(
                node
            )

            memory = self.quantum_memory(
                node
            )

            rows.append({
                "Node":
                    node.name,

                "Role":
                    node.functional_role,

                "Position (km)":
                    node.position,

                "Memory Capacity":
                    memory.capacity,

                "Memory Occupancy":
                    memory.occupancy,

                "ML-DSA Profile":
                    runtime.public_identity.parameter_set.value,

                "Key ID":
                    runtime.key_id,

                "Enrolled":
                    self.control_plane.is_enrolled(
                        node
                    ),

                "Authentication Status":
                    self.authentication_service
                    .authentication_status(
                        node
                    ).value,

                "Quantum Eligible":
                    self.is_quantum_eligible(
                        node
                    )
            })

        return pd.DataFrame(
            rows
        )

    def summary(
        self
    ) -> dict:

        return {
            "topology":
                self.topology.name,

            "simulation_seed":
                self.kernel.seed,

            "simulation_slot":
                self.kernel.current_slot,

            "controller_id":
                self.control_plane.controller_id,

            "ml_dsa_parameter_set":
                self.control_plane.parameter_set.value,

            "nodes":
                self.topology.node_count,

            "links":
                self.topology.link_count,

            "trusted_peers":
                self.control_plane.peer_count,

            "authenticated_nodes":
                len(
                    self.eligible_node_names
                ),

            "active_entanglement_resources":
                len(
                    self.entanglement_manager.active_resources
                )
        }

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "QuantumNetworkSimulator("
            f"topology='{self.topology.name}', "
            f"nodes={self.topology.node_count}, "
            f"links={self.topology.link_count}, "
            f"authenticated="
            f"{len(self.eligible_node_names)}/"
            f"{self.topology.node_count}, "
            f"slot={self.kernel.current_slot}"
            ")"
        )

In [ ]:
# @title Linear Chain Topology Builder


def build_linear_chain_topology(
    user_1_name: str,
    user_1_position_km: tuple[float, float],

    repeater_names: list[str],
    repeater_positions_km: list[
        tuple[float, float]
    ],

    user_2_name: str,
    user_2_position_km: tuple[float, float],

    topology_name: str = (
        "NARAD-QC-Linear-Verification"
    ),

    attenuation_db_per_km: float = (
        PHASE3_DEFAULT_FIBER_ATTENUATION_DB_PER_KM
    )
):
    """
    Build an ordered physical quantum chain:

        User 1
          -> Repeater 1
          -> Repeater 2
          -> ...
          -> Repeater N
          -> User 2

    Every coordinate unit represents exactly one kilometre.

    Returns
    -------
    topology : NetworkTopology

    ordered_path : tuple[NetworkNode, ...]
        Canonical node order from User 1 to User 2.
    """

    # =========================================================
    # Basic repeater input consistency
    # =========================================================

    if len(repeater_names) != len(
        repeater_positions_km
    ):
        raise ValueError(
            "Number of repeater names must match "
            "number of repeater coordinate pairs."
        )

    if len(repeater_names) < 1:
        raise ValueError(
            "Phase 3A.10 chain verification requires "
            "at least one repeater."
        )

    all_names = [
        user_1_name,
        *repeater_names,
        user_2_name
    ]

    normalized_names = [
        str(name).strip()
        for name in all_names
    ]

    if any(
        not name
        for name in normalized_names
    ):
        raise ValueError(
            "All user and repeater names must be non-empty."
        )

    if (
        len(set(normalized_names))
        != len(normalized_names)
    ):
        raise ValueError(
            "Every node in the verification chain must "
            "have a unique name."
        )

    # =========================================================
    # Create topology
    # =========================================================

    topology = NetworkTopology(
        name=topology_name
    )

    # Verification hardware baseline:
    #
    # 2 local quantum-memory slots per physical device.
    #
    # This is only a Phase-3A assembly fixture, not yet an
    # experimental memory-capacity assumption.
    user_1 = NetworkNode(
        name=user_1_name,
        x_km=user_1_position_km[0],
        y_km=user_1_position_km[1],
        endpoint_capable=True,
        swapping_capable=False,
        memory_capacity=2
    )

    topology.add_node(
        user_1
    )

    repeaters = []

    for (
        repeater_name,
        repeater_position
    ) in zip(
        repeater_names,
        repeater_positions_km
    ):

        repeater = NetworkNode(
            name=repeater_name,

            x_km=repeater_position[0],
            y_km=repeater_position[1],

            endpoint_capable=False,
            swapping_capable=True,

            memory_capacity=2
        )

        topology.add_node(
            repeater
        )

        repeaters.append(
            repeater
        )

    user_2 = NetworkNode(
        name=user_2_name,
        x_km=user_2_position_km[0],
        y_km=user_2_position_km[1],
        endpoint_capable=True,
        swapping_capable=False,
        memory_capacity=2
    )

    topology.add_node(
        user_2
    )

    ordered_path = (
        user_1,
        *repeaters,
        user_2
    )

    # =========================================================
    # Connect consecutive physical devices
    # =========================================================

    for left, right in zip(
        ordered_path[:-1],
        ordered_path[1:]
    ):

        link = PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=(
                attenuation_db_per_km
            )
        )

        topology.add_link(
            link
        )

    topology.validate_internal_consistency()
    topology.validate_path(
        ordered_path
    )

    return (
        topology,
        ordered_path
    )

In [ ]:
# @title Dynamic Physical Network Plot

import matplotlib.pyplot as plt


def plot_quantum_chain(
    topology: NetworkTopology,
    ordered_path
):
    """
    Plot one ordered NARAD-QC physical chain.

    Spatial scale:
        1 coordinate unit = 1 kilometre.

    Physical links are undirected.
    Link labels show Euclidean fibre distance in kilometres.
    """

    nodes = topology.validate_path(
        ordered_path
    )

    links = topology.path_links(
        nodes
    )

    node_count = len(
        nodes
    )

    figure_width = max(
        9,
        min(
            18,
            2.0 * node_count
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            figure_width,
            7
        )
    )

    # =========================================================
    # Physical Links
    # =========================================================

    for link in links:

        x_values = [
            link.node_a.x_km,
            link.node_b.x_km
        ]

        y_values = [
            link.node_a.y_km,
            link.node_b.y_km
        ]

        ax.plot(
            x_values,
            y_values,
            linewidth=2,
            zorder=1
        )

        midpoint_x = (
            link.node_a.x_km
            + link.node_b.x_km
        ) / 2.0

        midpoint_y = (
            link.node_a.y_km
            + link.node_b.y_km
        ) / 2.0

        ax.annotate(
            f"{link.distance_km:.2f} km",

            xy=(
                midpoint_x,
                midpoint_y
            ),

            xytext=(
                0,
                8
            ),

            textcoords="offset points",

            ha="center",
            va="bottom",

            fontsize=9,

            bbox={
                "boxstyle": "round,pad=0.2",
                "facecolor": "white",
                "alpha": 0.8,
                "edgecolor": "none"
            },

            zorder=4
        )

    # =========================================================
    # User Nodes
    # =========================================================

    user_nodes = [
        node
        for node in nodes
        if node.endpoint_capable
        and not node.swapping_capable
    ]

    ax.scatter(
        [
            node.x_km
            for node in user_nodes
        ],
        [
            node.y_km
            for node in user_nodes
        ],
        s=140,
        marker="o",
        label="User / Endpoint",
        zorder=3
    )

    # =========================================================
    # Repeater Nodes
    # =========================================================

    repeater_nodes = [
        node
        for node in nodes
        if node.swapping_capable
        and not node.endpoint_capable
    ]

    ax.scatter(
        [
            node.x_km
            for node in repeater_nodes
        ],
        [
            node.y_km
            for node in repeater_nodes
        ],
        s=140,
        marker="s",
        label="Quantum Repeater",
        zorder=3
    )

    # =========================================================
    # Node Labels
    # =========================================================

    for index, node in enumerate(
        nodes
    ):

        role_text = (
            "USER"
            if node.endpoint_capable
            else "REPEATER"
        )

        ax.annotate(
            f"{node.name}\n{role_text}",

            xy=(
                node.x_km,
                node.y_km
            ),

            xytext=(
                0,
                -15
            ),

            textcoords="offset points",

            ha="center",
            va="top",

            fontsize=9,
            fontweight="bold",

            zorder=5
        )

    # =========================================================
    # Spatial Scaling
    # =========================================================

    # Critical:
    # 1 unit on X and 1 unit on Y represent the same physical
    # distance. Since coordinates are supplied in kilometres,
    # one graph coordinate unit is exactly one kilometre.
    ax.set_aspect(
        "equal",
        adjustable="datalim"
    )

    ax.set_xlabel(
        "X Position (km) — 1 coordinate unit = 1 km"
    )

    ax.set_ylabel(
        "Y Position (km) — 1 coordinate unit = 1 km"
    )

    chain_text = " → ".join(
        node.name
        for node in nodes
    )

    ax.set_title(
        "NARAD-QC Physical Quantum Network\n"
        f"{chain_text}"
    )

    ax.grid(
        True,
        alpha=0.3
    )

    ax.legend()

    # Provide some breathing room around outer nodes.
    x_values = [
        node.x_km
        for node in nodes
    ]

    y_values = [
        node.y_km
        for node in nodes
    ]

    x_span = max(
        max(x_values) - min(x_values),
        1.0
    )

    y_span = max(
        max(y_values) - min(y_values),
        1.0
    )

    padding = max(
        3.0,
        0.08 * max(
            x_span,
            y_span
        )
    )

    ax.set_xlim(
        min(x_values) - padding,
        max(x_values) + padding
    )

    ax.set_ylim(
        min(y_values) - padding,
        max(y_values) + padding
    )

    plt.show()

    return (
        fig,
        ax
    )

In [ ]:
# @title Custom NARAD-QC Network Verification Form { display-mode: "form" }

# =============================================================
# USER 1
# =============================================================

user_1_name = "Alice" # @param {type:"string"}

user_1_x_km = 0 # @param {type:"number"}

user_1_y_km = 0 # @param {type:"number"}


# =============================================================
# REPEATERS
#
# Enter repeater names in physical chain order:
#
#     R1,R2,R3
#
# The FIRST repeater is connected to User 1.
# The LAST repeater is connected to User 2.
#
# Coordinates use the same order:
#
#     35,20;80,20;120,60
#
# means:
#
#     R1 = (35, 20) km
#     R2 = (80, 20) km
#     R3 = (120, 60) km
# =============================================================

repeater_nodes = "R1,R2,R3" # @param {type:"string"}

repeater_coordinates_km = "35,20;80,20;120,60" # @param {type:"string"}


# =============================================================
# USER 2
# =============================================================

user_2_name = "Bob" # @param {type:"string"}

user_2_x_km = 170 # @param {type:"number"}

user_2_y_km = 30 # @param {type:"number"}


# =============================================================
# Parse Form Input
# =============================================================

def _parse_repeater_names(
    repeater_text: str
) -> list[str]:

    names = [
        item.strip()
        for item in repeater_text.split(",")
        if item.strip()
    ]

    if not names:
        raise ValueError(
            "Please provide at least one repeater node."
        )

    return names


def _parse_coordinate_pairs(
    coordinate_text: str
) -> list[tuple[float, float]]:

    raw_pairs = [
        item.strip()
        for item in coordinate_text.split(";")
        if item.strip()
    ]

    coordinates = []

    for index, pair_text in enumerate(
        raw_pairs,
        start=1
    ):

        components = [
            value.strip()
            for value in pair_text.split(",")
        ]

        if len(components) != 2:
            raise ValueError(
                f"Repeater coordinate #{index} must have "
                f"exactly two values: x,y"
            )

        try:

            x_value = float(
                components[0]
            )

            y_value = float(
                components[1]
            )

        except ValueError as exc:

            raise ValueError(
                f"Repeater coordinate #{index} contains "
                f"a non-numeric value."
            ) from exc

        if (
            not math.isfinite(x_value)
            or not math.isfinite(y_value)
        ):
            raise ValueError(
                f"Repeater coordinate #{index} must contain "
                f"finite coordinates."
            )

        coordinates.append(
            (
                x_value,
                y_value
            )
        )

    return coordinates


parsed_repeater_names = (
    _parse_repeater_names(
        repeater_nodes
    )
)

parsed_repeater_coordinates = (
    _parse_coordinate_pairs(
        repeater_coordinates_km
    )
)


if (
    len(parsed_repeater_names)
    != len(parsed_repeater_coordinates)
):

    raise ValueError(
        "Repeater input mismatch: "
        f"{len(parsed_repeater_names)} repeater name(s) "
        f"were supplied but "
        f"{len(parsed_repeater_coordinates)} coordinate "
        f"pair(s) were supplied."
    )


# =============================================================
# Build User-Specified Physical Chain
# =============================================================

verification_topology, verification_path = (
    build_linear_chain_topology(

        user_1_name=(
            user_1_name.strip()
        ),

        user_1_position_km=(
            float(user_1_x_km),
            float(user_1_y_km)
        ),

        repeater_names=(
            parsed_repeater_names
        ),

        repeater_positions_km=(
            parsed_repeater_coordinates
        ),

        user_2_name=(
            user_2_name.strip()
        ),

        user_2_position_km=(
            float(user_2_x_km),
            float(user_2_y_km)
        ),

        topology_name=(
            "NARAD-QC-Phase3A10-Verification"
        )
    )
)


print("=" * 76)
print("CUSTOM NARAD-QC PHYSICAL CHAIN")
print("=" * 76)

print(
    "\nChain order:"
)

print(
    " -> ".join(
        node.name
        for node in verification_path
    )
)

print(
    "\nTotal physical path distance: "
    f"{verification_topology.path_distance_km(verification_path):.2f} km"
)

print(
    "\nEach coordinate unit in the graph represents exactly 1 km."
)


# =============================================================
# Display Physical Network
# =============================================================

plot_quantum_chain(
    verification_topology,
    verification_path
)

In [ ]:
# @title Full Quantum Network Simulator Verification


print("=" * 76)
print("NARAD-QC - Quantum Network Simulator Verification")
print("=" * 76)


# =============================================================
# Assemble the complete runtime world
#
# max_storage_slots=None intentionally:
# Phase 3A assembly verification does not impose an arbitrary
# quantum-memory lifetime experiment yet.
# =============================================================

verification_simulator = (
    QuantumNetworkSimulator(

        topology=(
            verification_topology
        ),

        simulation_seed=(
            310010
        ),

        controller_id=(
            "NARAD-QC-CONTROLLER"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=None
    )
)


# =============================================================
# Test 1 — Simulator assembled successfully
# =============================================================

assert isinstance(
    verification_simulator,
    QuantumNetworkSimulator
)

assert (
    verification_simulator.topology
    is verification_topology
)

print(
    "[PASS] QuantumNetworkSimulator assembled successfully."
)


# =============================================================
# Test 2 — Runtime matches physical topology
# =============================================================

assert (
    verification_simulator
    .validate_runtime_consistency()
    is True
)

assert (
    len(
        verification_simulator.device_runtimes
    )
    == verification_topology.node_count
)

assert (
    len(
        verification_simulator.quantum_memories
    )
    == verification_topology.node_count
)

assert (
    len(
        verification_simulator.authentication_clients
    )
    == verification_topology.node_count
)

print(
    "[PASS] Every physical node has exactly one runtime, "
    "authentication client, and quantum memory."
)


# =============================================================
# Test 3 — Controller enrolled every physical device
# =============================================================

assert (
    verification_simulator
    .control_plane.peer_count
    == verification_topology.node_count
)

for node in verification_topology.nodes:

    assert (
        verification_simulator
        .control_plane.is_enrolled(
            node
        )
    )

print(
    "[PASS] Every physical device is trusted-enrolled "
    "with the central control plane."
)


# =============================================================
# Test 4 — Enrollment alone grants NO quantum eligibility
# =============================================================

for node in verification_topology.nodes:

    assert not (
        verification_simulator
        .is_quantum_eligible(
            node
        )
    )

assert (
    len(
        verification_simulator
        .eligible_node_names
    )
    == 0
)

print(
    "[PASS] Newly provisioned devices cannot use quantum "
    "resources before mutual authentication."
)


# =============================================================
# Test 5 — Authorization gate blocks unauthenticated chain
# =============================================================

authorization_blocked = False

try:

    verification_simulator.require_quantum_eligibility(
        *verification_path
    )

except QuantumOperationAuthorizationError:
    authorization_blocked = True

assert authorization_blocked

print(
    "[PASS] Complete quantum chain is blocked while "
    "its devices are unauthenticated."
)


# =============================================================
# Test 6 — Mutually authenticate every physical device
# =============================================================

authentication_results = (
    verification_simulator
    .authenticate_all_nodes()
)

assert (
    len(authentication_results)
    == verification_topology.node_count
)

assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in authentication_results.values()
)

print(
    "[PASS] Every device completed fresh mutual "
    "ML-DSA authentication."
)


# =============================================================
# Test 7 — Entire chain becomes quantum eligible
# =============================================================

assert (
    verification_simulator
    .require_quantum_eligibility(
        *verification_path
    )
    is True
)

assert set(
    verification_simulator
    .eligible_node_names
) == {
    node.name
    for node in verification_topology.nodes
}

print(
    "[PASS] Fully authenticated chain is authorized "
    "for future quantum operations."
)


# =============================================================
# Test 8 — Every runtime uses the correct physical node
# =============================================================

for node in verification_topology.nodes:

    runtime = (
        verification_simulator
        .device_runtime(
            node
        )
    )

    memory = (
        verification_simulator
        .quantum_memory(
            node
        )
    )

    assert runtime.physical_node is node
    assert memory.owner is node

print(
    "[PASS] Static physical identity and runtime device "
    "identity are correctly bound."
)


# =============================================================
# Test 9 — Every node trusts exactly the assembled controller
# =============================================================

for node in verification_topology.nodes:

    runtime = (
        verification_simulator
        .device_runtime(
            node
        )
    )

    assert runtime.trusts_controller(
        verification_simulator
        .control_plane.controller_id
    )

print(
    "[PASS] Every physical device recognizes the network "
    "controller as its trust anchor."
)


# =============================================================
# Test 10 — Quantum memories are registered globally
# =============================================================

for node in verification_topology.nodes:

    simulator_memory = (
        verification_simulator
        .quantum_memory(
            node
        )
    )

    manager_memory = (
        verification_simulator
        .entanglement_manager
        .memory_for_node(
            node
        )
    )

    assert (
        simulator_memory
        is manager_memory
    )

print(
    "[PASS] Every local quantum memory is registered with "
    "the authoritative EntanglementManager."
)


# =============================================================
# Test 11 — No quantum resources exist merely from assembly
# =============================================================

assert (
    len(
        verification_simulator
        .entanglement_manager.resources
    )
    == 0
)

assert (
    len(
        verification_simulator
        .entanglement_manager.active_resources
    )
    == 0
)

print(
    "[PASS] Simulator assembly itself creates no "
    "entanglement resources."
)


# =============================================================
# Test 12 — Authentication invalidation immediately closes
#           the quantum authorization gate
# =============================================================

test_gate_node = (
    verification_path[1]
)

invalidated_record = (
    verification_simulator
    .authentication_service
    .invalidate_authentication(
        test_gate_node,
        reason=(
            "phase3a10_authorization_gate_test"
        )
    )
)

assert (
    invalidated_record.status
    == AuthenticationStatus.INVALIDATED
)

assert not (
    verification_simulator
    .is_quantum_eligible(
        test_gate_node
    )
)


gate_closed_after_invalidation = False

try:

    verification_simulator.require_quantum_eligibility(
        *verification_path
    )

except QuantumOperationAuthorizationError:
    gate_closed_after_invalidation = True

assert gate_closed_after_invalidation

print(
    "[PASS] Invalidating one repeater immediately blocks "
    "the entire chain from authorized quantum use."
)


# =============================================================
# Test 13 — Fresh reauthentication restores the chain
# =============================================================

restored_record = (
    verification_simulator
    .authenticate_node(
        test_gate_node
    )
)

assert (
    restored_record.status
    == AuthenticationStatus.AUTHENTICATED
)

assert (
    verification_simulator
    .require_quantum_eligibility(
        *verification_path
    )
    is True
)

print(
    "[PASS] Fresh mutual authentication restores "
    "the chain's quantum eligibility."
)


# =============================================================
# Test 14 — Final complete cross-component consistency
# =============================================================

assert (
    verification_simulator
    .validate_runtime_consistency()
    is True
)

print(
    "[PASS] Complete assembled NARAD-QC runtime is "
    "cross-layer consistent."
)


# =============================================================
# Runtime Inspection
# =============================================================

print("\nSIMULATOR")
print(
    verification_simulator
)


print("\nSIMULATOR SUMMARY")

print(
    verification_simulator.summary()
)


print("\nPHYSICAL TOPOLOGY")

display(
    verification_topology.nodes_dataframe()
)


print("\nPHYSICAL LINKS")

display(
    verification_topology.links_dataframe()
)


print("\nRUNTIME DEVICE STATE")

display(
    verification_simulator
    .nodes_runtime_dataframe()
)


print("\nTRUSTED PEER REGISTRY")

display(
    verification_simulator
    .control_plane
    .trusted_peer_registry_dataframe()
)


print("\nAUTHENTICATION HISTORY")

display(
    verification_simulator
    .authentication_service
    .authentication_dataframe()
)


print("\nSIMULATION EVENT LOG")

display(
    verification_simulator
    .kernel
    .events_dataframe()
)


# =============================================================
# Final Dynamic Network Graph
# =============================================================

print("\nFINAL AUTHENTICATED PHYSICAL NETWORK")

plot_quantum_chain(
    verification_topology,
    verification_path
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

### Phase 3B : 10-09-2026T11:20:52 -- 10-09-2026T13:14:16

In [ ]:
# @title Elementary Entanglement Physical Model

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional
import math
import numbers


# =============================================================
# Elementary Entanglement Errors
# =============================================================

class ElementaryEntanglementError(Exception):
    """
    Base exception for Phase-3B elementary entanglement.
    """
    pass


class ElementaryEntanglementModelError(
    ElementaryEntanglementError
):
    pass


class ElementaryEntanglementStorageError(
    ElementaryEntanglementError
):
    pass


# =============================================================
# Physical Model Interface
# =============================================================

class ElementaryEntanglementModel(ABC):
    """
    Abstract physical model for one elementary-entanglement
    generation attempt over one PhysicalQuantumLink.

    A model is intentionally STATELESS with respect to runtime
    attempts.

    Its responsibility is only to describe physical success
    probability.

    It does NOT:
    - draw random numbers;
    - advance simulation time;
    - create EntanglementResource objects;
    - mutate quantum memory;
    - authenticate nodes;
    - own attempt counters.
    """

    @property
    @abstractmethod
    def model_name(self) -> str:
        pass

    @abstractmethod
    def success_probability(
        self,
        link: PhysicalQuantumLink
    ) -> float:
        """
        Return probability that one physical elementary-
        entanglement attempt succeeds.
        """
        pass

    def validate_probability(
        self,
        probability: float
    ) -> float:

        if (
            isinstance(probability, bool)
            or not isinstance(
                probability,
                numbers.Real
            )
            or not math.isfinite(
                float(probability)
            )
        ):
            raise ElementaryEntanglementModelError(
                "Physical success probability must be "
                "a finite real number."
            )

        probability = float(
            probability
        )

        if not 0.0 <= probability <= 1.0:
            raise ElementaryEntanglementModelError(
                "Physical success probability must lie "
                "within [0, 1]."
            )

        return probability


# =============================================================
# Phase 1–2 Compatible Fibre-Survival Proxy
# =============================================================

class FiberSurvivalProxyModel(
    ElementaryEntanglementModel
):
    """
    Initial Phase-3B physical abstraction.

    One elementary-entanglement attempt succeeds with probability

        P_success = eta(d)

    where eta(d) is the complete-link fibre-survival probability
    already used by the validated Phase 1–2 attenuation model.
    """

    @property
    def model_name(self) -> str:
        return "FiberSurvivalProxyModel"

    def success_probability(
        self,
        link: PhysicalQuantumLink
    ) -> float:

        if not isinstance(
            link,
            PhysicalQuantumLink
        ):
            raise TypeError(
                "link must be a PhysicalQuantumLink."
            )

        return self.validate_probability(
            link.survival_probability
        )

    def __repr__(
        self
    ) -> str:

        return (
            "FiberSurvivalProxyModel("
            "P_success=full_link_survival_probability"
            ")"
        )

In [ ]:
# @title Elementary Entanglement Attempt Record


@dataclass(frozen=True)
class ElementaryEntanglementAttempt:
    """
    Immutable record of one physical elementary-entanglement
    generation attempt.

    The random draw is retained for reproducibility/debugging
    of seeded physical simulations.

    No cryptographic randomness is represented here.
    """

    attempt_index: int

    simulation_slot: int

    link_id: str

    node_a_name: str
    node_b_name: str

    model_name: str

    success_probability: float

    random_draw: float

    success: bool

    resource_id: Optional[str] = None

    def __post_init__(self):

        if (
            isinstance(self.attempt_index, bool)
            or not isinstance(
                self.attempt_index,
                int
            )
            or self.attempt_index < 0
        ):
            raise ValueError(
                "attempt_index must be a non-negative integer."
            )

        if (
            isinstance(self.simulation_slot, bool)
            or not isinstance(
                self.simulation_slot,
                int
            )
            or self.simulation_slot < 0
        ):
            raise ValueError(
                "simulation_slot must be a non-negative integer."
            )

        if not 0.0 <= (
            self.success_probability
        ) <= 1.0:
            raise ValueError(
                "success_probability must lie within [0, 1]."
            )

        if not 0.0 <= (
            self.random_draw
        ) < 1.0:
            raise ValueError(
                "random_draw must lie within [0, 1)."
            )

        if self.success:

            if (
                not isinstance(self.resource_id, str)
                or not self.resource_id.strip()
            ):
                raise ValueError(
                    "Successful attempt requires resource_id."
                )

        else:

            if self.resource_id is not None:
                raise ValueError(
                    "Failed attempt cannot have resource_id."
                )

In [ ]:
# @title Elementary Entanglement Runtime Statistics


class ElementaryEntanglementStatistics:
    """
    Runtime statistics for elementary-entanglement attempts.
    """

    def __init__(self):

        self._attempts: list[
            ElementaryEntanglementAttempt
        ] = []

    def record(
        self,
        attempt: ElementaryEntanglementAttempt
    ) -> None:

        if not isinstance(
            attempt,
            ElementaryEntanglementAttempt
        ):
            raise TypeError(
                "attempt must be an "
                "ElementaryEntanglementAttempt."
            )

        self._attempts.append(
            attempt
        )

    @property
    def attempts(
        self
    ) -> tuple[
        ElementaryEntanglementAttempt, ...
    ]:

        return tuple(
            self._attempts
        )

    @property
    def total_attempts(
        self
    ) -> int:

        return len(
            self._attempts
        )

    @property
    def total_successes(
        self
    ) -> int:

        return sum(
            attempt.success
            for attempt
            in self._attempts
        )

    @property
    def total_failures(
        self
    ) -> int:

        return (
            self.total_attempts
            - self.total_successes
        )

    @property
    def empirical_success_rate(
        self
    ) -> float:

        if self.total_attempts == 0:
            return 0.0

        return (
            self.total_successes
            / self.total_attempts
        )

    def attempts_for_link(
        self,
        link_or_id
    ) -> tuple[
        ElementaryEntanglementAttempt, ...
    ]:

        link_id = (
            link_or_id.link_id
            if isinstance(
                link_or_id,
                PhysicalQuantumLink
            )
            else str(link_or_id)
        )

        return tuple(
            attempt
            for attempt
            in self._attempts
            if attempt.link_id
            == link_id
        )

    def link_summary_dataframe(
        self
    ) -> pd.DataFrame:

        link_ids = sorted({
            attempt.link_id
            for attempt
            in self._attempts
        })

        rows = []

        for link_id in link_ids:

            attempts = (
                self.attempts_for_link(
                    link_id
                )
            )

            successes = sum(
                attempt.success
                for attempt
                in attempts
            )

            total = len(
                attempts
            )

            rows.append({
                "Link ID":
                    link_id,

                "Attempts":
                    total,

                "Successes":
                    successes,

                "Failures":
                    total - successes,

                "Empirical Success Rate":
                    (
                        successes / total
                        if total
                        else 0.0
                    )
            })

        return pd.DataFrame(
            rows
        )

    def attempts_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for attempt in self._attempts:

            rows.append({
                "Attempt":
                    attempt.attempt_index,

                "Simulation Slot":
                    attempt.simulation_slot,

                "Link ID":
                    attempt.link_id,

                "Node A":
                    attempt.node_a_name,

                "Node B":
                    attempt.node_b_name,

                "Model":
                    attempt.model_name,

                "P(success)":
                    attempt.success_probability,

                "Random Draw":
                    attempt.random_draw,

                "Success":
                    attempt.success,

                "Resource ID":
                    attempt.resource_id
            })

        return pd.DataFrame(
            rows
        )

    def __repr__(
        self
    ) -> str:

        return (
            "ElementaryEntanglementStatistics("
            f"attempts={self.total_attempts}, "
            f"successes={self.total_successes}, "
            f"failures={self.total_failures}"
            ")"
        )

In [ ]:
# @title Elementary Entanglement Single-Attempt Service


class ElementaryEntanglementService:
    """
    Executes authorized elementary-entanglement generation
    attempts inside one QuantumNetworkSimulator.

    Responsibilities
    ----------------
    - validate the physical link belongs to the simulator;
    - enforce mutual-authentication authorization;
    - ensure endpoint memory can accept a new resource;
    - obtain physical success probability from a model;
    - draw one stochastic result from SimulationKernel.rng;
    - create a global EntanglementResource only after success;
    - record attempt statistics and simulation events.

    The service deliberately does NOT:
    - advance simulation time;
    - retry automatically;
    - schedule multiple links;
    - perform entanglement swapping;
    - determine routes.

    Those responsibilities belong to later Phase-3 components.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        model: ElementaryEntanglementModel
    ):

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            model,
            ElementaryEntanglementModel
        ):
            raise TypeError(
                "model must be an "
                "ElementaryEntanglementModel."
            )

        self.simulator = simulator
        self.model = model

        self.statistics = (
            ElementaryEntanglementStatistics()
        )

    # =========================================================
    # Link Validation
    # =========================================================

    def _validate_link(
        self,
        link: PhysicalQuantumLink
    ) -> PhysicalQuantumLink:

        if not isinstance(
            link,
            PhysicalQuantumLink
        ):
            raise TypeError(
                "link must be a PhysicalQuantumLink."
            )

        # Exact canonical topology object required.
        canonical_link = (
            self.simulator.topology
            .get_link_by_id(
                link.link_id
            )
        )

        if canonical_link is not link:
            raise TopologyConsistencyError(
                f"Link '{link.link_id}' is not the "
                f"canonical link object owned by this "
                f"simulation topology."
            )

        return canonical_link

    # =========================================================
    # Storage Preflight
    # =========================================================

    def _require_storage_capacity(
        self,
        link: PhysicalQuantumLink
    ) -> None:

        memory_a = (
            self.simulator.quantum_memory(
                link.node_a
            )
        )

        memory_b = (
            self.simulator.quantum_memory(
                link.node_b
            )
        )

        full_nodes = []

        if memory_a.is_full:
            full_nodes.append(
                link.node_a.name
            )

        if memory_b.is_full:
            full_nodes.append(
                link.node_b.name
            )

        if full_nodes:
            raise ElementaryEntanglementStorageError(
                "Elementary entanglement attempt denied "
                "because no storage slot is available at: "
                + ", ".join(full_nodes)
            )

    # =========================================================
    # One Physical Attempt
    # =========================================================

    def attempt_once(
        self,
        link: PhysicalQuantumLink
    ) -> ElementaryEntanglementAttempt:
        """
        Execute exactly ONE physical elementary-entanglement
        generation attempt.

        IMPORTANT
        ---------
        This method does NOT advance SimulationKernel.current_slot.

        Multiple links may later call attempt_once() during the
        same global simulation slot to represent parallel
        generation attempts.
        """

        self.simulator.validate_runtime_consistency()

        link = self._validate_link(
            link
        )

        # =====================================================
        # Authorization gate
        # =====================================================

        self.simulator.require_quantum_eligibility(
            link.node_a,
            link.node_b
        )

        # =====================================================
        # Memory preflight
        #
        # If a result could not be stored, the control plane
        # should not authorize the physical attempt.
        # Therefore this is NOT counted as a physical attempt.
        # =====================================================

        self._require_storage_capacity(
            link
        )

        # =====================================================
        # Physical model
        # =====================================================

        probability = (
            self.model.success_probability(
                link
            )
        )

        # =====================================================
        # Physical stochastic outcome
        #
        # This is exactly where the reproducible simulation RNG
        # belongs.
        # =====================================================

        random_draw = float(
            self.simulator.kernel.rng.random()
        )

        success = (
            random_draw
            < probability
        )

        # =====================================================
        # Successful physical generation creates global
        # entanglement + both local halves atomically.
        # =====================================================

        resource = None

        if success:

            resource = (
                self.simulator
                .entanglement_manager
                .create_elementary_resource(
                    link
                )
            )

        # =====================================================
        # Immutable physical-attempt record
        # =====================================================

        attempt = ElementaryEntanglementAttempt(

            attempt_index=(
                self.statistics.total_attempts
            ),

            simulation_slot=(
                self.simulator.kernel.current_slot
            ),

            link_id=(
                link.link_id
            ),

            node_a_name=(
                link.node_a.name
            ),

            node_b_name=(
                link.node_b.name
            ),

            model_name=(
                self.model.model_name
            ),

            success_probability=(
                probability
            ),

            random_draw=(
                random_draw
            ),

            success=(
                success
            ),

            resource_id=(
                resource.resource_id
                if resource is not None
                else None
            )
        )

        self.statistics.record(
            attempt
        )

        # =====================================================
        # Simulation event
        # =====================================================

        event_type = (
            "ELEMENTARY_ENTANGLEMENT_SUCCEEDED"
            if success
            else "ELEMENTARY_ENTANGLEMENT_FAILED"
        )

        self.simulator.kernel.emit(

            event_type=event_type,

            component=(
                "ElementaryEntanglementService"
            ),

            node_names=(
                link.node_a.name,
                link.node_b.name
            ),

            link_ids=(
                link.link_id,
            ),

            resource_ids=(
                (resource.resource_id,)
                if resource is not None
                else ()
            ),

            message=(
                f"Elementary entanglement attempt "
                f"{attempt.attempt_index} on "
                f"'{link.link_id}' "
                f"{'succeeded' if success else 'failed'}."
            ),

            metadata={
                "model":
                    self.model.model_name,

                "success_probability":
                    probability,

                "random_draw":
                    random_draw,

                "attempt_index":
                    attempt.attempt_index
            }
        )

        return attempt

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "ElementaryEntanglementService("
            f"model='{self.model.model_name}', "
            f"attempts="
            f"{self.statistics.total_attempts}"
            ")"
        )

In [ ]:
# @title Elementary Entanglement Verification


print("=" * 76)
print("NARAD-QC - Elementary Entanglement Verification")
print("=" * 76)


# =============================================================
# Physical test topology
#
# Q-A -------- 50 km -------- Q-B
#
# 0.2 dB/km:
#
# total attenuation = 10 dB
#
# P(success) = 10^(-10/10) = 0.1
# =============================================================

elementary_test_topology = NetworkTopology(
    name="Phase3B-Test-Network"
)


elementary_node_a = NetworkNode(
    name="Elementary-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


elementary_node_b = NetworkNode(
    name="Elementary-B",
    x_km=50,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


elementary_test_topology.add_node(
    elementary_node_a
)

elementary_test_topology.add_node(
    elementary_node_b
)


elementary_test_link = PhysicalQuantumLink(

    node_a=elementary_node_a,

    node_b=elementary_node_b,

    attenuation_db_per_km=0.2
)


elementary_test_topology.add_link(
    elementary_test_link
)


# =============================================================
# Assemble simulator
# =============================================================

elementary_test_simulator = (
    QuantumNetworkSimulator(

        topology=(
            elementary_test_topology
        ),

        simulation_seed=(
            314159
        ),

        controller_id=(
            "PHASE3B1-CONTROLLER"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=None
    )
)


# =============================================================
# Physical model
# =============================================================

elementary_test_model = (
    FiberSurvivalProxyModel()
)


elementary_service = (
    ElementaryEntanglementService(

        simulator=(
            elementary_test_simulator
        ),

        model=(
            elementary_test_model
        )
    )
)


# =============================================================
# Test 1 — Physical model preserves attenuation baseline
# =============================================================

expected_probability = 0.1

actual_probability = (
    elementary_test_model
    .success_probability(
        elementary_test_link
    )
)

assert math.isclose(
    actual_probability,
    expected_probability,
    rel_tol=1e-12
)

print(
    "[PASS] FiberSurvivalProxyModel preserves the "
    "validated Phase 1–2 attenuation probability."
)


# =============================================================
# Test 2 — Enrollment alone cannot generate entanglement
# =============================================================

assert (
    elementary_test_simulator
    .control_plane.is_enrolled(
        elementary_node_a
    )
)

assert not (
    elementary_test_simulator
    .is_quantum_eligible(
        elementary_node_a
    )
)


attempt_count_before = (
    elementary_service
    .statistics.total_attempts
)


unauthorized_blocked = False

try:

    elementary_service.attempt_once(
        elementary_test_link
    )

except QuantumOperationAuthorizationError:

    unauthorized_blocked = True


assert unauthorized_blocked

assert (
    elementary_service
    .statistics.total_attempts
    == attempt_count_before
)

print(
    "[PASS] Unauthenticated devices cannot trigger "
    "a physical entanglement attempt."
)


# =============================================================
# Test 3 — Mutually authenticate both physical devices
# =============================================================

auth_results = (
    elementary_test_simulator
    .authenticate_all_nodes()
)

assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in auth_results.values()
)

assert (
    elementary_test_simulator
    .require_quantum_eligibility(
        elementary_node_a,
        elementary_node_b
    )
)

print(
    "[PASS] Both link endpoints completed mutual "
    "ML-DSA authentication."
)


# =============================================================
# Test 4 — One physical attempt does NOT advance time
# =============================================================

slot_before_attempt = (
    elementary_test_simulator
    .kernel.current_slot
)

first_attempt = (
    elementary_service.attempt_once(
        elementary_test_link
    )
)

slot_after_attempt = (
    elementary_test_simulator
    .kernel.current_slot
)

assert (
    slot_before_attempt
    == slot_after_attempt
)

assert (
    first_attempt.simulation_slot
    == slot_before_attempt
)

assert (
    elementary_service
    .statistics.total_attempts
    == 1
)

print(
    "[PASS] Individual link attempt does not advance "
    "the global simulation clock."
)


# =============================================================
# Test 5 — Stochastic decision matches recorded draw
# =============================================================

assert (
    first_attempt.success
    ==
    (
        first_attempt.random_draw
        <
        first_attempt.success_probability
    )
)

print(
    "[PASS] Attempt success is determined exactly by "
    "the seeded physical RNG and model probability."
)


# =============================================================
# Test 6 — Resource exists iff physical attempt succeeded
# =============================================================

if first_attempt.success:

    first_resource = (
        elementary_test_simulator
        .entanglement_manager
        .get_resource(
            first_attempt.resource_id
        )
    )

    assert (
        first_resource.kind
        == EntanglementKind.ELEMENTARY
    )

    assert (
        first_resource.status
        == EntanglementStatus.ACTIVE
    )

    assert (
        elementary_test_simulator
        .quantum_memory(
            elementary_node_a
        )
        .contains(
            first_attempt.resource_id
        )
    )

    assert (
        elementary_test_simulator
        .quantum_memory(
            elementary_node_b
        )
        .contains(
            first_attempt.resource_id
        )
    )

else:

    assert (
        first_attempt.resource_id
        is None
    )

    assert (
        len(
            elementary_test_simulator
            .entanglement_manager
            .resources
        )
        == 0
    )

print(
    "[PASS] Global entanglement resource is created "
    "if and only if physical generation succeeds."
)


# =============================================================
# Test 7 — Physical link owns no runtime attempt counters
# =============================================================

assert not hasattr(
    elementary_test_link,
    "attempts"
)

assert not hasattr(
    elementary_test_link,
    "bell_pair_attempts"
)

assert not hasattr(
    elementary_test_link,
    "successes"
)

assert not hasattr(
    elementary_test_link,
    "failures"
)

print(
    "[PASS] Runtime attempt statistics remain outside "
    "the immutable physical link."
)


# =============================================================
# Test 8 — Attempt statistics are internally consistent
# =============================================================

assert (
    elementary_service
    .statistics.total_successes
    +
    elementary_service
    .statistics.total_failures
    ==
    elementary_service
    .statistics.total_attempts
)

print(
    "[PASS] Elementary-entanglement statistics are "
    "internally consistent."
)


# =============================================================
# Test 9 — Physical/global runtime remains consistent
# =============================================================

assert (
    elementary_test_simulator
    .validate_runtime_consistency()
    is True
)

print(
    "[PASS] Simulator remains cross-layer consistent "
    "after the physical attempt."
)


# =============================================================
# Inspection
# =============================================================

print("\nPHYSICAL MODEL")
print(
    elementary_test_model
)


print("\nELEMENTARY SERVICE")
print(
    elementary_service
)


print("\nATTEMPT HISTORY")

display(
    elementary_service
    .statistics
    .attempts_dataframe()
)


print("\nLINK STATISTICS")

display(
    elementary_service
    .statistics
    .link_summary_dataframe()
)


print("\nGLOBAL ENTANGLEMENT RESOURCES")

display(
    elementary_test_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\nNODE A MEMORY")

display(
    elementary_test_simulator
    .quantum_memory(
        elementary_node_a
    )
    .slots_dataframe()
)


print("\nNODE B MEMORY")

display(
    elementary_test_simulator
    .quantum_memory(
        elementary_node_b
    )
    .slots_dataframe()
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

In [ ]:
# @title Elementary Link Generation Process State

from dataclasses import dataclass
from enum import Enum
from typing import Optional
import numbers


# =============================================================
# Process Errors
# =============================================================

class ElementaryGenerationProcessError(
    ElementaryEntanglementError
):
    """
    Base exception for elementary-link generation processes.
    """
    pass


class ElementaryGenerationLimitError(
    ElementaryGenerationProcessError
):
    """
    Raised when a requested generation run reaches its
    configured simulation-slot limit before becoming ready.
    """
    pass


class ElementaryGenerationStateError(
    ElementaryGenerationProcessError
):
    """
    Raised when a requested lifecycle operation is invalid
    for the current elementary-link process state.
    """
    pass


# =============================================================
# Elementary-Link Lifecycle State
# =============================================================

class ElementaryLinkProcessState(str, Enum):

    INACTIVE = "INACTIVE"

    # A resource is required but does not currently exist.
    SEEKING = "SEEKING"

    # One ACTIVE elementary resource exists and is waiting
    # in endpoint quantum memories.
    READY = "READY"


class ElementaryLinkStepAction(str, Enum):

    INACTIVE = "INACTIVE"

    ATTEMPT_FAILED = "ATTEMPT_FAILED"

    RESOURCE_CREATED = "RESOURCE_CREATED"

    RESOURCE_WAITING = "RESOURCE_WAITING"

    RESOURCE_EXPIRED = "RESOURCE_EXPIRED"

    RESOURCE_EXPIRED_AND_REGENERATED = (
        "RESOURCE_EXPIRED_AND_REGENERATED"
    )

    EXTERNAL_TERMINATION_DETECTED = (
        "EXTERNAL_TERMINATION_DETECTED"
    )


# =============================================================
# Immutable Per-Slot Process Result
# =============================================================

@dataclass(frozen=True)
class ElementaryLinkProcessStep:
    """
    Immutable record describing what one elementary-link
    generation process did during one global simulation slot.

    This is a scheduler-level record.

    If a physical generation attempt occurred, `attempt`
    contains the corresponding lower-layer
    ElementaryEntanglementAttempt.
    """

    simulation_slot: int

    link_id: str

    state_before: ElementaryLinkProcessState
    state_after: ElementaryLinkProcessState

    action: ElementaryLinkStepAction

    attempt: Optional[
        ElementaryEntanglementAttempt
    ] = None

    active_resource_id: Optional[str] = None

    terminated_resource_id: Optional[str] = None

    terminated_resource_status: Optional[
        EntanglementStatus
    ] = None

    def __post_init__(self):

        if (
            isinstance(self.simulation_slot, bool)
            or not isinstance(
                self.simulation_slot,
                numbers.Integral
            )
            or self.simulation_slot < 0
        ):
            raise ValueError(
                "simulation_slot must be a non-negative integer."
            )

In [ ]:
# @title  Elementary Link Generation Process

class ElementaryLinkGenerationProcess:
    """
    Stateful elementary-entanglement generation process for
    ONE physical link.

    Lifecycle
    ---------
    SEEKING:
        One physical generation attempt may occur per global slot.

    READY:
        One valid elementary entanglement resource exists.
        No further generation attempts occur while it remains ACTIVE.

    EXPIRATION:
        If the stored resource exceeds its quantum-memory lifetime,
        EntanglementManager expires it globally and this process
        becomes SEEKING again.

    INACTIVE:
        Higher-level scheduling does not currently request an
        elementary resource from this link.

    Future swapping
    ---------------
    A later swapping-tree scheduler will control demand when an
    elementary resource is consumed by a BSM.

    This class deliberately does NOT perform entanglement swapping.

    Important timing rule
    ---------------------
    process_current_slot() NEVER advances SimulationKernel time.

    This allows many independent elementary links to be processed
    during the same global simulation slot.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        service: ElementaryEntanglementService,
        link: PhysicalQuantumLink,
        demand_active: bool = True
    ):

        # =====================================================
        # Simulator / Service Validation
        # =====================================================

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            service,
            ElementaryEntanglementService
        ):
            raise TypeError(
                "service must be an "
                "ElementaryEntanglementService."
            )

        if service.simulator is not simulator:
            raise ValueError(
                "ElementaryEntanglementService must belong "
                "to the same QuantumNetworkSimulator."
            )

        if not isinstance(
            link,
            PhysicalQuantumLink
        ):
            raise TypeError(
                "link must be a PhysicalQuantumLink."
            )

        canonical_link = (
            simulator.topology
            .get_link_by_id(
                link.link_id
            )
        )

        if canonical_link is not link:
            raise TopologyConsistencyError(
                f"Link '{link.link_id}' is not the canonical "
                f"physical link owned by this simulator."
            )

        if not isinstance(
            demand_active,
            bool
        ):
            raise TypeError(
                "demand_active must be bool."
            )

        self.simulator = simulator
        self.service = service
        self.link = link

        self._demand_active = (
            demand_active
        )

        self._active_resource_id: Optional[
            str
        ] = None

        self._step_history: list[
            ElementaryLinkProcessStep
        ] = []

    # =========================================================
    # Demand State
    # =========================================================

    @property
    def demand_active(
        self
    ) -> bool:

        return self._demand_active

    def activate_demand(
        self
    ) -> None:
        """
        Request one usable elementary entanglement resource.

        Future swapping-tree execution will use this mechanism
        to activate/deactivate elementary leaves as needed.
        """

        self._demand_active = True

    def deactivate_demand(
        self
    ) -> None:
        """
        Stop requesting NEW elementary-entanglement generation.

        Existing ACTIVE entanglement is not destroyed merely by
        deactivating demand.

        Resource destruction/consumption belongs to the
        EntanglementManager or future swapping engine.
        """

        self._demand_active = False

    # =========================================================
    # Resource Inspection
    # =========================================================

    @property
    def active_resource_id(
        self
    ) -> Optional[str]:

        if self._active_resource_id is None:
            return None

        resource = (
            self.simulator
            .entanglement_manager
            .get_resource(
                self._active_resource_id
            )
        )

        if resource.is_active:
            return resource.resource_id

        return None

    @property
    def active_resource(
        self
    ) -> Optional[
        EntanglementResource
    ]:

        resource_id = (
            self.active_resource_id
        )

        if resource_id is None:
            return None

        return (
            self.simulator
            .entanglement_manager
            .get_resource(
                resource_id
            )
        )

    # =========================================================
    # Dynamic Process State
    # =========================================================

    @property
    def state(
        self
    ) -> ElementaryLinkProcessState:

        if not self.demand_active:
            return (
                ElementaryLinkProcessState.INACTIVE
            )

        if self.active_resource is not None:
            return (
                ElementaryLinkProcessState.READY
            )

        return (
            ElementaryLinkProcessState.SEEKING
        )

    # =========================================================
    # One Global-Slot Processing Step
    # =========================================================

    def process_current_slot(
        self
    ) -> ElementaryLinkProcessStep:
        """
        Process this elementary link exactly once during the
        CURRENT global simulation slot.

        Order
        -----
        1. Synchronize memory expiration.
        2. Reconcile any previously tracked resource.
        3. If still ACTIVE -> wait, no physical attempt.
        4. If no resource is required -> remain inactive.
        5. If unresolved -> perform exactly one physical attempt.

        This method NEVER advances simulation time.
        """

        self.simulator.validate_runtime_consistency()

        current_slot = (
            self.simulator.kernel.current_slot
        )

        state_before = self.state

        previous_resource_id = (
            self._active_resource_id
        )

        terminated_resource_id = None
        terminated_resource_status = None

        # =====================================================
        # STEP 1 — Synchronize hard memory expiration
        #
        # Existing resources are checked at the START of the
        # global slot.
        #
        # A resource created during this same slot therefore has
        # age 0 and cannot immediately expire.
        # =====================================================

        self.simulator.entanglement_manager\
            .synchronize_expirations()

        # =====================================================
        # STEP 2 — Reconcile tracked resource lifecycle
        # =====================================================

        if previous_resource_id is not None:

            previous_resource = (
                self.simulator
                .entanglement_manager
                .get_resource(
                    previous_resource_id
                )
            )

            if not previous_resource.is_active:

                terminated_resource_id = (
                    previous_resource.resource_id
                )

                terminated_resource_status = (
                    previous_resource.status
                )

                self._active_resource_id = None

                # -------------------------------------------------
                # Future special case:
                #
                # CONSUMED will be created by swapping/other
                # higher-layer operations.
                #
                # We deliberately do NOT immediately regenerate
                # in the SAME slot after observing CONSUMED.
                #
                # The future swapping-tree scheduler must decide
                # whether this leaf should be requested again.
                # -------------------------------------------------

                if (
                    previous_resource.status
                    == EntanglementStatus.CONSUMED
                ):

                    step = ElementaryLinkProcessStep(

                        simulation_slot=current_slot,

                        link_id=self.link.link_id,

                        state_before=state_before,

                        state_after=self.state,

                        action=(
                            ElementaryLinkStepAction
                            .EXTERNAL_TERMINATION_DETECTED
                        ),

                        attempt=None,

                        active_resource_id=None,

                        terminated_resource_id=(
                            terminated_resource_id
                        ),

                        terminated_resource_status=(
                            terminated_resource_status
                        )
                    )

                    self._step_history.append(
                        step
                    )

                    return step

        # =====================================================
        # STEP 3 — A valid resource already exists
        #
        # Waiting protocol:
        #
        # DO NOT generate another elementary pair.
        # =====================================================

        if self.active_resource is not None:

            step = ElementaryLinkProcessStep(

                simulation_slot=current_slot,

                link_id=self.link.link_id,

                state_before=state_before,

                state_after=(
                    ElementaryLinkProcessState.READY
                ),

                action=(
                    ElementaryLinkStepAction
                    .RESOURCE_WAITING
                ),

                attempt=None,

                active_resource_id=(
                    self.active_resource.resource_id
                ),

                terminated_resource_id=None,

                terminated_resource_status=None
            )

            self._step_history.append(
                step
            )

            return step

        # =====================================================
        # STEP 4 — Higher layer currently requests nothing
        # =====================================================

        if not self.demand_active:

            step = ElementaryLinkProcessStep(

                simulation_slot=current_slot,

                link_id=self.link.link_id,

                state_before=state_before,

                state_after=(
                    ElementaryLinkProcessState.INACTIVE
                ),

                action=(
                    ElementaryLinkStepAction.INACTIVE
                )
            )

            self._step_history.append(
                step
            )

            return step

        # =====================================================
        # STEP 5 — SEEKING: exactly one physical attempt
        # =====================================================

        attempt = (
            self.service.attempt_once(
                self.link
            )
        )

        if attempt.success:

            self._active_resource_id = (
                attempt.resource_id
            )

            if (
                terminated_resource_status
                == EntanglementStatus.EXPIRED
            ):

                action = (
                    ElementaryLinkStepAction
                    .RESOURCE_EXPIRED_AND_REGENERATED
                )

            else:

                action = (
                    ElementaryLinkStepAction
                    .RESOURCE_CREATED
                )

        else:

            if (
                terminated_resource_status
                == EntanglementStatus.EXPIRED
            ):

                action = (
                    ElementaryLinkStepAction
                    .RESOURCE_EXPIRED
                )

            else:

                action = (
                    ElementaryLinkStepAction
                    .ATTEMPT_FAILED
                )

        step = ElementaryLinkProcessStep(

            simulation_slot=current_slot,

            link_id=self.link.link_id,

            state_before=state_before,

            state_after=self.state,

            action=action,

            attempt=attempt,

            active_resource_id=(
                self.active_resource_id
            ),

            terminated_resource_id=(
                terminated_resource_id
            ),

            terminated_resource_status=(
                terminated_resource_status
            )
        )

        self._step_history.append(
            step
        )

        return step

    # =========================================================
    # Convenience Driver — Repeated Attempts Until Ready
    # =========================================================

    def run_until_ready(
        self,
        max_slots: int
    ) -> EntanglementResource:
        """
        Repeatedly process one global slot and advance time until
        one ACTIVE elementary resource becomes available.

        This helper is valid for a SINGLE-LINK experiment.

        Phase 3B.3 will replace this sequential driver with a
        multi-link scheduler that processes many links during
        every global slot before advancing the clock once.
        """

        if (
            isinstance(max_slots, bool)
            or not isinstance(
                max_slots,
                numbers.Integral
            )
            or max_slots <= 0
        ):
            raise ValueError(
                "max_slots must be a positive integer."
            )

        if not self.demand_active:
            raise ElementaryGenerationStateError(
                "Cannot run generation while demand is inactive."
            )

        max_slots = int(
            max_slots
        )

        for _ in range(
            max_slots
        ):

            step = self.process_current_slot()

            if (
                step.state_after
                == ElementaryLinkProcessState.READY
            ):

                resource = self.active_resource

                if resource is None:
                    raise RuntimeError(
                        "Process entered READY state without "
                        "an ACTIVE resource."
                    )

                return resource

            # One complete global slot has now elapsed.
            self.simulator.kernel.advance_slot()

        raise ElementaryGenerationLimitError(
            f"Link '{self.link.link_id}' did not produce "
            f"a ready elementary resource within "
            f"{max_slots} simulation slots."
        )

    # =========================================================
    # History / Statistics
    # =========================================================

    @property
    def step_history(
        self
    ) -> tuple[
        ElementaryLinkProcessStep, ...
    ]:

        return tuple(
            self._step_history
        )

    def step_history_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for step_index, step in enumerate(
            self._step_history
        ):

            rows.append({

                "Step":
                    step_index,

                "Simulation Slot":
                    step.simulation_slot,

                "Link ID":
                    step.link_id,

                "State Before":
                    step.state_before.value,

                "Action":
                    step.action.value,

                "State After":
                    step.state_after.value,

                "Physical Attempt":
                    (
                        step.attempt.attempt_index
                        if step.attempt is not None
                        else None
                    ),

                "Attempt Success":
                    (
                        step.attempt.success
                        if step.attempt is not None
                        else None
                    ),

                "Active Resource":
                    step.active_resource_id,

                "Terminated Resource":
                    step.terminated_resource_id,

                "Termination Status":
                    (
                        step.terminated_resource_status.value
                        if (
                            step.terminated_resource_status
                            is not None
                        )
                        else None
                    )
            })

        return pd.DataFrame(
            rows
        )

    def __repr__(
        self
    ) -> str:

        return (
            "ElementaryLinkGenerationProcess("
            f"link='{self.link.link_id}', "
            f"state='{self.state.value}', "
            f"resource={self.active_resource_id}"
            ")"
        )

In [ ]:
# @title Repeated Generation and Waiting Verification

print("=" * 76)
print("NARAD-QC — Elementary Link Lifecycle Verification")
print("=" * 76)


# =============================================================
# Controlled physical network
#
# Gen-A ---------------- 50 km ---------------- Gen-B
#
# alpha = 0.2 dB/km
#
# P(success) = 0.1 per physical attempt
#
# Memory lifetime = 3 global simulation slots
# =============================================================

generation_topology = NetworkTopology(
    name="Phase3B2-Test-Network"
)


generation_node_a = NetworkNode(
    name="Generation-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


generation_node_b = NetworkNode(
    name="Generation-B",
    x_km=50,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


generation_topology.add_node(
    generation_node_a
)

generation_topology.add_node(
    generation_node_b
)


generation_link = PhysicalQuantumLink(
    node_a=generation_node_a,
    node_b=generation_node_b,
    attenuation_db_per_km=0.2
)


generation_topology.add_link(
    generation_link
)


# =============================================================
# Fresh simulation
# =============================================================

generation_simulator = QuantumNetworkSimulator(

    topology=generation_topology,

    simulation_seed=20260302,

    controller_id="PHASE3B2-CONTROLLER",

    parameter_set=(
        MLDSAParameterSet.ML_DSA_65
    ),

    max_storage_slots=3
)


# =============================================================
# Authenticate both physical devices first
# =============================================================

generation_simulator.authenticate_all_nodes()

assert generation_simulator.require_quantum_eligibility(
    generation_node_a,
    generation_node_b
)

print(
    "[PASS] Both physical devices are mutually authenticated "
    "before elementary generation."
)


# =============================================================
# Physical service + lifecycle process
# =============================================================

generation_model = (
    FiberSurvivalProxyModel()
)


generation_service = (
    ElementaryEntanglementService(
        simulator=generation_simulator,
        model=generation_model
    )
)


generation_process = (
    ElementaryLinkGenerationProcess(
        simulator=generation_simulator,
        service=generation_service,
        link=generation_link,
        demand_active=True
    )
)


# =============================================================
# Test 1 — Initial state is SEEKING
# =============================================================

assert (
    generation_process.state
    == ElementaryLinkProcessState.SEEKING
)

assert (
    generation_process.active_resource
    is None
)

print(
    "[PASS] Elementary link begins unresolved in SEEKING state."
)


# =============================================================
# Test 2 — Repeated one-attempt-per-slot generation
# =============================================================

generation_start_slot = (
    generation_simulator.kernel.current_slot
)


first_ready_resource = (
    generation_process.run_until_ready(
        max_slots=1000
    )
)


assert first_ready_resource.is_active

assert (
    first_ready_resource.kind
    == EntanglementKind.ELEMENTARY
)

assert (
    generation_process.state
    == ElementaryLinkProcessState.READY
)

assert (
    generation_service.statistics.total_attempts
    >= 1
)

print(
    "[PASS] Repeated physical attempts continue until "
    "an elementary resource becomes ready."
)


# =============================================================
# Test 3 — Attempt count matches actual processed failures
#          plus final success
# =============================================================

attempts_to_first_success = (
    generation_service
    .statistics.total_attempts
)

assert (
    generation_service
    .statistics.total_successes
    == 1
)

assert (
    generation_service
    .statistics.total_failures
    == attempts_to_first_success - 1
)

print(
    "[PASS] Physical attempt statistics correctly capture "
    "repeated failures followed by success."
)


# =============================================================
# Test 4 — Resource exists in both endpoint memories
# =============================================================

first_resource_id = (
    first_ready_resource.resource_id
)


assert (
    generation_simulator
    .quantum_memory(
        generation_node_a
    )
    .contains(
        first_resource_id
    )
)

assert (
    generation_simulator
    .quantum_memory(
        generation_node_b
    )
    .contains(
        first_resource_id
    )
)

print(
    "[PASS] Successful elementary resource is stored at "
    "both physical endpoints."
)


# =============================================================
# Test 5 — READY link does NOT generate another Bell pair
# =============================================================

attempt_count_before_wait = (
    generation_service
    .statistics.total_attempts
)


wait_step_same_slot = (
    generation_process
    .process_current_slot()
)


assert (
    wait_step_same_slot.action
    == ElementaryLinkStepAction.RESOURCE_WAITING
)

assert (
    generation_service
    .statistics.total_attempts
    == attempt_count_before_wait
)

print(
    "[PASS] READY elementary link performs no redundant "
    "generation attempt while its EP waits."
)


# =============================================================
# Test 6 — Waiting resource ages through global slots
# =============================================================

creation_slot = (
    first_ready_resource.created_slot
)

assert (
    generation_simulator
    .quantum_memory(
        generation_node_a
    )
    .age_slots(
        first_resource_id
    )
    == 0
)


generation_simulator.kernel.advance_slot()

wait_step_age_1 = (
    generation_process
    .process_current_slot()
)

assert (
    wait_step_age_1.action
    == ElementaryLinkStepAction.RESOURCE_WAITING
)

assert (
    generation_simulator
    .quantum_memory(
        generation_node_a
    )
    .age_slots(
        first_resource_id
    )
    == 1
)


generation_simulator.kernel.advance_slot()

wait_step_age_2 = (
    generation_process
    .process_current_slot()
)

assert (
    wait_step_age_2.action
    == ElementaryLinkStepAction.RESOURCE_WAITING
)

assert (
    generation_simulator
    .quantum_memory(
        generation_node_a
    )
    .age_slots(
        first_resource_id
    )
    == 2
)

assert (
    generation_service
    .statistics.total_attempts
    == attempt_count_before_wait
)

print(
    "[PASS] Waiting EP ages while generation remains paused."
)


# =============================================================
# Test 7 — At age == 3 the old resource expires
# =============================================================

generation_simulator.kernel.advance_slot()

assert (
    generation_simulator.kernel.current_slot
    == creation_slot + 3
)


attempt_count_before_expiry_step = (
    generation_service
    .statistics.total_attempts
)


expiry_step = (
    generation_process
    .process_current_slot()
)


expired_resource = (
    generation_simulator
    .entanglement_manager
    .get_resource(
        first_resource_id
    )
)


assert (
    expired_resource.status
    == EntanglementStatus.EXPIRED
)

assert not (
    generation_simulator
    .quantum_memory(
        generation_node_a
    )
    .contains(
        first_resource_id
    )
)

assert not (
    generation_simulator
    .quantum_memory(
        generation_node_b
    )
    .contains(
        first_resource_id
    )
)


# Because demand remains active, exactly one fresh physical
# generation attempt is permitted in the same slot after expiry.
assert (
    generation_service
    .statistics.total_attempts
    == attempt_count_before_expiry_step + 1
)

assert (
    expiry_step.terminated_resource_id
    == first_resource_id
)

assert (
    expiry_step.terminated_resource_status
    == EntanglementStatus.EXPIRED
)

print(
    "[PASS] Expired waiting EP is removed globally and "
    "the elementary link immediately resumes generation."
)


# =============================================================
# Test 8 — After expiry, process is either:
#
# SEEKING : replacement attempt failed
#
# or
#
# READY   : replacement attempt succeeded immediately
# =============================================================

assert generation_process.state in (
    ElementaryLinkProcessState.SEEKING,
    ElementaryLinkProcessState.READY
)

print(
    "[PASS] Post-expiry lifecycle correctly returns to "
    "generation demand."
)


# =============================================================
# Test 9 — Eventually obtain a fresh replacement resource
# =============================================================

replacement_resource = (
    generation_process.run_until_ready(
        max_slots=1000
    )
)


assert replacement_resource.is_active

assert (
    replacement_resource.resource_id
    != first_resource_id
)

assert (
    replacement_resource.created_slot
    >= expired_resource.ended_slot
)

print(
    "[PASS] Expired elementary resource is eventually "
    "replaced by a fresh active resource."
)


# =============================================================
# Test 10 — Old resource remains preserved in history
# =============================================================

assert (
    generation_simulator
    .entanglement_manager
    .get_resource(
        first_resource_id
    )
    .status
    == EntanglementStatus.EXPIRED
)

assert (
    generation_simulator
    .entanglement_manager
    .get_resource(
        replacement_resource.resource_id
    )
    .status
    == EntanglementStatus.ACTIVE
)

print(
    "[PASS] Resource history distinguishes expired and "
    "currently active elementary entanglement."
)


# =============================================================
# Test 11 — Cross-layer consistency remains valid
# =============================================================

assert (
    generation_simulator
    .validate_runtime_consistency()
    is True
)

print(
    "[PASS] Complete runtime remains cross-layer consistent "
    "after expiration and regeneration."
)


# =============================================================
# Inspection
# =============================================================

print("\nELEMENTARY LINK PROCESS")
print(
    generation_process
)


print(
    "\nFirst resource:"
)
print(
    first_ready_resource
)


print(
    "\nReplacement resource:"
)
print(
    replacement_resource
)


print("\nPROCESS STEP HISTORY")

display(
    generation_process
    .step_history_dataframe()
)


print("\nPHYSICAL ATTEMPT HISTORY")

display(
    generation_service
    .statistics
    .attempts_dataframe()
)


print("\nGLOBAL RESOURCE HISTORY")

display(
    generation_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\nNODE A MEMORY")

display(
    generation_simulator
    .quantum_memory(
        generation_node_a
    )
    .slots_dataframe()
)


print("\nNODE B MEMORY")

display(
    generation_simulator
    .quantum_memory(
        generation_node_b
    )
    .slots_dataframe()
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

In [ ]:
# @title Parallel Generation Slot Record

from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class ParallelElementaryGenerationSlot:
    """
    Immutable scheduler-level record of one global simulation
    slot across an ordered physical path.

    All contained ElementaryLinkProcessStep objects must belong
    to the same simulation slot.

    The record represents logical concurrency:
    individual Python calls may execute sequentially, but all
    represented physical operations occur within the same
    simulation slot.
    """

    simulation_slot: int

    steps: tuple[
        ElementaryLinkProcessStep, ...
    ]

    all_links_ready: bool

    ready_link_ids: tuple[str, ...]

    unresolved_link_ids: tuple[str, ...]

    active_resource_ids: tuple[str, ...]

    def __post_init__(self):

        if (
            isinstance(self.simulation_slot, bool)
            or not isinstance(
                self.simulation_slot,
                int
            )
            or self.simulation_slot < 0
        ):
            raise ValueError(
                "simulation_slot must be a non-negative integer."
            )

        if not isinstance(
            self.steps,
            tuple
        ):
            raise TypeError(
                "steps must be a tuple."
            )

        step_link_ids = []

        for step in self.steps:

            if not isinstance(
                step,
                ElementaryLinkProcessStep
            ):
                raise TypeError(
                    "Every slot entry must be an "
                    "ElementaryLinkProcessStep."
                )

            if (
                step.simulation_slot
                != self.simulation_slot
            ):
                raise ValueError(
                    "All parallel link steps must belong "
                    "to the same global simulation slot."
                )

            step_link_ids.append(
                step.link_id
            )

        if (
            len(step_link_ids)
            != len(set(step_link_ids))
        ):
            raise ValueError(
                "A physical link may be processed at most "
                "once per parallel simulation slot."
            )

    # =========================================================
    # Convenience Metrics
    # =========================================================

    @property
    def physical_attempt_count(
        self
    ) -> int:

        return sum(
            step.attempt is not None
            for step in self.steps
        )

    @property
    def successful_generation_count(
        self
    ) -> int:

        return sum(
            (
                step.attempt is not None
                and step.attempt.success
            )
            for step in self.steps
        )

    @property
    def failed_generation_count(
        self
    ) -> int:

        return sum(
            (
                step.attempt is not None
                and not step.attempt.success
            )
            for step in self.steps
        )

    @property
    def waiting_link_count(
        self
    ) -> int:

        return sum(
            step.action
            == ElementaryLinkStepAction.RESOURCE_WAITING
            for step in self.steps
        )

In [ ]:
# @title Parallel Elementary Generation Scheduler


class ParallelElementaryGenerationScheduler:
    """
    Coordinate elementary-entanglement generation across every
    consecutive physical link of one ordered quantum-network path.

    Core slot semantics
    -------------------
    During one global slot:

        1. Synchronize all memory expirations globally.
        2. Process every elementary-link lifecycle exactly once.
        3. Each SEEKING link may perform at most one attempt.
        4. Each READY link waits without regenerating.
        5. No link advances SimulationKernel time.
        6. The scheduler returns the complete slot result.
        7. A higher-level driver advances the global clock once
           only if another slot is required.

    This implements logical parallelism rather than Python-thread
    concurrency.

    Elementary-resource rule
    ------------------------
    Every managed process corresponds to exactly one authoritative
    PhysicalQuantumLink connecting consecutive path nodes.

    No non-adjacent elementary resource may be generated.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        service: ElementaryEntanglementService,
        ordered_path,
        demand_active: bool = True
    ):

        # =====================================================
        # Core Object Validation
        # =====================================================

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            service,
            ElementaryEntanglementService
        ):
            raise TypeError(
                "service must be an "
                "ElementaryEntanglementService."
            )

        if service.simulator is not simulator:
            raise ValueError(
                "ElementaryEntanglementService must belong "
                "to the supplied QuantumNetworkSimulator."
            )

        if not isinstance(
            demand_active,
            bool
        ):
            raise TypeError(
                "demand_active must be bool."
            )

        self.simulator = simulator
        self.service = service

        # =====================================================
        # Resolve authoritative ordered physical path
        # =====================================================

        self.ordered_path = (
            simulator.topology
            .validate_path(
                ordered_path
            )
        )

        self.path_links = (
            simulator.topology
            .path_links(
                self.ordered_path
            )
        )

        if len(self.path_links) < 1:
            raise ValueError(
                "Parallel elementary generation requires "
                "at least one physical link."
            )

        # =====================================================
        # One lifecycle process per ADJACENT physical link
        # =====================================================

        self._processes: dict[
            str,
            ElementaryLinkGenerationProcess
        ] = {}

        for link in self.path_links:

            process = (
                ElementaryLinkGenerationProcess(
                    simulator=self.simulator,
                    service=self.service,
                    link=link,
                    demand_active=demand_active
                )
            )

            self._processes[
                link.link_id
            ] = process

        self._slot_history: list[
            ParallelElementaryGenerationSlot
        ] = []

    # =========================================================
    # Process Access
    # =========================================================

    @property
    def processes(
        self
    ) -> tuple[
        ElementaryLinkGenerationProcess, ...
    ]:
        """
        Processes in physical path order.
        """

        return tuple(
            self._processes[
                link.link_id
            ]
            for link in self.path_links
        )

    def process_for_link(
        self,
        link_or_id
    ) -> ElementaryLinkGenerationProcess:

        if isinstance(
            link_or_id,
            PhysicalQuantumLink
        ):
            link_id = (
                link_or_id.link_id
            )

        elif (
            isinstance(link_or_id, str)
            and link_or_id.strip()
        ):
            link_id = (
                link_or_id.strip()
            )

        else:
            raise TypeError(
                "Expected PhysicalQuantumLink or link ID."
            )

        if link_id not in self._processes:
            raise TopologyLinkNotFoundError(
                f"Link '{link_id}' is not managed by "
                f"this parallel scheduler."
            )

        return self._processes[
            link_id
        ]

    # =========================================================
    # Readiness
    # =========================================================

    @property
    def ready_link_ids(
        self
    ) -> tuple[str, ...]:

        return tuple(
            process.link.link_id
            for process in self.processes
            if (
                process.state
                == ElementaryLinkProcessState.READY
            )
        )

    @property
    def unresolved_link_ids(
        self
    ) -> tuple[str, ...]:

        return tuple(
            process.link.link_id
            for process in self.processes
            if (
                process.state
                != ElementaryLinkProcessState.READY
            )
        )

    @property
    def all_links_ready(
        self
    ) -> bool:

        return all(
            process.state
            == ElementaryLinkProcessState.READY
            for process in self.processes
        )

    # =========================================================
    # Ready Elementary Resources
    # =========================================================

    def ready_resources(
        self
    ) -> tuple[
        EntanglementResource, ...
    ]:
        """
        Return ACTIVE elementary resources in physical path order.

        Raises if the complete elementary path is not ready.
        """

        if not self.all_links_ready:
            raise ElementaryGenerationStateError(
                "Not every physical link currently has a "
                "ready elementary entanglement resource."
            )

        resources = []

        for process in self.processes:

            resource = (
                process.active_resource
            )

            if resource is None:
                raise RuntimeError(
                    "READY process has no ACTIVE resource."
                )

            # -------------------------------------------------
            # HARD elementary-entanglement adjacency invariant
            # -------------------------------------------------

            if (
                resource.kind
                != EntanglementKind.ELEMENTARY
            ):
                raise EntanglementConsistencyError(
                    f"Resource '{resource.resource_id}' "
                    f"managed as an elementary leaf is not "
                    f"an ELEMENTARY resource."
                )

            expected_endpoints = tuple(
                sorted(
                    (
                        process.link.node_a.name,
                        process.link.node_b.name
                    )
                )
            )

            if (
                resource.endpoint_names
                != expected_endpoints
            ):
                raise EntanglementConsistencyError(
                    f"Elementary resource "
                    f"'{resource.resource_id}' does not "
                    f"connect the endpoints of its physical link."
                )

            if (
                resource.source_link_ids
                != (
                    process.link.link_id,
                )
            ):
                raise EntanglementConsistencyError(
                    f"Elementary resource "
                    f"'{resource.resource_id}' has incorrect "
                    f"physical-link provenance."
                )

            resources.append(
                resource
            )

        return tuple(
            resources
        )

    # =========================================================
    # One Parallel Global Slot
    # =========================================================

    def process_current_slot(
        self
    ) -> ParallelElementaryGenerationSlot:
        """
        Process every managed physical link during the same
        global simulation slot.

        IMPORTANT:
        This method does NOT advance SimulationKernel time.
        """

        self.simulator.validate_runtime_consistency()

        current_slot = (
            self.simulator.kernel.current_slot
        )

        # =====================================================
        # Global slot-start expiration phase
        #
        # Every expired resource is resolved before ANY fresh
        # elementary-generation operation for this slot.
        # =====================================================

        self.simulator\
            .entanglement_manager\
            .synchronize_expirations()

        # =====================================================
        # Process each elementary link exactly once
        #
        # Python execution is sequential for deterministic RNG
        # ordering, but every step receives the SAME global slot.
        # =====================================================

        steps = []

        for process in self.processes:

            step = (
                process.process_current_slot()
            )

            if (
                step.simulation_slot
                != current_slot
            ):
                raise RuntimeError(
                    "Elementary process changed the global "
                    "simulation clock during a parallel slot."
                )

            steps.append(
                step
            )

        # =====================================================
        # Global clock MUST still be unchanged
        # =====================================================

        if (
            self.simulator.kernel.current_slot
            != current_slot
        ):
            raise RuntimeError(
                "An elementary-link process advanced the "
                "global clock inside a parallel slot."
            )

        ready_link_ids = (
            self.ready_link_ids
        )

        unresolved_link_ids = (
            self.unresolved_link_ids
        )

        active_resource_ids = tuple(
            process.active_resource.resource_id
            for process in self.processes
            if process.active_resource is not None
        )

        slot_result = (
            ParallelElementaryGenerationSlot(

                simulation_slot=current_slot,

                steps=tuple(
                    steps
                ),

                all_links_ready=(
                    len(unresolved_link_ids) == 0
                ),

                ready_link_ids=(
                    ready_link_ids
                ),

                unresolved_link_ids=(
                    unresolved_link_ids
                ),

                active_resource_ids=(
                    active_resource_ids
                )
            )
        )

        self._slot_history.append(
            slot_result
        )

        # =====================================================
        # Scheduler event
        # =====================================================

        self.simulator.kernel.emit(

            event_type=(
                "PARALLEL_ELEMENTARY_SLOT_PROCESSED"
            ),

            component=(
                "ParallelElementaryGenerationScheduler"
            ),

            node_names=tuple(
                node.name
                for node in self.ordered_path
            ),

            link_ids=tuple(
                link.link_id
                for link in self.path_links
            ),

            resource_ids=(
                active_resource_ids
            ),

            message=(
                f"Processed {len(self.path_links)} "
                f"elementary links during simulation "
                f"slot {current_slot}."
            ),

            metadata={
                "physical_attempts":
                    slot_result.physical_attempt_count,

                "generation_successes":
                    slot_result.successful_generation_count,

                "generation_failures":
                    slot_result.failed_generation_count,

                "ready_links":
                    len(ready_link_ids),

                "unresolved_links":
                    len(unresolved_link_ids),

                "all_links_ready":
                    slot_result.all_links_ready
            }
        )

        return slot_result

    # =========================================================
    # Run Until Entire Elementary Path Is Ready
    # =========================================================

    def run_until_all_ready(
        self,
        max_slots: int
    ) -> tuple[
        EntanglementResource, ...
    ]:
        """
        Execute parallel elementary-link generation until every
        physical link in the ordered path simultaneously holds
        one ACTIVE elementary entanglement resource.

        Slot semantics
        --------------
        If all links become ready during slot t, this method
        returns while SimulationKernel.current_slot is still t.

        That is intentional.

        A future swapping scheduler may therefore consume those
        resources later in the SAME global slot after elementary
        generation has completed.

        If the path is not yet ready, global time advances once
        after the complete parallel slot.
        """

        if (
            isinstance(max_slots, bool)
            or not isinstance(
                max_slots,
                int
            )
            or max_slots <= 0
        ):
            raise ValueError(
                "max_slots must be a positive integer."
            )

        for _ in range(
            max_slots
        ):

            slot_result = (
                self.process_current_slot()
            )

            if slot_result.all_links_ready:

                resources = (
                    self.ready_resources()
                )

                self.simulator.kernel.emit(

                    event_type=(
                        "ELEMENTARY_PATH_READY"
                    ),

                    component=(
                        "ParallelElementaryGenerationScheduler"
                    ),

                    node_names=tuple(
                        node.name
                        for node in self.ordered_path
                    ),

                    link_ids=tuple(
                        link.link_id
                        for link in self.path_links
                    ),

                    resource_ids=tuple(
                        resource.resource_id
                        for resource in resources
                    ),

                    message=(
                        "Every physical link in the ordered "
                        "path now holds an active elementary "
                        "entanglement resource."
                    ),

                    metadata={
                        "ready_slot":
                            self.simulator.kernel.current_slot,

                        "elementary_resources":
                            len(resources)
                    }
                )

                return resources

            # -------------------------------------------------
            # The complete parallel slot has finished.
            #
            # Only NOW does global simulation time advance.
            # -------------------------------------------------

            self.simulator.kernel.advance_slot()

        raise ElementaryGenerationLimitError(
            "The complete physical path did not obtain "
            f"simultaneously ready elementary resources "
            f"within {max_slots} global simulation slots."
        )

    # =========================================================
    # History
    # =========================================================

    @property
    def slot_history(
        self
    ) -> tuple[
        ParallelElementaryGenerationSlot, ...
    ]:

        return tuple(
            self._slot_history
        )

    def slot_history_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for slot in self._slot_history:

            rows.append({

                "Simulation Slot":
                    slot.simulation_slot,

                "Physical Links":
                    len(slot.steps),

                "Physical Attempts":
                    slot.physical_attempt_count,

                "Generation Successes":
                    slot.successful_generation_count,

                "Generation Failures":
                    slot.failed_generation_count,

                "Waiting Links":
                    slot.waiting_link_count,

                "Ready Links":
                    len(
                        slot.ready_link_ids
                    ),

                "Unresolved Links":
                    len(
                        slot.unresolved_link_ids
                    ),

                "All Links Ready":
                    slot.all_links_ready,

                "Active Resources":
                    len(
                        slot.active_resource_ids
                    )
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Current Path State
    # =========================================================

    def link_state_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for (
            link,
            process
        ) in zip(
            self.path_links,
            self.processes
        ):

            resource = (
                process.active_resource
            )

            rows.append({

                "Link ID":
                    link.link_id,

                "Node A":
                    link.node_a.name,

                "Node B":
                    link.node_b.name,

                "Distance (km)":
                    link.distance_km,

                "P(success)":
                    self.service.model
                    .success_probability(
                        link
                    ),

                "Process State":
                    process.state.value,

                "Active Resource":
                    (
                        resource.resource_id
                        if resource is not None
                        else None
                    ),

                "Resource Kind":
                    (
                        resource.kind.value
                        if resource is not None
                        else None
                    )
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "ParallelElementaryGenerationScheduler("
            f"links={len(self.path_links)}, "
            f"ready={len(self.ready_link_ids)}/"
            f"{len(self.path_links)}, "
            f"slot={self.simulator.kernel.current_slot}"
            ")"
        )

In [ ]:
# @title Parallel Elementary Generation Verification


print("=" * 76)
print("NARAD-QC — Parallel Elementary Generation Verification")
print("=" * 76)


# =============================================================
# Physical topology
#
# Parallel-A --5km-- Parallel-R1 --5km-- Parallel-R2 --5km-- Parallel-B
#
# Every segment:
#
# attenuation = 1 dB
# P(success)  = 10^(-1/10) ~= 0.794328
# =============================================================

parallel_topology = NetworkTopology(
    name="Phase3B3-Parallel-Test-Network"
)


parallel_a = NetworkNode(
    name="Parallel-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


parallel_r1 = NetworkNode(
    name="Parallel-R1",
    x_km=10,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


parallel_r2 = NetworkNode(
    name="Parallel-R2",
    x_km=20,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


parallel_b = NetworkNode(
    name="Parallel-B",
    x_km=30,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


for node in (
    parallel_a,
    parallel_r1,
    parallel_r2,
    parallel_b
):

    parallel_topology.add_node(
        node
    )


parallel_path = (
    parallel_a,
    parallel_r1,
    parallel_r2,
    parallel_b
)


for left, right in zip(
    parallel_path[:-1],
    parallel_path[1:]
):

    parallel_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert parallel_topology.validate_path(
    parallel_path
)

print(
    "[PASS] Ordered repeater path contains only "
    "authoritative adjacent physical links."
)


# =============================================================
# Assemble simulation
#
# 10-slot memory lifetime is deliberately finite.
# With these short segments it is comfortably sufficient for
# this controlled parallelism verification.
# =============================================================

parallel_simulator = (
    QuantumNetworkSimulator(

        topology=parallel_topology,

        simulation_seed=12345,

        controller_id=(
            "PHASE3B3-CONTROLLER"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=10
    )
)


# =============================================================
# Test 1 — Nothing is quantum eligible before authentication
# =============================================================

assert not any(
    parallel_simulator.is_quantum_eligible(
        node
    )
    for node in parallel_path
)

print(
    "[PASS] Parallel generation begins with the "
    "authentication gate closed."
)


# =============================================================
# Authenticate every participating physical device
# =============================================================

parallel_auth_results = (
    parallel_simulator
    .authenticate_all_nodes()
)

assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in parallel_auth_results.values()
)

assert (
    parallel_simulator
    .require_quantum_eligibility(
        *parallel_path
    )
)

print(
    "[PASS] Every endpoint and repeater completed mutual "
    "ML-DSA authentication."
)


# =============================================================
# Physical generation service
# =============================================================

parallel_model = (
    FiberSurvivalProxyModel()
)


parallel_service = (
    ElementaryEntanglementService(

        simulator=parallel_simulator,

        model=parallel_model
    )
)


parallel_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=parallel_simulator,

        service=parallel_service,

        ordered_path=parallel_path,

        demand_active=True
    )
)


# =============================================================
# Test 2 — One process exists per adjacent elementary link
# =============================================================

assert (
    len(
        parallel_scheduler.processes
    )
    == len(parallel_path) - 1
)

assert tuple(
    process.link
    for process in parallel_scheduler.processes
) == parallel_topology.path_links(
    parallel_path
)

print(
    "[PASS] Scheduler creates exactly one elementary process "
    "for every adjacent physical-link segment."
)


# =============================================================
# Test 3 — No non-adjacent elementary link exists
# =============================================================

assert not (
    parallel_topology.contains_link(
        parallel_a,
        parallel_r2
    )
)

assert not (
    parallel_topology.contains_link(
        parallel_a,
        parallel_b
    )
)

assert not (
    parallel_topology.contains_link(
        parallel_r1,
        parallel_b
    )
)

print(
    "[PASS] Elementary generation topology contains no "
    "non-adjacent physical shortcuts."
)


# # =============================================================
# # Test 4 — Physical probability is correct on every segment
# # =============================================================

# expected_segment_probability = (
#     10 ** (-1.0 / 10.0)
# )


# for link in parallel_scheduler.path_links:

#     probability = (
#         parallel_model
#         .success_probability(
#             link
#         )
#     )

#     assert math.isclose(
#         probability,
#         expected_segment_probability,
#         rel_tol=1e-12
#     )

# print(
#     "[PASS] Every 5 km elementary link uses the correct "
#     "fibre-survival probability."
# )


# =============================================================
# Test 5 — Initial states are all SEEKING
# =============================================================

assert all(
    process.state
    == ElementaryLinkProcessState.SEEKING
    for process in parallel_scheduler.processes
)

print(
    "[PASS] All elementary links initially seek "
    "their first Bell pair."
)


# =============================================================
# Test 6 — Process one COMPLETE parallel slot
#
# Every unresolved link must get exactly one opportunity,
# but the global clock must remain unchanged.
# =============================================================

slot_before = (
    parallel_simulator.kernel.current_slot
)


attempts_before = (
    parallel_service
    .statistics.total_attempts
)


first_parallel_slot = (
    parallel_scheduler
    .process_current_slot()
)


slot_after = (
    parallel_simulator.kernel.current_slot
)


assert (
    slot_before
    == slot_after
)

assert (
    first_parallel_slot.simulation_slot
    == slot_before
)

assert (
    first_parallel_slot.physical_attempt_count
    == len(
        parallel_scheduler.path_links
    )
)

assert (
    parallel_service
    .statistics.total_attempts
    - attempts_before
    == len(
        parallel_scheduler.path_links
    )
)


for step in first_parallel_slot.steps:

    assert (
        step.simulation_slot
        == slot_before
    )

    assert (
        step.attempt is not None
    )


print(
    "[PASS] Every unresolved physical link receives one "
    "generation attempt within the SAME global slot."
)


# =============================================================
# Test 7 — No physical link was processed twice
# =============================================================

slot_link_ids = tuple(
    step.link_id
    for step in first_parallel_slot.steps
)


assert (
    len(slot_link_ids)
    == len(set(slot_link_ids))
)

print(
    "[PASS] No physical elementary link is processed more "
    "than once per global simulation slot."
)


# =============================================================
# Test 8 — Continue parallel execution until EVERY physical
#          segment has a ready Bell pair.
# =============================================================

if not first_parallel_slot.all_links_ready:

    # Slot 0 has now completely finished.
    parallel_simulator.kernel.advance_slot()


parallel_resources = (
    parallel_scheduler.run_until_all_ready(
        max_slots=100
    )
)


assert (
    parallel_scheduler.all_links_ready
)

assert (
    len(parallel_resources)
    == len(
        parallel_scheduler.path_links
    )
)

print(
    "[PASS] Parallel scheduler eventually generates one "
    "simultaneously ready Bell pair on every physical link."
)


# =============================================================
# Test 9 — Every resource is an ELEMENTARY adjacent-node EP
# =============================================================

for link, resource in zip(
    parallel_scheduler.path_links,
    parallel_resources
):

    assert (
        resource.kind
        == EntanglementKind.ELEMENTARY
    )

    assert (
        resource.status
        == EntanglementStatus.ACTIVE
    )

    assert (
        resource.endpoint_names
        == tuple(
            sorted(
                (
                    link.node_a.name,
                    link.node_b.name
                )
            )
        )
    )

    assert (
        resource.source_link_ids
        == (
            link.link_id,
        )
    )

print(
    "[PASS] Every generated Bell pair connects exactly the "
    "two adjacent nodes of its physical link."
)


# =============================================================
# Test 10 — No swapped/non-local pair exists yet
# =============================================================

assert all(
    resource.kind
    == EntanglementKind.ELEMENTARY
    for resource in (
        parallel_simulator
        .entanglement_manager
        .active_resources
    )
)

assert not any(
    (
        resource.endpoint_names
        == tuple(
            sorted(
                (
                    parallel_a.name,
                    parallel_b.name
                )
            )
        )
    )
    for resource in (
        parallel_simulator
        .entanglement_manager
        .active_resources
    )
)

print(
    "[PASS] Phase 3B creates only elementary adjacent-node "
    "entanglement; no end-to-end pair is fabricated."
)


# =============================================================
# Test 11 — Exactly one ACTIVE EP exists for every path link
# =============================================================

assert (
    len(
        parallel_simulator
        .entanglement_manager
        .active_resources
    )
    == len(
        parallel_scheduler.path_links
    )
)

print(
    "[PASS] Complete path contains exactly one active "
    "elementary EP per physical-link segment."
)


# =============================================================
# Test 12 — Local memory occupancy is physically sensible
#
# Endpoint:
#     one adjacent EP half
#
# Intermediate repeater:
#     one left EP half + one right EP half
# =============================================================

assert (
    parallel_simulator
    .quantum_memory(
        parallel_a
    )
    .occupancy
    == 1
)

assert (
    parallel_simulator
    .quantum_memory(
        parallel_b
    )
    .occupancy
    == 1
)

assert (
    parallel_simulator
    .quantum_memory(
        parallel_r1
    )
    .occupancy
    == 2
)

assert (
    parallel_simulator
    .quantum_memory(
        parallel_r2
    )
    .occupancy
    == 2
)

print(
    "[PASS] Endpoints store one entangled half while each "
    "intermediate repeater stores its two adjacent halves."
)


# =============================================================
# Test 13 — READY links no longer attempt generation
# =============================================================

attempt_count_when_ready = (
    parallel_service
    .statistics.total_attempts
)


ready_slot = (
    parallel_scheduler
    .process_current_slot()
)


assert (
    ready_slot.all_links_ready
)

assert (
    ready_slot.physical_attempt_count
    == 0
)

assert (
    parallel_service
    .statistics.total_attempts
    == attempt_count_when_ready
)

assert all(
    step.action
    == ElementaryLinkStepAction.RESOURCE_WAITING
    for step in ready_slot.steps
)

print(
    "[PASS] Fully ready path waits without producing "
    "redundant Bell pairs."
)


# =============================================================
# Test 14 — Complete runtime consistency
# =============================================================

assert (
    parallel_simulator
    .validate_runtime_consistency()
    is True
)

print(
    "[PASS] Parallel elementary generation preserves "
    "complete cross-layer runtime consistency."
)


# =============================================================
# Final inspection
# =============================================================

print("\nPARALLEL SCHEDULER")
print(
    parallel_scheduler
)


print(
    "\nREADY ELEMENTARY RESOURCES"
)

for resource in parallel_resources:

    print(
        resource
    )


print(
    "\nCURRENT ELEMENTARY LINK STATE"
)

display(
    parallel_scheduler
    .link_state_dataframe()
)


print(
    "\nPARALLEL SLOT HISTORY"
)

display(
    parallel_scheduler
    .slot_history_dataframe()
)


print(
    "\nPHYSICAL ATTEMPT HISTORY"
)

display(
    parallel_service
    .statistics
    .attempts_dataframe()
)


print(
    "\nGLOBAL ENTANGLEMENT RESOURCE HISTORY"
)

display(
    parallel_simulator
    .entanglement_manager
    .resources_dataframe()
)


print(
    "\nENDPOINT A MEMORY"
)

display(
    parallel_simulator
    .quantum_memory(
        parallel_a
    )
    .slots_dataframe()
)


print(
    "\nREPEATER R1 MEMORY"
)

display(
    parallel_simulator
    .quantum_memory(
        parallel_r1
    )
    .slots_dataframe()
)


print(
    "\nREPEATER R2 MEMORY"
)

display(
    parallel_simulator
    .quantum_memory(
        parallel_r2
    )
    .slots_dataframe()
)


print(
    "\nENDPOINT B MEMORY"
)

display(
    parallel_simulator
    .quantum_memory(
        parallel_b
    )
    .slots_dataframe()
)


print("\n" + "=" * 76)
print("Verification completed successfully.")
print("=" * 76)

### Phase 3C : 12-09-2026T21:32:42 -- 14-09-2026T19:18:52

In [ ]:
# @title  Age-Preserving Quantum Memory Rebinding

def _phase3c_rebind_local_half_preserving_age(
    self,
    old_resource_id: str,
    new_resource_id: str,
    new_remote_node: NetworkNode,
    reason: str = "entanglement_swap_success"
) -> LocalEntanglementHalf:
    """
    Reassociate one surviving physical qubit from an old
    entanglement resource to a new resource.

    Crucially, stored_at_slot is PRESERVED.

    This models the outer qubits surviving a successful
    entanglement swap.

    It does not represent creating a new physical qubit.
    """

    if (
        not isinstance(old_resource_id, str)
        or not old_resource_id.strip()
    ):
        raise ValueError(
            "old_resource_id must be a non-empty string."
        )

    if (
        not isinstance(new_resource_id, str)
        or not new_resource_id.strip()
    ):
        raise ValueError(
            "new_resource_id must be a non-empty string."
        )

    if not isinstance(
        new_remote_node,
        NetworkNode
    ):
        raise TypeError(
            "new_remote_node must be a NetworkNode."
        )

    old_resource_id = old_resource_id.strip()
    new_resource_id = new_resource_id.strip()

    if (
        new_remote_node.name
        == self.owner.name
    ):
        raise QuantumMemoryError(
            "A local entanglement half cannot be "
            "entangled with its own memory owner."
        )

    if self.contains(
        new_resource_id
    ):
        raise QuantumMemoryDuplicateResourceError(
            f"Memory '{self.owner.name}' already contains "
            f"resource '{new_resource_id}'."
        )

    slot_index = self.find_slot(
        old_resource_id
    )

    old_half = self.get_local_half(
        old_resource_id
    )

    new_half = LocalEntanglementHalf(
        resource_id=new_resource_id,
        remote_node_name=new_remote_node.name,

        # CRITICAL:
        # physical qubit age does not reset after swapping.
        stored_at_slot=old_half.stored_at_slot
    )

    self._slots[
        slot_index
    ] = new_half

    self.kernel.emit(
        event_type="MEMORY_HALF_REBOUND",
        component="QuantumMemory",
        node_names=(
            self.owner.name,
            new_remote_node.name
        ),
        resource_ids=(
            old_resource_id,
            new_resource_id
        ),
        message=(
            f"Surviving local qubit at "
            f"'{self.owner.name}' rebound from "
            f"'{old_resource_id}' to "
            f"'{new_resource_id}'."
        ),
        metadata={
            "memory_slot":
                self.slot_id(
                    slot_index
                ),

            "preserved_stored_at_slot":
                old_half.stored_at_slot,

            "reason":
                str(reason)
        }
    )

    return new_half


# Extend the existing Phase-3A QuantumMemory API.
QuantumMemory.rebind_local_half_preserving_age = _phase3c_rebind_local_half_preserving_age

In [ ]:
# @title  Atomic Entanglement Swap Resource Transaction

from dataclasses import replace


def _phase3c_execute_swap_transaction(
    self,
    left_resource_id: str,
    right_resource_id: str,
    swap_node: NetworkNode,
    success: bool
) -> Optional[EntanglementResource]:
    """
    Atomically apply the resource consequences of one BSM.

    Inputs
    ------
    E(A,B) + E(B,C)

    Success
    -------
    - consume both child resources;
    - destroy both local B qubits;
    - preserve A and C physical qubits and their ages;
    - rebind them into new E(A,C);
    - create one SWAPPED EntanglementResource.

    Failure
    -------
    - BSM consumes the child entanglement resources;
    - all associated local halves are released;
    - no parent resource is created.

    This function does NOT determine whether the BSM succeeds.
    """

    if not isinstance(
        swap_node,
        NetworkNode
    ):
        raise TypeError(
            "swap_node must be a NetworkNode."
        )

    if not isinstance(
        success,
        bool
    ):
        raise TypeError(
            "success must be bool."
        )

    if (
        left_resource_id
        == right_resource_id
    ):
        raise EntanglementConsistencyError(
            "A swap requires two distinct entanglement resources."
        )

    left = self.get_resource(
        left_resource_id
    )

    right = self.get_resource(
        right_resource_id
    )

    # =========================================================
    # Child Resources Must Still Be ACTIVE
    # =========================================================

    if not left.is_active:
        raise EntanglementResourceStateError(
            f"Left resource '{left.resource_id}' is not ACTIVE."
        )

    if not right.is_active:
        raise EntanglementResourceStateError(
            f"Right resource '{right.resource_id}' is not ACTIVE."
        )

    self._assert_active_storage_consistency(
        left
    )

    self._assert_active_storage_consistency(
        right
    )

    # =========================================================
    # Exactly One Shared Endpoint
    # =========================================================

    shared_nodes = (
        set(left.endpoint_names)
        &
        set(right.endpoint_names)
    )

    if shared_nodes != {
        swap_node.name
    }:
        raise EntanglementConsistencyError(
            "Swap operands must share exactly the requested "
            "swapping node."
        )

    if not swap_node.swapping_capable:
        raise EntanglementConsistencyError(
            f"Node '{swap_node.name}' is not swapping-capable."
        )

    outer_left_name = (
        left.other_endpoint_name(
            swap_node.name
        )
    )

    outer_right_name = (
        right.other_endpoint_name(
            swap_node.name
        )
    )

    if (
        outer_left_name
        == outer_right_name
    ):
        raise EntanglementConsistencyError(
            "Entanglement swapping may not create a "
            "self-entangled network resource."
        )

    # =========================================================
    # Child Physical Provenance Must Be Disjoint
    # =========================================================

    if (
        set(left.source_link_ids)
        &
        set(right.source_link_ids)
    ):
        raise EntanglementConsistencyError(
            "Swap operands reuse physical-link provenance. "
            "This would create an invalid/cyclic resource."
        )

    outer_left_memory = (
        self.memory_for_node(
            outer_left_name
        )
    )

    outer_right_memory = (
        self.memory_for_node(
            outer_right_name
        )
    )

    swap_memory = (
        self.memory_for_node(
            swap_node.name
        )
    )

    # =========================================================
    # Transaction Snapshot
    #
    # Resources are immutable, so a shallow dictionary copy is
    # sufficient. Memory slot records are frozen dataclasses.
    # =========================================================

    resources_snapshot = dict(
        self._resources
    )

    next_number_snapshot = (
        self._next_resource_number
    )

    relevant_memories = {
        outer_left_name:
            outer_left_memory,

        swap_node.name:
            swap_memory,

        outer_right_name:
            outer_right_memory
    }

    memory_snapshots = {
        node_name: list(
            memory._slots
        )
        for node_name, memory
        in relevant_memories.items()
    }

    current_slot = (
        self.kernel.current_slot
    )

    try:

        # =====================================================
        # BSM FAILURE
        #
        # Both input EPs have been consumed by the attempted
        # measurement. No new E(A,C) exists.
        # =====================================================

        if not success:

            for resource in (
                left,
                right
            ):

                for endpoint_name in (
                    resource.endpoint_names
                ):

                    memory = (
                        self.memory_for_node(
                            endpoint_name
                        )
                    )

                    memory.remove_local_half(
                        resource.resource_id,
                        reason=(
                            "failed_entanglement_swap"
                        )
                    )

                self._resources[
                    resource.resource_id
                ] = replace(

                    resource,

                    status=(
                        EntanglementStatus.CONSUMED
                    ),

                    ended_slot=current_slot,

                    termination_reason=(
                        f"bsm_failure_at_{swap_node.name}"
                    )
                )

            self.kernel.emit(
                event_type="ENTANGLEMENT_SWAP_FAILED",
                component="EntanglementManager",
                node_names=(
                    outer_left_name,
                    swap_node.name,
                    outer_right_name
                ),
                resource_ids=(
                    left.resource_id,
                    right.resource_id
                ),
                message=(
                    f"BSM at '{swap_node.name}' failed; "
                    f"both child entanglement resources "
                    f"were consumed."
                )
            )

            return None

        # =====================================================
        # BSM SUCCESS
        # =====================================================

        parent_resource_id = (
            f"ENT-{self._next_resource_number:06d}"
        )

        self._next_resource_number += 1

        parent_endpoints = tuple(
            sorted(
                (
                    outer_left_name,
                    outer_right_name
                )
            )
        )

        # Provenance records all underlying elementary links.
        source_link_ids = tuple(
            dict.fromkeys(
                left.source_link_ids
                +
                right.source_link_ids
            )
        )

        parent = EntanglementResource(

            resource_id=(
                parent_resource_id
            ),

            endpoint_a_name=(
                parent_endpoints[0]
            ),

            endpoint_b_name=(
                parent_endpoints[1]
            ),

            kind=(
                EntanglementKind.SWAPPED
            ),

            created_slot=(
                current_slot
            ),

            source_link_ids=(
                source_link_ids
            ),

            parent_resource_ids=(
                left.resource_id,
                right.resource_id
            ),

            swap_node_name=(
                swap_node.name
            )
        )

        outer_left_node = (
            outer_left_memory.owner
        )

        outer_right_node = (
            outer_right_memory.owner
        )

        # =====================================================
        # Preserve Outer Physical Qubits
        #
        # Their stored_at_slot values MUST survive.
        # =====================================================

        outer_left_memory\
            .rebind_local_half_preserving_age(

                old_resource_id=(
                    left.resource_id
                ),

                new_resource_id=(
                    parent.resource_id
                ),

                new_remote_node=(
                    outer_right_node
                )
            )

        outer_right_memory\
            .rebind_local_half_preserving_age(

                old_resource_id=(
                    right.resource_id
                ),

                new_resource_id=(
                    parent.resource_id
                ),

                new_remote_node=(
                    outer_left_node
                )
            )

        # =====================================================
        # BSM Consumes Both Physical Qubits at Swap Node
        # =====================================================

        swap_memory.remove_local_half(
            left.resource_id,
            reason="successful_entanglement_swap_bsm"
        )

        swap_memory.remove_local_half(
            right.resource_id,
            reason="successful_entanglement_swap_bsm"
        )

        # =====================================================
        # Children Become Historical CONSUMED Resources
        # =====================================================

        self._resources[
            left.resource_id
        ] = replace(

            left,

            status=(
                EntanglementStatus.CONSUMED
            ),

            ended_slot=current_slot,

            termination_reason=(
                f"successful_swap_at_{swap_node.name}"
            )
        )

        self._resources[
            right.resource_id
        ] = replace(

            right,

            status=(
                EntanglementStatus.CONSUMED
            ),

            ended_slot=current_slot,

            termination_reason=(
                f"successful_swap_at_{swap_node.name}"
            )
        )

        # Parent becomes authoritative ACTIVE resource.
        self._resources[
            parent.resource_id
        ] = parent

        self._assert_active_storage_consistency(
            parent
        )

        self.kernel.emit(
            event_type="ENTANGLEMENT_SWAP_SUCCEEDED",
            component="EntanglementManager",
            node_names=(
                outer_left_name,
                swap_node.name,
                outer_right_name
            ),
            resource_ids=(
                left.resource_id,
                right.resource_id,
                parent.resource_id
            ),
            message=(
                f"BSM at '{swap_node.name}' created "
                f"swapped resource "
                f"'{parent.resource_id}'."
            ),
            metadata={
                "left_child":
                    left.resource_id,

                "right_child":
                    right.resource_id,

                "parent":
                    parent.resource_id
            }
        )

        return parent

    except Exception:

        # =====================================================
        # Atomic Rollback
        # =====================================================

        self._resources.clear()

        self._resources.update(
            resources_snapshot
        )

        self._next_resource_number = (
            next_number_snapshot
        )

        for (
            node_name,
            slots
        ) in memory_snapshots.items():

            relevant_memories[
                node_name
            ]._slots = list(
                slots
            )

        raise


EntanglementManager.execute_swap_transaction = (
    _phase3c_execute_swap_transaction
)

In [ ]:
# @title  Entanglement Fidelity Registry


@dataclass(frozen=True)
class EntanglementFidelityRecord:
    """
    Quality metadata associated with an entanglement resource.

    This record is deliberately separate from the resource's
    lifecycle state.
    """

    resource_id: str

    fidelity: float

    model_name: str

    recorded_slot: int

    parent_resource_ids: tuple[str, ...] = ()

    def __post_init__(self):

        if not (
            0.0 <= self.fidelity <= 1.0
        ):
            raise ValueError(
                "Entanglement fidelity must lie within [0, 1]."
            )


class EntanglementFidelityRegistry:
    """
    Sidecar registry for entanglement quality.

    Phase 3C.1 uses the ideal baseline:

        F = 1.0

    Later physical models may replace this assumption without
    changing EntanglementResource lifecycle semantics.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator
    ):

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        self.simulator = simulator

        self._records: dict[
            str,
            EntanglementFidelityRecord
        ] = {}

    def register(
        self,
        resource: EntanglementResource,
        fidelity: float,
        model_name: str
    ) -> EntanglementFidelityRecord:

        if not isinstance(
            resource,
            EntanglementResource
        ):
            raise TypeError(
                "resource must be an EntanglementResource."
            )

        if (
            resource.resource_id
            in self._records
        ):
            raise ValueError(
                f"Fidelity already registered for "
                f"'{resource.resource_id}'."
            )

        record = EntanglementFidelityRecord(

            resource_id=(
                resource.resource_id
            ),

            fidelity=float(
                fidelity
            ),

            model_name=str(
                model_name
            ),

            recorded_slot=(
                self.simulator.kernel.current_slot
            ),

            parent_resource_ids=(
                resource.parent_resource_ids
            )
        )

        self._records[
            resource.resource_id
        ] = record

        return record

    def register_ideal(
        self,
        resource: EntanglementResource
    ) -> EntanglementFidelityRecord:

        return self.register(
            resource=resource,
            fidelity=1.0,
            model_name="IDEAL_FIDELITY_BASELINE"
        )

    def get(
        self,
        resource_or_id
    ) -> EntanglementFidelityRecord:

        resource_id = (
            resource_or_id.resource_id
            if isinstance(
                resource_or_id,
                EntanglementResource
            )
            else str(resource_or_id)
        )

        if resource_id not in self._records:
            raise KeyError(
                f"No fidelity record exists for "
                f"'{resource_id}'."
            )

        return self._records[
            resource_id
        ]

    def fidelity(
        self,
        resource_or_id
    ) -> float:

        return self.get(
            resource_or_id
        ).fidelity

    def dataframe(
        self
    ) -> pd.DataFrame:

        return pd.DataFrame([
            {
                "Resource ID":
                    record.resource_id,

                "Fidelity":
                    record.fidelity,

                "Model":
                    record.model_name,

                "Recorded Slot":
                    record.recorded_slot,

                "Parents":
                    record.parent_resource_ids
            }
            for record
            in self._records.values()
        ])

In [ ]:
# @title  Entanglement Swapping Physics Models

from abc import ABC, abstractmethod


class EntanglementSwapModelError(Exception):
    pass


class EntanglementSwapModel(ABC):
    """
    Physical model for one atomic Bell-state-measurement swap.

    Model responsibility:
        - BSM success probability;
        - fidelity transformation.

    The model does NOT mutate resources.
    """

    @property
    @abstractmethod
    def model_name(self) -> str:
        pass

    @abstractmethod
    def success_probability(
        self,
        swap_node: NetworkNode,
        left_resource: EntanglementResource,
        right_resource: EntanglementResource
    ) -> float:
        pass

    @abstractmethod
    def output_fidelity(
        self,
        left_fidelity: float,
        right_fidelity: float
    ) -> float:
        pass


class IdealEntanglementSwapModel(
    EntanglementSwapModel
):
    """
    Ideal Phase-3C baseline.

        P(BSM success) = 1
        perfect input EPs -> perfect output EP

    No gate noise, detector inefficiency, or operational
    fidelity loss is modeled here.
    """

    @property
    def model_name(self) -> str:
        return "IdealEntanglementSwapModel"

    def success_probability(
        self,
        swap_node,
        left_resource,
        right_resource
    ) -> float:

        return 1.0

    def output_fidelity(
        self,
        left_fidelity: float,
        right_fidelity: float
    ) -> float:

        # This baseline intentionally assumes perfect inputs.
        if (
            not math.isclose(
                left_fidelity,
                1.0,
                rel_tol=0.0,
                abs_tol=1e-12
            )
            or not math.isclose(
                right_fidelity,
                1.0,
                rel_tol=0.0,
                abs_tol=1e-12
            )
        ):
            raise EntanglementSwapModelError(
                "The ideal Phase-3C baseline expects "
                "ideal input Bell pairs."
            )

        return 1.0


class VerificationAlwaysFailSwapModel(
    EntanglementSwapModel
):
    """
    TEST-ONLY model used to verify destructive BSM-failure
    resource semantics.

    This is NOT a physical production model.
    """

    @property
    def model_name(self) -> str:
        return "VerificationAlwaysFailSwapModel"

    def success_probability(
        self,
        swap_node,
        left_resource,
        right_resource
    ) -> float:

        return 0.0

    def output_fidelity(
        self,
        left_fidelity,
        right_fidelity
    ) -> float:

        raise EntanglementSwapModelError("A failed BSM creates no output fidelity.")

In [ ]:
# @title  Atomic Entanglement Swap Engine


@dataclass(frozen=True)
class EntanglementSwapAttempt:
    """
    Immutable record of one attempted BSM swap.
    """

    attempt_index: int

    simulation_slot: int

    swap_node_name: str

    left_resource_id: str
    right_resource_id: str

    model_name: str

    success_probability: float

    random_draw: Optional[float]

    success: bool

    output_resource_id: Optional[str]

    left_fidelity: float
    right_fidelity: float

    output_fidelity: Optional[float]


class EntanglementSwapEngine:
    """
    Execute ONE atomic entanglement swap at a time.

    Physics decides success.

    EntanglementManager performs the resource transaction.

    The control plane authorizes participating devices.

    No global simulation-clock advancement occurs here.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        model: EntanglementSwapModel,
        fidelity_registry: EntanglementFidelityRegistry
    ):

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            model,
            EntanglementSwapModel
        ):
            raise TypeError(
                "model must be an EntanglementSwapModel."
            )

        if not isinstance(
            fidelity_registry,
            EntanglementFidelityRegistry
        ):
            raise TypeError(
                "fidelity_registry must be an "
                "EntanglementFidelityRegistry."
            )

        if (
            fidelity_registry.simulator
            is not simulator
        ):
            raise ValueError(
                "Fidelity registry belongs to another simulator."
            )

        self.simulator = simulator
        self.model = model
        self.fidelity_registry = (
            fidelity_registry
        )

        self._attempts: list[
            EntanglementSwapAttempt
        ] = []

    # =========================================================
    # Atomic Swap
    # =========================================================

    def attempt_swap(
        self,
        left_resource_id: str,
        right_resource_id: str,
        swap_node
    ) -> EntanglementSwapAttempt:

        self.simulator.validate_runtime_consistency()

        # Resolve expirations BEFORE deciding whether the
        # candidate resources still physically exist.
        self.simulator\
            .entanglement_manager\
            .synchronize_expirations()

        manager = (
            self.simulator
            .entanglement_manager
        )

        left = manager.get_resource(
            left_resource_id
        )

        right = manager.get_resource(
            right_resource_id
        )

        swap_node = (
            self.simulator.topology
            .get_node(
                swap_node
            )
        )

        # =====================================================
        # Live Resource Validation
        # =====================================================

        if not left.is_active:
            raise EntanglementResourceStateError(
                f"Resource '{left.resource_id}' is not ACTIVE."
            )

        if not right.is_active:
            raise EntanglementResourceStateError(
                f"Resource '{right.resource_id}' is not ACTIVE."
            )

        shared_nodes = (
            set(left.endpoint_names)
            &
            set(right.endpoint_names)
        )

        if shared_nodes != {
            swap_node.name
        }:
            raise EntanglementConsistencyError(
                "Swap operands must share exactly one "
                "swapping endpoint."
            )

        if not swap_node.swapping_capable:
            raise EntanglementConsistencyError(
                f"Node '{swap_node.name}' lacks "
                f"swapping capability."
            )

        outer_left_name = (
            left.other_endpoint_name(
                swap_node.name
            )
        )

        outer_right_name = (
            right.other_endpoint_name(
                swap_node.name
            )
        )

        outer_left_node = (
            self.simulator.topology
            .get_node(
                outer_left_name
            )
        )

        outer_right_node = (
            self.simulator.topology
            .get_node(
                outer_right_name
            )
        )

        # =====================================================
        # NARAD-QC Authentication Gate
        # =====================================================

        self.simulator.require_quantum_eligibility(
            outer_left_node,
            swap_node,
            outer_right_node
        )

        # =====================================================
        # Fidelity Inputs
        # =====================================================

        left_fidelity = (
            self.fidelity_registry
            .fidelity(
                left
            )
        )

        right_fidelity = (
            self.fidelity_registry
            .fidelity(
                right
            )
        )

        # =====================================================
        # Swapping Physics
        # =====================================================

        probability = float(
            self.model.success_probability(
                swap_node=swap_node,
                left_resource=left,
                right_resource=right
            )
        )

        if not (
            0.0 <= probability <= 1.0
        ):
            raise EntanglementSwapModelError(
                "Swap success probability must lie in [0, 1]."
            )

        # Deterministic endpoints do not need to consume RNG.
        if probability <= 0.0:

            random_draw = None
            success = False

        elif probability >= 1.0:

            random_draw = None
            success = True

        else:

            random_draw = float(
                self.simulator.kernel.rng.random()
            )

            success = (
                random_draw
                < probability
            )

        # =====================================================
        # Atomic Resource Transaction
        # =====================================================

        output_resource = (
            manager.execute_swap_transaction(

                left_resource_id=(
                    left.resource_id
                ),

                right_resource_id=(
                    right.resource_id
                ),

                swap_node=(
                    swap_node
                ),

                success=(
                    success
                )
            )
        )

        output_fidelity = None

        if success:

            if output_resource is None:
                raise RuntimeError(
                    "Successful BSM produced no parent resource."
                )

            output_fidelity = (
                self.model.output_fidelity(
                    left_fidelity=left_fidelity,
                    right_fidelity=right_fidelity
                )
            )

            self.fidelity_registry.register(

                resource=output_resource,

                fidelity=output_fidelity,

                model_name=(
                    self.model.model_name
                )
            )

        else:

            if output_resource is not None:
                raise RuntimeError(
                    "Failed BSM unexpectedly created "
                    "a parent resource."
                )

        attempt = EntanglementSwapAttempt(

            attempt_index=len(
                self._attempts
            ),

            simulation_slot=(
                self.simulator.kernel.current_slot
            ),

            swap_node_name=(
                swap_node.name
            ),

            left_resource_id=(
                left.resource_id
            ),

            right_resource_id=(
                right.resource_id
            ),

            model_name=(
                self.model.model_name
            ),

            success_probability=(
                probability
            ),

            random_draw=(
                random_draw
            ),

            success=(
                success
            ),

            output_resource_id=(
                output_resource.resource_id
                if output_resource is not None
                else None
            ),

            left_fidelity=(
                left_fidelity
            ),

            right_fidelity=(
                right_fidelity
            ),

            output_fidelity=(
                output_fidelity
            )
        )

        self._attempts.append(
            attempt
        )

        self.simulator.kernel.emit(

            event_type=(
                "ATOMIC_SWAP_ATTEMPT_SUCCEEDED"
                if success
                else
                "ATOMIC_SWAP_ATTEMPT_FAILED"
            ),

            component="EntanglementSwapEngine",

            node_names=(
                outer_left_name,
                swap_node.name,
                outer_right_name
            ),

            resource_ids=tuple(
                resource_id
                for resource_id in (
                    left.resource_id,
                    right.resource_id,
                    (
                        output_resource.resource_id
                        if output_resource is not None
                        else None
                    )
                )
                if resource_id is not None
            ),

            message=(
                f"Atomic BSM swap at "
                f"'{swap_node.name}' "
                f"{'succeeded' if success else 'failed'}."
            ),

            metadata={
                "model":
                    self.model.model_name,

                "success_probability":
                    probability,

                "random_draw":
                    random_draw,

                "output_fidelity":
                    output_fidelity
            }
        )

        return attempt

    # =========================================================
    # Inspection
    # =========================================================

    @property
    def attempts(
        self
    ) -> tuple[
        EntanglementSwapAttempt, ...
    ]:

        return tuple(
            self._attempts
        )

    def attempts_dataframe(
        self
    ) -> pd.DataFrame:

        return pd.DataFrame([
            {
                "Attempt":
                    attempt.attempt_index,

                "Slot":
                    attempt.simulation_slot,

                "Swap Node":
                    attempt.swap_node_name,

                "Left Resource":
                    attempt.left_resource_id,

                "Right Resource":
                    attempt.right_resource_id,

                "Model":
                    attempt.model_name,

                "P(success)":
                    attempt.success_probability,

                "Random Draw":
                    attempt.random_draw,

                "Success":
                    attempt.success,

                "Output Resource":
                    attempt.output_resource_id,

                "Left Fidelity":
                    attempt.left_fidelity,

                "Right Fidelity":
                    attempt.right_fidelity,

                "Output Fidelity":
                    attempt.output_fidelity
            }
            for attempt
            in self._attempts
        ])

In [ ]:
# @title Ideal Atomic Entanglement Swap Verification


print("=" * 76)
print("Ideal Atomic Swapping Verification")
print("=" * 76)


# =============================================================
# Physical topology
#
# Swap-A ----5 km---- Swap-R1 ----5 km---- Swap-C
#
# There is NO physical Swap-A--Swap-C link.
# =============================================================

swap_topology = NetworkTopology(
    name="Phase3C1-Ideal-Swap-Network"
)


swap_a = NetworkNode(
    name="Swap-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


swap_r1 = NetworkNode(
    name="Swap-R1",
    x_km=5,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


swap_c = NetworkNode(
    name="Swap-C",
    x_km=10,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


for node in (
    swap_a,
    swap_r1,
    swap_c
):
    swap_topology.add_node(
        node
    )


swap_path = (
    swap_a,
    swap_r1,
    swap_c
)


for left, right in zip(
    swap_path[:-1],
    swap_path[1:]
):

    swap_topology.add_link(
        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert not swap_topology.contains_link(
    swap_a,
    swap_c
)

print(
    "[PASS] Remote endpoints have no direct physical link."
)


# =============================================================
# Assemble / authenticate network
# =============================================================

swap_simulator = QuantumNetworkSimulator(

    topology=swap_topology,

    simulation_seed=310001,

    controller_id=(
        "PHASE3C1-CONTROLLER"
    ),

    parameter_set=(
        MLDSAParameterSet.ML_DSA_65
    ),

    max_storage_slots=20
)


swap_simulator.authenticate_all_nodes()

assert swap_simulator.require_quantum_eligibility(
    *swap_path
)

print(
    "[PASS] All three physical devices are mutually authenticated."
)


# =============================================================
# Generate both elementary Bell pairs in parallel
# =============================================================

swap_generation_service = (
    ElementaryEntanglementService(

        simulator=swap_simulator,

        model=FiberSurvivalProxyModel()
    )
)


swap_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=swap_simulator,

        service=swap_generation_service,

        ordered_path=swap_path
    )
)


elementary_swap_resources = (
    swap_generation_scheduler
    .run_until_all_ready(
        max_slots=100
    )
)


assert (
    len(elementary_swap_resources)
    == 2
)


left_ep = (
    elementary_swap_resources[0]
)

right_ep = (
    elementary_swap_resources[1]
)


assert left_ep.endpoint_names == tuple(
    sorted(
        (
            swap_a.name,
            swap_r1.name
        )
    )
)

assert right_ep.endpoint_names == tuple(
    sorted(
        (
            swap_r1.name,
            swap_c.name
        )
    )
)

print(
    "[PASS] Two adjacent elementary Bell pairs are ready."
)


# =============================================================
# Ideal fidelity baseline
# =============================================================

swap_fidelity = (
    EntanglementFidelityRegistry(
        swap_simulator
    )
)


swap_fidelity.register_ideal(
    left_ep
)

swap_fidelity.register_ideal(
    right_ep
)


assert (
    swap_fidelity.fidelity(
        left_ep
    )
    == 1.0
)

assert (
    swap_fidelity.fidelity(
        right_ep
    )
    == 1.0
)

print(
    "[PASS] Elementary Bell pairs begin with ideal fidelity 1.0."
)


# =============================================================
# Record physical ages before BSM
# =============================================================

outer_a_half_before = (
    swap_simulator
    .quantum_memory(
        swap_a
    )
    .get_local_half(
        left_ep.resource_id
    )
)


outer_c_half_before = (
    swap_simulator
    .quantum_memory(
        swap_c
    )
    .get_local_half(
        right_ep.resource_id
    )
)


a_original_storage_slot = (
    outer_a_half_before.stored_at_slot
)

c_original_storage_slot = (
    outer_c_half_before.stored_at_slot
)


# =============================================================
# Test authentication gate on swapping
# =============================================================

swap_simulator\
    .authentication_service\
    .invalidate_authentication(
        swap_r1,
        reason="phase3c_swap_gate_test"
    )


ideal_swap_engine = (
    EntanglementSwapEngine(

        simulator=swap_simulator,

        model=(
            IdealEntanglementSwapModel()
        ),

        fidelity_registry=(
            swap_fidelity
        )
    )
)


blocked = False

try:

    ideal_swap_engine.attempt_swap(
        left_resource_id=(
            left_ep.resource_id
        ),

        right_resource_id=(
            right_ep.resource_id
        ),

        swap_node=(
            swap_r1
        )
    )

except QuantumOperationAuthorizationError:

    blocked = True


assert blocked

assert left_ep.resource_id in {
    resource.resource_id
    for resource
    in swap_simulator
    .entanglement_manager
    .active_resources
}

assert right_ep.resource_id in {
    resource.resource_id
    for resource
    in swap_simulator
    .entanglement_manager
    .active_resources
}

print(
    "[PASS] Unauthenticated repeater cannot trigger a BSM "
    "or consume quantum resources."
)


# =============================================================
# Freshly authenticate repeater again
# =============================================================

swap_simulator.authenticate_node(
    swap_r1
)


# =============================================================
# Execute ideal BSM
# =============================================================

slot_before_swap = (
    swap_simulator.kernel.current_slot
)


ideal_swap_attempt = (
    ideal_swap_engine.attempt_swap(

        left_resource_id=(
            left_ep.resource_id
        ),

        right_resource_id=(
            right_ep.resource_id
        ),

        swap_node=(
            swap_r1
        )
    )
)


assert ideal_swap_attempt.success

assert (
    ideal_swap_attempt.success_probability
    == 1.0
)

assert (
    ideal_swap_attempt.random_draw
    is None
)

assert (
    swap_simulator.kernel.current_slot
    == slot_before_swap
)

print(
    "[PASS] Ideal atomic BSM succeeds deterministically "
    "without advancing global simulation time."
)


# =============================================================
# Parent resource
# =============================================================

parent_ep = (
    swap_simulator
    .entanglement_manager
    .get_resource(
        ideal_swap_attempt.output_resource_id
    )
)


assert (
    parent_ep.kind
    == EntanglementKind.SWAPPED
)

assert (
    parent_ep.status
    == EntanglementStatus.ACTIVE
)

assert (
    parent_ep.endpoint_names
    == tuple(
        sorted(
            (
                swap_a.name,
                swap_c.name
            )
        )
    )
)

assert (
    parent_ep.parent_resource_ids
    == (
        left_ep.resource_id,
        right_ep.resource_id
    )
)

assert (
    parent_ep.swap_node_name
    == swap_r1.name
)

print(
    "[PASS] BSM creates remote E(Swap-A, Swap-C) "
    "without a physical A--C link."
)


# =============================================================
# Child resources are consumed
# =============================================================

left_after = (
    swap_simulator
    .entanglement_manager
    .get_resource(
        left_ep.resource_id
    )
)

right_after = (
    swap_simulator
    .entanglement_manager
    .get_resource(
        right_ep.resource_id
    )
)


assert (
    left_after.status
    == EntanglementStatus.CONSUMED
)

assert (
    right_after.status
    == EntanglementStatus.CONSUMED
)

print(
    "[PASS] Both child EP relationships are consumed by the BSM."
)


# =============================================================
# Repeater's measured qubits are gone
# =============================================================

assert (
    swap_simulator
    .quantum_memory(
        swap_r1
    )
    .occupancy
    == 0
)

print(
    "[PASS] Bell-state measurement consumes both repeater qubits."
)


# =============================================================
# Outer qubits survive
# =============================================================

assert (
    swap_simulator
    .quantum_memory(
        swap_a
    )
    .occupancy
    == 1
)

assert (
    swap_simulator
    .quantum_memory(
        swap_c
    )
    .occupancy
    == 1
)


a_parent_half = (
    swap_simulator
    .quantum_memory(
        swap_a
    )
    .get_local_half(
        parent_ep.resource_id
    )
)


c_parent_half = (
    swap_simulator
    .quantum_memory(
        swap_c
    )
    .get_local_half(
        parent_ep.resource_id
    )
)


assert (
    a_parent_half.remote_node_name
    == swap_c.name
)

assert (
    c_parent_half.remote_node_name
    == swap_a.name
)

print(
    "[PASS] Outer physical qubits survive and are rebound "
    "into the swapped EP."
)


# =============================================================
# CRITICAL TEST — Swapping must NOT reset physical qubit age
# =============================================================

assert (
    a_parent_half.stored_at_slot
    == a_original_storage_slot
)

assert (
    c_parent_half.stored_at_slot
    == c_original_storage_slot
)

print(
    "[PASS] Successful swapping preserves the original "
    "storage age of surviving outer qubits."
)


# =============================================================
# Ideal fidelity
# =============================================================

assert math.isclose(
    swap_fidelity.fidelity(
        parent_ep
    ),
    1.0,
    abs_tol=1e-12
)

print(
    "[PASS] Ideal swapping preserves fidelity F = 1.0."
)


# =============================================================
# Resource provenance
# =============================================================

assert set(
    parent_ep.source_link_ids
) == set(
    left_ep.source_link_ids
    +
    right_ep.source_link_ids
)

print(
    "[PASS] Swapped resource retains complete physical-link "
    "provenance."
)


# =============================================================
# Runtime consistency
# =============================================================

assert (
    swap_simulator
    .validate_runtime_consistency()
    is True
)

print(
    "[PASS] Complete runtime remains consistent after swapping."
)


# =============================================================
# Inspection
# =============================================================

print("\nSWAP ATTEMPT")

display(
    ideal_swap_engine
    .attempts_dataframe()
)


print("\nENTANGLEMENT RESOURCE HISTORY")

display(
    swap_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\nFIDELITY HISTORY")

display(
    swap_fidelity
    .dataframe()
)


print("\nSWAP NODE MEMORY")

display(
    swap_simulator
    .quantum_memory(
        swap_r1
    )
    .slots_dataframe()
)


print("\nOUTER A MEMORY")

display(
    swap_simulator
    .quantum_memory(
        swap_a
    )
    .slots_dataframe()
)


print("\nOUTER C MEMORY")

display(
    swap_simulator
    .quantum_memory(
        swap_c
    )
    .slots_dataframe()
)


print("\n" + "=" * 76)
print("Ideal swapping verification completed successfully.")
print("=" * 76)

In [ ]:
# @title Delayed Swap Age-Preservation Verification


print("=" * 80)
print("Delayed Swap Age-Preservation Verification")
print("=" * 80)


# ============================================================================
# PURPOSE
# ============================================================================
#
# This test specifically verifies that:
#
#     entanglement swapping creates a NEW entanglement relationship
#
# while:
#
#     the surviving OUTER PHYSICAL QUBITS retain their original storage ages.
#
#
# We deliberately:
#
#   1. generate E(A,R1) and E(R1,C),
#   2. wait several global simulation slots,
#   3. verify that the outer qubits have aged,
#   4. perform an ideal BSM,
#   5. verify that the new parent resource is created NOW,
#   6. verify that the physical qubits were NOT "reborn" NOW.
#
# ============================================================================


# ============================================================================
# Physical topology
#
# Age-A ----5 km---- Age-R1 ----5 km---- Age-C
#
# There is deliberately NO physical Age-A--Age-C link.
# ============================================================================

age_test_topology = NetworkTopology(
    name="Phase3C1-Delayed-Age-Test"
)


age_a = NetworkNode(
    name="Age-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


age_r1 = NetworkNode(
    name="Age-R1",
    x_km=5,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


age_c = NetworkNode(
    name="Age-C",
    x_km=10,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


for node in (
    age_a,
    age_r1,
    age_c
):
    age_test_topology.add_node(
        node
    )


age_path = (
    age_a,
    age_r1,
    age_c
)


for left, right in zip(
    age_path[:-1],
    age_path[1:]
):

    age_test_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert age_test_topology.validate_path(
    age_path
)


assert not age_test_topology.contains_link(
    age_a,
    age_c
)


print(
    "[PASS] Test topology contains only the two adjacent "
    "physical links; no direct outer-node fibre exists."
)


# ============================================================================
# Assemble a completely fresh simulator
#
# Memory lifetime is deliberately much larger than our waiting interval.
#
# We want the Bell pairs to AGE,
# but NOT EXPIRE before the swap.
# ============================================================================

age_test_simulator = QuantumNetworkSimulator(

    topology=age_test_topology,

    simulation_seed=31415926,

    controller_id="PHASE3C1-AGE-CONTROLLER",

    parameter_set=(
        MLDSAParameterSet.ML_DSA_65
    ),

    max_storage_slots=20
)


# ============================================================================
# Authenticate every participant
# ============================================================================

age_auth_results = (
    age_test_simulator
    .authenticate_all_nodes()
)


assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in age_auth_results.values()
)


assert age_test_simulator.require_quantum_eligibility(
    *age_path
)


print(
    "[PASS] Endpoint and repeater devices are mutually authenticated."
)


# ============================================================================
# Generate the two adjacent elementary Bell pairs in parallel
# ============================================================================

age_generation_service = (
    ElementaryEntanglementService(

        simulator=age_test_simulator,

        model=FiberSurvivalProxyModel()
    )
)


age_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=age_test_simulator,

        service=age_generation_service,

        ordered_path=age_path,

        demand_active=True
    )
)


age_elementary_resources = (
    age_generation_scheduler
    .run_until_all_ready(
        max_slots=100
    )
)


assert len(
    age_elementary_resources
) == 2


age_left_ep = (
    age_elementary_resources[0]
)

age_right_ep = (
    age_elementary_resources[1]
)


assert (
    age_left_ep.kind
    == EntanglementKind.ELEMENTARY
)

assert (
    age_right_ep.kind
    == EntanglementKind.ELEMENTARY
)

assert age_left_ep.is_active
assert age_right_ep.is_active


print(
    "[PASS] Two adjacent ACTIVE elementary Bell pairs are ready."
)


# ============================================================================
# Fidelity sidecar
# ============================================================================

age_fidelity_registry = (
    EntanglementFidelityRegistry(
        age_test_simulator
    )
)


age_fidelity_registry.register_ideal(
    age_left_ep
)

age_fidelity_registry.register_ideal(
    age_right_ep
)


assert math.isclose(
    age_fidelity_registry.fidelity(
        age_left_ep
    ),
    1.0,
    abs_tol=1e-12
)


assert math.isclose(
    age_fidelity_registry.fidelity(
        age_right_ep
    ),
    1.0,
    abs_tol=1e-12
)


print(
    "[PASS] Both elementary resources begin at ideal fidelity F = 1.0."
)


# ============================================================================
# Capture PRE-SWAP physical-qubit state
# ============================================================================

age_a_memory = (
    age_test_simulator
    .quantum_memory(
        age_a
    )
)

age_r1_memory = (
    age_test_simulator
    .quantum_memory(
        age_r1
    )
)

age_c_memory = (
    age_test_simulator
    .quantum_memory(
        age_c
    )
)


age_a_half_before = (
    age_a_memory
    .get_local_half(
        age_left_ep.resource_id
    )
)


age_c_half_before = (
    age_c_memory
    .get_local_half(
        age_right_ep.resource_id
    )
)


original_a_storage_slot = (
    age_a_half_before.stored_at_slot
)

original_c_storage_slot = (
    age_c_half_before.stored_at_slot
)


generation_ready_slot = (
    age_test_simulator.kernel.current_slot
)


a_age_before_wait = (
    age_a_memory.age_slots(
        age_left_ep.resource_id
    )
)


c_age_before_wait = (
    age_c_memory.age_slots(
        age_right_ep.resource_id
    )
)


print("\nPRE-WAIT PHYSICAL-QUBIT STATE")

print(
    f"A-side physical qubit stored at slot: "
    f"{original_a_storage_slot}"
)

print(
    f"C-side physical qubit stored at slot: "
    f"{original_c_storage_slot}"
)

print(
    f"Current simulation slot: "
    f"{generation_ready_slot}"
)

print(
    f"A-side age: "
    f"{a_age_before_wait} slots"
)

print(
    f"C-side age: "
    f"{c_age_before_wait} slots"
)


# ============================================================================
# DELIBERATELY WAIT
#
# No new generation.
# No swap.
#
# We only advance global physical simulation time.
# ============================================================================

WAIT_SLOTS_BEFORE_SWAP = 3


age_test_simulator.kernel.advance_slot(
    WAIT_SLOTS_BEFORE_SWAP
)


swap_slot = (
    age_test_simulator.kernel.current_slot
)


assert (
    swap_slot
    == generation_ready_slot
    + WAIT_SLOTS_BEFORE_SWAP
)


print(
    f"\n[INFO] Intentionally delayed swapping by "
    f"{WAIT_SLOTS_BEFORE_SWAP} global slots."
)


# ============================================================================
# Verify resources are still ACTIVE after waiting
# ============================================================================

age_test_simulator\
    .entanglement_manager\
    .synchronize_expirations()


age_left_before_swap = (
    age_test_simulator
    .entanglement_manager
    .get_resource(
        age_left_ep.resource_id
    )
)


age_right_before_swap = (
    age_test_simulator
    .entanglement_manager
    .get_resource(
        age_right_ep.resource_id
    )
)


assert age_left_before_swap.is_active
assert age_right_before_swap.is_active


print(
    "[PASS] Both elementary Bell pairs remain ACTIVE "
    "after the deliberate waiting interval."
)


# ============================================================================
# Their PHYSICAL QUBITS must now be older
# ============================================================================

a_age_immediately_before_swap = (
    age_a_memory.age_slots(
        age_left_ep.resource_id
    )
)


c_age_immediately_before_swap = (
    age_c_memory.age_slots(
        age_right_ep.resource_id
    )
)


assert (
    a_age_immediately_before_swap
    == (
        swap_slot
        - original_a_storage_slot
    )
)


assert (
    c_age_immediately_before_swap
    == (
        swap_slot
        - original_c_storage_slot
    )
)


assert (
    a_age_immediately_before_swap
    >= WAIT_SLOTS_BEFORE_SWAP
)


assert (
    c_age_immediately_before_swap
    >= WAIT_SLOTS_BEFORE_SWAP
)


assert (
    a_age_immediately_before_swap
    > 0
)


assert (
    c_age_immediately_before_swap
    > 0
)


print(
    "[PASS] Outer physical qubits accumulated non-zero "
    "memory age before the BSM."
)


# ============================================================================
# Re-read local halves immediately before swapping
# ============================================================================

a_half_immediately_before_swap = (
    age_a_memory
    .get_local_half(
        age_left_ep.resource_id
    )
)


c_half_immediately_before_swap = (
    age_c_memory
    .get_local_half(
        age_right_ep.resource_id
    )
)


assert (
    a_half_immediately_before_swap.stored_at_slot
    == original_a_storage_slot
)


assert (
    c_half_immediately_before_swap.stored_at_slot
    == original_c_storage_slot
)


print(
    "[PASS] Waiting changes qubit age without changing "
    "its original stored_at_slot."
)


# ============================================================================
# Construct IDEAL swap engine
# ============================================================================

age_swap_engine = (
    EntanglementSwapEngine(

        simulator=age_test_simulator,

        model=(
            IdealEntanglementSwapModel()
        ),

        fidelity_registry=(
            age_fidelity_registry
        )
    )
)


# ============================================================================
# Perform BSM at Age-R1
# ============================================================================

slot_immediately_before_swap_call = (
    age_test_simulator.kernel.current_slot
)


age_swap_attempt = (
    age_swap_engine.attempt_swap(

        left_resource_id=(
            age_left_ep.resource_id
        ),

        right_resource_id=(
            age_right_ep.resource_id
        ),

        swap_node=(
            age_r1
        )
    )
)


slot_immediately_after_swap_call = (
    age_test_simulator.kernel.current_slot
)


# ============================================================================
# Ideal swap itself must not advance simulation time
# ============================================================================

assert (
    slot_immediately_before_swap_call
    == slot_immediately_after_swap_call
)


assert (
    slot_immediately_after_swap_call
    == swap_slot
)


print(
    "[PASS] Ideal atomic BSM does not advance the global "
    "simulation clock."
)


# ============================================================================
# Swap outcome must be deterministic success
# ============================================================================

assert age_swap_attempt.success

assert math.isclose(
    age_swap_attempt.success_probability,
    1.0,
    abs_tol=1e-12
)

assert (
    age_swap_attempt.random_draw
    is None
)


print(
    "[PASS] Ideal BSM succeeds deterministically with "
    "P(swap) = 1."
)


# ============================================================================
# Obtain newly created SWAPPED parent
# ============================================================================

age_parent_ep = (
    age_test_simulator
    .entanglement_manager
    .get_resource(
        age_swap_attempt.output_resource_id
    )
)


assert (
    age_parent_ep.kind
    == EntanglementKind.SWAPPED
)

assert age_parent_ep.is_active


assert (
    age_parent_ep.endpoint_names
    == tuple(
        sorted(
            (
                age_a.name,
                age_c.name
            )
        )
    )
)


print(
    "[PASS] Successful BSM creates remote E(A,C)."
)


# ============================================================================
# CRITICAL DISTINCTION #1
#
# The ENTANGLEMENT RELATIONSHIP is new.
#
# Therefore:
#
#     parent.created_slot == swap slot
# ============================================================================

assert (
    age_parent_ep.created_slot
    == swap_slot
)


assert (
    age_parent_ep.created_slot
    > original_a_storage_slot
)


assert (
    age_parent_ep.created_slot
    > original_c_storage_slot
)


print(
    "[PASS] New swapped entanglement relationship is created "
    "at the later BSM slot."
)


# ============================================================================
# CRITICAL DISTINCTION #2
#
# The OUTER PHYSICAL QUBITS are NOT new.
#
# Their original storage timestamps MUST survive the swap.
# ============================================================================

age_a_parent_half = (
    age_a_memory
    .get_local_half(
        age_parent_ep.resource_id
    )
)


age_c_parent_half = (
    age_c_memory
    .get_local_half(
        age_parent_ep.resource_id
    )
)


assert (
    age_a_parent_half.stored_at_slot
    == original_a_storage_slot
)


assert (
    age_c_parent_half.stored_at_slot
    == original_c_storage_slot
)


assert (
    age_a_parent_half.stored_at_slot
    != age_parent_ep.created_slot
)


assert (
    age_c_parent_half.stored_at_slot
    != age_parent_ep.created_slot
)


print(
    "[PASS] Successful swapping DOES NOT reset surviving "
    "outer-qubit storage timestamps."
)


# ============================================================================
# CRITICAL DISTINCTION #3
#
# Physical qubit ages must remain unchanged across the
# zero-duration ideal BSM.
# ============================================================================

a_age_after_swap = (
    age_a_memory.age_slots(
        age_parent_ep.resource_id
    )
)


c_age_after_swap = (
    age_c_memory.age_slots(
        age_parent_ep.resource_id
    )
)


assert (
    a_age_after_swap
    == a_age_immediately_before_swap
)


assert (
    c_age_after_swap
    == c_age_immediately_before_swap
)


assert (
    a_age_after_swap
    == (
        swap_slot
        - original_a_storage_slot
    )
)


assert (
    c_age_after_swap
    == (
        swap_slot
        - original_c_storage_slot
    )
)


print(
    "[PASS] Physical qubit ages are exactly preserved "
    "through the ideal BSM."
)


# ============================================================================
# Old elementary resource IDs must disappear from OUTER memories
# ============================================================================

assert not age_a_memory.contains(
    age_left_ep.resource_id
)


assert not age_c_memory.contains(
    age_right_ep.resource_id
)


assert age_a_memory.contains(
    age_parent_ep.resource_id
)


assert age_c_memory.contains(
    age_parent_ep.resource_id
)


print(
    "[PASS] Outer memories are rebound from child resource IDs "
    "to the new parent resource ID."
)


# ============================================================================
# Remote-node relationships must be updated correctly
# ============================================================================

assert (
    age_a_parent_half.remote_node_name
    == age_c.name
)


assert (
    age_c_parent_half.remote_node_name
    == age_a.name
)


print(
    "[PASS] Surviving outer halves now correctly reference "
    "each other as remote entanglement endpoints."
)


# ============================================================================
# Both repeater qubits must have been measured/consumed
# ============================================================================

assert (
    age_r1_memory.occupancy
    == 0
)


assert not age_r1_memory.contains(
    age_left_ep.resource_id
)


assert not age_r1_memory.contains(
    age_right_ep.resource_id
)


print(
    "[PASS] Bell-state measurement consumes both local "
    "repeater qubits."
)


# ============================================================================
# Both child resources must be historical CONSUMED resources
# ============================================================================

age_left_after_swap = (
    age_test_simulator
    .entanglement_manager
    .get_resource(
        age_left_ep.resource_id
    )
)


age_right_after_swap = (
    age_test_simulator
    .entanglement_manager
    .get_resource(
        age_right_ep.resource_id
    )
)


assert (
    age_left_after_swap.status
    == EntanglementStatus.CONSUMED
)


assert (
    age_right_after_swap.status
    == EntanglementStatus.CONSUMED
)


assert (
    age_left_after_swap.ended_slot
    == swap_slot
)


assert (
    age_right_after_swap.ended_slot
    == swap_slot
)


print(
    "[PASS] Child entanglement relationships terminate exactly "
    "at the BSM slot."
)


# ============================================================================
# Parent provenance must retain both elementary links
# ============================================================================

assert (
    age_parent_ep.parent_resource_ids
    == (
        age_left_ep.resource_id,
        age_right_ep.resource_id
    )
)


assert set(
    age_parent_ep.source_link_ids
) == set(
    age_left_ep.source_link_ids
    +
    age_right_ep.source_link_ids
)


assert (
    age_parent_ep.swap_node_name
    == age_r1.name
)


print(
    "[PASS] Swapped resource retains correct child-resource "
    "and physical-link provenance."
)


# ============================================================================
# There must STILL be no physical A--C link
# ============================================================================

assert not age_test_topology.contains_link(
    age_a,
    age_c
)


print(
    "[PASS] Remote entanglement does not fabricate a physical "
    "outer-node fibre link."
)


# ============================================================================
# Fidelity remains ideal
# ============================================================================

assert math.isclose(
    age_fidelity_registry.fidelity(
        age_parent_ep
    ),
    1.0,
    abs_tol=1e-12
)


print(
    "[PASS] Ideal delayed swapping still preserves F = 1.0."
)


# ============================================================================
# Memory occupancy after successful swap
# ============================================================================

assert (
    age_a_memory.occupancy
    == 1
)


assert (
    age_r1_memory.occupancy
    == 0
)


assert (
    age_c_memory.occupancy
    == 1
)


print(
    "[PASS] Post-swap physical memory occupancy is "
    "A=1, R1=0, C=1."
)


# ============================================================================
# Global ACTIVE-resource state
#
# There should now be exactly ONE active resource:
#
#     E(A,C)
# ============================================================================

active_after_swap = (
    age_test_simulator
    .entanglement_manager
    .active_resources
)


assert (
    len(active_after_swap)
    == 1
)


assert (
    active_after_swap[0].resource_id
    == age_parent_ep.resource_id
)


print(
    "[PASS] The swapped E(A,C) is the only remaining "
    "ACTIVE entanglement resource."
)


# ============================================================================
# Full cross-layer consistency
# ============================================================================

assert (
    age_test_simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Complete NARAD-QC runtime remains cross-layer "
    "consistent after delayed swapping."
)


# ============================================================================
# Explicit diagnostic summary
# ============================================================================

print("\n" + "-" * 80)
print("AGE-PRESERVATION DIAGNOSTIC")
print("-" * 80)


age_diagnostic = pd.DataFrame([
    {
        "Outer Node":
            age_a.name,

        "Original Resource":
            age_left_ep.resource_id,

        "Original Stored Slot":
            original_a_storage_slot,

        "Swap Slot":
            swap_slot,

        "Parent Resource":
            age_parent_ep.resource_id,

        "Parent Created Slot":
            age_parent_ep.created_slot,

        "Parent Half Stored Slot":
            age_a_parent_half.stored_at_slot,

        "Age Before Swap":
            a_age_immediately_before_swap,

        "Age After Swap":
            a_age_after_swap,

        "Age Preserved":
            (
                a_age_after_swap
                == a_age_immediately_before_swap
            )
    },

    {
        "Outer Node":
            age_c.name,

        "Original Resource":
            age_right_ep.resource_id,

        "Original Stored Slot":
            original_c_storage_slot,

        "Swap Slot":
            swap_slot,

        "Parent Resource":
            age_parent_ep.resource_id,

        "Parent Created Slot":
            age_parent_ep.created_slot,

        "Parent Half Stored Slot":
            age_c_parent_half.stored_at_slot,

        "Age Before Swap":
            c_age_immediately_before_swap,

        "Age After Swap":
            c_age_after_swap,

        "Age Preserved":
            (
                c_age_after_swap
                == c_age_immediately_before_swap
            )
    }
])


display(
    age_diagnostic
)


print("\nRESOURCE HISTORY")

display(
    age_test_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\nFIDELITY HISTORY")

display(
    age_fidelity_registry
    .dataframe()
)


print("\nOUTER A MEMORY")

display(
    age_a_memory
    .slots_dataframe()
)


print("\nSWAP NODE MEMORY")

display(
    age_r1_memory
    .slots_dataframe()
)


print("\nOUTER C MEMORY")

display(
    age_c_memory
    .slots_dataframe()
)


print("\nSWAP ATTEMPT")

display(
    age_swap_engine
    .attempts_dataframe()
)


print("\n" + "=" * 80)
print(
    "Delayed-swap age-preservation verification "
    "completed successfully."
)
print("=" * 80)

In [ ]:
# @title Paper-Inspired Atomic BSM Model

import math
import numbers


# =============================================================
# Paper-Inspired Baseline Parameters
# =============================================================

PHASE3C_DEFAULT_ATOMIC_BSM_SUCCESS_PROBABILITY = 0.40

PHASE3C_DEFAULT_ATOMIC_BSM_LATENCY_US = 10.0


# =============================================================
# Stochastic Atomic BSM Model
# =============================================================

class PaperInspiredAtomicBSMModel(
    EntanglementSwapModel
):
    """
    Stochastic atomic Bell-state-measurement model.

    Default baseline
    ----------------
        p_b = 0.40
        t_b = 10 microseconds

    These values follow the evaluation baseline used in the
    mentor paper and must NOT be interpreted as universal
    quantum-repeater hardware constants.

    Phase 3C.2 models:
        - stochastic BSM success/failure;
        - destructive resource loss on BSM failure.

    Phase 3C.2 does NOT yet model:
        - operation-driven fidelity degradation;
        - memory decoherence;
        - explicit Bell-measurement outcomes;
        - physical-time-to-simulation-slot conversion.

    Successful swaps therefore preserve the current ideal
    fidelity assumption F = 1 for ideal input Bell pairs.
    """

    def __init__(
        self,
        atomic_bsm_success_probability: float = (
            PHASE3C_DEFAULT_ATOMIC_BSM_SUCCESS_PROBABILITY
        ),
        atomic_bsm_latency_us: float = (
            PHASE3C_DEFAULT_ATOMIC_BSM_LATENCY_US
        )
    ):

        # =====================================================
        # Success Probability
        # =====================================================

        if (
            isinstance(
                atomic_bsm_success_probability,
                bool
            )
            or not isinstance(
                atomic_bsm_success_probability,
                numbers.Real
            )
            or not math.isfinite(
                float(
                    atomic_bsm_success_probability
                )
            )
        ):
            raise ValueError(
                "atomic_bsm_success_probability must be "
                "a finite real number."
            )

        atomic_bsm_success_probability = float(
            atomic_bsm_success_probability
        )

        if not (
            0.0
            <= atomic_bsm_success_probability
            <= 1.0
        ):
            raise ValueError(
                "Atomic BSM success probability must "
                "lie within [0, 1]."
            )

        # =====================================================
        # Physical BSM Latency Metadata
        # =====================================================

        if (
            isinstance(
                atomic_bsm_latency_us,
                bool
            )
            or not isinstance(
                atomic_bsm_latency_us,
                numbers.Real
            )
            or not math.isfinite(
                float(
                    atomic_bsm_latency_us
                )
            )
            or float(
                atomic_bsm_latency_us
            ) < 0.0
        ):
            raise ValueError(
                "atomic_bsm_latency_us must be a "
                "non-negative finite real number."
            )

        self._p_b = float(
            atomic_bsm_success_probability
        )

        self._t_b_us = float(
            atomic_bsm_latency_us
        )

    # =========================================================
    # Metadata
    # =========================================================

    @property
    def model_name(
        self
    ) -> str:

        return (
            "PaperInspiredAtomicBSMModel"
            f"(p_b={self._p_b:.3f})"
        )

    @property
    def atomic_bsm_success_probability(
        self
    ) -> float:

        return self._p_b

    @property
    def atomic_bsm_latency_us(
        self
    ) -> float:

        return self._t_b_us

    # =========================================================
    # Swap Success Physics
    # =========================================================

    def success_probability(
        self,
        swap_node: NetworkNode,
        left_resource: EntanglementResource,
        right_resource: EntanglementResource
    ) -> float:

        if not isinstance(
            swap_node,
            NetworkNode
        ):
            raise TypeError(
                "swap_node must be a NetworkNode."
            )

        if not isinstance(
            left_resource,
            EntanglementResource
        ):
            raise TypeError(
                "left_resource must be an "
                "EntanglementResource."
            )

        if not isinstance(
            right_resource,
            EntanglementResource
        ):
            raise TypeError(
                "right_resource must be an "
                "EntanglementResource."
            )

        return self._p_b

    # =========================================================
    # Fidelity
    # =========================================================

    def output_fidelity(
        self,
        left_fidelity: float,
        right_fidelity: float
    ) -> float:
        """
        Phase 3C.2 deliberately isolates stochastic BSM
        success from fidelity degradation.

        Therefore ideal inputs remain ideal on successful BSM.

        Non-ideal input fidelity belongs to the later physical
        fidelity model rather than inventing an unsupported
        transformation here.
        """

        if (
            not math.isclose(
                float(left_fidelity),
                1.0,
                rel_tol=0.0,
                abs_tol=1e-12
            )
            or not math.isclose(
                float(right_fidelity),
                1.0,
                rel_tol=0.0,
                abs_tol=1e-12
            )
        ):
            raise EntanglementSwapModelError(
                "Phase 3C.2 stochastic BSM model currently "
                "expects ideal input fidelities. Physical "
                "fidelity degradation is modeled separately."
            )

        return 1.0

    # =========================================================
    # Inspection
    # =========================================================

    def parameter_summary(
        self
    ) -> dict:

        return {
            "model":
                self.model_name,

            "atomic_bsm_success_probability":
                self._p_b,

            "atomic_bsm_failure_probability":
                1.0 - self._p_b,

            "atomic_bsm_latency_us":
                self._t_b_us,

            "fidelity_model":
                "ideal-preservation baseline"
        }

    def __repr__(
        self
    ) -> str:

        return (
            "PaperInspiredAtomicBSMModel("
            f"p_b={self._p_b:.3f}, "
            f"failure_probability="
            f"{1.0 - self._p_b:.3f}, "
            f"t_b={self._t_b_us:.2f} us"
            ")"
        )

In [ ]:
# @title  Stochastic Atomic BSM End-to-End Verification


print("=" * 80)
print("Stochastic Atomic BSM Verification")
print("=" * 80)


# ============================================================================
# Physical topology
#
# Physical-A ----5 km---- Physical-R ----5 km---- Physical-C
#
# Elementary EPs remain generated using the Phase-3B fibre-survival proxy.
#
# Atomic swapping:
#
#       p_b = 0.40
#
# ============================================================================

physical_swap_topology = NetworkTopology(
    name="Phase3C2-Stochastic-BSM-Network"
)


physical_a = NetworkNode(
    name="Physical-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


physical_r = NetworkNode(
    name="Physical-R",
    x_km=5,
    y_km=0,
    endpoint_capable=False,
    swapping_capable=True,
    memory_capacity=2
)


physical_c = NetworkNode(
    name="Physical-C",
    x_km=10,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


for node in (
    physical_a,
    physical_r,
    physical_c
):

    physical_swap_topology.add_node(
        node
    )


physical_swap_path = (
    physical_a,
    physical_r,
    physical_c
)


for left, right in zip(
    physical_swap_path[:-1],
    physical_swap_path[1:]
):

    physical_swap_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert physical_swap_topology.validate_path(
    physical_swap_path
)


assert not physical_swap_topology.contains_link(
    physical_a,
    physical_c
)


print(
    "[PASS] Physical topology contains only adjacent "
    "elementary links."
)


# ============================================================================
# Assemble simulator
# ============================================================================

physical_swap_simulator = (
    QuantumNetworkSimulator(

        topology=physical_swap_topology,

        simulation_seed=3202402,

        controller_id=(
            "PHASE3C2-CONTROLLER"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        # Large enough to avoid memory expiry interfering
        # with this specific BSM probability verification.
        max_storage_slots=100
    )
)


# ============================================================================
# Mutual device authentication
# ============================================================================

physical_authentication = (
    physical_swap_simulator
    .authenticate_all_nodes()
)


assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in physical_authentication.values()
)


assert physical_swap_simulator.require_quantum_eligibility(
    *physical_swap_path
)


print(
    "[PASS] All devices are mutually authenticated before "
    "any quantum operation."
)


# ============================================================================
# Elementary generation layer
# ============================================================================

physical_generation_service = (
    ElementaryEntanglementService(

        simulator=physical_swap_simulator,

        model=FiberSurvivalProxyModel()
    )
)


physical_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=physical_swap_simulator,

        service=physical_generation_service,

        ordered_path=physical_swap_path,

        demand_active=True
    )
)


# ============================================================================
# Fidelity sidecar
# ============================================================================

physical_fidelity_registry = (
    EntanglementFidelityRegistry(
        physical_swap_simulator
    )
)


# ============================================================================
# Stochastic BSM model
# ============================================================================

physical_bsm_model = (
    PaperInspiredAtomicBSMModel()
)


assert math.isclose(
    physical_bsm_model.atomic_bsm_success_probability,
    0.40,
    abs_tol=1e-12
)


assert math.isclose(
    physical_bsm_model.atomic_bsm_latency_us,
    10.0,
    abs_tol=1e-12
)


print(
    "[PASS] Physical BSM model uses paper-inspired "
    "p_b = 0.40 and t_b = 10 us."
)


# ============================================================================
# Swap engine
# ============================================================================

physical_swap_engine = (
    EntanglementSwapEngine(

        simulator=physical_swap_simulator,

        model=physical_bsm_model,

        fidelity_registry=(
            physical_fidelity_registry
        )
    )
)


# ============================================================================
# Helper:
#
# Obtain exactly two fresh adjacent elementary resources.
# ============================================================================

def _phase3c2_prepare_fresh_children():

    resources = (
        physical_generation_scheduler
        .run_until_all_ready(
            max_slots=500
        )
    )

    assert len(resources) == 2

    left_resource = resources[0]
    right_resource = resources[1]

    assert left_resource.is_active
    assert right_resource.is_active

    assert (
        left_resource.kind
        == EntanglementKind.ELEMENTARY
    )

    assert (
        right_resource.kind
        == EntanglementKind.ELEMENTARY
    )

    # ---------------------------------------------------------
    # Register ideal elementary fidelities if these are freshly
    # generated resources that the fidelity registry has not
    # yet seen.
    # ---------------------------------------------------------

    for resource in resources:

        try:

            physical_fidelity_registry.get(
                resource
            )

        except KeyError:

            physical_fidelity_registry.register_ideal(
                resource
            )

    return (
        left_resource,
        right_resource
    )


# ============================================================================
# We deliberately keep executing COMPLETE physical cycles
# until BOTH stochastic outcomes have actually occurred.
#
# With:
#
#       P(success) = 0.4
#       P(failure) = 0.6
#
# observing both within 50 trials should be overwhelmingly likely.
# ============================================================================

observed_success = False
observed_failure = False

first_success_attempt = None
first_failure_attempt = None

MAX_BSM_CYCLES = 50


for bsm_cycle in range(
    MAX_BSM_CYCLES
):

    # ========================================================================
    # Fresh adjacent Bell pairs
    # ========================================================================

    child_left, child_right = (
        _phase3c2_prepare_fresh_children()
    )


    assert (
        physical_r
        in physical_swap_path
    )


    # ========================================================================
    # Both repeater memory slots should currently be occupied
    # ========================================================================

    assert (
        physical_swap_simulator
        .quantum_memory(
            physical_r
        )
        .occupancy
        == 2
    )


    # ========================================================================
    # Snapshot BSM slot
    # ========================================================================

    bsm_slot_before = (
        physical_swap_simulator
        .kernel.current_slot
    )


    # ========================================================================
    # REAL stochastic atomic BSM
    # ========================================================================

    stochastic_attempt = (
        physical_swap_engine
        .attempt_swap(

            left_resource_id=(
                child_left.resource_id
            ),

            right_resource_id=(
                child_right.resource_id
            ),

            swap_node=(
                physical_r
            )
        )
    )


    bsm_slot_after = (
        physical_swap_simulator
        .kernel.current_slot
    )


    # ========================================================================
    # BSM itself still does NOT redefine global slot duration
    # ========================================================================

    assert (
        bsm_slot_before
        == bsm_slot_after
    )


    assert (
        stochastic_attempt.simulation_slot
        == bsm_slot_before
    )


    # ========================================================================
    # For 0 < p_b < 1, a real simulation RNG draw MUST occur
    # ========================================================================

    assert (
        stochastic_attempt.random_draw
        is not None
    )


    assert (
        0.0
        <= stochastic_attempt.random_draw
        < 1.0
    )


    # ========================================================================
    # The stochastic decision must exactly follow:
    #
    #       success <=> random draw < p_b
    # ========================================================================

    assert (
        stochastic_attempt.success
        ==
        (
            stochastic_attempt.random_draw
            <
            physical_bsm_model
            .atomic_bsm_success_probability
        )
    )


    assert math.isclose(
        stochastic_attempt.success_probability,
        0.40,
        abs_tol=1e-12
    )


    # ========================================================================
    # Regardless of outcome, BOTH child relationships were consumed
    # by the attempted BSM.
    # ========================================================================

    child_left_after = (
        physical_swap_simulator
        .entanglement_manager
        .get_resource(
            child_left.resource_id
        )
    )


    child_right_after = (
        physical_swap_simulator
        .entanglement_manager
        .get_resource(
            child_right.resource_id
        )
    )


    assert (
        child_left_after.status
        == EntanglementStatus.CONSUMED
    )


    assert (
        child_right_after.status
        == EntanglementStatus.CONSUMED
    )


    assert (
        child_left_after.ended_slot
        == bsm_slot_before
    )


    assert (
        child_right_after.ended_slot
        == bsm_slot_before
    )


    # ========================================================================
    # Repeater BSM qubits are always consumed.
    # ========================================================================

    assert (
        physical_swap_simulator
        .quantum_memory(
            physical_r
        )
        .occupancy
        == 0
    )


    # ========================================================================
    # SUCCESS BRANCH
    # ========================================================================

    if stochastic_attempt.success:

        observed_success = True

        if first_success_attempt is None:

            first_success_attempt = (
                stochastic_attempt
            )


        assert (
            stochastic_attempt.output_resource_id
            is not None
        )


        parent_resource = (
            physical_swap_simulator
            .entanglement_manager
            .get_resource(
                stochastic_attempt.output_resource_id
            )
        )


        assert (
            parent_resource.kind
            == EntanglementKind.SWAPPED
        )


        assert parent_resource.is_active


        assert (
            parent_resource.endpoint_names
            == tuple(
                sorted(
                    (
                        physical_a.name,
                        physical_c.name
                    )
                )
            )
        )


        assert (
            physical_swap_simulator
            .quantum_memory(
                physical_a
            )
            .contains(
                parent_resource.resource_id
            )
        )


        assert (
            physical_swap_simulator
            .quantum_memory(
                physical_c
            )
            .contains(
                parent_resource.resource_id
            )
        )


        # Fidelity is intentionally still ideal in 3C.2.
        assert math.isclose(
            physical_fidelity_registry
            .fidelity(
                parent_resource
            ),
            1.0,
            abs_tol=1e-12
        )


        # -----------------------------------------------------
        # Verification cleanup only.
        #
        # Remove the successful remote parent so the outer
        # memories are available for the next independent
        # physical BSM trial.
        # -----------------------------------------------------

        physical_swap_simulator\
            .entanglement_manager\
            .discard_resource(

                parent_resource.resource_id,

                reason=(
                    "phase3c2_repeated_trial_cleanup"
                )
            )


        assert (
            physical_swap_simulator
            .quantum_memory(
                physical_a
            )
            .occupancy
            == 0
        )


        assert (
            physical_swap_simulator
            .quantum_memory(
                physical_c
            )
            .occupancy
            == 0
        )


    # ========================================================================
    # FAILURE BRANCH
    # ========================================================================

    else:

        observed_failure = True

        if first_failure_attempt is None:

            first_failure_attempt = (
                stochastic_attempt
            )


        # -----------------------------------------------------
        # Failed BSM must create NO parent.
        # -----------------------------------------------------

        assert (
            stochastic_attempt.output_resource_id
            is None
        )


        # -----------------------------------------------------
        # Outer halves no longer constitute usable entanglement
        # after failed BSM, so they must also have been released.
        # -----------------------------------------------------

        assert (
            physical_swap_simulator
            .quantum_memory(
                physical_a
            )
            .occupancy
            == 0
        )


        assert (
            physical_swap_simulator
            .quantum_memory(
                physical_c
            )
            .occupancy
            == 0
        )


        # -----------------------------------------------------
        # There must be NO new active swapped E(A,C).
        # -----------------------------------------------------

        assert not any(

            (
                resource.kind
                == EntanglementKind.SWAPPED

                and resource.is_active

                and resource.endpoint_names
                == tuple(
                    sorted(
                        (
                            physical_a.name,
                            physical_c.name
                        )
                    )
                )
            )

            for resource in (
                physical_swap_simulator
                .entanglement_manager
                .resources
            )
        )


    # ========================================================================
    # IMPORTANT ROUND BOUNDARY
    #
    # The elementary lifecycle processes still remember their
    # child resource IDs.
    #
    # They must FIRST observe that those resources were consumed.
    #
    # No immediate same-round regeneration is allowed.
    # ========================================================================

    physical_attempts_before_detection = (
        physical_generation_service
        .statistics.total_attempts
    )


    termination_detection_slot = (
        physical_generation_scheduler
        .process_current_slot()
    )


    assert (
        termination_detection_slot
        .physical_attempt_count
        == 0
    )


    assert all(

        step.action
        == (
            ElementaryLinkStepAction
            .EXTERNAL_TERMINATION_DETECTED
        )

        for step in (
            termination_detection_slot.steps
        )
    )


    assert (
        physical_generation_service
        .statistics.total_attempts
        == physical_attempts_before_detection
    )


    # ========================================================================
    # Stop once both stochastic branches have genuinely occurred.
    # ========================================================================

    if (
        observed_success
        and observed_failure
    ):

        break


    # ========================================================================
    # Move to the NEXT logical global slot before regeneration.
    # ========================================================================

    physical_swap_simulator\
        .kernel\
        .advance_slot()


# ============================================================================
# Both outcomes must genuinely have been observed
# ============================================================================

assert observed_success

assert observed_failure

assert (
    first_success_attempt
    is not None
)

assert (
    first_failure_attempt
    is not None
)


print(
    "[PASS] Real stochastic execution produced both "
    "atomic-BSM success and atomic-BSM failure."
)


# ============================================================================
# Failure draw must belong to the failure region
# ============================================================================

assert (
    first_failure_attempt.random_draw
    >= 0.40
)


assert not (
    first_failure_attempt.success
)


assert (
    first_failure_attempt.output_resource_id
    is None
)


print(
    "[PASS] Observed BSM failure satisfies random_draw >= p_b "
    "and creates no parent resource."
)


# ============================================================================
# Success draw must belong to the success region
# ============================================================================

assert (
    first_success_attempt.random_draw
    < 0.40
)


assert (
    first_success_attempt.success
)


assert (
    first_success_attempt.output_resource_id
    is not None
)


print(
    "[PASS] Observed BSM success satisfies random_draw < p_b "
    "and creates a swapped parent."
)


# ============================================================================
# Every BSM attempt must satisfy the stochastic decision law
# ============================================================================

for attempt in (
    physical_swap_engine.attempts
):

    assert (
        attempt.random_draw
        is not None
    )

    assert (
        attempt.success
        ==
        (
            attempt.random_draw
            <
            attempt.success_probability
        )
    )


print(
    "[PASS] Every physical BSM attempt obeys "
    "success iff random_draw < p_b."
)


# ============================================================================
# No BSM operation itself advanced global simulation time.
#
# Slot advancement occurred only at explicit round boundaries.
# ============================================================================

assert all(
    attempt.simulation_slot >= 0
    for attempt
    in physical_swap_engine.attempts
)


print(
    "[PASS] Atomic BSM physics remains separate from "
    "discrete simulation-clock advancement."
)


# ============================================================================
# Complete resource consistency
# ============================================================================

assert (
    physical_swap_simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Runtime remains cross-layer consistent through "
    "stochastic BSM successes, failures, and resource loss."
)


# ============================================================================
# Diagnostic summary
# ============================================================================

print("\nPHYSICAL BSM MODEL")
print(
    physical_bsm_model
)


print("\nMODEL PARAMETERS")

print(
    physical_bsm_model
    .parameter_summary()
)


print("\nALL STOCHASTIC BSM ATTEMPTS")

display(
    physical_swap_engine
    .attempts_dataframe()
)


print("\nENTANGLEMENT RESOURCE HISTORY")

display(
    physical_swap_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\nELEMENTARY GENERATION SLOT HISTORY")

display(
    physical_generation_scheduler
    .slot_history_dataframe()
)


print("\n" + "=" * 80)
print(
    "Stochastic atomic BSM end-to-end verification "
    "completed successfully."
)
print("=" * 80)

In [ ]:
# @title Atomic BSM Probability Statistical Verification


print("=" * 80)
print("Atomic BSM Probability Statistical Verification")
print("=" * 80)


# ============================================================================
# Independent seeded physical RNG
#
# This is simulation randomness — correctly SimulationKernel.rng,
# NOT cryptographic randomness.
# ============================================================================

bsm_probability_kernel = (
    SimulationKernel(
        seed=320240200
    )
)


bsm_probability_model = (
    PaperInspiredAtomicBSMModel()
)


p_b = (
    bsm_probability_model
    .atomic_bsm_success_probability
)


assert math.isclose(
    p_b,
    0.40,
    abs_tol=1e-12
)


# ============================================================================
# Monte Carlo BSM decisions
# ============================================================================

N_BSM_MONTE_CARLO = 10000


bsm_draws = (
    bsm_probability_kernel
    .rng
    .random(
        N_BSM_MONTE_CARLO
    )
)


bsm_successes = (
    bsm_draws
    < p_b
)


observed_successes = int(
    bsm_successes.sum()
)


observed_failures = (
    N_BSM_MONTE_CARLO
    - observed_successes
)


empirical_p_b = (
    observed_successes
    / N_BSM_MONTE_CARLO
)


# ============================================================================
# Binomial sampling uncertainty
#
# sigma = sqrt(p(1-p)/N)
#
# We use a generous six-sigma acceptance window because this
# is a simulator sanity check, not a formal hypothesis test.
# ============================================================================

binomial_sigma = math.sqrt(
    (
        p_b
        * (1.0 - p_b)
    )
    / N_BSM_MONTE_CARLO
)


acceptance_tolerance = (
    6.0
    * binomial_sigma
)


assert (
    abs(
        empirical_p_b
        - p_b
    )
    <= acceptance_tolerance
)


print(
    "[PASS] Empirical atomic-BSM success rate is "
    "statistically consistent with p_b = 0.40."
)


# ============================================================================
# Failure rate must correspondingly approach 0.60
# ============================================================================

empirical_failure_probability = (
    observed_failures
    / N_BSM_MONTE_CARLO
)


assert math.isclose(
    empirical_failure_probability,
    1.0 - empirical_p_b,
    abs_tol=1e-12
)


assert (
    abs(
        empirical_failure_probability
        - 0.60
    )
    <= acceptance_tolerance
)


print(
    "[PASS] Empirical destructive-BSM failure rate is "
    "statistically consistent with 0.60."
)


# ============================================================================
# Diagnostics
# ============================================================================

bsm_probability_summary = pd.DataFrame([
    {
        "Trials":
            N_BSM_MONTE_CARLO,

        "Target P(success)":
            p_b,

        "Observed Successes":
            observed_successes,

        "Observed Failures":
            observed_failures,

        "Empirical P(success)":
            empirical_p_b,

        "Empirical P(failure)":
            empirical_failure_probability,

        "Binomial Sigma":
            binomial_sigma,

        "Acceptance Tolerance":
            acceptance_tolerance,

        "Paper BSM Latency (us)":
            bsm_probability_model
            .atomic_bsm_latency_us
    }
])


display(
    bsm_probability_summary
)


print("\n" + "=" * 80)
print(
    "Atomic BSM statistical verification "
    "completed successfully."
)
print("=" * 80)

In [ ]:
# @title  Parallel Swap Candidate Representation

from dataclasses import dataclass
from typing import Optional
import itertools


# =============================================================
# Control-Plane Swap Scheduling Errors
# =============================================================

class ParallelSwapSchedulingError(Exception):
    """
    Base error for control-plane parallel swap scheduling.
    """
    pass


class SwapCandidateConsistencyError(
    ParallelSwapSchedulingError
):
    pass


class SwapSelectionConflictError(
    ParallelSwapSchedulingError
):
    pass


# =============================================================
# Candidate
# =============================================================

@dataclass(frozen=True)
class SwapCandidate:
    """
    One currently valid control-plane opportunity for a BSM.

    Example:

        E(A,R2) + E(R2,R4)

    produces a candidate:

        swap_node = R2
        outer_left = A
        outer_right = R4

    No quantum operation has occurred merely because this
    candidate exists.
    """

    candidate_id: str

    discovered_slot: int

    swap_node_name: str
    swap_node_path_index: int

    left_resource_id: str
    right_resource_id: str

    outer_left_name: str
    outer_right_name: str

    left_span_indices: tuple[int, int]
    right_span_indices: tuple[int, int]
    combined_span_indices: tuple[int, int]

    left_source_link_ids: tuple[str, ...]
    right_source_link_ids: tuple[str, ...]

    oldest_input_half_age_slots: int

    # =========================================================
    # Resources Used by Candidate
    # =========================================================

    @property
    def input_resource_ids(
        self
    ) -> tuple[str, str]:

        return (
            self.left_resource_id,
            self.right_resource_id
        )

    @property
    def combined_span_hops(
        self
    ) -> int:

        return (
            self.combined_span_indices[1]
            - self.combined_span_indices[0]
        )

    # =========================================================
    # Conflict Definition
    # =========================================================

    def conflicts_with(
        self,
        other: "SwapCandidate"
    ) -> bool:
        """
        Two swaps cannot execute in the same parallel round if:

        1. they consume any common entanglement resource, OR
        2. they request two BSMs at the same repeater.

        Sharing an OUTER endpoint is allowed if distinct local
        qubits/resources are involved.
        """

        if not isinstance(
            other,
            SwapCandidate
        ):
            raise TypeError(
                "other must be a SwapCandidate."
            )

        if (
            self.swap_node_name
            == other.swap_node_name
        ):
            return True

        return bool(
            set(self.input_resource_ids)
            &
            set(other.input_resource_ids)
        )

In [ ]:
# @title  Max-Parallel Oldest-First Swap Policy


class MaxParallelOldestFirstSwapPolicy:
    """
    Control-plane policy for choosing a conflict-free BSM set.

    Optimization priorities
    -----------------------
    1. Maximum number of swaps in the current round.
    2. Among equal-cardinality solutions, maximize total
       oldest-input-qubit age.
    3. Deterministic left-to-right tie-breaking.

    For Phase 3C.3 we use an exact subset search because the
    candidate sets are small.

    This is a baseline control policy, not a claim of globally
    optimal quantum-network scheduling.
    """

    @property
    def policy_name(
        self
    ) -> str:

        return "MAX_PARALLEL_OLDEST_FIRST"

    # =========================================================
    # Selection
    # =========================================================

    def select(
        self,
        candidates
    ) -> tuple[SwapCandidate, ...]:

        candidates = tuple(
            candidates
        )

        if not candidates:
            return ()

        for candidate in candidates:

            if not isinstance(
                candidate,
                SwapCandidate
            ):
                raise TypeError(
                    "All candidates must be SwapCandidate objects."
                )

        # Deterministic search order.
        ordered_candidates = tuple(
            sorted(
                candidates,
                key=lambda c: (
                    c.swap_node_path_index,
                    c.candidate_id
                )
            )
        )

        best_selection = ()

        # =====================================================
        # Selection Comparison
        # =====================================================

        def is_better(
            proposed,
            current_best
        ):

            # -------------------------------------------------
            # Primary objective:
            # maximize number of simultaneous BSMs.
            # -------------------------------------------------

            if (
                len(proposed)
                != len(current_best)
            ):
                return (
                    len(proposed)
                    > len(current_best)
                )

            # -------------------------------------------------
            # Secondary objective:
            # prioritize older input physical qubits.
            # -------------------------------------------------

            proposed_age_score = sum(
                candidate.oldest_input_half_age_slots
                for candidate in proposed
            )

            best_age_score = sum(
                candidate.oldest_input_half_age_slots
                for candidate in current_best
            )

            if (
                proposed_age_score
                != best_age_score
            ):
                return (
                    proposed_age_score
                    > best_age_score
                )

            # -------------------------------------------------
            # Deterministic final tie-break.
            # -------------------------------------------------

            proposed_positions = tuple(
                candidate.swap_node_path_index
                for candidate in proposed
            )

            best_positions = tuple(
                candidate.swap_node_path_index
                for candidate in current_best
            )

            return (
                proposed_positions
                < best_positions
            )

        # =====================================================
        # Exact Conflict-Free Search
        # =====================================================

        def search(
            index,
            chosen
        ):

            nonlocal best_selection

            if index >= len(
                ordered_candidates
            ):

                proposed = tuple(
                    chosen
                )

                if is_better(
                    proposed,
                    best_selection
                ):
                    best_selection = proposed

                return

            candidate = (
                ordered_candidates[index]
            )

            # -------------------------------------------------
            # Option 1 — skip candidate
            # -------------------------------------------------

            search(
                index + 1,
                chosen
            )

            # -------------------------------------------------
            # Option 2 — select if conflict-free
            # -------------------------------------------------

            if all(
                not candidate.conflicts_with(
                    selected
                )
                for selected in chosen
            ):

                chosen.append(
                    candidate
                )

                search(
                    index + 1,
                    chosen
                )

                chosen.pop()

        search(
            0,
            []
        )

        return tuple(
            sorted(
                best_selection,
                key=lambda c: (
                    c.swap_node_path_index,
                    c.candidate_id
                )
            )
        )

    def __repr__(
        self
    ) -> str:

        return (
            "MaxParallelOldestFirstSwapPolicy("
            "objective=max_parallelism_then_age"
            ")"
        )

In [ ]:
# @title Parallel Swap Round Record


@dataclass(frozen=True)
class ParallelSwapRound:
    """
    Immutable audit record for one control-plane swap round.

    Candidate discovery and selection are performed against a
    snapshot of resources that existed BEFORE the round began.

    Newly created resources cannot become inputs until a later
    round.
    """

    round_index: int

    simulation_slot: int

    policy_name: str

    snapshot_resource_ids: tuple[str, ...]

    discovered_candidates: tuple[
        SwapCandidate, ...
    ]

    selected_candidates: tuple[
        SwapCandidate, ...
    ]

    attempts: tuple[
        EntanglementSwapAttempt, ...
    ]

    output_resource_ids: tuple[str, ...]

    def __post_init__(self):

        # =====================================================
        # Every selected candidate must have existed in the
        # discovered candidate set.
        # =====================================================

        discovered_ids = {
            candidate.candidate_id
            for candidate
            in self.discovered_candidates
        }

        for candidate in self.selected_candidates:

            if (
                candidate.candidate_id
                not in discovered_ids
            ):
                raise ValueError(
                    "Selected swap candidate was not part "
                    "of the round-start candidate snapshot."
                )

        # =====================================================
        # Selected candidates must be pairwise conflict-free.
        # =====================================================

        for left_index in range(
            len(self.selected_candidates)
        ):

            for right_index in range(
                left_index + 1,
                len(self.selected_candidates)
            ):

                if (
                    self.selected_candidates[
                        left_index
                    ]
                    .conflicts_with(
                        self.selected_candidates[
                            right_index
                        ]
                    )
                ):
                    raise SwapSelectionConflictError(
                        "Parallel swap round contains "
                        "conflicting candidates."
                    )

        # =====================================================
        # All BSM attempts belong to same simulation slot.
        # =====================================================

        for attempt in self.attempts:

            if (
                attempt.simulation_slot
                != self.simulation_slot
            ):
                raise ValueError(
                    "All BSM attempts in one parallel round "
                    "must share the same simulation slot."
                )

    # =========================================================
    # Metrics
    # =========================================================

    @property
    def candidate_count(
        self
    ) -> int:

        return len(
            self.discovered_candidates
        )

    @property
    def selected_count(
        self
    ) -> int:

        return len(
            self.selected_candidates
        )

    @property
    def success_count(
        self
    ) -> int:

        return sum(
            attempt.success
            for attempt in self.attempts
        )

    @property
    def failure_count(
        self
    ) -> int:

        return (
            len(self.attempts)
            - self.success_count
        )

In [ ]:
# @title  Control-Plane Parallel Swap Candidate Scheduler


class ControlPlaneParallelSwapScheduler:
    """
    Control-plane component responsible for observing live
    entanglement state and commanding conflict-free BSM batches.

    Architectural boundary
    ----------------------
    CONTROL PLANE:
        decides WHICH valid swaps should be attempted.

    PHYSICAL SWAP ENGINE:
        decides WHETHER each commanded BSM succeeds.

    Candidate discovery always uses LIVE ACTIVE resources from
    EntanglementManager.

    Historical generation records are never used as proof that
    a resource still exists.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        swap_engine: EntanglementSwapEngine,
        ordered_path,
        policy: Optional[
            MaxParallelOldestFirstSwapPolicy
        ] = None
    ):

        # =====================================================
        # Core Validation
        # =====================================================

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            swap_engine,
            EntanglementSwapEngine
        ):
            raise TypeError(
                "swap_engine must be an EntanglementSwapEngine."
            )

        if (
            swap_engine.simulator
            is not simulator
        ):
            raise ValueError(
                "Swap engine must belong to the same "
                "QuantumNetworkSimulator."
            )

        self.simulator = simulator

        self.control_plane = (
            simulator.control_plane
        )

        self.swap_engine = (
            swap_engine
        )

        self.policy = (
            policy
            if policy is not None
            else MaxParallelOldestFirstSwapPolicy()
        )

        # =====================================================
        # Ordered Physical Path
        # =====================================================

        self.ordered_path = (
            simulator.topology
            .validate_path(
                ordered_path
            )
        )

        self.path_links = (
            simulator.topology
            .path_links(
                self.ordered_path
            )
        )

        self._node_index = {
            node.name: index
            for index, node
            in enumerate(
                self.ordered_path
            )
        }

        self._round_history: list[
            ParallelSwapRound
        ] = []

    # =========================================================
    # Resource Span Resolution
    # =========================================================

    def _resource_span(
        self,
        resource: EntanglementResource
    ) -> tuple[int, int]:
        """
        Validate that one ACTIVE resource represents one
        contiguous entangled span over this ordered path.

        Returns:
            (left_path_index, right_path_index)
        """

        if not isinstance(
            resource,
            EntanglementResource
        ):
            raise TypeError(
                "resource must be an EntanglementResource."
            )

        endpoint_a = (
            resource.endpoint_a_name
        )

        endpoint_b = (
            resource.endpoint_b_name
        )

        if (
            endpoint_a not in self._node_index
            or endpoint_b not in self._node_index
        ):
            raise SwapCandidateConsistencyError(
                f"Resource '{resource.resource_id}' has "
                f"an endpoint outside this ordered path."
            )

        index_a = (
            self._node_index[
                endpoint_a
            ]
        )

        index_b = (
            self._node_index[
                endpoint_b
            ]
        )

        left_index = min(
            index_a,
            index_b
        )

        right_index = max(
            index_a,
            index_b
        )

        if (
            right_index
            <= left_index
        ):
            raise SwapCandidateConsistencyError(
                "Entanglement resource has invalid path span."
            )

        expected_link_ids = tuple(
            link.link_id
            for link in (
                self.path_links[
                    left_index:right_index
                ]
            )
        )

        # A valid path resource must represent exactly the
        # physical links between its endpoints.
        if (
            len(resource.source_link_ids)
            != len(expected_link_ids)
            or set(resource.source_link_ids)
            != set(expected_link_ids)
        ):
            raise SwapCandidateConsistencyError(
                f"Resource '{resource.resource_id}' does "
                f"not represent a contiguous physical span "
                f"of this ordered path."
            )

        return (
            left_index,
            right_index
        )

    # =========================================================
    # Physical-Qubit Age Inspection
    # =========================================================

    def _oldest_half_age(
        self,
        resource: EntanglementResource
    ) -> int:
        """
        Return the age of the oldest surviving physical half
        associated with one ACTIVE entanglement resource.
        """

        ages = []

        for endpoint_name in (
            resource.endpoint_names
        ):

            memory = (
                self.simulator
                .quantum_memory(
                    endpoint_name
                )
            )

            ages.append(
                int(
                    memory.age_slots(
                        resource.resource_id
                    )
                )
            )

        return max(
            ages
        )

    # =========================================================
    # Candidate Construction
    # =========================================================

    def _build_candidate(
        self,
        resource_a: EntanglementResource,
        resource_b: EntanglementResource
    ) -> Optional[SwapCandidate]:

        # =====================================================
        # Must share exactly one entanglement endpoint.
        # =====================================================

        shared_names = (
            set(
                resource_a.endpoint_names
            )
            &
            set(
                resource_b.endpoint_names
            )
        )

        if len(
            shared_names
        ) != 1:

            return None

        swap_node_name = next(
            iter(
                shared_names
            )
        )

        swap_node = (
            self.simulator.topology
            .get_node(
                swap_node_name
            )
        )

        # =====================================================
        # Shared node must physically support swapping.
        # =====================================================

        if not swap_node.swapping_capable:
            return None

        # =====================================================
        # Resolve contiguous path spans.
        # =====================================================

        span_a = (
            self._resource_span(
                resource_a
            )
        )

        span_b = (
            self._resource_span(
                resource_b
            )
        )

        swap_index = (
            self._node_index[
                swap_node_name
            ]
        )

        # =====================================================
        # Resources must occupy opposite sides of swap node.
        #
        # Valid:
        #
        #     [i ---- k] + [k ---- j]
        #
        # Invalid overlap:
        #
        #     [i ---------- j]
        #          [k --- j]
        # =====================================================

        if (
            span_a[1]
            == swap_index
            and span_b[0]
            == swap_index
        ):

            left_resource = (
                resource_a
            )

            right_resource = (
                resource_b
            )

            left_span = span_a
            right_span = span_b

        elif (
            span_b[1]
            == swap_index
            and span_a[0]
            == swap_index
        ):

            left_resource = (
                resource_b
            )

            right_resource = (
                resource_a
            )

            left_span = span_b
            right_span = span_a

        else:

            return None

        # =====================================================
        # No physical-link provenance overlap.
        # =====================================================

        if (
            set(
                left_resource.source_link_ids
            )
            &
            set(
                right_resource.source_link_ids
            )
        ):
            return None

        outer_left_name = (
            self.ordered_path[
                left_span[0]
            ].name
        )

        outer_right_name = (
            self.ordered_path[
                right_span[1]
            ].name
        )

        # =====================================================
        # Authorization Gate
        #
        # Candidate exists physically, but control plane will
        # not schedule it unless all participating devices are
        # currently authenticated.
        # =====================================================

        participating_nodes = (
            outer_left_name,
            swap_node_name,
            outer_right_name
        )

        if not all(
            self.simulator
            .is_quantum_eligible(
                node_name
            )
            for node_name
            in participating_nodes
        ):
            return None

        # =====================================================
        # Age/Urgency
        # =====================================================

        oldest_age = max(
            self._oldest_half_age(
                left_resource
            ),
            self._oldest_half_age(
                right_resource
            )
        )

        candidate_id = (
            f"SWAP::{swap_node_name}::"
            f"{left_resource.resource_id}::"
            f"{right_resource.resource_id}"
        )

        return SwapCandidate(

            candidate_id=(
                candidate_id
            ),

            discovered_slot=(
                self.simulator.kernel.current_slot
            ),

            swap_node_name=(
                swap_node_name
            ),

            swap_node_path_index=(
                swap_index
            ),

            left_resource_id=(
                left_resource.resource_id
            ),

            right_resource_id=(
                right_resource.resource_id
            ),

            outer_left_name=(
                outer_left_name
            ),

            outer_right_name=(
                outer_right_name
            ),

            left_span_indices=(
                left_span
            ),

            right_span_indices=(
                right_span
            ),

            combined_span_indices=(
                left_span[0],
                right_span[1]
            ),

            left_source_link_ids=(
                left_resource.source_link_ids
            ),

            right_source_link_ids=(
                right_resource.source_link_ids
            ),

            oldest_input_half_age_slots=(
                oldest_age
            )
        )

    # =========================================================
    # Candidate Discovery
    # =========================================================

    def discover_candidates(
        self,
        resource_snapshot=None
    ) -> tuple[
        SwapCandidate, ...
    ]:
        """
        Discover all currently valid swap opportunities from
        LIVE ACTIVE entanglement resources.

        If resource_snapshot is supplied, discovery is frozen
        to that explicit round-start snapshot.
        """

        self.simulator.validate_runtime_consistency()

        self.simulator\
            .entanglement_manager\
            .synchronize_expirations()

        if resource_snapshot is None:

            resources = tuple(
                self.simulator
                .entanglement_manager
                .active_resources
            )

        else:

            resources = tuple(
                resource_snapshot
            )

        # Only active resources can be candidates.
        resources = tuple(
            resource
            for resource in resources
            if resource.is_active
        )

        candidates = []

        for (
            resource_a,
            resource_b
        ) in itertools.combinations(
            resources,
            2
        ):

            candidate = (
                self._build_candidate(
                    resource_a,
                    resource_b
                )
            )

            if candidate is not None:

                candidates.append(
                    candidate
                )

        return tuple(
            sorted(
                candidates,
                key=lambda candidate: (
                    candidate.swap_node_path_index,
                    candidate.left_resource_id,
                    candidate.right_resource_id
                )
            )
        )

    # =========================================================
    # Candidate Selection
    # =========================================================

    def select_parallel_candidates(
        self,
        candidates
    ) -> tuple[
        SwapCandidate, ...
    ]:

        selected = (
            self.policy.select(
                candidates
            )
        )

        # Defensive conflict verification.
        for left_index in range(
            len(selected)
        ):

            for right_index in range(
                left_index + 1,
                len(selected)
            ):

                if (
                    selected[
                        left_index
                    ]
                    .conflicts_with(
                        selected[
                            right_index
                        ]
                    )
                ):

                    raise SwapSelectionConflictError(
                        "Swap-selection policy returned "
                        "conflicting BSM commands."
                    )

        return selected

    # =========================================================
    # Execute ONE Parallel Swap Round
    # =========================================================

    def execute_parallel_round(
        self
    ) -> ParallelSwapRound:
        """
        Execute one complete control-plane swap round.

        Round ordering
        --------------
        1. Synchronize expiry.
        2. Snapshot active resources.
        3. Discover valid candidates from that snapshot.
        4. Select conflict-free commands.
        5. Execute selected BSMs.
        6. Do NOT rediscover candidates inside this round.
        7. Do NOT advance the global simulation clock.

        Consequently, newly generated parent resources are
        unavailable until the next swap round.
        """

        self.simulator.validate_runtime_consistency()

        self.simulator\
            .entanglement_manager\
            .synchronize_expirations()

        round_slot = (
            self.simulator.kernel.current_slot
        )

        # =====================================================
        # Immutable conceptual round-start snapshot
        # =====================================================

        snapshot_resources = tuple(
            self.simulator
            .entanglement_manager
            .active_resources
        )

        snapshot_resource_ids = tuple(
            resource.resource_id
            for resource
            in snapshot_resources
        )

        # =====================================================
        # Control-Plane Observation
        # =====================================================

        candidates = (
            self.discover_candidates(
                snapshot_resources
            )
        )

        # =====================================================
        # Control-Plane Decision
        # =====================================================

        selected = (
            self.select_parallel_candidates(
                candidates
            )
        )

        # =====================================================
        # Physical BSM Execution
        #
        # Python calls occur sequentially for deterministic
        # simulation execution, but every BSM belongs to the
        # same logical swap round and same simulation slot.
        # =====================================================

        attempts = []

        for candidate in selected:

            # Inputs MUST have existed before round start.
            assert (
                candidate.left_resource_id
                in snapshot_resource_ids
            )

            assert (
                candidate.right_resource_id
                in snapshot_resource_ids
            )

            attempt = (
                self.swap_engine
                .attempt_swap(

                    left_resource_id=(
                        candidate.left_resource_id
                    ),

                    right_resource_id=(
                        candidate.right_resource_id
                    ),

                    swap_node=(
                        candidate.swap_node_name
                    )
                )
            )

            if (
                attempt.simulation_slot
                != round_slot
            ):
                raise RuntimeError(
                    "Swap engine changed global simulation "
                    "time inside a parallel round."
                )

            attempts.append(
                attempt
            )

        # =====================================================
        # Clock Must Remain Unchanged
        # =====================================================

        if (
            self.simulator.kernel.current_slot
            != round_slot
        ):
            raise RuntimeError(
                "Parallel swap round unexpectedly advanced "
                "the global simulation clock."
            )

        output_resource_ids = tuple(
            attempt.output_resource_id
            for attempt in attempts
            if attempt.output_resource_id
            is not None
        )

        # =====================================================
        # Critical Anti-Cascade Rule
        #
        # No newly created resource may have been used as an
        # input during the same round.
        # =====================================================

        selected_input_ids = {
            resource_id
            for candidate in selected
            for resource_id
            in candidate.input_resource_ids
        }

        if (
            set(output_resource_ids)
            &
            selected_input_ids
        ):
            raise RuntimeError(
                "A newly created parent resource was reused "
                "inside the same swap round."
            )

        round_record = ParallelSwapRound(

            round_index=len(
                self._round_history
            ),

            simulation_slot=(
                round_slot
            ),

            policy_name=(
                self.policy.policy_name
            ),

            snapshot_resource_ids=(
                snapshot_resource_ids
            ),

            discovered_candidates=(
                candidates
            ),

            selected_candidates=(
                selected
            ),

            attempts=tuple(
                attempts
            ),

            output_resource_ids=(
                output_resource_ids
            )
        )

        self._round_history.append(
            round_record
        )

        # =====================================================
        # Immutable Simulation Event
        # =====================================================

        self.simulator.kernel.emit(

            event_type=(
                "CONTROL_PLANE_PARALLEL_SWAP_ROUND"
            ),

            component=(
                "ClassicalControlPlane."
                "ParallelSwapScheduler"
            ),

            node_names=tuple(
                node.name
                for node in self.ordered_path
            ),

            resource_ids=tuple(
                output_resource_ids
            ),

            message=(
                f"Control plane executed parallel swap "
                f"round {round_record.round_index} with "
                f"{round_record.selected_count} BSM command(s)."
            ),

            metadata={
                "policy":
                    self.policy.policy_name,

                "candidates_discovered":
                    round_record.candidate_count,

                "candidates_selected":
                    round_record.selected_count,

                "swap_successes":
                    round_record.success_count,

                "swap_failures":
                    round_record.failure_count
            }
        )

        return round_record

    # =========================================================
    # Active Entanglement Frontier
    # =========================================================

    def active_frontier_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        active_resources = tuple(
            self.simulator
            .entanglement_manager
            .active_resources
        )

        sortable = []

        for resource in active_resources:

            try:

                span = (
                    self._resource_span(
                        resource
                    )
                )

            except SwapCandidateConsistencyError:

                continue

            sortable.append(
                (
                    span,
                    resource
                )
            )

        sortable.sort(
            key=lambda item: (
                item[0][0],
                item[0][1]
            )
        )

        for span, resource in sortable:

            rows.append({

                "Resource ID":
                    resource.resource_id,

                "Kind":
                    resource.kind.value,

                "Endpoint A":
                    resource.endpoint_a_name,

                "Endpoint B":
                    resource.endpoint_b_name,

                "Path Span":
                    f"{span[0]} -> {span[1]}",

                "Span Hops":
                    span[1] - span[0],

                "Source Links":
                    resource.source_link_ids,

                "Created Slot":
                    resource.created_slot,

                "Oldest Half Age":
                    self._oldest_half_age(
                        resource
                    )
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Candidate Inspection
    # =========================================================

    def candidates_dataframe(
        self,
        candidates=None
    ) -> pd.DataFrame:

        if candidates is None:

            candidates = (
                self.discover_candidates()
            )

        rows = []

        for candidate in candidates:

            rows.append({

                "Candidate ID":
                    candidate.candidate_id,

                "Swap Node":
                    candidate.swap_node_name,

                "Swap Node Index":
                    candidate.swap_node_path_index,

                "Left Resource":
                    candidate.left_resource_id,

                "Right Resource":
                    candidate.right_resource_id,

                "Outer Left":
                    candidate.outer_left_name,

                "Outer Right":
                    candidate.outer_right_name,

                "Combined Span":
                    candidate.combined_span_indices,

                "Combined Hops":
                    candidate.combined_span_hops,

                "Oldest Input Age":
                    candidate.oldest_input_half_age_slots
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Round History
    # =========================================================

    @property
    def round_history(
        self
    ) -> tuple[
        ParallelSwapRound, ...
    ]:

        return tuple(
            self._round_history
        )

    def round_history_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for round_record in (
            self._round_history
        ):

            rows.append({

                "Round":
                    round_record.round_index,

                "Simulation Slot":
                    round_record.simulation_slot,

                "Candidates":
                    round_record.candidate_count,

                "Selected":
                    round_record.selected_count,

                "Successes":
                    round_record.success_count,

                "Failures":
                    round_record.failure_count,

                "Selected Swap Nodes":
                    tuple(
                        candidate.swap_node_name
                        for candidate
                        in round_record.selected_candidates
                    ),

                "Outputs":
                    round_record.output_resource_ids
            })

        return pd.DataFrame(
            rows
        )

    # =========================================================
    # Representation
    # =========================================================

    def __repr__(
        self
    ) -> str:

        return (
            "ControlPlaneParallelSwapScheduler("
            f"path_nodes={len(self.ordered_path)}, "
            f"repeaters={len(self.ordered_path) - 2}, "
            f"policy='{self.policy.policy_name}', "
            f"rounds={len(self._round_history)}"
            ")"
        )

In [ ]:
# @title Six-Repeater Parallel Swap Verification


print("=" * 84)
print("Six-Repeater Parallel Swap Verification")
print("=" * 84)


# ============================================================================
# NETWORK
#
# User-A -- R1 -- R2 -- R3 -- R4 -- R5 -- R6 -- User-B
#
# 8 physical nodes
# 6 repeaters
# 7 elementary links
#
# Every physical segment = 5 km
# ============================================================================

parallel_swap_topology = NetworkTopology(
    name="Phase3C3-Six-Repeater-Network"
)


parallel_swap_a = NetworkNode(
    name="User-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


parallel_swap_repeaters = []


for repeater_number in range(
    1,
    7
):

    repeater = NetworkNode(

        name=f"R{repeater_number}",

        x_km=(
            5 * repeater_number
        ),

        y_km=0,

        endpoint_capable=False,

        swapping_capable=True,

        memory_capacity=2
    )

    parallel_swap_repeaters.append(
        repeater
    )


parallel_swap_b = NetworkNode(
    name="User-B",
    x_km=35,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


parallel_swap_path = (
    parallel_swap_a,
    *parallel_swap_repeaters,
    parallel_swap_b
)


for node in parallel_swap_path:

    parallel_swap_topology.add_node(
        node
    )


for left, right in zip(
    parallel_swap_path[:-1],
    parallel_swap_path[1:]
):

    parallel_swap_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert (
    parallel_swap_topology.node_count
    == 8
)


assert (
    parallel_swap_topology.link_count
    == 7
)


assert (
    len(
        parallel_swap_repeaters
    )
    == 6
)


assert parallel_swap_topology.validate_path(
    parallel_swap_path
)


assert not (
    parallel_swap_topology.contains_link(
        parallel_swap_a,
        parallel_swap_b
    )
)


print(
    "[PASS] Six-repeater physical path contains 8 nodes "
    "and 7 adjacent quantum links."
)


# ============================================================================
# Assemble simulation
#
# Large finite memory lifetime so this test isolates parallel
# swap-control behavior rather than expiration behavior.
# ============================================================================

parallel_swap_simulator = (
    QuantumNetworkSimulator(

        topology=(
            parallel_swap_topology
        ),

        simulation_seed=(
            330303
        ),

        controller_id=(
            "PHASE3C3-CONTROLLER"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=100
    )
)


# ============================================================================
# Authentication
# ============================================================================

parallel_swap_auth_results = (
    parallel_swap_simulator
    .authenticate_all_nodes()
)


assert (
    len(
        parallel_swap_auth_results
    )
    == 8
)


assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in parallel_swap_auth_results.values()
)


assert (
    parallel_swap_simulator
    .require_quantum_eligibility(
        *parallel_swap_path
    )
)


print(
    "[PASS] Both users and all six repeaters are mutually "
    "authenticated and quantum eligible."
)


# ============================================================================
# Parallel elementary generation
# ============================================================================

parallel_swap_generation_service = (
    ElementaryEntanglementService(

        simulator=(
            parallel_swap_simulator
        ),

        model=(
            FiberSurvivalProxyModel()
        )
    )
)


parallel_swap_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=(
            parallel_swap_simulator
        ),

        service=(
            parallel_swap_generation_service
        ),

        ordered_path=(
            parallel_swap_path
        ),

        demand_active=True
    )
)


elementary_frontier = (
    parallel_swap_generation_scheduler
    .run_until_all_ready(
        max_slots=200
    )
)


assert (
    len(
        elementary_frontier
    )
    == 7
)


assert all(
    resource.kind
    == EntanglementKind.ELEMENTARY
    for resource
    in elementary_frontier
)


assert all(
    resource.is_active
    for resource
    in elementary_frontier
)


print(
    "[PASS] All seven adjacent physical links hold ACTIVE "
    "elementary Bell pairs."
)


# ============================================================================
# Physical memory shape before swapping
# ============================================================================

assert (
    parallel_swap_simulator
    .quantum_memory(
        parallel_swap_a
    )
    .occupancy
    == 1
)


assert (
    parallel_swap_simulator
    .quantum_memory(
        parallel_swap_b
    )
    .occupancy
    == 1
)


for repeater in (
    parallel_swap_repeaters
):

    assert (
        parallel_swap_simulator
        .quantum_memory(
            repeater
        )
        .occupancy
        == 2
    )


print(
    "[PASS] Initial repeater memory state contains two "
    "elementary halves per intermediate node."
)


# ============================================================================
# Ideal fidelity records for all elementary EPs
# ============================================================================

parallel_swap_fidelity = (
    EntanglementFidelityRegistry(
        parallel_swap_simulator
    )
)


for resource in (
    elementary_frontier
):

    parallel_swap_fidelity.register_ideal(
        resource
    )


assert all(
    math.isclose(
        parallel_swap_fidelity
        .fidelity(
            resource
        ),
        1.0,
        abs_tol=1e-12
    )
    for resource
    in elementary_frontier
)


print(
    "[PASS] All seven elementary resources begin at "
    "ideal fidelity F = 1.0."
)


# ============================================================================
# IDEAL BSM ENGINE
#
# We deliberately use ideal swapping here.
#
# Phase 3C.3 is testing CONTROL-PLANE SCHEDULING,
# not re-testing stochastic p_b behavior from 3C.2.
# ============================================================================

parallel_swap_engine = (
    EntanglementSwapEngine(

        simulator=(
            parallel_swap_simulator
        ),

        model=(
            IdealEntanglementSwapModel()
        ),

        fidelity_registry=(
            parallel_swap_fidelity
        )
    )
)


# ============================================================================
# CONTROL-PLANE PARALLEL SCHEDULER
# ============================================================================

parallel_swap_controller = (
    ControlPlaneParallelSwapScheduler(

        simulator=(
            parallel_swap_simulator
        ),

        swap_engine=(
            parallel_swap_engine
        ),

        ordered_path=(
            parallel_swap_path
        ),

        policy=(
            MaxParallelOldestFirstSwapPolicy()
        )
    )
)


assert (
    parallel_swap_controller.control_plane
    is parallel_swap_simulator.control_plane
)


print(
    "[PASS] Parallel swapping authority is attached to the "
    "network's ClassicalControlPlane."
)


# ============================================================================
# TEST 1 — INITIAL CANDIDATE DISCOVERY
#
# Seven elementary resources:
#
# E0 E1 E2 E3 E4 E5 E6
#
# Valid BSM nodes:
#
# R1 R2 R3 R4 R5 R6
#
# Therefore exactly six candidate swaps must exist.
# ============================================================================

initial_candidates = (
    parallel_swap_controller
    .discover_candidates()
)


assert (
    len(
        initial_candidates
    )
    == 6
)


assert set(
    candidate.swap_node_name
    for candidate
    in initial_candidates
) == {
    "R1",
    "R2",
    "R3",
    "R4",
    "R5",
    "R6"
}


print(
    "[PASS] Control plane discovers all six valid "
    "first-level BSM opportunities."
)


# ============================================================================
# TEST 2 — CONTROL PLANE MUST SELECT MAXIMUM PARALLEL SET
#
# From six neighbouring candidates over seven resources,
# maximum non-conflicting cardinality is three.
# ============================================================================

initial_selected = (
    parallel_swap_controller
    .select_parallel_candidates(
        initial_candidates
    )
)


assert (
    len(
        initial_selected
    )
    == 3
)


# Pairwise conflict freedom.
for i in range(
    len(initial_selected)
):

    for j in range(
        i + 1,
        len(initial_selected)
    ):

        assert not (
            initial_selected[i]
            .conflicts_with(
                initial_selected[j]
            )
        )


print(
    "[PASS] Control plane selects the maximum set of "
    "three mutually non-conflicting BSMs."
)


# ============================================================================
# TEST 3 — EXECUTE PARALLEL SWAP ROUND 1
# ============================================================================

round_1_slot = (
    parallel_swap_simulator
    .kernel.current_slot
)


round_1 = (
    parallel_swap_controller
    .execute_parallel_round()
)


assert (
    round_1.simulation_slot
    == round_1_slot
)


assert (
    parallel_swap_simulator
    .kernel.current_slot
    == round_1_slot
)


assert (
    round_1.candidate_count
    == 6
)


assert (
    round_1.selected_count
    == 3
)


assert (
    round_1.success_count
    == 3
)


assert (
    round_1.failure_count
    == 0
)


print(
    "[PASS] Round 1 executes three ideal BSMs in the "
    "same logical simulation slot."
)


# ============================================================================
# TEST 4 — NEW PARENTS WERE NOT USED INSIDE ROUND 1
# ============================================================================

round_1_snapshot = set(
    round_1.snapshot_resource_ids
)


round_1_input_ids = {
    resource_id
    for candidate
    in round_1.selected_candidates
    for resource_id
    in candidate.input_resource_ids
}


round_1_output_ids = set(
    round_1.output_resource_ids
)


assert (
    round_1_input_ids
    <= round_1_snapshot
)


assert not (
    round_1_output_ids
    &
    round_1_snapshot
)


assert not (
    round_1_output_ids
    &
    round_1_input_ids
)


print(
    "[PASS] Newly created swapped resources cannot cascade "
    "into another BSM during the same swap round."
)


# ============================================================================
# TEST 5 — ACTIVE FRONTIER SHOULD NOW CONTAIN FOUR RESOURCES
#
# Seven initial resources
# - six consumed as three pairs
# + three parents
#
# = four active resources.
# ============================================================================

active_after_round_1 = (
    parallel_swap_simulator
    .entanglement_manager
    .active_resources
)


assert (
    len(
        active_after_round_1
    )
    == 4
)


print(
    "[PASS] First parallel swap round compresses seven "
    "active spans into four."
)


print("\nACTIVE FRONTIER AFTER ROUND 1")

display(
    parallel_swap_controller
    .active_frontier_dataframe()
)


# ============================================================================
# ADVANCE ONE GLOBAL ROUND BOUNDARY
#
# A newly created parent becomes eligible only after this
# explicit round boundary.
# ============================================================================

parallel_swap_simulator\
    .kernel\
    .advance_slot()


# ============================================================================
# TEST 6 — SECOND-LEVEL CANDIDATE DISCOVERY
#
# Four ordered active spans create three boundaries/candidates.
# ============================================================================

round_2_candidates = (
    parallel_swap_controller
    .discover_candidates()
)


assert (
    len(
        round_2_candidates
    )
    == 3
)


round_2_selected = (
    parallel_swap_controller
    .select_parallel_candidates(
        round_2_candidates
    )
)


assert (
    len(
        round_2_selected
    )
    == 2
)


print(
    "[PASS] Second swap level exposes three candidates and "
    "two can execute concurrently."
)


# ============================================================================
# TEST 7 — EXECUTE ROUND 2
# ============================================================================

round_2 = (
    parallel_swap_controller
    .execute_parallel_round()
)


assert (
    round_2.candidate_count
    == 3
)


assert (
    round_2.selected_count
    == 2
)


assert (
    round_2.success_count
    == 2
)


assert (
    round_2.failure_count
    == 0
)


assert (
    len(
        parallel_swap_simulator
        .entanglement_manager
        .active_resources
    )
    == 2
)


print(
    "[PASS] Round 2 performs two non-conflicting higher-level "
    "BSMs and leaves two active spans."
)


print("\nACTIVE FRONTIER AFTER ROUND 2")

display(
    parallel_swap_controller
    .active_frontier_dataframe()
)


# ============================================================================
# Advance to third swap round.
# ============================================================================

parallel_swap_simulator\
    .kernel\
    .advance_slot()


# ============================================================================
# TEST 8 — FINAL CANDIDATE
# ============================================================================

round_3_candidates = (
    parallel_swap_controller
    .discover_candidates()
)


assert (
    len(
        round_3_candidates
    )
    == 1
)


assert (
    round_3_candidates[0]
    .combined_span_indices
    == (
        0,
        7
    )
)


print(
    "[PASS] Two surviving subtrees expose exactly one "
    "final BSM candidate."
)


# ============================================================================
# TEST 9 — EXECUTE FINAL ROUND
# ============================================================================

round_3 = (
    parallel_swap_controller
    .execute_parallel_round()
)


assert (
    round_3.selected_count
    == 1
)


assert (
    round_3.success_count
    == 1
)


assert (
    len(
        parallel_swap_simulator
        .entanglement_manager
        .active_resources
    )
    == 1
)


print(
    "[PASS] Third control-plane swap round creates the "
    "single final entangled span."
)


# ============================================================================
# FINAL END-TO-END RESOURCE
# ============================================================================

final_resource = (
    parallel_swap_simulator
    .entanglement_manager
    .active_resources[0]
)


assert (
    final_resource.kind
    == EntanglementKind.SWAPPED
)


assert (
    final_resource.endpoint_names
    == tuple(
        sorted(
            (
                parallel_swap_a.name,
                parallel_swap_b.name
            )
        )
    )
)


assert (
    len(
        final_resource.source_link_ids
    )
    == 7
)


assert set(
    final_resource.source_link_ids
) == {
    link.link_id
    for link in (
        parallel_swap_topology
        .path_links(
            parallel_swap_path
        )
    )
}


print(
    "[PASS] Final resource entangles User-A and User-B "
    "across all seven underlying physical links."
)


# ============================================================================
# There is STILL no direct physical A--B link.
# ============================================================================

assert not (
    parallel_swap_topology
    .contains_link(
        parallel_swap_a,
        parallel_swap_b
    )
)


print(
    "[PASS] End-to-end entanglement exists without "
    "fabricating a direct physical User-A--User-B link."
)


# ============================================================================
# TOTAL BSM COUNT
#
# With six repeaters, exactly six successful BSMs are needed:
#
#     3 + 2 + 1 = 6
# ============================================================================

assert (
    len(
        parallel_swap_engine.attempts
    )
    == 6
)


assert all(
    attempt.success
    for attempt
    in parallel_swap_engine.attempts
)


assert (
    tuple(
        round_record.selected_count
        for round_record
        in parallel_swap_controller.round_history
    )
    == (
        3,
        2,
        1
    )
)


print(
    "[PASS] Six repeater BSMs are completed in a "
    "3 -> 2 -> 1 parallel-round structure."
)


# ============================================================================
# FINAL FIDELITY
# ============================================================================

assert math.isclose(
    parallel_swap_fidelity
    .fidelity(
        final_resource
    ),
    1.0,
    abs_tol=1e-12
)


print(
    "[PASS] Ideal parallel swapping preserves final "
    "end-to-end fidelity F = 1.0."
)


# ============================================================================
# FINAL QUANTUM MEMORY STATE
#
# User-A : one surviving outer qubit
# User-B : one surviving outer qubit
#
# Every repeater:
#     all BSM-participating qubits have been consumed.
# ============================================================================

assert (
    parallel_swap_simulator
    .quantum_memory(
        parallel_swap_a
    )
    .occupancy
    == 1
)


assert (
    parallel_swap_simulator
    .quantum_memory(
        parallel_swap_b
    )
    .occupancy
    == 1
)


for repeater in (
    parallel_swap_repeaters
):

    assert (
        parallel_swap_simulator
        .quantum_memory(
            repeater
        )
        .occupancy
        == 0
    )


print(
    "[PASS] Final memory state contains only the two "
    "surviving end-user qubits."
)


# ============================================================================
# NO FURTHER SWAP CANDIDATES
# ============================================================================

assert (
    len(
        parallel_swap_controller
        .discover_candidates()
    )
    == 0
)


print(
    "[PASS] Control plane finds no further BSM candidate "
    "after end-to-end entanglement is established."
)


# ============================================================================
# CROSS-LAYER CONSISTENCY
# ============================================================================

assert (
    parallel_swap_simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Complete six-repeater NARAD-QC runtime remains "
    "cross-layer consistent."
)


# ============================================================================
# Diagnostics
# ============================================================================

print("\nCONTROL-PLANE PARALLEL SWAP SCHEDULER")

print(
    parallel_swap_controller
)


print("\nPARALLEL SWAP ROUND HISTORY")

display(
    parallel_swap_controller
    .round_history_dataframe()
)


print("\nALL BSM ATTEMPTS")

display(
    parallel_swap_engine
    .attempts_dataframe()
)


print("\nFINAL ACTIVE ENTANGLEMENT FRONTIER")

display(
    parallel_swap_controller
    .active_frontier_dataframe()
)


print("\nCOMPLETE RESOURCE HISTORY")

display(
    parallel_swap_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\nFINAL FIDELITY HISTORY")

display(
    parallel_swap_fidelity
    .dataframe()
)


print("\n" + "=" * 84)
print(
    "Six-repeater parallel swap verification "
    "completed successfully."
)
print("=" * 84)

In [ ]:
# @title Dynamic NARAD-QC End-to-End Demo Animation

# import math
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
# from matplotlib.lines import Line2D
# from IPython.display import HTML, display


# # =============================================================================
# # NARAD-QC PHASE 3C.3 DEMO ANIMATION
# #
# # PURE OBSERVER:
# #
# #   - does NOT rerun quantum physics
# #   - does NOT consume SimulationKernel RNG
# #   - does NOT emit simulation events
# #   - does NOT mutate topology
# #   - does NOT modify quantum memory
# #   - does NOT modify authentication state
# #   - does NOT create/discard resources
# #
# # It reconstructs the animation only from the already-completed
# # Phase 3C.3 verification state/history.
# # =============================================================================


# def animate_narad_qc_phase3c3(
#     simulator,
#     ordered_path,
#     authentication_results,
#     generation_scheduler,
#     swap_controller,
#     interval_ms=1000
# ):
#     """
#     Build a Matplotlib FuncAnimation of the actual recorded NARAD-QC
#     Phase-3C.3 execution.

#     Visual semantics
#     ----------------
#     User nodes:
#         squares

#     Repeater nodes:
#         circles

#     Physical fibre:
#         solid line

#     ACTIVE elementary Bell pair:
#         one dotted line

#     ACTIVE swapped entanglement:
#         two parallel dotted lines

#     Authentication:
#         node appearance changes after its recorded ML-DSA
#         authentication stage is reached.

#     Timing
#     ------
#     interval_ms controls presentation delay only.

#     It is NOT physical network time and is NOT automatically equal
#     to SimulationKernel slot duration.
#     """

#     # =========================================================================
#     # Defensive copies / observer snapshots
#     # =========================================================================

#     path = tuple(
#         ordered_path
#     )

#     path_links = tuple(
#         simulator.topology.path_links(
#             path
#         )
#     )

#     manager = (
#         simulator.entanglement_manager
#     )

#     generation_history = tuple(
#         generation_scheduler.slot_history
#     )

#     swap_history = tuple(
#         swap_controller.round_history
#     )

#     # =========================================================================
#     # Coordinate helper
#     # =========================================================================

#     def node_xy(node):

#         if (
#             hasattr(node, "x_km")
#             and hasattr(node, "y_km")
#         ):
#             return (
#                 float(node.x_km),
#                 float(node.y_km)
#             )

#         if hasattr(
#             node,
#             "position"
#         ):
#             x, y = node.position

#             return (
#                 float(x),
#                 float(y)
#             )

#         raise AttributeError(
#             f"Cannot determine coordinates for node "
#             f"'{node.name}'."
#         )


#     positions = {
#         node.name:
#             node_xy(node)
#         for node in path
#     }


#     node_by_name = {
#         node.name:
#             node
#         for node in path
#     }


#     # =========================================================================
#     # Plot scale
#     # =========================================================================

#     xs = [
#         positions[node.name][0]
#         for node in path
#     ]

#     ys = [
#         positions[node.name][1]
#         for node in path
#     ]


#     x_span = max(
#         max(xs) - min(xs),
#         1.0
#     )

#     y_span = max(
#         max(ys) - min(ys),
#         1.0
#     )


#     overall_span = max(
#         x_span,
#         y_span,
#         1.0
#     )


#     # Entanglement lines are offset slightly from the physical fibre
#     # so both remain visible.
#     single_entanglement_offset = max(
#         0.12,
#         overall_span * 0.006
#     )

#     swapped_double_offset = max(
#         0.18,
#         overall_span * 0.009
#     )


#     # =========================================================================
#     # Authentication-result normalization
#     # =========================================================================

#     auth_result_by_name = {}

#     if authentication_results is not None:

#         for key, record in authentication_results.items():

#             if isinstance(
#                 key,
#                 NetworkNode
#             ):
#                 name = key.name

#             else:
#                 name = str(key)

#             auth_result_by_name[
#                 name
#             ] = record


#     def authentication_succeeded(
#         node_name
#     ):

#         record = auth_result_by_name.get(
#             node_name
#         )

#         if record is None:
#             return False

#         status = getattr(
#             record,
#             "status",
#             None
#         )

#         if status is None:
#             return True

#         if isinstance(
#             status,
#             AuthenticationStatus
#         ):
#             return (
#                 status
#                 == AuthenticationStatus.AUTHENTICATED
#             )

#         return (
#             str(status).upper().endswith(
#                 "AUTHENTICATED"
#             )
#         )


#     # =========================================================================
#     # Resource helper
#     # =========================================================================

#     def get_resource(
#         resource_id
#     ):

#         return (
#             manager.get_resource(
#                 resource_id
#             )
#         )


#     # =========================================================================
#     # Reconstruct ACTIVE elementary frontier after each generation slot
#     #
#     # ParallelElementaryGenerationSlot already stored the IDs that were active
#     # at the end of that slot.
#     # =========================================================================

#     generation_frames = []

#     for slot_record in generation_history:

#         generation_frames.append({
#             "stage":
#                 "generation",

#             "title":
#                 (
#                     "Parallel Elementary Entanglement Generation "
#                     f"— Simulation Slot {slot_record.simulation_slot}"
#                 ),

#             "simulation_slot":
#                 slot_record.simulation_slot,

#             "resource_ids":
#                 tuple(
#                     slot_record.active_resource_ids
#                 ),

#             "physical_attempts":
#                 slot_record.physical_attempt_count,

#             "successes":
#                 slot_record.successful_generation_count,

#             "failures":
#                 slot_record.failed_generation_count,

#             "ready_links":
#                 len(
#                     slot_record.ready_link_ids
#                 ),

#             "total_links":
#                 len(
#                     path_links
#                 )
#         })


#     # =========================================================================
#     # Reconstruct ACTIVE entanglement frontier after each swap round
#     #
#     # A selected BSM always consumes BOTH inputs.
#     #
#     # On success:
#     #     parent appears.
#     #
#     # On failure:
#     #     no parent appears.
#     #
#     # This reconstruction works for both ideal and stochastic BSM histories.
#     # =========================================================================

#     swap_frames = []

#     for round_record in swap_history:

#         active_after_round = set(
#             round_record.snapshot_resource_ids
#         )

#         for candidate in (
#             round_record.selected_candidates
#         ):

#             active_after_round.discard(
#                 candidate.left_resource_id
#             )

#             active_after_round.discard(
#                 candidate.right_resource_id
#             )

#         for attempt in (
#             round_record.attempts
#         ):

#             if (
#                 attempt.output_resource_id
#                 is not None
#             ):
#                 active_after_round.add(
#                     attempt.output_resource_id
#                 )


#         swap_frames.append({
#             "stage":
#                 "swap",

#             "title":
#                 (
#                     "Control-Plane Parallel Swap Round "
#                     f"{round_record.round_index + 1}"
#                 ),

#             "simulation_slot":
#                 round_record.simulation_slot,

#             "resource_ids":
#                 tuple(
#                     sorted(
#                         active_after_round
#                     )
#                 ),

#             "candidates":
#                 round_record.candidate_count,

#             "selected":
#                 round_record.selected_count,

#             "successes":
#                 round_record.success_count,

#             "failures":
#                 round_record.failure_count,

#             "swap_nodes":
#                 tuple(
#                     candidate.swap_node_name
#                     for candidate
#                     in round_record.selected_candidates
#                 )
#         })


#     # =========================================================================
#     # Build Complete Presentation Timeline
#     # =========================================================================

#     frames = []


#     # -------------------------------------------------------------------------
#     # FRAME 0 — Empty network canvas
#     # -------------------------------------------------------------------------

#     frames.append({
#         "stage":
#             "bootstrap",

#         "title":
#             "NARAD-QC Network Initialization",

#         "visible_nodes":
#             (),

#         "visible_links":
#             (),

#         "authenticated_nodes":
#             (),

#         "resource_ids":
#             ()
#     })


#     # -------------------------------------------------------------------------
#     # NODE ESTABLISHMENT
#     #
#     # One node appears per presentation step.
#     # -------------------------------------------------------------------------

#     visible_node_names = []

#     for node in path:

#         visible_node_names.append(
#             node.name
#         )

#         frames.append({
#             "stage":
#                 "nodes",

#             "title":
#                 (
#                     "Establishing Quantum-Network Nodes "
#                     f"— {node.name}"
#                 ),

#             "visible_nodes":
#                 tuple(
#                     visible_node_names
#                 ),

#             "visible_links":
#                 (),

#             "authenticated_nodes":
#                 (),

#             "resource_ids":
#                 ()
#         })


#     # -------------------------------------------------------------------------
#     # PHYSICAL FIBRE ESTABLISHMENT
#     #
#     # One authoritative physical link appears per presentation step.
#     # -------------------------------------------------------------------------

#     visible_link_ids = []

#     for link in path_links:

#         visible_link_ids.append(
#             link.link_id
#         )

#         frames.append({
#             "stage":
#                 "fiber",

#             "title":
#                 (
#                     "Establishing Physical Quantum Fibre "
#                     f"— {link.link_id}"
#                 ),

#             "visible_nodes":
#                 tuple(
#                     node.name
#                     for node in path
#                 ),

#             "visible_links":
#                 tuple(
#                     visible_link_ids
#                 ),

#             "authenticated_nodes":
#                 (),

#             "resource_ids":
#                 ()
#         })


#     # -------------------------------------------------------------------------
#     # ML-DSA MUTUAL AUTHENTICATION
#     #
#     # Authentication is animated node by node for presentation clarity.
#     #
#     # This does NOT imply that authentication consumed physical simulation
#     # time in the original experiment.
#     # -------------------------------------------------------------------------

#     authenticated_names = []

#     for node in path:

#         if authentication_succeeded(
#             node.name
#         ):

#             authenticated_names.append(
#                 node.name
#             )

#         frames.append({
#             "stage":
#                 "authentication",

#             "title":
#                 (
#                     "ML-DSA Mutual Device Authentication "
#                     f"— {node.name}"
#                 ),

#             "visible_nodes":
#                 tuple(
#                     n.name
#                     for n in path
#                 ),

#             "visible_links":
#                 tuple(
#                     link.link_id
#                     for link in path_links
#                 ),

#             "authenticated_nodes":
#                 tuple(
#                     authenticated_names
#                 ),

#             "resource_ids":
#                 (),

#             "current_auth_node":
#                 node.name
#         })


#     # -------------------------------------------------------------------------
#     # ELEMENTARY ENTANGLEMENT GENERATION
#     #
#     # One animation frame per REAL parallel-generation simulation slot.
#     # -------------------------------------------------------------------------

#     for generation_frame in (
#         generation_frames
#     ):

#         frames.append({

#             **generation_frame,

#             "visible_nodes":
#                 tuple(
#                     node.name
#                     for node in path
#                 ),

#             "visible_links":
#                 tuple(
#                     link.link_id
#                     for link in path_links
#                 ),

#             "authenticated_nodes":
#                 tuple(
#                     node.name
#                     for node in path
#                     if authentication_succeeded(
#                         node.name
#                     )
#                 )
#         })


#     # -------------------------------------------------------------------------
#     # PARALLEL ENTANGLEMENT SWAPPING
#     #
#     # One frame per ACTUAL control-plane parallel swap round.
#     # -------------------------------------------------------------------------

#     for swap_frame in (
#         swap_frames
#     ):

#         frames.append({

#             **swap_frame,

#             "visible_nodes":
#                 tuple(
#                     node.name
#                     for node in path
#                 ),

#             "visible_links":
#                 tuple(
#                     link.link_id
#                     for link in path_links
#                 ),

#             "authenticated_nodes":
#                 tuple(
#                     node.name
#                     for node in path
#                     if authentication_succeeded(
#                         node.name
#                     )
#                 )
#         })


#     # -------------------------------------------------------------------------
#     # FINAL END-TO-END ENTANGLEMENT FRAME
#     # -------------------------------------------------------------------------

#     final_active_resources = tuple(
#         manager.active_resources
#     )

#     final_resource_ids = tuple(
#         resource.resource_id
#         for resource
#         in final_active_resources
#     )


#     frames.append({
#         "stage":
#             "complete",

#         "title":
#             "End-to-End Quantum Entanglement Established",

#         "visible_nodes":
#             tuple(
#                 node.name
#                 for node in path
#             ),

#         "visible_links":
#             tuple(
#                 link.link_id
#                 for link in path_links
#             ),

#         "authenticated_nodes":
#             tuple(
#                 node.name
#                 for node in path
#                 if authentication_succeeded(
#                     node.name
#                 )
#             ),

#         "resource_ids":
#             final_resource_ids
#     })


#     # =========================================================================
#     # Figure
#     # =========================================================================

#     fig, ax = plt.subplots(
#         figsize=(14, 7)
#     )


#     # =========================================================================
#     # Geometry Helper — Offset Line
#     # =========================================================================

#     def offset_segment(
#         node_a_name,
#         node_b_name,
#         offset
#     ):

#         x1, y1 = positions[
#             node_a_name
#         ]

#         x2, y2 = positions[
#             node_b_name
#         ]

#         dx = x2 - x1
#         dy = y2 - y1

#         length = math.hypot(
#             dx,
#             dy
#         )

#         if length == 0:

#             return (
#                 (x1, x2),
#                 (y1, y2)
#             )

#         # Unit perpendicular.
#         px = -dy / length
#         py = dx / length

#         return (
#             (
#                 x1 + px * offset,
#                 x2 + px * offset
#             ),
#             (
#                 y1 + py * offset,
#                 y2 + py * offset
#             )
#         )


#     # =========================================================================
#     # Physical Fibre Drawing
#     # =========================================================================

#     def draw_physical_link(
#         link
#     ):

#         x1, y1 = positions[
#             link.node_a.name
#         ]

#         x2, y2 = positions[
#             link.node_b.name
#         ]

#         ax.plot(
#             [x1, x2],
#             [y1, y2],
#             linestyle="-",
#             linewidth=2.0,
#             color="black",
#             zorder=1
#         )

#         midpoint_x = (
#             x1 + x2
#         ) / 2.0

#         midpoint_y = (
#             y1 + y2
#         ) / 2.0

#         ax.text(
#             midpoint_x,
#             midpoint_y - (
#                 overall_span * 0.025
#             ),
#             f"{link.distance_km:.1f} km",
#             fontsize=8,
#             ha="center",
#             va="top",
#             color="dimgray",
#             zorder=8
#         )


#     # =========================================================================
#     # Elementary EP — SINGLE dotted line
#     # =========================================================================

#     def draw_elementary_resource(
#         resource
#     ):

#         endpoint_a, endpoint_b = (
#             resource.endpoint_names
#         )

#         x_values, y_values = (
#             offset_segment(
#                 endpoint_a,
#                 endpoint_b,
#                 single_entanglement_offset
#             )
#         )

#         ax.plot(
#             x_values,
#             y_values,
#             linestyle=":",
#             linewidth=3.0,
#             color="tab:blue",
#             zorder=4
#         )

#         label_x = (
#             x_values[0]
#             + x_values[1]
#         ) / 2.0

#         label_y = (
#             y_values[0]
#             + y_values[1]
#         ) / 2.0

#         ax.text(
#             label_x,
#             label_y + (
#                 overall_span * 0.018
#             ),
#             resource.resource_id,
#             fontsize=7,
#             ha="center",
#             va="bottom",
#             color="tab:blue",
#             zorder=9
#         )


#     # =========================================================================
#     # Swapped EP — DOUBLE dotted line
#     # =========================================================================

#     def draw_swapped_resource(
#         resource
#     ):

#         endpoint_a, endpoint_b = (
#             resource.endpoint_names
#         )

#         upper_x, upper_y = (
#             offset_segment(
#                 endpoint_a,
#                 endpoint_b,
#                 swapped_double_offset
#             )
#         )

#         lower_x, lower_y = (
#             offset_segment(
#                 endpoint_a,
#                 endpoint_b,
#                 -swapped_double_offset
#             )
#         )

#         # First dotted strand.
#         ax.plot(
#             upper_x,
#             upper_y,
#             linestyle=":",
#             linewidth=2.5,
#             color="tab:purple",
#             zorder=5
#         )

#         # Second dotted strand.
#         ax.plot(
#             lower_x,
#             lower_y,
#             linestyle=":",
#             linewidth=2.5,
#             color="tab:purple",
#             zorder=5
#         )

#         label_x = (
#             upper_x[0]
#             + upper_x[1]
#         ) / 2.0

#         label_y = max(
#             (
#                 upper_y[0]
#                 + upper_y[1]
#             ) / 2.0,
#             (
#                 lower_y[0]
#                 + lower_y[1]
#             ) / 2.0
#         )

#         ax.text(
#             label_x,
#             label_y + (
#                 overall_span * 0.018
#             ),
#             resource.resource_id,
#             fontsize=8,
#             fontweight="bold",
#             ha="center",
#             va="bottom",
#             color="tab:purple",
#             zorder=9
#         )


#     # =========================================================================
#     # Node Drawing
#     # =========================================================================

#     def draw_node(
#         node,
#         authenticated,
#         current_auth_node=None,
#         final_stage=False
#     ):

#         x, y = positions[
#             node.name
#         ]

#         is_user = (
#             node.endpoint_capable
#             and not node.swapping_capable
#         )

#         marker = (
#             "s"
#             if is_user
#             else "o"
#         )


#         if authenticated:

#             face_color = (
#                 "palegreen"
#             )

#             edge_color = (
#                 "darkgreen"
#             )

#         else:

#             face_color = (
#                 "white"
#             )

#             edge_color = (
#                 "black"
#             )


#         # Highlight node currently undergoing authentication.
#         if (
#             current_auth_node
#             == node.name
#         ):

#             edge_color = (
#                 "darkorange"
#             )


#         # Final endpoints emphasized without changing square shape.
#         if (
#             final_stage
#             and is_user
#         ):

#             edge_width = 4.0
#             marker_size = 250

#         else:

#             edge_width = 2.0
#             marker_size = (
#                 210
#                 if is_user
#                 else 180
#             )


#         ax.scatter(
#             [x],
#             [y],
#             s=marker_size,
#             marker=marker,
#             facecolor=face_color,
#             edgecolor=edge_color,
#             linewidth=edge_width,
#             zorder=10
#         )


#         role_label = (
#             "USER"
#             if is_user
#             else "REPEATER"
#         )


#         auth_label = (
#             "\nML-DSA ✓"
#             if authenticated
#             else ""
#         )


#         ax.text(
#             x,
#             y - (
#                 overall_span * 0.045
#             ),
#             (
#                 f"{node.name}\n"
#                 f"{role_label}"
#                 f"{auth_label}"
#             ),
#             ha="center",
#             va="top",
#             fontsize=9,
#             zorder=11
#         )


#     # =========================================================================
#     # Status Panel
#     # =========================================================================

#     def draw_status_panel(
#         frame
#     ):

#         stage = frame[
#             "stage"
#         ]

#         lines = []


#         if stage == "bootstrap":

#             lines.append(
#                 "Initializing NARAD-QC..."
#             )


#         elif stage == "nodes":

#             lines.append(
#                 "Physical node registration"
#             )

#             lines.append(
#                 f"Nodes online: "
#                 f"{len(frame['visible_nodes'])}/{len(path)}"
#             )


#         elif stage == "fiber":

#             lines.append(
#                 "Authoritative fibre topology"
#             )

#             lines.append(
#                 f"Physical links established: "
#                 f"{len(frame['visible_links'])}/{len(path_links)}"
#             )


#         elif stage == "authentication":

#             lines.append(
#                 "ML-DSA mutual authentication"
#             )

#             lines.append(
#                 f"Authenticated devices: "
#                 f"{len(frame['authenticated_nodes'])}/{len(path)}"
#             )

#             lines.append(
#                 f"Current device: "
#                 f"{frame.get('current_auth_node', '')}"
#             )


#         elif stage == "generation":

#             lines.append(
#                 "Parallel elementary-entanglement generation"
#             )

#             lines.append(
#                 f"Simulation slot: "
#                 f"{frame['simulation_slot']}"
#             )

#             lines.append(
#                 f"Physical attempts this slot: "
#                 f"{frame['physical_attempts']}"
#             )

#             lines.append(
#                 f"Generation successes: "
#                 f"{frame['successes']}"
#             )

#             lines.append(
#                 f"Ready links: "
#                 f"{frame['ready_links']}/{frame['total_links']}"
#             )


#         elif stage == "swap":

#             lines.append(
#                 "Control-plane parallel entanglement swapping"
#             )

#             lines.append(
#                 f"Simulation slot: "
#                 f"{frame['simulation_slot']}"
#             )

#             lines.append(
#                 f"Candidates discovered: "
#                 f"{frame['candidates']}"
#             )

#             lines.append(
#                 f"Parallel BSMs selected: "
#                 f"{frame['selected']}"
#             )

#             lines.append(
#                 f"BSM successes: "
#                 f"{frame['successes']}"
#             )

#             if frame[
#                 "swap_nodes"
#             ]:

#                 lines.append(
#                     "Swap nodes: "
#                     + ", ".join(
#                         frame[
#                             "swap_nodes"
#                         ]
#                     )
#                 )


#         elif stage == "complete":

#             lines.append(
#                 "END-TO-END ENTANGLEMENT READY"
#             )

#             lines.append(
#                 f"{path[0].name} ↔ {path[-1].name}"
#             )

#             lines.append(
#                 "Physical fibre remains multi-hop"
#             )


#         ax.text(
#             0.02,
#             0.97,
#             "\n".join(
#                 lines
#             ),
#             transform=ax.transAxes,
#             ha="left",
#             va="top",
#             fontsize=10,
#             bbox=dict(
#                 boxstyle="round,pad=0.5",
#                 facecolor="white",
#                 edgecolor="gray",
#                 alpha=0.92
#             ),
#             zorder=20
#         )


#     # =========================================================================
#     # Legend
#     # =========================================================================

#     legend_handles = [

#         Line2D(
#             [0],
#             [0],
#             marker="s",
#             linestyle="None",
#             markerfacecolor="white",
#             markeredgecolor="black",
#             markersize=10,
#             label="User node"
#         ),

#         Line2D(
#             [0],
#             [0],
#             marker="o",
#             linestyle="None",
#             markerfacecolor="white",
#             markeredgecolor="black",
#             markersize=9,
#             label="Repeater"
#         ),

#         Line2D(
#             [0],
#             [0],
#             linestyle="-",
#             linewidth=2,
#             color="black",
#             label="Physical fibre"
#         ),

#         Line2D(
#             [0],
#             [0],
#             linestyle=":",
#             linewidth=3,
#             color="tab:blue",
#             label="Elementary Bell pair"
#         ),

#         Line2D(
#             [0],
#             [0],
#             linestyle=":",
#             linewidth=5,
#             color="tab:purple",
#             label="Swapped entanglement (double dotted)"
#         ),

#         Line2D(
#             [0],
#             [0],
#             marker="o",
#             linestyle="None",
#             markerfacecolor="palegreen",
#             markeredgecolor="darkgreen",
#             markersize=9,
#             label="ML-DSA authenticated"
#         )
#     ]


#     # =========================================================================
#     # Axis Limits
#     # =========================================================================

#     x_padding = max(
#         x_span * 0.08,
#         1.5
#     )

#     y_padding = max(
#         overall_span * 0.16,
#         3.0
#     )


#     # =========================================================================
#     # Animation Update
#     # =========================================================================

#     def update(
#         frame_index
#     ):

#         frame = frames[
#             frame_index
#         ]

#         ax.clear()


#         # ---------------------------------------------------------------------
#         # Canvas / scale
#         # ---------------------------------------------------------------------

#         ax.set_xlim(
#             min(xs) - x_padding,
#             max(xs) + x_padding
#         )

#         ax.set_ylim(
#             min(ys) - y_padding,
#             max(ys) + y_padding
#         )

#         ax.set_aspect(
#             "equal",
#             adjustable="box"
#         )

#         ax.grid(
#             True,
#             linestyle="--",
#             alpha=0.20
#         )

#         ax.set_xlabel(
#             "x-position (km)"
#         )

#         ax.set_ylabel(
#             "y-position (km)"
#         )


#         ax.set_title(
#             frame[
#                 "title"
#             ],
#             fontsize=15,
#             fontweight="bold",
#             pad=14
#         )


#         # ---------------------------------------------------------------------
#         # Physical fibre
#         # ---------------------------------------------------------------------

#         visible_link_ids = set(
#             frame.get(
#                 "visible_links",
#                 ()
#             )
#         )


#         for link in path_links:

#             if (
#                 link.link_id
#                 in visible_link_ids
#             ):

#                 draw_physical_link(
#                     link
#                 )


#         # ---------------------------------------------------------------------
#         # Entanglement frontier
#         #
#         # Resource state comes from recorded simulation history.
#         # ---------------------------------------------------------------------

#         for resource_id in (
#             frame.get(
#                 "resource_ids",
#                 ()
#             )
#         ):

#             resource = (
#                 get_resource(
#                     resource_id
#                 )
#             )


#             if (
#                 resource.kind
#                 == EntanglementKind.ELEMENTARY
#             ):

#                 draw_elementary_resource(
#                     resource
#                 )


#             elif (
#                 resource.kind
#                 == EntanglementKind.SWAPPED
#             ):

#                 draw_swapped_resource(
#                     resource
#                 )


#         # ---------------------------------------------------------------------
#         # Nodes
#         # ---------------------------------------------------------------------

#         visible_nodes = set(
#             frame.get(
#                 "visible_nodes",
#                 ()
#             )
#         )

#         authenticated_nodes = set(
#             frame.get(
#                 "authenticated_nodes",
#                 ()
#             )
#         )


#         for node in path:

#             if (
#                 node.name
#                 not in visible_nodes
#             ):
#                 continue


#             draw_node(

#                 node=node,

#                 authenticated=(
#                     node.name
#                     in authenticated_nodes
#                 ),

#                 current_auth_node=(
#                     frame.get(
#                         "current_auth_node"
#                     )
#                 ),

#                 final_stage=(
#                     frame["stage"]
#                     == "complete"
#                 )
#             )


#         # ---------------------------------------------------------------------
#         # Swap-node command markers
#         #
#         # Highlight the repeaters selected by the control plane
#         # during each parallel BSM round.
#         # ---------------------------------------------------------------------

#         if (
#             frame[
#                 "stage"
#             ]
#             == "swap"
#         ):

#             for swap_node_name in (
#                 frame[
#                     "swap_nodes"
#                 ]
#             ):

#                 x, y = positions[
#                     swap_node_name
#                 ]

#                 ax.scatter(
#                     [x],
#                     [y],
#                     s=430,
#                     marker="o",
#                     facecolors="none",
#                     edgecolors="red",
#                     linewidths=2.5,
#                     zorder=12
#                 )

#                 ax.text(
#                     x,
#                     y + (
#                         overall_span * 0.055
#                     ),
#                     "BSM",
#                     ha="center",
#                     va="bottom",
#                     fontsize=9,
#                     fontweight="bold",
#                     color="red",
#                     zorder=13
#                 )


#         # ---------------------------------------------------------------------
#         # Status
#         # ---------------------------------------------------------------------

#         draw_status_panel(
#             frame
#         )


#         # ---------------------------------------------------------------------
#         # Legend
#         # ---------------------------------------------------------------------

#         ax.legend(
#             handles=legend_handles,
#             loc="lower center",
#             bbox_to_anchor=(
#                 0.5,
#                 -0.20
#             ),
#             ncol=3,
#             frameon=True,
#             fontsize=9
#         )


#         # ---------------------------------------------------------------------
#         # Presentation-step counter
#         # ---------------------------------------------------------------------

#         ax.text(
#             0.98,
#             0.97,
#             (
#                 f"Demo Step "
#                 f"{frame_index + 1}/{len(frames)}"
#             ),
#             transform=ax.transAxes,
#             ha="right",
#             va="top",
#             fontsize=9,
#             color="dimgray"
#         )


#         return ()


#     # =========================================================================
#     # FuncAnimation
#     #
#     # interval=1000 ms -> one second presentation delay per step.
#     # =========================================================================

#     animation = FuncAnimation(

#         fig,

#         update,

#         frames=len(
#             frames
#         ),

#         interval=int(
#             interval_ms
#         ),

#         repeat=False,

#         blit=False
#     )


#     plt.close(
#         fig
#     )


#     return (
#         animation,
#         frames
#     )


# # =============================================================================
# # BUILD THE ANIMATION FROM THE ACTUAL PHASE 3C.3 VERIFICATION OBJECTS
# # =============================================================================

# narad_phase3c3_animation, narad_phase3c3_demo_frames = (
#     animate_narad_qc_phase3c3(

#         simulator=(
#             parallel_swap_simulator
#         ),

#         ordered_path=(
#             parallel_swap_path
#         ),

#         authentication_results=(
#             parallel_swap_auth_results
#         ),

#         generation_scheduler=(
#             parallel_swap_generation_scheduler
#         ),

#         swap_controller=(
#             parallel_swap_controller
#         ),

#         interval_ms=1000
#     )
# )


# # =============================================================================
# # Diagnostics
# # =============================================================================

# print(
#     "NARAD-QC dynamic demo created successfully."
# )

# print(
#     f"Total presentation frames: "
#     f"{len(narad_phase3c3_demo_frames)}"
# )

# print(
#     "Presentation delay per frame: 1.0 second"
# )

# print(
#     "IMPORTANT: animation delay is presentation time, "
#     "not physical simulation time."
# )


# # =============================================================================
# # DISPLAY IN COLAB / JUPYTER
# # =============================================================================

# display(
#     HTML(
#         narad_phase3c3_animation
#         .to_jshtml()
#     )
# )

In [ ]:
# @title  Werner-State Physical Fidelity Profile

from dataclasses import dataclass
import math
import numbers


class PhysicalFidelityModelError(Exception):
    """
    Base exception for Phase-3C.4 physical fidelity.
    """
    pass


@dataclass(frozen=True)
class WernerPhysicalFidelityProfile:
    """
    Network-level Werner-state fidelity approximation.

    Paper-inspired calibration anchors
    ----------------------------------
    Elementary-link Bell-pair fidelity:
        F_link = 0.995

    Memory benchmark:
        ideal Bell pair -> F = 0.90 after 1 second

    IMPORTANT
    ---------
    The paper uses NetSquid depolarization/dephasing models.

    NARAD-QC does NOT claim that this analytical exponential
    model reproduces NetSquid internally.

    Instead, we calibrate to the stated observable fidelity
    benchmark.

    slot_duration_s must therefore be supplied explicitly.
    It is a simulation-model assumption and must never be
    inferred from Python execution time.
    """

    elementary_fidelity: float = 0.995

    bell_fidelity_floor: float = 0.25

    memory_reference_initial_fidelity: float = 1.0

    memory_reference_fidelity: float = 0.90

    memory_reference_time_s: float = 1.0

    # Explicit physical-time interpretation of one simulation slot.
    slot_duration_s: float = 0.05

    # Additional local swapping-operation visibility.
    #
    # 1.0 means no extra BSM gate-noise penalty beyond the
    # Werner-state propagation of imperfect input resources.
    #
    # We deliberately leave this ideal until a defensible
    # hardware-specific value is supplied.
    swap_operation_visibility: float = 1.0


    def __post_init__(self):

        values = (
            self.elementary_fidelity,
            self.bell_fidelity_floor,
            self.memory_reference_initial_fidelity,
            self.memory_reference_fidelity,
            self.memory_reference_time_s,
            self.slot_duration_s,
            self.swap_operation_visibility
        )

        if not all(
            isinstance(value, numbers.Real)
            and not isinstance(value, bool)
            and math.isfinite(float(value))
            for value in values
        ):
            raise ValueError(
                "All physical-fidelity parameters must "
                "be finite real numbers."
            )

        floor = float(
            self.bell_fidelity_floor
        )

        if not (
            0.0 <= floor < 1.0
        ):
            raise ValueError(
                "Bell fidelity floor must lie in [0,1)."
            )

        for name, fidelity in (
            (
                "elementary_fidelity",
                self.elementary_fidelity
            ),
            (
                "memory_reference_initial_fidelity",
                self.memory_reference_initial_fidelity
            ),
            (
                "memory_reference_fidelity",
                self.memory_reference_fidelity
            )
        ):

            if not (
                floor
                <= float(fidelity)
                <= 1.0
            ):
                raise ValueError(
                    f"{name} must lie within "
                    f"[{floor}, 1]."
                )

        if not (
            self.memory_reference_fidelity
            <
            self.memory_reference_initial_fidelity
        ):
            raise ValueError(
                "Memory reference fidelity must be lower "
                "than its initial reference fidelity."
            )

        if (
            self.memory_reference_time_s
            <= 0.0
        ):
            raise ValueError(
                "memory_reference_time_s must be positive."
            )

        if (
            self.slot_duration_s
            <= 0.0
        ):
            raise ValueError(
                "slot_duration_s must be positive."
            )

        if not (
            0.0
            <= self.swap_operation_visibility
            <= 1.0
        ):
            raise ValueError(
                "swap_operation_visibility must lie in [0,1]."
            )

    # =====================================================================
    # Memory Decay Constant
    # =====================================================================

    @property
    def memory_decay_rate_per_s(
        self
    ) -> float:
        """
        Calibrate lambda from

            F(t) = F_floor +
                   (F0 - F_floor) exp(-lambda t)

        using the configured reference point.
        """

        floor = (
            self.bell_fidelity_floor
        )

        numerator = (
            self.memory_reference_fidelity
            - floor
        )

        denominator = (
            self.memory_reference_initial_fidelity
            - floor
        )

        ratio = (
            numerator
            / denominator
        )

        return (
            -math.log(
                ratio
            )
            /
            self.memory_reference_time_s
        )

    @property
    def memory_decay_time_constant_s(
        self
    ) -> float:

        return (
            1.0
            /
            self.memory_decay_rate_per_s
        )

    # =====================================================================
    # Fidelity / Werner Visibility
    # =====================================================================

    def validate_fidelity(
        self,
        fidelity
    ) -> float:

        if (
            isinstance(fidelity, bool)
            or not isinstance(
                fidelity,
                numbers.Real
            )
            or not math.isfinite(
                float(fidelity)
            )
        ):
            raise PhysicalFidelityModelError(
                "Fidelity must be a finite real number."
            )

        fidelity = float(
            fidelity
        )

        if not (
            self.bell_fidelity_floor
            <= fidelity
            <= 1.0
        ):
            raise PhysicalFidelityModelError(
                "Werner-state Bell-pair fidelity must lie "
                f"within [{self.bell_fidelity_floor}, 1]."
            )

        return fidelity

    def fidelity_to_visibility(
        self,
        fidelity
    ) -> float:

        fidelity = (
            self.validate_fidelity(
                fidelity
            )
        )

        return (
            (
                4.0 * fidelity
                - 1.0
            )
            / 3.0
        )

    def visibility_to_fidelity(
        self,
        visibility
    ) -> float:

        if not (
            0.0
            <= float(visibility)
            <= 1.0
        ):
            raise PhysicalFidelityModelError(
                "Werner visibility must lie in [0,1]."
            )

        return (
            0.25
            + 0.75
            * float(visibility)
        )

    # =====================================================================
    # Memory Decoherence
    # =====================================================================

    def memory_fidelity(
        self,
        initial_fidelity,
        elapsed_seconds
    ) -> float:

        initial_fidelity = (
            self.validate_fidelity(
                initial_fidelity
            )
        )

        if (
            isinstance(elapsed_seconds, bool)
            or not isinstance(
                elapsed_seconds,
                numbers.Real
            )
            or not math.isfinite(
                float(elapsed_seconds)
            )
            or float(elapsed_seconds) < 0.0
        ):
            raise PhysicalFidelityModelError(
                "elapsed_seconds must be a non-negative "
                "finite real number."
            )

        elapsed_seconds = float(
            elapsed_seconds
        )

        floor = (
            self.bell_fidelity_floor
        )

        result = (
            floor
            +
            (
                initial_fidelity
                - floor
            )
            * math.exp(
                -self.memory_decay_rate_per_s
                * elapsed_seconds
            )
        )

        return self.validate_fidelity(
            result
        )

    # =====================================================================
    # Simulation Slot -> Physical Time
    # =====================================================================

    def elapsed_seconds_for_slots(
        self,
        slot_count
    ) -> float:

        if (
            isinstance(slot_count, bool)
            or not isinstance(
                slot_count,
                numbers.Integral
            )
            or int(slot_count) < 0
        ):
            raise PhysicalFidelityModelError(
                "slot_count must be a non-negative integer."
            )

        return (
            int(slot_count)
            * self.slot_duration_s
        )

    # =====================================================================
    # Entanglement Swapping Fidelity
    # =====================================================================

    def swapped_fidelity(
        self,
        left_fidelity,
        right_fidelity
    ) -> float:
        """
        Werner-state approximation.

        Let:

            w = (4F - 1) / 3

        For ideal Bell-state swapping of two Werner states:

            w_out = w_left * w_right

        We additionally support an explicit local-operation
        visibility multiplier q_swap:

            w_out =
                w_left
                * w_right
                * q_swap

        q_swap defaults to 1.0 because we currently have no
        independently justified hardware-specific gate-noise
        parameter.
        """

        w_left = (
            self.fidelity_to_visibility(
                left_fidelity
            )
        )

        w_right = (
            self.fidelity_to_visibility(
                right_fidelity
            )
        )

        w_out = (
            w_left
            * w_right
            * self.swap_operation_visibility
        )

        # Numerical guard.
        w_out = min(
            max(
                w_out,
                0.0
            ),
            1.0
        )

        return (
            self.visibility_to_fidelity(
                w_out
            )
        )

    # =====================================================================
    # Diagnostics
    # =====================================================================

    def parameter_summary(
        self
    ) -> dict:

        return {
            "elementary_fidelity":
                self.elementary_fidelity,

            "bell_fidelity_floor":
                self.bell_fidelity_floor,

            "memory_reference":
                (
                    self.memory_reference_initial_fidelity,
                    self.memory_reference_fidelity,
                    self.memory_reference_time_s
                ),

            "memory_decay_rate_per_s":
                self.memory_decay_rate_per_s,

            "memory_decay_time_constant_s":
                self.memory_decay_time_constant_s,

            "slot_duration_s":
                self.slot_duration_s,

            "swap_operation_visibility":
                self.swap_operation_visibility
        }

In [ ]:
# @title  Dynamic Physical Fidelity Registry


class PhysicalEntanglementFidelityRegistry(
    EntanglementFidelityRegistry
):
    """
    Dynamic physical-fidelity sidecar.

    A stored fidelity record represents a fidelity CHECKPOINT:

        fidelity     = state quality at recorded_slot
        recorded_slot = time of that checkpoint

    Current fidelity is calculated deterministically from that
    checkpoint and the elapsed physical model time.

    IMPORTANT
    ---------
    recorded_slot is NOT physical-qubit birth time.

    Physical-qubit age remains governed by
    LocalEntanglementHalf.stored_at_slot.

    Swapping may therefore create a new fidelity checkpoint
    without resetting the age of the surviving physical qubits.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        profile: WernerPhysicalFidelityProfile
    ):

        super().__init__(
            simulator
        )

        if not isinstance(
            profile,
            WernerPhysicalFidelityProfile
        ):
            raise TypeError(
                "profile must be a "
                "WernerPhysicalFidelityProfile."
            )

        self.profile = (
            profile
        )

    # =====================================================================
    # Elementary Resource Registration
    # =====================================================================

    def register_elementary(
        self,
        resource: EntanglementResource
    ) -> EntanglementFidelityRecord:
        """
        Register an elementary EP at its ACTUAL creation slot.

        This matters when one elementary link succeeded several
        slots before the full path became ready.
        """

        if not isinstance(
            resource,
            EntanglementResource
        ):
            raise TypeError(
                "resource must be an EntanglementResource."
            )

        if (
            resource.kind
            != EntanglementKind.ELEMENTARY
        ):
            raise ValueError(
                "register_elementary requires an "
                "ELEMENTARY resource."
            )

        if (
            resource.resource_id
            in self._records
        ):
            raise ValueError(
                f"Fidelity already registered for "
                f"'{resource.resource_id}'."
            )

        record = (
            EntanglementFidelityRecord(

                resource_id=(
                    resource.resource_id
                ),

                fidelity=(
                    self.profile
                    .elementary_fidelity
                ),

                model_name=(
                    "PAPER_CALIBRATED_ELEMENTARY_FIDELITY"
                ),

                # Critical:
                # fidelity evolution starts when this EP
                # physically came into existence.
                recorded_slot=(
                    resource.created_slot
                ),

                parent_resource_ids=()
            )
        )

        self._records[
            resource.resource_id
        ] = record

        return record

    # =====================================================================
    # Historical Fidelity Evaluation
    # =====================================================================

    def fidelity_at_slot(
        self,
        resource_or_id,
        simulation_slot: int
    ) -> float:
        """
        Evaluate fidelity at an explicit simulation slot.

        Useful for auditing historical BSM attempts.
        """

        record = (
            self.get(
                resource_or_id
            )
        )

        if (
            isinstance(simulation_slot, bool)
            or not isinstance(
                simulation_slot,
                numbers.Integral
            )
        ):
            raise TypeError(
                "simulation_slot must be an integer."
            )

        simulation_slot = int(
            simulation_slot
        )

        if (
            simulation_slot
            < record.recorded_slot
        ):
            raise PhysicalFidelityModelError(
                "Cannot evaluate fidelity before its "
                "registered fidelity checkpoint."
            )

        elapsed_slots = (
            simulation_slot
            - record.recorded_slot
        )

        elapsed_seconds = (
            self.profile
            .elapsed_seconds_for_slots(
                elapsed_slots
            )
        )

        return (
            self.profile
            .memory_fidelity(

                initial_fidelity=(
                    record.fidelity
                ),

                elapsed_seconds=(
                    elapsed_seconds
                )
            )
        )

    # =====================================================================
    # Current Fidelity
    # =====================================================================

    def fidelity(
        self,
        resource_or_id
    ) -> float:
        """
        Override the Phase-3C.1 static lookup.

        Existing EntanglementSwapEngine therefore automatically
        receives CURRENT, decohered input fidelities.
        """

        return (
            self.fidelity_at_slot(

                resource_or_id,

                self.simulator
                .kernel
                .current_slot
            )
        )

    current_fidelity = fidelity

    # =====================================================================
    # Diagnostics
    # =====================================================================

    def dataframe(
        self
    ) -> pd.DataFrame:

        rows = []

        for record in (
            self._records.values()
        ):

            resource = (
                self.simulator
                .entanglement_manager
                .get_resource(
                    record.resource_id
                )
            )

            elapsed_slots = (
                self.simulator.kernel.current_slot
                - record.recorded_slot
            )

            elapsed_seconds = (
                self.profile
                .elapsed_seconds_for_slots(
                    max(
                        elapsed_slots,
                        0
                    )
                )
            )

            current_fidelity = None

            if resource.is_active:

                current_fidelity = (
                    self.fidelity(
                        record.resource_id
                    )
                )

            rows.append({

                "Resource ID":
                    record.resource_id,

                "Kind":
                    resource.kind.value,

                "Status":
                    resource.status.value,

                "Checkpoint Fidelity":
                    record.fidelity,

                "Checkpoint Slot":
                    record.recorded_slot,

                "Elapsed Slots":
                    max(
                        elapsed_slots,
                        0
                    ),

                "Elapsed Physical Time (s)":
                    elapsed_seconds,

                "Current Fidelity":
                    current_fidelity,

                "Model":
                    record.model_name,

                "Parents":
                    record.parent_resource_ids
            })

        return pd.DataFrame(
            rows
        )

In [ ]:
# @title  Werner-State Physical Atomic BSM Model


class WernerStateAtomicBSMModel(
    EntanglementSwapModel
):
    """
    Atomic-BSM model with:

        - configurable stochastic BSM success;
        - Werner-state fidelity propagation;
        - optional additional local operation visibility loss.

    Default BSM probability remains the paper-inspired:

        p_b = 0.40

    Fidelity and BSM success remain independent concepts.
    """

    def __init__(
        self,
        fidelity_profile: WernerPhysicalFidelityProfile,
        atomic_bsm_success_probability: float = (
            PHASE3C_DEFAULT_ATOMIC_BSM_SUCCESS_PROBABILITY
        ),
        atomic_bsm_latency_us: float = (
            PHASE3C_DEFAULT_ATOMIC_BSM_LATENCY_US
        )
    ):

        if not isinstance(
            fidelity_profile,
            WernerPhysicalFidelityProfile
        ):
            raise TypeError(
                "fidelity_profile must be a "
                "WernerPhysicalFidelityProfile."
            )

        if (
            isinstance(
                atomic_bsm_success_probability,
                bool
            )
            or not isinstance(
                atomic_bsm_success_probability,
                numbers.Real
            )
        ):
            raise TypeError(
                "atomic_bsm_success_probability must be real."
            )

        probability = float(
            atomic_bsm_success_probability
        )

        if not (
            0.0
            <= probability
            <= 1.0
        ):
            raise ValueError(
                "Atomic BSM probability must lie in [0,1]."
            )

        if (
            isinstance(
                atomic_bsm_latency_us,
                bool
            )
            or not isinstance(
                atomic_bsm_latency_us,
                numbers.Real
            )
            or not math.isfinite(
                float(
                    atomic_bsm_latency_us
                )
            )
            or float(
                atomic_bsm_latency_us
            ) < 0.0
        ):
            raise ValueError(
                "Atomic BSM latency must be a non-negative "
                "finite real number."
            )

        self.fidelity_profile = (
            fidelity_profile
        )

        self._p_b = probability

        self._t_b_us = float(
            atomic_bsm_latency_us
        )

    @property
    def model_name(
        self
    ) -> str:

        return (
            "WernerStateAtomicBSMModel"
            f"(p_b={self._p_b:.3f})"
        )

    @property
    def atomic_bsm_success_probability(
        self
    ) -> float:

        return self._p_b

    @property
    def atomic_bsm_latency_us(
        self
    ) -> float:

        return self._t_b_us

    # =====================================================================
    # BSM Success Physics
    # =====================================================================

    def success_probability(
        self,
        swap_node,
        left_resource,
        right_resource
    ) -> float:

        return self._p_b

    # =====================================================================
    # Fidelity Physics
    # =====================================================================

    def output_fidelity(
        self,
        left_fidelity: float,
        right_fidelity: float
    ) -> float:

        return (
            self.fidelity_profile
            .swapped_fidelity(
                left_fidelity,
                right_fidelity
            )
        )

    def __repr__(
        self
    ) -> str:

        return (
            "WernerStateAtomicBSMModel("
            f"p_b={self._p_b:.3f}, "
            f"t_b={self._t_b_us:.2f} us, "
            f"swap_visibility="
            f"{self.fidelity_profile.swap_operation_visibility:.6f}"
            ")"
        )

In [ ]:
# @title Physical Fidelity Mathematical Verification


print("=" * 82)
print("Fidelity Model Calibration Verification")
print("=" * 82)


physical_fidelity_profile = (
    WernerPhysicalFidelityProfile(

        elementary_fidelity=0.995,

        memory_reference_initial_fidelity=1.0,

        memory_reference_fidelity=0.90,

        memory_reference_time_s=1.0,

        # Explicit TEST physical clock.
        #
        # This is deliberately visible rather than hidden.
        slot_duration_s=0.05,

        swap_operation_visibility=1.0
    )
)


# ============================================================================
# Test 1 — Bell-state fidelity floor
# ============================================================================

assert math.isclose(
    physical_fidelity_profile.bell_fidelity_floor,
    0.25,
    abs_tol=1e-12
)

print(
    "[PASS] Completely depolarized Bell-state fidelity floor "
    "is F = 0.25."
)


# ============================================================================
# Test 2 — Paper-inspired elementary baseline
# ============================================================================

assert math.isclose(
    physical_fidelity_profile.elementary_fidelity,
    0.995,
    abs_tol=1e-12
)

print(
    "[PASS] Elementary Bell-pair fidelity baseline is F = 0.995."
)


# ============================================================================
# Test 3 — Zero elapsed time causes ZERO degradation
# ============================================================================

assert math.isclose(
    physical_fidelity_profile.memory_fidelity(
        1.0,
        0.0
    ),
    1.0,
    abs_tol=1e-12
)

assert math.isclose(
    physical_fidelity_profile.memory_fidelity(
        0.995,
        0.0
    ),
    0.995,
    abs_tol=1e-12
)

print(
    "[PASS] Fidelity is unchanged at zero storage time."
)


# ============================================================================
# Test 4 — Calibration point:
#
# Perfect Bell pair after 1 second -> F = 0.90
# ============================================================================

perfect_after_one_second = (
    physical_fidelity_profile
    .memory_fidelity(
        1.0,
        1.0
    )
)


assert math.isclose(
    perfect_after_one_second,
    0.90,
    abs_tol=1e-12
)

print(
    "[PASS] Memory-decoherence model exactly reproduces "
    "the F(1 s) = 0.90 calibration target."
)


# ============================================================================
# Test 5 — Decoherence must be monotonic
# ============================================================================

f_0 = (
    physical_fidelity_profile
    .memory_fidelity(
        0.995,
        0.0
    )
)

f_half = (
    physical_fidelity_profile
    .memory_fidelity(
        0.995,
        0.5
    )
)

f_1 = (
    physical_fidelity_profile
    .memory_fidelity(
        0.995,
        1.0
    )
)

f_2 = (
    physical_fidelity_profile
    .memory_fidelity(
        0.995,
        2.0
    )
)


assert (
    f_0
    > f_half
    > f_1
    > f_2
    > 0.25
)

print(
    "[PASS] Time-driven fidelity degradation is monotonic "
    "and asymptotically approaches F = 0.25."
)


# ============================================================================
# Test 6 — Fidelity <-> Werner visibility round trip
# ============================================================================

for test_fidelity in (
    1.0,
    0.995,
    0.90,
    0.75,
    0.50,
    0.25
):

    visibility = (
        physical_fidelity_profile
        .fidelity_to_visibility(
            test_fidelity
        )
    )

    reconstructed = (
        physical_fidelity_profile
        .visibility_to_fidelity(
            visibility
        )
    )

    assert math.isclose(
        reconstructed,
        test_fidelity,
        abs_tol=1e-12
    )


print(
    "[PASS] Fidelity/visibility mapping is mathematically reversible."
)


# ============================================================================
# Test 7 — Perfect inputs remain perfect under perfect local swapping
# ============================================================================

assert math.isclose(
    physical_fidelity_profile
    .swapped_fidelity(
        1.0,
        1.0
    ),
    1.0,
    abs_tol=1e-12
)

print(
    "[PASS] Perfect Werner inputs remain perfect under "
    "an ideal local Bell operation."
)


# ============================================================================
# Test 8 — Imperfect inputs degrade through swapping
# ============================================================================

swapped_995 = (
    physical_fidelity_profile
    .swapped_fidelity(
        0.995,
        0.995
    )
)


assert (
    0.25
    < swapped_995
    < 0.995
)

print(
    "[PASS] Two imperfect Bell pairs yield a lower-fidelity "
    "swapped Bell pair."
)


# ============================================================================
# Diagnostic sweep
# ============================================================================

fidelity_sweep = pd.DataFrame([
    {
        "Storage Time (s)":
            t,

        "F(initial=0.995)":
            physical_fidelity_profile
            .memory_fidelity(
                0.995,
                t
            ),

        "F(initial=1.0)":
            physical_fidelity_profile
            .memory_fidelity(
                1.0,
                t
            )
    }

    for t in (
        0.0,
        0.25,
        0.50,
        1.0,
        2.0,
        5.0,
        10.0
    )
])


print("\nPHYSICAL FIDELITY PROFILE")

print(
    physical_fidelity_profile
    .parameter_summary()
)


print("\nMEMORY-DECOHERENCE SWEEP")

display(
    fidelity_sweep
)


print("\nF(0.995, 0.995) after swapping:")

print(
    f"{swapped_995:.12f}"
)


print("\n" + "=" * 82)
print(
    "Fidelity mathematical verification completed successfully."
)
print("=" * 82)

In [ ]:
# @title Six-Repeater Physical Fidelity End-to-End Verification


print("=" * 86)
print("Six-Repeater Physical Fidelity Verification")
print("=" * 86)


# ============================================================================
# PHYSICAL NETWORK
#
# Fid-A -- R1 -- R2 -- R3 -- R4 -- R5 -- R6 -- Fid-B
#
# 6 repeaters
# 7 elementary physical links
# each link = 5 km
# ============================================================================

fidelity_topology = NetworkTopology(
    name="Phase3C4-Six-Repeater-Fidelity-Network"
)


fidelity_a = NetworkNode(
    name="Fid-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


fidelity_repeaters = []


for repeater_index in range(
    1,
    7
):

    fidelity_repeaters.append(

        NetworkNode(
            name=f"Fid-R{repeater_index}",
            x_km=5 * repeater_index,
            y_km=0,
            endpoint_capable=False,
            swapping_capable=True,
            memory_capacity=2
        )
    )


fidelity_b = NetworkNode(
    name="Fid-B",
    x_km=35,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


fidelity_path = (
    fidelity_a,
    *fidelity_repeaters,
    fidelity_b
)


for node in fidelity_path:

    fidelity_topology.add_node(
        node
    )


for left, right in zip(
    fidelity_path[:-1],
    fidelity_path[1:]
):

    fidelity_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert (
    fidelity_topology.node_count
    == 8
)


assert (
    fidelity_topology.link_count
    == 7
)


assert (
    len(
        fidelity_repeaters
    )
    == 6
)


print(
    "[PASS] Six-repeater fidelity test network contains "
    "8 physical nodes and 7 elementary links."
)


# ============================================================================
# Fresh simulator
#
# Large lifetime prevents the hard expiration threshold from
# interfering with this fidelity-only verification.
# ============================================================================

fidelity_simulator = (
    QuantumNetworkSimulator(

        topology=(
            fidelity_topology
        ),

        simulation_seed=(
            340404
        ),

        controller_id=(
            "PHASE3C4-CONTROLLER"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=1000
    )
)


# ============================================================================
# Authenticate all devices
# ============================================================================

fidelity_authentication = (
    fidelity_simulator
    .authenticate_all_nodes()
)


assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in fidelity_authentication.values()
)


assert (
    fidelity_simulator
    .require_quantum_eligibility(
        *fidelity_path
    )
)


print(
    "[PASS] All users and repeaters are mutually authenticated."
)


# ============================================================================
# Parallel elementary EP generation
# ============================================================================

fidelity_generation_service = (
    ElementaryEntanglementService(

        simulator=(
            fidelity_simulator
        ),

        model=(
            FiberSurvivalProxyModel()
        )
    )
)


fidelity_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=(
            fidelity_simulator
        ),

        service=(
            fidelity_generation_service
        ),

        ordered_path=(
            fidelity_path
        )
    )
)


fidelity_elementary_resources = (
    fidelity_generation_scheduler
    .run_until_all_ready(
        max_slots=500
    )
)


assert (
    len(
        fidelity_elementary_resources
    )
    == 7
)


print(
    "[PASS] Seven elementary Bell pairs are simultaneously ready."
)


# ============================================================================
# Dynamic physical-fidelity registry
# ============================================================================

fidelity_registry = (
    PhysicalEntanglementFidelityRegistry(

        simulator=(
            fidelity_simulator
        ),

        profile=(
            physical_fidelity_profile
        )
    )
)


# Register each elementary resource at its ACTUAL generation slot.
for resource in (
    fidelity_elementary_resources
):

    fidelity_registry.register_elementary(
        resource
    )


# ============================================================================
# TEST — Waiting time has already affected early EPs
# ============================================================================

elementary_fidelities_before_swapping = {
    resource.resource_id:
        fidelity_registry.fidelity(
            resource
        )
    for resource
    in fidelity_elementary_resources
}


assert all(
    (
        physical_fidelity_profile
        .bell_fidelity_floor
        <
        fidelity
        <=
        physical_fidelity_profile
        .elementary_fidelity
    )
    for fidelity
    in elementary_fidelities_before_swapping.values()
)


print(
    "[PASS] Elementary EP fidelity reflects actual waiting "
    "time since each resource's generation slot."
)


# ============================================================================
# Physical fidelity BSM model
#
# P_BSM = 1 here ONLY to isolate fidelity propagation.
#
# Full physical configuration remains p_b = 0.4 from Phase 3C.2.
# ============================================================================

fidelity_swap_model = (
    WernerStateAtomicBSMModel(

        fidelity_profile=(
            physical_fidelity_profile
        ),

        atomic_bsm_success_probability=1.0,

        atomic_bsm_latency_us=10.0
    )
)


fidelity_swap_engine = (
    EntanglementSwapEngine(

        simulator=(
            fidelity_simulator
        ),

        model=(
            fidelity_swap_model
        ),

        fidelity_registry=(
            fidelity_registry
        )
    )
)


# ============================================================================
# Control-plane parallel swap scheduler
# ============================================================================

fidelity_swap_scheduler = (
    ControlPlaneParallelSwapScheduler(

        simulator=(
            fidelity_simulator
        ),

        swap_engine=(
            fidelity_swap_engine
        ),

        ordered_path=(
            fidelity_path
        ),

        policy=(
            MaxParallelOldestFirstSwapPolicy()
        )
    )
)


# ============================================================================
# SWAP ROUND 1
# ============================================================================

fidelity_round_1 = (
    fidelity_swap_scheduler
    .execute_parallel_round()
)


assert (
    fidelity_round_1.selected_count
    == 3
)

assert (
    fidelity_round_1.success_count
    == 3
)


print(
    "[PASS] Fidelity-aware Round 1 performs three "
    "parallel BSMs."
)


# Explicit swap-round boundary.
fidelity_simulator.kernel.advance_slot()


# ============================================================================
# SWAP ROUND 2
# ============================================================================

fidelity_round_2 = (
    fidelity_swap_scheduler
    .execute_parallel_round()
)


assert (
    fidelity_round_2.selected_count
    == 2
)

assert (
    fidelity_round_2.success_count
    == 2
)


print(
    "[PASS] Fidelity-aware Round 2 performs two "
    "parallel BSMs."
)


fidelity_simulator.kernel.advance_slot()


# ============================================================================
# SWAP ROUND 3
# ============================================================================

fidelity_round_3 = (
    fidelity_swap_scheduler
    .execute_parallel_round()
)


assert (
    fidelity_round_3.selected_count
    == 1
)

assert (
    fidelity_round_3.success_count
    == 1
)


print(
    "[PASS] Fidelity-aware Round 3 performs the final BSM."
)


# ============================================================================
# There must now be exactly one active end-to-end EP
# ============================================================================

fidelity_active_resources = (
    fidelity_simulator
    .entanglement_manager
    .active_resources
)


assert (
    len(
        fidelity_active_resources
    )
    == 1
)


fidelity_final_resource = (
    fidelity_active_resources[0]
)


assert (
    fidelity_final_resource.endpoint_names
    == tuple(
        sorted(
            (
                fidelity_a.name,
                fidelity_b.name
            )
        )
    )
)


print(
    "[PASS] Final resource establishes Fid-A ↔ Fid-B "
    "entanglement."
)


# ============================================================================
# AUDIT EVERY BSM'S INPUT FIDELITY
#
# The swap engine's recorded input fidelity must equal:
#
# checkpoint fidelity
#     degraded over
# checkpoint_slot -> BSM_slot
# ============================================================================

for attempt in (
    fidelity_swap_engine.attempts
):

    expected_left = (
        fidelity_registry
        .fidelity_at_slot(

            attempt.left_resource_id,

            attempt.simulation_slot
        )
    )


    expected_right = (
        fidelity_registry
        .fidelity_at_slot(

            attempt.right_resource_id,

            attempt.simulation_slot
        )
    )


    assert math.isclose(
        attempt.left_fidelity,
        expected_left,
        rel_tol=0.0,
        abs_tol=1e-12
    )


    assert math.isclose(
        attempt.right_fidelity,
        expected_right,
        rel_tol=0.0,
        abs_tol=1e-12
    )


print(
    "[PASS] Every BSM consumed the correctly decohered "
    "current fidelity of both input resources."
)


# ============================================================================
# AUDIT EVERY SUCCESSFUL BSM'S OUTPUT FIDELITY
# ============================================================================

for attempt in (
    fidelity_swap_engine.attempts
):

    assert attempt.success

    expected_output = (
        physical_fidelity_profile
        .swapped_fidelity(

            attempt.left_fidelity,

            attempt.right_fidelity
        )
    )


    assert math.isclose(
        attempt.output_fidelity,
        expected_output,
        rel_tol=0.0,
        abs_tol=1e-12
    )


    parent_record = (
        fidelity_registry.get(
            attempt.output_resource_id
        )
    )


    assert math.isclose(
        parent_record.fidelity,
        expected_output,
        rel_tol=0.0,
        abs_tol=1e-12
    )


    assert (
        parent_record.recorded_slot
        == attempt.simulation_slot
    )


print(
    "[PASS] Every successful BSM obeys the Werner-state "
    "fidelity transformation exactly."
)


# ============================================================================
# FINAL FIDELITY MUST NOW BE NON-IDEAL
# ============================================================================

final_fidelity_at_creation = (
    fidelity_registry
    .fidelity(
        fidelity_final_resource
    )
)


assert (
    physical_fidelity_profile
    .bell_fidelity_floor
    <
    final_fidelity_at_creation
    <
    physical_fidelity_profile
    .elementary_fidelity
)


assert not math.isclose(
    final_fidelity_at_creation,
    1.0,
    abs_tol=1e-12
)


print(
    "[PASS] Six-repeater end-to-end fidelity is physically "
    "non-ideal and remains above the maximally mixed floor."
)


# ============================================================================
# Provenance remains complete
# ============================================================================

assert (
    len(
        fidelity_final_resource
        .source_link_ids
    )
    == 7
)


print(
    "[PASS] Physical fidelity modeling preserves complete "
    "seven-link entanglement provenance."
)


# ============================================================================
# CRITICAL AGE / FIDELITY-CHECKPOINT DISTINCTION
#
# Parent fidelity checkpoint is new,
# but surviving outer physical qubits must still carry their
# original stored_at_slot.
# ============================================================================

final_a_half = (
    fidelity_simulator
    .quantum_memory(
        fidelity_a
    )
    .get_local_half(
        fidelity_final_resource.resource_id
    )
)


final_b_half = (
    fidelity_simulator
    .quantum_memory(
        fidelity_b
    )
    .get_local_half(
        fidelity_final_resource.resource_id
    )
)


final_fidelity_record = (
    fidelity_registry.get(
        fidelity_final_resource
    )
)


assert (
    final_fidelity_record.recorded_slot
    == fidelity_final_resource.created_slot
)


assert (
    final_a_half.stored_at_slot
    <= final_fidelity_record.recorded_slot
)


assert (
    final_b_half.stored_at_slot
    <= final_fidelity_record.recorded_slot
)


print(
    "[PASS] Fidelity checkpoint creation does not reset "
    "the physical age of surviving end-user qubits."
)


# ============================================================================
# Deliberately wait another 1 physical-model second.
#
# With slot_duration = 0.05 s:
#
#     20 slots = 1 second
# ============================================================================

ONE_SECOND_SLOTS = int(
    round(
        1.0
        /
        physical_fidelity_profile
        .slot_duration_s
    )
)


assert math.isclose(
    physical_fidelity_profile
    .elapsed_seconds_for_slots(
        ONE_SECOND_SLOTS
    ),
    1.0,
    abs_tol=1e-12
)


final_checkpoint_fidelity = (
    final_fidelity_record.fidelity
)


fidelity_simulator\
    .kernel\
    .advance_slot(
        ONE_SECOND_SLOTS
    )


final_fidelity_after_one_second = (
    fidelity_registry
    .fidelity(
        fidelity_final_resource
    )
)


expected_final_after_one_second = (
    physical_fidelity_profile
    .memory_fidelity(

        final_checkpoint_fidelity,

        1.0
    )
)


assert math.isclose(
    final_fidelity_after_one_second,
    expected_final_after_one_second,
    rel_tol=0.0,
    abs_tol=1e-12
)


assert (
    final_fidelity_after_one_second
    <
    final_checkpoint_fidelity
)


print(
    "[PASS] Final end-to-end EP continues to decohere "
    "after establishment according to physical storage time."
)


# ============================================================================
# Fidelity is deterministic state quality — NOT a random event
# ============================================================================

repeated_fidelity_reads = tuple(

    fidelity_registry
    .fidelity(
        fidelity_final_resource
    )

    for _ in range(
        10
    )
)


assert all(
    math.isclose(
        value,
        repeated_fidelity_reads[0],
        abs_tol=1e-15
    )
    for value
    in repeated_fidelity_reads
)


print(
    "[PASS] Fidelity behaves as deterministic state quality, "
    "not as a probability-of-perfect-pair coin flip."
)


# ============================================================================
# Complete runtime consistency
# ============================================================================

assert (
    fidelity_simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Complete six-repeater runtime remains "
    "cross-layer consistent with physical fidelity enabled."
)


# ============================================================================
# Diagnostics
# ============================================================================

print("\nFIDELITY PROFILE")

print(
    physical_fidelity_profile
    .parameter_summary()
)


print("\nELEMENTARY FIDELITIES BEFORE SWAPPING")

display(
    pd.DataFrame([
        {
            "Resource ID":
                resource.resource_id,

            "Created Slot":
                resource.created_slot,

            "Current Slot Before Swapping":
                fidelity_round_1.simulation_slot,

            "Current Fidelity":
                elementary_fidelities_before_swapping[
                    resource.resource_id
                ]
        }

        for resource in (
            fidelity_elementary_resources
        )
    ])
)


print("\nALL PHYSICAL-FIDELITY BSM ATTEMPTS")

display(
    fidelity_swap_engine
    .attempts_dataframe()
)


print("\nPARALLEL SWAP ROUND HISTORY")

display(
    fidelity_swap_scheduler
    .round_history_dataframe()
)


print("\nCOMPLETE PHYSICAL FIDELITY REGISTRY")

display(
    fidelity_registry
    .dataframe()
)


print("\nFINAL RESOURCE")

print(
    fidelity_final_resource
)


print(
    "\nFinal fidelity at establishment:",
    f"{final_fidelity_at_creation:.12f}"
)


print(
    "Final fidelity after another 1 physical-model second:",
    f"{final_fidelity_after_one_second:.12f}"
)


print("\n" + "=" * 86)
print(
    "Six-repeater physical fidelity verification "
    "completed successfully."
)
print("=" * 86)

In [ ]:
# @title Failure-Aware Elementary Demand State

from dataclasses import dataclass
from enum import Enum
from typing import Optional


# =============================================================================
# Errors
# =============================================================================

class FailureAwareDemandError(Exception):
    """
    Base exception for Phase-3D failure-aware resource demand.
    """
    pass


class EntanglementCoverageConflictError(
    FailureAwareDemandError
):
    """
    Raised when multiple ACTIVE resources simultaneously claim
    the same elementary physical-link coverage.
    """
    pass


class EntanglementCoverageConsistencyError(
    FailureAwareDemandError
):
    pass


# =============================================================================
# Link Demand Classification
# =============================================================================

class ElementaryLinkDemandClass(str, Enum):

    # The physical link already has its own active elementary EP.
    ACTIVE_ELEMENTARY_READY = (
        "ACTIVE_ELEMENTARY_READY"
    )

    # The physical link has been absorbed into a larger active
    # swapped resource. Its elementary children must NOT regenerate.
    COVERED_BY_ACTIVE_PARENT = (
        "COVERED_BY_ACTIVE_PARENT"
    )

    # No active entanglement resource currently covers this
    # physical link. Fresh elementary generation is required.
    REGENERATION_REQUIRED = (
        "REGENERATION_REQUIRED"
    )


# =============================================================================
# One Link's Control-Plane Decision
# =============================================================================

@dataclass(frozen=True)
class ElementaryLinkDemandDecision:

    simulation_slot: int

    link_index: int

    link_id: str

    node_a_name: str
    node_b_name: str

    demand_class: ElementaryLinkDemandClass

    demand_active: bool

    covering_resource_id: Optional[str]

    covering_resource_kind: Optional[
        EntanglementKind
    ]

    def __post_init__(self):

        if self.link_index < 0:
            raise ValueError(
                "link_index must be non-negative."
            )

        if (
            self.demand_class
            == ElementaryLinkDemandClass.REGENERATION_REQUIRED
        ):

            if self.covering_resource_id is not None:
                raise ValueError(
                    "An uncovered link cannot have a "
                    "covering resource."
                )

        else:

            if self.covering_resource_id is None:
                raise ValueError(
                    "Covered/ready link must identify its "
                    "active covering resource."
                )


# =============================================================================
# Full Reconciliation Record
# =============================================================================

@dataclass(frozen=True)
class FailureAwareDemandReconciliation:

    reconciliation_index: int

    simulation_slot: int

    active_frontier_resource_ids: tuple[str, ...]

    decisions: tuple[
        ElementaryLinkDemandDecision, ...
    ]

    @property
    def regeneration_link_ids(
        self
    ) -> tuple[str, ...]:

        return tuple(
            decision.link_id
            for decision in self.decisions
            if (
                decision.demand_class
                == ElementaryLinkDemandClass.REGENERATION_REQUIRED
            )
        )

    @property
    def protected_link_ids(
        self
    ) -> tuple[str, ...]:

        return tuple(
            decision.link_id
            for decision in self.decisions
            if (
                decision.demand_class
                == ElementaryLinkDemandClass.COVERED_BY_ACTIVE_PARENT
            )
        )

    @property
    def ready_elementary_link_ids(
        self
    ) -> tuple[str, ...]:

        return tuple(
            decision.link_id
            for decision in self.decisions
            if (
                decision.demand_class
                == ElementaryLinkDemandClass.ACTIVE_ELEMENTARY_READY
            )
        )

In [ ]:
# @title  Failure-Aware Elementary Demand Controller


class FailureAwareElementaryDemandController:
    """
    Derive elementary-link generation demand from the ACTIVE
    entanglement frontier.

    Core principle
    --------------
    Entanglement provenance determines which elementary physical
    links are already represented by useful higher-level state.

    For each elementary physical link:

        ACTIVE elementary EP
            -> keep process active/READY
            -> no redundant generation

        covered by ACTIVE SWAPPED resource
            -> deactivate elementary generation demand

        covered by nothing
            -> activate regeneration demand

    This controller never:
        - performs quantum generation;
        - performs BSMs;
        - advances simulation time;
        - modifies EntanglementResource lifecycle state.

    It only controls elementary-link DEMAND.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        generation_scheduler: ParallelElementaryGenerationScheduler,
        ordered_path
    ):

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            generation_scheduler,
            ParallelElementaryGenerationScheduler
        ):
            raise TypeError(
                "generation_scheduler must be a "
                "ParallelElementaryGenerationScheduler."
            )

        if (
            generation_scheduler.simulator
            is not simulator
        ):
            raise ValueError(
                "Generation scheduler belongs to another simulator."
            )

        self.simulator = simulator

        self.generation_scheduler = (
            generation_scheduler
        )

        self.ordered_path = (
            simulator.topology
            .validate_path(
                ordered_path
            )
        )

        self.path_links = (
            simulator.topology
            .path_links(
                self.ordered_path
            )
        )

        # Ensure demand controller and generation scheduler refer
        # to exactly the same physical path.
        controller_path_names = tuple(
            node.name
            for node in self.ordered_path
        )

        scheduler_path_names = tuple(
            node.name
            for node in (
                generation_scheduler
                .ordered_path
            )
        )

        if (
            controller_path_names
            != scheduler_path_names
        ):
            raise ValueError(
                "Failure-aware demand controller and elementary "
                "generation scheduler must manage the same "
                "ordered physical path."
            )

        self._link_index_by_id = {
            link.link_id: index
            for index, link
            in enumerate(
                self.path_links
            )
        }

        self._history: list[
            FailureAwareDemandReconciliation
        ] = []

    # =========================================================================
    # Validate Resource Coverage
    # =========================================================================

    def _covered_link_indices(
        self,
        resource: EntanglementResource
    ) -> tuple[int, ...]:
        """
        Resolve which elementary physical links of this ordered
        path are represented by one ACTIVE resource.

        The resource provenance must correspond to one contiguous
        path interval.
        """

        if not isinstance(
            resource,
            EntanglementResource
        ):
            raise TypeError(
                "resource must be an EntanglementResource."
            )

        if not resource.is_active:
            raise EntanglementCoverageConsistencyError(
                f"Resource '{resource.resource_id}' is not ACTIVE."
            )

        source_link_ids = tuple(
            resource.source_link_ids
        )

        if not source_link_ids:

            raise EntanglementCoverageConsistencyError(
                f"ACTIVE resource '{resource.resource_id}' "
                f"has no physical-link provenance."
            )

        unknown_links = tuple(
            link_id
            for link_id in source_link_ids
            if (
                link_id
                not in self._link_index_by_id
            )
        )

        # A path-specific controller must never partially interpret
        # a resource that belongs to some different physical path.
        if unknown_links:

            raise EntanglementCoverageConsistencyError(
                f"Resource '{resource.resource_id}' contains "
                f"physical links outside the managed path: "
                f"{unknown_links}"
            )

        indices = tuple(
            sorted(
                self._link_index_by_id[
                    link_id
                ]
                for link_id
                in source_link_ids
            )
        )

        # No duplicate provenance.
        if (
            len(indices)
            != len(set(indices))
        ):

            raise EntanglementCoverageConsistencyError(
                f"Resource '{resource.resource_id}' repeats "
                f"physical-link provenance."
            )

        expected_indices = tuple(
            range(
                indices[0],
                indices[-1] + 1
            )
        )

        if indices != expected_indices:

            raise EntanglementCoverageConsistencyError(
                f"Resource '{resource.resource_id}' does not "
                f"represent one contiguous path interval."
            )

        # ---------------------------------------------------------------------
        # Validate resource endpoints against the boundaries of the
        # physical interval represented by provenance.
        # ---------------------------------------------------------------------

        expected_outer_names = {
            self.ordered_path[
                indices[0]
            ].name,

            self.ordered_path[
                indices[-1] + 1
            ].name
        }

        if (
            set(resource.endpoint_names)
            != expected_outer_names
        ):

            raise EntanglementCoverageConsistencyError(
                f"Resource '{resource.resource_id}' endpoints "
                f"do not match its physical provenance interval."
            )

        # ---------------------------------------------------------------------
        # Elementary resources must cover exactly one physical link.
        # ---------------------------------------------------------------------

        if (
            resource.kind
            == EntanglementKind.ELEMENTARY
            and len(indices) != 1
        ):

            raise EntanglementCoverageConsistencyError(
                f"Elementary resource '{resource.resource_id}' "
                f"covers more than one physical link."
            )

        return indices

    # =========================================================================
    # Analyze Current ACTIVE Frontier
    # =========================================================================

    def analyze_active_coverage(
        self
    ) -> dict[
        str,
        Optional[EntanglementResource]
    ]:
        """
        Map every elementary physical link to its single ACTIVE
        covering resource, or None if the link is currently
        uncovered.

        Overlapping ACTIVE resource coverage is illegal.
        """

        self.simulator\
            .entanglement_manager\
            .synchronize_expirations()

        coverage = {
            link.link_id: None
            for link in self.path_links
        }

        for resource in (
            self.simulator
            .entanglement_manager
            .active_resources
        ):

            indices = (
                self._covered_link_indices(
                    resource
                )
            )

            for link_index in indices:

                link = (
                    self.path_links[
                        link_index
                    ]
                )

                existing = (
                    coverage[
                        link.link_id
                    ]
                )

                if existing is not None:

                    raise EntanglementCoverageConflictError(
                        f"Physical link '{link.link_id}' is "
                        f"simultaneously covered by ACTIVE "
                        f"resources '{existing.resource_id}' and "
                        f"'{resource.resource_id}'."
                    )

                coverage[
                    link.link_id
                ] = resource

        return coverage

    # =========================================================================
    # Demand Reconciliation
    # =========================================================================

    def reconcile(
        self
    ) -> FailureAwareDemandReconciliation:
        """
        Recompute elementary-generation demand from the live
        entanglement frontier.

        This is the central failure-aware control operation.
        """

        self.simulator.validate_runtime_consistency()

        coverage = (
            self.analyze_active_coverage()
        )

        decisions = []

        for (
            link_index,
            link
        ) in enumerate(
            self.path_links
        ):

            process = (
                self.generation_scheduler
                .process_for_link(
                    link
                )
            )

            covering_resource = (
                coverage[
                    link.link_id
                ]
            )

            # =================================================================
            # CASE 1
            #
            # No active resource currently represents this physical link.
            #
            # The failed/expired/discarded subtree therefore requires
            # elementary regeneration.
            # =================================================================

            if covering_resource is None:

                process.activate_demand()

                decision = (
                    ElementaryLinkDemandDecision(

                        simulation_slot=(
                            self.simulator.kernel.current_slot
                        ),

                        link_index=(
                            link_index
                        ),

                        link_id=(
                            link.link_id
                        ),

                        node_a_name=(
                            link.node_a.name
                        ),

                        node_b_name=(
                            link.node_b.name
                        ),

                        demand_class=(
                            ElementaryLinkDemandClass
                            .REGENERATION_REQUIRED
                        ),

                        demand_active=True,

                        covering_resource_id=None,

                        covering_resource_kind=None
                    )
                )

            # =================================================================
            # CASE 2
            #
            # The active resource is itself the elementary Bell pair for
            # this physical link.
            #
            # Demand remains active because the link still "wants" one EP,
            # but the lifecycle process is READY and therefore performs
            # no redundant attempt.
            # =================================================================

            elif (
                covering_resource.kind
                == EntanglementKind.ELEMENTARY
            ):

                if (
                    tuple(
                        covering_resource
                        .source_link_ids
                    )
                    != (
                        link.link_id,
                    )
                ):

                    raise EntanglementCoverageConsistencyError(
                        "Elementary covering resource does not "
                        "match its physical link."
                    )

                process.activate_demand()

                decision = (
                    ElementaryLinkDemandDecision(

                        simulation_slot=(
                            self.simulator.kernel.current_slot
                        ),

                        link_index=(
                            link_index
                        ),

                        link_id=(
                            link.link_id
                        ),

                        node_a_name=(
                            link.node_a.name
                        ),

                        node_b_name=(
                            link.node_b.name
                        ),

                        demand_class=(
                            ElementaryLinkDemandClass
                            .ACTIVE_ELEMENTARY_READY
                        ),

                        demand_active=True,

                        covering_resource_id=(
                            covering_resource
                            .resource_id
                        ),

                        covering_resource_kind=(
                            covering_resource.kind
                        )
                    )
                )

            # =================================================================
            # CASE 3
            #
            # This elementary physical link is already represented by an
            # ACTIVE higher-level swapped resource.
            #
            # Its consumed children MUST NOT regenerate.
            # =================================================================

            elif (
                covering_resource.kind
                == EntanglementKind.SWAPPED
            ):

                process.deactivate_demand()

                decision = (
                    ElementaryLinkDemandDecision(

                        simulation_slot=(
                            self.simulator.kernel.current_slot
                        ),

                        link_index=(
                            link_index
                        ),

                        link_id=(
                            link.link_id
                        ),

                        node_a_name=(
                            link.node_a.name
                        ),

                        node_b_name=(
                            link.node_b.name
                        ),

                        demand_class=(
                            ElementaryLinkDemandClass
                            .COVERED_BY_ACTIVE_PARENT
                        ),

                        demand_active=False,

                        covering_resource_id=(
                            covering_resource
                            .resource_id
                        ),

                        covering_resource_kind=(
                            covering_resource.kind
                        )
                    )
                )

            else:

                raise EntanglementCoverageConsistencyError(
                    f"Unsupported entanglement kind "
                    f"'{covering_resource.kind}'."
                )

            decisions.append(
                decision
            )

        reconciliation = (
            FailureAwareDemandReconciliation(

                reconciliation_index=(
                    len(
                        self._history
                    )
                ),

                simulation_slot=(
                    self.simulator.kernel.current_slot
                ),

                active_frontier_resource_ids=tuple(
                    resource.resource_id
                    for resource
                    in (
                        self.simulator
                        .entanglement_manager
                        .active_resources
                    )
                ),

                decisions=tuple(
                    decisions
                )
            )
        )

        self._history.append(
            reconciliation
        )

        # =====================================================================
        # Immutable control-plane audit event
        # =====================================================================

        self.simulator.kernel.emit(

            event_type=(
                "FAILURE_AWARE_DEMAND_RECONCILED"
            ),

            component=(
                "ClassicalControlPlane."
                "FailureAwareDemandController"
            ),

            node_names=tuple(
                node.name
                for node
                in self.ordered_path
            ),

            link_ids=tuple(
                link.link_id
                for link
                in self.path_links
            ),

            resource_ids=(
                reconciliation
                .active_frontier_resource_ids
            ),

            message=(
                "Control plane reconciled elementary-link "
                "generation demand from the live "
                "entanglement frontier."
            ),

            metadata={
                "regeneration_links":
                    reconciliation
                    .regeneration_link_ids,

                "protected_links":
                    reconciliation
                    .protected_link_ids,

                "ready_elementary_links":
                    reconciliation
                    .ready_elementary_link_ids
            }
        )

        return reconciliation

    # =========================================================================
    # Inspection
    # =========================================================================

    @property
    def history(
        self
    ) -> tuple[
        FailureAwareDemandReconciliation, ...
    ]:

        return tuple(
            self._history
        )

    def decisions_dataframe(
        self,
        reconciliation=None
    ) -> pd.DataFrame:

        if reconciliation is None:

            if not self._history:

                return pd.DataFrame()

            reconciliation = (
                self._history[-1]
            )

        return pd.DataFrame([
            {
                "Link Index":
                    decision.link_index,

                "Link ID":
                    decision.link_id,

                "Node A":
                    decision.node_a_name,

                "Node B":
                    decision.node_b_name,

                "Demand Class":
                    decision.demand_class.value,

                "Demand Active":
                    decision.demand_active,

                "Covering Resource":
                    decision.covering_resource_id,

                "Covering Kind":
                    (
                        decision
                        .covering_resource_kind
                        .value

                        if (
                            decision
                            .covering_resource_kind
                            is not None
                        )

                        else None
                    )
            }

            for decision
            in reconciliation.decisions
        ])

    def history_dataframe(
        self
    ) -> pd.DataFrame:

        return pd.DataFrame([
            {
                "Reconciliation":
                    record.reconciliation_index,

                "Simulation Slot":
                    record.simulation_slot,

                "Active Frontier":
                    record.active_frontier_resource_ids,

                "Ready Elementary":
                    record.ready_elementary_link_ids,

                "Protected":
                    record.protected_link_ids,

                "Regeneration Required":
                    record.regeneration_link_ids
            }

            for record
            in self._history
        ])

In [ ]:
# @title  Six-Repeater Selective Regeneration Verification


print("=" * 88)
print(
    "Six-Repeater Failure-Aware Demand Verification"
)
print("=" * 88)


# =============================================================================
# NETWORK
#
# Demand-A -- R1 -- R2 -- R3 -- R4 -- R5 -- R6 -- Demand-B
#
# 7 elementary physical links
# =============================================================================

demand_topology = NetworkTopology(
    name="Phase3D1-Six-Repeater-Demand-Network"
)


demand_a = NetworkNode(
    name="Demand-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


demand_repeaters = []


for repeater_index in range(
    1,
    7
):

    demand_repeaters.append(

        NetworkNode(
            name=f"Demand-R{repeater_index}",
            x_km=5 * repeater_index,
            y_km=0,
            endpoint_capable=False,
            swapping_capable=True,
            memory_capacity=2
        )
    )


demand_b = NetworkNode(
    name="Demand-B",
    x_km=35,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


demand_path = (
    demand_a,
    *demand_repeaters,
    demand_b
)


for node in demand_path:

    demand_topology.add_node(
        node
    )


for left, right in zip(
    demand_path[:-1],
    demand_path[1:]
):

    demand_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert (
    demand_topology.node_count
    == 8
)

assert (
    demand_topology.link_count
    == 7
)


print(
    "[PASS] Six-repeater demand test contains "
    "8 nodes and 7 elementary physical links."
)


# =============================================================================
# Simulator
# =============================================================================

demand_simulator = (
    QuantumNetworkSimulator(

        topology=demand_topology,

        simulation_seed=350501,

        controller_id="PHASE3D1-CONTROLLER",

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=100
    )
)


demand_simulator.authenticate_all_nodes()


assert (
    demand_simulator
    .require_quantum_eligibility(
        *demand_path
    )
)


print(
    "[PASS] Entire physical path is mutually authenticated."
)


# =============================================================================
# Elementary generation
# =============================================================================

demand_generation_service = (
    ElementaryEntanglementService(

        simulator=demand_simulator,

        model=FiberSurvivalProxyModel()
    )
)


demand_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=demand_simulator,

        service=demand_generation_service,

        ordered_path=demand_path,

        demand_active=True
    )
)


initial_elementary_resources = (
    demand_generation_scheduler
    .run_until_all_ready(
        max_slots=500
    )
)


assert (
    len(
        initial_elementary_resources
    )
    == 7
)


print(
    "[PASS] All seven physical links initially hold "
    "elementary Bell pairs."
)


# =============================================================================
# Demand controller
# =============================================================================

demand_controller = (
    FailureAwareElementaryDemandController(

        simulator=demand_simulator,

        generation_scheduler=(
            demand_generation_scheduler
        ),

        ordered_path=demand_path
    )
)


# =============================================================================
# TEST 1
#
# Before any swapping, all seven links are represented directly
# by active elementary resources.
# =============================================================================

initial_reconciliation = (
    demand_controller.reconcile()
)


assert (
    len(
        initial_reconciliation
        .ready_elementary_link_ids
    )
    == 7
)


assert (
    len(
        initial_reconciliation
        .protected_link_ids
    )
    == 0
)


assert (
    len(
        initial_reconciliation
        .regeneration_link_ids
    )
    == 0
)


print(
    "[PASS] Initial frontier classifies all seven links "
    "as ready elementary resources."
)


# =============================================================================
# Ideal fidelity sidecar + ideal swap engine
#
# We isolate resource-demand behavior here.
# =============================================================================

demand_fidelity = (
    EntanglementFidelityRegistry(
        demand_simulator
    )
)


for resource in (
    initial_elementary_resources
):

    demand_fidelity.register_ideal(
        resource
    )


demand_swap_engine = (
    EntanglementSwapEngine(

        simulator=demand_simulator,

        model=(
            IdealEntanglementSwapModel()
        ),

        fidelity_registry=(
            demand_fidelity
        )
    )
)


demand_swap_scheduler = (
    ControlPlaneParallelSwapScheduler(

        simulator=demand_simulator,

        swap_engine=demand_swap_engine,

        ordered_path=demand_path,

        policy=(
            MaxParallelOldestFirstSwapPolicy()
        )
    )
)


# =============================================================================
# Execute first maximum-parallel round:
#
# 7 elementary resources
#       ↓
# 3 successful BSMs + 1 untouched elementary resource
#       ↓
# 4 active resources
# =============================================================================

first_swap_round = (
    demand_swap_scheduler
    .execute_parallel_round()
)


assert (
    first_swap_round.selected_count
    == 3
)

assert (
    first_swap_round.success_count
    == 3
)


assert (
    len(
        demand_simulator
        .entanglement_manager
        .active_resources
    )
    == 4
)


print(
    "[PASS] First parallel swap round creates three "
    "successful higher-level subtrees."
)


# =============================================================================
# TEST 2 — SUCCESSFUL SUBTREE PROTECTION
# =============================================================================

post_swap_reconciliation = (
    demand_controller.reconcile()
)


# Three two-link swapped parents protect 6 elementary links.
assert (
    len(
        post_swap_reconciliation
        .protected_link_ids
    )
    == 6
)


# One untouched elementary link remains directly ready.
assert (
    len(
        post_swap_reconciliation
        .ready_elementary_link_ids
    )
    == 1
)


# Absolutely nothing should regenerate.
assert (
    len(
        post_swap_reconciliation
        .regeneration_link_ids
    )
    == 0
)


print(
    "[PASS] Successful swapped subtrees suppress regeneration "
    "of all six consumed elementary children."
)


# =============================================================================
# Verify corresponding leaf demand flags.
# =============================================================================

for decision in (
    post_swap_reconciliation.decisions
):

    process = (
        demand_generation_scheduler
        .process_for_link(
            decision.link_id
        )
    )

    if (
        decision.demand_class
        == (
            ElementaryLinkDemandClass
            .COVERED_BY_ACTIVE_PARENT
        )
    ):

        assert not (
            process.demand_active
        )

    elif (
        decision.demand_class
        == (
            ElementaryLinkDemandClass
            .ACTIVE_ELEMENTARY_READY
        )
    ):

        assert (
            process.demand_active
        )


print(
    "[PASS] Elementary process demand flags match "
    "control-plane subtree coverage decisions."
)


# =============================================================================
# Choose exactly ONE successful swapped subtree and remove it.
#
# This simulates loss/expiration/failure of that higher-level
# useful state while leaving every other subtree untouched.
# =============================================================================

active_swapped_parents = tuple(

    resource

    for resource in (
        demand_simulator
        .entanglement_manager
        .active_resources
    )

    if (
        resource.kind
        == EntanglementKind.SWAPPED
    )
)


assert (
    len(
        active_swapped_parents
    )
    == 3
)


lost_parent = (
    active_swapped_parents[0]
)


lost_parent_source_links = tuple(
    lost_parent.source_link_ids
)


assert (
    len(
        lost_parent_source_links
    )
    == 2
)


print(
    "\n[INFO] Simulating loss of successful subtree:",
    lost_parent.resource_id
)

print(
    "[INFO] Underlying elementary links:",
    lost_parent_source_links
)


demand_simulator\
    .entanglement_manager\
    .discard_resource(

        lost_parent.resource_id,

        reason=(
            "phase3d1_selective_regeneration_verification"
        )
    )


assert (
    demand_simulator
    .entanglement_manager
    .get_resource(
        lost_parent.resource_id
    )
    .status
    == EntanglementStatus.DISCARDED
)


print(
    "[PASS] One higher-level subtree was removed while "
    "other successful subtrees remain intact."
)


# =============================================================================
# TEST 3 — ONLY THE LOST SUBTREE'S LEAVES REACTIVATE
# =============================================================================

failure_reconciliation = (
    demand_controller.reconcile()
)


assert set(
    failure_reconciliation
    .regeneration_link_ids
) == set(
    lost_parent_source_links
)


assert (
    len(
        failure_reconciliation
        .regeneration_link_ids
    )
    == 2
)


print(
    "[PASS] Control plane reactivates exactly the two "
    "elementary links belonging to the lost subtree."
)


# =============================================================================
# Surviving higher-level parents MUST remain protected.
# =============================================================================

surviving_swapped_parents = tuple(

    resource

    for resource in (
        demand_simulator
        .entanglement_manager
        .active_resources
    )

    if (
        resource.kind
        == EntanglementKind.SWAPPED
    )
)


assert (
    len(
        surviving_swapped_parents
    )
    == 2
)


surviving_parent_links = set()

for resource in (
    surviving_swapped_parents
):

    surviving_parent_links.update(
        resource.source_link_ids
    )


assert not (
    surviving_parent_links
    &
    set(
        failure_reconciliation
        .regeneration_link_ids
    )
)


print(
    "[PASS] Unaffected successful subtrees remain protected "
    "and are not regenerated."
)


# =============================================================================
# TEST 4
#
# First generation pass after reopening demand only lets old leaf
# processes observe that their previous resources were consumed.
#
# This preserves our Phase-3B/3C round-boundary rule:
#
#     consumed resource detected
#         ↓
#     no same-round regeneration
# =============================================================================

attempts_before_detection = (
    demand_generation_service
    .statistics.total_attempts
)


termination_detection_slot = (
    demand_generation_scheduler
    .process_current_slot()
)


assert (
    demand_generation_service
    .statistics.total_attempts
    == attempts_before_detection
)


print(
    "[PASS] Reopened leaf processes first detect historical "
    "resource consumption without same-round regeneration."
)


# =============================================================================
# Explicit next global slot.
# =============================================================================

demand_simulator\
    .kernel\
    .advance_slot()


# Reconcile again at the new slot.
demand_controller.reconcile()


# =============================================================================
# TEST 5
#
# Now only the two uncovered elementary links should make fresh
# physical attempts.
# =============================================================================

attempts_before_regeneration = (
    demand_generation_service
    .statistics.total_attempts
)


regeneration_slot = (
    demand_generation_scheduler
    .process_current_slot()
)


attempted_link_ids = tuple(

    step.link_id

    for step in (
        regeneration_slot.steps
    )

    if (
        step.attempt
        is not None
    )
)


assert set(
    attempted_link_ids
) == set(
    lost_parent_source_links
)


assert (
    demand_generation_service
    .statistics.total_attempts
    ==
    attempts_before_regeneration
    + 2
)


print(
    "[PASS] Only the failed subtree's two elementary links "
    "perform new physical generation attempts."
)


# =============================================================================
# Verify protected links made ZERO physical attempts.
# =============================================================================

protected_after_failure = set(
    failure_reconciliation
    .protected_link_ids
)


assert not (
    protected_after_failure
    &
    set(
        attempted_link_ids
    )
)


print(
    "[PASS] Protected successful subtrees consume zero "
    "redundant elementary-generation attempts."
)


# =============================================================================
# Runtime consistency
# =============================================================================

assert (
    demand_simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Complete runtime remains cross-layer consistent "
    "after selective subtree regeneration."
)


# =============================================================================
# Diagnostics
# =============================================================================

print("\nPOST-SWAP DEMAND STATE")

display(
    demand_controller
    .decisions_dataframe(
        post_swap_reconciliation
    )
)


print("\nAFTER SUBTREE LOSS")

display(
    demand_controller
    .decisions_dataframe(
        failure_reconciliation
    )
)


print("\nDEMAND RECONCILIATION HISTORY")

display(
    demand_controller
    .history_dataframe()
)


print("\nSELECTIVE REGENERATION SLOT")

display(
    pd.DataFrame([
        {
            "Link ID":
                step.link_id,

            "Action":
                step.action.value,

            "Physical Attempt":
                (
                    step.attempt.attempt_index
                    if step.attempt is not None
                    else None
                ),

            "Success":
                (
                    step.attempt.success
                    if step.attempt is not None
                    else None
                )
        }

        for step in (
            regeneration_slot.steps
        )
    ])
)


print("\nCURRENT ACTIVE ENTANGLEMENT FRONTIER")

display(
    demand_swap_scheduler
    .active_frontier_dataframe()
)


print("\n" + "=" * 88)
print(
    "Selective-regeneration verification "
    "completed successfully."
)
print("=" * 88)

In [ ]:
# @title  Failure-Aware End-to-End Execution Records

from dataclasses import dataclass
from enum import Enum
from typing import Optional


# =============================================================================
# Session Status
# =============================================================================

class EndToEndExecutionStatus(str, Enum):

    SUCCESS = "SUCCESS"

    MAX_SLOTS_EXCEEDED = "MAX_SLOTS_EXCEEDED"


# =============================================================================
# One Global Execution Slot
# =============================================================================

@dataclass(frozen=True)
class FailureAwareExecutionSlot:
    """
    Complete audit record of one autonomous NARAD-QC
    global execution slot.
    """

    execution_index: int

    simulation_slot: int

    demand_before_generation: (
        FailureAwareDemandReconciliation
    )

    generation_slot: (
        ParallelElementaryGenerationSlot
    )

    newly_registered_elementary_resource_ids: tuple[
        str, ...
    ]

    swap_round: Optional[
        ParallelSwapRound
    ]

    demand_after_swapping: (
        FailureAwareDemandReconciliation
    )

    active_frontier_resource_ids: tuple[
        str, ...
    ]

    end_to_end_resource_id: Optional[str]


    # =========================================================================
    # Elementary Generation Metrics
    # =========================================================================

    @property
    def elementary_attempt_count(
        self
    ) -> int:

        return (
            self.generation_slot
            .physical_attempt_count
        )


    @property
    def elementary_success_count(
        self
    ) -> int:

        return (
            self.generation_slot
            .successful_generation_count
        )


    @property
    def elementary_failure_count(
        self
    ) -> int:

        return (
            self.generation_slot
            .failed_generation_count
        )


    # =========================================================================
    # BSM Metrics
    # =========================================================================

    @property
    def bsm_attempt_count(
        self
    ) -> int:

        if self.swap_round is None:
            return 0

        return len(
            self.swap_round.attempts
        )


    @property
    def bsm_success_count(
        self
    ) -> int:

        if self.swap_round is None:
            return 0

        return (
            self.swap_round.success_count
        )


    @property
    def bsm_failure_count(
        self
    ) -> int:

        if self.swap_round is None:
            return 0

        return (
            self.swap_round.failure_count
        )


# =============================================================================
# Completed Session Result
# =============================================================================

@dataclass(frozen=True)
class FailureAwareEndToEndExecutionResult:

    status: EndToEndExecutionStatus

    start_slot: int

    end_slot: int

    processed_slot_count: int

    end_to_end_resource_id: Optional[str]

    final_fidelity: Optional[float]

    total_elementary_attempts: int

    total_elementary_successes: int

    total_elementary_failures: int

    total_bsm_attempts: int

    total_bsm_successes: int

    total_bsm_failures: int

    execution_history: tuple[
        FailureAwareExecutionSlot, ...
    ]


    @property
    def success(
        self
    ) -> bool:

        return (
            self.status
            == EndToEndExecutionStatus.SUCCESS
        )


    @property
    def simulation_slots_elapsed(
        self
    ) -> int:

        return (
            self.end_slot
            - self.start_slot
        )

In [ ]:
# @title  Autonomous Failure-Aware End-to-End Orchestrator


class FailureAwareEndToEndEntanglementOrchestrator:
    """
    Autonomous execution owner for one NARAD-QC end-to-end
    entanglement session.

    The orchestrator adds NO new quantum physics.

    It coordinates:

        FailureAwareElementaryDemandController
        ParallelElementaryGenerationScheduler
        PhysicalEntanglementFidelityRegistry
        ControlPlaneParallelSwapScheduler
        EntanglementSwapEngine

    into one complete failure-aware execution loop.
    """

    def __init__(
        self,
        simulator: QuantumNetworkSimulator,
        ordered_path,
        generation_scheduler: ParallelElementaryGenerationScheduler,
        demand_controller: FailureAwareElementaryDemandController,
        swap_scheduler: ControlPlaneParallelSwapScheduler,
        fidelity_registry: PhysicalEntanglementFidelityRegistry
    ):

        # =====================================================================
        # Core Validation
        # =====================================================================

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):
            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )

        if not isinstance(
            generation_scheduler,
            ParallelElementaryGenerationScheduler
        ):
            raise TypeError(
                "generation_scheduler must be a "
                "ParallelElementaryGenerationScheduler."
            )

        if not isinstance(
            demand_controller,
            FailureAwareElementaryDemandController
        ):
            raise TypeError(
                "demand_controller must be a "
                "FailureAwareElementaryDemandController."
            )

        if not isinstance(
            swap_scheduler,
            ControlPlaneParallelSwapScheduler
        ):
            raise TypeError(
                "swap_scheduler must be a "
                "ControlPlaneParallelSwapScheduler."
            )

        if not isinstance(
            fidelity_registry,
            PhysicalEntanglementFidelityRegistry
        ):
            raise TypeError(
                "fidelity_registry must be a "
                "PhysicalEntanglementFidelityRegistry."
            )


        # =====================================================================
        # Every subsystem must belong to this simulator.
        # =====================================================================

        if (
            generation_scheduler.simulator
            is not simulator
        ):
            raise ValueError(
                "Generation scheduler belongs to another simulator."
            )

        if (
            demand_controller.simulator
            is not simulator
        ):
            raise ValueError(
                "Demand controller belongs to another simulator."
            )

        if (
            swap_scheduler.simulator
            is not simulator
        ):
            raise ValueError(
                "Swap scheduler belongs to another simulator."
            )

        if (
            fidelity_registry.simulator
            is not simulator
        ):
            raise ValueError(
                "Fidelity registry belongs to another simulator."
            )


        self.simulator = simulator

        self.generation_scheduler = (
            generation_scheduler
        )

        self.demand_controller = (
            demand_controller
        )

        self.swap_scheduler = (
            swap_scheduler
        )

        self.fidelity_registry = (
            fidelity_registry
        )


        # =====================================================================
        # Authoritative ordered path
        # =====================================================================

        self.ordered_path = (
            simulator.topology
            .validate_path(
                ordered_path
            )
        )

        self.path_links = (
            simulator.topology
            .path_links(
                self.ordered_path
            )
        )


        expected_path_names = tuple(
            node.name
            for node in self.ordered_path
        )


        subsystem_paths = (

            tuple(
                node.name
                for node
                in generation_scheduler.ordered_path
            ),

            tuple(
                node.name
                for node
                in demand_controller.ordered_path
            ),

            tuple(
                node.name
                for node
                in swap_scheduler.ordered_path
            )
        )


        if not all(
            path_names
            == expected_path_names
            for path_names
            in subsystem_paths
        ):
            raise ValueError(
                "All Phase-3D subsystems must manage exactly "
                "the same ordered physical path."
            )


        self.source_name = (
            self.ordered_path[0].name
        )

        self.destination_name = (
            self.ordered_path[-1].name
        )


        self._path_link_ids = tuple(
            link.link_id
            for link in self.path_links
        )


        self._execution_history: list[
            FailureAwareExecutionSlot
        ] = []


    # =========================================================================
    # Fidelity Registration
    # =========================================================================

    def _is_fidelity_registered(
        self,
        resource
    ) -> bool:

        try:

            self.fidelity_registry.get(
                resource
            )

            return True

        except KeyError:

            return False


    def _register_unseen_active_elementary_resources(
        self
    ) -> tuple[str, ...]:
        """
        Register physical fidelity for elementary EPs created
        during the current generation step.

        Registration occurs BEFORE swapping, so any same-slot BSM
        receives a valid current fidelity.
        """

        registered_ids = []

        for resource in (
            self.simulator
            .entanglement_manager
            .active_resources
        ):

            if (
                resource.kind
                != EntanglementKind.ELEMENTARY
            ):
                continue

            if self._is_fidelity_registered(
                resource
            ):
                continue

            self.fidelity_registry\
                .register_elementary(
                    resource
                )

            registered_ids.append(
                resource.resource_id
            )

        return tuple(
            registered_ids
        )


    # =========================================================================
    # End-to-End Resource Detection
    # =========================================================================

    def _find_end_to_end_resource(
        self
    ) -> Optional[
        EntanglementResource
    ]:
        """
        An end-to-end resource must:

            - be ACTIVE;
            - connect the two requested users;
            - contain the exact physical-link provenance
              of the ordered path.

        Merely sharing the correct endpoint names is not enough.
        """

        target_endpoints = {
            self.source_name,
            self.destination_name
        }

        target_links = set(
            self._path_link_ids
        )

        matching = []

        for resource in (
            self.simulator
            .entanglement_manager
            .active_resources
        ):

            if (
                set(resource.endpoint_names)
                != target_endpoints
            ):
                continue

            if (
                len(
                    resource.source_link_ids
                )
                != len(
                    self._path_link_ids
                )
            ):
                continue

            if (
                set(
                    resource.source_link_ids
                )
                != target_links
            ):
                continue

            matching.append(
                resource
            )


        if len(
            matching
        ) > 1:

            raise RuntimeError(
                "Single-session orchestrator found multiple "
                "simultaneous full-path end-to-end resources."
            )


        if not matching:
            return None


        return matching[0]


    # =========================================================================
    # One Autonomous Global Slot
    # =========================================================================

    def process_current_slot(
        self
    ) -> FailureAwareExecutionSlot:
        """
        Execute exactly ONE complete global control/physics slot.

        IMPORTANT:
        This method does NOT advance the global simulation clock.
        """

        self.simulator\
            .validate_runtime_consistency()


        slot_at_start = (
            self.simulator.kernel.current_slot
        )


        # =====================================================================
        # 1. FAILURE-AWARE DEMAND RECONCILIATION
        #
        # synchronize_expirations() occurs inside reconciliation.
        # =====================================================================

        demand_before = (
            self.demand_controller
            .reconcile()
        )


        assert (
            self.simulator.kernel.current_slot
            == slot_at_start
        )


        # =====================================================================
        # 2. PARALLEL ELEMENTARY GENERATION
        #
        # Every demanded unresolved physical link receives at most
        # one attempt in this slot.
        # =====================================================================

        generation_slot = (
            self.generation_scheduler
            .process_current_slot()
        )


        assert (
            generation_slot.simulation_slot
            == slot_at_start
        )

        assert (
            self.simulator.kernel.current_slot
            == slot_at_start
        )


        # =====================================================================
        # 3. REGISTER FIDELITY OF NEWLY GENERATED ELEMENTARY EPS
        # =====================================================================

        new_elementary_fidelity_ids = (
            self
            ._register_unseen_active_elementary_resources()
        )


        # =====================================================================
        # 4. CONTROL PLANE OBSERVES THE NEW LIVE FRONTIER
        #
        # If adjacent compatible resources now exist, one parallel
        # BSM round may execute during this slot.
        #
        # Only ONE swapping round is permitted per global slot.
        # =====================================================================

        available_candidates = (
            self.swap_scheduler
            .discover_candidates()
        )


        swap_round = None


        if available_candidates:

            swap_round = (
                self.swap_scheduler
                .execute_parallel_round()
            )


            assert (
                swap_round.simulation_slot
                == slot_at_start
            )


        assert (
            self.simulator.kernel.current_slot
            == slot_at_start
        )


        # =====================================================================
        # 5. FAILURE-AWARE POST-BSM RECONCILIATION
        #
        # Successful subtrees become protected.
        #
        # Failed BSM input subtrees become uncovered and therefore
        # request regeneration.
        #
        # No generation occurs again in this same slot.
        # =====================================================================

        demand_after = (
            self.demand_controller
            .reconcile()
        )


        assert (
            self.simulator.kernel.current_slot
            == slot_at_start
        )


        # =====================================================================
        # 6. END-TO-END COMPLETION CHECK
        # =====================================================================

        end_to_end_resource = (
            self._find_end_to_end_resource()
        )


        active_frontier_ids = tuple(
            resource.resource_id
            for resource
            in (
                self.simulator
                .entanglement_manager
                .active_resources
            )
        )


        slot_record = (
            FailureAwareExecutionSlot(

                execution_index=(
                    len(
                        self._execution_history
                    )
                ),

                simulation_slot=(
                    slot_at_start
                ),

                demand_before_generation=(
                    demand_before
                ),

                generation_slot=(
                    generation_slot
                ),

                newly_registered_elementary_resource_ids=(
                    new_elementary_fidelity_ids
                ),

                swap_round=(
                    swap_round
                ),

                demand_after_swapping=(
                    demand_after
                ),

                active_frontier_resource_ids=(
                    active_frontier_ids
                ),

                end_to_end_resource_id=(
                    end_to_end_resource.resource_id
                    if end_to_end_resource
                    is not None
                    else None
                )
            )
        )


        self._execution_history.append(
            slot_record
        )


        # =====================================================================
        # Audit event
        # =====================================================================

        self.simulator.kernel.emit(

            event_type=(
                "FAILURE_AWARE_EXECUTION_SLOT_COMPLETED"
            ),

            component=(
                "FailureAwareEndToEndEntanglementOrchestrator"
            ),

            node_names=tuple(
                node.name
                for node
                in self.ordered_path
            ),

            link_ids=(
                self._path_link_ids
            ),

            resource_ids=(
                active_frontier_ids
            ),

            message=(
                f"Completed autonomous end-to-end execution "
                f"step {slot_record.execution_index}."
            ),

            metadata={
                "elementary_attempts":
                    slot_record
                    .elementary_attempt_count,

                "elementary_successes":
                    slot_record
                    .elementary_success_count,

                "elementary_failures":
                    slot_record
                    .elementary_failure_count,

                "bsm_attempts":
                    slot_record
                    .bsm_attempt_count,

                "bsm_successes":
                    slot_record
                    .bsm_success_count,

                "bsm_failures":
                    slot_record
                    .bsm_failure_count,

                "regeneration_links_after":
                    demand_after
                    .regeneration_link_ids,

                "end_to_end_resource":
                    slot_record
                    .end_to_end_resource_id
            }
        )


        self.simulator\
            .validate_runtime_consistency()


        return slot_record


    # =========================================================================
    # Autonomous Session
    # =========================================================================

    def run_until_end_to_end(
        self,
        max_slots: int = 5000
    ) -> FailureAwareEndToEndExecutionResult:
        """
        Run autonomously until a full-path end-to-end entangled
        resource exists or max_slots is exhausted.

        Global clock advancement occurs ONLY here, once after each
        incomplete execution slot.
        """

        if (
            isinstance(
                max_slots,
                bool
            )
            or not isinstance(
                max_slots,
                int
            )
            or max_slots <= 0
        ):
            raise ValueError(
                "max_slots must be a positive integer."
            )


        start_slot = (
            self.simulator.kernel.current_slot
        )


        # =====================================================================
        # Already Complete?
        # =====================================================================

        existing = (
            self._find_end_to_end_resource()
        )


        if existing is not None:

            final_fidelity = (
                self.fidelity_registry
                .fidelity(
                    existing
                )
            )

            return (
                FailureAwareEndToEndExecutionResult(

                    status=(
                        EndToEndExecutionStatus.SUCCESS
                    ),

                    start_slot=(
                        start_slot
                    ),

                    end_slot=(
                        start_slot
                    ),

                    processed_slot_count=0,

                    end_to_end_resource_id=(
                        existing.resource_id
                    ),

                    final_fidelity=(
                        final_fidelity
                    ),

                    total_elementary_attempts=0,

                    total_elementary_successes=0,

                    total_elementary_failures=0,

                    total_bsm_attempts=0,

                    total_bsm_successes=0,

                    total_bsm_failures=0,

                    execution_history=()
                )
            )


        run_history = []


        # =====================================================================
        # Main Autonomous Loop
        # =====================================================================

        for _ in range(
            max_slots
        ):

            step = (
                self.process_current_slot()
            )

            run_history.append(
                step
            )


            # =================================================================
            # SUCCESS
            # =================================================================

            if (
                step.end_to_end_resource_id
                is not None
            ):

                final_resource = (
                    self.simulator
                    .entanglement_manager
                    .get_resource(
                        step.end_to_end_resource_id
                    )
                )


                final_fidelity = (
                    self.fidelity_registry
                    .fidelity(
                        final_resource
                    )
                )


                result = (
                    FailureAwareEndToEndExecutionResult(

                        status=(
                            EndToEndExecutionStatus.SUCCESS
                        ),

                        start_slot=(
                            start_slot
                        ),

                        end_slot=(
                            self.simulator
                            .kernel
                            .current_slot
                        ),

                        processed_slot_count=(
                            len(
                                run_history
                            )
                        ),

                        end_to_end_resource_id=(
                            final_resource.resource_id
                        ),

                        final_fidelity=(
                            final_fidelity
                        ),

                        total_elementary_attempts=sum(
                            record.elementary_attempt_count
                            for record
                            in run_history
                        ),

                        total_elementary_successes=sum(
                            record.elementary_success_count
                            for record
                            in run_history
                        ),

                        total_elementary_failures=sum(
                            record.elementary_failure_count
                            for record
                            in run_history
                        ),

                        total_bsm_attempts=sum(
                            record.bsm_attempt_count
                            for record
                            in run_history
                        ),

                        total_bsm_successes=sum(
                            record.bsm_success_count
                            for record
                            in run_history
                        ),

                        total_bsm_failures=sum(
                            record.bsm_failure_count
                            for record
                            in run_history
                        ),

                        execution_history=tuple(
                            run_history
                        )
                    )
                )


                self.simulator.kernel.emit(

                    event_type=(
                        "END_TO_END_ENTANGLEMENT_ESTABLISHED"
                    ),

                    component=(
                        "FailureAwareEndToEndEntanglementOrchestrator"
                    ),

                    node_names=(
                        self.source_name,
                        self.destination_name
                    ),

                    link_ids=(
                        self._path_link_ids
                    ),

                    resource_ids=(
                        final_resource.resource_id,
                    ),

                    message=(
                        f"End-to-end entanglement established "
                        f"between {self.source_name} and "
                        f"{self.destination_name}."
                    ),

                    metadata={
                        "processed_slots":
                            result.processed_slot_count,

                        "elementary_attempts":
                            result.total_elementary_attempts,

                        "bsm_attempts":
                            result.total_bsm_attempts,

                        "bsm_failures":
                            result.total_bsm_failures,

                        "final_fidelity":
                            result.final_fidelity
                    }
                )


                return result


            # =================================================================
            # INCOMPLETE SESSION
            #
            # Exactly ONE global slot boundary.
            # =================================================================

            self.simulator\
                .kernel\
                .advance_slot()


        # =====================================================================
        # Maximum Slot Limit Reached
        # =====================================================================

        return (
            FailureAwareEndToEndExecutionResult(

                status=(
                    EndToEndExecutionStatus
                    .MAX_SLOTS_EXCEEDED
                ),

                start_slot=(
                    start_slot
                ),

                end_slot=(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                processed_slot_count=(
                    len(
                        run_history
                    )
                ),

                end_to_end_resource_id=None,

                final_fidelity=None,

                total_elementary_attempts=sum(
                    record.elementary_attempt_count
                    for record
                    in run_history
                ),

                total_elementary_successes=sum(
                    record.elementary_success_count
                    for record
                    in run_history
                ),

                total_elementary_failures=sum(
                    record.elementary_failure_count
                    for record
                    in run_history
                ),

                total_bsm_attempts=sum(
                    record.bsm_attempt_count
                    for record
                    in run_history
                ),

                total_bsm_successes=sum(
                    record.bsm_success_count
                    for record
                    in run_history
                ),

                total_bsm_failures=sum(
                    record.bsm_failure_count
                    for record
                    in run_history
                ),

                execution_history=tuple(
                    run_history
                )
            )
        )


    # =========================================================================
    # Inspection
    # =========================================================================

    @property
    def execution_history(
        self
    ) -> tuple[
        FailureAwareExecutionSlot, ...
    ]:

        return tuple(
            self._execution_history
        )


    def history_dataframe(
        self
    ) -> pd.DataFrame:

        rows = []


        for step in (
            self._execution_history
        ):

            swap_round = (
                step.swap_round
            )


            rows.append({

                "Execution Step":
                    step.execution_index,

                "Simulation Slot":
                    step.simulation_slot,

                "Regeneration Demand Before":
                    len(
                        step
                        .demand_before_generation
                        .regeneration_link_ids
                    ),

                "Elementary Attempts":
                    step.elementary_attempt_count,

                "Elementary Successes":
                    step.elementary_success_count,

                "Elementary Failures":
                    step.elementary_failure_count,

                "New Elementary Fidelity Records":
                    len(
                        step
                        .newly_registered_elementary_resource_ids
                    ),

                "Swap Candidates":
                    (
                        swap_round.candidate_count
                        if swap_round is not None
                        else 0
                    ),

                "Parallel BSMs":
                    step.bsm_attempt_count,

                "BSM Successes":
                    step.bsm_success_count,

                "BSM Failures":
                    step.bsm_failure_count,

                "Swap Nodes":
                    (
                        tuple(
                            candidate.swap_node_name
                            for candidate
                            in swap_round.selected_candidates
                        )

                        if swap_round is not None

                        else ()
                    ),

                "Regeneration Demand After":
                    len(
                        step
                        .demand_after_swapping
                        .regeneration_link_ids
                    ),

                "Active Frontier Size":
                    len(
                        step
                        .active_frontier_resource_ids
                    ),

                "End-to-End Resource":
                    step.end_to_end_resource_id
            })


        return pd.DataFrame(
            rows
        )


    def __repr__(
        self
    ) -> str:

        return (
            "FailureAwareEndToEndEntanglementOrchestrator("
            f"source='{self.source_name}', "
            f"destination='{self.destination_name}', "
            f"repeaters={len(self.ordered_path) - 2}, "
            f"executed_slots={len(self._execution_history)}"
            ")"
        )

In [ ]:
# @title Six-Repeater Autonomous Failure-Aware Verification


print("=" * 92)
print(
    "Full Failure-Aware End-to-End Execution Verification"
)
print("=" * 92)


# =============================================================================
# PHYSICAL NETWORK
#
# E2E-A -- R1 -- R2 -- R3 -- R4 -- R5 -- R6 -- E2E-B
#
# 8 nodes
# 6 repeaters
# 7 elementary fibre segments
#
# Each segment = 5 km
# =============================================================================

e2e_topology = NetworkTopology(
    name="Phase3D2-Full-Six-Repeater-Network"
)


e2e_a = NetworkNode(
    name="E2E-A",
    x_km=0,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


e2e_repeaters = []


for repeater_number in range(
    1,
    7
):

    e2e_repeaters.append(

        NetworkNode(
            name=f"E2E-R{repeater_number}",
            x_km=5 * repeater_number,
            y_km=0,
            endpoint_capable=False,
            swapping_capable=True,
            memory_capacity=2
        )
    )


e2e_b = NetworkNode(
    name="E2E-B",
    x_km=35,
    y_km=0,
    endpoint_capable=True,
    swapping_capable=False,
    memory_capacity=2
)


e2e_path = (
    e2e_a,
    *e2e_repeaters,
    e2e_b
)


for node in e2e_path:

    e2e_topology.add_node(
        node
    )


for left, right in zip(
    e2e_path[:-1],
    e2e_path[1:]
):

    e2e_topology.add_link(

        PhysicalQuantumLink(
            node_a=left,
            node_b=right,
            attenuation_db_per_km=0.2
        )
    )


assert (
    e2e_topology.node_count
    == 8
)


assert (
    e2e_topology.link_count
    == 7
)


assert not (
    e2e_topology.contains_link(
        e2e_a,
        e2e_b
    )
)


print(
    "[PASS] Six-repeater physical network contains "
    "8 nodes and 7 adjacent fibre links."
)


# =============================================================================
# SIMULATOR
#
# 100 slots × 0.05 s = 5 s maximum physical-qubit storage
# before hard expiration.
#
# Fidelity degrades continuously before that threshold.
# =============================================================================

e2e_simulator = (
    QuantumNetworkSimulator(

        topology=e2e_topology,

        simulation_seed=360602,

        controller_id="PHASE3D2-CONTROLLER",

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=100
    )
)


# =============================================================================
# DEVICE AUTHENTICATION
# =============================================================================

e2e_authentication = (
    e2e_simulator
    .authenticate_all_nodes()
)


assert all(
    record.status
    == AuthenticationStatus.AUTHENTICATED
    for record
    in e2e_authentication.values()
)


assert (
    e2e_simulator
    .require_quantum_eligibility(
        *e2e_path
    )
)


print(
    "[PASS] Both users and all six repeaters are "
    "ML-DSA mutually authenticated."
)


# =============================================================================
# ELEMENTARY GENERATION PHYSICS
# =============================================================================

e2e_generation_service = (
    ElementaryEntanglementService(

        simulator=e2e_simulator,

        model=FiberSurvivalProxyModel()
    )
)


# IMPORTANT:
#
# Demand starts OFF.
#
# The new failure-aware control plane is responsible for deciding
# which links are allowed to generate.
e2e_generation_scheduler = (
    ParallelElementaryGenerationScheduler(

        simulator=e2e_simulator,

        service=e2e_generation_service,

        ordered_path=e2e_path,

        demand_active=False
    )
)


print(
    "[PASS] Elementary generation begins under explicit "
    "control-plane demand authority."
)


# =============================================================================
# PHYSICAL FIDELITY MODEL
# =============================================================================

e2e_fidelity_profile = (
    WernerPhysicalFidelityProfile(

        elementary_fidelity=0.995,

        bell_fidelity_floor=0.25,

        memory_reference_initial_fidelity=1.0,

        memory_reference_fidelity=0.90,

        memory_reference_time_s=1.0,

        slot_duration_s=0.05,

        swap_operation_visibility=1.0
    )
)


e2e_fidelity_registry = (
    PhysicalEntanglementFidelityRegistry(

        simulator=e2e_simulator,

        profile=e2e_fidelity_profile
    )
)


print(
    "[PASS] Dynamic Werner-state physical fidelity model enabled."
)


# =============================================================================
# STOCHASTIC PHYSICAL BSM
#
# p_b = 0.40
# t_b = 10 us metadata
#
# Fidelity propagation:
# Werner-state model
# =============================================================================

e2e_swap_model = (
    WernerStateAtomicBSMModel(

        fidelity_profile=(
            e2e_fidelity_profile
        ),

        atomic_bsm_success_probability=0.40,

        atomic_bsm_latency_us=10.0
    )
)


e2e_swap_engine = (
    EntanglementSwapEngine(

        simulator=e2e_simulator,

        model=e2e_swap_model,

        fidelity_registry=(
            e2e_fidelity_registry
        )
    )
)


assert math.isclose(
    e2e_swap_model
    .atomic_bsm_success_probability,
    0.40,
    abs_tol=1e-12
)


print(
    "[PASS] Failure-prone atomic BSM physics enabled "
    "with p_b = 0.40."
)


# =============================================================================
# PARALLEL CONTROL-PLANE SWAPPING
# =============================================================================

e2e_swap_scheduler = (
    ControlPlaneParallelSwapScheduler(

        simulator=e2e_simulator,

        swap_engine=e2e_swap_engine,

        ordered_path=e2e_path,

        policy=(
            MaxParallelOldestFirstSwapPolicy()
        )
    )
)


print(
    "[PASS] Max-parallel oldest-first control policy enabled."
)


# =============================================================================
# FAILURE-AWARE ELEMENTARY DEMAND
# =============================================================================

e2e_demand_controller = (
    FailureAwareElementaryDemandController(

        simulator=e2e_simulator,

        generation_scheduler=(
            e2e_generation_scheduler
        ),

        ordered_path=e2e_path
    )
)


print(
    "[PASS] Failure-aware selective-regeneration controller enabled."
)


# =============================================================================
# AUTONOMOUS ORCHESTRATOR
# =============================================================================

e2e_orchestrator = (
    FailureAwareEndToEndEntanglementOrchestrator(

        simulator=e2e_simulator,

        ordered_path=e2e_path,

        generation_scheduler=(
            e2e_generation_scheduler
        ),

        demand_controller=(
            e2e_demand_controller
        ),

        swap_scheduler=(
            e2e_swap_scheduler
        ),

        fidelity_registry=(
            e2e_fidelity_registry
        )
    )
)


print(
    "[PASS] Full autonomous NARAD-QC execution stack assembled."
)


# =============================================================================
# START STATE
#
# There must be ZERO Bell pairs.
# =============================================================================

assert (
    len(
        e2e_simulator
        .entanglement_manager
        .active_resources
    )
    == 0
)


assert (
    e2e_orchestrator
    ._find_end_to_end_resource()
    is None
)


print(
    "[PASS] Session starts with zero entanglement resources."
)


# =============================================================================
# RUN AUTONOMOUSLY
#
# NO:
#   run_until_all_ready()
#   manual swap rounds
#   manual regeneration
#   manual tree
#
# The control plane owns the entire execution.
# =============================================================================

e2e_result = (
    e2e_orchestrator
    .run_until_end_to_end(
        max_slots=5000
    )
)


# =============================================================================
# SESSION MUST SUCCEED
# =============================================================================

assert (
    e2e_result.status
    == EndToEndExecutionStatus.SUCCESS
)


assert e2e_result.success


assert (
    e2e_result
    .end_to_end_resource_id
    is not None
)


assert (
    e2e_result
    .final_fidelity
    is not None
)


print(
    "[PASS] Autonomous control plane successfully established "
    "end-to-end entanglement."
)


# =============================================================================
# FINAL RESOURCE
# =============================================================================

e2e_final_resource = (
    e2e_simulator
    .entanglement_manager
    .get_resource(
        e2e_result
        .end_to_end_resource_id
    )
)


assert e2e_final_resource.is_active


assert (
    e2e_final_resource.kind
    == EntanglementKind.SWAPPED
)


assert (
    set(
        e2e_final_resource
        .endpoint_names
    )
    == {
        e2e_a.name,
        e2e_b.name
    }
)


print(
    "[PASS] Final ACTIVE resource directly entangles "
    "E2E-A and E2E-B."
)


# =============================================================================
# COMPLETE SEVEN-LINK PROVENANCE
# =============================================================================

expected_path_link_ids = {
    link.link_id
    for link in (
        e2e_topology
        .path_links(
            e2e_path
        )
    )
}


assert (
    len(
        e2e_final_resource
        .source_link_ids
    )
    == 7
)


assert (
    set(
        e2e_final_resource
        .source_link_ids
    )
    == expected_path_link_ids
)


print(
    "[PASS] Final Bell pair retains complete provenance "
    "across all seven physical fibre segments."
)


# =============================================================================
# NO FABRICATED PHYSICAL SHORTCUT
# =============================================================================

assert not (
    e2e_topology
    .contains_link(
        e2e_a,
        e2e_b
    )
)


print(
    "[PASS] End-to-end entanglement exists without "
    "a direct physical user-to-user fibre."
)


# =============================================================================
# PHYSICAL FIDELITY
# =============================================================================

assert (
    e2e_fidelity_profile
    .bell_fidelity_floor
    <
    e2e_result.final_fidelity
    <=
    e2e_fidelity_profile
    .elementary_fidelity
)


assert not math.isclose(
    e2e_result.final_fidelity,
    1.0,
    abs_tol=1e-12
)


print(
    "[PASS] Final end-to-end resource carries a "
    "non-ideal physical fidelity."
)


# =============================================================================
# EVERY BSM MUST FOLLOW STOCHASTIC p_b = 0.40
# =============================================================================

for attempt in (
    e2e_swap_engine.attempts
):

    assert math.isclose(
        attempt.success_probability,
        0.40,
        abs_tol=1e-12
    )


    assert (
        attempt.random_draw
        is not None
    )


    assert (
        attempt.success
        ==
        (
            attempt.random_draw
            < 0.40
        )
    )


print(
    "[PASS] Every atomic BSM obeys the configured "
    "stochastic success law."
)


# =============================================================================
# AT LEAST SIX SUCCESSFUL BSMs ARE REQUIRED
#
# Seven elementary leaves require six successful merge operations
# in the final realized binary swapping tree.
#
# Historical retries/expired branches may make this number larger.
# =============================================================================

assert (
    e2e_result
    .total_bsm_successes
    >= 6
)


print(
    "[PASS] End-to-end resource required at least the "
    "six successful BSM merges expected for seven leaves."
)


# =============================================================================
# FAILURE-AWARE RECOVERY INTEGRATION
#
# If this seeded run experiences ANY destructive BSM failure,
# that same execution slot must expose uncovered physical links
# for regeneration after the failed round.
# =============================================================================

failure_rounds = tuple(

    step

    for step in (
        e2e_result
        .execution_history
    )

    if (
        step.bsm_failure_count
        > 0
    )
)


for step in failure_rounds:

    assert (
        len(
            step
            .demand_after_swapping
            .regeneration_link_ids
        )
        > 0
    )


if failure_rounds:

    print(
        "[PASS] Destructive BSM failure occurred and the "
        "control plane automatically exposed only uncovered "
        "physical coverage for regeneration."
    )

else:

    print(
        "[INFO] This seeded execution happened to contain "
        "no BSM failure. Failure recovery was already "
        "independently verified in Phase 3D.1."
    )


# =============================================================================
# FIDELITY USED BY EVERY BSM MUST MATCH DYNAMIC REGISTRY VALUE
# AT THAT BSM SLOT.
# =============================================================================

for attempt in (
    e2e_swap_engine.attempts
):

    expected_left_fidelity = (
        e2e_fidelity_registry
        .fidelity_at_slot(

            attempt.left_resource_id,

            attempt.simulation_slot
        )
    )


    expected_right_fidelity = (
        e2e_fidelity_registry
        .fidelity_at_slot(

            attempt.right_resource_id,

            attempt.simulation_slot
        )
    )


    assert math.isclose(
        attempt.left_fidelity,
        expected_left_fidelity,
        rel_tol=0.0,
        abs_tol=1e-12
    )


    assert math.isclose(
        attempt.right_fidelity,
        expected_right_fidelity,
        rel_tol=0.0,
        abs_tol=1e-12
    )


print(
    "[PASS] Every BSM consumed the correct dynamically "
    "decohered input fidelities."
)


# =============================================================================
# SUCCESSFUL BSM OUTPUT FIDELITY
# =============================================================================

for attempt in (
    e2e_swap_engine.attempts
):

    if not attempt.success:
        continue


    expected_output_fidelity = (
        e2e_fidelity_profile
        .swapped_fidelity(

            attempt.left_fidelity,

            attempt.right_fidelity
        )
    )


    assert math.isclose(
        attempt.output_fidelity,
        expected_output_fidelity,
        rel_tol=0.0,
        abs_tol=1e-12
    )


print(
    "[PASS] Every successful BSM obeys the Werner-state "
    "fidelity transformation."
)


# =============================================================================
# FINAL ACTIVE FRONTIER
#
# Full-path end-to-end coverage means there should be exactly
# ONE ACTIVE entanglement resource in this single-session model.
# =============================================================================

final_active_resources = (
    e2e_simulator
    .entanglement_manager
    .active_resources
)


assert (
    len(
        final_active_resources
    )
    == 1
)


assert (
    final_active_resources[0]
    .resource_id
    == e2e_final_resource.resource_id
)


print(
    "[PASS] Successful completion leaves exactly one "
    "full-path ACTIVE entanglement resource."
)


# =============================================================================
# FINAL MEMORY STATE
#
# End users retain one outer qubit each.
# All repeater BSM qubits are gone.
# =============================================================================

assert (
    e2e_simulator
    .quantum_memory(
        e2e_a
    )
    .occupancy
    == 1
)


assert (
    e2e_simulator
    .quantum_memory(
        e2e_b
    )
    .occupancy
    == 1
)


for repeater in (
    e2e_repeaters
):

    assert (
        e2e_simulator
        .quantum_memory(
            repeater
        )
        .occupancy
        == 0
    )


print(
    "[PASS] Final quantum memory contains only the "
    "two surviving end-user qubits."
)


# =============================================================================
# THERE MUST BE NO FURTHER BSM CANDIDATE
# =============================================================================

assert (
    len(
        e2e_swap_scheduler
        .discover_candidates()
    )
    == 0
)


print(
    "[PASS] No further swap operation remains after "
    "end-to-end establishment."
)


# =============================================================================
# AT LEAST THREE SWAP ROUNDS
#
# With six repeaters, even perfect fully-ready execution cannot
# collapse seven leaves to one root in fewer than three logical
# BSM rounds under the anti-cascade rule.
# =============================================================================

actual_swap_round_count = sum(

    1

    for step in (
        e2e_result
        .execution_history
    )

    if (
        step.swap_round
        is not None
        and step.bsm_attempt_count > 0
    )
)


assert (
    actual_swap_round_count
    >= 3
)


print(
    "[PASS] Anti-cascade semantics preserve multiple "
    "logical swap rounds."
)


# =============================================================================
# CROSS-LAYER CONSISTENCY
# =============================================================================

assert (
    e2e_simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Complete authenticated, stochastic, "
    "failure-aware, fidelity-aware runtime is consistent."
)


# =============================================================================
# MODELED PHYSICAL COMPLETION TIME
#
# This comes ONLY from slot progression.
#
# BSM t_b = 10 us remains model metadata and is NOT secretly
# added to this discrete simulation clock.
# =============================================================================

modeled_completion_time_s = (
    e2e_result
    .simulation_slots_elapsed

    * e2e_fidelity_profile
    .slot_duration_s
)


# =============================================================================
# SUMMARY
# =============================================================================

e2e_summary = pd.DataFrame([
    {
        "Status":
            e2e_result.status.value,

        "Start Slot":
            e2e_result.start_slot,

        "End Slot":
            e2e_result.end_slot,

        "Processed Execution Slots":
            e2e_result.processed_slot_count,

        "Modeled Completion Time (s)":
            modeled_completion_time_s,

        "Elementary Attempts":
            e2e_result.total_elementary_attempts,

        "Elementary Successes":
            e2e_result.total_elementary_successes,

        "Elementary Failures":
            e2e_result.total_elementary_failures,

        "BSM Attempts":
            e2e_result.total_bsm_attempts,

        "BSM Successes":
            e2e_result.total_bsm_successes,

        "BSM Failures":
            e2e_result.total_bsm_failures,

        "Swap Rounds":
            actual_swap_round_count,

        "Final Resource":
            e2e_result.end_to_end_resource_id,

        "Final Fidelity":
            e2e_result.final_fidelity
    }
])


print("\nEND-TO-END SESSION SUMMARY")

display(
    e2e_summary
)


print("\nAUTONOMOUS EXECUTION TIMELINE")

display(
    e2e_orchestrator
    .history_dataframe()
)


print("\nCONTROL-PLANE DEMAND HISTORY")

display(
    e2e_demand_controller
    .history_dataframe()
)


print("\nPARALLEL SWAP HISTORY")

display(
    e2e_swap_scheduler
    .round_history_dataframe()
)


print("\nALL STOCHASTIC BSM ATTEMPTS")

display(
    e2e_swap_engine
    .attempts_dataframe()
)


print("\nCOMPLETE PHYSICAL FIDELITY HISTORY")

display(
    e2e_fidelity_registry
    .dataframe()
)


print("\nCOMPLETE ENTANGLEMENT RESOURCE HISTORY")

display(
    e2e_simulator
    .entanglement_manager
    .resources_dataframe()
)


print("\n" + "=" * 92)
print(
    "Full autonomous failure-aware end-to-end verification completed successfully."
)
print("=" * 92)

In [ ]:
# @title Benchmark Swapping Policies

from dataclasses import dataclass
from typing import Optional
import itertools


# =============================================================================
# SEQUENTIAL LEFT-TO-RIGHT POLICY
# =============================================================================

class SequentialLeftToRightSwapPolicy:
    """
    Baseline sequential swapping policy.

    Exactly ONE currently valid BSM is selected per swap round.

    Selection:
        left-most currently available candidate.

    This policy deliberately ignores:
        - parallelism,
        - resource age,
        - fidelity,
        - future tree balance.

    It exists as a simple benchmark.
    """

    @property
    def policy_name(
        self
    ) -> str:

        return "SEQUENTIAL_LEFT_TO_RIGHT"


    def select(
        self,
        candidates
    ) -> tuple[SwapCandidate, ...]:

        candidates = tuple(
            candidates
        )

        if not candidates:
            return ()

        for candidate in candidates:

            if not isinstance(
                candidate,
                SwapCandidate
            ):
                raise TypeError(
                    "All candidates must be SwapCandidate objects."
                )

        chosen = min(
            candidates,
            key=lambda candidate: (
                candidate.swap_node_path_index,
                candidate.combined_span_hops,
                candidate.candidate_id
            )
        )

        return (
            chosen,
        )


    def __repr__(
        self
    ) -> str:

        return (
            "SequentialLeftToRightSwapPolicy("
            "one_bsm_per_round=True"
            ")"
        )


# =============================================================================
# STATIC BALANCED TREE RULE
# =============================================================================

@dataclass(frozen=True)
class BalancedTreeMergeRule:

    rule_id: str

    level: int

    left_span: tuple[int, int]

    right_span: tuple[int, int]

    parent_span: tuple[int, int]

    swap_node_path_index: int


    def matches(
        self,
        candidate: SwapCandidate
    ) -> bool:

        return (
            candidate.left_span_indices
            == self.left_span

            and

            candidate.right_span_indices
            == self.right_span

            and

            candidate.combined_span_indices
            == self.parent_span

            and

            candidate.swap_node_path_index
            == self.swap_node_path_index
        )


# =============================================================================
# STATIC BALANCED TREE POLICY
# =============================================================================

class StaticBalancedTreeSwapPolicy:
    """
    Fixed balanced binary swapping-tree benchmark.

    IMPORTANT
    ---------
    This is NOT the mentor paper's optimized tree algorithm.

    It is simply a deterministic balanced-tree baseline.

    The tree is constructed before execution from the number of
    elementary physical links.

    The policy accepts ONLY BSM candidates that correspond to a
    pre-planned internal node of that tree.

    Independent planned merges may still execute in parallel.
    """

    def __init__(
        self,
        elementary_link_count: int
    ):

        if (
            isinstance(
                elementary_link_count,
                bool
            )
            or not isinstance(
                elementary_link_count,
                int
            )
            or elementary_link_count < 1
        ):
            raise ValueError(
                "elementary_link_count must be a positive integer."
            )

        self.elementary_link_count = (
            elementary_link_count
        )

        self._rules = []

        # Build the balanced tree recursively.
        self._build_subtree(
            0,
            elementary_link_count
        )

        self._rules = tuple(
            sorted(
                self._rules,
                key=lambda rule: (
                    rule.level,
                    rule.swap_node_path_index,
                    rule.parent_span
                )
            )
        )


    @property
    def policy_name(
        self
    ) -> str:

        return "STATIC_BALANCED_TREE"


    # =========================================================================
    # Tree Construction
    # =========================================================================

    def _build_subtree(
        self,
        left_boundary: int,
        right_boundary: int
    ) -> int:
        """
        Recursively construct a near-balanced binary tree over
        half-open leaf interval:

            [left_boundary, right_boundary)

        Returns the subtree height.

        One elementary physical link is a leaf with height 0.
        """

        leaf_count = (
            right_boundary
            - left_boundary
        )

        if leaf_count <= 1:
            return 0


        split = (
            left_boundary
            + leaf_count // 2
        )


        left_height = (
            self._build_subtree(
                left_boundary,
                split
            )
        )


        right_height = (
            self._build_subtree(
                split,
                right_boundary
            )
        )


        level = (
            1
            + max(
                left_height,
                right_height
            )
        )


        rule = BalancedTreeMergeRule(

            rule_id=(
                f"BALANCED::{left_boundary}:{split}:{right_boundary}"
            ),

            level=(
                level
            ),

            left_span=(
                left_boundary,
                split
            ),

            right_span=(
                split,
                right_boundary
            ),

            parent_span=(
                left_boundary,
                right_boundary
            ),

            swap_node_path_index=(
                split
            )
        )


        self._rules.append(
            rule
        )


        return level


    # =========================================================================
    # Plan Inspection
    # =========================================================================

    @property
    def planned_merges(
        self
    ) -> tuple[
        BalancedTreeMergeRule, ...
    ]:

        return self._rules


    @property
    def tree_depth(
        self
    ) -> int:

        if not self._rules:
            return 0

        return max(
            rule.level
            for rule in self._rules
        )


    def plan_dataframe(
        self
    ) -> pd.DataFrame:

        return pd.DataFrame([
            {
                "Rule":
                    rule.rule_id,

                "Tree Level":
                    rule.level,

                "Left Span":
                    rule.left_span,

                "Right Span":
                    rule.right_span,

                "Parent Span":
                    rule.parent_span,

                "Swap Node Index":
                    rule.swap_node_path_index
            }

            for rule in self._rules
        ])


    # =========================================================================
    # Candidate Selection
    # =========================================================================

    def select(
        self,
        candidates
    ) -> tuple[
        SwapCandidate, ...
    ]:

        candidates = tuple(
            candidates
        )

        if not candidates:
            return ()


        for candidate in candidates:

            if not isinstance(
                candidate,
                SwapCandidate
            ):
                raise TypeError(
                    "All candidates must be SwapCandidate objects."
                )


        # ---------------------------------------------------------------------
        # Only candidates explicitly contained in the pre-planned
        # balanced tree are eligible.
        # ---------------------------------------------------------------------

        eligible = []

        rule_for_candidate = {}


        for candidate in candidates:

            matching_rule = None

            for rule in self._rules:

                if rule.matches(
                    candidate
                ):

                    matching_rule = rule
                    break


            if matching_rule is None:
                continue


            eligible.append(
                candidate
            )

            rule_for_candidate[
                candidate.candidate_id
            ] = matching_rule


        if not eligible:
            return ()


        # ---------------------------------------------------------------------
        # Exact maximum conflict-free subset.
        #
        # Candidate counts in our current networks are tiny, so this is
        # preferable to introducing an unnecessary approximation.
        # ---------------------------------------------------------------------

        best = ()


        def score(
            selection
        ):

            # Primary:
            # maximum planned parallelism.
            cardinality = len(
                selection
            )

            # Secondary:
            # if equal, prefer lower-level tree operations.
            total_level = sum(
                rule_for_candidate[
                    candidate.candidate_id
                ].level
                for candidate
                in selection
            )

            # Final deterministic tie-break.
            positions = tuple(
                sorted(
                    candidate.swap_node_path_index
                    for candidate
                    in selection
                )
            )

            return (
                cardinality,
                -total_level,
                tuple(
                    -position
                    for position
                    in reversed(
                        positions
                    )
                )
            )


        for subset_size in range(
            1,
            len(eligible) + 1
        ):

            for subset in itertools.combinations(
                eligible,
                subset_size
            ):

                conflict_free = all(

                    not left.conflicts_with(
                        right
                    )

                    for left, right
                    in itertools.combinations(
                        subset,
                        2
                    )
                )


                if not conflict_free:
                    continue


                if (
                    not best
                    or score(subset)
                    > score(best)
                ):

                    best = tuple(
                        subset
                    )


        return tuple(
            sorted(
                best,
                key=lambda candidate: (
                    candidate.swap_node_path_index,
                    candidate.candidate_id
                )
            )
        )


    def __repr__(
        self
    ) -> str:

        return (
            "StaticBalancedTreeSwapPolicy("
            f"elementary_links={self.elementary_link_count}, "
            f"tree_depth={self.tree_depth}, "
            f"planned_merges={len(self._rules)}"
            ")"
        )

In [ ]:
# @title  Inspect Static Balanced Tree


phase3e_balanced_policy = (
    StaticBalancedTreeSwapPolicy(
        elementary_link_count=7
    )
)


assert (
    len(
        phase3e_balanced_policy
        .planned_merges
    )
    == 6
)


assert (
    phase3e_balanced_policy
    .tree_depth
    == 3
)


print(
    phase3e_balanced_policy
)


display(
    phase3e_balanced_policy
    .plan_dataframe()
)

In [ ]:
# @title  Realized Swapping Tree Metrics


def entanglement_resource_tree_depth(
    manager: EntanglementManager,
    resource_or_id
) -> int:
    """
    Number of BSM levels on the deepest branch of the
    realized provenance tree.

    Elementary resource:
        depth = 0

    Swapped resource:
        depth = 1 + max(parent depths)
    """

    if isinstance(
        resource_or_id,
        EntanglementResource
    ):

        resource = resource_or_id

    else:

        resource = (
            manager.get_resource(
                resource_or_id
            )
        )


    if (
        resource.kind
        == EntanglementKind.ELEMENTARY
    ):

        return 0


    if (
        not resource.parent_resource_ids
    ):

        raise RuntimeError(
            f"Swapped resource '{resource.resource_id}' "
            f"has no parent-resource provenance."
        )


    return (
        1
        + max(
            entanglement_resource_tree_depth(
                manager,
                parent_id
            )

            for parent_id
            in resource.parent_resource_ids
        )
    )


def entanglement_resource_tree_size(
    manager: EntanglementManager,
    resource_or_id
) -> dict:
    """
    Count only the resources belonging to the final realized tree.

    Historical failed/discarded branches that are not ancestors
    of the final resource are intentionally excluded.
    """

    visited = set()

    elementary_count = 0

    swapped_count = 0


    def visit(
        resource_or_id
    ):

        nonlocal elementary_count
        nonlocal swapped_count


        if isinstance(
            resource_or_id,
            EntanglementResource
        ):

            resource = (
                resource_or_id
            )

        else:

            resource = (
                manager.get_resource(
                    resource_or_id
                )
            )


        if (
            resource.resource_id
            in visited
        ):
            return


        visited.add(
            resource.resource_id
        )


        if (
            resource.kind
            == EntanglementKind.ELEMENTARY
        ):

            elementary_count += 1

            return


        swapped_count += 1


        for parent_id in (
            resource.parent_resource_ids
        ):

            visit(
                parent_id
            )


    visit(
        resource_or_id
    )


    return {
        "total_resources":
            len(
                visited
            ),

        "elementary_leaves":
            elementary_count,

        "successful_swaps_in_final_tree":
            swapped_count
    }

In [ ]:
# @title  Six-Repeater Structural Policy Verification


def run_phase3e_structural_policy_test(
    policy,
    seed: int
):

    # =========================================================================
    # Fresh 6-repeater topology
    # =========================================================================

    topology = NetworkTopology(
        name=(
            f"Phase3E-Structure-"
            f"{policy.policy_name}"
        )
    )


    node_a = NetworkNode(
        name="Policy-A",
        x_km=0,
        y_km=0,
        endpoint_capable=True,
        swapping_capable=False,
        memory_capacity=2
    )


    repeaters = tuple(

        NetworkNode(
            name=f"Policy-R{i}",
            x_km=5 * i,
            y_km=0,
            endpoint_capable=False,
            swapping_capable=True,
            memory_capacity=2
        )

        for i in range(
            1,
            7
        )
    )


    node_b = NetworkNode(
        name="Policy-B",
        x_km=35,
        y_km=0,
        endpoint_capable=True,
        swapping_capable=False,
        memory_capacity=2
    )


    path = (
        node_a,
        *repeaters,
        node_b
    )


    for node in path:

        topology.add_node(
            node
        )


    for left, right in zip(
        path[:-1],
        path[1:]
    ):

        topology.add_link(

            PhysicalQuantumLink(
                node_a=left,
                node_b=right,
                attenuation_db_per_km=0.2
            )
        )


    # =========================================================================
    # Simulator
    # =========================================================================

    simulator = QuantumNetworkSimulator(

        topology=topology,

        simulation_seed=seed,

        controller_id=(
            f"STRUCTURE-{policy.policy_name}"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=1000
    )


    simulator.authenticate_all_nodes()


    # =========================================================================
    # First create ALL seven elementary Bell pairs.
    #
    # Policy behavior begins only after the elementary frontier is ready.
    # =========================================================================

    generation_service = (
        ElementaryEntanglementService(

            simulator=simulator,

            model=(
                FiberSurvivalProxyModel()
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=simulator,

            service=generation_service,

            ordered_path=path,

            demand_active=True
        )
    )


    leaves = (
        generation_scheduler
        .run_until_all_ready(
            max_slots=500
        )
    )


    assert (
        len(
            leaves
        )
        == 7
    )


    # =========================================================================
    # Ideal fidelity + ideal BSM
    # =========================================================================

    fidelity_registry = (
        EntanglementFidelityRegistry(
            simulator
        )
    )


    for resource in leaves:

        fidelity_registry.register_ideal(
            resource
        )


    swap_engine = (
        EntanglementSwapEngine(

            simulator=simulator,

            model=(
                IdealEntanglementSwapModel()
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    swap_scheduler = (
        ControlPlaneParallelSwapScheduler(

            simulator=simulator,

            swap_engine=swap_engine,

            ordered_path=path,

            policy=policy
        )
    )


    # =========================================================================
    # Execute policy until one full-span resource remains.
    # =========================================================================

    selected_per_round = []

    swap_nodes_per_round = []


    for _ in range(
        20
    ):

        active = tuple(
            simulator
            .entanglement_manager
            .active_resources
        )


        full_span = tuple(

            resource

            for resource in active

            if (
                set(
                    resource.endpoint_names
                )
                == {
                    node_a.name,
                    node_b.name
                }

                and

                len(
                    resource.source_link_ids
                )
                == 7
            )
        )


        if full_span:
            break


        round_record = (
            swap_scheduler
            .execute_parallel_round()
        )


        if (
            round_record.selected_count
            == 0
        ):

            raise RuntimeError(
                f"Policy '{policy.policy_name}' entered "
                f"a structural deadlock."
            )


        selected_per_round.append(
            round_record.selected_count
        )


        swap_nodes_per_round.append(
            tuple(
                candidate.swap_node_name
                for candidate
                in round_record.selected_candidates
            )
        )


        simulator.kernel.advance_slot()


    else:

        raise RuntimeError(
            f"Policy '{policy.policy_name}' failed to "
            f"complete within 20 ideal swap rounds."
        )


    final_active = tuple(
        simulator
        .entanglement_manager
        .active_resources
    )


    assert (
        len(
            final_active
        )
        == 1
    )


    final_resource = (
        final_active[0]
    )


    depth = (
        entanglement_resource_tree_depth(
            simulator.entanglement_manager,
            final_resource
        )
    )


    tree_size = (
        entanglement_resource_tree_size(
            simulator.entanglement_manager,
            final_resource
        )
    )


    assert (
        tree_size[
            "elementary_leaves"
        ]
        == 7
    )


    assert (
        tree_size[
            "successful_swaps_in_final_tree"
        ]
        == 6
    )


    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    return {
        "Policy":
            policy.policy_name,

        "Swap Rounds":
            len(
                selected_per_round
            ),

        "Selected Per Round":
            tuple(
                selected_per_round
            ),

        "Swap Nodes Per Round":
            tuple(
                swap_nodes_per_round
            ),

        "Realized Tree Depth":
            depth,

        "Final Tree Leaves":
            tree_size[
                "elementary_leaves"
            ],

        "Final Tree Internal Swaps":
            tree_size[
                "successful_swaps_in_final_tree"
            ],

        "Final Fidelity":
            fidelity_registry
            .fidelity(
                final_resource
            )
    }


# =============================================================================
# THREE POLICY BASELINES
# =============================================================================

phase3e_structural_results = []


phase3e_structural_results.append(

    run_phase3e_structural_policy_test(

        policy=(
            SequentialLeftToRightSwapPolicy()
        ),

        seed=380001
    )
)


phase3e_structural_results.append(

    run_phase3e_structural_policy_test(

        policy=(
            StaticBalancedTreeSwapPolicy(
                elementary_link_count=7
            )
        ),

        seed=380001
    )
)


phase3e_structural_results.append(

    run_phase3e_structural_policy_test(

        policy=(
            MaxParallelOldestFirstSwapPolicy()
        ),

        seed=380001
    )
)


phase3e_structural_df = pd.DataFrame(
    phase3e_structural_results
)


display(
    phase3e_structural_df
)

sequential_row = (
    phase3e_structural_results[0]
)

assert (
    sequential_row[
        "Selected Per Round"
    ]
    == (
        1,
        1,
        1,
        1,
        1,
        1
    )
)

assert (
    sequential_row[
        "Realized Tree Depth"
    ]
    == 6
)


# Static balanced:
balanced_row = (
    phase3e_structural_results[1]
)

assert (
    balanced_row[
        "Selected Per Round"
    ]
    == (
        3,
        2,
        1
    )
)

assert (
    balanced_row[
        "Realized Tree Depth"
    ]
    == 3
)


# Adaptive:
adaptive_row = (
    phase3e_structural_results[2]
)

assert (
    adaptive_row[
        "Selected Per Round"
    ]
    == (
        3,
        2,
        1
    )
)

assert (
    adaptive_row[
        "Realized Tree Depth"
    ]
    == 3
)


print(
    "[PASS] Sequential policy requires six serial BSM rounds."
)

print(
    "[PASS] Static balanced policy completes through a "
    "3 -> 2 -> 1 planned parallel structure."
)

print(
    "[PASS] Adaptive max-parallel policy independently "
    "realizes a three-level tree without receiving one beforehand."
)

In [ ]:
# @title  Fresh Six-Repeater Policy Evaluation Stack


def build_phase3e_policy_stack(
    policy,
    simulation_seed: int,
    atomic_bsm_success_probability: float = 0.40,
    max_storage_slots: int = 100
):

    # =========================================================================
    # Topology
    # =========================================================================

    topology = NetworkTopology(
        name=(
            f"Phase3E-Physical-{policy.policy_name}"
        )
    )


    user_a = NetworkNode(
        name="Eval-A",
        x_km=0,
        y_km=0,
        endpoint_capable=True,
        swapping_capable=False,
        memory_capacity=2
    )


    repeaters = tuple(

        NetworkNode(
            name=f"Eval-R{i}",
            x_km=5 * i,
            y_km=0,
            endpoint_capable=False,
            swapping_capable=True,
            memory_capacity=2
        )

        for i in range(
            1,
            7
        )
    )


    user_b = NetworkNode(
        name="Eval-B",
        x_km=35,
        y_km=0,
        endpoint_capable=True,
        swapping_capable=False,
        memory_capacity=2
    )


    path = (
        user_a,
        *repeaters,
        user_b
    )


    for node in path:

        topology.add_node(
            node
        )


    for left, right in zip(
        path[:-1],
        path[1:]
    ):

        topology.add_link(

            PhysicalQuantumLink(
                node_a=left,
                node_b=right,
                attenuation_db_per_km=0.2
            )
        )


    # =========================================================================
    # Simulator
    # =========================================================================

    simulator = (
        QuantumNetworkSimulator(

            topology=topology,

            simulation_seed=(
                simulation_seed
            ),

            controller_id=(
                f"EVAL-{policy.policy_name}"
            ),

            parameter_set=(
                MLDSAParameterSet.ML_DSA_65
            ),

            max_storage_slots=(
                max_storage_slots
            )
        )
    )


    authentication = (
        simulator
        .authenticate_all_nodes()
    )


    assert all(
        record.status
        == AuthenticationStatus.AUTHENTICATED
        for record
        in authentication.values()
    )


    # =========================================================================
    # Elementary Generation
    # =========================================================================

    generation_service = (
        ElementaryEntanglementService(

            simulator=simulator,

            model=(
                FiberSurvivalProxyModel()
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=simulator,

            service=generation_service,

            ordered_path=path,

            demand_active=False
        )
    )


    # =========================================================================
    # Physical Fidelity
    # =========================================================================

    fidelity_profile = (
        WernerPhysicalFidelityProfile(

            elementary_fidelity=0.995,

            bell_fidelity_floor=0.25,

            memory_reference_initial_fidelity=1.0,

            memory_reference_fidelity=0.90,

            memory_reference_time_s=1.0,

            slot_duration_s=0.05,

            swap_operation_visibility=1.0
        )
    )


    fidelity_registry = (
        PhysicalEntanglementFidelityRegistry(

            simulator=simulator,

            profile=fidelity_profile
        )
    )


    # =========================================================================
    # Stochastic Atomic BSM
    # =========================================================================

    swap_model = (
        WernerStateAtomicBSMModel(

            fidelity_profile=(
                fidelity_profile
            ),

            atomic_bsm_success_probability=(
                atomic_bsm_success_probability
            ),

            atomic_bsm_latency_us=10.0
        )
    )


    swap_engine = (
        EntanglementSwapEngine(

            simulator=simulator,

            model=swap_model,

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    # =========================================================================
    # Policy-Controlled Swap Scheduler
    # =========================================================================

    swap_scheduler = (
        ControlPlaneParallelSwapScheduler(

            simulator=simulator,

            swap_engine=swap_engine,

            ordered_path=path,

            policy=policy
        )
    )


    # =========================================================================
    # Failure-Aware Regeneration
    # =========================================================================

    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=simulator,

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=path
        )
    )


    # =========================================================================
    # Autonomous Orchestrator
    # =========================================================================

    orchestrator = (
        FailureAwareEndToEndEntanglementOrchestrator(

            simulator=simulator,

            ordered_path=path,

            generation_scheduler=(
                generation_scheduler
            ),

            demand_controller=(
                demand_controller
            ),

            swap_scheduler=(
                swap_scheduler
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    return {
        "policy":
            policy,

        "topology":
            topology,

        "simulator":
            simulator,

        "path":
            path,

        "authentication":
            authentication,

        "generation_service":
            generation_service,

        "generation_scheduler":
            generation_scheduler,

        "fidelity_profile":
            fidelity_profile,

        "fidelity_registry":
            fidelity_registry,

        "swap_model":
            swap_model,

        "swap_engine":
            swap_engine,

        "swap_scheduler":
            swap_scheduler,

        "demand_controller":
            demand_controller,

        "orchestrator":
            orchestrator
    }

In [ ]:
# @title  Physical Policy Integration Smoke Test


def run_phase3e_policy_once(
    policy,
    simulation_seed: int,
    max_execution_slots: int = 10000
):

    stack = (
        build_phase3e_policy_stack(

            policy=policy,

            simulation_seed=(
                simulation_seed
            ),

            atomic_bsm_success_probability=0.40,

            max_storage_slots=100
        )
    )


    result = (
        stack[
            "orchestrator"
        ]
        .run_until_end_to_end(
            max_slots=(
                max_execution_slots
            )
        )
    )


    if not result.success:

        raise RuntimeError(
            f"Policy '{policy.policy_name}' did not "
            f"establish end-to-end entanglement within "
            f"{max_execution_slots} slots."
        )


    simulator = (
        stack[
            "simulator"
        ]
    )


    final_resource = (
        simulator
        .entanglement_manager
        .get_resource(
            result
            .end_to_end_resource_id
        )
    )


    final_tree_depth = (
        entanglement_resource_tree_depth(

            simulator
            .entanglement_manager,

            final_resource
        )
    )


    final_tree_size = (
        entanglement_resource_tree_size(

            simulator
            .entanglement_manager,

            final_resource
        )
    )


    swap_round_count = sum(

        1

        for step in (
            result.execution_history
        )

        if (
            step.swap_round
            is not None

            and

            step.bsm_attempt_count
            > 0
        )
    )


    modeled_time_s = (
        result
        .simulation_slots_elapsed

        *
        stack[
            "fidelity_profile"
        ]
        .slot_duration_s
    )


    # Total historical resource count includes retries and
    # destroyed/expired/discarded resources.
    historical_resource_count = len(
        simulator
        .entanglement_manager
        .resources
    )


    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    row = {
        "Policy":
            policy.policy_name,

        "Success":
            result.success,

        "Processed Slots":
            result.processed_slot_count,

        "Elapsed Simulation Slots":
            result.simulation_slots_elapsed,

        "Modeled Time (s)":
            modeled_time_s,

        "Elementary Attempts":
            result.total_elementary_attempts,

        "Elementary Successes":
            result.total_elementary_successes,

        "Elementary Failures":
            result.total_elementary_failures,

        "BSM Attempts":
            result.total_bsm_attempts,

        "BSM Successes":
            result.total_bsm_successes,

        "BSM Failures":
            result.total_bsm_failures,

        "Executed Swap Rounds":
            swap_round_count,

        "Final Tree Depth":
            final_tree_depth,

        "Final Tree Leaves":
            final_tree_size[
                "elementary_leaves"
            ],

        "Final Tree Swaps":
            final_tree_size[
                "successful_swaps_in_final_tree"
            ],

        "Historical Resources":
            historical_resource_count,

        "Final Fidelity":
            result.final_fidelity
    }


    return (
        row,
        stack,
        result
    )

phase3e_physical_runs = []


# IMPORTANT:
#
# We use the same initial seed for reproducibility.
#
# This does NOT mean the stochastic trajectories remain perfectly
# paired after policies begin making different decisions, because
# they may consume simulation RNG in different orders.
#
# Proper statistical comparison comes in Monte Carlo.


POLICY_SMOKE_SEED = 390001


# =============================================================================
# 1. Sequential
# =============================================================================

sequential_output = (
    run_phase3e_policy_once(

        SequentialLeftToRightSwapPolicy(),

        simulation_seed=(
            POLICY_SMOKE_SEED
        )
    )
)


phase3e_physical_runs.append(
    sequential_output
)


print(
    "[PASS] Sequential physical policy completed."
)


# =============================================================================
# 2. Static Balanced
# =============================================================================

balanced_output = (
    run_phase3e_policy_once(

        StaticBalancedTreeSwapPolicy(
            elementary_link_count=7
        ),

        simulation_seed=(
            POLICY_SMOKE_SEED
        )
    )
)


phase3e_physical_runs.append(
    balanced_output
)


print(
    "[PASS] Static-balanced physical policy completed."
)


# =============================================================================
# 3. Adaptive
# =============================================================================

adaptive_output = (
    run_phase3e_policy_once(

        MaxParallelOldestFirstSwapPolicy(),

        simulation_seed=(
            POLICY_SMOKE_SEED
        )
    )
)


phase3e_physical_runs.append(
    adaptive_output
)


print(
    "[PASS] Adaptive max-parallel oldest-first policy completed."
)


# =============================================================================
# Comparison
# =============================================================================

phase3e_physical_comparison = pd.DataFrame([
    output[0]
    for output
    in phase3e_physical_runs
])


display(
    phase3e_physical_comparison
)

### Phase 3 MC Distance Variance : 14-09-2026T19:39:21 -- 18-09-2026T13:16:38

In [ ]:
# @title Monte Carlo Configuration and Durable Checkpointing

from pathlib import Path
from collections import Counter

import os
import gc
import json
import math
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm


# =============================================================================
# EXPERIMENT DESIGN
# =============================================================================

PHASE3G_DISTANCES_KM = (
    100.0,
    200.0,
    250.0,
    500.0,
    750.0,
    1000.0,
    600,
    700
)

PHASE3G_POLICY_NAMES = (
    "SEQUENTIAL_LEFT_TO_RIGHT",
    "STATIC_BALANCED_TREE",
    "MAX_PARALLEL_OLDEST_FIRST"
)

PHASE3G_TRIALS_PER_POLICY_DISTANCE = 100

PHASE3G_REPEATER_COUNT = 8

# 8 repeaters imply:
#
# User-A -- R1 -- R2 -- ... -- R8 -- User-B
#
# = 9 elementary fibre segments.
PHASE3G_ELEMENTARY_LINK_COUNT = (
    PHASE3G_REPEATER_COUNT
    + 1
)

PHASE3G_ATOMIC_BSM_SUCCESS_PROBABILITY = 0.40

PHASE3G_MAX_STORAGE_SLOTS = 100

PHASE3G_SLOT_DURATION_S = 0.05

PHASE3G_ELEMENTARY_FIDELITY = 0.995

PHASE3G_FIBER_ATTENUATION_DB_PER_KM = 0.20

# ---------------------------------------------------------------------------
# Uniform censoring horizon.
#
# A trial that has not established E(User-A, User-B) by this point
# is recorded as TIMEOUT.
# ---------------------------------------------------------------------------

PHASE3G_MAX_EXECUTION_SLOTS = 20000


# ---------------------------------------------------------------------------
# Physical simulation seeds.
#
# The SAME seed is supplied to all three policies for a given
# (distance, trial) pair.
#
# Because policies eventually consume RNG in different orders, this is not
# perfect common-random-number coupling after their trajectories diverge.
# It does, however, provide deterministic and reproducible initial streams.
# ---------------------------------------------------------------------------

PHASE3G_BASE_SEED = 3700000


# ---------------------------------------------------------------------------
# Stop after checkpointing a programming/runtime ERROR.
#
# TIMEOUT is a valid Monte Carlo outcome and never raises.
# ---------------------------------------------------------------------------

PHASE3G_STOP_ON_ERROR = True


# =============================================================================
# RUN TAG
#
# IMPORTANT:
# Change this tag if you change major experimental parameters.
# =============================================================================

PHASE3G_RUN_TAG = (
    "phase3_mc_v1_"
    "8R_100trials_"
    "pb040_"
    "mem100_"
    "max20000"
)


# =============================================================================
# CHECKPOINT DIRECTORY
#
# Prefer mounted Google Drive when available.
# Otherwise fall back to local Colab storage.
# =============================================================================

drive_root = Path(
    "/content/drive/MyDrive"
)

if drive_root.exists():

    PHASE3G_CHECKPOINT_DIR = (
        drive_root
        / "NARAD_QC"
        / "Phase3"
        / PHASE3G_RUN_TAG
    )

else:

    PHASE3G_CHECKPOINT_DIR = (
        Path("/content")
        / "NARAD_QC"
        / "Phase3G"
        / PHASE3G_RUN_TAG
    )


PHASE3G_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PHASE3G_RESULTS_PATH = (
    PHASE3G_CHECKPOINT_DIR
    / "phase3g_trial_results.csv"
)

PHASE3G_CONFIG_PATH = (
    PHASE3G_CHECKPOINT_DIR
    / "phase3g_experiment_config.json"
)

PHASE3G_SUMMARY_PATH = (
    PHASE3G_CHECKPOINT_DIR
    / "phase3g_summary.csv"
)

PHASE3G_PLOT_DIR = (
    PHASE3G_CHECKPOINT_DIR
    / "plots"
)

PHASE3G_PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# REPRODUCIBLE EXPERIMENT CONFIGURATION
# =============================================================================

phase3g_config = {

    "run_tag":
        PHASE3G_RUN_TAG,

    "distances_km":
        list(
            PHASE3G_DISTANCES_KM
        ),

    "policies":
        list(
            PHASE3G_POLICY_NAMES
        ),

    "trials_per_policy_distance":
        PHASE3G_TRIALS_PER_POLICY_DISTANCE,

    "repeater_count":
        PHASE3G_REPEATER_COUNT,

    "elementary_link_count":
        PHASE3G_ELEMENTARY_LINK_COUNT,

    "atomic_bsm_success_probability":
        PHASE3G_ATOMIC_BSM_SUCCESS_PROBABILITY,

    "max_storage_slots":
        PHASE3G_MAX_STORAGE_SLOTS,

    "slot_duration_s":
        PHASE3G_SLOT_DURATION_S,

    "elementary_fidelity":
        PHASE3G_ELEMENTARY_FIDELITY,

    "fiber_attenuation_db_per_km":
        PHASE3G_FIBER_ATTENUATION_DB_PER_KM,

    "max_execution_slots":
        PHASE3G_MAX_EXECUTION_SLOTS,

    "base_seed":
        PHASE3G_BASE_SEED
}


# =============================================================================
# Protect against accidentally resuming with different parameters.
# =============================================================================

if PHASE3G_CONFIG_PATH.exists():

    with open(
        PHASE3G_CONFIG_PATH,
        "r"
    ) as config_file:

        existing_config = json.load(
            config_file
        )


    if (
        existing_config
        != phase3g_config
    ):

        raise RuntimeError(
            "Checkpoint directory already contains a DIFFERENT "
            "Phase-3G experiment configuration. Change "
            "PHASE3G_RUN_TAG before continuing."
        )

else:

    with open(
        PHASE3G_CONFIG_PATH,
        "w"
    ) as config_file:

        json.dump(
            phase3g_config,
            config_file,
            indent=2
        )


# =============================================================================
# ATOMIC DATAFRAME CHECKPOINT
# =============================================================================

def phase3g_atomic_write_csv(
    dataframe: pd.DataFrame,
    path: Path
):

    temporary_path = (
        path.with_name(
            path.name
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=False
    )


    os.replace(
        temporary_path,
        path
    )


# =============================================================================
# Load Existing Progress
# =============================================================================

if PHASE3G_RESULTS_PATH.exists():

    phase3g_results = pd.read_csv(
        PHASE3G_RESULTS_PATH
    )

    print(
        f"[INFO] Loaded "
        f"{len(phase3g_results)} checkpointed trial records."
    )

else:

    phase3g_results = pd.DataFrame()

    print(
        "[INFO] No existing trial checkpoint found."
    )


PHASE3G_TOTAL_EXPECTED_TRIALS = (
    len(
        PHASE3G_DISTANCES_KM
    )
    *
    len(
        PHASE3G_POLICY_NAMES
    )
    *
    PHASE3G_TRIALS_PER_POLICY_DISTANCE
)


print(
    "\nPhase 3G Monte Carlo experiment"
)

print(
    "Distances:",
    PHASE3G_DISTANCES_KM
)

print(
    "Policies:",
    PHASE3G_POLICY_NAMES
)

print(
    "Repeaters:",
    PHASE3G_REPEATER_COUNT
)

print(
    "Elementary links:",
    PHASE3G_ELEMENTARY_LINK_COUNT
)

print(
    "Trials per policy/distance:",
    PHASE3G_TRIALS_PER_POLICY_DISTANCE
)

print(
    "Total complete sessions:",
    PHASE3G_TOTAL_EXPECTED_TRIALS
)

print(
    "Checkpoint directory:",
    PHASE3G_CHECKPOINT_DIR
)

In [ ]:
# @title Distance Scenario Inspection


phase3g_scenario_rows = []


for total_distance_km in (
    PHASE3G_DISTANCES_KM
):

    segment_distance_km = (
        total_distance_km
        /
        PHASE3G_ELEMENTARY_LINK_COUNT
    )


    fiber_survival_probability = (
        10.0
        **
        (
            -(
                PHASE3G_FIBER_ATTENUATION_DB_PER_KM
                *
                segment_distance_km
            )
            /
            10.0
        )
    )


    phase3g_scenario_rows.append({

        "Total User Separation (km)":
            total_distance_km,

        "Repeaters":
            PHASE3G_REPEATER_COUNT,

        "Elementary Links":
            PHASE3G_ELEMENTARY_LINK_COUNT,

        "Segment Distance (km)":
            segment_distance_km,

        "Elementary Survival Proxy":
            fiber_survival_probability,

        "Mean Attempts / Elementary Success":
            (
                1.0
                /
                fiber_survival_probability
            )
    })


phase3g_scenarios = pd.DataFrame(
    phase3g_scenario_rows
)


display(
    phase3g_scenarios
)

In [ ]:
# @title Monte Carlo Policy Factory


def phase3g_make_policy(
    policy_name: str,
    elementary_link_count: int
):

    if (
        policy_name
        == "SEQUENTIAL_LEFT_TO_RIGHT"
    ):

        return (
            SequentialLeftToRightSwapPolicy()
        )


    if (
        policy_name
        == "STATIC_BALANCED_TREE"
    ):

        return (
            StaticBalancedTreeSwapPolicy(
                elementary_link_count=(
                    elementary_link_count
                )
            )
        )


    if (
        policy_name
        == "MAX_PARALLEL_OLDEST_FIRST"
    ):

        return (
            MaxParallelOldestFirstSwapPolicy()
        )


    raise ValueError(
        f"Unknown Phase-3G policy: "
        f"{policy_name}"
    )


# =============================================================================
# Structural sanity check for 9 leaves.
# =============================================================================

phase3g_balanced_sanity = (
    phase3g_make_policy(
        "STATIC_BALANCED_TREE",
        elementary_link_count=9
    )
)


assert (
    len(
        phase3g_balanced_sanity
        .planned_merges
    )
    == 8
)


assert (
    phase3g_balanced_sanity
    .tree_depth
    == 4
)


print(
    "[PASS] Nine-link static-balanced benchmark contains "
    "8 planned merges with tree depth 4."
)

In [ ]:
# @title Fresh Eight-Repeater Physical Trial Stack


def build_phase3g_trial_stack(
    policy_name: str,
    total_user_separation_km: float,
    simulation_seed: int
):

    # =========================================================================
    # Geometry
    # =========================================================================

    repeater_count = (
        PHASE3G_REPEATER_COUNT
    )

    elementary_link_count = (
        repeater_count
        + 1
    )


    segment_distance_km = (
        float(
            total_user_separation_km
        )
        /
        elementary_link_count
    )


    # =========================================================================
    # Policy
    # =========================================================================

    policy = (
        phase3g_make_policy(

            policy_name=(
                policy_name
            ),

            elementary_link_count=(
                elementary_link_count
            )
        )
    )


    # =========================================================================
    # Topology
    #
    # MC-A -- R1 -- ... -- R8 -- MC-B
    # =========================================================================

    topology = NetworkTopology(
        name=(
            f"Phase3G-"
            f"{policy_name}-"
            f"{total_user_separation_km:.1f}km"
        )
    )


    user_a = NetworkNode(

        name="MC-A",

        x_km=0.0,

        y_km=0.0,

        endpoint_capable=True,

        swapping_capable=False,

        memory_capacity=2
    )


    repeaters = tuple(

        NetworkNode(

            name=f"MC-R{i}",

            x_km=(
                segment_distance_km
                * i
            ),

            y_km=0.0,

            endpoint_capable=False,

            swapping_capable=True,

            memory_capacity=2
        )

        for i in range(
            1,
            repeater_count + 1
        )
    )


    user_b = NetworkNode(

        name="MC-B",

        x_km=float(
            total_user_separation_km
        ),

        y_km=0.0,

        endpoint_capable=True,

        swapping_capable=False,

        memory_capacity=2
    )


    path = (
        user_a,
        *repeaters,
        user_b
    )


    for node in path:

        topology.add_node(
            node
        )


    for left, right in zip(
        path[:-1],
        path[1:]
    ):

        topology.add_link(

            PhysicalQuantumLink(

                node_a=left,

                node_b=right,

                attenuation_db_per_km=(
                    PHASE3G_FIBER_ATTENUATION_DB_PER_KM
                )
            )
        )


    assert (
        topology.node_count
        == repeater_count + 2
    )


    assert (
        topology.link_count
        == elementary_link_count
    )


    # =========================================================================
    # Simulator
    # =========================================================================

    simulator = (
        QuantumNetworkSimulator(

            topology=topology,

            simulation_seed=(
                simulation_seed
            ),

            controller_id=(
                f"PHASE3G-{policy_name}"
            ),

            parameter_set=(
                MLDSAParameterSet.ML_DSA_65
            ),

            max_storage_slots=(
                PHASE3G_MAX_STORAGE_SLOTS
            )
        )
    )


    # =========================================================================
    # Authentication
    #
    # Fresh trial = fresh authenticated runtime.
    #
    # ML-DSA cryptographic randomness remains separate from the
    # simulation-physics RNG.
    # =========================================================================

    authentication = (
        simulator
        .authenticate_all_nodes()
    )


    assert all(
        record.status
        == AuthenticationStatus.AUTHENTICATED

        for record
        in authentication.values()
    )


    assert (
        simulator
        .require_quantum_eligibility(
            *path
        )
    )


    # =========================================================================
    # Elementary Entanglement Generation
    # =========================================================================

    generation_service = (
        ElementaryEntanglementService(

            simulator=simulator,

            model=(
                FiberSurvivalProxyModel()
            )
        )
    )


    # Demand begins OFF.
    #
    # FailureAwareElementaryDemandController owns generation authority.
    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=simulator,

            service=generation_service,

            ordered_path=path,

            demand_active=False
        )
    )


    # =========================================================================
    # Physical Fidelity
    # =========================================================================

    fidelity_profile = (
        WernerPhysicalFidelityProfile(

            elementary_fidelity=(
                PHASE3G_ELEMENTARY_FIDELITY
            ),

            bell_fidelity_floor=0.25,

            memory_reference_initial_fidelity=1.0,

            memory_reference_fidelity=0.90,

            memory_reference_time_s=1.0,

            slot_duration_s=(
                PHASE3G_SLOT_DURATION_S
            ),

            swap_operation_visibility=1.0
        )
    )


    fidelity_registry = (
        PhysicalEntanglementFidelityRegistry(

            simulator=simulator,

            profile=fidelity_profile
        )
    )


    # =========================================================================
    # Failure-Prone Atomic BSM
    # =========================================================================

    swap_model = (
        WernerStateAtomicBSMModel(

            fidelity_profile=(
                fidelity_profile
            ),

            atomic_bsm_success_probability=(
                PHASE3G_ATOMIC_BSM_SUCCESS_PROBABILITY
            ),

            atomic_bsm_latency_us=10.0
        )
    )


    swap_engine = (
        EntanglementSwapEngine(

            simulator=simulator,

            model=swap_model,

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    # =========================================================================
    # Policy-Controlled Parallel Scheduler
    # =========================================================================

    swap_scheduler = (
        ControlPlaneParallelSwapScheduler(

            simulator=simulator,

            swap_engine=swap_engine,

            ordered_path=path,

            policy=policy
        )
    )


    # =========================================================================
    # Failure-Aware Selective Regeneration
    # =========================================================================

    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=simulator,

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=path
        )
    )


    # =========================================================================
    # Autonomous Session Orchestrator
    # =========================================================================

    orchestrator = (
        FailureAwareEndToEndEntanglementOrchestrator(

            simulator=simulator,

            ordered_path=path,

            generation_scheduler=(
                generation_scheduler
            ),

            demand_controller=(
                demand_controller
            ),

            swap_scheduler=(
                swap_scheduler
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    # =========================================================================
    # Link-level physical metadata
    # =========================================================================

    physical_links = (
        topology.path_links(
            path
        )
    )


    link_success_probability = (
        physical_links[0]
        .survival_probability
    )


    return {

        "policy":
            policy,

        "topology":
            topology,

        "simulator":
            simulator,

        "path":
            path,

        "repeaters":
            repeaters,

        "generation_service":
            generation_service,

        "generation_scheduler":
            generation_scheduler,

        "fidelity_profile":
            fidelity_profile,

        "fidelity_registry":
            fidelity_registry,

        "swap_model":
            swap_model,

        "swap_engine":
            swap_engine,

        "swap_scheduler":
            swap_scheduler,

        "demand_controller":
            demand_controller,

        "orchestrator":
            orchestrator,

        "segment_distance_km":
            segment_distance_km,

        "elementary_success_probability":
            link_success_probability
    }

In [ ]:
# @title Single Monte Carlo Trial Runner


def run_phase3g_single_trial(
    policy_name: str,
    total_user_separation_km: float,
    trial_number: int,
    simulation_seed: int
):

    wall_start = (
        time.perf_counter()
    )


    stack = (
        build_phase3g_trial_stack(

            policy_name=(
                policy_name
            ),

            total_user_separation_km=(
                total_user_separation_km
            ),

            simulation_seed=(
                simulation_seed
            )
        )
    )


    simulator = (
        stack[
            "simulator"
        ]
    )


    result = (
        stack[
            "orchestrator"
        ]
        .run_until_end_to_end(

            max_slots=(
                PHASE3G_MAX_EXECUTION_SLOTS
            )
        )
    )


    # =========================================================================
    # Trial status
    # =========================================================================

    if result.success:

        trial_status = (
            "SUCCESS"
        )

    else:

        trial_status = (
            "TIMEOUT"
        )


    # =========================================================================
    # Historical Resource Statistics
    # =========================================================================

    all_resources = tuple(
        simulator
        .entanglement_manager
        .resources
    )


    resource_status_counts = Counter(
        resource.status.value
        for resource
        in all_resources
    )


    # =========================================================================
    # Swap Round Statistics
    # =========================================================================

    physical_swap_rounds = tuple(

        round_record

        for round_record in (
            stack[
                "swap_scheduler"
            ]
            .round_history
        )

        if (
            round_record.selected_count
            > 0
        )
    )


    swap_round_count = len(
        physical_swap_rounds
    )


    if swap_round_count > 0:

        mean_parallel_bsms = (
            result.total_bsm_attempts
            /
            swap_round_count
        )

    else:

        mean_parallel_bsms = 0.0


    # =========================================================================
    # Age-Aware Selection Metrics
    #
    # Meaningful for every policy because SwapCandidate stores age.
    # =========================================================================

    selected_candidate_ages = [

        candidate.oldest_input_half_age_slots

        for round_record in physical_swap_rounds

        for candidate
        in round_record.selected_candidates
    ]


    if selected_candidate_ages:

        mean_selected_oldest_age = float(
            np.mean(
                selected_candidate_ages
            )
        )

        max_selected_oldest_age = int(
            np.max(
                selected_candidate_ages
            )
        )

    else:

        mean_selected_oldest_age = np.nan

        max_selected_oldest_age = np.nan


    # =========================================================================
    # Failure / Regeneration Workload
    #
    # This is the sum of uncovered elementary-link demands observed
    # after orchestration steps.
    #
    # It is a workload metric, NOT a count of unique physical failures.
    # =========================================================================

    regeneration_demand_workload = sum(

        len(
            step
            .demand_after_swapping
            .regeneration_link_ids
        )

        for step
        in result.execution_history
    )


    bsm_failure_rounds = sum(

        1

        for step
        in result.execution_history

        if (
            step.bsm_failure_count
            > 0
        )
    )


    # =========================================================================
    # Successful Final Resource Metrics
    # =========================================================================

    final_tree_depth = np.nan

    final_tree_leaves = np.nan

    final_tree_swaps = np.nan

    final_fidelity = np.nan

    final_resource_id = None


    if result.success:

        final_resource_id = (
            result.end_to_end_resource_id
        )


        final_resource = (
            simulator
            .entanglement_manager
            .get_resource(
                final_resource_id
            )
        )


        final_tree_depth = (
            entanglement_resource_tree_depth(

                simulator
                .entanglement_manager,

                final_resource
            )
        )


        final_tree_size = (
            entanglement_resource_tree_size(

                simulator
                .entanglement_manager,

                final_resource
            )
        )


        final_tree_leaves = (
            final_tree_size[
                "elementary_leaves"
            ]
        )


        final_tree_swaps = (
            final_tree_size[
                "successful_swaps_in_final_tree"
            ]
        )


        final_fidelity = (
            result.final_fidelity
        )


        # -----------------------------------------------------
        # Eight repeaters / nine elementary links:
        #
        # final tree must contain
        #     9 leaves
        #     8 successful internal BSM merges.
        # -----------------------------------------------------

        assert (
            final_tree_leaves
            == PHASE3G_ELEMENTARY_LINK_COUNT
        )


        assert (
            final_tree_swaps
            == PHASE3G_REPEATER_COUNT
        )


    # =========================================================================
    # Runtime Consistency
    # =========================================================================

    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    # =========================================================================
    # Cross-check aggregate attempt accounting.
    # =========================================================================

    assert (
        stack[
            "generation_service"
        ]
        .statistics
        .total_attempts
        ==
        result.total_elementary_attempts
    )


    assert (
        len(
            stack[
                "swap_engine"
            ]
            .attempts
        )
        ==
        result.total_bsm_attempts
    )


    # =========================================================================
    # Timing
    # =========================================================================

    modeled_physical_time_s = (
        result.simulation_slots_elapsed

        *
        PHASE3G_SLOT_DURATION_S
    )


    wall_clock_seconds = (
        time.perf_counter()
        - wall_start
    )


    # =========================================================================
    # Trial Key
    # =========================================================================

    trial_key = (
        f"{policy_name}|"
        f"{float(total_user_separation_km):.1f}|"
        f"{int(trial_number)}"
    )


    return {

        "TrialKey":
            trial_key,

        "TrialStatus":
            trial_status,

        "Policy":
            policy_name,

        "TotalDistanceKm":
            float(
                total_user_separation_km
            ),

        "Trial":
            int(
                trial_number
            ),

        "SimulationSeed":
            int(
                simulation_seed
            ),

        "Repeaters":
            PHASE3G_REPEATER_COUNT,

        "ElementaryLinks":
            PHASE3G_ELEMENTARY_LINK_COUNT,

        "SegmentDistanceKm":
            stack[
                "segment_distance_km"
            ],

        "ElementarySuccessProbability":
            stack[
                "elementary_success_probability"
            ],

        "AtomicBSMSuccessProbability":
            PHASE3G_ATOMIC_BSM_SUCCESS_PROBABILITY,

        "MaxStorageSlots":
            PHASE3G_MAX_STORAGE_SLOTS,

        "Success":
            bool(
                result.success
            ),

        "ProcessedSlots":
            result.processed_slot_count,

        "ElapsedSimulationSlots":
            result.simulation_slots_elapsed,

        # -----------------------------------------------------
        # Restricted latency:
        #
        # Successful trial -> actual processed slots
        # Timeout          -> censoring horizon
        #
        # Useful when distant cases have substantial censoring.
        # -----------------------------------------------------

        "RestrictedCompletionSlots":
            (
                result.processed_slot_count

                if result.success

                else PHASE3G_MAX_EXECUTION_SLOTS
            ),

        "ModeledPhysicalTimeS":
            modeled_physical_time_s,

        "ElementaryAttempts":
            result.total_elementary_attempts,

        "ElementarySuccesses":
            result.total_elementary_successes,

        "ElementaryFailures":
            result.total_elementary_failures,

        "BSMAttempts":
            result.total_bsm_attempts,

        "BSMSuccesses":
            result.total_bsm_successes,

        "BSMFailures":
            result.total_bsm_failures,

        "BSMFailureRounds":
            bsm_failure_rounds,

        "SwapRounds":
            swap_round_count,

        "MeanParallelBSMsPerRound":
            mean_parallel_bsms,

        "RegenerationDemandWorkload":
            regeneration_demand_workload,

        "MeanSelectedOldestInputAge":
            mean_selected_oldest_age,

        "MaxSelectedOldestInputAge":
            max_selected_oldest_age,

        "HistoricalResources":
            len(
                all_resources
            ),

        "ConsumedResources":
            resource_status_counts.get(
                EntanglementStatus.CONSUMED.value,
                0
            ),

        "ExpiredResources":
            resource_status_counts.get(
                EntanglementStatus.EXPIRED.value,
                0
            ),

        "DiscardedResources":
            resource_status_counts.get(
                EntanglementStatus.DISCARDED.value,
                0
            ),

        "FinalResourceID":
            final_resource_id,

        "FinalTreeDepth":
            final_tree_depth,

        "FinalTreeLeaves":
            final_tree_leaves,

        "FinalTreeSwaps":
            final_tree_swaps,

        "FinalFidelity":
            final_fidelity,

        # Pure compute time. Never confuse with network physical time.
        "WallClockSeconds":
            wall_clock_seconds
    }

In [ ]:
# @title Full 2400-Trial Monte Carlo Experiment

# =============================================================================
# Current Valid Completed Trials
#
# ERROR records are deliberately NOT treated as complete.
# They will be retried after the bug is fixed.
# =============================================================================

def phase3g_completed_trial_keys(
    dataframe
):

    if dataframe.empty:

        return set()


    valid = dataframe[
        dataframe[
            "TrialStatus"
        ].isin(
            [
                "SUCCESS",
                "TIMEOUT"
            ]
        )
    ]


    return set(
        valid[
            "TrialKey"
        ].astype(
            str
        )
    )


completed_keys = (
    phase3g_completed_trial_keys(
        phase3g_results
    )
)


# =============================================================================
# Deterministic Physical Seed
#
# Same (distance, trial) seed across all three policies.
# =============================================================================

def phase3g_seed_for_trial(
    distance_index: int,
    trial_number: int
):

    return int(
        PHASE3G_BASE_SEED

        +
        distance_index
        * 100_000

        +
        trial_number
    )


# =============================================================================
# Upsert One Trial + Atomic Checkpoint
# =============================================================================

def phase3g_checkpoint_trial(
    current_results: pd.DataFrame,
    trial_row: dict
):

    trial_key = (
        trial_row[
            "TrialKey"
        ]
    )


    # Remove older ERROR/retry record with the same key.
    if (
        not current_results.empty
        and "TrialKey"
        in current_results.columns
    ):

        current_results = (
            current_results[
                current_results[
                    "TrialKey"
                ]
                != trial_key
            ]
            .copy()
        )


    current_results = pd.concat(
        [
            current_results,
            pd.DataFrame(
                [
                    trial_row
                ]
            )
        ],
        ignore_index=True
    )


    current_results = (
        current_results
        .sort_values(
            by=[
                "TotalDistanceKm",
                "Trial",
                "Policy"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # ---------------------------------------------------------
    # CRITICAL:
    # atomic checkpoint after EVERY COMPLETE trial.
    # ---------------------------------------------------------

    phase3g_atomic_write_csv(
        current_results,
        PHASE3G_RESULTS_PATH
    )


    return current_results


# =============================================================================
# Progress
# =============================================================================

already_complete = len(
    completed_keys
)


print(
    f"Already complete: "
    f"{already_complete}/"
    f"{PHASE3G_TOTAL_EXPECTED_TRIALS}"
)


progress = tqdm(

    total=(
        PHASE3G_TOTAL_EXPECTED_TRIALS
    ),

    initial=(
        already_complete
    ),

    desc=(
        "NARAD-QC Phase 3 Monte Carlo"
    )
)


session_wall_start = (
    time.perf_counter()
)


session_new_trials = 0


# =============================================================================
# Main Experiment
#
# Loop order:
#
# distance
#     trial
#         policy
#
# This keeps the three policies for the same physical seed close together.
# =============================================================================

for (
    distance_index,
    total_distance_km
) in enumerate(
    PHASE3G_DISTANCES_KM
):


    for trial_number in range(
        1,
        PHASE3G_TRIALS_PER_POLICY_DISTANCE
        + 1
    ):


        simulation_seed = (
            phase3g_seed_for_trial(

                distance_index=(
                    distance_index
                ),

                trial_number=(
                    trial_number
                )
            )
        )


        for policy_name in (
            PHASE3G_POLICY_NAMES
        ):


            trial_key = (
                f"{policy_name}|"
                f"{float(total_distance_km):.1f}|"
                f"{trial_number}"
            )


            # ================================================================
            # Resume support
            # ================================================================

            if (
                trial_key
                in completed_keys
            ):

                continue


            try:

                trial_row = (
                    run_phase3g_single_trial(

                        policy_name=(
                            policy_name
                        ),

                        total_user_separation_km=(
                            total_distance_km
                        ),

                        trial_number=(
                            trial_number
                        ),

                        simulation_seed=(
                            simulation_seed
                        )
                    )
                )


                # ============================================================
                # CHECKPOINT IMMEDIATELY
                # ============================================================

                phase3g_results = (
                    phase3g_checkpoint_trial(

                        current_results=(
                            phase3g_results
                        ),

                        trial_row=(
                            trial_row
                        )
                    )
                )


                completed_keys.add(
                    trial_key
                )


                session_new_trials += 1


                progress.update(
                    1
                )


                elapsed_wall = (
                    time.perf_counter()
                    - session_wall_start
                )


                average_wall_per_trial = (
                    elapsed_wall
                    /
                    session_new_trials
                )


                remaining_trials = (
                    PHASE3G_TOTAL_EXPECTED_TRIALS
                    - len(
                        completed_keys
                    )
                )


                estimated_remaining_s = (
                    average_wall_per_trial
                    *
                    remaining_trials
                )


                progress.set_postfix({

                    "distance_km":
                        int(
                            total_distance_km
                        ),

                    "trial":
                        trial_number,

                    "policy":
                        policy_name,

                    "status":
                        trial_row[
                            "TrialStatus"
                        ],

                    "sim_slots":
                        trial_row[
                            "ProcessedSlots"
                        ],

                    "eta_h":
                        round(
                            estimated_remaining_s
                            / 3600.0,
                            2
                        )
                })


            except Exception as exc:

                # ============================================================
                # Save the failure itself before stopping.
                # ============================================================

                error_row = {

                    "TrialKey":
                        trial_key,

                    "TrialStatus":
                        "ERROR",

                    "Policy":
                        policy_name,

                    "TotalDistanceKm":
                        float(
                            total_distance_km
                        ),

                    "Trial":
                        int(
                            trial_number
                        ),

                    "SimulationSeed":
                        int(
                            simulation_seed
                        ),

                    "ErrorType":
                        type(
                            exc
                        ).__name__,

                    "ErrorMessage":
                        str(
                            exc
                        ),

                    "Traceback":
                        traceback.format_exc()
                }


                phase3g_results = (
                    phase3g_checkpoint_trial(

                        current_results=(
                            phase3g_results
                        ),

                        trial_row=(
                            error_row
                        )
                    )
                )


                progress.close()


                print(
                    "\n[ERROR] Trial failed and was checkpointed:"
                )

                print(
                    error_row
                )


                if PHASE3G_STOP_ON_ERROR:

                    raise


            finally:

                # Fresh trial objects can be large because they contain
                # complete event/resource histories.
                gc.collect()


progress.close()


# =============================================================================
# Final Checkpoint Verification
# =============================================================================

phase3g_results = pd.read_csv(
    PHASE3G_RESULTS_PATH
)


valid_completed = (
    phase3g_results[
        phase3g_results[
            "TrialStatus"
        ].isin(
            [
                "SUCCESS",
                "TIMEOUT"
            ]
        )
    ]
)


print(
    "\nMonte Carlo execution finished."
)

print(
    "Checkpointed valid trials:",
    len(
        valid_completed
    )
)

print(
    "Expected:",
    PHASE3G_TOTAL_EXPECTED_TRIALS
)

print(
    "Checkpoint:",
    PHASE3G_RESULTS_PATH
)

In [ ]:
# @title Monte Carlo Integrity Verification


phase3g_results = pd.read_csv(
    PHASE3G_RESULTS_PATH
)


phase3g_valid_results = (
    phase3g_results[
        phase3g_results[
            "TrialStatus"
        ].isin(
            [
                "SUCCESS",
                "TIMEOUT"
            ]
        )
    ]
    .copy()
)


# =============================================================================
# Exactly 1,800 completed Monte Carlo sessions.
# =============================================================================

assert (
    len(
        phase3g_valid_results
    )
    ==
    PHASE3G_TOTAL_EXPECTED_TRIALS
)


# =============================================================================
# No duplicate trial keys.
# =============================================================================

assert (
    phase3g_valid_results[
        "TrialKey"
    ]
    .nunique()
    ==
    PHASE3G_TOTAL_EXPECTED_TRIALS
)


# =============================================================================
# Exactly 100 trials for each policy/distance combination.
# =============================================================================

phase3g_counts = (
    phase3g_valid_results
    .groupby(
        [
            "TotalDistanceKm",
            "Policy"
        ]
    )
    .size()
    .reset_index(
        name="Trials"
    )
)


assert all(
    phase3g_counts[
        "Trials"
    ]
    == 100
)


# =============================================================================
# Same 8-repeaters / 9-link architecture everywhere.
# =============================================================================

assert all(
    phase3g_valid_results[
        "Repeaters"
    ]
    == 8
)


assert all(
    phase3g_valid_results[
        "ElementaryLinks"
    ]
    == 9
)


# =============================================================================
# Successful final trees must contain 9 leaves and 8 internal swaps.
# =============================================================================

successful_rows = (
    phase3g_valid_results[
        phase3g_valid_results[
            "Success"
        ]
        == True
    ]
)


if len(
    successful_rows
) > 0:

    assert all(
        successful_rows[
            "FinalTreeLeaves"
        ]
        == 9
    )


    assert all(
        successful_rows[
            "FinalTreeSwaps"
        ]
        == 8
    )


print(
    "[PASS] Exactly 1,800 valid policy trials were completed."
)

print(
    "[PASS] Every policy/distance combination contains 100 trials."
)

print(
    "[PASS] Every scenario uses 8 repeaters and 9 elementary links."
)

print(
    "[PASS] Every successful final resource has a valid "
    "9-leaf / 8-swap realized tree."
)


display(
    phase3g_counts
)

In [ ]:
# @title  Monte Carlo Statistical Aggregation


phase3g_valid_results = (
    pd.read_csv(
        PHASE3G_RESULTS_PATH
    )
)


phase3g_valid_results = (
    phase3g_valid_results[
        phase3g_valid_results[
            "TrialStatus"
        ].isin(
            [
                "SUCCESS",
                "TIMEOUT"
            ]
        )
    ]
    .copy()
)


# =============================================================================
# 95% Mean Confidence Interval
#
# With n=100 this normal approximation is suitable for exploratory
# Monte Carlo reporting.
# =============================================================================

def phase3g_mean_ci95(
    values
):

    values = np.asarray(
        pd.Series(
            values
        )
        .dropna(),
        dtype=float
    )


    n = len(
        values
    )


    if n == 0:

        return (
            np.nan,
            np.nan,
            np.nan
        )


    mean = float(
        np.mean(
            values
        )
    )


    if n == 1:

        return (
            mean,
            mean,
            mean
        )


    sem = float(
        np.std(
            values,
            ddof=1
        )
        /
        math.sqrt(
            n
        )
    )


    half_width = (
        1.96
        * sem
    )


    return (
        mean,
        mean - half_width,
        mean + half_width
    )


# =============================================================================
# Wilson 95% CI for Success Probability
# =============================================================================

def phase3g_wilson_ci95(
    successes: int,
    trials: int
):

    if trials <= 0:

        return (
            np.nan,
            np.nan
        )


    z = 1.96

    p_hat = (
        successes
        /
        trials
    )


    denominator = (
        1.0
        +
        z**2
        /
        trials
    )


    center = (
        p_hat
        +
        z**2
        /
        (
            2.0
            * trials
        )
    ) / denominator


    half_width = (
        z
        /
        denominator
        *
        math.sqrt(
            (
                p_hat
                * (
                    1.0
                    - p_hat
                )
                /
                trials
            )
            +
            (
                z**2
                /
                (
                    4.0
                    * trials**2
                )
            )
        )
    )


    return (
        max(
            0.0,
            center - half_width
        ),

        min(
            1.0,
            center + half_width
        )
    )


# =============================================================================
# Aggregate
# =============================================================================

phase3g_summary_rows = []


for (
    total_distance_km,
    policy_name
), group in (
    phase3g_valid_results
    .groupby(
        [
            "TotalDistanceKm",
            "Policy"
        ],
        sort=True
    )
):


    group = group.copy()


    successful = (
        group[
            group[
                "Success"
            ]
            == True
        ]
    )


    n_trials = len(
        group
    )


    n_success = len(
        successful
    )


    success_rate = (
        n_success
        /
        n_trials
    )


    success_ci_low, success_ci_high = (
        phase3g_wilson_ci95(
            n_success,
            n_trials
        )
    )


    restricted_mean, restricted_low, restricted_high = (
        phase3g_mean_ci95(
            group[
                "RestrictedCompletionSlots"
            ]
        )
    )


    completion_mean, completion_low, completion_high = (
        phase3g_mean_ci95(
            successful[
                "ProcessedSlots"
            ]
        )
    )


    modeled_time_mean, modeled_time_low, modeled_time_high = (
        phase3g_mean_ci95(
            successful[
                "ModeledPhysicalTimeS"
            ]
        )
    )


    fidelity_mean, fidelity_low, fidelity_high = (
        phase3g_mean_ci95(
            successful[
                "FinalFidelity"
            ]
        )
    )


    elementary_attempt_mean, elementary_attempt_low, elementary_attempt_high = (
        phase3g_mean_ci95(
            group[
                "ElementaryAttempts"
            ]
        )
    )


    bsm_attempt_mean, bsm_attempt_low, bsm_attempt_high = (
        phase3g_mean_ci95(
            group[
                "BSMAttempts"
            ]
        )
    )


    bsm_failure_mean, _, _ = (
        phase3g_mean_ci95(
            group[
                "BSMFailures"
            ]
        )
    )


    expired_mean, expired_low, expired_high = (
        phase3g_mean_ci95(
            group[
                "ExpiredResources"
            ]
        )
    )


    tree_depth_mean, tree_depth_low, tree_depth_high = (
        phase3g_mean_ci95(
            successful[
                "FinalTreeDepth"
            ]
        )
    )


    wall_mean, _, _ = (
        phase3g_mean_ci95(
            group[
                "WallClockSeconds"
            ]
        )
    )


    phase3g_summary_rows.append({

        "TotalDistanceKm":
            total_distance_km,

        "Policy":
            policy_name,

        "Trials":
            n_trials,

        "Successes":
            n_success,

        "Timeouts":
            n_trials - n_success,

        "SuccessRate":
            success_rate,

        "SuccessCI95Low":
            success_ci_low,

        "SuccessCI95High":
            success_ci_high,

        # -----------------------------------------------------
        # All trials, including timeout penalty.
        # -----------------------------------------------------

        "RestrictedMeanCompletionSlots":
            restricted_mean,

        "RestrictedCompletionCI95Low":
            restricted_low,

        "RestrictedCompletionCI95High":
            restricted_high,

        # -----------------------------------------------------
        # Successful trials only.
        # -----------------------------------------------------

        "MeanCompletionSlotsSuccessOnly":
            completion_mean,

        "CompletionCI95Low":
            completion_low,

        "CompletionCI95High":
            completion_high,

        "MeanModeledTimeSSuccessOnly":
            modeled_time_mean,

        "ModeledTimeCI95Low":
            modeled_time_low,

        "ModeledTimeCI95High":
            modeled_time_high,

        "MeanFinalFidelitySuccessOnly":
            fidelity_mean,

        "FidelityCI95Low":
            fidelity_low,

        "FidelityCI95High":
            fidelity_high,

        # -----------------------------------------------------
        # Resource costs over ALL trials.
        # -----------------------------------------------------

        "MeanElementaryAttemptsPerTrial":
            elementary_attempt_mean,

        "ElementaryAttemptsCI95Low":
            elementary_attempt_low,

        "ElementaryAttemptsCI95High":
            elementary_attempt_high,

        "MeanBSMAttemptsPerTrial":
            bsm_attempt_mean,

        "BSMAttemptsCI95Low":
            bsm_attempt_low,

        "BSMAttemptsCI95High":
            bsm_attempt_high,

        "MeanBSMFailuresPerTrial":
            bsm_failure_mean,

        "MeanExpiredResourcesPerTrial":
            expired_mean,

        "ExpiredResourcesCI95Low":
            expired_low,

        "ExpiredResourcesCI95High":
            expired_high,

        "MeanFinalTreeDepthSuccessOnly":
            tree_depth_mean,

        "TreeDepthCI95Low":
            tree_depth_low,

        "TreeDepthCI95High":
            tree_depth_high,

        "MeanWallClockSecondsPerTrial":
            wall_mean
    })


phase3g_summary = pd.DataFrame(
    phase3g_summary_rows
)


phase3g_atomic_write_csv(
    phase3g_summary,
    PHASE3G_SUMMARY_PATH
)


display(
    phase3g_summary
)


print(
    "Summary saved to:",
    PHASE3G_SUMMARY_PATH
)

In [ ]:
# @title Monte Carlo Evaluation Graphs

# phase3g_summary = pd.read_csv(
#     PHASE3G_SUMMARY_PATH
# )


# PHASE3G_POLICY_DISPLAY_ORDER = (
#     "SEQUENTIAL_LEFT_TO_RIGHT",
#     "STATIC_BALANCED_TREE",
#     "MAX_PARALLEL_OLDEST_FIRST"
# )


# # =============================================================================
# # Generic Confidence-Interval Plot
# # =============================================================================

# def phase3g_plot_metric(

#     mean_column,

#     low_column,

#     high_column,

#     ylabel,

#     title,

#     filename,

#     y_min=None,

#     y_max=None,

#     log_y=False
# ):

#     fig, ax = plt.subplots(
#         figsize=(
#             10,
#             6
#         )
#     )


#     for policy_name in (
#         PHASE3G_POLICY_DISPLAY_ORDER
#     ):

#         subset = (
#             phase3g_summary[
#                 phase3g_summary[
#                     "Policy"
#                 ]
#                 == policy_name
#             ]
#             .sort_values(
#                 "TotalDistanceKm"
#             )
#         )


#         x = (
#             subset[
#                 "TotalDistanceKm"
#             ]
#             .to_numpy(
#                 dtype=float
#             )
#         )


#         y = (
#             subset[
#                 mean_column
#             ]
#             .to_numpy(
#                 dtype=float
#             )
#         )


#         lower = (
#             subset[
#                 low_column
#             ]
#             .to_numpy(
#                 dtype=float
#             )
#         )


#         upper = (
#             subset[
#                 high_column
#             ]
#             .to_numpy(
#                 dtype=float
#             )
#         )


#         lower_error = (
#             y
#             - lower
#         )


#         upper_error = (
#             upper
#             - y
#         )


#         ax.errorbar(

#             x,

#             y,

#             yerr=np.vstack(
#                 [
#                     lower_error,
#                     upper_error
#                 ]
#             ),

#             marker="o",

#             capsize=4,

#             label=(
#                 policy_name
#             )
#         )


#     ax.set_xlabel(
#         "Total user separation (km)"
#     )

#     ax.set_ylabel(
#         ylabel
#     )

#     ax.set_title(
#         title
#     )

#     ax.grid(
#         True,
#         alpha=0.3
#     )

#     ax.legend()


#     if (
#         y_min is not None
#         or y_max is not None
#     ):

#         current_low, current_high = (
#             ax.get_ylim()
#         )


#         ax.set_ylim(

#             y_min
#             if y_min is not None
#             else current_low,

#             y_max
#             if y_max is not None
#             else current_high
#         )


#     if log_y:

#         ax.set_yscale(
#             "log"
#         )


#     fig.tight_layout()


#     output_path = (
#         PHASE3G_PLOT_DIR
#         / filename
#     )


#     fig.savefig(
#         output_path,
#         dpi=200,
#         bbox_inches="tight"
#     )


#     plt.show()


#     print(
#         "Saved:",
#         output_path
#     )


# # =============================================================================
# # GRAPH 1 — Success Probability
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "SuccessRate"
#     ),

#     low_column=(
#         "SuccessCI95Low"
#     ),

#     high_column=(
#         "SuccessCI95High"
#     ),

#     ylabel=(
#         "End-to-end success probability"
#     ),

#     title=(
#         "NARAD-QC policy success probability vs user separation"
#     ),

#     filename=(
#         "01_success_probability.png"
#     ),

#     y_min=0.0,

#     y_max=1.0
# )


# # =============================================================================
# # GRAPH 2 — Restricted Completion Slots
# #
# # Includes timeout penalty and is therefore especially important
# # when success probability drops.
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "RestrictedMeanCompletionSlots"
#     ),

#     low_column=(
#         "RestrictedCompletionCI95Low"
#     ),

#     high_column=(
#         "RestrictedCompletionCI95High"
#     ),

#     ylabel=(
#         "Restricted mean completion slots"
#     ),

#     title=(
#         "NARAD-QC restricted completion cost vs user separation"
#     ),

#     filename=(
#         "02_restricted_completion_slots.png"
#     )
# )


# # =============================================================================
# # GRAPH 3 — Completion Slots, Successful Runs Only
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "MeanCompletionSlotsSuccessOnly"
#     ),

#     low_column=(
#         "CompletionCI95Low"
#     ),

#     high_column=(
#         "CompletionCI95High"
#     ),

#     ylabel=(
#         "Mean completion slots (successful trials)"
#     ),

#     title=(
#         "NARAD-QC successful-session completion latency"
#     ),

#     filename=(
#         "03_successful_completion_slots.png"
#     )
# )


# # =============================================================================
# # GRAPH 4 — Final Fidelity
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "MeanFinalFidelitySuccessOnly"
#     ),

#     low_column=(
#         "FidelityCI95Low"
#     ),

#     high_column=(
#         "FidelityCI95High"
#     ),

#     ylabel=(
#         "Mean final end-to-end fidelity"
#     ),

#     title=(
#         "NARAD-QC final entanglement fidelity vs user separation"
#     ),

#     filename=(
#         "04_final_fidelity.png"
#     ),

#     y_min=0.25,

#     y_max=1.0
# )


# # =============================================================================
# # GRAPH 5 — Elementary Generation Cost
# #
# # Log scale is useful because long-distance attempts can increase
# # by orders of magnitude.
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "MeanElementaryAttemptsPerTrial"
#     ),

#     low_column=(
#         "ElementaryAttemptsCI95Low"
#     ),

#     high_column=(
#         "ElementaryAttemptsCI95High"
#     ),

#     ylabel=(
#         "Mean elementary generation attempts / trial"
#     ),

#     title=(
#         "NARAD-QC elementary-entanglement resource cost"
#     ),

#     filename=(
#         "05_elementary_attempts.png"
#     ),

#     log_y=True
# )


# # =============================================================================
# # GRAPH 6 — Atomic BSM Cost
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "MeanBSMAttemptsPerTrial"
#     ),

#     low_column=(
#         "BSMAttemptsCI95Low"
#     ),

#     high_column=(
#         "BSMAttemptsCI95High"
#     ),

#     ylabel=(
#         "Mean atomic BSM attempts / trial"
#     ),

#     title=(
#         "NARAD-QC entanglement-swapping resource cost"
#     ),

#     filename=(
#         "06_bsm_attempts.png"
#     ),

#     log_y=True
# )


# # =============================================================================
# # GRAPH 7 — Memory Expiration Burden
# # =============================================================================

# phase3g_plot_metric(

#     mean_column=(
#         "MeanExpiredResourcesPerTrial"
#     ),

#     low_column=(
#         "ExpiredResourcesCI95Low"
#     ),

#     high_column=(
#         "ExpiredResourcesCI95High"
#     ),

#     ylabel=(
#         "Mean expired entanglement resources / trial"
#     ),

#     title=(
#         "NARAD-QC quantum-memory expiration burden"
#     ),

#     filename=(
#         "07_memory_expirations.png"
#     )
# )

In [ ]:
# @title  Policy Performance Map


phase3g_summary = pd.read_csv(
    PHASE3G_SUMMARY_PATH
)


phase3g_winner_rows = []


for distance_km in (
    PHASE3G_DISTANCES_KM
):

    subset = (
        phase3g_summary[
            phase3g_summary[
                "TotalDistanceKm"
            ]
            == distance_km
        ]
        .copy()
    )


    # Reliability:
    reliability_winner = (
        subset.loc[
            subset[
                "SuccessRate"
            ].idxmax()
        ]
    )


    # Censor-aware completion:
    latency_winner = (
        subset.loc[
            subset[
                "RestrictedMeanCompletionSlots"
            ].idxmin()
        ]
    )


    # Fidelity:
    fidelity_subset = (
        subset.dropna(
            subset=[
                "MeanFinalFidelitySuccessOnly"
            ]
        )
    )


    if len(
        fidelity_subset
    ) > 0:

        fidelity_winner = (
            fidelity_subset.loc[
                fidelity_subset[
                    "MeanFinalFidelitySuccessOnly"
                ].idxmax()
            ]
        )

        fidelity_policy = (
            fidelity_winner[
                "Policy"
            ]
        )

    else:

        fidelity_policy = None


    # Elementary-resource efficiency:
    generation_winner = (
        subset.loc[
            subset[
                "MeanElementaryAttemptsPerTrial"
            ].idxmin()
        ]
    )


    phase3g_winner_rows.append({

        "DistanceKm":
            distance_km,

        "Highest Success Rate":
            reliability_winner[
                "Policy"
            ],

        "Lowest Restricted Completion Cost":
            latency_winner[
                "Policy"
            ],

        "Highest Final Fidelity":
            fidelity_policy,

        "Lowest Elementary Attempt Cost":
            generation_winner[
                "Policy"
            ]
    })


phase3g_policy_map = pd.DataFrame(
    phase3g_winner_rows
)


display(
    phase3g_policy_map
)

### Phase 3 MC Repeater Count Variance : 19-09-2026T10:02:19 -- 21-09-2026T19:35:21

In [ ]:
# @title  Fixed-600-km Repeater Density Sweep Configuration

from pathlib import Path
from collections import Counter

import os
import gc
import json
import math
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm


# =============================================================================
# FIXED APPLICATION DISTANCE
# =============================================================================

PHASE3G2_TOTAL_DISTANCE_KM = 600.0


# =============================================================================
# INDEPENDENT VARIABLE
# =============================================================================

PHASE3G2_REPEATER_COUNTS = (
    3,
    4,
    5,
    6,7,8,9
)


# =============================================================================
# SAME THREE CONTROL POLICIES
# =============================================================================

PHASE3G2_POLICY_NAMES = (
    "SEQUENTIAL_LEFT_TO_RIGHT",
    "STATIC_BALANCED_TREE",
    "MAX_PARALLEL_OLDEST_FIRST"
)


PHASE3G2_TRIALS_PER_CONFIGURATION = 100


# =============================================================================
# KEEP PHYSICS IDENTICAL TO PHASE 3G.1
# =============================================================================

PHASE3G2_ATOMIC_BSM_SUCCESS_PROBABILITY = 0.40

PHASE3G2_MAX_STORAGE_SLOTS = 100

PHASE3G2_SLOT_DURATION_S = 0.05

PHASE3G2_ELEMENTARY_FIDELITY = 0.995

PHASE3G2_FIBER_ATTENUATION_DB_PER_KM = 0.20

PHASE3G2_MAX_EXECUTION_SLOTS = 20_000


PHASE3G2_BASE_SEED = 3_720_000

PHASE3G2_STOP_ON_ERROR = True


PHASE3G2_RUN_TAG = (
    "phase3_density_"
    "600km_"
    "R3-R6_"
    "100trials_"
    "pb040_"
    "mem100_"
    "max20000"
)


# =============================================================================
# CHECKPOINT LOCATION
# =============================================================================

drive_root = Path(
    "/content/drive/MyDrive"
)


if drive_root.exists():

    PHASE3G2_CHECKPOINT_DIR = (
        drive_root
        / "NARAD_QC"
        / "Phase3"
        / PHASE3G2_RUN_TAG
    )

    print(
        "[INFO] Durable Google Drive checkpointing enabled."
    )

else:

    PHASE3G2_CHECKPOINT_DIR = (
        Path("/content")
        / "NARAD_QC"
        / "Phase3G2"
        / PHASE3G2_RUN_TAG
    )

    print(
        "[WARNING] Google Drive not mounted; checkpoints "
        "will not survive full runtime deletion."
    )


PHASE3G2_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PHASE3G2_RESULTS_PATH = (
    PHASE3G2_CHECKPOINT_DIR
    / "phase3g2_trial_results.csv"
)


PHASE3G2_SUMMARY_PATH = (
    PHASE3G2_CHECKPOINT_DIR
    / "phase3g2_summary.csv"
)


PHASE3G2_CONFIG_PATH = (
    PHASE3G2_CHECKPOINT_DIR
    / "phase3g2_config.json"
)


PHASE3G2_PLOT_DIR = (
    PHASE3G2_CHECKPOINT_DIR
    / "plots"
)


PHASE3G2_PLOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# CONFIGURATION SNAPSHOT
# =============================================================================

phase3g2_config = {

    "total_distance_km":
        PHASE3G2_TOTAL_DISTANCE_KM,

    "repeater_counts":
        list(
            PHASE3G2_REPEATER_COUNTS
        ),

    "policies":
        list(
            PHASE3G2_POLICY_NAMES
        ),

    "trials_per_configuration":
        PHASE3G2_TRIALS_PER_CONFIGURATION,

    "atomic_bsm_success_probability":
        PHASE3G2_ATOMIC_BSM_SUCCESS_PROBABILITY,

    "max_storage_slots":
        PHASE3G2_MAX_STORAGE_SLOTS,

    "slot_duration_s":
        PHASE3G2_SLOT_DURATION_S,

    "elementary_fidelity":
        PHASE3G2_ELEMENTARY_FIDELITY,

    "fiber_attenuation_db_per_km":
        PHASE3G2_FIBER_ATTENUATION_DB_PER_KM,

    "max_execution_slots":
        PHASE3G2_MAX_EXECUTION_SLOTS,

    "base_seed":
        PHASE3G2_BASE_SEED
}


if PHASE3G2_CONFIG_PATH.exists():

    with open(
        PHASE3G2_CONFIG_PATH,
        "r"
    ) as file:

        previous_config = json.load(
            file
        )


    if previous_config != phase3g2_config:

        raise RuntimeError(
            "Existing Phase-3G.2 checkpoint uses a different "
            "configuration. Change PHASE3G2_RUN_TAG."
        )

else:

    with open(
        PHASE3G2_CONFIG_PATH,
        "w"
    ) as file:

        json.dump(
            phase3g2_config,
            file,
            indent=2
        )


# =============================================================================
# REUSE / DEFINE ATOMIC CSV CHECKPOINT
# =============================================================================

def phase3g2_atomic_write_csv(
    dataframe,
    path
):

    temp_path = path.with_name(
        path.name
        + ".tmp"
    )

    dataframe.to_csv(
        temp_path,
        index=False
    )

    os.replace(
        temp_path,
        path
    )


# =============================================================================
# RESUME
# =============================================================================

if PHASE3G2_RESULTS_PATH.exists():

    phase3g2_results = pd.read_csv(
        PHASE3G2_RESULTS_PATH
    )

else:

    phase3g2_results = pd.DataFrame()


PHASE3G2_EXPECTED_TRIALS = (
    len(
        PHASE3G2_REPEATER_COUNTS
    )
    *
    len(
        PHASE3G2_POLICY_NAMES
    )
    *
    PHASE3G2_TRIALS_PER_CONFIGURATION
)


print(
    "Fixed user separation:",
    PHASE3G2_TOTAL_DISTANCE_KM,
    "km"
)

print(
    "Repeater counts:",
    PHASE3G2_REPEATER_COUNTS
)

print(
    "Total Monte Carlo sessions:",
    PHASE3G2_EXPECTED_TRIALS
)

In [ ]:
# @title  Repeater Density Scenario Inspection


phase3g2_scenario_rows = []


for repeater_count in (
    PHASE3G2_REPEATER_COUNTS
):

    elementary_link_count = (
        repeater_count
        + 1
    )


    hop_distance_km = (
        PHASE3G2_TOTAL_DISTANCE_KM
        /
        elementary_link_count
    )


    elementary_success_probability = (
        10.0
        **
        (
            -(
                PHASE3G2_FIBER_ATTENUATION_DB_PER_KM
                *
                hop_distance_km
            )
            /
            10.0
        )
    )


    phase3g2_scenario_rows.append({

        "Repeaters":
            repeater_count,

        "Elementary Links":
            elementary_link_count,

        "Hop Distance (km)":
            hop_distance_km,

        "Elementary Success Proxy":
            elementary_success_probability,

        "Expected Attempts / Elementary Success":
            (
                1.0
                /
                elementary_success_probability
            ),

        "Successful BSMs Required in Final Tree":
            repeater_count
    })


phase3g2_scenarios = pd.DataFrame(
    phase3g2_scenario_rows
)


display(
    phase3g2_scenarios
)

In [ ]:
# @title Variable-Repeater Policy Factory


def phase3g2_make_policy(
    policy_name,
    elementary_link_count
):

    if (
        policy_name
        == "SEQUENTIAL_LEFT_TO_RIGHT"
    ):

        return (
            SequentialLeftToRightSwapPolicy()
        )


    if (
        policy_name
        == "STATIC_BALANCED_TREE"
    ):

        return (
            StaticBalancedTreeSwapPolicy(

                elementary_link_count=(
                    elementary_link_count
                )
            )
        )


    if (
        policy_name
        == "MAX_PARALLEL_OLDEST_FIRST"
    ):

        return (
            MaxParallelOldestFirstSwapPolicy()
        )


    raise ValueError(
        f"Unknown policy: {policy_name}"
    )

In [ ]:
# @title Variable-Density 600-km NARAD-QC Stack


def build_phase3g2_trial_stack(
    policy_name,
    repeater_count,
    simulation_seed
):

    # =========================================================================
    # Validate
    # =========================================================================

    if (
        isinstance(
            repeater_count,
            bool
        )
        or not isinstance(
            repeater_count,
            int
        )
        or repeater_count < 1
    ):

        raise ValueError(
            "repeater_count must be a positive integer."
        )


    elementary_link_count = (
        repeater_count
        + 1
    )


    hop_distance_km = (
        PHASE3G2_TOTAL_DISTANCE_KM
        /
        elementary_link_count
    )


    # =========================================================================
    # Policy
    # =========================================================================

    policy = (
        phase3g2_make_policy(

            policy_name=(
                policy_name
            ),

            elementary_link_count=(
                elementary_link_count
            )
        )
    )


    # =========================================================================
    # Physical topology
    # =========================================================================

    topology = NetworkTopology(

        name=(
            f"Phase3G2-600km-"
            f"R{repeater_count}-"
            f"{policy_name}"
        )
    )


    user_a = NetworkNode(

        name="Density-A",

        x_km=0.0,

        y_km=0.0,

        endpoint_capable=True,

        swapping_capable=False,

        memory_capacity=2
    )


    repeaters = tuple(

        NetworkNode(

            name=f"Density-R{i}",

            x_km=(
                hop_distance_km
                * i
            ),

            y_km=0.0,

            endpoint_capable=False,

            swapping_capable=True,

            memory_capacity=2
        )

        for i in range(
            1,
            repeater_count + 1
        )
    )


    user_b = NetworkNode(

        name="Density-B",

        x_km=(
            PHASE3G2_TOTAL_DISTANCE_KM
        ),

        y_km=0.0,

        endpoint_capable=True,

        swapping_capable=False,

        memory_capacity=2
    )


    path = (
        user_a,
        *repeaters,
        user_b
    )


    for node in path:

        topology.add_node(
            node
        )


    for left, right in zip(
        path[:-1],
        path[1:]
    ):

        topology.add_link(

            PhysicalQuantumLink(

                node_a=left,

                node_b=right,

                attenuation_db_per_km=(
                    PHASE3G2_FIBER_ATTENUATION_DB_PER_KM
                )
            )
        )


    assert (
        topology.node_count
        ==
        repeater_count + 2
    )


    assert (
        topology.link_count
        ==
        elementary_link_count
    )


    # =========================================================================
    # Simulator
    # =========================================================================

    simulator = (
        QuantumNetworkSimulator(

            topology=topology,

            simulation_seed=(
                simulation_seed
            ),

            controller_id=(
                f"PHASE3G2-"
                f"R{repeater_count}-"
                f"{policy_name}"
            ),

            parameter_set=(
                MLDSAParameterSet.ML_DSA_65
            ),

            max_storage_slots=(
                PHASE3G2_MAX_STORAGE_SLOTS
            )
        )
    )


    authentication = (
        simulator
        .authenticate_all_nodes()
    )


    assert all(
        record.status
        == AuthenticationStatus.AUTHENTICATED

        for record
        in authentication.values()
    )


    assert (
        simulator
        .require_quantum_eligibility(
            *path
        )
    )


    # =========================================================================
    # Elementary generation
    # =========================================================================

    generation_service = (
        ElementaryEntanglementService(

            simulator=simulator,

            model=(
                FiberSurvivalProxyModel()
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=simulator,

            service=generation_service,

            ordered_path=path,

            demand_active=False
        )
    )


    # =========================================================================
    # Fidelity
    # =========================================================================

    fidelity_profile = (
        WernerPhysicalFidelityProfile(

            elementary_fidelity=(
                PHASE3G2_ELEMENTARY_FIDELITY
            ),

            bell_fidelity_floor=0.25,

            memory_reference_initial_fidelity=1.0,

            memory_reference_fidelity=0.90,

            memory_reference_time_s=1.0,

            slot_duration_s=(
                PHASE3G2_SLOT_DURATION_S
            ),

            swap_operation_visibility=1.0
        )
    )


    fidelity_registry = (
        PhysicalEntanglementFidelityRegistry(

            simulator=simulator,

            profile=fidelity_profile
        )
    )


    # =========================================================================
    # Stochastic physical BSM
    # =========================================================================

    swap_model = (
        WernerStateAtomicBSMModel(

            fidelity_profile=(
                fidelity_profile
            ),

            atomic_bsm_success_probability=(
                PHASE3G2_ATOMIC_BSM_SUCCESS_PROBABILITY
            ),

            atomic_bsm_latency_us=10.0
        )
    )


    swap_engine = (
        EntanglementSwapEngine(

            simulator=simulator,

            model=swap_model,

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    # =========================================================================
    # Policy-controlled scheduler
    # =========================================================================

    swap_scheduler = (
        ControlPlaneParallelSwapScheduler(

            simulator=simulator,

            swap_engine=swap_engine,

            ordered_path=path,

            policy=policy
        )
    )


    # =========================================================================
    # Failure-aware demand
    # =========================================================================

    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=simulator,

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=path
        )
    )


    # =========================================================================
    # Autonomous execution
    # =========================================================================

    orchestrator = (
        FailureAwareEndToEndEntanglementOrchestrator(

            simulator=simulator,

            ordered_path=path,

            generation_scheduler=(
                generation_scheduler
            ),

            demand_controller=(
                demand_controller
            ),

            swap_scheduler=(
                swap_scheduler
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    physical_links = (
        topology.path_links(
            path
        )
    )


    elementary_probability = (
        physical_links[0]
        .survival_probability
    )


    return {

        "policy":
            policy,

        "topology":
            topology,

        "simulator":
            simulator,

        "path":
            path,

        "repeaters":
            repeaters,

        "generation_service":
            generation_service,

        "generation_scheduler":
            generation_scheduler,

        "fidelity_profile":
            fidelity_profile,

        "fidelity_registry":
            fidelity_registry,

        "swap_model":
            swap_model,

        "swap_engine":
            swap_engine,

        "swap_scheduler":
            swap_scheduler,

        "demand_controller":
            demand_controller,

        "orchestrator":
            orchestrator,

        "repeater_count":
            repeater_count,

        "elementary_link_count":
            elementary_link_count,

        "hop_distance_km":
            hop_distance_km,

        "elementary_success_probability":
            elementary_probability
    }

In [ ]:
# @title  Single Repeater-Density Monte Carlo Trial


def run_phase3g2_single_trial(
    policy_name,
    repeater_count,
    trial_number,
    simulation_seed
):

    wall_start = (
        time.perf_counter()
    )


    stack = (
        build_phase3g2_trial_stack(

            policy_name=(
                policy_name
            ),

            repeater_count=(
                repeater_count
            ),

            simulation_seed=(
                simulation_seed
            )
        )
    )


    simulator = (
        stack[
            "simulator"
        ]
    )


    result = (
        stack[
            "orchestrator"
        ]
        .run_until_end_to_end(

            max_slots=(
                PHASE3G2_MAX_EXECUTION_SLOTS
            )
        )
    )


    trial_status = (
        "SUCCESS"
        if result.success
        else "TIMEOUT"
    )


    # =========================================================================
    # Resource history
    # =========================================================================

    all_resources = tuple(
        simulator
        .entanglement_manager
        .resources
    )


    status_counts = Counter(
        resource.status.value

        for resource
        in all_resources
    )


    # =========================================================================
    # Swap rounds
    # =========================================================================

    swap_rounds = tuple(

        round_record

        for round_record in (
            stack[
                "swap_scheduler"
            ]
            .round_history
        )

        if (
            round_record.selected_count
            > 0
        )
    )


    swap_round_count = len(
        swap_rounds
    )


    mean_parallel_bsms = (

        result.total_bsm_attempts
        /
        swap_round_count

        if swap_round_count > 0

        else 0.0
    )


    # =========================================================================
    # Selected age statistics
    # =========================================================================

    selected_ages = [

        candidate.oldest_input_half_age_slots

        for round_record
        in swap_rounds

        for candidate
        in round_record.selected_candidates
    ]


    mean_selected_age = (
        float(
            np.mean(
                selected_ages
            )
        )

        if selected_ages

        else np.nan
    )


    max_selected_age = (
        int(
            np.max(
                selected_ages
            )
        )

        if selected_ages

        else np.nan
    )


    # =========================================================================
    # Failure-aware workload
    # =========================================================================

    regeneration_workload = sum(

        len(
            step
            .demand_after_swapping
            .regeneration_link_ids
        )

        for step
        in result.execution_history
    )


    # =========================================================================
    # Final tree
    # =========================================================================

    final_tree_depth = np.nan

    final_fidelity = np.nan

    final_tree_leaves = np.nan

    final_tree_swaps = np.nan


    if result.success:

        final_resource = (
            simulator
            .entanglement_manager
            .get_resource(
                result.end_to_end_resource_id
            )
        )


        final_tree_depth = (
            entanglement_resource_tree_depth(

                simulator
                .entanglement_manager,

                final_resource
            )
        )


        tree_size = (
            entanglement_resource_tree_size(

                simulator
                .entanglement_manager,

                final_resource
            )
        )


        final_tree_leaves = (
            tree_size[
                "elementary_leaves"
            ]
        )


        final_tree_swaps = (
            tree_size[
                "successful_swaps_in_final_tree"
            ]
        )


        final_fidelity = (
            result.final_fidelity
        )


        assert (
            final_tree_leaves
            ==
            stack[
                "elementary_link_count"
            ]
        )


        assert (
            final_tree_swaps
            ==
            repeater_count
        )


    # =========================================================================
    # Consistency
    # =========================================================================

    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    assert (
        stack[
            "generation_service"
        ]
        .statistics
        .total_attempts
        ==
        result.total_elementary_attempts
    )


    assert (
        len(
            stack[
                "swap_engine"
            ]
            .attempts
        )
        ==
        result.total_bsm_attempts
    )


    # =========================================================================
    # Timing
    # =========================================================================

    modeled_time_s = (
        result.simulation_slots_elapsed

        *
        PHASE3G2_SLOT_DURATION_S
    )


    wall_seconds = (
        time.perf_counter()
        - wall_start
    )


    trial_key = (
        f"R{repeater_count}|"
        f"{policy_name}|"
        f"{trial_number}"
    )


    return {

        "TrialKey":
            trial_key,

        "TrialStatus":
            trial_status,

        "Policy":
            policy_name,

        "TotalDistanceKm":
            PHASE3G2_TOTAL_DISTANCE_KM,

        "RepeaterCount":
            repeater_count,

        "ElementaryLinks":
            stack[
                "elementary_link_count"
            ],

        "HopDistanceKm":
            stack[
                "hop_distance_km"
            ],

        "ElementarySuccessProbability":
            stack[
                "elementary_success_probability"
            ],

        "Trial":
            trial_number,

        "SimulationSeed":
            simulation_seed,

        "Success":
            result.success,

        "ProcessedSlots":
            result.processed_slot_count,

        "RestrictedCompletionSlots":
            (
                result.processed_slot_count

                if result.success

                else PHASE3G2_MAX_EXECUTION_SLOTS
            ),

        "ModeledPhysicalTimeS":
            modeled_time_s,

        "ElementaryAttempts":
            result.total_elementary_attempts,

        "ElementarySuccesses":
            result.total_elementary_successes,

        "ElementaryFailures":
            result.total_elementary_failures,

        "BSMAttempts":
            result.total_bsm_attempts,

        "BSMSuccesses":
            result.total_bsm_successes,

        "BSMFailures":
            result.total_bsm_failures,

        "SwapRounds":
            swap_round_count,

        "MeanParallelBSMsPerRound":
            mean_parallel_bsms,

        "RegenerationDemandWorkload":
            regeneration_workload,

        "MeanSelectedOldestInputAge":
            mean_selected_age,

        "MaxSelectedOldestInputAge":
            max_selected_age,

        "HistoricalResources":
            len(
                all_resources
            ),

        "ExpiredResources":
            status_counts.get(
                EntanglementStatus.EXPIRED.value,
                0
            ),

        "DiscardedResources":
            status_counts.get(
                EntanglementStatus.DISCARDED.value,
                0
            ),

        "FinalTreeDepth":
            final_tree_depth,

        "FinalTreeLeaves":
            final_tree_leaves,

        "FinalTreeSwaps":
            final_tree_swaps,

        "FinalFidelity":
            final_fidelity,

        "WallClockSeconds":
            wall_seconds
    }

In [ ]:
# @title Full Repeater-Density Monte Carlo Runner


def phase3g2_completed_keys(
    dataframe
):

    if dataframe.empty:

        return set()


    valid = dataframe[
        dataframe[
            "TrialStatus"
        ].isin(
            [
                "SUCCESS",
                "TIMEOUT"
            ]
        )
    ]


    return set(
        valid[
            "TrialKey"
        ].astype(
            str
        )
    )


def phase3g2_seed_for_trial(
    repeater_index,
    trial_number
):

    return int(
        PHASE3G2_BASE_SEED
        +
        repeater_index
        * 100_000
        +
        trial_number
    )


def phase3g2_checkpoint_trial(
    current_results,
    trial_row
):

    trial_key = (
        trial_row[
            "TrialKey"
        ]
    )


    if (
        not current_results.empty
        and "TrialKey"
        in current_results.columns
    ):

        current_results = (
            current_results[
                current_results[
                    "TrialKey"
                ]
                != trial_key
            ]
            .copy()
        )


    current_results = pd.concat(
        [
            current_results,
            pd.DataFrame(
                [
                    trial_row
                ]
            )
        ],
        ignore_index=True
    )


    current_results = (
        current_results
        .sort_values(
            [
                "RepeaterCount",
                "Trial",
                "Policy"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    phase3g2_atomic_write_csv(
        current_results,
        PHASE3G2_RESULTS_PATH
    )


    return current_results


completed_keys = (
    phase3g2_completed_keys(
        phase3g2_results
    )
)


progress = tqdm(

    total=(
        PHASE3G2_EXPECTED_TRIALS
    ),

    initial=len(
        completed_keys
    ),

    desc=(
        "NARAD-QC Phase 3G.2 Density Sweep"
    )
)


session_start = (
    time.perf_counter()
)

new_trials = 0


for (
    repeater_index,
    repeater_count
) in enumerate(
    PHASE3G2_REPEATER_COUNTS
):


    for trial_number in range(
        1,
        PHASE3G2_TRIALS_PER_CONFIGURATION + 1
    ):


        simulation_seed = (
            phase3g2_seed_for_trial(

                repeater_index,
                trial_number
            )
        )


        for policy_name in (
            PHASE3G2_POLICY_NAMES
        ):


            trial_key = (
                f"R{repeater_count}|"
                f"{policy_name}|"
                f"{trial_number}"
            )


            if trial_key in completed_keys:

                continue


            try:

                trial_row = (
                    run_phase3g2_single_trial(

                        policy_name=(
                            policy_name
                        ),

                        repeater_count=(
                            repeater_count
                        ),

                        trial_number=(
                            trial_number
                        ),

                        simulation_seed=(
                            simulation_seed
                        )
                    )
                )


                phase3g2_results = (
                    phase3g2_checkpoint_trial(

                        phase3g2_results,

                        trial_row
                    )
                )


                completed_keys.add(
                    trial_key
                )


                new_trials += 1

                progress.update(
                    1
                )


                elapsed = (
                    time.perf_counter()
                    -
                    session_start
                )


                average_trial_wall = (
                    elapsed
                    /
                    new_trials
                )


                remaining = (
                    PHASE3G2_EXPECTED_TRIALS
                    -
                    len(
                        completed_keys
                    )
                )


                eta_hours = (
                    average_trial_wall
                    *
                    remaining
                    /
                    3600.0
                )


                progress.set_postfix({

                    "R":
                        repeater_count,

                    "hop_km":
                        round(
                            PHASE3G2_TOTAL_DISTANCE_KM
                            /
                            (
                                repeater_count
                                + 1
                            ),
                            1
                        ),

                    "trial":
                        trial_number,

                    "policy":
                        policy_name,

                    "status":
                        trial_row[
                            "TrialStatus"
                        ],

                    "eta_h":
                        round(
                            eta_hours,
                            2
                        )
                })


            except Exception as exc:

                error_row = {

                    "TrialKey":
                        trial_key,

                    "TrialStatus":
                        "ERROR",

                    "Policy":
                        policy_name,

                    "TotalDistanceKm":
                        PHASE3G2_TOTAL_DISTANCE_KM,

                    "RepeaterCount":
                        repeater_count,

                    "Trial":
                        trial_number,

                    "SimulationSeed":
                        simulation_seed,

                    "ErrorType":
                        type(
                            exc
                        ).__name__,

                    "ErrorMessage":
                        str(
                            exc
                        ),

                    "Traceback":
                        traceback.format_exc()
                }


                phase3g2_results = (
                    phase3g2_checkpoint_trial(

                        phase3g2_results,

                        error_row
                    )
                )


                progress.close()


                print(
                    "\n[ERROR] Trial checkpointed:"
                )

                print(
                    error_row
                )


                if PHASE3G2_STOP_ON_ERROR:

                    raise


            finally:

                gc.collect()


progress.close()


print(
    "\nPhase 3G.2 Monte Carlo complete."
)

print(
    "Checkpoint:",
    PHASE3G2_RESULTS_PATH
)

In [ ]:
# @title Density Sweep Integrity Verification


phase3g2_results = pd.read_csv(
    PHASE3G2_RESULTS_PATH
)


phase3g2_valid = (
    phase3g2_results[
        phase3g2_results[
            "TrialStatus"
        ].isin(
            [
                "SUCCESS",
                "TIMEOUT"
            ]
        )
    ]
    .copy()
)


assert (
    len(
        phase3g2_valid
    )
    ==
    PHASE3G2_EXPECTED_TRIALS
)


assert (
    phase3g2_valid[
        "TrialKey"
    ]
    .nunique()
    ==
    PHASE3G2_EXPECTED_TRIALS
)


phase3g2_counts = (
    phase3g2_valid
    .groupby(
        [
            "RepeaterCount",
            "Policy"
        ]
    )
    .size()
    .reset_index(
        name="Trials"
    )
)


assert all(
    phase3g2_counts[
        "Trials"
    ]
    == 100
)


successful = (
    phase3g2_valid[
        phase3g2_valid[
            "Success"
        ]
        == True
    ]
)


for _, row in successful.iterrows():

    assert (
        row[
            "FinalTreeLeaves"
        ]
        ==
        row[
            "RepeaterCount"
        ]
        + 1
    )


    assert (
        row[
            "FinalTreeSwaps"
        ]
        ==
        row[
            "RepeaterCount"
        ]
    )


print(
    "[PASS] All 1,200 density-sweep sessions are accounted for."
)

print(
    "[PASS] Every repeater/policy combination contains 100 trials."
)

print(
    "[PASS] All successful final trees have correct leaf and BSM counts."
)


display(
    phase3g2_counts
)

In [ ]:
# @title Statistical Aggregation


phase3g2_summary_rows = []


for (
    repeater_count,
    policy_name
), group in (
    phase3g2_valid
    .groupby(
        [
            "RepeaterCount",
            "Policy"
        ]
    )
):

    successful = (
        group[
            group[
                "Success"
            ]
            == True
        ]
    )


    n_trials = len(
        group
    )


    n_success = len(
        successful
    )


    success_rate = (
        n_success
        /
        n_trials
    )


    success_low, success_high = (
        phase3g_wilson_ci95(
            n_success,
            n_trials
        )
    )


    restricted_mean, restricted_low, restricted_high = (
        phase3g_mean_ci95(
            group[
                "RestrictedCompletionSlots"
            ]
        )
    )


    completion_mean, completion_low, completion_high = (
        phase3g_mean_ci95(
            successful[
                "ProcessedSlots"
            ]
        )
    )


    fidelity_mean, fidelity_low, fidelity_high = (
        phase3g_mean_ci95(
            successful[
                "FinalFidelity"
            ]
        )
    )


    elementary_mean, elementary_low, elementary_high = (
        phase3g_mean_ci95(
            group[
                "ElementaryAttempts"
            ]
        )
    )


    bsm_mean, bsm_low, bsm_high = (
        phase3g_mean_ci95(
            group[
                "BSMAttempts"
            ]
        )
    )


    expiration_mean, expiration_low, expiration_high = (
        phase3g_mean_ci95(
            group[
                "ExpiredResources"
            ]
        )
    )


    tree_depth_mean, tree_depth_low, tree_depth_high = (
        phase3g_mean_ci95(
            successful[
                "FinalTreeDepth"
            ]
        )
    )


    first_row = (
        group.iloc[0]
    )


    phase3g2_summary_rows.append({

        "RepeaterCount":
            repeater_count,

        "ElementaryLinks":
            int(
                first_row[
                    "ElementaryLinks"
                ]
            ),

        "HopDistanceKm":
            first_row[
                "HopDistanceKm"
            ],

        "ElementarySuccessProbability":
            first_row[
                "ElementarySuccessProbability"
            ],

        "Policy":
            policy_name,

        "Trials":
            n_trials,

        "Successes":
            n_success,

        "Timeouts":
            n_trials
            - n_success,

        "SuccessRate":
            success_rate,

        "SuccessCI95Low":
            success_low,

        "SuccessCI95High":
            success_high,

        "RestrictedMeanCompletionSlots":
            restricted_mean,

        "RestrictedCompletionCI95Low":
            restricted_low,

        "RestrictedCompletionCI95High":
            restricted_high,

        "MeanCompletionSlotsSuccessOnly":
            completion_mean,

        "CompletionCI95Low":
            completion_low,

        "CompletionCI95High":
            completion_high,

        "MeanFinalFidelitySuccessOnly":
            fidelity_mean,

        "FidelityCI95Low":
            fidelity_low,

        "FidelityCI95High":
            fidelity_high,

        "MeanElementaryAttemptsPerTrial":
            elementary_mean,

        "ElementaryAttemptsCI95Low":
            elementary_low,

        "ElementaryAttemptsCI95High":
            elementary_high,

        "MeanBSMAttemptsPerTrial":
            bsm_mean,

        "BSMAttemptsCI95Low":
            bsm_low,

        "BSMAttemptsCI95High":
            bsm_high,

        "MeanExpiredResourcesPerTrial":
            expiration_mean,

        "ExpiredResourcesCI95Low":
            expiration_low,

        "ExpiredResourcesCI95High":
            expiration_high,

        "MeanFinalTreeDepthSuccessOnly":
            tree_depth_mean,

        "TreeDepthCI95Low":
            tree_depth_low,

        "TreeDepthCI95High":
            tree_depth_high
    })


phase3g2_summary = pd.DataFrame(
    phase3g2_summary_rows
)


phase3g2_atomic_write_csv(
    phase3g2_summary,
    PHASE3G2_SUMMARY_PATH
)


display(
    phase3g2_summary
)

In [ ]:
# @title  Repeater Density Evaluation Graphs


# PHASE3G2_POLICY_DISPLAY_ORDER = (
#     "SEQUENTIAL_LEFT_TO_RIGHT",
#     "STATIC_BALANCED_TREE",
#     "MAX_PARALLEL_OLDEST_FIRST"
# )


# def phase3g2_plot_metric(
#     mean_column,
#     low_column,
#     high_column,
#     ylabel,
#     title,
#     filename,
#     y_min=None,
#     y_max=None,
#     log_y=False
# ):

#     fig, ax = plt.subplots(
#         figsize=(
#             10,
#             6
#         )
#     )


#     for policy_name in (
#         PHASE3G2_POLICY_DISPLAY_ORDER
#     ):

#         subset = (
#             phase3g2_summary[
#                 phase3g2_summary[
#                     "Policy"
#                 ]
#                 == policy_name
#             ]
#             .sort_values(
#                 "RepeaterCount"
#             )
#         )


#         x = subset[
#             "RepeaterCount"
#         ].to_numpy(
#             dtype=float
#         )


#         y = subset[
#             mean_column
#         ].to_numpy(
#             dtype=float
#         )


#         low = subset[
#             low_column
#         ].to_numpy(
#             dtype=float
#         )


#         high = subset[
#             high_column
#         ].to_numpy(
#             dtype=float
#         )


#         ax.errorbar(

#             x,

#             y,

#             yerr=np.vstack(
#                 [
#                     y - low,
#                     high - y
#                 ]
#             ),

#             marker="o",

#             capsize=4,

#             label=(
#                 policy_name
#             )
#         )


#     ax.set_xlabel(
#         "Number of repeaters across fixed 600 km path"
#     )

#     ax.set_ylabel(
#         ylabel
#     )

#     ax.set_title(
#         title
#     )

#     ax.set_xticks(
#         PHASE3G2_REPEATER_COUNTS
#     )

#     ax.grid(
#         True,
#         alpha=0.3
#     )

#     ax.legend()


#     if y_min is not None or y_max is not None:

#         current_low, current_high = (
#             ax.get_ylim()
#         )


#         ax.set_ylim(

#             y_min
#             if y_min is not None
#             else current_low,

#             y_max
#             if y_max is not None
#             else current_high
#         )


#     if log_y:

#         ax.set_yscale(
#             "log"
#         )


#     fig.tight_layout()


#     path = (
#         PHASE3G2_PLOT_DIR
#         /
#         filename
#     )


#     fig.savefig(
#         path,
#         dpi=200,
#         bbox_inches="tight"
#     )


#     plt.show()


# # =============================================================================
# # 1. Success probability
# # =============================================================================

# phase3g2_plot_metric(

#     "SuccessRate",

#     "SuccessCI95Low",

#     "SuccessCI95High",

#     "End-to-end success probability",

#     "NARAD-QC success probability vs repeater density at 600 km",

#     "01_success_vs_repeater_count.png",

#     y_min=0.0,

#     y_max=1.0
# )


# # =============================================================================
# # 2. Restricted completion cost
# # =============================================================================

# phase3g2_plot_metric(

#     "RestrictedMeanCompletionSlots",

#     "RestrictedCompletionCI95Low",

#     "RestrictedCompletionCI95High",

#     "Restricted mean completion slots",

#     "NARAD-QC completion cost vs repeater density at 600 km",

#     "02_completion_vs_repeater_count.png"
# )


# # =============================================================================
# # 3. Final fidelity
# # =============================================================================

# phase3g2_plot_metric(

#     "MeanFinalFidelitySuccessOnly",

#     "FidelityCI95Low",

#     "FidelityCI95High",

#     "Final end-to-end fidelity",

#     "NARAD-QC final fidelity vs repeater density at 600 km",

#     "03_fidelity_vs_repeater_count.png",

#     y_min=0.25,

#     y_max=1.0
# )


# # =============================================================================
# # 4. Elementary attempt burden
# # =============================================================================

# phase3g2_plot_metric(

#     "MeanElementaryAttemptsPerTrial",

#     "ElementaryAttemptsCI95Low",

#     "ElementaryAttemptsCI95High",

#     "Mean elementary attempts / trial",

#     "Elementary generation cost vs repeater density at 600 km",

#     "04_elementary_attempts_vs_repeater_count.png",

#     log_y=True
# )


# # =============================================================================
# # 5. BSM attempt burden
# # =============================================================================

# phase3g2_plot_metric(

#     "MeanBSMAttemptsPerTrial",

#     "BSMAttemptsCI95Low",

#     "BSMAttemptsCI95High",

#     "Mean atomic BSM attempts / trial",

#     "Swapping cost vs repeater density at 600 km",

#     "05_bsm_attempts_vs_repeater_count.png"
# )


# # =============================================================================
# # 6. Memory expiration burden
# # =============================================================================

# phase3g2_plot_metric(

#     "MeanExpiredResourcesPerTrial",

#     "ExpiredResourcesCI95Low",

#     "ExpiredResourcesCI95High",

#     "Mean expired resources / trial",

#     "Quantum-memory expiration burden vs repeater density at 600 km",

#     "06_expiration_vs_repeater_count.png"
# )

In [ ]:
# @title Success Probability vs Elementary Hop Distance


# fig, ax = plt.subplots(
#     figsize=(
#         10,
#         6
#     )
# )


# for policy_name in (
#     PHASE3G2_POLICY_DISPLAY_ORDER
# ):

#     subset = (
#         phase3g2_summary[
#             phase3g2_summary[
#                 "Policy"
#             ]
#             == policy_name
#         ]
#         .sort_values(
#             "HopDistanceKm"
#         )
#     )


#     x = (
#         subset[
#             "HopDistanceKm"
#         ]
#         .to_numpy(
#             dtype=float
#         )
#     )


#     y = (
#         subset[
#             "SuccessRate"
#         ]
#         .to_numpy(
#             dtype=float
#         )
#     )


#     low = (
#         subset[
#             "SuccessCI95Low"
#         ]
#         .to_numpy(
#             dtype=float
#         )
#     )


#     high = (
#         subset[
#             "SuccessCI95High"
#         ]
#         .to_numpy(
#             dtype=float
#         )
#     )


#     ax.errorbar(

#         x,

#         y,

#         yerr=np.vstack(
#             [
#                 y - low,
#                 high - y
#             ]
#         ),

#         marker="o",

#         capsize=4,

#         label=(
#             policy_name
#         )
#     )


# ax.set_xlabel(
#     "Elementary hop distance (km)"
# )

# ax.set_ylabel(
#     "End-to-end success probability"
# )

# ax.set_title(
#     "NARAD-QC success probability vs physical repeater spacing"
# )

# ax.set_ylim(
#     0.0,
#     1.0
# )

# ax.grid(
#     True,
#     alpha=0.3
# )

# ax.legend()

# fig.tight_layout()


# fig.savefig(

#     PHASE3G2_PLOT_DIR
#     / "07_success_vs_hop_distance.png",

#     dpi=200,

#     bbox_inches="tight"
# )


# plt.show()

### Latest Demo : 22-09-2026T03:26:54 -- 22-09-2026T05:17:32

In [ ]:
# @title Interactive Entanglement Demonstrator Imports

import math
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D

from IPython.display import display, HTML, clear_output

import ipywidgets as widgets


# ---------------------------------------------------------------------------
# Google Colab widget support
# ---------------------------------------------------------------------------

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass


# ---------------------------------------------------------------------------
# NARAD-QC frozen baseline assumptions
# ---------------------------------------------------------------------------

NARAD_DEMO_MAX_ELEMENTARY_HOP_KM = 100.0

NARAD_DEMO_MAX_STORAGE_SLOTS = 100

NARAD_DEMO_SLOT_DURATION_S = 0.05

NARAD_DEMO_ELEMENTARY_FIDELITY = 0.995

NARAD_DEMO_FIBER_ATTENUATION_DB_PER_KM = 0.20

NARAD_DEMO_BSM_SUCCESS_PROBABILITY = 0.40

NARAD_DEMO_MAX_EXECUTION_SLOTS = 20_000

NARAD_DEMO_DEFAULT_SEED = 3_720_001


# Presentation only.
NARAD_DEMO_FRAME_INTERVAL_MS = 1000


# Allow reasonably large JS animation embedding in Colab.
plt.rcParams["animation.embed_limit"] = 10000


# Prevent garbage collection of the active animation.
NARAD_DEMO_ACTIVE_ANIMATION = None


print(
    "[READY] Phase 3G.3 interactive demonstrator configuration loaded."
)

print(
    "Maximum permitted elementary hop:",
    NARAD_DEMO_MAX_ELEMENTARY_HOP_KM,
    "km"
)

In [ ]:
# @title Coordinate Parsing and Path Validation


def narad_demo_parse_repeater_coordinates(
    text
):

    coordinates = []

    stripped = text.strip()

    if not stripped:

        return tuple()


    for line_number, raw_line in enumerate(
        stripped.splitlines(),
        start=1
    ):

        line = raw_line.strip()

        if not line:
            continue


        parts = [
            part.strip()
            for part in line.split(",")
        ]


        if len(parts) != 2:

            raise ValueError(

                f"Repeater line {line_number} must contain exactly "
                f"two values in the form x,y.\n\n"

                f"Received: {raw_line!r}"
            )


        try:

            x = float(parts[0])
            y = float(parts[1])

        except ValueError as exc:

            raise ValueError(

                f"Repeater line {line_number} contains "
                f"non-numeric coordinates:\n{raw_line!r}"

            ) from exc


        if not (
            np.isfinite(x)
            and np.isfinite(y)
        ):

            raise ValueError(
                f"Repeater line {line_number} must contain finite values."
            )


        coordinates.append(
            (x, y)
        )


    return tuple(
        coordinates
    )



def narad_demo_euclidean_distance_km(
    point_a,
    point_b
):

    return float(
        math.hypot(
            point_b[0] - point_a[0],
            point_b[1] - point_a[1]
        )
    )



def narad_demo_validate_coordinates(
    user_a_coordinate,
    repeater_coordinates,
    user_b_coordinate
):

    ordered_coordinates = (

        user_a_coordinate,

        *repeater_coordinates,

        user_b_coordinate
    )


    ordered_names = (

        "User-A",

        *[
            f"R{i}"
            for i in range(
                1,
                len(repeater_coordinates) + 1
            )
        ],

        "User-B"
    )


    if len(
        set(ordered_coordinates)
    ) != len(
        ordered_coordinates
    ):

        raise ValueError(
            "Two network nodes cannot occupy exactly the same coordinate."
        )


    hop_rows = []

    violating_hops = []


    for index in range(
        len(ordered_coordinates) - 1
    ):

        left_name = ordered_names[index]

        right_name = ordered_names[index + 1]

        left_coordinate = ordered_coordinates[index]

        right_coordinate = ordered_coordinates[index + 1]


        distance_km = (
            narad_demo_euclidean_distance_km(
                left_coordinate,
                right_coordinate
            )
        )


        hop_rows.append({

            "From":
                left_name,

            "To":
                right_name,

            "DistanceKm":
                distance_km
        })


        if (
            distance_km
            >
            NARAD_DEMO_MAX_ELEMENTARY_HOP_KM
            + 1e-9
        ):

            violating_hops.append(
                (
                    left_name,
                    right_name,
                    distance_km
                )
            )


    if violating_hops:

        violation_text = "\n".join(

            f"  {left} -> {right}: {distance:.3f} km"

            for (
                left,
                right,
                distance
            )
            in violating_hops
        )


        raise ValueError(

            "The proposed path violates the frozen NARAD-QC "
            "elementary-link constraint:\n\n"

            f"    d_hop <= "
            f"{NARAD_DEMO_MAX_ELEMENTARY_HOP_KM:.1f} km\n\n"

            "Violating hops:\n"
            f"{violation_text}\n\n"

            "Add/reposition repeaters and run the demo again."
        )


    return {

        "ordered_names":
            ordered_names,

        "ordered_coordinates":
            ordered_coordinates,

        "hop_rows":
            tuple(
                hop_rows
            )
    }

In [ ]:
# @title Arbitrary Coordinate NARAD-QC Stack Builder


def narad_demo_make_policy(
    policy_name,
    elementary_link_count
):

    if (
        policy_name
        == "SEQUENTIAL_LEFT_TO_RIGHT"
    ):

        return (
            SequentialLeftToRightSwapPolicy()
        )


    if (
        policy_name
        == "STATIC_BALANCED_TREE"
    ):

        return (
            StaticBalancedTreeSwapPolicy(

                elementary_link_count=(
                    elementary_link_count
                )
            )
        )


    if (
        policy_name
        == "MAX_PARALLEL_OLDEST_FIRST"
    ):

        return (
            MaxParallelOldestFirstSwapPolicy()
        )


    raise ValueError(
        f"Unknown swapping policy: {policy_name}"
    )



def narad_demo_build_stack(
    user_a_coordinate,
    repeater_coordinates,
    user_b_coordinate,
    policy_name,
    simulation_seed
):

    validation = (
        narad_demo_validate_coordinates(

            user_a_coordinate=(
                user_a_coordinate
            ),

            repeater_coordinates=(
                repeater_coordinates
            ),

            user_b_coordinate=(
                user_b_coordinate
            )
        )
    )


    ordered_names = (
        validation[
            "ordered_names"
        ]
    )

    ordered_coordinates = (
        validation[
            "ordered_coordinates"
        ]
    )


    repeater_count = (
        len(
            repeater_coordinates
        )
    )

    elementary_link_count = (
        repeater_count
        + 1
    )


    # =======================================================================
    # Policy
    # =======================================================================

    policy = (
        narad_demo_make_policy(

            policy_name=(
                policy_name
            ),

            elementary_link_count=(
                elementary_link_count
            )
        )
    )


    # =======================================================================
    # Topology
    # =======================================================================

    topology = NetworkTopology(
        name="NARAD-QC-Interactive-Demo"
    )


    path_nodes = []


    for index, (
        node_name,
        coordinate
    ) in enumerate(
        zip(
            ordered_names,
            ordered_coordinates
        )
    ):

        is_endpoint = (
            index == 0
            or index == len(ordered_names) - 1
        )


        node = NetworkNode(

            name=node_name,

            x_km=float(
                coordinate[0]
            ),

            y_km=float(
                coordinate[1]
            ),

            endpoint_capable=(
                is_endpoint
            ),

            swapping_capable=(
                not is_endpoint
            ),

            memory_capacity=2
        )


        topology.add_node(
            node
        )


        path_nodes.append(
            node
        )


    path_nodes = tuple(
        path_nodes
    )


    for left_node, right_node in zip(
        path_nodes[:-1],
        path_nodes[1:]
    ):

        topology.add_link(

            PhysicalQuantumLink(

                node_a=left_node,

                node_b=right_node,

                attenuation_db_per_km=(
                    NARAD_DEMO_FIBER_ATTENUATION_DB_PER_KM
                )
            )
        )


    # =======================================================================
    # Simulator runtime
    # =======================================================================

    simulator = QuantumNetworkSimulator(

        topology=topology,

        simulation_seed=int(
            simulation_seed
        ),

        controller_id=(
            "NARAD-QC-INTERACTIVE-DEMO"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=(
            NARAD_DEMO_MAX_STORAGE_SLOTS
        )
    )


    # =======================================================================
    # Real mutual ML-DSA authentication
    # =======================================================================

    authentication_records = (
        simulator.authenticate_all_nodes()
    )


    if not all(

        record.status
        == AuthenticationStatus.AUTHENTICATED

        for record
        in authentication_records.values()
    ):

        raise RuntimeError(
            "Not all nodes completed ML-DSA authentication."
        )


    simulator.require_quantum_eligibility(
        *path_nodes
    )


    # =======================================================================
    # Elementary entanglement generation
    # =======================================================================

    generation_service = (
        ElementaryEntanglementService(

            simulator=simulator,

            model=(
                FiberSurvivalProxyModel()
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=simulator,

            service=(
                generation_service
            ),

            ordered_path=(
                path_nodes
            ),

            demand_active=False
        )
    )


    # =======================================================================
    # Physical scalar fidelity model
    # =======================================================================

    fidelity_profile = (
        WernerPhysicalFidelityProfile(

            elementary_fidelity=(
                NARAD_DEMO_ELEMENTARY_FIDELITY
            ),

            bell_fidelity_floor=0.25,

            memory_reference_initial_fidelity=1.0,

            memory_reference_fidelity=0.90,

            memory_reference_time_s=1.0,

            slot_duration_s=(
                NARAD_DEMO_SLOT_DURATION_S
            ),

            swap_operation_visibility=1.0
        )
    )


    fidelity_registry = (
        PhysicalEntanglementFidelityRegistry(

            simulator=simulator,

            profile=(
                fidelity_profile
            )
        )
    )


    # =======================================================================
    # Stochastic atomic BSM model
    # =======================================================================

    swap_model = (
        WernerStateAtomicBSMModel(

            fidelity_profile=(
                fidelity_profile
            ),

            atomic_bsm_success_probability=(
                NARAD_DEMO_BSM_SUCCESS_PROBABILITY
            ),

            atomic_bsm_latency_us=10.0
        )
    )


    swap_engine = (
        EntanglementSwapEngine(

            simulator=simulator,

            model=(
                swap_model
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    # =======================================================================
    # Parallel swap scheduler
    # =======================================================================

    swap_scheduler = (
        ControlPlaneParallelSwapScheduler(

            simulator=simulator,

            swap_engine=(
                swap_engine
            ),

            ordered_path=(
                path_nodes
            ),

            policy=(
                policy
            )
        )
    )


    # =======================================================================
    # Failure-aware elementary demand
    # =======================================================================

    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=simulator,

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=(
                path_nodes
            )
        )
    )


    # =======================================================================
    # Autonomous end-to-end orchestration
    # =======================================================================

    orchestrator = (
        FailureAwareEndToEndEntanglementOrchestrator(

            simulator=simulator,

            ordered_path=(
                path_nodes
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            demand_controller=(
                demand_controller
            ),

            swap_scheduler=(
                swap_scheduler
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    return {

        "validation":
            validation,

        "topology":
            topology,

        "simulator":
            simulator,

        "path":
            path_nodes,

        "policy":
            policy,

        "authentication_records":
            authentication_records,

        "generation_service":
            generation_service,

        "generation_scheduler":
            generation_scheduler,

        "fidelity_profile":
            fidelity_profile,

        "fidelity_registry":
            fidelity_registry,

        "swap_model":
            swap_model,

        "swap_engine":
            swap_engine,

        "swap_scheduler":
            swap_scheduler,

        "demand_controller":
            demand_controller,

        "orchestrator":
            orchestrator
    }

In [ ]:
# @title  Pure Observer Timeline Reconstruction


def narad_demo_enum_text(
    value
):

    if hasattr(
        value,
        "value"
    ):

        return str(
            value.value
        )

    return str(
        value
    )



def narad_demo_resource_id(
    resource
):

    return str(
        resource.resource_id
    )



def narad_demo_resource_kind(
    resource
):

    text = narad_demo_enum_text(
        resource.kind
    ).upper()


    if "ELEMENTARY" in text:

        return "ELEMENTARY"


    if "SWAPPED" in text:

        return "SWAPPED"


    return text



def narad_demo_resource_created_slot(
    resource
):

    return int(
        resource.created_slot
    )



def narad_demo_resource_endpoints(
    resource
):

    endpoints = tuple(
        resource.endpoint_names
    )


    if len(endpoints) != 2:

        raise RuntimeError(

            f"Resource {resource.resource_id} "
            f"does not have exactly two endpoints."
        )


    return endpoints



def narad_demo_terminal_slot(
    resource
):

    terminal_info = getattr(
        resource,
        "terminal_info",
        None
    )


    if terminal_info is None:

        return None


    if isinstance(
        terminal_info,
        dict
    ):

        for key in (
            "terminal_slot",
            "simulation_slot",
            "slot",
            "at_slot"
        ):

            if (
                key in terminal_info
                and terminal_info[key] is not None
            ):

                return int(
                    terminal_info[key]
                )


        return None


    for attribute_name in (
        "terminal_slot",
        "simulation_slot",
        "slot",
        "at_slot"
    ):

        if hasattr(
            terminal_info,
            attribute_name
        ):

            value = getattr(
                terminal_info,
                attribute_name
            )


            if value is not None:

                return int(
                    value
                )


    return None



def narad_demo_round_slot(
    round_record
):

    for attribute_name in (
        "simulation_slot",
        "slot"
    ):

        if hasattr(
            round_record,
            attribute_name
        ):

            return int(
                getattr(
                    round_record,
                    attribute_name
                )
            )


    raise AttributeError(
        "Could not determine swap-round simulation slot."
    )



def narad_demo_candidate_input_ids(
    candidate
):

    return (

        str(
            candidate.left_resource_id
        ),

        str(
            candidate.right_resource_id
        )
    )



def narad_demo_build_visual_trace(
    stack,
    result,
    policy_name
):

    simulator = (
        stack[
            "simulator"
        ]
    )


    manager = (
        simulator
        .entanglement_manager
    )


    resources = tuple(
        manager.resources
    )


    resource_by_id = {

        narad_demo_resource_id(resource):
            resource

        for resource
        in resources
    }


    # =======================================================================
    # Frames 0 and 1:
    # physical setup followed by actual authenticated state.
    # =======================================================================

    frames = [

        {
            "phase":
                "TOPOLOGY",

            "title":
                "1. Physical network topology established",

            "slot":
                None,

            "authenticated":
                False,

            "active_resource_ids":
                tuple(),

            "new_resource_ids":
                tuple(),

            "swap_input_ids":
                tuple(),

            "swap_output_ids":
                tuple(),

            "swap_node_names":
                tuple()
        },

        {
            "phase":
                "AUTHENTICATION",

            "title":
                "2. Mutual ML-DSA authentication completed",

            "slot":
                None,

            "authenticated":
                True,

            "active_resource_ids":
                tuple(),

            "new_resource_ids":
                tuple(),

            "swap_input_ids":
                tuple(),

            "swap_output_ids":
                tuple(),

            "swap_node_names":
                tuple()
        }
    ]


    # =======================================================================
    # Elementary creations by slot
    # =======================================================================

    elementary_created_by_slot = {}


    swapped_created_by_slot = {}


    for resource in resources:

        slot = (
            narad_demo_resource_created_slot(
                resource
            )
        )


        resource_id = (
            narad_demo_resource_id(
                resource
            )
        )


        kind = (
            narad_demo_resource_kind(
                resource
            )
        )


        if kind == "ELEMENTARY":

            elementary_created_by_slot.setdefault(
                slot,
                []
            ).append(
                resource_id
            )


        elif kind == "SWAPPED":

            swapped_created_by_slot.setdefault(
                slot,
                []
            ).append(
                resource_id
            )


    # =======================================================================
    # Actual BSM rounds
    # =======================================================================

    rounds_by_slot = {}


    selected_input_ids_by_slot = {}


    swap_nodes_by_slot = {}


    for round_record in (
        stack[
            "swap_scheduler"
        ]
        .round_history
    ):

        selected_candidates = tuple(
            getattr(
                round_record,
                "selected_candidates",
                tuple()
            )
        )


        if not selected_candidates:
            continue


        slot = (
            narad_demo_round_slot(
                round_record
            )
        )


        rounds_by_slot.setdefault(
            slot,
            []
        ).append(
            round_record
        )


        selected_ids = (
            selected_input_ids_by_slot.setdefault(
                slot,
                set()
            )
        )


        swap_nodes = (
            swap_nodes_by_slot.setdefault(
                slot,
                set()
            )
        )


        for candidate in (
            selected_candidates
        ):

            left_id, right_id = (
                narad_demo_candidate_input_ids(
                    candidate
                )
            )


            selected_ids.add(
                left_id
            )

            selected_ids.add(
                right_id
            )


            swap_node_name = getattr(
                candidate,
                "swap_node_name",
                None
            )


            if swap_node_name is None:

                swap_node_name = getattr(
                    candidate,
                    "swap_node",
                    None
                )


            if swap_node_name is not None:

                if hasattr(
                    swap_node_name,
                    "name"
                ):

                    swap_node_name = (
                        swap_node_name.name
                    )


                swap_nodes.add(
                    str(
                        swap_node_name
                    )
                )


    # =======================================================================
    # Non-BSM terminations:
    # expiry / discard etc.
    #
    # BSM-consumed resources are deliberately NOT removed here because
    # they must remain visible during the pre-BSM generation frame.
    # =======================================================================

    non_swap_terminations_by_slot = {}


    for resource in resources:

        resource_id = (
            narad_demo_resource_id(
                resource
            )
        )


        terminal_slot = (
            narad_demo_terminal_slot(
                resource
            )
        )


        if terminal_slot is None:
            continue


        bsm_consumed_this_slot = (

            resource_id
            in selected_input_ids_by_slot.get(
                terminal_slot,
                set()
            )
        )


        if bsm_consumed_this_slot:
            continue


        non_swap_terminations_by_slot.setdefault(
            terminal_slot,
            []
        ).append(
            resource_id
        )


    # =======================================================================
    # Replay only slots where the VISIBLE network state changes.
    # =======================================================================

    interesting_slots = sorted(

        set(
            elementary_created_by_slot.keys()
        )

        |

        set(
            swapped_created_by_slot.keys()
        )

        |

        set(
            rounds_by_slot.keys()
        )

        |

        set(
            non_swap_terminations_by_slot.keys()
        )
    )


    active_ids = set()


    for slot in interesting_slots:

        # -------------------------------------------------------------------
        # Expirations/discards occur before generation in our slot cycle.
        # -------------------------------------------------------------------

        removed_before_generation = tuple(
            non_swap_terminations_by_slot.get(
                slot,
                tuple()
            )
        )


        for resource_id in (
            removed_before_generation
        ):

            active_ids.discard(
                resource_id
            )


        # -------------------------------------------------------------------
        # Parallel elementary generation successes.
        # -------------------------------------------------------------------

        new_elementary_ids = tuple(
            elementary_created_by_slot.get(
                slot,
                tuple()
            )
        )


        for resource_id in (
            new_elementary_ids
        ):

            active_ids.add(
                resource_id
            )


        if (
            new_elementary_ids
            or removed_before_generation
        ):

            if new_elementary_ids:

                generation_title = (

                    "3. Parallel elementary entanglement generation "
                    f"— {len(new_elementary_ids)} new pair(s)"
                )

            else:

                generation_title = (

                    "3. Memory state update "
                    "— expired/discarded pair(s) removed"
                )


            frames.append({

                "phase":
                    "GENERATION",

                "title":
                    generation_title,

                "slot":
                    slot,

                "authenticated":
                    True,

                "active_resource_ids":
                    tuple(
                        sorted(
                            active_ids
                        )
                    ),

                "new_resource_ids":
                    new_elementary_ids,

                "swap_input_ids":
                    tuple(),

                "swap_output_ids":
                    tuple(),

                "swap_node_names":
                    tuple()
            })


        # -------------------------------------------------------------------
        # One control-plane parallel swap round per slot.
        #
        # Anti-cascade rule remains intact:
        # outputs of this round cannot become inputs again in this round.
        # -------------------------------------------------------------------

        if slot in rounds_by_slot:

            swap_input_ids = tuple(
                sorted(
                    selected_input_ids_by_slot.get(
                        slot,
                        set()
                    )
                )
            )


            # All attempted BSM inputs are consumed,
            # whether that BSM succeeds or fails.
            for resource_id in (
                swap_input_ids
            ):

                active_ids.discard(
                    resource_id
                )


            swap_output_ids = tuple(
                swapped_created_by_slot.get(
                    slot,
                    tuple()
                )
            )


            for resource_id in (
                swap_output_ids
            ):

                active_ids.add(
                    resource_id
                )


            bsm_attempt_count = (
                len(
                    swap_input_ids
                )
                // 2
            )


            frames.append({

                "phase":
                    "SWAPPING",

                "title":
                    (
                        "4. Parallel entanglement swapping "
                        f"— {bsm_attempt_count} BSM(s), "
                        f"{len(swap_output_ids)} successful output pair(s)"
                    ),

                "slot":
                    slot,

                "authenticated":
                    True,

                "active_resource_ids":
                    tuple(
                        sorted(
                            active_ids
                        )
                    ),

                "new_resource_ids":
                    tuple(),

                "swap_input_ids":
                    swap_input_ids,

                "swap_output_ids":
                    swap_output_ids,

                "swap_node_names":
                    tuple(
                        sorted(
                            swap_nodes_by_slot.get(
                                slot,
                                set()
                            )
                        )
                    )
            })


        # Defensive fallback:
        # swapped resource exists but round-history metadata was absent.
        elif slot in swapped_created_by_slot:

            for resource_id in (
                swapped_created_by_slot[slot]
            ):

                active_ids.add(
                    resource_id
                )


    # =======================================================================
    # Final presentation frame
    # =======================================================================

    if result.success:

        frames.append({

            "phase":
                "SUCCESS",

            "title":
                "5. End-to-end entanglement established",

            "slot":
                result.end_slot,

            "authenticated":
                True,

            "active_resource_ids":
                tuple(
                    sorted(
                        active_ids
                    )
                ),

            "new_resource_ids":
                tuple(),

            "swap_input_ids":
                tuple(),

            "swap_output_ids":
                tuple(),

            "swap_node_names":
                tuple()
        })


    else:

        frames.append({

            "phase":
                "TIMEOUT",

            "title":
                (
                    "Execution horizon reached before "
                    "end-to-end entanglement"
                ),

            "slot":
                result.end_slot,

            "authenticated":
                True,

            "active_resource_ids":
                tuple(
                    sorted(
                        active_ids
                    )
                ),

            "new_resource_ids":
                tuple(),

            "swap_input_ids":
                tuple(),

            "swap_output_ids":
                tuple(),

            "swap_node_names":
                tuple()
        })


    return {

        "frames":
            tuple(
                frames
            ),

        "resource_by_id":
            resource_by_id,

        "policy_name":
            policy_name
    }

In [ ]:
# @title Matplotlib FuncAnimation Renderer


def narad_demo_create_animation(
    stack,
    result,
    trace,
    frame_interval_ms=(
        NARAD_DEMO_FRAME_INTERVAL_MS
    )
):

    validation = (
        stack[
            "validation"
        ]
    )


    ordered_names = (
        validation[
            "ordered_names"
        ]
    )


    ordered_coordinates = (
        validation[
            "ordered_coordinates"
        ]
    )


    coordinate_by_name = {

        node_name:
            coordinate

        for (
            node_name,
            coordinate
        )
        in zip(
            ordered_names,
            ordered_coordinates
        )
    }


    frames = (
        trace[
            "frames"
        ]
    )


    resource_by_id = (
        trace[
            "resource_by_id"
        ]
    )


    policy_name = (
        trace[
            "policy_name"
        ]
    )


    # =======================================================================
    # Plot limits
    # =======================================================================

    xs = [
        coordinate[0]
        for coordinate
        in ordered_coordinates
    ]


    ys = [
        coordinate[1]
        for coordinate
        in ordered_coordinates
    ]


    x_span = max(
        max(xs) - min(xs),
        1.0
    )


    y_span = max(
        max(ys) - min(ys),
        1.0
    )


    x_pad = max(
        0.08 * x_span,
        8.0
    )


    y_pad = max(
        0.15 * y_span,
        8.0
    )


    fig, ax = plt.subplots(
        figsize=(
            13,
            7
        )
    )


    # =======================================================================
    # Static legend
    # =======================================================================

    legend_handles = [

        Line2D(
            [0],
            [0],
            linestyle="-",
            linewidth=2.2,
            color="0.20",
            label="Physical fibre"
        ),

        Line2D(
            [0],
            [0],
            linestyle=":",
            linewidth=2.6,
            color="tab:blue",
            label="Elementary entanglement"
        ),

        Line2D(
            [0],
            [0],
            linestyle=":",
            linewidth=3.0,
            color="tab:purple",
            label="Swapped entanglement"
        ),

        Line2D(
            [0],
            [0],
            marker="s",
            markersize=10,
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor="black",
            label="User node"
        ),

        Line2D(
            [0],
            [0],
            marker="o",
            markersize=10,
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="black",
            label="Repeater"
        ),

        Line2D(
            [0],
            [0],
            marker="o",
            markersize=10,
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="lightgreen",
            markeredgewidth=3,
            label="ML-DSA authenticated"
        )
    ]


    # =======================================================================
    # Frame renderer
    # =======================================================================

    def render_frame(
        frame_index
    ):

        ax.clear()


        frame = (
            frames[
                frame_index
            ]
        )


        authenticated = (
            frame[
                "authenticated"
            ]
        )


        active_resource_ids = (
            frame[
                "active_resource_ids"
            ]
        )


        new_resource_ids = set(
            frame[
                "new_resource_ids"
            ]
        )


        swap_output_ids = set(
            frame[
                "swap_output_ids"
            ]
        )


        swap_node_names = set(
            frame[
                "swap_node_names"
            ]
        )


        # ===================================================================
        # Physical fibres
        # ===================================================================

        for index in range(
            len(
                ordered_coordinates
            )
            - 1
        ):

            left = (
                ordered_coordinates[
                    index
                ]
            )

            right = (
                ordered_coordinates[
                    index + 1
                ]
            )


            ax.plot(

                [
                    left[0],
                    right[0]
                ],

                [
                    left[1],
                    right[1]
                ],

                linestyle="-",

                linewidth=2.3,

                color="0.20",

                zorder=1
            )


        # ===================================================================
        # Active entanglement resources
        # ===================================================================

        for resource_id in (
            active_resource_ids
        ):

            resource = (
                resource_by_id.get(
                    resource_id
                )
            )


            if resource is None:
                continue


            endpoint_a, endpoint_b = (
                narad_demo_resource_endpoints(
                    resource
                )
            )


            if (
                endpoint_a
                not in coordinate_by_name

                or

                endpoint_b
                not in coordinate_by_name
            ):

                continue


            coordinate_a = (
                coordinate_by_name[
                    endpoint_a
                ]
            )

            coordinate_b = (
                coordinate_by_name[
                    endpoint_b
                ]
            )


            kind = (
                narad_demo_resource_kind(
                    resource
                )
            )


            if kind == "ELEMENTARY":

                entanglement_color = (
                    "tab:blue"
                )

                base_width = 2.6


            else:

                entanglement_color = (
                    "tab:purple"
                )

                base_width = 3.0


            if (
                resource_id
                in new_resource_ids

                or

                resource_id
                in swap_output_ids
            ):

                linewidth = (
                    base_width
                    + 1.8
                )

                alpha = 1.0

            else:

                linewidth = (
                    base_width
                )

                alpha = 0.82


            ax.plot(

                [
                    coordinate_a[0],
                    coordinate_b[0]
                ],

                [
                    coordinate_a[1],
                    coordinate_b[1]
                ],

                linestyle=":",

                linewidth=linewidth,

                color=(
                    entanglement_color
                ),

                alpha=alpha,

                zorder=3
            )


        # ===================================================================
        # User nodes
        # ===================================================================

        endpoint_names = {

            ordered_names[0],
            ordered_names[-1]
        }


        for node_name in (
            endpoint_names
        ):

            x, y = (
                coordinate_by_name[
                    node_name
                ]
            )


            if authenticated:

                user_face = (
                    "lightgreen"
                )

                user_edge = (
                    "green"
                )

            else:

                user_face = (
                    "white"
                )

                user_edge = (
                    "black"
                )


            ax.scatter(

                [x],
                [y],

                marker="s",

                s=280,

                facecolors=(
                    user_face
                ),

                edgecolors=(
                    user_edge
                ),

                linewidths=2.4,

                zorder=6
            )


        # ===================================================================
        # Repeaters:
        # ALWAYS hollow circles.
        # Authentication is shown using their outline.
        # ===================================================================

        repeater_names = (
            ordered_names[
                1:-1
            ]
        )


        for node_name in (
            repeater_names
        ):

            x, y = (
                coordinate_by_name[
                    node_name
                ]
            )


            repeater_edge = (

                "lightgreen"
                if authenticated
                else "black"
            )


            repeater_width = (

                3.2
                if authenticated
                else 2.2
            )


            ax.scatter(

                [x],
                [y],

                marker="o",

                s=300,

                facecolors="none",

                edgecolors=(
                    repeater_edge
                ),

                linewidths=(
                    repeater_width
                ),

                zorder=6
            )


            # ---------------------------------------------------------------
            # Highlight repeaters currently executing a BSM.
            # Purely a presentation overlay.
            # ---------------------------------------------------------------

            if (
                node_name
                in swap_node_names
            ):

                ax.scatter(

                    [x],
                    [y],

                    marker="o",

                    s=470,

                    facecolors="none",

                    edgecolors="gold",

                    linewidths=2.3,

                    zorder=5
                )


        # ===================================================================
        # Labels
        # ===================================================================

        for node_name in (
            ordered_names
        ):

            x, y = (
                coordinate_by_name[
                    node_name
                ]
            )


            ax.annotate(

                node_name,

                xy=(
                    x,
                    y
                ),

                xytext=(
                    0,
                    14
                ),

                textcoords=(
                    "offset points"
                ),

                ha="center",

                va="bottom",

                fontsize=10,

                fontweight="bold",

                zorder=8
            )


        # ===================================================================
        # Frame status box
        # ===================================================================

        slot = (
            frame[
                "slot"
            ]
        )


        if slot is None:

            slot_text = (
                "Simulation slot: —"
            )

        else:

            slot_text = (
                f"Simulation slot: {slot}"
            )


        status_text = (

            f"{frame['title']}\n"
            f"{slot_text}\n"
            f"Policy: {policy_name}\n"
            f"Active entangled resources: "
            f"{len(active_resource_ids)}"
        )


        ax.text(

            0.015,
            0.985,

            status_text,

            transform=(
                ax.transAxes
            ),

            ha="left",

            va="top",

            fontsize=10,

            bbox={
                "boxstyle":
                    "round,pad=0.5",

                "facecolor":
                    "white",

                "edgecolor":
                    "0.5",

                "alpha":
                    0.92
            },

            zorder=20
        )


        # ===================================================================
        # Axes
        # ===================================================================

        ax.set_xlim(
            min(xs) - x_pad,
            max(xs) + x_pad
        )


        ax.set_ylim(
            min(ys) - y_pad,
            max(ys) + y_pad
        )


        ax.set_xlabel(
            "X coordinate (km)"
        )


        ax.set_ylabel(
            "Y coordinate (km)"
        )


        ax.set_title(
            "NARAD-QC — Authentication, Entanglement Generation "
            "and Parallel Swapping"
        )


        ax.set_aspect(
            "equal",
            adjustable="box"
        )


        ax.grid(
            True,
            alpha=0.20
        )


        ax.legend(

            handles=(
                legend_handles
            ),

            loc="upper right",

            framealpha=0.95
        )


        return tuple()


    animation = FuncAnimation(

        fig,

        render_frame,

        frames=len(
            frames
        ),

        interval=(
            frame_interval_ms
        ),

        repeat=False,

        blit=False
    )


    return (
        fig,
        animation
    )

In [ ]:
# @title Execute Interactive Demonstration


def narad_demo_run(
    user_a_coordinate,
    repeater_coordinates,
    user_b_coordinate,
    policy_name,
    simulation_seed
):

    print(
        "Building NARAD-QC network..."
    )


    stack = (
        narad_demo_build_stack(

            user_a_coordinate=(
                user_a_coordinate
            ),

            repeater_coordinates=(
                repeater_coordinates
            ),

            user_b_coordinate=(
                user_b_coordinate
            ),

            policy_name=(
                policy_name
            ),

            simulation_seed=(
                simulation_seed
            )
        )
    )


    print(
        "Topology valid under the 86 km elementary-hop constraint."
    )


    print(
        "Executing autonomous failure-aware entanglement delivery..."
    )


    result = (
        stack[
            "orchestrator"
        ]
        .run_until_end_to_end(

            max_slots=(
                NARAD_DEMO_MAX_EXECUTION_SLOTS
            )
        )
    )


    if (
        stack[
            "simulator"
        ]
        .validate_runtime_consistency()
        is not True
    ):

        raise RuntimeError(
            "Runtime consistency validation failed."
        )


    trace = (
        narad_demo_build_visual_trace(

            stack=(
                stack
            ),

            result=(
                result
            ),

            policy_name=(
                policy_name
            )
        )
    )


    fig, animation = (
        narad_demo_create_animation(

            stack=(
                stack
            ),

            result=(
                result
            ),

            trace=(
                trace
            )
        )
    )


    print()
    print(
        "------------------------------------------------------------"
    )

    print(
        "NARAD-QC DEMONSTRATION RESULT"
    )

    print(
        "------------------------------------------------------------"
    )

    print(
        "Policy:",
        policy_name
    )

    print(
        "Repeaters:",
        len(
            repeater_coordinates
        )
    )

    print(
        "Elementary links:",
        len(
            repeater_coordinates
        )
        + 1
    )

    print(
        "Result:",
        (
            "SUCCESS"
            if result.success
            else "TIMEOUT"
        )
    )

    print(
        "Processed slots:",
        result.processed_slot_count
    )

    print(
        "Elementary attempts:",
        result.total_elementary_attempts
    )

    print(
        "BSM attempts:",
        result.total_bsm_attempts
    )

    print(
        "BSM successes:",
        result.total_bsm_successes
    )

    print(
        "BSM failures:",
        result.total_bsm_failures
    )


    if result.success:

        print(
            "Final fidelity:",
            f"{result.final_fidelity:.6f}"
        )


    print(
        "Animation state-change frames:",
        len(
            trace[
                "frames"
            ]
        )
    )

    print(
        "------------------------------------------------------------"
    )


    return {

        "stack":
            stack,

        "result":
            result,

        "trace":
            trace,

        "figure":
            fig,

        "animation":
            animation
    }

In [ ]:
#@title Renderer

# Replace the original renderer cell with this one. The simulation helpers,
# constants and trace builder from earlier cells remain in scope.
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D

def narad_demo_create_animation(

    stack,
    result,
    trace,
    frame_interval_ms=(
        NARAD_DEMO_FRAME_INTERVAL_MS
    )
):

    validation = (
        stack[
            "validation"
        ]
    )


    ordered_names = (
        validation[
            "ordered_names"
        ]
    )


    ordered_coordinates = (
        validation[
            "ordered_coordinates"
        ]
    )


    coordinate_by_name = {

        node_name:
            coordinate

        for (
            node_name,
            coordinate
        )
        in zip(
            ordered_names,
            ordered_coordinates
        )
    }


    frames = (
        trace[
            "frames"
        ]
    )


    resource_by_id = (
        trace[
            "resource_by_id"
        ]
    )


    policy_name = (
        trace[
            "policy_name"
        ]
    )


    # =======================================================================
    # Plot limits
    # =======================================================================

    xs = [
        coordinate[0]
        for coordinate
        in ordered_coordinates
    ]


    ys = [
        coordinate[1]
        for coordinate
        in ordered_coordinates
    ]


    x_span = max(
        max(xs) - min(xs),
        1.0
    )


    y_span = max(
        max(ys) - min(ys),
        1.0
    )


    x_pad = max(
        0.08 * x_span,
        8.0
    )


    y_pad = max(
        0.15 * y_span,
        8.0
    )


    # The topology owns the large canvas; a separate panel owns all metadata.
    # A wide path needs a shorter figure; a vertically spread path gets height.
    figure_height = min(8.5, max(6.3, 5.4 + 2.4 * y_span / x_span))
    fig = plt.figure(figsize=(16, figure_height), facecolor="#f4f7fb")
    grid = fig.add_gridspec(
        1, 2, width_ratios=(4.7, 1.55),
        left=0.065, right=0.975, bottom=0.105, top=0.875, wspace=0.12,
    )
    ax = fig.add_subplot(grid[0, 0])
    info_ax = fig.add_subplot(grid[0, 1])
    fig.suptitle(
        "NARAD-QC  |  Entanglement across a repeater chain",
        x=0.065, y=0.965, ha="left", fontsize=18, fontweight="bold",
        color="#18263b",
    )
    fig.text(
        0.065, 0.913, "Authentication  ·  Pair generation  ·  Parallel swapping",
        ha="left", va="center", fontsize=10.5, color="#64748b",
    )


    # =======================================================================
    # Static legend
    # =======================================================================

    legend_handles = [

        Line2D(
            [0],
            [0],
            linestyle="-",
            linewidth=2.2,
            color="#94a3b8",
            label="Physical fibre"
        ),

        Line2D(
            [0],
            [0],
            linestyle=":",
            linewidth=2.6,
            color="#2979b8",
            label="Elementary entanglement"
        ),

        Line2D(
            [0],
            [0],
            linestyle=":",
            linewidth=3.0,
            color="#8251a8",
            label="Swapped entanglement"
        ),

        Line2D(
            [0],
            [0],
            marker="s",
            markersize=10,
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor="black",
            label="User node"
        ),

        Line2D(
            [0],
            [0],
            marker="o",
            markersize=10,
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="black",
            label="Repeater"
        ),

        Line2D(
            [0],
            [0],
            marker="o",
            markersize=10,
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="#48b691",
            markeredgewidth=3,
            label="ML-DSA authenticated"
        )
    ]


    # =======================================================================
    # Frame renderer
    # =======================================================================

    def render_frame(
        frame_index
    ):

        ax.clear()
        info_ax.clear()
        ax.set_facecolor("#ffffff")
        info_ax.set_facecolor("#eaf0f8")


        frame = (
            frames[
                frame_index
            ]
        )


        authenticated = (
            frame[
                "authenticated"
            ]
        )


        active_resource_ids = (
            frame[
                "active_resource_ids"
            ]
        )


        new_resource_ids = set(
            frame[
                "new_resource_ids"
            ]
        )


        swap_output_ids = set(
            frame[
                "swap_output_ids"
            ]
        )


        swap_node_names = set(
            frame[
                "swap_node_names"
            ]
        )


        # ===================================================================
        # Physical fibres
        # ===================================================================

        for index in range(
            len(
                ordered_coordinates
            )
            - 1
        ):

            left = (
                ordered_coordinates[
                    index
                ]
            )

            right = (
                ordered_coordinates[
                    index + 1
                ]
            )


            ax.plot(

                [
                    left[0],
                    right[0]
                ],

                [
                    left[1],
                    right[1]
                ],

                linestyle="-",

                linewidth=2.3,

                color="#94a3b8",

                zorder=1
            )


        # ===================================================================
        # Active entanglement resources
        # ===================================================================

        for resource_id in (
            active_resource_ids
        ):

            resource = (
                resource_by_id.get(
                    resource_id
                )
            )


            if resource is None:
                continue


            endpoint_a, endpoint_b = (
                narad_demo_resource_endpoints(
                    resource
                )
            )


            if (
                endpoint_a
                not in coordinate_by_name

                or

                endpoint_b
                not in coordinate_by_name
            ):

                continue


            coordinate_a = (
                coordinate_by_name[
                    endpoint_a
                ]
            )

            coordinate_b = (
                coordinate_by_name[
                    endpoint_b
                ]
            )


            kind = (
                narad_demo_resource_kind(
                    resource
                )
            )


            if kind == "ELEMENTARY":

                entanglement_color = (
                    "#2979b8"
                )

                base_width = 2.6


            else:

                entanglement_color = (
                    "#8251a8"
                )

                base_width = 3.0


            if (
                resource_id
                in new_resource_ids

                or

                resource_id
                in swap_output_ids
            ):

                linewidth = (
                    base_width
                    + 1.8
                )

                alpha = 1.0

            else:

                linewidth = (
                    base_width
                )

                alpha = 0.82


            ax.plot(

                [
                    coordinate_a[0],
                    coordinate_b[0]
                ],

                [
                    coordinate_a[1],
                    coordinate_b[1]
                ],

                linestyle=":",

                linewidth=linewidth,

                color=(
                    entanglement_color
                ),

                alpha=alpha,

                zorder=3
            )


        # ===================================================================
        # User nodes
        # ===================================================================

        endpoint_names = {

            ordered_names[0],
            ordered_names[-1]
        }


        for node_name in (
            endpoint_names
        ):

            x, y = (
                coordinate_by_name[
                    node_name
                ]
            )


            if authenticated:

                user_face = (
                    "#48b691"
                )

                user_edge = (
                    "#168263"
                )

            else:

                user_face = (
                    "white"
                )

                user_edge = (
                    "black"
                )


            ax.scatter(

                [x],
                [y],

                marker="s",

                s=280,

                facecolors=(
                    user_face
                ),

                edgecolors=(
                    user_edge
                ),

                linewidths=2.4,

                zorder=6
            )


        # ===================================================================
        # Repeaters:
        # ALWAYS hollow circles.
        # Authentication is shown using their outline.
        # ===================================================================

        repeater_names = (
            ordered_names[
                1:-1
            ]
        )


        for node_name in (
            repeater_names
        ):

            x, y = (
                coordinate_by_name[
                    node_name
                ]
            )


            repeater_edge = (

                "#48b691"
                if authenticated
                else "black"
            )


            repeater_width = (

                3.2
                if authenticated
                else 2.2
            )


            ax.scatter(

                [x],
                [y],

                marker="o",

                s=300,

                facecolors="none",

                edgecolors=(
                    repeater_edge
                ),

                linewidths=(
                    repeater_width
                ),

                zorder=6
            )


            # ---------------------------------------------------------------
            # Highlight repeaters currently executing a BSM.
            # Purely a presentation overlay.
            # ---------------------------------------------------------------

            if (
                node_name
                in swap_node_names
            ):

                ax.scatter(

                    [x],
                    [y],

                    marker="o",

                    s=470,

                    facecolors="none",

                    edgecolors="#e8a633",

                    linewidths=2.3,

                    zorder=5
                )


        # ===================================================================
        # Labels
        # ===================================================================

        for node_index, node_name in enumerate(ordered_names):

            x, y = (
                coordinate_by_name[
                    node_name
                ]
            )


            ax.annotate(

                node_name,

                xy=(
                    x,
                    y
                ),

                xytext=(0, -16 if node_index % 2 else 14),

                textcoords=(
                    "offset points"
                ),

                ha="center",

                va="top" if node_index % 2 else "bottom",

                fontsize=9.5,

                fontweight="bold",

                zorder=8
            )


        # ===================================================================
        # Frame status box
        # ===================================================================

        slot = (
            frame[
                "slot"
            ]
        )


        if slot is None:

            slot_text = (
                "Simulation slot: —"
            )

        else:

            slot_text = (
                f"Simulation slot: {slot}"
            )


        # Status lives outside the data axes, so it cannot cover network nodes.
        info_ax.set_xlim(0, 1)
        info_ax.set_ylim(0, 1)
        info_ax.set_xticks([])
        info_ax.set_yticks([])
        for spine in info_ax.spines.values():
            spine.set_visible(False)

        phase = frame["phase"].replace("_", " ").title()
        info_ax.text(0.07, 0.94, "CURRENT PHASE", transform=info_ax.transAxes,
                     fontsize=9, fontweight="bold", color="#64748b", va="top")
        info_ax.text(0.07, 0.895, phase, transform=info_ax.transAxes,
                     fontsize=15, fontweight="bold", color="#18263b", va="top")
        info_ax.text(0.07, 0.835, textwrap.fill(frame["title"], 29),
                     transform=info_ax.transAxes, fontsize=10.5,
                     color="#334155", va="top", linespacing=1.4)
        info_ax.plot([0.07, 0.93], [0.69, 0.69], transform=info_ax.transAxes,
                     color="#cbd5e1", linewidth=1)

        info_ax.text(0.07, 0.645, "SIMULATION", transform=info_ax.transAxes,
                     fontsize=9, fontweight="bold", color="#64748b", va="top")
        info_ax.text(0.07, 0.59, slot_text, transform=info_ax.transAxes,
                     fontsize=10, color="#18263b", va="top")
        info_ax.text(0.07, 0.535, "Active pairs: " + str(len(active_resource_ids)),
                     transform=info_ax.transAxes, fontsize=10, color="#18263b", va="top")
        info_ax.text(0.07, 0.48, "Policy: " + textwrap.fill(
                         policy_name.replace("_", " ").title(), 28),
                     transform=info_ax.transAxes, fontsize=9.5,
                     color="#475569", va="top", linespacing=1.35)

        info_ax.text(0.07, 0.365, "VISUAL KEY", transform=info_ax.transAxes,
                     fontsize=9, fontweight="bold", color="#64748b", va="top")
        info_ax.legend(handles=legend_handles, loc="upper left",
                       bbox_to_anchor=(0.05, 0.335), frameon=False,
                       fontsize=8.8, labelspacing=1.0, borderaxespad=0,
                       handlelength=2.3, handletextpad=0.8)


        # ===================================================================
        # Axes
        # ===================================================================

        ax.set_xlim(
            min(xs) - x_pad,
            max(xs) + x_pad
        )


        ax.set_ylim(
            min(ys) - y_pad,
            max(ys) + y_pad
        )


        ax.set_xlabel(
            "X coordinate (km)"
        )


        ax.set_ylabel(
            "Y coordinate (km)"
        )


        ax.set_aspect(
            "equal",
            adjustable="box"
        )


        ax.grid(True, color="#e2e8f0", linewidth=0.8)
        ax.tick_params(colors="#64748b", labelsize=9)
        ax.xaxis.label.set_color("#475569")
        ax.yaxis.label.set_color("#475569")
        for spine in ax.spines.values():
            spine.set_color("#e2e8f0")


        return tuple()


    animation = FuncAnimation(

        fig,

        render_frame,

        frames=len(
            frames
        ),

        interval=(
            frame_interval_ms
        ),

        repeat=False,

        blit=False
    )


    return (
        fig,
        animation
    )

In [ ]:
# @title Interactive NARAD-QC Demonstration Form


# ===========================================================================
# Input widgets
# ===========================================================================

demo_user_a_x = widgets.FloatText(

    value=0.0,

    description="User-A X:",

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="260px"
    )
)


demo_user_a_y = widgets.FloatText(

    value=0.0,

    description="User-A Y:",

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="260px"
    )
)


demo_user_b_x = widgets.FloatText(

    value=600.0,

    description="User-B X:",

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="260px"
    )
)


demo_user_b_y = widgets.FloatText(

    value=0.0,

    description="User-B Y:",

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="260px"
    )
)


demo_repeaters = widgets.Textarea(

    value=(
        "85.714, 0\n"
        "171.429, 0\n"
        "257.143, 0\n"
        "342.857, 0\n"
        "428.571, 0\n"
        "514.286, 0"
    ),

    description="Repeaters:",

    placeholder=(
        "One repeater per line, in path order:\n"
        "x,y\n"
        "x,y"
    ),

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="520px",
        height="175px"
    )
)


demo_policy = widgets.Dropdown(

    options=[

        (
            "Sequential Left-to-Right",
            "SEQUENTIAL_LEFT_TO_RIGHT"
        ),

        (
            "Static Balanced Tree",
            "STATIC_BALANCED_TREE"
        ),

        (
            "Max Parallel Oldest-First",
            "MAX_PARALLEL_OLDEST_FIRST"
        )
    ],

    value=(
        "MAX_PARALLEL_OLDEST_FIRST"
    ),

    description="Swap policy:",

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="420px"
    )
)


demo_seed = widgets.IntText(

    value=(
        NARAD_DEMO_DEFAULT_SEED
    ),

    description="Seed:",

    style={
        "description_width":
            "110px"
    },

    layout=widgets.Layout(
        width="260px"
    )
)


demo_run_button = widgets.Button(

    description="Run NARAD-QC Demo",

    button_style="success",

    icon="play",

    layout=widgets.Layout(
        width="230px",
        height="42px"
    )
)


demo_output = widgets.Output()


# ===========================================================================
# Click handler
# ===========================================================================

def narad_demo_on_run_clicked(
    button
):

    global NARAD_DEMO_ACTIVE_ANIMATION


    with demo_output:

        clear_output(
            wait=True
        )


        try:

            user_a_coordinate = (

                float(
                    demo_user_a_x.value
                ),

                float(
                    demo_user_a_y.value
                )
            )


            user_b_coordinate = (

                float(
                    demo_user_b_x.value
                ),

                float(
                    demo_user_b_y.value
                )
            )


            repeater_coordinates = (
                narad_demo_parse_repeater_coordinates(

                    demo_repeaters.value
                )
            )


            if len(
                repeater_coordinates
            ) < 1:

                raise ValueError(
                    "Enter at least one repeater coordinate."
                )


            print(
                "User-A:",
                user_a_coordinate
            )

            print(
                "Repeaters:",
                repeater_coordinates
            )

            print(
                "User-B:",
                user_b_coordinate
            )

            print(
                "Policy:",
                demo_policy.value
            )

            print()


            demo = (
                narad_demo_run(

                    user_a_coordinate=(
                        user_a_coordinate
                    ),

                    repeater_coordinates=(
                        repeater_coordinates
                    ),

                    user_b_coordinate=(
                        user_b_coordinate
                    ),

                    policy_name=(
                        demo_policy.value
                    ),

                    simulation_seed=(
                        int(
                            demo_seed.value
                        )
                    )
                )
            )


            NARAD_DEMO_ACTIVE_ANIMATION = (
                demo[
                    "animation"
                ]
            )


            animation_html = (
                NARAD_DEMO_ACTIVE_ANIMATION
                .to_jshtml()
            )


            plt.close(
                demo[
                    "figure"
                ]
            )


            display(
                HTML(
                    animation_html
                )
            )


        except Exception as exc:

            print(
                "NARAD-QC DEMO ERROR"
            )

            print(
                "-------------------"
            )

            print(
                type(
                    exc
                ).__name__
                + ":",
                exc
            )


demo_run_button.on_click(
    narad_demo_on_run_clicked
)


# ===========================================================================
# Form layout
# ===========================================================================

demo_heading = widgets.HTML(

    value="""
    <h3>NARAD-QC Interactive Entanglement Demonstrator</h3>

    <p>
    Coordinates are measured in kilometres:
    <b>1 coordinate unit = 1 km</b>.
    Repeater lines are interpreted in the exact order entered from
    User-A toward User-B.
    </p>

    <p>
    Every consecutive physical quantum link must satisfy
    <b>d ≤ 86 km</b>.
    </p>
    """
)


user_coordinate_box = widgets.HBox([

    widgets.VBox([
        demo_user_a_x,
        demo_user_a_y
    ]),

    widgets.VBox([
        demo_user_b_x,
        demo_user_b_y
    ])
])


demo_form = widgets.VBox([

    demo_heading,

    user_coordinate_box,

    demo_repeaters,

    demo_policy,

    demo_seed,

    demo_run_button,

    demo_output
])


display(
    demo_form
)

### Phase 3D : 30-09-2026T12:23:42 -- 05-10-2026T15:36:32
First failed attempt (RRI paper implementation) : 28-09-2026T13:05:33 -- 30-09-2026T11:24:53 <br>
Second attempt : 30-09-2026T12:23:42 -- 05-10-2026T15:36:32

In [ ]:
# @title QKD Quality Mathematics and Budget Configuration


from dataclasses import dataclass, asdict
from enum import Enum
import math

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# Bell-depolarized / isotropic scalar-state helpers
# ----------------------------------------------------------------

def narad_visibility_from_fidelity(fidelity: float) -> float:
    """
    Convert Bell-state target fidelity F into scalar visibility w:

        F = (1 + 3w) / 4

    Valid for the Bell-depolarized/isotropic scalar approximation
    used by the current NARAD-QC physical fidelity model.
    """

    fidelity = float(fidelity)

    if not math.isfinite(fidelity):
        raise ValueError("fidelity must be finite.")

    if fidelity < 0.25 - 1e-12 or fidelity > 1.0 + 1e-12:
        raise ValueError(
            "Current NARAD-QC Bell-depolarized fidelity must lie "
            "within [0.25, 1.0]."
        )

    visibility = (4.0 * fidelity - 1.0) / 3.0

    return float(
        min(
            1.0,
            max(
                0.0,
                visibility
            )
        )
    )


def narad_fidelity_from_visibility(visibility: float) -> float:

    visibility = float(visibility)

    if not math.isfinite(visibility):
        raise ValueError("visibility must be finite.")

    if visibility < -1e-12 or visibility > 1.0 + 1e-12:
        raise ValueError(
            "visibility must lie within [0, 1]."
        )

    visibility = min(
        1.0,
        max(
            0.0,
            visibility
        )
    )

    return float(
        (
            1.0
            +
            3.0 * visibility
        )
        /
        4.0
    )


def narad_qber_from_visibility(
    visibility: float
) -> float:
    """
    Same-basis bit-error proxy for the current symmetric
    Bell-depolarized state model:

        Q = (1 - w) / 2
    """

    visibility = float(visibility)

    if visibility < -1e-12 or visibility > 1.0 + 1e-12:
        raise ValueError(
            "visibility must lie within [0, 1]."
        )

    visibility = min(
        1.0,
        max(
            0.0,
            visibility
        )
    )

    return float(
        (
            1.0
            -
            visibility
        )
        /
        2.0
    )


def narad_qber_from_fidelity(
    fidelity: float
) -> float:

    return narad_qber_from_visibility(
        narad_visibility_from_fidelity(
            fidelity
        )
    )


def narad_quality_cost_from_visibility(
    visibility: float
) -> float:
    """
    C = -ln(w)

    Multiplicative visibility losses therefore become additive
    quality costs.
    """

    visibility = float(visibility)

    if visibility <= 0.0:
        return math.inf

    if visibility > 1.0 + 1e-12:
        raise ValueError(
            "visibility cannot exceed 1."
        )

    visibility = min(
        1.0,
        visibility
    )

    return float(
        -math.log(
            visibility
        )
    )


# ----------------------------------------------------------------
# QKD quality configuration
# ----------------------------------------------------------------

@dataclass(frozen=True)
class QKDQualityBudgetConfig:

    # Existing NARAD-QC BB84 acceptance limit
    qber_limit: float = 0.11

    # Current physical-fidelity model
    elementary_fidelity: float = 0.995

    # Current Phase-3C4 assumption:
    # ideal local swapping operation quality, while existence
    # remains stochastic through p_b separately.
    swap_operation_visibility: float = 1.0

    # Current experiment clock mapping
    slot_duration_s: float = 0.05

    # Memory model calibration
    bell_fidelity_floor: float = 0.25

    memory_reference_initial_fidelity: float = 1.0
    memory_reference_fidelity: float = 0.90
    memory_reference_time_s: float = 1.0

    numerical_tolerance: float = 1e-12


    def __post_init__(self):

        if not (
            0.0
            <= self.qber_limit
            < 0.5
        ):
            raise ValueError(
                "qber_limit must lie within [0, 0.5)."
            )

        if not (
            0.25
            <= self.elementary_fidelity
            <= 1.0
        ):
            raise ValueError(
                "elementary_fidelity must lie within [0.25, 1]."
            )

        if not (
            0.0
            <
            self.swap_operation_visibility
            <=
            1.0
        ):
            raise ValueError(
                "swap_operation_visibility must lie within (0, 1]."
            )

        if self.slot_duration_s <= 0.0:
            raise ValueError(
                "slot_duration_s must be positive."
            )

        if self.memory_reference_time_s <= 0.0:
            raise ValueError(
                "memory_reference_time_s must be positive."
            )


    @property
    def minimum_end_to_end_visibility(self):

        return float(
            1.0
            -
            2.0 * self.qber_limit
        )


    @property
    def minimum_end_to_end_fidelity(self):

        return narad_fidelity_from_visibility(
            self.minimum_end_to_end_visibility
        )


    @property
    def maximum_quality_cost(self):

        return narad_quality_cost_from_visibility(
            self.minimum_end_to_end_visibility
        )


    @property
    def elementary_visibility(self):

        return narad_visibility_from_fidelity(
            self.elementary_fidelity
        )


    @property
    def elementary_quality_cost(self):

        return narad_quality_cost_from_visibility(
            self.elementary_visibility
        )


    @property
    def swap_quality_cost(self):

        return narad_quality_cost_from_visibility(
            self.swap_operation_visibility
        )


    @property
    def memory_decay_rate_per_s(self):
        """
        Derive lambda directly from the same reference calibration
        used by the current WernerPhysicalFidelityProfile.

        Because:

            w(t) = w(0) exp(-lambda t)
        """

        w0 = narad_visibility_from_fidelity(
            self.memory_reference_initial_fidelity
        )

        w1 = narad_visibility_from_fidelity(
            self.memory_reference_fidelity
        )

        if w0 <= 0.0 or w1 <= 0.0:
            raise ValueError(
                "Reference fidelities must produce positive visibility."
            )

        ratio = w1 / w0

        if ratio <= 0.0 or ratio > 1.0:
            raise ValueError(
                "Memory reference fidelity must not improve with time."
            )

        return float(
            -math.log(
                ratio
            )
            /
            self.memory_reference_time_s
        )


    @property
    def memory_quality_cost_per_slot(self):

        return float(
            self.memory_decay_rate_per_s
            *
            self.slot_duration_s
        )


PHASE3H1_QUALITY_CONFIG = (
    QKDQualityBudgetConfig()
)

In [ ]:
# @title QKD Resource / Frontier Quality Assessment

class QKDResourceQualityState(Enum):

    INTERMEDIATE_VIABLE = "INTERMEDIATE_VIABLE"

    INTERMEDIATE_QUALITY_EXPIRED = (
        "INTERMEDIATE_QUALITY_EXPIRED"
    )

    END_TO_END_QKD_READY = (
        "END_TO_END_QKD_READY"
    )

    END_TO_END_QKD_REJECT = (
        "END_TO_END_QKD_REJECT"
    )


class QKDFrontierQualityState(Enum):

    VIABLE = "VIABLE"

    COLLECTIVELY_DOOMED = (
        "COLLECTIVELY_DOOMED"
    )


@dataclass(frozen=True)
class QKDResourceQualityAssessment:

    resource_id: str

    simulation_slot: int

    source_link_count: int

    span_start_link_index: int

    span_end_link_index: int

    missing_elementary_links: int

    remaining_successful_swaps_best_case: int

    current_fidelity: float

    current_visibility: float

    current_qber_proxy: float

    current_quality_cost: float

    minimum_future_quality_cost: float

    optimistic_total_quality_cost: float

    quality_cost_limit: float

    remaining_quality_slack: float

    optimistic_final_visibility: float

    optimistic_final_fidelity: float

    optimistic_final_qber: float

    optimistic_additional_wait_slots: float

    is_full_path_resource: bool

    can_still_reach_qkd_target: bool

    qkd_eligible_now: bool

    quality_expired: bool

    state: QKDResourceQualityState


@dataclass(frozen=True)
class QKDFrontierQualityAssessment:

    simulation_slot: int

    active_resource_count: int

    covered_elementary_links: int

    uncovered_elementary_links: int

    remaining_successful_swaps_best_case: int

    current_resource_quality_cost: float

    fresh_uncovered_link_quality_cost: float

    future_swap_quality_cost: float

    optimistic_total_quality_cost: float

    quality_cost_limit: float

    remaining_quality_slack: float

    optimistic_final_visibility: float

    optimistic_final_fidelity: float

    optimistic_final_qber: float

    remaining_quality_resource_slots: float

    can_still_reach_qkd_target: bool

    state: QKDFrontierQualityState


class QKDQualityBudgetEvaluator:
    """
    Read-only QKD quality evaluator.

    IMPORTANT:
    This class does NOT discard resources.
    It only tells the rest of NARAD-QC whether keeping a
    resource still makes physical/QKD sense.

    Resource test:
        Optimistic lower-cost completion assuming every missing
        elementary resource becomes available immediately and fresh.

    Therefore:

        If even the optimistic completion violates the QBER target,
        the resource is provably useless for this route under the
        current scalar fidelity model.

    Frontier test:
        Uses all currently active, non-overlapping resources together
        and assumes every uncovered physical link arrives fresh.
    """

    def __init__(
        self,
        topology,
        ordered_path,
        config=PHASE3H1_QUALITY_CONFIG
    ):

        self.topology = topology

        self.ordered_path = tuple(
            ordered_path
        )

        self.config = config


        if len(
            self.ordered_path
        ) < 2:

            raise ValueError(
                "ordered_path must contain at least two nodes."
            )


        # Validate against authoritative topology.
        self.topology.validate_path(
            self.ordered_path
        )


        self.path_links = tuple(
            self.topology.path_links(
                self.ordered_path
            )
        )


        self.path_link_ids = tuple(
            link.link_id

            for link
            in self.path_links
        )


        self.link_index_by_id = {

            link_id: index

            for index, link_id
            in enumerate(
                self.path_link_ids
            )
        }


        self.elementary_link_count = len(
            self.path_links
        )


        self.source_name = (
            self.ordered_path[0].name
        )

        self.destination_name = (
            self.ordered_path[-1].name
        )


    # ------------------------------------------------------------
    # Internal span validation
    # ------------------------------------------------------------

    def _resource_span(
        self,
        resource
    ):

        source_link_ids = tuple(
            resource.source_link_ids
        )


        if not source_link_ids:

            raise ValueError(
                f"{resource.resource_id} has no physical-link provenance."
            )


        if len(
            source_link_ids
        ) != len(
            set(
                source_link_ids
            )
        ):

            raise ValueError(
                f"{resource.resource_id} contains duplicate "
                "physical-link provenance."
            )


        try:

            link_indices = sorted(

                self.link_index_by_id[
                    link_id
                ]

                for link_id
                in source_link_ids
            )

        except KeyError as exc:

            raise ValueError(
                f"{resource.resource_id} contains a source link "
                "outside the evaluated path."
            ) from exc


        start_index = (
            link_indices[0]
        )

        end_index = (
            link_indices[-1]
        )


        expected_indices = list(
            range(
                start_index,
                end_index + 1
            )
        )


        if (
            link_indices
            != expected_indices
        ):

            raise ValueError(
                f"{resource.resource_id} does not represent a "
                "contiguous path interval."
            )


        expected_link_ids = {

            self.path_link_ids[index]

            for index
            in expected_indices
        }


        if (
            set(
                source_link_ids
            )
            != expected_link_ids
        ):

            raise ValueError(
                f"{resource.resource_id} provenance does not "
                "exactly match its path interval."
            )


        expected_endpoint_names = {

            self.ordered_path[
                start_index
            ].name,

            self.ordered_path[
                end_index + 1
            ].name
        }


        actual_endpoint_names = set(
            resource.endpoint_names
        )


        if (
            actual_endpoint_names
            != expected_endpoint_names
        ):

            raise ValueError(
                f"{resource.resource_id} endpoint/provenance "
                "inconsistency."
            )


        return (
            start_index,
            end_index,
            len(
                expected_indices
            )
        )


    # ------------------------------------------------------------
    # Single-resource optimistic viability
    # ------------------------------------------------------------

    def assess_resource(
        self,
        resource,
        current_fidelity,
        simulation_slot
    ):

        (
            span_start,
            span_end,
            span_link_count
        ) = self._resource_span(
            resource
        )


        current_fidelity = float(
            current_fidelity
        )


        current_visibility = (
            narad_visibility_from_fidelity(
                current_fidelity
            )
        )


        current_qber_proxy = (
            narad_qber_from_visibility(
                current_visibility
            )
        )


        current_quality_cost = (
            narad_quality_cost_from_visibility(
                current_visibility
            )
        )


        missing_links = (
            self.elementary_link_count
            -
            span_link_count
        )


        if missing_links < 0:

            raise RuntimeError(
                "Resource spans more links than the evaluated path."
            )


        # A contiguous resource spanning s leaves already contains
        # s-1 successful internal swaps.
        #
        # Completing an N-link path therefore still requires N-s
        # successful merges.
        remaining_swaps = (
            missing_links
        )


        minimum_future_quality_cost = (

            missing_links
            *
            self.config.elementary_quality_cost

            +

            remaining_swaps
            *
            self.config.swap_quality_cost
        )


        optimistic_total_quality_cost = (

            current_quality_cost
            +
            minimum_future_quality_cost
        )


        cost_limit = (
            self.config.maximum_quality_cost
        )


        remaining_slack = (

            cost_limit
            -
            optimistic_total_quality_cost
        )


        if math.isinf(
            optimistic_total_quality_cost
        ):

            optimistic_final_visibility = 0.0

        else:

            optimistic_final_visibility = float(
                math.exp(
                    -optimistic_total_quality_cost
                )
            )


        optimistic_final_fidelity = (
            narad_fidelity_from_visibility(
                optimistic_final_visibility
            )
        )


        optimistic_final_qber = (
            narad_qber_from_visibility(
                optimistic_final_visibility
            )
        )


        can_reach_target = bool(

            optimistic_final_qber
            <=
            (
                self.config.qber_limit
                +
                self.config.numerical_tolerance
            )
        )


        is_full_path_resource = bool(

            span_link_count
            ==
            self.elementary_link_count

            and

            set(
                resource.endpoint_names
            )
            ==
            {
                self.source_name,
                self.destination_name
            }
        )


        qkd_eligible_now = bool(

            is_full_path_resource
            and
            current_qber_proxy
            <=
            (
                self.config.qber_limit
                +
                self.config.numerical_tolerance
            )
        )


        quality_expired = (
            not can_reach_target
        )


        cost_per_slot = (
            self.config.memory_quality_cost_per_slot
        )


        if remaining_slack < 0.0:

            optimistic_wait_slots = 0.0

        elif cost_per_slot == 0.0:

            optimistic_wait_slots = math.inf

        else:

            optimistic_wait_slots = float(
                max(
                    0,
                    math.floor(
                        (
                            remaining_slack
                            /
                            cost_per_slot
                        )
                        +
                        1e-12
                    )
                )
            )


        if is_full_path_resource:

            if qkd_eligible_now:

                state = (
                    QKDResourceQualityState
                    .END_TO_END_QKD_READY
                )

            else:

                state = (
                    QKDResourceQualityState
                    .END_TO_END_QKD_REJECT
                )

        else:

            if can_reach_target:

                state = (
                    QKDResourceQualityState
                    .INTERMEDIATE_VIABLE
                )

            else:

                state = (
                    QKDResourceQualityState
                    .INTERMEDIATE_QUALITY_EXPIRED
                )


        return QKDResourceQualityAssessment(

            resource_id=(
                resource.resource_id
            ),

            simulation_slot=int(
                simulation_slot
            ),

            source_link_count=(
                span_link_count
            ),

            span_start_link_index=(
                span_start
            ),

            span_end_link_index=(
                span_end
            ),

            missing_elementary_links=(
                missing_links
            ),

            remaining_successful_swaps_best_case=(
                remaining_swaps
            ),

            current_fidelity=(
                current_fidelity
            ),

            current_visibility=(
                current_visibility
            ),

            current_qber_proxy=(
                current_qber_proxy
            ),

            current_quality_cost=(
                current_quality_cost
            ),

            minimum_future_quality_cost=(
                minimum_future_quality_cost
            ),

            optimistic_total_quality_cost=(
                optimistic_total_quality_cost
            ),

            quality_cost_limit=(
                cost_limit
            ),

            remaining_quality_slack=(
                remaining_slack
            ),

            optimistic_final_visibility=(
                optimistic_final_visibility
            ),

            optimistic_final_fidelity=(
                optimistic_final_fidelity
            ),

            optimistic_final_qber=(
                optimistic_final_qber
            ),

            optimistic_additional_wait_slots=(
                optimistic_wait_slots
            ),

            is_full_path_resource=(
                is_full_path_resource
            ),

            can_still_reach_qkd_target=(
                can_reach_target
            ),

            qkd_eligible_now=(
                qkd_eligible_now
            ),

            quality_expired=(
                quality_expired
            ),

            state=(
                state
            )
        )


    # ------------------------------------------------------------
    # Complete active-frontier optimistic viability
    # ------------------------------------------------------------

    def assess_frontier(
        self,
        active_resources,
        fidelity_lookup,
        simulation_slot
    ):

        active_resources = tuple(
            active_resources
        )


        covered_indices = set()

        current_resource_quality_cost = 0.0


        for resource in active_resources:

            (
                start_index,
                end_index,
                _
            ) = self._resource_span(
                resource
            )


            resource_indices = set(
                range(
                    start_index,
                    end_index + 1
                )
            )


            overlap = (
                covered_indices
                &
                resource_indices
            )


            if overlap:

                raise RuntimeError(
                    "Active frontier contains overlapping "
                    "entanglement provenance."
                )


            covered_indices.update(
                resource_indices
            )


            fidelity = float(
                fidelity_lookup(
                    resource
                )
            )


            visibility = (
                narad_visibility_from_fidelity(
                    fidelity
                )
            )


            current_resource_quality_cost += (
                narad_quality_cost_from_visibility(
                    visibility
                )
            )


        covered_count = len(
            covered_indices
        )


        uncovered_count = (
            self.elementary_link_count
            -
            covered_count
        )


        if uncovered_count < 0:

            raise RuntimeError(
                "Active frontier covers more links than exist."
            )


        fresh_uncovered_cost = (

            uncovered_count
            *
            self.config.elementary_quality_cost
        )


        # Current active resources count as existing components.
        # Every uncovered elementary link is optimistically assumed
        # to become one fresh component immediately.
        component_count = (

            len(
                active_resources
            )
            +
            uncovered_count
        )


        remaining_swaps = max(
            0,
            component_count - 1
        )


        future_swap_cost = (

            remaining_swaps
            *
            self.config.swap_quality_cost
        )


        optimistic_total_cost = (

            current_resource_quality_cost
            +
            fresh_uncovered_cost
            +
            future_swap_cost
        )


        cost_limit = (
            self.config.maximum_quality_cost
        )


        remaining_slack = (

            cost_limit
            -
            optimistic_total_cost
        )


        if math.isinf(
            optimistic_total_cost
        ):

            optimistic_visibility = 0.0

        else:

            optimistic_visibility = float(
                math.exp(
                    -optimistic_total_cost
                )
            )


        optimistic_fidelity = (
            narad_fidelity_from_visibility(
                optimistic_visibility
            )
        )


        optimistic_qber = (
            narad_qber_from_visibility(
                optimistic_visibility
            )
        )


        can_reach_target = bool(

            optimistic_qber
            <=
            (
                self.config.qber_limit
                +
                self.config.numerical_tolerance
            )
        )


        cost_per_slot = (
            self.config.memory_quality_cost_per_slot
        )


        # IMPORTANT:
        # This is a RESOURCE-SLOT budget, not necessarily wall-clock
        # slots. If 4 active resources each wait for one global slot,
        # approximately 4 resource-slots of budget are consumed.
        if remaining_slack < 0.0:

            remaining_resource_slots = 0.0

        elif cost_per_slot == 0.0:

            remaining_resource_slots = math.inf

        else:

            remaining_resource_slots = float(
                remaining_slack
                /
                cost_per_slot
            )


        state = (

            QKDFrontierQualityState.VIABLE

            if can_reach_target

            else

            QKDFrontierQualityState
            .COLLECTIVELY_DOOMED
        )


        return QKDFrontierQualityAssessment(

            simulation_slot=int(
                simulation_slot
            ),

            active_resource_count=len(
                active_resources
            ),

            covered_elementary_links=(
                covered_count
            ),

            uncovered_elementary_links=(
                uncovered_count
            ),

            remaining_successful_swaps_best_case=(
                remaining_swaps
            ),

            current_resource_quality_cost=float(
                current_resource_quality_cost
            ),

            fresh_uncovered_link_quality_cost=float(
                fresh_uncovered_cost
            ),

            future_swap_quality_cost=float(
                future_swap_cost
            ),

            optimistic_total_quality_cost=float(
                optimistic_total_cost
            ),

            quality_cost_limit=float(
                cost_limit
            ),

            remaining_quality_slack=float(
                remaining_slack
            ),

            optimistic_final_visibility=float(
                optimistic_visibility
            ),

            optimistic_final_fidelity=float(
                optimistic_fidelity
            ),

            optimistic_final_qber=float(
                optimistic_qber
            ),

            remaining_quality_resource_slots=float(
                remaining_resource_slots
            ),

            can_still_reach_qkd_target=(
                can_reach_target
            ),

            state=(
                state
            )
        )

In [ ]:
# @title Analytical Verification — 600 km / 6 Repeaters


phase3h1_cfg = (
    PHASE3H1_QUALITY_CONFIG
)


PHASE3H1_REPEATER_COUNT = 6

PHASE3H1_ELEMENTARY_LINK_COUNT = (
    PHASE3H1_REPEATER_COUNT
    +
    1
)


w_min = (
    phase3h1_cfg
    .minimum_end_to_end_visibility
)


f_min = (
    phase3h1_cfg
    .minimum_end_to_end_fidelity
)


c_max = (
    phase3h1_cfg
    .maximum_quality_cost
)


w_elem = (
    phase3h1_cfg
    .elementary_visibility
)


c_elem = (
    phase3h1_cfg
    .elementary_quality_cost
)


c_swap = (
    phase3h1_cfg
    .swap_quality_cost
)


lambda_memory = (
    phase3h1_cfg
    .memory_decay_rate_per_s
)


cost_per_slot = (
    phase3h1_cfg
    .memory_quality_cost_per_slot
)


# ----------------------------------------------------------------
# Completely fresh 7-link path
# ----------------------------------------------------------------

fresh_path_cost = (

    PHASE3H1_ELEMENTARY_LINK_COUNT
    *
    c_elem

    +

    (
        PHASE3H1_ELEMENTARY_LINK_COUNT
        -
        1
    )
    *
    c_swap
)


fresh_path_visibility = math.exp(
    -fresh_path_cost
)


fresh_path_fidelity = (
    narad_fidelity_from_visibility(
        fresh_path_visibility
    )
)


fresh_path_qber = (
    narad_qber_from_visibility(
        fresh_path_visibility
    )
)


remaining_memory_budget = (

    c_max
    -
    fresh_path_cost
)


equivalent_resource_slots = (

    remaining_memory_budget
    /
    cost_per_slot
)


# ----------------------------------------------------------------
# Verify the fresh-elementary optimistic quality deadline.
#
# This is NOT a universal 28-slot memory lifetime.
#
# Meaning:
# If ONE fresh elementary resource in this seven-link path
# were the only resource accumulating age, while all other six
# links appeared fresh and all future swaps added q_swap=1 quality,
# it has about 28 slots of optimistic remaining usefulness.
# ----------------------------------------------------------------

optimistic_wait_floor = int(
    math.floor(
        equivalent_resource_slots
        +
        1e-12
    )
)


deadline_rows = []


for age_slots in (
    optimistic_wait_floor,
    optimistic_wait_floor + 1
):

    aged_elementary_visibility = (

        w_elem
        *
        math.exp(
            -lambda_memory
            *
            phase3h1_cfg.slot_duration_s
            *
            age_slots
        )
    )


    optimistic_final_visibility = (

        aged_elementary_visibility

        *
        (
            w_elem
            **
            (
                PHASE3H1_ELEMENTARY_LINK_COUNT
                -
                1
            )
        )

        *
        (
            phase3h1_cfg
            .swap_operation_visibility

            **
            (
                PHASE3H1_ELEMENTARY_LINK_COUNT
                -
                1
            )
        )
    )


    deadline_rows.append({

        "ElementaryAgeSlots":
            age_slots,

        "ElementaryAgeSeconds":
            (
                age_slots
                *
                phase3h1_cfg.slot_duration_s
            ),

        "OptimisticFinalFidelity":
            narad_fidelity_from_visibility(
                optimistic_final_visibility
            ),

        "OptimisticFinalQBER":
            narad_qber_from_visibility(
                optimistic_final_visibility
            ),

        "PassesQKDTarget":
            (
                narad_qber_from_visibility(
                    optimistic_final_visibility
                )
                <=
                phase3h1_cfg.qber_limit
            )
    })


phase3h1_deadline_df = pd.DataFrame(
    deadline_rows
)


phase3h1_baseline_df = pd.DataFrame(
    [
        {
            "Metric":
                "QBER limit",

            "Value":
                phase3h1_cfg.qber_limit
        },

        {
            "Metric":
                "Required final visibility",

            "Value":
                w_min
        },

        {
            "Metric":
                "Required final fidelity",

            "Value":
                f_min
        },

        {
            "Metric":
                "Maximum quality cost",

            "Value":
                c_max
        },

        {
            "Metric":
                "Elementary fidelity",

            "Value":
                phase3h1_cfg.elementary_fidelity
        },

        {
            "Metric":
                "Elementary visibility",

            "Value":
                w_elem
        },

        {
            "Metric":
                "Elementary quality cost",

            "Value":
                c_elem
        },

        {
            "Metric":
                "Memory decay lambda / s",

            "Value":
                lambda_memory
        },

        {
            "Metric":
                "Memory quality cost / slot",

            "Value":
                cost_per_slot
        },

        {
            "Metric":
                "Fresh 7-link final fidelity",

            "Value":
                fresh_path_fidelity
        },

        {
            "Metric":
                "Fresh 7-link final QBER",

            "Value":
                fresh_path_qber
        },

        {
            "Metric":
                "Fresh-path quality cost",

            "Value":
                fresh_path_cost
        },

        {
            "Metric":
                "Remaining memory quality budget",

            "Value":
                remaining_memory_budget
        },

        {
            "Metric":
                "Equivalent one-resource wait slots",

            "Value":
                equivalent_resource_slots
        }
    ]
)


# ----------------------------------------------------------------
# Assertions
# ----------------------------------------------------------------

assert math.isclose(
    w_min,
    0.78,
    rel_tol=0.0,
    abs_tol=1e-12
)


assert math.isclose(
    f_min,
    0.835,
    rel_tol=0.0,
    abs_tol=1e-12
)


assert fresh_path_qber < (
    phase3h1_cfg.qber_limit
)


assert remaining_memory_budget > 0.0


assert (
    phase3h1_deadline_df
    .iloc[0][
        "PassesQKDTarget"
    ]
    == True
)


assert (
    phase3h1_deadline_df
    .iloc[1][
        "PassesQKDTarget"
    ]
    == False
)


print(
    "=" * 72
)

print(
    "QKD Quality-Budget Analytical Verification"
)

print(
    "=" * 72
)


display(
    phase3h1_baseline_df
)


print(
    "\nOptimistic fresh-elementary deadline boundary:"
)


display(
    phase3h1_deadline_df
)


print(
    "[PASS] QBER = 0.11 corresponds to F_min = 0.835 "
    "under the current Bell-depolarized model."
)

print(
    "[PASS] Completely fresh seven-link route is QKD-viable."
)

print(
    "[PASS] Quality cost remains additive under the current "
    "visibility model."
)

print(
    "[PASS] Optimistic single-resource quality deadline "
    "crosses between the two tested slots."
)

In [ ]:
# @title Live NARAD-QC Quality Snapshot


def phase3h1_active_resources(
    simulator
):

    active = (
        simulator
        .entanglement_manager
        .active_resources
    )

    if callable(
        active
    ):
        active = active()

    return tuple(
        active
    )


def phase3h1_current_fidelity(
    fidelity_registry,
    resource
):
    """
    Uses the current Phase-3C4 registry as the single authority
    for dynamic Bell-pair fidelity.
    """

    return float(
        fidelity_registry.fidelity(
            resource.resource_id
        )
    )


def phase3h1_build_evaluator_from_stack(
    stack,
    config=PHASE3H1_QUALITY_CONFIG
):

    return QKDQualityBudgetEvaluator(

        topology=(
            stack[
                "topology"
            ]
        ),

        ordered_path=(
            stack[
                "path"
            ]
        ),

        config=config
    )


def phase3h1_quality_snapshot(
    stack,
    simulation_slot
):

    simulator = (
        stack[
            "simulator"
        ]
    )

    fidelity_registry = (
        stack[
            "fidelity_registry"
        ]
    )


    evaluator = (
        phase3h1_build_evaluator_from_stack(
            stack
        )
    )


    active_resources = (
        phase3h1_active_resources(
            simulator
        )
    )


    assessments = []


    for resource in active_resources:

        current_fidelity = (
            phase3h1_current_fidelity(

                fidelity_registry,
                resource
            )
        )


        assessment = (
            evaluator.assess_resource(

                resource=resource,

                current_fidelity=(
                    current_fidelity
                ),

                simulation_slot=(
                    simulation_slot
                )
            )
        )


        row = asdict(
            assessment
        )


        row[
            "state"
        ] = (
            assessment
            .state
            .value
        )


        assessments.append(
            row
        )


    resource_dataframe = pd.DataFrame(
        assessments
    )


    frontier_assessment = (
        evaluator.assess_frontier(

            active_resources=(
                active_resources
            ),

            fidelity_lookup=(
                lambda resource:
                phase3h1_current_fidelity(
                    fidelity_registry,
                    resource
                )
            ),

            simulation_slot=(
                simulation_slot
            )
        )
    )


    frontier_row = asdict(
        frontier_assessment
    )


    frontier_row[
        "state"
    ] = (
        frontier_assessment
        .state
        .value
    )


    frontier_dataframe = pd.DataFrame(
        [
            frontier_row
        ]
    )


    return (
        resource_dataframe,
        frontier_dataframe
    )

In [ ]:
# @title Safe Quality-Expiration Candidate Detection


def phase3h1_safe_quality_expiration_candidates(
    resource_quality_dataframe
):
    """
    Returns ONLY resources that individually fail the optimistic
    best-case completion test.

    These are safe candidates for QKD-quality-driven discard.

    A collectively doomed frontier is intentionally NOT enough
    to choose a victim here; that choice belongs to Phase 3H.2.
    """

    if resource_quality_dataframe.empty:

        return (
            resource_quality_dataframe
            .copy()
        )


    return (
        resource_quality_dataframe[

            resource_quality_dataframe[
                "quality_expired"
            ]
            == True

        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

In [ ]:
# @title Public Lifecycle Adapters

from dataclasses import dataclass, asdict
import inspect

import pandas as pd


# ----------------------------------------------------------------
# EntanglementManager public discard adapter
# ----------------------------------------------------------------

def phase3h1_discard_resource_via_public_api(
    manager,
    resource_id,
    reason
):
    """
    Use an existing public EntanglementManager discard method.

    This helper intentionally does NOT mutate:
        manager._resources
        QuantumMemory internals
        EntanglementResource fields

    Phase 3H must preserve EntanglementManager as lifecycle authority.
    """

    candidate_method_names = (
        "discard_resource",
        "discard_active_resource",
        "discard_entanglement_resource"
    )


    for method_name in candidate_method_names:

        method = getattr(
            manager,
            method_name,
            None
        )


        if not callable(method):
            continue


        signature = inspect.signature(
            method
        )


        parameters = signature.parameters


        # --------------------------------------------------------
        # Resolve resource identifier parameter
        # --------------------------------------------------------

        resource_parameter = None

        for candidate in (
            "resource_id",
            "entanglement_id",
            "resource_or_id"
        ):

            if candidate in parameters:

                resource_parameter = candidate

                break


        # --------------------------------------------------------
        # Resolve reason parameter
        # --------------------------------------------------------

        reason_parameter = None

        for candidate in (
            "reason",
            "termination_reason",
            "discard_reason"
        ):

            if candidate in parameters:

                reason_parameter = candidate

                break


        # Preferred: fully named invocation.
        if resource_parameter is not None:

            kwargs = {
                resource_parameter:
                    resource_id
            }


            if reason_parameter is not None:

                kwargs[
                    reason_parameter
                ] = reason


            return method(
                **kwargs
            )


        # --------------------------------------------------------
        # Conservative positional fallback
        # --------------------------------------------------------

        positional_parameters = [

            parameter

            for parameter
            in parameters.values()

            if parameter.kind
            in (
                inspect.Parameter.POSITIONAL_ONLY,
                inspect.Parameter.POSITIONAL_OR_KEYWORD
            )
        ]


        if len(
            positional_parameters
        ) >= 1:

            kwargs = {}


            if reason_parameter is not None:

                kwargs[
                    reason_parameter
                ] = reason


            return method(
                resource_id,
                **kwargs
            )


    # ------------------------------------------------------------
    # Refuse hidden-state mutation.
    # ------------------------------------------------------------

    public_names = sorted(

        name

        for name in dir(
            manager
        )

        if (
            not name.startswith("_")
            and
            (
                "discard" in name.lower()
                or
                "remove" in name.lower()
                or
                "terminate" in name.lower()
            )
        )
    )


    raise RuntimeError(
        "Phase 3H.1 could not locate a compatible PUBLIC "
        "EntanglementManager discard method.\n"
        f"Relevant public methods found: {public_names}\n"
        "Do not patch manager._resources directly."
    )


# ----------------------------------------------------------------
# Demand reconciliation adapter
# ----------------------------------------------------------------

def phase3h1_reconcile_demand_via_public_api(
    demand_controller,
    simulation_slot=None
):
    """
    Invoke the existing failure-aware demand reconciliation
    without assuming a private API.
    """

    candidate_names = (
        "reconcile",
        "reconcile_demand",
        "reconcile_demands"
    )


    for method_name in candidate_names:

        method = getattr(
            demand_controller,
            method_name,
            None
        )


        if not callable(method):
            continue


        signature = inspect.signature(
            method
        )


        parameters = signature.parameters


        kwargs = {}


        if (
            simulation_slot is not None
        ):

            for slot_name in (
                "simulation_slot",
                "current_slot",
                "slot"
            ):

                if slot_name in parameters:

                    kwargs[
                        slot_name
                    ] = int(
                        simulation_slot
                    )

                    break


        return method(
            **kwargs
        )


    relevant_names = sorted(

        name

        for name in dir(
            demand_controller
        )

        if (
            not name.startswith("_")
            and
            "recon" in name.lower()
        )
    )


    raise RuntimeError(
        "Could not locate the public demand-reconciliation API.\n"
        f"Relevant public methods found: {relevant_names}"
    )


# ----------------------------------------------------------------
# Small API inspection helper
# ----------------------------------------------------------------

def phase3h1_lifecycle_api_dataframe(
    manager
):

    rows = []


    for name in sorted(
        dir(
            manager
        )
    ):

        if name.startswith("_"):
            continue


        if not (
            "discard" in name.lower()
            or
            "remove" in name.lower()
            or
            "expire" in name.lower()
        ):
            continue


        attribute = getattr(
            manager,
            name
        )


        if callable(
            attribute
        ):

            try:

                signature = str(
                    inspect.signature(
                        attribute
                    )
                )

            except Exception:

                signature = "UNAVAILABLE"

        else:

            signature = "NON_CALLABLE"


        rows.append({

            "PublicMember":
                name,

            "Signature":
                signature
        })


    return pd.DataFrame(
        rows
    )

In [ ]:
# @title Lifecycle API Preflight

phase3h1_api_stack = (
    build_phase3g2_trial_stack(

        policy_name=(
            "MAX_PARALLEL_OLDEST_FIRST"
        ),

        repeater_count=6,

        simulation_seed=7_310_001
    )
)


display(

    phase3h1_lifecycle_api_dataframe(

        phase3h1_api_stack[
            "simulator"
        ]
        .entanglement_manager
    )
)

In [ ]:
# @title QKD Quality-Aware Resource Lifecycle Controller


PHASE3H1_QUALITY_EXPIRY_REASON = (
    "qkd_quality_expired"
)


@dataclass(frozen=True)
class QKDQualityExpirationRecord:

    simulation_slot: int

    resource_id: str

    resource_kind: str

    source_link_count: int

    fidelity_before_discard: float

    qber_proxy_before_discard: float

    optimistic_final_fidelity: float

    optimistic_final_qber: float

    remaining_quality_slack: float

    quality_cost_limit: float

    reason: str


class QKDQualityLifecycleController:
    """
    Conservative QKD-aware lifecycle controller.

    Authority split:

        QKDQualityBudgetEvaluator
            decides whether a resource is still useful.

        QKDQualityLifecycleController
            decides whether QKD-quality discard should be requested.

        EntanglementManager
            remains the sole owner of actual resource mutation.

    IMPORTANT:
        Only individually doomed resources are discarded in Phase 3H.1.

        A COLLECTIVELY_DOOMED frontier is recorded but NOT repaired
        here. Choosing the best victim belongs to Phase 3H.2.
    """

    def __init__(
        self,
        stack,
        config=PHASE3H1_QUALITY_CONFIG
    ):

        self.stack = stack

        self.simulator = (
            stack[
                "simulator"
            ]
        )

        self.manager = (
            self.simulator
            .entanglement_manager
        )

        self.fidelity_registry = (
            stack[
                "fidelity_registry"
            ]
        )

        self.config = config


        self.evaluator = (
            phase3h1_build_evaluator_from_stack(

                stack=stack,

                config=config
            )
        )


        self._expiration_history = []


    @property
    def expiration_history(
        self
    ):

        return tuple(
            self._expiration_history
        )


    def expiration_history_dataframe(
        self
    ):

        return pd.DataFrame(

            [
                asdict(
                    record
                )

                for record
                in self._expiration_history
            ]
        )


    # ------------------------------------------------------------
    # Read-only audit
    # ------------------------------------------------------------

    def audit(
        self,
        simulation_slot
    ):

        return (
            phase3h1_quality_snapshot(

                stack=self.stack,

                simulation_slot=(
                    simulation_slot
                )
            )
        )


    # ------------------------------------------------------------
    # Conservative mutation sweep
    # ------------------------------------------------------------

    def expire_individually_doomed_resources(
        self,
        simulation_slot,
        dry_run=False
    ):

        simulation_slot = int(
            simulation_slot
        )


        # ========================================================
        # BEFORE
        # ========================================================

        (
            resource_quality_before,
            frontier_quality_before
        ) = self.audit(
            simulation_slot
        )


        candidates = (
            phase3h1_safe_quality_expiration_candidates(

                resource_quality_before
            )
        )


        if not candidates.empty:

            candidates = (
                candidates
                .sort_values(
                    "resource_id"
                )
                .reset_index(
                    drop=True
                )
            )


        discard_records = []


        # ========================================================
        # MUTATION
        # ========================================================

        if not dry_run:

            for _, row in candidates.iterrows():

                resource_id = str(
                    row[
                        "resource_id"
                    ]
                )


                # Resource may have changed only if some other layer
                # mutated it after our snapshot. Verify before acting.
                resource = (
                    self.manager
                    .get_resource(
                        resource_id
                    )
                )


                status_value = getattr(
                    resource.status,
                    "value",
                    str(
                        resource.status
                    )
                )


                if (
                    status_value
                    !=
                    EntanglementStatus.ACTIVE.value
                ):

                    continue


                resource_kind = getattr(
                    resource.kind,
                    "value",
                    str(
                        resource.kind
                    )
                )


                reason = (

                    f"{PHASE3H1_QUALITY_EXPIRY_REASON}"
                    f"_qber_limit_"
                    f"{self.config.qber_limit:.6f}"
                )


                # -----------------------------------------------
                # EntanglementManager remains lifecycle authority.
                # -----------------------------------------------

                phase3h1_discard_resource_via_public_api(

                    manager=self.manager,

                    resource_id=resource_id,

                    reason=reason
                )


                # -----------------------------------------------
                # Verify resulting historical state.
                # -----------------------------------------------

                discarded_resource = (
                    self.manager
                    .get_resource(
                        resource_id
                    )
                )


                discarded_status = getattr(
                    discarded_resource.status,
                    "value",
                    str(
                        discarded_resource.status
                    )
                )


                if (
                    discarded_status
                    !=
                    EntanglementStatus.DISCARDED.value
                ):

                    raise RuntimeError(
                        f"{resource_id} was requested for QKD-quality "
                        "discard but did not enter DISCARDED state."
                    )


                record = (
                    QKDQualityExpirationRecord(

                        simulation_slot=(
                            simulation_slot
                        ),

                        resource_id=(
                            resource_id
                        ),

                        resource_kind=(
                            resource_kind
                        ),

                        source_link_count=int(
                            row[
                                "source_link_count"
                            ]
                        ),

                        fidelity_before_discard=float(
                            row[
                                "current_fidelity"
                            ]
                        ),

                        qber_proxy_before_discard=float(
                            row[
                                "current_qber_proxy"
                            ]
                        ),

                        optimistic_final_fidelity=float(
                            row[
                                "optimistic_final_fidelity"
                            ]
                        ),

                        optimistic_final_qber=float(
                            row[
                                "optimistic_final_qber"
                            ]
                        ),

                        remaining_quality_slack=float(
                            row[
                                "remaining_quality_slack"
                            ]
                        ),

                        quality_cost_limit=float(
                            row[
                                "quality_cost_limit"
                            ]
                        ),

                        reason=(
                            reason
                        )
                    )
                )


                self._expiration_history.append(
                    record
                )


                discard_records.append(
                    asdict(
                        record
                    )
                )


            # Validate memory/resource mirrors after every complete sweep.
            validator = getattr(
                self.manager,
                "validate_all_active_resources",
                None
            )


            if callable(
                validator
            ):

                validator()


            runtime_validator = getattr(
                self.simulator,
                "validate_runtime_consistency",
                None
            )


            if callable(
                runtime_validator
            ):

                validation_result = (
                    runtime_validator()
                )


                if (
                    validation_result
                    is not None
                    and
                    validation_result
                    is not True
                ):

                    raise RuntimeError(
                        "Runtime consistency failed after "
                        "QKD-quality expiration sweep."
                    )


        # ========================================================
        # AFTER
        # ========================================================

        if dry_run:

            resource_quality_after = (
                resource_quality_before
                .copy()
            )

            frontier_quality_after = (
                frontier_quality_before
                .copy()
            )

        else:

            (
                resource_quality_after,
                frontier_quality_after
            ) = self.audit(
                simulation_slot
            )


        return {

            "simulation_slot":
                simulation_slot,

            "dry_run":
                bool(
                    dry_run
                ),

            "candidate_count":
                int(
                    len(
                        candidates
                    )
                ),

            "discard_count":
                int(
                    len(
                        discard_records
                    )
                ),

            "candidates":
                candidates,

            "discard_records":
                pd.DataFrame(
                    discard_records
                ),

            "resource_quality_before":
                resource_quality_before,

            "frontier_quality_before":
                frontier_quality_before,

            "resource_quality_after":
                resource_quality_after,

            "frontier_quality_after":
                frontier_quality_after
        }

In [ ]:
# @title Integration Verification — Entanglement Success != QKD Success


# print(
#     "=" * 78
# )

# print("QKD Quality-Aware Lifecycle Integration Verification")

# print(
#     "=" * 78
# )


# # ----------------------------------------------------------------
# # Reproducible 600 km / six-repeater physical stack
# # ----------------------------------------------------------------

# phase3h1_live_stack = (
#     build_phase3g2_trial_stack(

#         policy_name=(
#             "MAX_PARALLEL_OLDEST_FIRST"
#         ),

#         repeater_count=6,

#         simulation_seed=4_020_002
#     )
# )


# phase3h1_live_result = (

#     phase3h1_live_stack[
#         "orchestrator"
#     ]
#     .run_until_end_to_end(

#         max_slots=(
#             PHASE3G2_MAX_EXECUTION_SLOTS
#         )
#     )
# )


# assert (
#     phase3h1_live_result.success
# ), (
#     "The deterministic Phase-3H.1 integration seed did not "
#     "produce an ordinary end-to-end entanglement resource."
# )


# manager = (

#     phase3h1_live_stack[
#         "simulator"
#     ]
#     .entanglement_manager
# )


# final_resource = (

#     manager
#     .get_resource(

#         phase3h1_live_result
#         .end_to_end_resource_id
#     )
# )


# final_slot = int(
#     phase3h1_live_result.end_slot
# )


# final_fidelity = float(
#     phase3h1_live_result.final_fidelity
# )


# final_qber_proxy = (
#     narad_qber_from_fidelity(
#         final_fidelity
#     )
# )


# print(
#     "\nBASELINE DELIVERY RESULT"
# )

# print(
#     "Resource:",
#     final_resource.resource_id
# )

# print(
#     "End slot:",
#     final_slot
# )

# print(
#     "Final fidelity:",
#     final_fidelity
# )

# print(
#     "Predicted QBER proxy:",
#     final_qber_proxy
# )

# print(
#     "Ordinary entanglement success:",
#     phase3h1_live_result.success
# )


# # ----------------------------------------------------------------
# # New QKD-aware layer
# # ----------------------------------------------------------------

# phase3h1_lifecycle = (
#     QKDQualityLifecycleController(

#         stack=(
#             phase3h1_live_stack
#         )
#     )
# )


# (
#     resource_quality_before,
#     frontier_quality_before
# ) = (
#     phase3h1_lifecycle
#     .audit(
#         simulation_slot=(
#             final_slot
#         )
#     )
# )


# print(
#     "\nQKD QUALITY ASSESSMENT BEFORE MUTATION"
# )


# display(
#     resource_quality_before
# )


# display(
#     frontier_quality_before
# )


# # ----------------------------------------------------------------
# # The final resource should exist but be QKD-rejected.
# # ----------------------------------------------------------------

# assert (
#     len(
#         resource_quality_before
#     )
#     ==
#     1
# )


# final_assessment = (
#     resource_quality_before
#     .iloc[0]
# )


# assert (
#     final_assessment[
#         "is_full_path_resource"
#     ]
#     == True
# )


# assert (
#     final_assessment[
#         "qkd_eligible_now"
#     ]
#     == False
# )


# assert (
#     final_assessment[
#         "quality_expired"
#     ]
#     == True
# )


# assert (
#     final_assessment[
#         "state"
#     ]
#     ==
#     QKDResourceQualityState
#     .END_TO_END_QKD_REJECT
#     .value
# )


# # ----------------------------------------------------------------
# # First prove dry-run does NOT mutate anything.
# # ----------------------------------------------------------------

# dry_run = (
#     phase3h1_lifecycle
#     .expire_individually_doomed_resources(

#         simulation_slot=(
#             final_slot
#         ),

#         dry_run=True
#     )
# )


# assert (
#     dry_run[
#         "candidate_count"
#     ]
#     ==
#     1
# )


# assert (
#     dry_run[
#         "discard_count"
#     ]
#     ==
#     0
# )


# still_active = (
#     manager
#     .get_resource(
#         final_resource.resource_id
#     )
# )


# assert (
#     still_active.status
#     ==
#     EntanglementStatus.ACTIVE
# )


# print(
#     "[PASS] Dry-run identified the doomed QKD resource "
#     "without mutating the network."
# )


# # ----------------------------------------------------------------
# # Now execute the real quality-expiration sweep.
# # ----------------------------------------------------------------

# live_sweep = (
#     phase3h1_lifecycle
#     .expire_individually_doomed_resources(

#         simulation_slot=(
#             final_slot
#         ),

#         dry_run=False
#     )
# )


# assert (
#     live_sweep[
#         "discard_count"
#     ]
#     ==
#     1
# )


# discarded_resource = (
#     manager
#     .get_resource(
#         final_resource.resource_id
#     )
# )


# assert (
#     discarded_resource.status
#     ==
#     EntanglementStatus.DISCARDED
# )


# assert (
#     len(
#         phase3h1_active_resources(
#             phase3h1_live_stack[
#                 "simulator"
#             ]
#         )
#     )
#     ==
#     0
# )


# print(
#     "[PASS] QKD-ineligible end-to-end resource was "
#     "discarded through EntanglementManager."
# )


# # ----------------------------------------------------------------
# # Existing failure-aware demand logic must react naturally.
# # ----------------------------------------------------------------

# reconciliation = (
#     phase3h1_reconcile_demand_via_public_api(

#         demand_controller=(
#             phase3h1_live_stack[
#                 "demand_controller"
#             ]
#         ),

#         simulation_slot=(
#             final_slot
#         )
#     )
# )


# regeneration_link_ids = tuple(

#     getattr(
#         reconciliation,
#         "regeneration_link_ids"
#     )
# )


# assert (
#     len(
#         regeneration_link_ids
#     )
#     ==
#     7
# )


# assert (
#     set(
#         regeneration_link_ids
#     )
#     ==
#     set(

#         link.link_id

#         for link
#         in (
#             phase3h1_live_stack[
#                 "topology"
#             ]
#             .path_links(

#                 phase3h1_live_stack[
#                     "path"
#                 ]
#             )
#         )
#     )
# )


# print(
#     "[PASS] Discarding the unusable full-path resource "
#     "reactivated exactly its seven underlying elementary links."
# )


# # ----------------------------------------------------------------
# # Cross-layer integrity
# # ----------------------------------------------------------------

# assert (
#     phase3h1_live_stack[
#         "simulator"
#     ]
#     .validate_runtime_consistency()
#     is True
# )


# print(
#     "[PASS] Runtime remains cross-layer consistent."
# )


# print(
#     "\nQUALITY EXPIRATION HISTORY"
# )


# display(
#     phase3h1_lifecycle
#     .expiration_history_dataframe()
# )


# print(
#     "\nFRONTIER AFTER QKD-QUALITY DISCARD"
# )


# display(
#     live_sweep[
#         "frontier_quality_after"
#     ]
# )


# print(
#     "\n" + "=" * 78
# )

# print(
#     "Quality-aware lifecycle integration PASSED."
# )

# print(
#     "=" * 78
# )

In [ ]:
# @title Exact Minimum-Cost Collective QKD Quality Recovery Planner

from dataclasses import dataclass, asdict
from enum import Enum
from itertools import combinations
from typing import Optional
import math

import pandas as pd


# ================================================================
# Recovery Status
# ================================================================

class QKDCollectiveRecoveryStatus(str, Enum):

    NOT_REQUIRED = (
        "NOT_REQUIRED"
    )

    INDIVIDUAL_EXPIRY_REQUIRED = (
        "INDIVIDUAL_EXPIRY_REQUIRED"
    )

    PLAN_FOUND = (
        "PLAN_FOUND"
    )

    NO_FEASIBLE_RECOVERY = (
        "NO_FEASIBLE_RECOVERY"
    )


# ================================================================
# One Hypothetical Recovery Candidate
# ================================================================

@dataclass(frozen=True)
class QKDCollectiveRecoveryCandidate:

    simulation_slot: int

    discarded_resource_ids: tuple[str, ...]
    preserved_resource_ids: tuple[str, ...]

    discarded_resource_count: int

    incremental_regeneration_link_ids: tuple[str, ...]
    total_regeneration_link_ids: tuple[str, ...]

    incremental_regeneration_link_count: int
    total_regeneration_link_count: int

    incremental_expected_attempt_work: float
    total_expected_attempt_work: float

    discarded_current_quality_cost: float
    fresh_replacement_quality_cost: float
    estimated_quality_cost_recovered: float

    optimistic_total_quality_cost_after_repair: float
    quality_cost_limit: float
    remaining_quality_slack_after_repair: float

    optimistic_final_visibility_after_repair: float
    optimistic_final_fidelity_after_repair: float
    optimistic_final_qber_after_repair: float

    remaining_quality_resource_slots_after_repair: float

    feasible: bool


# ================================================================
# Full Planning Result
# ================================================================

@dataclass(frozen=True)
class QKDCollectiveRecoveryPlan:

    simulation_slot: int

    status: QKDCollectiveRecoveryStatus

    active_resource_ids: tuple[str, ...]

    individually_doomed_resource_ids: tuple[str, ...]

    current_frontier_state: str

    current_optimistic_final_fidelity: float
    current_optimistic_final_qber: float
    current_remaining_quality_slack: float

    candidate_count: int
    feasible_candidate_count: int

    chosen_candidate: Optional[
        QKDCollectiveRecoveryCandidate
    ]

    candidates: tuple[
        QKDCollectiveRecoveryCandidate, ...
    ]


# ================================================================
# Planner
# ================================================================

class QKDMinimumCostCollectiveRecoveryPlanner:
    """
    Exact read-only planner for a collectively QKD-doomed frontier.

    Important Phase-3H.2A rule
    --------------------------
    This class NEVER mutates entanglement state.

    It asks:

        "Which currently ACTIVE resource subset could be discarded
         and regenerated fresh so that the optimistic frontier once
         again satisfies the QKD target?"

    Objective
    ---------
    Primary:
        Minimize incremental expected elementary-generation work.

    Secondary:
        Minimize number of elementary links regenerated.

    Tertiary:
        Minimize number of discarded entanglement resources.

    Quaternary:
        Prefer larger restored quality slack.

    Final:
        Deterministic lexicographic resource-ID tie-break.

    Current elementary work proxy
    -----------------------------
        expected attempts on link l = 1 / p_l

    where p_l is the already-defined physical elementary-generation
    success probability proxy for that PhysicalQuantumLink.

    This is NOT wall-clock latency.
    """

    def __init__(
        self,
        stack,
        config=PHASE3H1_QUALITY_CONFIG,
        max_exact_frontier_resources=16
    ):

        self.stack = stack

        self.simulator = (
            stack[
                "simulator"
            ]
        )

        self.manager = (
            self.simulator
            .entanglement_manager
        )

        self.topology = (
            stack[
                "topology"
            ]
        )

        self.ordered_path = tuple(
            stack[
                "path"
            ]
        )

        self.fidelity_registry = (
            stack[
                "fidelity_registry"
            ]
        )

        self.config = config

        self.max_exact_frontier_resources = int(
            max_exact_frontier_resources
        )


        if (
            self.max_exact_frontier_resources
            < 1
        ):

            raise ValueError(
                "max_exact_frontier_resources must be positive."
            )


        self.evaluator = (
            QKDQualityBudgetEvaluator(

                topology=(
                    self.topology
                ),

                ordered_path=(
                    self.ordered_path
                ),

                config=(
                    self.config
                )
            )
        )


        self.path_links = tuple(
            self.topology
            .path_links(
                self.ordered_path
            )
        )


        self._link_by_id = {

            link.link_id:
                link

            for link
            in self.path_links
        }


        self._link_index_by_id = {

            link.link_id:
                index

            for index, link
            in enumerate(
                self.path_links
            )
        }


    # ============================================================
    # Current Fidelity
    # ============================================================

    def _current_fidelity(
        self,
        resource
    ) -> float:

        return float(
            self.fidelity_registry
            .fidelity(
                resource.resource_id
            )
        )


    # ============================================================
    # Elementary Regeneration Work Proxy
    # ============================================================

    def _link_expected_attempt_work(
        self,
        link_id
    ) -> float:

        link = (
            self._link_by_id[
                link_id
            ]
        )


        probability = float(
            link.survival_probability
        )


        if probability <= 0.0:

            return math.inf


        return float(
            1.0
            /
            probability
        )


    def _expected_attempt_work(
        self,
        link_ids
    ) -> float:

        return float(
            sum(

                self._link_expected_attempt_work(
                    link_id
                )

                for link_id
                in link_ids
            )
        )


    # ============================================================
    # Deterministic Physical-Link Ordering
    # ============================================================

    def _ordered_link_ids(
        self,
        link_ids
    ) -> tuple[str, ...]:

        link_ids = set(
            link_ids
        )


        return tuple(

            link.link_id

            for link
            in self.path_links

            if (
                link.link_id
                in link_ids
            )
        )


    # ============================================================
    # Candidate Ordering
    # ============================================================

    @staticmethod
    def _candidate_sort_key(
        candidate
    ):

        return (

            candidate
            .incremental_expected_attempt_work,

            candidate
            .incremental_regeneration_link_count,

            candidate
            .discarded_resource_count,

            -candidate
            .remaining_quality_slack_after_repair,

            candidate
            .discarded_resource_ids
        )


    # ============================================================
    # Main Planner
    # ============================================================

    def plan(
        self
    ) -> QKDCollectiveRecoveryPlan:

        simulation_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        # --------------------------------------------------------
        # Read current Phase-3H.1 quality state.
        # --------------------------------------------------------

        (
            resource_quality_df,
            frontier_quality_df
        ) = (
            phase3h1_quality_snapshot(

                stack=(
                    self.stack
                ),

                simulation_slot=(
                    simulation_slot
                )
            )
        )


        if frontier_quality_df.empty:

            raise RuntimeError(
                "QKD frontier assessment unexpectedly returned "
                "no frontier row."
            )


        active_resources = tuple(
            self.manager
            .active_resources
        )


        active_resource_ids = tuple(
            sorted(

                resource.resource_id

                for resource
                in active_resources
            )
        )


        # --------------------------------------------------------
        # Detect resources that Phase 3H.1 should handle first.
        # --------------------------------------------------------

        if resource_quality_df.empty:

            individually_doomed_ids = ()

        else:

            individually_doomed_ids = tuple(
                sorted(

                    resource_quality_df.loc[

                        resource_quality_df[
                            "quality_expired"
                        ]
                        == True,

                        "resource_id"
                    ]
                    .astype(
                        str
                    )
                    .tolist()
                )
            )


        frontier_row = (
            frontier_quality_df
            .iloc[0]
        )


        current_frontier_state = str(
            frontier_row[
                "state"
            ]
        )


        current_frontier_viable = bool(
            frontier_row[
                "can_still_reach_qkd_target"
            ]
        )


        # --------------------------------------------------------
        # Case 1:
        # Phase 3H.1 still has deterministic work to do.
        # --------------------------------------------------------

        if individually_doomed_ids:

            return QKDCollectiveRecoveryPlan(

                simulation_slot=(
                    simulation_slot
                ),

                status=(
                    QKDCollectiveRecoveryStatus
                    .INDIVIDUAL_EXPIRY_REQUIRED
                ),

                active_resource_ids=(
                    active_resource_ids
                ),

                individually_doomed_resource_ids=(
                    individually_doomed_ids
                ),

                current_frontier_state=(
                    current_frontier_state
                ),

                current_optimistic_final_fidelity=float(
                    frontier_row[
                        "optimistic_final_fidelity"
                    ]
                ),

                current_optimistic_final_qber=float(
                    frontier_row[
                        "optimistic_final_qber"
                    ]
                ),

                current_remaining_quality_slack=float(
                    frontier_row[
                        "remaining_quality_slack"
                    ]
                ),

                candidate_count=0,

                feasible_candidate_count=0,

                chosen_candidate=None,

                candidates=()
            )


        # --------------------------------------------------------
        # Case 2:
        # Current frontier already has enough QKD quality.
        # --------------------------------------------------------

        if current_frontier_viable:

            return QKDCollectiveRecoveryPlan(

                simulation_slot=(
                    simulation_slot
                ),

                status=(
                    QKDCollectiveRecoveryStatus
                    .NOT_REQUIRED
                ),

                active_resource_ids=(
                    active_resource_ids
                ),

                individually_doomed_resource_ids=(),

                current_frontier_state=(
                    current_frontier_state
                ),

                current_optimistic_final_fidelity=float(
                    frontier_row[
                        "optimistic_final_fidelity"
                    ]
                ),

                current_optimistic_final_qber=float(
                    frontier_row[
                        "optimistic_final_qber"
                    ]
                ),

                current_remaining_quality_slack=float(
                    frontier_row[
                        "remaining_quality_slack"
                    ]
                ),

                candidate_count=0,

                feasible_candidate_count=0,

                chosen_candidate=None,

                candidates=()
            )


        # --------------------------------------------------------
        # Exact subset search guard.
        # --------------------------------------------------------

        if (
            len(
                active_resources
            )
            >
            self.max_exact_frontier_resources
        ):

            raise RuntimeError(
                "Exact collective-recovery subset search would "
                f"require 2^{len(active_resources)} combinations. "
                "Increase max_exact_frontier_resources deliberately "
                "or introduce a later heuristic."
            )


        # --------------------------------------------------------
        # Validate current frontier and establish baseline
        # uncovered physical links.
        # --------------------------------------------------------

        covered_now = set()


        current_quality_cost_by_resource = {}


        for resource in active_resources:

            assessment = (
                self.evaluator
                .assess_resource(

                    resource=(
                        resource
                    ),

                    current_fidelity=(
                        self._current_fidelity(
                            resource
                        )
                    ),

                    simulation_slot=(
                        simulation_slot
                    )
                )
            )


            current_quality_cost_by_resource[
                resource.resource_id
            ] = (
                assessment
                .current_quality_cost
            )


            for link_id in (
                resource
                .source_link_ids
            ):

                if link_id in covered_now:

                    raise RuntimeError(
                        "Active frontier contains overlapping "
                        "physical-link provenance."
                    )

                covered_now.add(
                    link_id
                )


        baseline_uncovered_link_ids = (
            self._ordered_link_ids(

                set(
                    self._link_by_id
                )
                -
                covered_now
            )
        )


        candidates = []


        # ========================================================
        # Exact Non-Empty Discard-Subset Enumeration
        # ========================================================

        for subset_size in range(
            1,
            len(
                active_resources
            )
            + 1
        ):

            for discarded_resources in combinations(
                active_resources,
                subset_size
            ):

                discarded_resource_ids = tuple(
                    sorted(

                        resource.resource_id

                        for resource
                        in discarded_resources
                    )
                )


                discarded_id_set = set(
                    discarded_resource_ids
                )


                preserved_resources = tuple(

                    resource

                    for resource
                    in active_resources

                    if (
                        resource.resource_id
                        not in discarded_id_set
                    )
                )


                preserved_resource_ids = tuple(
                    sorted(

                        resource.resource_id

                        for resource
                        in preserved_resources
                    )
                )


                # ------------------------------------------------
                # Physical links exposed by this hypothetical
                # discard action.
                # ------------------------------------------------

                incremental_regeneration_link_set = set()


                for resource in discarded_resources:

                    incremental_regeneration_link_set.update(
                        resource.source_link_ids
                    )


                incremental_regeneration_link_ids = (
                    self._ordered_link_ids(
                        incremental_regeneration_link_set
                    )
                )


                total_regeneration_link_ids = (
                    self._ordered_link_ids(

                        set(
                            baseline_uncovered_link_ids
                        )
                        |
                        incremental_regeneration_link_set
                    )
                )


                # ------------------------------------------------
                # Hypothetical repaired frontier:
                #
                # preserve surviving resources,
                # assume every uncovered link reappears FRESH.
                # ------------------------------------------------

                repaired_frontier = (
                    self.evaluator
                    .assess_frontier(

                        active_resources=(
                            preserved_resources
                        ),

                        fidelity_lookup=(

                            lambda resource:
                            self._current_fidelity(
                                resource
                            )
                        ),

                        simulation_slot=(
                            simulation_slot
                        )
                    )
                )


                incremental_work = (
                    self._expected_attempt_work(

                        incremental_regeneration_link_ids
                    )
                )


                total_work = (
                    self._expected_attempt_work(

                        total_regeneration_link_ids
                    )
                )


                discarded_current_quality_cost = float(
                    sum(

                        current_quality_cost_by_resource[
                            resource.resource_id
                        ]

                        for resource
                        in discarded_resources
                    )
                )


                fresh_replacement_quality_cost = float(

                    len(
                        incremental_regeneration_link_ids
                    )

                    *
                    self.config
                    .elementary_quality_cost
                )


                quality_cost_recovered = float(

                    discarded_current_quality_cost
                    -
                    fresh_replacement_quality_cost
                )


                candidate = (
                    QKDCollectiveRecoveryCandidate(

                        simulation_slot=(
                            simulation_slot
                        ),

                        discarded_resource_ids=(
                            discarded_resource_ids
                        ),

                        preserved_resource_ids=(
                            preserved_resource_ids
                        ),

                        discarded_resource_count=len(
                            discarded_resource_ids
                        ),

                        incremental_regeneration_link_ids=(
                            incremental_regeneration_link_ids
                        ),

                        total_regeneration_link_ids=(
                            total_regeneration_link_ids
                        ),

                        incremental_regeneration_link_count=len(
                            incremental_regeneration_link_ids
                        ),

                        total_regeneration_link_count=len(
                            total_regeneration_link_ids
                        ),

                        incremental_expected_attempt_work=float(
                            incremental_work
                        ),

                        total_expected_attempt_work=float(
                            total_work
                        ),

                        discarded_current_quality_cost=(
                            discarded_current_quality_cost
                        ),

                        fresh_replacement_quality_cost=(
                            fresh_replacement_quality_cost
                        ),

                        estimated_quality_cost_recovered=(
                            quality_cost_recovered
                        ),

                        optimistic_total_quality_cost_after_repair=(
                            repaired_frontier
                            .optimistic_total_quality_cost
                        ),

                        quality_cost_limit=(
                            repaired_frontier
                            .quality_cost_limit
                        ),

                        remaining_quality_slack_after_repair=(
                            repaired_frontier
                            .remaining_quality_slack
                        ),

                        optimistic_final_visibility_after_repair=(
                            repaired_frontier
                            .optimistic_final_visibility
                        ),

                        optimistic_final_fidelity_after_repair=(
                            repaired_frontier
                            .optimistic_final_fidelity
                        ),

                        optimistic_final_qber_after_repair=(
                            repaired_frontier
                            .optimistic_final_qber
                        ),

                        remaining_quality_resource_slots_after_repair=(
                            repaired_frontier
                            .remaining_quality_resource_slots
                        ),

                        feasible=(
                            repaired_frontier
                            .can_still_reach_qkd_target
                        )
                    )
                )


                candidates.append(
                    candidate
                )


        candidates = tuple(
            candidates
        )


        feasible_candidates = tuple(

            candidate

            for candidate
            in candidates

            if candidate.feasible
        )


        # --------------------------------------------------------
        # No amount of simple regeneration can save this path.
        #
        # This happens, for example, if even a completely fresh
        # route violates the QKD quality threshold.
        # --------------------------------------------------------

        if not feasible_candidates:

            return QKDCollectiveRecoveryPlan(

                simulation_slot=(
                    simulation_slot
                ),

                status=(
                    QKDCollectiveRecoveryStatus
                    .NO_FEASIBLE_RECOVERY
                ),

                active_resource_ids=(
                    active_resource_ids
                ),

                individually_doomed_resource_ids=(),

                current_frontier_state=(
                    current_frontier_state
                ),

                current_optimistic_final_fidelity=float(
                    frontier_row[
                        "optimistic_final_fidelity"
                    ]
                ),

                current_optimistic_final_qber=float(
                    frontier_row[
                        "optimistic_final_qber"
                    ]
                ),

                current_remaining_quality_slack=float(
                    frontier_row[
                        "remaining_quality_slack"
                    ]
                ),

                candidate_count=len(
                    candidates
                ),

                feasible_candidate_count=0,

                chosen_candidate=None,

                candidates=(
                    candidates
                )
            )


        # --------------------------------------------------------
        # Select exact minimum-cost feasible repair.
        # --------------------------------------------------------

        chosen_candidate = min(

            feasible_candidates,

            key=(
                self._candidate_sort_key
            )
        )


        return QKDCollectiveRecoveryPlan(

            simulation_slot=(
                simulation_slot
            ),

            status=(
                QKDCollectiveRecoveryStatus
                .PLAN_FOUND
            ),

            active_resource_ids=(
                active_resource_ids
            ),

            individually_doomed_resource_ids=(),

            current_frontier_state=(
                current_frontier_state
            ),

            current_optimistic_final_fidelity=float(
                frontier_row[
                    "optimistic_final_fidelity"
                ]
            ),

            current_optimistic_final_qber=float(
                frontier_row[
                    "optimistic_final_qber"
                ]
            ),

            current_remaining_quality_slack=float(
                frontier_row[
                    "remaining_quality_slack"
                ]
            ),

            candidate_count=len(
                candidates
            ),

            feasible_candidate_count=len(
                feasible_candidates
            ),

            chosen_candidate=(
                chosen_candidate
            ),

            candidates=(
                candidates
            )
        )


    # ============================================================
    # Inspection
    # ============================================================

    def candidates_dataframe(
        self,
        plan,
        feasible_only=False
    ) -> pd.DataFrame:

        rows = []


        for candidate in (
            plan.candidates
        ):

            if (
                feasible_only
                and
                not candidate.feasible
            ):

                continue


            rows.append({

                "Discard Resources":
                    candidate
                    .discarded_resource_ids,

                "Preserve Resources":
                    candidate
                    .preserved_resource_ids,

                "Discard Count":
                    candidate
                    .discarded_resource_count,

                "Regenerate Links":
                    candidate
                    .incremental_regeneration_link_ids,

                "Regenerate Link Count":
                    candidate
                    .incremental_regeneration_link_count,

                "Incremental Expected Attempts":
                    candidate
                    .incremental_expected_attempt_work,

                "Quality Cost Recovered":
                    candidate
                    .estimated_quality_cost_recovered,

                "Optimistic Final Cost":
                    candidate
                    .optimistic_total_quality_cost_after_repair,

                "Remaining Slack":
                    candidate
                    .remaining_quality_slack_after_repair,

                "Optimistic Final Fidelity":
                    candidate
                    .optimistic_final_fidelity_after_repair,

                "Optimistic Final QBER":
                    candidate
                    .optimistic_final_qber_after_repair,

                "Remaining Quality Resource-Slots":
                    candidate
                    .remaining_quality_resource_slots_after_repair,

                "Feasible":
                    candidate
                    .feasible
            })


        dataframe = pd.DataFrame(
            rows
        )


        if not dataframe.empty:

            dataframe = (
                dataframe
                .sort_values(

                    by=[
                        "Feasible",
                        "Incremental Expected Attempts",
                        "Regenerate Link Count",
                        "Discard Count",
                        "Remaining Slack"
                    ],

                    ascending=[
                        False,
                        True,
                        True,
                        True,
                        False
                    ]
                )
                .reset_index(
                    drop=True
                )
            )


        return dataframe

In [ ]:
# @title Individually Viable Resources, Collectively Doomed Frontier

print(
    "=" * 86
)

print(
    "Minimum-Cost Collective Quality Recovery Verification"
)

print(
    "=" * 86
)


# ----------------------------------------------------------------
# Fresh 600-km / six-repeater stack
# ----------------------------------------------------------------

phase3h2a_stack = (
    build_phase3g2_trial_stack(

        policy_name=(
            "MAX_PARALLEL_OLDEST_FIRST"
        ),

        repeater_count=6,

        simulation_seed=7_320_001
    )
)


simulator = (
    phase3h2a_stack[
        "simulator"
    ]
)


manager = (
    simulator
    .entanglement_manager
)


fidelity_registry = (
    phase3h2a_stack[
        "fidelity_registry"
    ]
)


path_links = tuple(
    phase3h2a_stack[
        "topology"
    ]
    .path_links(

        phase3h2a_stack[
            "path"
        ]
    )
)


assert (
    len(
        path_links
    )
    ==
    7
)


# ----------------------------------------------------------------
# Create exactly one fresh elementary Bell pair on every link.
#
# No swapping.
# No stochastic generation.
#
# This isolates ONLY the collective quality-recovery logic.
# ----------------------------------------------------------------

created_resources = []


for link in path_links:

    resource = (
        manager
        .create_elementary_resource(
            link
        )
    )


    fidelity_registry.register_elementary(
        resource
    )


    created_resources.append(
        resource
    )


assert (
    len(
        manager.active_resources
    )
    ==
    7
)


manager.validate_all_active_resources()


assert (
    simulator.validate_runtime_consistency()
    is True
)


# ----------------------------------------------------------------
# Age ALL seven active resources by five global slots.
#
# Why five?
#
# At our current model:
#
#   Fresh 7-link cost      ~= 0.046823
#   Memory cost/slot       ~= 0.007155 per active resource
#
# After 5 slots:
#
#   C_frontier
#   ~= 0.046823 + 7 * 5 * 0.007155
#   ~= 0.297249
#
# which exceeds:
#
#   C_max ~= 0.248461
#
# Yet every individual elementary resource remains easily
# salvageable if the other six were regenerated fresh.
# ----------------------------------------------------------------

simulator.kernel.advance_slot(
    5
)


current_slot = (
    simulator.kernel.current_slot
)


assert (
    current_slot
    ==
    5
)


# ----------------------------------------------------------------
# Verify Phase 3H.1 diagnosis first.
# ----------------------------------------------------------------

(
    resource_quality_df,
    frontier_quality_df
) = (
    phase3h1_quality_snapshot(

        stack=(
            phase3h2a_stack
        ),

        simulation_slot=(
            current_slot
        )
    )
)


print(
    "\nINDIVIDUAL RESOURCE QUALITY"
)


display(
    resource_quality_df[
        [
            "resource_id",
            "current_fidelity",
            "current_qber_proxy",
            "remaining_quality_slack",
            "optimistic_final_qber",
            "quality_expired",
            "state"
        ]
    ]
)


print(
    "\nCOLLECTIVE FRONTIER QUALITY"
)


display(
    frontier_quality_df
)


# ----------------------------------------------------------------
# Every resource is individually viable.
# ----------------------------------------------------------------

assert (
    resource_quality_df[
        "quality_expired"
    ]
    .sum()
    ==
    0
)


assert all(

    resource_quality_df[
        "state"
    ]
    ==
    QKDResourceQualityState
    .INTERMEDIATE_VIABLE
    .value
)


print(
    "[PASS] Every five-slot-old elementary resource remains "
    "individually QKD-viable."
)


# ----------------------------------------------------------------
# But the seven together exceed the QKD quality budget.
# ----------------------------------------------------------------

frontier_row = (
    frontier_quality_df
    .iloc[0]
)


assert (
    frontier_row[
        "can_still_reach_qkd_target"
    ]
    == False
)


assert (
    frontier_row[
        "state"
    ]
    ==
    QKDFrontierQualityState
    .COLLECTIVELY_DOOMED
    .value
)


print(
    "[PASS] The complete active frontier is collectively QKD-doomed."
)


# ----------------------------------------------------------------
# Planner
# ----------------------------------------------------------------

planner = (
    QKDMinimumCostCollectiveRecoveryPlanner(

        stack=(
            phase3h2a_stack
        )
    )
)


plan = (
    planner
    .plan()
)


assert (
    plan.status
    ==
    QKDCollectiveRecoveryStatus
    .PLAN_FOUND
)


assert (
    plan.chosen_candidate
    is not None
)


chosen = (
    plan
    .chosen_candidate
)


print(
    "\nCHOSEN MINIMUM-COST REPAIR"
)


display(
    pd.DataFrame(
        [
            asdict(
                chosen
            )
        ]
    )
)


# ----------------------------------------------------------------
# Expected exact result:
#
# One replacement is not enough.
# Two fresh replacements ARE enough.
# ----------------------------------------------------------------

assert (
    chosen
    .discarded_resource_count
    ==
    2
)


assert (
    chosen
    .incremental_regeneration_link_count
    ==
    2
)


assert (
    len(
        chosen
        .preserved_resource_ids
    )
    ==
    5
)


assert (
    chosen
    .feasible
    is True
)


assert (
    chosen
    .optimistic_final_qber_after_repair
    <=
    PHASE3H1_QUALITY_CONFIG
    .qber_limit
)


print(
    "[PASS] Exact planner replaces only two of seven resources."
)


# ----------------------------------------------------------------
# Verify that EVERY one-resource replacement is still insufficient.
# ----------------------------------------------------------------

one_resource_candidates = tuple(

    candidate

    for candidate
    in plan.candidates

    if (
        candidate
        .discarded_resource_count
        ==
        1
    )
)


assert (
    len(
        one_resource_candidates
    )
    ==
    7
)


assert all(

    not candidate.feasible

    for candidate
    in one_resource_candidates
)


print(
    "[PASS] Replacing only one stale resource cannot restore "
    "the QKD target."
)


# ----------------------------------------------------------------
# Verify at least one two-resource repair works.
# ----------------------------------------------------------------

two_resource_feasible = tuple(

    candidate

    for candidate
    in plan.candidates

    if (
        candidate
        .discarded_resource_count
        ==
        2

        and

        candidate
        .feasible
    )
)


assert (
    len(
        two_resource_feasible
    )
    >
    0
)


print(
    "[PASS] Two-resource selective regeneration is sufficient."
)


# ----------------------------------------------------------------
# Physical-work validation.
#
# Equal 85.714-km links currently have equal survival probability.
# Therefore expected incremental work should be:
#
#     2 / p_elem
# ----------------------------------------------------------------

elementary_probability = float(

    phase3h2a_stack[
        "elementary_success_probability"
    ]
)


expected_two_link_work = (

    2.0
    /
    elementary_probability
)


assert math.isclose(

    chosen
    .incremental_expected_attempt_work,

    expected_two_link_work,

    rel_tol=1e-12,

    abs_tol=1e-12
)


print(
    "[PASS] Recovery cost equals the expected physical work "
    "of regenerating exactly two 85.714-km links."
)


# ----------------------------------------------------------------
# Read-only guarantee:
#
# Planner must not mutate a single resource.
# ----------------------------------------------------------------

active_after_planning = tuple(
    sorted(

        resource.resource_id

        for resource
        in manager.active_resources
    )
)


active_before_planning = tuple(
    sorted(

        resource.resource_id

        for resource
        in created_resources
    )
)


assert (
    active_after_planning
    ==
    active_before_planning
)


assert all(

    manager
    .get_resource(
        resource_id
    )
    .status
    ==
    EntanglementStatus.ACTIVE

    for resource_id
    in active_before_planning
)


manager.validate_all_active_resources()


assert (
    simulator.validate_runtime_consistency()
    is True
)


print(
    "[PASS] Recovery planning is completely read-only."
)


# ----------------------------------------------------------------
# Inspect best feasible candidates.
# ----------------------------------------------------------------

print(
    "\nTOP FEASIBLE RECOVERY CANDIDATES"
)

df1 = planner.candidates_dataframe(
    plan=plan,
    feasible_only=True
)

display(df1)

df1.to_csv("candidates.csv", index=False)

print(
    "\n" + "=" * 86
)

print(
    "Minimum-cost recovery verification PASSED."
)

print(
    "=" * 86
)

In [ ]:
# @title Selective Collective-Quality Recovery Lifecycle Controller

from dataclasses import dataclass, asdict
from enum import Enum
import math

import pandas as pd


PHASE3H2_COLLECTIVE_RECOVERY_REASON = (
    "qkd_collective_quality_recovery"
)


class QKDSelectiveRecoveryExecutionStatus(
    str,
    Enum
):

    NOT_REQUIRED = (
        "NOT_REQUIRED"
    )

    BLOCKED_BY_INDIVIDUAL_EXPIRY = (
        "BLOCKED_BY_INDIVIDUAL_EXPIRY"
    )

    NO_FEASIBLE_RECOVERY = (
        "NO_FEASIBLE_RECOVERY"
    )

    DRY_RUN_PLANNED = (
        "DRY_RUN_PLANNED"
    )

    EXECUTED = (
        "EXECUTED"
    )


@dataclass(frozen=True)
class QKDSelectiveRecoveryExecutionRecord:

    execution_index: int

    simulation_slot: int

    status: QKDSelectiveRecoveryExecutionStatus

    planner_status: QKDCollectiveRecoveryStatus

    dry_run: bool

    discarded_resource_ids: tuple[str, ...]

    preserved_resource_ids: tuple[str, ...]

    planned_regeneration_link_ids: tuple[str, ...]

    actual_regeneration_link_ids: tuple[str, ...]

    ready_elementary_link_ids: tuple[str, ...]

    protected_link_ids: tuple[str, ...]

    frontier_state_before: str

    frontier_state_after: str

    optimistic_final_qber_before: float

    optimistic_final_qber_after: float

    quality_slack_before: float

    quality_slack_after: float

    termination_reason: str | None


class QKDSelectiveRecoveryLifecycleController:
    """
    Execute an already-validated Phase-3H.2A collective recovery plan.

    Authority separation
    --------------------
    Planner:
        Decides WHICH ACTIVE resources should be sacrificed.

    This controller:
        Requests those sacrifices and verifies the intended result.

    EntanglementManager:
        Remains the ONLY authority that actually changes
        EntanglementResource / QuantumMemory lifecycle state.

    FailureAwareElementaryDemandController:
        Recomputes which physical elementary links now require
        regeneration.

    The controller itself:
        - does NOT clear quantum memories manually;
        - does NOT modify generation-process private state;
        - does NOT generate Bell pairs;
        - does NOT perform swapping;
        - does NOT advance simulation time.
    """

    def __init__(
        self,
        stack,
        planner=None,
        config=PHASE3H1_QUALITY_CONFIG
    ):

        self.stack = stack

        self.simulator = (
            stack[
                "simulator"
            ]
        )

        self.manager = (
            self.simulator
            .entanglement_manager
        )

        self.demand_controller = (
            stack[
                "demand_controller"
            ]
        )

        self.fidelity_registry = (
            stack[
                "fidelity_registry"
            ]
        )

        self.config = config


        if planner is None:

            planner = (
                QKDMinimumCostCollectiveRecoveryPlanner(

                    stack=stack,

                    config=config
                )
            )


        self.planner = planner

        self._history = []


    # ============================================================
    # History
    # ============================================================

    @property
    def history(
        self
    ):

        return tuple(
            self._history
        )


    def history_dataframe(
        self
    ):

        return pd.DataFrame(

            [
                {
                    **asdict(
                        record
                    ),

                    "status":
                        record.status.value,

                    "planner_status":
                        record.planner_status.value
                }

                for record
                in self._history
            ]
        )


    # ============================================================
    # Helpers
    # ============================================================

    def _active_resource_ids(
        self
    ) -> tuple[str, ...]:

        return tuple(
            sorted(

                resource.resource_id

                for resource
                in self.manager.active_resources
            )
        )


    @staticmethod
    def _frontier_row(
        frontier_dataframe
    ):

        if frontier_dataframe.empty:

            raise RuntimeError(
                "QKD frontier audit returned no row."
            )

        return (
            frontier_dataframe
            .iloc[0]
        )


    # ============================================================
    # Execution
    # ============================================================

    def execute(
        self,
        dry_run=False
    ):

        if not isinstance(
            dry_run,
            bool
        ):

            raise TypeError(
                "dry_run must be bool."
            )


        current_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        # --------------------------------------------------------
        # Planner + independent audit before any mutation.
        # --------------------------------------------------------

        plan = (
            self.planner
            .plan()
        )


        (
            resource_quality_before,
            frontier_quality_before
        ) = (
            phase3h1_quality_snapshot(

                stack=self.stack,

                simulation_slot=(
                    current_slot
                )
            )
        )


        frontier_before_row = (
            self._frontier_row(
                frontier_quality_before
            )
        )


        # --------------------------------------------------------
        # The plan must refer to exactly the state that exists now.
        # --------------------------------------------------------

        if (
            plan.simulation_slot
            != current_slot
        ):

            raise RuntimeError(
                "Collective recovery plan is stale: "
                "simulation slot changed before execution."
            )


        live_active_ids = (
            self._active_resource_ids()
        )


        if (
            live_active_ids
            != plan.active_resource_ids
        ):

            raise RuntimeError(
                "Collective recovery plan is stale: "
                "ACTIVE entanglement frontier changed."
            )


        # ========================================================
        # Planner says 3H.1 must act first.
        # ========================================================

        if (
            plan.status
            ==
            QKDCollectiveRecoveryStatus
            .INDIVIDUAL_EXPIRY_REQUIRED
        ):

            record = (
                QKDSelectiveRecoveryExecutionRecord(

                    execution_index=len(
                        self._history
                    ),

                    simulation_slot=(
                        current_slot
                    ),

                    status=(
                        QKDSelectiveRecoveryExecutionStatus
                        .BLOCKED_BY_INDIVIDUAL_EXPIRY
                    ),

                    planner_status=(
                        plan.status
                    ),

                    dry_run=(
                        dry_run
                    ),

                    discarded_resource_ids=(),

                    preserved_resource_ids=(
                        plan.active_resource_ids
                    ),

                    planned_regeneration_link_ids=(),

                    actual_regeneration_link_ids=(),

                    ready_elementary_link_ids=(),

                    protected_link_ids=(),

                    frontier_state_before=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    frontier_state_after=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    optimistic_final_qber_before=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    optimistic_final_qber_after=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    quality_slack_before=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    quality_slack_after=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    termination_reason=None
                )
            )


            self._history.append(
                record
            )


            return {

                "record":
                    record,

                "plan":
                    plan,

                "resource_quality_before":
                    resource_quality_before,

                "frontier_quality_before":
                    frontier_quality_before,

                "resource_quality_after":
                    resource_quality_before.copy(),

                "frontier_quality_after":
                    frontier_quality_before.copy(),

                "reconciliation":
                    None
            }


        # ========================================================
        # Nothing needs repairing.
        # ========================================================

        if (
            plan.status
            ==
            QKDCollectiveRecoveryStatus
            .NOT_REQUIRED
        ):

            record = (
                QKDSelectiveRecoveryExecutionRecord(

                    execution_index=len(
                        self._history
                    ),

                    simulation_slot=(
                        current_slot
                    ),

                    status=(
                        QKDSelectiveRecoveryExecutionStatus
                        .NOT_REQUIRED
                    ),

                    planner_status=(
                        plan.status
                    ),

                    dry_run=(
                        dry_run
                    ),

                    discarded_resource_ids=(),

                    preserved_resource_ids=(
                        plan.active_resource_ids
                    ),

                    planned_regeneration_link_ids=(),

                    actual_regeneration_link_ids=(),

                    ready_elementary_link_ids=(),

                    protected_link_ids=(),

                    frontier_state_before=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    frontier_state_after=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    optimistic_final_qber_before=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    optimistic_final_qber_after=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    quality_slack_before=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    quality_slack_after=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    termination_reason=None
                )
            )


            self._history.append(
                record
            )


            return {

                "record":
                    record,

                "plan":
                    plan,

                "resource_quality_before":
                    resource_quality_before,

                "frontier_quality_before":
                    frontier_quality_before,

                "resource_quality_after":
                    resource_quality_before.copy(),

                "frontier_quality_after":
                    frontier_quality_before.copy(),

                "reconciliation":
                    None
            }


        # ========================================================
        # No regeneration-only repair can restore QKD viability.
        # ========================================================

        if (
            plan.status
            ==
            QKDCollectiveRecoveryStatus
            .NO_FEASIBLE_RECOVERY
        ):

            record = (
                QKDSelectiveRecoveryExecutionRecord(

                    execution_index=len(
                        self._history
                    ),

                    simulation_slot=(
                        current_slot
                    ),

                    status=(
                        QKDSelectiveRecoveryExecutionStatus
                        .NO_FEASIBLE_RECOVERY
                    ),

                    planner_status=(
                        plan.status
                    ),

                    dry_run=(
                        dry_run
                    ),

                    discarded_resource_ids=(),

                    preserved_resource_ids=(
                        plan.active_resource_ids
                    ),

                    planned_regeneration_link_ids=(),

                    actual_regeneration_link_ids=(),

                    ready_elementary_link_ids=(),

                    protected_link_ids=(),

                    frontier_state_before=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    frontier_state_after=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    optimistic_final_qber_before=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    optimistic_final_qber_after=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    quality_slack_before=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    quality_slack_after=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    termination_reason=None
                )
            )


            self._history.append(
                record
            )


            return {

                "record":
                    record,

                "plan":
                    plan,

                "resource_quality_before":
                    resource_quality_before,

                "frontier_quality_before":
                    frontier_quality_before,

                "resource_quality_after":
                    resource_quality_before.copy(),

                "frontier_quality_after":
                    frontier_quality_before.copy(),

                "reconciliation":
                    None
            }


        # --------------------------------------------------------
        # From this point onward a concrete repair MUST exist.
        # --------------------------------------------------------

        if (
            plan.status
            !=
            QKDCollectiveRecoveryStatus
            .PLAN_FOUND
        ):

            raise RuntimeError(
                f"Unsupported collective recovery planner status: "
                f"{plan.status}"
            )


        candidate = (
            plan.chosen_candidate
        )


        if candidate is None:

            raise RuntimeError(
                "PLAN_FOUND has no chosen recovery candidate."
            )


        # --------------------------------------------------------
        # Full preflight before irreversible lifecycle mutation.
        # --------------------------------------------------------

        self.manager\
            .validate_all_active_resources()


        assert (
            self.simulator
            .validate_runtime_consistency()
            is True
        )


        chosen_ids = tuple(
            candidate
            .discarded_resource_ids
        )


        preserved_ids = tuple(
            candidate
            .preserved_resource_ids
        )


        for resource_id in chosen_ids:

            resource = (
                self.manager
                .get_resource(
                    resource_id
                )
            )

            if (
                resource.status
                !=
                EntanglementStatus.ACTIVE
            ):

                raise RuntimeError(
                    f"Planned victim '{resource_id}' "
                    "is no longer ACTIVE."
                )


        # Preserve exact object identity and current fidelity so we
        # can prove untouched resources actually remained untouched.
        preserved_objects = {

            resource_id:
                self.manager
                .get_resource(
                    resource_id
                )

            for resource_id
            in preserved_ids
        }


        preserved_fidelities = {

            resource_id:
                float(
                    self.fidelity_registry
                    .fidelity(
                        resource_id
                    )
                )

            for resource_id
            in preserved_ids
        }


        # ========================================================
        # Dry-run
        # ========================================================

        if dry_run:

            record = (
                QKDSelectiveRecoveryExecutionRecord(

                    execution_index=len(
                        self._history
                    ),

                    simulation_slot=(
                        current_slot
                    ),

                    status=(
                        QKDSelectiveRecoveryExecutionStatus
                        .DRY_RUN_PLANNED
                    ),

                    planner_status=(
                        plan.status
                    ),

                    dry_run=True,

                    discarded_resource_ids=(
                        chosen_ids
                    ),

                    preserved_resource_ids=(
                        preserved_ids
                    ),

                    planned_regeneration_link_ids=(
                        candidate
                        .total_regeneration_link_ids
                    ),

                    actual_regeneration_link_ids=(),

                    ready_elementary_link_ids=(),

                    protected_link_ids=(),

                    frontier_state_before=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    frontier_state_after=str(
                        frontier_before_row[
                            "state"
                        ]
                    ),

                    optimistic_final_qber_before=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    optimistic_final_qber_after=float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                    quality_slack_before=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    quality_slack_after=float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                    termination_reason=None
                )
            )


            self._history.append(
                record
            )


            return {

                "record":
                    record,

                "plan":
                    plan,

                "resource_quality_before":
                    resource_quality_before,

                "frontier_quality_before":
                    frontier_quality_before,

                "resource_quality_after":
                    resource_quality_before.copy(),

                "frontier_quality_after":
                    frontier_quality_before.copy(),

                "reconciliation":
                    None
            }


        # ========================================================
        # Real selective lifecycle mutation
        # ========================================================

        execution_index = len(
            self._history
        )


        termination_reason = (

            f"{PHASE3H2_COLLECTIVE_RECOVERY_REASON}"
            f"_execution_{execution_index:04d}"
            f"_qber_limit_{self.config.qber_limit:.6f}"
        )


        discarded_ids = []


        for resource_id in chosen_ids:

            discarded_resource = (
                self.manager
                .discard_resource(

                    resource_id=(
                        resource_id
                    ),

                    reason=(
                        termination_reason
                    )
                )
            )


            if (
                discarded_resource.status
                !=
                EntanglementStatus.DISCARDED
            ):

                raise RuntimeError(
                    f"Resource '{resource_id}' failed to enter "
                    "DISCARDED state."
                )


            discarded_ids.append(
                resource_id
            )


        discarded_ids = tuple(
            discarded_ids
        )


        # --------------------------------------------------------
        # Manager/local-memory state must still be valid before
        # demand is recalculated.
        # --------------------------------------------------------

        self.manager\
            .validate_all_active_resources()


        assert (
            self.simulator
            .validate_runtime_consistency()
            is True
        )


        # ========================================================
        # Recompute elementary generation demand from the ACTUAL
        # surviving frontier.
        # ========================================================

        reconciliation = (
            self.demand_controller
            .reconcile()
        )


        actual_regeneration_link_ids = tuple(
            reconciliation
            .regeneration_link_ids
        )


        expected_regeneration_link_ids = tuple(
            candidate
            .total_regeneration_link_ids
        )


        if (
            actual_regeneration_link_ids
            !=
            expected_regeneration_link_ids
        ):

            raise RuntimeError(
                "Selective recovery plan and failure-aware demand "
                "reconciliation disagree.\n"
                f"Planned regeneration: "
                f"{expected_regeneration_link_ids}\n"
                f"Actual regeneration: "
                f"{actual_regeneration_link_ids}"
            )


        # ========================================================
        # Prove untouched resources were genuinely untouched.
        # ========================================================

        active_after_ids = (
            self._active_resource_ids()
        )


        if (
            active_after_ids
            !=
            tuple(
                sorted(
                    preserved_ids
                )
            )
        ):

            raise RuntimeError(
                "Selective recovery affected resources outside "
                "the chosen discard set."
            )


        for resource_id in preserved_ids:

            resource_after = (
                self.manager
                .get_resource(
                    resource_id
                )
            )


            if (
                resource_after
                is not
                preserved_objects[
                    resource_id
                ]
            ):

                raise RuntimeError(
                    f"Preserved resource '{resource_id}' "
                    "was unexpectedly replaced."
                )


            fidelity_after = float(
                self.fidelity_registry
                .fidelity(
                    resource_id
                )
            )


            if not math.isclose(

                fidelity_after,

                preserved_fidelities[
                    resource_id
                ],

                rel_tol=0.0,

                abs_tol=1e-15
            ):

                raise RuntimeError(
                    f"Preserved resource '{resource_id}' "
                    "changed fidelity despite no time advancement."
                )


        # --------------------------------------------------------
        # This lifecycle operation must consume zero global slots.
        # --------------------------------------------------------

        if (
            self.simulator
            .kernel
            .current_slot
            !=
            current_slot
        ):

            raise RuntimeError(
                "Selective quality recovery illegally advanced "
                "simulation time."
            )


        # ========================================================
        # Independent QKD audit after mutation.
        #
        # Uncovered links are optimistically treated as fresh,
        # exactly as in the Phase-3H.2A planning model.
        # ========================================================

        (
            resource_quality_after,
            frontier_quality_after
        ) = (
            phase3h1_quality_snapshot(

                stack=self.stack,

                simulation_slot=(
                    current_slot
                )
            )
        )


        frontier_after_row = (
            self._frontier_row(
                frontier_quality_after
            )
        )


        if not bool(
            frontier_after_row[
                "can_still_reach_qkd_target"
            ]
        ):

            raise RuntimeError(
                "Executed recovery plan did not restore optimistic "
                "QKD frontier viability."
            )


        if (
            float(
                frontier_after_row[
                    "optimistic_final_qber"
                ]
            )
            >
            (
                self.config.qber_limit
                +
                self.config.numerical_tolerance
            )
        ):

            raise RuntimeError(
                "Executed recovery plan still exceeds QBER limit."
            )


        # Planner prediction must match the actually exposed
        # frontier after the chosen resources were removed.
        if not math.isclose(

            float(
                frontier_after_row[
                    "optimistic_final_qber"
                ]
            ),

            candidate
            .optimistic_final_qber_after_repair,

            rel_tol=0.0,

            abs_tol=1e-12
        ):

            raise RuntimeError(
                "Planner-predicted repaired QBER does not match "
                "actual post-discard frontier audit."
            )


        if not math.isclose(

            float(
                frontier_after_row[
                    "remaining_quality_slack"
                ]
            ),

            candidate
            .remaining_quality_slack_after_repair,

            rel_tol=0.0,

            abs_tol=1e-12
        ):

            raise RuntimeError(
                "Planner-predicted quality slack does not match "
                "actual post-discard frontier audit."
            )


        # ========================================================
        # Final consistency
        # ========================================================

        self.manager\
            .validate_all_active_resources()


        assert (
            self.simulator
            .validate_runtime_consistency()
            is True
        )


        # ========================================================
        # Audit event
        # ========================================================

        self.simulator.kernel.emit(

            event_type=(
                "QKD_COLLECTIVE_QUALITY_RECOVERY_EXECUTED"
            ),

            component=(
                "ClassicalControlPlane."
                "QKDSelectiveRecoveryLifecycleController"
            ),

            node_names=tuple(
                node.name

                for node
                in self.stack[
                    "path"
                ]
            ),

            link_ids=(
                actual_regeneration_link_ids
            ),

            resource_ids=(
                discarded_ids
            ),

            message=(
                "Control plane executed the exact minimum-cost "
                "collective QKD-quality recovery plan."
            ),

            metadata={
                "planner_status":
                    plan.status.value,

                "discarded_resource_count":
                    len(
                        discarded_ids
                    ),

                "preserved_resource_count":
                    len(
                        preserved_ids
                    ),

                "planned_regeneration_links":
                    expected_regeneration_link_ids,

                "actual_regeneration_links":
                    actual_regeneration_link_ids,

                "qber_before":
                    float(
                        frontier_before_row[
                            "optimistic_final_qber"
                        ]
                    ),

                "qber_after_optimistic":
                    float(
                        frontier_after_row[
                            "optimistic_final_qber"
                        ]
                    ),

                "quality_slack_before":
                    float(
                        frontier_before_row[
                            "remaining_quality_slack"
                        ]
                    ),

                "quality_slack_after":
                    float(
                        frontier_after_row[
                            "remaining_quality_slack"
                        ]
                    )
            }
        )


        record = (
            QKDSelectiveRecoveryExecutionRecord(

                execution_index=(
                    execution_index
                ),

                simulation_slot=(
                    current_slot
                ),

                status=(
                    QKDSelectiveRecoveryExecutionStatus
                    .EXECUTED
                ),

                planner_status=(
                    plan.status
                ),

                dry_run=False,

                discarded_resource_ids=(
                    discarded_ids
                ),

                preserved_resource_ids=(
                    preserved_ids
                ),

                planned_regeneration_link_ids=(
                    expected_regeneration_link_ids
                ),

                actual_regeneration_link_ids=(
                    actual_regeneration_link_ids
                ),

                ready_elementary_link_ids=tuple(
                    reconciliation
                    .ready_elementary_link_ids
                ),

                protected_link_ids=tuple(
                    reconciliation
                    .protected_link_ids
                ),

                frontier_state_before=str(
                    frontier_before_row[
                        "state"
                    ]
                ),

                frontier_state_after=str(
                    frontier_after_row[
                        "state"
                    ]
                ),

                optimistic_final_qber_before=float(
                    frontier_before_row[
                        "optimistic_final_qber"
                    ]
                ),

                optimistic_final_qber_after=float(
                    frontier_after_row[
                        "optimistic_final_qber"
                    ]
                ),

                quality_slack_before=float(
                    frontier_before_row[
                        "remaining_quality_slack"
                    ]
                ),

                quality_slack_after=float(
                    frontier_after_row[
                        "remaining_quality_slack"
                    ]
                ),

                termination_reason=(
                    termination_reason
                )
            )
        )


        self._history.append(
            record
        )


        return {

            "record":
                record,

            "plan":
                plan,

            "resource_quality_before":
                resource_quality_before,

            "frontier_quality_before":
                frontier_quality_before,

            "resource_quality_after":
                resource_quality_after,

            "frontier_quality_after":
                frontier_quality_after,

            "reconciliation":
                reconciliation
        }

In [ ]:
# @title Integrated Synchronized-Frontier Test Builder

class Phase3H2BootstrapAlwaysSuccessModel(
    ElementaryEntanglementModel
):
    """
    TEST-ONLY model used for exactly one bootstrap slot.

    Purpose:
        Construct seven simultaneously fresh elementary resources
        through the REAL elementary-generation scheduler so every
        ElementaryLinkGenerationProcess owns its proper resource.

    This model is removed immediately afterward.

    It is NOT a NARAD-QC physical model and must never be used in
    experiments or Monte Carlo evaluation.
    """

    @property
    def model_name(
        self
    ) -> str:

        return (
            "PHASE3H2_TEST_ONLY_BOOTSTRAP_ALWAYS_SUCCESS"
        )


    def success_probability(
        self,
        link
    ) -> float:

        if not isinstance(
            link,
            PhysicalQuantumLink
        ):

            raise TypeError(
                "link must be a PhysicalQuantumLink."
            )


        return self.validate_probability(
            1.0
        )


def build_phase3h2_integrated_collectively_doomed_stack(
    simulation_seed=7_320_101,
    age_slots=5
):
    """
    Build the controlled Phase-3H.2 verification state:

        600 km
        6 repeaters
        7 elementary links

    Slot 0:
        All seven elementary resources are generated through the
        actual generation scheduler using the TEST-ONLY bootstrap
        model.

    Immediately afterward:
        Generation physics is restored to FiberSurvivalProxyModel.

    Then:
        All seven resources age together for `age_slots`.

    At age_slots=5:
        every resource should remain individually QKD-viable,
        while the complete frontier is collectively doomed.
    """

    if (
        isinstance(
            age_slots,
            bool
        )
        or not isinstance(
            age_slots,
            int
        )
        or age_slots < 0
    ):

        raise ValueError(
            "age_slots must be a non-negative integer."
        )


    # ============================================================
    # Base physical / authenticated / fidelity / swapping stack
    # ============================================================

    base_stack = (
        build_phase3g2_trial_stack(

            policy_name=(
                "MAX_PARALLEL_OLDEST_FIRST"
            ),

            repeater_count=6,

            simulation_seed=(
                simulation_seed
            )
        )
    )


    simulator = (
        base_stack[
            "simulator"
        ]
    )


    path = (
        base_stack[
            "path"
        ]
    )


    fidelity_registry = (
        base_stack[
            "fidelity_registry"
        ]
    )


    # ============================================================
    # Replace only the elementary generation control components
    # for this isolated verification stack.
    # ============================================================

    bootstrap_service = (
        ElementaryEntanglementService(

            simulator=(
                simulator
            ),

            model=(
                Phase3H2BootstrapAlwaysSuccessModel()
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=(
                simulator
            ),

            service=(
                bootstrap_service
            ),

            ordered_path=(
                path
            ),

            demand_active=False
        )
    )


    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=(
                simulator
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=(
                path
            )
        )
    )


    # Rebuild the autonomous orchestrator too so the returned stack
    # remains internally coherent for later 3H experiments.
    orchestrator = (
        FailureAwareEndToEndEntanglementOrchestrator(

            simulator=(
                simulator
            ),

            ordered_path=(
                path
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            demand_controller=(
                demand_controller
            ),

            swap_scheduler=(
                base_stack[
                    "swap_scheduler"
                ]
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    stack = dict(
        base_stack
    )


    stack[
        "generation_service"
    ] = bootstrap_service


    stack[
        "generation_scheduler"
    ] = generation_scheduler


    stack[
        "demand_controller"
    ] = demand_controller


    stack[
        "orchestrator"
    ] = orchestrator


    # ============================================================
    # Empty frontier -> demand all seven physical links.
    # ============================================================

    initial_reconciliation = (
        demand_controller
        .reconcile()
    )


    assert (
        len(
            initial_reconciliation
            .regeneration_link_ids
        )
        ==
        7
    )


    # ============================================================
    # ONE synchronized bootstrap slot.
    # ============================================================

    bootstrap_slot = (
        generation_scheduler
        .process_current_slot()
    )


    assert (
        bootstrap_slot.simulation_slot
        ==
        0
    )


    assert (
        bootstrap_slot.physical_attempt_count
        ==
        7
    )


    assert (
        bootstrap_slot.successful_generation_count
        ==
        7
    )


    assert (
        bootstrap_slot.failed_generation_count
        ==
        0
    )


    assert (
        len(
            simulator
            .entanglement_manager
            .active_resources
        )
        ==
        7
    )


    # ============================================================
    # Register the seven elementary fidelity checkpoints.
    # ============================================================

    for step in (
        bootstrap_slot
        .steps
    ):

        assert (
            step.attempt
            is not None
        )


        assert (
            step.attempt.success
            is True
        )


        resource = (
            simulator
            .entanglement_manager
            .get_resource(
                step.attempt
                .resource_id
            )
        )


        fidelity_registry\
            .register_elementary(
                resource
            )


    # ============================================================
    # CRITICAL:
    # Restore the actual Phase-3 physical generation model BEFORE
    # any quality recovery or regeneration is allowed.
    # ============================================================

    bootstrap_service.model = (
        FiberSurvivalProxyModel()
    )


    assert (
        bootstrap_service
        .model
        .model_name
        ==
        "FiberSurvivalProxyModel"
    )


    # ============================================================
    # Verify all seven processes genuinely own READY resources.
    # ============================================================

    ready_reconciliation = (
        demand_controller
        .reconcile()
    )


    assert (
        len(
            ready_reconciliation
            .ready_elementary_link_ids
        )
        ==
        7
    )


    assert (
        len(
            ready_reconciliation
            .regeneration_link_ids
        )
        ==
        0
    )


    assert (
        generation_scheduler
        .all_links_ready
        is True
    )


    assert all(

        process.state
        ==
        ElementaryLinkProcessState.READY

        for process
        in generation_scheduler.processes
    )


    # ============================================================
    # Age together.
    # ============================================================

    if age_slots > 0:

        simulator\
            .kernel\
            .advance_slot(
                age_slots
            )


    assert (
        simulator
        .kernel
        .current_slot
        ==
        age_slots
    )


    simulator\
        .entanglement_manager\
        .validate_all_active_resources()


    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    return {

        "stack":
            stack,

        "bootstrap_slot":
            bootstrap_slot,

        "initial_reconciliation":
            initial_reconciliation,

        "ready_reconciliation":
            ready_reconciliation
    }

In [ ]:
# @title Selective Quality-Recovery Integration Verification

print(
    "=" * 90
)

print(
    "Selective QKD Quality-Recovery Integration Verification"
)

print(
    "=" * 90
)


# ================================================================
# Build integrated 7-resource / five-slot-old doomed frontier
# ================================================================

fixture = (
    build_phase3h2_integrated_collectively_doomed_stack(

        simulation_seed=(
            7_320_101
        ),

        age_slots=5
    )
)


phase3h2b_stack = (
    fixture[
        "stack"
    ]
)


simulator = (
    phase3h2b_stack[
        "simulator"
    ]
)


manager = (
    simulator
    .entanglement_manager
)


generation_scheduler = (
    phase3h2b_stack[
        "generation_scheduler"
    ]
)


fidelity_registry = (
    phase3h2b_stack[
        "fidelity_registry"
    ]
)


current_slot = (
    simulator
    .kernel
    .current_slot
)


assert (
    current_slot
    ==
    5
)


# ================================================================
# Confirm the exact Phase-3H.2A starting condition.
# ================================================================

(
    initial_resource_quality,
    initial_frontier_quality
) = (
    phase3h1_quality_snapshot(

        stack=(
            phase3h2b_stack
        ),

        simulation_slot=(
            current_slot
        )
    )
)


assert (
    len(
        initial_resource_quality
    )
    ==
    7
)


assert (
    initial_resource_quality[
        "quality_expired"
    ]
    .sum()
    ==
    0
)


assert (
    initial_frontier_quality
    .iloc[0][
        "state"
    ]
    ==
    QKDFrontierQualityState
    .COLLECTIVELY_DOOMED
    .value
)


print(
    "[PASS] Seven synchronized resources are individually viable "
    "but collectively QKD-doomed."
)


# ================================================================
# Planner + lifecycle controller
# ================================================================

planner = (
    QKDMinimumCostCollectiveRecoveryPlanner(

        stack=(
            phase3h2b_stack
        )
    )
)


recovery_controller = (
    QKDSelectiveRecoveryLifecycleController(

        stack=(
            phase3h2b_stack
        ),

        planner=(
            planner
        )
    )
)


# ================================================================
# Dry-run first.
# ================================================================

active_ids_before = tuple(
    sorted(

        resource.resource_id

        for resource
        in manager.active_resources
    )
)


dry_run_result = (
    recovery_controller
    .execute(
        dry_run=True
    )
)


dry_record = (
    dry_run_result[
        "record"
    ]
)


dry_plan = (
    dry_run_result[
        "plan"
    ]
)


assert (
    dry_record.status
    ==
    QKDSelectiveRecoveryExecutionStatus
    .DRY_RUN_PLANNED
)


assert (
    dry_plan.status
    ==
    QKDCollectiveRecoveryStatus
    .PLAN_FOUND
)


assert (
    dry_plan
    .chosen_candidate
    is not None
)


assert (
    dry_plan
    .chosen_candidate
    .discarded_resource_count
    ==
    2
)


assert (
    recovery_controller
    ._active_resource_ids()
    ==
    active_ids_before
)


assert all(

    manager
    .get_resource(
        resource_id
    )
    .status
    ==
    EntanglementStatus.ACTIVE

    for resource_id
    in active_ids_before
)


print(
    "[PASS] Dry-run selected exactly two victims without "
    "mutating quantum state."
)


# ================================================================
# Real selective recovery.
# ================================================================

execution_result = (
    recovery_controller
    .execute(
        dry_run=False
    )
)


record = (
    execution_result[
        "record"
    ]
)


plan = (
    execution_result[
        "plan"
    ]
)


chosen = (
    plan
    .chosen_candidate
)


reconciliation = (
    execution_result[
        "reconciliation"
    ]
)


assert (
    record.status
    ==
    QKDSelectiveRecoveryExecutionStatus
    .EXECUTED
)


assert (
    chosen
    is not None
)


victim_ids = tuple(
    chosen
    .discarded_resource_ids
)


preserved_ids = tuple(
    chosen
    .preserved_resource_ids
)


planned_regeneration_links = tuple(
    chosen
    .total_regeneration_link_ids
)


# ================================================================
# Exactly two victims; exactly five survivors.
# ================================================================

assert (
    len(
        victim_ids
    )
    ==
    2
)


assert (
    len(
        preserved_ids
    )
    ==
    5
)


assert (
    len(
        manager.active_resources
    )
    ==
    5
)


assert (
    tuple(
        sorted(

            resource.resource_id

            for resource
            in manager.active_resources
        )
    )
    ==
    tuple(
        sorted(
            preserved_ids
        )
    )
)


print(
    "[PASS] Exactly two planned resources were removed and "
    "exactly five were preserved."
)


# ================================================================
# Victim lifecycle verification.
# ================================================================

for resource_id in victim_ids:

    resource = (
        manager
        .get_resource(
            resource_id
        )
    )


    assert (
        resource.status
        ==
        EntanglementStatus.DISCARDED
    )


    assert (
        resource.ended_slot
        ==
        current_slot
    )


    assert (
        resource.termination_reason
        ==
        record.termination_reason
    )


print(
    "[PASS] Both selected victims entered DISCARDED through "
    "EntanglementManager at the current slot."
)


# ================================================================
# Preserved resources really remain ACTIVE.
# ================================================================

for resource_id in preserved_ids:

    assert (
        manager
        .get_resource(
            resource_id
        )
        .status
        ==
        EntanglementStatus.ACTIVE
    )


print(
    "[PASS] All five non-selected resources remain ACTIVE."
)


# ================================================================
# Failure-aware demand must expose exactly the planned links.
# ================================================================

assert (
    tuple(
        reconciliation
        .regeneration_link_ids
    )
    ==
    planned_regeneration_links
)


assert (
    len(
        reconciliation
        .regeneration_link_ids
    )
    ==
    2
)


assert (
    len(
        reconciliation
        .ready_elementary_link_ids
    )
    ==
    5
)


assert (
    len(
        reconciliation
        .protected_link_ids
    )
    ==
    0
)


print(
    "[PASS] Failure-aware demand exposed exactly two elementary "
    "links for regeneration and preserved five ready links."
)


# ================================================================
# Generation-process state must agree with the control plane.
# ================================================================

regeneration_link_set = set(
    reconciliation
    .regeneration_link_ids
)


ready_link_set = set(
    reconciliation
    .ready_elementary_link_ids
)


for process in (
    generation_scheduler
    .processes
):

    if (
        process.link.link_id
        in regeneration_link_set
    ):

        assert (
            process.state
            ==
            ElementaryLinkProcessState.SEEKING
        )

    elif (
        process.link.link_id
        in ready_link_set
    ):

        assert (
            process.state
            ==
            ElementaryLinkProcessState.READY
        )

    else:

        raise AssertionError(
            "Unexpected link classification after selective recovery."
        )


print(
    "[PASS] Generation processes agree with recovery demand: "
    "2 SEEKING, 5 READY."
)


# ================================================================
# Planner prediction == actually exposed frontier.
# ================================================================

frontier_after = (
    execution_result[
        "frontier_quality_after"
    ]
    .iloc[0]
)


assert (
    frontier_after[
        "state"
    ]
    ==
    QKDFrontierQualityState
    .VIABLE
    .value
)


assert (
    frontier_after[
        "can_still_reach_qkd_target"
    ]
    == True
)


assert (
    float(
        frontier_after[
            "optimistic_final_qber"
        ]
    )
    <=
    PHASE3H1_QUALITY_CONFIG
    .qber_limit
)


assert math.isclose(

    float(
        frontier_after[
            "optimistic_final_qber"
        ]
    ),

    chosen
    .optimistic_final_qber_after_repair,

    rel_tol=0.0,

    abs_tol=1e-12
)


assert math.isclose(

    float(
        frontier_after[
            "remaining_quality_slack"
        ]
    ),

    chosen
    .remaining_quality_slack_after_repair,

    rel_tol=0.0,

    abs_tol=1e-12
)


print(
    "[PASS] Actual post-discard frontier exactly matches the "
    "planner's predicted QKD repair."
)


# ================================================================
# Zero-time lifecycle operation.
# ================================================================

assert (
    simulator
    .kernel
    .current_slot
    ==
    current_slot
)


print(
    "[PASS] Planning, selective discard, and demand reconciliation "
    "advanced zero simulation slots."
)


# ================================================================
# NOW permit one real physical elementary-generation slot.
#
# FiberSurvivalProxyModel has already been restored.
#
# Expected:
#     exactly two physical attempts;
#     exactly five resources simply wait.
# ================================================================

assert (
    phase3h2b_stack[
        "generation_service"
    ]
    .model
    .model_name
    ==
    "FiberSurvivalProxyModel"
)


selective_generation_slot = (
    generation_scheduler
    .process_current_slot()
)


assert (
    selective_generation_slot
    .simulation_slot
    ==
    current_slot
)


assert (
    selective_generation_slot
    .physical_attempt_count
    ==
    2
)


assert (
    selective_generation_slot
    .waiting_link_count
    ==
    5
)


attempted_link_ids = tuple(

    step.link_id

    for step
    in selective_generation_slot.steps

    if (
        step.attempt
        is not None
    )
)


assert (
    attempted_link_ids
    ==
    planned_regeneration_links
)


print(
    "[PASS] The physical generation layer attempted ONLY the two "
    "links selected for quality recovery."
)


# ================================================================
# Register any successful regenerated Bell pairs with the existing
# physical-fidelity sidecar.
#
# There may be 0, 1, or 2 successes because the real fibre model is
# now active. The test does NOT require lucky stochastic outcomes.
# ================================================================

successful_regenerated_ids = []


for step in (
    selective_generation_slot
    .steps
):

    if (
        step.attempt
        is not None
        and
        step.attempt.success
    ):

        resource = (
            manager
            .get_resource(
                step.attempt
                .resource_id
            )
        )


        fidelity_registry\
            .register_elementary(
                resource
            )


        successful_regenerated_ids.append(
            resource.resource_id
        )


successful_regenerated_ids = tuple(
    successful_regenerated_ids
)


print(
    "[INFO] Real fibre-model regeneration successes in this slot:",
    len(
        successful_regenerated_ids
    ),
    "/ 2"
)


# ================================================================
# Final cross-layer consistency.
# ================================================================

manager\
    .validate_all_active_resources()


assert (
    simulator
    .validate_runtime_consistency()
    is True
)


assert (
    simulator
    .kernel
    .current_slot
    ==
    current_slot
)


print(
    "[PASS] Runtime remains cross-layer consistent after "
    "selective physical regeneration attempts."
)


# ================================================================
# Diagnostics
# ================================================================

print(
    "\nSELECTIVE RECOVERY EXECUTION"
)


display(
    pd.DataFrame(
        [
            {
                **asdict(
                    record
                ),

                "status":
                    record.status.value,

                "planner_status":
                    record.planner_status.value
            }
        ]
    )
)


print(
    "\nPOST-RECOVERY DEMAND STATE"
)


display(
    phase3h2b_stack[
        "demand_controller"
    ]
    .decisions_dataframe(
        reconciliation
    )
)


print(
    "\nSELECTIVE PHYSICAL GENERATION SLOT"
)


display(
    pd.DataFrame(
        [
            {
                "Link ID":
                    step.link_id,

                "State Before":
                    step.state_before.value,

                "Action":
                    step.action.value,

                "State After":
                    step.state_after.value,

                "Physical Attempt":
                    (
                        step.attempt
                        is not None
                    ),

                "Attempt Success":
                    (
                        step.attempt.success

                        if (
                            step.attempt
                            is not None
                        )

                        else None
                    ),

                "Terminated Resource":
                    step.terminated_resource_id,

                "Termination Status":
                    (
                        step
                        .terminated_resource_status
                        .value

                        if (
                            step
                            .terminated_resource_status
                            is not None
                        )

                        else None
                    ),

                "Active Resource":
                    step.active_resource_id
            }

            for step
            in selective_generation_slot.steps
        ]
    )
)


print(
    "\nQKD FRONTIER AFTER SELECTIVE DISCARD"
)


display(
    execution_result[
        "frontier_quality_after"
    ]
)


print(
    "\nRECOVERY CONTROLLER HISTORY"
)


display(
    recovery_controller
    .history_dataframe()
)


print(
    "\n" + "=" * 90
)

print(
    "Selective quality-recovery integration PASSED."
)

print(
    "=" * 90
)

In [ ]:
# @title Autonomous QKD-Ready Execution Records

from dataclasses import dataclass
from enum import Enum
from typing import Optional

import pandas as pd


class QKDReadyExecutionStatus(
    str,
    Enum
):

    SUCCESS = (
        "SUCCESS"
    )

    MAX_SLOTS_EXCEEDED = (
        "MAX_SLOTS_EXCEEDED"
    )


@dataclass(frozen=True)
class QKDReadyExecutionSlot:

    execution_index: int

    simulation_slot: int

    # Existing Phase-3D orchestration record
    base_execution_slot: FailureAwareExecutionSlot

    # Pre-physics quality maintenance
    pre_individual_quality_discard_count: int

    pre_collective_recovery_status: str

    pre_collective_discarded_resource_ids: tuple[str, ...]

    # Post-physics quality maintenance
    post_individual_quality_discard_count: int

    post_collective_recovery_status: str

    post_collective_discarded_resource_ids: tuple[str, ...]

    # Final QKD admission state
    end_to_end_resource_id: Optional[str]

    final_fidelity: Optional[float]

    predicted_qber: Optional[float]

    qkd_eligible: bool


    @property
    def elementary_attempt_count(
        self
    ) -> int:

        return (
            self.base_execution_slot
            .elementary_attempt_count
        )


    @property
    def elementary_success_count(
        self
    ) -> int:

        return (
            self.base_execution_slot
            .elementary_success_count
        )


    @property
    def elementary_failure_count(
        self
    ) -> int:

        return (
            self.base_execution_slot
            .elementary_failure_count
        )


    @property
    def bsm_attempt_count(
        self
    ) -> int:

        return (
            self.base_execution_slot
            .bsm_attempt_count
        )


    @property
    def bsm_success_count(
        self
    ) -> int:

        return (
            self.base_execution_slot
            .bsm_success_count
        )


    @property
    def bsm_failure_count(
        self
    ) -> int:

        return (
            self.base_execution_slot
            .bsm_failure_count
        )


@dataclass(frozen=True)
class QKDReadyEndToEndExecutionResult:

    status: QKDReadyExecutionStatus

    start_slot: int
    end_slot: int

    processed_slot_count: int

    end_to_end_resource_id: Optional[str]

    final_fidelity: Optional[float]

    predicted_qber: Optional[float]

    total_elementary_attempts: int

    total_elementary_successes: int

    total_elementary_failures: int

    total_bsm_attempts: int

    total_bsm_successes: int

    total_bsm_failures: int

    total_individual_quality_discards: int

    total_collective_recovery_executions: int

    total_collective_resources_discarded: int

    execution_history: tuple[
        QKDReadyExecutionSlot, ...
    ]


    @property
    def success(
        self
    ) -> bool:

        return (
            self.status
            ==
            QKDReadyExecutionStatus.SUCCESS
        )

In [ ]:
# @title Autonomous QKD-Ready End-to-End Orchestrator


class QKDReadyEndToEndEntanglementOrchestrator(
    FailureAwareEndToEndEntanglementOrchestrator
):
    """
    Phase-3H closed-loop execution owner.

    Reuses the complete validated Phase-3D physical execution slot:

        demand
            ->
        elementary generation
            ->
        fidelity registration
            ->
        one parallel BSM round
            ->
        failure-aware reconciliation

    and surrounds it with QKD-quality control:

        pre-slot quality maintenance
            ->
        Phase-3D physical slot
            ->
        post-slot quality maintenance
            ->
        QKD admission test

    SUCCESS now means:

        full-path Bell resource exists
        AND
        predicted QBER <= configured QKD limit.

    Merely establishing end-to-end entanglement is no longer enough.
    """

    def __init__(
        self,
        stack,
        config=PHASE3H1_QUALITY_CONFIG
    ):

        self.stack = stack

        self.config = config


        super().__init__(

            simulator=(
                stack[
                    "simulator"
                ]
            ),

            ordered_path=(
                stack[
                    "path"
                ]
            ),

            generation_scheduler=(
                stack[
                    "generation_scheduler"
                ]
            ),

            demand_controller=(
                stack[
                    "demand_controller"
                ]
            ),

            swap_scheduler=(
                stack[
                    "swap_scheduler"
                ]
            ),

            fidelity_registry=(
                stack[
                    "fidelity_registry"
                ]
            )
        )


        # ========================================================
        # Quality evaluator
        # ========================================================

        self.quality_evaluator = (
            QKDQualityBudgetEvaluator(

                topology=(
                    stack[
                        "topology"
                    ]
                ),

                ordered_path=(
                    stack[
                        "path"
                    ]
                ),

                config=(
                    config
                )
            )
        )


        # ========================================================
        # Phase-3H.1 individual expiration
        # ========================================================

        self.individual_quality_controller = (
            QKDQualityLifecycleController(

                stack=(
                    stack
                ),

                config=(
                    config
                )
            )
        )


        # ========================================================
        # Phase-3H.2 collective repair
        # ========================================================

        self.collective_recovery_planner = (
            QKDMinimumCostCollectiveRecoveryPlanner(

                stack=(
                    stack
                ),

                config=(
                    config
                )
            )
        )


        self.collective_recovery_controller = (
            QKDSelectiveRecoveryLifecycleController(

                stack=(
                    stack
                ),

                planner=(
                    self.collective_recovery_planner
                ),

                config=(
                    config
                )
            )
        )


        self._qkd_execution_history = []


    # ============================================================
    # Quality Maintenance
    # ============================================================

    def _apply_quality_maintenance(
        self,
        stage
    ):
        """
        Execute conservative quality maintenance at the current slot.

        Order matters:

            1. Remove individually hopeless resources.
            2. Reconcile generation ownership/demand if that happened.
            3. Only then evaluate collective minimum-cost recovery.

        No global clock advancement occurs.
        """

        current_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        slot_before = (
            current_slot
        )


        # --------------------------------------------------------
        # Individual optimistic-impossibility discard
        # --------------------------------------------------------

        individual_result = (
            self.individual_quality_controller
            .expire_individually_doomed_resources(

                simulation_slot=(
                    current_slot
                ),

                dry_run=False
            )
        )


        individual_discard_count = int(
            individual_result[
                "discard_count"
            ]
        )


        if (
            individual_discard_count
            >
            0
        ):

            self.demand_controller\
                .reconcile()


        # --------------------------------------------------------
        # Collective minimum-cost repair
        # --------------------------------------------------------

        collective_result = (
            self.collective_recovery_controller
            .execute(
                dry_run=False
            )
        )


        collective_record = (
            collective_result[
                "record"
            ]
        )


        # --------------------------------------------------------
        # Demand state must reflect all possible quality mutations.
        # --------------------------------------------------------

        final_reconciliation = (
            self.demand_controller
            .reconcile()
        )


        # --------------------------------------------------------
        # Absolutely no hidden clock advancement.
        # --------------------------------------------------------

        if (
            self.simulator
            .kernel
            .current_slot
            !=
            slot_before
        ):

            raise RuntimeError(
                f"{stage} QKD-quality maintenance illegally "
                "advanced the global simulation clock."
            )


        self.simulator\
            .entanglement_manager\
            .validate_all_active_resources()


        assert (
            self.simulator
            .validate_runtime_consistency()
            is True
        )


        return {

            "stage":
                str(
                    stage
                ),

            "individual":
                individual_result,

            "individual_discard_count":
                individual_discard_count,

            "collective":
                collective_result,

            "collective_record":
                collective_record,

            "final_reconciliation":
                final_reconciliation
        }


    # ============================================================
    # QKD Admission
    # ============================================================

    def _assess_end_to_end_qkd_resource(
        self
    ):

        resource = (
            self._find_end_to_end_resource()
        )


        if (
            resource
            is None
        ):

            return (
                None,
                None
            )


        current_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        current_fidelity = float(
            self.fidelity_registry
            .fidelity(
                resource
            )
        )


        assessment = (
            self.quality_evaluator
            .assess_resource(

                resource=(
                    resource
                ),

                current_fidelity=(
                    current_fidelity
                ),

                simulation_slot=(
                    current_slot
                )
            )
        )


        return (
            resource,
            assessment
        )


    # ============================================================
    # ONE Closed-Loop Global Slot
    # ============================================================

    def process_qkd_ready_current_slot(
        self
    ) -> QKDReadyExecutionSlot:

        self.simulator\
            .validate_runtime_consistency()


        slot_at_start = int(
            self.simulator
            .kernel
            .current_slot
        )


        # ========================================================
        # 1. Synchronize ordinary physical expiry/demand first.
        #
        # This ensures quality logic never reasons over a resource
        # that should already have physically expired.
        # ========================================================

        self.demand_controller\
            .reconcile()


        assert (
            self.simulator
            .kernel
            .current_slot
            ==
            slot_at_start
        )


        # ========================================================
        # 2. PRE-PHYSICS QKD QUALITY MAINTENANCE
        # ========================================================

        pre_quality = (
            self._apply_quality_maintenance(
                stage=(
                    "PRE_PHYSICS"
                )
            )
        )


        # ========================================================
        # 3. EXISTING VALIDATED PHASE-3D PHYSICAL SLOT
        #
        # Internally:
        #
        #   demand reconciliation
        #   parallel generation
        #   elementary fidelity registration
        #   at most ONE parallel swap round
        #   post-BSM reconciliation
        #
        # No clock advancement.
        # ========================================================

        base_step = (
            super()
            .process_current_slot()
        )


        assert (
            base_step.simulation_slot
            ==
            slot_at_start
        )


        assert (
            self.simulator
            .kernel
            .current_slot
            ==
            slot_at_start
        )


        # ========================================================
        # 4. POST-PHYSICS QKD QUALITY MAINTENANCE
        #
        # Important cases:
        #
        # - stale end-to-end pair was produced;
        # - future q_swap < 1 makes a new parent too weak;
        # - a changed frontier becomes collectively doomed.
        #
        # No second generation attempt is allowed in this slot.
        # ========================================================

        post_quality = (
            self._apply_quality_maintenance(
                stage=(
                    "POST_PHYSICS"
                )
            )
        )


        assert (
            self.simulator
            .kernel
            .current_slot
            ==
            slot_at_start
        )


        # ========================================================
        # 5. QKD-READY END-TO-END ADMISSION
        # ========================================================

        (
            end_to_end_resource,
            qkd_assessment
        ) = (
            self
            ._assess_end_to_end_qkd_resource()
        )


        if (
            end_to_end_resource
            is None
        ):

            end_to_end_resource_id = None

            final_fidelity = None

            predicted_qber = None

            qkd_eligible = False


        else:

            final_fidelity = float(
                qkd_assessment
                .current_fidelity
            )


            predicted_qber = float(
                qkd_assessment
                .current_qber_proxy
            )


            qkd_eligible = bool(
                qkd_assessment
                .qkd_eligible_now
            )


            # After post-quality maintenance, any full-path resource
            # that remains ACTIVE should already satisfy admission.
            if not qkd_eligible:

                raise RuntimeError(
                    "A non-QKD-eligible full-path resource survived "
                    "post-physics quality maintenance."
                )


            end_to_end_resource_id = (
                end_to_end_resource
                .resource_id
            )


        # ========================================================
        # 6. Record
        # ========================================================

        pre_collective_record = (
            pre_quality[
                "collective_record"
            ]
        )


        post_collective_record = (
            post_quality[
                "collective_record"
            ]
        )


        step = (
            QKDReadyExecutionSlot(

                execution_index=len(
                    self._qkd_execution_history
                ),

                simulation_slot=(
                    slot_at_start
                ),

                base_execution_slot=(
                    base_step
                ),

                pre_individual_quality_discard_count=(
                    pre_quality[
                        "individual_discard_count"
                    ]
                ),

                pre_collective_recovery_status=(
                    pre_collective_record
                    .status
                    .value
                ),

                pre_collective_discarded_resource_ids=(
                    pre_collective_record
                    .discarded_resource_ids
                ),

                post_individual_quality_discard_count=(
                    post_quality[
                        "individual_discard_count"
                    ]
                ),

                post_collective_recovery_status=(
                    post_collective_record
                    .status
                    .value
                ),

                post_collective_discarded_resource_ids=(
                    post_collective_record
                    .discarded_resource_ids
                ),

                end_to_end_resource_id=(
                    end_to_end_resource_id
                ),

                final_fidelity=(
                    final_fidelity
                ),

                predicted_qber=(
                    predicted_qber
                ),

                qkd_eligible=(
                    qkd_eligible
                )
            )
        )


        self._qkd_execution_history.append(
            step
        )


        # ========================================================
        # 7. Audit event
        # ========================================================

        self.simulator.kernel.emit(

            event_type=(
                "QKD_READY_EXECUTION_SLOT_COMPLETED"
            ),

            component=(
                "QKDReadyEndToEndEntanglementOrchestrator"
            ),

            node_names=tuple(
                node.name

                for node
                in self.ordered_path
            ),

            link_ids=tuple(
                self._path_link_ids
            ),

            resource_ids=tuple(
                resource.resource_id

                for resource
                in (
                    self.simulator
                    .entanglement_manager
                    .active_resources
                )
            ),

            message=(
                f"Completed QKD-aware execution slot "
                f"{step.execution_index}."
            ),

            metadata={
                "pre_individual_quality_discards":
                    step
                    .pre_individual_quality_discard_count,

                "pre_collective_recovery":
                    step
                    .pre_collective_recovery_status,

                "elementary_attempts":
                    step
                    .elementary_attempt_count,

                "bsm_attempts":
                    step
                    .bsm_attempt_count,

                "post_individual_quality_discards":
                    step
                    .post_individual_quality_discard_count,

                "post_collective_recovery":
                    step
                    .post_collective_recovery_status,

                "qkd_eligible":
                    step
                    .qkd_eligible,

                "predicted_qber":
                    step
                    .predicted_qber
            }
        )


        self.simulator\
            .validate_runtime_consistency()


        return step


    # ============================================================
    # Autonomous Session
    # ============================================================

    def run_until_qkd_ready(
        self,
        max_slots=5000
    ) -> QKDReadyEndToEndExecutionResult:

        if (
            isinstance(
                max_slots,
                bool
            )
            or not isinstance(
                max_slots,
                int
            )
            or max_slots <= 0
        ):

            raise ValueError(
                "max_slots must be a positive integer."
            )


        start_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        run_history = []


        # ========================================================
        # Existing full-path state?
        # ========================================================

        self._apply_quality_maintenance(
            stage=(
                "SESSION_START"
            )
        )


        (
            existing_resource,
            existing_assessment
        ) = (
            self
            ._assess_end_to_end_qkd_resource()
        )


        if (
            existing_resource
            is not None
        ):

            assert (
                existing_assessment
                .qkd_eligible_now
            )


            return (
                QKDReadyEndToEndExecutionResult(

                    status=(
                        QKDReadyExecutionStatus
                        .SUCCESS
                    ),

                    start_slot=(
                        start_slot
                    ),

                    end_slot=(
                        start_slot
                    ),

                    processed_slot_count=0,

                    end_to_end_resource_id=(
                        existing_resource
                        .resource_id
                    ),

                    final_fidelity=float(
                        existing_assessment
                        .current_fidelity
                    ),

                    predicted_qber=float(
                        existing_assessment
                        .current_qber_proxy
                    ),

                    total_elementary_attempts=0,

                    total_elementary_successes=0,

                    total_elementary_failures=0,

                    total_bsm_attempts=0,

                    total_bsm_successes=0,

                    total_bsm_failures=0,

                    total_individual_quality_discards=0,

                    total_collective_recovery_executions=0,

                    total_collective_resources_discarded=0,

                    execution_history=()
                )
            )


        # ========================================================
        # Main closed-loop execution
        # ========================================================

        for _ in range(
            max_slots
        ):

            step = (
                self
                .process_qkd_ready_current_slot()
            )


            run_history.append(
                step
            )


            # ====================================================
            # QKD SUCCESS
            # ====================================================

            if (
                step.qkd_eligible
            ):

                result = (
                    QKDReadyEndToEndExecutionResult(

                        status=(
                            QKDReadyExecutionStatus
                            .SUCCESS
                        ),

                        start_slot=(
                            start_slot
                        ),

                        end_slot=int(
                            self.simulator
                            .kernel
                            .current_slot
                        ),

                        processed_slot_count=len(
                            run_history
                        ),

                        end_to_end_resource_id=(
                            step
                            .end_to_end_resource_id
                        ),

                        final_fidelity=(
                            step
                            .final_fidelity
                        ),

                        predicted_qber=(
                            step
                            .predicted_qber
                        ),

                        total_elementary_attempts=sum(
                            record
                            .elementary_attempt_count

                            for record
                            in run_history
                        ),

                        total_elementary_successes=sum(
                            record
                            .elementary_success_count

                            for record
                            in run_history
                        ),

                        total_elementary_failures=sum(
                            record
                            .elementary_failure_count

                            for record
                            in run_history
                        ),

                        total_bsm_attempts=sum(
                            record
                            .bsm_attempt_count

                            for record
                            in run_history
                        ),

                        total_bsm_successes=sum(
                            record
                            .bsm_success_count

                            for record
                            in run_history
                        ),

                        total_bsm_failures=sum(
                            record
                            .bsm_failure_count

                            for record
                            in run_history
                        ),

                        total_individual_quality_discards=sum(

                            record
                            .pre_individual_quality_discard_count

                            +

                            record
                            .post_individual_quality_discard_count

                            for record
                            in run_history
                        ),

                        total_collective_recovery_executions=sum(

                            int(
                                record
                                .pre_collective_recovery_status
                                ==
                                QKDSelectiveRecoveryExecutionStatus
                                .EXECUTED
                                .value
                            )

                            +

                            int(
                                record
                                .post_collective_recovery_status
                                ==
                                QKDSelectiveRecoveryExecutionStatus
                                .EXECUTED
                                .value
                            )

                            for record
                            in run_history
                        ),

                        total_collective_resources_discarded=sum(

                            len(
                                record
                                .pre_collective_discarded_resource_ids
                            )

                            +

                            len(
                                record
                                .post_collective_discarded_resource_ids
                            )

                            for record
                            in run_history
                        ),

                        execution_history=tuple(
                            run_history
                        )
                    )
                )


                self.simulator.kernel.emit(

                    event_type=(
                        "QKD_READY_END_TO_END_ENTANGLEMENT_ESTABLISHED"
                    ),

                    component=(
                        "QKDReadyEndToEndEntanglementOrchestrator"
                    ),

                    node_names=(
                        self.source_name,
                        self.destination_name
                    ),

                    link_ids=tuple(
                        self._path_link_ids
                    ),

                    resource_ids=(
                        result
                        .end_to_end_resource_id,
                    ),

                    message=(
                        "Established end-to-end entanglement "
                        "that satisfies the configured QKD "
                        "quality admission threshold."
                    ),

                    metadata={
                        "final_fidelity":
                            result.final_fidelity,

                        "predicted_qber":
                            result.predicted_qber,

                        "qber_limit":
                            self.config.qber_limit,

                        "processed_slots":
                            result.processed_slot_count,

                        "elementary_attempts":
                            result.total_elementary_attempts,

                        "bsm_attempts":
                            result.total_bsm_attempts,

                        "collective_recoveries":
                            result
                            .total_collective_recovery_executions
                    }
                )


                return result


            # ====================================================
            # Incomplete:
            # exactly ONE global time boundary.
            # ====================================================

            self.simulator\
                .kernel\
                .advance_slot()


        # ========================================================
        # Timeout
        # ========================================================

        return (
            QKDReadyEndToEndExecutionResult(

                status=(
                    QKDReadyExecutionStatus
                    .MAX_SLOTS_EXCEEDED
                ),

                start_slot=(
                    start_slot
                ),

                end_slot=int(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                processed_slot_count=len(
                    run_history
                ),

                end_to_end_resource_id=None,

                final_fidelity=None,

                predicted_qber=None,

                total_elementary_attempts=sum(
                    record
                    .elementary_attempt_count

                    for record
                    in run_history
                ),

                total_elementary_successes=sum(
                    record
                    .elementary_success_count

                    for record
                    in run_history
                ),

                total_elementary_failures=sum(
                    record
                    .elementary_failure_count

                    for record
                    in run_history
                ),

                total_bsm_attempts=sum(
                    record
                    .bsm_attempt_count

                    for record
                    in run_history
                ),

                total_bsm_successes=sum(
                    record
                    .bsm_success_count

                    for record
                    in run_history
                ),

                total_bsm_failures=sum(
                    record
                    .bsm_failure_count

                    for record
                    in run_history
                ),

                total_individual_quality_discards=sum(

                    record
                    .pre_individual_quality_discard_count

                    +

                    record
                    .post_individual_quality_discard_count

                    for record
                    in run_history
                ),

                total_collective_recovery_executions=sum(

                    int(
                        record
                        .pre_collective_recovery_status
                        ==
                        QKDSelectiveRecoveryExecutionStatus
                        .EXECUTED
                        .value
                    )

                    +

                    int(
                        record
                        .post_collective_recovery_status
                        ==
                        QKDSelectiveRecoveryExecutionStatus
                        .EXECUTED
                        .value
                    )

                    for record
                    in run_history
                ),

                total_collective_resources_discarded=sum(

                    len(
                        record
                        .pre_collective_discarded_resource_ids
                    )

                    +

                    len(
                        record
                        .post_collective_discarded_resource_ids
                    )

                    for record
                    in run_history
                ),

                execution_history=tuple(
                    run_history
                )
            )
        )


    # ============================================================
    # Inspection
    # ============================================================

    @property
    def qkd_execution_history(
        self
    ):

        return tuple(
            self._qkd_execution_history
        )


    def qkd_history_dataframe(
        self
    ):

        return pd.DataFrame(
            [
                {
                    "Execution Step":
                        step.execution_index,

                    "Simulation Slot":
                        step.simulation_slot,

                    "Pre Individual Discards":
                        step
                        .pre_individual_quality_discard_count,

                    "Pre Collective Recovery":
                        step
                        .pre_collective_recovery_status,

                    "Pre Collective Victims":
                        step
                        .pre_collective_discarded_resource_ids,

                    "Elementary Attempts":
                        step.elementary_attempt_count,

                    "Elementary Successes":
                        step.elementary_success_count,

                    "Parallel BSM Attempts":
                        step.bsm_attempt_count,

                    "BSM Successes":
                        step.bsm_success_count,

                    "BSM Failures":
                        step.bsm_failure_count,

                    "Post Individual Discards":
                        step
                        .post_individual_quality_discard_count,

                    "Post Collective Recovery":
                        step
                        .post_collective_recovery_status,

                    "Post Collective Victims":
                        step
                        .post_collective_discarded_resource_ids,

                    "QKD Eligible":
                        step.qkd_eligible,

                    "Final Fidelity":
                        step.final_fidelity,

                    "Predicted QBER":
                        step.predicted_qber
                }

                for step
                in self._qkd_execution_history
            ]
        )

In [ ]:
# @title Closed-Loop Quality-Aware Orchestration Verification

print(
    "=" * 92
)

print(
    "Autonomous QKD-Ready Orchestration Verification"
)

print(
    "=" * 92
)


# ----------------------------------------------------------------
# Build the same internally consistent symmetric doomed frontier.
# ----------------------------------------------------------------

fixture = (
    build_phase3h2_integrated_collectively_doomed_stack(

        simulation_seed=(
            7_320_201
        ),

        age_slots=5
    )
)


phase3h2c_stack = (
    fixture[
        "stack"
    ]
)


simulator = (
    phase3h2c_stack[
        "simulator"
    ]
)


manager = (
    simulator
    .entanglement_manager
)


assert (
    simulator
    .kernel
    .current_slot
    ==
    5
)


# ----------------------------------------------------------------
# Verify starting state.
# ----------------------------------------------------------------

(
    start_resource_quality,
    start_frontier_quality
) = (
    phase3h1_quality_snapshot(

        stack=(
            phase3h2c_stack
        ),

        simulation_slot=(
            simulator
            .kernel
            .current_slot
        )
    )
)


assert (
    start_resource_quality[
        "quality_expired"
    ]
    .sum()
    ==
    0
)


assert (
    start_frontier_quality
    .iloc[0][
        "state"
    ]
    ==
    QKDFrontierQualityState
    .COLLECTIVELY_DOOMED
    .value
)


print(
    "[PASS] Starting frontier is individually viable but "
    "collectively QKD-doomed."
)


# ----------------------------------------------------------------
# New autonomous orchestrator.
# ----------------------------------------------------------------

qkd_orchestrator = (
    QKDReadyEndToEndEntanglementOrchestrator(

        stack=(
            phase3h2c_stack
        )
    )
)


slot_before = (
    simulator
    .kernel
    .current_slot
)


step = (
    qkd_orchestrator
    .process_qkd_ready_current_slot()
)


# ----------------------------------------------------------------
# Entire closed-loop control slot must remain at the same time.
# ----------------------------------------------------------------

assert (
    simulator
    .kernel
    .current_slot
    ==
    slot_before
)


assert (
    step.simulation_slot
    ==
    slot_before
)


print(
    "[PASS] Complete quality-control + physical execution "
    "advanced zero hidden simulation slots."
)


# ----------------------------------------------------------------
# Pre-physics recovery must have executed.
# ----------------------------------------------------------------

assert (
    step
    .pre_collective_recovery_status
    ==
    QKDSelectiveRecoveryExecutionStatus
    .EXECUTED
    .value
)


assert (
    len(
        step
        .pre_collective_discarded_resource_ids
    )
    ==
    2
)


print(
    "[PASS] Autonomous pre-physics control surgically replaced "
    "exactly two resources."
)


# ----------------------------------------------------------------
# Base physical layer may attempt only the two newly exposed links.
# ----------------------------------------------------------------

assert (
    step
    .elementary_attempt_count
    ==
    2
)


print(
    "[PASS] Elementary generation remained restricted to the "
    "two recovery links."
)


# ----------------------------------------------------------------
# Crucial new behavior:
#
# Existing healthy neighbouring resources are still permitted to
# swap in the SAME slot, even if the two regeneration attempts fail.
# ----------------------------------------------------------------

print(
    "[INFO] Parallel BSM attempts in same slot:",
    step.bsm_attempt_count
)


print(
    "[INFO] BSM successes:",
    step.bsm_success_count
)


print(
    "[INFO] BSM failures:",
    step.bsm_failure_count
)


assert (
    step.bsm_attempt_count
    >= 1
)


print(
    "[PASS] Preserved useful resources were immediately allowed "
    "to continue progressing through swapping."
)


# ----------------------------------------------------------------
# QKD admission invariant.
# ----------------------------------------------------------------

if (
    step.end_to_end_resource_id
    is not None
):

    assert (
        step.qkd_eligible
        is True
    )


    assert (
        step.predicted_qber
        <=
        PHASE3H1_QUALITY_CONFIG
        .qber_limit
    )


    print(
        "[PASS] Any surviving full-path result is genuinely "
        "QKD-eligible."
    )

else:

    assert (
        step.qkd_eligible
        is False
    )


    print(
        "[PASS] No ordinary-but-unusable end-to-end pair was "
        "misreported as QKD success."
    )


# ----------------------------------------------------------------
# Complete runtime consistency.
# ----------------------------------------------------------------

manager\
    .validate_all_active_resources()


assert (
    simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Runtime remains cross-layer consistent."
)


print(
    "\nQKD-AWARE EXECUTION SLOT"
)


display(
    qkd_orchestrator
    .qkd_history_dataframe()
)


print(
    "\nFINAL LIVE QKD FRONTIER"
)


final_resource_quality, final_frontier_quality = (
    phase3h1_quality_snapshot(

        stack=(
            phase3h2c_stack
        ),

        simulation_slot=(
            simulator
            .kernel
            .current_slot
        )
    )
)


display(
    final_frontier_quality
)


print(
    "\nACTIVE RESOURCES AFTER CLOSED-LOOP SLOT"
)


display(
    pd.DataFrame(
        [
            {
                "Resource":
                    resource.resource_id,

                "Kind":
                    resource.kind.value,

                "Endpoints":
                    resource.endpoint_names,

                "Source Links":
                    resource.source_link_ids,

                "Current Fidelity":
                    phase3h2c_stack[
                        "fidelity_registry"
                    ]
                    .fidelity(
                        resource
                    )
            }

            for resource
            in manager.active_resources
        ]
    )
)


print(
    "\n" + "=" * 92
)

print(
    "Autonomous QKD-ready orchestration PASSED."
)

print(
    "=" * 92
)

In [ ]:
# @title Explicit Physical Timing Calibration Model

from dataclasses import dataclass
from enum import Enum
import math

import pandas as pd


class ElementaryAttemptTimingGeometry(
    str,
    Enum
):

    # Paper-inspired:
    #
    # endpoints -> midpoint optical BSM
    # midpoint   -> endpoints herald
    #
    # Critical propagation path:
    #
    #     d/2 + d/2 = d
    #
    MIDPOINT_HERALDED = (
        "MIDPOINT_HERALDED"
    )

    # More conservative generic stop-and-wait interpretation:
    #
    #     full-link outward propagation
    #     +
    #     full-link acknowledgement propagation
    #
    ENDPOINT_ROUND_TRIP = (
        "ENDPOINT_ROUND_TRIP"
    )


@dataclass(frozen=True)
class NARADPhysicalTimingProfile:
    """
    Physical-time calibration sidecar for NARAD-QC.

    IMPORTANT
    ---------
    This object does NOT:

        - alter SimulationKernel slot ordering;
        - alter elementary success probability;
        - alter BSM success probability;
        - mutate memory;
        - mutate fidelity;
        - alter generation scheduling.

    It only answers:

        "If one logical physical operation happened,
         approximately how much modeled physical time
         should that operation represent?"

    Phase 3H.3A is therefore completely read-only.
    """

    # ------------------------------------------------------------
    # Fibre propagation
    # ------------------------------------------------------------

    fiber_group_velocity_km_per_s: float = (
        200_000.0
    )

    # ------------------------------------------------------------
    # Paper-inspired local operation anchors
    # ------------------------------------------------------------

    atom_photon_generation_time_s: float = (
        50e-6
    )

    atomic_bsm_latency_s: float = (
        10e-6
    )

    # Optical-BSM processing latency is not specified by our
    # current NARAD-QC model.
    #
    # Do NOT invent one silently.
    optical_bsm_processing_time_s: float = (
        0.0
    )

    # ------------------------------------------------------------
    # Additional hardware/control overhead
    #
    # Left explicit for later sensitivity sweeps.
    # ------------------------------------------------------------

    local_control_overhead_s: float = (
        0.0
    )

    # ------------------------------------------------------------
    # Frozen historical experiment mapping
    # ------------------------------------------------------------

    legacy_slot_duration_s: float = (
        0.05
    )


    def __post_init__(
        self
    ):

        values = {

            "fiber_group_velocity_km_per_s":
                self.fiber_group_velocity_km_per_s,

            "atom_photon_generation_time_s":
                self.atom_photon_generation_time_s,

            "atomic_bsm_latency_s":
                self.atomic_bsm_latency_s,

            "optical_bsm_processing_time_s":
                self.optical_bsm_processing_time_s,

            "local_control_overhead_s":
                self.local_control_overhead_s,

            "legacy_slot_duration_s":
                self.legacy_slot_duration_s
        }


        for name, value in values.items():

            if (
                isinstance(
                    value,
                    bool
                )
                or not isinstance(
                    value,
                    (int, float)
                )
                or not math.isfinite(
                    float(
                        value
                    )
                )
            ):

                raise ValueError(
                    f"{name} must be a finite real number."
                )


        if (
            self.fiber_group_velocity_km_per_s
            <= 0.0
        ):

            raise ValueError(
                "fiber_group_velocity_km_per_s "
                "must be positive."
            )


        for name in (

            "atom_photon_generation_time_s",
            "atomic_bsm_latency_s",
            "optical_bsm_processing_time_s",
            "local_control_overhead_s"

        ):

            if (
                getattr(
                    self,
                    name
                )
                <
                0.0
            ):

                raise ValueError(
                    f"{name} cannot be negative."
                )


        if (
            self.legacy_slot_duration_s
            <= 0.0
        ):

            raise ValueError(
                "legacy_slot_duration_s must be positive."
            )


    # ============================================================
    # Propagation
    # ============================================================

    def one_full_link_propagation_s(
        self,
        distance_km
    ) -> float:

        distance_km = float(
            distance_km
        )


        if (
            not math.isfinite(
                distance_km
            )
            or distance_km < 0.0
        ):

            raise ValueError(
                "distance_km must be a non-negative "
                "finite number."
            )


        return float(

            distance_km
            /
            self.fiber_group_velocity_km_per_s
        )


    def propagation_critical_path_s(
        self,
        distance_km,
        geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    ) -> float:

        if not isinstance(
            geometry,
            ElementaryAttemptTimingGeometry
        ):

            geometry = (
                ElementaryAttemptTimingGeometry(
                    geometry
                )
            )


        one_link_time = (
            self.one_full_link_propagation_s(
                distance_km
            )
        )


        if (
            geometry
            ==
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        ):

            # d/2 quantum outward
            # +
            # d/2 classical herald return
            return float(
                one_link_time
            )


        if (
            geometry
            ==
            ElementaryAttemptTimingGeometry
            .ENDPOINT_ROUND_TRIP
        ):

            return float(
                2.0
                *
                one_link_time
            )


        raise RuntimeError(
            "Unsupported timing geometry."
        )


    # ============================================================
    # Elementary Attempt Cycle
    # ============================================================

    def elementary_attempt_cycle_s(
        self,
        distance_km,
        geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    ) -> float:
        """
        One stop-and-wait elementary-generation opportunity.

        Current abstraction:

            local atom-photon preparation
            +
            propagation/heralding critical path
            +
            optional optical-BSM processing
            +
            explicitly configured control overhead

        Local operations at the two endpoints are assumed parallel,
        so atom_photon_generation_time_s is NOT doubled.
        """

        return float(

            self.atom_photon_generation_time_s

            +

            self.propagation_critical_path_s(

                distance_km=(
                    distance_km
                ),

                geometry=(
                    geometry
                )
            )

            +

            self.optical_bsm_processing_time_s

            +

            self.local_control_overhead_s
        )


    # ============================================================
    # Expected Link Generation Latency
    # ============================================================

    def expected_generation_latency_s(
        self,
        distance_km,
        success_probability,
        geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    ) -> float:

        success_probability = float(
            success_probability
        )


        if not (
            0.0
            <=
            success_probability
            <=
            1.0
        ):

            raise ValueError(
                "success_probability must lie in [0,1]."
            )


        if (
            success_probability
            ==
            0.0
        ):

            return math.inf


        return float(

            self.elementary_attempt_cycle_s(

                distance_km=(
                    distance_km
                ),

                geometry=(
                    geometry
                )
            )

            /

            success_probability
        )


    # ============================================================
    # Legacy Timing
    # ============================================================

    def legacy_expected_generation_latency_s(
        self,
        success_probability
    ) -> float:

        success_probability = float(
            success_probability
        )


        if (
            success_probability
            ==
            0.0
        ):

            return math.inf


        return float(

            self.legacy_slot_duration_s

            /
            success_probability
        )


PHASE3H3_TIMING_PROFILE = (
    NARADPhysicalTimingProfile()
)

In [ ]:
# @title Timing Audit Across the Existing 600-km Density Sweep

def phase3h3_elementary_success_proxy(
    distance_km,
    attenuation_db_per_km=(
        PHASE3G2_FIBER_ATTENUATION_DB_PER_KM
    )
):

    return float(

        10.0
        **
        (
            -
            (
                float(
                    attenuation_db_per_km
                )
                *
                float(
                    distance_km
                )
            )
            /
            10.0
        )
    )


timing_rows = []


for repeater_count in range(
    3,
    10
):

    elementary_links = (
        repeater_count
        +
        1
    )


    hop_distance_km = (

        PHASE3G2_TOTAL_DISTANCE_KM
        /
        elementary_links
    )


    p_elem = (
        phase3h3_elementary_success_proxy(
            hop_distance_km
        )
    )


    midpoint_cycle_s = (
        PHASE3H3_TIMING_PROFILE
        .elementary_attempt_cycle_s(

            distance_km=(
                hop_distance_km
            ),

            geometry=(
                ElementaryAttemptTimingGeometry
                .MIDPOINT_HERALDED
            )
        )
    )


    conservative_cycle_s = (
        PHASE3H3_TIMING_PROFILE
        .elementary_attempt_cycle_s(

            distance_km=(
                hop_distance_km
            ),

            geometry=(
                ElementaryAttemptTimingGeometry
                .ENDPOINT_ROUND_TRIP
            )
        )
    )


    midpoint_expected_s = (
        PHASE3H3_TIMING_PROFILE
        .expected_generation_latency_s(

            distance_km=(
                hop_distance_km
            ),

            success_probability=(
                p_elem
            ),

            geometry=(
                ElementaryAttemptTimingGeometry
                .MIDPOINT_HERALDED
            )
        )
    )


    conservative_expected_s = (
        PHASE3H3_TIMING_PROFILE
        .expected_generation_latency_s(

            distance_km=(
                hop_distance_km
            ),

            success_probability=(
                p_elem
            ),

            geometry=(
                ElementaryAttemptTimingGeometry
                .ENDPOINT_ROUND_TRIP
            )
        )
    )


    legacy_expected_s = (
        PHASE3H3_TIMING_PROFILE
        .legacy_expected_generation_latency_s(
            p_elem
        )
    )


    timing_rows.append({

        "RepeaterCount":
            repeater_count,

        "ElementaryLinks":
            elementary_links,

        "HopDistanceKm":
            hop_distance_km,

        "ElementarySuccessProbability":
            p_elem,

        "ExpectedAttempts":
            (
                1.0
                /
                p_elem
            ),

        "LegacyAttemptCycleMs":
            (
                PHASE3H3_TIMING_PROFILE
                .legacy_slot_duration_s
                *
                1000.0
            ),

        "MidpointAttemptCycleMs":
            (
                midpoint_cycle_s
                *
                1000.0
            ),

        "ConservativeAttemptCycleMs":
            (
                conservative_cycle_s
                *
                1000.0
            ),

        "LegacyExpectedGenerationMs":
            (
                legacy_expected_s
                *
                1000.0
            ),

        "MidpointExpectedGenerationMs":
            (
                midpoint_expected_s
                *
                1000.0
            ),

        "ConservativeExpectedGenerationMs":
            (
                conservative_expected_s
                *
                1000.0
            ),

        "LegacyVsMidpointSlowdown":
            (
                legacy_expected_s
                /
                midpoint_expected_s
            ),

        "LegacyVsConservativeSlowdown":
            (
                legacy_expected_s
                /
                conservative_expected_s
            )
    })


phase3h3_density_timing_df = (
    pd.DataFrame(
        timing_rows
    )
)


display(
    phase3h3_density_timing_df
)


# ================================================================
# R6 / 600-km calibration checkpoint
# ================================================================

r6_timing = (

    phase3h3_density_timing_df[

        phase3h3_density_timing_df[
            "RepeaterCount"
        ]
        ==
        6

    ]
    .iloc[0]
)


assert math.isclose(

    float(
        r6_timing[
            "HopDistanceKm"
        ]
    ),

    600.0 / 7.0,

    rel_tol=0.0,

    abs_tol=1e-12
)


assert (
    float(
        r6_timing[
            "MidpointAttemptCycleMs"
        ]
    )
    <
    float(
        r6_timing[
            "LegacyAttemptCycleMs"
        ]
    )
)


assert (
    float(
        r6_timing[
            "ConservativeAttemptCycleMs"
        ]
    )
    <
    float(
        r6_timing[
            "LegacyAttemptCycleMs"
        ]
    )
)


print(
    "=" * 78
)

print(
    "600-km Density Timing Audit"
)

print(
    "=" * 78
)


print(
    "R6 hop distance (km):",
    r6_timing[
        "HopDistanceKm"
    ]
)


print(
    "R6 current P_elem:",
    r6_timing[
        "ElementarySuccessProbability"
    ]
)


print(
    "Legacy 50-ms expected link generation latency:",
    f"{r6_timing['LegacyExpectedGenerationMs']:.3f} ms"
)


print(
    "Midpoint-heralded expected link generation latency:",
    f"{r6_timing['MidpointExpectedGenerationMs']:.3f} ms"
)


print(
    "Conservative endpoint-round-trip expected latency:",
    f"{r6_timing['ConservativeExpectedGenerationMs']:.3f} ms"
)


print(
    "Legacy / midpoint timing inflation:",
    f"{r6_timing['LegacyVsMidpointSlowdown']:.2f}x"
)


print(
    "Legacy / conservative timing inflation:",
    f"{r6_timing['LegacyVsConservativeSlowdown']:.2f}x"
)

In [ ]:
# @title Timing -> Memory Quality Cost Audit


quality_cfg = (
    PHASE3H1_QUALITY_CONFIG
)


memory_lambda = (
    quality_cfg
    .memory_decay_rate_per_s
)


r6_hop_km = (
    600.0
    /
    7.0
)


r6_p_elem = (
    phase3h3_elementary_success_proxy(
        r6_hop_km
    )
)


legacy_wait_s = (
    PHASE3H3_TIMING_PROFILE
    .legacy_expected_generation_latency_s(
        r6_p_elem
    )
)


midpoint_wait_s = (
    PHASE3H3_TIMING_PROFILE
    .expected_generation_latency_s(

        distance_km=(
            r6_hop_km
        ),

        success_probability=(
            r6_p_elem
        ),

        geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    )
)


conservative_wait_s = (
    PHASE3H3_TIMING_PROFILE
    .expected_generation_latency_s(

        distance_km=(
            r6_hop_km
        ),

        success_probability=(
            r6_p_elem
        ),

        geometry=(
            ElementaryAttemptTimingGeometry
            .ENDPOINT_ROUND_TRIP
        )
    )
)


def quality_cost_for_wait(
    wait_s
):

    return float(

        memory_lambda
        *
        float(
            wait_s
        )
    )


audit_rows = []


for timing_name, wait_s in (

    (
        "LEGACY_50MS_SLOT",
        legacy_wait_s
    ),

    (
        "MIDPOINT_HERALDED",
        midpoint_wait_s
    ),

    (
        "CONSERVATIVE_ENDPOINT_ROUND_TRIP",
        conservative_wait_s
    )

):

    waiting_quality_cost = (
        quality_cost_for_wait(
            wait_s
        )
    )


    optimistic_total_cost = (

        # Existing fresh seven-link intrinsic cost
        7
        *
        quality_cfg
        .elementary_quality_cost

        +

        waiting_quality_cost
    )


    optimistic_visibility = float(
        math.exp(
            -optimistic_total_cost
        )
    )


    audit_rows.append({

        "TimingModel":
            timing_name,

        "ExpectedWaitSeconds":
            wait_s,

        "ExpectedWaitMilliseconds":
            (
                wait_s
                *
                1000.0
            ),

        "MemoryQualityCostFromOneExpectedWait":
            waiting_quality_cost,

        "FreshSevenLinkIntrinsicCost":
            (
                7
                *
                quality_cfg
                .elementary_quality_cost
            ),

        "OptimisticTotalCost":
            optimistic_total_cost,

        "OptimisticFidelity":
            narad_fidelity_from_visibility(
                optimistic_visibility
            ),

        "OptimisticQBER":
            narad_qber_from_visibility(
                optimistic_visibility
            ),

        "PassesQKDTarget":
            (
                narad_qber_from_visibility(
                    optimistic_visibility
                )
                <=
                quality_cfg
                .qber_limit
            )
    })


phase3h3_quality_timing_df = (
    pd.DataFrame(
        audit_rows
    )
)


display(
    phase3h3_quality_timing_df
)


legacy_row = (
    phase3h3_quality_timing_df[
        phase3h3_quality_timing_df[
            "TimingModel"
        ]
        ==
        "LEGACY_50MS_SLOT"
    ]
    .iloc[0]
)


midpoint_row = (
    phase3h3_quality_timing_df[
        phase3h3_quality_timing_df[
            "TimingModel"
        ]
        ==
        "MIDPOINT_HERALDED"
    ]
    .iloc[0]
)


conservative_row = (
    phase3h3_quality_timing_df[
        phase3h3_quality_timing_df[
            "TimingModel"
        ]
        ==
        "CONSERVATIVE_ENDPOINT_ROUND_TRIP"
    ]
    .iloc[0]
)


assert (
    midpoint_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
    <
    legacy_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
)


assert (
    conservative_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
    <
    legacy_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
)


print(
    "=" * 78
)

print(
    "Timing / QKD Quality-Cost Audit"
)

print(
    "=" * 78
)


print(
    "Memory decay rate lambda:",
    memory_lambda,
    "1/s"
)


print(
    "Legacy one-wait quality cost:",
    legacy_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
)


print(
    "Midpoint one-wait quality cost:",
    midpoint_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
)


print(
    "Conservative one-wait quality cost:",
    conservative_row[
        "MemoryQualityCostFromOneExpectedWait"
    ]
)


print(
    "[PASS] Physical timing interpretation materially changes "
    "the memory-quality budget."
)

In [ ]:
# @title Continuous Modeled-Physical-Time Ledger

from dataclasses import dataclass
import math

import pandas as pd


@dataclass(frozen=True)
class PhysicalTimeAdvanceRecord:

    advance_index: int

    simulation_slot: int

    phase_name: str

    start_time_s: float

    duration_s: float

    end_time_s: float

    link_ids: tuple[str, ...]

    resource_ids: tuple[str, ...]

    metadata: dict


class PhysicalTimeLedger:
    """
    Continuous modeled-physical-time sidecar.

    SimulationKernel.current_slot remains responsible ONLY for
    deterministic logical ordering.

    This ledger owns:

        modeled physical seconds.

    Therefore:

        logical slot != physical duration

    A single logical slot may contain several physical phases:

        elementary generation
        BSM execution
        heralding / control
        etc.

    Advancing this ledger NEVER advances SimulationKernel.current_slot.
    """

    def __init__(
        self,
        kernel
    ):

        if not isinstance(
            kernel,
            SimulationKernel
        ):

            raise TypeError(
                "kernel must be a SimulationKernel."
            )


        self.kernel = kernel

        self._current_time_s = 0.0

        self._history = []


    # ============================================================
    # Current Physical Time
    # ============================================================

    @property
    def current_time_s(
        self
    ) -> float:

        return float(
            self._current_time_s
        )


    # ============================================================
    # Explicit Physical-Time Advancement
    # ============================================================

    def advance(
        self,
        duration_s,
        phase_name,
        link_ids=(),
        resource_ids=(),
        metadata=None
    ) -> PhysicalTimeAdvanceRecord:

        duration_s = float(
            duration_s
        )


        if (
            not math.isfinite(
                duration_s
            )
            or duration_s < 0.0
        ):

            raise ValueError(
                "duration_s must be a non-negative finite number."
            )


        if (
            not isinstance(
                phase_name,
                str
            )
            or not phase_name.strip()
        ):

            raise ValueError(
                "phase_name must be a non-empty string."
            )


        logical_slot_before = (
            self.kernel.current_slot
        )


        start_time_s = float(
            self._current_time_s
        )


        end_time_s = (
            start_time_s
            +
            duration_s
        )


        self._current_time_s = float(
            end_time_s
        )


        record = (
            PhysicalTimeAdvanceRecord(

                advance_index=len(
                    self._history
                ),

                simulation_slot=(
                    logical_slot_before
                ),

                phase_name=(
                    phase_name.strip()
                ),

                start_time_s=(
                    start_time_s
                ),

                duration_s=(
                    duration_s
                ),

                end_time_s=(
                    end_time_s
                ),

                link_ids=tuple(
                    link_ids
                ),

                resource_ids=tuple(
                    resource_ids
                ),

                metadata=dict(
                    metadata or {}
                )
            )
        )


        self._history.append(
            record
        )


        # Physical time must NEVER secretly advance logical order.
        if (
            self.kernel.current_slot
            != logical_slot_before
        ):

            raise RuntimeError(
                "PhysicalTimeLedger changed SimulationKernel slot."
            )


        self.kernel.emit(

            event_type=(
                "MODELED_PHYSICAL_TIME_ADVANCED"
            ),

            component=(
                "PhysicalTimeLedger"
            ),

            link_ids=(
                record.link_ids
            ),

            resource_ids=(
                record.resource_ids
            ),

            message=(
                f"Advanced modeled physical time by "
                f"{duration_s:.9f} s during "
                f"{record.phase_name}."
            ),

            metadata={
                "phase_name":
                    record.phase_name,

                "start_time_s":
                    record.start_time_s,

                "duration_s":
                    record.duration_s,

                "end_time_s":
                    record.end_time_s
            }
        )


        return record


    # ============================================================
    # History
    # ============================================================

    @property
    def history(
        self
    ):

        return tuple(
            self._history
        )


    def history_dataframe(
        self
    ):

        return pd.DataFrame(
            [
                {
                    "Advance Index":
                        record.advance_index,

                    "Simulation Slot":
                        record.simulation_slot,

                    "Phase":
                        record.phase_name,

                    "Start Physical Time (s)":
                        record.start_time_s,

                    "Duration (s)":
                        record.duration_s,

                    "End Physical Time (s)":
                        record.end_time_s,

                    "Links":
                        record.link_ids,

                    "Resources":
                        record.resource_ids,

                    "Metadata":
                        record.metadata
                }

                for record
                in self._history
            ]
        )

In [ ]:
# @title Continuous-Time Physical Fidelity Registry

class PhysicalTimeAwareEntanglementFidelityRegistry(
    PhysicalEntanglementFidelityRegistry
):
    """
    Continuous-time replacement for the fixed-slot-duration
    interpretation of PhysicalEntanglementFidelityRegistry.

    Compatibility
    -------------
    EntanglementFidelityRecord remains unchanged:

        recorded_slot

    still records logical event ordering.

    A separate physical checkpoint:

        recorded_time_s

    stores the actual modeled physical time.

    SwapEngine compatibility is preserved because it already calls:

        fidelity_registry.fidelity(...)
        fidelity_registry.register(...)

    which are overridden here.
    """

    def __init__(
        self,
        simulator,
        profile,
        time_ledger
    ):

        super().__init__(

            simulator=(
                simulator
            ),

            profile=(
                profile
            )
        )


        if not isinstance(
            time_ledger,
            PhysicalTimeLedger
        ):

            raise TypeError(
                "time_ledger must be a PhysicalTimeLedger."
            )


        if (
            time_ledger.kernel
            is not
            simulator.kernel
        ):

            raise ValueError(
                "time_ledger and fidelity registry must share "
                "the same SimulationKernel."
            )


        self.time_ledger = (
            time_ledger
        )


        self._recorded_time_s = {}


    # ============================================================
    # Generic Fidelity Checkpoint
    # ============================================================

    def register(
        self,
        resource,
        fidelity,
        model_name
    ):

        record = (
            super()
            .register(

                resource=(
                    resource
                ),

                fidelity=(
                    fidelity
                ),

                model_name=(
                    model_name
                )
            )
        )


        self._recorded_time_s[
            resource.resource_id
        ] = float(
            self.time_ledger
            .current_time_s
        )


        return record


    # ============================================================
    # Elementary Fidelity Checkpoint
    # ============================================================

    def register_elementary(
        self,
        resource
    ):

        record = (
            super()
            .register_elementary(
                resource
            )
        )


        self._recorded_time_s[
            resource.resource_id
        ] = float(
            self.time_ledger
            .current_time_s
        )


        return record


    # ============================================================
    # Physical Checkpoint Inspection
    # ============================================================

    def recorded_time_s(
        self,
        resource_or_id
    ) -> float:

        resource_id = (

            resource_or_id.resource_id

            if isinstance(
                resource_or_id,
                EntanglementResource
            )

            else str(
                resource_or_id
            )
        )


        if (
            resource_id
            not in self._recorded_time_s
        ):

            raise KeyError(
                f"No physical-time fidelity checkpoint exists "
                f"for '{resource_id}'."
            )


        return float(
            self._recorded_time_s[
                resource_id
            ]
        )


    # ============================================================
    # Fidelity at Explicit Physical Time
    # ============================================================

    def fidelity_at_time(
        self,
        resource_or_id,
        physical_time_s
    ) -> float:

        physical_time_s = float(
            physical_time_s
        )


        if (
            not math.isfinite(
                physical_time_s
            )
            or physical_time_s < 0.0
        ):

            raise ValueError(
                "physical_time_s must be a non-negative "
                "finite number."
            )


        record = (
            self.get(
                resource_or_id
            )
        )


        checkpoint_time_s = (
            self.recorded_time_s(
                resource_or_id
            )
        )


        if (
            physical_time_s
            <
            checkpoint_time_s
            -
            1e-15
        ):

            raise PhysicalFidelityModelError(
                "Cannot evaluate fidelity before its "
                "physical checkpoint time."
            )


        elapsed_seconds = max(

            0.0,

            physical_time_s
            -
            checkpoint_time_s
        )


        return (
            self.profile
            .memory_fidelity(

                initial_fidelity=(
                    record.fidelity
                ),

                elapsed_seconds=(
                    elapsed_seconds
                )
            )
        )


    # ============================================================
    # Current Fidelity
    # ============================================================

    def fidelity(
        self,
        resource_or_id
    ) -> float:

        return (
            self.fidelity_at_time(

                resource_or_id=(
                    resource_or_id
                ),

                physical_time_s=(
                    self.time_ledger
                    .current_time_s
                )
            )
        )


    current_fidelity = fidelity


    # ============================================================
    # Deliberately Disable Slot-Based Physical Evaluation
    # ============================================================

    def fidelity_at_slot(
        self,
        resource_or_id,
        simulation_slot
    ):

        raise PhysicalFidelityModelError(
            "PhysicalTimeAwareEntanglementFidelityRegistry "
            "does not infer physical seconds from logical slots. "
            "Use fidelity_at_time(...)."
        )


    # ============================================================
    # Diagnostics
    # ============================================================

    def dataframe(
        self
    ):

        rows = []


        for record in (
            self._records
            .values()
        ):

            resource = (
                self.simulator
                .entanglement_manager
                .get_resource(
                    record.resource_id
                )
            )


            checkpoint_time_s = (
                self.recorded_time_s(
                    record.resource_id
                )
            )


            elapsed_physical_time_s = max(

                0.0,

                self.time_ledger
                .current_time_s

                -
                checkpoint_time_s
            )


            current_fidelity = None


            if (
                resource.is_active
            ):

                current_fidelity = (
                    self.fidelity(
                        record.resource_id
                    )
                )


            rows.append({

                "Resource ID":
                    record.resource_id,

                "Kind":
                    resource.kind.value,

                "Status":
                    resource.status.value,

                "Checkpoint Fidelity":
                    record.fidelity,

                "Checkpoint Logical Slot":
                    record.recorded_slot,

                "Checkpoint Physical Time (s)":
                    checkpoint_time_s,

                "Current Physical Time (s)":
                    self.time_ledger
                    .current_time_s,

                "Elapsed Physical Time (s)":
                    elapsed_physical_time_s,

                "Current Fidelity":
                    current_fidelity,

                "Model":
                    record.model_name,

                "Parents":
                    record.parent_resource_ids
            })


        return pd.DataFrame(
            rows
        )

In [ ]:
# @title Logical-Time / Physical-Time Decoupling Verification

print(
    "=" * 88
)

print(
    "Logical / Physical Time Decoupling Verification"
)

print(
    "=" * 88
)


# ----------------------------------------------------------------
# Use the existing R6 physical topology only as a convenient
# authenticated test fixture.
# ----------------------------------------------------------------

stack = (
    build_phase3g2_trial_stack(

        policy_name=(
            "MAX_PARALLEL_OLDEST_FIRST"
        ),

        repeater_count=6,

        simulation_seed=(
            7_330_001
        )
    )
)


simulator = (
    stack[
        "simulator"
    ]
)


manager = (
    simulator
    .entanglement_manager
)


path = (
    stack[
        "path"
    ]
)


path_links = tuple(
    stack[
        "topology"
    ]
    .path_links(
        path
    )
)


fidelity_profile = (
    stack[
        "fidelity_profile"
    ]
)


# ----------------------------------------------------------------
# New continuous physical-time sidecars.
# ----------------------------------------------------------------

physical_time = (
    PhysicalTimeLedger(

        kernel=(
            simulator.kernel
        )
    )
)


continuous_registry = (
    PhysicalTimeAwareEntanglementFidelityRegistry(

        simulator=(
            simulator
        ),

        profile=(
            fidelity_profile
        ),

        time_ledger=(
            physical_time
        )
    )
)


# ----------------------------------------------------------------
# Create two adjacent elementary resources at physical time t = 0.
# ----------------------------------------------------------------

left_resource = (
    manager
    .create_elementary_resource(
        path_links[0]
    )
)


right_resource = (
    manager
    .create_elementary_resource(
        path_links[1]
    )
)


continuous_registry\
    .register_elementary(
        left_resource
    )


continuous_registry\
    .register_elementary(
        right_resource
    )


initial_left_fidelity = (
    continuous_registry
    .fidelity(
        left_resource
    )
)


initial_right_fidelity = (
    continuous_registry
    .fidelity(
        right_resource
    )
)


assert math.isclose(

    initial_left_fidelity,

    fidelity_profile
    .elementary_fidelity,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    initial_right_fidelity,

    fidelity_profile
    .elementary_fidelity,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Two elementary resources begin at the configured "
    "elementary fidelity."
)


# ================================================================
# TEST 1:
# Advance LOGICAL slots only.
#
# Physical time remains exactly zero.
# Fidelity must therefore remain unchanged.
# ================================================================

simulator\
    .kernel\
    .advance_slot(
        7
    )


assert (
    simulator
    .kernel
    .current_slot
    ==
    7
)


assert math.isclose(

    physical_time
    .current_time_s,

    0.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    continuous_registry
    .fidelity(
        left_resource
    ),

    initial_left_fidelity,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    continuous_registry
    .fidelity(
        right_resource
    ),

    initial_right_fidelity,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Seven logical scheduling slots caused ZERO "
    "physical decoherence."
)


# ================================================================
# TEST 2:
# Advance PHYSICAL time only.
#
# Use one R6 midpoint-heralded elementary-attempt cycle.
# Logical slot must remain 7.
# ================================================================

r6_cycle_s = (
    PHASE3H3_TIMING_PROFILE
    .elementary_attempt_cycle_s(

        distance_km=(
            path_links[0]
            .distance_km
        ),

        geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    )
)


physical_time\
    .advance(

        duration_s=(
            r6_cycle_s
        ),

        phase_name=(
            "VERIFICATION_ELEMENTARY_CYCLE"
        ),

        link_ids=(
            path_links[0].link_id,
            path_links[1].link_id
        )
    )


assert (
    simulator
    .kernel
    .current_slot
    ==
    7
)


assert math.isclose(

    physical_time
    .current_time_s,

    r6_cycle_s,

    rel_tol=0.0,

    abs_tol=1e-15
)


expected_after_cycle = (
    fidelity_profile
    .memory_fidelity(

        initial_fidelity=(
            fidelity_profile
            .elementary_fidelity
        ),

        elapsed_seconds=(
            r6_cycle_s
        )
    )
)


actual_left_after_cycle = (
    continuous_registry
    .fidelity(
        left_resource
    )
)


actual_right_after_cycle = (
    continuous_registry
    .fidelity(
        right_resource
    )
)


assert math.isclose(

    actual_left_after_cycle,

    expected_after_cycle,

    rel_tol=0.0,

    abs_tol=1e-12
)


assert math.isclose(

    actual_right_after_cycle,

    expected_after_cycle,

    rel_tol=0.0,

    abs_tol=1e-12
)


print(
    "[PASS] Physical-time advancement caused exactly the "
    "predicted memory decoherence without changing logical slot."
)


# ================================================================
# TEST 3:
# Existing SwapEngine compatibility.
#
# Use deterministic p_b = 1 ONLY for this verification.
# ================================================================

verification_swap_model = (
    WernerStateAtomicBSMModel(

        fidelity_profile=(
            fidelity_profile
        ),

        atomic_bsm_success_probability=1.0,

        atomic_bsm_latency_us=10.0
    )
)


verification_swap_engine = (
    EntanglementSwapEngine(

        simulator=(
            simulator
        ),

        model=(
            verification_swap_model
        ),

        fidelity_registry=(
            continuous_registry
        )
    )
)


left_fidelity_at_swap = (
    continuous_registry
    .fidelity(
        left_resource
    )
)


right_fidelity_at_swap = (
    continuous_registry
    .fidelity(
        right_resource
    )
)


swap_attempt = (
    verification_swap_engine
    .attempt_swap(

        left_resource_id=(
            left_resource
            .resource_id
        ),

        right_resource_id=(
            right_resource
            .resource_id
        ),

        swap_node=(
            path[1]
            .name
        )
    )
)


assert (
    swap_attempt.success
    is True
)


assert (
    swap_attempt.output_resource_id
    is not None
)


expected_swap_output = (
    verification_swap_model
    .output_fidelity(

        left_fidelity=(
            left_fidelity_at_swap
        ),

        right_fidelity=(
            right_fidelity_at_swap
        )
    )
)


assert math.isclose(

    swap_attempt
    .output_fidelity,

    expected_swap_output,

    rel_tol=0.0,

    abs_tol=1e-12
)


assert math.isclose(

    continuous_registry
    .recorded_time_s(
        swap_attempt
        .output_resource_id
    ),

    physical_time
    .current_time_s,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Existing atomic SwapEngine registered its output "
    "at the correct physical-time checkpoint."
)


# ================================================================
# TEST 4:
# BSM physical latency advances physical time only.
# ================================================================

logical_slot_before_bsm_latency = (
    simulator
    .kernel
    .current_slot
)


swap_checkpoint_fidelity = (
    swap_attempt
    .output_fidelity
)


physical_time\
    .advance(

        duration_s=(
            PHASE3H3_TIMING_PROFILE
            .atomic_bsm_latency_s
        ),

        phase_name=(
            "VERIFICATION_ATOMIC_BSM_LATENCY"
        ),

        resource_ids=(
            swap_attempt
            .output_resource_id,
        )
    )


assert (
    simulator
    .kernel
    .current_slot
    ==
    logical_slot_before_bsm_latency
)


expected_after_bsm_latency = (
    fidelity_profile
    .memory_fidelity(

        initial_fidelity=(
            swap_checkpoint_fidelity
        ),

        elapsed_seconds=(
            PHASE3H3_TIMING_PROFILE
            .atomic_bsm_latency_s
        )
    )
)


actual_after_bsm_latency = (
    continuous_registry
    .fidelity(
        swap_attempt
        .output_resource_id
    )
)


assert math.isclose(

    actual_after_bsm_latency,

    expected_after_bsm_latency,

    rel_tol=0.0,

    abs_tol=1e-12
)


print(
    "[PASS] Atomic-BSM physical latency aged the swapped "
    "resource without advancing logical scheduling time."
)


# ================================================================
# Cross-layer consistency.
# ================================================================

manager\
    .validate_all_active_resources()


assert (
    simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] Runtime remains cross-layer consistent."
)


# ================================================================
# Diagnostics
# ================================================================

print(
    "\nPHYSICAL TIME HISTORY"
)


display(
    physical_time
    .history_dataframe()
)


print(
    "\nCONTINUOUS-TIME FIDELITY REGISTRY"
)


display(
    continuous_registry
    .dataframe()
)


print(
    "\n" + "=" * 88
)

print(
    "Physical-time decoupling PASSED."
)

print(
    "=" * 88
)

In [ ]:
# @title Public Entanglement-Manager Expiration Transition


def _phase3h3_expire_resource(
    self,
    resource_id,
    reason="physical_memory_lifetime_exceeded"
):
    """
    Public lifecycle transition for explicit physical-memory expiry.

    EntanglementManager remains the sole authority that:
        - removes both endpoint halves;
        - updates global resource status;
        - records ended_slot;
        - emits lifecycle events.

    Higher-level timing controllers decide WHEN expiry is warranted.
    """

    return (
        self._terminate_resource(

            resource_id=(
                resource_id
            ),

            terminal_status=(
                EntanglementStatus.EXPIRED
            ),

            reason=(
                reason
            )
        )
    )


# Add only if an equivalent public API does not already exist.
if not hasattr(
    EntanglementManager,
    "expire_resource"
):

    EntanglementManager.expire_resource = (
        _phase3h3_expire_resource
    )


assert callable(
    getattr(
        EntanglementManager,
        "expire_resource",
        None
    )
)


print(
    "[PASS] EntanglementManager now exposes a public "
    "EXPIRED lifecycle transition."
)

In [ ]:
# @title Physical-Time Quantum-Memory Lifetime Controller

from dataclasses import dataclass, asdict
import math

import pandas as pd


# ----------------------------------------------------------------
# Preserve the old experiment's intended hard lifetime in SECONDS.
#
# This is a model parameter, not a universal hardware claim.
# ----------------------------------------------------------------

PHASE3H3_MAX_STORAGE_TIME_S = (

    PHASE3G2_MAX_STORAGE_SLOTS
    *
    PHASE3G2_SLOT_DURATION_S
)


assert math.isclose(

    PHASE3H3_MAX_STORAGE_TIME_S,

    5.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


@dataclass(frozen=True)
class PhysicalQubitLifetimeAssessment:

    simulation_slot: int

    physical_time_s: float

    resource_id: str

    resource_kind: str

    endpoint_a_name: str
    endpoint_b_name: str

    endpoint_a_birth_time_s: float
    endpoint_b_birth_time_s: float

    endpoint_a_age_s: float
    endpoint_b_age_s: float

    oldest_physical_qubit_age_s: float

    max_storage_time_s: float

    remaining_hard_lifetime_s: float

    physically_expired: bool


@dataclass(frozen=True)
class PhysicalQubitExpirationRecord:

    simulation_slot: int

    physical_time_s: float

    resource_id: str

    oldest_physical_qubit_age_s: float

    max_storage_time_s: float

    reason: str


class PhysicalQubitLifetimeController:
    """
    Hard physical-memory lifetime authority.

    Key principle
    -------------
    Entanglement-resource identity may change after swapping,
    but surviving OUTER PHYSICAL QUBITS do not become young again.

    Therefore hard-memory age follows endpoint-qubit ancestry.

    Example:

        elementary EP created at t = 0
                |
                | waits 4 s
                v
        successful swap at t = 4 s
                |
                v
        new SWAPPED resource

    The new resource has a fidelity checkpoint at t = 4 s,
    but its surviving outer qubit is still physically 4 s old.

    This controller reconstructs that age through
    parent_resource_ids.
    """

    def __init__(
        self,
        simulator,
        fidelity_registry,
        time_ledger,
        max_storage_time_s=(
            PHASE3H3_MAX_STORAGE_TIME_S
        ),
        numerical_tolerance=1e-12
    ):

        if not isinstance(
            simulator,
            QuantumNetworkSimulator
        ):

            raise TypeError(
                "simulator must be a QuantumNetworkSimulator."
            )


        if not isinstance(
            fidelity_registry,
            PhysicalTimeAwareEntanglementFidelityRegistry
        ):

            raise TypeError(
                "fidelity_registry must be a "
                "PhysicalTimeAwareEntanglementFidelityRegistry."
            )


        if not isinstance(
            time_ledger,
            PhysicalTimeLedger
        ):

            raise TypeError(
                "time_ledger must be a PhysicalTimeLedger."
            )


        if (
            fidelity_registry.simulator
            is not simulator
        ):

            raise ValueError(
                "Fidelity registry belongs to another simulator."
            )


        if (
            time_ledger.kernel
            is not simulator.kernel
        ):

            raise ValueError(
                "Physical-time ledger belongs to another kernel."
            )


        max_storage_time_s = float(
            max_storage_time_s
        )


        if (
            not math.isfinite(
                max_storage_time_s
            )
            or
            max_storage_time_s <= 0.0
        ):

            raise ValueError(
                "max_storage_time_s must be a positive "
                "finite number."
            )


        self.simulator = simulator

        self.manager = (
            simulator
            .entanglement_manager
        )

        self.fidelity_registry = (
            fidelity_registry
        )

        self.time_ledger = (
            time_ledger
        )

        self.max_storage_time_s = (
            max_storage_time_s
        )

        self.numerical_tolerance = float(
            numerical_tolerance
        )


        self._birth_time_cache = {}

        self._expiration_history = []


    # ============================================================
    # Physical Birth-Time Reconstruction
    # ============================================================

    def endpoint_qubit_birth_time_s(
        self,
        resource_or_id,
        endpoint_name
    ) -> float:
        """
        Return the physical creation/storage time of the qubit
        currently surviving at one endpoint of a resource.

        ELEMENTARY:
            birth time == elementary fidelity checkpoint time.

        SWAPPED:
            recursively follow the unique parent that contained
            this OUTER endpoint.

        This preserves physical-qubit age through arbitrary
        successful swap depth.
        """

        resource = (

            resource_or_id

            if isinstance(
                resource_or_id,
                EntanglementResource
            )

            else
            self.manager.get_resource(
                str(
                    resource_or_id
                )
            )
        )


        endpoint_name = str(
            endpoint_name
        )


        if (
            endpoint_name
            not in
            resource.endpoint_names
        ):

            raise ValueError(
                f"Endpoint '{endpoint_name}' is not part of "
                f"resource '{resource.resource_id}'."
            )


        cache_key = (
            resource.resource_id,
            endpoint_name
        )


        if (
            cache_key
            in self._birth_time_cache
        ):

            return float(
                self._birth_time_cache[
                    cache_key
                ]
            )


        # --------------------------------------------------------
        # Elementary physical qubit:
        # its storage begins when the elementary resource exists.
        # --------------------------------------------------------

        if (
            resource.kind
            ==
            EntanglementKind.ELEMENTARY
        ):

            birth_time_s = float(
                self.fidelity_registry
                .recorded_time_s(
                    resource
                )
            )


        # --------------------------------------------------------
        # Swapped resource:
        # find which parent carried this surviving outer qubit.
        # --------------------------------------------------------

        elif (
            resource.kind
            ==
            EntanglementKind.SWAPPED
        ):

            parents = tuple(

                self.manager
                .get_resource(
                    parent_id
                )

                for parent_id
                in resource.parent_resource_ids
            )


            containing_parents = tuple(

                parent

                for parent
                in parents

                if (
                    endpoint_name
                    in parent.endpoint_names
                )
            )


            if (
                len(
                    containing_parents
                )
                !=
                1
            ):

                raise RuntimeError(
                    "Could not uniquely reconstruct surviving "
                    "outer-qubit ancestry for "
                    f"resource '{resource.resource_id}', "
                    f"endpoint '{endpoint_name}'."
                )


            birth_time_s = (
                self.endpoint_qubit_birth_time_s(

                    containing_parents[0],

                    endpoint_name
                )
            )


        else:

            raise RuntimeError(
                f"Unsupported entanglement kind: "
                f"{resource.kind}"
            )


        self._birth_time_cache[
            cache_key
        ] = float(
            birth_time_s
        )


        return float(
            birth_time_s
        )


    # ============================================================
    # Resource Assessment
    # ============================================================

    def assess_resource(
        self,
        resource_or_id
    ) -> PhysicalQubitLifetimeAssessment:

        resource = (

            resource_or_id

            if isinstance(
                resource_or_id,
                EntanglementResource
            )

            else
            self.manager.get_resource(
                str(
                    resource_or_id
                )
            )
        )


        if not resource.is_active:

            raise ValueError(
                "Physical lifetime assessment requires an "
                "ACTIVE resource."
            )


        current_time_s = float(
            self.time_ledger
            .current_time_s
        )


        endpoint_a = (
            resource.endpoint_a_name
        )

        endpoint_b = (
            resource.endpoint_b_name
        )


        birth_a = (
            self.endpoint_qubit_birth_time_s(

                resource,
                endpoint_a
            )
        )


        birth_b = (
            self.endpoint_qubit_birth_time_s(

                resource,
                endpoint_b
            )
        )


        age_a = max(

            0.0,

            current_time_s
            -
            birth_a
        )


        age_b = max(

            0.0,

            current_time_s
            -
            birth_b
        )


        oldest_age = max(
            age_a,
            age_b
        )


        remaining = (

            self.max_storage_time_s
            -
            oldest_age
        )


        physically_expired = (

            oldest_age

            >=

            (
                self.max_storage_time_s
                -
                self.numerical_tolerance
            )
        )


        return (
            PhysicalQubitLifetimeAssessment(

                simulation_slot=int(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                physical_time_s=(
                    current_time_s
                ),

                resource_id=(
                    resource.resource_id
                ),

                resource_kind=(
                    resource.kind.value
                ),

                endpoint_a_name=(
                    endpoint_a
                ),

                endpoint_b_name=(
                    endpoint_b
                ),

                endpoint_a_birth_time_s=(
                    birth_a
                ),

                endpoint_b_birth_time_s=(
                    birth_b
                ),

                endpoint_a_age_s=(
                    age_a
                ),

                endpoint_b_age_s=(
                    age_b
                ),

                oldest_physical_qubit_age_s=(
                    oldest_age
                ),

                max_storage_time_s=(
                    self.max_storage_time_s
                ),

                remaining_hard_lifetime_s=(
                    remaining
                ),

                physically_expired=(
                    physically_expired
                )
            )
        )


    # ============================================================
    # Complete Active-Frontier Audit
    # ============================================================

    def audit_active_resources(
        self
    ) -> pd.DataFrame:

        rows = []


        for resource in sorted(

            self.manager.active_resources,

            key=(
                lambda item:
                item.resource_id
            )
        ):

            rows.append(
                asdict(
                    self.assess_resource(
                        resource
                    )
                )
            )


        return pd.DataFrame(
            rows
        )


    # ============================================================
    # Physical Expiration Synchronization
    # ============================================================

    def synchronize_expirations(
        self
    ):

        assessments = tuple(

            self.assess_resource(
                resource
            )

            for resource
            in sorted(

                self.manager.active_resources,

                key=(
                    lambda item:
                    item.resource_id
                )
            )
        )


        expired_now = []


        for assessment in assessments:

            if not (
                assessment
                .physically_expired
            ):

                continue


            reason = (

                "physical_memory_lifetime_exceeded"
                f"_limit_{self.max_storage_time_s:.9f}s"
            )


            terminal_resource = (
                self.manager
                .expire_resource(

                    resource_id=(
                        assessment
                        .resource_id
                    ),

                    reason=(
                        reason
                    )
                )
            )


            if (
                terminal_resource.status
                !=
                EntanglementStatus.EXPIRED
            ):

                raise RuntimeError(
                    "Physical memory expiration failed to enter "
                    "EXPIRED state."
                )


            record = (
                PhysicalQubitExpirationRecord(

                    simulation_slot=(
                        assessment
                        .simulation_slot
                    ),

                    physical_time_s=(
                        assessment
                        .physical_time_s
                    ),

                    resource_id=(
                        assessment
                        .resource_id
                    ),

                    oldest_physical_qubit_age_s=(
                        assessment
                        .oldest_physical_qubit_age_s
                    ),

                    max_storage_time_s=(
                        self.max_storage_time_s
                    ),

                    reason=(
                        reason
                    )
                )
            )


            self._expiration_history.append(
                record
            )


            expired_now.append(
                assessment.resource_id
            )


        self.manager\
            .validate_all_active_resources()


        assert (
            self.simulator
            .validate_runtime_consistency()
            is True
        )


        if expired_now:

            self.simulator.kernel.emit(

                event_type=(
                    "PHYSICAL_MEMORY_EXPIRATION_SYNCHRONIZED"
                ),

                component=(
                    "PhysicalQubitLifetimeController"
                ),

                resource_ids=tuple(
                    expired_now
                ),

                message=(
                    "Expired entanglement resources whose "
                    "surviving physical qubits exceeded the "
                    "hard modeled storage lifetime."
                ),

                metadata={
                    "physical_time_s":
                        self.time_ledger
                        .current_time_s,

                    "max_storage_time_s":
                        self.max_storage_time_s,

                    "expired_count":
                        len(
                            expired_now
                        )
                }
            )


        return tuple(
            expired_now
        )


    # ============================================================
    # History
    # ============================================================

    @property
    def expiration_history(
        self
    ):

        return tuple(
            self._expiration_history
        )


    def expiration_history_dataframe(
        self
    ):

        return pd.DataFrame(
            [
                asdict(
                    record
                )

                for record
                in self._expiration_history
            ]
        )

In [ ]:
# @title Physical-Qubit Lifetime Preservation Through Swapping

print(
    "=" * 92
)

print("Physical Quantum-Memory Lifetime Verification"
)

print(
    "=" * 92
)


# ================================================================
# Base authenticated R6 environment.
#
# Logical-slot expiry cannot interfere because this verification
# never advances the logical clock.
# ================================================================

stack = (
    build_phase3g2_trial_stack(

        policy_name=(
            "MAX_PARALLEL_OLDEST_FIRST"
        ),

        repeater_count=6,

        simulation_seed=(
            7_330_101
        )
    )
)


simulator = (
    stack[
        "simulator"
    ]
)


manager = (
    simulator
    .entanglement_manager
)


path = (
    stack[
        "path"
    ]
)


path_links = tuple(
    stack[
        "topology"
    ]
    .path_links(
        path
    )
)


fidelity_profile = (
    stack[
        "fidelity_profile"
    ]
)


# ================================================================
# Continuous physical-time infrastructure.
# ================================================================

physical_time = (
    PhysicalTimeLedger(

        kernel=(
            simulator.kernel
        )
    )
)


continuous_registry = (
    PhysicalTimeAwareEntanglementFidelityRegistry(

        simulator=(
            simulator
        ),

        profile=(
            fidelity_profile
        ),

        time_ledger=(
            physical_time
        )
    )
)


lifetime_controller = (
    PhysicalQubitLifetimeController(

        simulator=(
            simulator
        ),

        fidelity_registry=(
            continuous_registry
        ),

        time_ledger=(
            physical_time
        ),

        max_storage_time_s=(
            PHASE3H3_MAX_STORAGE_TIME_S
        )
    )
)


assert math.isclose(

    lifetime_controller
    .max_storage_time_s,

    5.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


# ================================================================
# Create two adjacent elementary EPs at:
#
#     logical slot = 0
#     physical time = 0
# ================================================================

left_resource = (
    manager
    .create_elementary_resource(
        path_links[0]
    )
)


right_resource = (
    manager
    .create_elementary_resource(
        path_links[1]
    )
)


continuous_registry\
    .register_elementary(
        left_resource
    )


continuous_registry\
    .register_elementary(
        right_resource
    )


assert math.isclose(

    lifetime_controller
    .endpoint_qubit_birth_time_s(

        left_resource,
        left_resource.endpoint_a_name
    ),

    0.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    lifetime_controller
    .endpoint_qubit_birth_time_s(

        right_resource,
        right_resource.endpoint_b_name
    ),

    0.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Elementary outer qubits have physical birth time t=0."
)


# ================================================================
# Advance to 4.999 s.
# ================================================================

physical_time\
    .advance(

        duration_s=(
            4.999
        ),

        phase_name=(
            "VERIFICATION_PRE_SWAP_STORAGE"
        ),

        resource_ids=(
            left_resource.resource_id,
            right_resource.resource_id
        )
    )


assert (
    simulator
    .kernel
    .current_slot
    ==
    0
)


expired_before_swap = (
    lifetime_controller
    .synchronize_expirations()
)


assert (
    expired_before_swap
    ==
    ()
)


print(
    "[PASS] Resources remain alive immediately below the "
    "5-second hard physical-memory limit."
)


# ================================================================
# Perform deterministic successful swap AT t = 4.999 s.
# ================================================================

verification_swap_model = (
    WernerStateAtomicBSMModel(

        fidelity_profile=(
            fidelity_profile
        ),

        atomic_bsm_success_probability=1.0,

        atomic_bsm_latency_us=10.0
    )
)


verification_swap_engine = (
    EntanglementSwapEngine(

        simulator=(
            simulator
        ),

        model=(
            verification_swap_model
        ),

        fidelity_registry=(
            continuous_registry
        )
    )
)


swap_attempt = (
    verification_swap_engine
    .attempt_swap(

        left_resource_id=(
            left_resource
            .resource_id
        ),

        right_resource_id=(
            right_resource
            .resource_id
        ),

        swap_node=(
            path[1]
            .name
        )
    )
)


assert (
    swap_attempt.success
    is True
)


parent_resource = (
    manager
    .get_resource(
        swap_attempt
        .output_resource_id
    )
)


assert (
    parent_resource.status
    ==
    EntanglementStatus.ACTIVE
)


# ================================================================
# The PARENT fidelity checkpoint is new...
# ================================================================

assert math.isclose(

    continuous_registry
    .recorded_time_s(
        parent_resource
    ),

    4.999,

    rel_tol=0.0,

    abs_tol=1e-12
)


# ================================================================
# ...but its surviving OUTER PHYSICAL QUBITS are NOT new.
# ================================================================

birth_a = (
    lifetime_controller
    .endpoint_qubit_birth_time_s(

        parent_resource,
        parent_resource.endpoint_a_name
    )
)


birth_b = (
    lifetime_controller
    .endpoint_qubit_birth_time_s(

        parent_resource,
        parent_resource.endpoint_b_name
    )
)


assert math.isclose(

    birth_a,

    0.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    birth_b,

    0.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


assessment_after_swap = (
    lifetime_controller
    .assess_resource(
        parent_resource
    )
)


assert math.isclose(

    assessment_after_swap
    .oldest_physical_qubit_age_s,

    4.999,

    rel_tol=0.0,

    abs_tol=1e-12
)


assert (
    assessment_after_swap
    .physically_expired
    is False
)


print(
    "[PASS] Successful swapping created a new entanglement "
    "resource WITHOUT resetting surviving physical-qubit age."
)


# ================================================================
# Advance only another 2 ms:
#
#     physical time = 5.001 s
#
# Logical slot still remains zero.
# ================================================================

physical_time\
    .advance(

        duration_s=(
            0.002
        ),

        phase_name=(
            "VERIFICATION_POST_SWAP_STORAGE"
        ),

        resource_ids=(
            parent_resource.resource_id,
        )
    )


assert math.isclose(

    physical_time
    .current_time_s,

    5.001,

    rel_tol=0.0,

    abs_tol=1e-12
)


assert (
    simulator
    .kernel
    .current_slot
    ==
    0
)


assessment_before_expiry = (
    lifetime_controller
    .assess_resource(
        parent_resource
    )
)


assert (
    assessment_before_expiry
    .physically_expired
    is True
)


assert math.isclose(

    assessment_before_expiry
    .oldest_physical_qubit_age_s,

    5.001,

    rel_tol=0.0,

    abs_tol=1e-12
)


print(
    "[PASS] New parent resource correctly inherits "
    "5.001-second-old outer physical qubits."
)


# ================================================================
# Synchronize physical expiration.
# ================================================================

expired_now = (
    lifetime_controller
    .synchronize_expirations()
)


assert (
    expired_now
    ==
    (
        parent_resource.resource_id,
    )
)


expired_parent = (
    manager
    .get_resource(
        parent_resource.resource_id
    )
)


assert (
    expired_parent.status
    ==
    EntanglementStatus.EXPIRED
)


assert (
    expired_parent.ended_slot
    ==
    0
)


assert (
    "physical_memory_lifetime_exceeded"
    in expired_parent.termination_reason
)


print(
    "[PASS] Parent expired from PHYSICAL qubit age while "
    "logical slot remained unchanged."
)


# ================================================================
# Parent's two physical halves must have been removed atomically.
# ================================================================

assert (
    len(
        manager.active_resources
    )
    ==
    0
)


manager\
    .validate_all_active_resources()


assert (
    simulator
    .validate_runtime_consistency()
    is True
)


print(
    "[PASS] EntanglementManager removed both endpoint halves "
    "and runtime remains consistent."
)


# ================================================================
# Diagnostics
# ================================================================

print(
    "\nPARENT LIFETIME ASSESSMENT BEFORE EXPIRY"
)


display(
    pd.DataFrame(
        [
            asdict(
                assessment_before_expiry
            )
        ]
    )
)


print(
    "\nPHYSICAL EXPIRATION HISTORY"
)


display(
    lifetime_controller
    .expiration_history_dataframe()
)


print(
    "\nPHYSICAL TIME HISTORY"
)


display(
    physical_time
    .history_dataframe()
)


print(
    "\nRESOURCE HISTORY"
)


display(
    manager
    .resources_dataframe()
)


print(
    "\n" + "=" * 92
)

print(
    "Physical memory-lifetime verification PASSED."
)

print(
    "=" * 92
)

In [ ]:
# @title Physical-Time-Compatible Swap Policy + Barrier-Aware Elementary Fidelity Checkpoints

import itertools
import math

from dataclasses import dataclass, asdict
from typing import Optional

import pandas as pd


# ================================================================
# Maximum Parallelism WITHOUT Logical-Slot Age Semantics
# ================================================================

class MaxParallelDeterministicSwapPolicy:
    """
    Physical-time-safe baseline swap policy.

    Optimization priorities
    -----------------------
    1. Maximize number of simultaneous non-conflicting BSMs.
    2. Deterministic left-to-right / candidate-ID tie-break.

    Deliberately NOT used:
        oldest_input_half_age_slots

    Logical slot age is no longer interpreted as physical age.
    """

    @property
    def policy_name(
        self
    ) -> str:

        return (
            "MAX_PARALLEL_DETERMINISTIC"
        )


    def select(
        self,
        candidates
    ):

        candidates = tuple(
            candidates
        )


        if not candidates:

            return ()


        for candidate in candidates:

            if not isinstance(
                candidate,
                SwapCandidate
            ):

                raise TypeError(
                    "All candidates must be SwapCandidate objects."
                )


        ordered_candidates = tuple(
            sorted(

                candidates,

                key=lambda candidate: (
                    candidate.swap_node_path_index,
                    candidate.candidate_id
                )
            )
        )


        best_selection = ()


        def deterministic_key(
            selection
        ):

            return tuple(

                (
                    candidate.swap_node_path_index,
                    candidate.candidate_id
                )

                for candidate
                in selection
            )


        def is_better(
            proposed,
            current_best
        ):

            # ----------------------------------------------------
            # Primary:
            # maximum simultaneous BSM count.
            # ----------------------------------------------------

            if (
                len(
                    proposed
                )
                !=
                len(
                    current_best
                )
            ):

                return (
                    len(
                        proposed
                    )
                    >
                    len(
                        current_best
                    )
                )


            # ----------------------------------------------------
            # Secondary:
            # deterministic only.
            # ----------------------------------------------------

            return (
                deterministic_key(
                    proposed
                )
                <
                deterministic_key(
                    current_best
                )
            )


        def search(
            index,
            chosen
        ):

            nonlocal best_selection


            if (
                index
                >=
                len(
                    ordered_candidates
                )
            ):

                proposed = tuple(
                    chosen
                )


                if is_better(
                    proposed,
                    best_selection
                ):

                    best_selection = (
                        proposed
                    )


                return


            candidate = (
                ordered_candidates[
                    index
                ]
            )


            # Skip.
            search(
                index + 1,
                chosen
            )


            # Select if conflict-free.
            if all(

                not candidate
                .conflicts_with(
                    selected
                )

                for selected
                in chosen
            ):

                chosen.append(
                    candidate
                )


                search(
                    index + 1,
                    chosen
                )


                chosen.pop()


        search(
            0,
            []
        )


        return tuple(
            sorted(

                best_selection,

                key=lambda candidate: (
                    candidate.swap_node_path_index,
                    candidate.candidate_id
                )
            )
        )


# ================================================================
# Barrier-Aware Continuous-Time Fidelity Registry
# ================================================================

class BarrierAwarePhysicalTimeFidelityRegistry(
    PhysicalTimeAwareEntanglementFidelityRegistry
):
    """
    Extends the Phase-3H.3B continuous-time registry.

    Why?
    ----
    Parallel elementary attempts can have DIFFERENT physical
    durations on heterogeneous links.

    A shorter link may finish before the global parallel barrier.

    Therefore:

        resource physical birth/checkpoint time
        !=
        necessarily the final barrier time.

    Example:

        30-km link finishes at t = 0.20 ms
        85-km link finishes at t = 0.48 ms

    The control plane synchronizes at 0.48 ms, but the first
    Bell pair has already waited for 0.28 ms.

    That waiting must count toward:
        - memory decoherence;
        - hard qubit lifetime.
    """

    def register_elementary_at_time(
        self,
        resource,
        physical_time_s
    ):

        physical_time_s = float(
            physical_time_s
        )


        if (
            not math.isfinite(
                physical_time_s
            )
            or physical_time_s < 0.0
        ):

            raise ValueError(
                "physical_time_s must be a non-negative "
                "finite number."
            )


        if (
            physical_time_s
            >
            self.time_ledger.current_time_s
            +
            1e-15
        ):

            raise ValueError(
                "Cannot register an elementary resource "
                "at a future physical time."
            )


        # --------------------------------------------------------
        # Reuse all existing physical-fidelity validation and
        # record creation.
        # --------------------------------------------------------

        record = (
            super()
            .register_elementary(
                resource
            )
        )


        # Parent method records the current barrier time.
        #
        # Replace ONLY the physical checkpoint with the resource's
        # actual link-completion time.
        self._recorded_time_s[
            resource.resource_id
        ] = (
            physical_time_s
        )


        return record


# ================================================================
# Verification
# ================================================================

test_policy = (
    MaxParallelDeterministicSwapPolicy()
)


assert (
    test_policy.policy_name
    ==
    "MAX_PARALLEL_DETERMINISTIC"
)


print(
    "[PASS] Swap policy maximizes parallelism "
    "without treating logical slots as physical age."
)

print(
    "[PASS] Elementary fidelity registry supports per-link "
    "physical completion times inside one parallel barrier."
)

In [ ]:
# @title Fully Physical-Timed Autonomous QKD Orchestrator

@dataclass(frozen=True)
class PhysicalTimedQKDExecutionSlot:

    execution_index: int

    simulation_slot: int

    physical_start_time_s: float

    generation_duration_s: float

    post_generation_time_s: float

    bsm_duration_s: float

    physical_end_time_s: float

    pre_physical_expirations: tuple[str, ...]
    post_generation_physical_expirations: tuple[str, ...]
    post_bsm_physical_expirations: tuple[str, ...]

    pre_individual_quality_discards: int
    mid_individual_quality_discards: int
    post_individual_quality_discards: int

    pre_collective_recovery_status: str
    mid_collective_recovery_status: str
    post_collective_recovery_status: str

    pre_collective_victims: tuple[str, ...]
    mid_collective_victims: tuple[str, ...]
    post_collective_victims: tuple[str, ...]

    base_execution_slot: FailureAwareExecutionSlot

    end_to_end_resource_id: Optional[str]

    final_fidelity: Optional[float]

    predicted_qber: Optional[float]

    qkd_eligible: bool


    @property
    def elementary_attempt_count(
        self
    ):

        return (
            self.base_execution_slot
            .elementary_attempt_count
        )


    @property
    def elementary_success_count(
        self
    ):

        return (
            self.base_execution_slot
            .elementary_success_count
        )


    @property
    def elementary_failure_count(
        self
    ):

        return (
            self.base_execution_slot
            .elementary_failure_count
        )


    @property
    def bsm_attempt_count(
        self
    ):

        return (
            self.base_execution_slot
            .bsm_attempt_count
        )


    @property
    def bsm_success_count(
        self
    ):

        return (
            self.base_execution_slot
            .bsm_success_count
        )


    @property
    def bsm_failure_count(
        self
    ):

        return (
            self.base_execution_slot
            .bsm_failure_count
        )


@dataclass(frozen=True)
class PhysicalTimedQKDExecutionResult:

    status: QKDReadyExecutionStatus

    start_slot: int
    end_slot: int

    start_physical_time_s: float
    end_physical_time_s: float

    processed_slot_count: int

    end_to_end_resource_id: Optional[str]

    final_fidelity: Optional[float]

    predicted_qber: Optional[float]

    total_elementary_attempts: int
    total_elementary_successes: int
    total_elementary_failures: int

    total_bsm_attempts: int
    total_bsm_successes: int
    total_bsm_failures: int

    total_hard_physical_expirations: int

    total_individual_quality_discards: int

    total_collective_recovery_executions: int

    total_collective_resources_discarded: int

    execution_history: tuple[
        PhysicalTimedQKDExecutionSlot, ...
    ]


    @property
    def success(
        self
    ):

        return (
            self.status
            ==
            QKDReadyExecutionStatus.SUCCESS
        )


    @property
    def physical_elapsed_s(
        self
    ):

        return (
            self.end_physical_time_s
            -
            self.start_physical_time_s
        )


class PhysicalTimedQKDReadyOrchestrator(
    QKDReadyEndToEndEntanglementOrchestrator
):
    """
    Phase 3H.3D autonomous execution owner.

    Logical time:
        SimulationKernel.current_slot

    Physical time:
        PhysicalTimeLedger.current_time_s

    Hard physical lifetime:
        PhysicalQubitLifetimeController

    QKD quality:
        Phase 3H.1 + 3H.2 controllers

    One logical execution round:

        hard-expiry sync
            ->
        quality maintenance
            ->
        elementary generation
            ->
        PHYSICAL generation latency
            ->
        exact elementary birth checkpoints
            ->
        hard-expiry sync
            ->
        quality maintenance
            ->
        one parallel BSM round
            ->
        PHYSICAL BSM latency
            ->
        hard-expiry sync
            ->
        quality maintenance
            ->
        QKD admission

    The logical slot advances only outside this method.
    """

    def __init__(
        self,
        stack,
        timing_profile=(
            PHASE3H3_TIMING_PROFILE
        ),
        generation_timing_geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        ),
        config=(
            PHASE3H1_QUALITY_CONFIG
        )
    ):

        super().__init__(

            stack=(
                stack
            ),

            config=(
                config
            )
        )


        self.time_ledger = (
            stack[
                "physical_time_ledger"
            ]
        )


        self.physical_lifetime_controller = (
            stack[
                "physical_lifetime_controller"
            ]
        )


        self.timing_profile = (
            timing_profile
        )


        if not isinstance(
            generation_timing_geometry,
            ElementaryAttemptTimingGeometry
        ):

            generation_timing_geometry = (
                ElementaryAttemptTimingGeometry(
                    generation_timing_geometry
                )
            )


        self.generation_timing_geometry = (
            generation_timing_geometry
        )


        self._path_link_by_id = {

            link.link_id:
                link

            for link
            in (
                stack[
                    "topology"
                ]
                .path_links(
                    stack[
                        "path"
                    ]
                )
            )
        }


        self._physical_execution_history = []


    # ============================================================
    # Physical Hard-Expiry Synchronization
    # ============================================================

    def _synchronize_physical_expiry(
        self
    ):

        slot_before = (
            self.simulator
            .kernel
            .current_slot
        )


        expired = (
            self.physical_lifetime_controller
            .synchronize_expirations()
        )


        if expired:

            self.demand_controller\
                .reconcile()


        if (
            self.simulator
            .kernel
            .current_slot
            !=
            slot_before
        ):

            raise RuntimeError(
                "Physical hard-expiry synchronization "
                "advanced logical time."
            )


        return tuple(
            expired
        )


    # ============================================================
    # Generation Physical-Time Phase
    # ============================================================

    def _complete_generation_phase(
        self,
        generation_slot
    ):

        phase_start_s = float(
            self.time_ledger
            .current_time_s
        )


        attempted_steps = tuple(

            step

            for step
            in generation_slot.steps

            if (
                step.attempt
                is not None
            )
        )


        duration_by_link = {}


        for step in attempted_steps:

            link = (
                self._path_link_by_id[
                    step.link_id
                ]
            )


            duration_by_link[
                step.link_id
            ] = (
                self.timing_profile
                .elementary_attempt_cycle_s(

                    distance_km=(
                        link.distance_km
                    ),

                    geometry=(
                        self.generation_timing_geometry
                    )
                )
            )


        generation_duration_s = (

            max(
                duration_by_link.values()
            )

            if duration_by_link

            else 0.0
        )


        # --------------------------------------------------------
        # Parallel phase barrier:
        #
        # physical phase duration = longest participating attempt.
        # --------------------------------------------------------

        if (
            generation_duration_s
            >
            0.0
        ):

            self.time_ledger\
                .advance(

                    duration_s=(
                        generation_duration_s
                    ),

                    phase_name=(
                        "PARALLEL_ELEMENTARY_GENERATION"
                    ),

                    link_ids=tuple(
                        step.link_id
                        for step
                        in attempted_steps
                    ),

                    metadata={
                        "geometry":
                            self
                            .generation_timing_geometry
                            .value,

                        "attempt_durations_s":
                            dict(
                                duration_by_link
                            ),

                        "parallel_barrier_duration_s":
                            generation_duration_s
                    }
                )


        # --------------------------------------------------------
        # Register successful Bell pairs at THEIR OWN completion
        # times, not blindly at the longest-link barrier.
        # --------------------------------------------------------

        registered_ids = []


        for step in attempted_steps:

            if not step.attempt.success:

                continue


            resource = (
                self.simulator
                .entanglement_manager
                .get_resource(
                    step.attempt
                    .resource_id
                )
            )


            completion_time_s = (

                phase_start_s

                +

                duration_by_link[
                    step.link_id
                ]
            )


            self.fidelity_registry\
                .register_elementary_at_time(

                    resource=(
                        resource
                    ),

                    physical_time_s=(
                        completion_time_s
                    )
                )


            registered_ids.append(
                resource.resource_id
            )


        return (
            float(
                generation_duration_s
            ),
            tuple(
                registered_ids
            )
        )


    # ============================================================
    # Parallel BSM Physical-Time Phase
    # ============================================================

    def _execute_bsm_phase(
        self
    ):

        candidates = (
            self.swap_scheduler
            .discover_candidates()
        )


        if not candidates:

            return (
                None,
                0.0
            )


        swap_round = (
            self.swap_scheduler
            .execute_parallel_round()
        )


        if (
            len(
                swap_round.attempts
            )
            ==
            0
        ):

            return (
                swap_round,
                0.0
            )


        # --------------------------------------------------------
        # All current atomic BSMs use the same model latency.
        #
        # Because the selected BSMs execute in parallel:
        #
        #     phase duration = one BSM latency
        #
        # NOT:
        #
        #     number_of_BSMs * latency
        # --------------------------------------------------------

        bsm_duration_s = (

            float(
                self.swap_engine
                .model
                .atomic_bsm_latency_us
            )

            *
            1e-6
        )


        self.time_ledger\
            .advance(

                duration_s=(
                    bsm_duration_s
                ),

                phase_name=(
                    "PARALLEL_ATOMIC_BSM"
                ),

                resource_ids=tuple(
                    swap_round
                    .output_resource_ids
                ),

                metadata={
                    "parallel_bsm_attempts":
                        len(
                            swap_round.attempts
                        ),

                    "parallel_bsm_successes":
                        swap_round.success_count,

                    "parallel_bsm_failures":
                        swap_round.failure_count,

                    "atomic_bsm_latency_us":
                        self.swap_engine
                        .model
                        .atomic_bsm_latency_us
                }
            )


        return (
            swap_round,
            float(
                bsm_duration_s
            )
        )


    # ============================================================
    # One Fully Physical-Timed Logical Round
    # ============================================================

    def process_physical_current_slot(
        self
    ):

        self.simulator\
            .validate_runtime_consistency()


        slot_at_start = int(
            self.simulator
            .kernel
            .current_slot
        )


        physical_start_s = float(
            self.time_ledger
            .current_time_s
        )


        # ========================================================
        # 1. HARD PHYSICAL EXPIRY
        # ========================================================

        pre_expired = (
            self
            ._synchronize_physical_expiry()
        )


        # ========================================================
        # 2. PRE-PHYSICS QKD QUALITY MAINTENANCE
        # ========================================================

        pre_quality = (
            self._apply_quality_maintenance(
                stage=(
                    "PHYSICAL_PRE_GENERATION"
                )
            )
        )


        demand_before = (
            self.demand_controller
            .reconcile()
        )


        # ========================================================
        # 3. PARALLEL ELEMENTARY GENERATION
        # ========================================================

        generation_slot = (
            self.generation_scheduler
            .process_current_slot()
        )


        assert (
            generation_slot.simulation_slot
            ==
            slot_at_start
        )


        # ========================================================
        # 4. ADVANCE REAL MODELED GENERATION TIME
        # ========================================================

        (
            generation_duration_s,
            registered_elementary_ids
        ) = (
            self._complete_generation_phase(
                generation_slot
            )
        )


        post_generation_time_s = float(
            self.time_ledger
            .current_time_s
        )


        # ========================================================
        # 5. HARD EXPIRY AFTER GENERATION WAIT
        # ========================================================

        post_generation_expired = (
            self
            ._synchronize_physical_expiry()
        )


        self.demand_controller\
            .reconcile()


        # ========================================================
        # 6. MID-SLOT QKD QUALITY MAINTENANCE
        #
        # Resources may have become QKD-useless while waiting for
        # the generation phase barrier.
        #
        # If selective repair occurs here, no second elementary
        # generation attempt is allowed until the NEXT logical slot.
        # ========================================================

        mid_quality = (
            self._apply_quality_maintenance(
                stage=(
                    "PHYSICAL_POST_GENERATION"
                )
            )
        )


        # ========================================================
        # 7. ONE MAXIMUM-PARALLEL BSM ROUND
        # ========================================================

        (
            swap_round,
            bsm_duration_s
        ) = (
            self._execute_bsm_phase()
        )


        # ========================================================
        # 8. HARD EXPIRY AFTER BSM LATENCY
        # ========================================================

        post_bsm_expired = (
            self
            ._synchronize_physical_expiry()
        )


        self.demand_controller\
            .reconcile()


        # ========================================================
        # 9. POST-BSM QKD QUALITY MAINTENANCE
        # ========================================================

        post_quality = (
            self._apply_quality_maintenance(
                stage=(
                    "PHYSICAL_POST_BSM"
                )
            )
        )


        demand_after = (
            self.demand_controller
            .reconcile()
        )


        # ========================================================
        # 10. QKD ADMISSION
        # ========================================================

        (
            end_to_end_resource,
            qkd_assessment
        ) = (
            self
            ._assess_end_to_end_qkd_resource()
        )


        if (
            end_to_end_resource
            is None
        ):

            end_to_end_resource_id = None

            final_fidelity = None

            predicted_qber = None

            qkd_eligible = False


        else:

            if not (
                qkd_assessment
                .qkd_eligible_now
            ):

                raise RuntimeError(
                    "A non-QKD-eligible full-path resource "
                    "survived final quality maintenance."
                )


            end_to_end_resource_id = (
                end_to_end_resource
                .resource_id
            )


            final_fidelity = float(
                qkd_assessment
                .current_fidelity
            )


            predicted_qber = float(
                qkd_assessment
                .current_qber_proxy
            )


            qkd_eligible = True


        # ========================================================
        # 11. Build existing Phase-3D-compatible execution record
        # ========================================================

        active_frontier_ids = tuple(

            resource.resource_id

            for resource
            in (
                self.simulator
                .entanglement_manager
                .active_resources
            )
        )


        base_step = (
            FailureAwareExecutionSlot(

                execution_index=len(
                    self._execution_history
                ),

                simulation_slot=(
                    slot_at_start
                ),

                demand_before_generation=(
                    demand_before
                ),

                generation_slot=(
                    generation_slot
                ),

                newly_registered_elementary_resource_ids=(
                    registered_elementary_ids
                ),

                swap_round=(
                    swap_round
                ),

                demand_after_swapping=(
                    demand_after
                ),

                active_frontier_resource_ids=(
                    active_frontier_ids
                ),

                end_to_end_resource_id=(
                    end_to_end_resource_id
                )
            )
        )


        self._execution_history.append(
            base_step
        )


        # ========================================================
        # 12. Physical-time-aware execution record
        # ========================================================

        pre_collective = (
            pre_quality[
                "collective_record"
            ]
        )


        mid_collective = (
            mid_quality[
                "collective_record"
            ]
        )


        post_collective = (
            post_quality[
                "collective_record"
            ]
        )


        physical_step = (
            PhysicalTimedQKDExecutionSlot(

                execution_index=len(
                    self._physical_execution_history
                ),

                simulation_slot=(
                    slot_at_start
                ),

                physical_start_time_s=(
                    physical_start_s
                ),

                generation_duration_s=(
                    generation_duration_s
                ),

                post_generation_time_s=(
                    post_generation_time_s
                ),

                bsm_duration_s=(
                    bsm_duration_s
                ),

                physical_end_time_s=float(
                    self.time_ledger
                    .current_time_s
                ),

                pre_physical_expirations=(
                    pre_expired
                ),

                post_generation_physical_expirations=(
                    post_generation_expired
                ),

                post_bsm_physical_expirations=(
                    post_bsm_expired
                ),

                pre_individual_quality_discards=int(
                    pre_quality[
                        "individual_discard_count"
                    ]
                ),

                mid_individual_quality_discards=int(
                    mid_quality[
                        "individual_discard_count"
                    ]
                ),

                post_individual_quality_discards=int(
                    post_quality[
                        "individual_discard_count"
                    ]
                ),

                pre_collective_recovery_status=(
                    pre_collective
                    .status
                    .value
                ),

                mid_collective_recovery_status=(
                    mid_collective
                    .status
                    .value
                ),

                post_collective_recovery_status=(
                    post_collective
                    .status
                    .value
                ),

                pre_collective_victims=(
                    pre_collective
                    .discarded_resource_ids
                ),

                mid_collective_victims=(
                    mid_collective
                    .discarded_resource_ids
                ),

                post_collective_victims=(
                    post_collective
                    .discarded_resource_ids
                ),

                base_execution_slot=(
                    base_step
                ),

                end_to_end_resource_id=(
                    end_to_end_resource_id
                ),

                final_fidelity=(
                    final_fidelity
                ),

                predicted_qber=(
                    predicted_qber
                ),

                qkd_eligible=(
                    qkd_eligible
                )
            )
        )


        self._physical_execution_history.append(
            physical_step
        )


        # ========================================================
        # 13. Absolutely no logical advancement inside the round
        # ========================================================

        if (
            self.simulator
            .kernel
            .current_slot
            !=
            slot_at_start
        ):

            raise RuntimeError(
                "Physical-timed execution round illegally "
                "advanced logical simulation time."
            )


        self.simulator\
            .validate_runtime_consistency()


        return physical_step


    # ============================================================
    # Full Autonomous Session
    # ============================================================

    def run_until_qkd_ready_physical(
        self,
        max_slots=20_000
    ):

        if (
            isinstance(
                max_slots,
                bool
            )
            or not isinstance(
                max_slots,
                int
            )
            or max_slots <= 0
        ):

            raise ValueError(
                "max_slots must be a positive integer."
            )


        start_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        start_physical_time_s = float(
            self.time_ledger
            .current_time_s
        )


        history = []


        # --------------------------------------------------------
        # Fresh session synchronization.
        # --------------------------------------------------------

        self._synchronize_physical_expiry()


        self._apply_quality_maintenance(
            stage=(
                "PHYSICAL_SESSION_START"
            )
        )


        (
            existing_resource,
            existing_assessment
        ) = (
            self
            ._assess_end_to_end_qkd_resource()
        )


        if (
            existing_resource
            is not None
        ):

            assert (
                existing_assessment
                .qkd_eligible_now
            )


            return (
                PhysicalTimedQKDExecutionResult(

                    status=(
                        QKDReadyExecutionStatus
                        .SUCCESS
                    ),

                    start_slot=(
                        start_slot
                    ),

                    end_slot=(
                        start_slot
                    ),

                    start_physical_time_s=(
                        start_physical_time_s
                    ),

                    end_physical_time_s=(
                        self.time_ledger
                        .current_time_s
                    ),

                    processed_slot_count=0,

                    end_to_end_resource_id=(
                        existing_resource
                        .resource_id
                    ),

                    final_fidelity=float(
                        existing_assessment
                        .current_fidelity
                    ),

                    predicted_qber=float(
                        existing_assessment
                        .current_qber_proxy
                    ),

                    total_elementary_attempts=0,

                    total_elementary_successes=0,

                    total_elementary_failures=0,

                    total_bsm_attempts=0,

                    total_bsm_successes=0,

                    total_bsm_failures=0,

                    total_hard_physical_expirations=0,

                    total_individual_quality_discards=0,

                    total_collective_recovery_executions=0,

                    total_collective_resources_discarded=0,

                    execution_history=()
                )
            )


        # ========================================================
        # Main loop
        # ========================================================

        for _ in range(
            max_slots
        ):

            step = (
                self
                .process_physical_current_slot()
            )


            history.append(
                step
            )


            if step.qkd_eligible:

                status = (
                    QKDReadyExecutionStatus
                    .SUCCESS
                )

                break


            # ----------------------------------------------------
            # Logical progression ONLY.
            #
            # No implicit seconds are associated with this call.
            # ----------------------------------------------------

            self.simulator\
                .kernel\
                .advance_slot()


        else:

            status = (
                QKDReadyExecutionStatus
                .MAX_SLOTS_EXCEEDED
            )


        # ========================================================
        # Aggregate
        # ========================================================

        success = (
            status
            ==
            QKDReadyExecutionStatus.SUCCESS
        )


        final_step = (
            history[-1]
            if history
            else None
        )


        total_hard_expirations = sum(

            len(
                step.pre_physical_expirations
            )

            +

            len(
                step.post_generation_physical_expirations
            )

            +

            len(
                step.post_bsm_physical_expirations
            )

            for step
            in history
        )


        total_individual_discards = sum(

            step.pre_individual_quality_discards

            +

            step.mid_individual_quality_discards

            +

            step.post_individual_quality_discards

            for step
            in history
        )


        collective_statuses = (

            QKDSelectiveRecoveryExecutionStatus
            .EXECUTED
            .value
        )


        total_collective_executions = sum(

            int(
                step.pre_collective_recovery_status
                ==
                collective_statuses
            )

            +

            int(
                step.mid_collective_recovery_status
                ==
                collective_statuses
            )

            +

            int(
                step.post_collective_recovery_status
                ==
                collective_statuses
            )

            for step
            in history
        )


        total_collective_victims = sum(

            len(
                step.pre_collective_victims
            )

            +

            len(
                step.mid_collective_victims
            )

            +

            len(
                step.post_collective_victims
            )

            for step
            in history
        )


        return (
            PhysicalTimedQKDExecutionResult(

                status=(
                    status
                ),

                start_slot=(
                    start_slot
                ),

                end_slot=int(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                start_physical_time_s=(
                    start_physical_time_s
                ),

                end_physical_time_s=float(
                    self.time_ledger
                    .current_time_s
                ),

                processed_slot_count=len(
                    history
                ),

                end_to_end_resource_id=(

                    final_step
                    .end_to_end_resource_id

                    if success

                    else None
                ),

                final_fidelity=(

                    final_step
                    .final_fidelity

                    if success

                    else None
                ),

                predicted_qber=(

                    final_step
                    .predicted_qber

                    if success

                    else None
                ),

                total_elementary_attempts=sum(
                    step.elementary_attempt_count
                    for step
                    in history
                ),

                total_elementary_successes=sum(
                    step.elementary_success_count
                    for step
                    in history
                ),

                total_elementary_failures=sum(
                    step.elementary_failure_count
                    for step
                    in history
                ),

                total_bsm_attempts=sum(
                    step.bsm_attempt_count
                    for step
                    in history
                ),

                total_bsm_successes=sum(
                    step.bsm_success_count
                    for step
                    in history
                ),

                total_bsm_failures=sum(
                    step.bsm_failure_count
                    for step
                    in history
                ),

                total_hard_physical_expirations=(
                    total_hard_expirations
                ),

                total_individual_quality_discards=(
                    total_individual_discards
                ),

                total_collective_recovery_executions=(
                    total_collective_executions
                ),

                total_collective_resources_discarded=(
                    total_collective_victims
                ),

                execution_history=tuple(
                    history
                )
            )
        )


    # ============================================================
    # Diagnostics
    # ============================================================

    @property
    def physical_execution_history(
        self
    ):

        return tuple(
            self._physical_execution_history
        )


    def physical_history_dataframe(
        self
    ):

        return pd.DataFrame(
            [
                {
                    "Execution Step":
                        step.execution_index,

                    "Logical Slot":
                        step.simulation_slot,

                    "Physical Start (s)":
                        step.physical_start_time_s,

                    "Generation Duration (ms)":
                        step.generation_duration_s
                        *
                        1000.0,

                    "BSM Duration (us)":
                        step.bsm_duration_s
                        *
                        1e6,

                    "Physical End (s)":
                        step.physical_end_time_s,

                    "Elementary Attempts":
                        step.elementary_attempt_count,

                    "Elementary Successes":
                        step.elementary_success_count,

                    "BSM Attempts":
                        step.bsm_attempt_count,

                    "BSM Successes":
                        step.bsm_success_count,

                    "Hard Expirations":
                        (
                            len(
                                step.pre_physical_expirations
                            )
                            +
                            len(
                                step.post_generation_physical_expirations
                            )
                            +
                            len(
                                step.post_bsm_physical_expirations
                            )
                        ),

                    "Individual Quality Discards":
                        (
                            step.pre_individual_quality_discards
                            +
                            step.mid_individual_quality_discards
                            +
                            step.post_individual_quality_discards
                        ),

                    "Collective Recovery Pre":
                        step
                        .pre_collective_recovery_status,

                    "Collective Recovery Mid":
                        step
                        .mid_collective_recovery_status,

                    "Collective Recovery Post":
                        step
                        .post_collective_recovery_status,

                    "QKD Eligible":
                        step.qkd_eligible,

                    "Final Fidelity":
                        step.final_fidelity,

                    "Predicted QBER":
                        step.predicted_qber
                }

                for step
                in self._physical_execution_history
            ]
        )

In [ ]:
# @title Real Physically Timed NARAD-QC Stack + 600-km / R6 Experiment

PHASE3H3D_MAX_ELEMENTARY_HOP_KM = (
  500.0
)


def build_phase3h3d_physical_stack(
    total_distance_km=600.0,
    repeater_count=6,
    simulation_seed=4_020_002,
    generation_timing_geometry=(
        ElementaryAttemptTimingGeometry
        .MIDPOINT_HERALDED
    )
):

    total_distance_km = float(
        total_distance_km
    )


    if (
        not math.isfinite(
            total_distance_km
        )
        or total_distance_km <= 0.0
    ):

        raise ValueError(
            "total_distance_km must be positive and finite."
        )


    if (
        isinstance(
            repeater_count,
            bool
        )
        or not isinstance(
            repeater_count,
            int
        )
        or repeater_count < 1
    ):

        raise ValueError(
            "repeater_count must be a positive integer."
        )


    elementary_link_count = (
        repeater_count
        +
        1
    )


    hop_distance_km = (

        total_distance_km
        /
        elementary_link_count
    )


    # ------------------------------------------------------------
    # Frozen direct elementary-hop deployment constraint.
    # ------------------------------------------------------------

    if (
        hop_distance_km
        >
        PHASE3H3D_MAX_ELEMENTARY_HOP_KM
        +
        1e-12
    ):

        raise ValueError(
            f"Uniform hop distance {hop_distance_km:.6f} km "
            f"exceeds the frozen NARAD-QC elementary-hop "
            f"baseline of "
            f"{PHASE3H3D_MAX_ELEMENTARY_HOP_KM:.1f} km. "
            "Increase repeater density or use another route."
        )


    # ============================================================
    # Physical Topology
    # ============================================================

    topology = (
        NetworkTopology(

            name=(
                f"Phase3H3D-"
                f"{total_distance_km:.3f}km-"
                f"R{repeater_count}"
            )
        )
    )


    user_a = (
        NetworkNode(

            name="Physical-A",

            x_km=0.0,

            y_km=0.0,

            endpoint_capable=True,

            swapping_capable=False,

            memory_capacity=2
        )
    )


    repeaters = tuple(

        NetworkNode(

            name=f"Physical-R{i}",

            x_km=(
                hop_distance_km
                *
                i
            ),

            y_km=0.0,

            endpoint_capable=False,

            swapping_capable=True,

            memory_capacity=2
        )

        for i
        in range(
            1,
            repeater_count + 1
        )
    )


    user_b = (
        NetworkNode(

            name="Physical-B",

            x_km=(
                total_distance_km
            ),

            y_km=0.0,

            endpoint_capable=True,

            swapping_capable=False,

            memory_capacity=2
        )
    )


    path = (
        user_a,
        *repeaters,
        user_b
    )


    for node in path:

        topology.add_node(
            node
        )


    for left, right in zip(
        path[:-1],
        path[1:]
    ):

        topology.add_link(

            PhysicalQuantumLink(

                node_a=(
                    left
                ),

                node_b=(
                    right
                ),

                attenuation_db_per_km=(
                    PHASE3G2_FIBER_ATTENUATION_DB_PER_KM
                )
            )
        )


    # ============================================================
    # Simulator
    #
    # CRITICAL:
    #
    #     max_storage_slots = None
    #
    # Hard lifetime is now owned by the physical-time controller.
    # ============================================================

    simulator = (
        QuantumNetworkSimulator(

            topology=(
                topology
            ),

            simulation_seed=(
                simulation_seed
            ),

            controller_id=(
                f"PHASE3H3D-R{repeater_count}"
            ),

            parameter_set=(
                MLDSAParameterSet.ML_DSA_65
            ),

            max_storage_slots=None
        )
    )


    authentication = (
        simulator
        .authenticate_all_nodes()
    )


    assert all(

        record.status
        ==
        AuthenticationStatus.AUTHENTICATED

        for record
        in authentication.values()
    )


    assert (
        simulator
        .require_quantum_eligibility(
            *path
        )
    )


    # ============================================================
    # Elementary Generation
    # ============================================================

    generation_service = (
        ElementaryEntanglementService(

            simulator=(
                simulator
            ),

            model=(
                FiberSurvivalProxyModel()
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=(
                simulator
            ),

            service=(
                generation_service
            ),

            ordered_path=(
                path
            ),

            demand_active=False
        )
    )


    # ============================================================
    # Fidelity Physics
    #
    # The old profile slot_duration_s field remains present only
    # for backwards-compatible profile construction.
    #
    # The continuous-time registry NEVER calls fidelity_at_slot().
    # ============================================================

    fidelity_profile = (
        WernerPhysicalFidelityProfile(

            elementary_fidelity=(
                PHASE3G2_ELEMENTARY_FIDELITY
            ),

            bell_fidelity_floor=0.25,

            memory_reference_initial_fidelity=1.0,

            memory_reference_fidelity=0.90,

            memory_reference_time_s=1.0,

            slot_duration_s=(
                PHASE3G2_SLOT_DURATION_S
            ),

            swap_operation_visibility=1.0
        )
    )


    physical_time_ledger = (
        PhysicalTimeLedger(

            kernel=(
                simulator.kernel
            )
        )
    )


    fidelity_registry = (
        BarrierAwarePhysicalTimeFidelityRegistry(

            simulator=(
                simulator
            ),

            profile=(
                fidelity_profile
            ),

            time_ledger=(
                physical_time_ledger
            )
        )
    )


    # ============================================================
    # Physical BSM
    # ============================================================

    swap_model = (
        WernerStateAtomicBSMModel(

            fidelity_profile=(
                fidelity_profile
            ),

            atomic_bsm_success_probability=(
                PHASE3G2_ATOMIC_BSM_SUCCESS_PROBABILITY
            ),

            atomic_bsm_latency_us=10.0
        )
    )


    swap_engine = (
        EntanglementSwapEngine(

            simulator=(
                simulator
            ),

            model=(
                swap_model
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    # ============================================================
    # Physical-Time-Safe Control Policy
    # ============================================================

    policy = (
        MaxParallelDeterministicSwapPolicy()
    )


    swap_scheduler = (
        ControlPlaneParallelSwapScheduler(

            simulator=(
                simulator
            ),

            swap_engine=(
                swap_engine
            ),

            ordered_path=(
                path
            ),

            policy=(
                policy
            )
        )
    )


    # ============================================================
    # Failure-Aware Demand
    # ============================================================

    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=(
                simulator
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=(
                path
            )
        )
    )


    # ============================================================
    # Assemble stack before higher control layers
    # ============================================================

    physical_links = tuple(
        topology
        .path_links(
            path
        )
    )


    stack = {

        "policy":
            policy,

        "topology":
            topology,

        "simulator":
            simulator,

        "path":
            path,

        "repeaters":
            repeaters,

        "generation_service":
            generation_service,

        "generation_scheduler":
            generation_scheduler,

        "fidelity_profile":
            fidelity_profile,

        "fidelity_registry":
            fidelity_registry,

        "swap_model":
            swap_model,

        "swap_engine":
            swap_engine,

        "swap_scheduler":
            swap_scheduler,

        "demand_controller":
            demand_controller,

        "physical_time_ledger":
            physical_time_ledger,

        "timing_profile":
            PHASE3H3_TIMING_PROFILE,

        "generation_timing_geometry":
            generation_timing_geometry,

        "total_distance_km":
            total_distance_km,

        "repeater_count":
            repeater_count,

        "elementary_link_count":
            elementary_link_count,

        "hop_distance_km":
            hop_distance_km,

        "elementary_success_probability":
            physical_links[0]
            .survival_probability
    }


    # ============================================================
    # Hard Physical Qubit Lifetime
    # ============================================================

    physical_lifetime_controller = (
        PhysicalQubitLifetimeController(

            simulator=(
                simulator
            ),

            fidelity_registry=(
                fidelity_registry
            ),

            time_ledger=(
                physical_time_ledger
            ),

            max_storage_time_s=(
                PHASE3H3_MAX_STORAGE_TIME_S
            )
        )
    )


    stack[
        "physical_lifetime_controller"
    ] = (
        physical_lifetime_controller
    )


    # ============================================================
    # Fully Physical-Timed QKD Orchestrator
    # ============================================================

    orchestrator = (
        PhysicalTimedQKDReadyOrchestrator(

            stack=(
                stack
            ),

            timing_profile=(
                PHASE3H3_TIMING_PROFILE
            ),

            generation_timing_geometry=(
                generation_timing_geometry
            )
        )
    )


    stack[
        "orchestrator"
    ] = (
        orchestrator
    )


    # ============================================================
    # Construction invariants
    # ============================================================

    assert all(

        memory.max_storage_slots
        is None

        for memory
        in simulator.quantum_memories.values()
    )


    assert math.isclose(

        physical_time_ledger
        .current_time_s,

        0.0,

        rel_tol=0.0,

        abs_tol=1e-15
    )


    assert (
        simulator
        .kernel
        .current_slot
        ==
        0
    )


    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    return stack

In [ ]:
# @title Bug Repatch

def _phase3h3d_corrected_execute_bsm_phase(
    self
):

    candidates = (
        self.swap_scheduler
        .discover_candidates()
    )


    if not candidates:

        return (
            None,
            0.0
        )


    swap_round = (
        self.swap_scheduler
        .execute_parallel_round()
    )


    if (
        len(
            swap_round.attempts
        )
        ==
        0
    ):

        return (
            swap_round,
            0.0
        )


    # ============================================================
    # AUTHORITATIVE OWNERSHIP
    #
    # Orchestrator
    #     -> SwapScheduler
    #         -> SwapEngine
    #             -> Physical BSM Model
    #
    # The parent orchestrator intentionally owns the scheduler,
    # not a duplicate swap-engine reference.
    # ============================================================

    swap_engine = (
        self.swap_scheduler
        .swap_engine
    )


    swap_model = (
        swap_engine
        .model
    )


    if not hasattr(
        swap_model,
        "atomic_bsm_latency_us"
    ):

        raise RuntimeError(
            "Physical swap model does not expose "
            "atomic_bsm_latency_us."
        )


    atomic_bsm_latency_us = float(
        swap_model
        .atomic_bsm_latency_us
    )


    if (
        not math.isfinite(
            atomic_bsm_latency_us
        )
        or atomic_bsm_latency_us < 0.0
    ):

        raise RuntimeError(
            "atomic_bsm_latency_us must be a "
            "non-negative finite value."
        )


    # ------------------------------------------------------------
    # All selected conflict-free BSMs execute concurrently.
    #
    # Therefore:
    #
    #     phase duration = ONE BSM latency
    #
    # rather than:
    #
    #     number_of_BSMs * BSM latency
    # ------------------------------------------------------------

    bsm_duration_s = (

        atomic_bsm_latency_us
        *
        1e-6
    )


    self.time_ledger\
        .advance(

            duration_s=(
                bsm_duration_s
            ),

            phase_name=(
                "PARALLEL_ATOMIC_BSM"
            ),

            resource_ids=tuple(
                swap_round
                .output_resource_ids
            ),

            metadata={

                "parallel_bsm_attempts":
                    len(
                        swap_round.attempts
                    ),

                "parallel_bsm_successes":
                    swap_round
                    .success_count,

                "parallel_bsm_failures":
                    swap_round
                    .failure_count,

                "atomic_bsm_latency_us":
                    atomic_bsm_latency_us
            }
        )


    return (
        swap_round,
        float(
            bsm_duration_s
        )
    )


# ================================================================
# Install corrected method
# ================================================================

PhysicalTimedQKDReadyOrchestrator\
    ._execute_bsm_phase = (
        _phase3h3d_corrected_execute_bsm_phase
    )


# ================================================================
# Sanity-check ownership path before another physical experiment
# ================================================================

_test_stack = (
    build_phase3h3d_physical_stack(

        total_distance_km=600.0,

        repeater_count=6,

        simulation_seed=7_330_999,

        generation_timing_geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    )
)


_test_orchestrator = (
    _test_stack[
        "orchestrator"
    ]
)


assert (
    _test_orchestrator
    .swap_scheduler
    .swap_engine

    is

    _test_stack[
        "swap_engine"
    ]
)


assert (
    _test_orchestrator
    .swap_scheduler
    .swap_engine
    .model

    is

    _test_stack[
        "swap_model"
    ]
)


assert math.isclose(

    _test_orchestrator
    .swap_scheduler
    .swap_engine
    .model
    .atomic_bsm_latency_us,

    10.0,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Physical-timed orchestrator resolves the "
    "authoritative swap engine through SwapScheduler."
)

print(
    "[PASS] Atomic BSM latency resolved correctly:",
    _test_orchestrator
    .swap_scheduler
    .swap_engine
    .model
    .atomic_bsm_latency_us,
    "us"
)

print(
    "[PASS] Phase 3H.3D BSM ownership patch installed."
)

In [ ]:
# @title Real Physically Timed NARAD-QC Stack

print(
    "=" * 94
)

print(
    "Clean Fully Physical-Timed 600-km / R6 Run"
)

print(
    "=" * 94
)

phase3h3d_stack = (
    build_phase3h3d_physical_stack(

        total_distance_km=600.0,

        repeater_count=6,

        simulation_seed=4_020_002,

        generation_timing_geometry=(
            ElementaryAttemptTimingGeometry
            .MIDPOINT_HERALDED
        )
    )
)


# orchestrator = (
#     phase3h3d_stack[
#         "orchestrator"
#     ]
# )


# result = (
#     orchestrator
#     .run_until_qkd_ready_physical(

#         max_slots=20_000
#     )
# )


# summary = pd.DataFrame(
#     [
#         {
#             "Status":
#                 result.status.value,

#             "Logical Slots Processed":
#                 result.processed_slot_count,

#             "Final Logical Slot":
#                 result.end_slot,

#             "Physical Elapsed (s)":
#                 result.physical_elapsed_s,

#             "Physical Elapsed (ms)":
#                 result.physical_elapsed_s
#                 *
#                 1000.0,

#             "Elementary Attempts":
#                 result.total_elementary_attempts,

#             "Elementary Successes":
#                 result.total_elementary_successes,

#             "Elementary Failures":
#                 result.total_elementary_failures,

#             "BSM Attempts":
#                 result.total_bsm_attempts,

#             "BSM Successes":
#                 result.total_bsm_successes,

#             "BSM Failures":
#                 result.total_bsm_failures,

#             "Hard Physical Expirations":
#                 result
#                 .total_hard_physical_expirations,

#             "Individual Quality Discards":
#                 result
#                 .total_individual_quality_discards,

#             "Collective Recovery Executions":
#                 result
#                 .total_collective_recovery_executions,

#             "Collective Resources Discarded":
#                 result
#                 .total_collective_resources_discarded,

#             "Final Fidelity":
#                 result.final_fidelity,

#             "Predicted QBER":
#                 result.predicted_qber,

#             "QKD Limit":
#                 PHASE3H1_QUALITY_CONFIG
#                 .qber_limit
#         }
#     ]
# )


# display(
#     summary
# )


# print(
#     "\nFINAL 20 PHYSICAL-TIMED EXECUTION ROUNDS"
# )


# display(
#     orchestrator
#     .physical_history_dataframe()
#     .tail(
#         20
#     )
# )


# print(
#     "\nFINAL PHYSICAL TIME"
# )

# print(
#     phase3h3d_stack[
#         "physical_time_ledger"
#     ]
#     .current_time_s,
#     "seconds"
# )


# print(
#     "\nFINAL ACTIVE QKD FRONTIER"
# )


# resource_quality_df, frontier_quality_df = (
#     phase3h1_quality_snapshot(

#         stack=(
#             phase3h3d_stack
#         ),

#         simulation_slot=(
#             phase3h3d_stack[
#                 "simulator"
#             ]
#             .kernel
#             .current_slot
#         )
#     )
# )


# display(
#     frontier_quality_df
# )


# phase3h3d_stack[
#     "simulator"
# ]\
# .entanglement_manager\
# .validate_all_active_resources()


# assert (
#     phase3h3d_stack[
#         "simulator"
#     ]
#     .validate_runtime_consistency()
#     is True
# )


# if result.success:

#     assert (
#         result.predicted_qber
#         <=
#         PHASE3H1_QUALITY_CONFIG
#         .qber_limit
#         +
#         PHASE3H1_QUALITY_CONFIG
#         .numerical_tolerance
#     )


#     print(
#         "\n[PASS] PHYSICALLY TIMED QKD DELIVERY ACHIEVED."
#     )


# else:

#     print(
#         "\n[INFO] No QKD-ready pair within execution horizon."
#     )


# print(
#     "\n" + "=" * 94
# )

# print(
#     "Execution completed."
# )

# print(
#     "=" * 94
# )

In [ ]:
# @title Explicit Elementary-Link Multiplexing Data Model

from dataclasses import dataclass
from typing import Optional

import math
import pandas as pd


class ElementaryMultiplexingConfig:
    """
    Per-link optical multiplexing configuration.

    m_l = number of independent elementary-generation optical modes
          attempted for physical link l during ONE generation cycle.

    Important
    ---------
    Multiple mode successes do NOT imply multiple stored Bell pairs.

    Current Phase-3I baseline:

        m optical attempts
            ->
        zero or more heralded successes
            ->
        retain at most ONE elementary entanglement resource

    This preserves the current quantum-memory architecture while
    adding explicit multiplexed optical opportunity.
    """

    def __init__(
        self,
        default_mode_count=1,
        link_mode_counts=None
    ):

        if (
            isinstance(
                default_mode_count,
                bool
            )
            or not isinstance(
                default_mode_count,
                int
            )
            or default_mode_count < 1
        ):

            raise ValueError(
                "default_mode_count must be a positive integer."
            )


        self.default_mode_count = int(
            default_mode_count
        )


        self._link_mode_counts = {}


        for link_id, mode_count in dict(
            link_mode_counts or {}
        ).items():

            if (
                not isinstance(
                    link_id,
                    str
                )
                or not link_id.strip()
            ):

                raise ValueError(
                    "Every multiplexing link ID must be "
                    "a non-empty string."
                )


            if (
                isinstance(
                    mode_count,
                    bool
                )
                or not isinstance(
                    mode_count,
                    int
                )
                or mode_count < 1
            ):

                raise ValueError(
                    "Every per-link mode count must be "
                    "a positive integer."
                )


            self._link_mode_counts[
                link_id.strip()
            ] = int(
                mode_count
            )


    def mode_count_for(
        self,
        link_or_id
    ) -> int:

        link_id = (

            link_or_id.link_id

            if isinstance(
                link_or_id,
                PhysicalQuantumLink
            )

            else str(
                link_or_id
            )
        )


        return int(

            self._link_mode_counts
            .get(
                link_id,
                self.default_mode_count
            )
        )


    def dataframe(
        self,
        links
    ):

        return pd.DataFrame(
            [
                {
                    "Link ID":
                        link.link_id,

                    "Distance (km)":
                        link.distance_km,

                    "Mode Count":
                        self.mode_count_for(
                            link
                        )
                }

                for link
                in links
            ]
        )


@dataclass(
    frozen=True
)
class MultiplexedElementaryEntanglementAttempt(
    ElementaryEntanglementAttempt
):
    """
    Aggregate record for ONE multiplexed elementary-link cycle.

    Compatibility
    -------------
    This subclasses ElementaryEntanglementAttempt so the existing:

        ElementaryLinkGenerationProcess
        ParallelElementaryGenerationScheduler
        FailureAwareElementaryDemandController

    can continue consuming:

        attempt.success
        attempt.resource_id
        attempt.success_probability
        attempt.random_draw

    without architectural rewrites.

    Physical detail is retained explicitly below.
    """

    base_mode_success_probability: float = 0.0

    mode_count: int = 1

    mode_random_draws: tuple[
        float, ...
    ] = ()

    successful_mode_indices: tuple[
        int, ...
    ] = ()

    selected_mode_index: Optional[
        int
    ] = None


    def __post_init__(
        self
    ):

        super().__post_init__()


        if not (
            0.0
            <=
            self.base_mode_success_probability
            <=
            1.0
        ):

            raise ValueError(
                "base_mode_success_probability must lie in [0,1]."
            )


        if (
            isinstance(
                self.mode_count,
                bool
            )
            or not isinstance(
                self.mode_count,
                int
            )
            or self.mode_count < 1
        ):

            raise ValueError(
                "mode_count must be a positive integer."
            )


        if (
            len(
                self.mode_random_draws
            )
            !=
            self.mode_count
        ):

            raise ValueError(
                "mode_random_draws length must equal mode_count."
            )


        expected_successful_indices = tuple(

            index

            for index, draw
            in enumerate(
                self.mode_random_draws
            )

            if (
                draw
                <
                self.base_mode_success_probability
            )
        )


        if (
            expected_successful_indices
            !=
            self.successful_mode_indices
        ):

            raise ValueError(
                "successful_mode_indices disagree with "
                "explicit mode random draws."
            )


        expected_success = bool(
            expected_successful_indices
        )


        if (
            bool(
                self.success
            )
            !=
            expected_success
        ):

            raise ValueError(
                "Aggregate success disagrees with mode outcomes."
            )


        if self.success:

            if (
                self.selected_mode_index
                not in
                self.successful_mode_indices
            ):

                raise ValueError(
                    "selected_mode_index must identify a "
                    "successful physical mode."
                )

        else:

            if (
                self.selected_mode_index
                is not None
            ):

                raise ValueError(
                    "Failed multiplexed cycle cannot select a mode."
                )


    @property
    def successful_mode_count(
        self
    ) -> int:

        return len(
            self.successful_mode_indices
        )


    @property
    def unretained_success_count(
        self
    ) -> int:

        if not self.success:

            return 0


        return max(

            0,

            self.successful_mode_count
            -
            1
        )

In [ ]:
# @title Multiplexed Elementary Entanglement Service


class MultiplexedElementaryEntanglementService(
    ElementaryEntanglementService
):
    """
    Explicit m-mode elementary-entanglement generation service.

    During ONE requested link-generation cycle:

        1. Validate/authenticate link.
        2. Verify storage exists for ONE retained Bell pair.
        3. Resolve per-link mode count m_l.
        4. Draw m_l independent physical Bernoulli mode outcomes.
        5. If >=1 succeeds:
               retain the lowest-index successful mode;
               create exactly ONE EntanglementResource.
        6. Record all mode outcomes.
        7. Return one scheduler-compatible aggregate attempt.

    No additional physical time is introduced here.

    Physical phase timing still belongs to the Phase-3H.3D
    orchestrator.
    """

    def __init__(
        self,
        simulator,
        model,
        multiplexing_config=None
    ):

        super().__init__(

            simulator=(
                simulator
            ),

            model=(
                model
            )
        )


        self.multiplexing_config = (

            multiplexing_config

            if multiplexing_config
            is not None

            else ElementaryMultiplexingConfig(
                default_mode_count=1
            )
        )


        if not isinstance(
            self.multiplexing_config,
            ElementaryMultiplexingConfig
        ):

            raise TypeError(
                "multiplexing_config must be an "
                "ElementaryMultiplexingConfig."
            )


        self._multiplexed_attempt_history = []


    # ============================================================
    # Numerically Stable Combined Probability
    # ============================================================

    @staticmethod
    def multiplexed_success_probability(
        base_probability,
        mode_count
    ):

        p = float(
            base_probability
        )


        m = int(
            mode_count
        )


        if not (
            0.0
            <= p
            <= 1.0
        ):

            raise ValueError(
                "base_probability must lie in [0,1]."
            )


        if m < 1:

            raise ValueError(
                "mode_count must be positive."
            )


        if p == 0.0:

            return 0.0


        if p == 1.0:

            return 1.0


        # 1 - (1-p)^m
        return float(

            -
            math.expm1(

                m
                *
                math.log1p(
                    -p
                )
            )
        )


    # ============================================================
    # One Multiplexed Physical Cycle
    # ============================================================

    def attempt_once(
        self,
        link
    ):

        self.simulator\
            .validate_runtime_consistency()


        link = (
            self._validate_link(
                link
            )
        )


        # --------------------------------------------------------
        # Authentication remains mandatory.
        # --------------------------------------------------------

        self.simulator\
            .require_quantum_eligibility(

                link.node_a,
                link.node_b
            )


        # --------------------------------------------------------
        # We retain at most ONE success.
        #
        # Therefore only one storage pair is required.
        # --------------------------------------------------------

        self._require_storage_capacity(
            link
        )


        base_probability = float(

            self.model
            .success_probability(
                link
            )
        )


        mode_count = int(

            self.multiplexing_config
            .mode_count_for(
                link
            )
        )


        cycle_probability = (
            self.multiplexed_success_probability(

                base_probability=(
                    base_probability
                ),

                mode_count=(
                    mode_count
                )
            )
        )


        # ========================================================
        # EXPLICIT PHYSICAL MODE DRAWS
        # ========================================================

        mode_random_draws = tuple(

            float(
                self.simulator
                .kernel
                .rng
                .random()
            )

            for _
            in range(
                mode_count
            )
        )


        successful_mode_indices = tuple(

            index

            for index, draw
            in enumerate(
                mode_random_draws
            )

            if (
                draw
                <
                base_probability
            )
        )


        success = bool(
            successful_mode_indices
        )


        selected_mode_index = (

            successful_mode_indices[0]

            if success

            else None
        )


        # ========================================================
        # Equivalent aggregate Uniform(0,1) draw
        #
        # U_min has CDF:
        #
        #     G(x) = 1 - (1-x)^m
        #
        # Therefore:
        #
        #     U_cycle = G(U_min)
        #
        # is uniform, and:
        #
        #     U_cycle < P_cycle
        #
        # iff at least one physical mode succeeds.
        # ========================================================

        minimum_mode_draw = min(
            mode_random_draws
        )


        equivalent_cycle_draw = (

            1.0

            -

            (
                1.0
                -
                minimum_mode_draw
            )
            **
            mode_count
        )


        # ========================================================
        # Retain ONE heralded Bell pair
        # ========================================================

        resource = None


        if success:

            resource = (

                self.simulator
                .entanglement_manager
                .create_elementary_resource(
                    link
                )
            )


        attempt = (
            MultiplexedElementaryEntanglementAttempt(

                attempt_index=(
                    self.statistics
                    .total_attempts
                ),

                simulation_slot=(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                link_id=(
                    link.link_id
                ),

                node_a_name=(
                    link.node_a.name
                ),

                node_b_name=(
                    link.node_b.name
                ),

                model_name=(

                    "MULTIPLEXED["

                    +
                    self.model
                    .model_name

                    +
                    "]"
                ),

                success_probability=(
                    cycle_probability
                ),

                random_draw=(
                    equivalent_cycle_draw
                ),

                success=(
                    success
                ),

                resource_id=(

                    resource.resource_id

                    if resource
                    is not None

                    else None
                ),

                base_mode_success_probability=(
                    base_probability
                ),

                mode_count=(
                    mode_count
                ),

                mode_random_draws=(
                    mode_random_draws
                ),

                successful_mode_indices=(
                    successful_mode_indices
                ),

                selected_mode_index=(
                    selected_mode_index
                )
            )
        )


        # Existing aggregate statistics remain usable.
        self.statistics.record(
            attempt
        )


        self._multiplexed_attempt_history.append(
            attempt
        )


        self.simulator.kernel.emit(

            event_type=(

                "MULTIPLEXED_ELEMENTARY_ENTANGLEMENT_SUCCEEDED"

                if success

                else
                "MULTIPLEXED_ELEMENTARY_ENTANGLEMENT_FAILED"
            ),

            component=(
                "MultiplexedElementaryEntanglementService"
            ),

            node_names=(
                link.node_a.name,
                link.node_b.name
            ),

            link_ids=(
                link.link_id,
            ),

            resource_ids=(

                (
                    resource.resource_id,
                )

                if resource
                is not None

                else ()
            ),

            message=(

                f"Multiplexed elementary cycle on "
                f"'{link.link_id}' used {mode_count} modes; "
                f"{len(successful_mode_indices)} succeeded."
            ),

            metadata={

                "base_mode_success_probability":
                    base_probability,

                "mode_count":
                    mode_count,

                "cycle_success_probability":
                    cycle_probability,

                "successful_mode_indices":
                    successful_mode_indices,

                "successful_mode_count":
                    len(
                        successful_mode_indices
                    ),

                "selected_mode_index":
                    selected_mode_index,

                "unretained_success_count":
                    attempt
                    .unretained_success_count,

                "equivalent_cycle_draw":
                    equivalent_cycle_draw
            }
        )


        return attempt


    # ============================================================
    # Physical Mode Statistics
    # ============================================================

    @property
    def multiplexed_attempt_history(
        self
    ):

        return tuple(
            self._multiplexed_attempt_history
        )


    @property
    def total_generation_cycles(
        self
    ):

        return len(
            self._multiplexed_attempt_history
        )


    @property
    def total_mode_attempts(
        self
    ):

        return sum(

            attempt.mode_count

            for attempt
            in self._multiplexed_attempt_history
        )


    @property
    def total_mode_successes(
        self
    ):

        return sum(

            attempt
            .successful_mode_count

            for attempt
            in self._multiplexed_attempt_history
        )


    @property
    def total_unretained_mode_successes(
        self
    ):

        return sum(

            attempt
            .unretained_success_count

            for attempt
            in self._multiplexed_attempt_history
        )


    @property
    def empirical_mode_success_rate(
        self
    ):

        if (
            self.total_mode_attempts
            ==
            0
        ):

            return 0.0


        return (

            self.total_mode_successes

            /
            self.total_mode_attempts
        )


    def multiplexing_dataframe(
        self
    ):

        return pd.DataFrame(
            [
                {
                    "Cycle":
                        attempt
                        .attempt_index,

                    "Slot":
                        attempt
                        .simulation_slot,

                    "Link":
                        attempt
                        .link_id,

                    "Modes":
                        attempt
                        .mode_count,

                    "Base P":
                        attempt
                        .base_mode_success_probability,

                    "Cycle P":
                        attempt
                        .success_probability,

                    "Successful Modes":
                        attempt
                        .successful_mode_count,

                    "Successful Mode Indices":
                        attempt
                        .successful_mode_indices,

                    "Selected Mode":
                        attempt
                        .selected_mode_index,

                    "Unretained Successes":
                        attempt
                        .unretained_success_count,

                    "Cycle Success":
                        attempt
                        .success,

                    "Resource":
                        attempt
                        .resource_id
                }

                for attempt
                in self._multiplexed_attempt_history
            ]
        )

In [ ]:
# @title Multiplexing Verification

print(
    "=" * 88
)

print(
    "Explicit Elementary-Link Multiplexing Verification"
)

print(
    "=" * 88
)


# ================================================================
# PART A
# m = 1 must reproduce the original single-mode stochastic draw.
# ================================================================

seed = (
    7_310_001
)


baseline_stack = (
    build_phase3h3d_physical_stack(

        total_distance_km=600.0,

        repeater_count=6,

        simulation_seed=(
            seed
        )
    )
)


multiplex_stack = (
    build_phase3h3d_physical_stack(

        total_distance_km=600.0,

        repeater_count=6,

        simulation_seed=(
            seed
        )
    )
)


baseline_link = (

    baseline_stack[
        "topology"
    ]
    .path_links(

        baseline_stack[
            "path"
        ]
    )[0]
)


multiplex_link = (

    multiplex_stack[
        "topology"
    ]
    .path_links(

        multiplex_stack[
            "path"
        ]
    )[0]
)


baseline_attempt = (

    baseline_stack[
        "generation_service"
    ]
    .attempt_once(
        baseline_link
    )
)


m1_service = (
    MultiplexedElementaryEntanglementService(

        simulator=(
            multiplex_stack[
                "simulator"
            ]
        ),

        model=(
            FiberSurvivalProxyModel()
        ),

        multiplexing_config=(
            ElementaryMultiplexingConfig(
                default_mode_count=1
            )
        )
    )
)


m1_attempt = (
    m1_service
    .attempt_once(
        multiplex_link
    )
)


assert (
    m1_attempt
    .mode_count
    ==
    1
)


assert math.isclose(

    m1_attempt
    .base_mode_success_probability,

    baseline_attempt
    .success_probability,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    m1_attempt
    .success_probability,

    baseline_attempt
    .success_probability,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    m1_attempt
    .mode_random_draws[0],

    baseline_attempt
    .random_draw,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    m1_attempt
    .random_draw,

    baseline_attempt
    .random_draw,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert (
    m1_attempt.success
    ==
    baseline_attempt.success
)


print(
    "[PASS] m=1 multiplexing is exactly backward-compatible "
    "with the original single-mode physical draw."
)


# ================================================================
# PART B
# R6, m=8 statistical verification.
# ================================================================

stat_stack = (
    build_phase3h3d_physical_stack(

        total_distance_km=600.0,

        repeater_count=6,

        simulation_seed=(
            7_310_008
        )
    )
)


stat_simulator = (
    stat_stack[
        "simulator"
    ]
)


stat_manager = (
    stat_simulator
    .entanglement_manager
)


stat_link = (

    stat_stack[
        "topology"
    ]
    .path_links(

        stat_stack[
            "path"
        ]
    )[0]
)


mode_count = 8


stat_service = (
    MultiplexedElementaryEntanglementService(

        simulator=(
            stat_simulator
        ),

        model=(
            FiberSurvivalProxyModel()
        ),

        multiplexing_config=(
            ElementaryMultiplexingConfig(

                default_mode_count=(
                    mode_count
                )
            )
        )
    )
)


base_probability = float(
    stat_link
    .survival_probability
)


theoretical_cycle_probability = (
    stat_service
    .multiplexed_success_probability(

        base_probability=(
            base_probability
        ),

        mode_count=(
            mode_count
        )
    )
)


trial_count = (
    5_000
)


cycle_successes = 0


for _ in range(
    trial_count
):

    attempt = (
        stat_service
        .attempt_once(
            stat_link
        )
    )


    if attempt.success:

        cycle_successes += 1


        # Statistical test retains no generated Bell pair between
        # independent cycles.
        stat_manager\
            .discard_resource(

                resource_id=(
                    attempt
                    .resource_id
                ),

                reason=(
                    "phase3i1_statistical_trial_cleanup"
                )
            )


    # Logical ordering only.
    #
    # No physical-time meaning is attached to this increment.
    stat_simulator\
        .kernel\
        .advance_slot()


empirical_cycle_probability = (

    cycle_successes
    /
    trial_count
)


empirical_mode_probability = (
    stat_service
    .empirical_mode_success_rate
)


cycle_sigma = math.sqrt(

    theoretical_cycle_probability

    *
    (
        1.0
        -
        theoretical_cycle_probability
    )

    /
    trial_count
)


mode_trial_count = (
    stat_service
    .total_mode_attempts
)


mode_sigma = math.sqrt(

    base_probability

    *
    (
        1.0
        -
        base_probability
    )

    /
    mode_trial_count
)


assert (
    abs(
        empirical_cycle_probability
        -
        theoretical_cycle_probability
    )
    <=
    (
        6.0
        *
        cycle_sigma
    )
)


assert (
    abs(
        empirical_mode_probability
        -
        base_probability
    )
    <=
    (
        6.0
        *
        mode_sigma
    )
)


assert (
    stat_service
    .total_mode_attempts
    ==
    trial_count
    *
    mode_count
)


assert (
    len(
        stat_manager
        .active_resources
    )
    ==
    0
)


stat_manager\
    .validate_all_active_resources()


assert (
    stat_simulator
    .validate_runtime_consistency()
    is True
)


verification_df = (
    pd.DataFrame(
        [
            {
                "Hop Distance (km)":
                    stat_link
                    .distance_km,

                "Mode Count":
                    mode_count,

                "Base P(mode success)":
                    base_probability,

                "Theoretical P(cycle success)":
                    theoretical_cycle_probability,

                "Empirical P(cycle success)":
                    empirical_cycle_probability,

                "Total Cycles":
                    trial_count,

                "Total Optical Mode Attempts":
                    stat_service
                    .total_mode_attempts,

                "Total Mode Successes":
                    stat_service
                    .total_mode_successes,

                "Empirical P(mode success)":
                    empirical_mode_probability,

                "Unretained Successful Modes":
                    stat_service
                    .total_unretained_mode_successes
            }
        ]
    )
)


display(
    verification_df
)


print(
    "[PASS] Explicit mode-level statistics preserve the "
    "underlying single-mode probability."
)


print(
    "[PASS] Multiplexed cycle success agrees statistically with:"
)

print(
    "       P_m = 1 - (1 - p)^m"
)


print(
    "[PASS] At most one Bell resource was retained per "
    "multiplexed generation cycle."
)


print(
    "\n" + "=" * 88
)

print(
    "Multiplexing verification PASSED."
)

print(
    "=" * 88
)

In [ ]:
# @title Multiplexed Physical-Time NARAD-QC Stack

def build_phase3i2_multiplexed_stack(
    total_distance_km=600.0,
    repeater_count=6,
    simulation_seed=4_020_002,
    default_mode_count=1,
    link_mode_counts=None,
    generation_timing_geometry=(
        ElementaryAttemptTimingGeometry
        .MIDPOINT_HERALDED
    )
):

    # ============================================================
    # Start from the already validated physical-time architecture.
    # ============================================================

    stack = (
        build_phase3h3d_physical_stack(

            total_distance_km=(
                total_distance_km
            ),

            repeater_count=(
                repeater_count
            ),

            simulation_seed=(
                simulation_seed
            ),

            generation_timing_geometry=(
                generation_timing_geometry
            )
        )
    )


    simulator = (
        stack[
            "simulator"
        ]
    )


    path = (
        stack[
            "path"
        ]
    )


    # ============================================================
    # PRECONDITION
    #
    # We are rewiring BEFORE execution begins.
    # ============================================================

    assert (
        simulator
        .kernel
        .current_slot
        ==
        0
    )


    assert math.isclose(

        stack[
            "physical_time_ledger"
        ]
        .current_time_s,

        0.0,

        rel_tol=0.0,

        abs_tol=1e-15
    )


    assert (
        len(
            simulator
            .entanglement_manager
            .active_resources
        )
        ==
        0
    )


    # ============================================================
    # Multiplexing Configuration
    # ============================================================

    multiplexing_config = (
        ElementaryMultiplexingConfig(

            default_mode_count=(
                default_mode_count
            ),

            link_mode_counts=(
                link_mode_counts
            )
        )
    )


    # ============================================================
    # Replace ONLY elementary generation service.
    # ============================================================

    generation_service = (
        MultiplexedElementaryEntanglementService(

            simulator=(
                simulator
            ),

            model=(
                FiberSurvivalProxyModel()
            ),

            multiplexing_config=(
                multiplexing_config
            )
        )
    )


    # ============================================================
    # New scheduler/process objects must point to the new service.
    # ============================================================

    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=(
                simulator
            ),

            service=(
                generation_service
            ),

            ordered_path=(
                path
            ),

            demand_active=False
        )
    )


    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=(
                simulator
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=(
                path
            )
        )
    )


    # ============================================================
    # Update stack ownership.
    # ============================================================

    stack[
        "multiplexing_config"
    ] = (
        multiplexing_config
    )


    stack[
        "generation_service"
    ] = (
        generation_service
    )


    stack[
        "generation_scheduler"
    ] = (
        generation_scheduler
    )


    stack[
        "demand_controller"
    ] = (
        demand_controller
    )


    # ============================================================
    # Rebuild ONLY the top-level orchestrator so it observes the
    # multiplexed generation subsystem.
    #
    # Fidelity, swap engine, swap scheduler, time ledger and
    # physical lifetime controller remain exactly the validated
    # Phase-3H.3D instances.
    # ============================================================

    orchestrator = (
        PhysicalTimedQKDReadyOrchestrator(

            stack=(
                stack
            ),

            timing_profile=(
                stack[
                    "timing_profile"
                ]
            ),

            generation_timing_geometry=(
                generation_timing_geometry
            ),

            config=(
                PHASE3H1_QUALITY_CONFIG
            )
        )
    )


    stack[
        "orchestrator"
    ] = (
        orchestrator
    )


    # ============================================================
    # Cross-layer ownership checks.
    # ============================================================

    assert (
        orchestrator
        .generation_scheduler

        is

        generation_scheduler
    )


    assert (
        orchestrator
        .demand_controller

        is

        demand_controller
    )


    assert (
        generation_scheduler
        .service

        is

        generation_service
    )


    assert (
        simulator
        .validate_runtime_consistency()
        is True
    )


    return stack

In [ ]:
# @title Multiplexed Physical-Layer Accounting

@dataclass(frozen=True)
class MultiplexedNetworkAccounting:

    generation_cycles: int

    optical_mode_attempts: int

    successful_generation_cycles: int

    successful_optical_modes: int

    retained_elementary_resources: int

    unretained_successful_modes: int

    average_modes_per_generation_cycle: float

    empirical_mode_success_rate: float

    empirical_cycle_success_rate: float


def phase3i2_network_accounting(
    stack
):

    service = (
        stack[
            "generation_service"
        ]
    )


    if not isinstance(
        service,
        MultiplexedElementaryEntanglementService
    ):

        raise TypeError(
            "Stack does not use the multiplexed "
            "elementary-generation service."
        )


    attempts = (
        service
        .multiplexed_attempt_history
    )


    generation_cycles = len(
        attempts
    )


    successful_generation_cycles = sum(

        attempt.success

        for attempt
        in attempts
    )


    retained_elementary_resources = (
        successful_generation_cycles
    )


    average_modes = (

        service.total_mode_attempts
        /
        generation_cycles

        if generation_cycles

        else 0.0
    )


    empirical_cycle_success_rate = (

        successful_generation_cycles
        /
        generation_cycles

        if generation_cycles

        else 0.0
    )


    return (
        MultiplexedNetworkAccounting(

            generation_cycles=(
                generation_cycles
            ),

            optical_mode_attempts=(
                service
                .total_mode_attempts
            ),

            successful_generation_cycles=(
                successful_generation_cycles
            ),

            successful_optical_modes=(
                service
                .total_mode_successes
            ),

            retained_elementary_resources=(
                retained_elementary_resources
            ),

            unretained_successful_modes=(
                service
                .total_unretained_mode_successes
            ),

            average_modes_per_generation_cycle=(
                average_modes
            ),

            empirical_mode_success_rate=(
                service
                .empirical_mode_success_rate
            ),

            empirical_cycle_success_rate=(
                empirical_cycle_success_rate
            )
        )
    )


def phase3i2_accounting_dataframe(
    stack
):

    accounting = (
        phase3i2_network_accounting(
            stack
        )
    )


    return pd.DataFrame(
        [
            asdict(
                accounting
            )
        ]
    )

In [ ]:
# @title Full Physical-Time Multiplexing Integration Verification

# print(
#     "=" * 94
# )

# print(
#     "Full Physical-Time Multiplexing Integration"
# )

# print(
#     "=" * 94
# )


# # ================================================================
# # TEST 1
# # m = 1 full-network regression
# # ================================================================

# m1_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=4_020_002,

#         default_mode_count=1
#     )
# )


# m1_result = (
#     m1_stack[
#         "orchestrator"
#     ]
#     .run_until_qkd_ready_physical(

#         max_slots=20_000
#     )
# )


# assert (
#     m1_result.success
#     is True
# )


# # Frozen Phase-3H.3D result for same seed.
# assert (
#     m1_result.processed_slot_count
#     ==
#     1494
# )


# assert math.isclose(

#     m1_result.physical_elapsed_s,

#     0.7151571428571549,

#     rel_tol=0.0,

#     abs_tol=1e-12
# )


# assert math.isclose(

#     m1_result.final_fidelity,

#     0.952856,

#     rel_tol=0.0,

#     abs_tol=1e-6
# )


# assert math.isclose(

#     m1_result.predicted_qber,

#     0.031429,

#     rel_tol=0.0,

#     abs_tol=1e-6
# )


# m1_accounting = (
#     phase3i2_network_accounting(
#         m1_stack
#     )
# )


# assert (
#     m1_accounting
#     .optical_mode_attempts
#     ==
#     m1_accounting
#     .generation_cycles
# )


# print(
#     "[PASS] m=1 full-network multiplexed stack exactly "
#     "reproduces the validated Phase-3H.3D physical run."
# )


# # ================================================================
# # TEST 2
# # m = 8 full-network physical run
# # ================================================================

# m8_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=4_020_002,

#         default_mode_count=8
#     )
# )


# m8_result = (
#     m8_stack[
#         "orchestrator"
#     ]
#     .run_until_qkd_ready_physical(

#         max_slots=20_000
#     )
# )


# assert (
#     m8_result.success
#     is True
# )


# m8_accounting = (
#     phase3i2_network_accounting(
#         m8_stack
#     )
# )


# # Every multiplexed link cycle uses exactly eight optical modes.
# assert (
#     m8_accounting
#     .optical_mode_attempts
#     ==
#     8
#     *
#     m8_accounting
#     .generation_cycles
# )


# # ================================================================
# # Physical timing invariant
# #
# # Multiplexing must NOT multiply propagation/heralding duration.
# # ================================================================

# m8_history = (
#     m8_stack[
#         "orchestrator"
#     ]
#     .physical_history_dataframe()
# )


# nonzero_generation_rows = (

#     m8_history[

#         m8_history[
#             "Generation Duration (ms)"
#         ]
#         >
#         0.0

#     ]
# )


# expected_r6_cycle_ms = (

#     PHASE3H3_TIMING_PROFILE
#     .elementary_attempt_cycle_s(

#         distance_km=(
#             600.0 / 7.0
#         ),

#         geometry=(
#             ElementaryAttemptTimingGeometry
#             .MIDPOINT_HERALDED
#         )
#     )

#     *
#     1000.0
# )


# assert all(

#     math.isclose(

#         float(
#             duration_ms
#         ),

#         expected_r6_cycle_ms,

#         rel_tol=0.0,

#         abs_tol=1e-12
#     )

#     for duration_ms
#     in (
#         nonzero_generation_rows[
#             "Generation Duration (ms)"
#         ]
#     )
# )


# print(
#     "[PASS] m=8 uses eight independent optical modes "
#     "without multiplying one generation-cycle latency."
# )


# # ================================================================
# # Comparison
# # ================================================================

# comparison = pd.DataFrame(
#     [
#         {
#             "Mode Count":
#                 1,

#             "Success":
#                 m1_result.success,

#             "Logical Slots":
#                 m1_result.processed_slot_count,

#             "Physical Time (s)":
#                 m1_result.physical_elapsed_s,

#             "Generation Cycles":
#                 m1_accounting
#                 .generation_cycles,

#             "Optical Mode Attempts":
#                 m1_accounting
#                 .optical_mode_attempts,

#             "Successful Modes":
#                 m1_accounting
#                 .successful_optical_modes,

#             "Unretained Successful Modes":
#                 m1_accounting
#                 .unretained_successful_modes,

#             "BSM Attempts":
#                 m1_result.total_bsm_attempts,

#             "Final Fidelity":
#                 m1_result.final_fidelity,

#             "Predicted QBER":
#                 m1_result.predicted_qber
#         },

#         {
#             "Mode Count":
#                 8,

#             "Success":
#                 m8_result.success,

#             "Logical Slots":
#                 m8_result.processed_slot_count,

#             "Physical Time (s)":
#                 m8_result.physical_elapsed_s,

#             "Generation Cycles":
#                 m8_accounting
#                 .generation_cycles,

#             "Optical Mode Attempts":
#                 m8_accounting
#                 .optical_mode_attempts,

#             "Successful Modes":
#                 m8_accounting
#                 .successful_optical_modes,

#             "Unretained Successful Modes":
#                 m8_accounting
#                 .unretained_successful_modes,

#             "BSM Attempts":
#                 m8_result.total_bsm_attempts,

#             "Final Fidelity":
#                 m8_result.final_fidelity,

#             "Predicted QBER":
#                 m8_result.predicted_qber
#         }
#     ]
# )


# display(
#     comparison
# )


# print(
#     "\nM=8 MULTIPLEXING ACCOUNTING"
# )


# display(
#     phase3i2_accounting_dataframe(
#         m8_stack
#     )
# )


# print(
#     "\nM=8 FINAL 20 PHYSICAL EXECUTION ROUNDS"
# )


# display(
#     m8_stack[
#         "orchestrator"
#     ]
#     .physical_history_dataframe()
#     .tail(
#         20
#     )
# )


# # ================================================================
# # Runtime
# # ================================================================

# m8_stack[
#     "simulator"
# ]\
# .entanglement_manager\
# .validate_all_active_resources()


# assert (
#     m8_stack[
#         "simulator"
#     ]
#     .validate_runtime_consistency()
#     is True
# )


# print(
#     "[PASS] Multiplexed execution remains cross-layer consistent."
# )


# print(
#     "\n" + "=" * 94
# )

# print(
#     "Physical-time multiplexing integration PASSED."
# )

# print(
#     "=" * 94
# )

In [ ]:
# @title Multiplexing-Aware Collective Recovery Planner

class MultiplexingAwareQKDMinimumCostCollectiveRecoveryPlanner(
    QKDMinimumCostCollectiveRecoveryPlanner
):
    """
    Phase-3I extension of the exact Phase-3H.2 planner.

    Existing candidate search, QKD feasibility evaluation,
    deterministic tie-breaking, and lifecycle semantics remain
    unchanged.

    Only elementary regeneration work changes.

    Single mode:
        E[generation cycles] = 1 / p

    m multiplexed modes:
        P_cycle = 1 - (1-p)^m

        E[generation cycles] = 1 / P_cycle

    IMPORTANT
    ---------
    The inherited field:

        incremental_expected_attempt_work

    now means expected LINK-LEVEL GENERATION-CYCLE work.

    Optical mode expenditure is tracked separately and is NOT
    silently folded into the latency objective.
    """

    def __init__(
        self,
        stack,
        config=PHASE3H1_QUALITY_CONFIG,
        max_exact_frontier_resources=16
    ):

        super().__init__(

            stack=stack,

            config=config,

            max_exact_frontier_resources=(
                max_exact_frontier_resources
            )
        )


        self.multiplexing_config = (
            stack.get(
                "multiplexing_config",
                None
            )
        )


    # ============================================================
    # Per-Link Mode Count
    # ============================================================

    def _mode_count_for_link(
        self,
        link
    ) -> int:

        if (
            self.multiplexing_config
            is None
        ):

            return 1


        return int(

            self.multiplexing_config
            .mode_count_for(
                link
            )
        )


    # ============================================================
    # Multiplexed Cycle Probability
    # ============================================================

    def _link_cycle_success_probability(
        self,
        link_id
    ) -> float:

        link = (
            self._link_by_id[
                link_id
            ]
        )


        p = float(
            link.survival_probability
        )


        m = (
            self._mode_count_for_link(
                link
            )
        )


        if p <= 0.0:

            return 0.0


        if p >= 1.0:

            return 1.0


        return float(

            -
            math.expm1(

                m
                *
                math.log1p(
                    -p
                )
            )
        )


    # ============================================================
    # Planner Objective:
    # Expected LINK-LEVEL GENERATION CYCLES
    # ============================================================

    def _link_expected_attempt_work(
        self,
        link_id
    ) -> float:

        probability = (
            self._link_cycle_success_probability(
                link_id
            )
        )


        if probability <= 0.0:

            return math.inf


        return float(
            1.0
            /
            probability
        )


    # ============================================================
    # Separate Optical Resource Accounting
    # ============================================================

    def link_expected_optical_mode_work(
        self,
        link_id
    ) -> float:

        link = (
            self._link_by_id[
                link_id
            ]
        )


        mode_count = (
            self._mode_count_for_link(
                link
            )
        )


        cycle_probability = (
            self._link_cycle_success_probability(
                link_id
            )
        )


        if cycle_probability <= 0.0:

            return math.inf


        return float(

            mode_count

            /
            cycle_probability
        )


    def expected_optical_mode_work(
        self,
        link_ids
    ) -> float:

        return float(

            sum(

                self.link_expected_optical_mode_work(
                    link_id
                )

                for link_id
                in link_ids
            )
        )

In [ ]:
# @title Install Multiplexing-Aware Recovery into a Stack

def phase3i2_install_multiplexing_aware_recovery(
    stack,
    config=PHASE3H1_QUALITY_CONFIG
):

    orchestrator = (
        stack[
            "orchestrator"
        ]
    )


    planner = (
        MultiplexingAwareQKDMinimumCostCollectiveRecoveryPlanner(

            stack=stack,

            config=config
        )
    )


    controller = (
        QKDSelectiveRecoveryLifecycleController(

            stack=stack,

            planner=planner,

            config=config
        )
    )


    orchestrator\
        .collective_recovery_planner = (
            planner
        )


    orchestrator\
        .collective_recovery_controller = (
            controller
        )


    stack[
        "collective_recovery_planner"
    ] = (
        planner
    )


    stack[
        "collective_recovery_controller"
    ] = (
        controller
    )


    return planner

In [ ]:
# @title Multiplexing-Aware Recovery Cost Verification

# print(
#     "=" * 88
# )

# print(
#     "Multiplexing-Aware Recovery Cost Verification"
# )

# print(
#     "=" * 88
# )


# # ================================================================
# # m = 1
# # ================================================================

# recovery_m1_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=7_312_001,

#         default_mode_count=1
#     )
# )


# planner_m1 = (
#     phase3i2_install_multiplexing_aware_recovery(
#         recovery_m1_stack
#     )
# )


# link_m1 = (

#     recovery_m1_stack[
#         "topology"
#     ]
#     .path_links(

#         recovery_m1_stack[
#             "path"
#         ]
#     )[0]
# )


# m1_cycle_probability = (
#     planner_m1
#     ._link_cycle_success_probability(
#         link_m1.link_id
#     )
# )


# m1_expected_cycles = (
#     planner_m1
#     ._link_expected_attempt_work(
#         link_m1.link_id
#     )
# )


# m1_expected_optical = (
#     planner_m1
#     .link_expected_optical_mode_work(
#         link_m1.link_id
#     )
# )


# assert math.isclose(

#     m1_cycle_probability,

#     link_m1
#     .survival_probability,

#     rel_tol=0.0,

#     abs_tol=1e-15
# )


# assert math.isclose(

#     m1_expected_cycles,

#     1.0
#     /
#     link_m1
#     .survival_probability,

#     rel_tol=0.0,

#     abs_tol=1e-12
# )


# assert math.isclose(

#     m1_expected_cycles,

#     m1_expected_optical,

#     rel_tol=0.0,

#     abs_tol=1e-12
# )


# print(
#     "[PASS] m=1 recovery work exactly reproduces "
#     "the original 1/p planner cost."
# )


# # ================================================================
# # m = 8
# # ================================================================

# recovery_m8_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=7_312_008,

#         default_mode_count=8
#     )
# )


# planner_m8 = (
#     phase3i2_install_multiplexing_aware_recovery(
#         recovery_m8_stack
#     )
# )


# link_m8 = (

#     recovery_m8_stack[
#         "topology"
#     ]
#     .path_links(

#         recovery_m8_stack[
#             "path"
#         ]
#     )[0]
# )


# base_p = float(
#     link_m8
#     .survival_probability
# )


# expected_p8 = float(

#     1.0

#     -
#     (
#         1.0
#         -
#         base_p
#     )
#     **
#     8
# )


# actual_p8 = (
#     planner_m8
#     ._link_cycle_success_probability(
#         link_m8.link_id
#     )
# )


# expected_cycle_work = (
#     1.0
#     /
#     expected_p8
# )


# expected_optical_work = (

#     8.0

#     /
#     expected_p8
# )


# actual_cycle_work = (
#     planner_m8
#     ._link_expected_attempt_work(
#         link_m8.link_id
#     )
# )


# actual_optical_work = (
#     planner_m8
#     .link_expected_optical_mode_work(
#         link_m8.link_id
#     )
# )


# assert math.isclose(

#     actual_p8,

#     expected_p8,

#     rel_tol=0.0,

#     abs_tol=1e-15
# )


# assert math.isclose(

#     actual_cycle_work,

#     expected_cycle_work,

#     rel_tol=0.0,

#     abs_tol=1e-12
# )


# assert math.isclose(

#     actual_optical_work,

#     expected_optical_work,

#     rel_tol=0.0,

#     abs_tol=1e-12
# )


# assert (
#     actual_cycle_work
#     <
#     m1_expected_cycles
# )


# assert (
#     actual_optical_work
#     >
#     actual_cycle_work
# )


# verification = (
#     pd.DataFrame(
#         [
#             {
#                 "Mode Count":
#                     1,

#                 "Base Mode P":
#                     base_p,

#                 "Cycle Success P":
#                     m1_cycle_probability,

#                 "Expected Generation Cycles":
#                     m1_expected_cycles,

#                 "Expected Optical Mode Attempts":
#                     m1_expected_optical
#             },

#             {
#                 "Mode Count":
#                     8,

#                 "Base Mode P":
#                     base_p,

#                 "Cycle Success P":
#                     actual_p8,

#                 "Expected Generation Cycles":
#                     actual_cycle_work,

#                 "Expected Optical Mode Attempts":
#                     actual_optical_work
#             }
#         ]
#     )
# )


# display(
#     verification
# )


# assert (
#     recovery_m8_stack[
#         "orchestrator"
#     ]
#     .collective_recovery_planner

#     is

#     planner_m8
# )


# assert (
#     recovery_m8_stack[
#         "orchestrator"
#     ]
#     .collective_recovery_controller
#     .planner

#     is

#     planner_m8
# )


# print(
#     "[PASS] QKD selective recovery now understands "
#     "multiplexed generation probability."
# )


# print(
#     "[PASS] Recovery distinguishes expected generation-cycle "
#     "work from optical-mode expenditure."
# )


# print(
#     "\n" + "=" * 88
# )

# print(
#     "Recovery integration PASSED."
# )

# print(
#     "=" * 88
# )

### Phase 3E : 05-10-2026T15:39:52 -- 13-10-2026T15:07:32
QKD Integration

In [ ]:
# @title QKD Consumption Lifecycle + Entanglement-Based Measurement Record

from dataclasses import dataclass, asdict, replace
from typing import Optional

import math
import numpy as np
import pandas as pd


# ================================================================
# Public CONSUMED lifecycle transition
#
# EXPIRED / DISCARDED already have public manager transitions.
#
# QKD measurement is fundamentally different:
#
#     the resource was USED successfully.
#
# Therefore its terminal state must be CONSUMED rather than
# DISCARDED.
# ================================================================

def _phase3j_consume_resource(
    self,
    resource_id,
    reason="qkd_measurement_consumed"
):
    """
    Consume one ACTIVE entanglement resource.

    Both physical endpoint halves are removed atomically and the
    global resource enters EntanglementStatus.CONSUMED.

    This is used when endpoint measurement irreversibly destroys
    an end-to-end Bell pair during QKD.
    """

    resource = (
        self.get_resource(
            resource_id
        )
    )


    if not resource.is_active:

        raise EntanglementResourceStateError(
            f"Resource '{resource_id}' is already "
            f"{resource.status.value}."
        )


    # ------------------------------------------------------------
    # Full preflight BEFORE mutation.
    # ------------------------------------------------------------

    self._assert_active_storage_consistency(
        resource
    )


    memory_a = (
        self.memory_for_node(
            resource.endpoint_a_name
        )
    )


    memory_b = (
        self.memory_for_node(
            resource.endpoint_b_name
        )
    )


    # ------------------------------------------------------------
    # Measurement consumes both endpoint physical qubits.
    # ------------------------------------------------------------

    memory_a.remove_local_half(

        resource.resource_id,

        reason=(
            "qkd_measurement_consumed"
        )
    )


    memory_b.remove_local_half(

        resource.resource_id,

        reason=(
            "qkd_measurement_consumed"
        )
    )


    terminal_resource = (
        replace(

            resource,

            status=(
                EntanglementStatus.CONSUMED
            ),

            ended_slot=(
                self.kernel.current_slot
            ),

            termination_reason=str(
                reason
            )
        )
    )


    self._resources[
        resource.resource_id
    ] = (
        terminal_resource
    )


    self.kernel.emit(

        event_type=(
            "ENTANGLEMENT_RESOURCE_CONSUMED"
        ),

        component=(
            "EntanglementManager"
        ),

        node_names=(
            resource.endpoint_a_name,
            resource.endpoint_b_name
        ),

        resource_ids=(
            resource.resource_id,
        ),

        message=(
            f"Entanglement resource "
            f"'{resource.resource_id}' was consumed by QKD."
        ),

        metadata={
            "ended_slot":
                self.kernel.current_slot,

            "reason":
                str(
                    reason
                )
        }
    )


    return terminal_resource


if not hasattr(
    EntanglementManager,
    "consume_resource"
):

    EntanglementManager.consume_resource = (
        _phase3j_consume_resource
    )


assert callable(
    getattr(
        EntanglementManager,
        "consume_resource",
        None
    )
)


# ================================================================
# One actual entanglement-based QKD measurement
# ================================================================

@dataclass(frozen=True)
class NARADQKDPairMeasurement:

    pair_index: int

    resource_id: str

    delivery_start_slot: int
    delivery_end_slot: int

    delivery_start_physical_time_s: float
    delivery_end_physical_time_s: float

    delivery_physical_time_s: float

    elementary_generation_cycles: int

    optical_mode_attempts: int

    bsm_attempts: int

    fidelity: float

    visibility: float

    predicted_qber: float

    alice_basis: int
    bob_basis: int

    basis_match: bool

    alice_bit: int
    bob_bit: int

    bit_error: bool

    consumed: bool

In [ ]:
# @title Werner Bell-Pair Measurement Model + NARAD Entanglement-Based QKD Channel

def narad_sample_werner_bell_measurement(
    fidelity,
    alice_basis,
    bob_basis,
    protocol_rng
):
    """
    Sample endpoint measurement outcomes for the currently frozen
    Bell-depolarized / Werner resource model.

    Basis convention
    ----------------
        0 = Z
        1 = X

    For |Phi+>-centered Werner states:

        same basis:
            P(A != B) = Q = (1-w)/2

        different bases:
            no useful key correlation;
            Bob is sampled independently.

    This is NOT a full density-matrix simulator.
    It is the exact stochastic measurement abstraction implied by
    the scalar Werner model currently used by NARAD-QC.
    """

    fidelity = float(
        fidelity
    )


    alice_basis = int(
        alice_basis
    )


    bob_basis = int(
        bob_basis
    )


    if alice_basis not in (
        0,
        1
    ):

        raise ValueError(
            "alice_basis must be 0 (Z) or 1 (X)."
        )


    if bob_basis not in (
        0,
        1
    ):

        raise ValueError(
            "bob_basis must be 0 (Z) or 1 (X)."
        )


    visibility = (
        narad_visibility_from_fidelity(
            fidelity
        )
    )


    predicted_qber = (
        narad_qber_from_visibility(
            visibility
        )
    )


    alice_bit = int(

        protocol_rng
        .integers(
            0,
            2
        )
    )


    basis_match = (
        alice_basis
        ==
        bob_basis
    )


    if basis_match:

        bit_error = bool(

            protocol_rng.random()

            <
            predicted_qber
        )


        bob_bit = (

            alice_bit
            ^
            int(
                bit_error
            )
        )


    else:

        # Cross-basis Werner correlations vanish.
        bob_bit = int(

            protocol_rng
            .integers(
                0,
                2
            )
        )


        # Not a sifted-key error.
        bit_error = False


    return {
        "visibility":
            float(
                visibility
            ),

        "predicted_qber":
            float(
                predicted_qber
            ),

        "alice_bit":
            int(
                alice_bit
            ),

        "bob_bit":
            int(
                bob_bit
            ),

        "basis_match":
            bool(
                basis_match
            ),

        "bit_error":
            bool(
                bit_error
            )
    }


class NARADQKDEntanglementChannel:
    """
    Protocol bridge between the completed NARAD delivery stack and
    entanglement-based QKD.

    One call to deliver_and_measure():

        1. requests one QKD-ready end-to-end entangled resource;
        2. records its network-layer fidelity / predicted QBER;
        3. chooses or accepts Alice/Bob measurement bases;
        4. samples endpoint Bell-pair measurement;
        5. consumes the physical entanglement resource;
        6. reconciles delivery demand for the next pair;
        7. advances logical ordering exactly once.

    Physical statistics remain physical.
    QKD statistics remain protocol-level.
    """

    def __init__(
        self,
        stack,
        protocol_seed=7_400_001,
        measurement_latency_s=0.0
    ):

        self.stack = stack


        self.simulator = (
            stack[
                "simulator"
            ]
        )


        self.orchestrator = (
            stack[
                "orchestrator"
            ]
        )


        self.manager = (
            self.simulator
            .entanglement_manager
        )


        self.time_ledger = (
            stack[
                "physical_time_ledger"
            ]
        )


        self.generation_service = (
            stack[
                "generation_service"
            ]
        )


        if not isinstance(
            self.generation_service,
            MultiplexedElementaryEntanglementService
        ):

            raise TypeError(
                "Phase 3J requires the multiplexed "
                "elementary-generation service."
            )


        if (
            isinstance(
                protocol_seed,
                bool
            )
            or not isinstance(
                protocol_seed,
                (int, np.integer)
            )
            or protocol_seed < 0
        ):

            raise ValueError(
                "protocol_seed must be a non-negative integer."
            )


        self.protocol_seed = int(
            protocol_seed
        )


        self.protocol_rng = (
            np.random.default_rng(
                self.protocol_seed
            )
        )


        measurement_latency_s = float(
            measurement_latency_s
        )


        if (
            not math.isfinite(
                measurement_latency_s
            )
            or measurement_latency_s < 0.0
        ):

            raise ValueError(
                "measurement_latency_s must be non-negative."
            )


        # Explicitly zero for the current baseline.
        #
        # We will not invent detector/readout latency without a
        # defensible hardware model.
        self.measurement_latency_s = (
            measurement_latency_s
        )


        self._measurement_history = []


    # ============================================================
    # Basis Sampling
    # ============================================================

    def _sample_basis(
        self,
        z_probability
    ):

        z_probability = float(
            z_probability
        )


        if not (
            0.0
            <=
            z_probability
            <=
            1.0
        ):

            raise ValueError(
                "z_probability must lie within [0,1]."
            )


        # 0 = Z
        # 1 = X
        return (

            0

            if (
                self.protocol_rng.random()
                <
                z_probability
            )

            else 1
        )


    # ============================================================
    # One Delivered Bell Pair -> One QKD Measurement
    # ============================================================

    def deliver_and_measure(
        self,
        alice_basis=None,
        bob_basis=None,
        z_probability=0.5,
        max_slots_per_pair=20_000
    ):

        pair_index = len(
            self._measurement_history
        )


        delivery_start_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        delivery_start_time_s = float(
            self.time_ledger
            .current_time_s
        )


        optical_before = (
            self.generation_service
            .total_mode_attempts
        )


        # ========================================================
        # Request ONE genuinely QKD-ready resource.
        # ========================================================

        delivery_result = (
            self.orchestrator
            .run_until_qkd_ready_physical(

                max_slots=(
                    max_slots_per_pair
                )
            )
        )


        if not delivery_result.success:

            raise RuntimeError(
                "NARAD delivery layer failed to produce a "
                "QKD-ready end-to-end pair."
            )


        if (
            delivery_result
            .end_to_end_resource_id
            is None
        ):

            raise RuntimeError(
                "QKD-ready delivery succeeded without a "
                "resource ID."
            )


        resource = (
            self.manager
            .get_resource(

                delivery_result
                .end_to_end_resource_id
            )
        )


        if not resource.is_active:

            raise RuntimeError(
                "Delivered QKD resource is not ACTIVE."
            )


        fidelity = float(
            delivery_result
            .final_fidelity
        )


        predicted_qber = float(
            delivery_result
            .predicted_qber
        )


        if (
            predicted_qber
            >
            PHASE3H1_QUALITY_CONFIG.qber_limit
            +
            PHASE3H1_QUALITY_CONFIG.numerical_tolerance
        ):

            raise RuntimeError(
                "Network delivered a pair above the "
                "QKD admission threshold."
            )


        # ========================================================
        # Optional QKD measurement latency.
        #
        # Current baseline = 0 by explicit assumption.
        # ========================================================

        if (
            self.measurement_latency_s
            >
            0.0
        ):

            self.time_ledger.advance(

                duration_s=(
                    self.measurement_latency_s
                ),

                phase_name=(
                    "ENDPOINT_QKD_MEASUREMENT"
                ),

                resource_ids=(
                    resource.resource_id,
                )
            )


            # Measurement latency can age the pair.
            #
            # Re-evaluate its current physical fidelity.
            fidelity = float(

                self.stack[
                    "fidelity_registry"
                ]
                .fidelity(
                    resource
                )
            )


            predicted_qber = float(

                narad_qber_from_fidelity(
                    fidelity
                )
            )


            if (
                predicted_qber
                >
                PHASE3H1_QUALITY_CONFIG.qber_limit
                +
                PHASE3H1_QUALITY_CONFIG.numerical_tolerance
            ):

                raise RuntimeError(
                    "QKD pair became inadmissible during "
                    "measurement latency."
                )


        # ========================================================
        # QKD basis choices
        # ========================================================

        if alice_basis is None:

            alice_basis = (
                self._sample_basis(
                    z_probability
                )
            )


        if bob_basis is None:

            bob_basis = (
                self._sample_basis(
                    z_probability
                )
            )


        measurement = (
            narad_sample_werner_bell_measurement(

                fidelity=(
                    fidelity
                ),

                alice_basis=(
                    alice_basis
                ),

                bob_basis=(
                    bob_basis
                ),

                protocol_rng=(
                    self.protocol_rng
                )
            )
        )


        optical_after = (
            self.generation_service
            .total_mode_attempts
        )


        # ========================================================
        # Physical Bell pair is irreversibly consumed.
        # ========================================================

        consumed_resource = (
            self.manager
            .consume_resource(

                resource_id=(
                    resource.resource_id
                ),

                reason=(
                    "endpoint_qkd_measurement"
                )
            )
        )


        if (
            consumed_resource.status
            !=
            EntanglementStatus.CONSUMED
        ):

            raise RuntimeError(
                "QKD measurement failed to consume "
                "the Bell resource."
            )


        # Reconcile the newly uncovered elementary path.
        self.orchestrator\
            .demand_controller\
            .reconcile()


        physical_end_time_s = float(
            self.time_ledger
            .current_time_s
        )


        record = (
            NARADQKDPairMeasurement(

                pair_index=(
                    pair_index
                ),

                resource_id=(
                    resource.resource_id
                ),

                delivery_start_slot=(
                    delivery_start_slot
                ),

                delivery_end_slot=int(
                    delivery_result
                    .end_slot
                ),

                delivery_start_physical_time_s=(
                    delivery_start_time_s
                ),

                delivery_end_physical_time_s=(
                    physical_end_time_s
                ),

                delivery_physical_time_s=(
                    physical_end_time_s
                    -
                    delivery_start_time_s
                ),

                elementary_generation_cycles=int(
                    delivery_result
                    .total_elementary_attempts
                ),

                optical_mode_attempts=int(
                    optical_after
                    -
                    optical_before
                ),

                bsm_attempts=int(
                    delivery_result
                    .total_bsm_attempts
                ),

                fidelity=(
                    fidelity
                ),

                visibility=float(
                    measurement[
                        "visibility"
                    ]
                ),

                predicted_qber=(
                    predicted_qber
                ),

                alice_basis=int(
                    alice_basis
                ),

                bob_basis=int(
                    bob_basis
                ),

                basis_match=(
                    measurement[
                        "basis_match"
                    ]
                ),

                alice_bit=(
                    measurement[
                        "alice_bit"
                    ]
                ),

                bob_bit=(
                    measurement[
                        "bob_bit"
                    ]
                ),

                bit_error=(
                    measurement[
                        "bit_error"
                    ]
                ),

                consumed=True
            )
        )


        self._measurement_history.append(
            record
        )


        self.simulator.kernel.emit(

            event_type=(
                "QKD_BELL_PAIR_MEASURED"
            ),

            component=(
                "NARADQKDEntanglementChannel"
            ),

            node_names=(
                resource.endpoint_a_name,
                resource.endpoint_b_name
            ),

            resource_ids=(
                resource.resource_id,
            ),

            message=(
                "One QKD-ready end-to-end Bell pair "
                "was measured and consumed."
            ),

            metadata={
                "pair_index":
                    pair_index,

                "fidelity":
                    fidelity,

                "predicted_qber":
                    predicted_qber,

                "alice_basis":
                    int(
                        alice_basis
                    ),

                "bob_basis":
                    int(
                        bob_basis
                    ),

                "basis_match":
                    measurement[
                        "basis_match"
                    ],

                "bit_error":
                    measurement[
                        "bit_error"
                    ],

                "optical_mode_attempts":
                    int(
                        optical_after
                        -
                        optical_before
                    )
            }
        )


        # ========================================================
        # Close this protocol-use cycle.
        #
        # Logical slots remain ordering only.
        #
        # Advancing once ensures the NEXT pair-delivery request
        # cannot reprocess the same elementary links twice in one
        # logical slot.
        # ========================================================

        self.simulator\
            .kernel\
            .advance_slot()


        self.simulator\
            .validate_runtime_consistency()


        return record


    # ============================================================
    # History
    # ============================================================

    @property
    def measurement_history(
        self
    ):

        return tuple(
            self._measurement_history
        )


    def measurement_history_dataframe(
        self
    ):

        return pd.DataFrame(
            [
                asdict(
                    record
                )

                for record
                in self._measurement_history
            ]
        )

In [ ]:
# @title Entanglement-to-QKD Bridge Verification

# print(
#     "=" * 92
# )

# print(
#     "NARAD Entanglement-to-QKD Verification"
# )

# print(
#     "=" * 92
# )


# # ================================================================
# # TEST 1
# # Werner measurement statistics
# # ================================================================

# test_fidelity = (
#     0.952856
# )


# expected_qber = (
#     narad_qber_from_fidelity(
#         test_fidelity
#     )
# )


# measurement_rng = (
#     np.random.default_rng(
#         7_401_001
#     )
# )


# sample_count = (
#     50_000
# )


# errors = 0


# for _ in range(
#     sample_count
# ):

#     sample = (
#         narad_sample_werner_bell_measurement(

#             fidelity=(
#                 test_fidelity
#             ),

#             alice_basis=0,

#             bob_basis=0,

#             protocol_rng=(
#                 measurement_rng
#             )
#         )
#     )


#     errors += int(
#         sample[
#             "bit_error"
#         ]
#     )


# empirical_qber = (

#     errors
#     /
#     sample_count
# )


# sigma = math.sqrt(

#     expected_qber

#     *
#     (
#         1.0
#         -
#         expected_qber
#     )

#     /
#     sample_count
# )


# assert (
#     abs(
#         empirical_qber
#         -
#         expected_qber
#     )

#     <=

#     6.0
#     *
#     sigma
# )


# print(
#     "[PASS] Same-basis Bell measurements reproduce the "
#     "Werner-model QBER statistically."
# )


# print(
#     "Expected QBER:",
#     expected_qber
# )


# print(
#     "Empirical QBER:",
#     empirical_qber
# )


# # ================================================================
# # TEST 2
# # Build REAL 600-km / R6 / m=8 delivery stack
# # ================================================================

# qkd_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=4_030_001,

#         default_mode_count=8,

#         generation_timing_geometry=(
#             ElementaryAttemptTimingGeometry
#             .MIDPOINT_HERALDED
#         )
#     )
# )


# # Multiplexing-aware selective recovery.
# phase3i2_install_multiplexing_aware_recovery(
#     qkd_stack
# )


# qkd_channel = (
#     NARADQKDEntanglementChannel(

#         stack=(
#             qkd_stack
#         ),

#         protocol_seed=(
#             7_401_008
#         ),

#         measurement_latency_s=0.0
#     )
# )


# # ================================================================
# # Force same-basis Z/Z measurement for this integration test.
# #
# # Later the protocol layer samples bases naturally.
# # ================================================================

# pair_record = (
#     qkd_channel
#     .deliver_and_measure(

#         alice_basis=0,

#         bob_basis=0,

#         max_slots_per_pair=20_000
#     )
# )


# assert (
#     pair_record
#     .basis_match
#     is True
# )


# assert (
#     pair_record
#     .predicted_qber
#     <=
#     PHASE3H1_QUALITY_CONFIG
#     .qber_limit
#     +
#     PHASE3H1_QUALITY_CONFIG
#     .numerical_tolerance
# )


# assert (
#     pair_record
#     .consumed
#     is True
# )


# terminal_resource = (
#     qkd_stack[
#         "simulator"
#     ]
#     .entanglement_manager
#     .get_resource(
#         pair_record
#         .resource_id
#     )
# )


# assert (
#     terminal_resource.status
#     ==
#     EntanglementStatus.CONSUMED
# )


# # Full path Bell pair is gone after measurement.
# assert (
#     len(
#         qkd_stack[
#             "simulator"
#         ]
#         .entanglement_manager
#         .active_resources
#     )
#     ==
#     0
# )


# assert (
#     qkd_stack[
#         "simulator"
#     ]
#     .validate_runtime_consistency()
#     is True
# )


# print(
#     "[PASS] NARAD produced one genuinely QKD-ready "
#     "600-km Bell pair."
# )


# print(
#     "[PASS] Endpoint QKD measurement consumed the physical "
#     "end-to-end entanglement resource."
# )


# print(
#     "[PASS] Multiplexed physical resource accounting survived "
#     "the QKD handoff."
# )


# print(
#     "\nREAL NARAD-QKD PAIR MEASUREMENT"
# )


# display(
#     qkd_channel
#     .measurement_history_dataframe()
# )


# print(
#     "\nRESOURCE AFTER QKD MEASUREMENT"
# )


# display(
#     qkd_stack[
#         "simulator"
#     ]
#     .entanglement_manager
#     .resources_dataframe()
#     .tail(
#         1
#     )
# )


# print(
#     "\n" + "=" * 92
# )

# print(
#     "Entanglement-to-QKD bridge PASSED."
# )

# print(
#     "=" * 92
# )

In [ ]:
# @title 80:20 Biased-Basis BBM92 Block Configuration

from dataclasses import dataclass, asdict
from collections import deque

import math
import numpy as np
import pandas as pd


def narad_binary_entropy(
    probability
):
    """
    Binary Shannon entropy h2(p).

    Returned in bits.
    """

    p = float(
        probability
    )


    if not (
        0.0 <= p <= 1.0
    ):

        raise ValueError(
            "probability must lie in [0,1]."
        )


    if (
        p == 0.0
        or
        p == 1.0
    ):

        return 0.0


    return float(

        -p
        *
        math.log2(
            p
        )

        -
        (
            1.0 - p
        )
        *
        math.log2(
            1.0 - p
        )
    )


@dataclass(frozen=True)
class BiasedBBM92Config:
    """
    Phase-3J biased entanglement-based QKD configuration.

    Basis convention
    ----------------
        0 = Z  -> key-generation basis
        1 = X  -> parameter-estimation basis

    Current baseline
    ----------------
        P(Z) = 0.80
        P(X) = 0.20

    Security note
    -------------
    This is a simulation-level finite-block construction.

    parameter_estimation_margin is an explicit conservative
    engineering margin under the current symmetric Werner model;
    it is NOT a composable finite-key security proof.
    """

    target_secret_bits: int = 128

    z_probability: float = 0.80

    qber_abort_threshold: float = 0.11

    minimum_parameter_estimation_bits: int = 24

    parameter_estimation_margin: float = 0.01

    security_margin_bits: int = 32

    reconciliation_efficiency_forecast: float = 1.16

    collection_headroom_bits: int = 32

    collection_check_interval_pairs: int = 16

    max_delivered_pairs: int = 5_000

    max_slots_per_pair: int = 20_000

    cascade_passes: int = 8


    def __post_init__(
        self
    ):

        if (
            isinstance(
                self.target_secret_bits,
                bool
            )
            or self.target_secret_bits <= 0
        ):

            raise ValueError(
                "target_secret_bits must be positive."
            )


        # if not (
        #     0.5
        #     <
        #     self.z_probability
        #     <
        #     1.0
        # ):

        #     raise ValueError(
        #         "z_probability must lie strictly within (0.5,1)."
        #     )


        if not (
            0.0
            <
            self.qber_abort_threshold
            <
            0.5
        ):

            raise ValueError(
                "qber_abort_threshold must lie in (0,0.5)."
            )


        if (
            self.minimum_parameter_estimation_bits
            <
            1
        ):

            raise ValueError(
                "minimum_parameter_estimation_bits "
                "must be positive."
            )


        if not (
            0.0
            <=
            self.parameter_estimation_margin
            <
            0.5
        ):

            raise ValueError(
                "parameter_estimation_margin must lie in [0,0.5)."
            )


        if (
            self.security_margin_bits
            <
            0
        ):

            raise ValueError(
                "security_margin_bits cannot be negative."
            )


        if (
            self.reconciliation_efficiency_forecast
            <
            1.0
        ):

            raise ValueError(
                "reconciliation_efficiency_forecast "
                "must be >= 1."
            )


PHASE3J_BIASED_BBM92_CONFIG = (
    BiasedBBM92Config()
)


@dataclass(frozen=True)
class BiasedBBM92CollectionSummary:

    delivered_pairs: int

    zz_key_basis_bits: int

    xx_parameter_estimation_bits: int

    mismatched_basis_pairs: int

    observed_x_qber: float

    mean_predicted_qber: float

    phase_error_estimate: float

    forecast_ec_leakage_bits: int

    forecast_extractable_bits: int

    collection_ready: bool

    physical_time_s: float

    elementary_generation_cycles: int

    optical_mode_attempts: int

    bsm_attempts: int


def phase3j_bbm92_collection_summary(
    measurements,
    config=PHASE3J_BIASED_BBM92_CONFIG
):
    """
    Read-only session-progress calculation.

    XX outcomes:
        sacrificed entirely for parameter estimation.

    ZZ outcomes:
        retained as candidate raw key material.

    Mismatched ZX/XZ:
        discarded during basis sifting.
    """

    measurements = tuple(
        measurements
    )


    zz = tuple(

        record

        for record
        in measurements

        if (
            record.alice_basis == 0
            and
            record.bob_basis == 0
        )
    )


    xx = tuple(

        record

        for record
        in measurements

        if (
            record.alice_basis == 1
            and
            record.bob_basis == 1
        )
    )


    mismatched = (

        len(
            measurements
        )

        -
        len(
            zz
        )

        -
        len(
            xx
        )
    )


    observed_x_qber = (

        sum(
            int(
                record.alice_bit
                !=
                record.bob_bit
            )

            for record
            in xx
        )

        /
        len(
            xx
        )

        if xx

        else 1.0
    )


    mean_predicted_qber = (

        sum(
            record.predicted_qber

            for record
            in measurements
        )

        /
        len(
            measurements
        )

        if measurements

        else 0.0
    )


    # ============================================================
    # Phase-error estimate
    #
    # Under the currently frozen symmetric Werner model:
    #
    #     X and Z same-basis QBERs are equal in expectation.
    #
    # Use the worse of:
    #     observed XX QBER
    #     average network-predicted QBER
    #
    # and add an explicit engineering PE margin.
    # ============================================================

    if (
        len(
            xx
        )
        >=
        config.minimum_parameter_estimation_bits
    ):

        phase_error_estimate = min(

            0.5,

            max(
                observed_x_qber,
                mean_predicted_qber
            )

            +
            config.parameter_estimation_margin
        )


    else:

        phase_error_estimate = 0.5


    n_key = len(
        zz
    )


    if (
        phase_error_estimate
        <
        0.5
    ):

        h_phase = (
            narad_binary_entropy(
                phase_error_estimate
            )
        )


        forecast_ec_leakage_bits = int(
            math.ceil(

                config
                .reconciliation_efficiency_forecast

                *
                n_key

                *
                h_phase
            )
        )


        forecast_extractable_bits = int(
            math.floor(

                n_key
                *
                (
                    1.0
                    -
                    h_phase
                )

                -
                forecast_ec_leakage_bits

                -
                config.security_margin_bits
            )
        )


    else:

        forecast_ec_leakage_bits = (
            n_key
        )

        forecast_extractable_bits = 0


    collection_ready = (

        len(
            xx
        )
        >=
        config.minimum_parameter_estimation_bits

        and

        phase_error_estimate
        <=
        config.qber_abort_threshold

        and

        forecast_extractable_bits
        >=
        (
            config.target_secret_bits
            +
            config.collection_headroom_bits
        )
    )


    physical_time_s = sum(

        record.delivery_physical_time_s

        for record
        in measurements
    )


    elementary_generation_cycles = sum(

        record.elementary_generation_cycles

        for record
        in measurements
    )


    optical_mode_attempts = sum(

        record.optical_mode_attempts

        for record
        in measurements
    )


    bsm_attempts = sum(

        record.bsm_attempts

        for record
        in measurements
    )


    return (
        BiasedBBM92CollectionSummary(

            delivered_pairs=len(
                measurements
            ),

            zz_key_basis_bits=(
                len(
                    zz
                )
            ),

            xx_parameter_estimation_bits=(
                len(
                    xx
                )
            ),

            mismatched_basis_pairs=(
                mismatched
            ),

            observed_x_qber=float(
                observed_x_qber
            ),

            mean_predicted_qber=float(
                mean_predicted_qber
            ),

            phase_error_estimate=float(
                phase_error_estimate
            ),

            forecast_ec_leakage_bits=int(
                forecast_ec_leakage_bits
            ),

            forecast_extractable_bits=int(
                max(
                    0,
                    forecast_extractable_bits
                )
            ),

            collection_ready=bool(
                collection_ready
            ),

            physical_time_s=float(
                physical_time_s
            ),

            elementary_generation_cycles=int(
                elementary_generation_cycles
            ),

            optical_mode_attempts=int(
                optical_mode_attempts
            ),

            bsm_attempts=int(
                bsm_attempts
            )
        )
    )

In [ ]:
# @title Repeat NARAD Delivery Until BBM92 Block Is Distillable

def collect_biased_bbm92_block(
    qkd_channel,
    config=PHASE3J_BIASED_BBM92_CONFIG,
    verbose=True
):
    """
    Repeatedly request, measure and consume end-to-end Bell pairs.

    Basis policy:
        Alice: Z with probability 0.80
        Bob:   Z with probability 0.80

    ZZ -> candidate raw key
    XX -> parameter estimation
    ZX/XZ -> sifted away

    Stops only after the forecast says enough entropy remains for:

        EC leakage
        +
        privacy amplification
        +
        explicit security margin
        +
        requested 128 secret bits
    """

    if not isinstance(
        qkd_channel,
        NARADQKDEntanglementChannel
    ):

        raise TypeError(
            "qkd_channel must be a "
            "NARADQKDEntanglementChannel."
        )


    measurements = []


    for pair_number in range(
        1,
        config.max_delivered_pairs + 1
    ):

        record = (
            qkd_channel
            .deliver_and_measure(

                alice_basis=None,

                bob_basis=None,

                z_probability=(
                    config.z_probability
                ),

                max_slots_per_pair=(
                    config.max_slots_per_pair
                )
            )
        )


        measurements.append(
            record
        )


        should_check = (

            pair_number
            %
            config.collection_check_interval_pairs
            ==
            0
        )


        if not should_check:

            continue


        summary = (
            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    config
                )
            )
        )


        if verbose:

            x_qber_text = (

                f"{summary.observed_x_qber:.2%}"

                if (
                    summary
                    .xx_parameter_estimation_bits
                    >
                    0
                )

                else "N/A"
            )


            print(

                f"Pairs={summary.delivered_pairs} | "
                f"ZZ={summary.zz_key_basis_bits} | "
                f"XX={summary.xx_parameter_estimation_bits} | "
                f"mismatch={summary.mismatched_basis_pairs} | "
                f"Q_X={x_qber_text} | "
                f"Q_phase={summary.phase_error_estimate:.2%} | "
                f"forecast_secret="
                f"{summary.forecast_extractable_bits}"
            )


        if (
            summary.collection_ready
        ):

            return (
                tuple(
                    measurements
                ),
                summary
            )


    final_summary = (
        phase3j_bbm92_collection_summary(

            measurements=(
                measurements
            ),

            config=(
                config
            )
        )
    )


    raise RuntimeError(

        "Biased-BBM92 collection reached "
        f"{config.max_delivered_pairs} delivered Bell pairs "
        "without accumulating a distillable block. "
        f"Final forecast="
        f"{final_summary.forecast_extractable_bits} bits."
    )

In [ ]:
# @title Cascade-Style Error Reconciliation With Leakage Accounting

@dataclass(frozen=True)
class NARADCascadeResult:

    success: bool

    input_length: int

    estimated_qber: float

    passes_used: int

    corrections: int

    parity_bits_disclosed: int

    residual_errors: int

    alice_reconciled_bits: tuple[
        int, ...
    ]

    bob_reconciled_bits: tuple[
        int, ...
    ]


def _narad_block_parity(
    bits,
    indices
):

    parity = 0


    for index in indices:

        parity ^= int(
            bits[
                index
            ]
        )


    return parity


def narad_cascade_reconcile(
    alice_bits,
    bob_bits,
    estimated_qber,
    protocol_rng,
    passes=8
):
    """
    Practical Cascade-style interactive reconciliation model.

    Every parity comparison is counted as one publicly disclosed
    bit and therefore contributes to EC leakage.

    This implementation includes:
        - random permutation each pass;
        - increasing block sizes;
        - parity mismatch detection;
        - binary search correction;
        - backtracking into earlier passes.

    It NEVER silently forces Bob's key equal to Alice's.
    """

    alice = tuple(
        int(
            bit
        )

        for bit
        in alice_bits
    )


    bob = [

        int(
            bit
        )

        for bit
        in bob_bits
    ]


    if (
        len(
            alice
        )
        !=
        len(
            bob
        )
    ):

        raise ValueError(
            "Alice and Bob raw keys must have equal length."
        )


    n = len(
        alice
    )


    if n == 0:

        raise ValueError(
            "Cannot reconcile an empty raw key."
        )


    q = max(

        1.0 / n,

        min(
            0.25,
            float(
                estimated_qber
            )
        )
    )


    initial_block_size = max(

        2,

        int(
            round(
                0.73
                /
                q
            )
        )
    )


    initial_block_size = min(
        initial_block_size,
        n
    )


    parity_bits_disclosed = 0

    corrections = 0


    pass_blocks = []

    pass_index_to_block = []


    # ============================================================
    # Public parity comparison
    # ============================================================

    def parity_mismatch(
        indices
    ):

        nonlocal parity_bits_disclosed


        parity_bits_disclosed += 1


        return (

            _narad_block_parity(
                alice,
                indices
            )

            !=

            _narad_block_parity(
                bob,
                indices
            )
        )


    # ============================================================
    # Binary search within a block already KNOWN to have odd error
    # parity.
    # ============================================================

    def locate_and_correct(
        indices
    ):

        nonlocal parity_bits_disclosed
        nonlocal corrections


        search = list(
            indices
        )


        while (
            len(
                search
            )
            >
            1
        ):

            midpoint = (
                len(
                    search
                )
                //
                2
            )


            left = (
                search[
                    :midpoint
                ]
            )


            right = (
                search[
                    midpoint:
                ]
            )


            parity_bits_disclosed += 1


            left_mismatch = (

                _narad_block_parity(
                    alice,
                    left
                )

                !=

                _narad_block_parity(
                    bob,
                    left
                )
            )


            search = (

                left

                if left_mismatch

                else right
            )


        corrected_index = (
            search[0]
        )


        bob[
            corrected_index
        ] ^= 1


        corrections += 1


        return corrected_index


    # ============================================================
    # Cascade passes
    # ============================================================

    for pass_number in range(
        passes
    ):

        block_size = min(

            n,

            initial_block_size
            *
            (
                2
                **
                pass_number
            )
        )


        permutation = (
            protocol_rng
            .permutation(
                n
            )
            .tolist()
        )


        blocks = tuple(

            tuple(
                permutation[
                    start:
                    start + block_size
                ]
            )

            for start
            in range(
                0,
                n,
                block_size
            )
        )


        index_to_block = {}


        for block_number, block in enumerate(
            blocks
        ):

            for original_index in block:

                index_to_block[
                    original_index
                ] = (
                    block_number
                )


        pass_blocks.append(
            blocks
        )


        pass_index_to_block.append(
            index_to_block
        )


        # --------------------------------------------------------
        # Backtracking queue.
        # --------------------------------------------------------

        def backtrack(
            corrected_index,
            previous_pass_limit
        ):

            queue = deque()


            for earlier_pass in range(
                previous_pass_limit
            ):

                queue.append(
                    (
                        earlier_pass,

                        pass_index_to_block[
                            earlier_pass
                        ][
                            corrected_index
                        ]
                    )
                )


            while queue:

                earlier_pass, block_number = (
                    queue.popleft()
                )


                earlier_block = (
                    pass_blocks[
                        earlier_pass
                    ][
                        block_number
                    ]
                )


                if not parity_mismatch(
                    earlier_block
                ):

                    continue


                newly_corrected_index = (
                    locate_and_correct(
                        earlier_block
                    )
                )


                for even_earlier_pass in range(
                    earlier_pass
                ):

                    queue.append(
                        (
                            even_earlier_pass,

                            pass_index_to_block[
                                even_earlier_pass
                            ][
                                newly_corrected_index
                            ]
                        )
                    )


        # --------------------------------------------------------
        # Process every block of this pass.
        # --------------------------------------------------------

        for block in blocks:

            if not parity_mismatch(
                block
            ):

                continue


            corrected_index = (
                locate_and_correct(
                    block
                )
            )


            backtrack(

                corrected_index=(
                    corrected_index
                ),

                previous_pass_limit=(
                    pass_number
                )
            )


        # Early termination is allowed only after exact equality.
        if all(

            alice[index]
            ==
            bob[index]

            for index
            in range(
                n
            )
        ):

            return (
                NARADCascadeResult(

                    success=True,

                    input_length=(
                        n
                    ),

                    estimated_qber=(
                        q
                    ),

                    passes_used=(
                        pass_number
                        +
                        1
                    ),

                    corrections=(
                        corrections
                    ),

                    parity_bits_disclosed=(
                        parity_bits_disclosed
                    ),

                    residual_errors=0,

                    alice_reconciled_bits=(
                        alice
                    ),

                    bob_reconciled_bits=tuple(
                        bob
                    )
                )
            )


    residual_errors = sum(

        int(
            alice[index]
            !=
            bob[index]
        )

        for index
        in range(
            n
        )
    )


    return (
        NARADCascadeResult(

            success=(
                residual_errors
                ==
                0
            ),

            input_length=(
                n
            ),

            estimated_qber=(
                q
            ),

            passes_used=(
                passes
            ),

            corrections=(
                corrections
            ),

            parity_bits_disclosed=(
                parity_bits_disclosed
            ),

            residual_errors=(
                residual_errors
            ),

            alice_reconciled_bits=(
                alice
            ),

            bob_reconciled_bits=tuple(
                bob
            )
        )
    )

In [ ]:
# @title Privacy Amplification + Complete 128-bit NARAD-QC Session

def narad_toeplitz_hash(
    input_bits,
    output_length,
    protocol_rng
):
    """
    Universal-2 Toeplitz hashing.

    Returns:
        output_bits
        public_seed

    Seed length:
        n + l - 1
    """

    input_bits = tuple(
        int(
            bit
        )

        for bit
        in input_bits
    )


    n = len(
        input_bits
    )


    output_length = int(
        output_length
    )


    if not (
        1
        <=
        output_length
        <=
        n
    ):

        raise ValueError(
            "output_length must lie within [1, input length]."
        )


    seed = (
        protocol_rng
        .integers(

            0,
            2,

            size=(
                n
                +
                output_length
                -
                1
            ),

            dtype=np.uint8
        )
    )


    first_column = (
        seed[
            :output_length
        ]
    )


    first_row = np.concatenate(
        (
            seed[
                :1
            ],

            seed[
                output_length:
            ]
        )
    )


    source = np.asarray(

        input_bits,

        dtype=np.uint8
    )


    output = np.zeros(

        output_length,

        dtype=np.uint8
    )


    for row in range(
        output_length
    ):

        accumulator = 0


        for column in range(
            n
        ):

            if (
                row
                >=
                column
            ):

                matrix_value = (
                    first_column[
                        row
                        -
                        column
                    ]
                )


            else:

                matrix_value = (
                    first_row[
                        column
                        -
                        row
                    ]
                )


            accumulator ^= (

                int(
                    matrix_value
                )

                &

                int(
                    source[
                        column
                    ]
                )
            )


        output[
            row
        ] = (
            accumulator
        )


    return (
        tuple(
            int(
                bit
            )

            for bit
            in output.tolist()
        ),

        tuple(
            int(
                bit
            )

            for bit
            in seed.tolist()
        )
    )


@dataclass(frozen=True)
class NARADBiasedBBM92SessionResult:

    success: bool

    delivered_pairs: int

    zz_raw_key_bits: int

    xx_parameter_estimation_bits: int

    mismatched_basis_pairs: int

    observed_x_qber: float

    actual_z_qber_diagnostic: float

    mean_network_predicted_qber: float

    phase_error_estimate: float

    cascade_passes: int

    cascade_corrections: int

    ec_leakage_bits: int

    security_margin_bits: int

    maximum_extractable_bits: int

    final_secret_bits: int

    physical_time_s: float

    elementary_generation_cycles: int

    optical_mode_attempts: int

    bsm_attempts: int

    alice_key_bytes: bytes

    bob_key_bytes: bytes


def distill_biased_bbm92_session(
    qkd_channel,
    measurements,
    collection_summary,
    config=PHASE3J_BIASED_BBM92_CONFIG
):

    measurements = tuple(
        measurements
    )


    zz = tuple(

        record

        for record
        in measurements

        if (
            record.alice_basis == 0
            and
            record.bob_basis == 0
        )
    )


    xx = tuple(

        record

        for record
        in measurements

        if (
            record.alice_basis == 1
            and
            record.bob_basis == 1
        )
    )


    if (
        len(
            xx
        )
        <
        config.minimum_parameter_estimation_bits
    ):

        raise RuntimeError(
            "Insufficient XX parameter-estimation data."
        )


    if (
        collection_summary
        .phase_error_estimate

        >
        config.qber_abort_threshold
    ):

        raise RuntimeError(
            "Estimated phase error exceeds QKD threshold."
        )


    alice_raw = tuple(

        record.alice_bit

        for record
        in zz
    )


    bob_raw = tuple(

        record.bob_bit

        for record
        in zz
    )


    actual_z_qber = (

        sum(

            int(
                a != b
            )

            for a, b
            in zip(
                alice_raw,
                bob_raw
            )
        )

        /
        len(
            alice_raw
        )
    )


    # ============================================================
    # ERROR RECONCILIATION
    # ============================================================

    reconciliation = (
        narad_cascade_reconcile(

            alice_bits=(
                alice_raw
            ),

            bob_bits=(
                bob_raw
            ),

            estimated_qber=(
                collection_summary
                .phase_error_estimate
            ),

            protocol_rng=(
                qkd_channel
                .protocol_rng
            ),

            passes=(
                config
                .cascade_passes
            )
        )
    )


    if not reconciliation.success:

        raise RuntimeError(

            "Cascade reconciliation failed. "
            f"Residual errors="
            f"{reconciliation.residual_errors}."
        )


    assert (

        reconciliation
        .alice_reconciled_bits

        ==

        reconciliation
        .bob_reconciled_bits
    )


    # ============================================================
    # SECRET ENTROPY BUDGET
    # ============================================================

    phase_entropy = (
        narad_binary_entropy(

            collection_summary
            .phase_error_estimate
        )
    )


    maximum_extractable_bits = int(
        math.floor(

            len(
                alice_raw
            )

            *
            (
                1.0
                -
                phase_entropy
            )

            -
            reconciliation
            .parity_bits_disclosed

            -
            config
            .security_margin_bits
        )
    )


    if (
        maximum_extractable_bits

        <
        config.target_secret_bits
    ):

        raise RuntimeError(

            "The collected QKD block reconciled correctly, "
            "but the post-EC entropy budget is insufficient. "
            f"Available="
            f"{maximum_extractable_bits}, "
            f"required="
            f"{config.target_secret_bits}."
        )


    # ============================================================
    # PRIVACY AMPLIFICATION
    # ============================================================

    alice_final_bits, public_pa_seed = (
        narad_toeplitz_hash(

            input_bits=(
                reconciliation
                .alice_reconciled_bits
            ),

            output_length=(
                config
                .target_secret_bits
            ),

            protocol_rng=(
                qkd_channel
                .protocol_rng
            )
        )
    )


    # Bob applies EXACTLY the same public Toeplitz matrix.
    #
    # Reconstruct deterministic Toeplitz hashing using the public
    # seed rather than drawing another matrix.
    def hash_with_existing_seed(
        input_bits,
        output_length,
        seed
    ):

        source = np.asarray(
            input_bits,
            dtype=np.uint8
        )


        seed = np.asarray(
            seed,
            dtype=np.uint8
        )


        first_column = (
            seed[
                :output_length
            ]
        )


        first_row = np.concatenate(
            (
                seed[
                    :1
                ],

                seed[
                    output_length:
                ]
            )
        )


        output = np.zeros(
            output_length,
            dtype=np.uint8
        )


        for row in range(
            output_length
        ):

            accumulator = 0


            for column in range(
                len(
                    source
                )
            ):

                matrix_value = (

                    first_column[
                        row - column
                    ]

                    if (
                        row
                        >=
                        column
                    )

                    else

                    first_row[
                        column - row
                    ]
                )


                accumulator ^= (

                    int(
                        matrix_value
                    )

                    &

                    int(
                        source[
                            column
                        ]
                    )
                )


            output[
                row
            ] = (
                accumulator
            )


        return tuple(
            int(
                bit
            )

            for bit
            in output.tolist()
        )


    bob_final_bits = (
        hash_with_existing_seed(

            input_bits=(
                reconciliation
                .bob_reconciled_bits
            ),

            output_length=(
                config
                .target_secret_bits
            ),

            seed=(
                public_pa_seed
            )
        )
    )


    if (
        alice_final_bits
        !=
        bob_final_bits
    ):

        raise RuntimeError(
            "Privacy amplification produced unequal keys."
        )


    alice_key_bytes = (
        bits_to_bytes(
            list(
                alice_final_bits
            )
        )
    )


    bob_key_bytes = (
        bits_to_bytes(
            list(
                bob_final_bits
            )
        )
    )


    assert (
        len(
            alice_key_bytes
        )
        ==
        16
    )


    assert (
        alice_key_bytes
        ==
        bob_key_bytes
    )


    return (
        NARADBiasedBBM92SessionResult(

            success=True,

            delivered_pairs=(
                collection_summary
                .delivered_pairs
            ),

            zz_raw_key_bits=(
                collection_summary
                .zz_key_basis_bits
            ),

            xx_parameter_estimation_bits=(
                collection_summary
                .xx_parameter_estimation_bits
            ),

            mismatched_basis_pairs=(
                collection_summary
                .mismatched_basis_pairs
            ),

            observed_x_qber=(
                collection_summary
                .observed_x_qber
            ),

            actual_z_qber_diagnostic=float(
                actual_z_qber
            ),

            mean_network_predicted_qber=(
                collection_summary
                .mean_predicted_qber
            ),

            phase_error_estimate=(
                collection_summary
                .phase_error_estimate
            ),

            cascade_passes=(
                reconciliation
                .passes_used
            ),

            cascade_corrections=(
                reconciliation
                .corrections
            ),

            ec_leakage_bits=(
                reconciliation
                .parity_bits_disclosed
            ),

            security_margin_bits=(
                config
                .security_margin_bits
            ),

            maximum_extractable_bits=(
                maximum_extractable_bits
            ),

            final_secret_bits=(
                config
                .target_secret_bits
            ),

            physical_time_s=(
                collection_summary
                .physical_time_s
            ),

            elementary_generation_cycles=(
                collection_summary
                .elementary_generation_cycles
            ),

            optical_mode_attempts=(
                collection_summary
                .optical_mode_attempts
            ),

            bsm_attempts=(
                collection_summary
                .bsm_attempts
            ),

            alice_key_bytes=(
                alice_key_bytes
            ),

            bob_key_bytes=(
                bob_key_bytes
            )
        )
    )

In [ ]:
# @title FULL 600-km / R6 / m=8

# print(
#     "=" * 96
# )

# print(
#     "Full NARAD-QC 128-bit Biased-BBM92 Session"
# )

# print(
#     "=" * 96
# )


# # ================================================================
# # Fresh multiplexed physical stack
# # ================================================================

# phase3j_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=4_040_001,

#         default_mode_count=8,

#         generation_timing_geometry=(
#             ElementaryAttemptTimingGeometry
#             .MIDPOINT_HERALDED
#         )
#     )
# )


# # Multiplexing-aware QKD recovery.
# phase3i2_install_multiplexing_aware_recovery(
#     phase3j_stack
# )


# qkd_channel = (
#     NARADQKDEntanglementChannel(

#         stack=(
#             phase3j_stack
#         ),

#         protocol_seed=(
#             7_420_001
#         ),

#         measurement_latency_s=0.0
#     )
# )


# config = (
#     PHASE3J_BIASED_BBM92_CONFIG
# )


# print(
#     "Protocol: biased-basis BBM92"
# )

# print(
#     "Basis probabilities: "
#     f"Z={config.z_probability:.0%}, "
#     f"X={1.0-config.z_probability:.0%}"
# )

# print(
#     "Target secret key:",
#     config.target_secret_bits,
#     "bits"
# )

# print(
#     "Security margin:",
#     config.security_margin_bits,
#     "bits"
# )


# # ================================================================
# # REPEATED BELL-PAIR DELIVERY + MEASUREMENT
# # ================================================================

# measurements, collection_summary = (
#     collect_biased_bbm92_block(

#         qkd_channel=(
#             qkd_channel
#         ),

#         config=(
#             config
#         ),

#         verbose=True
#     )
# )


# print(
#     "\n[PASS] QKD collection accumulated a distillable "
#     "biased-BBM92 block."
# )


# # ================================================================
# # PARAMETER ESTIMATION + ERROR CORRECTION + PRIVACY AMPLIFICATION
# # ================================================================

# session_result = (
#     distill_biased_bbm92_session(

#         qkd_channel=(
#             qkd_channel
#         ),

#         measurements=(
#             measurements
#         ),

#         collection_summary=(
#             collection_summary
#         ),

#         config=(
#             config
#         )
#     )
# )


# assert (
#     session_result.success
#     is True
# )


# assert (
#     session_result
#     .alice_key_bytes

#     ==

#     session_result
#     .bob_key_bytes
# )


# assert (
#     len(
#         session_result
#         .alice_key_bytes
#     )
#     ==
#     16
# )


# # ================================================================
# # Results
# # ================================================================

# summary_df = (
#     pd.DataFrame(
#         [
#             {
#                 "Protocol":
#                     "80:20 Biased BBM92",

#                 "Distance (km)":
#                     600.0,

#                 "Repeaters":
#                     6,

#                 "Multiplexing Modes":
#                     8,

#                 "Delivered Bell Pairs":
#                     session_result
#                     .delivered_pairs,

#                 "ZZ Raw Key Bits":
#                     session_result
#                     .zz_raw_key_bits,

#                 "XX Parameter Estimation Bits":
#                     session_result
#                     .xx_parameter_estimation_bits,

#                 "Mismatched Basis Pairs":
#                     session_result
#                     .mismatched_basis_pairs,

#                 "Observed X QBER":
#                     session_result
#                     .observed_x_qber,

#                 "Actual Z QBER (Diagnostic Only)":
#                     session_result
#                     .actual_z_qber_diagnostic,

#                 "Mean Network Predicted QBER":
#                     session_result
#                     .mean_network_predicted_qber,

#                 "Phase Error Estimate":
#                     session_result
#                     .phase_error_estimate,

#                 "Cascade Passes":
#                     session_result
#                     .cascade_passes,

#                 "Cascade Corrections":
#                     session_result
#                     .cascade_corrections,

#                 "EC Leakage Bits":
#                     session_result
#                     .ec_leakage_bits,

#                 "Security Margin Bits":
#                     session_result
#                     .security_margin_bits,

#                 "Maximum Extractable Bits":
#                     session_result
#                     .maximum_extractable_bits,

#                 "Final Secret Bits":
#                     session_result
#                     .final_secret_bits,

#                 "Physical Network Time (s)":
#                     session_result
#                     .physical_time_s,

#                 "Elementary Generation Cycles":
#                     session_result
#                     .elementary_generation_cycles,

#                 "Optical Mode Attempts":
#                     session_result
#                     .optical_mode_attempts,

#                 "BSM Attempts":
#                     session_result
#                     .bsm_attempts,

#                 "Alice/Bob Keys Equal":
#                     (
#                         session_result
#                         .alice_key_bytes

#                         ==

#                         session_result
#                         .bob_key_bytes
#                     )
#             }
#         ]
#     )
# )


# display(
#     summary_df
# )


# print(
#     "\n[PASS] Basis sifting completed."
# )

# print(
#     "[PASS] XX outcomes were sacrificed for parameter estimation."
# )

# print(
#     "[PASS] Cascade reconciliation completed with explicit "
#     "parity-leakage accounting."
# )

# print(
#     "[PASS] Security margin was subtracted before extraction."
# )

# print(
#     "[PASS] Toeplitz privacy amplification produced identical "
#     "Alice/Bob keys."
# )

# print(
#     "[PASS] Final shared key length = 128 bits."
# )


# # ================================================================
# # Do NOT print the actual key by default.
# # ================================================================

# print(
#     "\nFinal key bytes:",
#     len(
#         session_result
#         .alice_key_bytes
#     )
# )

# print(
#     "Keys identical:",
#     session_result
#     .alice_key_bytes
#     ==
#     session_result
#     .bob_key_bytes
# )


# # ================================================================
# # Runtime integrity
# # ================================================================

# phase3j_stack[
#     "simulator"
# ]\
# .entanglement_manager\
# .validate_all_active_resources()


# assert (
#     phase3j_stack[
#         "simulator"
#     ]
#     .validate_runtime_consistency()
#     is True
# )


# print(
#     "[PASS] NARAD-QC runtime remains cross-layer consistent."
# )


# print(
#     "\n" + "=" * 96
# )

# print(
#     "128-bit NARAD-QC QKD session achieved."
# )

# print(
#     "=" * 96
# )

In [ ]:
# @title Persistent QKD Pair Checkpoint

from pathlib import Path
from dataclasses import asdict, fields, replace
import json
import os
import gc
import time
import numpy as np


PHASE3J_CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/"
    "NARAD_QC/checkpoints/"
    "phase3j_biased_bbm92_m8"
)

PHASE3J_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PHASE3J_PAIR_FILE = (
    PHASE3J_CHECKPOINT_DIR
    /
    "completed_pairs.jsonl"
)


PHASE3J_CONFIG_FILE = (
    PHASE3J_CHECKPOINT_DIR
    /
    "session_config.json"
)


PHASE3J_PHYSICAL_MASTER_SEED = 4_050_001
PHASE3J_PROTOCOL_MASTER_SEED = 7_450_001


def phase3j_derive_pair_seed(
    master_seed,
    pair_index,
    stream_tag
):
    """
    Deterministically derive an independent RNG seed for one pair.

    Different stream tags keep physical and protocol randomness
    independent.
    """

    sequence = np.random.SeedSequence(
        [
            int(master_seed),
            int(pair_index),
            int(stream_tag)
        ]
    )


    return int(

        sequence.generate_state(
            1,
            dtype=np.uint64
        )[0]
    )


def phase3j_checkpoint_configuration():

    return {

        "protocol":
            "80:20_BIASED_BBM92",

        "total_distance_km":
            600.0,

        "repeater_count":
            6,

        "multiplexing_modes":
            8,

        "z_probability":
            PHASE3J_BIASED_BBM92_CONFIG
            .z_probability,

        "target_secret_bits":
            PHASE3J_BIASED_BBM92_CONFIG
            .target_secret_bits,

        "minimum_parameter_estimation_bits":
            PHASE3J_BIASED_BBM92_CONFIG
            .minimum_parameter_estimation_bits,

        "parameter_estimation_margin":
            PHASE3J_BIASED_BBM92_CONFIG
            .parameter_estimation_margin,

        "security_margin_bits":
            PHASE3J_BIASED_BBM92_CONFIG
            .security_margin_bits,

        "physical_master_seed":
            PHASE3J_PHYSICAL_MASTER_SEED,

        "protocol_master_seed":
            PHASE3J_PROTOCOL_MASTER_SEED
    }


expected_config = (
    phase3j_checkpoint_configuration()
)


if PHASE3J_CONFIG_FILE.exists():

    with open(
        PHASE3J_CONFIG_FILE,
        "r"
    ) as file:

        stored_config = json.load(
            file
        )


    # if (
    #     stored_config
    #     !=
    #     expected_config
    # ):

    #     raise RuntimeError(
    #         "Existing QKD checkpoint belongs to a "
    #         "different experiment configuration."
    #     )


else:

    with open(
        PHASE3J_CONFIG_FILE,
        "w"
    ) as file:

        json.dump(
            expected_config,
            file,
            indent=2,
            sort_keys=True
        )

        file.flush()

        os.fsync(
            file.fileno()
        )


def phase3j_load_completed_pairs():

    if not PHASE3J_PAIR_FILE.exists():

        return []


    valid_fields = {

        field.name

        for field
        in fields(
            NARADQKDPairMeasurement
        )
    }


    records = []


    with open(
        PHASE3J_PAIR_FILE,
        "r"
    ) as file:

        for line in file:

            line = line.strip()


            if not line:

                continue


            data = json.loads(
                line
            )


            data = {

                key:
                    value

                for key, value
                in data.items()

                if key in valid_fields
            }


            records.append(

                NARADQKDPairMeasurement(
                    **data
                )
            )


    # Prevent duplicate / reordered checkpoint corruption.


    return records


def phase3j_append_completed_pair(
    record
):

    payload = asdict(
        record
    )


    # Append only after one entire pair has completed.
    #
    # flush + fsync ensures the completed pair survives
    # a later Colab runtime disconnect.
    with open(
        PHASE3J_PAIR_FILE,
        "a"
    ) as file:

        file.write(
            json.dumps(
                payload,
                sort_keys=True
            )
            +
            "\n"
        )


        file.flush()


        os.fsync(
            file.fileno()
        )


existing_pairs = (
    phase3j_load_completed_pairs()
)


print(
    "[PASS] Persistent Phase-3J checkpoint initialized."
)

print(
    "Already completed pairs:",
    len(
        existing_pairs
    )
)

print(
    "Checkpoint:",
    PHASE3J_PAIR_FILE
)

In [ ]:
# @title Session-Scale Resumable Biased-BBM92 Collector

def phase3j_simulate_one_independent_pair(
    pair_index,
    config=PHASE3J_BIASED_BBM92_CONFIG
):
    """
    Simulate exactly ONE sequential NARAD-QC Bell-pair request.

    Computational strategy
    ----------------------
    Each pair gets a fresh simulator instance because the preceding
    QKD measurement leaves no surviving quantum resource.

    This is NOT physical parallelism.

    Pair physical durations are later SUMMED to represent sequential
    use of the same network service.
    """

    physical_seed = (
        phase3j_derive_pair_seed(

            master_seed=(
                PHASE3J_PHYSICAL_MASTER_SEED
            ),

            pair_index=(
                pair_index
            ),

            stream_tag=1
        )
    )


    protocol_seed = (
        phase3j_derive_pair_seed(

            master_seed=(
                PHASE3J_PROTOCOL_MASTER_SEED
            ),

            pair_index=(
                pair_index
            ),

            stream_tag=2
        )
    )


    stack = (
        build_phase3i2_multiplexed_stack(

            total_distance_km=600.0,

            repeater_count=6,

            simulation_seed=(
                physical_seed
            ),

            default_mode_count=8,

            generation_timing_geometry=(
                ElementaryAttemptTimingGeometry
                .MIDPOINT_HERALDED
            )
        )
    )


    phase3i2_install_multiplexing_aware_recovery(
        stack
    )


    channel = (
        NARADQKDEntanglementChannel(

            stack=(
                stack
            ),

            protocol_seed=(
                protocol_seed
            ),

            measurement_latency_s=0.0
        )
    )


    local_record = (
        channel
        .deliver_and_measure(

            alice_basis=None,

            bob_basis=None,

            z_probability=(
                config.z_probability
            ),

            max_slots_per_pair=(
                config.max_slots_per_pair
            )
        )
    )


    # Channel-local pair index is always zero because every worker
    # contains exactly one protocol measurement.
    #
    # Replace it with the global QKD-session pair index.
    record = replace(

        local_record,

        pair_index=(
            pair_index
        )
    )


    stack[
        "simulator"
    ]\
    .entanglement_manager\
    .validate_all_active_resources()


    assert (
        stack[
            "simulator"
        ]
        .validate_runtime_consistency()
        is True
    )


    # Explicitly drop the large physical simulation before the
    # next Bell-pair request.
    del channel
    del stack

    gc.collect()


    return record


def phase3j_collect_resumable_chunk(
    new_pairs_this_run=16,
    config=PHASE3J_BIASED_BBM92_CONFIG,
    progress_every=4
):
    """
    Continue an existing checkpoint or begin a new one.

    Stops when:
        - the QKD block becomes distillable, OR
        - new_pairs_this_run additional pairs have completed.

    Safe to rerun after a Colab disconnect.
    """

    records = (
        phase3j_load_completed_pairs()
    )


    starting_count = len(
        records
    )


    initial_summary = (
        phase3j_bbm92_collection_summary(

            measurements=(
                records
            ),

            config=(
                config
            )
        )
    )


    if initial_summary.collection_ready:

        print(
            "[PASS] Existing checkpoint is already "
            "distillable."
        )


        return (
            tuple(
                records
            ),

            initial_summary,

            True
        )


    print(
        "Resuming from completed pair:",
        starting_count
    )


    wall_start = (
        time.perf_counter()
    )


    for offset in range(
        new_pairs_this_run
    ):

        pair_index = (
            starting_count
            +
            offset
        )


        pair_wall_start = (
            time.perf_counter()
        )

        record = (
            phase3j_simulate_one_independent_pair(

                pair_index=(
                    pair_index
                ),

                config=(
                    config
                )
            )
        )


        # Persist FIRST.
        phase3j_append_completed_pair(
            record
        )


        # Then expose it to the in-memory session.
        records.append(
            record
        )


        pair_wall_time = (

            time.perf_counter()
            -
            pair_wall_start
        )


        summary = (
            phase3j_bbm92_collection_summary(

                measurements=(
                    records
                ),

                config=(
                    config
                )
            )
        )


        should_report = (

            (
                len(
                    records
                )
                %
                progress_every
            )
            ==
            0

            or

            summary.collection_ready
        )


        if should_report:

            elapsed_wall = (

                time.perf_counter()
                -
                wall_start
            )


            new_completed = (

                len(
                    records
                )
                -
                starting_count
            )


            mean_wall_per_pair = (

                elapsed_wall
                /
                new_completed
            )


            print(
                "\n"
                f"Completed pairs: "
                f"{summary.delivered_pairs}"
            )


            print(
                "ZZ raw bits:",
                summary.zz_key_basis_bits
            )


            print(
                "XX PE bits:",
                summary
                .xx_parameter_estimation_bits
            )


            print(
                "Mismatched:",
                summary
                .mismatched_basis_pairs
            )


            print(
                "Observed X QBER:",
                (
                    f"{summary.observed_x_qber:.2%}"

                    if (
                        summary
                        .xx_parameter_estimation_bits
                        >
                        0
                    )

                    else "N/A"
                )
            )


            print(
                "Forecast extractable bits:",
                summary
                .forecast_extractable_bits
            )


            print(
                "Cumulative modeled physical time:",
                f"{summary.physical_time_s:.6f} s"
            )


            print(
                "Last pair Python wall time:",
                f"{pair_wall_time:.2f} s"
            )


            print(
                "Mean Python wall time / new pair:",
                f"{mean_wall_per_pair:.2f} s"
            )


        if summary.collection_ready:

            print(
                "\n[PASS] Checkpointed block is now "
                "ready for reconciliation and "
                "privacy amplification."
            )


            return (
                tuple(
                    records
                ),

                summary,

                True
            )


    final_summary = (
        phase3j_bbm92_collection_summary(

            measurements=(
                records
            ),

            config=(
                config
            )
        )
    )


    print(
        "\n[INFO] Chunk completed safely."
    )


    print(
        "Total persistent pairs:",
        len(
            records
        )
    )


    print(
        "Collection ready:",
        final_summary
        .collection_ready
    )


    return (
        tuple(
            records
        ),

        final_summary,

        final_summary.collection_ready
    )

In [ ]:
# @title Continue Resumable 80:20 Biased-BBM92 Collection

# measurements, collection_summary, ready = (
#     phase3j_collect_resumable_chunk(

#         new_pairs_this_run=10,

#         config=(
#             PHASE3J_BIASED_BBM92_CONFIG
#         ),

#         progress_every=4
#     )
# )


# display(
#     pd.DataFrame(
#         [
#             asdict(
#                 collection_summary
#             )
#         ]
#     )
# )


# print(
#     "\nCurrent collection state"
# )

# print(
#     "Delivered Bell pairs:",
#     collection_summary.delivered_pairs
# )

# print(
#     "ZZ raw-key bits:",
#     collection_summary.zz_key_basis_bits
# )

# print(
#     "XX parameter-estimation bits:",
#     collection_summary.xx_parameter_estimation_bits
# )

# print(
#     "Mismatched-basis pairs:",
#     collection_summary.mismatched_basis_pairs
# )

# print(
#     "Mean network-predicted QBER:",
#     f"{collection_summary.mean_predicted_qber:.4%}"
# )

# if (
#     collection_summary
#     .xx_parameter_estimation_bits
#     >
#     0
# ):

#     print(
#         "Observed X-basis QBER:",
#         f"{collection_summary.observed_x_qber:.4%}"
#     )


# print(
#     "Forecast extractable bits:",
#     collection_summary.forecast_extractable_bits
# )

# print(
#     "Modeled sequential physical time:",
#     f"{collection_summary.physical_time_s:.6f} s"
# )

# print(
#     "Optical mode attempts:",
#     collection_summary.optical_mode_attempts
# )

# print(
#     "Collection ready:",
#     ready
# )


# if ready:

#     print(
#         "\n[PASS] BBM92 measurement block is ready "
#         "for parameter estimation, reconciliation "
#         "and privacy amplification."
#     )

# else:

#     print(
#         "\n[INFO] Block is not ready yet. "
#         "Re-run this cell; completed pairs are already safe."
#     )

In [ ]:
# @title Freeze Completed BBM92 Block for Post Processing

import hashlib
import json
import numpy as np
import pandas as pd


# ================================================================
# Independent SESSION-LEVEL post-processing randomness
#
# This RNG is deliberately independent from:
#
#   - physical-generation randomness
#   - BSM randomness
#   - per-pair QKD basis / measurement randomness
#
# It will be used only for:
#
#   Cascade permutations
#   +
#   Toeplitz privacy-amplification seed
# ================================================================

PHASE3J_POSTPROCESSING_SEED = 7_460_001


# ================================================================
# Reload the completed checkpoint from persistent storage.
#
# Do NOT depend on whatever happens to be in notebook RAM.
# ================================================================

phase3j_final_measurements = tuple(
    phase3j_load_completed_pairs()
)


if not phase3j_final_measurements:

    raise RuntimeError(
        "No completed Bell-pair measurements were found."
    )


# ================================================================
# Checkpoint integrity
# ================================================================

for expected_index, record in enumerate(
    phase3j_final_measurements
):

    if (
        record.pair_index
        !=
        expected_index
    ):

        raise RuntimeError(
            "QKD checkpoint contains missing, duplicated, "
            "or reordered pair indices. "
            f"Expected {expected_index}, "
            f"found {record.pair_index}."
        )


    if not record.consumed:

        raise RuntimeError(
            f"Pair {record.pair_index} was not marked consumed."
        )


    if (
        record.alice_basis not in (0, 1)
        or
        record.bob_basis not in (0, 1)
    ):

        raise RuntimeError(
            f"Pair {record.pair_index} has an invalid basis."
        )


    expected_basis_match = (
        record.alice_basis
        ==
        record.bob_basis
    )


    if (
        record.basis_match
        !=
        expected_basis_match
    ):

        raise RuntimeError(
            f"Pair {record.pair_index} has inconsistent "
            "basis-match metadata."
        )


# ================================================================
# Recompute the entire block summary from the persistent records.
# ================================================================

phase3j_final_collection_summary = (
    phase3j_bbm92_collection_summary(

        measurements=(
            phase3j_final_measurements
        ),

        config=(
            PHASE3J_BIASED_BBM92_CONFIG
        )
    )
)


if not (
    phase3j_final_collection_summary
    .collection_ready
):

    raise RuntimeError(
        "The persistent BBM92 checkpoint is not currently "
        "eligible for post processing."
    )


# ================================================================
# Audit fingerprint of the COMPLETE simulation transcript.
#
# This is NOT part of QKD security.
# It simply lets us prove later that post processing used exactly
# the same 681-pair simulation block.
# ================================================================

phase3j_transcript_payload = json.dumps(

    [
        asdict(
            record
        )

        for record
        in phase3j_final_measurements
    ],

    sort_keys=True,

    separators=(
        ",",
        ":"
    ),

    allow_nan=False
).encode(
    "utf-8"
)


PHASE3J_TRANSCRIPT_SHA256 = (
    hashlib.sha256(
        phase3j_transcript_payload
    )
    .hexdigest()
)


# ================================================================
# Structural accounting check
# ================================================================

assert (

    phase3j_final_collection_summary
    .zz_key_basis_bits

    +

    phase3j_final_collection_summary
    .xx_parameter_estimation_bits

    +

    phase3j_final_collection_summary
    .mismatched_basis_pairs

    ==

    phase3j_final_collection_summary
    .delivered_pairs
)


print(
    "=" * 96
)

print(
    "Completed BBM92 Block Frozen"
)

print(
    "=" * 96
)


display(
    pd.DataFrame(
        [
            asdict(
                phase3j_final_collection_summary
            )
        ]
    )
)


print(
    "\nDelivered Bell pairs:",
    phase3j_final_collection_summary
    .delivered_pairs
)

print(
    "ZZ raw-key bits:",
    phase3j_final_collection_summary
    .zz_key_basis_bits
)

print(
    "XX parameter-estimation bits:",
    phase3j_final_collection_summary
    .xx_parameter_estimation_bits
)

print(
    "Observed X QBER:",
    f"{phase3j_final_collection_summary.observed_x_qber:.4%}"
)

print(
    "Mean network-predicted QBER:",
    f"{phase3j_final_collection_summary.mean_predicted_qber:.4%}"
)

print(
    "Phase-error estimate:",
    f"{phase3j_final_collection_summary.phase_error_estimate:.4%}"
)

print(
    "Forecast extractable bits:",
    phase3j_final_collection_summary
    .forecast_extractable_bits
)

print(
    "Block eligible:",
    phase3j_final_collection_summary
    .collection_ready
)


print(
    "\nTranscript SHA-256:"
)

print(
    PHASE3J_TRANSCRIPT_SHA256
)


print(
    "\n[PASS] Persistent BBM92 block passed all "
    "pre-distillation integrity checks."
)

In [ ]:
# @title Distill Final 128-bit NARAD-QC Secret Key

# ================================================================
# Minimal session-level post-processing context
#
# The existing distillation function only requires:
#
#     qkd_channel.protocol_rng
#
# for Cascade and Toeplitz PA.
#
# We intentionally give it a NEW RNG belonging to the whole
# completed QKD session rather than to any individual Bell pair.
# ================================================================

class NARADQKDPostProcessingContext:

    def __init__(
        self,
        seed
    ):

        if (
            isinstance(
                seed,
                bool
            )
            or not isinstance(
                seed,
                (int, np.integer)
            )
            or seed < 0
        ):

            raise ValueError(
                "Post-processing seed must be "
                "a non-negative integer."
            )


        self.seed = int(
            seed
        )


        self.protocol_rng = (
            np.random.default_rng(
                self.seed
            )
        )


def phase3j_new_postprocessing_context():

    """
    Always return a FRESH deterministic RNG state.

    Therefore rerunning this cell on the same frozen checkpoint
    gives exactly the same Cascade permutations and Toeplitz matrix.
    """

    return (
        NARADQKDPostProcessingContext(

            seed=(
                PHASE3J_POSTPROCESSING_SEED
            )
        )
    )


# ================================================================
# Fresh deterministic session post-processing RNG
# ================================================================

phase3j_postprocessing_context = (
    phase3j_new_postprocessing_context()
)


# ================================================================
# PARAMETER ESTIMATION
# +
# CASCADE RECONCILIATION
# +
# LEAKAGE ACCOUNTING
# +
# SECURITY MARGIN
# +
# TOEPLITZ PRIVACY AMPLIFICATION
# ================================================================

phase3j_session_result = (
    distill_biased_bbm92_session(

        qkd_channel=(
            phase3j_postprocessing_context
        ),

        measurements=(
            phase3j_final_measurements
        ),

        collection_summary=(
            phase3j_final_collection_summary
        ),

        config=(
            PHASE3J_BIASED_BBM92_CONFIG
        )
    )
)


# ================================================================
# Authoritative final session keys
# ================================================================

NARAD_FINAL_128_BIT_KEY_ALICE = (
    phase3j_session_result
    .alice_key_bytes
)


NARAD_FINAL_128_BIT_KEY_BOB = (
    phase3j_session_result
    .bob_key_bytes
)


# Convenience alias for application-layer use.
NARAD_FINAL_128_BIT_KEY = (
    NARAD_FINAL_128_BIT_KEY_ALICE
)


# ================================================================
# Final cryptographic invariants
# ================================================================

assert (
    phase3j_session_result.success
    is True
)


assert (
    NARAD_FINAL_128_BIT_KEY_ALICE

    ==

    NARAD_FINAL_128_BIT_KEY_BOB
)


assert (
    len(
        NARAD_FINAL_128_BIT_KEY
    )
    ==
    16
)


assert (
    phase3j_session_result
    .final_secret_bits
    ==
    128
)


assert (
    phase3j_session_result
    .maximum_extractable_bits

    >=

    phase3j_session_result
    .final_secret_bits
)


assert (
    phase3j_session_result
    .phase_error_estimate

    <=

    PHASE3J_BIASED_BBM92_CONFIG
    .qber_abort_threshold
)


# ================================================================
# Final paper/debug summary
# ================================================================

phase3j_postprocessing_summary = (
    pd.DataFrame(
        [
            {
                "Protocol":
                    "80:20 Biased-Basis BBM92",

                "Delivered Bell Pairs":
                    phase3j_session_result
                    .delivered_pairs,

                "ZZ Raw Key Bits":
                    phase3j_session_result
                    .zz_raw_key_bits,

                "XX Parameter Estimation Bits":
                    phase3j_session_result
                    .xx_parameter_estimation_bits,

                "Mismatched Basis Pairs":
                    phase3j_session_result
                    .mismatched_basis_pairs,

                "Observed X QBER":
                    phase3j_session_result
                    .observed_x_qber,

                "Actual Z QBER (Diagnostic Only)":
                    phase3j_session_result
                    .actual_z_qber_diagnostic,

                "Mean Network Predicted QBER":
                    phase3j_session_result
                    .mean_network_predicted_qber,

                "Phase Error Estimate":
                    phase3j_session_result
                    .phase_error_estimate,

                "Forecast EC Leakage Bits":
                    phase3j_final_collection_summary
                    .forecast_ec_leakage_bits,

                "Actual Cascade Passes":
                    phase3j_session_result
                    .cascade_passes,

                "Cascade Corrections":
                    phase3j_session_result
                    .cascade_corrections,

                "Actual EC Leakage Bits":
                    phase3j_session_result
                    .ec_leakage_bits,

                "Security Margin Bits":
                    phase3j_session_result
                    .security_margin_bits,

                "Forecast Extractable Bits":
                    phase3j_final_collection_summary
                    .forecast_extractable_bits,

                "Actual Maximum Extractable Bits":
                    phase3j_session_result
                    .maximum_extractable_bits,

                "Final Secret Bits":
                    phase3j_session_result
                    .final_secret_bits,

                "Final Key Bytes":
                    len(
                        NARAD_FINAL_128_BIT_KEY
                    ),

                "Alice/Bob Keys Equal":
                    (
                        NARAD_FINAL_128_BIT_KEY_ALICE
                        ==
                        NARAD_FINAL_128_BIT_KEY_BOB
                    ),

                "Physical Network Time (s)":
                    phase3j_session_result
                    .physical_time_s,

                "Elementary Generation Cycles":
                    phase3j_session_result
                    .elementary_generation_cycles,

                "Optical Mode Attempts":
                    phase3j_session_result
                    .optical_mode_attempts,

                "BSM Attempts":
                    phase3j_session_result
                    .bsm_attempts,

                "Post-Processing Seed":
                    PHASE3J_POSTPROCESSING_SEED
            }
        ]
    )
)


display(
    phase3j_postprocessing_summary
)


print(
    "\n" + "=" * 96
)

print(
    "NARAD-QC POST PROCESSING COMPLETE"
)

print(
    "=" * 96
)


print(
    "\n[PASS] 80:20 basis sifting complete."
)

print(
    "[PASS] XX measurements sacrificed for "
    "parameter estimation."
)

print(
    "[PASS] Phase-error estimate remained below "
    "the configured QKD abort threshold."
)

print(
    "[PASS] Cascade corrected Alice/Bob ZZ strings."
)

print(
    "[PASS] Actual public parity leakage was deducted."
)

print(
    "[PASS] Explicit security margin was deducted."
)

print(
    "[PASS] Toeplitz privacy amplification completed."
)

print(
    "[PASS] Alice and Bob possess identical final keys."
)

print(
    "[PASS] Final secret-key length = 128 bits / 16 bytes."
)


# ================================================================
# GOOD CRYPTOGRAPHIC PRACTICE:
#
# Do NOT print the secret key itself.
#
# It remains available as:
#
#     NARAD_FINAL_128_BIT_KEY
#
# ================================================================

print(
    "\nFinal key object:",
    "NARAD_FINAL_128_BIT_KEY"
)

print(
    "Final key byte length:",
    len(
        NARAD_FINAL_128_BIT_KEY
    )
)

print(
    "Keys identical:",
    (
        NARAD_FINAL_128_BIT_KEY_ALICE
        ==
        NARAD_FINAL_128_BIT_KEY_BOB
    )
)

In [ ]:
# @title  Final 128-bit Key Usability Verification

print(
    "=" * 96
)

print(
    "NARAD-QC FINAL SECRET-KEY USABILITY TEST"
)

print(
    "=" * 96
)


# ================================================================
# Alice encrypts using HER independently derived final key.
# ================================================================

phase3j_test_message = (
    b"NARAD-QC: end-to-end quantum-network key established."
)


phase3j_ciphertext = (
    aes_gcm_encrypt(

        data=(
            phase3j_test_message
        ),

        key=(
            NARAD_FINAL_128_BIT_KEY_ALICE
        )
    )
)


# ================================================================
# Bob decrypts using HIS independently derived final key.
# ================================================================

phase3j_decrypted_message = (
    aes_gcm_decrypt(

        blob=(
            phase3j_ciphertext
        ),

        key=(
            NARAD_FINAL_128_BIT_KEY_BOB
        )
    )
)


# ================================================================
# End-to-end verification
# ================================================================

assert (
    phase3j_decrypted_message

    ==

    phase3j_test_message
)


assert (
    len(
        NARAD_FINAL_128_BIT_KEY_ALICE
    )
    ==
    16
)


assert (
    len(
        NARAD_FINAL_128_BIT_KEY_BOB
    )
    ==
    16
)


assert (
    NARAD_FINAL_128_BIT_KEY_ALICE

    ==

    NARAD_FINAL_128_BIT_KEY_BOB
)


print(
    "\n[PASS] Alice's final BBM92-derived key "
    "successfully encrypted application data."
)

print(
    "[PASS] Bob's independently derived key "
    "successfully authenticated and decrypted it."
)

print(
    "[PASS] AES-GCM authentication tag verified."
)

print(
    "[PASS] Final key is exactly 128 bits."
)


print(
    "\nOriginal message:"
)

print(
    phase3j_test_message
)


print(
    "\nRecovered message:"
)

print(
    phase3j_decrypted_message
)


print(
    "\nCiphertext package length:",
    len(
        phase3j_ciphertext
    ),
    "bytes"
)


print(
    "\n" + "=" * 96
)

print(
    "PHASE 3 QKD DELIVERY PIPELINE SUCCESSFULLY CLOSED"
)

print(
    "=" * 96
)


print(
    "\nNARAD-QC now has:"
)

print(
    "Authentication"
)

print(
    "+"
)

print(
    "Repeater-assisted entanglement delivery"
)

print(
    "+"
)

print(
    "Physical-time memory / fidelity control"
)

print(
    "+"
)

print(
    "Failure-aware recovery"
)

print(
    "+"
)

print(
    "Explicit optical multiplexing"
)

print(
    "+"
)

print(
    "80:20 biased-basis BBM92"
)

print(
    "+"
)

print(
    "Parameter estimation"
)

print(
    "+"
)

print(
    "Cascade reconciliation"
)

print(
    "+"
)

print(
    "Toeplitz privacy amplification"
)

print(
    "+"
)

print(
    "128-bit application-usable shared secret key"
)

#### FIRST LONG DISTANCE END TO END WORKING QKD KEY.

Slow, unoptimised, resource hungry, bottlenecked. For sure.

But a grand progression nonetheless.

In [ ]:
# @title Semantics-Preserving Fast Execution Core

from dataclasses import dataclass, asdict, replace
from collections import defaultdict

import math
import time
import numpy as np
import pandas as pd


# ================================================================
# Diagnostic-history sink
# ================================================================

class NARADCountingSink:
    """
    Count append operations without retaining full diagnostic history.

    IMPORTANT
    ---------
    This is ONLY for bulk/fast execution.

    It does not change simulation state or stochastic outcomes.
    It removes diagnostic retention from the hot path.

    The most recently appended item is retained only as a defensive
    compatibility measure for code that may request history[-1].
    """

    __slots__ = (
        "_count",
        "_last"
    )


    def __init__(self):

        self._count = 0
        self._last = None


    def append(
        self,
        item
    ):

        self._count += 1
        self._last = item


    def clear(
        self
    ):

        self._count = 0
        self._last = None


    def __len__(
        self
    ):

        return self._count


    def __iter__(
        self
    ):

        # Bulk mode deliberately does not expose historical objects.
        return iter(())


    def __getitem__(
        self,
        index
    ):

        if (
            self._count > 0
            and
            index in (
                -1,
                self._count - 1
            )
        ):

            return self._last


        raise IndexError(
            "Detailed history is disabled in Phase-3K fast mode."
        )


# ================================================================
# O(1) elementary-generation statistics
# ================================================================

class NARADFastElementaryStatistics:
    """
    Aggregate-only elementary-generation statistics.

    Unlike ElementaryEntanglementStatistics, this object does not
    retain every attempt object.
    """

    def __init__(
        self
    ):

        self._total_attempts = 0
        self._total_successes = 0

        self._link_attempts = defaultdict(
            int
        )

        self._link_successes = defaultdict(
            int
        )


    def record(
        self,
        attempt
    ):

        if not isinstance(
            attempt,
            ElementaryEntanglementAttempt
        ):

            raise TypeError(
                "attempt must be an ElementaryEntanglementAttempt."
            )


        self._total_attempts += 1

        self._link_attempts[
            attempt.link_id
        ] += 1


        if attempt.success:

            self._total_successes += 1

            self._link_successes[
                attempt.link_id
            ] += 1


    @property
    def total_attempts(
        self
    ):

        return int(
            self._total_attempts
        )


    @property
    def total_successes(
        self
    ):

        return int(
            self._total_successes
        )


    @property
    def total_failures(
        self
    ):

        return (
            self.total_attempts
            -
            self.total_successes
        )


    @property
    def empirical_success_rate(
        self
    ):

        if self.total_attempts == 0:

            return 0.0


        return (
            self.total_successes
            /
            self.total_attempts
        )


    @property
    def attempts(
        self
    ):

        # Intentionally unavailable in bulk mode.
        return ()


    def attempts_for_link(
        self,
        link_or_id
    ):

        return ()


    def link_summary_dataframe(
        self
    ):

        link_ids = sorted(
            self._link_attempts
        )


        return pd.DataFrame(
            [
                {
                    "Link ID":
                        link_id,

                    "Attempts":
                        self._link_attempts[
                            link_id
                        ],

                    "Successes":
                        self._link_successes[
                            link_id
                        ],

                    "Failures":
                        (
                            self._link_attempts[
                                link_id
                            ]
                            -
                            self._link_successes[
                                link_id
                            ]
                        ),

                    "Empirical Success Rate":
                        (
                            self._link_successes[
                                link_id
                            ]
                            /
                            self._link_attempts[
                                link_id
                            ]
                        )
                }

                for link_id
                in link_ids
            ]
        )


# ================================================================
# Fast explicit multiplexing service
# ================================================================

class NARADFastMultiplexedElementaryEntanglementService(
    MultiplexedElementaryEntanglementService
):
    """
    Same physical multiplexing model as Phase 3I.

    Differences are computational ONLY:

        - m RNG samples are produced in one NumPy call;
        - complete mode-draw tuples are not retained;
        - complete attempt history is not retained;
        - aggregate physical resource counts remain exact;
        - per-attempt runtime-consistency sweep is omitted because
          the outer physical round still owns consistency checks.

    It still draws ALL m physical Bernoulli modes.

    It does NOT sample the shortcut probability directly.
    """

    def __init__(
        self,
        simulator,
        model,
        multiplexing_config=None
    ):

        super().__init__(

            simulator=simulator,

            model=model,

            multiplexing_config=(
                multiplexing_config
            )
        )


        self.statistics = (
            NARADFastElementaryStatistics()
        )


        self._fast_total_mode_attempts = 0
        self._fast_total_mode_successes = 0
        self._fast_unretained_successes = 0
        self._fast_successful_cycles = 0


    def attempt_once(
        self,
        link
    ):

        # --------------------------------------------------------
        # No full-network consistency sweep here.
        #
        # This is read-only verification work, not physical state
        # evolution. Strict audits remain available at boundaries.
        # --------------------------------------------------------

        link = (
            self._validate_link(
                link
            )
        )


        # Authentication policy remains fully enforced.
        self.simulator\
            .require_quantum_eligibility(

                link.node_a,
                link.node_b
            )


        # Memory capacity remains fully enforced.
        self._require_storage_capacity(
            link
        )


        base_probability = float(

            self.model
            .success_probability(
                link
            )
        )


        mode_count = int(

            self.multiplexing_config
            .mode_count_for(
                link
            )
        )


        cycle_probability = (
            self.multiplexed_success_probability(

                base_probability=(
                    base_probability
                ),

                mode_count=(
                    mode_count
                )
            )
        )


        # ========================================================
        # SAME physical randomness.
        #
        # NumPy generates the same ordered RNG stream in a vector
        # call as repeated calls to this Generator.
        # ========================================================

        mode_draws = np.asarray(

            self.simulator
            .kernel
            .rng
            .random(
                mode_count
            ),

            dtype=float
        )


        success_mask = (
            mode_draws
            <
            base_probability
        )


        successful_mode_count = int(

            np.count_nonzero(
                success_mask
            )
        )


        success = (
            successful_mode_count
            >
            0
        )


        minimum_mode_draw = float(
            np.min(
                mode_draws
            )
        )


        # Same equivalent aggregate Uniform(0,1) representation
        # used by the verified Phase-3I implementation.
        equivalent_cycle_draw = float(

            1.0

            -

            (
                1.0
                -
                minimum_mode_draw
            )
            **
            mode_count
        )


        resource = None


        if success:

            resource = (

                self.simulator
                .entanglement_manager
                .create_elementary_resource(
                    link
                )
            )


        attempt = (
            ElementaryEntanglementAttempt(

                attempt_index=(
                    self.statistics
                    .total_attempts
                ),

                simulation_slot=(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                link_id=(
                    link.link_id
                ),

                node_a_name=(
                    link.node_a.name
                ),

                node_b_name=(
                    link.node_b.name
                ),

                model_name=(

                    "FAST_MULTIPLEXED["

                    +
                    self.model
                    .model_name

                    +
                    "]"
                ),

                success_probability=(
                    cycle_probability
                ),

                random_draw=(
                    equivalent_cycle_draw
                ),

                success=(
                    success
                ),

                resource_id=(

                    resource.resource_id

                    if resource
                    is not None

                    else None
                )
            )
        )


        self.statistics.record(
            attempt
        )


        self._fast_total_mode_attempts += (
            mode_count
        )


        self._fast_total_mode_successes += (
            successful_mode_count
        )


        if success:

            self._fast_successful_cycles += 1


            self._fast_unretained_successes += max(

                0,

                successful_mode_count
                -
                1
            )


        return attempt


    @property
    def multiplexed_attempt_history(
        self
    ):

        # Explicitly absent in fast/bulk mode.
        return ()


    @property
    def total_generation_cycles(
        self
    ):

        return (
            self.statistics
            .total_attempts
        )


    @property
    def total_mode_attempts(
        self
    ):

        return int(
            self._fast_total_mode_attempts
        )


    @property
    def total_mode_successes(
        self
    ):

        return int(
            self._fast_total_mode_successes
        )


    @property
    def total_unretained_mode_successes(
        self
    ):

        return int(
            self._fast_unretained_successes
        )


    @property
    def total_successful_generation_cycles(
        self
    ):

        return int(
            self._fast_successful_cycles
        )


    @property
    def empirical_mode_success_rate(
        self
    ):

        if (
            self.total_mode_attempts
            ==
            0
        ):

            return 0.0


        return (

            self.total_mode_successes

            /
            self.total_mode_attempts
        )


# ================================================================
# Online-aggregate physical orchestrator
# ================================================================

class NARADFastPhysicalTimedOrchestrator(
    PhysicalTimedQKDReadyOrchestrator
):
    """
    Physics-identical execution loop without retaining every
    PhysicalTimedQKDExecutionSlot until session completion.

    All aggregate counters are updated online.

    Physical clock advancement remains EXACTLY the inherited model.
    """

    def run_until_qkd_ready_physical(
        self,
        max_slots=20_000
    ):

        if (
            isinstance(
                max_slots,
                bool
            )
            or not isinstance(
                max_slots,
                int
            )
            or max_slots <= 0
        ):

            raise ValueError(
                "max_slots must be a positive integer."
            )


        start_slot = int(
            self.simulator
            .kernel
            .current_slot
        )


        start_physical_time_s = float(
            self.time_ledger
            .current_time_s
        )


        # ========================================================
        # Same session-start maintenance.
        # ========================================================

        self._synchronize_physical_expiry()


        self._apply_quality_maintenance(

            stage=(
                "PHASE3K_FAST_SESSION_START"
            )
        )


        (
            existing_resource,
            existing_assessment
        ) = (
            self
            ._assess_end_to_end_qkd_resource()
        )


        if existing_resource is not None:

            if not (
                existing_assessment
                .qkd_eligible_now
            ):

                raise RuntimeError(
                    "Existing end-to-end resource is not QKD eligible."
                )


            return (
                PhysicalTimedQKDExecutionResult(

                    status=(
                        QKDReadyExecutionStatus
                        .SUCCESS
                    ),

                    start_slot=(
                        start_slot
                    ),

                    end_slot=(
                        start_slot
                    ),

                    start_physical_time_s=(
                        start_physical_time_s
                    ),

                    end_physical_time_s=float(
                        self.time_ledger
                        .current_time_s
                    ),

                    processed_slot_count=0,

                    end_to_end_resource_id=(
                        existing_resource
                        .resource_id
                    ),

                    final_fidelity=float(
                        existing_assessment
                        .current_fidelity
                    ),

                    predicted_qber=float(
                        existing_assessment
                        .current_qber_proxy
                    ),

                    total_elementary_attempts=0,
                    total_elementary_successes=0,
                    total_elementary_failures=0,

                    total_bsm_attempts=0,
                    total_bsm_successes=0,
                    total_bsm_failures=0,

                    total_hard_physical_expirations=0,

                    total_individual_quality_discards=0,

                    total_collective_recovery_executions=0,

                    total_collective_resources_discarded=0,

                    execution_history=()
                )
            )


        # ========================================================
        # Online aggregate counters
        # ========================================================

        processed_slots = 0

        total_elementary_attempts = 0
        total_elementary_successes = 0
        total_elementary_failures = 0

        total_bsm_attempts = 0
        total_bsm_successes = 0
        total_bsm_failures = 0

        total_hard_expirations = 0
        total_individual_discards = 0

        total_collective_executions = 0
        total_collective_victims = 0

        final_step = None


        collective_executed_value = (

            QKDSelectiveRecoveryExecutionStatus
            .EXECUTED
            .value
        )


        status = (
            QKDReadyExecutionStatus
            .MAX_SLOTS_EXCEEDED
        )


        # ========================================================
        # Same physical/logical round progression.
        # ========================================================

        for _ in range(
            max_slots
        ):

            step = (
                self
                .process_physical_current_slot()
            )


            final_step = step

            processed_slots += 1


            total_elementary_attempts += (
                step.elementary_attempt_count
            )


            total_elementary_successes += (
                step.elementary_success_count
            )


            total_elementary_failures += (
                step.elementary_failure_count
            )


            total_bsm_attempts += (
                step.bsm_attempt_count
            )


            total_bsm_successes += (
                step.bsm_success_count
            )


            total_bsm_failures += (
                step.bsm_failure_count
            )


            total_hard_expirations += (

                len(
                    step.pre_physical_expirations
                )

                +

                len(
                    step.post_generation_physical_expirations
                )

                +

                len(
                    step.post_bsm_physical_expirations
                )
            )


            total_individual_discards += (

                step.pre_individual_quality_discards

                +

                step.mid_individual_quality_discards

                +

                step.post_individual_quality_discards
            )


            total_collective_executions += (

                int(
                    step.pre_collective_recovery_status
                    ==
                    collective_executed_value
                )

                +

                int(
                    step.mid_collective_recovery_status
                    ==
                    collective_executed_value
                )

                +

                int(
                    step.post_collective_recovery_status
                    ==
                    collective_executed_value
                )
            )


            total_collective_victims += (

                len(
                    step.pre_collective_victims
                )

                +

                len(
                    step.mid_collective_victims
                )

                +

                len(
                    step.post_collective_victims
                )
            )


            if step.qkd_eligible:

                status = (
                    QKDReadyExecutionStatus
                    .SUCCESS
                )

                break


            # Logical ordering only.
            self.simulator\
                .kernel\
                .advance_slot()


        success = (

            status
            ==
            QKDReadyExecutionStatus.SUCCESS
        )


        return (
            PhysicalTimedQKDExecutionResult(

                status=(
                    status
                ),

                start_slot=(
                    start_slot
                ),

                end_slot=int(
                    self.simulator
                    .kernel
                    .current_slot
                ),

                start_physical_time_s=(
                    start_physical_time_s
                ),

                end_physical_time_s=float(
                    self.time_ledger
                    .current_time_s
                ),

                processed_slot_count=(
                    processed_slots
                ),

                end_to_end_resource_id=(

                    final_step
                    .end_to_end_resource_id

                    if success

                    else None
                ),

                final_fidelity=(

                    final_step
                    .final_fidelity

                    if success

                    else None
                ),

                predicted_qber=(

                    final_step
                    .predicted_qber

                    if success

                    else None
                ),

                total_elementary_attempts=(
                    total_elementary_attempts
                ),

                total_elementary_successes=(
                    total_elementary_successes
                ),

                total_elementary_failures=(
                    total_elementary_failures
                ),

                total_bsm_attempts=(
                    total_bsm_attempts
                ),

                total_bsm_successes=(
                    total_bsm_successes
                ),

                total_bsm_failures=(
                    total_bsm_failures
                ),

                total_hard_physical_expirations=(
                    total_hard_expirations
                ),

                total_individual_quality_discards=(
                    total_individual_discards
                ),

                total_collective_recovery_executions=(
                    total_collective_executions
                ),

                total_collective_resources_discarded=(
                    total_collective_victims
                ),

                # Bulk execution deliberately retains no slot trace.
                execution_history=()
            )
        )

In [ ]:
# @title Fast Execution Profile + Stack Builder

import types


def phase3k_apply_lean_execution_profile(
    stack
):
    """
    Remove DIAGNOSTIC work from the hot path.

    Does NOT modify:

        - RNG seed/state
        - success probabilities
        - physical timing
        - fidelity decay
        - memory lifetime
        - BSM probability
        - swap policy
        - quality thresholds
        - demand/recovery decisions

    Strict validators are saved and can still be run at boundaries.
    """

    if stack.get(
        "_phase3k_fast_profile_applied",
        False
    ):

        return stack


    simulator = (
        stack[
            "simulator"
        ]
    )


    manager = (
        simulator
        .entanglement_manager
    )


    kernel = (
        simulator
        .kernel
    )


    ledger = (
        stack[
            "physical_time_ledger"
        ]
    )


    # ============================================================
    # Save strict methods before replacing hot-path validation.
    # ============================================================

    stack[
        "_phase3k_strict_methods"
    ] = {

        "simulator_validate":
            simulator
            .validate_runtime_consistency,

        "manager_validate":
            manager
            .validate_all_active_resources,

        "kernel_emit":
            kernel.emit,

        "ledger_advance":
            ledger.advance
    }


    # ============================================================
    # Quiet event mode
    #
    # State mutation does NOT depend on event storage.
    # ============================================================

    kernel._phase3k_suppressed_event_count = 0


    def quiet_emit(
        *args,
        **kwargs
    ):

        kernel._phase3k_suppressed_event_count += 1

        return None


    kernel.emit = (
        quiet_emit
    )


    # ============================================================
    # Hot-path full-network consistency sweeps
    #
    # Internal mutation checks still remain.
    #
    # We run the saved STRICT versions at pair boundaries.
    # ============================================================

    simulator.validate_runtime_consistency = (
        lambda: True
    )


    manager.validate_all_active_resources = (
        lambda: True
    )


    # ============================================================
    # Lean physical-time advancement
    #
    # Same physical seconds.
    # No change whatsoever to the clock model.
    # ============================================================

    ledger._history = (
        NARADCountingSink()
    )


    def lean_advance(
        duration_s,
        phase_name,
        link_ids=(),
        resource_ids=(),
        metadata=None
    ):

        duration_s = float(
            duration_s
        )


        if (
            not math.isfinite(
                duration_s
            )
            or duration_s < 0.0
        ):

            raise ValueError(
                "duration_s must be a non-negative finite number."
            )


        if (
            not isinstance(
                phase_name,
                str
            )
            or not phase_name.strip()
        ):

            raise ValueError(
                "phase_name must be a non-empty string."
            )


        logical_slot_before = (
            kernel.current_slot
        )


        start_time_s = float(
            ledger._current_time_s
        )


        end_time_s = (

            start_time_s
            +
            duration_s
        )


        ledger._current_time_s = float(
            end_time_s
        )


        record = (
            PhysicalTimeAdvanceRecord(

                advance_index=len(
                    ledger._history
                ),

                simulation_slot=(
                    logical_slot_before
                ),

                phase_name=(
                    phase_name.strip()
                ),

                start_time_s=(
                    start_time_s
                ),

                duration_s=(
                    duration_s
                ),

                end_time_s=(
                    end_time_s
                ),

                link_ids=tuple(
                    link_ids
                ),

                resource_ids=tuple(
                    resource_ids
                ),

                # Metadata is diagnostically irrelevant in bulk mode.
                metadata={}
            )
        )


        ledger._history.append(
            record
        )


        if (
            kernel.current_slot
            !=
            logical_slot_before
        ):

            raise RuntimeError(
                "Physical-time advancement changed logical time."
            )


        return record


    ledger.advance = (
        lean_advance
    )


    # ============================================================
    # Remove persistent diagnostic histories.
    # ============================================================

    generation_scheduler = (
        stack[
            "generation_scheduler"
        ]
    )


    for process in (
        generation_scheduler
        .processes
    ):

        process._step_history = (
            NARADCountingSink()
        )


    generation_scheduler._slot_history = (
        NARADCountingSink()
    )


    demand_controller = (
        stack[
            "demand_controller"
        ]
    )


    if hasattr(
        demand_controller,
        "_history"
    ):

        demand_controller._history = (
            NARADCountingSink()
        )


    swap_scheduler = (
        stack[
            "swap_scheduler"
        ]
    )


    if hasattr(
        swap_scheduler,
        "_round_history"
    ):

        swap_scheduler._round_history = (
            NARADCountingSink()
        )


    orchestrator = (
        stack[
            "orchestrator"
        ]
    )


    for attribute_name in (

        "_execution_history",

        "_qkd_execution_history",

        "_physical_execution_history"
    ):

        if hasattr(
            orchestrator,
            attribute_name
        ):

            setattr(

                orchestrator,

                attribute_name,

                NARADCountingSink()
            )


    recovery_controller = stack.get(
        "collective_recovery_controller"
    )


    if (
        recovery_controller
        is not None
        and
        hasattr(
            recovery_controller,
            "_history"
        )
    ):

        recovery_controller._history = (
            NARADCountingSink()
        )


    stack[
        "_phase3k_fast_profile_applied"
    ] = True


    return stack


def phase3k_strict_boundary_audit(
    stack
):
    """
    Temporarily restore the original expensive validators and run them.

    Intended for:
        - end of one Bell-pair delivery;
        - end of one worker job;
        - benchmark equivalence tests.
    """

    methods = stack.get(
        "_phase3k_strict_methods"
    )


    if methods is None:

        raise RuntimeError(
            "Fast execution profile has not been applied."
        )


    simulator = (
        stack[
            "simulator"
        ]
    )


    manager = (
        simulator
        .entanglement_manager
    )


    fast_simulator_validate = (
        simulator
        .validate_runtime_consistency
    )


    fast_manager_validate = (
        manager
        .validate_all_active_resources
    )


    try:

        manager.validate_all_active_resources = (
            methods[
                "manager_validate"
            ]
        )


        simulator.validate_runtime_consistency = (
            methods[
                "simulator_validate"
            ]
        )


        assert (
            manager
            .validate_all_active_resources()
            is True
        )


        assert (
            simulator
            .validate_runtime_consistency()
            is True
        )


    finally:

        simulator.validate_runtime_consistency = (
            fast_simulator_validate
        )


        manager.validate_all_active_resources = (
            fast_manager_validate
        )


    return True


def build_phase3k_fast_stack(
    total_distance_km=600.0,
    repeater_count=6,
    simulation_seed=4_060_001,
    default_mode_count=8,
    link_mode_counts=None,
    generation_timing_geometry=(
        ElementaryAttemptTimingGeometry
        .MIDPOINT_HERALDED
    )
):
    """
    Phase-3K fast backend.

    Starts from the fully validated Phase-3I physical architecture
    and changes execution bookkeeping only.
    """

    stack = (
        build_phase3i2_multiplexed_stack(

            total_distance_km=(
                total_distance_km
            ),

            repeater_count=(
                repeater_count
            ),

            simulation_seed=(
                simulation_seed
            ),

            default_mode_count=(
                default_mode_count
            ),

            link_mode_counts=(
                link_mode_counts
            ),

            generation_timing_geometry=(
                generation_timing_geometry
            )
        )
    )


    simulator = (
        stack[
            "simulator"
        ]
    )


    path = (
        stack[
            "path"
        ]
    )


    multiplexing_config = (
        stack[
            "multiplexing_config"
        ]
    )


    # ============================================================
    # Replace generation service with the semantics-identical
    # aggregate-only implementation.
    # ============================================================

    generation_service = (
        NARADFastMultiplexedElementaryEntanglementService(

            simulator=(
                simulator
            ),

            model=(
                FiberSurvivalProxyModel()
            ),

            multiplexing_config=(
                multiplexing_config
            )
        )
    )


    generation_scheduler = (
        ParallelElementaryGenerationScheduler(

            simulator=(
                simulator
            ),

            service=(
                generation_service
            ),

            ordered_path=(
                path
            ),

            demand_active=False
        )
    )


    demand_controller = (
        FailureAwareElementaryDemandController(

            simulator=(
                simulator
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=(
                path
            )
        )
    )


    stack[
        "generation_service"
    ] = (
        generation_service
    )


    stack[
        "generation_scheduler"
    ] = (
        generation_scheduler
    )


    stack[
        "demand_controller"
    ] = (
        demand_controller
    )


    # ============================================================
    # Fast online-aggregate orchestrator.
    # ============================================================

    orchestrator = (
        NARADFastPhysicalTimedOrchestrator(

            stack=(
                stack
            ),

            timing_profile=(
                stack[
                    "timing_profile"
                ]
            ),

            generation_timing_geometry=(
                generation_timing_geometry
            ),

            config=(
                PHASE3H1_QUALITY_CONFIG
            )
        )
    )


    stack[
        "orchestrator"
    ] = (
        orchestrator
    )


    # Same multiplexing-aware quality-recovery cost model.
    phase3i2_install_multiplexing_aware_recovery(
        stack
    )


    phase3k_apply_lean_execution_profile(
        stack
    )


    assert (
        stack[
            "generation_scheduler"
        ]
        .service

        is

        generation_service
    )


    assert (
        stack[
            "orchestrator"
        ]
        .generation_scheduler

        is

        generation_scheduler
    )


    return stack

In [ ]:
# @title Full Backend vs Fast Backend Equivalence Test

# print(
#     "=" * 96
# )

# print(
#     "Same-Seed Physics Equivalence Benchmark"
# )

# print(
#     "=" * 96
# )


# PHASE3K_TEST_PHYSICAL_SEED = (
#     4_020_002
# )

# PHASE3K_TEST_PROTOCOL_SEED = (
#     7_470_002
# )


# # ================================================================
# # ORIGINAL fully diagnostic backend
# # ================================================================

# full_wall_start = (
#     time.perf_counter()
# )


# full_stack = (
#     build_phase3i2_multiplexed_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=(
#             PHASE3K_TEST_PHYSICAL_SEED
#         ),

#         default_mode_count=8
#     )
# )


# phase3i2_install_multiplexing_aware_recovery(
#     full_stack
# )


# full_channel = (
#     NARADQKDEntanglementChannel(

#         stack=(
#             full_stack
#         ),

#         protocol_seed=(
#             PHASE3K_TEST_PROTOCOL_SEED
#         ),

#         measurement_latency_s=0.0
#     )
# )


# full_record = (
#     full_channel
#     .deliver_and_measure(

#         alice_basis=0,

#         bob_basis=0,

#         max_slots_per_pair=20_000
#     )
# )


# full_wall_s = (
#     time.perf_counter()
#     -
#     full_wall_start
# )


# # ================================================================
# # FAST backend
# # ================================================================

# fast_wall_start = (
#     time.perf_counter()
# )


# fast_stack = (
#     build_phase3k_fast_stack(

#         total_distance_km=600.0,

#         repeater_count=6,

#         simulation_seed=(
#             PHASE3K_TEST_PHYSICAL_SEED
#         ),

#         default_mode_count=8
#     )
# )


# fast_channel = (
#     NARADQKDEntanglementChannel(

#         stack=(
#             fast_stack
#         ),

#         protocol_seed=(
#             PHASE3K_TEST_PROTOCOL_SEED
#         ),

#         measurement_latency_s=0.0
#     )
# )


# fast_record = (
#     fast_channel
#     .deliver_and_measure(

#         alice_basis=0,

#         bob_basis=0,

#         max_slots_per_pair=20_000
#     )
# )


# fast_wall_s = (
#     time.perf_counter()
#     -
#     fast_wall_start
# )


# # ================================================================
# # STRICT boundary validation
# # ================================================================

# assert (
#     phase3k_strict_boundary_audit(
#         fast_stack
#     )
#     is True
# )


# # ================================================================
# # STOCHASTIC + PHYSICAL equivalence
# # ================================================================

# assert (
#     full_record.delivery_end_slot
#     ==
#     fast_record.delivery_end_slot
# )


# assert (
#     full_record.elementary_generation_cycles
#     ==
#     fast_record.elementary_generation_cycles
# )


# assert (
#     full_record.optical_mode_attempts
#     ==
#     fast_record.optical_mode_attempts
# )


# assert (
#     full_record.bsm_attempts
#     ==
#     fast_record.bsm_attempts
# )


# assert math.isclose(

#     full_record.delivery_physical_time_s,

#     fast_record.delivery_physical_time_s,

#     rel_tol=0.0,

#     abs_tol=1e-15
# )


# assert math.isclose(

#     full_record.fidelity,

#     fast_record.fidelity,

#     rel_tol=0.0,

#     abs_tol=1e-15
# )


# assert math.isclose(

#     full_record.predicted_qber,

#     fast_record.predicted_qber,

#     rel_tol=0.0,

#     abs_tol=1e-15
# )


# assert (
#     full_record.alice_bit
#     ==
#     fast_record.alice_bit
# )


# assert (
#     full_record.bob_bit
#     ==
#     fast_record.bob_bit
# )


# # ================================================================
# # Benchmark
# # ================================================================

# phase3k_backend_benchmark = pd.DataFrame(
#     [
#         {
#             "Backend":
#                 "FULL_DIAGNOSTIC",

#             "Python Wall Time (s)":
#                 full_wall_s,

#             "Logical End Slot":
#                 full_record.delivery_end_slot,

#             "Modeled Physical Time (s)":
#                 full_record.delivery_physical_time_s,

#             "Generation Cycles":
#                 full_record.elementary_generation_cycles,

#             "Optical Mode Attempts":
#                 full_record.optical_mode_attempts,

#             "BSM Attempts":
#                 full_record.bsm_attempts,

#             "Fidelity":
#                 full_record.fidelity,

#             "Predicted QBER":
#                 full_record.predicted_qber
#         },

#         {
#             "Backend":
#                 "FAST_BULK",

#             "Python Wall Time (s)":
#                 fast_wall_s,

#             "Logical End Slot":
#                 fast_record.delivery_end_slot,

#             "Modeled Physical Time (s)":
#                 fast_record.delivery_physical_time_s,

#             "Generation Cycles":
#                 fast_record.elementary_generation_cycles,

#             "Optical Mode Attempts":
#                 fast_record.optical_mode_attempts,

#             "BSM Attempts":
#                 fast_record.bsm_attempts,

#             "Fidelity":
#                 fast_record.fidelity,

#             "Predicted QBER":
#                 fast_record.predicted_qber
#         }
#     ]
# )


# display(
#     phase3k_backend_benchmark
# )


# phase3k_execution_speedup = (

#     full_wall_s
#     /
#     fast_wall_s

#     if fast_wall_s > 0.0

#     else float(
#         "inf"
#     )
# )


# print(
#     "\nPython execution speed-up:",
#     f"{phase3k_execution_speedup:.2f}x"
# )


# print(
#     "\n[PASS] Same seed produced identical physical "
#     "delivery trajectory."
# )

# print(
#     "[PASS] Logical slots unchanged."
# )

# print(
#     "[PASS] Modeled physical seconds unchanged."
# )

# print(
#     "[PASS] Optical resource expenditure unchanged."
# )

# print(
#     "[PASS] BSM workload unchanged."
# )

# print(
#     "[PASS] Fidelity and QBER unchanged."
# )

# print(
#     "[PASS] Strict end-of-pair runtime audit passed."
# )


# print(
#     "\n" + "=" * 96
# )

# print(
#     "FAST BACKEND VALIDATED."
# )

# print(
#     "=" * 96
# )

In [ ]:
# @title  Resource-Bounded Multi-Pair Hardware Lanes

from concurrent.futures import (
    ProcessPoolExecutor,
    as_completed
)

import multiprocessing as mp
import os
import gc


PHASE3K_PARALLEL_PHYSICAL_MASTER_SEED = (
    4_080_001
)

PHASE3K_PARALLEL_PROTOCOL_MASTER_SEED = (
    7_480_001
)


@dataclass(frozen=True)
class NARADParallelLaneBatchSummary:

    parallel_lanes: int

    cpu_workers: int

    repeater_count: int

    elementary_link_count: int

    multiplexing_modes_per_lane_per_link: int

    memory_slots_per_node_per_lane: int

    provisioned_memory_slots_networkwide: int

    provisioned_optical_modes_per_link: int

    provisioned_optical_modes_networkwide: int

    provisioned_bsm_units_networkwide: int

    sequential_physical_work_s: float

    parallel_batch_physical_time_s: float

    physical_parallel_speedup: float

    delivered_pair_throughput_per_s: float

    python_wall_time_s: float

    total_generation_cycles: int

    total_optical_mode_attempts: int

    total_bsm_attempts: int

    mean_fidelity: float

    mean_predicted_qber: float


def phase3k_simulate_one_fast_hardware_lane(
    pair_index,
    total_distance_km=600.0,
    repeater_count=6,
    mode_count=8,
    z_probability=0.80,
    max_slots_per_pair=20_000
):
    """
    Simulate ONE physically independent hardware lane.

    Each lane has its own:
        - quantum memories,
        - optical multiplexing modes,
        - entanglement resources,
        - BSM operations,
        - seeded physical RNG.

    Therefore simultaneously running K lanes corresponds to
    explicitly provisioning K copies of these resources.
    """

    physical_seed = (
        phase3j_derive_pair_seed(

            master_seed=(
                PHASE3K_PARALLEL_PHYSICAL_MASTER_SEED
            ),

            pair_index=(
                pair_index
            ),

            stream_tag=301
        )
    )


    protocol_seed = (
        phase3j_derive_pair_seed(

            master_seed=(
                PHASE3K_PARALLEL_PROTOCOL_MASTER_SEED
            ),

            pair_index=(
                pair_index
            ),

            stream_tag=302
        )
    )


    stack = (
        build_phase3k_fast_stack(

            total_distance_km=(
                total_distance_km
            ),

            repeater_count=(
                repeater_count
            ),

            simulation_seed=(
                physical_seed
            ),

            default_mode_count=(
                mode_count
            )
        )
    )


    channel = (
        NARADQKDEntanglementChannel(

            stack=(
                stack
            ),

            protocol_seed=(
                protocol_seed
            ),

            measurement_latency_s=0.0
        )
    )


    local_record = (
        channel
        .deliver_and_measure(

            alice_basis=None,

            bob_basis=None,

            z_probability=(
                z_probability
            ),

            max_slots_per_pair=(
                max_slots_per_pair
            )
        )
    )


    record = replace(

        local_record,

        pair_index=(
            pair_index
        )
    )


    # Strict full consistency sweep at hardware-lane boundary.
    phase3k_strict_boundary_audit(
        stack
    )


    payload = asdict(
        record
    )


    del channel
    del stack

    gc.collect()


    return payload


def phase3k_run_parallel_hardware_lane_batch(
    pair_start_index=0,
    parallel_lanes=8,
    cpu_workers=None,
    total_distance_km=600.0,
    repeater_count=6,
    mode_count=8,
    z_probability=0.80,
    max_slots_per_pair=20_000
):
    """
    Execute one PHYSICAL batch containing K simultaneous pair lanes.

    Physical interpretation
    -----------------------
    All K lanes start at t=0 using dedicated provisioned hardware.

    Therefore:

        batch physical elapsed time
            =
        max(individual lane completion times)

    NOT:

        sum(individual lane completion times)

    Python execution parallelism is separate and is controlled by
    cpu_workers.
    """

    if (
        isinstance(
            parallel_lanes,
            bool
        )
        or not isinstance(
            parallel_lanes,
            int
        )
        or parallel_lanes < 1
    ):

        raise ValueError(
            "parallel_lanes must be a positive integer."
        )


    if cpu_workers is None:

        cpu_workers = min(

            parallel_lanes,

            max(
                1,
                min(
                    4,
                    os.cpu_count()
                    or
                    1
                )
            )
        )


    cpu_workers = int(
        cpu_workers
    )


    pair_indices = tuple(

        range(

            pair_start_index,

            pair_start_index
            +
            parallel_lanes
        )
    )


    wall_start = (
        time.perf_counter()
    )


    payloads = []


    # ============================================================
    # Computational parallelization
    #
    # This does NOT determine modeled physical parallelism.
    # ============================================================

    if cpu_workers == 1:

        for pair_index in pair_indices:

            payloads.append(

                phase3k_simulate_one_fast_hardware_lane(

                    pair_index=(
                        pair_index
                    ),

                    total_distance_km=(
                        total_distance_km
                    ),

                    repeater_count=(
                        repeater_count
                    ),

                    mode_count=(
                        mode_count
                    ),

                    z_probability=(
                        z_probability
                    ),

                    max_slots_per_pair=(
                        max_slots_per_pair
                    )
                )
            )


    else:

        try:

            context = (
                mp.get_context(
                    "fork"
                )
            )


            with ProcessPoolExecutor(

                max_workers=(
                    cpu_workers
                ),

                mp_context=(
                    context
                )
            ) as executor:

                futures = {

                    executor.submit(

                        phase3k_simulate_one_fast_hardware_lane,

                        pair_index,

                        total_distance_km,

                        repeater_count,

                        mode_count,

                        z_probability,

                        max_slots_per_pair

                    ):
                    pair_index

                    for pair_index
                    in pair_indices
                }


                for future in as_completed(
                    futures
                ):

                    payloads.append(
                        future.result()
                    )


        except Exception as error:

            print(
                "[WARN] Process-pool execution failed."
            )

            print(
                "Reason:",
                repr(
                    error
                )
            )

            print(
                "[INFO] Falling back to sequential Python execution. "
                "The PHYSICAL lane model is unchanged."
            )


            payloads = []


            for pair_index in pair_indices:

                payloads.append(

                    phase3k_simulate_one_fast_hardware_lane(

                        pair_index=(
                            pair_index
                        ),

                        total_distance_km=(
                            total_distance_km
                        ),

                        repeater_count=(
                            repeater_count
                        ),

                        mode_count=(
                            mode_count
                        ),

                        z_probability=(
                            z_probability
                        ),

                        max_slots_per_pair=(
                            max_slots_per_pair
                        )
                    )
                )


            cpu_workers = 1


    python_wall_time_s = (

        time.perf_counter()
        -
        wall_start
    )


    payloads.sort(

        key=lambda payload:
            payload[
                "pair_index"
            ]
    )


    records = tuple(

        NARADQKDPairMeasurement(
            **payload
        )

        for payload
        in payloads
    )


    individual_times = tuple(

        record.delivery_physical_time_s

        for record
        in records
    )


    sequential_physical_work_s = float(

        sum(
            individual_times
        )
    )


    parallel_batch_physical_time_s = float(

        max(
            individual_times
        )
    )


    physical_parallel_speedup = (

        sequential_physical_work_s

        /

        parallel_batch_physical_time_s
    )


    delivered_pair_throughput = (

        len(
            records
        )

        /

        parallel_batch_physical_time_s
    )


    elementary_link_count = (
        repeater_count
        +
        1
    )


    node_count = (
        repeater_count
        +
        2
    )


    # Current validated stack provisions 2 slots at every node.
    memory_slots_per_node_per_lane = 2


    provisioned_memory_slots_networkwide = (

        node_count

        *

        memory_slots_per_node_per_lane

        *

        parallel_lanes
    )


    provisioned_optical_modes_per_link = (

        mode_count

        *

        parallel_lanes
    )


    provisioned_optical_modes_networkwide = (

        elementary_link_count

        *

        provisioned_optical_modes_per_link
    )


    # One independent BSM capability at each repeater per lane.
    provisioned_bsm_units_networkwide = (

        repeater_count

        *

        parallel_lanes
    )


    summary = (
        NARADParallelLaneBatchSummary(

            parallel_lanes=(
                parallel_lanes
            ),

            cpu_workers=(
                cpu_workers
            ),

            repeater_count=(
                repeater_count
            ),

            elementary_link_count=(
                elementary_link_count
            ),

            multiplexing_modes_per_lane_per_link=(
                mode_count
            ),

            memory_slots_per_node_per_lane=(
                memory_slots_per_node_per_lane
            ),

            provisioned_memory_slots_networkwide=(
                provisioned_memory_slots_networkwide
            ),

            provisioned_optical_modes_per_link=(
                provisioned_optical_modes_per_link
            ),

            provisioned_optical_modes_networkwide=(
                provisioned_optical_modes_networkwide
            ),

            provisioned_bsm_units_networkwide=(
                provisioned_bsm_units_networkwide
            ),

            sequential_physical_work_s=(
                sequential_physical_work_s
            ),

            parallel_batch_physical_time_s=(
                parallel_batch_physical_time_s
            ),

            physical_parallel_speedup=(
                physical_parallel_speedup
            ),

            delivered_pair_throughput_per_s=(
                delivered_pair_throughput
            ),

            python_wall_time_s=(
                python_wall_time_s
            ),

            total_generation_cycles=sum(

                record.elementary_generation_cycles

                for record
                in records
            ),

            total_optical_mode_attempts=sum(

                record.optical_mode_attempts

                for record
                in records
            ),

            total_bsm_attempts=sum(

                record.bsm_attempts

                for record
                in records
            ),

            mean_fidelity=float(

                np.mean(
                    [
                        record.fidelity

                        for record
                        in records
                    ]
                )
            ),

            mean_predicted_qber=float(

                np.mean(
                    [
                        record.predicted_qber

                        for record
                        in records
                    ]
                )
            )
        )
    )


    return (
        records,
        summary
    )

In [ ]:
# @title Eight-Lane Parallel Delivery Pilot

# print(
#     "=" * 100
# )

# print(
#     "Eight-Lane Multi-Pair Quantum-Network Pilot"
# )

# print(
#     "=" * 100
# )


# phase3k_parallel_records, phase3k_parallel_summary = (
#     phase3k_run_parallel_hardware_lane_batch(

#         pair_start_index=0,

#         parallel_lanes=8,

#         # Computational workers only.
#         # Physical lanes remain 8 regardless of this value.
#         cpu_workers=min(
#             4,
#             os.cpu_count()
#             or
#             1
#         ),

#         total_distance_km=600.0,

#         repeater_count=6,

#         mode_count=8,

#         z_probability=(
#             PHASE3J_BIASED_BBM92_CONFIG
#             .z_probability
#         ),

#         max_slots_per_pair=20_000
#     )
# )


# # ================================================================
# # QKD-quality sanity
# # ================================================================

# assert (
#     len(
#         phase3k_parallel_records
#     )
#     ==
#     8
# )


# assert all(

#     record.consumed

#     for record
#     in phase3k_parallel_records
# )


# assert all(

#     record.predicted_qber

#     <=

#     PHASE3H1_QUALITY_CONFIG
#     .qber_limit

#     +

#     PHASE3H1_QUALITY_CONFIG
#     .numerical_tolerance

#     for record
#     in phase3k_parallel_records
# )


# # ================================================================
# # Summary
# # ================================================================

# display(
#     pd.DataFrame(
#         [
#             asdict(
#                 phase3k_parallel_summary
#             )
#         ]
#     )
# )


# phase3k_parallel_pair_df = pd.DataFrame(
#     [
#         {
#             "Pair":
#                 record.pair_index,

#             "Basis A":
#                 (
#                     "Z"
#                     if record.alice_basis == 0
#                     else "X"
#                 ),

#             "Basis B":
#                 (
#                     "Z"
#                     if record.bob_basis == 0
#                     else "X"
#                 ),

#             "Basis Match":
#                 record.basis_match,

#             "Physical Delivery Time (s)":
#                 record.delivery_physical_time_s,

#             "Generation Cycles":
#                 record.elementary_generation_cycles,

#             "Optical Mode Attempts":
#                 record.optical_mode_attempts,

#             "BSM Attempts":
#                 record.bsm_attempts,

#             "Fidelity":
#                 record.fidelity,

#             "Predicted QBER":
#                 record.predicted_qber
#         }

#         for record
#         in phase3k_parallel_records
#     ]
# )


# display(
#     phase3k_parallel_pair_df
# )


# print(
#     "\nPHYSICAL INTERPRETATION"
# )


# print(
#     "Independent pair lanes:",
#     phase3k_parallel_summary
#     .parallel_lanes
# )


# print(
#     "Provisioned memory slots:",
#     phase3k_parallel_summary
#     .provisioned_memory_slots_networkwide
# )


# print(
#     "Provisioned optical modes / physical link:",
#     phase3k_parallel_summary
#     .provisioned_optical_modes_per_link
# )


# print(
#     "Provisioned BSM units:",
#     phase3k_parallel_summary
#     .provisioned_bsm_units_networkwide
# )


# print(
#     "\nSequential physical work:",
#     f"{phase3k_parallel_summary.sequential_physical_work_s:.6f} s"
# )


# print(
#     "Parallel batch physical completion:",
#     f"{phase3k_parallel_summary.parallel_batch_physical_time_s:.6f} s"
# )


# print(
#     "Physical parallel speed-up:",
#     f"{phase3k_parallel_summary.physical_parallel_speedup:.2f}x"
# )


# print(
#     "Delivered-pair throughput:",
#     f"{phase3k_parallel_summary.delivered_pair_throughput_per_s:.3f} pairs/s"
# )


# print(
#     "Python wall time:",
#     f"{phase3k_parallel_summary.python_wall_time_s:.3f} s"
# )


# print(
#     "\n[PASS] Eight independent QKD-ready Bell-pair flows "
#     "were simulated with explicit hardware provisioning."
# )


# print(
#     "[PASS] Physical batch time used MAX(lane times), "
#     "not SUM(lane times)."
# )


# print(
#     "[PASS] Optical-mode cost remained explicitly counted."
# )


# print(
#     "[PASS] Memory and BSM hardware scale explicitly with lane count."
# )


# print(
#     "\n" + "=" * 100
# )

# print(
#     "EIGHT-LANE PILOT COMPLETE."
# )

# print(
#     "=" * 100
# )

In [ ]:
# @title Work-Conserving Asynchronous Quantum Lanes

from dataclasses import dataclass, asdict

import heapq
import math
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class NARADAsyncLaneAssignment:

    pair_index: int

    lane_id: int

    physical_start_time_s: float

    service_time_s: float

    physical_end_time_s: float

    fidelity: float

    predicted_qber: float

    elementary_generation_cycles: int

    optical_mode_attempts: int

    bsm_attempts: int


@dataclass(frozen=True)
class NARADAsyncServiceSummary:

    delivered_pairs: int

    parallel_lanes: int

    sequential_physical_work_s: float

    synchronized_batch_time_s: float

    asynchronous_makespan_s: float

    ideal_lower_bound_s: float

    synchronized_batch_throughput_pairs_per_s: float

    asynchronous_throughput_pairs_per_s: float

    sequential_to_async_speedup: float

    async_lane_utilization: float

    batch_lane_utilization: float

    total_lane_idle_time_s: float

    total_generation_cycles: int

    total_optical_mode_attempts: int

    total_bsm_attempts: int

    mean_fidelity: float

    mean_predicted_qber: float


def phase3k_synchronized_batch_time(
    records,
    parallel_lanes
):
    """
    Old batch-barrier model.

    Every group of K pairs begins together.
    The next group cannot begin until the slowest pair in the
    current group finishes.
    """

    records = tuple(
        sorted(
            records,
            key=lambda record:
                record.pair_index
        )
    )


    total_time_s = 0.0


    for start in range(
        0,
        len(records),
        parallel_lanes
    ):

        batch = (
            records[
                start:
                start + parallel_lanes
            ]
        )


        if not batch:

            continue


        total_time_s += max(

            record.delivery_physical_time_s

            for record
            in batch
        )


    return float(
        total_time_s
    )


def phase3k_schedule_work_conserving(
    records,
    parallel_lanes
):
    """
    FCFS work-conserving scheduler over statically partitioned
    quantum hardware lanes.

    IMPORTANT
    ---------
    The scheduler does NOT know future pair service times.

    Pair requests are considered in pair_index order.

    Each new request goes to the lane that is currently available
    earliest.

    The realized service duration is used only AFTER lane assignment
    to determine when that lane becomes available again.

    Therefore this is NOT shortest-job-first or oracle scheduling.
    """

    if (
        isinstance(
            parallel_lanes,
            bool
        )
        or not isinstance(
            parallel_lanes,
            int
        )
        or parallel_lanes < 1
    ):

        raise ValueError(
            "parallel_lanes must be a positive integer."
        )


    records = tuple(
        sorted(
            records,
            key=lambda record:
                record.pair_index
        )
    )


    if not records:

        raise ValueError(
            "At least one delivered-pair record is required."
        )


    # ============================================================
    # (available_time, lane_id)
    #
    # lane_id gives deterministic tie-breaking.
    # ============================================================

    lane_heap = [

        (
            0.0,
            lane_id
        )

        for lane_id
        in range(
            parallel_lanes
        )
    ]


    heapq.heapify(
        lane_heap
    )


    assignments = []


    for record in records:

        (
            lane_available_time_s,
            lane_id
        ) = heapq.heappop(
            lane_heap
        )


        physical_start_time_s = float(
            lane_available_time_s
        )


        service_time_s = float(
            record.delivery_physical_time_s
        )


        physical_end_time_s = (

            physical_start_time_s
            +
            service_time_s
        )


        assignment = (
            NARADAsyncLaneAssignment(

                pair_index=(
                    record.pair_index
                ),

                lane_id=(
                    lane_id
                ),

                physical_start_time_s=(
                    physical_start_time_s
                ),

                service_time_s=(
                    service_time_s
                ),

                physical_end_time_s=(
                    physical_end_time_s
                ),

                fidelity=(
                    record.fidelity
                ),

                predicted_qber=(
                    record.predicted_qber
                ),

                elementary_generation_cycles=(
                    record.elementary_generation_cycles
                ),

                optical_mode_attempts=(
                    record.optical_mode_attempts
                ),

                bsm_attempts=(
                    record.bsm_attempts
                )
            )
        )


        assignments.append(
            assignment
        )


        heapq.heappush(

            lane_heap,

            (
                physical_end_time_s,
                lane_id
            )
        )


    # ============================================================
    # Aggregate physical service metrics
    # ============================================================

    sequential_work_s = float(

        sum(
            assignment.service_time_s

            for assignment
            in assignments
        )
    )


    asynchronous_makespan_s = float(

        max(
            assignment.physical_end_time_s

            for assignment
            in assignments
        )
    )


    synchronized_batch_time_s = (
        phase3k_synchronized_batch_time(

            records=(
                records
            ),

            parallel_lanes=(
                parallel_lanes
            )
        )
    )


    maximum_single_pair_time_s = max(

        assignment.service_time_s

        for assignment
        in assignments
    )


    ideal_lower_bound_s = max(

        maximum_single_pair_time_s,

        sequential_work_s
        /
        parallel_lanes
    )


    total_available_lane_time_s = (

        parallel_lanes

        *
        asynchronous_makespan_s
    )


    total_idle_time_s = (

        total_available_lane_time_s

        -
        sequential_work_s
    )


    async_utilization = (

        sequential_work_s

        /
        total_available_lane_time_s
    )


    batch_utilization = (

        sequential_work_s

        /
        (
            parallel_lanes
            *
            synchronized_batch_time_s
        )
    )


    summary = (
        NARADAsyncServiceSummary(

            delivered_pairs=len(
                records
            ),

            parallel_lanes=(
                parallel_lanes
            ),

            sequential_physical_work_s=(
                sequential_work_s
            ),

            synchronized_batch_time_s=(
                synchronized_batch_time_s
            ),

            asynchronous_makespan_s=(
                asynchronous_makespan_s
            ),

            ideal_lower_bound_s=(
                ideal_lower_bound_s
            ),

            synchronized_batch_throughput_pairs_per_s=(

                len(
                    records
                )

                /
                synchronized_batch_time_s
            ),

            asynchronous_throughput_pairs_per_s=(

                len(
                    records
                )

                /
                asynchronous_makespan_s
            ),

            sequential_to_async_speedup=(

                sequential_work_s

                /
                asynchronous_makespan_s
            ),

            async_lane_utilization=(
                async_utilization
            ),

            batch_lane_utilization=(
                batch_utilization
            ),

            total_lane_idle_time_s=(
                total_idle_time_s
            ),

            total_generation_cycles=sum(

                record.elementary_generation_cycles

                for record
                in records
            ),

            total_optical_mode_attempts=sum(

                record.optical_mode_attempts

                for record
                in records
            ),

            total_bsm_attempts=sum(

                record.bsm_attempts

                for record
                in records
            ),

            mean_fidelity=float(

                np.mean(
                    [
                        record.fidelity

                        for record
                        in records
                    ]
                )
            ),

            mean_predicted_qber=float(

                np.mean(
                    [
                        record.predicted_qber

                        for record
                        in records
                    ]
                )
            )
        )
    )


    return (
        tuple(
            assignments
        ),

        summary
    )

In [ ]:
# @title Build Async Pilot Dataset Without Process-Pool Overhead

# import time


# PHASE3K_ASYNC_TOTAL_PAIRS = 32
# PHASE3K_ASYNC_LANES = 8


# # ================================================================
# # Reuse the already completed eight-lane pilot.
# # ================================================================

# existing_records_by_index = {}


# if (
#     "phase3k_parallel_records"
#     in globals()
# ):

#     for record in phase3k_parallel_records:

#         existing_records_by_index[
#             record.pair_index
#         ] = record


# required_pair_indices = tuple(

#     range(
#         PHASE3K_ASYNC_TOTAL_PAIRS
#     )
# )


# missing_pair_indices = tuple(

#     pair_index

#     for pair_index
#     in required_pair_indices

#     if (
#         pair_index
#         not in
#         existing_records_by_index
#     )
# )


# print(
#     "=" * 92
# )

# print(
#     "Sequential Trajectory Generation"
# )

# print(
#     "=" * 92
# )


# print(
#     "Required pairs:",
#     len(
#         required_pair_indices
#     )
# )


# print(
#     "Already available:",
#     (
#         len(
#             required_pair_indices
#         )
#         -
#         len(
#             missing_pair_indices
#         )
#     )
# )


# print(
#     "New trajectories required:",
#     len(
#         missing_pair_indices
#     )
# )


# wall_start = (
#     time.perf_counter()
# )


# completed_this_run = 0


# for pair_index in missing_pair_indices:

#     pair_wall_start = (
#         time.perf_counter()
#     )


#     record = (
#         phase3k_simulate_one_fast_hardware_lane(

#             pair_index=(
#                 pair_index
#             ),

#             total_distance_km=600.0,

#             repeater_count=6,

#             mode_count=8,

#             z_probability=(
#                 PHASE3J_BIASED_BBM92_CONFIG
#                 .z_probability
#             ),

#             max_slots_per_pair=20_000
#         )
#     )


#     # Worker returns a dict payload.
#     if isinstance(
#         record,
#         dict
#     ):

#         record = (
#             NARADQKDPairMeasurement(
#                 **record
#             )
#         )


#     existing_records_by_index[
#         pair_index
#     ] = record


#     completed_this_run += 1


#     pair_wall_s = (

#         time.perf_counter()
#         -
#         pair_wall_start
#     )


#     elapsed_s = (

#         time.perf_counter()
#         -
#         wall_start
#     )


#     mean_s = (

#         elapsed_s
#         /
#         completed_this_run
#     )


#     remaining = (

#         len(
#             missing_pair_indices
#         )
#         -
#         completed_this_run
#     )


#     estimated_remaining_s = (

#         mean_s
#         *
#         remaining
#     )


#     print(
#         f"Pair {pair_index:>3} complete | "
#         f"physical={record.delivery_physical_time_s:.6f} s | "
#         f"cycles={record.elementary_generation_cycles:>5} | "
#         f"Python={pair_wall_s:>7.2f} s | "
#         f"ETA≈{estimated_remaining_s / 60.0:>6.2f} min"
#     )


# phase3k_async_python_wall_s = (

#     time.perf_counter()
#     -
#     wall_start
# )


# phase3k_async_records = tuple(

#     existing_records_by_index[
#         pair_index
#     ]

#     for pair_index
#     in required_pair_indices
# )


# assert (
#     len(
#         phase3k_async_records
#     )
#     ==
#     PHASE3K_ASYNC_TOTAL_PAIRS
# )


# assert (

#     tuple(
#         record.pair_index

#         for record
#         in phase3k_async_records
#     )

#     ==

#     required_pair_indices
# )


# print(
#     "\n" + "=" * 92
# )

# print(
#     "[PASS] 32 independent physical trajectories ready."
# )

# print(
#     "Python generation time:",
#     f"{phase3k_async_python_wall_s:.3f} s"
# )

# print(
#     "=" * 92
# )

In [ ]:
# @title Async Work-Conserving Throughput Verification

# print(
#     "=" * 104
# )

# print(
#     "Work-Conserving Asynchronous Quantum-Lane Pilot"
# )

# print(
#     "=" * 104
# )


# (
#     phase3k_async_assignments,
#     phase3k_async_summary
# ) = (
#     phase3k_schedule_work_conserving(

#         records=(
#             phase3k_async_records
#         ),

#         parallel_lanes=(
#             PHASE3K_ASYNC_LANES
#         )
#     )
# )


# # ================================================================
# # Core invariants
# # ================================================================

# assert (
#     phase3k_async_summary
#     .asynchronous_makespan_s

#     <=

#     phase3k_async_summary
#     .synchronized_batch_time_s
#     +
#     1e-15
# )


# assert (
#     phase3k_async_summary
#     .async_lane_utilization

#     >=

#     phase3k_async_summary
#     .batch_lane_utilization
#     -
#     1e-15
# )


# assert (
#     phase3k_async_summary
#     .asynchronous_makespan_s

#     >=

#     phase3k_async_summary
#     .ideal_lower_bound_s
#     -
#     1e-15
# )


# # Every physical pair is delivered exactly once.
# assert (

#     sorted(
#         assignment.pair_index

#         for assignment
#         in phase3k_async_assignments
#     )

#     ==

#     list(
#         range(
#             PHASE3K_ASYNC_TOTAL_PAIRS
#         )
#     )
# )


# # ================================================================
# # Check for illegal overlapping work INSIDE one physical lane.
# # ================================================================

# for lane_id in range(
#     PHASE3K_ASYNC_LANES
# ):

#     lane_jobs = sorted(

#         (
#             assignment

#             for assignment
#             in phase3k_async_assignments

#             if (
#                 assignment.lane_id
#                 ==
#                 lane_id
#             )
#         ),

#         key=lambda assignment:
#             assignment.physical_start_time_s
#     )


#     for previous, current in zip(

#         lane_jobs[:-1],
#         lane_jobs[1:]
#     ):

#         assert (

#             current.physical_start_time_s

#             >=

#             previous.physical_end_time_s
#             -
#             1e-15
#         )


# # ================================================================
# # Summary comparison
# # ================================================================

# comparison_df = pd.DataFrame(
#     [
#         {
#             "Execution Model":
#                 "SYNCHRONIZED_BATCH_BARRIER",

#             "Physical Completion Time (s)":
#                 phase3k_async_summary
#                 .synchronized_batch_time_s,

#             "Throughput (pairs/s)":
#                 phase3k_async_summary
#                 .synchronized_batch_throughput_pairs_per_s,

#             "Lane Utilization":
#                 phase3k_async_summary
#                 .batch_lane_utilization
#         },

#         {
#             "Execution Model":
#                 "WORK_CONSERVING_ASYNC",

#             "Physical Completion Time (s)":
#                 phase3k_async_summary
#                 .asynchronous_makespan_s,

#             "Throughput (pairs/s)":
#                 phase3k_async_summary
#                 .asynchronous_throughput_pairs_per_s,

#             "Lane Utilization":
#                 phase3k_async_summary
#                 .async_lane_utilization
#         }
#     ]
# )


# display(
#     comparison_df
# )


# # ================================================================
# # Per-job asynchronous timeline
# # ================================================================

# assignment_df = pd.DataFrame(
#     [
#         {
#             "Pair":
#                 assignment.pair_index,

#             "Lane":
#                 assignment.lane_id,

#             "Start (s)":
#                 assignment.physical_start_time_s,

#             "Service Time (s)":
#                 assignment.service_time_s,

#             "Finish (s)":
#                 assignment.physical_end_time_s,

#             "Generation Cycles":
#                 assignment.elementary_generation_cycles,

#             "Optical Mode Attempts":
#                 assignment.optical_mode_attempts,

#             "BSM Attempts":
#                 assignment.bsm_attempts,

#             "Fidelity":
#                 assignment.fidelity,

#             "Predicted QBER":
#                 assignment.predicted_qber
#         }

#         for assignment
#         in phase3k_async_assignments
#     ]
# )


# display(
#     assignment_df
# )


# # ================================================================
# # Completion order
# #
# # This is the order in which BBM92 measurement bits become
# # available to the classical session.
# # ================================================================

# completion_order_df = (

#     assignment_df
#     .sort_values(
#         by=[
#             "Finish (s)",
#             "Lane",
#             "Pair"
#         ]
#     )
#     .reset_index(
#         drop=True
#     )
# )


# print(
#     "\nFIRST 16 PHYSICAL COMPLETIONS"
# )


# display(
#     completion_order_df
#     .head(
#         16
#     )
# )


# # ================================================================
# # Hardware provisioning is UNCHANGED from our 8-lane pilot.
# # ================================================================

# memory_slots = (

#     8      # physical nodes

#     *
#     2      # slots per node per lane

#     *
#     PHASE3K_ASYNC_LANES
# )


# optical_modes_per_link = (

#     8      # m = 8

#     *
#     PHASE3K_ASYNC_LANES
# )


# bsm_units = (

#     6      # repeaters

#     *
#     PHASE3K_ASYNC_LANES
# )


# print(
#     "\nPHYSICAL HARDWARE"
# )


# print(
#     "Parallel hardware lanes:",
#     PHASE3K_ASYNC_LANES
# )


# print(
#     "Provisioned memory slots:",
#     memory_slots
# )


# print(
#     "Provisioned optical modes / physical link:",
#     optical_modes_per_link
# )


# print(
#     "Provisioned BSM units:",
#     bsm_units
# )


# print(
#     "\nTHROUGHPUT"
# )


# print(
#     "Sequential physical work:",
#     f"{phase3k_async_summary.sequential_physical_work_s:.6f} s"
# )


# print(
#     "Old synchronized-batch completion:",
#     f"{phase3k_async_summary.synchronized_batch_time_s:.6f} s"
# )


# print(
#     "Async work-conserving completion:",
#     f"{phase3k_async_summary.asynchronous_makespan_s:.6f} s"
# )


# print(
#     "Ideal non-preemptive lower bound:",
#     f"{phase3k_async_summary.ideal_lower_bound_s:.6f} s"
# )


# print(
#     "Batch throughput:",
#     (
#         f"{phase3k_async_summary.synchronized_batch_throughput_pairs_per_s:.3f} pairs/s"
#     )
# )


# print(
#     "Async throughput:",
#     (
#         f"{phase3k_async_summary.asynchronous_throughput_pairs_per_s:.3f} pairs/s"
#     )
# )


# print(
#     "Sequential -> async physical speed-up:",
#     f"{phase3k_async_summary.sequential_to_async_speedup:.2f}x"
# )


# print(
#     "Old batch lane utilization:",
#     f"{phase3k_async_summary.batch_lane_utilization:.2%}"
# )


# print(
#     "Async lane utilization:",
#     f"{phase3k_async_summary.async_lane_utilization:.2%}"
# )


# print(
#     "Total async lane idle time:",
#     f"{phase3k_async_summary.total_lane_idle_time_s:.6f} lane-s"
# )


# async_vs_batch_gain = (

#     phase3k_async_summary
#     .asynchronous_throughput_pairs_per_s

#     /

#     phase3k_async_summary
#     .synchronized_batch_throughput_pairs_per_s
# )


# print(
#     "Async throughput gain over batch barriers:",
#     f"{async_vs_batch_gain:.2f}x"
# )


# print(
#     "\n[PASS] No physical lane executed two Bell-pair jobs simultaneously."
# )


# print(
#     "[PASS] Finished lanes immediately accepted waiting pair requests."
# )


# print(
#     "[PASS] No future service duration was used to choose a lane."
# )


# print(
#     "[PASS] Optical, memory and BSM provisioning stayed fixed."
# )


# print(
#     "[PASS] Async improvement therefore comes from utilization, "
#     "not extra hardware."
# )


# print(
#     "\n" + "=" * 104
# )

# print(
#     "ASYNCHRONOUS LANE SERVICE VALIDATED."
# )

# print(
#     "=" * 104
# )

In [ ]:
# @title  Async Secret-Key Experiment Inputs + Pair Cache

from dataclasses import (
    dataclass,
    asdict,
    replace,
    fields
)

from pathlib import Path

import heapq
import json
import math
import os
import time

import numpy as np
import pandas as pd


# ================================================================
# USER INPUTS
# ================================================================

PHASE3K_KEY_PARALLEL_LANES = 50

PHASE3K_KEY_TARGET_SECRET_BITS = 128


# ================================================================
# AES permits exactly:
#
#       AES-128
#       AES-192
#       AES-256
# ================================================================

if (
    PHASE3K_KEY_TARGET_SECRET_BITS
    not in
    (
        128,
        192,
        256
    )
):

    raise ValueError(

        "For an AES-usable final key, "
        "TARGET_SECRET_BITS must be one of "
        "{128, 192, 256}."
    )


if (
    isinstance(
        PHASE3K_KEY_PARALLEL_LANES,
        bool
    )
    or not isinstance(
        PHASE3K_KEY_PARALLEL_LANES,
        int
    )
    or PHASE3K_KEY_PARALLEL_LANES < 1
):

    raise ValueError(
        "PHASE3K_KEY_PARALLEL_LANES must be positive."
    )


# ================================================================
# Frozen physical experiment parameters
# ================================================================

PHASE3K_KEY_TOTAL_DISTANCE_KM = 600.0

PHASE3K_KEY_REPEATER_COUNT = 6

PHASE3K_KEY_MODE_COUNT = 8

PHASE3K_KEY_MAX_COMPLETED_PAIRS = 5_000

PHASE3K_KEY_PROGRESS_EVERY = 16

PHASE3K_KEY_POSTPROCESS_RETRY_EVERY = 8


# ================================================================
# Preserve the same deterministic pair stream used by Phase 3K.
#
# This is useful because:
#
#       K = 1
#       K = 8
#       K = 16
#
# can then see the SAME underlying stochastic pair trajectories.
# ================================================================

PHASE3K_PARALLEL_PHYSICAL_MASTER_SEED = (
    4_080_001
)

PHASE3K_PARALLEL_PROTOCOL_MASTER_SEED = (
    7_480_001
)


# Completely separate post-processing randomness.
PHASE3K_KEY_POSTPROCESS_MASTER_SEED = (
    7_490_001
)


# ================================================================
# Create experiment-specific BBM92 configuration
# ================================================================

PHASE3K_KEY_QKD_CONFIG = replace(

    PHASE3J_BIASED_BBM92_CONFIG,

    target_secret_bits=(
        PHASE3K_KEY_TARGET_SECRET_BITS
    ),

    max_delivered_pairs=(
        PHASE3K_KEY_MAX_COMPLETED_PAIRS
    )
)


# ================================================================
# Persistent pair-trajectory cache
#
# Pair trajectories are independent of the number of lanes.
#
# Therefore:
#
#   generate pair 137 once
#   ->
#   reuse pair 137 for K=8, K=16, etc.
#
# This is VERY useful for fair scaling experiments.
# ================================================================

drive_root = Path(
    "/content/drive/MyDrive"
)


if drive_root.exists():

    PHASE3K_KEY_CACHE_DIR = (

        drive_root
        /
        "NARAD_QC"
        /
        "checkpoints"
        /
        "phase3k_async_key_experiment"
    )

else:

    PHASE3K_KEY_CACHE_DIR = Path(
        "/content/phase3k_async_key_experiment"
    )


PHASE3K_KEY_CACHE_DIR.mkdir(

    parents=True,

    exist_ok=True
)


PHASE3K_KEY_PAIR_CACHE_FILE = (

    PHASE3K_KEY_CACHE_DIR
    /
    (
        "pair_cache_"
        "600km_"
        "R6_"
        "m8_"
        "z80_"
        "phys4080001_"
        "prot7480001.jsonl"
    )
)


PHASE3K_KEY_PAIR_CACHE = {}


valid_measurement_fields = {

    field.name

    for field
    in fields(
        NARADQKDPairMeasurement
    )
}


# ================================================================
# Load existing persistent trajectories
# ================================================================

if PHASE3K_KEY_PAIR_CACHE_FILE.exists():

    with open(

        PHASE3K_KEY_PAIR_CACHE_FILE,

        "r",

        encoding="utf-8"

    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):

            line = line.strip()


            if not line:

                continue


            try:

                payload = json.loads(
                    line
                )

            except json.JSONDecodeError:

                print(
                    "[WARN] Ignoring incomplete final cache line:",
                    line_number
                )

                break


            filtered_payload = {

                key:
                    value

                for key, value
                in payload.items()

                if (
                    key
                    in
                    valid_measurement_fields
                )
            }


            record = (
                NARADQKDPairMeasurement(
                    **filtered_payload
                )
            )


            existing = (
                PHASE3K_KEY_PAIR_CACHE
                .get(
                    record.pair_index
                )
            )


            if (
                existing is not None
                and
                existing != record
            ):

                raise RuntimeError(

                    "Conflicting cached trajectory for "
                    f"pair {record.pair_index}."
                )


            PHASE3K_KEY_PAIR_CACHE[
                record.pair_index
            ] = record


# ================================================================
# Import trajectories already present in RAM from our previous
# 32-pair experiment.
# ================================================================

if (
    "phase3k_async_records"
    in globals()
):

    for record in phase3k_async_records:

        pair_index = (
            record.pair_index
        )


        if (
            pair_index
            not in
            PHASE3K_KEY_PAIR_CACHE
        ):

            PHASE3K_KEY_PAIR_CACHE[
                pair_index
            ] = record


            with open(

                PHASE3K_KEY_PAIR_CACHE_FILE,

                "a",

                encoding="utf-8"

            ) as file:

                file.write(

                    json.dumps(
                        asdict(
                            record
                        ),
                        sort_keys=True
                    )

                    +
                    "\n"
                )


                file.flush()

                os.fsync(
                    file.fileno()
                )


print(
    "=" * 96
)

print(
    "Phase 3K.4 — Dynamic Async Secret-Key Experiment"
)

print(
    "=" * 96
)


print(
    "Parallel quantum lanes:",
    PHASE3K_KEY_PARALLEL_LANES
)


print(
    "Requested AES secret:",
    PHASE3K_KEY_TARGET_SECRET_BITS,
    "bits"
)


print(
    "AES key bytes:",
    PHASE3K_KEY_TARGET_SECRET_BITS // 8
)


print(
    "Existing cached physical trajectories:",
    len(
        PHASE3K_KEY_PAIR_CACHE
    )
)


print(
    "Cache:",
    PHASE3K_KEY_PAIR_CACHE_FILE
)

In [ ]:
# @title Configurable BBM92 Distillation

def phase3k_hash_with_existing_toeplitz_seed(
    input_bits,
    output_length,
    seed
):

    source = np.asarray(

        input_bits,

        dtype=np.uint8
    )


    seed = np.asarray(

        seed,

        dtype=np.uint8
    )


    first_column = (
        seed[
            :output_length
        ]
    )


    first_row = np.concatenate(
        (
            seed[
                :1
            ],

            seed[
                output_length:
            ]
        )
    )


    output = np.zeros(

        output_length,

        dtype=np.uint8
    )


    for row in range(
        output_length
    ):

        accumulator = 0


        for column in range(
            len(
                source
            )
        ):

            matrix_value = (

                first_column[
                    row - column
                ]

                if (
                    row
                    >=
                    column
                )

                else

                first_row[
                    column - row
                ]
            )


            accumulator ^= (

                int(
                    matrix_value
                )

                &

                int(
                    source[
                        column
                    ]
                )
            )


        output[
            row
        ] = accumulator


    return tuple(

        int(
            bit
        )

        for bit
        in output.tolist()
    )


def phase3k_distill_configurable_bbm92_key(
    measurements,
    collection_summary,
    config,
    postprocess_rng
):
    """
    Actual post-processing gate.

    Success here means:

        parameter estimation passed
        +
        Cascade reconciliation succeeded
        +
        ACTUAL parity leakage was deducted
        +
        security margin was deducted
        +
        requested secret entropy exists
        +
        Toeplitz PA generated exactly TARGET_SECRET_BITS
        +
        Alice == Bob

    No forecast alone can produce a key.
    """

    measurements = tuple(
        measurements
    )


    zz = tuple(

        record

        for record
        in measurements

        if (
            record.alice_basis == 0
            and
            record.bob_basis == 0
        )
    )


    xx = tuple(

        record

        for record
        in measurements

        if (
            record.alice_basis == 1
            and
            record.bob_basis == 1
        )
    )


    if (
        len(
            xx
        )
        <
        config.minimum_parameter_estimation_bits
    ):

        raise RuntimeError(
            "INSUFFICIENT_PARAMETER_ESTIMATION"
        )


    if (
        collection_summary
        .phase_error_estimate

        >

        config.qber_abort_threshold
    ):

        raise RuntimeError(
            "PHASE_ERROR_ABOVE_THRESHOLD"
        )


    alice_raw = tuple(

        record.alice_bit

        for record
        in zz
    )


    bob_raw = tuple(

        record.bob_bit

        for record
        in zz
    )


    if not alice_raw:

        raise RuntimeError(
            "NO_ZZ_RAW_KEY_BITS"
        )


    actual_z_qber = (

        sum(

            int(
                alice_bit
                !=
                bob_bit
            )

            for alice_bit, bob_bit
            in zip(
                alice_raw,
                bob_raw
            )
        )

        /

        len(
            alice_raw
        )
    )


    # ============================================================
    # CASCADE
    # ============================================================

    reconciliation = (
        narad_cascade_reconcile(

            alice_bits=(
                alice_raw
            ),

            bob_bits=(
                bob_raw
            ),

            estimated_qber=(
                collection_summary
                .phase_error_estimate
            ),

            protocol_rng=(
                postprocess_rng
            ),

            passes=(
                config
                .cascade_passes
            )
        )
    )


    if not reconciliation.success:

        raise RuntimeError(

            "CASCADE_RECONCILIATION_FAILED:"
            f"{reconciliation.residual_errors}"
        )


    if (

        reconciliation
        .alice_reconciled_bits

        !=

        reconciliation
        .bob_reconciled_bits
    ):

        raise RuntimeError(
            "POST_CASCADE_KEYS_DIFFER"
        )


    # ============================================================
    # ACTUAL SECRET ENTROPY BUDGET
    # ============================================================

    phase_entropy = (
        narad_binary_entropy(

            collection_summary
            .phase_error_estimate
        )
    )


    maximum_extractable_bits = int(

        math.floor(

            len(
                alice_raw
            )

            *

            (
                1.0
                -
                phase_entropy
            )

            -

            reconciliation
            .parity_bits_disclosed

            -

            config
            .security_margin_bits
        )
    )


    if (
        maximum_extractable_bits
        <
        config.target_secret_bits
    ):

        raise RuntimeError(

            "INSUFFICIENT_POST_EC_ENTROPY:"
            f"{maximum_extractable_bits}"
        )


    # ============================================================
    # TOEPLITZ PRIVACY AMPLIFICATION
    # ============================================================

    (
        alice_final_bits,
        public_pa_seed
    ) = (
        narad_toeplitz_hash(

            input_bits=(

                reconciliation
                .alice_reconciled_bits
            ),

            output_length=(
                config
                .target_secret_bits
            ),

            protocol_rng=(
                postprocess_rng
            )
        )
    )


    bob_final_bits = (
        phase3k_hash_with_existing_toeplitz_seed(

            input_bits=(

                reconciliation
                .bob_reconciled_bits
            ),

            output_length=(
                config
                .target_secret_bits
            ),

            seed=(
                public_pa_seed
            )
        )
    )


    if (
        alice_final_bits
        !=
        bob_final_bits
    ):

        raise RuntimeError(
            "PRIVACY_AMPLIFICATION_KEYS_DIFFER"
        )


    alice_key_bytes = (
        bits_to_bytes(

            list(
                alice_final_bits
            )
        )
    )


    bob_key_bytes = (
        bits_to_bytes(

            list(
                bob_final_bits
            )
        )
    )


    expected_key_bytes = (

        config.target_secret_bits
        //
        8
    )


    if (
        len(
            alice_key_bytes
        )
        !=
        expected_key_bytes
    ):

        raise RuntimeError(

            "Unexpected final key length: "
            f"{len(alice_key_bytes)} bytes."
        )


    if (
        alice_key_bytes
        !=
        bob_key_bytes
    ):

        raise RuntimeError(
            "ALICE_BOB_FINAL_KEYS_DIFFER"
        )


    return (
        NARADBiasedBBM92SessionResult(

            success=True,

            delivered_pairs=(
                collection_summary
                .delivered_pairs
            ),

            zz_raw_key_bits=(
                collection_summary
                .zz_key_basis_bits
            ),

            xx_parameter_estimation_bits=(
                collection_summary
                .xx_parameter_estimation_bits
            ),

            mismatched_basis_pairs=(
                collection_summary
                .mismatched_basis_pairs
            ),

            observed_x_qber=(
                collection_summary
                .observed_x_qber
            ),

            actual_z_qber_diagnostic=float(
                actual_z_qber
            ),

            mean_network_predicted_qber=(
                collection_summary
                .mean_predicted_qber
            ),

            phase_error_estimate=(
                collection_summary
                .phase_error_estimate
            ),

            cascade_passes=(
                reconciliation
                .passes_used
            ),

            cascade_corrections=(
                reconciliation
                .corrections
            ),

            ec_leakage_bits=(
                reconciliation
                .parity_bits_disclosed
            ),

            security_margin_bits=(
                config
                .security_margin_bits
            ),

            maximum_extractable_bits=(
                maximum_extractable_bits
            ),

            final_secret_bits=(
                config
                .target_secret_bits
            ),

            physical_time_s=(
                collection_summary
                .physical_time_s
            ),

            elementary_generation_cycles=(
                collection_summary
                .elementary_generation_cycles
            ),

            optical_mode_attempts=(
                collection_summary
                .optical_mode_attempts
            ),

            bsm_attempts=(
                collection_summary
                .bsm_attempts
            ),

            alice_key_bytes=(
                alice_key_bytes
            ),

            bob_key_bytes=(
                bob_key_bytes
            )
        )
    )


print(
    "[PASS] Configurable AES-128/192/256 "
    "BBM92 distiller ready."
)

In [ ]:
# @title  Continuous Async Bell-Pair Service Until Secret Key Exists

@dataclass(frozen=True)
class NARADAsyncKeyCompletion:

    completion_index: int

    pair_index: int

    lane_id: int

    physical_start_time_s: float

    service_time_s: float

    physical_finish_time_s: float


def phase3k_append_pair_to_cache(
    record
):

    pair_index = (
        record.pair_index
    )


    if (
        pair_index
        in
        PHASE3K_KEY_PAIR_CACHE
    ):

        return


    PHASE3K_KEY_PAIR_CACHE[
        pair_index
    ] = record


    with open(

        PHASE3K_KEY_PAIR_CACHE_FILE,

        "a",

        encoding="utf-8"

    ) as file:

        file.write(

            json.dumps(
                asdict(
                    record
                ),
                sort_keys=True
            )

            +
            "\n"
        )


        file.flush()

        os.fsync(
            file.fileno()
        )


def phase3k_get_or_simulate_pair(
    pair_index
):
    """
    Return one complete physical Bell-pair trajectory.

    Cached trajectory:
        almost zero Python cost.

    New trajectory:
        run the validated Phase-3K fast physical simulator.
    """

    if (
        pair_index
        in
        PHASE3K_KEY_PAIR_CACHE
    ):

        return (
            PHASE3K_KEY_PAIR_CACHE[
                pair_index
            ],

            True,

            0.0
        )


    wall_start = (
        time.perf_counter()
    )


    payload = (
        phase3k_simulate_one_fast_hardware_lane(

            pair_index=(
                pair_index
            ),

            total_distance_km=(
                PHASE3K_KEY_TOTAL_DISTANCE_KM
            ),

            repeater_count=(
                PHASE3K_KEY_REPEATER_COUNT
            ),

            mode_count=(
                PHASE3K_KEY_MODE_COUNT
            ),

            z_probability=(
                PHASE3K_KEY_QKD_CONFIG
                .z_probability
            ),

            max_slots_per_pair=(
                PHASE3K_KEY_QKD_CONFIG
                .max_slots_per_pair
            )
        )
    )


    if isinstance(
        payload,
        dict
    ):

        record = (
            NARADQKDPairMeasurement(
                **payload
            )
        )

    else:

        record = payload


    python_wall_s = (

        time.perf_counter()
        -
        wall_start
    )


    phase3k_append_pair_to_cache(
        record
    )


    return (
        record,
        False,
        python_wall_s
    )


def phase3k_make_postprocess_rng(
    delivered_pairs,
    parallel_lanes,
    target_secret_bits
):

    seed_sequence = (
        np.random.SeedSequence(
            [
                PHASE3K_KEY_POSTPROCESS_MASTER_SEED,

                int(
                    delivered_pairs
                ),

                int(
                    parallel_lanes
                ),

                int(
                    target_secret_bits
                )
            ]
        )
    )


    return np.random.default_rng(
        seed_sequence
    )


def phase3k_run_async_until_secret_key(
    parallel_lanes,
    config,
    progress_every=16,
    postprocess_retry_every=8
):
    """
    Continuous FCFS / work-conserving quantum service.

    MODELED PHYSICS
    ---------------
    K physical lanes operate simultaneously.

    Each lane owns:
        2 memory slots per network node
        m optical modes per elementary link
        1 BSM unit per repeater

    When one lane completes:
        its Bell-pair measurement becomes available,
        then that lane immediately receives the next request.

    STOPPING CONDITION
    ------------------
    Stop ONLY after actual post-processing can generate exactly
    config.target_secret_bits secret bits.

    Python computation time and modeled physical time are reported
    separately.
    """

    if (
        isinstance(
            parallel_lanes,
            bool
        )
        or not isinstance(
            parallel_lanes,
            int
        )
        or parallel_lanes < 1
    ):

        raise ValueError(
            "parallel_lanes must be positive."
        )


    max_pairs = int(
        config.max_delivered_pairs
    )


    python_wall_start = (
        time.perf_counter()
    )


    # ============================================================
    # Priority queue:
    #
    # (
    #   physical_finish_time,
    #   lane_id,
    #   pair_index,
    #   physical_start_time,
    #   measurement_record
    # )
    # ============================================================

    active_heap = []


    completed_measurements = []

    completion_log = []


    next_pair_index = 0

    newly_simulated_count = 0

    cached_trajectory_count = 0

    trajectory_python_time_s = 0.0


    # ============================================================
    # t = 0
    #
    # Fill every physical hardware lane simultaneously.
    # ============================================================

    initial_jobs = min(

        parallel_lanes,

        max_pairs
    )


    print(
        "=" * 104
    )

    print(
        "NARAD-QC Continuous Asynchronous Secret-Key Session"
    )

    print(
        "=" * 104
    )


    print(
        "Parallel lanes:",
        parallel_lanes
    )


    print(
        "Target secret:",
        config.target_secret_bits,
        "bits"
    )


    print(
        "Basis bias:",
        f"Z={config.z_probability:.0%}, "
        f"X={1.0-config.z_probability:.0%}"
    )


    print(
        "\nLaunching",
        initial_jobs,
        "simultaneous physical lanes..."
    )


    for lane_id in range(
        initial_jobs
    ):

        (
            record,
            from_cache,
            pair_python_s
        ) = (
            phase3k_get_or_simulate_pair(
                next_pair_index
            )
        )


        if from_cache:

            cached_trajectory_count += 1

        else:

            newly_simulated_count += 1


        trajectory_python_time_s += (
            pair_python_s
        )


        service_time_s = float(
            record.delivery_physical_time_s
        )


        heapq.heappush(

            active_heap,

            (
                service_time_s,
                lane_id,
                next_pair_index,
                0.0,
                record
            )
        )


        next_pair_index += 1


    final_session_result = None

    final_collection_summary = None

    last_postprocess_attempt_at = -10**9


    # ============================================================
    # DISCRETE EVENT LOOP
    # ============================================================

    while active_heap:

        (
            physical_finish_time_s,
            lane_id,
            pair_index,
            physical_start_time_s,
            record
        ) = (
            heapq.heappop(
                active_heap
            )
        )


        # ========================================================
        # One Bell pair has physically completed.
        #
        # QKD sees records in PHYSICAL COMPLETION ORDER.
        # ========================================================

        completed_measurements.append(
            record
        )


        completion_log.append(

            NARADAsyncKeyCompletion(

                completion_index=(
                    len(
                        completed_measurements
                    )
                    -
                    1
                ),

                pair_index=(
                    pair_index
                ),

                lane_id=(
                    lane_id
                ),

                physical_start_time_s=(
                    physical_start_time_s
                ),

                service_time_s=(
                    record.delivery_physical_time_s
                ),

                physical_finish_time_s=(
                    physical_finish_time_s
                )
            )
        )


        # ========================================================
        # Recompute QKD collection state using ONLY physically
        # completed Bell pairs.
        # ========================================================

        raw_summary = (
            phase3j_bbm92_collection_summary(

                measurements=(
                    completed_measurements
                ),

                config=(
                    config
                )
            )
        )


        # IMPORTANT:
        #
        # The old collection helper sums individual pair service
        # durations because it was originally sequential.
        #
        # In this asynchronous model the real network elapsed time
        # is the current completion event time.
        final_collection_summary = replace(

            raw_summary,

            physical_time_s=(
                float(
                    physical_finish_time_s
                )
            )
        )


        delivered = (
            final_collection_summary
            .delivered_pairs
        )


        # ========================================================
        # Progress display
        # ========================================================

        if (
            delivered == 1
            or
            delivered % progress_every == 0
            or
            final_collection_summary.collection_ready
        ):

            python_elapsed_s = (

                time.perf_counter()
                -
                python_wall_start
            )


            print(
                f"\n[COMPLETE {delivered:>4}] "
                f"physical={physical_finish_time_s:>9.4f} s | "
                f"lane={lane_id:>3} | "
                f"ZZ={final_collection_summary.zz_key_basis_bits:>4} | "
                f"XX={final_collection_summary.xx_parameter_estimation_bits:>3}"
            )


            if (
                final_collection_summary
                .xx_parameter_estimation_bits

                >=

                config
                .minimum_parameter_estimation_bits
            ):

                print(
                    "             "
                    f"Qx="
                    f"{final_collection_summary.observed_x_qber:.3%} | "
                    f"phase="
                    f"{final_collection_summary.phase_error_estimate:.3%} | "
                    f"forecast="
                    f"{final_collection_summary.forecast_extractable_bits} bits"
                )


            print(
                "             "
                f"in-flight={len(active_heap)} | "
                f"Python wall={python_elapsed_s/60.0:.2f} min | "
                f"new trajectories={newly_simulated_count} | "
                f"cached={cached_trajectory_count}"
            )


        # ========================================================
        # ACTUAL POST-PROCESSING GATE
        #
        # Forecast readiness merely permits us to TRY.
        # It cannot terminate the experiment.
        # ========================================================

        should_try_postprocessing = (

            final_collection_summary
            .collection_ready

            and

            (
                delivered
                -
                last_postprocess_attempt_at
            )
            >=
            postprocess_retry_every
        )


        if should_try_postprocessing:

            last_postprocess_attempt_at = (
                delivered
            )


            print(
                "\n[CANDIDATE] Forecast says the block may support",
                config.target_secret_bits,
                "secret bits."
            )


            print(
                "[POST] Running actual Cascade + leakage accounting "
                "+ Toeplitz privacy amplification..."
            )


            postprocess_rng = (
                phase3k_make_postprocess_rng(

                    delivered_pairs=(
                        delivered
                    ),

                    parallel_lanes=(
                        parallel_lanes
                    ),

                    target_secret_bits=(
                        config.target_secret_bits
                    )
                )
            )


            try:

                candidate_result = (
                    phase3k_distill_configurable_bbm92_key(

                        measurements=(
                            completed_measurements
                        ),

                        collection_summary=(
                            final_collection_summary
                        ),

                        config=(
                            config
                        ),

                        postprocess_rng=(
                            postprocess_rng
                        )
                    )
                )


            except RuntimeError as error:

                error_text = str(
                    error
                )


                if (
                    error_text.startswith(
                        "INSUFFICIENT_POST_EC_ENTROPY"
                    )
                    or
                    error_text.startswith(
                        "CASCADE_RECONCILIATION_FAILED"
                    )
                ):

                    print(
                        "[POST] Not enough usable key material yet:"
                    )

                    print(
                        "      ",
                        error_text
                    )

                    print(
                        "[POST] Continuing quantum delivery..."
                    )


                else:

                    raise


            else:

                final_session_result = (
                    candidate_result
                )


                print(
                    "\n[PASS] ACTUAL post-processing produced "
                    f"{config.target_secret_bits} secret bits."
                )


                # DO NOT launch another job.
                #
                # Key became available at THIS physical completion
                # event.
                break


        # ========================================================
        # No key yet.
        #
        # Work-conserving behavior:
        # immediately refill the lane that just became free.
        # ========================================================

        if (
            delivered
            >=
            max_pairs
        ):

            break


        if (
            next_pair_index
            <
            max_pairs
        ):

            (
                next_record,
                from_cache,
                pair_python_s
            ) = (
                phase3k_get_or_simulate_pair(
                    next_pair_index
                )
            )


            if from_cache:

                cached_trajectory_count += 1

            else:

                newly_simulated_count += 1


            trajectory_python_time_s += (
                pair_python_s
            )


            next_start_time_s = float(
                physical_finish_time_s
            )


            next_finish_time_s = (

                next_start_time_s

                +

                next_record
                .delivery_physical_time_s
            )


            heapq.heappush(

                active_heap,

                (
                    next_finish_time_s,
                    lane_id,
                    next_pair_index,
                    next_start_time_s,
                    next_record
                )
            )


            next_pair_index += 1


    # ============================================================
    # Final condition
    # ============================================================

    python_wall_time_s = (

        time.perf_counter()
        -
        python_wall_start
    )


    if (
        final_session_result
        is None
    ):

        raise RuntimeError(

            "Maximum delivered-pair budget was reached before "
            f"a {config.target_secret_bits}-bit post-processed "
            "secret key became available."
        )


    # ============================================================
    # Physical hardware accounting
    # ============================================================

    node_count = (

        PHASE3K_KEY_REPEATER_COUNT
        +
        2
    )


    elementary_link_count = (

        PHASE3K_KEY_REPEATER_COUNT
        +
        1
    )


    memory_slots_networkwide = (

        node_count

        *
        2

        *
        parallel_lanes
    )


    optical_modes_per_link = (

        PHASE3K_KEY_MODE_COUNT

        *
        parallel_lanes
    )


    optical_modes_networkwide = (

        elementary_link_count

        *
        optical_modes_per_link
    )


    bsm_units_networkwide = (

        PHASE3K_KEY_REPEATER_COUNT

        *
        parallel_lanes
    )


    # ============================================================
    # Note:
    #
    # At key completion, some other lanes may still contain
    # in-flight pair jobs. Their completed trajectories are NOT
    # included in the QKD transcript.
    #
    # This is correct for key timing.
    # ============================================================

    experiment_result = {

        "parallel_lanes":
            parallel_lanes,

        "target_secret_bits":
            config.target_secret_bits,

        "key_bytes":
            len(
                final_session_result
                .alice_key_bytes
            ),

        "completed_pairs_used":
            len(
                completed_measurements
            ),

        "pair_jobs_launched":
            next_pair_index,

        "inflight_jobs_at_key_completion":
            len(
                active_heap
            ),

        "zz_raw_bits":
            final_session_result
            .zz_raw_key_bits,

        "xx_parameter_estimation_bits":
            final_session_result
            .xx_parameter_estimation_bits,

        "observed_x_qber":
            final_session_result
            .observed_x_qber,

        "phase_error_estimate":
            final_session_result
            .phase_error_estimate,

        "actual_ec_leakage_bits":
            final_session_result
            .ec_leakage_bits,

        "maximum_extractable_bits":
            final_session_result
            .maximum_extractable_bits,

        "final_secret_bits":
            final_session_result
            .final_secret_bits,

        "MODELED_KEY_ESTABLISHMENT_TIME_S":
            final_session_result
            .physical_time_s,

        "modeled_completed_pair_rate_per_s":
            (
                len(
                    completed_measurements
                )

                /
                final_session_result
                .physical_time_s
            ),

        "python_wall_time_s":
            python_wall_time_s,

        "newly_simulated_trajectories":
            newly_simulated_count,

        "cached_trajectories_used":
            cached_trajectory_count,

        "memory_slots_networkwide":
            memory_slots_networkwide,

        "optical_modes_per_physical_link":
            optical_modes_per_link,

        "optical_modes_networkwide":
            optical_modes_networkwide,

        "bsm_units_networkwide":
            bsm_units_networkwide
    }


    return {

        "experiment":
            experiment_result,

        "session_result":
            final_session_result,

        "collection_summary":
            final_collection_summary,

        "measurements":
            tuple(
                completed_measurements
            ),

        "completion_log":
            tuple(
                completion_log
            ),

        "inflight_heap":
            tuple(
                active_heap
            )
    }


print(
    "[PASS] Continuous asynchronous key-generation engine ready."
)

In [ ]:
# @title RUN: Async Network -> Post Processing -> AES Key

print(
    "\n" + "=" * 104
)

print(
    "STARTING COMPLETE NARAD-QC SECRET-KEY EXPERIMENT"
)

print(
    "=" * 104
)


PHASE3K_KEY_EXPERIMENT = (
    phase3k_run_async_until_secret_key(

        parallel_lanes=(
            PHASE3K_KEY_PARALLEL_LANES
        ),

        config=(
            PHASE3K_KEY_QKD_CONFIG
        ),

        progress_every=(
            PHASE3K_KEY_PROGRESS_EVERY
        ),

        postprocess_retry_every=(
            PHASE3K_KEY_POSTPROCESS_RETRY_EVERY
        )
    )
)


PHASE3K_FINAL_SESSION = (

    PHASE3K_KEY_EXPERIMENT[
        "session_result"
    ]
)


NARAD_ASYNC_SECRET_KEY_ALICE = (

    PHASE3K_FINAL_SESSION
    .alice_key_bytes
)


NARAD_ASYNC_SECRET_KEY_BOB = (

    PHASE3K_FINAL_SESSION
    .bob_key_bytes
)


NARAD_ASYNC_SECRET_KEY = (
    NARAD_ASYNC_SECRET_KEY_ALICE
)


assert (

    NARAD_ASYNC_SECRET_KEY_ALICE

    ==

    NARAD_ASYNC_SECRET_KEY_BOB
)


assert (

    len(
        NARAD_ASYNC_SECRET_KEY
    )

    *

    8

    ==

    PHASE3K_KEY_TARGET_SECRET_BITS
)


# ================================================================
# REAL AES-GCM USABILITY TEST
# ================================================================

phase3k_key_test_message = (

    f"NARAD-QC async {PHASE3K_KEY_PARALLEL_LANES}-lane "
    f"{PHASE3K_KEY_TARGET_SECRET_BITS}-bit secret established."

).encode(
    "utf-8"
)


phase3k_key_test_ciphertext = (
    aes_gcm_encrypt(

        data=(
            phase3k_key_test_message
        ),

        key=(
            NARAD_ASYNC_SECRET_KEY_ALICE
        )
    )
)


phase3k_key_test_plaintext = (
    aes_gcm_decrypt(

        blob=(
            phase3k_key_test_ciphertext
        ),

        key=(
            NARAD_ASYNC_SECRET_KEY_BOB
        )
    )
)


assert (

    phase3k_key_test_plaintext

    ==

    phase3k_key_test_message
)


# ================================================================
# Final experiment report
# ================================================================

phase3k_final_experiment_df = pd.DataFrame(
    [
        PHASE3K_KEY_EXPERIMENT[
            "experiment"
        ]
    ]
)


display(
    phase3k_final_experiment_df
)


print(
    "\n" + "=" * 104
)

print(
    "NARAD-QC ASYNCHRONOUS SECRET KEY ESTABLISHED"
)

print(
    "=" * 104
)


print(
    "\nParallel lanes:",
    PHASE3K_KEY_PARALLEL_LANES
)


print(
    "Final AES key:",
    PHASE3K_FINAL_SESSION.final_secret_bits,
    "bits /",
    len(
        NARAD_ASYNC_SECRET_KEY
    ),
    "bytes"
)


print(
    "Bell pairs physically completed:",
    PHASE3K_FINAL_SESSION
    .delivered_pairs
)


print(
    "ZZ raw-key bits:",
    PHASE3K_FINAL_SESSION
    .zz_raw_key_bits
)


print(
    "XX PE bits:",
    PHASE3K_FINAL_SESSION
    .xx_parameter_estimation_bits
)


print(
    "Observed X QBER:",
    f"{PHASE3K_FINAL_SESSION.observed_x_qber:.4%}"
)


print(
    "Phase-error estimate:",
    f"{PHASE3K_FINAL_SESSION.phase_error_estimate:.4%}"
)


print(
    "Actual Cascade leakage:",
    PHASE3K_FINAL_SESSION
    .ec_leakage_bits,
    "bits"
)


print(
    "Actual post-EC extractable budget:",
    PHASE3K_FINAL_SESSION
    .maximum_extractable_bits,
    "bits"
)


print(
    "\n*** MODELED PHYSICAL TIME TO USABLE KEY ***"
)


print(
    f"{PHASE3K_FINAL_SESSION.physical_time_s:.6f} seconds"
)


print(
    "\nPython wall time:"
)


print(
    f"{PHASE3K_KEY_EXPERIMENT['experiment']['python_wall_time_s']:.3f} seconds"
)


print(
    "\n[PASS] Parameter estimation passed."
)

print(
    "[PASS] Cascade reconciliation passed."
)

print(
    "[PASS] Actual EC leakage was deducted."
)

print(
    "[PASS] Privacy amplification produced the requested key."
)

print(
    "[PASS] Alice and Bob possess identical keys."
)

print(
    "[PASS] AES-GCM encryption/decryption succeeded."
)


print(
    f"\nSecret key not shown."
)


print(
    "\n" + "=" * 104
)

print(
    "EXPERIMENT COMPLETE"
)

print(
    "=" * 104
)

In [ ]:
# @title CPU Trajectory-Farm Benchmark Configuration

import os
import gc
import math
import time
import threading
import multiprocessing as mp

from dataclasses import (
    asdict,
    is_dataclass
)

import numpy as np
import pandas as pd
import psutil


# ================================================================
# BENCHMARK WORKLOAD
# ================================================================

PHASE3K_FARM_PAIR_COUNT = 48

# Start beyond the pair indices used in our previous key experiment.
PHASE3K_FARM_PAIR_START = 1_200

PHASE3K_FARM_PAIR_INDICES = tuple(

    range(
        PHASE3K_FARM_PAIR_START,
        PHASE3K_FARM_PAIR_START
        +
        PHASE3K_FARM_PAIR_COUNT
    )
)


# ================================================================
# WORKER COUNTS
#
# Machine:
#   6 physical cores
#   12 logical CPUs
#
# We explicitly test beyond the physical-core count to discover
# whether SMT / Hyper-Threading helps this workload.
# ================================================================

PHASE3K_FARM_WORKER_COUNTS = (
    1,
    6,
    8,
    12,
)


# ================================================================
# FROZEN PHYSICAL MODEL
# ================================================================

PHASE3K_FARM_DISTANCE_KM = 600.0

PHASE3K_FARM_REPEATER_COUNT = 6

PHASE3K_FARM_MODE_COUNT = 8

PHASE3K_FARM_Z_PROBABILITY = (
    PHASE3J_BIASED_BBM92_CONFIG
    .z_probability
)

PHASE3K_FARM_MAX_SLOTS_PER_PAIR = 20_000


# ================================================================
# Strict semantic-equivalence fields
#
# resource_id is deliberately excluded:
# object identity is not physics.
# ================================================================

PHASE3K_FARM_EXACT_FIELDS = (

    "pair_index",

    "delivery_start_slot",
    "delivery_end_slot",

    "elementary_generation_cycles",
    "optical_mode_attempts",
    "bsm_attempts",

    "alice_basis",
    "bob_basis",

    "basis_match",

    "alice_bit",
    "bob_bit",

    "bit_error",
    "consumed"
)


PHASE3K_FARM_FLOAT_FIELDS = (

    "delivery_start_physical_time_s",
    "delivery_end_physical_time_s",

    "delivery_physical_time_s",

    "fidelity",
    "visibility",
    "predicted_qber"
)


PHASE3K_FARM_FLOAT_ABS_TOL = 1e-15


# ================================================================
# Environment report
# ================================================================

physical_cpus = (
    psutil.cpu_count(
        logical=False
    )
)

logical_cpus = (
    psutil.cpu_count(
        logical=True
    )
)

memory = (
    psutil.virtual_memory()
)


print(
    "=" * 100
)

print(
    "CPU Trajectory-Farm Scaling Benchmark"
)

print(
    "=" * 100
)


print(
    "Physical CPUs :",
    physical_cpus
)

print(
    "Logical CPUs  :",
    logical_cpus
)

print(
    "Available RAM :",
    f"{memory.available / (1024**3):.2f} GiB"
)

print(
    "Pair workload :",
    len(
        PHASE3K_FARM_PAIR_INDICES
    )
)

print(
    "Pair IDs      :",
    (
        PHASE3K_FARM_PAIR_INDICES[0],
        PHASE3K_FARM_PAIR_INDICES[-1]
    )
)

print(
    "Worker sweep  :",
    PHASE3K_FARM_WORKER_COUNTS
)


assert (
    PHASE3K_FARM_PAIR_COUNT
    %
    12
    ==
    0
)


assert (
    hasattr(
        globals().get(
            "phase3k_simulate_one_fast_hardware_lane",
            None
        ),
        "__call__"
    )
)


print(
    "\n[PASS] Benchmark configuration ready."
)

In [ ]:
# @title  Persistent CPU Trajectory Farm

# ================================================================
# Worker-local state
# ================================================================

_PHASE3K_WORKER_THREAD_LIMITER = None


def phase3k_farm_worker_initializer():
    """
    Configure each worker as a single-threaded simulation process.

    Why?
    ----
    We want:

        6 worker processes
        ~= 6 occupied CPU cores

    NOT:

        6 Python workers
        x hidden BLAS/OpenMP threads
        = severe oversubscription.
    """

    global _PHASE3K_WORKER_THREAD_LIMITER


    os.environ[
        "OMP_NUM_THREADS"
    ] = "1"

    os.environ[
        "MKL_NUM_THREADS"
    ] = "1"

    os.environ[
        "OPENBLAS_NUM_THREADS"
    ] = "1"

    os.environ[
        "NUMEXPR_NUM_THREADS"
    ] = "1"


    # Existing NumPy thread pools may already have been initialized
    # before fork. threadpoolctl handles those as well.
    try:

        from threadpoolctl import (
            threadpool_limits
        )


        _PHASE3K_WORKER_THREAD_LIMITER = (
            threadpool_limits(
                limits=1
            )
        )


        _PHASE3K_WORKER_THREAD_LIMITER.__enter__()


    except Exception:

        _PHASE3K_WORKER_THREAD_LIMITER = None


def phase3k_farm_normalize_payload(
    result
):

    if isinstance(
        result,
        dict
    ):

        return dict(
            result
        )


    if is_dataclass(
        result
    ):

        return asdict(
            result
        )


    raise TypeError(

        "Trajectory simulator returned unsupported type: "
        f"{type(result)}"
    )


def phase3k_farm_worker(
    pair_index
):
    """
    Calculate ONE independent physical Bell-pair trajectory.

    The worker:
        - does not read the trajectory cache
        - does not write Google Drive
        - does not modify parent state
        - returns only a compact Python payload
    """

    pair_index = int(
        pair_index
    )


    wall_start = (
        time.perf_counter()
    )


    cpu_start = (
        time.process_time()
    )


    result = (
        phase3k_simulate_one_fast_hardware_lane(

            pair_index=(
                pair_index
            ),

            total_distance_km=(
                PHASE3K_FARM_DISTANCE_KM
            ),

            repeater_count=(
                PHASE3K_FARM_REPEATER_COUNT
            ),

            mode_count=(
                PHASE3K_FARM_MODE_COUNT
            ),

            z_probability=(
                PHASE3K_FARM_Z_PROBABILITY
            ),

            max_slots_per_pair=(
                PHASE3K_FARM_MAX_SLOTS_PER_PAIR
            )
        )
    )


    payload = (
        phase3k_farm_normalize_payload(
            result
        )
    )


    cpu_s = (

        time.process_time()
        -
        cpu_start
    )


    wall_s = (

        time.perf_counter()
        -
        wall_start
    )


    return {

        "pair_index":
            pair_index,

        "worker_pid":
            os.getpid(),

        "worker_cpu_s":
            float(
                cpu_s
            ),

        "worker_wall_s":
            float(
                wall_s
            ),

        "payload":
            payload
    }


# ================================================================
# Lightweight system monitor
# ================================================================

class NARADFarmSystemMonitor:

    def __init__(
        self,
        interval_s=0.25
    ):

        self.interval_s = float(
            interval_s
        )

        self.cpu_samples = []

        self.available_ram_samples_gib = []

        self._stop_event = (
            threading.Event()
        )

        self._thread = None


    def _run(
        self
    ):

        # Prime psutil's CPU measurement.
        psutil.cpu_percent(
            interval=None
        )


        while not self._stop_event.wait(
            self.interval_s
        ):

            self.cpu_samples.append(

                psutil.cpu_percent(
                    interval=None
                )
            )


            vm = (
                psutil.virtual_memory()
            )


            self.available_ram_samples_gib.append(

                vm.available
                /
                (1024**3)
            )


    def start(
        self
    ):

        self._thread = (
            threading.Thread(

                target=(
                    self._run
                ),

                daemon=True
            )
        )


        self._thread.start()


    def stop(
        self
    ):

        self._stop_event.set()


        if (
            self._thread
            is not None
        ):

            self._thread.join(
                timeout=2.0
            )


    def summary(
        self
    ):

        if self.cpu_samples:

            mean_cpu = float(
                np.mean(
                    self.cpu_samples
                )
            )

            peak_cpu = float(
                np.max(
                    self.cpu_samples
                )
            )

        else:

            mean_cpu = float(
                "nan"
            )

            peak_cpu = float(
                "nan"
            )


        if (
            self.available_ram_samples_gib
        ):

            minimum_available_ram = float(

                np.min(
                    self.available_ram_samples_gib
                )
            )

        else:

            minimum_available_ram = float(
                "nan"
            )


        return {

            "mean_system_cpu_percent":
                mean_cpu,

            "peak_system_cpu_percent":
                peak_cpu,

            "minimum_available_ram_gib":
                minimum_available_ram
        }


# ================================================================
# One complete worker-count benchmark
# ================================================================

def phase3k_run_trajectory_farm(
    pair_indices,
    workers,
    progress_every=8
):

    pair_indices = tuple(

        int(
            pair_index
        )

        for pair_index
        in pair_indices
    )


    workers = int(
        workers
    )


    if workers < 1:

        raise ValueError(
            "workers must be positive."
        )


    if (
        workers
        >
        psutil.cpu_count(
            logical=True
        )
    ):

        raise ValueError(

            "Requested more workers than "
            "available logical CPUs."
        )


    print(
        "\n" + "-" * 100
    )

    print(
        f"Starting trajectory farm: "
        f"W={workers}, "
        f"N={len(pair_indices)}"
    )

    print(
        "-" * 100
    )


    # Clean parent-side garbage before FORK.
    gc.collect()


    monitor = (
        NARADFarmSystemMonitor()
    )


    context = (
        mp.get_context(
            "fork"
        )
    )


    results = []


    wall_start = (
        time.perf_counter()
    )


    monitor.start()


    try:

        # --------------------------------------------------------
        # Pool workers are persistent.
        #
        # chunksize=1 gives dynamic load balancing.
        # --------------------------------------------------------

        with context.Pool(

            processes=(
                workers
            ),

            initializer=(
                phase3k_farm_worker_initializer
            )

        ) as pool:


            iterator = (
                pool.imap_unordered(

                    phase3k_farm_worker,

                    pair_indices,

                    chunksize=1
                )
            )


            for completed_count, result in enumerate(
                iterator,
                start=1
            ):

                results.append(
                    result
                )


                if (
                    completed_count == 1
                    or
                    completed_count % progress_every == 0
                    or
                    completed_count == len(
                        pair_indices
                    )
                ):

                    elapsed_s = (

                        time.perf_counter()
                        -
                        wall_start
                    )


                    rate = (

                        completed_count
                        /
                        elapsed_s
                    )


                    print(

                        f"[W={workers:>2}] "
                        f"{completed_count:>2}/"
                        f"{len(pair_indices)} complete | "
                        f"wall={elapsed_s:>8.2f}s | "
                        f"rate={rate:.3f} pairs/s"
                    )


    finally:

        monitor.stop()


    total_wall_s = (

        time.perf_counter()
        -
        wall_start
    )


    results.sort(

        key=lambda item:
            item[
                "pair_index"
            ]
    )


    # ============================================================
    # Sanity: exactly one result per requested pair.
    # ============================================================

    returned_indices = tuple(

        result[
            "pair_index"
        ]

        for result
        in results
    )


    if (

        returned_indices

        !=

        tuple(
            sorted(
                pair_indices
            )
        )
    ):

        raise RuntimeError(

            "Trajectory farm did not return "
            "exactly one result per pair index."
        )


    total_worker_cpu_s = float(

        sum(

            result[
                "worker_cpu_s"
            ]

            for result
            in results
        )
    )


    # Number of fully occupied CPU cores represented by the
    # measured process CPU time.
    effective_busy_cores = (

        total_worker_cpu_s
        /
        total_wall_s
    )


    monitor_summary = (
        monitor.summary()
    )


    worker_pids = tuple(

        sorted(

            set(

                result[
                    "worker_pid"
                ]

                for result
                in results
            )
        )
    )


    summary = {

        "workers":
            workers,

        "pairs":
            len(
                results
            ),

        "wall_time_s":
            total_wall_s,

        "pairs_per_s":

            len(
                results
            )
            /
            total_wall_s,

        "total_worker_cpu_s":
            total_worker_cpu_s,

        "effective_busy_cores":
            effective_busy_cores,

        "worker_processes_observed":
            len(
                worker_pids
            ),

        "mean_system_cpu_percent":
            monitor_summary[
                "mean_system_cpu_percent"
            ],

        "peak_system_cpu_percent":
            monitor_summary[
                "peak_system_cpu_percent"
            ],

        "minimum_available_ram_gib":
            monitor_summary[
                "minimum_available_ram_gib"
            ]
    }


    return (
        tuple(
            results
        ),
        summary
    )


print(
    "[PASS] Persistent trajectory-farm backend ready."
)

In [ ]:
# @title  Strict Trajectory Equivalence Checker

def phase3k_compare_trajectory_payloads(
    reference,
    candidate,
    abs_tol=PHASE3K_FARM_FLOAT_ABS_TOL
):

    differences = []


    # ============================================================
    # Exact discrete quantities
    # ============================================================

    for field_name in (
        PHASE3K_FARM_EXACT_FIELDS
    ):

        if (
            field_name
            not in reference
            or
            field_name
            not in candidate
        ):

            differences.append(

                (
                    field_name,
                    "MISSING_FIELD"
                )
            )

            continue


        if (

            reference[
                field_name
            ]

            !=

            candidate[
                field_name
            ]
        ):

            differences.append(

                (
                    field_name,

                    reference[
                        field_name
                    ],

                    candidate[
                        field_name
                    ]
                )
            )


    # ============================================================
    # Floating physical quantities
    # ============================================================

    for field_name in (
        PHASE3K_FARM_FLOAT_FIELDS
    ):

        if (
            field_name
            not in reference
            or
            field_name
            not in candidate
        ):

            differences.append(

                (
                    field_name,
                    "MISSING_FIELD"
                )
            )

            continue


        if not math.isclose(

            float(
                reference[
                    field_name
                ]
            ),

            float(
                candidate[
                    field_name
                ]
            ),

            rel_tol=0.0,

            abs_tol=(
                abs_tol
            )
        ):

            differences.append(

                (
                    field_name,

                    reference[
                        field_name
                    ],

                    candidate[
                        field_name
                    ]
                )
            )


    return tuple(
        differences
    )


def phase3k_validate_farm_against_reference(
    reference_by_index,
    candidate_results
):

    failures = []


    for result in candidate_results:

        pair_index = int(
            result[
                "pair_index"
            ]
        )


        candidate_payload = (
            result[
                "payload"
            ]
        )


        if (
            pair_index
            not in
            reference_by_index
        ):

            failures.append(

                (
                    pair_index,
                    "PAIR_NOT_IN_REFERENCE"
                )
            )

            continue


        differences = (
            phase3k_compare_trajectory_payloads(

                reference=(
                    reference_by_index[
                        pair_index
                    ]
                ),

                candidate=(
                    candidate_payload
                )
            )
        )


        if differences:

            failures.append(

                (
                    pair_index,
                    differences
                )
            )


    return tuple(
        failures
    )


print(
    "[PASS] Strict cross-worker equivalence checker ready."
)

In [ ]:
# @title RUN Full CPU Scaling Sweep

# print(
#     "=" * 110
# )

# print(
#     "NARAD-QC Bell-Pair Trajectory Farm — CPU Scaling Sweep"
# )

# print(
#     "=" * 110
# )


# PHASE3K_FARM_ALL_RESULTS = {}

# PHASE3K_FARM_BENCHMARK_ROWS = []

# PHASE3K_FARM_REFERENCE_BY_INDEX = None

# PHASE3K_FARM_BASELINE_WALL_S = None


# for workers in (
#     PHASE3K_FARM_WORKER_COUNTS
# ):

#     (
#         farm_results,
#         farm_summary
#     ) = (
#         phase3k_run_trajectory_farm(

#             pair_indices=(
#                 PHASE3K_FARM_PAIR_INDICES
#             ),

#             workers=(
#                 workers
#             ),

#             progress_every=8
#         )
#     )


#     # ============================================================
#     # W=1 becomes the golden physics reference.
#     # ============================================================

#     if (
#         workers
#         ==
#         1
#     ):

#         PHASE3K_FARM_REFERENCE_BY_INDEX = {

#             int(
#                 result[
#                     "pair_index"
#                 ]
#             ):
#             result[
#                 "payload"
#             ]

#             for result
#             in farm_results
#         }


#         PHASE3K_FARM_BASELINE_WALL_S = (

#             farm_summary[
#                 "wall_time_s"
#             ]
#         )


#         equivalence_failures = ()

#         physics_equivalent = True


#         print(
#             "\n[REFERENCE] W=1 golden trajectory set frozen."
#         )


#     else:

#         equivalence_failures = (
#             phase3k_validate_farm_against_reference(

#                 reference_by_index=(
#                     PHASE3K_FARM_REFERENCE_BY_INDEX
#                 ),

#                 candidate_results=(
#                     farm_results
#                 )
#             )
#         )


#         physics_equivalent = (

#             len(
#                 equivalence_failures
#             )

#             ==
#             0
#         )


#         if not physics_equivalent:

#             print(
#                 "\n[FAIL] Physics equivalence failed."
#             )

#             print(
#                 "First mismatch:"
#             )

#             print(
#                 equivalence_failures[0]
#             )


#             raise AssertionError(

#                 f"W={workers} changed at least one "
#                 "Bell-pair trajectory."
#             )


#         print(

#             f"\n[PASS] W={workers} reproduced all "
#             f"{PHASE3K_FARM_PAIR_COUNT} "
#             "reference trajectories."
#         )


#     # ============================================================
#     # Scaling metrics
#     # ============================================================

#     speedup = (

#         PHASE3K_FARM_BASELINE_WALL_S

#         /

#         farm_summary[
#             "wall_time_s"
#         ]
#     )


#     parallel_efficiency = (

#         speedup
#         /
#         workers
#     )


#     benchmark_row = {

#         **farm_summary,

#         "speedup_vs_w1":
#             speedup,

#         "parallel_efficiency":
#             parallel_efficiency,

#         "physics_equivalent":
#             physics_equivalent
#     }


#     PHASE3K_FARM_BENCHMARK_ROWS.append(
#         benchmark_row
#     )


#     PHASE3K_FARM_ALL_RESULTS[
#         workers
#     ] = farm_results


#     print(
#         "\nRESULT"
#     )

#     print(
#         "Workers:",
#         workers
#     )

#     print(
#         "Wall time:",
#         f"{farm_summary['wall_time_s']:.3f} s"
#     )

#     print(
#         "Throughput:",
#         f"{farm_summary['pairs_per_s']:.4f} pairs/s"
#     )

#     print(
#         "Speed-up vs W=1:",
#         f"{speedup:.3f}x"
#     )

#     print(
#         "Parallel efficiency:",
#         f"{parallel_efficiency:.2%}"
#     )

#     print(
#         "Effective busy CPU cores:",
#         f"{farm_summary['effective_busy_cores']:.2f}"
#     )

#     print(
#         "Mean total-machine CPU:",
#         f"{farm_summary['mean_system_cpu_percent']:.1f}%"
#     )

#     print(
#         "Peak total-machine CPU:",
#         f"{farm_summary['peak_system_cpu_percent']:.1f}%"
#     )

#     print(
#         "Minimum free RAM:",
#         f"{farm_summary['minimum_available_ram_gib']:.2f} GiB"
#     )


#     print(
#         "\n" + "=" * 110
#     )


# PHASE3K_FARM_BENCHMARK_DF = (
#     pd.DataFrame(
#         PHASE3K_FARM_BENCHMARK_ROWS
#     )
# )


# display(

#     PHASE3K_FARM_BENCHMARK_DF[
#         [
#             "workers",
#             "pairs",
#             "wall_time_s",
#             "pairs_per_s",
#             "speedup_vs_w1",
#             "parallel_efficiency",
#             "effective_busy_cores",
#             "mean_system_cpu_percent",
#             "peak_system_cpu_percent",
#             "minimum_available_ram_gib",
#             "physics_equivalent"
#         ]
#     ]
# )


# print(
#     "\n[PASS] ALL REQUESTED CPU CONFIGURATIONS "
#     "PRESERVED THE BELL-PAIR PHYSICS."
# )

In [ ]:
# @title  Select Fastest Valid Trajectory-Farm Configuration

# valid_results = (

#     PHASE3K_FARM_BENCHMARK_DF[
#         PHASE3K_FARM_BENCHMARK_DF[
#             "physics_equivalent"
#         ]
#     ]
#     .copy()
# )


# best_index = (

#     valid_results[
#         "pairs_per_s"
#     ]
#     .idxmax()
# )


# PHASE3K_OPTIMAL_CPU_WORKERS = int(

#     valid_results
#     .loc[
#         best_index,
#         "workers"
#     ]
# )


# best_row = (

#     valid_results
#     .loc[
#         best_index
#     ]
# )


# baseline_row = (

#     valid_results[
#         valid_results[
#             "workers"
#         ]
#         ==
#         1
#     ]
#     .iloc[
#         0
#     ]
# )


# print(
#     "=" * 100
# )

# print(
#     "NARAD-QC TRAJECTORY FARM — BEST CPU CONFIGURATION"
# )

# print(
#     "=" * 100
# )


# print(
#     "Recommended CPU workers:",
#     PHASE3K_OPTIMAL_CPU_WORKERS
# )


# print(
#     "W=1 throughput:",
#     f"{baseline_row['pairs_per_s']:.4f} pairs/s"
# )


# print(
#     "Best throughput:",
#     f"{best_row['pairs_per_s']:.4f} pairs/s"
# )


# print(
#     "Measured speed-up:",
#     f"{best_row['speedup_vs_w1']:.3f}x"
# )


# print(
#     "Parallel efficiency:",
#     f"{best_row['parallel_efficiency']:.2%}"
# )


# print(
#     "Effective occupied cores:",
#     f"{best_row['effective_busy_cores']:.2f}"
# )


# print(
#     "Mean machine CPU utilisation:",
#     f"{best_row['mean_system_cpu_percent']:.1f}%"
# )


# print(
#     "Minimum available RAM:",
#     f"{best_row['minimum_available_ram_gib']:.2f} GiB"
# )


# assert (
#     bool(
#         best_row[
#             "physics_equivalent"
#         ]
#     )
# )


# print(
#     "\n[PASS] Best-performing configuration "
#     "preserves every validated trajectory."
# )


# print(
#     "\nNEXT BACKEND SETTING:"
# )


# print(
#     "PHASE3K_OPTIMAL_CPU_WORKERS =",
#     PHASE3K_OPTIMAL_CPU_WORKERS
# )


# print(
#     "=" * 100
# )

In [ ]:
# @title Fresh 4-Worker Bell-Pair Trajectory Provider

import os
import gc
import math
import time
import multiprocessing as mp

from dataclasses import (
    dataclass,
    asdict,
    is_dataclass,
    replace
)

import numpy as np
import pandas as pd
import psutil


# ================================================================
# FROZEN END-TO-END EXPERIMENT
# ================================================================

PHASE3K6_PARALLEL_QUANTUM_LANES = 50

PHASE3K6_CPU_WORKERS = 4

PHASE3K6_TARGET_SECRET_BITS = 128

PHASE3K6_LOOKAHEAD_TRAJECTORIES = 16

PHASE3K6_PROGRESS_EVERY = 16

PHASE3K6_POSTPROCESS_RETRY_EVERY = 8

PHASE3K6_MAX_DELIVERED_PAIRS = 5_000


# ================================================================
# Physics
# ================================================================

PHASE3K6_DISTANCE_KM = 600.0

PHASE3K6_REPEATER_COUNT = 6

PHASE3K6_MODE_COUNT = 8

PHASE3K6_MAX_SLOTS_PER_PAIR = 20_000


# ================================================================
# QKD configuration
# ================================================================

PHASE3K6_QKD_CONFIG = replace(

    PHASE3J_BIASED_BBM92_CONFIG,

    target_secret_bits=(
        PHASE3K6_TARGET_SECRET_BITS
    ),

    max_delivered_pairs=(
        PHASE3K6_MAX_DELIVERED_PAIRS
    )
)


# ================================================================
# API sanity checks
# ================================================================

required_callables = (
    "phase3k_simulate_one_fast_hardware_lane",
    "phase3j_bbm92_collection_summary",
    "phase3k_distill_configurable_bbm92_key",
    "phase3k_make_postprocess_rng",
    "aes_gcm_encrypt",
    "aes_gcm_decrypt"
)


for function_name in required_callables:

    function_object = globals().get(
        function_name
    )

    if not callable(
        function_object
    ):

        raise RuntimeError(
            f"Required function missing: {function_name}"
        )


# ================================================================
# Worker initializer
# ================================================================

_PHASE3K6_THREAD_LIMITER = None


def phase3k6_worker_initializer():

    global _PHASE3K6_THREAD_LIMITER


    # One simulation process = one CPU thread.
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"


    try:

        from threadpoolctl import (
            threadpool_limits
        )


        _PHASE3K6_THREAD_LIMITER = (
            threadpool_limits(
                limits=1
            )
        )


        _PHASE3K6_THREAD_LIMITER.__enter__()


    except Exception:

        _PHASE3K6_THREAD_LIMITER = None


def phase3k6_normalize_payload(
    result
):

    if isinstance(
        result,
        dict
    ):

        return dict(
            result
        )


    if is_dataclass(
        result
    ):

        return asdict(
            result
        )


    raise TypeError(
        f"Unsupported trajectory result: {type(result)}"
    )


def phase3k6_worker_simulate_pair(
    pair_index
):

    pair_index = int(
        pair_index
    )


    wall_start = (
        time.perf_counter()
    )


    cpu_start = (
        time.process_time()
    )


    result = (
        phase3k_simulate_one_fast_hardware_lane(

            pair_index=(
                pair_index
            ),

            total_distance_km=(
                PHASE3K6_DISTANCE_KM
            ),

            repeater_count=(
                PHASE3K6_REPEATER_COUNT
            ),

            mode_count=(
                PHASE3K6_MODE_COUNT
            ),

            z_probability=(
                PHASE3K6_QKD_CONFIG
                .z_probability
            ),

            max_slots_per_pair=(
                PHASE3K6_MAX_SLOTS_PER_PAIR
            )
        )
    )


    payload = (
        phase3k6_normalize_payload(
            result
        )
    )


    return {

        "pair_index":
            pair_index,

        "worker_pid":
            os.getpid(),

        "worker_cpu_s":
            float(
                time.process_time()
                -
                cpu_start
            ),

        "worker_wall_s":
            float(
                time.perf_counter()
                -
                wall_start
            ),

        "payload":
            payload
    }


# ================================================================
# Persistent trajectory farm
# ================================================================

class NARADFreshTrajectoryFarm:

    def __init__(
        self,
        workers=4,
        lookahead=16
    ):

        self.workers = int(
            workers
        )

        self.lookahead = int(
            lookahead
        )


        self.context = (
            mp.get_context(
                "fork"
            )
        )


        self.pool = (
            self.context.Pool(

                processes=(
                    self.workers
                ),

                initializer=(
                    phase3k6_worker_initializer
                )
            )
        )


        # pair_index -> AsyncResult
        self.pending = {}


        self.submitted_indices = set()


        self.submitted_count = 0

        self.retrieved_count = 0


        self.total_worker_cpu_s = 0.0

        self.total_worker_wall_s = 0.0

        self.total_parent_wait_s = 0.0


        self.worker_pids = set()


        self.closed = False


    def submit(
        self,
        pair_index
    ):

        pair_index = int(
            pair_index
        )


        if (
            pair_index
            in
            self.submitted_indices
        ):

            return


        async_result = (
            self.pool.apply_async(

                phase3k6_worker_simulate_pair,

                (
                    pair_index,
                )
            )
        )


        self.pending[
            pair_index
        ] = async_result


        self.submitted_indices.add(
            pair_index
        )


        self.submitted_count += 1


    def submit_through(
        self,
        final_pair_index
    ):

        final_pair_index = int(
            final_pair_index
        )


        if final_pair_index < 0:

            return


        start_index = (

            max(
                self.submitted_indices
            )
            +
            1

            if self.submitted_indices

            else 0
        )


        for pair_index in range(

            start_index,

            final_pair_index
            +
            1
        ):

            self.submit(
                pair_index
            )


    def maintain_window(
        self,
        next_physical_pair_index
    ):

        final_needed = (

            int(
                next_physical_pair_index
            )

            +
            self.lookahead

            -
            1
        )


        self.submit_through(
            final_needed
        )


    def get(
        self,
        pair_index
    ):

        pair_index = int(
            pair_index
        )


        if (
            pair_index
            not in
            self.submitted_indices
        ):

            self.submit(
                pair_index
            )


        wait_start = (
            time.perf_counter()
        )


        result = (
            self.pending[
                pair_index
            ]
            .get()
        )


        wait_s = (

            time.perf_counter()
            -
            wait_start
        )


        self.total_parent_wait_s += (
            wait_s
        )


        self.retrieved_count += 1


        self.total_worker_cpu_s += (
            result[
                "worker_cpu_s"
            ]
        )


        self.total_worker_wall_s += (
            result[
                "worker_wall_s"
            ]
        )


        self.worker_pids.add(

            result[
                "worker_pid"
            ]
        )


        del self.pending[
            pair_index
        ]


        payload = (
            result[
                "payload"
            ]
        )


        record = (
            NARADQKDPairMeasurement(
                **payload
            )
        )


        return record


    def snapshot(
        self
    ):

        return {

            "cpu_workers":
                self.workers,

            "compute_jobs_submitted":
                self.submitted_count,

            "trajectories_retrieved":
                self.retrieved_count,

            "compute_jobs_pending":
                len(
                    self.pending
                ),

            "worker_processes_seen":
                len(
                    self.worker_pids
                ),

            "total_worker_cpu_s":
                self.total_worker_cpu_s,

            "sum_worker_wall_s":
                self.total_worker_wall_s,

            "parent_waiting_for_trajectories_s":
                self.total_parent_wait_s
        }


    def shutdown(
        self
    ):

        if self.closed:

            return


        # We intentionally terminate speculative trajectories that
        # are irrelevant once the AES key has been established.
        self.pool.terminate()

        self.pool.join()


        self.closed = True


print(
    "=" * 100
)

print(
    "Fresh Parallel End-to-End Benchmark"
)

print(
    "=" * 100
)

print(
    "Quantum lanes :",
    PHASE3K6_PARALLEL_QUANTUM_LANES
)

print(
    "CPU workers   :",
    PHASE3K6_CPU_WORKERS
)

print(
    "Look-ahead    :",
    PHASE3K6_LOOKAHEAD_TRAJECTORIES
)

print(
    "Secret target :",
    PHASE3K6_TARGET_SECRET_BITS,
    "bits"
)

print(
    "Physical CPUs :",
    psutil.cpu_count(
        logical=False
    )
)

print(
    "Logical CPUs  :",
    psutil.cpu_count(
        logical=True
    )
)

print(
    "Free RAM      :",
    f"{psutil.virtual_memory().available / (1024**3):.2f} GiB"
)

print(
    "\n[PASS] Fresh trajectory provider ready."
)

In [ ]:
# @title  Farm-Backed 50-Lane BBM92 Session

@dataclass(frozen=True)
class NARADPhase3K6Completion:

    completion_index: int

    pair_index: int

    lane_id: int

    physical_start_time_s: float

    service_time_s: float

    physical_finish_time_s: float


def phase3k6_run_fresh_async_key_session():

    parallel_lanes = (
        PHASE3K6_PARALLEL_QUANTUM_LANES
    )

    config = (
        PHASE3K6_QKD_CONFIG
    )


    farm = (
        NARADFreshTrajectoryFarm(

            workers=(
                PHASE3K6_CPU_WORKERS
            ),

            lookahead=(
                PHASE3K6_LOOKAHEAD_TRAJECTORIES
            )
        )
    )


    python_wall_start = (
        time.perf_counter()
    )


    active_heap = []

    completed_measurements = []

    completion_log = []


    next_pair_index = 0


    final_session_result = None

    final_collection_summary = None


    last_postprocess_attempt_at = (
        -10**9
    )


    print(
        "\n" + "=" * 108
    )

    print(
        "NARAD-QC FRESH 50-LANE / 4-WORKER SECRET-KEY SESSION"
    )

    print(
        "=" * 108
    )


    print(
        "Quantum lanes:",
        parallel_lanes
    )

    print(
        "CPU trajectory workers:",
        PHASE3K6_CPU_WORKERS
    )

    print(
        "Target secret:",
        config.target_secret_bits,
        "bits"
    )

    print(
        "Basis bias:",
        f"Z={config.z_probability:.0%}, "
        f"X={1.0-config.z_probability:.0%}"
    )


    try:

        # ========================================================
        # INITIAL PHYSICAL SATURATION
        #
        # We need the service times for all 50 initially active
        # physical lanes before we know the first event.
        # ========================================================

        farm.submit_through(

            parallel_lanes

            +
            PHASE3K6_LOOKAHEAD_TRAJECTORIES

            -
            1
        )


        print(
            "\nComputing initial 50 simultaneous "
            "physical-lane trajectories..."
        )


        initial_start = (
            time.perf_counter()
        )


        for lane_id in range(
            parallel_lanes
        ):

            record = (
                farm.get(
                    next_pair_index
                )
            )


            start_time_s = 0.0


            finish_time_s = float(

                record
                .delivery_physical_time_s
            )


            heapq.heappush(

                active_heap,

                (
                    finish_time_s,
                    lane_id,
                    next_pair_index,
                    start_time_s,
                    record
                )
            )


            next_pair_index += 1


            farm.maintain_window(
                next_pair_index
            )


            if (
                next_pair_index % 10
                ==
                0
            ):

                elapsed = (

                    time.perf_counter()
                    -
                    initial_start
                )


                print(

                    f"  Initial lanes ready: "
                    f"{next_pair_index:>2}/50 | "
                    f"Python={elapsed:.1f}s"
                )


        print(
            "\n[PASS] All 50 physical lanes active at modeled t=0."
        )


        # ========================================================
        # CONTINUOUS DISCRETE-EVENT EXECUTION
        # ========================================================

        while active_heap:

            (
                physical_finish_time_s,
                lane_id,
                pair_index,
                physical_start_time_s,
                record
            ) = (
                heapq.heappop(
                    active_heap
                )
            )


            completed_measurements.append(
                record
            )


            completion_log.append(

                NARADPhase3K6Completion(

                    completion_index=(

                        len(
                            completed_measurements
                        )
                        -
                        1
                    ),

                    pair_index=(
                        pair_index
                    ),

                    lane_id=(
                        lane_id
                    ),

                    physical_start_time_s=(
                        physical_start_time_s
                    ),

                    service_time_s=(

                        record
                        .delivery_physical_time_s
                    ),

                    physical_finish_time_s=(
                        physical_finish_time_s
                    )
                )
            )


            raw_summary = (
                phase3j_bbm92_collection_summary(

                    measurements=(
                        completed_measurements
                    ),

                    config=(
                        config
                    )
                )
            )


            # Sequential helper originally SUMMED service time.
            # In our asynchronous system the correct elapsed network
            # time is the current physical completion event.
            collection_summary = replace(

                raw_summary,

                physical_time_s=(
                    float(
                        physical_finish_time_s
                    )
                )
            )


            final_collection_summary = (
                collection_summary
            )


            delivered = (
                collection_summary
                .delivered_pairs
            )


            # ====================================================
            # Progress
            # ====================================================

            if (
                delivered == 1
                or
                delivered
                %
                PHASE3K6_PROGRESS_EVERY
                ==
                0
                or
                collection_summary.collection_ready
            ):

                python_elapsed = (

                    time.perf_counter()
                    -
                    python_wall_start
                )


                farm_state = (
                    farm.snapshot()
                )


                print(

                    f"\n[COMPLETE {delivered:>4}] "
                    f"physical="
                    f"{physical_finish_time_s:>8.4f}s | "
                    f"ZZ="
                    f"{collection_summary.zz_key_basis_bits:>4} | "
                    f"XX="
                    f"{collection_summary.xx_parameter_estimation_bits:>3}"
                )


                if (

                    collection_summary
                    .xx_parameter_estimation_bits

                    >=

                    config
                    .minimum_parameter_estimation_bits
                ):

                    print(

                        "                "
                        f"Qx="
                        f"{collection_summary.observed_x_qber:.3%} | "
                        f"phase="
                        f"{collection_summary.phase_error_estimate:.3%} | "
                        f"forecast="
                        f"{collection_summary.forecast_extractable_bits} bits"
                    )


                print(

                    "                "
                    f"in-flight={len(active_heap):>2} | "
                    f"Python={python_elapsed/60.0:>6.2f} min | "
                    f"CPU-jobs submitted="
                    f"{farm_state['compute_jobs_submitted']} | "
                    f"retrieved="
                    f"{farm_state['trajectories_retrieved']}"
                )


            # ====================================================
            # ACTUAL POST-PROCESSING
            # ====================================================

            should_try_postprocessing = (

                collection_summary
                .collection_ready

                and

                (
                    delivered
                    -
                    last_postprocess_attempt_at
                )

                >=

                PHASE3K6_POSTPROCESS_RETRY_EVERY
            )


            if should_try_postprocessing:

                last_postprocess_attempt_at = (
                    delivered
                )


                print(
                    "\n[CANDIDATE] Forecast permits actual "
                    "post-processing."
                )


                postprocess_rng = (
                    phase3k_make_postprocess_rng(

                        delivered_pairs=(
                            delivered
                        ),

                        parallel_lanes=(
                            parallel_lanes
                        ),

                        target_secret_bits=(
                            config.target_secret_bits
                        )
                    )
                )


                try:

                    candidate = (
                        phase3k_distill_configurable_bbm92_key(

                            measurements=(
                                completed_measurements
                            ),

                            collection_summary=(
                                collection_summary
                            ),

                            config=(
                                config
                            ),

                            postprocess_rng=(
                                postprocess_rng
                            )
                        )
                    )


                except RuntimeError as error:

                    error_text = str(
                        error
                    )


                    if (
                        error_text.startswith(
                            "INSUFFICIENT_POST_EC_ENTROPY"
                        )

                        or

                        error_text.startswith(
                            "CASCADE_RECONCILIATION_FAILED"
                        )
                    ):

                        print(
                            "[POST] Candidate not yet usable:",
                            error_text
                        )


                    else:

                        raise


                else:

                    final_session_result = (
                        candidate
                    )


                    print(

                        "\n[PASS] Actual post-processing produced "
                        f"{config.target_secret_bits} secret bits."
                    )


                    break


            # ====================================================
            # WORK-CONSERVING REFILL
            #
            # Same physical lane becomes immediately busy again.
            # ====================================================

            if (
                next_pair_index
                >=
                config.max_delivered_pairs
            ):

                break


            next_record = (
                farm.get(
                    next_pair_index
                )
            )


            next_start_time_s = float(
                physical_finish_time_s
            )


            next_finish_time_s = (

                next_start_time_s

                +

                next_record
                .delivery_physical_time_s
            )


            heapq.heappush(

                active_heap,

                (
                    next_finish_time_s,
                    lane_id,
                    next_pair_index,
                    next_start_time_s,
                    next_record
                )
            )


            next_pair_index += 1


            farm.maintain_window(
                next_pair_index
            )


        python_wall_s = (

            time.perf_counter()
            -
            python_wall_start
        )


        if (
            final_session_result
            is None
        ):

            raise RuntimeError(

                "Fresh farm-backed session exhausted its "
                "pair budget before establishing the key."
            )


        farm_metrics = (
            farm.snapshot()
        )


        # ========================================================
        # Hardware provisioning
        # ========================================================

        node_count = (

            PHASE3K6_REPEATER_COUNT
            +
            2
        )


        elementary_links = (

            PHASE3K6_REPEATER_COUNT
            +
            1
        )


        memory_slots_networkwide = (

            node_count
            *
            2
            *
            parallel_lanes
        )


        optical_modes_per_link = (

            PHASE3K6_MODE_COUNT
            *
            parallel_lanes
        )


        optical_modes_networkwide = (

            optical_modes_per_link
            *
            elementary_links
        )


        bsm_units_networkwide = (

            PHASE3K6_REPEATER_COUNT
            *
            parallel_lanes
        )


        result = {

            "parallel_quantum_lanes":
                parallel_lanes,

            "cpu_workers":
                PHASE3K6_CPU_WORKERS,

            "target_secret_bits":
                config.target_secret_bits,

            "completed_pairs":
                final_session_result.delivered_pairs,

            "physical_pair_jobs_launched":
                next_pair_index,

            "physical_jobs_inflight_at_key":
                len(
                    active_heap
                ),

            "zz_bits":
                final_session_result.zz_raw_key_bits,

            "xx_bits":
                final_session_result.xx_parameter_estimation_bits,

            "observed_x_qber":
                final_session_result.observed_x_qber,

            "phase_error_estimate":
                final_session_result.phase_error_estimate,

            "actual_ec_leakage_bits":
                final_session_result.ec_leakage_bits,

            "maximum_extractable_bits":
                final_session_result.maximum_extractable_bits,

            "MODELED_KEY_TIME_S":
                final_session_result.physical_time_s,

            "PYTHON_WALL_TIME_S":
                python_wall_s,

            "REALTIME_FACTOR":
                (
                    python_wall_s
                    /
                    final_session_result.physical_time_s
                ),

            "modeled_pairs_per_s":
                (
                    final_session_result.delivered_pairs
                    /
                    final_session_result.physical_time_s
                ),

            "fresh_trajectories_retrieved":
                farm_metrics[
                    "trajectories_retrieved"
                ],

            "compute_jobs_submitted":
                farm_metrics[
                    "compute_jobs_submitted"
                ],

            "speculative_compute_jobs_at_stop":
                farm_metrics[
                    "compute_jobs_pending"
                ],

            "worker_processes_seen":
                farm_metrics[
                    "worker_processes_seen"
                ],

            "total_worker_cpu_s":
                farm_metrics[
                    "total_worker_cpu_s"
                ],

            "sum_worker_wall_s":
                farm_metrics[
                    "sum_worker_wall_s"
                ],

            "parent_waiting_for_trajectories_s":
                farm_metrics[
                    "parent_waiting_for_trajectories_s"
                ],

            "memory_slots_networkwide":
                memory_slots_networkwide,

            "optical_modes_per_link":
                optical_modes_per_link,

            "optical_modes_networkwide":
                optical_modes_networkwide,

            "bsm_units_networkwide":
                bsm_units_networkwide
        }


        return {

            "summary":
                result,

            "session_result":
                final_session_result,

            "collection_summary":
                final_collection_summary,

            "measurements":
                tuple(
                    completed_measurements
                ),

            "completion_log":
                tuple(
                    completion_log
                )
        }


    finally:

        farm.shutdown()


print(
    "[PASS] Fresh farm-backed BBM92 runner ready."
)

In [ ]:
# @title RUN Fresh 50-Lane / 4-Worker AES-128 Experiment

# print(
#     "\n" + "=" * 110
# )

# print(
#     "STARTING FRESH NARAD-QC END-TO-END PERFORMANCE RUN"
# )

# print(
#     "=" * 110
# )


# PHASE3K6_RESULT = (
#     phase3k6_run_fresh_async_key_session()
# )


# PHASE3K6_SESSION = (

#     PHASE3K6_RESULT[
#         "session_result"
#     ]
# )


# PHASE3K6_SECRET_ALICE = (

#     PHASE3K6_SESSION
#     .alice_key_bytes
# )


# PHASE3K6_SECRET_BOB = (

#     PHASE3K6_SESSION
#     .bob_key_bytes
# )


# assert (
#     PHASE3K6_SECRET_ALICE
#     ==
#     PHASE3K6_SECRET_BOB
# )


# assert (
#     len(
#         PHASE3K6_SECRET_ALICE
#     )
#     ==
#     16
# )


# # ================================================================
# # AES-GCM usability proof
# # ================================================================

# test_plaintext = (

#     b"NARAD-QC fresh trajectory-farm "
#     b"secret-key validation."
# )


# ciphertext = (
#     aes_gcm_encrypt(

#         data=(
#             test_plaintext
#         ),

#         key=(
#             PHASE3K6_SECRET_ALICE
#         )
#     )
# )


# recovered = (
#     aes_gcm_decrypt(

#         blob=(
#             ciphertext
#         ),

#         key=(
#             PHASE3K6_SECRET_BOB
#         )
#     )
# )


# assert (
#     recovered
#     ==
#     test_plaintext
# )


# # ================================================================
# # Report
# # ================================================================

# PHASE3K6_SUMMARY_DF = pd.DataFrame(
#     [
#         PHASE3K6_RESULT[
#             "summary"
#         ]
#     ]
# )


# display(
#     PHASE3K6_SUMMARY_DF
# )


# print(
#     "\n" + "=" * 110
# )

# print(
#     "FRESH 50-LANE / 4-WORKER NARAD-QC KEY ESTABLISHED"
# )

# print(
#     "=" * 110
# )


# summary = (
#     PHASE3K6_RESULT[
#         "summary"
#     ]
# )


# print(
#     "\nCompleted Bell pairs:",
#     summary[
#         "completed_pairs"
#     ]
# )


# print(
#     "Physical jobs launched:",
#     summary[
#         "physical_pair_jobs_launched"
#     ]
# )


# print(
#     "ZZ raw bits:",
#     summary[
#         "zz_bits"
#     ]
# )


# print(
#     "XX PE bits:",
#     summary[
#         "xx_bits"
#     ]
# )


# print(
#     "Observed X QBER:",
#     f"{summary['observed_x_qber']:.4%}"
# )


# print(
#     "Phase estimate:",
#     f"{summary['phase_error_estimate']:.4%}"
# )


# print(
#     "Cascade leakage:",
#     summary[
#         "actual_ec_leakage_bits"
#     ],
#     "bits"
# )


# print(
#     "Maximum extractable:",
#     summary[
#         "maximum_extractable_bits"
#     ],
#     "bits"
# )


# print(
#     "\nMODELED PHYSICAL KEY TIME:"
# )

# print(
#     f"{summary['MODELED_KEY_TIME_S']:.6f} s"
# )


# print(
#     "\nPYTHON WALL TIME:"
# )

# print(
#     f"{summary['PYTHON_WALL_TIME_S']:.3f} s"
# )


# print(
#     "\nREALTIME FACTOR:"
# )

# print(
#     f"{summary['REALTIME_FACTOR']:.2f}x"
# )


# print(
#     "\nFresh trajectory throughput:"
# )

# print(
#     f"{summary['fresh_trajectories_retrieved'] / summary['PYTHON_WALL_TIME_S']:.4f} "
#     "pairs/s"
# )


# print(
#     "Worker processes actually used:",
#     summary[
#         "worker_processes_seen"
#     ]
# )


# print(
#     "Parent time blocked waiting for trajectories:",
#     f"{summary['parent_waiting_for_trajectories_s']:.3f} s"
# )


# print(
#     "\n[PASS] Actual PE passed."
# )

# print(
#     "[PASS] Actual Cascade reconciliation passed."
# )

# print(
#     "[PASS] Actual leakage was deducted."
# )

# print(
#     "[PASS] Toeplitz privacy amplification passed."
# )

# print(
#     "[PASS] Alice == Bob."
# )

# print(
#     "[PASS] AES-GCM encryption/decryption passed."
# )


# print(
#     "\nSecret key intentionally NOT printed."
# )


# print(
#     "\n" + "=" * 110
# )

In [ ]:
# @title  Cached-vs-Fresh Physics Regression

# CACHED_GOLDEN_COMPLETED_PAIRS = 961

# CACHED_GOLDEN_PHYSICAL_TIME_S = (
#     4.362119
# )

# CACHED_GOLDEN_ZZ_BITS = 627

# CACHED_GOLDEN_XX_BITS = 42

# CACHED_GOLDEN_X_QBER = (
#     0.047619
# )

# CACHED_GOLDEN_PHASE_ERROR = (
#     0.057619
# )


# fresh = (
#     PHASE3K6_RESULT[
#         "summary"
#     ]
# )


# comparison = pd.DataFrame(
#     [
#         {
#             "Metric":
#                 "Completed Bell pairs",

#             "Cached golden":
#                 CACHED_GOLDEN_COMPLETED_PAIRS,

#             "Fresh 4-worker":
#                 fresh[
#                     "completed_pairs"
#                 ]
#         },

#         {
#             "Metric":
#                 "Modeled key time (s)",

#             "Cached golden":
#                 CACHED_GOLDEN_PHYSICAL_TIME_S,

#             "Fresh 4-worker":
#                 fresh[
#                     "MODELED_KEY_TIME_S"
#                 ]
#         },

#         {
#             "Metric":
#                 "ZZ bits",

#             "Cached golden":
#                 CACHED_GOLDEN_ZZ_BITS,

#             "Fresh 4-worker":
#                 fresh[
#                     "zz_bits"
#                 ]
#         },

#         {
#             "Metric":
#                 "XX bits",

#             "Cached golden":
#                 CACHED_GOLDEN_XX_BITS,

#             "Fresh 4-worker":
#                 fresh[
#                     "xx_bits"
#                 ]
#         },

#         {
#             "Metric":
#                 "Observed X QBER",

#             "Cached golden":
#                 CACHED_GOLDEN_X_QBER,

#             "Fresh 4-worker":
#                 fresh[
#                     "observed_x_qber"
#                 ]
#         },

#         {
#             "Metric":
#                 "Phase error",

#             "Cached golden":
#                 CACHED_GOLDEN_PHASE_ERROR,

#             "Fresh 4-worker":
#                 fresh[
#                     "phase_error_estimate"
#                 ]
#         }
#     ]
# )


# display(
#     comparison
# )


# assert (
#     fresh[
#         "completed_pairs"
#     ]
#     ==
#     CACHED_GOLDEN_COMPLETED_PAIRS
# )


# assert (
#     fresh[
#         "zz_bits"
#     ]
#     ==
#     CACHED_GOLDEN_ZZ_BITS
# )


# assert (
#     fresh[
#         "xx_bits"
#     ]
#     ==
#     CACHED_GOLDEN_XX_BITS
# )


# assert math.isclose(

#     fresh[
#         "MODELED_KEY_TIME_S"
#     ],

#     CACHED_GOLDEN_PHYSICAL_TIME_S,

#     rel_tol=0.0,

#     abs_tol=1e-6
# )


# print(
#     "\n[PASS] Fresh multiprocessing run reproduced "
#     "the cached 50-lane physical experiment."
# )


# print(
#     "[PASS] CPU concurrency changed computational time, "
#     "not modeled quantum-network behavior."
# )

## Phase 4 : 13-10-2026T15:20:18 --

### Phase 4A : 14-10-2026T12:00:31 -- 14-10-2026T12:55:52

Create the testbed

In [ ]:
# @title Operator-Controlled Routing Hardware Profiles

from dataclasses import dataclass
import math
import numbers
import pandas as pd


# ================================================================
# Operator-facing node roles
# ================================================================

PHASE4_ROLE_USER = "USER"
PHASE4_ROLE_REPEATER = "REPEATER"

PHASE4_VALID_NODE_ROLES = {
    PHASE4_ROLE_USER,
    PHASE4_ROLE_REPEATER
}


# ================================================================
# Phase-4 graph errors
# ================================================================

class Phase4GraphError(Exception):
    """
    Base exception for the operator-controlled Phase-4 graph.
    """
    pass


class Phase4NodeProfileError(Phase4GraphError):
    pass


class Phase4LinkProfileError(Phase4GraphError):
    pass


class Phase4OperatorConsistencyError(Phase4GraphError):
    pass


# ================================================================
# Node routing / hardware profile
# ================================================================

@dataclass
class Phase4NodeProfile:
    """
    Operator-visible hardware/control profile for one physical node.

    IMPORTANT
    ---------
    This object does NOT replace NetworkNode.

    NetworkNode remains the immutable physical identity.

    This sidecar stores Phase-4 resource properties that can later
    be consumed by routing.
    """

    node_name: str

    role: str

    active: bool

    # ------------------------------------------------------------
    # Quantum memory
    # ------------------------------------------------------------

    memory_slots_total: int
    memory_slots_available: int

    # Exponential Werner-memory decay parameter:
    #
    #     w(t) = w0 * exp(-lambda * t)
    #
    memory_decay_rate_per_s: float

    # Optional hard physical storage lifetime.
    memory_hard_lifetime_s: float

    # ------------------------------------------------------------
    # Bell-state measurement resources
    #
    # USER nodes do not perform repeater swapping, so all three
    # BSM quantities must be zero for users.
    # ------------------------------------------------------------

    bsm_units_total: int
    bsm_units_available: int

    bsm_success_probability: float


    def __post_init__(self):

        # ========================================================
        # Identity
        # ========================================================

        if (
            not isinstance(self.node_name, str)
            or not self.node_name.strip()
        ):
            raise Phase4NodeProfileError(
                "node_name must be a non-empty string."
            )

        self.node_name = (
            self.node_name.strip()
        )


        # ========================================================
        # Role
        # ========================================================

        if self.role not in PHASE4_VALID_NODE_ROLES:

            raise Phase4NodeProfileError(
                "role must be USER or REPEATER."
            )


        if not isinstance(
            self.active,
            bool
        ):

            raise Phase4NodeProfileError(
                "active must be bool."
            )


        # ========================================================
        # Memory capacities
        # ========================================================

        for field_name, value in (

            (
                "memory_slots_total",
                self.memory_slots_total
            ),

            (
                "memory_slots_available",
                self.memory_slots_available
            )

        ):

            if (
                isinstance(value, bool)
                or not isinstance(
                    value,
                    numbers.Integral
                )
            ):

                raise Phase4NodeProfileError(
                    f"{field_name} must be an integer."
                )


            if int(value) < 0:

                raise Phase4NodeProfileError(
                    f"{field_name} cannot be negative."
                )


        self.memory_slots_total = int(
            self.memory_slots_total
        )

        self.memory_slots_available = int(
            self.memory_slots_available
        )


        if (
            self.memory_slots_available
            >
            self.memory_slots_total
        ):

            raise Phase4NodeProfileError(
                "Available memory cannot exceed total memory."
            )


        # Repeater swapping requires at least two memory slots.
        if (
            self.role == PHASE4_ROLE_REPEATER
            and self.memory_slots_total < 2
        ):

            raise Phase4NodeProfileError(
                "A repeater requires at least "
                "2 quantum-memory slots."
            )


        # ========================================================
        # Memory coherence parameters
        # ========================================================

        for field_name, value in (

            (
                "memory_decay_rate_per_s",
                self.memory_decay_rate_per_s
            ),

            (
                "memory_hard_lifetime_s",
                self.memory_hard_lifetime_s
            )

        ):

            if (
                isinstance(value, bool)
                or not isinstance(
                    value,
                    numbers.Real
                )
                or not math.isfinite(
                    float(value)
                )
            ):

                raise Phase4NodeProfileError(
                    f"{field_name} must be a finite real number."
                )


            if float(value) < 0:

                raise Phase4NodeProfileError(
                    f"{field_name} cannot be negative."
                )


        self.memory_decay_rate_per_s = float(
            self.memory_decay_rate_per_s
        )

        self.memory_hard_lifetime_s = float(
            self.memory_hard_lifetime_s
        )


        # ========================================================
        # BSM hardware
        # ========================================================

        for field_name, value in (

            (
                "bsm_units_total",
                self.bsm_units_total
            ),

            (
                "bsm_units_available",
                self.bsm_units_available
            )

        ):

            if (
                isinstance(value, bool)
                or not isinstance(
                    value,
                    numbers.Integral
                )
            ):

                raise Phase4NodeProfileError(
                    f"{field_name} must be an integer."
                )


            if int(value) < 0:

                raise Phase4NodeProfileError(
                    f"{field_name} cannot be negative."
                )


        self.bsm_units_total = int(
            self.bsm_units_total
        )

        self.bsm_units_available = int(
            self.bsm_units_available
        )


        if (
            self.bsm_units_available
            >
            self.bsm_units_total
        ):

            raise Phase4NodeProfileError(
                "Available BSM units cannot exceed total BSM units."
            )


        if (
            isinstance(
                self.bsm_success_probability,
                bool
            )
            or not isinstance(
                self.bsm_success_probability,
                numbers.Real
            )
        ):

            raise Phase4NodeProfileError(
                "bsm_success_probability must be real."
            )


        self.bsm_success_probability = float(
            self.bsm_success_probability
        )


        if not (
            0.0
            <=
            self.bsm_success_probability
            <=
            1.0
        ):

            raise Phase4NodeProfileError(
                "bsm_success_probability must lie in [0, 1]."
            )


        # ========================================================
        # USER / REPEATER semantic invariant
        # ========================================================

        if self.role == PHASE4_ROLE_USER:

            if (
                self.bsm_units_total != 0
                or self.bsm_units_available != 0
                or self.bsm_success_probability != 0.0
            ):

                raise Phase4NodeProfileError(
                    "USER nodes cannot own repeater BSM resources."
                )


# ================================================================
# Physical fibre routing profile
# ================================================================

@dataclass
class Phase4LinkProfile:
    """
    Operator-controlled resource profile for one physical fibre.

    The actual fibre object remains PhysicalQuantumLink.
    """

    link_id: str

    active: bool

    multiplexing_modes_total: int
    multiplexing_modes_available: int


    def __post_init__(self):

        if (
            not isinstance(self.link_id, str)
            or not self.link_id.strip()
        ):

            raise Phase4LinkProfileError(
                "link_id must be a non-empty string."
            )


        self.link_id = (
            self.link_id.strip()
        )


        if not isinstance(
            self.active,
            bool
        ):

            raise Phase4LinkProfileError(
                "active must be bool."
            )


        for field_name, value in (

            (
                "multiplexing_modes_total",
                self.multiplexing_modes_total
            ),

            (
                "multiplexing_modes_available",
                self.multiplexing_modes_available
            )

        ):

            if (
                isinstance(value, bool)
                or not isinstance(
                    value,
                    numbers.Integral
                )
            ):

                raise Phase4LinkProfileError(
                    f"{field_name} must be an integer."
                )


            if int(value) < 0:

                raise Phase4LinkProfileError(
                    f"{field_name} cannot be negative."
                )


        self.multiplexing_modes_total = int(
            self.multiplexing_modes_total
        )

        self.multiplexing_modes_available = int(
            self.multiplexing_modes_available
        )


        if (
            self.multiplexing_modes_available
            >
            self.multiplexing_modes_total
        ):

            raise Phase4LinkProfileError(
                "Available multiplexing modes cannot "
                "exceed provisioned modes."
            )


print(
    "[PASS] Phase-4 operator hardware profiles defined."
)

In [ ]:
# @title Operator-Controlled Quantum Network Builder

class OperatorQuantumNetwork:
    """
    Operator-facing construction layer for NARAD-QC Phase 4.

    Architecture
    ------------
    NetworkTopology:
        authoritative physical graph

    Phase4NodeProfile:
        routing-visible node resource state

    Phase4LinkProfile:
        routing-visible fibre resource state

    The builder does NOT yet:
        - discover paths;
        - score paths;
        - authenticate paths;
        - execute quantum delivery.
    """

    def __init__(
        self,
        name="NARAD-QC Phase-4 Network"
    ):

        self.topology = (
            NetworkTopology(
                name=name
            )
        )


        self._node_profiles = {}

        self._link_profiles = {}


    # ============================================================
    # USER creation
    # ============================================================

    def add_user(
        self,
        name,
        x_km,
        y_km,

        memory_slots=1,

        memory_decay_rate_per_s=0.1431,

        memory_hard_lifetime_s=5.0,

        active=True
    ):

        """
        Add an endpoint-only USER node.

        USER:
            endpoint_capable = True
            swapping_capable = False
        """

        node = NetworkNode(

            name=name,

            x_km=x_km,

            y_km=y_km,

            endpoint_capable=True,

            swapping_capable=False,

            memory_capacity=memory_slots
        )


        profile = Phase4NodeProfile(

            node_name=node.name,

            role=PHASE4_ROLE_USER,

            active=active,

            memory_slots_total=(
                memory_slots
            ),

            memory_slots_available=(
                memory_slots
            ),

            memory_decay_rate_per_s=(
                memory_decay_rate_per_s
            ),

            memory_hard_lifetime_s=(
                memory_hard_lifetime_s
            ),

            bsm_units_total=0,

            bsm_units_available=0,

            bsm_success_probability=0.0
        )


        # Add only after both objects have validated.
        self.topology.add_node(
            node
        )


        self._node_profiles[
            node.name
        ] = profile


        return node


    # ============================================================
    # REPEATER creation
    # ============================================================

    def add_repeater(
        self,
        name,
        x_km,
        y_km,

        memory_slots,

        bsm_units=1,

        bsm_success_probability=0.4,

        memory_decay_rate_per_s=0.1431,

        memory_hard_lifetime_s=5.0,

        active=True
    ):

        """
        Add a swapping-only REPEATER node.

        REPEATER:
            endpoint_capable = False
            swapping_capable = True
        """

        node = NetworkNode(

            name=name,

            x_km=x_km,

            y_km=y_km,

            endpoint_capable=False,

            swapping_capable=True,

            memory_capacity=memory_slots
        )


        profile = Phase4NodeProfile(

            node_name=node.name,

            role=PHASE4_ROLE_REPEATER,

            active=active,

            memory_slots_total=(
                memory_slots
            ),

            memory_slots_available=(
                memory_slots
            ),

            memory_decay_rate_per_s=(
                memory_decay_rate_per_s
            ),

            memory_hard_lifetime_s=(
                memory_hard_lifetime_s
            ),

            bsm_units_total=(
                bsm_units
            ),

            bsm_units_available=(
                bsm_units
            ),

            bsm_success_probability=(
                bsm_success_probability
            )
        )


        self.topology.add_node(
            node
        )


        self._node_profiles[
            node.name
        ] = profile


        return node


    # ============================================================
    # Operator-created fibre
    # ============================================================

    def connect_fiber(
        self,
        node_a_name,
        node_b_name,

        multiplexing_modes=8,

        attenuation_db_per_km=(
            PHASE3_DEFAULT_FIBER_ATTENUATION_DB_PER_KM
        ),

        active=True
    ):

        """
        Physically connect two existing nodes.

        The operator decides WHICH edges exist.

        Distance is still calculated by the validated Phase-3
        PhysicalQuantumLink from node coordinates.
        """

        node_a = (
            self.topology.get_node(
                node_a_name
            )
        )


        node_b = (
            self.topology.get_node(
                node_b_name
            )
        )


        link = PhysicalQuantumLink(

            node_a=node_a,

            node_b=node_b,

            attenuation_db_per_km=(
                attenuation_db_per_km
            )
        )


        profile = Phase4LinkProfile(

            link_id=(
                link.link_id
            ),

            active=active,

            multiplexing_modes_total=(
                multiplexing_modes
            ),

            multiplexing_modes_available=(
                multiplexing_modes
            )
        )


        self.topology.add_link(
            link
        )


        self._link_profiles[
            link.link_id
        ] = profile


        return link


    # ============================================================
    # Profile access
    # ============================================================

    def node_profile(
        self,
        node_name
    ):

        canonical_node = (
            self.topology.get_node(
                node_name
            )
        )


        return self._node_profiles[
            canonical_node.name
        ]


    def link_profile(
        self,
        node_a_name,
        node_b_name
    ):

        link = (
            self.topology.get_link(

                node_a_name,
                node_b_name
            )
        )


        return self._link_profiles[
            link.link_id
        ]


    # ============================================================
    # Operator inspection tables
    # ============================================================

    def node_table(
        self
    ):

        rows = []


        for node_name in sorted(
            self._node_profiles
        ):

            node = (
                self.topology.get_node(
                    node_name
                )
            )


            profile = (
                self._node_profiles[
                    node_name
                ]
            )


            rows.append(
                {

                    "node":
                        node.name,

                    "role":
                        profile.role,

                    "x_km":
                        node.x_km,

                    "y_km":
                        node.y_km,

                    "active":
                        profile.active,

                    "memory_total":
                        profile.memory_slots_total,

                    "memory_available":
                        profile.memory_slots_available,

                    "memory_decay_rate_per_s":
                        profile.memory_decay_rate_per_s,

                    "memory_hard_lifetime_s":
                        profile.memory_hard_lifetime_s,

                    "bsm_units_total":
                        profile.bsm_units_total,

                    "bsm_units_available":
                        profile.bsm_units_available,

                    "bsm_success_probability":
                        profile.bsm_success_probability
                }
            )


        return pd.DataFrame(
            rows
        )


    def link_table(
        self
    ):

        rows = []


        for link_id in sorted(
            self._link_profiles
        ):

            profile = (
                self._link_profiles[
                    link_id
                ]
            )


            link = (
                self.topology._links[
                    link_id
                ]
            )


            rows.append(
                {

                    "link_id":
                        link.link_id,

                    "node_a":
                        link.node_a.name,

                    "node_b":
                        link.node_b.name,

                    "distance_km":
                        link.distance_km,

                    "attenuation_db_per_km":
                        link.attenuation_db_per_km,

                    "survival_probability":
                        link.survival_probability,

                    "active":
                        profile.active,

                    "mux_total":
                        profile.multiplexing_modes_total,

                    "mux_available":
                        profile.multiplexing_modes_available
                }
            )


        return pd.DataFrame(
            rows
        )


print(
    "[PASS] OperatorQuantumNetwork builder defined."
)

In [ ]:
# @title Build First Operator-Controlled NARAD-QC Graph

phase4_network = (
    OperatorQuantumNetwork(
        name="NARAD-QC Phase-4 Routing Testbed"
    )
)


# ================================================================
# USERS
# ================================================================

phase4_network.add_user(

    name="Alice",

    x_km=0,
    y_km=0,

    memory_slots=2
)


phase4_network.add_user(

    name="Bob",

    x_km=300,
    y_km=0,

    memory_slots=2
)


# Charlie exists deliberately.
#
# Later:
# Alice -> ... -> Charlie -> ... -> Bob
#
# MUST be rejected as a route because Charlie is a USER and would
# appear internally.
phase4_network.add_user(

    name="Charlie",

    x_km=150,
    y_km=85,

    memory_slots=2
)


# ================================================================
# REPEATERS
# ================================================================

phase4_network.add_repeater(

    name="R1",

    x_km=55,
    y_km=20,

    memory_slots=8,

    bsm_units=2
)


phase4_network.add_repeater(

    name="R2",

    x_km=120,
    y_km=15,

    memory_slots=8,

    bsm_units=2
)


phase4_network.add_repeater(

    name="R3",

    x_km=205,
    y_km=18,

    memory_slots=8,

    bsm_units=2
)


phase4_network.add_repeater(

    name="R4",

    x_km=75,
    y_km=-55,

    memory_slots=12,

    bsm_units=3
)


phase4_network.add_repeater(

    name="R5",

    x_km=165,
    y_km=-65,

    memory_slots=12,

    bsm_units=3
)


phase4_network.add_repeater(

    name="R6",

    x_km=245,
    y_km=-45,

    memory_slots=12,

    bsm_units=3
)


# ================================================================
# OPERATOR-SELECTED FIBRE EDGES
#
# Nothing is auto-connected.
# Every cable below physically exists ONLY because the operator
# explicitly requests it.
# ================================================================

# Upper route:
#
# Alice -> R1 -> R2 -> R3 -> Bob

phase4_network.connect_fiber(
    "Alice",
    "R1",
    multiplexing_modes=8
)

phase4_network.connect_fiber(
    "R1",
    "R2",
    multiplexing_modes=8
)

phase4_network.connect_fiber(
    "R2",
    "R3",
    multiplexing_modes=8
)

phase4_network.connect_fiber(
    "R3",
    "Bob",
    multiplexing_modes=8
)


# Lower route:
#
# Alice -> R4 -> R5 -> R6 -> Bob

phase4_network.connect_fiber(
    "Alice",
    "R4",
    multiplexing_modes=16
)

phase4_network.connect_fiber(
    "R4",
    "R5",
    multiplexing_modes=16
)

phase4_network.connect_fiber(
    "R5",
    "R6",
    multiplexing_modes=16
)

phase4_network.connect_fiber(
    "R6",
    "Bob",
    multiplexing_modes=16
)


# Cross-links create additional legal candidate routes later.

phase4_network.connect_fiber(
    "R1",
    "R5",
    multiplexing_modes=6
)

phase4_network.connect_fiber(
    "R2",
    "R5",
    multiplexing_modes=10
)

phase4_network.connect_fiber(
    "R3",
    "R6",
    multiplexing_modes=12
)


# ================================================================
# Deliberately dangerous USER-transit possibility.
#
# Graph connectivity exists physically:
#
# Charlie -> R1
# Charlie -> R2
# Charlie -> R3
#
# But Charlie must NEVER be accepted as an internal routing node.
# ================================================================

phase4_network.connect_fiber(
    "R2",
    "Charlie",
    multiplexing_modes=8
)

phase4_network.connect_fiber(
    "Charlie",
    "R3",
    multiplexing_modes=8
)

phase4_network.connect_fiber(
    "Charlie",
    "R1",
    multiplexing_modes=8
)

print(
    "[PASS] Operator-defined Phase-4 physical graph created."
)

In [ ]:
# @title  Inspect Operator-Controlled Network

print(
    "=" * 100
)

print(
    "NARAD-QC PHASE 4 — OPERATOR-CONTROLLED NETWORK"
)

print(
    "=" * 100
)


print(
    "\nNODES"
)

display(
    phase4_network.node_table()
)


print(
    "\nPHYSICAL FIBRE LINKS"
)

display(
    phase4_network.link_table()
)


print(
    "\nTopology name:"
)

print(
    phase4_network.topology.name
)


print(
    "\n[PASS] Operator network inspection complete."
)

In [ ]:
# @title Structural Graph Verification

print(
    "=" * 100
)

print(
    "PHASE 4A STRUCTURAL VERIFICATION"
)

print(
    "=" * 100
)


# ================================================================
# Role invariants
# ================================================================

for node_name, profile in (
    phase4_network
    ._node_profiles
    .items()
):

    node = (
        phase4_network
        .topology
        .get_node(
            node_name
        )
    )


    if profile.role == PHASE4_ROLE_USER:

        assert (
            node.endpoint_capable
            is True
        )

        assert (
            node.swapping_capable
            is False
        )

        assert (
            profile.bsm_units_total
            ==
            0
        )


    elif (
        profile.role
        ==
        PHASE4_ROLE_REPEATER
    ):

        assert (
            node.endpoint_capable
            is False
        )

        assert (
            node.swapping_capable
            is True
        )

        assert (
            profile.memory_slots_total
            >=
            2
        )

        assert (
            profile.bsm_units_total
            >=
            1
        )


print(
    "[PASS] USER and REPEATER semantics are disjoint."
)


# ================================================================
# Physical-link/profile consistency
# ================================================================

for link_id, profile in (
    phase4_network
    ._link_profiles
    .items()
):

    assert (
        link_id
        in
        phase4_network
        .topology
        ._links
    )

    assert (
        profile.link_id
        ==
        link_id
    )

    assert (
        profile.multiplexing_modes_available
        <=
        profile.multiplexing_modes_total
    )


print(
    "[PASS] Every routing link profile maps to one "
    "authoritative physical fibre."
)


# ================================================================
# Deliberate user-transit structure exists.
#
# We are NOT rejecting a path yet because path discovery does not
# belong to Phase 4A.1.
#
# We merely prove Charlie is physically connected such that our
# next routing test will have something dangerous to reject.
# ================================================================

assert (
    phase4_network
    .topology
    .contains_link(
        "R2",
        "Charlie"
    )
)


assert (
    phase4_network
    .topology
    .contains_link(
        "Charlie",
        "R3"
    )
)


assert (
    phase4_network
    .node_profile(
        "Charlie"
    )
    .role
    ==
    PHASE4_ROLE_USER
)


print(
    "[PASS] Deliberate USER-transit trap exists "
    "for the next path-validation stage."
)


print(
    "\n" + "=" * 100
)

print(
    "PHASE 4A PASSED"
)

print(
    "=" * 100
)

### Phase 4B : 14-10-2026T12:56:2026 -- 14-10-2026T14:03:41

Validate paths

In [ ]:
# @title  Quantum Path Discovery Types

from dataclasses import dataclass
import pandas as pd


# ================================================================
# Routing errors
# ================================================================

class Phase4PathDiscoveryError(Phase4GraphError):
    """
    Base exception for Phase-4 path discovery.
    """
    pass


class Phase4InvalidEndpointError(
    Phase4PathDiscoveryError
):
    pass


class Phase4NoValidPathError(
    Phase4PathDiscoveryError
):
    pass


# ================================================================
# Rejected-path audit record
# ================================================================

@dataclass(frozen=True)
class Phase4RejectedPath:

    node_names: tuple[str, ...]

    reason: str

    offending_user: str | None


# ================================================================
# Valid candidate path
# ================================================================

@dataclass(frozen=True)
class Phase4CandidatePath:

    path_index: int

    sender: str

    receiver: str

    node_names: tuple[str, ...]

    repeater_names: tuple[str, ...]

    link_ids: tuple[str, ...]

    hop_count: int

    repeater_count: int


    @property
    def display_path(self) -> str:

        return " -> ".join(
            self.node_names
        )


print(
    "[PASS] Phase-4 path discovery types defined."
)

In [ ]:
# @title Quantum Route Identification Helpers

def phase4_validate_routing_endpoint(
    network,
    node_name
):
    """
    Validate that a requested routing endpoint:

        1. physically exists;
        2. has a Phase-4 profile;
        3. is a USER;
        4. is active.

    Returns the canonical NetworkNode.
    """

    node = (
        network
        .topology
        .get_node(
            node_name
        )
    )


    profile = (
        network
        .node_profile(
            node.name
        )
    )


    if (
        profile.role
        !=
        PHASE4_ROLE_USER
    ):

        raise Phase4InvalidEndpointError(

            f"Routing endpoint '{node.name}' "
            "must be a USER node."
        )


    if not profile.active:

        raise Phase4InvalidEndpointError(

            f"Routing endpoint '{node.name}' "
            "is currently inactive."
        )


    return node


def phase4_find_internal_user(
    network,
    node_names
):
    """
    Return the first internal USER node in a supplied path.

    Endpoints are deliberately excluded.
    """

    node_names = tuple(
        node_names
    )


    if len(node_names) <= 2:

        return None


    for node_name in node_names[1:-1]:

        profile = (
            network
            .node_profile(
                node_name
            )
        )


        if (
            profile.role
            ==
            PHASE4_ROLE_USER
        ):

            return node_name


    return None


def phase4_is_repeater_only_internal_path(
    network,
    node_names
):
    """
    True iff every INTERNAL node is a REPEATER.
    """

    return (
        phase4_find_internal_user(
            network,
            node_names
        )
        is None
    )


print(
    "[PASS] Endpoint and internal-node semantics ready."
)

In [ ]:
# @title  Enumerate All Simple Physical Paths

def phase4_enumerate_simple_physical_paths(
    network,
    sender_name,
    receiver_name,
    max_hops=None
):
    """
    Enumerate every simple PHYSICAL path between two nodes.

    This function performs topology traversal only.

    It does NOT yet impose:
        - internal USER restrictions;
        - QKD quality checks;
        - resource feasibility;
        - routing scores;
        - authentication.

    DFS is deterministic because NetworkTopology.neighbours()
    returns neighbours in sorted name order.
    """

    sender = (
        phase4_validate_routing_endpoint(
            network,
            sender_name
        )
    )


    receiver = (
        phase4_validate_routing_endpoint(
            network,
            receiver_name
        )
    )


    if sender.name == receiver.name:

        raise Phase4InvalidEndpointError(
            "Sender and receiver must be different USER nodes."
        )


    if max_hops is not None:

        if (
            isinstance(max_hops, bool)
            or not isinstance(
                max_hops,
                int
            )
            or max_hops < 1
        ):

            raise ValueError(
                "max_hops must be a positive integer or None."
            )


    discovered_paths = []


    # ------------------------------------------------------------
    # path stores canonical node names.
    # ------------------------------------------------------------

    def dfs(
        current_name,
        path
    ):

        current_hops = (
            len(path)
            -
            1
        )


        if (
            max_hops is not None
            and current_hops > max_hops
        ):

            return


        if (
            current_name
            ==
            receiver.name
        ):

            validated = (
                network
                .topology
                .validate_path(
                    path
                )
            )


            discovered_paths.append(

                tuple(
                    node.name
                    for node in validated
                )
            )

            return


        current_node = (
            network
            .topology
            .get_node(
                current_name
            )
        )


        for neighbour in (
            network
            .topology
            .neighbours(
                current_node
            )
        ):

            neighbour_name = (
                neighbour.name
            )


            # Simple-path invariant:
            # never revisit a node.
            if (
                neighbour_name
                in
                path
            ):

                continue


            next_path = (

                path
                +
                (
                    neighbour_name,
                )
            )


            next_hops = (
                len(next_path)
                -
                1
            )


            if (
                max_hops is not None
                and next_hops > max_hops
            ):

                continue


            dfs(
                neighbour_name,
                next_path
            )


    dfs(
        sender.name,
        (
            sender.name,
        )
    )


    return tuple(
        discovered_paths
    )


print(
    "[PASS] Simple physical-path enumerator ready."
)

In [ ]:
# @title  Filter Repeater-Only Candidate Routes

def phase4_build_candidate_path(
    network,
    path_index,
    node_names
):

    node_names = tuple(
        node_names
    )


    sender = (
        node_names[0]
    )

    receiver = (
        node_names[-1]
    )


    repeater_names = tuple(

        node_name

        for node_name
        in node_names[1:-1]

        if (
            network
            .node_profile(
                node_name
            )
            .role

            ==
            PHASE4_ROLE_REPEATER
        )
    )


    link_ids = []


    for node_a, node_b in zip(
        node_names[:-1],
        node_names[1:]
    ):

        link = (
            network
            .topology
            .get_link(
                node_a,
                node_b
            )
        )


        link_ids.append(
            link.link_id
        )


    return Phase4CandidatePath(

        path_index=(
            int(
                path_index
            )
        ),

        sender=(
            sender
        ),

        receiver=(
            receiver
        ),

        node_names=(
            node_names
        ),

        repeater_names=(
            repeater_names
        ),

        link_ids=(
            tuple(
                link_ids
            )
        ),

        hop_count=(
            len(node_names)
            -
            1
        ),

        repeater_count=(
            len(
                repeater_names
            )
        )
    )


def phase4_discover_valid_quantum_paths(
    network,
    sender_name,
    receiver_name,
    max_hops=None
):
    """
    Discover all physical simple paths and divide them into:

        valid:
            sender USER
            internal REPEATERS only
            receiver USER

        rejected:
            contains another USER internally
    """

    physical_paths = (
        phase4_enumerate_simple_physical_paths(

            network=(
                network
            ),

            sender_name=(
                sender_name
            ),

            receiver_name=(
                receiver_name
            ),

            max_hops=(
                max_hops
            )
        )
    )


    valid_paths = []

    rejected_paths = []


    for node_names in physical_paths:

        offending_user = (
            phase4_find_internal_user(

                network,
                node_names
            )
        )


        if offending_user is not None:

            rejected_paths.append(

                Phase4RejectedPath(

                    node_names=(
                        tuple(
                            node_names
                        )
                    ),

                    reason=(
                        "INTERNAL_USER_NOT_ALLOWED"
                    ),

                    offending_user=(
                        offending_user
                    )
                )
            )

            continue


        valid_paths.append(

            phase4_build_candidate_path(

                network=(
                    network
                ),

                path_index=(
                    len(valid_paths)
                ),

                node_names=(
                    node_names
                )
            )
        )


    if not valid_paths:

        raise Phase4NoValidPathError(

            f"No repeater-only quantum route exists "
            f"between '{sender_name}' and "
            f"'{receiver_name}'."
        )


    return (
        tuple(
            valid_paths
        ),

        tuple(
            rejected_paths
        ),

        tuple(
            physical_paths
        )
    )


print(
    "[PASS] Repeater-only candidate-path filter ready."
)

In [ ]:
# @title  Discover Alice-to-Bob Quantum Routes

PHASE4_SENDER = "Alice"

PHASE4_RECEIVER = "Bob"


(
    PHASE4_VALID_PATHS,
    PHASE4_REJECTED_PATHS,
    PHASE4_ALL_PHYSICAL_PATHS
) = (
    phase4_discover_valid_quantum_paths(

        network=(
            phase4_network
        ),

        sender_name=(
            PHASE4_SENDER
        ),

        receiver_name=(
            PHASE4_RECEIVER
        )
    )
)


print(
    "=" * 100
)

print(
    "NARAD-QC PHASE 4B : PATH DISCOVERY"
)

print(
    "=" * 100
)


print(
    "\nSender:",
    PHASE4_SENDER
)

print(
    "Receiver:",
    PHASE4_RECEIVER
)


print(
    "\nAll physical simple paths:",
    len(
        PHASE4_ALL_PHYSICAL_PATHS
    )
)


print(
    "Valid repeater-only paths:",
    len(
        PHASE4_VALID_PATHS
    )
)


print(
    "Rejected USER-transit paths:",
    len(
        PHASE4_REJECTED_PATHS
    )
)


print(
    "\n[PASS] Alice-to-Bob path discovery completed."
)

In [ ]:
# @title Valid Candidate Path Table

valid_rows = []


for path in PHASE4_VALID_PATHS:

    valid_rows.append(
        {

            "path_index":
                path.path_index,

            "path":
                path.display_path,

            "hop_count":
                path.hop_count,

            "repeater_count":
                path.repeater_count,

            "repeaters":
                ", ".join(
                    path.repeater_names
                ),

            "link_ids":
                " | ".join(
                    path.link_ids
                )
        }
    )


PHASE4_VALID_PATH_DF = (
    pd.DataFrame(
        valid_rows
    )
)


display(
    PHASE4_VALID_PATH_DF
)


print(
    f"\n[PASS] {len(PHASE4_VALID_PATHS)} "
    "routing-valid candidate path(s) retained."
)

In [ ]:
# @title  Rejected USER-Transit Path Audit

rejected_rows = []


for index, rejected in enumerate(
    PHASE4_REJECTED_PATHS
):

    rejected_rows.append(
        {

            "rejected_index":
                index,

            "path":
                " -> ".join(
                    rejected.node_names
                ),

            "reason":
                rejected.reason,

            "offending_user":
                rejected.offending_user
        }
    )


PHASE4_REJECTED_PATH_DF = (
    pd.DataFrame(
        rejected_rows
    )
)


display(
    PHASE4_REJECTED_PATH_DF
)


assert (
    any(

        rejected.offending_user
        ==
        "Charlie"

        for rejected
        in PHASE4_REJECTED_PATHS
    )
)


print(
    "\n[PASS] Charlie was detected as an "
    "illegal internal USER."
)

In [ ]:
# @title Candidate Route Verification

print(
    "=" * 100
)

print(
    "PHASE 4B ROUTING-SEMANTIC VERIFICATION"
)

print(
    "=" * 100
)


# ================================================================
# Every retained path:
#
# USER -> REPEATER* -> USER
# ================================================================

for path in PHASE4_VALID_PATHS:

    # ------------------------------------------------------------
    # Endpoints must be the selected users.
    # ------------------------------------------------------------

    assert (
        path.node_names[0]
        ==
        PHASE4_SENDER
    )


    assert (
        path.node_names[-1]
        ==
        PHASE4_RECEIVER
    )


    assert (
        phase4_network
        .node_profile(
            path.node_names[0]
        )
        .role

        ==
        PHASE4_ROLE_USER
    )


    assert (
        phase4_network
        .node_profile(
            path.node_names[-1]
        )
        .role

        ==
        PHASE4_ROLE_USER
    )


    # ------------------------------------------------------------
    # Every internal node must be a repeater.
    # ------------------------------------------------------------

    for internal_name in (
        path.node_names[1:-1]
    ):

        assert (
            phase4_network
            .node_profile(
                internal_name
            )
            .role

            ==
            PHASE4_ROLE_REPEATER
        )


    # ------------------------------------------------------------
    # No repeated nodes.
    # ------------------------------------------------------------

    assert (
        len(
            set(
                path.node_names
            )
        )

        ==

        len(
            path.node_names
        )
    )


    # ------------------------------------------------------------
    # Every consecutive physical edge must exist.
    # ------------------------------------------------------------

    for node_a, node_b in zip(

        path.node_names[:-1],

        path.node_names[1:]
    ):

        assert (
            phase4_network
            .topology
            .contains_link(
                node_a,
                node_b
            )
        )


    # ------------------------------------------------------------
    # Re-use Phase-3 topology validator as a second independent
    # structural check.
    # ------------------------------------------------------------

    validated = (
        phase4_network
        .topology
        .validate_path(
            path.node_names
        )
    )


    assert (
        tuple(
            node.name
            for node in validated
        )

        ==

        path.node_names
    )


print(
    "[PASS] Every retained route is a simple "
    "physical path."
)


print(
    "[PASS] Every retained route has USER endpoints."
)


print(
    "[PASS] Every internal routing node is a REPEATER."
)


print(
    "[PASS] No intermediate USER survives candidate filtering."
)


# ================================================================
# Rejected path validation
# ================================================================

assert (
    len(
        PHASE4_REJECTED_PATHS
    )
    >
    0
)


for rejected in PHASE4_REJECTED_PATHS:

    assert (
        rejected.offending_user
        is not None
    )


    assert (
        phase4_network
        .node_profile(
            rejected.offending_user
        )
        .role

        ==
        PHASE4_ROLE_USER
    )


print(
    "[PASS] USER-transit rejection audit is valid."
)


print(
    "\n" + "=" * 100
)

print(
    "PHASE 4B PASSED"
)

print(
    "=" * 100
)

### Phase 4C : 14-10-2026T14:17:03 -- 14-10-2026T16:17:33

Metric Calculation

In [ ]:
# @title Candidate Path Metric Record

from dataclasses import dataclass
import math
import pandas as pd


@dataclass(frozen=True)
class Phase4PathMetrics:

    path_index: int

    path: str

    node_names: tuple[str, ...]

    # ------------------------------------------------------------
    # Topology
    # ------------------------------------------------------------

    total_distance_km: float

    hop_count: int

    repeater_count: int


    # ------------------------------------------------------------
    # Quantum-quality prediction
    #
    # "fresh" means:
    # no memory waiting penalty has yet been applied.
    # ------------------------------------------------------------

    predicted_werner_visibility: float

    predicted_fidelity: float

    predicted_qber: float


    # ------------------------------------------------------------
    # Bottleneck physical resources
    # ------------------------------------------------------------

    multiplexing_modes_available: int

    memory_slots_available: int | None

    bsm_units_available: int | None


    # ------------------------------------------------------------
    # BSM diagnostic
    #
    # Not independently weighted yet.
    # ------------------------------------------------------------

    bsm_success_product: float


    # ------------------------------------------------------------
    # Physical-service estimates
    # ------------------------------------------------------------

    parallel_generation_latency_proxy_s: float

    bsm_latency_proxy_s: float

    expected_pair_delivery_proxy_s: float


    # ------------------------------------------------------------
    # Memory-quality exposure
    #
    # Dimensionless:
    #
    #     sum(lambda_r) * T_delivery_proxy
    #
    # Conservative routing proxy, NOT exact physical ageing.
    # ------------------------------------------------------------

    memory_coherence_cost: float


    # ------------------------------------------------------------
    # Hard feasibility diagnostics
    # ------------------------------------------------------------

    all_nodes_active: bool

    all_links_active: bool

    multiplexing_feasible: bool

    memory_feasible: bool

    bsm_feasible: bool

    qkd_quality_feasible: bool

    statically_feasible: bool


print(
    "[PASS] Phase-4 candidate metric record defined."
)

In [ ]:
# @title Frozen Routing Metric Configuration

PHASE4_METRIC_QUALITY_CONFIG = (
    PHASE3H1_QUALITY_CONFIG
)

PHASE4_METRIC_TIMING_PROFILE = (
    PHASE3H3_TIMING_PROFILE
)


# ================================================================
# Sanity checks
# ================================================================

assert math.isclose(

    PHASE4_METRIC_QUALITY_CONFIG
    .qber_limit,

    0.11,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert (

    PHASE4_METRIC_QUALITY_CONFIG
    .elementary_fidelity

    >
    0.25
)


assert (

    PHASE4_METRIC_TIMING_PROFILE
    .fiber_group_velocity_km_per_s

    >
    0.0
)


print(
    "=" * 90
)

print(
    "PHASE 4 ROUTING METRIC MODEL"
)

print(
    "=" * 90
)


print(
    "Elementary fidelity:",
    PHASE4_METRIC_QUALITY_CONFIG
    .elementary_fidelity
)


print(
    "Elementary visibility:",
    PHASE4_METRIC_QUALITY_CONFIG
    .elementary_visibility
)


print(
    "Swap operation visibility:",
    PHASE4_METRIC_QUALITY_CONFIG
    .swap_operation_visibility
)


print(
    "QBER feasibility limit:",
    PHASE4_METRIC_QUALITY_CONFIG
    .qber_limit
)


print(
    "Fibre velocity:",
    PHASE4_METRIC_TIMING_PROFILE
    .fiber_group_velocity_km_per_s,
    "km/s"
)


print(
    "Atom-photon generation latency:",
    PHASE4_METRIC_TIMING_PROFILE
    .atom_photon_generation_time_s,
    "s"
)


print(
    "Atomic BSM latency:",
    PHASE4_METRIC_TIMING_PROFILE
    .atomic_bsm_latency_s,
    "s"
)


print(
    "\n[PASS] Phase-4 metric physics frozen."
)

In [ ]:
# @title  Elementary Link Routing Metrics

def phase4_link_metric_snapshot(
    network,
    node_a_name,
    node_b_name
):

    link = (
        network
        .topology
        .get_link(
            node_a_name,
            node_b_name
        )
    )


    profile = (
        network
        .link_profile(
            node_a_name,
            node_b_name
        )
    )


    # ============================================================
    # Single-mode physical survival probability
    # ============================================================

    p_single = float(
        link.survival_probability
    )


    modes_available = int(
        profile
        .multiplexing_modes_available
    )


    # ============================================================
    # Multiplexed cycle success probability
    #
    # Exact current Phase-3I abstraction:
    #
    # P_m = 1 - (1-p)^m
    # ============================================================

    if modes_available <= 0:

        p_cycle = 0.0

    else:

        p_cycle = (

            1.0
            -
            (
                1.0
                -
                p_single
            )
            **
            modes_available
        )


    # Numerical guard.
    p_cycle = min(
        max(
            float(
                p_cycle
            ),
            0.0
        ),
        1.0
    )


    # ============================================================
    # One physical generation opportunity
    # ============================================================

    attempt_cycle_s = (

        PHASE4_METRIC_TIMING_PROFILE
        .elementary_attempt_cycle_s(

            distance_km=(
                link.distance_km
            )
        )
    )


    # ============================================================
    # Expected elementary-link latency
    # ============================================================

    if p_cycle <= 0.0:

        expected_generation_latency_s = (
            math.inf
        )

    else:

        expected_generation_latency_s = (

            PHASE4_METRIC_TIMING_PROFILE
            .expected_generation_latency_s(

                distance_km=(
                    link.distance_km
                ),

                success_probability=(
                    p_cycle
                )
            )
        )


    return {

        "link_id":
            link.link_id,

        "distance_km":
            float(
                link.distance_km
            ),

        "attenuation_db_per_km":
            float(
                link.attenuation_db_per_km
            ),

        "single_mode_success_probability":
            p_single,

        "multiplexing_modes_available":
            modes_available,

        "multiplexed_cycle_success_probability":
            p_cycle,

        "attempt_cycle_s":
            float(
                attempt_cycle_s
            ),

        "expected_generation_latency_s":
            float(
                expected_generation_latency_s
            ),

        "active":
            bool(
                profile.active
            )
    }


print(
    "[PASS] Elementary-link metric calculator ready."
)

In [ ]:
# @title Fresh Path Werner Quality Predictor

def phase4_predict_fresh_path_quality(
    candidate_path
):

    link_count = int(
        candidate_path.hop_count
    )


    repeater_count = int(
        candidate_path.repeater_count
    )


    if link_count < 1:

        raise ValueError(
            "Candidate path must contain at least one link."
        )


    w_elementary = float(

        PHASE4_METRIC_QUALITY_CONFIG
        .elementary_visibility
    )


    q_swap = float(

        PHASE4_METRIC_QUALITY_CONFIG
        .swap_operation_visibility
    )


    # ============================================================
    # Fresh end-to-end Werner visibility
    # ============================================================

    predicted_visibility = (

        (
            w_elementary
            **
            link_count
        )

        *

        (
            q_swap
            **
            repeater_count
        )
    )


    predicted_visibility = min(
        max(
            float(
                predicted_visibility
            ),
            0.0
        ),
        1.0
    )


    # ============================================================
    # Convert visibility -> fidelity / QBER
    # ============================================================

    predicted_fidelity = (

        0.25
        +
        0.75
        *
        predicted_visibility
    )


    predicted_qber = (

        1.0
        -
        predicted_visibility

    ) / 2.0


    return {

        "predicted_werner_visibility":
            predicted_visibility,

        "predicted_fidelity":
            float(
                predicted_fidelity
            ),

        "predicted_qber":
            float(
                predicted_qber
            )
    }


print(
    "[PASS] Fresh-path Werner predictor ready."
)

In [ ]:
# @title Path Resource Bottleneck Metrics

def phase4_path_resource_metrics(
    network,
    candidate_path
):

    # ============================================================
    # Link multiplexing
    # ============================================================

    mux_values = []


    for node_a, node_b in zip(

        candidate_path.node_names[:-1],
        candidate_path.node_names[1:]
    ):

        profile = (
            network
            .link_profile(
                node_a,
                node_b
            )
        )


        mux_values.append(

            int(
                profile
                .multiplexing_modes_available
            )
        )


    multiplexing_bottleneck = (
        min(
            mux_values
        )
    )


    # ============================================================
    # Repeater resources
    # ============================================================

    memory_values = []

    bsm_values = []

    bsm_probabilities = []

    memory_decay_rates = []

    repeater_active_states = []


    for repeater_name in (
        candidate_path.repeater_names
    ):

        profile = (
            network
            .node_profile(
                repeater_name
            )
        )


        memory_values.append(

            int(
                profile
                .memory_slots_available
            )
        )


        bsm_values.append(

            int(
                profile
                .bsm_units_available
            )
        )


        bsm_probabilities.append(

            float(
                profile
                .bsm_success_probability
            )
        )


        memory_decay_rates.append(

            float(
                profile
                .memory_decay_rate_per_s
            )
        )


        repeater_active_states.append(

            bool(
                profile.active
            )
        )


    # ============================================================
    # Direct USER-to-USER path support
    # ============================================================

    if memory_values:

        memory_bottleneck = (
            min(
                memory_values
            )
        )

        bsm_bottleneck = (
            min(
                bsm_values
            )
        )

        bsm_success_product = (
            math.prod(
                bsm_probabilities
            )
        )

    else:

        memory_bottleneck = None

        bsm_bottleneck = None

        bsm_success_product = 1.0


    return {

        "multiplexing_modes_available":
            multiplexing_bottleneck,

        "memory_slots_available":
            memory_bottleneck,

        "bsm_units_available":
            bsm_bottleneck,

        "bsm_success_product":
            float(
                bsm_success_product
            ),

        "memory_decay_rates":
            tuple(
                memory_decay_rates
            ),

        "all_repeaters_active":
            all(
                repeater_active_states
            )
    }


print(
    "[PASS] Resource bottleneck calculator ready."
)

In [ ]:
# @title  Entanglement Delivery-Time Proxy

def phase4_path_delivery_time_proxy(
    network,
    candidate_path
):

    link_snapshots = []


    # ============================================================
    # Elementary generation
    # ============================================================

    for node_a, node_b in zip(

        candidate_path.node_names[:-1],
        candidate_path.node_names[1:]
    ):

        snapshot = (
            phase4_link_metric_snapshot(

                network,
                node_a,
                node_b
            )
        )


        link_snapshots.append(
            snapshot
        )


    generation_latencies = [

        snapshot[
            "expected_generation_latency_s"
        ]

        for snapshot
        in link_snapshots
    ]


    parallel_generation_latency_proxy_s = (

        max(
            generation_latencies
        )

        if generation_latencies

        else 0.0
    )


    # ============================================================
    # BSM processing / retry proxy
    # ============================================================

    bsm_latency_proxy_s = 0.0


    for repeater_name in (
        candidate_path.repeater_names
    ):

        profile = (
            network
            .node_profile(
                repeater_name
            )
        )


        p_b = float(
            profile
            .bsm_success_probability
        )


        if p_b <= 0.0:

            bsm_latency_proxy_s = (
                math.inf
            )

            break


        bsm_latency_proxy_s += (

            PHASE4_METRIC_TIMING_PROFILE
            .atomic_bsm_latency_s

            /

            p_b
        )


    expected_pair_delivery_proxy_s = (

        parallel_generation_latency_proxy_s

        +

        bsm_latency_proxy_s
    )


    return {

        "link_snapshots":
            tuple(
                link_snapshots
            ),

        "parallel_generation_latency_proxy_s":
            float(
                parallel_generation_latency_proxy_s
            ),

        "bsm_latency_proxy_s":
            float(
                bsm_latency_proxy_s
            ),

        "expected_pair_delivery_proxy_s":
            float(
                expected_pair_delivery_proxy_s
            )
    }


print(
    "[PASS] End-to-end delivery-time proxy ready."
)

In [ ]:
# @title  Memory Coherence Exposure Cost

def phase4_memory_coherence_cost(
    network,
    candidate_path,
    expected_delivery_time_s
):

    expected_delivery_time_s = float(
        expected_delivery_time_s
    )


    if not math.isfinite(
        expected_delivery_time_s
    ):

        return math.inf


    total_decay_rate = 0.0


    for repeater_name in (
        candidate_path.repeater_names
    ):

        profile = (
            network
            .node_profile(
                repeater_name
            )
        )


        total_decay_rate += float(

            profile
            .memory_decay_rate_per_s
        )


    return float(

        total_decay_rate
        *
        expected_delivery_time_s
    )


print(
    "[PASS] Memory coherence-cost model ready."
)

In [ ]:
# @title  Complete Candidate Path Metric Evaluator

def phase4_evaluate_candidate_path(
    network,
    candidate_path
):

    # ============================================================
    # Topology metrics
    # ============================================================

    total_distance_km = 0.0

    link_active_states = []


    for node_a, node_b in zip(

        candidate_path.node_names[:-1],
        candidate_path.node_names[1:]
    ):

        link = (
            network
            .topology
            .get_link(
                node_a,
                node_b
            )
        )


        profile = (
            network
            .link_profile(
                node_a,
                node_b
            )
        )


        total_distance_km += float(
            link.distance_km
        )


        link_active_states.append(
            bool(
                profile.active
            )
        )


    # ============================================================
    # Node activity
    # ============================================================

    node_active_states = [

        bool(
            network
            .node_profile(
                node_name
            )
            .active
        )

        for node_name
        in candidate_path.node_names
    ]


    # ============================================================
    # Quantum quality
    # ============================================================

    quality = (
        phase4_predict_fresh_path_quality(
            candidate_path
        )
    )


    # ============================================================
    # Physical resources
    # ============================================================

    resources = (
        phase4_path_resource_metrics(

            network,
            candidate_path
        )
    )


    # ============================================================
    # Physical timing proxy
    # ============================================================

    timing = (
        phase4_path_delivery_time_proxy(

            network,
            candidate_path
        )
    )


    # ============================================================
    # Memory coherence
    # ============================================================

    memory_cost = (
        phase4_memory_coherence_cost(

            network=(
                network
            ),

            candidate_path=(
                candidate_path
            ),

            expected_delivery_time_s=(

                timing[
                    "expected_pair_delivery_proxy_s"
                ]
            )
        )
    )


    # ============================================================
    # Hard feasibility
    # ============================================================

    all_nodes_active = all(
        node_active_states
    )


    all_links_active = all(
        link_active_states
    )


    multiplexing_feasible = (

        resources[
            "multiplexing_modes_available"
        ]

        >
        0
    )


    # Direct paths contain no repeaters.
    if (
        candidate_path.repeater_count
        ==
        0
    ):

        memory_feasible = True

        bsm_feasible = True

    else:

        memory_feasible = (

            resources[
                "memory_slots_available"
            ]

            is not None

            and

            resources[
                "memory_slots_available"
            ]

            >=
            2
        )


        bsm_feasible = (

            resources[
                "bsm_units_available"
            ]

            is not None

            and

            resources[
                "bsm_units_available"
            ]

            >=
            1

            and

            resources[
                "bsm_success_product"
            ]

            >
            0.0
        )


    qkd_quality_feasible = (

        quality[
            "predicted_qber"
        ]

        <=

        PHASE4_METRIC_QUALITY_CONFIG
        .qber_limit
    )


    statically_feasible = all(
        (
            all_nodes_active,
            all_links_active,
            multiplexing_feasible,
            memory_feasible,
            bsm_feasible,
            qkd_quality_feasible
        )
    )


    return Phase4PathMetrics(

        path_index=(
            candidate_path.path_index
        ),

        path=(
            candidate_path.display_path
        ),

        node_names=(
            candidate_path.node_names
        ),

        total_distance_km=(
            float(
                total_distance_km
            )
        ),

        hop_count=(
            candidate_path.hop_count
        ),

        repeater_count=(
            candidate_path.repeater_count
        ),

        predicted_werner_visibility=(

            quality[
                "predicted_werner_visibility"
            ]
        ),

        predicted_fidelity=(

            quality[
                "predicted_fidelity"
            ]
        ),

        predicted_qber=(

            quality[
                "predicted_qber"
            ]
        ),

        multiplexing_modes_available=(

            resources[
                "multiplexing_modes_available"
            ]
        ),

        memory_slots_available=(

            resources[
                "memory_slots_available"
            ]
        ),

        bsm_units_available=(

            resources[
                "bsm_units_available"
            ]
        ),

        bsm_success_product=(

            resources[
                "bsm_success_product"
            ]
        ),

        parallel_generation_latency_proxy_s=(

            timing[
                "parallel_generation_latency_proxy_s"
            ]
        ),

        bsm_latency_proxy_s=(

            timing[
                "bsm_latency_proxy_s"
            ]
        ),

        expected_pair_delivery_proxy_s=(

            timing[
                "expected_pair_delivery_proxy_s"
            ]
        ),

        memory_coherence_cost=(
            memory_cost
        ),

        all_nodes_active=(
            all_nodes_active
        ),

        all_links_active=(
            all_links_active
        ),

        multiplexing_feasible=(
            multiplexing_feasible
        ),

        memory_feasible=(
            memory_feasible
        ),

        bsm_feasible=(
            bsm_feasible
        ),

        qkd_quality_feasible=(
            qkd_quality_feasible
        ),

        statically_feasible=(
            statically_feasible
        )
    )


print(
    "[PASS] Complete candidate-path evaluator ready."
)

In [ ]:
# @title  Evaluate All Candidate Paths

PHASE4_PATH_METRICS = tuple(

    phase4_evaluate_candidate_path(

        network=(
            phase4_network
        ),

        candidate_path=(
            candidate_path
        )
    )

    for candidate_path
    in PHASE4_VALID_PATHS
)


assert (

    len(
        PHASE4_PATH_METRICS
    )

    ==

    len(
        PHASE4_VALID_PATHS
    )
)


print(
    f"[PASS] Evaluated "
    f"{len(PHASE4_PATH_METRICS)} "
    "valid Alice-to-Bob candidate paths."
)

In [ ]:
# @title  Candidate Quantum Routing Metric Table

metric_rows = []


for metrics in (
    PHASE4_PATH_METRICS
):

    metric_rows.append(
        {

            "path_index":
                metrics.path_index,

            "path":
                metrics.path,

            "distance_km":
                metrics.total_distance_km,

            "hops":
                metrics.hop_count,

            "repeaters":
                metrics.repeater_count,

            "pred_visibility":
                metrics.predicted_werner_visibility,

            "pred_fidelity":
                metrics.predicted_fidelity,

            "pred_qber":
                metrics.predicted_qber,

            "mux_available":
                metrics.multiplexing_modes_available,

            "memory_available":
                metrics.memory_slots_available,

            "bsm_available":
                metrics.bsm_units_available,

            "bsm_success_product":
                metrics.bsm_success_product,

            "generation_latency_proxy_ms":

                metrics
                .parallel_generation_latency_proxy_s

                *
                1000.0,

            "delivery_proxy_ms":

                metrics
                .expected_pair_delivery_proxy_s

                *
                1000.0,

            "memory_coherence_cost":
                metrics.memory_coherence_cost,

            "qkd_feasible":
                metrics.qkd_quality_feasible,

            "static_feasible":
                metrics.statically_feasible
        }
    )


PHASE4_PATH_METRICS_DF = (
    pd.DataFrame(
        metric_rows
    )
)


display(
    PHASE4_PATH_METRICS_DF
)


print(
    "\n[PASS] Candidate routing metric table created."
)

In [ ]:
# @title Hard Feasibility Audit

print(
    "=" * 100
)

print(
    "FEASIBILITY AUDIT"
)

print(
    "=" * 100
)


for metrics in (
    PHASE4_PATH_METRICS
):

    print(
        f"\nPath {metrics.path_index}: "
        f"{metrics.path}"
    )


    print(
        "  Nodes active       :",
        metrics.all_nodes_active
    )


    print(
        "  Links active       :",
        metrics.all_links_active
    )


    print(
        "  Multiplexing       :",
        metrics.multiplexing_feasible
    )


    print(
        "  Memory             :",
        metrics.memory_feasible
    )


    print(
        "  BSM                :",
        metrics.bsm_feasible
    )


    print(
        "  QKD quality        :",
        metrics.qkd_quality_feasible
    )


    print(
        "  STATICALLY FEASIBLE:",
        metrics.statically_feasible
    )


    if not (
        metrics.statically_feasible
    ):

        print(
            "  -> This path MUST NOT reach weighted scoring."
        )


PHASE4_STATICALLY_FEASIBLE_PATHS = tuple(

    metrics

    for metrics
    in PHASE4_PATH_METRICS

    if metrics.statically_feasible
)


print(
    "\n" + "-" * 100
)


print(
    "Candidate paths:",
    len(
        PHASE4_PATH_METRICS
    )
)


print(
    "Statically feasible:",
    len(
        PHASE4_STATICALLY_FEASIBLE_PATHS
    )
)


assert (
    len(
        PHASE4_STATICALLY_FEASIBLE_PATHS
    )
    >
    0
)


print(
    "\n[PASS] At least one route survives "
    "the complete static feasibility gate."
)

In [ ]:
# @title  Metric Model Verification

print(
    "=" * 100
)

print(
    "METRIC MODEL VERIFICATION"
)

print(
    "=" * 100
)


for metrics in (
    PHASE4_PATH_METRICS
):

    # ============================================================
    # Distance must be positive.
    # ============================================================

    assert (
        metrics.total_distance_km
        >
        0.0
    )


    # ============================================================
    # Path structure.
    # ============================================================

    assert (

        metrics.hop_count

        ==

        len(
            metrics.node_names
        )
        -
        1
    )


    assert (

        metrics.repeater_count

        ==

        max(
            metrics.hop_count
            -
            1,

            0
        )
    )


    # ============================================================
    # Werner consistency.
    # ============================================================

    expected_fidelity = (

        0.25

        +

        0.75
        *
        metrics.predicted_werner_visibility
    )


    expected_qber = (

        1.0

        -

        metrics.predicted_werner_visibility

    ) / 2.0


    assert math.isclose(

        metrics.predicted_fidelity,

        expected_fidelity,

        rel_tol=0.0,

        abs_tol=1e-15
    )


    assert math.isclose(

        metrics.predicted_qber,

        expected_qber,

        rel_tol=0.0,

        abs_tol=1e-15
    )


    # ============================================================
    # Resource bounds.
    # ============================================================

    assert (
        metrics.multiplexing_modes_available
        >=
        0
    )


    if (
        metrics.memory_slots_available
        is not None
    ):

        assert (
            metrics.memory_slots_available
            >=
            0
        )


    if (
        metrics.bsm_units_available
        is not None
    ):

        assert (
            metrics.bsm_units_available
            >=
            0
        )


    # ============================================================
    # Delivery proxy must include generation latency.
    # ============================================================

    assert (

        metrics.expected_pair_delivery_proxy_s

        >=

        metrics.parallel_generation_latency_proxy_s
    )


    # ============================================================
    # Memory coherence cost cannot be negative.
    # ============================================================

    assert (
        metrics.memory_coherence_cost
        >=
        0.0
    )


print(
    "[PASS] Topology metric invariants hold."
)

print(
    "[PASS] Werner visibility/fidelity/QBER mapping holds."
)

print(
    "[PASS] Resource bottleneck metrics are valid."
)

print(
    "[PASS] Delivery-time proxy is physically non-negative."
)

print(
    "[PASS] Memory coherence cost is non-negative."
)


print(
    "\n" + "=" * 100
)

print(
    "PHASE 4C PASSED"
)

print(
    "=" * 100
)

### Phase 4D :

Weight Calibration Experiment

In [ ]:
# @title Empirical Routing Calibration Design

from dataclasses import dataclass, asdict
from itertools import product

import math
import time
import numpy as np
import pandas as pd


# ================================================================
# Calibration factors
# ================================================================

PHASE4_CAL_DISTANCE_KM = (
    200.0,
    300.0,
    400.0,
    500.0,
    600.0
)

PHASE4_CAL_REPEATER_COUNTS = (
    2,
    4,
    6,
    8
)

PHASE4_CAL_MODE_COUNTS = (
    1,
    4,
    8,
    16
)


# Pilot first.
PHASE4_CAL_TRIALS_PER_CONFIG = 8

PHASE4_CAL_MAX_SLOTS_PER_PAIR = 20_000

PHASE4_CAL_Z_PROBABILITY = 0.80


# ================================================================
# Deterministic calibration pair-index namespace
#
# Deliberately kept far away from Phase-3K production pair IDs.
# ================================================================

PHASE4_CAL_PAIR_INDEX_BASE = 10_000_000


PHASE4_CAL_CONFIGS = tuple(

    (
        float(distance_km),
        int(repeater_count),
        int(mode_count)
    )

    for (
        distance_km,
        repeater_count,
        mode_count
    )

    in product(
        PHASE4_CAL_DISTANCE_KM,
        PHASE4_CAL_REPEATER_COUNTS,
        PHASE4_CAL_MODE_COUNTS
    )
)


PHASE4_CAL_TOTAL_TRAJECTORIES = (

    len(
        PHASE4_CAL_CONFIGS
    )

    *

    PHASE4_CAL_TRIALS_PER_CONFIG
)


print(
    "=" * 94
)

print(
    "NARAD-QC PHASE 4 — EMPIRICAL ROUTING CALIBRATION PILOT"
)

print(
    "=" * 94
)

print(
    "Distances          :",
    PHASE4_CAL_DISTANCE_KM
)

print(
    "Repeater counts    :",
    PHASE4_CAL_REPEATER_COUNTS
)

print(
    "Multiplexing modes :",
    PHASE4_CAL_MODE_COUNTS
)

print(
    "Configurations     :",
    len(
        PHASE4_CAL_CONFIGS
    )
)

print(
    "Trials/config      :",
    PHASE4_CAL_TRIALS_PER_CONFIG
)

print(
    "Total trajectories :",
    PHASE4_CAL_TOTAL_TRAJECTORIES
)

print(
    "\n[PASS] Calibration design frozen."
)

In [ ]:
# @title Deterministic Calibration Scenario Identity

def phase4_cal_pair_index(
    config_index,
    trial_index
):

    config_index = int(
        config_index
    )

    trial_index = int(
        trial_index
    )


    if config_index < 0:

        raise ValueError(
            "config_index cannot be negative."
        )


    if not (
        0
        <=
        trial_index
        <
        PHASE4_CAL_TRIALS_PER_CONFIG
    ):

        raise ValueError(
            "trial_index out of calibration range."
        )


    return int(

        PHASE4_CAL_PAIR_INDEX_BASE

        +

        config_index
        *
        PHASE4_CAL_TRIALS_PER_CONFIG

        +

        trial_index
    )


# Unique deterministic identity test.
all_pair_ids = [

    phase4_cal_pair_index(
        config_index,
        trial_index
    )

    for config_index
    in range(
        len(
            PHASE4_CAL_CONFIGS
        )
    )

    for trial_index
    in range(
        PHASE4_CAL_TRIALS_PER_CONFIG
    )
]


assert (
    len(
        all_pair_ids
    )

    ==

    len(
        set(
            all_pair_ids
        )
    )
)


print(
    "[PASS] Every calibration trajectory has "
    "a unique deterministic pair index."
)

In [ ]:
# @title Calibration Pre-Execution Routing Features

def phase4_calibration_features(
    total_distance_km,
    repeater_count,
    mode_count
):

    total_distance_km = float(
        total_distance_km
    )

    repeater_count = int(
        repeater_count
    )

    mode_count = int(
        mode_count
    )


    link_count = (
        repeater_count
        +
        1
    )


    elementary_distance_km = (

        total_distance_km

        /

        link_count
    )


    # ============================================================
    # Phase-3 fibre survival proxy
    # ============================================================

    attenuation_db_per_km = float(
        PHASE3_DEFAULT_FIBER_ATTENUATION_DB_PER_KM
    )


    p_single = float(

        10.0
        **
        (
            -
            attenuation_db_per_km
            *
            elementary_distance_km
            /
            10.0
        )
    )


    p_mux = float(

        1.0

        -

        (
            1.0
            -
            p_single
        )
        **
        mode_count
    )


    # ============================================================
    # Expected elementary generation latency
    # ============================================================

    expected_link_generation_s = (

        PHASE4_METRIC_TIMING_PROFILE
        .expected_generation_latency_s(

            distance_km=(
                elementary_distance_km
            ),

            success_probability=(
                p_mux
            )
        )
    )


    # Equal chain => every elementary link has same expectation.
    generation_proxy_s = float(
        expected_link_generation_s
    )


    # ============================================================
    # Current BSM retry-time proxy
    # p_b = 0.4 in our validated Phase-3 model.
    # ============================================================

    bsm_success_probability = 0.4


    bsm_proxy_s = float(

        repeater_count

        *

        (
            PHASE4_METRIC_TIMING_PROFILE
            .atomic_bsm_latency_s

            /

            bsm_success_probability
        )
    )


    delivery_proxy_s = float(

        generation_proxy_s

        +

        bsm_proxy_s
    )


    # ============================================================
    # Fresh Werner quality
    # ============================================================

    w_elem = float(

        PHASE4_METRIC_QUALITY_CONFIG
        .elementary_visibility
    )


    q_swap = float(

        PHASE4_METRIC_QUALITY_CONFIG
        .swap_operation_visibility
    )


    predicted_visibility = float(

        (
            w_elem
            **
            link_count
        )

        *

        (
            q_swap
            **
            repeater_count
        )
    )


    predicted_fidelity = float(

        0.25

        +

        0.75
        *
        predicted_visibility
    )


    predicted_qber = float(

        (
            1.0
            -
            predicted_visibility
        )

        /
        2.0
    )


    # ============================================================
    # Conservative coherence exposure proxy
    # ============================================================

    lambda_memory = float(

        PHASE4_METRIC_QUALITY_CONFIG
        .memory_decay_rate_per_s
    )


    memory_coherence_cost = float(

        repeater_count

        *
        lambda_memory

        *
        delivery_proxy_s
    )


    return {

        "total_distance_km":
            total_distance_km,

        "repeater_count":
            repeater_count,

        "hop_count":
            link_count,

        "mode_count":
            mode_count,

        "elementary_distance_km":
            elementary_distance_km,

        "single_mode_success_probability":
            p_single,

        "multiplexed_cycle_success_probability":
            p_mux,

        "predicted_visibility":
            predicted_visibility,

        "predicted_fidelity":
            predicted_fidelity,

        "predicted_qber":
            predicted_qber,

        "generation_proxy_s":
            generation_proxy_s,

        "delivery_proxy_s":
            delivery_proxy_s,

        "memory_coherence_cost":
            memory_coherence_cost
    }


# Quick sanity point.
_phase4_cal_test = (
    phase4_calibration_features(
        total_distance_km=600.0,
        repeater_count=6,
        mode_count=8
    )
)


assert (
    _phase4_cal_test[
        "hop_count"
    ]
    ==
    7
)


assert (
    0.0
    <
    _phase4_cal_test[
        "predicted_visibility"
    ]
    <=
    1.0
)


print(
    "[PASS] Calibration router-feature generator ready."
)

In [ ]:
# @title Execute One Calibration Trajectory

def phase4_run_one_calibration_trajectory(
    config_index,
    trial_index,
    total_distance_km,
    repeater_count,
    mode_count
):

    pair_index = (
        phase4_cal_pair_index(
            config_index,
            trial_index
        )
    )


    features = (
        phase4_calibration_features(

            total_distance_km=(
                total_distance_km
            ),

            repeater_count=(
                repeater_count
            ),

            mode_count=(
                mode_count
            )
        )
    )


    wall_start = (
        time.perf_counter()
    )


    payload = (
        phase3k_simulate_one_fast_hardware_lane(

            pair_index=(
                pair_index
            ),

            total_distance_km=(
                total_distance_km
            ),

            repeater_count=(
                repeater_count
            ),

            mode_count=(
                mode_count
            ),

            z_probability=(
                PHASE4_CAL_Z_PROBABILITY
            ),

            max_slots_per_pair=(
                PHASE4_CAL_MAX_SLOTS_PER_PAIR
            )
        )
    )


    python_wall_s = float(

        time.perf_counter()
        -
        wall_start
    )


    # The Phase-3K API returns an asdict payload.
    record = (
        NARADQKDPairMeasurement(
            **payload
        )
    )


    actual_delivery_s = float(
        record.delivery_physical_time_s
    )


    actual_pair_rate_per_s = float(

        1.0
        /
        actual_delivery_s
    )


    return {

        "config_index":
            int(
                config_index
            ),

        "trial_index":
            int(
                trial_index
            ),

        "pair_index":
            int(
                pair_index
            ),

        **features,

        # ========================================================
        # Actual Phase-3 outcomes
        # ========================================================

        "actual_delivery_s":
            actual_delivery_s,

        "actual_pair_rate_per_s":
            actual_pair_rate_per_s,

        "actual_fidelity":
            float(
                record.fidelity
            ),

        "actual_visibility":
            float(
                record.visibility
            ),

        "actual_predicted_qber":
            float(
                record.predicted_qber
            ),

        "elementary_generation_cycles":
            int(
                record.elementary_generation_cycles
            ),

        "optical_mode_attempts":
            int(
                record.optical_mode_attempts
            ),

        "bsm_attempts":
            int(
                record.bsm_attempts
            ),

        "python_wall_s":
            python_wall_s
    }


print(
    "[PASS] One-trajectory calibration executor ready."
)

In [ ]:
# @title Calibration Test

# PHASE4_CAL_SMOKE = (
#     phase4_run_one_calibration_trajectory(

#         config_index=0,

#         trial_index=0,

#         total_distance_km=(
#             PHASE4_CAL_CONFIGS[0][0]
#         ),

#         repeater_count=(
#             PHASE4_CAL_CONFIGS[0][1]
#         ),

#         mode_count=(
#             PHASE4_CAL_CONFIGS[0][2]
#         )
#     )
# )


# display(
#     pd.DataFrame(
#         [
#             PHASE4_CAL_SMOKE
#         ]
#     )
# )


# assert (
#     PHASE4_CAL_SMOKE[
#         "actual_delivery_s"
#     ]
#     >
#     0.0
# )


# assert (
#     0.0
#     <=
#     PHASE4_CAL_SMOKE[
#         "actual_visibility"
#     ]
#     <=
#     1.0
# )


# print(
#     "\n[PASS] Calibration smoke trajectory completed."
# )

In [ ]:
# @title Four-Worker Calibration Backend

import os
import gc
import json
import time
import math
import multiprocessing as mp

from dataclasses import is_dataclass, asdict

import numpy as np
import pandas as pd


# ================================================================
# Computational configuration
# ================================================================

PHASE4_CAL_CPU_WORKERS = 4

PHASE4_CAL_PROGRESS_EVERY = 16


# ================================================================
# Worker thread limiter
#
# One process should not silently start a BLAS/OpenMP thread farm.
# ================================================================

_PHASE4_CAL_THREAD_LIMITER = None


def phase4_cal_worker_initializer():

    global _PHASE4_CAL_THREAD_LIMITER


    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"


    try:

        from threadpoolctl import (
            threadpool_limits
        )


        _PHASE4_CAL_THREAD_LIMITER = (
            threadpool_limits(
                limits=1
            )
        )


        _PHASE4_CAL_THREAD_LIMITER.__enter__()


    except Exception:

        _PHASE4_CAL_THREAD_LIMITER = None


# ================================================================
# One scenario worker
# ================================================================

def phase4_calibration_worker(
    scenario
):
    """
    Execute exactly one deterministic calibration trajectory.

    scenario:
        (
            config_index,
            trial_index,
            total_distance_km,
            repeater_count,
            mode_count
        )

    IMPORTANT:
        - worker performs no file writes;
        - worker modifies no parent state;
        - pair_index remains deterministic;
        - returned payload is compact and serializable.
    """

    (
        config_index,
        trial_index,
        total_distance_km,
        repeater_count,
        mode_count
    ) = scenario


    cpu_start = (
        time.process_time()
    )


    wall_start = (
        time.perf_counter()
    )


    result = (
        phase4_run_one_calibration_trajectory(

            config_index=(
                int(
                    config_index
                )
            ),

            trial_index=(
                int(
                    trial_index
                )
            ),

            total_distance_km=(
                float(
                    total_distance_km
                )
            ),

            repeater_count=(
                int(
                    repeater_count
                )
            ),

            mode_count=(
                int(
                    mode_count
                )
            )
        )
    )


    result = dict(
        result
    )


    result[
        "worker_pid"
    ] = int(
        os.getpid()
    )


    result[
        "worker_cpu_s"
    ] = float(

        time.process_time()
        -
        cpu_start
    )


    result[
        "worker_total_wall_s"
    ] = float(

        time.perf_counter()
        -
        wall_start
    )


    return result


print(
    "[PASS] Four-worker calibration backend ready."
)

In [ ]:
# @title Frozen Calibration Scenario

PHASE4_CAL_SCENARIOS = tuple(

    (
        config_index,
        trial_index,
        distance_km,
        repeater_count,
        mode_count
    )

    for config_index, (
        distance_km,
        repeater_count,
        mode_count
    )

    in enumerate(
        PHASE4_CAL_CONFIGS
    )

    for trial_index
    in range(
        PHASE4_CAL_TRIALS_PER_CONFIG
    )
)


assert (
    len(
        PHASE4_CAL_SCENARIOS
    )

    ==
    PHASE4_CAL_TOTAL_TRAJECTORIES
)


# ================================================================
# Deterministic pair-ID uniqueness
# ================================================================

_manifest_pair_ids = [

    phase4_cal_pair_index(
        config_index,
        trial_index
    )

    for (
        config_index,
        trial_index,
        _,
        _,
        _
    )
    in PHASE4_CAL_SCENARIOS
]


assert (
    len(
        _manifest_pair_ids
    )

    ==
    len(
        set(
            _manifest_pair_ids
        )
    )
)


PHASE4_CAL_MANIFEST_DF = pd.DataFrame(

    [

        {

            "config_index":
                config_index,

            "trial_index":
                trial_index,

            "pair_index":
                phase4_cal_pair_index(
                    config_index,
                    trial_index
                ),

            "total_distance_km":
                distance_km,

            "repeater_count":
                repeater_count,

            "hop_count":
                repeater_count + 1,

            "mode_count":
                mode_count

        }

        for (
            config_index,
            trial_index,
            distance_km,
            repeater_count,
            mode_count
        )
        in PHASE4_CAL_SCENARIOS
    ]
)


display(
    PHASE4_CAL_MANIFEST_DF.head(
        12
    )
)


print(
    "\nTotal scenarios:",
    len(
        PHASE4_CAL_SCENARIOS
    )
)


print(
    "[PASS] Calibration manifest frozen."
)

In [ ]:
# @title Calibration Checkpoint Location

from pathlib import Path


if Path(
    "/content/drive/MyDrive"
).exists():

    PHASE4_CAL_CHECKPOINT_DIR = Path(

        "/content/drive/MyDrive/"
        "NARAD_QC/checkpoints/"
        "phase4_routing_calibration"
    )

    PHASE4_CAL_CHECKPOINT_IS_PERSISTENT = True

else:

    PHASE4_CAL_CHECKPOINT_DIR = Path(

        "/content/"
        "NARAD_QC_phase4_routing_calibration"
    )

    PHASE4_CAL_CHECKPOINT_IS_PERSISTENT = False


PHASE4_CAL_CHECKPOINT_DIR.mkdir(

    parents=True,

    exist_ok=True
)


PHASE4_CAL_JSONL_PATH = (

    PHASE4_CAL_CHECKPOINT_DIR
    /
    "phase4_calibration_raw.jsonl"
)


PHASE4_CAL_FINAL_CSV_PATH = (

    PHASE4_CAL_CHECKPOINT_DIR
    /
    "phase4_calibration_raw.csv"
)


PHASE4_CAL_SUMMARY_CSV_PATH = (

    PHASE4_CAL_CHECKPOINT_DIR
    /
    "phase4_calibration_summary.csv"
)

PHASE4_CAL_ONLY_SUCCESS_AWARE_SUMMARY_CSV_PATH = (

    PHASE4_CAL_CHECKPOINT_DIR
    /
    "phase4_calibration_summary_only_success.csv"
)


print(
    "Checkpoint directory:"
)

print(
    PHASE4_CAL_CHECKPOINT_DIR
)


print(
    "\nPersistent across runtime resets:",
    PHASE4_CAL_CHECKPOINT_IS_PERSISTENT
)


if not PHASE4_CAL_CHECKPOINT_IS_PERSISTENT:

    print(
        "\n[WARNING] Google Drive is not mounted. "
        "Checkpointing works, but /content will disappear "
        "when the runtime resets."
    )


print(
    "\n[PASS] Calibration checkpoint location ready."
)

In [ ]:
# @title  Resume-Safe Calibration Checkpoint Loader

def phase4_load_calibration_checkpoint():

    records_by_pair = {}


    if not (
        PHASE4_CAL_JSONL_PATH.exists()
    ):

        return records_by_pair


    with open(

        PHASE4_CAL_JSONL_PATH,

        "r",

        encoding="utf-8"

    ) as handle:


        for line_number, line in enumerate(

            handle,

            start=1
        ):

            line = line.strip()


            if not line:

                continue


            try:

                record = (
                    json.loads(
                        line
                    )
                )


            except json.JSONDecodeError:

                # ------------------------------------------------
                # A runtime interruption can theoretically leave
                # the final line incomplete.
                #
                # Ignore ONLY a malformed final line.
                # ------------------------------------------------

                print(
                    f"[WARNING] Ignoring malformed checkpoint "
                    f"line {line_number}."
                )

                continue


            pair_index = int(
                record[
                    "pair_index"
                ]
            )


            records_by_pair[
                pair_index
            ] = record


    return records_by_pair


PHASE4_CAL_EXISTING_RECORDS = (
    phase4_load_calibration_checkpoint()
)


print(
    "Already checkpointed trajectories:",
    len(
        PHASE4_CAL_EXISTING_RECORDS
    )
)


print(
    "[PASS] Resume-safe checkpoint loader ready."
)

In [ ]:
# @title  Calibration Checkpoint Consistency Audit

def phase4_expected_manifest_lookup():

    return {

        phase4_cal_pair_index(
            config_index,
            trial_index
        ):
        {

            "config_index":
                int(
                    config_index
                ),

            "trial_index":
                int(
                    trial_index
                ),

            "total_distance_km":
                float(
                    distance_km
                ),

            "repeater_count":
                int(
                    repeater_count
                ),

            "mode_count":
                int(
                    mode_count
                )

        }

        for (
            config_index,
            trial_index,
            distance_km,
            repeater_count,
            mode_count
        )
        in PHASE4_CAL_SCENARIOS
    }


PHASE4_CAL_EXPECTED_LOOKUP = (
    phase4_expected_manifest_lookup()
)


for pair_index, record in (
    PHASE4_CAL_EXISTING_RECORDS.items()
):

    if (
        pair_index
        not in
        PHASE4_CAL_EXPECTED_LOOKUP
    ):

        raise RuntimeError(

            "Checkpoint contains an unexpected calibration "
            f"pair_index: {pair_index}"
        )


    expected = (

        PHASE4_CAL_EXPECTED_LOOKUP[
            pair_index
        ]
    )


    assert (
        int(
            record[
                "config_index"
            ]
        )

        ==

        expected[
            "config_index"
        ]
    )


    assert (
        int(
            record[
                "trial_index"
            ]
        )

        ==

        expected[
            "trial_index"
        ]
    )


    assert math.isclose(

        float(
            record[
                "total_distance_km"
            ]
        ),

        expected[
            "total_distance_km"
        ],

        rel_tol=0.0,

        abs_tol=1e-12
    )


    assert (
        int(
            record[
                "repeater_count"
            ]
        )

        ==

        expected[
            "repeater_count"
        ]
    )


    assert (
        int(
            record[
                "mode_count"
            ]
        )

        ==

        expected[
            "mode_count"
        ]
    )


print(
    "[PASS] Existing checkpoint matches "
    "the frozen calibration manifest."
)

In [ ]:
# @title  Missing Calibration Workload

PHASE4_CAL_COMPLETED_PAIR_IDS = set(

    PHASE4_CAL_EXISTING_RECORDS.keys()
)


PHASE4_CAL_MISSING_SCENARIOS = tuple(

    scenario

    for scenario
    in PHASE4_CAL_SCENARIOS

    if (

        phase4_cal_pair_index(

            scenario[0],
            scenario[1]
        )

        not in

        PHASE4_CAL_COMPLETED_PAIR_IDS
    )
)


print(
    "Total required :",
    PHASE4_CAL_TOTAL_TRAJECTORIES
)


print(
    "Already done   :",
    len(
        PHASE4_CAL_COMPLETED_PAIR_IDS
    )
)


print(
    "Still missing  :",
    len(
        PHASE4_CAL_MISSING_SCENARIOS
    )
)


print(
    "[PASS] Resume workload prepared."
)

In [ ]:
# @title  Failure-Aware Calibration Trajectory Runner

def phase4_run_one_calibration_trajectory(
    config_index,
    trial_index,
    total_distance_km,
    repeater_count,
    mode_count
):

    pair_index = (
        phase4_cal_pair_index(
            config_index,
            trial_index
        )
    )


    features = (
        phase4_calibration_features(

            total_distance_km=(
                total_distance_km
            ),

            repeater_count=(
                repeater_count
            ),

            mode_count=(
                mode_count
            )
        )
    )


    wall_start = (
        time.perf_counter()
    )


    try:

        payload = (
            phase3k_simulate_one_fast_hardware_lane(

                pair_index=(
                    pair_index
                ),

                total_distance_km=(
                    total_distance_km
                ),

                repeater_count=(
                    repeater_count
                ),

                mode_count=(
                    mode_count
                ),

                z_probability=(
                    PHASE4_CAL_Z_PROBABILITY
                ),

                max_slots_per_pair=(
                    PHASE4_CAL_MAX_SLOTS_PER_PAIR
                )
            )
        )


    except RuntimeError as error:

        error_message = str(
            error
        )


        # ========================================================
        # We suppress ONLY the known physical service-budget
        # exhaustion condition.
        #
        # Programming bugs / unexpected RuntimeErrors still crash,
        # which is exactly what we want.
        # ========================================================

        expected_failure_message = (
            "NARAD delivery layer failed to produce "
            "a QKD-ready end-to-end pair."
        )


        if (
            expected_failure_message
            not in
            error_message
        ):

            raise


        python_wall_s = float(

            time.perf_counter()
            -
            wall_start
        )


        return {

            "config_index":
                int(
                    config_index
                ),

            "trial_index":
                int(
                    trial_index
                ),

            "pair_index":
                int(
                    pair_index
                ),

            **features,

            # ====================================================
            # Outcome
            # ====================================================

            "delivery_success":
                False,

            "service_budget_exhausted":
                True,

            "max_slots_per_pair":
                int(
                    PHASE4_CAL_MAX_SLOTS_PER_PAIR
                ),

            "failure_reason":
                "QKD_READY_PAIR_NOT_PRODUCED_WITHIN_SLOT_BUDGET",

            # ====================================================
            # Undefined because no pair was delivered.
            # ====================================================

            "actual_delivery_s":
                float("nan"),

            "actual_pair_rate_per_s":
                0.0,

            "actual_fidelity":
                float("nan"),

            "actual_visibility":
                float("nan"),

            "actual_predicted_qber":
                float("nan"),

            "elementary_generation_cycles":
                float("nan"),

            "optical_mode_attempts":
                float("nan"),

            "bsm_attempts":
                float("nan"),

            "python_wall_s":
                python_wall_s
        }


    # ============================================================
    # Successful trajectory
    # ============================================================

    record = (
        NARADQKDPairMeasurement(
            **payload
        )
    )


    actual_delivery_s = float(
        record.delivery_physical_time_s
    )


    actual_pair_rate_per_s = float(

        1.0
        /
        actual_delivery_s
    )


    python_wall_s = float(

        time.perf_counter()
        -
        wall_start
    )


    return {

        "config_index":
            int(
                config_index
            ),

        "trial_index":
            int(
                trial_index
            ),

        "pair_index":
            int(
                pair_index
            ),

        **features,

        # ========================================================
        # Outcome
        # ========================================================

        "delivery_success":
            True,

        "service_budget_exhausted":
            False,

        "max_slots_per_pair":
            int(
                PHASE4_CAL_MAX_SLOTS_PER_PAIR
            ),

        "failure_reason":
            None,

        # ========================================================
        # Actual Phase-3 outcomes
        # ========================================================

        "actual_delivery_s":
            actual_delivery_s,

        "actual_pair_rate_per_s":
            actual_pair_rate_per_s,

        "actual_fidelity":
            float(
                record.fidelity
            ),

        "actual_visibility":
            float(
                record.visibility
            ),

        "actual_predicted_qber":
            float(
                record.predicted_qber
            ),

        "elementary_generation_cycles":
            int(
                record.elementary_generation_cycles
            ),

        "optical_mode_attempts":
            int(
                record.optical_mode_attempts
            ),

        "bsm_attempts":
            int(
                record.bsm_attempts
            ),

        "python_wall_s":
            python_wall_s
    }


print(
    "[PASS] Calibration runner now records "
    "physical service-budget failures instead of crashing."
)

In [ ]:
# @title  JSON-Safe Calibration Records

def phase4_json_safe_record(
    record
):

    clean = {}


    for key, value in record.items():

        if isinstance(
            value,
            (float, np.floating)
        ):

            if not math.isfinite(
                float(value)
            ):

                clean[key] = None

            else:

                clean[key] = float(
                    value
                )

        elif isinstance(
            value,
            np.integer
        ):

            clean[key] = int(
                value
            )

        else:

            clean[key] = value


    return clean


# ================================================================
# Redefine worker with JSON-safe output
# ================================================================

def phase4_calibration_worker(
    scenario
):

    (
        config_index,
        trial_index,
        total_distance_km,
        repeater_count,
        mode_count
    ) = scenario


    cpu_start = (
        time.process_time()
    )


    wall_start = (
        time.perf_counter()
    )


    result = (
        phase4_run_one_calibration_trajectory(

            config_index=(
                int(
                    config_index
                )
            ),

            trial_index=(
                int(
                    trial_index
                )
            ),

            total_distance_km=(
                float(
                    total_distance_km
                )
            ),

            repeater_count=(
                int(
                    repeater_count
                )
            ),

            mode_count=(
                int(
                    mode_count
                )
            )
        )
    )


    result = (
        phase4_json_safe_record(
            result
        )
    )


    result[
        "worker_pid"
    ] = int(
        os.getpid()
    )


    result[
        "worker_cpu_s"
    ] = float(

        time.process_time()
        -
        cpu_start
    )


    result[
        "worker_total_wall_s"
    ] = float(

        time.perf_counter()
        -
        wall_start
    )


    return result


print(
    "[PASS] Worker output is now explicit JSON-safe data."
)

In [ ]:
# @title  Run Four-Worker Calibration Farm

def phase4_run_calibration_farm(
    scenarios,
    workers=PHASE4_CAL_CPU_WORKERS,
    progress_every=PHASE4_CAL_PROGRESS_EVERY
):

    scenarios = tuple(
        scenarios
    )


    workers = int(
        workers
    )


    if workers < 1:

        raise ValueError(
            "workers must be positive."
        )


    if not scenarios:

        print(
            "[INFO] No missing calibration trajectories."
        )

        return []


    print(
        "=" * 100
    )

    print(
        "NARAD-QC PHASE 4 — CALIBRATION TRAJECTORY FARM"
    )

    print(
        "=" * 100
    )


    print(
        "Workers            :",
        workers
    )


    print(
        "Missing trajectories:",
        len(
            scenarios
        )
    )


    print(
        "Checkpoint         :",
        PHASE4_CAL_JSONL_PATH
    )


    # Parent cleanup before fork.
    gc.collect()


    context = (
        mp.get_context(
            "fork"
        )
    )


    completed_records = []


    farm_wall_start = (
        time.perf_counter()
    )


    with context.Pool(

        processes=(
            workers
        ),

        initializer=(
            phase4_cal_worker_initializer
        )

    ) as pool:


        iterator = (

            pool.imap_unordered(

                phase4_calibration_worker,

                scenarios,

                chunksize=1
            )
        )


        # ========================================================
        # ONLY the parent opens the checkpoint for writing.
        # ========================================================

        with open(

            PHASE4_CAL_JSONL_PATH,

            "a",

            encoding="utf-8"

        ) as checkpoint_handle:


            for completed_count, record in enumerate(

                iterator,

                start=1

            ):


                # ------------------------------------------------
                # Append one complete JSON record.
                # ------------------------------------------------

                checkpoint_handle.write(

                    json.dumps(
                        record,
                        sort_keys=True
                    )

                    +

                    "\n"
                )


                # Flush immediately for interruption safety.
                checkpoint_handle.flush()


                completed_records.append(
                    record
                )


                if (

                    completed_count
                    %
                    progress_every
                    ==
                    0

                    or

                    completed_count
                    ==
                    len(
                        scenarios
                    )
                ):


                    elapsed_s = (

                        time.perf_counter()
                        -
                        farm_wall_start
                    )


                    rate = (

                        completed_count
                        /
                        elapsed_s

                        if elapsed_s > 0.0

                        else float(
                            "nan"
                        )
                    )


                    remaining = (

                        len(
                            scenarios
                        )
                        -
                        completed_count
                    )


                    eta_s = (

                        remaining
                        /
                        rate

                        if (
                            rate > 0.0
                            and math.isfinite(
                                rate
                            )
                        )

                        else float(
                            "nan"
                        )
                    )


                    total_checkpointed = (

                        len(
                            PHASE4_CAL_COMPLETED_PAIR_IDS
                        )

                        +
                        completed_count
                    )


                    print(

                        f"[CAL] "
                        f"{total_checkpointed}/"
                        f"{PHASE4_CAL_TOTAL_TRAJECTORIES} "
                        f"checkpointed | "

                        f"this run "
                        f"{completed_count}/"
                        f"{len(scenarios)} | "

                        f"{rate:.3f} trajectories/s | "

                        f"elapsed "
                        f"{elapsed_s/60.0:.2f} min | "

                        f"ETA "
                        f"{eta_s/60.0:.2f} min"
                    )


    farm_wall_s = (

        time.perf_counter()
        -
        farm_wall_start
    )


    print(
        "\n" + "=" * 100
    )


    print(
        "CALIBRATION FARM RUN COMPLETE"
    )


    print(
        "=" * 100
    )


    print(
        "New trajectories:",
        len(
            completed_records
        )
    )


    print(
        "Farm wall time:",
        f"{farm_wall_s:.3f} s"
    )


    if farm_wall_s > 0.0:

        print(
            "Fresh throughput:",
            f"{len(completed_records)/farm_wall_s:.4f} "
            "trajectories/s"
        )


    return completed_records


PHASE4_CAL_NEW_RECORDS = (
    phase4_run_calibration_farm(

        scenarios=(
            PHASE4_CAL_MISSING_SCENARIOS
        ),

        workers=(
            PHASE4_CAL_CPU_WORKERS
        )
    )
)

In [ ]:
# @title  Reload Complete Calibration Dataset

PHASE4_CAL_ALL_RECORDS_BY_PAIR = (
    phase4_load_calibration_checkpoint()
)


assert (

    len(
        PHASE4_CAL_ALL_RECORDS_BY_PAIR
    )

    <=

    PHASE4_CAL_TOTAL_TRAJECTORIES
)


PHASE4_CAL_RAW_DF = (

    pd.DataFrame(

        list(
            PHASE4_CAL_ALL_RECORDS_BY_PAIR.values()
        )
    )

    .sort_values(

        by=[
            "config_index",
            "trial_index"
        ]

    )

    .reset_index(
        drop=True
    )
)


print(
    "Loaded calibration trajectories:",
    len(
        PHASE4_CAL_RAW_DF
    )
)


display(
    PHASE4_CAL_RAW_DF.head(
        10
    )
)

In [ ]:
# @title  Failure-Aware Dataset Integrity Audit

print(
    "=" * 100
)

print(
    "PHASE 4 CALIBRATION DATASET INTEGRITY"
)

print(
    "=" * 100
)


assert (
    PHASE4_CAL_RAW_DF[
        "pair_index"
    ]
    .is_unique
)


# ================================================================
# Delivery outcome column must exist
#
# Old checkpoint rows were generated before delivery_success was
# introduced, and all of those rows were necessarily successful.
# ================================================================

if (
    "delivery_success"
    not in
    PHASE4_CAL_RAW_DF.columns
):

    PHASE4_CAL_RAW_DF[
        "delivery_success"
    ] = True

else:

    PHASE4_CAL_RAW_DF[
        "delivery_success"
    ] = (

        PHASE4_CAL_RAW_DF[
            "delivery_success"
        ]
        .fillna(
            True
        )
        .astype(
            bool
        )
    )


successful_df = (

    PHASE4_CAL_RAW_DF[
        PHASE4_CAL_RAW_DF[
            "delivery_success"
        ]
    ]

    .copy()
)


failed_df = (

    PHASE4_CAL_RAW_DF[
        ~PHASE4_CAL_RAW_DF[
            "delivery_success"
        ]
    ]

    .copy()
)


# ================================================================
# Successful trajectories
# ================================================================

assert (
    successful_df[
        "actual_delivery_s"
    ]
    .gt(
        0.0
    )
    .all()
)


assert (
    successful_df[
        "actual_visibility"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


assert (
    successful_df[
        "actual_fidelity"
    ]
    .between(
        0.25,
        1.0
    )
    .all()
)


assert (
    successful_df[
        "actual_predicted_qber"
    ]
    .between(
        0.0,
        0.5
    )
    .all()
)


assert np.allclose(

    successful_df[
        "actual_pair_rate_per_s"
    ]
    .to_numpy(),

    (
        1.0
        /
        successful_df[
            "actual_delivery_s"
        ]
        .to_numpy()
    ),

    rtol=1e-12,

    atol=1e-12
)


# ================================================================
# Failed trajectories
# ================================================================

if len(
    failed_df
) > 0:

    assert (
        failed_df[
            "actual_delivery_s"
        ]
        .isna()
        .all()
    )


    assert (
        failed_df[
            "actual_pair_rate_per_s"
        ]
        .fillna(
            0.0
        )
        .eq(
            0.0
        )
        .all()
    )


print(
    "[PASS] Pair indices are unique."
)

print(
    "[PASS] Successful trajectory measurements are valid."
)

print(
    "[PASS] Failed service-budget trajectories are preserved."
)


print(
    "\nSuccessful:",
    len(
        successful_df
    )
)


print(
    "Failed within slot budget:",
    len(
        failed_df
    )
)


print(
    "Observed success rate:",
    f"{len(successful_df) / len(PHASE4_CAL_RAW_DF):.4%}"
)


if (
    len(
        PHASE4_CAL_RAW_DF
    )
    ==
    PHASE4_CAL_TOTAL_TRAJECTORIES
):

    print(
        "\n[PASS] COMPLETE 640-TRAJECTORY "
        "CALIBRATION DATASET."
    )

In [ ]:
# @title Configuration-Level Monte Carlo Summary

PHASE4_CAL_SUMMARY_DF = (

    PHASE4_CAL_RAW_DF

    .groupby(

        [
            "config_index",
            "total_distance_km",
            "repeater_count",
            "hop_count",
            "mode_count",
            "elementary_distance_km",
            "predicted_visibility",
            "predicted_fidelity",
            "predicted_qber",
            "delivery_proxy_s",
            "memory_coherence_cost"
        ],

        as_index=False
    )

    .agg(

        completed_trials=(
            "pair_index",
            "count"
        ),

        mean_actual_delivery_s=(
            "actual_delivery_s",
            "mean"
        ),

        std_actual_delivery_s=(
            "actual_delivery_s",
            "std"
        ),

        median_actual_delivery_s=(
            "actual_delivery_s",
            "median"
        ),

        min_actual_delivery_s=(
            "actual_delivery_s",
            "min"
        ),

        max_actual_delivery_s=(
            "actual_delivery_s",
            "max"
        ),

        mean_actual_pair_rate_per_s=(
            "actual_pair_rate_per_s",
            "mean"
        ),

        mean_actual_visibility=(
            "actual_visibility",
            "mean"
        ),

        std_actual_visibility=(
            "actual_visibility",
            "std"
        ),

        mean_actual_fidelity=(
            "actual_fidelity",
            "mean"
        ),

        mean_actual_qber=(
            "actual_predicted_qber",
            "mean"
        ),

        mean_generation_cycles=(
            "elementary_generation_cycles",
            "mean"
        ),

        mean_optical_attempts=(
            "optical_mode_attempts",
            "mean"
        ),

        mean_bsm_attempts=(
            "bsm_attempts",
            "mean"
        )
    )
)


display(
    PHASE4_CAL_SUMMARY_DF.head(
        20
    )
)


print(
    "\nConfigurations summarized:",
    len(
        PHASE4_CAL_SUMMARY_DF
    )
)

In [ ]:
# @title  Save Calibration Datasets

PHASE4_CAL_RAW_DF.to_csv(

    PHASE4_CAL_FINAL_CSV_PATH,

    index=False
)


PHASE4_CAL_SUMMARY_DF.to_csv(

    PHASE4_CAL_SUMMARY_CSV_PATH,

    index=False
)


print(
    "Raw dataset:"
)

print(
    PHASE4_CAL_FINAL_CSV_PATH
)


print(
    "\nConfiguration summary:"
)

print(
    PHASE4_CAL_SUMMARY_CSV_PATH
)


print(
    "\n[PASS] Calibration datasets saved."
)

In [ ]:
# @title  Raw Calibration Dataset

import math
import numpy as np
import pandas as pd


# ================================================================
# OPTION A:
# If PHASE4_CAL_RAW_DF already exists in memory, it is used.
#
# OPTION B:
# Otherwise, point this at the saved raw CSV.
# ================================================================

if (
    "PHASE4_CAL_RAW_DF"
    in globals()
    and isinstance(
        PHASE4_CAL_RAW_DF,
        pd.DataFrame
    )
):

    phase4_raw = (
        PHASE4_CAL_RAW_DF
        .copy()
    )

else:

    # Change this only if your actual path differs.
    PHASE4_CAL_RAW_INPUT_PATH = (
        "/content/drive/MyDrive/"
        "NARAD_QC/checkpoints/"
        "phase4_routing_calibration/"
        "phase4_calibration_raw.csv"
    )


    phase4_raw = (
        pd.read_csv(
            PHASE4_CAL_RAW_INPUT_PATH
        )
    )


print(
    "Raw rows:",
    len(
        phase4_raw
    )
)


print(
    "Raw columns:",
    len(
        phase4_raw.columns
    )
)


display(
    phase4_raw.head()
)

In [ ]:
# @title Canonical Delivery Outcome

required_columns = {

    "config_index",
    "trial_index",
    "pair_index",
    "total_distance_km",
    "repeater_count",
    "hop_count",
    "mode_count",
    "elementary_distance_km",
    "predicted_visibility",
    "predicted_fidelity",
    "predicted_qber",
    "delivery_proxy_s",
    "memory_coherence_cost",
    "actual_delivery_s",
    "actual_pair_rate_per_s",
    "actual_fidelity",
    "actual_visibility",
    "actual_predicted_qber"
}


missing_columns = (

    required_columns
    -
    set(
        phase4_raw.columns
    )
)


if missing_columns:

    raise RuntimeError(

        "Raw calibration dataset is missing required columns: "

        +

        ", ".join(
            sorted(
                missing_columns
            )
        )
    )


# ================================================================
# Canonical physical delivery-success definition
# ================================================================

_actual_delivery = pd.to_numeric(

    phase4_raw[
        "actual_delivery_s"
    ],

    errors="coerce"
)


phase4_raw[
    "delivery_success_canonical"
] = (

    _actual_delivery.notna()

    &

    np.isfinite(
        _actual_delivery
    )

    &

    (
        _actual_delivery
        >
        0.0
    )
)


# ================================================================
# Failure label
# ================================================================

phase4_raw[
    "delivery_failure_canonical"
] = (

    ~phase4_raw[
        "delivery_success_canonical"
    ]
)


# ================================================================
# Effective throughput:
#
# successful trajectory -> actual 1/T
# failed trajectory     -> 0
#
# This is NOT conditional on success.
# ================================================================

phase4_raw[
    "effective_pair_rate_per_s"
] = np.where(

    phase4_raw[
        "delivery_success_canonical"
    ],

    pd.to_numeric(
        phase4_raw[
            "actual_pair_rate_per_s"
        ],
        errors="coerce"
    ),

    0.0
)


print(
    "Canonical successes:",
    int(
        phase4_raw[
            "delivery_success_canonical"
        ]
        .sum()
    )
)


print(
    "Canonical failures:",
    int(
        phase4_raw[
            "delivery_failure_canonical"
        ]
        .sum()
    )
)


print(
    "Observed global service success:",
    f"{phase4_raw['delivery_success_canonical'].mean():.4%}"
)


print(
    "\n[PASS] Success/failure outcome canonicalized."
)

In [ ]:
# @title Raw Dataset Integrity Audit

print(
    "=" * 100
)

print(
    "RAW CALIBRATION INTEGRITY"
)

print(
    "=" * 100
)


assert (
    len(
        phase4_raw
    )
    ==
    PHASE4_CAL_TOTAL_TRAJECTORIES
)


assert (
    phase4_raw[
        "pair_index"
    ]
    .is_unique
)


# Every configuration should contain exactly the intended
# number of Monte Carlo trials.
trial_counts = (

    phase4_raw

    .groupby(
        "config_index"
    )

    .size()
)


assert (
    trial_counts
    .eq(
        PHASE4_CAL_TRIALS_PER_CONFIG
    )
    .all()
)


assert (
    phase4_raw[
        "config_index"
    ]
    .nunique()

    ==

    len(
        PHASE4_CAL_CONFIGS
    )
)


successful_raw = (

    phase4_raw[
        phase4_raw[
            "delivery_success_canonical"
        ]
    ]

    .copy()
)


failed_raw = (

    phase4_raw[
        phase4_raw[
            "delivery_failure_canonical"
        ]
    ]

    .copy()
)


assert (
    successful_raw[
        "actual_delivery_s"
    ]
    .gt(
        0.0
    )
    .all()
)


assert (
    successful_raw[
        "actual_visibility"
    ]
    .between(
        0.0,
        1.0
    )
    .all()
)


assert (
    successful_raw[
        "actual_fidelity"
    ]
    .between(
        0.25,
        1.0
    )
    .all()
)


print(
    "[PASS] Exactly",
    PHASE4_CAL_TOTAL_TRAJECTORIES,
    "trajectories are present."
)


print(
    "[PASS] Every pair index is unique."
)


print(
    "[PASS] Every configuration contains exactly",
    PHASE4_CAL_TRIALS_PER_CONFIG,
    "trials."
)


print(
    "[PASS] Successful physical outputs are valid."
)

In [ ]:
# @title  Wilson Success-Probability Interval

def phase4_wilson_interval(
    successes,
    trials,
    z=1.959963984540054
):
    """
    Two-sided Wilson score interval for a Bernoulli proportion.
    """

    successes = int(
        successes
    )

    trials = int(
        trials
    )


    if trials <= 0:

        return (
            float("nan"),
            float("nan")
        )


    p_hat = (

        successes
        /
        trials
    )


    z2 = (
        z
        *
        z
    )


    denominator = (

        1.0

        +

        z2
        /
        trials
    )


    center = (

        p_hat

        +

        z2
        /
        (
            2.0
            *
            trials
        )
    ) / denominator


    half_width = (

        z

        *

        math.sqrt(

            (
                p_hat
                *
                (
                    1.0
                    -
                    p_hat
                )
                /
                trials
            )

            +

            (
                z2
                /
                (
                    4.0
                    *
                    trials
                    *
                    trials
                )
            )
        )

        /

        denominator
    )


    return (

        max(
            0.0,
            center - half_width
        ),

        min(
            1.0,
            center + half_width
        )
    )


# Sanity tests
lo, hi = (
    phase4_wilson_interval(
        8,
        8
    )
)


assert (
    0.0
    <=
    lo
    <=
    1.0
)


assert (
    0.0
    <=
    hi
    <=
    1.0
)


assert (
    lo
    <=
    hi
)


print(
    "[PASS] Wilson confidence interval helper ready."
)

In [ ]:
# @title  Configuration Summary Builder

def phase4_summarize_calibration_config(
    group
):

    group = group.copy()


    successes = (

        group[
            group[
                "delivery_success_canonical"
            ]
        ]

        .copy()
    )


    total_trials = int(
        len(
            group
        )
    )


    successful_trials = int(
        len(
            successes
        )
    )


    failed_trials = (

        total_trials
        -
        successful_trials
    )


    service_success_probability = (

        successful_trials

        /

        total_trials
    )


    (
        success_ci_low,
        success_ci_high
    ) = phase4_wilson_interval(

        successful_trials,

        total_trials
    )


    # ============================================================
    # Helper for success-conditioned statistics
    # ============================================================

    def success_stat(
        column,
        statistic
    ):

        if successful_trials == 0:

            return float(
                "nan"
            )


        series = pd.to_numeric(

            successes[
                column
            ],

            errors="coerce"
        )


        if statistic == "mean":

            return float(
                series.mean()
            )


        if statistic == "std":

            return float(
                series.std(
                    ddof=1
                )
            )


        if statistic == "median":

            return float(
                series.median()
            )


        if statistic == "min":

            return float(
                series.min()
            )


        if statistic == "max":

            return float(
                series.max()
            )


        if statistic == "q25":

            return float(
                series.quantile(
                    0.25
                )
            )


        if statistic == "q75":

            return float(
                series.quantile(
                    0.75
                )
            )


        raise ValueError(
            f"Unsupported statistic: {statistic}"
        )


    first = (
        group.iloc[0]
    )


    # ============================================================
    # Proxy bias relative to actual successful delivery.
    # ============================================================

    proxy_delivery_s = float(
        first[
            "delivery_proxy_s"
        ]
    )


    mean_actual_delivery_s = (
        success_stat(
            "actual_delivery_s",
            "mean"
        )
    )


    median_actual_delivery_s = (
        success_stat(
            "actual_delivery_s",
            "median"
        )
    )


    if (
        successful_trials > 0
        and proxy_delivery_s > 0.0
    ):

        mean_actual_to_proxy_ratio = (

            mean_actual_delivery_s

            /

            proxy_delivery_s
        )


        median_actual_to_proxy_ratio = (

            median_actual_delivery_s

            /

            proxy_delivery_s
        )


    else:

        mean_actual_to_proxy_ratio = (
            float("nan")
        )

        median_actual_to_proxy_ratio = (
            float("nan")
        )


    # ============================================================
    # Delivered visibility loss relative to fresh prediction
    # ============================================================

    mean_actual_visibility = (
        success_stat(
            "actual_visibility",
            "mean"
        )
    )


    if successful_trials > 0:

        mean_visibility_delta = (

            mean_actual_visibility

            -

            float(
                first[
                    "predicted_visibility"
                ]
            )
        )

    else:

        mean_visibility_delta = (
            float("nan")
        )


    return pd.Series(
        {

            # ----------------------------------------------------
            # Configuration identity
            # ----------------------------------------------------

            "config_index":
                int(
                    first[
                        "config_index"
                    ]
                ),

            "total_distance_km":
                float(
                    first[
                        "total_distance_km"
                    ]
                ),

            "repeater_count":
                int(
                    first[
                        "repeater_count"
                    ]
                ),

            "hop_count":
                int(
                    first[
                        "hop_count"
                    ]
                ),

            "mode_count":
                int(
                    first[
                        "mode_count"
                    ]
                ),

            "elementary_distance_km":
                float(
                    first[
                        "elementary_distance_km"
                    ]
                ),


            # ----------------------------------------------------
            # Pre-execution router prediction
            # ----------------------------------------------------

            "single_mode_success_probability":
                float(
                    first[
                        "single_mode_success_probability"
                    ]
                ),

            "multiplexed_cycle_success_probability":
                float(
                    first[
                        "multiplexed_cycle_success_probability"
                    ]
                ),

            "predicted_visibility":
                float(
                    first[
                        "predicted_visibility"
                    ]
                ),

            "predicted_fidelity":
                float(
                    first[
                        "predicted_fidelity"
                    ]
                ),

            "predicted_qber":
                float(
                    first[
                        "predicted_qber"
                    ]
                ),

            "generation_proxy_s":
                float(
                    first[
                        "generation_proxy_s"
                    ]
                ),

            "delivery_proxy_s":
                proxy_delivery_s,

            "memory_coherence_cost":
                float(
                    first[
                        "memory_coherence_cost"
                    ]
                ),


            # ----------------------------------------------------
            # Finite service-budget outcome
            # ----------------------------------------------------

            "total_trials":
                total_trials,

            "successful_trials":
                successful_trials,

            "failed_trials":
                failed_trials,

            "service_success_probability":
                float(
                    service_success_probability
                ),

            "service_success_ci95_low":
                float(
                    success_ci_low
                ),

            "service_success_ci95_high":
                float(
                    success_ci_high
                ),


            # ----------------------------------------------------
            # Delivery time | success
            # ----------------------------------------------------

            "mean_actual_delivery_s":
                mean_actual_delivery_s,

            "std_actual_delivery_s":
                success_stat(
                    "actual_delivery_s",
                    "std"
                ),

            "median_actual_delivery_s":
                median_actual_delivery_s,

            "q25_actual_delivery_s":
                success_stat(
                    "actual_delivery_s",
                    "q25"
                ),

            "q75_actual_delivery_s":
                success_stat(
                    "actual_delivery_s",
                    "q75"
                ),

            "min_actual_delivery_s":
                success_stat(
                    "actual_delivery_s",
                    "min"
                ),

            "max_actual_delivery_s":
                success_stat(
                    "actual_delivery_s",
                    "max"
                ),


            # ----------------------------------------------------
            # Pair-rate statistics
            # ----------------------------------------------------

            "mean_successful_pair_rate_per_s":
                success_stat(
                    "actual_pair_rate_per_s",
                    "mean"
                ),

            # Failed trials are explicitly zero here.
            "mean_effective_pair_rate_per_s":
                float(
                    group[
                        "effective_pair_rate_per_s"
                    ]
                    .mean()
                ),


            # ----------------------------------------------------
            # Delivered quality | success
            # ----------------------------------------------------

            "mean_actual_visibility":
                mean_actual_visibility,

            "std_actual_visibility":
                success_stat(
                    "actual_visibility",
                    "std"
                ),

            "mean_actual_fidelity":
                success_stat(
                    "actual_fidelity",
                    "mean"
                ),

            "mean_actual_qber":
                success_stat(
                    "actual_predicted_qber",
                    "mean"
                ),


            # ----------------------------------------------------
            # Work | success
            # ----------------------------------------------------

            "mean_generation_cycles":
                success_stat(
                    "elementary_generation_cycles",
                    "mean"
                ),

            "mean_optical_attempts":
                success_stat(
                    "optical_mode_attempts",
                    "mean"
                ),

            "mean_bsm_attempts":
                success_stat(
                    "bsm_attempts",
                    "mean"
                ),


            # ----------------------------------------------------
            # Proxy-calibration diagnostics
            # ----------------------------------------------------

            "mean_actual_to_proxy_ratio":
                float(
                    mean_actual_to_proxy_ratio
                ),

            "median_actual_to_proxy_ratio":
                float(
                    median_actual_to_proxy_ratio
                ),

            "mean_visibility_delta":
                float(
                    mean_visibility_delta
                )
        }
    )


print(
    "[PASS] Failure-aware configuration summarizer ready."
)

In [ ]:
# @title  Improved Calibration Summary

PHASE4_CAL_IMPROVED_SUMMARY_DF = (

    phase4_raw

    .groupby(
        "config_index",
        sort=True,
        group_keys=False
    ).apply(
        phase4_summarize_calibration_config,
        include_groups=True
    ).reset_index(
        drop=True
    ).sort_values(
        "config_index"
    ).reset_index(
        drop=True
    )
)


assert (
    len(
        PHASE4_CAL_IMPROVED_SUMMARY_DF
    )

    ==
    len(
        PHASE4_CAL_CONFIGS
    )
)


display(
    PHASE4_CAL_IMPROVED_SUMMARY_DF
)


print(
    "\n[PASS] Improved configuration summary created."
)

In [ ]:
# @title  Service-Budget Failure Audit

PHASE4_CAL_FAILURE_AUDIT_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[

        PHASE4_CAL_IMPROVED_SUMMARY_DF[
            "failed_trials"
        ]
        >
        0
    ]

    [
        [
            "config_index",
            "total_distance_km",
            "repeater_count",
            "hop_count",
            "mode_count",
            "elementary_distance_km",

            "successful_trials",
            "failed_trials",
            "service_success_probability",
            "service_success_ci95_low",
            "service_success_ci95_high",

            "median_actual_delivery_s",
            "mean_actual_delivery_s",
            "mean_effective_pair_rate_per_s",

            "predicted_visibility",
            "mean_actual_visibility",

            "delivery_proxy_s",
            "median_actual_to_proxy_ratio"
        ]
    ]

    .sort_values(

        by=[
            "service_success_probability",
            "total_distance_km",
            "repeater_count",
            "mode_count"
        ],

        ascending=[
            True,
            False,
            True,
            True
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_CAL_FAILURE_AUDIT_DF
)


print(
    "\nConfigurations with >=1 service-budget failure:",
    len(
        PHASE4_CAL_FAILURE_AUDIT_DF
    )
)

In [ ]:
# @title  Calibration Analysis Table

PHASE4_CAL_ANALYSIS_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        [
            "config_index",

            "total_distance_km",
            "repeater_count",
            "hop_count",
            "mode_count",
            "elementary_distance_km",

            "multiplexed_cycle_success_probability",
            "predicted_visibility",
            "predicted_qber",
            "delivery_proxy_s",
            "memory_coherence_cost",

            "successful_trials",
            "failed_trials",
            "service_success_probability",

            "median_actual_delivery_s",
            "mean_actual_delivery_s",
            "mean_effective_pair_rate_per_s",

            "mean_actual_visibility",
            "mean_actual_fidelity",
            "mean_actual_qber",

            "median_actual_to_proxy_ratio",
            "mean_visibility_delta"
        ]
    ]

    .copy()
)


display(
    PHASE4_CAL_ANALYSIS_DF
)


print(
    "\n[PASS] Compact analysis dataset ready."
)

In [ ]:
# @title  Global Calibration Outcome Summary

total_trials = int(
    len(
        phase4_raw
    )
)


total_successes = int(

    phase4_raw[
        "delivery_success_canonical"
    ]
    .sum()
)


total_failures = (

    total_trials
    -
    total_successes
)


global_success_probability = (

    total_successes
    /
    total_trials
)


(
    global_ci_low,
    global_ci_high
) = phase4_wilson_interval(

    total_successes,
    total_trials
)


print(
    "=" * 100
)

print(
    "NARAD-QC PHASE 4 CALIBRATION — GLOBAL OUTCOME"
)

print(
    "=" * 100
)


print(
    "Total trajectories :",
    total_trials
)


print(
    "Successful         :",
    total_successes
)


print(
    "Budget failures    :",
    total_failures
)


print(
    "Service success    :",
    f"{global_success_probability:.4%}"
)


print(
    "Wilson 95% CI      :",
    f"[{global_ci_low:.4%}, "
    f"{global_ci_high:.4%}]"
)


print(
    "\nConfigurations with failures:",
    int(

        (
            PHASE4_CAL_IMPROVED_SUMMARY_DF[
                "failed_trials"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Completely failed configurations:",
    int(

        (
            PHASE4_CAL_IMPROVED_SUMMARY_DF[
                "successful_trials"
            ]
            ==
            0
        )
        .sum()
    )
)


print(
    "\n[PASS] Global finite-budget service outcome computed."
)

In [ ]:
# @title  Save Improved Summary

from pathlib import Path


if (
    "PHASE4_CAL_CHECKPOINT_DIR"
    in globals()
):

    improved_output_dir = Path(
        PHASE4_CAL_CHECKPOINT_DIR
    )

else:

    improved_output_dir = Path(
        "/content"
    )


PHASE4_CAL_IMPROVED_SUMMARY_PATH = (

    improved_output_dir
    /
    "phase4_calibration_summary_improved.csv"
)


PHASE4_CAL_FAILURE_AUDIT_PATH = (

    improved_output_dir
    /
    "phase4_calibration_failure_audit.csv"
)


PHASE4_CAL_IMPROVED_SUMMARY_DF.to_csv(

    PHASE4_CAL_IMPROVED_SUMMARY_PATH,

    index=False
)


PHASE4_CAL_FAILURE_AUDIT_DF.to_csv(

    PHASE4_CAL_FAILURE_AUDIT_PATH,

    index=False
)


print(
    "Improved summary:"
)

print(
    PHASE4_CAL_IMPROVED_SUMMARY_PATH
)


print(
    "\nFailure audit:"
)

print(
    PHASE4_CAL_FAILURE_AUDIT_PATH
)


print(
    "\n[PASS] Failure-aware calibration artifacts saved."
)

In [ ]:
# @title  Calibration Feature / Outcome Sets

PHASE4_CAL_PRIMITIVE_FEATURES = [

    "total_distance_km",
    "repeater_count",
    "mode_count",
    "elementary_distance_km"
]


PHASE4_CAL_DERIVED_FEATURES = [

    "single_mode_success_probability",
    "multiplexed_cycle_success_probability",

    "predicted_visibility",
    "predicted_qber",

    "delivery_proxy_s",
    "memory_coherence_cost"
]


PHASE4_CAL_OUTCOMES = [

    "service_success_probability",

    "median_actual_delivery_s",

    "mean_effective_pair_rate_per_s",

    "mean_actual_visibility",

    "mean_visibility_delta"
]


PHASE4_CAL_DIAGNOSTIC_COLUMNS = (

    PHASE4_CAL_PRIMITIVE_FEATURES

    +

    PHASE4_CAL_DERIVED_FEATURES

    +

    PHASE4_CAL_OUTCOMES
)


PHASE4_CAL_DIAGNOSTIC_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        PHASE4_CAL_DIAGNOSTIC_COLUMNS
    ]

    .copy()
)


print(
    "[PASS] Calibration diagnostic feature set ready."
)

In [ ]:
# @title  Feature Correlation Audit

PHASE4_CAL_PEARSON_CORR = (

    PHASE4_CAL_DIAGNOSTIC_DF
    .corr(
        method="pearson"
    )
)


PHASE4_CAL_SPEARMAN_CORR = (

    PHASE4_CAL_DIAGNOSTIC_DF
    .corr(
        method="spearman"
    )
)


print(
    "=" * 100
)

print(
    "PEARSON CORRELATION"
)

print(
    "=" * 100
)


display(
    PHASE4_CAL_PEARSON_CORR.round(
        4
    )
)


print(
    "\n" + "=" * 100
)

print(
    "SPEARMAN CORRELATION"
)

print(
    "=" * 100
)


display(
    PHASE4_CAL_SPEARMAN_CORR.round(
        4
    )
)

In [ ]:
# @title  Redundant Feature Pair Detection

PHASE4_REDUNDANCY_THRESHOLD = 0.95


routing_feature_columns = (

    PHASE4_CAL_PRIMITIVE_FEATURES

    +

    PHASE4_CAL_DERIVED_FEATURES
)


redundant_pairs = []


for i, feature_a in enumerate(
    routing_feature_columns
):

    for feature_b in (
        routing_feature_columns[
            i + 1:
        ]
    ):

        pearson = float(

            PHASE4_CAL_PEARSON_CORR.loc[
                feature_a,
                feature_b
            ]
        )


        spearman = float(

            PHASE4_CAL_SPEARMAN_CORR.loc[
                feature_a,
                feature_b
            ]
        )


        if (

            abs(
                spearman
            )

            >=

            PHASE4_REDUNDANCY_THRESHOLD
        ):

            redundant_pairs.append(
                {

                    "feature_a":
                        feature_a,

                    "feature_b":
                        feature_b,

                    "pearson":
                        pearson,

                    "spearman":
                        spearman
                }
            )


PHASE4_REDUNDANT_FEATURE_PAIRS_DF = (

    pd.DataFrame(
        redundant_pairs
    )

    .sort_values(

        by="spearman",

        key=lambda s:
            s.abs(),

        ascending=False
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_REDUNDANT_FEATURE_PAIRS_DF
)


print(
    "\n[PASS] Highly redundant routing features identified."
)

In [ ]:
# @title  Feature-to-Outcome Correlation Table

feature_outcome_rows = []


for feature in routing_feature_columns:

    row = {
        "feature":
            feature
    }


    for outcome in PHASE4_CAL_OUTCOMES:

        row[
            f"pearson__{outcome}"
        ] = float(

            PHASE4_CAL_PEARSON_CORR.loc[
                feature,
                outcome
            ]
        )


        row[
            f"spearman__{outcome}"
        ] = float(

            PHASE4_CAL_SPEARMAN_CORR.loc[
                feature,
                outcome
            ]
        )


    feature_outcome_rows.append(
        row
    )


PHASE4_FEATURE_OUTCOME_CORR_DF = (

    pd.DataFrame(
        feature_outcome_rows
    )
)


display(
    PHASE4_FEATURE_OUTCOME_CORR_DF.round(
        4
    )
)


print(
    "[PASS] Feature-to-service relationship table created."
)

In [ ]:
# @title  Repeater Response Surface

PHASE4_REPEATER_RESPONSE_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        [
            "total_distance_km",
            "mode_count",
            "repeater_count",
            "hop_count",
            "elementary_distance_km",

            "service_success_probability",
            "median_actual_delivery_s",
            "mean_effective_pair_rate_per_s",

            "mean_actual_visibility"
        ]
    ]

    .sort_values(
        [
            "total_distance_km",
            "mode_count",
            "repeater_count"
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_REPEATER_RESPONSE_DF
)

In [ ]:
# @title  Empirical Best Repeater Count

best_rows = []


for (
    distance_km,
    mode_count
), group in (

    PHASE4_CAL_IMPROVED_SUMMARY_DF

    .groupby(
        [
            "total_distance_km",
            "mode_count"
        ]
    )
):

    valid = (

        group[
            group[
                "successful_trials"
            ]
            >
            0
        ]

        .copy()
    )


    if len(
        valid
    ) == 0:

        continue


    best_index = (

        valid[
            "median_actual_delivery_s"
        ]
        .idxmin()
    )


    best = (

        valid.loc[
            best_index
        ]
    )


    best_rows.append(
        {

            "total_distance_km":
                float(
                    distance_km
                ),

            "mode_count":
                int(
                    mode_count
                ),

            "best_repeater_count":
                int(
                    best[
                        "repeater_count"
                    ]
                ),

            "best_hop_count":
                int(
                    best[
                        "hop_count"
                    ]
                ),

            "best_elementary_distance_km":
                float(
                    best[
                        "elementary_distance_km"
                    ]
                ),

            "best_median_delivery_s":
                float(
                    best[
                        "median_actual_delivery_s"
                    ]
                ),

            "service_success_probability":
                float(
                    best[
                        "service_success_probability"
                    ]
                )
        }
    )


PHASE4_EMPIRICAL_BEST_REPEATERS_DF = (

    pd.DataFrame(
        best_rows
    )

    .sort_values(
        [
            "total_distance_km",
            "mode_count"
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_EMPIRICAL_BEST_REPEATERS_DF
)

In [ ]:
# @title  Elementary Span Response

PHASE4_ELEMENTARY_SPAN_RESPONSE_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        [
            "elementary_distance_km",
            "mode_count",
            "repeater_count",
            "total_distance_km",

            "single_mode_success_probability",
            "multiplexed_cycle_success_probability",

            "service_success_probability",
            "median_actual_delivery_s",
            "mean_effective_pair_rate_per_s",
            "mean_actual_visibility"
        ]
    ]

    .sort_values(
        [
            "elementary_distance_km",
            "mode_count"
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_ELEMENTARY_SPAN_RESPONSE_DF
)

In [ ]:
# @title  Delivery Proxy Error

PHASE4_PROXY_ERROR_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        PHASE4_CAL_IMPROVED_SUMMARY_DF[
            "successful_trials"
        ]
        >
        0
    ]

    [
        [
            "config_index",

            "total_distance_km",
            "repeater_count",
            "mode_count",
            "elementary_distance_km",

            "delivery_proxy_s",
            "median_actual_delivery_s",
            "mean_actual_delivery_s",

            "median_actual_to_proxy_ratio",
            "mean_actual_to_proxy_ratio",

            "service_success_probability"
        ]
    ]

    .copy()
)


PHASE4_PROXY_ERROR_DF[
    "median_proxy_relative_error"
] = (

    (
        PHASE4_PROXY_ERROR_DF[
            "delivery_proxy_s"
        ]

        -

        PHASE4_PROXY_ERROR_DF[
            "median_actual_delivery_s"
        ]
    )

    /

    PHASE4_PROXY_ERROR_DF[
        "median_actual_delivery_s"
    ]
)


display(

    PHASE4_PROXY_ERROR_DF

    .sort_values(
        "median_actual_to_proxy_ratio",
        ascending=False
    )
)

In [ ]:
# @title  Delivered Visibility Degradation

PHASE4_VISIBILITY_DEGRADATION_DF = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        PHASE4_CAL_IMPROVED_SUMMARY_DF[
            "successful_trials"
        ]
        >
        0
    ]

    [
        [
            "config_index",

            "total_distance_km",
            "repeater_count",
            "mode_count",
            "elementary_distance_km",

            "predicted_visibility",
            "mean_actual_visibility",
            "mean_visibility_delta",

            "median_actual_delivery_s",
            "memory_coherence_cost"
        ]
    ]

    .copy()

    .sort_values(
        "mean_visibility_delta"
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_VISIBILITY_DEGRADATION_DF
)

In [ ]:
# @title  Save Diagnostic Tables

from pathlib import Path


if (
    "PHASE4_CAL_CHECKPOINT_DIR"
    in globals()
):

    phase4_diag_dir = Path(
        PHASE4_CAL_CHECKPOINT_DIR
    )

else:

    phase4_diag_dir = Path(
        "/content"
    )


PHASE4_REDUNDANT_FEATURE_PAIRS_DF.to_csv(

    phase4_diag_dir
    /
    "phase4_redundant_feature_pairs.csv",

    index=False
)


PHASE4_FEATURE_OUTCOME_CORR_DF.to_csv(

    phase4_diag_dir
    /
    "phase4_feature_outcome_correlations.csv",

    index=False
)


PHASE4_EMPIRICAL_BEST_REPEATERS_DF.to_csv(

    phase4_diag_dir
    /
    "phase4_empirical_best_repeaters.csv",

    index=False
)


PHASE4_PROXY_ERROR_DF.to_csv(

    phase4_diag_dir
    /
    "phase4_proxy_error_analysis.csv",

    index=False
)


PHASE4_VISIBILITY_DEGRADATION_DF.to_csv(

    phase4_diag_dir
    /
    "phase4_visibility_degradation.csv",

    index=False
)


print(
    "[PASS] Diagnostics saved."
)

#### Correlation Study

In [ ]:
# @title  Cross-Validated Calibration Models: Setup

import math
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures
)

from sklearn.linear_model import (
    Ridge,
    LogisticRegression
)

from sklearn.model_selection import (
    KFold,
    GroupKFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    log_loss,
    brier_score_loss
)


# ================================================================
# Minimal physical predictor basis
# ================================================================

PHASE4_CAL_MODEL_FEATURES = [

    "elementary_distance_km",
    "repeater_count",
    "multiplexed_cycle_success_probability"
]


# ================================================================
# Reproducible CV configuration
# ================================================================

PHASE4_CAL_CV_SPLITS = 5

PHASE4_CAL_CV_RANDOM_SEED = 4_200_001


print(
    "Policy-C calibration feature basis:"
)

for feature in PHASE4_CAL_MODEL_FEATURES:

    print(
        "  -",
        feature
    )


print(
    "\n[PASS] Model setup ready."
)

In [ ]:
# @title Prepare Latency and Visibility-Loss Datasets

phase4_config_model_df = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF[
        PHASE4_CAL_IMPROVED_SUMMARY_DF[
            "successful_trials"
        ]
        >
        0
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


# ================================================================
# Latency target
# ================================================================

phase4_config_model_df[
    "target_log_median_delivery"
] = np.log(

    phase4_config_model_df[
        "median_actual_delivery_s"
    ]
)


# ================================================================
# Quality target
#
# Negative means actual delivered visibility is worse than
# fresh predicted visibility.
# ================================================================

phase4_config_model_df[
    "target_visibility_delta"
] = (

    phase4_config_model_df[
        "mean_actual_visibility"
    ]

    -

    phase4_config_model_df[
        "predicted_visibility"
    ]
)


assert (
    np.isfinite(
        phase4_config_model_df[
            "target_log_median_delivery"
        ]
    )
    .all()
)


assert (
    np.isfinite(
        phase4_config_model_df[
            "target_visibility_delta"
        ]
    )
    .all()
)


print(
    "Latency/quality configurations:",
    len(
        phase4_config_model_df
    )
)


print(
    "All-failure configurations excluded from conditional models:",
    len(
        PHASE4_CAL_IMPROVED_SUMMARY_DF
    )
    -
    len(
        phase4_config_model_df
    )
)


print(
    "\n[PASS] Conditional service datasets prepared."
)

In [ ]:
# @title  Candidate Regression Models

PHASE4_REGRESSION_MODELS = {

    "linear_ridge": Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "ridge",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    ),

    "quadratic_ridge": Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),

            (
                "scale",
                StandardScaler()
            ),

            (
                "ridge",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    )
}


print(
    "[PASS] Candidate regression models defined."
)

In [ ]:
# @title  Cross-Validated Regression Evaluator

def phase4_cross_validate_regression(
    dataframe,
    feature_columns,
    target_column,
    models,
    splits=5,
    random_seed=4_200_001
):

    X = (

        dataframe[
            feature_columns
        ]

        .to_numpy(
            dtype=float
        )
    )


    y = (

        dataframe[
            target_column
        ]

        .to_numpy(
            dtype=float
        )
    )


    cv = KFold(

        n_splits=(
            splits
        ),

        shuffle=True,

        random_state=(
            random_seed
        )
    )


    model_rows = []

    prediction_frames = []


    for model_name, model_template in models.items():

        oof_pred = np.full(

            len(
                dataframe
            ),

            np.nan,

            dtype=float
        )


        fold_rows = []


        for fold_index, (
            train_index,
            test_index
        ) in enumerate(
            cv.split(
                X
            ),
            start=1
        ):

            model = clone(
                model_template
            )


            model.fit(

                X[
                    train_index
                ],

                y[
                    train_index
                ]
            )


            fold_pred = model.predict(

                X[
                    test_index
                ]
            )


            oof_pred[
                test_index
            ] = fold_pred


            fold_rows.append(
                {

                    "model":
                        model_name,

                    "fold":
                        fold_index,

                    "mae":
                        mean_absolute_error(
                            y[
                                test_index
                            ],
                            fold_pred
                        ),

                    "rmse":
                        math.sqrt(
                            mean_squared_error(
                                y[
                                    test_index
                                ],
                                fold_pred
                            )
                        ),

                    "r2":
                        r2_score(
                            y[
                                test_index
                            ],
                            fold_pred
                        )
                }
            )


        assert (
            np.isfinite(
                oof_pred
            )
            .all()
        )


        overall_mae = (
            mean_absolute_error(
                y,
                oof_pred
            )
        )


        overall_rmse = math.sqrt(

            mean_squared_error(
                y,
                oof_pred
            )
        )


        overall_r2 = (
            r2_score(
                y,
                oof_pred
            )
        )


        model_rows.append(
            {

                "model":
                    model_name,

                "oof_mae":
                    overall_mae,

                "oof_rmse":
                    overall_rmse,

                "oof_r2":
                    overall_r2,

                "fold_mae_mean":
                    float(
                        np.mean(
                            [
                                row[
                                    "mae"
                                ]
                                for row
                                in fold_rows
                            ]
                        )
                    ),

                "fold_mae_std":
                    float(
                        np.std(
                            [
                                row[
                                    "mae"
                                ]
                                for row
                                in fold_rows
                            ],
                            ddof=1
                        )
                    ),

                "fold_rmse_mean":
                    float(
                        np.mean(
                            [
                                row[
                                    "rmse"
                                ]
                                for row
                                in fold_rows
                            ]
                        )
                    ),

                "fold_rmse_std":
                    float(
                        np.std(
                            [
                                row[
                                    "rmse"
                                ]
                                for row
                                in fold_rows
                            ],
                            ddof=1
                        )
                    )
            }
        )


        pred_frame = dataframe[
            [
                "config_index",
                *feature_columns
            ]
        ].copy()


        pred_frame[
            "model"
        ] = model_name


        pred_frame[
            "actual_target"
        ] = y


        pred_frame[
            "oof_prediction"
        ] = oof_pred


        prediction_frames.append(
            pred_frame
        )


    results_df = (

        pd.DataFrame(
            model_rows
        )

        .sort_values(
            "oof_rmse"
        )

        .reset_index(
            drop=True
        )
    )


    predictions_df = (

        pd.concat(
            prediction_frames,
            ignore_index=True
        )
    )


    return (
        results_df,
        predictions_df
    )


print(
    "[PASS] Regression CV evaluator ready."
)

In [ ]:
# @title  Cross-Validated Latency Models

(
    PHASE4_LATENCY_MODEL_RESULTS_DF,
    PHASE4_LATENCY_OOF_DF
) = phase4_cross_validate_regression(

    dataframe=(
        phase4_config_model_df
    ),

    feature_columns=(
        PHASE4_CAL_MODEL_FEATURES
    ),

    target_column=(
        "target_log_median_delivery"
    ),

    models=(
        PHASE4_REGRESSION_MODELS
    ),

    splits=(
        PHASE4_CAL_CV_SPLITS
    ),

    random_seed=(
        PHASE4_CAL_CV_RANDOM_SEED
    )
)


# ================================================================
# Back-transform log predictions
# ================================================================

PHASE4_LATENCY_OOF_DF[
    "actual_delivery_s"
] = np.exp(

    PHASE4_LATENCY_OOF_DF[
        "actual_target"
    ]
)


PHASE4_LATENCY_OOF_DF[
    "predicted_delivery_s"
] = np.exp(

    PHASE4_LATENCY_OOF_DF[
        "oof_prediction"
    ]
)


PHASE4_LATENCY_SECONDS_RESULTS = []


for model_name, group in (

    PHASE4_LATENCY_OOF_DF

    .groupby(
        "model"
    )
):

    actual = (

        group[
            "actual_delivery_s"
        ]
        .to_numpy()
    )


    predicted = (

        group[
            "predicted_delivery_s"
        ]
        .to_numpy()
    )


    PHASE4_LATENCY_SECONDS_RESULTS.append(
        {

            "model":
                model_name,

            "mae_seconds":
                mean_absolute_error(
                    actual,
                    predicted
                ),

            "rmse_seconds":
                math.sqrt(
                    mean_squared_error(
                        actual,
                        predicted
                    )
                ),

            "median_absolute_percentage_error":
                float(
                    np.median(
                        np.abs(
                            predicted
                            -
                            actual
                        )
                        /
                        actual
                    )
                )
        }
    )


PHASE4_LATENCY_SECONDS_RESULTS_DF = (

    pd.DataFrame(
        PHASE4_LATENCY_SECONDS_RESULTS
    )

    .sort_values(
        "rmse_seconds"
    )

    .reset_index(
        drop=True
    )
)


print(
    "=" * 100
)

print(
    "LATENCY MODEL — LOG-SPACE CV"
)

print(
    "=" * 100
)


display(
    PHASE4_LATENCY_MODEL_RESULTS_DF
)


print(
    "\n" + "=" * 100
)

print(
    "LATENCY MODEL — BACK-TRANSFORMED SECONDS"
)

print(
    "=" * 100
)


display(
    PHASE4_LATENCY_SECONDS_RESULTS_DF
)

In [ ]:
# @title  Cross-Validated Visibility-Loss Models

(
    PHASE4_VISIBILITY_MODEL_RESULTS_DF,
    PHASE4_VISIBILITY_OOF_DF
) = phase4_cross_validate_regression(

    dataframe=(
        phase4_config_model_df
    ),

    feature_columns=(
        PHASE4_CAL_MODEL_FEATURES
    ),

    target_column=(
        "target_visibility_delta"
    ),

    models=(
        PHASE4_REGRESSION_MODELS
    ),

    splits=(
        PHASE4_CAL_CV_SPLITS
    ),

    random_seed=(
        PHASE4_CAL_CV_RANDOM_SEED
        +
        101
    )
)


print(
    "=" * 100
)

print(
    "VISIBILITY-LOSS MODEL — CROSS VALIDATION"
)

print(
    "=" * 100
)


display(
    PHASE4_VISIBILITY_MODEL_RESULTS_DF
)

In [ ]:
# @title  Prepare Reliability Dataset

phase4_reliability_df = (

    phase4_raw[
        [
            "config_index",

            "elementary_distance_km",
            "repeater_count",
            "multiplexed_cycle_success_probability",

            "delivery_success_canonical"
        ]
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


phase4_reliability_df[
    "reliability_target"
] = (

    phase4_reliability_df[
        "delivery_success_canonical"
    ]

    .astype(
        int
    )
)


print(
    "Reliability trajectories:",
    len(
        phase4_reliability_df
    )
)


print(
    "Successes:",
    int(
        phase4_reliability_df[
            "reliability_target"
        ]
        .sum()
    )
)


print(
    "Failures:",
    int(
        (
            1
            -
            phase4_reliability_df[
                "reliability_target"
            ]
        )
        .sum()
    )
)


print(
    "\n[PASS] Reliability dataset prepared."
)

In [ ]:
# @title  Reliability Candidate Models

PHASE4_RELIABILITY_MODELS = {

    "linear_logistic": Pipeline(
        [
            (
                "scale",
                StandardScaler()
            ),

            (
                "logistic",
                LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=10_000,
                    solver="lbfgs"
                )
            )
        ]
    ),

    "quadratic_logistic": Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),

            (
                "scale",
                StandardScaler()
            ),

            (
                "logistic",
                LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=10_000,
                    solver="lbfgs"
                )
            )
        ]
    )
}


print(
    "[PASS] Reliability candidate models ready."
)

In [ ]:
# @title  Grouped Reliability Cross Validation

def phase4_cross_validate_reliability(
    dataframe,
    feature_columns,
    target_column,
    group_column,
    models,
    splits=5
):

    X = (

        dataframe[
            feature_columns
        ]

        .to_numpy(
            dtype=float
        )
    )


    y = (

        dataframe[
            target_column
        ]

        .to_numpy(
            dtype=int
        )
    )


    groups = (

        dataframe[
            group_column
        ]

        .to_numpy()
    )


    cv = GroupKFold(
        n_splits=(
            splits
        )
    )


    result_rows = []

    prediction_frames = []


    for model_name, model_template in models.items():

        oof_probability = np.full(

            len(
                dataframe
            ),

            np.nan,

            dtype=float
        )


        for (
            train_index,
            test_index
        ) in cv.split(
            X,
            y,
            groups
        ):

            model = clone(
                model_template
            )


            model.fit(

                X[
                    train_index
                ],

                y[
                    train_index
                ]
            )


            probabilities = (

                model.predict_proba(
                    X[
                        test_index
                    ]
                )[
                    :,
                    1
                ]
            )


            oof_probability[
                test_index
            ] = probabilities


        assert (
            np.isfinite(
                oof_probability
            )
            .all()
        )


        # --------------------------------------------------------
        # Metrics
        # --------------------------------------------------------

        auc = (
            roc_auc_score(
                y,
                oof_probability
            )
        )


        brier = (
            brier_score_loss(
                y,
                oof_probability
            )
        )


        ll = (
            log_loss(
                y,
                oof_probability,
                labels=[
                    0,
                    1
                ]
            )
        )


        result_rows.append(
            {

                "model":
                    model_name,

                "oof_roc_auc":
                    auc,

                "oof_brier_score":
                    brier,

                "oof_log_loss":
                    ll
            }
        )


        pred_df = dataframe[
            [
                "config_index",
                *feature_columns,
                target_column
            ]
        ].copy()


        pred_df[
            "model"
        ] = model_name


        pred_df[
            "oof_success_probability"
        ] = oof_probability


        prediction_frames.append(
            pred_df
        )


    results_df = (

        pd.DataFrame(
            result_rows
        )

        .sort_values(
            [
                "oof_brier_score",
                "oof_log_loss"
            ]
        )

        .reset_index(
            drop=True
        )
    )


    predictions_df = (

        pd.concat(
            prediction_frames,
            ignore_index=True
        )
    )


    return (
        results_df,
        predictions_df
    )


(
    PHASE4_RELIABILITY_MODEL_RESULTS_DF,
    PHASE4_RELIABILITY_OOF_DF
) = phase4_cross_validate_reliability(

    dataframe=(
        phase4_reliability_df
    ),

    feature_columns=(
        PHASE4_CAL_MODEL_FEATURES
    ),

    target_column=(
        "reliability_target"
    ),

    group_column=(
        "config_index"
    ),

    models=(
        PHASE4_RELIABILITY_MODELS
    ),

    splits=(
        PHASE4_CAL_CV_SPLITS
    )
)


display(
    PHASE4_RELIABILITY_MODEL_RESULTS_DF
)

In [ ]:
# @title  Configuration-Level Reliability Calibration

reliability_config_rows = []


for model_name, model_group in (

    PHASE4_RELIABILITY_OOF_DF

    .groupby(
        "model"
    )
):

    grouped = (

        model_group

        .groupby(
            "config_index",
            as_index=False
        )

        .agg(

            observed_success_probability=(
                "reliability_target",
                "mean"
            ),

            predicted_success_probability=(
                "oof_success_probability",
                "mean"
            ),

            elementary_distance_km=(
                "elementary_distance_km",
                "first"
            ),

            repeater_count=(
                "repeater_count",
                "first"
            ),

            multiplexed_cycle_success_probability=(
                "multiplexed_cycle_success_probability",
                "first"
            )
        )
    )


    grouped[
        "model"
    ] = model_name


    grouped[
        "absolute_probability_error"
    ] = np.abs(

        grouped[
            "predicted_success_probability"
        ]

        -

        grouped[
            "observed_success_probability"
        ]
    )


    reliability_config_rows.append(
        grouped
    )


PHASE4_RELIABILITY_CONFIG_OOF_DF = (

    pd.concat(
        reliability_config_rows,
        ignore_index=True
    )
)


PHASE4_RELIABILITY_CONFIG_RESULTS = []


for model_name, group in (

    PHASE4_RELIABILITY_CONFIG_OOF_DF

    .groupby(
        "model"
    )
):

    PHASE4_RELIABILITY_CONFIG_RESULTS.append(
        {

            "model":
                model_name,

            "config_probability_mae":
                mean_absolute_error(

                    group[
                        "observed_success_probability"
                    ],

                    group[
                        "predicted_success_probability"
                    ]
                ),

            "config_probability_rmse":
                math.sqrt(

                    mean_squared_error(

                        group[
                            "observed_success_probability"
                        ],

                        group[
                            "predicted_success_probability"
                        ]
                    )
                )
        }
    )


PHASE4_RELIABILITY_CONFIG_RESULTS_DF = (

    pd.DataFrame(
        PHASE4_RELIABILITY_CONFIG_RESULTS
    )

    .sort_values(
        "config_probability_rmse"
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4_RELIABILITY_CONFIG_RESULTS_DF
)

In [ ]:
# @title  Select Cross-Validated Winning Models

PHASE4_BEST_LATENCY_MODEL_NAME = str(

    PHASE4_LATENCY_MODEL_RESULTS_DF
    .iloc[0][
        "model"
    ]
)


PHASE4_BEST_VISIBILITY_MODEL_NAME = str(

    PHASE4_VISIBILITY_MODEL_RESULTS_DF
    .iloc[0][
        "model"
    ]
)


PHASE4_BEST_RELIABILITY_MODEL_NAME = str(

    PHASE4_RELIABILITY_CONFIG_RESULTS_DF
    .iloc[0][
        "model"
    ]
)


print(
    "Best latency model    :",
    PHASE4_BEST_LATENCY_MODEL_NAME
)


print(
    "Best visibility model :",
    PHASE4_BEST_VISIBILITY_MODEL_NAME
)


print(
    "Best reliability model:",
    PHASE4_BEST_RELIABILITY_MODEL_NAME
)


print(
    "\n[PASS] Winning model family chosen strictly from out-of-fold performance."
)

In [ ]:
# @title  Fit Final Calibration Models

PHASE4_FINAL_LATENCY_MODEL = clone(

    PHASE4_REGRESSION_MODELS[
        PHASE4_BEST_LATENCY_MODEL_NAME
    ]
)


PHASE4_FINAL_VISIBILITY_MODEL = clone(

    PHASE4_REGRESSION_MODELS[
        PHASE4_BEST_VISIBILITY_MODEL_NAME
    ]
)


PHASE4_FINAL_RELIABILITY_MODEL = clone(

    PHASE4_RELIABILITY_MODELS[
        PHASE4_BEST_RELIABILITY_MODEL_NAME
    ]
)


# ================================================================
# Latency
# ================================================================

X_config = (

    phase4_config_model_df[
        PHASE4_CAL_MODEL_FEATURES
    ]

    .to_numpy(
        dtype=float
    )
)


PHASE4_FINAL_LATENCY_MODEL.fit(

    X_config,

    phase4_config_model_df[
        "target_log_median_delivery"
    ]
    .to_numpy(
        dtype=float
    )
)


# ================================================================
# Visibility loss
# ================================================================

PHASE4_FINAL_VISIBILITY_MODEL.fit(

    X_config,

    phase4_config_model_df[
        "target_visibility_delta"
    ]
    .to_numpy(
        dtype=float
    )
)


# ================================================================
# Reliability
# ================================================================

X_reliability = (

    phase4_reliability_df[
        PHASE4_CAL_MODEL_FEATURES
    ]

    .to_numpy(
        dtype=float
    )
)


PHASE4_FINAL_RELIABILITY_MODEL.fit(

    X_reliability,

    phase4_reliability_df[
        "reliability_target"
    ]
    .to_numpy(
        dtype=int
    )
)


print(
    "[PASS] Final calibration models fitted."
)

In [ ]:
# @title  Unified Empirical Service Predictor

def phase4_policy_c_predict_service(
    elementary_distance_km,
    repeater_count,
    multiplexed_cycle_success_probability,
    fresh_predicted_visibility
):

    feature_vector = np.array(
        [[
            float(
                elementary_distance_km
            ),

            float(
                repeater_count
            ),

            float(
                multiplexed_cycle_success_probability
            )
        ]],
        dtype=float
    )


    # ============================================================
    # Reliability
    # ================================================================

    predicted_service_success_probability = float(

        PHASE4_FINAL_RELIABILITY_MODEL
        .predict_proba(
            feature_vector
        )[0, 1]
    )


    # ============================================================
    # Conditional delivery latency
    # ================================================================

    predicted_log_delivery = float(

        PHASE4_FINAL_LATENCY_MODEL
        .predict(
            feature_vector
        )[0]
    )


    predicted_median_delivery_s = float(

        math.exp(
            predicted_log_delivery
        )
    )


    # ============================================================
    # Delivered visibility
    # ================================================================

    predicted_visibility_delta = float(

        PHASE4_FINAL_VISIBILITY_MODEL
        .predict(
            feature_vector
        )[0]
    )


    predicted_delivered_visibility = float(

        fresh_predicted_visibility

        +

        predicted_visibility_delta
    )


    predicted_delivered_visibility = float(

        np.clip(

            predicted_delivered_visibility,

            0.0,

            1.0
        )
    )


    predicted_delivered_qber = float(

        (
            1.0
            -
            predicted_delivered_visibility
        )

        /
        2.0
    )


    return {

        "predicted_service_success_probability":
            predicted_service_success_probability,

        "predicted_median_delivery_s_given_success":
            predicted_median_delivery_s,

        "predicted_visibility_delta":
            predicted_visibility_delta,

        "predicted_delivered_visibility":
            predicted_delivered_visibility,

        "predicted_delivered_qber":
            predicted_delivered_qber
    }


print(
    "[PASS] Unified Policy-C service predictor ready."
)

In [ ]:
# @title  Policy-C Predictor Sanity Test

_test_row = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF
    .iloc[
        0
    ]
)


PHASE4_POLICY_C_TEST_PREDICTION = (

    phase4_policy_c_predict_service(

        elementary_distance_km=(
            _test_row[
                "elementary_distance_km"
            ]
        ),

        repeater_count=(
            _test_row[
                "repeater_count"
            ]
        ),

        multiplexed_cycle_success_probability=(
            _test_row[
                "multiplexed_cycle_success_probability"
            ]
        ),

        fresh_predicted_visibility=(
            _test_row[
                "predicted_visibility"
            ]
        )
    )
)


print(
    "Observed config:"
)

print(
    "  service success =",
    _test_row[
        "service_success_probability"
    ]
)

print(
    "  median delivery =",
    _test_row[
        "median_actual_delivery_s"
    ]
)

print(
    "  actual visibility =",
    _test_row[
        "mean_actual_visibility"
    ]
)


print(
    "\nPolicy-C prediction:"
)

for key, value in (
    PHASE4_POLICY_C_TEST_PREDICTION.items()
):

    print(
        f"  {key} = {value}"
    )


assert (
    0.0
    <=
    PHASE4_POLICY_C_TEST_PREDICTION[
        "predicted_service_success_probability"
    ]
    <=
    1.0
)


assert (
    PHASE4_POLICY_C_TEST_PREDICTION[
        "predicted_median_delivery_s_given_success"
    ]
    >
    0.0
)


assert (
    0.0
    <=
    PHASE4_POLICY_C_TEST_PREDICTION[
        "predicted_delivered_visibility"
    ]
    <=
    1.0
)


print(
    "\n[PASS] Policy-C service predictor sanity checks passed."
)

In [ ]:
# @title  Model Coefficient Audit

def phase4_extract_model_coefficients(
    fitted_pipeline,
    base_feature_names
):

    pipeline_steps = (
        fitted_pipeline.named_steps
    )


    if "poly" in pipeline_steps:

        feature_names = (

            pipeline_steps[
                "poly"
            ]
            .get_feature_names_out(
                base_feature_names
            )
        )

    else:

        feature_names = np.array(
            base_feature_names
        )


    if "ridge" in pipeline_steps:

        estimator = (
            pipeline_steps[
                "ridge"
            ]
        )

    elif "logistic" in pipeline_steps:

        estimator = (
            pipeline_steps[
                "logistic"
            ]
        )

    else:

        raise RuntimeError(
            "Unsupported fitted estimator."
        )


    coefficient_array = np.ravel(
        estimator.coef_
    )


    result_df = pd.DataFrame(
        {

            "feature":
                feature_names,

            "standardized_coefficient":
                coefficient_array,

            "absolute_coefficient":
                np.abs(
                    coefficient_array
                )
        }
    )


    result_df = (

        result_df

        .sort_values(
            "absolute_coefficient",
            ascending=False
        )

        .reset_index(
            drop=True
        )
    )


    return result_df


PHASE4_LATENCY_COEFFICIENTS_DF = (

    phase4_extract_model_coefficients(

        PHASE4_FINAL_LATENCY_MODEL,

        PHASE4_CAL_MODEL_FEATURES
    )
)


PHASE4_VISIBILITY_COEFFICIENTS_DF = (

    phase4_extract_model_coefficients(

        PHASE4_FINAL_VISIBILITY_MODEL,

        PHASE4_CAL_MODEL_FEATURES
    )
)


PHASE4_RELIABILITY_COEFFICIENTS_DF = (

    phase4_extract_model_coefficients(

        PHASE4_FINAL_RELIABILITY_MODEL,

        PHASE4_CAL_MODEL_FEATURES
    )
)


print(
    "=" * 100
)

print(
    "LATENCY MODEL COEFFICIENTS"
)

print(
    "=" * 100
)


display(
    PHASE4_LATENCY_COEFFICIENTS_DF
)


print(
    "\n" + "=" * 100
)

print(
    "VISIBILITY-LOSS MODEL COEFFICIENTS"
)

print(
    "=" * 100
)


display(
    PHASE4_VISIBILITY_COEFFICIENTS_DF
)


print(
    "\n" + "=" * 100
)

print(
    "RELIABILITY MODEL COEFFICIENTS"
)

print(
    "=" * 100
)


display(
    PHASE4_RELIABILITY_COEFFICIENTS_DF
)

In [ ]:
# @title  Save Cross-Validation Results

if (
    "PHASE4_CAL_CHECKPOINT_DIR"
    in globals()
):

    phase4_model_dir = Path(
        PHASE4_CAL_CHECKPOINT_DIR
    )

else:

    phase4_model_dir = Path(
        "/content"
    )


PHASE4_LATENCY_MODEL_RESULTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_latency_model_cv.csv",

    index=False
)


PHASE4_LATENCY_SECONDS_RESULTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_latency_model_seconds_cv.csv",

    index=False
)


PHASE4_VISIBILITY_MODEL_RESULTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_visibility_model_cv.csv",

    index=False
)


PHASE4_RELIABILITY_MODEL_RESULTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_reliability_model_cv.csv",

    index=False
)


PHASE4_RELIABILITY_CONFIG_RESULTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_reliability_config_cv.csv",

    index=False
)


PHASE4_LATENCY_COEFFICIENTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_latency_coefficients.csv",

    index=False
)


PHASE4_VISIBILITY_COEFFICIENTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_visibility_coefficients.csv",

    index=False
)


PHASE4_RELIABILITY_COEFFICIENTS_DF.to_csv(

    phase4_model_dir
    /
    "phase4_reliability_coefficients.csv",

    index=False
)


print(
    "[PASS] All diagnostics saved."
)

In [ ]:
# @title  Correct Reliability Coefficient Export

# ================================================================
# Refit explicitly using the selected linear logistic model
# ================================================================

PHASE4_FINAL_RELIABILITY_MODEL = clone(

    PHASE4_RELIABILITY_MODELS[
        "linear_logistic"
    ]
)


X_reliability = (

    phase4_reliability_df[
        PHASE4_CAL_MODEL_FEATURES
    ]

    .to_numpy(
        dtype=float
    )
)


y_reliability = (

    phase4_reliability_df[
        "reliability_target"
    ]

    .to_numpy(
        dtype=int
    )
)


PHASE4_FINAL_RELIABILITY_MODEL.fit(

    X_reliability,

    y_reliability
)


# ================================================================
# Extract standardized linear coefficients
# ================================================================

PHASE4_RELIABILITY_LINEAR_COEFFICIENTS_DF = (

    phase4_extract_model_coefficients(

        PHASE4_FINAL_RELIABILITY_MODEL,

        PHASE4_CAL_MODEL_FEATURES
    )
)


display(
    PHASE4_RELIABILITY_LINEAR_COEFFICIENTS_DF
)


assert (
    len(
        PHASE4_RELIABILITY_LINEAR_COEFFICIENTS_DF
    )

    ==

    len(
        PHASE4_CAL_MODEL_FEATURES
    )
)


expected_features = set(
    PHASE4_CAL_MODEL_FEATURES
)


assert (
    set(
        PHASE4_RELIABILITY_LINEAR_COEFFICIENTS_DF[
            "feature"
        ]
    )

    ==

    expected_features
)


print(
    "[PASS] Reliability coefficients now correspond "
    "to the frozen linear-logistic model."
)

In [ ]:
# @title Reliability Calibration Setup

import math
import numpy as np
import pandas as pd

from sklearn.base import clone

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.linear_model import (
    LogisticRegression
)

from sklearn.model_selection import (
    GroupKFold
)

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss
)


# ================================================================
# Frozen reliability feature basis
# ================================================================

PHASE4E4_RELIABILITY_FEATURES = [

    "elementary_distance_km",
    "repeater_count",
    "multiplexed_cycle_success_probability"
]


# ================================================================
# Cross-validation configuration
# ================================================================

PHASE4E4_OUTER_SPLITS = 5

PHASE4E4_INNER_SPLITS = 4


# ================================================================
# Base reliability model
#
# This is the model already selected in Phase 4E.3.
# ================================================================

PHASE4E4_BASE_RELIABILITY_MODEL = Pipeline(
    [
        (
            "scale",
            StandardScaler()
        ),

        (
            "logistic",
            LogisticRegression(

                C=1.0,

                class_weight="balanced",

                max_iter=10_000,

                solver="lbfgs"
            )
        )
    ]
)


print(
    "=" * 96
)

print(
    "NARAD-QC PHASE 4E.4 — RELIABILITY CALIBRATION"
)

print(
    "=" * 96
)


print(
    "Features:"
)

for feature in PHASE4E4_RELIABILITY_FEATURES:

    print(
        "  -",
        feature
    )


print(
    "\nOuter grouped folds:",
    PHASE4E4_OUTER_SPLITS
)


print(
    "Inner grouped folds:",
    PHASE4E4_INNER_SPLITS
)


print(
    "\n[PASS] Configuration ready."
)

In [ ]:
# @title  Reliability Calibration Dataset

required_columns = {

    "config_index",

    "elementary_distance_km",
    "repeater_count",
    "multiplexed_cycle_success_probability",

    "delivery_success_canonical"
}


missing = (

    required_columns
    -
    set(
        phase4_raw.columns
    )
)


if missing:

    raise RuntimeError(

        "Missing required reliability columns: "

        +

        ", ".join(
            sorted(
                missing
            )
        )
    )


PHASE4E4_RELIABILITY_DF = (

    phase4_raw[
        [
            "config_index",

            *PHASE4E4_RELIABILITY_FEATURES,

            "delivery_success_canonical"
        ]
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


PHASE4E4_RELIABILITY_DF[
    "target_success"
] = (

    PHASE4E4_RELIABILITY_DF[
        "delivery_success_canonical"
    ]

    .astype(
        int
    )
)


assert (
    len(
        PHASE4E4_RELIABILITY_DF
    )
    ==
    640
)


assert (
    PHASE4E4_RELIABILITY_DF[
        "config_index"
    ]
    .nunique()
    ==
    80
)


print(
    "Trajectories:",
    len(
        PHASE4E4_RELIABILITY_DF
    )
)


print(
    "Configurations:",
    PHASE4E4_RELIABILITY_DF[
        "config_index"
    ]
    .nunique()
)


print(
    "Successes:",
    int(
        PHASE4E4_RELIABILITY_DF[
            "target_success"
        ]
        .sum()
    )
)


print(
    "Failures:",
    int(
        (
            1
            -
            PHASE4E4_RELIABILITY_DF[
                "target_success"
            ]
        )
        .sum()
    )
)


print(
    "\n[PASS] Reliability calibration dataset ready."
)

In [ ]:
# @title  Platt Probability Calibrator

def phase4e4_fit_platt_calibrator(
    decision_scores,
    targets
):

    decision_scores = np.asarray(
        decision_scores,
        dtype=float
    ).reshape(
        -1,
        1
    )


    targets = np.asarray(
        targets,
        dtype=int
    )


    if (
        len(
            np.unique(
                targets
            )
        )
        !=
        2
    ):

        raise RuntimeError(
            "Platt calibration requires both success and failure samples."
        )


    # Very weak regularization:
    # effectively ordinary sigmoid calibration.
    calibrator = LogisticRegression(

        C=1_000_000.0,

        class_weight=None,

        max_iter=10_000,

        solver="lbfgs"
    )


    calibrator.fit(

        decision_scores,

        targets
    )


    return calibrator


print(
    "[PASS] Platt calibration helper ready."
)

In [ ]:
# @title  Nested Grouped Probability Calibration

def phase4e4_nested_grouped_calibration(
    dataframe,
    feature_columns,
    target_column,
    group_column,
    base_model,
    outer_splits=5,
    inner_splits=4
):

    X = (

        dataframe[
            feature_columns
        ]

        .to_numpy(
            dtype=float
        )
    )


    y = (

        dataframe[
            target_column
        ]

        .to_numpy(
            dtype=int
        )
    )


    groups = (

        dataframe[
            group_column
        ]

        .to_numpy()
    )


    raw_oof_probability = np.full(

        len(
            dataframe
        ),

        np.nan,

        dtype=float
    )


    calibrated_oof_probability = np.full(

        len(
            dataframe
        ),

        np.nan,

        dtype=float
    )


    raw_oof_score = np.full(

        len(
            dataframe
        ),

        np.nan,

        dtype=float
    )


    outer_fold_id = np.full(

        len(
            dataframe
        ),

        -1,

        dtype=int
    )


    outer_cv = GroupKFold(

        n_splits=(
            outer_splits
        )
    )


    for outer_fold, (
        outer_train_idx,
        outer_test_idx
    ) in enumerate(

        outer_cv.split(
            X,
            y,
            groups
        ),

        start=1
    ):

        X_outer_train = X[
            outer_train_idx
        ]


        y_outer_train = y[
            outer_train_idx
        ]


        groups_outer_train = groups[
            outer_train_idx
        ]


        # ========================================================
        # INNER OOF SCORES
        #
        # These train the probability calibrator without allowing
        # the base classifier to score its own training samples.
        # ========================================================

        inner_oof_scores = np.full(

            len(
                outer_train_idx
            ),

            np.nan,

            dtype=float
        )


        inner_cv = GroupKFold(

            n_splits=(
                inner_splits
            )
        )


        for (
            inner_train_local,
            inner_valid_local
        ) in inner_cv.split(

            X_outer_train,

            y_outer_train,

            groups_outer_train
        ):

            inner_model = clone(
                base_model
            )


            inner_model.fit(

                X_outer_train[
                    inner_train_local
                ],

                y_outer_train[
                    inner_train_local
                ]
            )


            inner_oof_scores[
                inner_valid_local
            ] = (

                inner_model
                .decision_function(

                    X_outer_train[
                        inner_valid_local
                    ]
                )
            )


        assert (
            np.isfinite(
                inner_oof_scores
            )
            .all()
        )


        # ========================================================
        # Fit Platt sigmoid using natural training prevalence.
        # ========================================================

        calibrator = (
            phase4e4_fit_platt_calibrator(

                inner_oof_scores,

                y_outer_train
            )
        )


        # ========================================================
        # Train base reliability model on full outer training set.
        # ========================================================

        outer_model = clone(
            base_model
        )


        outer_model.fit(

            X[
                outer_train_idx
            ],

            y[
                outer_train_idx
            ]
        )


        outer_scores = (

            outer_model
            .decision_function(

                X[
                    outer_test_idx
                ]
            )
        )


        outer_raw_probability = (

            outer_model
            .predict_proba(

                X[
                    outer_test_idx
                ]
            )[
                :,
                1
            ]
        )


        outer_calibrated_probability = (

            calibrator
            .predict_proba(

                outer_scores.reshape(
                    -1,
                    1
                )
            )[
                :,
                1
            ]
        )


        raw_oof_score[
            outer_test_idx
        ] = outer_scores


        raw_oof_probability[
            outer_test_idx
        ] = outer_raw_probability


        calibrated_oof_probability[
            outer_test_idx
        ] = outer_calibrated_probability


        outer_fold_id[
            outer_test_idx
        ] = outer_fold


    assert (
        np.isfinite(
            raw_oof_probability
        )
        .all()
    )


    assert (
        np.isfinite(
            calibrated_oof_probability
        )
        .all()
    )


    output = dataframe.copy()


    output[
        "outer_fold"
    ] = outer_fold_id


    output[
        "raw_oof_decision_score"
    ] = raw_oof_score


    output[
        "raw_oof_success_probability"
    ] = raw_oof_probability


    output[
        "calibrated_oof_success_probability"
    ] = calibrated_oof_probability


    return output


PHASE4E4_OOF_DF = (

    phase4e4_nested_grouped_calibration(

        dataframe=(
            PHASE4E4_RELIABILITY_DF
        ),

        feature_columns=(
            PHASE4E4_RELIABILITY_FEATURES
        ),

        target_column=(
            "target_success"
        ),

        group_column=(
            "config_index"
        ),

        base_model=(
            PHASE4E4_BASE_RELIABILITY_MODEL
        ),

        outer_splits=(
            PHASE4E4_OUTER_SPLITS
        ),

        inner_splits=(
            PHASE4E4_INNER_SPLITS
        )
    )
)


print(
    "[PASS] Nested grouped OOF calibration completed."
)

In [ ]:
# @title  Raw vs Calibrated Reliability Performance

y_true = (

    PHASE4E4_OOF_DF[
        "target_success"
    ]

    .to_numpy(
        dtype=int
    )
)


raw_probability = (

    PHASE4E4_OOF_DF[
        "raw_oof_success_probability"
    ]

    .to_numpy(
        dtype=float
    )
)


calibrated_probability = (

    PHASE4E4_OOF_DF[
        "calibrated_oof_success_probability"
    ]

    .to_numpy(
        dtype=float
    )
)


PHASE4E4_CALIBRATION_METRICS_DF = pd.DataFrame(
    [
        {

            "probability_type":
                "raw_balanced_logistic",

            "roc_auc":
                roc_auc_score(
                    y_true,
                    raw_probability
                ),

            "brier_score":
                brier_score_loss(
                    y_true,
                    raw_probability
                ),

            "log_loss":
                log_loss(
                    y_true,
                    raw_probability,
                    labels=[
                        0,
                        1
                    ]
                )
        },

        {

            "probability_type":
                "nested_platt_calibrated",

            "roc_auc":
                roc_auc_score(
                    y_true,
                    calibrated_probability
                ),

            "brier_score":
                brier_score_loss(
                    y_true,
                    calibrated_probability
                ),

            "log_loss":
                log_loss(
                    y_true,
                    calibrated_probability,
                    labels=[
                        0,
                        1
                    ]
                )
        }
    ]
)


display(
    PHASE4E4_CALIBRATION_METRICS_DF
)


print(
    "[PASS] Raw/calibrated probability comparison complete."
)

In [ ]:
# @title Configuration-Level Calibrated Reliability

PHASE4E4_CONFIG_RELIABILITY_DF = (

    PHASE4E4_OOF_DF

    .groupby(
        "config_index",
        as_index=False
    )

    .agg(

        observed_service_success_probability=(
            "target_success",
            "mean"
        ),

        successful_trials=(
            "target_success",
            "sum"
        ),

        total_trials=(
            "target_success",
            "count"
        ),

        calibrated_service_probability=(
            "calibrated_oof_success_probability",
            "mean"
        ),

        raw_service_probability=(
            "raw_oof_success_probability",
            "mean"
        ),

        elementary_distance_km=(
            "elementary_distance_km",
            "first"
        ),

        repeater_count=(
            "repeater_count",
            "first"
        ),

        multiplexed_cycle_success_probability=(
            "multiplexed_cycle_success_probability",
            "first"
        ),

        outer_fold=(
            "outer_fold",
            "first"
        )
    )
)


PHASE4E4_CONFIG_RELIABILITY_DF[
    "failed_trials"
] = (

    PHASE4E4_CONFIG_RELIABILITY_DF[
        "total_trials"
    ]

    -

    PHASE4E4_CONFIG_RELIABILITY_DF[
        "successful_trials"
    ]
)


PHASE4E4_CONFIG_RELIABILITY_DF[
    "absolute_probability_error"
] = np.abs(

    PHASE4E4_CONFIG_RELIABILITY_DF[
        "calibrated_service_probability"
    ]

    -

    PHASE4E4_CONFIG_RELIABILITY_DF[
        "observed_service_success_probability"
    ]
)


display(

    PHASE4E4_CONFIG_RELIABILITY_DF

    .sort_values(
        "calibrated_service_probability"
    )

    .reset_index(
        drop=True
    )
)


config_mae = (

    PHASE4E4_CONFIG_RELIABILITY_DF[
        "absolute_probability_error"
    ]
    .mean()
)


config_rmse = math.sqrt(

    np.mean(

        (
            PHASE4E4_CONFIG_RELIABILITY_DF[
                "calibrated_service_probability"
            ]

            -

            PHASE4E4_CONFIG_RELIABILITY_DF[
                "observed_service_success_probability"
            ]
        )
        **
        2
    )
)


print(
    "\nConfiguration probability MAE :",
    f"{config_mae:.6f}"
)


print(
    "Configuration probability RMSE:",
    f"{config_rmse:.6f}"
)

In [ ]:
# @title  Reliability Calibration Table

calibration_table_source = (

    PHASE4E4_CONFIG_RELIABILITY_DF
    .copy()
)


# ================================================================
# Quantile bins
# ================================================================

calibration_table_source[
    "probability_bin"
] = pd.qcut(

    calibration_table_source[
        "calibrated_service_probability"
    ],

    q=5,

    duplicates="drop"
)


PHASE4E4_RELIABILITY_TABLE_DF = (

    calibration_table_source

    .groupby(
        "probability_bin",
        observed=False,
        as_index=False
    )

    .agg(

        configurations=(
            "config_index",
            "count"
        ),

        mean_predicted_probability=(
            "calibrated_service_probability",
            "mean"
        ),

        mean_observed_probability=(
            "observed_service_success_probability",
            "mean"
        ),

        total_successes=(
            "successful_trials",
            "sum"
        ),

        total_trials=(
            "total_trials",
            "sum"
        )
    )
)


PHASE4E4_RELIABILITY_TABLE_DF[
    "pooled_observed_probability"
] = (

    PHASE4E4_RELIABILITY_TABLE_DF[
        "total_successes"
    ]

    /

    PHASE4E4_RELIABILITY_TABLE_DF[
        "total_trials"
    ]
)


PHASE4E4_RELIABILITY_TABLE_DF[
    "absolute_calibration_gap"
] = np.abs(

    PHASE4E4_RELIABILITY_TABLE_DF[
        "mean_predicted_probability"
    ]

    -

    PHASE4E4_RELIABILITY_TABLE_DF[
        "pooled_observed_probability"
    ]
)


display(
    PHASE4E4_RELIABILITY_TABLE_DF
)

In [ ]:
# @title  Reliability Admission Threshold Sweep

def phase4e4_threshold_metrics(
    threshold,
    trajectory_df,
    config_df
):

    threshold = float(
        threshold
    )


    # ============================================================
    # TRAJECTORY LEVEL
    # ================================================================

    y = (

        trajectory_df[
            "target_success"
        ]

        .to_numpy(
            dtype=int
        )
    )


    p = (

        trajectory_df[
            "calibrated_oof_success_probability"
        ]

        .to_numpy(
            dtype=float
        )
    )


    admitted = (
        p
        >=
        threshold
    )


    success_mask = (
        y
        ==
        1
    )


    failure_mask = (
        y
        ==
        0
    )


    total_successes = int(
        success_mask.sum()
    )


    total_failures = int(
        failure_mask.sum()
    )


    retained_successes = int(
        (
            admitted
            &
            success_mask
        )
        .sum()
    )


    rejected_failures = int(
        (
            (~admitted)
            &
            failure_mask
        )
        .sum()
    )


    success_retention = (

        retained_successes
        /
        total_successes

        if total_successes > 0

        else float("nan")
    )


    failure_rejection = (

        rejected_failures
        /
        total_failures

        if total_failures > 0

        else float("nan")
    )


    balanced_accuracy = (

        0.5
        *
        (
            success_retention
            +
            failure_rejection
        )
    )


    youden_j = (

        success_retention
        +
        failure_rejection
        -
        1.0
    )


    admitted_trajectory_count = int(
        admitted.sum()
    )


    admitted_trajectory_success_rate = (

        float(
            y[
                admitted
            ]
            .mean()
        )

        if admitted_trajectory_count > 0

        else float("nan")
    )


    # ============================================================
    # CONFIGURATION LEVEL
    # ================================================================

    config_admitted = (

        config_df[
            "calibrated_service_probability"
        ]

        >=

        threshold
    )


    admitted_config_df = (

        config_df[
            config_admitted
        ]
    )


    admitted_config_count = int(
        len(
            admitted_config_df
        )
    )


    config_coverage = (

        admitted_config_count

        /

        len(
            config_df
        )
    )


    if admitted_config_count > 0:

        admitted_config_mean_observed_success = float(

            admitted_config_df[
                "observed_service_success_probability"
            ]
            .mean()
        )


        admitted_config_pooled_success = float(

            admitted_config_df[
                "successful_trials"
            ]
            .sum()

            /

            admitted_config_df[
                "total_trials"
            ]
            .sum()
        )


    else:

        admitted_config_mean_observed_success = (
            float("nan")
        )

        admitted_config_pooled_success = (
            float("nan")
        )


    return {

        "threshold":
            threshold,

        "success_retention":
            success_retention,

        "failure_rejection":
            failure_rejection,

        "balanced_accuracy":
            balanced_accuracy,

        "youden_j":
            youden_j,

        "trajectory_admission_fraction":
            float(
                admitted.mean()
            ),

        "admitted_trajectory_success_rate":
            admitted_trajectory_success_rate,

        "configurations_admitted":
            admitted_config_count,

        "configuration_coverage":
            config_coverage,

        "admitted_config_mean_observed_success":
            admitted_config_mean_observed_success,

        "admitted_config_pooled_success":
            admitted_config_pooled_success
    }


# ================================================================
# Candidate thresholds
#
# Include both a dense grid and all observed config probabilities.
# ================================================================

candidate_thresholds = np.unique(

    np.concatenate(
        [

            np.linspace(
                0.01,
                0.99,
                99
            ),

            PHASE4E4_CONFIG_RELIABILITY_DF[
                "calibrated_service_probability"
            ]
            .to_numpy(
                dtype=float
            )
        ]
    )
)


threshold_rows = [

    phase4e4_threshold_metrics(

        threshold,

        PHASE4E4_OOF_DF,

        PHASE4E4_CONFIG_RELIABILITY_DF
    )

    for threshold
    in candidate_thresholds
]


PHASE4E4_THRESHOLD_SWEEP_DF = (

    pd.DataFrame(
        threshold_rows
    )

    .sort_values(
        "threshold"
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4E4_THRESHOLD_SWEEP_DF
)

In [ ]:
# @title  Select Empirical Admission Threshold

PHASE4E4_THRESHOLD_RANKED_DF = (

    PHASE4E4_THRESHOLD_SWEEP_DF

    .sort_values(

        by=[
            "youden_j",
            "admitted_config_pooled_success",
            "configuration_coverage"
        ],

        ascending=[
            False,
            False,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


PHASE4E4_SELECTED_THRESHOLD_ROW = (

    PHASE4E4_THRESHOLD_RANKED_DF
    .iloc[
        0
    ]
)


PHASE4E4_SERVICE_ADMISSION_THRESHOLD = float(

    PHASE4E4_SELECTED_THRESHOLD_ROW[
        "threshold"
    ]
)


print(
    "=" * 100
)

print(
    "NARAD-QC EMPIRICAL RELIABILITY ADMISSION THRESHOLD"
)

print(
    "=" * 100
)


print(
    "Selected threshold:",
    f"{PHASE4E4_SERVICE_ADMISSION_THRESHOLD:.6f}"
)


print(
    "\nYouden J:",
    f"{PHASE4E4_SELECTED_THRESHOLD_ROW['youden_j']:.6f}"
)


print(
    "Success retention:",
    f"{PHASE4E4_SELECTED_THRESHOLD_ROW['success_retention']:.4%}"
)


print(
    "Failure rejection:",
    f"{PHASE4E4_SELECTED_THRESHOLD_ROW['failure_rejection']:.4%}"
)


print(
    "Configuration coverage:",
    f"{PHASE4E4_SELECTED_THRESHOLD_ROW['configuration_coverage']:.4%}"
)


print(
    "Observed success among admitted trajectories:",
    f"{PHASE4E4_SELECTED_THRESHOLD_ROW['admitted_trajectory_success_rate']:.4%}"
)


print(
    "Pooled observed success among admitted configurations:",
    f"{PHASE4E4_SELECTED_THRESHOLD_ROW['admitted_config_pooled_success']:.4%}"
)


print(
    "\n[PASS] Data-derived admission threshold selected."
)

In [ ]:
# @title  Admission Decision Audit

PHASE4E4_ADMISSION_AUDIT_DF = (

    PHASE4E4_CONFIG_RELIABILITY_DF
    .copy()
)


PHASE4E4_ADMISSION_AUDIT_DF[
    "reliability_admitted"
] = (

    PHASE4E4_ADMISSION_AUDIT_DF[
        "calibrated_service_probability"
    ]

    >=

    PHASE4E4_SERVICE_ADMISSION_THRESHOLD
)


PHASE4E4_ADMISSION_AUDIT_DF = (

    PHASE4E4_ADMISSION_AUDIT_DF

    .sort_values(

        by=[
            "reliability_admitted",
            "calibrated_service_probability"
        ],

        ascending=[
            True,
            True
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4E4_ADMISSION_AUDIT_DF[
        [
            "config_index",

            "elementary_distance_km",
            "repeater_count",
            "multiplexed_cycle_success_probability",

            "calibrated_service_probability",
            "observed_service_success_probability",

            "successful_trials",
            "failed_trials",

            "reliability_admitted"
        ]
    ]
)


print(
    "\nRejected configurations:",
    int(
        (
            ~PHASE4E4_ADMISSION_AUDIT_DF[
                "reliability_admitted"
            ]
        )
        .sum()
    )
)


print(
    "Admitted configurations:",
    int(
        PHASE4E4_ADMISSION_AUDIT_DF[
            "reliability_admitted"
        ]
        .sum()
    )
)

In [ ]:
# @title  Bootstrap Admission-Threshold Stability

# PHASE4E4_BOOTSTRAP_REPLICATES = 2_000

# PHASE4E4_BOOTSTRAP_SEED = 4_240_001


# rng = np.random.default_rng(

#     PHASE4E4_BOOTSTRAP_SEED
# )


# config_ids = (

#     PHASE4E4_CONFIG_RELIABILITY_DF[
#         "config_index"
#     ]

#     .to_numpy()
# )


# bootstrap_thresholds = []


# # ================================================================
# # Use the already-computed OOF probabilities.
# #
# # We are estimating threshold-selection stability, not retraining
# # the physical predictor 2,000 times.
# # ================================================================

# for bootstrap_index in range(

#     PHASE4E4_BOOTSTRAP_REPLICATES
# ):

#     sampled_config_ids = rng.choice(

#         config_ids,

#         size=len(
#             config_ids
#         ),

#         replace=True
#     )


#     sampled_trajectory_parts = []

#     sampled_config_parts = []


#     for synthetic_index, config_id in enumerate(
#         sampled_config_ids
#     ):

#         trajectory_part = (

#             PHASE4E4_OOF_DF[
#                 PHASE4E4_OOF_DF[
#                     "config_index"
#                 ]
#                 ==
#                 config_id
#             ]

#             .copy()
#         )


#         config_part = (

#             PHASE4E4_CONFIG_RELIABILITY_DF[
#                 PHASE4E4_CONFIG_RELIABILITY_DF[
#                     "config_index"
#                 ]
#                 ==
#                 config_id
#             ]

#             .copy()
#         )


#         # Give repeated sampled configurations independent
#         # bootstrap identities.
#         trajectory_part[
#             "bootstrap_config_id"
#         ] = synthetic_index


#         config_part[
#             "bootstrap_config_id"
#         ] = synthetic_index


#         sampled_trajectory_parts.append(
#             trajectory_part
#         )


#         sampled_config_parts.append(
#             config_part
#         )


#     boot_traj = pd.concat(

#         sampled_trajectory_parts,

#         ignore_index=True
#     )


#     boot_config = pd.concat(

#         sampled_config_parts,

#         ignore_index=True
#     )


#     # ------------------------------------------------------------
#     # Skip pathological bootstrap samples containing only one class.
#     # ------------------------------------------------------------

#     if (

#         boot_traj[
#             "target_success"
#         ]
#         .nunique()

#         <
#         2
#     ):

#         continue


#     boot_rows = [

#         phase4e4_threshold_metrics(

#             threshold,

#             boot_traj,

#             boot_config
#         )

#         for threshold
#         in candidate_thresholds
#     ]


#     boot_df = pd.DataFrame(
#         boot_rows
#     )


#     boot_best = (

#         boot_df

#         .sort_values(

#             by=[
#                 "youden_j",
#                 "admitted_config_pooled_success",
#                 "configuration_coverage"
#             ],

#             ascending=[
#                 False,
#                 False,
#                 False
#             ]
#         )

#         .iloc[
#             0
#         ]
#     )


#     bootstrap_thresholds.append(

#         float(
#             boot_best[
#                 "threshold"
#             ]
#         )
#     )


# PHASE4E4_BOOTSTRAP_THRESHOLDS = np.array(

#     bootstrap_thresholds,

#     dtype=float
# )


# print(
#     "Valid bootstrap replicates:",
#     len(
#         PHASE4E4_BOOTSTRAP_THRESHOLDS
#     )
# )


# print(
#     "\nThreshold median:",
#     f"{np.median(PHASE4E4_BOOTSTRAP_THRESHOLDS):.6f}"
# )


# print(
#     "Threshold IQR:",
#     (
#         f"["
#         f"{np.quantile(PHASE4E4_BOOTSTRAP_THRESHOLDS, 0.25):.6f}, "
#         f"{np.quantile(PHASE4E4_BOOTSTRAP_THRESHOLDS, 0.75):.6f}"
#         f"]"
#     )
# )


# print(
#     "Bootstrap 95% interval:",
#     (
#         f"["
#         f"{np.quantile(PHASE4E4_BOOTSTRAP_THRESHOLDS, 0.025):.6f}, "
#         f"{np.quantile(PHASE4E4_BOOTSTRAP_THRESHOLDS, 0.975):.6f}"
#         f"]"
#     )
# )


# print(
#     "\n[PASS] Admission-threshold stability analysis complete."
# )

In [ ]:
# @title  Threshold Operating Neighborhood

selected_threshold = (

    PHASE4E4_SERVICE_ADMISSION_THRESHOLD
)


PHASE4E4_THRESHOLD_NEIGHBORHOOD_DF = (

    PHASE4E4_THRESHOLD_SWEEP_DF[
        (
            PHASE4E4_THRESHOLD_SWEEP_DF[
                "threshold"
            ]

            >=

            selected_threshold
            -
            0.10
        )

        &

        (
            PHASE4E4_THRESHOLD_SWEEP_DF[
                "threshold"
            ]

            <=

            selected_threshold
            +
            0.10
        )
    ]

    .copy()

    .sort_values(
        "threshold"
    )

    .reset_index(
        drop=True
    )
)


display(
    PHASE4E4_THRESHOLD_NEIGHBORHOOD_DF
)

In [ ]:
# @title  Fit Final Calibrated Reliability Model

X_full = (

    PHASE4E4_RELIABILITY_DF[
        PHASE4E4_RELIABILITY_FEATURES
    ]

    .to_numpy(
        dtype=float
    )
)


y_full = (

    PHASE4E4_RELIABILITY_DF[
        "target_success"
    ]

    .to_numpy(
        dtype=int
    )
)


groups_full = (

    PHASE4E4_RELIABILITY_DF[
        "config_index"
    ]

    .to_numpy()
)


# ================================================================
# Generate grouped OOF scores over the complete calibration set.
# ================================================================

full_oof_scores = np.full(

    len(
        PHASE4E4_RELIABILITY_DF
    ),

    np.nan,

    dtype=float
)


full_calibration_cv = GroupKFold(

    n_splits=5
)


for (
    train_index,
    valid_index
) in full_calibration_cv.split(

    X_full,

    y_full,

    groups_full
):

    model = clone(
        PHASE4E4_BASE_RELIABILITY_MODEL
    )


    model.fit(

        X_full[
            train_index
        ],

        y_full[
            train_index
        ]
    )


    full_oof_scores[
        valid_index
    ] = (

        model
        .decision_function(

            X_full[
                valid_index
            ]
        )
    )


assert (
    np.isfinite(
        full_oof_scores
    )
    .all()
)


# ================================================================
# Final probability calibrator
# ================================================================

PHASE4_FINAL_RELIABILITY_CALIBRATOR = (

    phase4e4_fit_platt_calibrator(

        full_oof_scores,

        y_full
    )
)


# ================================================================
# Final base model
# ================================================================

PHASE4_FINAL_RELIABILITY_MODEL = clone(

    PHASE4E4_BASE_RELIABILITY_MODEL
)


PHASE4_FINAL_RELIABILITY_MODEL.fit(

    X_full,

    y_full
)


print(
    "[PASS] Final calibrated reliability model fitted."
)

In [ ]:
# @title  Calibrated Reliability Prediction API

def phase4_predict_calibrated_service_probability(
    elementary_distance_km,
    repeater_count,
    multiplexed_cycle_success_probability
):

    feature_vector = np.array(
        [[

            float(
                elementary_distance_km
            ),

            float(
                repeater_count
            ),

            float(
                multiplexed_cycle_success_probability
            )

        ]],

        dtype=float
    )


    raw_decision_score = float(

        PHASE4_FINAL_RELIABILITY_MODEL
        .decision_function(
            feature_vector
        )[0]
    )


    calibrated_probability = float(

        PHASE4_FINAL_RELIABILITY_CALIBRATOR
        .predict_proba(

            np.array(
                [[
                    raw_decision_score
                ]]
            )
        )[0, 1]
    )


    admitted = (

        calibrated_probability

        >=

        PHASE4E4_SERVICE_ADMISSION_THRESHOLD
    )


    return {

        "calibrated_service_probability":
            calibrated_probability,

        "service_admission_threshold":
            float(
                PHASE4E4_SERVICE_ADMISSION_THRESHOLD
            ),

        "reliability_admitted":
            bool(
                admitted
            )
    }


print(
    "[PASS] Calibrated reliability API ready."
)

In [ ]:
# @title  Reliability API Sanity Check

easy_config = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF

    .sort_values(

        by=[
            "service_success_probability",
            "median_actual_delivery_s"
        ],

        ascending=[
            False,
            True
        ]
    )

    .iloc[
        0
    ]
)


hard_config = (

    PHASE4_CAL_IMPROVED_SUMMARY_DF

    .sort_values(

        by=[
            "service_success_probability",
            "median_actual_delivery_s"
        ],

        ascending=[
            True,
            False
        ],

        na_position="last"
    )

    .iloc[
        0
    ]
)


easy_prediction = (

    phase4_predict_calibrated_service_probability(

        easy_config[
            "elementary_distance_km"
        ],

        easy_config[
            "repeater_count"
        ],

        easy_config[
            "multiplexed_cycle_success_probability"
        ]
    )
)


hard_prediction = (

    phase4_predict_calibrated_service_probability(

        hard_config[
            "elementary_distance_km"
        ],

        hard_config[
            "repeater_count"
        ],

        hard_config[
            "multiplexed_cycle_success_probability"
        ]
    )
)


print(
    "EASY CONFIG"
)

print(
    "Observed success:",
    easy_config[
        "service_success_probability"
    ]
)

print(
    easy_prediction
)


print(
    "\nHARD CONFIG"
)

print(
    "Observed success:",
    hard_config[
        "service_success_probability"
    ]
)

print(
    hard_prediction
)


assert (

    easy_prediction[
        "calibrated_service_probability"
    ]

    >

    hard_prediction[
        "calibrated_service_probability"
    ]
)


print(
    "\n[PASS] Calibrated reliability ordering is physically sensible."
)

In [ ]:
# @title  Save Reliability Calibration Artifacts

# from pathlib import Path


# if (
#     "PHASE4_CAL_CHECKPOINT_DIR"
#     in globals()
# ):

#     phase4e4_output_dir = Path(
#         PHASE4_CAL_CHECKPOINT_DIR
#     )

# else:

#     phase4e4_output_dir = Path(
#         "/content"
#     )


# PHASE4E4_CALIBRATION_METRICS_DF.to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_calibration_metrics.csv",

#     index=False
# )


# PHASE4E4_CONFIG_RELIABILITY_DF.to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_calibrated_configs.csv",

#     index=False
# )


# PHASE4E4_RELIABILITY_TABLE_DF.to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_calibration_table.csv",

#     index=False
# )


# PHASE4E4_THRESHOLD_SWEEP_DF.to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_threshold_sweep.csv",

#     index=False
# )


# PHASE4E4_ADMISSION_AUDIT_DF.to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_admission_audit.csv",

#     index=False
# )


# pd.DataFrame(
#     {
#         "bootstrap_threshold":
#             PHASE4E4_BOOTSTRAP_THRESHOLDS
#     }
# ).to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_threshold_bootstrap.csv",

#     index=False
# )


# PHASE4E4_THRESHOLD_NEIGHBORHOOD_DF.to_csv(

#     phase4e4_output_dir
#     /
#     "phase4_reliability_threshold_neighborhood.csv",

#     index=False
# )


# print(
#     "[PASS] Diagnostics models saved."
# )

### Phase 4E : 14-10-2026T16:33:53 -- 14-10-2026T16:45:22

Routing Policy

In [ ]:
# @title Baseline Equalised Weighted Routing Policy

from dataclasses import dataclass
import math
import pandas as pd


PHASE4_ROUTING_WEIGHTS = {

    "distance":
        0.125,

    "hops":
        0.125,

    "visibility":
        0.125,

    "multiplexing":
        0.125,

    "memory":
        0.125,

    "bsm":
        0.125,

    "delivery_time":
        0.125,

    "memory_coherence":
        0.125
}


# ================================================================
# Validate operator policy
# ================================================================

expected_weight_keys = {

    "distance",
    "hops",
    "visibility",
    "multiplexing",
    "memory",
    "bsm",
    "delivery_time",
    "memory_coherence"
}


assert (
    set(
        PHASE4_ROUTING_WEIGHTS
    )
    ==
    expected_weight_keys
)


for metric_name, weight in (
    PHASE4_ROUTING_WEIGHTS.items()
):

    assert (
        isinstance(
            weight,
            (int, float)
        )
    )

    assert (
        math.isfinite(
            float(weight)
        )
    )

    assert (
        float(weight)
        >=
        0.0
    )


assert math.isclose(

    sum(
        PHASE4_ROUTING_WEIGHTS.values()
    ),

    1.0,

    rel_tol=0.0,

    abs_tol=1e-12
)


print(
    "=" * 90
)

print(
    "PHASE 4 EQUALISED WEIGHTED ROUTING POLICY"
)

print(
    "=" * 90
)


for metric_name, weight in (
    PHASE4_ROUTING_WEIGHTS.items()
):

    print(
        f"{metric_name:20s}: "
        f"{weight:.3f}"
    )


print(
    "\n[PASS] Baseline equalised operator routing policy accepted."
)

In [ ]:
# @title Real Parameter Based Weighted Routing Policy

from dataclasses import dataclass
import math
import pandas as pd


PHASE4_ROUTING_WEIGHTS = {

    "distance":
        0.05,

    "hops":
        0.08,

    "visibility":
        0.24,

    "multiplexing":
        0.22,

    "memory":
        0.1,

    "bsm":
        0.08,

    "delivery_time":
        0.15,

    "memory_coherence":
        0.08
}


# ================================================================
# Validate operator policy
# ================================================================

expected_weight_keys = {

    "distance",
    "hops",
    "visibility",
    "multiplexing",
    "memory",
    "bsm",
    "delivery_time",
    "memory_coherence"
}


assert (
    set(
        PHASE4_ROUTING_WEIGHTS
    )
    ==
    expected_weight_keys
)


for metric_name, weight in (
    PHASE4_ROUTING_WEIGHTS.items()
):

    assert (
        isinstance(
            weight,
            (int, float)
        )
    )

    assert (
        math.isfinite(
            float(weight)
        )
    )

    assert (
        float(weight)
        >=
        0.0
    )


assert math.isclose(

    sum(
        PHASE4_ROUTING_WEIGHTS.values()
    ),

    1.0,

    rel_tol=0.0,

    abs_tol=1e-12
)


print(
    "=" * 90
)

print(
    "PHASE 4 QUANTUM PARAMS ROUTING POLICY"
)

print(
    "=" * 90
)


for metric_name, weight in (
    PHASE4_ROUTING_WEIGHTS.items()
):

    print(
        f"{metric_name:20s}: "
        f"{weight:.3f}"
    )


print(
    "\n[PASS] Quantum aware operator routing policy accepted."
)

In [ ]:
# @title Routing Metric Normalization

def phase4_normalize_benefit(
    value,
    minimum,
    maximum
):
    """
    Larger raw value is better.
    """

    value = float(
        value
    )

    minimum = float(
        minimum
    )

    maximum = float(
        maximum
    )


    if math.isclose(
        minimum,
        maximum,
        rel_tol=0.0,
        abs_tol=1e-15
    ):

        # Every candidate is identical in this dimension.
        # Giving all candidates 1.0 preserves neutrality.
        return 1.0


    normalized = (

        value
        -
        minimum

    ) / (

        maximum
        -
        minimum
    )


    return float(

        min(
            max(
                normalized,
                0.0
            ),
            1.0
        )
    )


def phase4_normalize_cost(
    value,
    minimum,
    maximum
):
    """
    Smaller raw value is better.
    """

    return float(

        1.0

        -

        phase4_normalize_benefit(

            value=value,
            minimum=minimum,
            maximum=maximum
        )
    )


print(
    "[PASS] Benefit/cost normalization functions ready."
)

In [ ]:
# @title  Feasible-Set Normalization Bounds

PHASE4_SCOREABLE_PATHS = tuple(

    metrics

    for metrics
    in PHASE4_PATH_METRICS

    if metrics.statically_feasible
)


if not PHASE4_SCOREABLE_PATHS:

    raise Phase4NoValidPathError(
        "No statically feasible path is available for scoring."
    )


def phase4_metric_bounds(
    values
):

    values = tuple(
        float(value)
        for value in values
    )


    return (
        min(values),
        max(values)
    )


PHASE4_NORMALIZATION_BOUNDS = {

    "distance":
        phase4_metric_bounds(
            metrics.total_distance_km
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "hops":
        phase4_metric_bounds(
            metrics.hop_count
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "visibility":
        phase4_metric_bounds(
            metrics.predicted_werner_visibility
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "multiplexing":
        phase4_metric_bounds(
            metrics.multiplexing_modes_available
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "memory":
        phase4_metric_bounds(
            metrics.memory_slots_available
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "bsm":
        phase4_metric_bounds(
            metrics.bsm_units_available
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "delivery_time":
        phase4_metric_bounds(
            metrics.expected_pair_delivery_proxy_s
            for metrics
            in PHASE4_SCOREABLE_PATHS
        ),

    "memory_coherence":
        phase4_metric_bounds(
            metrics.memory_coherence_cost
            for metrics
            in PHASE4_SCOREABLE_PATHS
        )
}


print(
    "=" * 90
)

print(
    "NORMALIZATION BOUNDS"
)

print(
    "=" * 90
)


for metric_name, bounds in (
    PHASE4_NORMALIZATION_BOUNDS.items()
):

    print(
        f"{metric_name:20s}: "
        f"{bounds[0]:.9f} -> "
        f"{bounds[1]:.9f}"
    )


print(
    "\n[PASS] Feasible-set normalization bounds frozen."
)

In [ ]:
# @title  Normalized Candidate Metric Vector

@dataclass(frozen=True)
class Phase4NormalizedMetrics:

    path_index: int

    distance: float

    hops: float

    visibility: float

    multiplexing: float

    memory: float

    bsm: float

    delivery_time: float

    memory_coherence: float


def phase4_normalize_path_metrics(
    metrics
):

    bounds = (
        PHASE4_NORMALIZATION_BOUNDS
    )


    # ============================================================
    # COST METRICS
    # smaller is better
    # ============================================================

    distance_score = (
        phase4_normalize_cost(

            metrics.total_distance_km,

            *bounds[
                "distance"
            ]
        )
    )


    hop_score = (
        phase4_normalize_cost(

            metrics.hop_count,

            *bounds[
                "hops"
            ]
        )
    )


    delivery_score = (
        phase4_normalize_cost(

            metrics.expected_pair_delivery_proxy_s,

            *bounds[
                "delivery_time"
            ]
        )
    )


    memory_coherence_score = (
        phase4_normalize_cost(

            metrics.memory_coherence_cost,

            *bounds[
                "memory_coherence"
            ]
        )
    )


    # ============================================================
    # BENEFIT METRICS
    # larger is better
    # ============================================================

    visibility_score = (
        phase4_normalize_benefit(

            metrics.predicted_werner_visibility,

            *bounds[
                "visibility"
            ]
        )
    )


    multiplexing_score = (
        phase4_normalize_benefit(

            metrics.multiplexing_modes_available,

            *bounds[
                "multiplexing"
            ]
        )
    )


    memory_score = (
        phase4_normalize_benefit(

            metrics.memory_slots_available,

            *bounds[
                "memory"
            ]
        )
    )


    bsm_score = (
        phase4_normalize_benefit(

            metrics.bsm_units_available,

            *bounds[
                "bsm"
            ]
        )
    )


    return Phase4NormalizedMetrics(

        path_index=(
            metrics.path_index
        ),

        distance=(
            distance_score
        ),

        hops=(
            hop_score
        ),

        visibility=(
            visibility_score
        ),

        multiplexing=(
            multiplexing_score
        ),

        memory=(
            memory_score
        ),

        bsm=(
            bsm_score
        ),

        delivery_time=(
            delivery_score
        ),

        memory_coherence=(
            memory_coherence_score
        )
    )


PHASE4_NORMALIZED_METRICS = tuple(

    phase4_normalize_path_metrics(
        metrics
    )

    for metrics
    in PHASE4_SCOREABLE_PATHS
)


print(
    "[PASS] Every feasible route normalized to [0,1]."
)

In [ ]:
# @title  Weighted Quantum Path Score

@dataclass(frozen=True)
class Phase4PathScore:

    rank: int

    path_index: int

    path: str

    score: float

    normalized: Phase4NormalizedMetrics


def phase4_weighted_score(
    normalized_metrics
):

    weights = (
        PHASE4_ROUTING_WEIGHTS
    )


    score = (

        weights["distance"]
        *
        normalized_metrics.distance

        +

        weights["hops"]
        *
        normalized_metrics.hops

        +

        weights["visibility"]
        *
        normalized_metrics.visibility

        +

        weights["multiplexing"]
        *
        normalized_metrics.multiplexing

        +

        weights["memory"]
        *
        normalized_metrics.memory

        +

        weights["bsm"]
        *
        normalized_metrics.bsm

        +

        weights["delivery_time"]
        *
        normalized_metrics.delivery_time

        +

        weights["memory_coherence"]
        *
        normalized_metrics.memory_coherence
    )


    return float(
        score
    )


# ================================================================
# Raw unranked score records
# ================================================================

_score_records = []


metrics_by_index = {

    metrics.path_index:
        metrics

    for metrics
    in PHASE4_SCOREABLE_PATHS
}


for normalized in (
    PHASE4_NORMALIZED_METRICS
):

    raw_metrics = (
        metrics_by_index[
            normalized.path_index
        ]
    )


    score = (
        phase4_weighted_score(
            normalized
        )
    )


    _score_records.append(
        (
            normalized,
            raw_metrics,
            score
        )
    )


# ================================================================
# Deterministic ranking
#
# Tie break:
#   1. higher weighted score
#   2. lower delivery proxy
#   3. lower distance
#   4. lower path index
# ================================================================

_score_records.sort(

    key=lambda item: (

        -item[2],

        item[1]
        .expected_pair_delivery_proxy_s,

        item[1]
        .total_distance_km,

        item[1]
        .path_index
    )
)


PHASE4_RANKED_PATHS = tuple(

    Phase4PathScore(

        rank=(
            rank
        ),

        path_index=(
            raw_metrics.path_index
        ),

        path=(
            raw_metrics.path
        ),

        score=(
            score
        ),

        normalized=(
            normalized
        )
    )

    for rank, (
        normalized,
        raw_metrics,
        score
    )

    in enumerate(
        _score_records,
        start=1
    )
)


print(
    "[PASS] Weighted path ranking completed."
)

In [ ]:
# @title  Quantum Routing Leaderboard

# leaderboard_rows = []


# for result in (
#     PHASE4_RANKED_PATHS
# ):

#     raw = (
#         metrics_by_index[
#             result.path_index
#         ]
#     )


#     normalized = (
#         result.normalized
#     )


#     leaderboard_rows.append(
#         {

#             "rank":
#                 result.rank,

#             "path_index":
#                 result.path_index,

#             "path":
#                 result.path,

#             "score":
#                 result.score,

#             "N_distance":
#                 normalized.distance,

#             "N_hops":
#                 normalized.hops,

#             "N_visibility":
#                 normalized.visibility,

#             "N_mux":
#                 normalized.multiplexing,

#             "N_memory":
#                 normalized.memory,

#             "N_bsm":
#                 normalized.bsm,

#             "N_delivery":
#                 normalized.delivery_time,

#             "N_coherence":
#                 normalized.memory_coherence,

#             "distance_km":
#                 raw.total_distance_km,

#             "delivery_ms":

#                 raw
#                 .expected_pair_delivery_proxy_s
#                 *
#                 1000.0,

#             "pred_qber":
#                 raw.predicted_qber
#         }
#     )


# PHASE4_ROUTING_LEADERBOARD_DF = (
#     pd.DataFrame(
#         leaderboard_rows
#     )
# )


# display(
#     PHASE4_ROUTING_LEADERBOARD_DF
# )


# print(
#     "\n[PASS] Routing leaderboard generated."
# )

In [ ]:
# @title  Baseline Route Proposal

# PHASE4_PROPOSED_ROUTE_SCORE = (
#     PHASE4_RANKED_PATHS[0]
# )


# PHASE4_PROPOSED_ROUTE = next(

#     candidate

#     for candidate
#     in PHASE4_VALID_PATHS

#     if (
#         candidate.path_index
#         ==
#         PHASE4_PROPOSED_ROUTE_SCORE.path_index
#     )
# )


# PHASE4_PROPOSED_ROUTE_METRICS = (

#     metrics_by_index[
#         PHASE4_PROPOSED_ROUTE.path_index
#     ]
# )


# print(
#     "=" * 100
# )

# print(
#     "NARAD-QC BASELINE ROUTE PROPOSAL"
# )

# print(
#     "=" * 100
# )


# print(
#     "\nRank:",
#     PHASE4_PROPOSED_ROUTE_SCORE.rank
# )


# print(
#     "Path index:",
#     PHASE4_PROPOSED_ROUTE.path_index
# )


# print(
#     "Route:"
# )

# print(
#     "  ",
#     PHASE4_PROPOSED_ROUTE.display_path
# )


# print(
#     "\nWeighted score:",
#     f"{PHASE4_PROPOSED_ROUTE_SCORE.score:.6f}"
# )


# print(
#     "Total fibre distance:",
#     f"{PHASE4_PROPOSED_ROUTE_METRICS.total_distance_km:.3f} km"
# )


# print(
#     "Predicted Werner visibility:",
#     f"{PHASE4_PROPOSED_ROUTE_METRICS.predicted_werner_visibility:.6f}"
# )


# print(
#     "Predicted QBER:",
#     f"{PHASE4_PROPOSED_ROUTE_METRICS.predicted_qber:.3%}"
# )


# print(
#     "Available multiplexing:",
#     PHASE4_PROPOSED_ROUTE_METRICS
#     .multiplexing_modes_available
# )


# print(
#     "Available memory:",
#     PHASE4_PROPOSED_ROUTE_METRICS
#     .memory_slots_available
# )


# print(
#     "Available BSM units:",
#     PHASE4_PROPOSED_ROUTE_METRICS
#     .bsm_units_available
# )


# print(
#     "Delivery proxy:",
#     f"{PHASE4_PROPOSED_ROUTE_METRICS.expected_pair_delivery_proxy_s * 1000:.3f} ms"
# )


# print(
#     "\nSTATUS:"
# )

# print(
#     "  PROPOSED — NOT YET AUTHENTICATED"
# )


# print(
#     "\n[PASS] Control plane autonomously proposed "
#     "its highest-ranked quantum route."
# )

In [ ]:
# @title  Weighted Routing Verification

# print(
#     "=" * 100
# )

# print(
#     "ROUTING SCORE VERIFICATION"
# )

# print(
#     "=" * 100
# )


# assert (
#     len(
#         PHASE4_RANKED_PATHS
#     )
#     ==
#     len(
#         PHASE4_SCOREABLE_PATHS
#     )
# )


# for result in (
#     PHASE4_RANKED_PATHS
# ):

#     assert (
#         0.0
#         <=
#         result.score
#         <=
#         1.0
#     )


#     normalized = (
#         result.normalized
#     )


#     for value in (

#         normalized.distance,
#         normalized.hops,
#         normalized.visibility,
#         normalized.multiplexing,
#         normalized.memory,
#         normalized.bsm,
#         normalized.delivery_time,
#         normalized.memory_coherence

#     ):

#         assert (
#             0.0
#             <=
#             value
#             <=
#             1.0
#         )


# # Scores must be descending.
# scores = [

#     result.score

#     for result
#     in PHASE4_RANKED_PATHS
# ]


# assert (

#     scores

#     ==

#     sorted(
#         scores,
#         reverse=True
#     )
# )


# # Winner must have survived all hard gates.
# assert (
#     PHASE4_PROPOSED_ROUTE_METRICS
#     .statically_feasible
# )


# print(
#     "[PASS] Every normalized value lies in [0,1]."
# )

# print(
#     "[PASS] Every weighted score lies in [0,1]."
# )

# print(
#     "[PASS] Ranking order is deterministic."
# )

# print(
#     "[PASS] Proposed route survived all hard feasibility gates."
# )


# print(
#     "\n" + "=" * 100
# )

# print(
#     "PHASE 4 PASSED"
# )

# print(
#     "=" * 100
# )

In [ ]:
# @title Policy C: Empirically Calibrated Quantum-Service Routing

from dataclasses import dataclass
import math
import numpy as np
import pandas as pd


# ======================================================================
# NARAD-QC POLICY C
#
# Empirically Calibrated Quantum-Service Routing
#
# For each already statically/QKD-feasible path:
#
#   1. Extract heterogeneous-path bottleneck physics:
#
#        d_b = max elementary-link distance
#
#        P_b = min_e [1 - (1 - eta_e)^m_e]
#
#   2. Feed:
#
#        [d_b, repeater_count, P_b]
#
#      into the calibrated Phase-4E service models.
#
#   3. Predict:
#
#        p_service
#        T_pair | success
#        delivered visibility
#
#   4. Apply the existing QKD quality boundary to predicted delivered
#      quality.
#
#   5. Rank surviving paths using:
#
#        S_C =
#            p_service
#            --------------------  *  U_quality
#            T_pair | success
#
#      where:
#
#        U_quality =
#            clip(
#                (w_delivered - w_min)
#                /
#                (1 - w_min),
#                0,
#                1
#            )
#
#      and:
#
#        w_min = 1 - 2 * Q_max
#
#
# IMPORTANT:
#   - NO hand-picked routing weights.
#   - NO candidate-set min-max normalization.
#   - Reliability remains continuous; there is NO hard reliability gate
#     because Phase-4E.4 showed threshold instability.
#   - Static hardware feasibility remains inherited from Phase 4C.
#   - Delivered-QKD quality remains a hard gate.
#
# ======================================================================


# ======================================================================
# API / prerequisite checks
# ======================================================================

required_objects = (

    "phase4_network",

    "PHASE4_SCOREABLE_PATHS",

    "PHASE4_VALID_PATHS",

    "metrics_by_index",

    "phase4_link_metric_snapshot",

    "PHASE4_FINAL_RELIABILITY_MODEL",

    "PHASE4_FINAL_RELIABILITY_CALIBRATOR",

    "PHASE4_FINAL_LATENCY_MODEL",

    "PHASE4_FINAL_VISIBILITY_MODEL",

    "PHASE4_METRIC_QUALITY_CONFIG"
)


for object_name in required_objects:

    if object_name not in globals():

        raise RuntimeError(

            "Policy C prerequisite missing: "
            f"{object_name}"
        )


if not PHASE4_SCOREABLE_PATHS:

    raise Phase4NoValidPathError(

        "No statically feasible path exists for Policy C."
    )


# ======================================================================
# Frozen QKD quality boundary
# ======================================================================

PHASE4_POLICY_C_QBER_LIMIT = float(

    PHASE4_METRIC_QUALITY_CONFIG
    .qber_limit
)


PHASE4_POLICY_C_MIN_VISIBILITY = float(

    1.0

    -

    2.0
    *
    PHASE4_POLICY_C_QBER_LIMIT
)


assert math.isclose(

    PHASE4_POLICY_C_MIN_VISIBILITY,

    1.0
    -
    2.0
    *
    PHASE4_POLICY_C_QBER_LIMIT,

    rel_tol=0.0,

    abs_tol=1e-15
)


# ======================================================================
# Optional calibration-domain bounds
#
# These are diagnostics only.
#
# A path is NOT rejected merely for falling outside the empirical
# calibration range. Such a prediction is flagged as extrapolation.
# ======================================================================

if (
    "PHASE4_CAL_IMPROVED_SUMMARY_DF"
    in globals()
):

    PHASE4_POLICY_C_CALIBRATION_BOUNDS = {

        "elementary_distance_km": (

            float(

                PHASE4_CAL_IMPROVED_SUMMARY_DF[
                    "elementary_distance_km"
                ]
                .min()
            ),

            float(

                PHASE4_CAL_IMPROVED_SUMMARY_DF[
                    "elementary_distance_km"
                ]
                .max()
            )
        ),

        "repeater_count": (

            float(

                PHASE4_CAL_IMPROVED_SUMMARY_DF[
                    "repeater_count"
                ]
                .min()
            ),

            float(

                PHASE4_CAL_IMPROVED_SUMMARY_DF[
                    "repeater_count"
                ]
                .max()
            )
        ),

        "multiplexed_cycle_success_probability": (

            float(

                PHASE4_CAL_IMPROVED_SUMMARY_DF[
                    "multiplexed_cycle_success_probability"
                ]
                .min()
            ),

            float(

                PHASE4_CAL_IMPROVED_SUMMARY_DF[
                    "multiplexed_cycle_success_probability"
                ]
                .max()
            )
        )
    }

else:

    PHASE4_POLICY_C_CALIBRATION_BOUNDS = None


# ======================================================================
# Policy-C result object
# ======================================================================

@dataclass(frozen=True)
class Phase4PolicyCPathScore:

    rank: int

    path_index: int

    path: str


    # --------------------------------------------------------------
    # Heterogeneous-path empirical features
    # --------------------------------------------------------------

    bottleneck_elementary_distance_km: float

    bottleneck_multiplexed_success_probability: float

    repeater_count: int


    # --------------------------------------------------------------
    # Existing analytical fresh-state prediction
    # --------------------------------------------------------------

    fresh_predicted_visibility: float

    fresh_predicted_qber: float


    # --------------------------------------------------------------
    # Empirically calibrated service prediction
    # --------------------------------------------------------------

    predicted_service_probability: float

    predicted_median_delivery_s: float

    predicted_visibility_delta: float

    predicted_delivered_visibility: float

    predicted_delivered_qber: float


    # --------------------------------------------------------------
    # Policy-C decision quantities
    # --------------------------------------------------------------

    empirical_qkd_feasible: bool

    quality_utility: float

    score: float


    # --------------------------------------------------------------
    # Calibration diagnostics
    # --------------------------------------------------------------

    within_calibration_domain: bool

    extrapolation_fields: tuple


# ======================================================================
# Candidate lookup
# ======================================================================

PHASE4_POLICY_C_CANDIDATE_BY_INDEX = {

    candidate.path_index:
        candidate

    for candidate
    in PHASE4_VALID_PATHS
}


# ======================================================================
# Heterogeneous-path empirical feature extraction
#
# Calibration used homogeneous chains.
#
# For a heterogeneous routing path we conservatively transfer the model
# using:
#
#   d_b = maximum elementary-link distance
#
#   P_b = minimum multiplexed cycle success probability
#
# This transfer assumption MUST remain documented in the paper.
# ======================================================================

def phase4_policy_c_path_empirical_features(
    candidate_path
):

    link_snapshots = []


    for (
        node_a_name,
        node_b_name
    ) in zip(

        candidate_path.node_names[:-1],

        candidate_path.node_names[1:]
    ):

        snapshot = (

            phase4_link_metric_snapshot(

                phase4_network,

                node_a_name,

                node_b_name
            )
        )


        link_snapshots.append(
            snapshot
        )


    if not link_snapshots:

        raise RuntimeError(

            "Policy C received a path containing no physical links."
        )


    bottleneck_distance_km = float(

        max(

            snapshot[
                "distance_km"
            ]

            for snapshot
            in link_snapshots
        )
    )


    bottleneck_cycle_success = float(

        min(

            snapshot[
                "multiplexed_cycle_success_probability"
            ]

            for snapshot
            in link_snapshots
        )
    )


    return {

        "bottleneck_elementary_distance_km":
            bottleneck_distance_km,

        "bottleneck_multiplexed_success_probability":
            bottleneck_cycle_success,

        "repeater_count":
            int(
                candidate_path.repeater_count
            ),

        "link_snapshots":
            tuple(
                link_snapshots
            )
    }


# ======================================================================
# Calibration-domain audit
# ======================================================================

def phase4_policy_c_calibration_domain_audit(
    bottleneck_elementary_distance_km,
    repeater_count,
    bottleneck_multiplexed_success_probability
):

    if (
        PHASE4_POLICY_C_CALIBRATION_BOUNDS
        is None
    ):

        return (
            True,
            tuple()
        )


    values = {

        "elementary_distance_km":
            float(
                bottleneck_elementary_distance_km
            ),

        "repeater_count":
            float(
                repeater_count
            ),

        "multiplexed_cycle_success_probability":
            float(
                bottleneck_multiplexed_success_probability
            )
    }


    outside = []


    for (
        feature_name,
        value
    ) in values.items():

        lower, upper = (

            PHASE4_POLICY_C_CALIBRATION_BOUNDS[
                feature_name
            ]
        )


        if (

            value < lower

            or

            value > upper
        ):

            outside.append(
                feature_name
            )


    return (

        len(
            outside
        )
        ==
        0,

        tuple(
            outside
        )
    )


# ======================================================================
# Calibrated reliability predictor
# ======================================================================

def phase4_policy_c_predict_reliability(
    feature_vector
):

    raw_decision_score = float(

        PHASE4_FINAL_RELIABILITY_MODEL
        .decision_function(
            feature_vector
        )[0]
    )


    calibrated_probability = float(

        PHASE4_FINAL_RELIABILITY_CALIBRATOR
        .predict_proba(

            np.array(
                [[
                    raw_decision_score
                ]],
                dtype=float
            )
        )[0, 1]
    )


    return float(

        np.clip(

            calibrated_probability,

            0.0,

            1.0
        )
    )


# ======================================================================
# Complete Policy-C service prediction
# ======================================================================

def phase4_policy_c_predict_path(
    candidate_path,
    raw_metrics
):

    empirical_features = (

        phase4_policy_c_path_empirical_features(

            candidate_path
        )
    )


    d_b = float(

        empirical_features[
            "bottleneck_elementary_distance_km"
        ]
    )


    p_b = float(

        empirical_features[
            "bottleneck_multiplexed_success_probability"
        ]
    )


    repeater_count = int(

        empirical_features[
            "repeater_count"
        ]
    )


    feature_vector = np.array(
        [[

            d_b,

            float(
                repeater_count
            ),

            p_b

        ]],

        dtype=float
    )


    # ==============================================================
    # 1. Calibrated finite-budget service probability
    # ==============================================================

    predicted_service_probability = (

        phase4_policy_c_predict_reliability(

            feature_vector
        )
    )


    # ==============================================================
    # 2. Conditional median pair-delivery time
    #
    # Latency model predicts log(T).
    # ==============================================================

    predicted_log_delivery_s = float(

        PHASE4_FINAL_LATENCY_MODEL
        .predict(
            feature_vector
        )[0]
    )


    predicted_median_delivery_s = float(

        math.exp(
            predicted_log_delivery_s
        )
    )


    if (

        not math.isfinite(
            predicted_median_delivery_s
        )

        or

        predicted_median_delivery_s
        <=
        0.0
    ):

        raise RuntimeError(

            "Policy C latency predictor returned an invalid "
            "conditional delivery time."
        )


    # ==============================================================
    # 3. Empirical visibility degradation
    # ==============================================================

    predicted_visibility_delta = float(

        PHASE4_FINAL_VISIBILITY_MODEL
        .predict(
            feature_vector
        )[0]
    )


    fresh_visibility = float(

        raw_metrics
        .predicted_werner_visibility
    )


    fresh_qber = float(

        raw_metrics
        .predicted_qber
    )


    predicted_delivered_visibility = float(

        np.clip(

            fresh_visibility

            +

            predicted_visibility_delta,

            0.0,

            1.0
        )
    )


    predicted_delivered_qber = float(

        (
            1.0

            -

            predicted_delivered_visibility
        )

        /

        2.0
    )


    # ==============================================================
    # Existing QKD threshold applied to empirical delivered quality
    # ==============================================================

    empirical_qkd_feasible = bool(

        predicted_delivered_qber

        <=

        (
            PHASE4_POLICY_C_QBER_LIMIT

            +

            PHASE4_METRIC_QUALITY_CONFIG
            .numerical_tolerance
        )
    )


    # ==============================================================
    # Quality utility
    #
    # 0 at the QKD visibility boundary.
    # 1 at ideal visibility.
    # ==============================================================

    quality_utility = float(

        np.clip(

            (
                predicted_delivered_visibility

                -

                PHASE4_POLICY_C_MIN_VISIBILITY
            )

            /

            (
                1.0

                -

                PHASE4_POLICY_C_MIN_VISIBILITY
            ),

            0.0,

            1.0
        )
    )


    # ==============================================================
    # Policy-C score
    #
    # Reliability-adjusted, quality-adjusted entanglement service
    # rate.
    #
    # Units: approximately useful-pair service opportunities / second.
    #
    # If empirical delivered quality fails QKD admission, score = 0.
    # ==============================================================

    if empirical_qkd_feasible:

        score = float(

            (
                predicted_service_probability

                /

                predicted_median_delivery_s
            )

            *

            quality_utility
        )

    else:

        score = 0.0


    # ==============================================================
    # Calibration-domain diagnostic
    # ==============================================================

    (
        within_calibration_domain,
        extrapolation_fields
    ) = (

        phase4_policy_c_calibration_domain_audit(

            bottleneck_elementary_distance_km=(
                d_b
            ),

            repeater_count=(
                repeater_count
            ),

            bottleneck_multiplexed_success_probability=(
                p_b
            )
        )
    )


    return {

        "path_index":
            int(
                raw_metrics.path_index
            ),

        "path":
            str(
                raw_metrics.path
            ),

        "bottleneck_elementary_distance_km":
            d_b,

        "bottleneck_multiplexed_success_probability":
            p_b,

        "repeater_count":
            repeater_count,

        "fresh_predicted_visibility":
            fresh_visibility,

        "fresh_predicted_qber":
            fresh_qber,

        "predicted_service_probability":
            predicted_service_probability,

        "predicted_median_delivery_s":
            predicted_median_delivery_s,

        "predicted_visibility_delta":
            predicted_visibility_delta,

        "predicted_delivered_visibility":
            predicted_delivered_visibility,

        "predicted_delivered_qber":
            predicted_delivered_qber,

        "empirical_qkd_feasible":
            empirical_qkd_feasible,

        "quality_utility":
            quality_utility,

        "score":
            score,

        "within_calibration_domain":
            bool(
                within_calibration_domain
            ),

        "extrapolation_fields":
            tuple(
                extrapolation_fields
            )
    }


# ======================================================================
# Evaluate every already-statically-feasible path
# ======================================================================

_policy_c_records = []


for raw_metrics in (

    PHASE4_SCOREABLE_PATHS
):

    candidate_path = (

        PHASE4_POLICY_C_CANDIDATE_BY_INDEX[
            raw_metrics.path_index
        ]
    )


    prediction = (

        phase4_policy_c_predict_path(

            candidate_path=(
                candidate_path
            ),

            raw_metrics=(
                raw_metrics
            )
        )
    )


    _policy_c_records.append(
        prediction
    )


# ======================================================================
# Empirical delivered-QKD quality gate
#
# Reliability is NOT a hard gate.
# ======================================================================

PHASE4_POLICY_C_EMPIRICALLY_QKD_FEASIBLE = tuple(

    record

    for record
    in _policy_c_records

    if record[
        "empirical_qkd_feasible"
    ]
)


PHASE4_POLICY_C_EMPIRICALLY_QKD_REJECTED = tuple(

    record

    for record
    in _policy_c_records

    if not record[
        "empirical_qkd_feasible"
    ]
)


if not PHASE4_POLICY_C_EMPIRICALLY_QKD_FEASIBLE:

    raise Phase4NoValidPathError(

        "Policy C predicts that no statically feasible route "
        "retains QKD-admissible delivered quality."
    )


# ======================================================================
# Deterministic ranking
#
# 1. higher Policy-C service score
# 2. higher calibrated reliability
# 3. lower predicted conditional delivery time
# 4. higher delivered visibility
# 5. lower path index
# ======================================================================

_policy_c_sorted = sorted(

    PHASE4_POLICY_C_EMPIRICALLY_QKD_FEASIBLE,

    key=lambda record: (

        -record[
            "score"
        ],

        -record[
            "predicted_service_probability"
        ],

        record[
            "predicted_median_delivery_s"
        ],

        -record[
            "predicted_delivered_visibility"
        ],

        record[
            "path_index"
        ]
    )
)


PHASE4_POLICY_C_RANKED_PATHS = tuple(

    Phase4PolicyCPathScore(

        rank=rank,

        **record

    )

    for rank, record
    in enumerate(

        _policy_c_sorted,

        start=1
    )
)


# ======================================================================
# Leaderboard
# ======================================================================

PHASE4_POLICY_C_LEADERBOARD_DF = pd.DataFrame(

    [

        {

            "rank":
                result.rank,

            "path_index":
                result.path_index,

            "path":
                result.path,

            "policy_c_score_per_s":
                result.score,

            "service_probability":
                result.predicted_service_probability,

            "predicted_delivery_ms":
                (
                    result
                    .predicted_median_delivery_s
                    *
                    1000.0
                ),

            "fresh_visibility":
                result.fresh_predicted_visibility,

            "delivered_visibility":
                result.predicted_delivered_visibility,

            "delivered_qber":
                result.predicted_delivered_qber,

            "quality_utility":
                result.quality_utility,

            "bottleneck_link_km":
                result
                .bottleneck_elementary_distance_km,

            "bottleneck_mux_success":
                result
                .bottleneck_multiplexed_success_probability,

            "repeaters":
                result.repeater_count,

            "within_calibration_domain":
                result.within_calibration_domain,

            "extrapolation_fields":
                (
                    ", ".join(
                        result.extrapolation_fields
                    )

                    if result.extrapolation_fields

                    else ""
                )
        }

        for result
        in PHASE4_POLICY_C_RANKED_PATHS
    ]
)


display(
    PHASE4_POLICY_C_LEADERBOARD_DF
)


# ======================================================================
# Autonomous route proposal
# ======================================================================

PHASE4_POLICY_C_PROPOSED_ROUTE_SCORE = (

    PHASE4_POLICY_C_RANKED_PATHS[
        0
    ]
)


PHASE4_POLICY_C_PROPOSED_ROUTE = next(

    candidate

    for candidate
    in PHASE4_VALID_PATHS

    if (

        candidate.path_index

        ==

        PHASE4_POLICY_C_PROPOSED_ROUTE_SCORE
        .path_index
    )
)


PHASE4_POLICY_C_PROPOSED_ROUTE_METRICS = (

    metrics_by_index[

        PHASE4_POLICY_C_PROPOSED_ROUTE
        .path_index
    ]
)


# ======================================================================
# Compatibility aliases
#
# These intentionally expose the selected Policy-C result under the
# generic names expected by later routing/authentication stages.
#
# DO NOT run the OLD weighted-normalization verification cell afterward;
# Policy-C scores are service-rate quantities, not [0,1] weights.
# ======================================================================

PHASE4_RANKED_PATHS = (

    PHASE4_POLICY_C_RANKED_PATHS
)


PHASE4_ROUTING_LEADERBOARD_DF = (

    PHASE4_POLICY_C_LEADERBOARD_DF
)


PHASE4_PROPOSED_ROUTE_SCORE = (

    PHASE4_POLICY_C_PROPOSED_ROUTE_SCORE
)


PHASE4_PROPOSED_ROUTE = (

    PHASE4_POLICY_C_PROPOSED_ROUTE
)


PHASE4_PROPOSED_ROUTE_METRICS = (

    PHASE4_POLICY_C_PROPOSED_ROUTE_METRICS
)


# ======================================================================
# Policy-C verification
# ======================================================================

assert (

    len(
        PHASE4_POLICY_C_RANKED_PATHS
    )

    ==

    len(
        PHASE4_POLICY_C_EMPIRICALLY_QKD_FEASIBLE
    )
)


assert all(

    result.score
    >=
    0.0

    for result
    in PHASE4_POLICY_C_RANKED_PATHS
)


assert all(

    0.0
    <=
    result.predicted_service_probability
    <=
    1.0

    for result
    in PHASE4_POLICY_C_RANKED_PATHS
)


assert all(

    result.predicted_median_delivery_s
    >
    0.0

    for result
    in PHASE4_POLICY_C_RANKED_PATHS
)


assert all(

    result.predicted_delivered_qber

    <=

    (
        PHASE4_POLICY_C_QBER_LIMIT

        +

        PHASE4_METRIC_QUALITY_CONFIG
        .numerical_tolerance
    )

    for result
    in PHASE4_POLICY_C_RANKED_PATHS
)


assert all(

    0.0
    <=
    result.quality_utility
    <=
    1.0

    for result
    in PHASE4_POLICY_C_RANKED_PATHS
)


# Scores must be monotonically non-increasing.
_policy_c_scores = [

    result.score

    for result
    in PHASE4_POLICY_C_RANKED_PATHS
]


assert (

    _policy_c_scores

    ==

    sorted(
        _policy_c_scores,
        reverse=True
    )
)


# Winner must still satisfy original hard Phase-4 feasibility.
assert (

    PHASE4_POLICY_C_PROPOSED_ROUTE_METRICS
    .statically_feasible
)


# ======================================================================
# Final proposal report
# ======================================================================

winner = (

    PHASE4_POLICY_C_PROPOSED_ROUTE_SCORE
)


print(
    "\n" + "=" * 104
)

print(
    "NARAD-QC POLICY C — EMPIRICALLY CALIBRATED QUANTUM-SERVICE ROUTING"
)

print(
    "=" * 104
)


print(
    "\nSelected path:"
)

print(
    " ",
    winner.path
)


print(
    "\nPath index:",
    winner.path_index
)


print(
    "Rank:",
    winner.rank
)


print(
    "\nBottleneck elementary link:",
    f"{winner.bottleneck_elementary_distance_km:.3f} km"
)


print(
    "Bottleneck multiplexed cycle success:",
    f"{winner.bottleneck_multiplexed_success_probability:.6f}"
)


print(
    "Repeater count:",
    winner.repeater_count
)


print(
    "\nCalibrated service probability:",
    f"{winner.predicted_service_probability:.6f}"
)


print(
    "Predicted median delivery:",
    f"{winner.predicted_median_delivery_s * 1000.0:.3f} ms"
)


print(
    "Fresh predicted visibility:",
    f"{winner.fresh_predicted_visibility:.6f}"
)


print(
    "Empirically predicted delivered visibility:",
    f"{winner.predicted_delivered_visibility:.6f}"
)


print(
    "Empirically predicted delivered QBER:",
    f"{winner.predicted_delivered_qber:.3%}"
)


print(
    "Quality headroom utility:",
    f"{winner.quality_utility:.6f}"
)


print(
    "\nPolicy-C score:",
    f"{winner.score:.6f} quality-adjusted service/s"
)


print(
    "Inside empirical calibration domain:",
    winner.within_calibration_domain
)


if not winner.within_calibration_domain:

    print(
        "Extrapolated features:",
        winner.extrapolation_fields
    )


print(
    "\nSTATUS:"
)

print(
    "  PROPOSED — NOT YET AUTHENTICATED"
)


print(
    "\nEmpirical QKD-quality rejections:",
    len(
        PHASE4_POLICY_C_EMPIRICALLY_QKD_REJECTED
    )
)


print(
    "\n[PASS] Static feasibility inherited from Phase 4C."
)

print(
    "[PASS] Reliability estimated using calibrated Phase-4E model."
)

print(
    "[PASS] Latency estimated using cross-validated quadratic Ridge."
)

print(
    "[PASS] Delivered visibility estimated using cross-validated quadratic Ridge."
)

print(
    "[PASS] Existing QKD threshold applied to predicted delivered quality."
)

print(
    "[PASS] No hand-selected routing coefficients are used."
)

print(
    "[PASS] Reliability remains continuous; no unstable hard threshold is imposed."
)

print(
    "[PASS] Policy-C ranking is deterministic."
)


print(
    "\n" + "=" * 104
)

print(
    "POLICY C SUCCESSFUL"
)

print(
    "=" * 104
)

## Phase 5 :

Full NARAD-QC Integration

All components established successfully

In [ ]:
# @title NARAD-QC — COMPLETE END-TO-END KEY ESTABLISHMENT API

from dataclasses import dataclass
from typing import Optional
import math
import hashlib
import numpy as np


# ======================================================================
# NARAD-QC COMPLETE SERVICE API
#
# Public entry point:
#
#     narad_qc_establish_key(
#         network,
#         sender_name,
#         receiver_name
#     )
#
#
# COMPLETE CONTROL FLOW
# ---------------------
#
# Operator network
#       ↓
# sender / receiver validation
#       ↓
# Phase-4 repeater-only path discovery
#       ↓
# Phase-4 hard resource + fresh-QKD feasibility
#       ↓
# Policy-C empirical service prediction
#       ↓
# Policy-C ranking
#       ↓
# highest-ranked candidate
#       ↓
# build actual heterogeneous Phase-3 execution stack
#       ↓
# fresh ML-DSA mutual authentication of EVERY selected node
#       ↓
# hard quantum-eligibility authorization
#       ↓
# Bell-pair generation + multiplexing
#       ↓
# failure-aware swapping + recovery
#       ↓
# BBM92 measurement / sifting / PE
#       ↓
# Cascade reconciliation
#       ↓
# Toeplitz privacy amplification
#       ↓
# Alice == Bob
#       ↓
# RETURN SECRET KEY
#
#
# If a candidate fails authentication or physical/QKD execution:
#
#       try the next Policy-C-ranked candidate
#
#
# IMPORTANT
# ---------
# The returned key is NEVER printed by this API.
#
# ======================================================================


# ======================================================================
# Exceptions
# ======================================================================

class NARADQCServiceError(RuntimeError):
    """Base error for complete NARAD-QC service establishment."""
    pass


class NARADQCNoExecutableRouteError(NARADQCServiceError):
    """No Policy-C route could complete authenticated QKD service."""
    pass


class NARADQCExecutionModelError(NARADQCServiceError):
    """Selected operator route is incompatible with frozen Phase-3 physics."""
    pass


# ======================================================================
# Audit records
# ======================================================================

@dataclass(frozen=True)
class NARADQCRouteAttempt:

    rank: int

    path_index: int

    path: str

    policy_c_score: float

    authenticated: bool

    qkd_completed: bool

    success: bool

    failure_stage: Optional[str]

    failure_reason: Optional[str]


@dataclass(frozen=True)
class NARADQCKeyEstablishmentResult:

    success: bool

    sender: str

    receiver: str

    selected_rank: int

    selected_path_index: int

    selected_path: str

    selected_node_names: tuple[str, ...]

    policy_c_score: float

    predicted_service_probability: float

    predicted_median_delivery_s: float

    predicted_delivered_visibility: float

    predicted_delivered_qber: float

    delivered_pairs: int

    zz_raw_key_bits: int

    xx_parameter_estimation_bits: int

    observed_x_qber: float

    phase_error_estimate: float

    ec_leakage_bits: int

    maximum_extractable_bits: int

    final_secret_bits: int

    modeled_physical_time_s: float

    elementary_generation_cycles: int

    optical_mode_attempts: int

    bsm_attempts: int

    authentication_session_ids: tuple[str, ...]

    attempted_routes: tuple[NARADQCRouteAttempt, ...]

    # --------------------------------------------------------------
    # FINAL SECRET
    #
    # Do not display / log this object carelessly.
    # --------------------------------------------------------------

    key: bytes


# ======================================================================
# Required API audit
# ======================================================================

_NARAD_REQUIRED_CALLABLES = (

    "phase4_discover_valid_quantum_paths",

    "phase4_evaluate_candidate_path",

    "phase4_link_metric_snapshot",

    "phase3i2_install_multiplexing_aware_recovery",

    "phase3j_bbm92_collection_summary",

    "phase3k_distill_configurable_bbm92_key"
)


for _name in _NARAD_REQUIRED_CALLABLES:

    if not callable(
        globals().get(
            _name
        )
    ):

        raise RuntimeError(
            f"NARAD-QC prerequisite missing: {_name}"
        )


_NARAD_REQUIRED_OBJECTS = (

    "PHASE4_FINAL_RELIABILITY_MODEL",

    "PHASE4_FINAL_RELIABILITY_CALIBRATOR",

    "PHASE4_FINAL_LATENCY_MODEL",

    "PHASE4_FINAL_VISIBILITY_MODEL",

    "PHASE4_METRIC_QUALITY_CONFIG",

    "PHASE3J_BIASED_BBM92_CONFIG",

    "PHASE3H1_QUALITY_CONFIG",

    "PHASE3H3_TIMING_PROFILE"
)


for _name in _NARAD_REQUIRED_OBJECTS:

    if _name not in globals():

        raise RuntimeError(
            f"NARAD-QC prerequisite missing: {_name}"
        )


# ======================================================================
# Seed domains
#
# These are NOT secret cryptographic values.
#
# They make the simulator experiment reproducible while keeping:
#
#     physical RNG
#     QKD protocol RNG
#     post-processing RNG
#
# logically separated.
# ======================================================================

NARAD_API_DEFAULT_PHYSICAL_SEED = 4_900_001

NARAD_API_DEFAULT_PROTOCOL_SEED = 7_900_001

NARAD_API_DEFAULT_POSTPROCESS_SEED = 7_910_001


def _narad_derive_seed(
    master_seed,
    path_index,
    stream_tag
):

    sequence = np.random.SeedSequence(
        [
            int(
                master_seed
            ),

            int(
                path_index
            ),

            int(
                stream_tag
            )
        ]
    )


    return int(

        sequence.generate_state(
            1,
            dtype=np.uint32
        )[0]
    )


# ======================================================================
# Generic Policy-C feature extraction
#
# NO dependency on global `phase4_network`.
# ======================================================================

def _narad_policy_c_features(
    network,
    candidate_path
):

    snapshots = []


    for (
        node_a_name,
        node_b_name
    ) in zip(

        candidate_path.node_names[:-1],

        candidate_path.node_names[1:]
    ):

        snapshots.append(

            phase4_link_metric_snapshot(

                network,

                node_a_name,

                node_b_name
            )
        )


    if not snapshots:

        raise NARADQCServiceError(
            "Candidate path contains no physical link."
        )


    bottleneck_distance_km = float(

        max(

            snapshot[
                "distance_km"
            ]

            for snapshot
            in snapshots
        )
    )


    bottleneck_mux_success = float(

        min(

            snapshot[
                "multiplexed_cycle_success_probability"
            ]

            for snapshot
            in snapshots
        )
    )


    return {

        "bottleneck_elementary_distance_km":
            bottleneck_distance_km,

        "bottleneck_multiplexed_success_probability":
            bottleneck_mux_success,

        "repeater_count":
            int(
                candidate_path.repeater_count
            ),

        "link_snapshots":
            tuple(
                snapshots
            )
    }


# ======================================================================
# Generic Policy-C candidate evaluator
# ======================================================================

def _narad_policy_c_evaluate_candidate(
    network,
    candidate_path,
    raw_metrics
):

    features = (

        _narad_policy_c_features(

            network,

            candidate_path
        )
    )


    d_b = float(

        features[
            "bottleneck_elementary_distance_km"
        ]
    )


    p_b = float(

        features[
            "bottleneck_multiplexed_success_probability"
        ]
    )


    repeater_count = int(

        features[
            "repeater_count"
        ]
    )


    x = np.array(
        [[

            d_b,

            float(
                repeater_count
            ),

            p_b

        ]],

        dtype=float
    )


    # ==================================================================
    # CALIBRATED RELIABILITY
    # ==================================================================

    raw_reliability_score = float(

        PHASE4_FINAL_RELIABILITY_MODEL
        .decision_function(
            x
        )[0]
    )


    predicted_service_probability = float(

        PHASE4_FINAL_RELIABILITY_CALIBRATOR
        .predict_proba(

            np.array(
                [[
                    raw_reliability_score
                ]],
                dtype=float
            )

        )[0, 1]
    )


    predicted_service_probability = float(

        np.clip(

            predicted_service_probability,

            0.0,

            1.0
        )
    )


    # ==================================================================
    # CONDITIONAL DELIVERY LATENCY
    # ==================================================================

    predicted_log_delivery_s = float(

        PHASE4_FINAL_LATENCY_MODEL
        .predict(
            x
        )[0]
    )


    predicted_delivery_s = float(

        math.exp(
            predicted_log_delivery_s
        )
    )


    if (

        not math.isfinite(
            predicted_delivery_s
        )

        or

        predicted_delivery_s <= 0.0
    ):

        raise NARADQCServiceError(
            "Policy-C latency predictor returned invalid latency."
        )


    # ==================================================================
    # DELIVERED VISIBILITY
    # ==================================================================

    predicted_visibility_delta = float(

        PHASE4_FINAL_VISIBILITY_MODEL
        .predict(
            x
        )[0]
    )


    fresh_visibility = float(

        raw_metrics
        .predicted_werner_visibility
    )


    delivered_visibility = float(

        np.clip(

            fresh_visibility
            +
            predicted_visibility_delta,

            0.0,

            1.0
        )
    )


    delivered_qber = float(

        (
            1.0
            -
            delivered_visibility
        )

        /
        2.0
    )


    qber_limit = float(

        PHASE4_METRIC_QUALITY_CONFIG
        .qber_limit
    )


    qkd_feasible = bool(

        delivered_qber

        <=

        (
            qber_limit

            +

            PHASE4_METRIC_QUALITY_CONFIG
            .numerical_tolerance
        )
    )


    minimum_visibility = float(

        1.0

        -

        2.0
        *
        qber_limit
    )


    quality_utility = float(

        np.clip(

            (
                delivered_visibility

                -

                minimum_visibility
            )

            /

            (
                1.0

                -

                minimum_visibility
            ),

            0.0,

            1.0
        )
    )


    score = (

        float(

            (
                predicted_service_probability

                /

                predicted_delivery_s
            )

            *

            quality_utility
        )

        if qkd_feasible

        else 0.0
    )


    return {

        "candidate":
            candidate_path,

        "metrics":
            raw_metrics,

        "score":
            score,

        "predicted_service_probability":
            predicted_service_probability,

        "predicted_median_delivery_s":
            predicted_delivery_s,

        "predicted_delivered_visibility":
            delivered_visibility,

        "predicted_delivered_qber":
            delivered_qber,

        "quality_utility":
            quality_utility,

        "empirical_qkd_feasible":
            qkd_feasible,

        "bottleneck_elementary_distance_km":
            d_b,

        "bottleneck_multiplexed_success_probability":
            p_b
    }


# ======================================================================
# Route discovery + Policy-C ranking
# ======================================================================

def _narad_rank_routes_policy_c(
    network,
    sender_name,
    receiver_name,
    max_hops=None
):

    (
        valid_paths,
        rejected_paths,
        physical_paths
    ) = phase4_discover_valid_quantum_paths(

        network=(
            network
        ),

        sender_name=(
            sender_name
        ),

        receiver_name=(
            receiver_name
        ),

        max_hops=(
            max_hops
        )
    )


    evaluated = []


    for candidate in valid_paths:

        metrics = (

            phase4_evaluate_candidate_path(

                network=(
                    network
                ),

                candidate_path=(
                    candidate
                )
            )
        )


        # ----------------------------------------------------------
        # Original Phase-4 hard gates.
        # ----------------------------------------------------------

        if not metrics.statically_feasible:

            continue


        policy_c = (

            _narad_policy_c_evaluate_candidate(

                network=(
                    network
                ),

                candidate_path=(
                    candidate
                ),

                raw_metrics=(
                    metrics
                )
            )
        )


        # ----------------------------------------------------------
        # Empirically predicted DELIVERED QKD quality remains hard.
        # ----------------------------------------------------------

        if not policy_c[
            "empirical_qkd_feasible"
        ]:

            continue


        evaluated.append(
            policy_c
        )


    if not evaluated:

        raise Phase4NoValidPathError(

            f"No Policy-C executable QKD route exists "
            f"between '{sender_name}' and '{receiver_name}'."
        )


    evaluated.sort(

        key=lambda record: (

            -record[
                "score"
            ],

            -record[
                "predicted_service_probability"
            ],

            record[
                "predicted_median_delivery_s"
            ],

            -record[
                "predicted_delivered_visibility"
            ],

            record[
                "candidate"
            ]
            .path_index
        )
    )


    return (
        tuple(
            evaluated
        ),

        tuple(
            valid_paths
        ),

        tuple(
            rejected_paths
        ),

        tuple(
            physical_paths
        )
    )


# ======================================================================
# Phase-4 -> Phase-3 execution-model consistency audit
# ======================================================================

def _narad_validate_execution_model(
    network,
    candidate_path
):
    """
    Validate operator-side path properties against physics currently
    implemented by the frozen Phase-3 executor.

    Current Phase-3 physical execution uses one common atomic-BSM
    success probability.

    We refuse to silently ignore a heterogeneous operator p_b.
    """

    expected_bsm_probability = float(

        PHASE3G2_ATOMIC_BSM_SUCCESS_PROBABILITY
    )


    for repeater_name in (

        candidate_path.repeater_names
    ):

        profile = (

            network
            .node_profile(
                repeater_name
            )
        )


        if not math.isclose(

            float(
                profile
                .bsm_success_probability
            ),

            expected_bsm_probability,

            rel_tol=0.0,

            abs_tol=1e-12
        ):

            raise NARADQCExecutionModelError(

                f"Repeater '{repeater_name}' declares "
                f"p_b={profile.bsm_success_probability}, "
                f"but the frozen Phase-3 executor currently "
                f"implements p_b="
                f"{expected_bsm_probability}. "
                f"Heterogeneous BSM probabilities are not "
                f"silently approximated."
            )


    return True


# ======================================================================
# Build the ACTUAL selected physical route
#
# This is the crucial Phase-4 -> Phase-3 adapter.
#
# It DOES NOT create:
#
#     Physical-A -> uniformly spaced Physical-R* -> Physical-B
#
# Instead it uses:
#
#     network.topology
#
# and the exact selected node objects + exact physical links.
# ======================================================================

def _narad_build_selected_path_stack(
    network,
    candidate_path,
    simulation_seed
):

    _narad_validate_execution_model(

        network,

        candidate_path
    )


    topology = (
        network.topology
    )


    path = tuple(

        topology.get_node(
            node_name
        )

        for node_name
        in candidate_path.node_names
    )


    # ==================================================================
    # Authoritative topology validation
    # ==================================================================

    topology.validate_path(
        path
    )


    # ==================================================================
    # Assemble Phase-3 simulator over the REAL operator topology.
    #
    # All devices are provisioned.
    #
    # They are NOT all authenticated here.
    # Authentication is performed only for the selected route below.
    # ==================================================================

    simulator = QuantumNetworkSimulator(

        topology=(
            topology
        ),

        simulation_seed=(
            int(
                simulation_seed
            )
        ),

        controller_id=(

            "NARAD-QC-END-TO-END-"
            f"{candidate_path.sender}-"
            f"{candidate_path.receiver}"
        ),

        parameter_set=(
            MLDSAParameterSet.ML_DSA_65
        ),

        max_storage_slots=None
    )


    # ==================================================================
    # EXACT PER-LINK AVAILABLE MULTIPLEXING
    # ==================================================================

    link_mode_counts = {}


    physical_links = []


    for (
        left_name,
        right_name
    ) in zip(

        candidate_path.node_names[:-1],

        candidate_path.node_names[1:]
    ):

        link = (

            topology.get_link(

                left_name,

                right_name
            )
        )


        profile = (

            network.link_profile(

                left_name,

                right_name
            )
        )


        modes = int(

            profile
            .multiplexing_modes_available
        )


        if modes <= 0:

            raise NARADQCExecutionModelError(

                f"Selected route link '{link.link_id}' has "
                f"no available multiplexing mode."
            )


        link_mode_counts[
            link.link_id
        ] = modes


        physical_links.append(
            link
        )


    multiplexing_config = ElementaryMultiplexingConfig(

        default_mode_count=1,

        link_mode_counts=(
            link_mode_counts
        )
    )


    # ==================================================================
    # Multiplexed elementary entanglement
    # ==================================================================

    generation_service = (

        MultiplexedElementaryEntanglementService(

            simulator=(
                simulator
            ),

            model=(
                FiberSurvivalProxyModel()
            ),

            multiplexing_config=(
                multiplexing_config
            )
        )
    )


    generation_scheduler = (

        ParallelElementaryGenerationScheduler(

            simulator=(
                simulator
            ),

            service=(
                generation_service
            ),

            ordered_path=(
                path
            ),

            demand_active=False
        )
    )


    # ==================================================================
    # Failure-aware elementary demand
    # ==================================================================

    demand_controller = (

        FailureAwareElementaryDemandController(

            simulator=(
                simulator
            ),

            generation_scheduler=(
                generation_scheduler
            ),

            ordered_path=(
                path
            )
        )
    )


    # ==================================================================
    # Frozen Phase-3 Werner fidelity physics
    # ==================================================================

    fidelity_profile = (

        WernerPhysicalFidelityProfile(

            elementary_fidelity=(
                PHASE3G2_ELEMENTARY_FIDELITY
            ),

            bell_fidelity_floor=0.25,

            memory_reference_initial_fidelity=1.0,

            memory_reference_fidelity=0.90,

            memory_reference_time_s=1.0,

            slot_duration_s=(
                PHASE3G2_SLOT_DURATION_S
            ),

            swap_operation_visibility=1.0
        )
    )


    physical_time_ledger = (

        PhysicalTimeLedger(

            kernel=(
                simulator.kernel
            )
        )
    )


    fidelity_registry = (

        BarrierAwarePhysicalTimeFidelityRegistry(

            simulator=(
                simulator
            ),

            profile=(
                fidelity_profile
            ),

            time_ledger=(
                physical_time_ledger
            )
        )
    )


    # ==================================================================
    # Bell-state measurement / entanglement swapping
    # ==================================================================

    swap_model = (

        WernerStateAtomicBSMModel(

            fidelity_profile=(
                fidelity_profile
            ),

            atomic_bsm_success_probability=(

                PHASE3G2_ATOMIC_BSM_SUCCESS_PROBABILITY
            ),

            atomic_bsm_latency_us=10.0
        )
    )


    swap_engine = (

        EntanglementSwapEngine(

            simulator=(
                simulator
            ),

            model=(
                swap_model
            ),

            fidelity_registry=(
                fidelity_registry
            )
        )
    )


    swap_policy = (

        MaxParallelDeterministicSwapPolicy()
    )


    swap_scheduler = (

        ControlPlaneParallelSwapScheduler(

            simulator=(
                simulator
            ),

            swap_engine=(
                swap_engine
            ),

            ordered_path=(
                path
            ),

            policy=(
                swap_policy
            )
        )
    )


    # ==================================================================
    # Stack object expected by the existing Phase-3 orchestrators
    # ==================================================================

    stack = {

        "policy":
            swap_policy,

        "topology":
            topology,

        "simulator":
            simulator,

        "path":
            path,

        "repeaters":
            tuple(
                path[
                    1:-1
                ]
            ),

        "physical_links":
            tuple(
                physical_links
            ),

        "generation_service":
            generation_service,

        "generation_scheduler":
            generation_scheduler,

        "multiplexing_config":
            multiplexing_config,

        "fidelity_profile":
            fidelity_profile,

        "fidelity_registry":
            fidelity_registry,

        "swap_model":
            swap_model,

        "swap_engine":
            swap_engine,

        "swap_scheduler":
            swap_scheduler,

        "demand_controller":
            demand_controller,

        "physical_time_ledger":
            physical_time_ledger,

        "timing_profile":
            PHASE3H3_TIMING_PROFILE,

        "generation_timing_geometry":
            (
                ElementaryAttemptTimingGeometry
                .MIDPOINT_HERALDED
            ),

        "total_distance_km":
            float(

                sum(

                    link.distance_km

                    for link
                    in physical_links
                )
            ),

        "repeater_count":
            int(
                candidate_path.repeater_count
            ),

        "elementary_link_count":
            len(
                physical_links
            ),

        # Heterogeneous route:
        "hop_distance_km":
            None,

        # Backwards-compatible conservative scalar:
        "elementary_success_probability":
            float(

                min(

                    link.survival_probability

                    for link
                    in physical_links
                )
            ),

        # Correct heterogeneous representation:
        "elementary_success_probabilities":
            tuple(

                float(
                    link.survival_probability
                )

                for link
                in physical_links
            )
    }


    # ==================================================================
    # Hard physical qubit lifetime
    # ==================================================================

    lifetime_controller = (

        PhysicalQubitLifetimeController(

            simulator=(
                simulator
            ),

            fidelity_registry=(
                fidelity_registry
            ),

            time_ledger=(
                physical_time_ledger
            ),

            max_storage_time_s=(
                PHASE3H3_MAX_STORAGE_TIME_S
            )
        )
    )


    stack[
        "physical_lifetime_controller"
    ] = lifetime_controller


    # ==================================================================
    # QKD-ready physical orchestrator
    # ==================================================================

    orchestrator = (

        PhysicalTimedQKDReadyOrchestrator(

            stack=(
                stack
            ),

            timing_profile=(
                PHASE3H3_TIMING_PROFILE
            ),

            generation_timing_geometry=(

                ElementaryAttemptTimingGeometry
                .MIDPOINT_HERALDED
            ),

            config=(
                PHASE3H1_QUALITY_CONFIG
            )
        )
    )


    stack[
        "orchestrator"
    ] = orchestrator


    # ==================================================================
    # Multiplexing-aware selective recovery
    # ==================================================================

    phase3i2_install_multiplexing_aware_recovery(

        stack,

        config=(
            PHASE3H1_QUALITY_CONFIG
        )
    )


    simulator.validate_runtime_consistency()


    return stack


# ======================================================================
# Fresh mutual ML-DSA authentication of selected route
# ======================================================================

def _narad_authenticate_selected_path(
    stack
):

    simulator = (
        stack[
            "simulator"
        ]
    )


    path = (
        stack[
            "path"
        ]
    )


    records = []


    for node in path:

        record = (

            simulator.authenticate_node(
                node
            )
        )


        records.append(
            record
        )


        if (

            record.status

            !=

            AuthenticationStatus.AUTHENTICATED
        ):

            raise QuantumOperationAuthorizationError(

                f"Fresh mutual authentication failed for "
                f"selected-path node '{node.name}'. "
                f"Reason={record.reason}"
            )


    # ==================================================================
    # HARD authorization gate
    # ==================================================================

    simulator.require_quantum_eligibility(

        *path
    )


    return tuple(
        records
    )


# ======================================================================
# COMPLETE BBM92 COLLECTION + ACTUAL POST-PROCESSING
#
# Important difference from merely calling
# collect_biased_bbm92_block():
#
# Forecast readiness only ALLOWS a post-processing attempt.
#
# It does not terminate the session.
#
# We stop only once ACTUAL:
#
#     Cascade
#     + leakage accounting
#     + entropy budget
#     + Toeplitz PA
#
# produces the requested key.
# ======================================================================

def _narad_run_qkd_until_actual_key(
    stack,
    qkd_config,
    protocol_seed,
    postprocess_master_seed,
    verbose
):

    qkd_channel = (

        NARADQKDEntanglementChannel(

            stack=(
                stack
            ),

            protocol_seed=(
                int(
                    protocol_seed
                )
            ),

            measurement_latency_s=0.0
        )
    )


    measurements = []


    last_postprocess_attempt_at = 0


    final_summary = None

    final_result = None


    for pair_number in range(

        1,

        int(
            qkd_config
            .max_delivered_pairs
        )
        +
        1
    ):

        measurement = (

            qkd_channel
            .deliver_and_measure(

                alice_basis=None,

                bob_basis=None,

                z_probability=(
                    qkd_config
                    .z_probability
                ),

                max_slots_per_pair=(
                    qkd_config
                    .max_slots_per_pair
                )
            )
        )


        measurements.append(
            measurement
        )


        # ----------------------------------------------------------
        # Don't recompute full block every pair unless useful.
        # ----------------------------------------------------------

        should_check = (

            pair_number

            %

            qkd_config
            .collection_check_interval_pairs

            ==

            0
        )


        if not should_check:

            continue


        summary = (

            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    qkd_config
                )
            )
        )


        final_summary = summary


        if verbose:

            x_qber_text = (

                f"{summary.observed_x_qber:.3%}"

                if (

                    summary
                    .xx_parameter_estimation_bits

                    >
                    0
                )

                else "N/A"
            )


            print(

                f"[QKD {summary.delivered_pairs:>4}] "
                f"ZZ={summary.zz_key_basis_bits:>4} | "
                f"XX={summary.xx_parameter_estimation_bits:>3} | "
                f"QX={x_qber_text} | "
                f"Qphase={summary.phase_error_estimate:.3%} | "
                f"forecast="
                f"{summary.forecast_extractable_bits}"
            )


        # ----------------------------------------------------------
        # Forecast is only permission to TRY actual post processing.
        # ----------------------------------------------------------

        if not summary.collection_ready:

            continue


        # Avoid repeatedly retrying on identical blocks.
        if (

            pair_number

            <=

            last_postprocess_attempt_at
        ):

            continue


        last_postprocess_attempt_at = (
            pair_number
        )


        # Independent post-processing RNG for THIS block.
        postprocess_rng = (

            np.random.default_rng(

                np.random.SeedSequence(
                    [
                        int(
                            postprocess_master_seed
                        ),

                        int(
                            pair_number
                        ),

                        int(
                            qkd_config.target_secret_bits
                        )
                    ]
                )
            )
        )


        try:

            final_result = (

                phase3k_distill_configurable_bbm92_key(

                    measurements=(
                        tuple(
                            measurements
                        )
                    ),

                    collection_summary=(
                        summary
                    ),

                    config=(
                        qkd_config
                    ),

                    postprocess_rng=(
                        postprocess_rng
                    )
                )
            )


        except RuntimeError as error:

            # ------------------------------------------------------
            # These are expected "need more entropy / reconciliation"
            # outcomes. Continue collecting Bell pairs.
            #
            # Physical delivery failures are NOT caught here.
            # ------------------------------------------------------

            message = str(
                error
            )


            retryable_prefixes = (

                "INSUFFICIENT_PARAMETER_ESTIMATION",

                "CASCADE_RECONCILIATION_FAILED",

                "INSUFFICIENT_POST_EC_ENTROPY"
            )


            if not any(

                message.startswith(
                    prefix
                )

                for prefix
                in retryable_prefixes
            ):

                raise


            if verbose:

                print(

                    "         "
                    "[POSTPROCESS RETRY REQUIRED] "
                    f"{message}"
                )


            continue


        # ==========================================================
        # ACTUAL SECRET KEY EXISTS
        # ==========================================================

        if (

            final_result.success

            and

            final_result.alice_key_bytes

            ==

            final_result.bob_key_bytes
        ):

            return (

                final_result,

                tuple(
                    measurements
                ),

                summary
            )


    # ==================================================================
    # Exhausted QKD pair budget
    # ==================================================================

    if final_summary is None:

        final_summary = (

            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    qkd_config
                )
            )
        )


    raise NARADQCServiceError(

        "QKD pair budget exhausted before actual "
        "post-processing produced the requested secret key. "
        f"Delivered={len(measurements)}, "
        f"forecast_extractable="
        f"{final_summary.forecast_extractable_bits}."
    )


# ======================================================================
# PUBLIC API
# ======================================================================

def narad_qc_establish_key(
    network,
    sender_name,
    receiver_name,
    *,
    qkd_config=PHASE3J_BIASED_BBM92_CONFIG,
    max_hops=None,
    physical_seed=NARAD_API_DEFAULT_PHYSICAL_SEED,
    protocol_seed=NARAD_API_DEFAULT_PROTOCOL_SEED,
    postprocess_seed=NARAD_API_DEFAULT_POSTPROCESS_SEED,
    verbose=True,
    return_details=False
):
    """
    COMPLETE NARAD-QC key-establishment API.

    Parameters
    ----------
    network
        OperatorQuantumNetwork containing the authoritative node/link graph.

    sender_name
        USER endpoint name.

    receiver_name
        USER endpoint name.

    qkd_config
        BiasedBBM92Config. Defaults to the frozen Phase-3J configuration.

    max_hops
        Optional Phase-4 path-discovery bound.

    physical_seed
        Reproducible simulator physical-RNG master seed.

    protocol_seed
        Reproducible QKD protocol-RNG master seed.

    postprocess_seed
        Independent post-processing RNG master seed.

    verbose
        Print CONTROL/PROTOCOL progress.
        The secret key itself is NEVER printed.

    return_details
        False:
            return only final key bytes.

        True:
            return NARADQCKeyEstablishmentResult.

    Returns
    -------
    bytes
        Final shared secret key, by default.

    OR

    NARADQCKeyEstablishmentResult
        Full audit result when return_details=True.


    Architecture
    ------------
    Phase 4:
        discovers + ranks possible paths.

    ML-DSA:
        freshly authenticates every selected path participant.

    Phase 3:
        generates Bell pairs, performs swapping/recovery,
        BBM92, reconciliation and privacy amplification.

    Route fallback:
        if a candidate cannot authenticate or cannot complete
        physical/QKD service, the next Policy-C-ranked route is tried.
    """

    # ==================================================================
    # Basic input validation
    # ==================================================================

    if not isinstance(
        network,
        OperatorQuantumNetwork
    ):

        raise TypeError(

            "network must be an OperatorQuantumNetwork."
        )


    if (

        not isinstance(
            sender_name,
            str
        )

        or

        not sender_name.strip()
    ):

        raise ValueError(
            "sender_name must be a non-empty string."
        )


    if (

        not isinstance(
            receiver_name,
            str
        )

        or

        not receiver_name.strip()
    ):

        raise ValueError(
            "receiver_name must be a non-empty string."
        )


    sender_name = (
        sender_name.strip()
    )


    receiver_name = (
        receiver_name.strip()
    )


    if sender_name == receiver_name:

        raise ValueError(

            "Sender and receiver must be different nodes."
        )


    # ==================================================================
    # PHASE 4 — PATH DISCOVERY + POLICY C
    # ==================================================================

    (
        ranked_routes,
        valid_paths,
        rejected_paths,
        physical_paths
    ) = (

        _narad_rank_routes_policy_c(

            network=(
                network
            ),

            sender_name=(
                sender_name
            ),

            receiver_name=(
                receiver_name
            ),

            max_hops=(
                max_hops
            )
        )
    )


    if verbose:

        print(
            "=" * 108
        )

        print(
            "NARAD-QC COMPLETE END-TO-END SERVICE"
        )

        print(
            "=" * 108
        )


        print(

            f"\nREQUEST: "
            f"{sender_name} -> {receiver_name}"
        )


        print(

            "\nPhysical simple paths:",
            len(
                physical_paths
            )
        )


        print(

            "Role-valid quantum paths:",
            len(
                valid_paths
            )
        )


        print(

            "Policy-C executable paths:",
            len(
                ranked_routes
            )
        )


        print(
            "\nPOLICY-C RANKING"
        )


        for rank, route in enumerate(

            ranked_routes,

            start=1
        ):

            candidate = (
                route[
                    "candidate"
                ]
            )


            print(

                f"  #{rank:<2} "
                f"{candidate.display_path} | "
                f"S={route['score']:.6f} | "
                f"Psvc="
                f"{route['predicted_service_probability']:.4f} | "
                f"T="
                f"{route['predicted_median_delivery_s']*1000:.3f}ms | "
                f"Qdel="
                f"{route['predicted_delivered_qber']:.3%}"
            )


    # ==================================================================
    # Ranked authentication / execution fallback
    # ==================================================================

    route_attempts = []


    for rank, route in enumerate(

        ranked_routes,

        start=1
    ):

        candidate = (
            route[
                "candidate"
            ]
        )


        if verbose:

            print(
                "\n" + "-" * 108
            )

            print(

                f"ATTEMPTING ROUTE #{rank}: "
                f"{candidate.display_path}"
            )

            print(
                "-" * 108
            )


        route_physical_seed = (

            _narad_derive_seed(

                master_seed=(
                    physical_seed
                ),

                path_index=(
                    candidate.path_index
                ),

                stream_tag=101
            )
        )


        route_protocol_seed = (

            _narad_derive_seed(

                master_seed=(
                    protocol_seed
                ),

                path_index=(
                    candidate.path_index
                ),

                stream_tag=202
            )
        )


        route_postprocess_seed = (

            _narad_derive_seed(

                master_seed=(
                    postprocess_seed
                ),

                path_index=(
                    candidate.path_index
                ),

                stream_tag=303
            )
        )


        # ==============================================================
        # Build ACTUAL routed Phase-3 stack
        # ==============================================================

        try:

            stack = (

                _narad_build_selected_path_stack(

                    network=(
                        network
                    ),

                    candidate_path=(
                        candidate
                    ),

                    simulation_seed=(
                        route_physical_seed
                    )
                )
            )


        except Exception as error:

            route_attempts.append(

                NARADQCRouteAttempt(

                    rank=rank,

                    path_index=(
                        candidate.path_index
                    ),

                    path=(
                        candidate.display_path
                    ),

                    policy_c_score=float(
                        route[
                            "score"
                        ]
                    ),

                    authenticated=False,

                    qkd_completed=False,

                    success=False,

                    failure_stage=(
                        "STACK_CONSTRUCTION"
                    ),

                    failure_reason=(
                        repr(
                            error
                        )
                    )
                )
            )


            if verbose:

                print(

                    "[ROUTE REJECTED — EXECUTION MODEL] ",
                    repr(
                        error
                    )
                )


            continue


        # ==============================================================
        # Fresh mutual ML-DSA authentication
        # ==============================================================

        try:

            authentication_records = (

                _narad_authenticate_selected_path(

                    stack
                )
            )


        except Exception as error:

            route_attempts.append(

                NARADQCRouteAttempt(

                    rank=rank,

                    path_index=(
                        candidate.path_index
                    ),

                    path=(
                        candidate.display_path
                    ),

                    policy_c_score=float(
                        route[
                            "score"
                        ]
                    ),

                    authenticated=False,

                    qkd_completed=False,

                    success=False,

                    failure_stage=(
                        "ML_DSA_AUTHENTICATION"
                    ),

                    failure_reason=(
                        repr(
                            error
                        )
                    )
                )
            )


            if verbose:

                print(

                    "[ROUTE REJECTED — AUTHENTICATION] ",
                    repr(
                        error
                    )
                )


            continue


        if verbose:

            print(

                "[PASS] Fresh mutual ML-DSA authentication "
                "completed for every selected-path node."
            )


            print(

                "[PASS] Quantum-operation authorization gate OPEN."
            )


        # ==============================================================
        # PHASE 3 — BELL PAIRS -> QKD -> POST PROCESSING
        # ==============================================================

        try:

            (
                qkd_result,
                measurements,
                collection_summary
            ) = (

                _narad_run_qkd_until_actual_key(

                    stack=(
                        stack
                    ),

                    qkd_config=(
                        qkd_config
                    ),

                    protocol_seed=(
                        route_protocol_seed
                    ),

                    postprocess_master_seed=(
                        route_postprocess_seed
                    ),

                    verbose=(
                        verbose
                    )
                )
            )


        except Exception as error:

            route_attempts.append(

                NARADQCRouteAttempt(

                    rank=rank,

                    path_index=(
                        candidate.path_index
                    ),

                    path=(
                        candidate.display_path
                    ),

                    policy_c_score=float(
                        route[
                            "score"
                        ]
                    ),

                    authenticated=True,

                    qkd_completed=False,

                    success=False,

                    failure_stage=(
                        "PHYSICAL_OR_QKD_SERVICE"
                    ),

                    failure_reason=(
                        repr(
                            error
                        )
                    )
                )
            )


            if verbose:

                print(

                    "[ROUTE FAILED DURING PHYSICAL/QKD SERVICE] ",
                    repr(
                        error
                    )
                )


                print(

                    "Trying next Policy-C-ranked route..."
                )


            continue


        # ==============================================================
        # FINAL KEY INVARIANTS
        # ==============================================================

        if not qkd_result.success:

            raise NARADQCServiceError(

                "Phase-3 QKD result returned success=False "
                "after post-processing."
            )


        if (

            qkd_result.alice_key_bytes

            !=

            qkd_result.bob_key_bytes
        ):

            raise NARADQCServiceError(

                "Final Alice/Bob keys disagree."
            )


        final_key = bytes(

            qkd_result
            .alice_key_bytes
        )


        expected_key_bytes = (

            int(
                qkd_config
                .target_secret_bits
            )

            // 8
        )


        if (

            len(
                final_key
            )

            !=

            expected_key_bytes
        ):

            raise NARADQCServiceError(

                "Final key length disagrees with "
                "requested secret-bit target."
            )


        route_attempts.append(

            NARADQCRouteAttempt(

                rank=rank,

                path_index=(
                    candidate.path_index
                ),

                path=(
                    candidate.display_path
                ),

                policy_c_score=float(
                    route[
                        "score"
                    ]
                ),

                authenticated=True,

                qkd_completed=True,

                success=True,

                failure_stage=None,

                failure_reason=None
            )
        )


        # ==============================================================
        # Build non-secret audit metadata
        # ==============================================================

        authentication_session_ids = tuple(

            record.session_id

            for record
            in authentication_records
        )


        result = (

            NARADQCKeyEstablishmentResult(

                success=True,

                sender=(
                    sender_name
                ),

                receiver=(
                    receiver_name
                ),

                selected_rank=(
                    rank
                ),

                selected_path_index=(
                    candidate.path_index
                ),

                selected_path=(
                    candidate.display_path
                ),

                selected_node_names=tuple(
                    candidate.node_names
                ),

                policy_c_score=float(
                    route[
                        "score"
                    ]
                ),

                predicted_service_probability=float(

                    route[
                        "predicted_service_probability"
                    ]
                ),

                predicted_median_delivery_s=float(

                    route[
                        "predicted_median_delivery_s"
                    ]
                ),

                predicted_delivered_visibility=float(

                    route[
                        "predicted_delivered_visibility"
                    ]
                ),

                predicted_delivered_qber=float(

                    route[
                        "predicted_delivered_qber"
                    ]
                ),

                delivered_pairs=int(

                    qkd_result
                    .delivered_pairs
                ),

                zz_raw_key_bits=int(

                    qkd_result
                    .zz_raw_key_bits
                ),

                xx_parameter_estimation_bits=int(

                    qkd_result
                    .xx_parameter_estimation_bits
                ),

                observed_x_qber=float(

                    qkd_result
                    .observed_x_qber
                ),

                phase_error_estimate=float(

                    qkd_result
                    .phase_error_estimate
                ),

                ec_leakage_bits=int(

                    qkd_result
                    .ec_leakage_bits
                ),

                maximum_extractable_bits=int(

                    qkd_result
                    .maximum_extractable_bits
                ),

                final_secret_bits=int(

                    qkd_result
                    .final_secret_bits
                ),

                modeled_physical_time_s=float(

                    qkd_result
                    .physical_time_s
                ),

                elementary_generation_cycles=int(

                    qkd_result
                    .elementary_generation_cycles
                ),

                optical_mode_attempts=int(

                    qkd_result
                    .optical_mode_attempts
                ),

                bsm_attempts=int(

                    qkd_result
                    .bsm_attempts
                ),

                authentication_session_ids=(
                    authentication_session_ids
                ),

                attempted_routes=tuple(
                    route_attempts
                ),

                key=(
                    final_key
                )
            )
        )


        if verbose:

            print(
                "\n" + "=" * 108
            )

            print(
                "NARAD-QC END-TO-END KEY ESTABLISHMENT SUCCESS"
            )

            print(
                "=" * 108
            )


            print(

                "\nSelected route:",
                candidate.display_path
            )


            print(

                "Policy-C rank:",
                rank
            )


            print(

                "Delivered Bell pairs:",
                qkd_result.delivered_pairs
            )


            print(

                "Observed X-basis QBER:",
                f"{qkd_result.observed_x_qber:.4%}"
            )


            print(

                "Phase-error estimate:",
                f"{qkd_result.phase_error_estimate:.4%}"
            )


            print(

                "Cascade leakage:",
                qkd_result.ec_leakage_bits,
                "bits"
            )


            print(

                "Maximum extractable:",
                qkd_result.maximum_extractable_bits,
                "bits"
            )


            print(

                "Final secret:",
                qkd_result.final_secret_bits,
                "bits"
            )


            print(

                "Modeled physical time:",
                f"{qkd_result.physical_time_s:.6f} s"
            )


            print(
                "\n[PASS] Alice and Bob hold identical secret keys."
            )


            print(
                "[SECURITY] Secret key value intentionally NOT printed."
            )


            print(
                "\n" + "=" * 108
            )


        return (

            result

            if return_details

            else final_key
        )


    # ==================================================================
    # No ranked route completed service
    # ==================================================================

    failure_lines = [

        (
            f"rank={attempt.rank}, "
            f"path={attempt.path}, "
            f"stage={attempt.failure_stage}, "
            f"reason={attempt.failure_reason}"
        )

        for attempt
        in route_attempts
    ]


    raise NARADQCNoExecutableRouteError(

        "Every Policy-C-ranked route failed before secret-key "
        "establishment.\n"

        +

        "\n".join(
            failure_lines
        )
    )


print(
    "=" * 108
)

print(
    "NARAD-QC COMPLETE CALLABLE API READY"
)

print(
    "=" * 108
)

print(
    "\nPublic function:"
)

print(
    "    narad_qc_establish_key(network, sender_name, receiver_name)"
)

print(
    "\nPipeline:"
)

print(
    "    Operator Graph"
)

print(
    "       -> Phase-4 Path Discovery"
)

print(
    "       -> Policy-C Routing"
)

print(
    "       -> Fresh ML-DSA Mutual Authentication"
)

print(
    "       -> Quantum Authorization"
)

print(
    "       -> Bell-Pair Establishment"
)

print(
    "       -> Multiplexed Failure-Aware Entanglement Swapping"
)

print(
    "       -> Biased BBM92"
)

print(
    "       -> Parameter Estimation"
)

print(
    "       -> Cascade Reconciliation"
)

print(
    "       -> Toeplitz Privacy Amplification"
)

print(
    "       -> Shared Secret Key"
)

print(
    "\n[PASS] NARAD-QC integration layer compiled."
)

In [ ]:
# @title NARAD-QC — FIRST COMPLETE NETWORK SERVICE REQUEST

# narad_key = narad_qc_establish_key(

#     network=phase4_network,

#     sender_name="Charlie",

#     receiver_name="Bob",

#     verbose=True
# )


# assert isinstance(
#     narad_key,
#     bytes
# )


# assert len(
#     narad_key
# ) == 16


# print(
#     "\n[PASS] Application received a 128-bit NARAD-QC session key."
# )

# print(
#     "[SECURITY] Key value intentionally hidden."
# )

In [ ]:
# print(narad_key.hex())

## Phase 6

API Exposure

In [ ]:
# @title NARAD-QC API — Install Dependencies

!pip -q install fastapi uvicorn nest-asyncio pydantic
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("[PASS] FastAPI dependencies installed.")

In [ ]:
# @title NARAD-QC — FastAPI Service Layer

import base64
import hashlib
import os
import secrets
import threading
import time
import uuid

from dataclasses import asdict, replace
from typing import Optional, Any

from fastapi import (
    FastAPI,
    HTTPException
)

from fastapi.middleware.cors import CORSMiddleware

from pydantic import (
    BaseModel,
    Field
)


# ======================================================================
# API
# ======================================================================

NARAD_API_VERSION = "1.0.0"

app = FastAPI(

    title="NARAD-QC API",

    description=(
        "Network Architecture for Routing, Authentication and "
        "Delivery in Quantum Communications"
    ),

    version=NARAD_API_VERSION
)


# ======================================================================
# CORS
#
# For a temporary research demo we allow all origins.
#
# For deployment later:
# replace "*" with the actual frontend origin.
# ======================================================================

app.add_middleware(

    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=False,

    allow_methods=["*"],

    allow_headers=["*"]
)


# ======================================================================
# Runtime session store
#
# IMPORTANT:
# Keys remain SERVER-SIDE.
# The REST API never returns raw session keys.
# ======================================================================

NARAD_API_SESSIONS = {}

NARAD_API_SESSION_LOCK = threading.Lock()


# ======================================================================
# Request / response schemas
# ======================================================================

class QKDConfigRequest(BaseModel):

    target_secret_bits: int = Field(
        128,
        ge=8,
        description="Desired final secret-key length in bits."
    )

    z_probability: float = Field(
        0.80,
        gt=0.0,
        lt=1.0
    )

    qber_abort_threshold: float = Field(
        0.11,
        gt=0.0,
        lt=0.5
    )

    minimum_parameter_estimation_bits: int = Field(
        24,
        ge=1
    )

    parameter_estimation_margin: float = Field(
        0.01,
        ge=0.0,
        lt=0.5
    )

    security_margin_bits: int = Field(
        32,
        ge=0
    )

    reconciliation_efficiency_forecast: float = Field(
        1.16,
        ge=1.0
    )

    collection_headroom_bits: int = Field(
        32,
        ge=0
    )

    collection_check_interval_pairs: int = Field(
        16,
        ge=1
    )

    max_delivered_pairs: int = Field(
        5000,
        ge=1
    )

    max_slots_per_pair: int = Field(
        20000,
        ge=1
    )

    cascade_passes: int = Field(
        8,
        ge=1
    )


class ExecutionConfigRequest(BaseModel):

    cpu_workers: int = Field(
        1,
        ge=1,
        le=64
    )

    parallel_quantum_lanes: int = Field(
        1,
        ge=1,
        le=512
    )

    max_hops: Optional[int] = Field(
        None,
        ge=1
    )

    physical_seed: int = (
        NARAD_API_DEFAULT_PHYSICAL_SEED
    )

    protocol_seed: int = (
        NARAD_API_DEFAULT_PROTOCOL_SEED
    )

    postprocess_seed: int = (
        NARAD_API_DEFAULT_POSTPROCESS_SEED
    )


class RoutePreviewRequest(BaseModel):

    sender: str

    receiver: str

    max_hops: Optional[int] = None


class SessionEstablishRequest(BaseModel):

    sender: str

    receiver: str

    qkd: QKDConfigRequest = QKDConfigRequest()

    execution: ExecutionConfigRequest = (
        ExecutionConfigRequest()
    )


class MessageSendRequest(BaseModel):

    session_id: str

    sender: str

    receiver: str

    message: str


class MessageDecryptRequest(BaseModel):

    session_id: str

    ciphertext_b64: str


# ======================================================================
# Helpers
# ======================================================================

def narad_api_make_qkd_config(
    request: QKDConfigRequest
):

    # ================================================================
    # AES only accepts 128 / 192 / 256-bit keys in the current
    # application layer.
    # ================================================================

    if request.target_secret_bits not in (
        128,
        192,
        256
    ):

        raise ValueError(

            "target_secret_bits must be one of "
            "128, 192, or 256 for AES-GCM message service."
        )


    return replace(

        PHASE3J_BIASED_BBM92_CONFIG,

        target_secret_bits=(
            request.target_secret_bits
        ),

        z_probability=(
            request.z_probability
        ),

        qber_abort_threshold=(
            request.qber_abort_threshold
        ),

        minimum_parameter_estimation_bits=(
            request.minimum_parameter_estimation_bits
        ),

        parameter_estimation_margin=(
            request.parameter_estimation_margin
        ),

        security_margin_bits=(
            request.security_margin_bits
        ),

        reconciliation_efficiency_forecast=(
            request.reconciliation_efficiency_forecast
        ),

        collection_headroom_bits=(
            request.collection_headroom_bits
        ),

        collection_check_interval_pairs=(
            request.collection_check_interval_pairs
        ),

        max_delivered_pairs=(
            request.max_delivered_pairs
        ),

        max_slots_per_pair=(
            request.max_slots_per_pair
        ),

        cascade_passes=(
            request.cascade_passes
        )
    )


def narad_api_public_result(
    result
):

    """
    Convert the complete internal result into safe public metadata.

    Deliberately excludes:
        result.key
    """

    return {

        "success":
            result.success,

        "sender":
            result.sender,

        "receiver":
            result.receiver,

        "selected_rank":
            result.selected_rank,

        "selected_path_index":
            result.selected_path_index,

        "selected_path":
            result.selected_path,

        "selected_node_names":
            list(
                result.selected_node_names
            ),

        "policy_c_score":
            result.policy_c_score,

        "predicted_service_probability":
            result.predicted_service_probability,

        "predicted_median_delivery_s":
            result.predicted_median_delivery_s,

        "predicted_delivered_visibility":
            result.predicted_delivered_visibility,

        "predicted_delivered_qber":
            result.predicted_delivered_qber,

        "delivered_pairs":
            result.delivered_pairs,

        "zz_raw_key_bits":
            result.zz_raw_key_bits,

        "xx_parameter_estimation_bits":
            result.xx_parameter_estimation_bits,

        "observed_x_qber":
            result.observed_x_qber,

        "phase_error_estimate":
            result.phase_error_estimate,

        "ec_leakage_bits":
            result.ec_leakage_bits,

        "maximum_extractable_bits":
            result.maximum_extractable_bits,

        "final_secret_bits":
            result.final_secret_bits,

        "modeled_physical_time_s":
            result.modeled_physical_time_s,

        "elementary_generation_cycles":
            result.elementary_generation_cycles,

        "optical_mode_attempts":
            result.optical_mode_attempts,

        "bsm_attempts":
            result.bsm_attempts,

        "authentication_sessions":
            len(
                result.authentication_session_ids
            ),

        "routes_attempted":
            [

                asdict(
                    attempt
                )

                for attempt
                in result.attempted_routes
            ]
    }


def narad_api_get_session(
    session_id
):

    with NARAD_API_SESSION_LOCK:

        session = (

            NARAD_API_SESSIONS.get(
                session_id
            )
        )


    if session is None:

        raise HTTPException(

            status_code=404,

            detail="Unknown or expired NARAD-QC session."
        )


    return session


# ======================================================================
# HEALTH
# ======================================================================

@app.get("/health")
def health():

    return {

        "status":
            "ok",

        "service":
            "NARAD-QC",

        "version":
            NARAD_API_VERSION,

        "active_sessions":
            len(
                NARAD_API_SESSIONS
            )
    }


# ======================================================================
# CAPABILITIES
# ======================================================================

@app.get("/capabilities")
def capabilities():

    return {

        "routing": {

            "policy":
                "Policy C",

            "dynamic_sender_receiver":
                True,

            "max_hops":
                True,

            "ranked_route_fallback":
                True
        },


        "authentication": {

            "algorithm":
                "ML-DSA-65",

            "fresh_per_route":
                True,

            "hard_quantum_authorization_gate":
                True
        },


        "qkd": {

            "protocol":
                "Biased BBM92",

            "configurable":
                [

                    "target_secret_bits",

                    "z_probability",

                    "qber_abort_threshold",

                    "minimum_parameter_estimation_bits",

                    "parameter_estimation_margin",

                    "security_margin_bits",

                    "reconciliation_efficiency_forecast",

                    "collection_headroom_bits",

                    "collection_check_interval_pairs",

                    "max_delivered_pairs",

                    "max_slots_per_pair",

                    "cascade_passes"
                ]
        },


        "messaging": {

            "cipher":
                "AES-GCM",

            "plaintext_input":
                "UTF-8",

            "ciphertext_transport":
                "Base64",

            "raw_key_exposed":
                False
        },


        # ============================================================
        # IMPORTANT SCIENTIFIC HONESTY:
        #
        # These Phase-3K controls exist in the project, but the current
        # heterogeneous routed executor does not yet use the farm.
        # ============================================================

        "execution": {

            "heterogeneous_routed_executor": {

                "cpu_workers_applied":
                    1,

                "parallel_quantum_lanes_applied":
                    1
            },

            "phase3k_homogeneous_backend_available":
                True,

            "requested_worker_fields_exposed":
                True,

            "note": (
                "CPU workers and physical quantum lanes are exposed "
                "for GUI configuration, but the current heterogeneous "
                "Policy-C routed executor runs one physical service "
                "lane. They are not silently treated as active."
            )
        }
    }


# ======================================================================
# NETWORK
# ======================================================================

@app.get("/network")
def network_summary():

    return {

        "name":
            phase4_network
            .topology
            .name,

        "node_count":
            phase4_network
            .topology
            .node_count,

        "link_count":
            phase4_network
            .topology
            .link_count
    }


@app.get("/nodes")
def nodes():

    rows = []


    for node in (

        phase4_network
        .topology
        .nodes
    ):

        profile = (

            phase4_network
            .node_profile(
                node.name
            )
        )


        rows.append({

            "name":
                node.name,

            "role":
                profile.role,

            "active":
                profile.active,

            "x_km":
                node.x_km,

            "y_km":
                node.y_km,

            "memory_slots_total":
                profile.memory_slots_total,

            "memory_slots_available":
                profile.memory_slots_available,

            "bsm_units_total":
                profile.bsm_units_total,

            "bsm_units_available":
                profile.bsm_units_available,

            "bsm_success_probability":
                profile.bsm_success_probability
        })


    return {
        "nodes": rows
    }


# ======================================================================
# ROUTE PREVIEW
#
# Does NOT establish a key.
# Excellent for GUI visualization.
# ======================================================================

@app.post("/routes/preview")
def preview_routes(
    request: RoutePreviewRequest
):

    try:

        (
            ranked,
            valid_paths,
            rejected_paths,
            physical_paths
        ) = _narad_rank_routes_policy_c(

            network=(
                phase4_network
            ),

            sender_name=(
                request.sender
            ),

            receiver_name=(
                request.receiver
            ),

            max_hops=(
                request.max_hops
            )
        )


        rows = []


        for rank, record in enumerate(
            ranked,
            start=1
        ):

            candidate = (
                record[
                    "candidate"
                ]
            )


            rows.append({

                "rank":
                    rank,

                "path_index":
                    candidate.path_index,

                "path":
                    candidate.display_path,

                "nodes":
                    list(
                        candidate.node_names
                    ),

                "repeaters":
                    list(
                        candidate.repeater_names
                    ),

                "score":
                    record[
                        "score"
                    ],

                "service_probability":
                    record[
                        "predicted_service_probability"
                    ],

                "predicted_delivery_s":
                    record[
                        "predicted_median_delivery_s"
                    ],

                "predicted_visibility":
                    record[
                        "predicted_delivered_visibility"
                    ],

                "predicted_qber":
                    record[
                        "predicted_delivered_qber"
                    ],

                "bottleneck_elementary_distance_km":
                    record[
                        "bottleneck_elementary_distance_km"
                    ],

                "bottleneck_mux_success":
                    record[
                        "bottleneck_multiplexed_success_probability"
                    ]
            })


        return {

            "sender":
                request.sender,

            "receiver":
                request.receiver,

            "physical_paths":
                len(
                    physical_paths
                ),

            "role_valid_paths":
                len(
                    valid_paths
                ),

            "rejected_user_transit_paths":
                len(
                    rejected_paths
                ),

            "ranked_routes":
                rows
        }


    except Exception as error:

        raise HTTPException(

            status_code=400,

            detail=str(
                error
            )
        )


# ======================================================================
# ESTABLISH NARAD-QC SESSION
# ======================================================================

@app.post("/session/establish")
def establish_session(
    request: SessionEstablishRequest
):

    try:

        qkd_config = (

            narad_api_make_qkd_config(
                request.qkd
            )
        )


        # ============================================================
        # CURRENT EXECUTOR LIMITATION
        #
        # Accept requested values so GUI can experiment with the API
        # schema, but report what is actually applied.
        # ============================================================

        requested_cpu_workers = (
            request.execution.cpu_workers
        )

        requested_parallel_lanes = (
            request.execution.parallel_quantum_lanes
        )


        result = (

            narad_qc_establish_key(

                network=(
                    phase4_network
                ),

                sender_name=(
                    request.sender
                ),

                receiver_name=(
                    request.receiver
                ),

                qkd_config=(
                    qkd_config
                ),

                max_hops=(
                    request.execution.max_hops
                ),

                physical_seed=(
                    request.execution.physical_seed
                ),

                protocol_seed=(
                    request.execution.protocol_seed
                ),

                postprocess_seed=(
                    request.execution.postprocess_seed
                ),

                verbose=False,

                return_details=True
            )
        )


        session_id = (
            str(
                uuid.uuid4()
            )
        )


        now = time.time()


        with NARAD_API_SESSION_LOCK:

            NARAD_API_SESSIONS[
                session_id
            ] = {

                "session_id":
                    session_id,

                "created_at_unix":
                    now,

                "sender":
                    request.sender,

                "receiver":
                    request.receiver,

                # KEY NEVER RETURNED
                "key":
                    bytes(
                        result.key
                    ),

                "qkd_config":
                    qkd_config,

                "result":
                    result,

                "message_count":
                    0
            }


        response = {

            "session_id":
                session_id,

            "status":
                "ESTABLISHED",

            "result":
                narad_api_public_result(
                    result
                ),

            "execution": {

                "requested_cpu_workers":
                    requested_cpu_workers,

                "applied_cpu_workers":
                    1,

                "requested_parallel_quantum_lanes":
                    requested_parallel_lanes,

                "applied_parallel_quantum_lanes":
                    1,

                "note": (
                    "Current heterogeneous routed executor is "
                    "single-lane. Requested Phase-3K parallelism "
                    "is recorded but not silently applied."
                )
            }
        }


        return response


    except HTTPException:

        raise


    except Exception as error:

        raise HTTPException(

            status_code=400,

            detail=(

                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ======================================================================
# SESSION STATUS
# ======================================================================

@app.get("/session/{session_id}")
def get_session(
    session_id: str
):

    session = (
        narad_api_get_session(
            session_id
        )
    )


    return {

        "session_id":
            session_id,

        "status":
            "ESTABLISHED",

        "sender":
            session[
                "sender"
            ],

        "receiver":
            session[
                "receiver"
            ],

        "message_count":
            session[
                "message_count"
            ],

        "result":
            narad_api_public_result(

                session[
                    "result"
                ]
            )
    }


# ======================================================================
# SEND ENCRYPTED APPLICATION MESSAGE
# ======================================================================

@app.post("/message/send")
def send_message(
    request: MessageSendRequest
):

    session = (

        narad_api_get_session(
            request.session_id
        )
    )


    # ================================================================
    # Endpoint binding
    # ================================================================================

    if (

        request.sender
        !=
        session[
            "sender"
        ]

        or

        request.receiver
        !=
        session[
            "receiver"
        ]
    ):

        raise HTTPException(

            status_code=403,

            detail=(
                "Message endpoints do not match "
                "the authenticated NARAD-QC session."
            )
        )


    plaintext = (

        request.message
        .encode(
            "utf-8"
        )
    )


    encrypted_blob = (

        aes_gcm_encrypt(

            plaintext,

            session[
                "key"
            ]
        )
    )


    ciphertext_b64 = (

        base64.b64encode(
            encrypted_blob
        )
        .decode(
            "ascii"
        )
    )


    with NARAD_API_SESSION_LOCK:

        session[
            "message_count"
        ] += 1


        message_index = (
            session[
                "message_count"
            ]
        )


    return {

        "session_id":
            request.session_id,

        "message_index":
            message_index,

        "sender":
            request.sender,

        "receiver":
            request.receiver,

        "cipher":
            "AES-GCM",

        "ciphertext_b64":
            ciphertext_b64,

        "plaintext_bytes":
            len(
                plaintext
            ),

        "encrypted_blob_bytes":
            len(
                encrypted_blob
            )
    }


# ======================================================================
# RECEIVER-SIDE MESSAGE DECRYPTION
#
# Useful for your GUI:
#
# left panel  : Alice sends
# middle      : encrypted channel/blob
# right panel : Bob decrypts
# ======================================================================

@app.post("/message/decrypt")
def decrypt_message(
    request: MessageDecryptRequest
):

    session = (

        narad_api_get_session(
            request.session_id
        )
    )


    try:

        encrypted_blob = (

            base64.b64decode(

                request.ciphertext_b64,

                validate=True
            )
        )


        plaintext = (

            aes_gcm_decrypt(

                encrypted_blob,

                session[
                    "key"
                ]
            )
        )


        text = (

            plaintext.decode(
                "utf-8"
            )
        )


    except Exception:

        raise HTTPException(

            status_code=400,

            detail=(
                "AES-GCM authentication/decryption failed."
            )
        )


    return {

        "session_id":
            request.session_id,

        "receiver":
            session[
                "receiver"
            ],

        "verified":
            True,

        "message":
            text
    }


# ======================================================================
# DESTROY SESSION
#
# Removes server-side key material.
# ======================================================================

@app.delete("/session/{session_id}")
def destroy_session(
    session_id: str
):

    with NARAD_API_SESSION_LOCK:

        session = (

            NARAD_API_SESSIONS.pop(
                session_id,
                None
            )
        )


    if session is None:

        raise HTTPException(

            status_code=404,

            detail="Unknown session."
        )


    return {

        "session_id":
            session_id,

        "status":
            "DESTROYED",

        "key_material_retained":
            False
    }


print("=" * 100)
print("NARAD-QC FASTAPI SERVICE READY")
print("=" * 100)

print("\nEndpoints:")
print("  GET    /health")
print("  GET    /capabilities")
print("  GET    /network")
print("  GET    /nodes")
print("  POST   /routes/preview")
print("  POST   /session/establish")
print("  GET    /session/{session_id}")
print("  POST   /message/send")
print("  POST   /message/decrypt")
print("  DELETE /session/{session_id}")

print("\n[PASS] Secret keys remain server-side.")

In [ ]:
# @title  Core Semantic Hardening

from contextvars import ContextVar
from dataclasses import replace
import math
import numpy as np


# ======================================================================
# NARAD-QC PHASE 6.1A
#
# Fixes:
#
#   BUG 1:
#       Request-level BBM92 QBER threshold previously affected only
#       protocol post-processing.
#
#       It now also governs:
#
#           Phase-4 fresh quality admission
#           Policy-C delivered-quality admission
#           Policy-C quality utility
#           Phase-3 QKD-ready delivery admission
#           selective recovery planning
#           BBM92 protocol abort
#
#
#   BUG 2:
#       If max_delivered_pairs was not an exact multiple of
#       collection_check_interval_pairs, the final partial block
#       could be discarded without one last ACTUAL distillation try.
#
#       The final delivered pair now ALWAYS triggers:
#
#           summary
#           -> readiness
#           -> actual Cascade
#           -> entropy accounting
#           -> Toeplitz PA
#
#       if eligible.
#
# ======================================================================


# ======================================================================
# Preserve the already-validated pre-hardening implementations.
#
# Guards make this cell safe to re-run.
# ======================================================================

if (
    "_NARAD_PRE_6_1A_ESTABLISH_KEY"
    not in globals()
):

    _NARAD_PRE_6_1A_ESTABLISH_KEY = (
        narad_qc_establish_key
    )


if (
    "_NARAD_PRE_6_1A_BUILD_SELECTED_PATH_STACK"
    not in globals()
):

    _NARAD_PRE_6_1A_BUILD_SELECTED_PATH_STACK = (
        _narad_build_selected_path_stack
    )


# ======================================================================
# Request-local QKD configuration
#
# ContextVar rather than a mutable ordinary global:
#
#   - safe across nested calls;
#   - automatically reset;
#   - avoids permanently mutating frozen Phase-3/4 baseline globals.
#
# The old notebook defaults therefore remain reproducible.
# ======================================================================

_NARAD_REQUEST_QKD_CONFIG = ContextVar(

    "NARAD_REQUEST_QKD_CONFIG",

    default=None
)


def _narad_current_qkd_config():

    request_config = (
        _NARAD_REQUEST_QKD_CONFIG.get()
    )


    if request_config is None:

        return (
            PHASE3J_BIASED_BBM92_CONFIG
        )


    return request_config


def _narad_current_quality_config():

    qkd_config = (
        _narad_current_qkd_config()
    )


    return replace(

        PHASE3H1_QUALITY_CONFIG,

        qber_limit=float(

            qkd_config
            .qber_abort_threshold
        )
    )


# ======================================================================
# Policy-C candidate evaluator
#
# QBER threshold is now REQUEST-SCOPED.
# ======================================================================

def _narad_policy_c_evaluate_candidate(
    network,
    candidate_path,
    raw_metrics
):

    features = (

        _narad_policy_c_features(

            network,

            candidate_path
        )
    )


    d_b = float(

        features[
            "bottleneck_elementary_distance_km"
        ]
    )


    p_b = float(

        features[
            "bottleneck_multiplexed_success_probability"
        ]
    )


    repeater_count = int(

        features[
            "repeater_count"
        ]
    )


    x = np.array(
        [[

            d_b,

            float(
                repeater_count
            ),

            p_b

        ]],

        dtype=float
    )


    # ==================================================================
    # 1. Calibrated service reliability
    # ==================================================================

    raw_reliability_score = float(

        PHASE4_FINAL_RELIABILITY_MODEL
        .decision_function(
            x
        )[0]
    )


    predicted_service_probability = float(

        PHASE4_FINAL_RELIABILITY_CALIBRATOR
        .predict_proba(

            np.array(
                [[
                    raw_reliability_score
                ]],

                dtype=float
            )

        )[0, 1]
    )


    predicted_service_probability = float(

        np.clip(

            predicted_service_probability,

            0.0,

            1.0
        )
    )


    # ==================================================================
    # 2. Conditional successful-pair latency
    # ==================================================================

    predicted_log_delivery_s = float(

        PHASE4_FINAL_LATENCY_MODEL
        .predict(
            x
        )[0]
    )


    predicted_delivery_s = float(

        math.exp(
            predicted_log_delivery_s
        )
    )


    if (

        not math.isfinite(
            predicted_delivery_s
        )

        or

        predicted_delivery_s
        <=
        0.0
    ):

        raise NARADQCServiceError(

            "Policy-C latency predictor returned "
            "invalid latency."
        )


    # ==================================================================
    # 3. Empirical delivered visibility
    # ==================================================================

    predicted_visibility_delta = float(

        PHASE4_FINAL_VISIBILITY_MODEL
        .predict(
            x
        )[0]
    )


    fresh_visibility = float(

        raw_metrics
        .predicted_werner_visibility
    )


    delivered_visibility = float(

        np.clip(

            fresh_visibility
            +
            predicted_visibility_delta,

            0.0,

            1.0
        )
    )


    delivered_qber = float(

        (
            1.0
            -
            delivered_visibility
        )

        /
        2.0
    )


    # ==================================================================
    # REQUEST-SCOPED QBER QUALITY BOUNDARY
    # ==================================================================

    quality_config = (
        _narad_current_quality_config()
    )


    qber_limit = float(

        quality_config
        .qber_limit
    )


    numerical_tolerance = float(

        quality_config
        .numerical_tolerance
    )


    qkd_feasible = bool(

        delivered_qber

        <=

        (
            qber_limit
            +
            numerical_tolerance
        )
    )


    minimum_visibility = float(

        quality_config
        .minimum_end_to_end_visibility
    )


    # ==================================================================
    # Quality utility:
    #
    #       0 at requested QBER boundary
    #       1 at ideal visibility
    # ==================================================================

    quality_utility = float(

        np.clip(

            (
                delivered_visibility

                -

                minimum_visibility
            )

            /

            (
                1.0

                -

                minimum_visibility
            ),

            0.0,

            1.0
        )
    )


    score = (

        float(

            (
                predicted_service_probability

                /

                predicted_delivery_s
            )

            *

            quality_utility
        )

        if qkd_feasible

        else 0.0
    )


    # ==================================================================
    # Calibration-domain diagnostic
    #
    # Do NOT reject on extrapolation.
    # Only flag it.
    # ==================================================================

    if callable(

        globals().get(
            "phase4_policy_c_calibration_domain_audit"
        )
    ):

        (
            within_calibration_domain,
            extrapolation_fields
        ) = (

            phase4_policy_c_calibration_domain_audit(

                bottleneck_elementary_distance_km=(
                    d_b
                ),

                repeater_count=(
                    repeater_count
                ),

                bottleneck_multiplexed_success_probability=(
                    p_b
                )
            )
        )


    else:

        within_calibration_domain = True

        extrapolation_fields = tuple()


    return {

        "candidate":
            candidate_path,

        "metrics":
            raw_metrics,

        "score":
            score,

        "predicted_service_probability":
            predicted_service_probability,

        "predicted_median_delivery_s":
            predicted_delivery_s,

        "predicted_visibility_delta":
            predicted_visibility_delta,

        "predicted_delivered_visibility":
            delivered_visibility,

        "predicted_delivered_qber":
            delivered_qber,

        "quality_utility":
            quality_utility,

        "empirical_qkd_feasible":
            qkd_feasible,

        "requested_qber_limit":
            qber_limit,

        "bottleneck_elementary_distance_km":
            d_b,

        "bottleneck_multiplexed_success_probability":
            p_b,

        "within_calibration_domain":
            bool(
                within_calibration_domain
            ),

        "extrapolation_fields":
            tuple(
                extrapolation_fields
            )
    }


# ======================================================================
# Policy-C route discovery/ranking
#
# IMPORTANT:
#
# We must NOT use:
#
#       metrics.statically_feasible
#
# directly anymore because that boolean was calculated earlier using
# the frozen 0.11 routing threshold.
#
# Instead:
#
#       static RESOURCE feasibility
#           +
#       fresh quality under REQUEST threshold
#           +
#       empirical delivered quality under REQUEST threshold
#
# ======================================================================

def _narad_rank_routes_policy_c(
    network,
    sender_name,
    receiver_name,
    max_hops=None
):

    (
        valid_paths,
        rejected_paths,
        physical_paths
    ) = (

        phase4_discover_valid_quantum_paths(

            network=(
                network
            ),

            sender_name=(
                sender_name
            ),

            receiver_name=(
                receiver_name
            ),

            max_hops=(
                max_hops
            )
        )
    )


    quality_config = (
        _narad_current_quality_config()
    )


    qber_limit = float(

        quality_config
        .qber_limit
    )


    tolerance = float(

        quality_config
        .numerical_tolerance
    )


    evaluated = []


    for candidate in valid_paths:

        metrics = (

            phase4_evaluate_candidate_path(

                network=(
                    network
                ),

                candidate_path=(
                    candidate
                )
            )
        )


        # ==========================================================
        # STATIC HARDWARE / CONTROL FEASIBILITY
        #
        # Deliberately excludes old qkd_quality_feasible boolean.
        # ==========================================================

        resource_feasible = all(
            (
                bool(
                    metrics.all_nodes_active
                ),

                bool(
                    metrics.all_links_active
                ),

                bool(
                    metrics.multiplexing_feasible
                ),

                bool(
                    metrics.memory_feasible
                ),

                bool(
                    metrics.bsm_feasible
                )
            )
        )


        if not resource_feasible:

            continue


        # ==========================================================
        # FRESH QUALITY GATE — REQUEST threshold
        # ==========================================================

        fresh_qkd_feasible = bool(

            float(
                metrics.predicted_qber
            )

            <=

            (
                qber_limit
                +
                tolerance
            )
        )


        if not fresh_qkd_feasible:

            continue


        # ==========================================================
        # EMPIRICAL DELIVERED QUALITY + POLICY-C SCORE
        # ==========================================================

        policy_c = (

            _narad_policy_c_evaluate_candidate(

                network=(
                    network
                ),

                candidate_path=(
                    candidate
                ),

                raw_metrics=(
                    metrics
                )
            )
        )


        if not policy_c[
            "empirical_qkd_feasible"
        ]:

            continue


        evaluated.append(
            policy_c
        )


    if not evaluated:

        raise Phase4NoValidPathError(

            f"No Policy-C executable QKD route exists "
            f"between '{sender_name}' and '{receiver_name}' "
            f"under requested QBER limit "
            f"{qber_limit:.6f}."
        )


    evaluated.sort(

        key=lambda record: (

            -record[
                "score"
            ],

            -record[
                "predicted_service_probability"
            ],

            record[
                "predicted_median_delivery_s"
            ],

            -record[
                "predicted_delivered_visibility"
            ],

            record[
                "candidate"
            ]
            .path_index
        )
    )


    return (

        tuple(
            evaluated
        ),

        tuple(
            valid_paths
        ),

        tuple(
            rejected_paths
        ),

        tuple(
            physical_paths
        )
    )


# ======================================================================
# Phase-4 -> Phase-3 adapter patch
#
# Reuse the already-working exact heterogeneous stack builder.
#
# Then replace ONLY the QKD-quality orchestrator/recovery objects with
# request-scoped quality configuration.
#
# Physical topology, RNG, generation, fidelity, swap model, memory,
# timing ledger and multiplexing remain untouched.
# ======================================================================

def _narad_build_selected_path_stack(
    network,
    candidate_path,
    simulation_seed
):

    stack = (

        _NARAD_PRE_6_1A_BUILD_SELECTED_PATH_STACK(

            network=(
                network
            ),

            candidate_path=(
                candidate_path
            ),

            simulation_seed=(
                simulation_seed
            )
        )
    )


    quality_config = (
        _narad_current_quality_config()
    )


    # ==================================================================
    # Rebuild top-level QKD-ready orchestrator with the request's
    # quality boundary.
    # ==================================================================

    orchestrator = (

        PhysicalTimedQKDReadyOrchestrator(

            stack=(
                stack
            ),

            timing_profile=(
                stack[
                    "timing_profile"
                ]
            ),

            generation_timing_geometry=(

                stack[
                    "generation_timing_geometry"
                ]
            ),

            config=(
                quality_config
            )
        )
    )


    stack[
        "orchestrator"
    ] = orchestrator


    # ==================================================================
    # Reinstall recovery using SAME request-scoped QBER budget.
    # ==================================================================

    phase3i2_install_multiplexing_aware_recovery(

        stack,

        config=(
            quality_config
        )
    )


    # ==================================================================
    # Audit metadata
    # ==================================================================

    stack[
        "request_quality_config"
    ] = quality_config


    stack[
        "request_qber_limit"
    ] = float(

        quality_config
        .qber_limit
    )


    stack[
        "simulator"
    ].validate_runtime_consistency()


    return stack


# ======================================================================
# QKD collection checkpoint predicate
# ======================================================================

def _narad_should_check_qkd_block(
    pair_number,
    config
):

    pair_number = int(
        pair_number
    )


    max_pairs = int(

        config
        .max_delivered_pairs
    )


    check_interval = int(

        config
        .collection_check_interval_pairs
    )


    return bool(

        (
            pair_number
            %
            check_interval
            ==
            0
        )

        or

        (
            pair_number
            ==
            max_pairs
        )
    )


# ======================================================================
# Fixed QKD collection + actual distillation loop
#
# Only semantic change from working Phase-5 implementation:
#
#       LAST PAIR ALWAYS GETS CHECKED.
# ======================================================================

def _narad_run_qkd_until_actual_key(
    stack,
    qkd_config,
    protocol_seed,
    postprocess_master_seed,
    verbose
):

    qkd_channel = (

        NARADQKDEntanglementChannel(

            stack=(
                stack
            ),

            protocol_seed=(
                int(
                    protocol_seed
                )
            ),

            measurement_latency_s=0.0
        )
    )


    measurements = []


    last_postprocess_attempt_at = 0


    final_summary = None


    for pair_number in range(

        1,

        int(
            qkd_config
            .max_delivered_pairs
        )
        +
        1
    ):

        measurement = (

            qkd_channel
            .deliver_and_measure(

                alice_basis=None,

                bob_basis=None,

                z_probability=(

                    qkd_config
                    .z_probability
                ),

                max_slots_per_pair=(

                    qkd_config
                    .max_slots_per_pair
                )
            )
        )


        measurements.append(
            measurement
        )


        # ==========================================================
        # FIX:
        #
        # Check:
        #   - ordinary interval boundary
        #   - OR final pair budget boundary
        # ==========================================================

        should_check = (

            _narad_should_check_qkd_block(

                pair_number=(
                    pair_number
                ),

                config=(
                    qkd_config
                )
            )
        )


        if not should_check:

            continue


        summary = (

            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    qkd_config
                )
            )
        )


        final_summary = summary


        if verbose:

            x_qber_text = (

                f"{summary.observed_x_qber:.3%}"

                if (

                    summary
                    .xx_parameter_estimation_bits

                    >
                    0
                )

                else "N/A"
            )


            print(

                f"[QKD {summary.delivered_pairs:>4}] "
                f"ZZ={summary.zz_key_basis_bits:>4} | "
                f"XX={summary.xx_parameter_estimation_bits:>3} | "
                f"QX={x_qber_text} | "
                f"Qphase="
                f"{summary.phase_error_estimate:.3%} | "
                f"forecast="
                f"{summary.forecast_extractable_bits}"
            )


        # ==========================================================
        # Forecast readiness only permits REAL post-processing.
        # ==========================================================

        if not summary.collection_ready:

            continue


        if (

            pair_number

            <=

            last_postprocess_attempt_at
        ):

            continue


        last_postprocess_attempt_at = (
            pair_number
        )


        postprocess_rng = (

            np.random.default_rng(

                np.random.SeedSequence(
                    [

                        int(
                            postprocess_master_seed
                        ),

                        int(
                            pair_number
                        ),

                        int(
                            qkd_config
                            .target_secret_bits
                        )
                    ]
                )
            )
        )


        try:

            final_result = (

                phase3k_distill_configurable_bbm92_key(

                    measurements=(
                        tuple(
                            measurements
                        )
                    ),

                    collection_summary=(
                        summary
                    ),

                    config=(
                        qkd_config
                    ),

                    postprocess_rng=(
                        postprocess_rng
                    )
                )
            )


        except RuntimeError as error:

            message = str(
                error
            )


            retryable_prefixes = (

                "INSUFFICIENT_PARAMETER_ESTIMATION",

                "CASCADE_RECONCILIATION_FAILED",

                "INSUFFICIENT_POST_EC_ENTROPY"
            )


            if not any(

                message.startswith(
                    prefix
                )

                for prefix
                in retryable_prefixes
            ):

                raise


            if verbose:

                print(

                    "         "
                    "[POSTPROCESS RETRY REQUIRED] "
                    f"{message}"
                )


            continue


        # ==========================================================
        # ACTUAL SECRET KEY EXISTS
        # ==========================================================

        if (

            final_result.success

            and

            final_result.alice_key_bytes

            ==

            final_result.bob_key_bytes
        ):

            return (

                final_result,

                tuple(
                    measurements
                ),

                summary
            )


    # ==================================================================
    # Exhausted pair budget
    #
    # final_summary is now guaranteed to correspond to the full block
    # because the final pair ALWAYS triggers a summary.
    # ==================================================================

    if final_summary is None:

        final_summary = (

            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    qkd_config
                )
            )
        )


    raise NARADQCServiceError(

        "QKD pair budget exhausted before actual "
        "post-processing produced the requested secret key. "
        f"Delivered={len(measurements)}, "
        f"forecast_extractable="
        f"{final_summary.forecast_extractable_bits}."
    )


# ======================================================================
# Thin wrapper around ALREADY-WORKING public API
#
# This is intentionally tiny.
#
# The validated Phase-5 orchestration remains untouched.
# We simply make its helpers observe this request's QKD configuration.
# ======================================================================

def narad_qc_establish_key(
    network,
    sender_name,
    receiver_name,
    *,
    qkd_config=PHASE3J_BIASED_BBM92_CONFIG,
    max_hops=None,
    physical_seed=NARAD_API_DEFAULT_PHYSICAL_SEED,
    protocol_seed=NARAD_API_DEFAULT_PROTOCOL_SEED,
    postprocess_seed=NARAD_API_DEFAULT_POSTPROCESS_SEED,
    verbose=True,
    return_details=False
):

    # ==================================================================
    # One request = one consistent QBER/security configuration.
    # ==================================================================

    request_token = (

        _NARAD_REQUEST_QKD_CONFIG.set(
            qkd_config
        )
    )


    try:

        return (

            _NARAD_PRE_6_1A_ESTABLISH_KEY(

                network=(
                    network
                ),

                sender_name=(
                    sender_name
                ),

                receiver_name=(
                    receiver_name
                ),

                qkd_config=(
                    qkd_config
                ),

                max_hops=(
                    max_hops
                ),

                physical_seed=(
                    physical_seed
                ),

                protocol_seed=(
                    protocol_seed
                ),

                postprocess_seed=(
                    postprocess_seed
                ),

                verbose=(
                    verbose
                ),

                return_details=(
                    return_details
                )
            )
        )


    finally:

        _NARAD_REQUEST_QKD_CONFIG.reset(
            request_token
        )


print(
    "=" * 100
)

print(
    "NARAD-QC PHASE 6.1A — CORE SEMANTIC HARDENING"
)

print(
    "=" * 100
)

print(
    "\n[PASS] Request QBER threshold now propagates into Policy C."
)

print(
    "[PASS] Request QBER threshold now propagates into Phase-3 delivery."
)

print(
    "[PASS] Request QBER threshold now propagates into recovery planning."
)

print(
    "[PASS] BBM92 and network delivery now share one quality boundary."
)

print(
    "[PASS] Final partial QKD block now receives an actual distillation attempt."
)

print(
    "[PASS] Historical frozen global configurations remain unchanged."
)

In [ ]:
# @title  Hardening Regression Tests

from dataclasses import replace
import math


print(
    "=" * 100
)

print(
    "REGRESSION TESTS"
)

print(
    "=" * 100
)


# ======================================================================
# TEST 1 — frozen globals must NOT have been mutated
# ======================================================================

assert math.isclose(

    PHASE3H1_QUALITY_CONFIG
    .qber_limit,

    0.11,

    rel_tol=0.0,

    abs_tol=1e-15
)


assert math.isclose(

    PHASE4_METRIC_QUALITY_CONFIG
    .qber_limit,

    0.11,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Frozen historical QBER globals remain 0.11."
)


# ======================================================================
# TEST 2 — request-local QBER quality configuration
# ======================================================================

test_qkd_08 = replace(

    PHASE3J_BIASED_BBM92_CONFIG,

    qber_abort_threshold=0.08
)


token = (

    _NARAD_REQUEST_QKD_CONFIG.set(
        test_qkd_08
    )
)


try:

    test_quality_config = (

        _narad_current_quality_config()
    )


    assert math.isclose(

        test_quality_config
        .qber_limit,

        0.08,

        rel_tol=0.0,

        abs_tol=1e-15
    )


    assert math.isclose(

        test_quality_config
        .minimum_end_to_end_visibility,

        0.84,

        rel_tol=0.0,

        abs_tol=1e-15
    )


finally:

    _NARAD_REQUEST_QKD_CONFIG.reset(
        token
    )


print(
    "[PASS] Per-request QBER threshold correctly creates "
    "a request-local network quality budget."
)


# ======================================================================
# TEST 3 — ContextVar resets cleanly
# ======================================================================

assert math.isclose(

    _narad_current_quality_config()
    .qber_limit,

    PHASE3J_BIASED_BBM92_CONFIG
    .qber_abort_threshold,

    rel_tol=0.0,

    abs_tol=1e-15
)


print(
    "[PASS] Request quality state does not leak after request completion."
)


# ======================================================================
# TEST 4 — ordinary collection interval
# ======================================================================

test_config = replace(

    PHASE3J_BIASED_BBM92_CONFIG,

    max_delivered_pairs=100,

    collection_check_interval_pairs=16
)


assert (

    _narad_should_check_qkd_block(
        16,
        test_config
    )

    is True
)


assert (

    _narad_should_check_qkd_block(
        96,
        test_config
    )

    is True
)


assert (

    _narad_should_check_qkd_block(
        97,
        test_config
    )

    is False
)


# THE BUG FIX:
assert (

    _narad_should_check_qkd_block(
        100,
        test_config
    )

    is True
)


print(
    "[PASS] Final non-multiple block boundary is checked."
)


# ======================================================================
# TEST 5 — interval larger than whole pair budget
# ======================================================================

test_config_large_interval = replace(

    PHASE3J_BIASED_BBM92_CONFIG,

    max_delivered_pairs=10,

    collection_check_interval_pairs=16
)


for pair_number in range(
    1,
    10
):

    assert (

        _narad_should_check_qkd_block(

            pair_number,

            test_config_large_interval
        )

        is False
    )


assert (

    _narad_should_check_qkd_block(
        10,
        test_config_large_interval
    )

    is True
)


print(
    "[PASS] Pair budget smaller than collection interval still "
    "gets a final post-processing opportunity."
)


# ======================================================================
# TEST 6 — Policy-C requested-QBER sensitivity
#
# No physics execution:
# just route prediction/ranking.
# ======================================================================

baseline_token = (

    _NARAD_REQUEST_QKD_CONFIG.set(

        replace(

            PHASE3J_BIASED_BBM92_CONFIG,

            qber_abort_threshold=0.11
        )
    )
)


try:

    (
        baseline_routes,
        _,
        _,
        _
    ) = (

        _narad_rank_routes_policy_c(

            network=(
                phase4_network
            ),

            sender_name="Alice",

            receiver_name="Bob"
        )
    )


finally:

    _NARAD_REQUEST_QKD_CONFIG.reset(
        baseline_token
    )


strict_token = (

    _NARAD_REQUEST_QKD_CONFIG.set(

        replace(

            PHASE3J_BIASED_BBM92_CONFIG,

            qber_abort_threshold=0.08
        )
    )
)


try:

    (
        strict_routes,
        _,
        _,
        _
    ) = (

        _narad_rank_routes_policy_c(

            network=(
                phase4_network
            ),

            sender_name="Alice",

            receiver_name="Bob"
        )
    )


finally:

    _NARAD_REQUEST_QKD_CONFIG.reset(
        strict_token
    )


assert all(

    route[
        "predicted_delivered_qber"
    ]

    <=

    0.11
    +
    1e-12

    for route
    in baseline_routes
)


assert all(

    route[
        "predicted_delivered_qber"
    ]

    <=

    0.08
    +
    1e-12

    for route
    in strict_routes
)


print(
    "[PASS] Policy-C route admission observes request-level QBER."
)


print(
    "\nBaseline QBER=11% executable routes:",
    len(
        baseline_routes
    )
)


print(
    "Strict QBER=8% executable routes:",
    len(
        strict_routes
    )
)


print(
    "\nBaseline winner:",
    baseline_routes[0][
        "candidate"
    ].display_path
)


print(
    "Strict winner:",
    strict_routes[0][
        "candidate"
    ].display_path
)


print(
    "\n" + "=" * 100
)

print(
    "REGRESSION SUITE PASSED"
)

print(
    "=" * 100
)

In [ ]:
# @title   Runtime and Session Hardening

import itertools
import time
import uuid

from fastapi import HTTPException


print("=" * 100)
print("NARAD-QC PHASE 6.1B — RUNTIME AND SESSION HARDENING")
print("=" * 100)


# ======================================================================
# SESSION LIFETIME
#
# Research/demo default:
#
#     30 minutes absolute lifetime
#
# This is intentionally configurable through one constant.
#
# IMPORTANT:
# Deleting a Python bytes object from the session dictionary is NOT
# claimed to be cryptographic RAM zeroization.
# ======================================================================

NARAD_API_SESSION_TTL_S = 30.0 * 60.0


if (
    not math.isfinite(
        NARAD_API_SESSION_TTL_S
    )

    or

    NARAD_API_SESSION_TTL_S <= 0.0
):

    raise RuntimeError(
        "NARAD_API_SESSION_TTL_S must be finite and positive."
    )


# ======================================================================
# SESSION CLEANUP
# ======================================================================

def _narad_api_session_expired(
    session,
    *,
    now=None
):

    if now is None:

        now = time.time()


    created_at = float(

        session[
            "created_at_unix"
        ]
    )


    age_s = max(

        0.0,

        float(now)
        -
        created_at
    )


    return bool(

        age_s

        >=

        NARAD_API_SESSION_TTL_S
    )


def _narad_api_prune_expired_sessions(
    *,
    now=None
):

    if now is None:

        now = time.time()


    expired_ids = []


    with NARAD_API_SESSION_LOCK:

        for (
            session_id,
            session
        ) in list(

            NARAD_API_SESSIONS.items()
        ):

            if _narad_api_session_expired(

                session,

                now=now
            ):

                expired_ids.append(
                    session_id
                )


        for session_id in expired_ids:

            NARAD_API_SESSIONS.pop(

                session_id,

                None
            )


    return tuple(
        expired_ids
    )


# ======================================================================
# REPLACE SESSION LOOKUP HELPER
#
# Existing endpoints call this helper dynamically, so replacing the
# helper hardens:
#
#     GET    /session/{id}
#     POST   /message/send
#     POST   /message/decrypt
#
# without duplicating those route implementations.
# ======================================================================

def narad_api_get_session(
    session_id
):

    session_id = str(
        session_id
    ).strip()


    if not session_id:

        raise HTTPException(

            status_code=404,

            detail="Unknown or expired NARAD-QC session."
        )


    now = time.time()


    with NARAD_API_SESSION_LOCK:

        session = (

            NARAD_API_SESSIONS.get(
                session_id
            )
        )


        if session is None:

            raise HTTPException(

                status_code=404,

                detail="Unknown or expired NARAD-QC session."
            )


        if _narad_api_session_expired(

            session,

            now=now
        ):

            NARAD_API_SESSIONS.pop(

                session_id,

                None
            )


            raise HTTPException(

                status_code=404,

                detail="Unknown or expired NARAD-QC session."
            )


        # --------------------------------------------------------------
        # Diagnostic only.
        #
        # Lifetime remains ABSOLUTE from created_at_unix.
        # Access does not extend the session.
        # --------------------------------------------------------------

        session[
            "last_accessed_at_unix"
        ] = now


        return session


# ======================================================================
# READINESS
#
# /health remains the cheap liveness endpoint.
#
# /ready performs:
#
#   1. required global dependency checks
#   2. callable checks
#   3. network structural checks
#   4. fitted Policy-C model checks
#   5. existence of at least one currently executable USER -> USER route
#
# It DOES NOT execute physical QKD.
# ======================================================================

_NARAD_READINESS_REQUIRED_OBJECTS = (

    "phase4_network",

    "PHASE3J_BIASED_BBM92_CONFIG",

    "PHASE3H1_QUALITY_CONFIG",

    "PHASE4_FINAL_RELIABILITY_MODEL",

    "PHASE4_FINAL_RELIABILITY_CALIBRATOR",

    "PHASE4_FINAL_LATENCY_MODEL",

    "PHASE4_FINAL_VISIBILITY_MODEL"
)


_NARAD_READINESS_REQUIRED_CALLABLES = (

    "narad_qc_establish_key",

    "_narad_rank_routes_policy_c",

    "phase4_discover_valid_quantum_paths",

    "phase4_evaluate_candidate_path",

    "phase3k_distill_configurable_bbm92_key",

    "aes_gcm_encrypt",

    "aes_gcm_decrypt"
)


def _narad_api_role_value(
    role
):

    value = getattr(

        role,

        "value",

        role
    )


    return str(
        value
    ).upper()


def _narad_api_active_user_names():

    user_names = []


    for node in (

        phase4_network
        .topology
        .nodes
    ):

        profile = (

            phase4_network
            .node_profile(
                node.name
            )
        )


        if (

            bool(
                profile.active
            )

            and

            _narad_api_role_value(
                profile.role
            )
            ==
            "USER"
        ):

            user_names.append(
                node.name
            )


    return tuple(
        sorted(
            user_names
        )
    )


def narad_api_runtime_readiness():

    missing_objects = [

        name

        for name
        in _NARAD_READINESS_REQUIRED_OBJECTS

        if name not in globals()
    ]


    missing_callables = [

        name

        for name
        in _NARAD_READINESS_REQUIRED_CALLABLES

        if not callable(
            globals().get(
                name
            )
        )
    ]


    if (
        missing_objects

        or

        missing_callables
    ):

        return {

            "ready":
                False,

            "reason":
                "MISSING_RUNTIME_DEPENDENCIES",

            "missing_objects":
                missing_objects,

            "missing_callables":
                missing_callables
        }


    try:

        topology = (
            phase4_network
            .topology
        )


        node_count = int(
            topology.node_count
        )


        link_count = int(
            topology.link_count
        )


        if (

            node_count < 2

            or

            link_count < 1
        ):

            return {

                "ready":
                    False,

                "reason":
                    "NETWORK_NOT_INITIALIZED",

                "node_count":
                    node_count,

                "link_count":
                    link_count
            }


        user_names = (
            _narad_api_active_user_names()
        )


        if len(
            user_names
        ) < 2:

            return {

                "ready":
                    False,

                "reason":
                    "INSUFFICIENT_ACTIVE_USER_ENDPOINTS",

                "active_users":
                    list(
                        user_names
                    )
            }


        # ==============================================================
        # Find ONE currently scoreable/executable USER-to-USER route.
        #
        # This exercises route discovery + Policy C prediction only.
        # It does NOT run authentication or physical QKD.
        # ==============================================================

        route_probe = None


        for (
            sender_name,
            receiver_name
        ) in itertools.combinations(

            user_names,

            2
        ):

            try:

                (
                    ranked,
                    _,
                    _,
                    _
                ) = (

                    _narad_rank_routes_policy_c(

                        network=(
                            phase4_network
                        ),

                        sender_name=(
                            sender_name
                        ),

                        receiver_name=(
                            receiver_name
                        )
                    )
                )


                if ranked:

                    winner = ranked[0]


                    route_probe = {

                        "sender":
                            sender_name,

                        "receiver":
                            receiver_name,

                        "candidate_count":
                            len(
                                ranked
                            ),

                        "winner":
                            (
                                winner[
                                    "candidate"
                                ]
                                .display_path
                            )
                    }


                    break


            except Exception:

                # Another endpoint pair may still be executable.
                continue


        if route_probe is None:

            return {

                "ready":
                    False,

                "reason":
                    "NO_EXECUTABLE_POLICY_C_ROUTE",

                "active_users":
                    list(
                        user_names
                    )
            }


        expired_removed = (

            _narad_api_prune_expired_sessions()
        )


        with NARAD_API_SESSION_LOCK:

            active_sessions = len(
                NARAD_API_SESSIONS
            )


        return {

            "ready":
                True,

            "service":
                "NARAD-QC",

            "version":
                NARAD_API_VERSION,

            "node_count":
                node_count,

            "link_count":
                link_count,

            "active_users":
                list(
                    user_names
                ),

            "route_probe":
                route_probe,

            "active_sessions":
                active_sessions,

            "expired_sessions_pruned":
                len(
                    expired_removed
                ),

            "session_ttl_s":
                NARAD_API_SESSION_TTL_S
        }


    except Exception as error:

        return {

            "ready":
                False,

            "reason":
                "READINESS_CHECK_FAILED",

            "error_type":
                type(error).__name__,

            "error":
                str(
                    error
                )
        }


# ======================================================================
# RUNTIME GENERATION
#
# Captured by /ready so server startup can distinguish the CURRENT app
# object from a stale app still running on port 8000.
# ======================================================================

NARAD_API_RUNTIME_GENERATION = (
    uuid.uuid4().hex
)


# ======================================================================
# ADD /ready ONCE PER APP INSTANCE
# ======================================================================

_ready_route_exists = any(

    getattr(
        route,
        "path",
        None
    )
    ==
    "/ready"

    and

    "GET"
    in
    (
        getattr(
            route,
            "methods",
            set()
        )
        or
        set()
    )

    for route
    in app.routes
)


if not _ready_route_exists:

    @app.get("/ready")
    def narad_api_ready(
        _runtime_generation=(
            NARAD_API_RUNTIME_GENERATION
        )
    ):

        report = (
            narad_api_runtime_readiness()
        )


        report[
            "runtime_generation"
        ] = _runtime_generation


        if not report[
            "ready"
        ]:

            raise HTTPException(

                status_code=503,

                detail=report
            )


        return report


print("\n[PASS] Real session TTL is active.")
print("[PASS] Expired sessions are removed on access/pruning.")
print("[PASS] /health remains liveness-only.")
print("[PASS] /ready now performs runtime readiness validation.")
print("[PASS] Readiness includes a Policy-C route probe without physical QKD.")

In [ ]:
# @title Runtime Hardening Regression Tests

import time

from fastapi import HTTPException


print("=" * 100)
print("REGRESSION TESTS")
print("=" * 100)


# ======================================================================
# TEST 1 — fresh session remains accessible
# ======================================================================

_test_live_session_id = (
    "__narad_test_live_session__"
)


with NARAD_API_SESSION_LOCK:

    NARAD_API_SESSIONS[
        _test_live_session_id
    ] = {

        "session_id":
            _test_live_session_id,

        "created_at_unix":
            time.time(),

        "sender":
            "TEST_A",

        "receiver":
            "TEST_B",

        "key":
            b"\x00" * 16,

        "qkd_config":
            None,

        "result":
            None,

        "message_count":
            0
    }


try:

    _test_live_session = (

        narad_api_get_session(
            _test_live_session_id
        )
    )


    assert (

        _test_live_session[
            "session_id"
        ]

        ==

        _test_live_session_id
    )


    assert (

        "last_accessed_at_unix"

        in

        _test_live_session
    )


finally:

    with NARAD_API_SESSION_LOCK:

        NARAD_API_SESSIONS.pop(

            _test_live_session_id,

            None
        )


print(
    "[PASS] Fresh session remains accessible."
)


# ======================================================================
# TEST 2 — expired session is rejected AND deleted
# ======================================================================

_test_expired_session_id = (
    "__narad_test_expired_session__"
)


with NARAD_API_SESSION_LOCK:

    NARAD_API_SESSIONS[
        _test_expired_session_id
    ] = {

        "session_id":
            _test_expired_session_id,

        "created_at_unix":
            (
                time.time()

                -

                NARAD_API_SESSION_TTL_S

                -

                1.0
            ),

        "sender":
            "TEST_A",

        "receiver":
            "TEST_B",

        "key":
            b"\x00" * 16,

        "qkd_config":
            None,

        "result":
            None,

        "message_count":
            0
    }


_expiry_exception_seen = False


try:

    narad_api_get_session(
        _test_expired_session_id
    )


except HTTPException as error:

    _expiry_exception_seen = True


    assert (
        error.status_code
        ==
        404
    )


assert (
    _expiry_exception_seen
    is True
)


with NARAD_API_SESSION_LOCK:

    assert (

        _test_expired_session_id

        not in

        NARAD_API_SESSIONS
    )


print(
    "[PASS] Expired session is rejected and removed."
)


# ======================================================================
# TEST 3 — bulk pruning
# ======================================================================

_test_bulk_expired_id = (
    "__narad_test_bulk_expired__"
)


with NARAD_API_SESSION_LOCK:

    NARAD_API_SESSIONS[
        _test_bulk_expired_id
    ] = {

        "session_id":
            _test_bulk_expired_id,

        "created_at_unix":
            (
                time.time()

                -

                NARAD_API_SESSION_TTL_S

                -

                10.0
            ),

        "sender":
            "TEST_A",

        "receiver":
            "TEST_B",

        "key":
            b"\x00" * 16,

        "qkd_config":
            None,

        "result":
            None,

        "message_count":
            0
    }


_pruned = (

    _narad_api_prune_expired_sessions()
)


assert (

    _test_bulk_expired_id

    in

    _pruned
)


with NARAD_API_SESSION_LOCK:

    assert (

        _test_bulk_expired_id

        not in

        NARAD_API_SESSIONS
    )


print(
    "[PASS] Bulk expiry pruning works."
)


# ======================================================================
# TEST 4 — readiness actually examines runtime
# ======================================================================

_readiness = (

    narad_api_runtime_readiness()
)


assert (
    isinstance(
        _readiness,
        dict
    )
)


assert (
    "ready"
    in
    _readiness
)


if not _readiness[
    "ready"
]:

    raise AssertionError(

        "NARAD-QC runtime is not ready: "
        f"{_readiness}"
    )


assert (

    _readiness[
        "node_count"
    ]

    >=
    2
)


assert (

    _readiness[
        "link_count"
    ]

    >=
    1
)


assert (

    _readiness[
        "route_probe"
    ]

    is not None
)


print(
    "[PASS] Runtime readiness validates dependencies, "
    "topology and an executable Policy-C route."
)


# ======================================================================
# TEST 5 — /ready exists exactly once
# ======================================================================

_ready_routes = [

    route

    for route
    in app.routes

    if (

        getattr(
            route,
            "path",
            None
        )

        ==

        "/ready"
    )
]


assert (
    len(
        _ready_routes
    )
    ==
    1
)


print(
    "[PASS] /ready is registered exactly once."
)


print("\n" + "=" * 100)
print("REGRESSION SUITE PASSED")
print("=" * 100)

In [ ]:
# @title  API Contract Hardening

from dataclasses import asdict
from fastapi import HTTPException


print("=" * 100)
print("NARAD-QC PHASE 6.1C — API CONTRACT HARDENING")
print("=" * 100)


# ======================================================================
# 1. CANONICAL ENDPOINT NAMES
#
# Phase 5 already resolves/normalizes node names.
#
# Phase 6 previously stored raw request strings, so:
#
#       " Alice "
#
# could successfully establish a session but later be stored literally
# as " Alice ", producing avoidable endpoint-binding mismatches.
#
# Every API boundary now resolves through the authoritative topology.
# ======================================================================

def _narad_api_canonical_node_name(
    name
):

    if not isinstance(
        name,
        str
    ):

        raise ValueError(
            "Node name must be a string."
        )


    normalized = (
        name.strip()
    )


    if not normalized:

        raise ValueError(
            "Node name cannot be empty."
        )


    node = (

        phase4_network
        .topology
        .get_node(
            normalized
        )
    )


    return node.name


def _narad_api_canonical_endpoints(
    sender,
    receiver
):

    sender = (
        _narad_api_canonical_node_name(
            sender
        )
    )


    receiver = (
        _narad_api_canonical_node_name(
            receiver
        )
    )


    if sender == receiver:

        raise ValueError(
            "Sender and receiver must be distinct nodes."
        )


    return (
        sender,
        receiver
    )


# ======================================================================
# 2. FASTAPI ROUTE REPLACEMENT HELPER
#
# Important:
#
# Merely redefining a Python function after @app.get/@app.post does NOT
# reliably replace the callable already captured by FastAPI.
#
# Therefore affected routes are deliberately removed and re-registered.
#
# This helper also makes the hardening cell safe to rerun.
# ======================================================================

def _narad_api_remove_route(
    path,
    method
):

    method = (
        str(method)
        .upper()
    )


    retained = []


    for route in app.router.routes:

        route_path = getattr(
            route,
            "path",
            None
        )


        route_methods = (

            getattr(
                route,
                "methods",
                set()
            )

            or

            set()
        )


        if (

            route_path == path

            and

            method in route_methods
        ):

            continue


        retained.append(
            route
        )


    app.router.routes[:] = (
        retained
    )


# ======================================================================
# 3. COMPLETE NETWORK LINK SERIALIZATION
#
# Codex correctly observed that /nodes + /network link_count cannot
# reconstruct the actual physical graph.
#
# /links now exposes every authoritative physical fibre, including
# inactive ones and their routing-visible resource state.
# ======================================================================

def _narad_api_link_rows():

    rows = []


    for link in sorted(

        phase4_network
        .topology
        .links,

        key=lambda item: item.link_id
    ):

        profile = (

            phase4_network
            .link_profile(

                link.node_a.name,

                link.node_b.name
            )
        )


        rows.append({

            "link_id":
                link.link_id,

            "node_a":
                link.node_a.name,

            "node_b":
                link.node_b.name,

            "distance_km":
                float(
                    link.distance_km
                ),

            "attenuation_db_per_km":
                float(
                    link.attenuation_db_per_km
                ),

            "total_attenuation_db":
                float(
                    link.total_attenuation_db
                ),

            "survival_probability":
                float(
                    link.survival_probability
                ),

            "loss_probability":
                float(
                    link.loss_probability
                ),

            "active":
                bool(
                    profile.active
                ),

            "multiplexing_modes_total":
                int(
                    profile
                    .multiplexing_modes_total
                ),

            "multiplexing_modes_available":
                int(
                    profile
                    .multiplexing_modes_available
                )
        })


    return rows


_narad_api_remove_route(
    "/links",
    "GET"
)


@app.get("/links")
def links():

    rows = (
        _narad_api_link_rows()
    )


    return {

        "link_count":
            len(
                rows
            ),

        "links":
            rows
    }


# ======================================================================
# 4. COMPLETE ROUTE AUDIT
#
# Existing preview returned only:
#
#     executable Policy-C routes
#     +
#     count of paths rejected because of internal USER nodes
#
# It did NOT expose:
#
#     which physical paths were USER-transit rejected
#     hardware/resource rejection reasons
#     fresh-QBER rejection
#     empirical delivered-QBER rejection
#     calibration/extrapolation diagnostic
#
# This helper evaluates ALL role-valid candidates and preserves why
# each candidate survived or failed.
# ======================================================================

def _narad_api_route_audit(
    sender,
    receiver,
    max_hops=None
):

    (
        sender,
        receiver
    ) = (

        _narad_api_canonical_endpoints(

            sender,

            receiver
        )
    )


    (
        valid_paths,
        rejected_paths,
        physical_paths
    ) = (

        phase4_discover_valid_quantum_paths(

            network=(
                phase4_network
            ),

            sender_name=(
                sender
            ),

            receiver_name=(
                receiver
            ),

            max_hops=(
                max_hops
            )
        )
    )


    quality_config = (
        _narad_current_quality_config()
    )


    qber_limit = float(

        quality_config
        .qber_limit
    )


    tolerance = float(

        quality_config
        .numerical_tolerance
    )


    # ------------------------------------------------------------------
    # Physical paths rejected because another USER occurs internally.
    # ------------------------------------------------------------------

    role_rejections = []


    for rejected in rejected_paths:

        role_rejections.append({

            "nodes":
                list(
                    rejected.node_names
                ),

            "path":
                " -> ".join(
                    rejected.node_names
                ),

            "reason":
                rejected.reason,

            "offending_user":
                rejected.offending_user
        })


    candidate_audit = []

    executable_records = []


    # ------------------------------------------------------------------
    # Every role-valid candidate.
    # ------------------------------------------------------------------

    for candidate in valid_paths:

        metrics = (

            phase4_evaluate_candidate_path(

                network=(
                    phase4_network
                ),

                candidate_path=(
                    candidate
                )
            )
        )


        rejection_reasons = []


        # ==============================================================
        # HARDWARE / CONTROL RESOURCE FEASIBILITY
        # ==============================================================

        if not bool(
            metrics.all_nodes_active
        ):

            rejection_reasons.append(
                "INACTIVE_NODE"
            )


        if not bool(
            metrics.all_links_active
        ):

            rejection_reasons.append(
                "INACTIVE_LINK"
            )


        if not bool(
            metrics.multiplexing_feasible
        ):

            rejection_reasons.append(
                "INSUFFICIENT_MULTIPLEXING"
            )


        if not bool(
            metrics.memory_feasible
        ):

            rejection_reasons.append(
                "INSUFFICIENT_MEMORY"
            )


        if not bool(
            metrics.bsm_feasible
        ):

            rejection_reasons.append(
                "INSUFFICIENT_BSM_RESOURCES"
            )


        resource_feasible = (

            len(
                rejection_reasons
            )
            ==
            0
        )


        # ==============================================================
        # REQUEST-SCOPED FRESH QUALITY GATE
        # ==============================================================

        fresh_qkd_feasible = bool(

            float(
                metrics.predicted_qber
            )

            <=

            (
                qber_limit
                +
                tolerance
            )
        )


        if (

            resource_feasible

            and

            not fresh_qkd_feasible
        ):

            rejection_reasons.append(
                "FRESH_QBER_EXCEEDS_REQUEST_LIMIT"
            )


        policy_c = None


        # ==============================================================
        # EMPIRICAL POLICY-C QUALITY
        # ==============================================================

        if (

            resource_feasible

            and

            fresh_qkd_feasible
        ):

            policy_c = (

                _narad_policy_c_evaluate_candidate(

                    network=(
                        phase4_network
                    ),

                    candidate_path=(
                        candidate
                    ),

                    raw_metrics=(
                        metrics
                    )
                )
            )


            if not policy_c[
                "empirical_qkd_feasible"
            ]:

                rejection_reasons.append(
                    "DELIVERED_QBER_EXCEEDS_REQUEST_LIMIT"
                )


        executable = bool(

            resource_feasible

            and

            fresh_qkd_feasible

            and

            policy_c is not None

            and

            policy_c[
                "empirical_qkd_feasible"
            ]
        )


        if executable:

            executable_records.append(
                policy_c
            )


        audit_row = {

            "path_index":
                int(
                    candidate.path_index
                ),

            "path":
                candidate.display_path,

            "nodes":
                list(
                    candidate.node_names
                ),

            "repeaters":
                list(
                    candidate.repeater_names
                ),

            "hop_count":
                int(
                    candidate.hop_count
                ),

            "repeater_count":
                int(
                    candidate.repeater_count
                ),

            "total_distance_km":
                float(
                    metrics.total_distance_km
                ),

            "fresh_predicted_visibility":
                float(
                    metrics.predicted_werner_visibility
                ),

            "fresh_predicted_qber":
                float(
                    metrics.predicted_qber
                ),

            "requested_qber_limit":
                qber_limit,

            "resource_feasible":
                resource_feasible,

            "fresh_qkd_feasible":
                fresh_qkd_feasible,

            "executable":
                executable,

            "rejection_reasons":
                list(
                    rejection_reasons
                )
        }


        if policy_c is not None:

            audit_row.update({

                "policy_c_score":
                    float(
                        policy_c[
                            "score"
                        ]
                    ),

                "predicted_service_probability":
                    float(
                        policy_c[
                            "predicted_service_probability"
                        ]
                    ),

                "predicted_median_delivery_s":
                    float(
                        policy_c[
                            "predicted_median_delivery_s"
                        ]
                    ),

                "predicted_delivered_visibility":
                    float(
                        policy_c[
                            "predicted_delivered_visibility"
                        ]
                    ),

                "predicted_delivered_qber":
                    float(
                        policy_c[
                            "predicted_delivered_qber"
                        ]
                    ),

                "within_calibration_domain":
                    bool(
                        policy_c[
                            "within_calibration_domain"
                        ]
                    ),

                "extrapolation_fields":
                    list(
                        policy_c[
                            "extrapolation_fields"
                        ]
                    )
            })


        else:

            audit_row.update({

                "policy_c_score":
                    None,

                "predicted_service_probability":
                    None,

                "predicted_median_delivery_s":
                    None,

                "predicted_delivered_visibility":
                    None,

                "predicted_delivered_qber":
                    None,

                "within_calibration_domain":
                    None,

                "extrapolation_fields":
                    []
            })


        candidate_audit.append(
            audit_row
        )


    # ==================================================================
    # Rank executable candidates exactly as Policy C does.
    # ==================================================================

    executable_records.sort(

        key=lambda record: (

            -record[
                "score"
            ],

            -record[
                "predicted_service_probability"
            ],

            record[
                "predicted_median_delivery_s"
            ],

            -record[
                "predicted_delivered_visibility"
            ],

            record[
                "candidate"
            ]
            .path_index
        )
    )


    rank_by_path_index = {

        record[
            "candidate"
        ].path_index:
            rank

        for rank, record
        in enumerate(
            executable_records,
            start=1
        )
    }


    audit_by_index = {

        row[
            "path_index"
        ]:
            row

        for row
        in candidate_audit
    }


    ranked_routes = []


    for rank, record in enumerate(

        executable_records,

        start=1
    ):

        candidate = (
            record[
                "candidate"
            ]
        )


        ranked_routes.append({

            "rank":
                rank,

            "path_index":
                candidate.path_index,

            "path":
                candidate.display_path,

            "nodes":
                list(
                    candidate.node_names
                ),

            "repeaters":
                list(
                    candidate.repeater_names
                ),

            "score":
                float(
                    record[
                        "score"
                    ]
                ),

            "service_probability":
                float(
                    record[
                        "predicted_service_probability"
                    ]
                ),

            "predicted_delivery_s":
                float(
                    record[
                        "predicted_median_delivery_s"
                    ]
                ),

            "predicted_visibility":
                float(
                    record[
                        "predicted_delivered_visibility"
                    ]
                ),

            "predicted_qber":
                float(
                    record[
                        "predicted_delivered_qber"
                    ]
                ),

            "requested_qber_limit":
                float(
                    record[
                        "requested_qber_limit"
                    ]
                ),

            "bottleneck_elementary_distance_km":
                float(
                    record[
                        "bottleneck_elementary_distance_km"
                    ]
                ),

            "bottleneck_mux_success":
                float(
                    record[
                        "bottleneck_multiplexed_success_probability"
                    ]
                ),

            "within_calibration_domain":
                bool(
                    record[
                        "within_calibration_domain"
                    ]
                ),

            "extrapolation_fields":
                list(
                    record[
                        "extrapolation_fields"
                    ]
                )
        })


    for row in candidate_audit:

        row[
            "rank"
        ] = (

            rank_by_path_index.get(

                row[
                    "path_index"
                ]
            )
        )


    return {

        "sender":
            sender,

        "receiver":
            receiver,

        "requested_qber_limit":
            qber_limit,

        "physical_paths":
            len(
                physical_paths
            ),

        "role_valid_paths":
            len(
                valid_paths
            ),

        "rejected_user_transit_paths":
            len(
                rejected_paths
            ),

        "executable_route_count":
            len(
                executable_records
            ),

        "status":
            (
                "EXECUTABLE_ROUTES_AVAILABLE"

                if executable_records

                else

                "NO_EXECUTABLE_ROUTE"
            ),

        "ranked_routes":
            ranked_routes,

        "role_rejections":
            role_rejections,

        "candidate_audit":
            candidate_audit
    }


# ======================================================================
# 5. REPLACE /routes/preview
# ======================================================================

_narad_api_remove_route(
    "/routes/preview",
    "POST"
)


@app.post("/routes/preview")
def preview_routes(
    request: RoutePreviewRequest
):

    try:

        return (

            _narad_api_route_audit(

                sender=(
                    request.sender
                ),

                receiver=(
                    request.receiver
                ),

                max_hops=(
                    request.max_hops
                )
            )
        )


    except HTTPException:

        raise


    except Exception as error:

        raise HTTPException(

            status_code=400,

            detail=(

                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ======================================================================
# 6. REPLACE /session/establish
#
# Main semantic change:
#
# Session binding now stores result.sender/result.receiver, i.e. the
# canonical endpoints actually used by Phase 5.
# ======================================================================

_narad_api_remove_route(
    "/session/establish",
    "POST"
)


@app.post("/session/establish")
def establish_session(
    request: SessionEstablishRequest
):

    try:

        (
            canonical_sender,
            canonical_receiver
        ) = (

            _narad_api_canonical_endpoints(

                request.sender,

                request.receiver
            )
        )


        qkd_config = (

            narad_api_make_qkd_config(
                request.qkd
            )
        )


        requested_cpu_workers = (
            request.execution.cpu_workers
        )


        requested_parallel_lanes = (
            request.execution.parallel_quantum_lanes
        )


        result = (

            narad_qc_establish_key(

                network=(
                    phase4_network
                ),

                sender_name=(
                    canonical_sender
                ),

                receiver_name=(
                    canonical_receiver
                ),

                qkd_config=(
                    qkd_config
                ),

                max_hops=(
                    request.execution.max_hops
                ),

                physical_seed=(
                    request.execution.physical_seed
                ),

                protocol_seed=(
                    request.execution.protocol_seed
                ),

                postprocess_seed=(
                    request.execution.postprocess_seed
                ),

                verbose=False,

                return_details=True
            )
        )


        # Remove expired sessions before inserting new one.
        _narad_api_prune_expired_sessions()


        session_id = (
            str(
                uuid.uuid4()
            )
        )


        now = time.time()


        with NARAD_API_SESSION_LOCK:

            NARAD_API_SESSIONS[
                session_id
            ] = {

                "session_id":
                    session_id,

                "created_at_unix":
                    now,

                "last_accessed_at_unix":
                    now,

                # ==================================================
                # Store actual canonical execution endpoints.
                # ==================================================

                "sender":
                    result.sender,

                "receiver":
                    result.receiver,

                # KEY NEVER RETURNED
                "key":
                    bytes(
                        result.key
                    ),

                "qkd_config":
                    qkd_config,

                "result":
                    result,

                "message_count":
                    0
            }


        return {

            "session_id":
                session_id,

            "status":
                "ESTABLISHED",

            "sender":
                result.sender,

            "receiver":
                result.receiver,

            "session_ttl_s":
                NARAD_API_SESSION_TTL_S,

            "result":
                narad_api_public_result(
                    result
                ),

            "execution": {

                "requested_cpu_workers":
                    requested_cpu_workers,

                "applied_cpu_workers":
                    1,

                "requested_parallel_quantum_lanes":
                    requested_parallel_lanes,

                "applied_parallel_quantum_lanes":
                    1,

                "note": (
                    "Current heterogeneous routed executor is "
                    "single-lane. Requested Phase-3K parallelism "
                    "is recorded but not silently applied."
                )
            }
        }


    except HTTPException:

        raise


    except Exception as error:

        raise HTTPException(

            status_code=400,

            detail=(

                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ======================================================================
# 7. REPLACE /message/send
#
# Bind using canonical node identity rather than literal raw strings.
# ======================================================================

_narad_api_remove_route(
    "/message/send",
    "POST"
)


@app.post("/message/send")
def send_message(
    request: MessageSendRequest
):

    session = (

        narad_api_get_session(
            request.session_id
        )
    )


    try:

        (
            sender,
            receiver
        ) = (

            _narad_api_canonical_endpoints(

                request.sender,

                request.receiver
            )
        )


    except Exception:

        raise HTTPException(

            status_code=403,

            detail=(
                "Message endpoints do not match "
                "the authenticated NARAD-QC session."
            )
        )


    if (

        sender
        !=
        session[
            "sender"
        ]

        or

        receiver
        !=
        session[
            "receiver"
        ]
    ):

        raise HTTPException(

            status_code=403,

            detail=(
                "Message endpoints do not match "
                "the authenticated NARAD-QC session."
            )
        )


    plaintext = (

        request.message
        .encode(
            "utf-8"
        )
    )


    encrypted_blob = (

        aes_gcm_encrypt(

            plaintext,

            session[
                "key"
            ]
        )
    )


    ciphertext_b64 = (

        base64.b64encode(
            encrypted_blob
        )
        .decode(
            "ascii"
        )
    )


    with NARAD_API_SESSION_LOCK:

        session[
            "message_count"
        ] += 1


        message_index = (
            session[
                "message_count"
            ]
        )


    return {

        "session_id":
            request.session_id,

        "message_index":
            message_index,

        "sender":
            sender,

        "receiver":
            receiver,

        "cipher":
            "AES-GCM",

        "ciphertext_b64":
            ciphertext_b64,

        "plaintext_bytes":
            len(
                plaintext
            ),

        "encrypted_blob_bytes":
            len(
                encrypted_blob
            )
    }


# ======================================================================
# 8. REPLACE DELETE /session/{session_id}
#
# Do NOT claim secure RAM zeroization.
#
# What we can truthfully guarantee:
#
#     session store reference removed
#
# What Python does NOT guarantee:
#
#     immediate/verified memory overwrite of immutable bytes
# ======================================================================

_narad_api_remove_route(
    "/session/{session_id}",
    "DELETE"
)


@app.delete("/session/{session_id}")
def destroy_session(
    session_id: str
):

    session_id = (
        str(
            session_id
        ).strip()
    )


    with NARAD_API_SESSION_LOCK:

        session = (

            NARAD_API_SESSIONS.pop(

                session_id,

                None
            )
        )


    if session is None:

        raise HTTPException(

            status_code=404,

            detail=(
                "Unknown or already-destroyed session."
            )
        )


    return {

        "session_id":
            session_id,

        "status":
            "DESTROYED",

        "session_store_reference_removed":
            True,

        "memory_zeroization_guaranteed":
            False,

        "note": (
            "The server-side session reference was removed. "
            "Python does not guarantee cryptographic zeroization "
            "of previously allocated immutable key bytes."
        )
    }


# ======================================================================
# 9. CONTRACT METADATA
# ======================================================================

NARAD_API_CONTRACT_HARDENING_VERSION = (
    "6.1C"
)


print("\n[PASS] Endpoint names are canonicalized at the API boundary.")
print("[PASS] /links exposes the complete physical fibre graph.")
print("[PASS] Route preview exposes complete rejection diagnostics.")
print("[PASS] Policy-C calibration extrapolation is exposed publicly.")
print("[PASS] Session endpoint binding stores canonical execution names.")
print("[PASS] Session destruction no longer claims RAM zeroization.")

In [ ]:
# @title  API Contract Regression Tests

print("=" * 100)
print("API CONTRACT REGRESSION TESTS")
print("=" * 100)


# ======================================================================
# TEST 1 — canonical endpoint normalization
# ======================================================================

assert (

    _narad_api_canonical_node_name(
        " Alice "
    )

    ==

    "Alice"
)


assert (

    _narad_api_canonical_node_name(
        "Bob"
    )

    ==

    "Bob"
)


(
    _test_sender,
    _test_receiver
) = (

    _narad_api_canonical_endpoints(

        " Alice ",

        " Bob "
    )
)


assert (
    _test_sender
    ==
    "Alice"
)


assert (
    _test_receiver
    ==
    "Bob"
)


print(
    "[PASS] Endpoint names normalize to authoritative topology names."
)


# ======================================================================
# TEST 2 — /links covers every physical topology edge
# ======================================================================

_test_links = (
    _narad_api_link_rows()
)


assert (

    len(
        _test_links
    )

    ==

    phase4_network
    .topology
    .link_count
)


_test_link_ids = {

    row[
        "link_id"
    ]

    for row
    in _test_links
}


_topology_link_ids = {

    link.link_id

    for link
    in phase4_network
    .topology
    .links
}


assert (
    _test_link_ids
    ==
    _topology_link_ids
)


print(
    "[PASS] /links serialization covers the complete physical graph."
)


# ======================================================================
# TEST 3 — route preview accepts whitespace but returns canonical names
# ======================================================================

_test_preview = (

    _narad_api_route_audit(

        sender=" Alice ",

        receiver=" Bob "
    )
)


assert (
    _test_preview[
        "sender"
    ]
    ==
    "Alice"
)


assert (
    _test_preview[
        "receiver"
    ]
    ==
    "Bob"
)


assert (
    _test_preview[
        "physical_paths"
    ]
    >=
    _test_preview[
        "role_valid_paths"
    ]
)


assert (

    _test_preview[
        "executable_route_count"
    ]

    ==

    len(
        _test_preview[
            "ranked_routes"
        ]
    )
)


print(
    "[PASS] Route preview canonicalizes endpoints and reports consistent counts."
)


# ======================================================================
# TEST 4 — every role-valid candidate appears in complete audit
# ======================================================================

assert (

    len(
        _test_preview[
            "candidate_audit"
        ]
    )

    ==

    _test_preview[
        "role_valid_paths"
    ]
)


for row in (

    _test_preview[
        "candidate_audit"
    ]
):

    assert (
        "rejection_reasons"
        in
        row
    )


    assert (
        "within_calibration_domain"
        in
        row
    )


    assert (
        "extrapolation_fields"
        in
        row
    )


print(
    "[PASS] Every role-valid route has explicit feasibility/rejection audit."
)


# ======================================================================
# TEST 5 — every INTERNAL-USER rejection is individually exposed
# ======================================================================

assert (

    len(
        _test_preview[
            "role_rejections"
        ]
    )

    ==

    _test_preview[
        "rejected_user_transit_paths"
    ]
)


for row in (

    _test_preview[
        "role_rejections"
    ]
):

    assert (
        row[
            "reason"
        ]
        ==
        "INTERNAL_USER_NOT_ALLOWED"
    )


    assert (
        row[
            "offending_user"
        ]
        is not None
    )


print(
    "[PASS] Internal-USER path rejections are individually auditable."
)


# ======================================================================
# TEST 6 — executable rows expose calibration diagnostics
# ======================================================================

for route in (

    _test_preview[
        "ranked_routes"
    ]
):

    assert (
        isinstance(
            route[
                "within_calibration_domain"
            ],
            bool
        )
    )


    assert (
        isinstance(
            route[
                "extrapolation_fields"
            ],
            list
        )
    )


    assert (

        route[
            "predicted_qber"
        ]

        <=

        route[
            "requested_qber_limit"
        ]
        +
        1e-12
    )


print(
    "[PASS] Ranked routes expose QBER boundary and calibration-domain diagnostics."
)


# ======================================================================
# TEST 7 — affected FastAPI routes occur exactly once
# ======================================================================

_expected_routes = (

    (
        "/links",
        "GET"
    ),

    (
        "/routes/preview",
        "POST"
    ),

    (
        "/session/establish",
        "POST"
    ),

    (
        "/message/send",
        "POST"
    ),

    (
        "/session/{session_id}",
        "DELETE"
    )
)


for (
    expected_path,
    expected_method
) in _expected_routes:

    matching = [

        route

        for route
        in app.routes

        if (

            getattr(
                route,
                "path",
                None
            )

            ==

            expected_path

            and

            expected_method
            in
            (
                getattr(
                    route,
                    "methods",
                    set()
                )

                or

                set()
            )
        )
    ]


    assert (

        len(
            matching
        )

        ==

        1
    ), (

        f"{expected_method} {expected_path} "
        f"registered {len(matching)} times."
    )


print(
    "[PASS] Hardened FastAPI routes are registered exactly once."
)


# ======================================================================
# TEST 8 — destruction contract is scientifically honest
# ======================================================================

_test_destroy_session_id = (
    "__narad_6_1c_destroy_test__"
)


with NARAD_API_SESSION_LOCK:

    NARAD_API_SESSIONS[
        _test_destroy_session_id
    ] = {

        "session_id":
            _test_destroy_session_id,

        "created_at_unix":
            time.time(),

        "last_accessed_at_unix":
            time.time(),

        "sender":
            "Alice",

        "receiver":
            "Bob",

        "key":
            b"\x42" * 16,

        "qkd_config":
            None,

        "result":
            None,

        "message_count":
            0
    }


_test_destroy_response = (

    destroy_session(
        _test_destroy_session_id
    )
)


assert (

    _test_destroy_response[
        "session_store_reference_removed"
    ]

    is True
)


assert (

    _test_destroy_response[
        "memory_zeroization_guaranteed"
    ]

    is False
)


assert (

    "key_material_retained"

    not in

    _test_destroy_response
)


with NARAD_API_SESSION_LOCK:

    assert (

        _test_destroy_session_id

        not in

        NARAD_API_SESSIONS
    )


print(
    "[PASS] Session destruction contract no longer overclaims memory erasure."
)


# ======================================================================
# TEST 9 — ranked path ordering matches candidate audit rank
# ======================================================================

_rank_lookup = {

    row[
        "path_index"
    ]:
        row[
            "rank"
        ]

    for row
    in _test_preview[
        "candidate_audit"
    ]

    if row[
        "executable"
    ]
}


for route in (

    _test_preview[
        "ranked_routes"
    ]
):

    assert (

        _rank_lookup[
            route[
                "path_index"
            ]
        ]

        ==

        route[
            "rank"
        ]
    )


print(
    "[PASS] Ranked-route output and complete candidate audit agree."
)


print("\n" + "=" * 100)
print("REGRESSION SUITE PASSED")
print("=" * 100)

In [ ]:
# @title  Establishment Safety and Idempotency

import hashlib
import json
import threading
import time
import uuid

from fastapi import Header, HTTPException


print("=" * 100)
print("ESTABLISHMENT SAFETY AND IDEMPOTENCY")
print("=" * 100)


# ======================================================================
# DESIGN
#
# Fixes two related Phase-6 operational problems:
#
#   1. A timeout/retry of POST /session/establish previously had no
#      idempotency semantics. The retry could create a second QKD run.
#
#   2. Concurrent establishments previously executed independent
#      simulator runtime states against the same operator topology.
#
# Current research-backend policy:
#
#   - optional client Idempotency-Key;
#   - same key + same request => exactly one establishment execution;
#   - same key + different request => HTTP 409;
#   - duplicate while original runs => wait for original result;
#   - completed duplicate => return original result;
#   - establishment physics is serialized globally until a true
#     network-wide resource allocator exists.
#
# Ordinary reads/messages are NOT globally serialized.
# ======================================================================


NARAD_API_IDEMPOTENCY_TTL_S = (
    NARAD_API_SESSION_TTL_S
)


# ======================================================================
# GLOBAL ESTABLISHMENT EXECUTION LOCK
#
# This is intentionally separate from NARAD_API_SESSION_LOCK.
#
# The session lock protects dictionary bookkeeping.
# This lock protects the semantic assumption that the current routed
# simulator does not model simultaneous competing network sessions.
# ======================================================================

NARAD_API_ESTABLISHMENT_EXECUTION_LOCK = (
    threading.Lock()
)


# ======================================================================
# IDEMPOTENCY REGISTRY
#
# No raw key material is stored here.
#
# Successful records contain only the same PUBLIC response returned by
# the HTTP endpoint.
# ======================================================================

NARAD_API_IDEMPOTENCY_LOCK = (
    threading.RLock()
)


NARAD_API_IDEMPOTENCY_RECORDS = {}


# ======================================================================
# PYDANTIC-COMPATIBLE SERIALIZATION
#
# Works with both Pydantic v1-style .dict() and v2-style .model_dump().
# ======================================================================

def _narad_api_model_to_plain(
    value
):

    if hasattr(
        value,
        "model_dump"
    ):

        return value.model_dump()


    if hasattr(
        value,
        "dict"
    ):

        return value.dict()


    return value


# ======================================================================
# CANONICAL ESTABLISHMENT FINGERPRINT
#
# Whitespace aliases must hash identically:
#
#       " Alice " -> "Alice"
#
# because 6.1C made those semantically identical requests.
# ======================================================================

def _narad_api_establishment_payload(
    request
):

    (
        sender,
        receiver
    ) = (

        _narad_api_canonical_endpoints(

            request.sender,

            request.receiver
        )
    )


    return {

        "sender":
            sender,

        "receiver":
            receiver,

        "qkd":
            _narad_api_model_to_plain(
                request.qkd
            ),

        "execution":
            _narad_api_model_to_plain(
                request.execution
            )
    }


def _narad_api_establishment_fingerprint(
    request
):

    payload = (

        _narad_api_establishment_payload(
            request
        )
    )


    encoded = (

        json.dumps(

            payload,

            sort_keys=True,

            separators=(
                ",",
                ":"
            ),

            default=str
        )
        .encode(
            "utf-8"
        )
    )


    return hashlib.sha256(
        encoded
    ).hexdigest()


# ======================================================================
# IDEMPOTENCY KEY VALIDATION
# ======================================================================

def _narad_api_normalize_idempotency_key(
    value
):

    if value is None:

        return None


    value = (
        str(value)
        .strip()
    )


    if not value:

        raise HTTPException(

            status_code=400,

            detail=(
                "Idempotency-Key cannot be empty."
            )
        )


    if len(value) > 128:

        raise HTTPException(

            status_code=400,

            detail=(
                "Idempotency-Key exceeds 128 characters."
            )
        )


    return value


# ======================================================================
# IDEMPOTENCY REGISTRY CLEANUP
# ======================================================================

def _narad_api_prune_idempotency_records(
    *,
    now=None
):

    if now is None:

        now = time.time()


    removed = []


    with NARAD_API_IDEMPOTENCY_LOCK:

        for (
            key,
            record
        ) in list(

            NARAD_API_IDEMPOTENCY_RECORDS.items()
        ):

            # NEVER delete an in-flight record.
            if (
                record[
                    "status"
                ]
                ==
                "IN_PROGRESS"
            ):

                continue


            age_s = (

                float(now)

                -

                float(
                    record[
                        "updated_at_unix"
                    ]
                )
            )


            if (

                age_s

                >=

                NARAD_API_IDEMPOTENCY_TTL_S
            ):

                NARAD_API_IDEMPOTENCY_RECORDS.pop(

                    key,

                    None
                )


                removed.append(
                    key
                )


    return tuple(
        removed
    )


# ======================================================================
# PUBLIC RECORD VIEW
# ======================================================================

def _narad_api_public_idempotency_record(
    key,
    record
):

    output = {

        "idempotency_key":
            key,

        "status":
            record[
                "status"
            ],

        "created_at_unix":
            float(
                record[
                    "created_at_unix"
                ]
            ),

        "updated_at_unix":
            float(
                record[
                    "updated_at_unix"
                ]
            )
    }


    if (
        record[
            "status"
        ]
        ==
        "SUCCEEDED"
    ):

        response = (
            record[
                "response"
            ]
        )


        output[
            "session_id"
        ] = (

            response.get(
                "session_id"
            )
        )


    elif (
        record[
            "status"
        ]
        ==
        "FAILED"
    ):

        output[
            "error"
        ] = (
            record[
                "error"
            ]
        )


    return output


# ======================================================================
# CORE SESSION ESTABLISHMENT
#
# This contains the actual one-time physics/service execution.
#
# Keeping it separate lets idempotency wrap around it cleanly.
# ======================================================================

def _narad_api_execute_establishment_once(
    request
):

    (
        canonical_sender,
        canonical_receiver
    ) = (

        _narad_api_canonical_endpoints(

            request.sender,

            request.receiver
        )
    )


    qkd_config = (

        narad_api_make_qkd_config(
            request.qkd
        )
    )


    requested_cpu_workers = (
        request.execution.cpu_workers
    )


    requested_parallel_lanes = (
        request.execution.parallel_quantum_lanes
    )


    # ==================================================================
    # IMPORTANT:
    #
    # Current simulator does not contain a genuine global allocator for
    # competing simultaneous sessions.
    #
    # Until such an allocator exists, serialize PHYSICAL ESTABLISHMENT
    # rather than pretending independent simulator stacks compete for
    # the same resources.
    # ==================================================================

    with NARAD_API_ESTABLISHMENT_EXECUTION_LOCK:

        result = (

            narad_qc_establish_key(

                network=(
                    phase4_network
                ),

                sender_name=(
                    canonical_sender
                ),

                receiver_name=(
                    canonical_receiver
                ),

                qkd_config=(
                    qkd_config
                ),

                max_hops=(
                    request.execution.max_hops
                ),

                physical_seed=(
                    request.execution.physical_seed
                ),

                protocol_seed=(
                    request.execution.protocol_seed
                ),

                postprocess_seed=(
                    request.execution.postprocess_seed
                ),

                verbose=False,

                return_details=True
            )
        )


    _narad_api_prune_expired_sessions()


    session_id = (
        str(
            uuid.uuid4()
        )
    )


    now = time.time()


    with NARAD_API_SESSION_LOCK:

        NARAD_API_SESSIONS[
            session_id
        ] = {

            "session_id":
                session_id,

            "created_at_unix":
                now,

            "last_accessed_at_unix":
                now,

            "sender":
                result.sender,

            "receiver":
                result.receiver,

            # RAW KEY REMAINS SERVER-SIDE ONLY
            "key":
                bytes(
                    result.key
                ),

            "qkd_config":
                qkd_config,

            "result":
                result,

            "message_count":
                0
        }


    return {

        "session_id":
            session_id,

        "status":
            "ESTABLISHED",

        "sender":
            result.sender,

        "receiver":
            result.receiver,

        "session_ttl_s":
            NARAD_API_SESSION_TTL_S,

        "result":
            narad_api_public_result(
                result
            ),

        "execution": {

            "requested_cpu_workers":
                requested_cpu_workers,

            "applied_cpu_workers":
                1,

            "requested_parallel_quantum_lanes":
                requested_parallel_lanes,

            "applied_parallel_quantum_lanes":
                1,

            "establishment_concurrency":
                "SERIALIZED",

            "note": (
                "Current heterogeneous routed executor is "
                "single-lane and establishment runs are serialized "
                "until a network-wide concurrent resource allocator "
                "is implemented."
            )
        }
    }


# ======================================================================
# IDEMPOTENT ESTABLISHMENT WRAPPER
# ======================================================================

def _narad_api_establish_idempotent(
    request,
    idempotency_key=None
):

    idempotency_key = (

        _narad_api_normalize_idempotency_key(
            idempotency_key
        )
    )


    # ==================================================================
    # No key supplied:
    #
    # Preserve backward compatibility.
    #
    # Request still benefits from global establishment serialization,
    # but automatic retry is NOT safe because there is no identity tying
    # two HTTP requests together.
    # ==================================================================

    if idempotency_key is None:

        response = (

            _narad_api_execute_establishment_once(
                request
            )
        )


        response[
            "idempotency"
        ] = {

            "protected":
                False,

            "key":
                None,

            "replayed":
                False,

            "note": (
                "No Idempotency-Key was supplied. "
                "Automatic retry after an uncertain timeout is unsafe."
            )
        }


        return response


    fingerprint = (

        _narad_api_establishment_fingerprint(
            request
        )
    )


    _narad_api_prune_idempotency_records()


    owner = False


    with NARAD_API_IDEMPOTENCY_LOCK:

        record = (

            NARAD_API_IDEMPOTENCY_RECORDS.get(
                idempotency_key
            )
        )


        if record is None:

            now = time.time()


            record = {

                "fingerprint":
                    fingerprint,

                "status":
                    "IN_PROGRESS",

                "created_at_unix":
                    now,

                "updated_at_unix":
                    now,

                "event":
                    threading.Event(),

                "response":
                    None,

                "error":
                    None
            }


            NARAD_API_IDEMPOTENCY_RECORDS[
                idempotency_key
            ] = record


            owner = True


        else:

            if (

                record[
                    "fingerprint"
                ]

                !=

                fingerprint
            ):

                raise HTTPException(

                    status_code=409,

                    detail=(
                        "Idempotency-Key was already used with "
                        "a different establishment request."
                    )
                )


    # ==================================================================
    # FIRST CALLER OWNS THE EXECUTION
    # ==================================================================

    if owner:

        try:

            response = (

                _narad_api_execute_establishment_once(
                    request
                )
            )


            response[
                "idempotency"
            ] = {

                "protected":
                    True,

                "key":
                    idempotency_key,

                "replayed":
                    False
            }


            with NARAD_API_IDEMPOTENCY_LOCK:

                record[
                    "status"
                ] = "SUCCEEDED"


                record[
                    "response"
                ] = response


                record[
                    "updated_at_unix"
                ] = time.time()


                record[
                    "event"
                ].set()


            return response


        except HTTPException as error:

            public_error = {

                "status_code":
                    int(
                        error.status_code
                    ),

                "detail":
                    error.detail
            }


            with NARAD_API_IDEMPOTENCY_LOCK:

                record[
                    "status"
                ] = "FAILED"


                record[
                    "error"
                ] = public_error


                record[
                    "updated_at_unix"
                ] = time.time()


                record[
                    "event"
                ].set()


            raise


        except Exception as error:

            public_error = {

                "status_code":
                    400,

                "detail": (
                    f"{type(error).__name__}: "
                    f"{error}"
                )
            }


            with NARAD_API_IDEMPOTENCY_LOCK:

                record[
                    "status"
                ] = "FAILED"


                record[
                    "error"
                ] = public_error


                record[
                    "updated_at_unix"
                ] = time.time()


                record[
                    "event"
                ].set()


            raise


    # ==================================================================
    # DUPLICATE CALL
    #
    # If original call is still running, wait for THAT call rather than
    # performing another physical/QKD execution.
    # ==================================================================

    event = (
        record[
            "event"
        ]
    )


    event.wait()


    with NARAD_API_IDEMPOTENCY_LOCK:

        status = (
            record[
                "status"
            ]
        )


        if status == "SUCCEEDED":

            # shallow/deep-safe JSON roundtrip copy;
            # public response contains no key bytes.
            replay_response = (

                json.loads(

                    json.dumps(
                        record[
                            "response"
                        ]
                    )
                )
            )


            replay_response[
                "idempotency"
            ][
                "replayed"
            ] = True


            return replay_response


        if status == "FAILED":

            error = (
                record[
                    "error"
                ]
            )


            raise HTTPException(

                status_code=int(
                    error[
                        "status_code"
                    ]
                ),

                detail=(
                    error[
                        "detail"
                    ]
                )
            )


    raise HTTPException(

        status_code=500,

        detail=(
            "Invalid NARAD-QC idempotency state."
        )
    )


# ======================================================================
# REPLACE POST /session/establish
# ======================================================================

_narad_api_remove_route(
    "/session/establish",
    "POST"
)


@app.post("/session/establish")
def establish_session(
    request: SessionEstablishRequest,

    idempotency_key: str | None = Header(

        default=None,

        alias="Idempotency-Key"
    )
):

    try:

        return (

            _narad_api_establish_idempotent(

                request,

                idempotency_key=(
                    idempotency_key
                )
            )
        )


    except HTTPException:

        raise


    except Exception as error:

        raise HTTPException(

            status_code=400,

            detail=(

                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ======================================================================
# STATUS / RECOVERY LOOKUP
#
# This does NOT provide live physical-stage percentages.
#
# It only tells the truth:
#
#     IN_PROGRESS
#     SUCCEEDED
#     FAILED
#
# which is enough for uncertain timeout recovery.
# ======================================================================

_narad_api_remove_route(
    "/establishment/{idempotency_key}",
    "GET"
)


@app.get("/establishment/{idempotency_key}")
def establishment_status(
    idempotency_key: str
):

    idempotency_key = (

        _narad_api_normalize_idempotency_key(
            idempotency_key
        )
    )


    _narad_api_prune_idempotency_records()


    with NARAD_API_IDEMPOTENCY_LOCK:

        record = (

            NARAD_API_IDEMPOTENCY_RECORDS.get(
                idempotency_key
            )
        )


        if record is None:

            raise HTTPException(

                status_code=404,

                detail=(
                    "Unknown or expired establishment "
                    "idempotency key."
                )
            )


        return (

            _narad_api_public_idempotency_record(

                idempotency_key,

                record
            )
        )


NARAD_API_ESTABLISHMENT_HARDENING_VERSION = (
    "6.1D"
)


print()
print("[PASS] Idempotent establishment support installed.")
print("[PASS] Duplicate keyed requests execute QKD at most once.")
print("[PASS] Key reuse with a different request returns HTTP 409.")
print("[PASS] Establishment status/recovery endpoint installed.")
print("[PASS] Physical establishment execution is globally serialized.")
print("[PASS] Unkeyed requests remain backward-compatible but explicitly unsafe to retry.")

In [ ]:
# @title  Establishment Safety Regression Tests

from types import SimpleNamespace
import threading
import time

from fastapi import HTTPException


print("=" * 100)
print("PHASE 6.1D ESTABLISHMENT SAFETY REGRESSION TESTS")
print("=" * 100)


# ======================================================================
# TEST REQUEST
#
# Reuse actual Phase-6 Pydantic models.
# ======================================================================

_test_request = (

    SessionEstablishRequest(

        sender=" Alice ",

        receiver=" Bob "
    )
)


# ======================================================================
# TEST 1 — canonical requests produce same fingerprint
# ======================================================================

_test_request_canonical = (

    SessionEstablishRequest(

        sender="Alice",

        receiver="Bob"
    )
)


assert (

    _narad_api_establishment_fingerprint(
        _test_request
    )

    ==

    _narad_api_establishment_fingerprint(
        _test_request_canonical
    )
)


print(
    "[PASS] Semantically identical canonical endpoint requests "
    "produce the same fingerprint."
)


# ======================================================================
# Preserve real executor.
# ======================================================================

_real_execute_establishment_once = (

    _narad_api_execute_establishment_once
)


# ======================================================================
# FAKE EXECUTOR
#
# Counts calls and deliberately sleeps briefly so two threads overlap.
# ======================================================================

_fake_counter_lock = (
    threading.Lock()
)


_fake_execution_count = 0


def _fake_execute_establishment_once(
    request
):

    global _fake_execution_count


    with _fake_counter_lock:

        _fake_execution_count += 1


        current_execution = (
            _fake_execution_count
        )


    time.sleep(
        0.20
    )


    return {

        "session_id":
            "FAKE-SESSION-001",

        "status":
            "ESTABLISHED",

        "sender":
            "Alice",

        "receiver":
            "Bob",

        "execution_number":
            current_execution,

        "result":
            {
                "success":
                    True
            }
    }


# Temporarily install fake.
_narad_api_execute_establishment_once = (
    _fake_execute_establishment_once
)


try:

    # ==================================================================
    # TEST 2 — concurrent duplicate keyed calls execute exactly once
    # ==================================================================

    _test_key = (
        "__narad_6_1d_concurrent__"
    )


    with NARAD_API_IDEMPOTENCY_LOCK:

        NARAD_API_IDEMPOTENCY_RECORDS.pop(

            _test_key,

            None
        )


    _fake_execution_count = 0

    _thread_results = []

    _thread_errors = []


    def _invoke_duplicate():

        try:

            response = (

                _narad_api_establish_idempotent(

                    _test_request,

                    idempotency_key=(
                        _test_key
                    )
                )
            )


            _thread_results.append(
                response
            )


        except Exception as error:

            _thread_errors.append(
                error
            )


    thread_a = threading.Thread(
        target=_invoke_duplicate
    )


    thread_b = threading.Thread(
        target=_invoke_duplicate
    )


    thread_a.start()

    time.sleep(
        0.03
    )

    thread_b.start()


    thread_a.join(
        timeout=5.0
    )


    thread_b.join(
        timeout=5.0
    )


    assert not thread_a.is_alive()

    assert not thread_b.is_alive()


    assert (
        _thread_errors
        ==
        []
    )


    assert (
        _fake_execution_count
        ==
        1
    )


    assert (
        len(
            _thread_results
        )
        ==
        2
    )


    assert (

        _thread_results[0][
            "session_id"
        ]

        ==

        _thread_results[1][
            "session_id"
        ]

        ==

        "FAKE-SESSION-001"
    )


    replay_flags = sorted(

        response[
            "idempotency"
        ][
            "replayed"
        ]

        for response
        in _thread_results
    )


    assert (
        replay_flags
        ==
        [
            False,
            True
        ]
    )


    print(
        "[PASS] Concurrent duplicate keyed requests execute exactly once."
    )


    # ==================================================================
    # TEST 3 — completed retry returns original result without execution
    # ==================================================================

    before = (
        _fake_execution_count
    )


    replay = (

        _narad_api_establish_idempotent(

            _test_request,

            idempotency_key=(
                _test_key
            )
        )
    )


    after = (
        _fake_execution_count
    )


    assert (
        before
        ==
        after
        ==
        1
    )


    assert (
        replay[
            "session_id"
        ]
        ==
        "FAKE-SESSION-001"
    )


    assert (

        replay[
            "idempotency"
        ][
            "replayed"
        ]

        is True
    )


    print(
        "[PASS] Completed retry replays original response "
        "without another establishment."
    )


    # ==================================================================
    # TEST 4 — same key with different request is rejected
    # ==================================================================

    conflicting_request = (

        SessionEstablishRequest(

            sender="Alice",

            receiver="Bob"
        )
    )


    # Modify a real request parameter without depending on exact
    # Pydantic constructor defaults.
    if hasattr(
        conflicting_request.execution,
        "physical_seed"
    ):

        conflicting_request.execution.physical_seed = (

            int(
                conflicting_request
                .execution
                .physical_seed
            )

            +
            1
        )


    conflict_seen = False


    try:

        _narad_api_establish_idempotent(

            conflicting_request,

            idempotency_key=(
                _test_key
            )
        )


    except HTTPException as error:

        conflict_seen = True


        assert (
            error.status_code
            ==
            409
        )


    assert (
        conflict_seen
        is True
    )


    assert (
        _fake_execution_count
        ==
        1
    )


    print(
        "[PASS] Same idempotency key cannot be reused "
        "with a different request."
    )


    # ==================================================================
    # TEST 5 — status lookup data contains no key material
    # ==================================================================

    with NARAD_API_IDEMPOTENCY_LOCK:

        record = (

            NARAD_API_IDEMPOTENCY_RECORDS[
                _test_key
            ]
        )


        public_record = (

            _narad_api_public_idempotency_record(

                _test_key,

                record
            )
        )


    assert (
        public_record[
            "status"
        ]
        ==
        "SUCCEEDED"
    )


    assert (
        public_record[
            "session_id"
        ]
        ==
        "FAKE-SESSION-001"
    )


    public_text = (
        json.dumps(
            public_record
        )
    )


    assert (
        '"key"'
        not in
        public_text
    )


    print(
        "[PASS] Establishment status exposes no raw key material."
    )


    # ==================================================================
    # TEST 6 — unkeyed calls remain NON-idempotent by design
    # ==================================================================

    _fake_execution_count = 0


    first_unkeyed = (

        _narad_api_establish_idempotent(

            _test_request,

            idempotency_key=None
        )
    )


    second_unkeyed = (

        _narad_api_establish_idempotent(

            _test_request,

            idempotency_key=None
        )
    )


    assert (
        _fake_execution_count
        ==
        2
    )


    assert (

        first_unkeyed[
            "idempotency"
        ][
            "protected"
        ]

        is False
    )


    assert (

        second_unkeyed[
            "idempotency"
        ][
            "protected"
        ]

        is False
    )


    print(
        "[PASS] Unkeyed establishment remains explicitly "
        "non-idempotent and backward-compatible."
    )


finally:

    # ==================================================================
    # ALWAYS RESTORE REAL PHYSICS EXECUTOR
    # ==================================================================

    _narad_api_execute_establishment_once = (

        _real_execute_establishment_once
    )


    with NARAD_API_IDEMPOTENCY_LOCK:

        for test_key in (
            "__narad_6_1d_concurrent__",
        ):

            NARAD_API_IDEMPOTENCY_RECORDS.pop(

                test_key,

                None
            )


# ======================================================================
# TEST 7 — hardened FastAPI routes registered exactly once
# ======================================================================

for (
    expected_path,
    expected_method
) in (

    (
        "/session/establish",
        "POST"
    ),

    (
        "/establishment/{idempotency_key}",
        "GET"
    )
):

    matching = [

        route

        for route
        in app.routes

        if (

            getattr(
                route,
                "path",
                None
            )

            ==

            expected_path

            and

            expected_method
            in
            (
                getattr(
                    route,
                    "methods",
                    set()
                )

                or

                set()
            )
        )
    ]


    assert (
        len(
            matching
        )
        ==
        1
    )


print(
    "[PASS] Idempotent establishment API routes "
    "are registered exactly once."
)


print("\n" + "=" * 100)
print("REGRESSION SUITE PASSED")
print("=" * 100)

In [ ]:
# @title  HTTP Access Boundary Hardening

import hmac
import secrets

from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse


print("=" * 100)
print("HTTP ACCESS BOUNDARY HARDENING")
print("=" * 100)


# ======================================================================
# PURPOSE
#
# IMPORTANT SECURITY BOUNDARY:
#
#   ML-DSA authentication inside NARAD-QC
#       !=
#   authentication of the HTTP/API caller
#
# Phase 3A authenticates simulated network identities.
#
# This Phase 6.1E layer protects the externally exposed FastAPI
# interface itself.
#
# This is intentionally a research/demo operator-access mechanism,
# NOT a full multi-user IAM/OAuth system.
# ======================================================================


# ======================================================================
# API ACCESS KEY
#
# Preserve the key if this cell is rerun in the same runtime.
# This prevents accidentally invalidating the GUI/client every rerun.
# ======================================================================

if (
    "NARAD_API_ACCESS_KEY"
    not in globals()
):

    NARAD_API_ACCESS_KEY = (
        secrets.token_urlsafe(32)
    )


NARAD_API_ACCESS_HEADER = (
    "X-NARAD-API-Key"
)


# ======================================================================
# PUBLIC PATHS
#
# These remain accessible without credentials so:
#
#   - liveness works;
#   - readiness works;
#   - hardened Cloudflare verification still works;
#   - Swagger/OpenAPI can describe the service.
#
# Everything else is protected.
# ======================================================================

NARAD_API_PUBLIC_EXACT_PATHS = frozenset(
    {
        "/",
        "/health",
        "/ready",
        "/docs",
        "/redoc",
        "/openapi.json",
        "/docs/oauth2-redirect",
    }
)


def _narad_api_path_is_public(
    path
):

    path = str(
        path
    )


    return bool(

        path
        in
        NARAD_API_PUBLIC_EXACT_PATHS
    )


# ======================================================================
# CONSTANT-TIME CREDENTIAL CHECK
# ======================================================================

def _narad_api_access_key_valid(
    candidate
):

    if candidate is None:

        return False


    candidate = str(
        candidate
    )


    if not candidate:

        return False


    return hmac.compare_digest(

        candidate,

        NARAD_API_ACCESS_KEY
    )


# ======================================================================
# ACCESS MIDDLEWARE
# ======================================================================

class NARADAPIAccessBoundaryMiddleware(
    BaseHTTPMiddleware
):

    async def dispatch(
        self,
        request,
        call_next
    ):

        path = (
            request.url.path
        )


        # --------------------------------------------------------------
        # Public operational/documentation endpoints.
        # --------------------------------------------------------------

        if _narad_api_path_is_public(
            path
        ):

            return await call_next(
                request
            )


        supplied_key = (

            request.headers.get(
                NARAD_API_ACCESS_HEADER
            )
        )


        if not _narad_api_access_key_valid(
            supplied_key
        ):

            return JSONResponse(

                status_code=401,

                content={

                    "detail":
                        "Valid NARAD-QC API access key required.",

                    "authentication_scheme":
                        "API_KEY",

                    "header":
                        NARAD_API_ACCESS_HEADER
                },

                headers={

                    "Cache-Control":
                        "no-store"
                }
            )


        response = await call_next(
            request
        )


        # --------------------------------------------------------------
        # Avoid caching authenticated research/session responses.
        # --------------------------------------------------------------

        response.headers[
            "Cache-Control"
        ] = "no-store"


        return response


# ======================================================================
# INSTALL / REINSTALL MIDDLEWARE SAFELY
#
# FastAPI/Starlette normally forbids add_middleware() after the app has
# already started once.
#
# Since this notebook evolves incrementally, remove any previous
# NARAD access-boundary middleware directly from user_middleware and
# rebuild the ASGI middleware stack.
#
# Other existing middleware is preserved.
# ======================================================================

app.user_middleware[:] = [

    middleware

    for middleware
    in app.user_middleware

    if (

        getattr(
            middleware,
            "cls",
            None
        )

        is not

        NARADAPIAccessBoundaryMiddleware
    )
]


app.user_middleware.insert(

    0,

    Middleware(
        NARADAPIAccessBoundaryMiddleware
    )
)


app.middleware_stack = (
    app.build_middleware_stack()
)


# ======================================================================
# SECURITY METADATA
# ======================================================================

NARAD_API_ACCESS_BOUNDARY_VERSION = (
    "6.1E"
)


NARAD_API_AUTHENTICATION_MODEL = {

    "http_api_authentication":
        "API_KEY",

    "header":
        NARAD_API_ACCESS_HEADER,

    "node_authentication":
        "ML-DSA-65 simulated mutual authentication",

    "boundaries_are_distinct":
        True,

    "public_paths":
        sorted(
            NARAD_API_PUBLIC_EXACT_PATHS
        )
}


print()
print("[PASS] HTTP caller authentication boundary installed.")
print("[PASS] ML-DSA node authentication remains unchanged.")
print("[PASS] /health and /ready remain public.")
print("[PASS] NARAD-QC operational/session endpoints now require API key.")
print("[PASS] Authenticated responses are marked no-store.")

print("\n" + "=" * 100)
print("NARAD-QC API ACCESS KEY")
print("=" * 100)

print(
    NARAD_API_ACCESS_KEY
)

print("=" * 100)

print(
    "\nUse this HTTP header for protected requests:"
)

print(
    f"{NARAD_API_ACCESS_HEADER}: "
    f"{NARAD_API_ACCESS_KEY}"
)

print(
    "\nIMPORTANT: treat this value as a runtime secret. "
    "Do not hard-code it into a public repository."
)

In [ ]:
# @title  HTTP Access Boundary Regression Tests

from fastapi.testclient import TestClient


print("=" * 100)
print("HTTP ACCESS REGRESSION TESTS")
print("=" * 100)


# ======================================================================
# TEST 1 — key properties
# ======================================================================

assert isinstance(
    NARAD_API_ACCESS_KEY,
    str
)


assert (
    len(
        NARAD_API_ACCESS_KEY
    )
    >=
    32
)


assert (

    _narad_api_access_key_valid(
        NARAD_API_ACCESS_KEY
    )

    is True
)


assert (

    _narad_api_access_key_valid(
        "definitely-wrong-key"
    )

    is False
)


assert (

    _narad_api_access_key_valid(
        None
    )

    is False
)


print(
    "[PASS] API access-key validator accepts only the active runtime key."
)


# ======================================================================
# TEST 2 — middleware installed exactly once
# ======================================================================

_access_middlewares = [

    middleware

    for middleware
    in app.user_middleware

    if (

        getattr(
            middleware,
            "cls",
            None
        )

        is

        NARADAPIAccessBoundaryMiddleware
    )
]


assert (
    len(
        _access_middlewares
    )
    ==
    1
)


print(
    "[PASS] Access middleware is installed exactly once."
)


# ======================================================================
# TEST CLIENT
# ======================================================================

_test_client = (
    TestClient(
        app
    )
)


# ======================================================================
# TEST 3 — public /health remains accessible
# ======================================================================

_health_response = (

    _test_client.get(
        "/health"
    )
)


assert (
    _health_response.status_code
    ==
    200
)


print(
    "[PASS] /health remains publicly accessible."
)


# ======================================================================
# TEST 4 — public /ready remains accessible
# ======================================================================

_ready_response = (

    _test_client.get(
        "/ready"
    )
)


assert (
    _ready_response.status_code
    ==
    200
), (
    _ready_response.text
)


_ready_json = (
    _ready_response.json()
)


assert (
    _ready_json[
        "ready"
    ]

    is True
)


print(
    "[PASS] /ready remains publicly accessible."
)


# ======================================================================
# TEST 5 — protected endpoint without key is rejected
#
# /nodes is safe and cheap for this test:
# no physical QKD execution.
# ======================================================================

_no_key_response = (

    _test_client.get(
        "/nodes"
    )
)


assert (
    _no_key_response.status_code
    ==
    401
)


_no_key_json = (
    _no_key_response.json()
)


assert (

    _no_key_json[
        "authentication_scheme"
    ]

    ==

    "API_KEY"
)


print(
    "[PASS] Protected endpoint rejects missing API key."
)


# ======================================================================
# TEST 6 — wrong key is rejected
# ======================================================================

_wrong_key_response = (

    _test_client.get(

        "/nodes",

        headers={

            NARAD_API_ACCESS_HEADER:
                "wrong-key"
        }
    )
)


assert (
    _wrong_key_response.status_code
    ==
    401
)


print(
    "[PASS] Protected endpoint rejects incorrect API key."
)


# ======================================================================
# TEST 7 — correct key grants access
# ======================================================================

_auth_headers = {

    NARAD_API_ACCESS_HEADER:
        NARAD_API_ACCESS_KEY
}


_valid_response = (

    _test_client.get(

        "/nodes",

        headers=(
            _auth_headers
        )
    )
)


assert (
    _valid_response.status_code
    ==
    200
), (
    _valid_response.text
)


print(
    "[PASS] Protected endpoint accepts valid API key."
)


# ======================================================================
# TEST 8 — /links also protected
# ======================================================================

_links_without_key = (

    _test_client.get(
        "/links"
    )
)


assert (
    _links_without_key.status_code
    ==
    401
)


_links_with_key = (

    _test_client.get(

        "/links",

        headers=(
            _auth_headers
        )
    )
)


assert (
    _links_with_key.status_code
    ==
    200
)


print(
    "[PASS] Physical topology endpoint is protected."
)


# ======================================================================
# TEST 9 — route preview protected without running QKD
# ======================================================================

_preview_without_key = (

    _test_client.post(

        "/routes/preview",

        json={

            "sender":
                "Alice",

            "receiver":
                "Bob"
        }
    )
)


assert (
    _preview_without_key.status_code
    ==
    401
)


_preview_with_key = (

    _test_client.post(

        "/routes/preview",

        headers=(
            _auth_headers
        ),

        json={

            "sender":
                "Alice",

            "receiver":
                "Bob"
        }
    )
)


assert (
    _preview_with_key.status_code
    ==
    200
), (
    _preview_with_key.text
)


print(
    "[PASS] Route preview requires authenticated HTTP caller."
)


# ======================================================================
# TEST 10 — Swagger/OpenAPI remains public
# ======================================================================

_openapi_response = (

    _test_client.get(
        "/openapi.json"
    )
)


assert (
    _openapi_response.status_code
    ==
    200
)


print(
    "[PASS] OpenAPI schema remains publicly reachable."
)


# ======================================================================
# TEST 11 — authenticated responses disable caching
# ======================================================================

assert (

    _valid_response.headers.get(
        "cache-control"
    )

    ==

    "no-store"
)


print(
    "[PASS] Authenticated API responses are marked no-store."
)


print("\n" + "=" * 100)
print("REGRESSION SUITE PASSED")
print("=" * 100)

In [ ]:
# @title Start NARAD-QC API Server — Hardened

import json
import threading
import time
import urllib.error
import urllib.request

import nest_asyncio
import uvicorn


nest_asyncio.apply()


NARAD_API_HOST = "127.0.0.1"
NARAD_API_PORT = 8001

NARAD_API_LOCAL_URL = (
    f"http://{NARAD_API_HOST}:{NARAD_API_PORT}"
)


# ======================================================================
# HTTP helper
# ======================================================================

def _narad_http_json(
    url,
    *,
    timeout_s=2.0
):

    request = urllib.request.Request(

        url,

        headers={
            "User-Agent":
                "NARAD-QC-runtime-probe/1.0"
        }
    )


    with urllib.request.urlopen(

        request,

        timeout=float(
            timeout_s
        )
    ) as response:

        payload = (
            response.read()
        )


    return json.loads(

        payload.decode(
            "utf-8"
        )
    )


# ======================================================================
# Stop a PREVIOUS SERVER CREATED BY THIS HARDENED CELL.
#
# This makes rerunning this cell predictable.
# ======================================================================

_previous_server = globals().get(

    "NARAD_API_UVICORN_SERVER"
)


_previous_thread = globals().get(

    "NARAD_API_SERVER_THREAD"
)


if (

    _previous_server is not None

    and

    _previous_thread is not None

    and

    _previous_thread.is_alive()
):

    print(
        "[INFO] Stopping previous managed NARAD-QC API server..."
    )


    _previous_server.should_exit = True


    _previous_thread.join(
        timeout=5.0
    )


    if _previous_thread.is_alive():

        raise RuntimeError(

            "Previous managed API server did not stop cleanly."
        )


# ======================================================================
# Detect a stale/legacy server already occupying port 8000.
# ======================================================================

_existing_payload = None


try:

    _existing_payload = (

        _narad_http_json(

            NARAD_API_LOCAL_URL
            +
            "/ready",

            timeout_s=1.0
        )
    )


except Exception:

    _existing_payload = None


if _existing_payload is not None:

    existing_generation = (

        _existing_payload.get(
            "runtime_generation"
        )
    )


    if (

        existing_generation

        !=

        NARAD_API_RUNTIME_GENERATION
    ):

        raise RuntimeError(

            "Port 8000 is serving a stale NARAD-QC app generation. "
            "A legacy Uvicorn server from an older service cell is "
            "still alive. Restart the notebook runtime once, run the "
            "notebook through the hardening cells, then start this "
            "server cell again."
        )


    print(
        "[PASS] Current NARAD-QC app generation is already serving."
    )


else:

    # ==================================================================
    # Managed Uvicorn server
    # ==================================================================

    NARAD_API_UVICORN_CONFIG = (

        uvicorn.Config(

            app,

            host=NARAD_API_HOST,

            port=NARAD_API_PORT,

            log_level="info"
        )
    )


    NARAD_API_UVICORN_SERVER = (

        uvicorn.Server(

            NARAD_API_UVICORN_CONFIG
        )
    )


    def _run_narad_api_server():

        NARAD_API_UVICORN_SERVER.run()


    NARAD_API_SERVER_THREAD = (

        threading.Thread(

            target=(
                _run_narad_api_server
            ),

            name=(
                "NARAD-QC-API-MANAGED"
            ),

            daemon=True
        )
    )


    NARAD_API_SERVER_THREAD.start()


    # ==================================================================
    # REAL readiness probe.
    #
    # No arbitrary "sleep(2) => PASS".
    # ==================================================================

    deadline = (

        time.monotonic()
        +
        20.0
    )


    last_error = None

    ready_payload = None


    while (

        time.monotonic()
        <
        deadline
    ):

        try:

            ready_payload = (

                _narad_http_json(

                    NARAD_API_LOCAL_URL
                    +
                    "/ready",

                    timeout_s=1.0
                )
            )


            if (

                ready_payload.get(
                    "ready"
                )

                is True

                and

                ready_payload.get(
                    "runtime_generation"
                )

                ==

                NARAD_API_RUNTIME_GENERATION
            ):

                break


        except Exception as error:

            last_error = error


        time.sleep(
            0.20
        )


    if ready_payload is None:

        NARAD_API_UVICORN_SERVER.should_exit = True


        raise RuntimeError(

            "NARAD-QC API did not become ready. "
            f"Last probe error: {last_error}"
        )


    if (

        ready_payload.get(
            "runtime_generation"
        )

        !=

        NARAD_API_RUNTIME_GENERATION
    ):

        NARAD_API_UVICORN_SERVER.should_exit = True


        raise RuntimeError(

            "Port 8000 responded, but it is not the current "
            "NARAD-QC application generation."
        )


print("\n" + "=" * 100)

print(
    "[PASS] NARAD-QC API passed real HTTP readiness probe."
)

print(
    "Local API:"
)

print(
    NARAD_API_LOCAL_URL
)

print(
    "\nLocal readiness:"
)

print(
    NARAD_API_LOCAL_URL
    +
    "/ready"
)

print(
    "\nSwagger UI:"
)

print(
    NARAD_API_LOCAL_URL
    +
    "/docs"
)

print("=" * 100)

In [ ]:
# @title  Runtime Parallelism + Live QKD Telemetry

from contextvars import ContextVar
from dataclasses import asdict, replace
import gc
import math
import multiprocessing as mp
import os
import time


print("=" * 100)
print("RUNTIME PARALLELISM + LIVE QKD TELEMETRY")
print("=" * 100)


# ======================================================================
# CONTROLLER WORKER CAP
#
# This is intentionally separate from os.cpu_count().
#
# Example:
#
#     Colab/runtime exposes 8 logical CPUs
#     but controller knows 4 is the safe operating maximum.
#
# The GUI may request any value <= this cap.
# ======================================================================

if "NARAD_CONTROLLER_MAX_CPU_WORKERS" not in globals():

    NARAD_CONTROLLER_MAX_CPU_WORKERS = min(

        4,

        max(
            1,
            os.cpu_count()
            or
            1
        )
    )


# ======================================================================
# REQUEST-LOCAL EXECUTION STATE
#
# Mutable dict is intentional:
# actual worker fallback information can be written back into the same
# audit object which Phase 6 later returns to the GUI.
# ======================================================================

_NARAD_EXECUTION_CONTEXT = ContextVar(

    "NARAD_EXECUTION_CONTEXT",

    default=None
)


_NARAD_PROGRESS_CALLBACK = ContextVar(

    "NARAD_PROGRESS_CALLBACK",

    default=None
)


# ======================================================================
# Preserve already-hardened implementations once.
# ======================================================================

if (
    "_NARAD_PRE_6_1F_BUILD_SELECTED_PATH_STACK"
    not in globals()
):

    _NARAD_PRE_6_1F_BUILD_SELECTED_PATH_STACK = (
        _narad_build_selected_path_stack
    )


if (
    "_NARAD_PRE_6_1F_ESTABLISH_KEY"
    not in globals()
):

    _NARAD_PRE_6_1F_ESTABLISH_KEY = (
        narad_qc_establish_key
    )


# ======================================================================
# WORKER CAPABILITY
# ======================================================================

def _narad_runtime_cpu_count():

    return max(
        1,
        int(
            os.cpu_count()
            or
            1
        )
    )


def _narad_applied_cpu_workers(
    requested_workers,
    parallel_quantum_lanes
):

    requested_workers = max(
        1,
        int(
            requested_workers
        )
    )


    parallel_quantum_lanes = max(
        1,
        int(
            parallel_quantum_lanes
        )
    )


    runtime_limit = (
        _narad_runtime_cpu_count()
    )


    controller_limit = max(
        1,
        int(
            NARAD_CONTROLLER_MAX_CPU_WORKERS
        )
    )


    # ==============================================================
    # IMPORTANT:
    #
    # Independent CPU workers are useful only when there are
    # independent physical lane trajectories available to compute.
    #
    # One physical lane remains one causal execution stream.
    # ==============================================================

    return max(

        1,

        min(
            requested_workers,
            parallel_quantum_lanes,
            runtime_limit,
            controller_limit
        )
    )


def _narad_execution_state():

    state = (
        _NARAD_EXECUTION_CONTEXT.get()
    )


    if state is None:

        return {

            "requested_cpu_workers":
                1,

            "requested_parallel_quantum_lanes":
                1,

            "applied_cpu_workers":
                1,

            "applied_parallel_quantum_lanes":
                1,

            "worker_backend":
                "SEQUENTIAL"
        }


    return state


# ======================================================================
# TELEMETRY
# ======================================================================

def _narad_safe_float(
    value
):

    if value is None:

        return None


    value = float(
        value
    )


    if not math.isfinite(
        value
    ):

        return None


    return value


def _narad_progress_payload(
    stage,
    qkd_config,
    summary=None,
    **extra
):

    payload = {

        "stage":
            str(
                stage
            ),

        "timestamp_unix":
            time.time(),

        "target_secret_bits":
            int(
                qkd_config.target_secret_bits
            ),

        "max_delivered_pairs":
            int(
                qkd_config.max_delivered_pairs
            )
    }


    if summary is not None:

        forecast = int(

            summary
            .forecast_extractable_bits
        )


        target = int(

            qkd_config
            .target_secret_bits
        )


        payload.update({

            "delivered_pairs":
                int(
                    summary.delivered_pairs
                ),

            "zz_bits":
                int(
                    summary.zz_key_basis_bits
                ),

            "xx_bits":
                int(
                    summary.xx_parameter_estimation_bits
                ),

            # Don't display "100% QBER" merely because no XX sample
            # exists yet.
            "observed_x_qber":
                (
                    _narad_safe_float(
                        summary.observed_x_qber
                    )

                    if (
                        summary
                        .xx_parameter_estimation_bits

                        >
                        0
                    )

                    else None
                ),

            "mean_predicted_qber":
                _narad_safe_float(
                    summary.mean_predicted_qber
                ),

            "phase_error_estimate":
                _narad_safe_float(
                    summary.phase_error_estimate
                ),

            "forecast_extractable_bits":
                forecast,

            "forecast_secret_fraction":
                float(

                    max(
                        0.0,

                        min(
                            1.0,

                            forecast
                            /
                            max(
                                1,
                                target
                            )
                        )
                    )
                ),

            "pair_budget_fraction":
                float(

                    max(
                        0.0,

                        min(
                            1.0,

                            summary.delivered_pairs
                            /
                            max(
                                1,
                                qkd_config
                                .max_delivered_pairs
                            )
                        )
                    )
                ),

            "collection_ready":
                bool(
                    summary.collection_ready
                ),

            "modeled_physical_time_s":
                _narad_safe_float(
                    summary.physical_time_s
                ),

            "elementary_generation_cycles":
                int(
                    summary
                    .elementary_generation_cycles
                ),

            "optical_mode_attempts":
                int(
                    summary
                    .optical_mode_attempts
                ),

            "bsm_attempts":
                int(
                    summary.bsm_attempts
                )
        })


    payload.update(
        extra
    )


    return payload


def _narad_emit_progress(
    stage,
    qkd_config,
    summary=None,
    **extra
):

    callback = (
        _NARAD_PROGRESS_CALLBACK.get()
    )


    if callback is None:

        return


    payload = (

        _narad_progress_payload(

            stage=(
                stage
            ),

            qkd_config=(
                qkd_config
            ),

            summary=(
                summary
            ),

            **extra
        )
    )


    # Telemetry must NEVER be able to kill the physics execution.
    try:

        callback(
            payload
        )

    except Exception as error:

        print(
            "[WARN] NARAD telemetry callback failed:",
            repr(
                error
            )
        )


# ======================================================================
# ADD ROUTE METADATA TO THE ALREADY-HARDENED STACK
#
# No physical behavior is changed.
#
# The metadata allows worker processes to reconstruct independent copies
# of this exact heterogeneous Policy-C route.
# ======================================================================

def _narad_build_selected_path_stack(
    network,
    candidate_path,
    simulation_seed
):

    stack = (

        _NARAD_PRE_6_1F_BUILD_SELECTED_PATH_STACK(

            network=(
                network
            ),

            candidate_path=(
                candidate_path
            ),

            simulation_seed=(
                simulation_seed
            )
        )
    )


    stack[
        "_narad_network"
    ] = network


    stack[
        "_narad_candidate_path"
    ] = candidate_path


    stack[
        "_narad_route_physical_seed"
    ] = int(
        simulation_seed
    )


    return stack


# ======================================================================
# ROUTED TRAJECTORY FARM
#
# Physical interpretation:
#
# each job =
#     one independent provisioned quantum-lane trajectory
#     over the exact heterogeneous selected route.
#
# CPU workers != physical lanes.
#
# CPU workers merely calculate the independent lane trajectories in
# parallel Python processes.
# ======================================================================

_NARAD_ROUTED_FARM_CONTEXT = None

_NARAD_ROUTED_WORKER_THREAD_LIMITER = None


def _narad_routed_worker_initializer():

    global _NARAD_ROUTED_WORKER_THREAD_LIMITER


    try:

        from threadpoolctl import (
            threadpool_limits
        )


        _NARAD_ROUTED_WORKER_THREAD_LIMITER = (

            threadpool_limits(
                limits=1
            )
        )


    except Exception:

        _NARAD_ROUTED_WORKER_THREAD_LIMITER = None


def _narad_routed_worker_simulate_pair(
    pair_index
):

    global _NARAD_ROUTED_FARM_CONTEXT


    if _NARAD_ROUTED_FARM_CONTEXT is None:

        raise RuntimeError(
            "NARAD routed farm context is not initialized."
        )


    pair_index = int(
        pair_index
    )


    context = (
        _NARAD_ROUTED_FARM_CONTEXT
    )


    wall_start = (
        time.perf_counter()
    )


    cpu_start = (
        time.process_time()
    )


    qkd_config = (
        context[
            "qkd_config"
        ]
    )


    qkd_token = (

        _NARAD_REQUEST_QKD_CONFIG.set(
            qkd_config
        )
    )


    try:

        physical_seed = (

            _narad_derive_seed(

                master_seed=(
                    context[
                        "physical_seed"
                    ]
                ),

                path_index=(
                    pair_index
                ),

                stream_tag=(
                    61_001
                )
            )
        )


        protocol_seed = (

            _narad_derive_seed(

                master_seed=(
                    context[
                        "protocol_seed"
                    ]
                ),

                path_index=(
                    pair_index
                ),

                stream_tag=(
                    61_002
                )
            )
        )


        stack = (

            _narad_build_selected_path_stack(

                network=(
                    context[
                        "network"
                    ]
                ),

                candidate_path=(
                    context[
                        "candidate_path"
                    ]
                ),

                simulation_seed=(
                    physical_seed
                )
            )
        )


        # The outer route has already authenticated successfully.
        #
        # Each independently reconstructed physical-lane simulator
        # still needs its own local authorization gate opened before
        # quantum operations are legal.
        _narad_authenticate_selected_path(
            stack
        )


        channel = (

            NARADQKDEntanglementChannel(

                stack=(
                    stack
                ),

                protocol_seed=(
                    protocol_seed
                ),

                measurement_latency_s=0.0
            )
        )


        local_record = (

            channel
            .deliver_and_measure(

                alice_basis=None,

                bob_basis=None,

                z_probability=(
                    qkd_config
                    .z_probability
                ),

                max_slots_per_pair=(
                    qkd_config
                    .max_slots_per_pair
                )
            )
        )


        record = replace(

            local_record,

            pair_index=(
                pair_index
            )
        )


        payload = (
            asdict(
                record
            )
        )


        del channel
        del stack


        gc.collect()


        return {

            "pair_index":
                pair_index,

            "worker_pid":
                os.getpid(),

            "worker_cpu_s":
                float(
                    time.process_time()
                    -
                    cpu_start
                ),

            "worker_wall_s":
                float(
                    time.perf_counter()
                    -
                    wall_start
                ),

            "payload":
                payload
        }


    finally:

        _NARAD_REQUEST_QKD_CONFIG.reset(
            qkd_token
        )


class NARADRoutedTrajectoryFarm:

    def __init__(
        self,
        workers,
        lookahead
    ):

        self.requested_workers = max(
            1,
            int(
                workers
            )
        )


        self.applied_workers = (
            self.requested_workers
        )


        self.lookahead = max(
            1,
            int(
                lookahead
            )
        )


        self.pool = None

        self.pending = {}

        self.submitted_indices = set()

        self.worker_pids = set()

        self.submitted_count = 0

        self.retrieved_count = 0

        self.direct_fallback = False

        self.closed = False


        if self.requested_workers > 1:

            try:

                context = (
                    mp.get_context(
                        "fork"
                    )
                )


                self.pool = (

                    context.Pool(

                        processes=(
                            self.requested_workers
                        ),

                        initializer=(
                            _narad_routed_worker_initializer
                        )
                    )
                )


            except Exception as error:

                print(
                    "[WARN] Routed process farm unavailable:",
                    repr(
                        error
                    )
                )

                print(
                    "[INFO] Falling back to one CPU worker while "
                    "preserving requested physical-lane semantics."
                )


                self.pool = None

                self.direct_fallback = True

                self.applied_workers = 1


        else:

            self.direct_fallback = True

            self.applied_workers = 1


    def submit(
        self,
        pair_index
    ):

        pair_index = int(
            pair_index
        )


        if (
            pair_index
            in
            self.submitted_indices
        ):

            return


        self.submitted_indices.add(
            pair_index
        )


        self.submitted_count += 1


        if self.pool is not None:

            self.pending[
                pair_index
            ] = (

                self.pool
                .apply_async(

                    _narad_routed_worker_simulate_pair,

                    (
                        pair_index,
                    )
                )
            )


    def submit_through(
        self,
        final_pair_index
    ):

        final_pair_index = int(
            final_pair_index
        )


        if final_pair_index < 0:

            return


        start = (

            max(
                self.submitted_indices
            )
            +
            1

            if self.submitted_indices

            else 0
        )


        for pair_index in range(

            start,

            final_pair_index
            +
            1
        ):

            self.submit(
                pair_index
            )


    def maintain_window(
        self,
        next_pair_index,
        maximum_pair_index
    ):

        final_needed = min(

            int(
                maximum_pair_index
            ),

            int(
                next_pair_index
            )
            +
            self.lookahead
            -
            1
        )


        self.submit_through(
            final_needed
        )


    def _switch_to_direct_fallback(
        self,
        error
    ):

        print(
            "[WARN] Routed worker process failed:",
            repr(
                error
            )
        )


        print(
            "[INFO] Continuing with one CPU worker. "
            "Physical quantum-lane count is unchanged."
        )


        if self.pool is not None:

            try:

                self.pool.terminate()

                self.pool.join()

            except Exception:

                pass


        self.pool = None

        self.pending = {}

        self.direct_fallback = True

        self.applied_workers = 1


    def get(
        self,
        pair_index
    ):

        pair_index = int(
            pair_index
        )


        if (
            pair_index
            not in
            self.submitted_indices
        ):

            self.submit(
                pair_index
            )


        if self.pool is None:

            result = (

                _narad_routed_worker_simulate_pair(
                    pair_index
                )
            )


        else:

            try:

                result = (

                    self.pending[
                        pair_index
                    ]
                    .get()
                )


            except Exception as error:

                self._switch_to_direct_fallback(
                    error
                )


                result = (

                    _narad_routed_worker_simulate_pair(
                        pair_index
                    )
                )


        self.worker_pids.add(

            int(
                result[
                    "worker_pid"
                ]
            )
        )


        self.retrieved_count += 1


        self.pending.pop(
            pair_index,
            None
        )


        return (

            NARADQKDPairMeasurement(

                **result[
                    "payload"
                ]
            )
        )


    def snapshot(
        self
    ):

        pool_processes = 0


        if self.pool is not None:

            try:

                pool_processes = len(
                    self.pool._pool
                )

            except Exception:

                pool_processes = (
                    self.applied_workers
                )


        return {

            "requested_cpu_workers":
                self.requested_workers,

            "applied_cpu_workers":
                self.applied_workers,

            "worker_processes_created":
                pool_processes,

            "worker_processes_seen":
                len(
                    self.worker_pids
                ),

            "jobs_submitted":
                self.submitted_count,

            "trajectories_retrieved":
                self.retrieved_count,

            "jobs_pending":
                len(
                    self.pending
                ),

            "direct_fallback":
                bool(
                    self.direct_fallback
                )
        }


    def shutdown(
        self
    ):

        if self.closed:

            return


        if self.pool is not None:

            try:

                # Speculative trajectories are no longer relevant
                # once the requested key exists.
                self.pool.terminate()

                self.pool.join()

            except Exception:

                pass


        self.closed = True


# ======================================================================
# POST-PROCESSING HELPER
# ======================================================================

_NARAD_RETRYABLE_POSTPROCESS_PREFIXES = (

    "INSUFFICIENT_PARAMETER_ESTIMATION",

    "CASCADE_RECONCILIATION_FAILED",

    "INSUFFICIENT_POST_EC_ENTROPY"
)


def _narad_try_actual_postprocessing(
    measurements,
    summary,
    qkd_config,
    postprocess_master_seed
):

    pair_number = int(
        summary.delivered_pairs
    )


    postprocess_rng = (

        np.random.default_rng(

            np.random.SeedSequence(
                [

                    int(
                        postprocess_master_seed
                    ),

                    pair_number,

                    int(
                        qkd_config
                        .target_secret_bits
                    )
                ]
            )
        )
    )


    try:

        result = (

            phase3k_distill_configurable_bbm92_key(

                measurements=(
                    tuple(
                        measurements
                    )
                ),

                collection_summary=(
                    summary
                ),

                config=(
                    qkd_config
                ),

                postprocess_rng=(
                    postprocess_rng
                )
            )
        )


    except RuntimeError as error:

        message = str(
            error
        )


        if not any(

            message.startswith(
                prefix
            )

            for prefix
            in
            _NARAD_RETRYABLE_POSTPROCESS_PREFIXES
        ):

            raise


        return (
            None,
            message
        )


    if (

        result.success

        and

        result.alice_key_bytes

        ==

        result.bob_key_bytes
    ):

        return (
            result,
            None
        )


    return (
        None,
        "POSTPROCESS_DID_NOT_PRODUCE_MATCHING_SECRET"
    )


# ======================================================================
# SEQUENTIAL ROUTED EXECUTOR WITH LIVE TELEMETRY
#
# Physics is equivalent to the existing one-lane routed executor.
# ======================================================================

def _narad_run_qkd_sequential_with_progress(
    stack,
    qkd_config,
    protocol_seed,
    postprocess_master_seed,
    verbose
):

    execution_state = (
        _narad_execution_state()
    )


    execution_state[
        "applied_cpu_workers"
    ] = 1


    execution_state[
        "applied_parallel_quantum_lanes"
    ] = 1


    execution_state[
        "worker_backend"
    ] = "SEQUENTIAL_CAUSAL_ROUTE"


    qkd_channel = (

        NARADQKDEntanglementChannel(

            stack=(
                stack
            ),

            protocol_seed=(
                int(
                    protocol_seed
                )
            ),

            measurement_latency_s=0.0
        )
    )


    measurements = []

    final_summary = None


    _narad_emit_progress(

        "QKD_COLLECTION",

        qkd_config,

        route=(
            stack[
                "_narad_candidate_path"
            ]
            .display_path
        ),

        applied_cpu_workers=1,

        applied_parallel_quantum_lanes=1
    )


    for pair_number in range(

        1,

        int(
            qkd_config.max_delivered_pairs
        )
        +
        1
    ):

        measurement = (

            qkd_channel
            .deliver_and_measure(

                alice_basis=None,

                bob_basis=None,

                z_probability=(
                    qkd_config.z_probability
                ),

                max_slots_per_pair=(
                    qkd_config
                    .max_slots_per_pair
                )
            )
        )


        measurements.append(
            measurement
        )


        if not _narad_should_check_qkd_block(

            pair_number=(
                pair_number
            ),

            config=(
                qkd_config
            )
        ):

            continue


        summary = (

            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    qkd_config
                )
            )
        )


        final_summary = (
            summary
        )


        _narad_emit_progress(

            "QKD_COLLECTION",

            qkd_config,

            summary=(
                summary
            ),

            applied_cpu_workers=1,

            applied_parallel_quantum_lanes=1
        )


        if not summary.collection_ready:

            continue


        _narad_emit_progress(

            "QKD_POSTPROCESS",

            qkd_config,

            summary=(
                summary
            )
        )


        (
            final_result,
            retry_reason
        ) = (

            _narad_try_actual_postprocessing(

                measurements=(
                    measurements
                ),

                summary=(
                    summary
                ),

                qkd_config=(
                    qkd_config
                ),

                postprocess_master_seed=(
                    postprocess_master_seed
                )
            )
        )


        if final_result is not None:

            _narad_emit_progress(

                "COMPLETE",

                qkd_config,

                summary=(
                    summary
                ),

                final_secret_bits=int(
                    final_result.final_secret_bits
                )
            )


            return (
                final_result,
                tuple(
                    measurements
                ),
                summary
            )


        _narad_emit_progress(

            "QKD_COLLECTION",

            qkd_config,

            summary=(
                summary
            ),

            postprocess_retry_reason=(
                retry_reason
            )
        )


    if final_summary is None:

        final_summary = (

            phase3j_bbm92_collection_summary(

                measurements=(
                    measurements
                ),

                config=(
                    qkd_config
                )
            )
        )


    raise NARADQCServiceError(

        "QKD pair budget exhausted before actual "
        "post-processing produced the requested secret key. "
        f"Delivered={len(measurements)}, "
        f"forecast_extractable="
        f"{final_summary.forecast_extractable_bits}."
    )


# ======================================================================
# PARALLEL PHYSICAL-LANE EXECUTOR
#
# The physical lanes are independent provisioned copies of the exact
# selected heterogeneous route.
#
# Modeled network time follows completion events:
#
#       NOT sum(all trajectory times)
#
# but concurrent lane completion order.
# ======================================================================

def _narad_run_qkd_parallel_routed_lanes(
    stack,
    qkd_config,
    protocol_seed,
    postprocess_master_seed,
    verbose
):

    global _NARAD_ROUTED_FARM_CONTEXT


    execution_state = (
        _narad_execution_state()
    )


    requested_workers = int(

        execution_state[
            "requested_cpu_workers"
        ]
    )


    requested_lanes = int(

        execution_state[
            "requested_parallel_quantum_lanes"
        ]
    )


    applied_workers = (

        _narad_applied_cpu_workers(

            requested_workers,

            requested_lanes
        )
    )


    applied_lanes = max(
        1,
        requested_lanes
    )


    execution_state[
        "applied_cpu_workers"
    ] = applied_workers


    execution_state[
        "applied_parallel_quantum_lanes"
    ] = applied_lanes


    execution_state[
        "worker_backend"
    ] = "ROUTED_TRAJECTORY_FARM"


    candidate_path = (
        stack[
            "_narad_candidate_path"
        ]
    )


    _NARAD_ROUTED_FARM_CONTEXT = {

        "network":
            stack[
                "_narad_network"
            ],

        "candidate_path":
            candidate_path,

        "qkd_config":
            qkd_config,

        "physical_seed":
            int(
                stack[
                    "_narad_route_physical_seed"
                ]
            ),

        "protocol_seed":
            int(
                protocol_seed
            )
    }


    lookahead = max(

        applied_lanes,

        applied_workers
        *
        4
    )


    farm = (

        NARADRoutedTrajectoryFarm(

            workers=(
                applied_workers
            ),

            lookahead=(
                lookahead
            )
        )
    )


    # Pool creation may itself fall back to direct mode.
    execution_state[
        "applied_cpu_workers"
    ] = (
        farm.applied_workers
    )


    max_pairs = int(
        qkd_config.max_delivered_pairs
    )


    active_lanes = min(
        applied_lanes,
        max_pairs
    )


    active_heap = []

    measurements = []

    next_pair_index = 0

    final_summary = None

    last_physical_finish_s = 0.0


    try:

        initial_prefetch_end = min(

            max_pairs
            -
            1,

            active_lanes
            +
            lookahead
            -
            1
        )


        farm.submit_through(
            initial_prefetch_end
        )


        # ==========================================================
        # Populate each physical lane at modeled t = 0.
        # ==========================================================

        import heapq


        for lane_id in range(
            active_lanes
        ):

            record = (

                farm.get(
                    next_pair_index
                )
            )


            finish_s = float(
                record.delivery_physical_time_s
            )


            heapq.heappush(

                active_heap,

                (
                    finish_s,
                    lane_id,
                    next_pair_index,
                    record
                )
            )


            next_pair_index += 1


            farm.maintain_window(

                next_pair_index=(
                    next_pair_index
                ),

                maximum_pair_index=(
                    max_pairs
                    -
                    1
                )
            )


        _narad_emit_progress(

            "QKD_COLLECTION",

            qkd_config,

            route=(
                candidate_path
                .display_path
            ),

            applied_cpu_workers=(
                farm.applied_workers
            ),

            applied_parallel_quantum_lanes=(
                active_lanes
            ),

            worker_state=(
                farm.snapshot()
            )
        )


        # ==========================================================
        # Continuous completion-event execution.
        # ==========================================================

        while active_heap:

            (
                physical_finish_s,
                lane_id,
                trajectory_index,
                record
            ) = (

                heapq.heappop(
                    active_heap
                )
            )


            last_physical_finish_s = float(
                physical_finish_s
            )


            completion_index = len(
                measurements
            )


            completed_record = replace(

                record,

                pair_index=(
                    completion_index
                )
            )


            measurements.append(
                completed_record
            )


            delivered = len(
                measurements
            )


            should_check = (

                _narad_should_check_qkd_block(

                    pair_number=(
                        delivered
                    ),

                    config=(
                        qkd_config
                    )
                )
            )


            if should_check:

                raw_summary = (

                    phase3j_bbm92_collection_summary(

                        measurements=(
                            measurements
                        ),

                        config=(
                            qkd_config
                        )
                    )
                )


                summary = replace(

                    raw_summary,

                    physical_time_s=(
                        float(
                            physical_finish_s
                        )
                    )
                )


                final_summary = (
                    summary
                )


                execution_state[
                    "applied_cpu_workers"
                ] = (
                    farm.applied_workers
                )


                _narad_emit_progress(

                    "QKD_COLLECTION",

                    qkd_config,

                    summary=(
                        summary
                    ),

                    applied_cpu_workers=(
                        farm.applied_workers
                    ),

                    applied_parallel_quantum_lanes=(
                        active_lanes
                    ),

                    worker_state=(
                        farm.snapshot()
                    )
                )


                if summary.collection_ready:

                    _narad_emit_progress(

                        "QKD_POSTPROCESS",

                        qkd_config,

                        summary=(
                            summary
                        ),

                        worker_state=(
                            farm.snapshot()
                        )
                    )


                    (
                        final_result,
                        retry_reason
                    ) = (

                        _narad_try_actual_postprocessing(

                            measurements=(
                                measurements
                            ),

                            summary=(
                                summary
                            ),

                            qkd_config=(
                                qkd_config
                            ),

                            postprocess_master_seed=(
                                postprocess_master_seed
                            )
                        )
                    )


                    if final_result is not None:

                        _narad_emit_progress(

                            "COMPLETE",

                            qkd_config,

                            summary=(
                                summary
                            ),

                            final_secret_bits=int(
                                final_result
                                .final_secret_bits
                            ),

                            worker_state=(
                                farm.snapshot()
                            )
                        )


                        return (

                            final_result,

                            tuple(
                                measurements
                            ),

                            summary
                        )


                    _narad_emit_progress(

                        "QKD_COLLECTION",

                        qkd_config,

                        summary=(
                            summary
                        ),

                        postprocess_retry_reason=(
                            retry_reason
                        )
                    )


            # ======================================================
            # Refill the physical lane which just became free.
            # ======================================================

            if next_pair_index < max_pairs:

                next_record = (

                    farm.get(
                        next_pair_index
                    )
                )


                next_finish_s = (

                    float(
                        physical_finish_s
                    )

                    +

                    float(
                        next_record
                        .delivery_physical_time_s
                    )
                )


                heapq.heappush(

                    active_heap,

                    (
                        next_finish_s,
                        lane_id,
                        next_pair_index,
                        next_record
                    )
                )


                next_pair_index += 1


                farm.maintain_window(

                    next_pair_index=(
                        next_pair_index
                    ),

                    maximum_pair_index=(
                        max_pairs
                        -
                        1
                    )
                )


        if final_summary is None:

            raw_summary = (

                phase3j_bbm92_collection_summary(

                    measurements=(
                        measurements
                    ),

                    config=(
                        qkd_config
                    )
                )
            )


            final_summary = replace(

                raw_summary,

                physical_time_s=(
                    float(
                        last_physical_finish_s
                    )
                )
            )


        raise NARADQCServiceError(

            "QKD pair budget exhausted before actual "
            "post-processing produced the requested secret key. "
            f"Delivered={len(measurements)}, "
            f"forecast_extractable="
            f"{final_summary.forecast_extractable_bits}."
        )


    finally:

        execution_state[
            "applied_cpu_workers"
        ] = (
            farm.applied_workers
        )


        execution_state[
            "worker_snapshot"
        ] = (
            farm.snapshot()
        )


        farm.shutdown()


        _NARAD_ROUTED_FARM_CONTEXT = None


# ======================================================================
# GLOBAL QKD EXECUTOR PATCH
# ======================================================================

def _narad_run_qkd_until_actual_key(
    stack,
    qkd_config,
    protocol_seed,
    postprocess_master_seed,
    verbose
):

    execution_state = (
        _narad_execution_state()
    )


    requested_lanes = int(

        execution_state[
            "requested_parallel_quantum_lanes"
        ]
    )


    if requested_lanes <= 1:

        return (

            _narad_run_qkd_sequential_with_progress(

                stack=(
                    stack
                ),

                qkd_config=(
                    qkd_config
                ),

                protocol_seed=(
                    protocol_seed
                ),

                postprocess_master_seed=(
                    postprocess_master_seed
                ),

                verbose=(
                    verbose
                )
            )
        )


    return (

        _narad_run_qkd_parallel_routed_lanes(

            stack=(
                stack
            ),

            qkd_config=(
                qkd_config
            ),

            protocol_seed=(
                protocol_seed
            ),

            postprocess_master_seed=(
                postprocess_master_seed
            ),

            verbose=(
                verbose
            )
        )
    )


# ======================================================================
# PUBLIC ESTABLISHMENT WRAPPER
#
# Existing Phase-5 orchestration remains intact.
# Only execution/progress context is added around it.
# ======================================================================

def narad_qc_establish_key(
    network,
    sender_name,
    receiver_name,
    *,
    qkd_config=PHASE3J_BIASED_BBM92_CONFIG,
    max_hops=None,
    physical_seed=NARAD_API_DEFAULT_PHYSICAL_SEED,
    protocol_seed=NARAD_API_DEFAULT_PROTOCOL_SEED,
    postprocess_seed=NARAD_API_DEFAULT_POSTPROCESS_SEED,
    verbose=True,
    return_details=False,
    cpu_workers=1,
    parallel_quantum_lanes=1,
    progress_callback=None,
    execution_audit=None
):

    requested_workers = max(
        1,
        int(
            cpu_workers
        )
    )


    requested_lanes = max(
        1,
        int(
            parallel_quantum_lanes
        )
    )


    if execution_audit is None:

        execution_audit = {}


    execution_audit.update({

        "requested_cpu_workers":
            requested_workers,

        "requested_parallel_quantum_lanes":
            requested_lanes,

        "applied_cpu_workers":
            _narad_applied_cpu_workers(
                requested_workers,
                requested_lanes
            ),

        "applied_parallel_quantum_lanes":
            requested_lanes,

        "worker_backend":
            (
                "ROUTED_TRAJECTORY_FARM"

                if requested_lanes > 1

                else "SEQUENTIAL_CAUSAL_ROUTE"
            )
    })


    execution_token = (

        _NARAD_EXECUTION_CONTEXT.set(
            execution_audit
        )
    )


    progress_token = (

        _NARAD_PROGRESS_CALLBACK.set(
            progress_callback
        )
    )


    try:

        return (

            _NARAD_PRE_6_1F_ESTABLISH_KEY(

                network=(
                    network
                ),

                sender_name=(
                    sender_name
                ),

                receiver_name=(
                    receiver_name
                ),

                qkd_config=(
                    qkd_config
                ),

                max_hops=(
                    max_hops
                ),

                physical_seed=(
                    physical_seed
                ),

                protocol_seed=(
                    protocol_seed
                ),

                postprocess_seed=(
                    postprocess_seed
                ),

                verbose=(
                    verbose
                ),

                return_details=(
                    return_details
                )
            )
        )


    finally:

        _NARAD_PROGRESS_CALLBACK.reset(
            progress_token
        )


        _NARAD_EXECUTION_CONTEXT.reset(
            execution_token
        )


print()
print("[PASS] Request-scoped CPU worker configuration installed.")
print("[PASS] Routed multi-lane trajectory farm installed.")
print("[PASS] Single causal lane remains scientifically single-worker.")
print("[PASS] BBM92 progress telemetry installed.")
print("[PASS] ZZ/XX/QBER/forecast telemetry comes from real collection summaries.")

In [ ]:
# @title API Worker + Progress Wiring

from contextvars import ContextVar
import time


print("=" * 100)
print("API WORKER + PROGRESS WIRING")
print("=" * 100)


# ======================================================================
# ACTIVE IDEMPOTENCY KEY
#
# Lets the synchronous establishment path publish telemetry into the
# correct 6.1D status record without changing the public API contract.
# ======================================================================

_NARAD_ACTIVE_IDEMPOTENCY_KEY = ContextVar(

    "NARAD_ACTIVE_IDEMPOTENCY_KEY",

    default=None
)


# ======================================================================
# Preserve 6.1D idempotency function.
# ======================================================================

if (
    "_NARAD_PRE_6_1F_ESTABLISH_IDEMPOTENT"
    not in globals()
):

    _NARAD_PRE_6_1F_ESTABLISH_IDEMPOTENT = (
        _narad_api_establish_idempotent
    )


if (
    "_NARAD_PRE_6_1F_PUBLIC_IDEMPOTENCY_RECORD"
    not in globals()
):

    _NARAD_PRE_6_1F_PUBLIC_IDEMPOTENCY_RECORD = (
        _narad_api_public_idempotency_record
    )


# ======================================================================
# Context wrapper around 6.1D.
# ======================================================================

def _narad_api_establish_idempotent(
    request,
    idempotency_key=None
):

    normalized_key = (

        _narad_api_normalize_idempotency_key(
            idempotency_key
        )
    )


    token = (

        _NARAD_ACTIVE_IDEMPOTENCY_KEY.set(
            normalized_key
        )
    )


    try:

        return (

            _NARAD_PRE_6_1F_ESTABLISH_IDEMPOTENT(

                request,

                idempotency_key=(
                    normalized_key
                )
            )
        )


    finally:

        _NARAD_ACTIVE_IDEMPOTENCY_KEY.reset(
            token
        )


# ======================================================================
# TELEMETRY SINK
# ======================================================================

def _narad_api_progress_callback_for_current_request():

    idempotency_key = (

        _NARAD_ACTIVE_IDEMPOTENCY_KEY.get()
    )


    if idempotency_key is None:

        return None


    def publish(
        payload
    ):

        with NARAD_API_IDEMPOTENCY_LOCK:

            record = (

                NARAD_API_IDEMPOTENCY_RECORDS.get(
                    idempotency_key
                )
            )


            if record is None:

                return


            record[
                "progress"
            ] = dict(
                payload
            )


            record[
                "updated_at_unix"
            ] = time.time()


    return publish


# ======================================================================
# PUBLIC STATUS RECORD NOW INCLUDES LIVE PROGRESS
# ======================================================================

def _narad_api_public_idempotency_record(
    key,
    record
):

    output = (

        _NARAD_PRE_6_1F_PUBLIC_IDEMPOTENCY_RECORD(

            key,

            record
        )
    )


    progress = (

        record.get(
            "progress"
        )
    )


    if progress is not None:

        output[
            "progress"
        ] = dict(
            progress
        )


    else:

        output[
            "progress"
        ] = None


    return output


# ======================================================================
# REPLACE ONE-TIME EXECUTION IMPLEMENTATION
#
# Existing 6.1D idempotency wrapper dynamically calls this symbol, so
# no duplicate POST route is needed.
# ======================================================================

def _narad_api_execute_establishment_once(
    request
):

    (
        canonical_sender,
        canonical_receiver
    ) = (

        _narad_api_canonical_endpoints(

            request.sender,

            request.receiver
        )
    )


    qkd_config = (

        narad_api_make_qkd_config(
            request.qkd
        )
    )


    requested_cpu_workers = int(
        request.execution.cpu_workers
    )


    requested_parallel_lanes = int(
        request.execution.parallel_quantum_lanes
    )


    execution_audit = {}


    progress_callback = (

        _narad_api_progress_callback_for_current_request()
    )


    if progress_callback is not None:

        progress_callback({

            "stage":
                "ROUTE_SELECTION_AND_AUTHENTICATION",

            "timestamp_unix":
                time.time(),

            "requested_cpu_workers":
                requested_cpu_workers,

            "requested_parallel_quantum_lanes":
                requested_parallel_lanes,

            "target_secret_bits":
                int(
                    qkd_config.target_secret_bits
                )
        })


    # ==================================================================
    # 6.1D global establishment serialization remains intact.
    #
    # One NARAD network establishment at a time;
    # internal CPU workers may parallelize that one request.
    # ==================================================================

    with NARAD_API_ESTABLISHMENT_EXECUTION_LOCK:

        result = (

            narad_qc_establish_key(

                network=(
                    phase4_network
                ),

                sender_name=(
                    canonical_sender
                ),

                receiver_name=(
                    canonical_receiver
                ),

                qkd_config=(
                    qkd_config
                ),

                max_hops=(
                    request.execution.max_hops
                ),

                physical_seed=(
                    request.execution.physical_seed
                ),

                protocol_seed=(
                    request.execution.protocol_seed
                ),

                postprocess_seed=(
                    request.execution.postprocess_seed
                ),

                verbose=False,

                return_details=True,

                cpu_workers=(
                    requested_cpu_workers
                ),

                parallel_quantum_lanes=(
                    requested_parallel_lanes
                ),

                progress_callback=(
                    progress_callback
                ),

                execution_audit=(
                    execution_audit
                )
            )
        )


    _narad_api_prune_expired_sessions()


    session_id = (
        str(
            uuid.uuid4()
        )
    )


    now = time.time()


    with NARAD_API_SESSION_LOCK:

        NARAD_API_SESSIONS[
            session_id
        ] = {

            "session_id":
                session_id,

            "created_at_unix":
                now,

            "last_accessed_at_unix":
                now,

            "sender":
                result.sender,

            "receiver":
                result.receiver,

            "key":
                bytes(
                    result.key
                ),

            "qkd_config":
                qkd_config,

            "result":
                result,

            "execution":
                dict(
                    execution_audit
                ),

            "message_count":
                0
        }


    if progress_callback is not None:

        progress_callback({

            "stage":
                "SESSION_ESTABLISHED",

            "timestamp_unix":
                time.time(),

            "target_secret_bits":
                int(
                    qkd_config.target_secret_bits
                ),

            "final_secret_bits":
                int(
                    result.final_secret_bits
                ),

            "delivered_pairs":
                int(
                    result.delivered_pairs
                ),

            "zz_bits":
                int(
                    result.zz_raw_key_bits
                ),

            "xx_bits":
                int(
                    result.xx_parameter_estimation_bits
                ),

            "forecast_secret_fraction":
                1.0,

            "pair_budget_fraction":
                float(

                    result.delivered_pairs
                    /
                    max(
                        1,
                        qkd_config.max_delivered_pairs
                    )
                ),

            "applied_cpu_workers":
                int(
                    execution_audit[
                        "applied_cpu_workers"
                    ]
                ),

            "applied_parallel_quantum_lanes":
                int(
                    execution_audit[
                        "applied_parallel_quantum_lanes"
                    ]
                )
        })


    return {

        "session_id":
            session_id,

        "status":
            "ESTABLISHED",

        "sender":
            result.sender,

        "receiver":
            result.receiver,

        "session_ttl_s":
            NARAD_API_SESSION_TTL_S,

        "result":
            narad_api_public_result(
                result
            ),

        "execution": {

            **execution_audit,

            "runtime_logical_cpus":
                _narad_runtime_cpu_count(),

            "controller_worker_limit":
                int(
                    NARAD_CONTROLLER_MAX_CPU_WORKERS
                ),

            "establishment_concurrency":
                "SERIALIZED_BETWEEN_SESSIONS",

            "note": (
                "CPU workers parallelize independent routed "
                "quantum-lane trajectories. CPU worker count and "
                "physical quantum-lane count are distinct controls."
            )
        }
    }


# ======================================================================
# REPLACE /capabilities
#
# The old response intentionally said applied workers = 1 because that
# was true before this patch. It would now be stale.
# ======================================================================

_narad_api_remove_route(
    "/capabilities",
    "GET"
)


@app.get("/capabilities")
def capabilities():

    return {

        "routing": {

            "policy":
                "Policy C",

            "dynamic_sender_receiver":
                True,

            "max_hops":
                True,

            "ranked_route_fallback":
                True
        },


        "authentication": {

            "algorithm":
                "ML-DSA-65",

            "fresh_per_route":
                True,

            "hard_quantum_authorization_gate":
                True
        },


        "qkd": {

            "protocol":
                "Biased BBM92",

            "live_progress":
                True,

            "progress_fields": [

                "delivered_pairs",
                "zz_bits",
                "xx_bits",
                "observed_x_qber",
                "phase_error_estimate",
                "forecast_extractable_bits",
                "forecast_secret_fraction",
                "collection_ready",
                "modeled_physical_time_s"
            ]
        },


        "execution": {

            "heterogeneous_routed_executor": {

                "parallel_cpu_workers_supported":
                    True,

                "parallel_quantum_lanes_supported":
                    True,

                "runtime_logical_cpus":
                    _narad_runtime_cpu_count(),

                "controller_worker_limit":
                    int(
                        NARAD_CONTROLLER_MAX_CPU_WORKERS
                    ),

                "worker_rule":
                    (
                        "applied_cpu_workers = min("
                        "requested_cpu_workers, "
                        "requested_parallel_quantum_lanes, "
                        "runtime_logical_cpus, "
                        "controller_worker_limit)"
                    )
            },

            "session_establishments":
                "SERIALIZED",

            "requested_worker_fields_applied":
                True
        },


        "messaging": {

            "cipher":
                "AES-GCM",

            "plaintext_input":
                "UTF-8",

            "ciphertext_transport":
                "Base64",

            "raw_key_exposed":
                False
        }
    }


# ======================================================================
# OPTIONAL CONVENIENCE ENDPOINT FOR GUI
# ======================================================================

_narad_api_remove_route(
    "/runtime/capacity",
    "GET"
)


@app.get("/runtime/capacity")
def runtime_capacity():

    maximum = min(

        _narad_runtime_cpu_count(),

        int(
            NARAD_CONTROLLER_MAX_CPU_WORKERS
        )
    )


    return {

        "runtime_logical_cpus":
            _narad_runtime_cpu_count(),

        "controller_worker_limit":
            int(
                NARAD_CONTROLLER_MAX_CPU_WORKERS
            ),

        "max_safe_cpu_workers":
            int(
                maximum
            ),

        "supported_cpu_worker_values":
            list(
                range(
                    1,
                    maximum
                    +
                    1
                )
            ),

        "note": (
            "CPU worker count is computational parallelism. "
            "parallel_quantum_lanes is physical simulated "
            "hardware parallelism."
        )
    }


NARAD_API_PARALLEL_PROGRESS_VERSION = (
    "6.1F"
)


print()
print("[PASS] Requested CPU workers now propagate into routed execution.")
print("[PASS] /capabilities now reports real worker support.")
print("[PASS] /runtime/capacity exposes the controller/runtime limit.")
print("[PASS] Establishment status now carries live QKD telemetry.")
print("[PASS] Session-to-session serialization remains intact.")

In [ ]:
# @title Parallelism + Progress Regression Tests

from types import SimpleNamespace
import time


print("=" * 100)
print("PARALLELISM + PROGRESS REGRESSION TESTS")
print("=" * 100)


# ======================================================================
# TEST 1 — controller/runtime worker cap
# ======================================================================

_runtime_cpus = (
    _narad_runtime_cpu_count()
)


assert (
    _runtime_cpus
    >=
    1
)


assert (
    NARAD_CONTROLLER_MAX_CPU_WORKERS
    >=
    1
)


print(
    "[PASS] Runtime/controller worker limits are valid."
)


# ======================================================================
# TEST 2 — four requested workers are actually eligible when lanes allow
# ======================================================================

_expected = min(

    4,

    8,

    _runtime_cpus,

    NARAD_CONTROLLER_MAX_CPU_WORKERS
)


assert (

    _narad_applied_cpu_workers(
        4,
        8
    )

    ==

    _expected
)


print(
    f"[PASS] Request workers=4, lanes=8 -> applied workers={_expected}."
)


# ======================================================================
# TEST 3 — workers do NOT silently create physical lanes
# ======================================================================

assert (

    _narad_applied_cpu_workers(
        4,
        1
    )

    ==

    1
)


print(
    "[PASS] workers=4 with one physical lane remains one-worker causal execution."
)


# ======================================================================
# TEST 4 — telemetry payload uses genuine BBM92 counters
# ======================================================================

_fake_summary = SimpleNamespace(

    delivered_pairs=320,

    zz_key_basis_bits=205,

    xx_parameter_estimation_bits=17,

    observed_x_qber=0.038,

    mean_predicted_qber=0.041,

    phase_error_estimate=0.051,

    forecast_extractable_bits=71,

    collection_ready=False,

    physical_time_s=2.75,

    elementary_generation_cycles=1550,

    optical_mode_attempts=8220,

    bsm_attempts=640
)


_fake_config = SimpleNamespace(

    target_secret_bits=128,

    max_delivered_pairs=1200
)


_payload = (

    _narad_progress_payload(

        stage="QKD_COLLECTION",

        qkd_config=(
            _fake_config
        ),

        summary=(
            _fake_summary
        )
    )
)


assert (
    _payload[
        "delivered_pairs"
    ]
    ==
    320
)


assert (
    _payload[
        "zz_bits"
    ]
    ==
    205
)


assert (
    _payload[
        "xx_bits"
    ]
    ==
    17
)


assert (
    _payload[
        "forecast_extractable_bits"
    ]
    ==
    71
)


assert abs(

    _payload[
        "forecast_secret_fraction"
    ]

    -

    (
        71
        /
        128
    )

) < 1e-12


print(
    "[PASS] QKD progress payload reports real BBM92 accumulation state."
)


# ======================================================================
# TEST 5 — idempotency progress registry wiring
# ======================================================================

_test_progress_key = (
    "__narad_6_1f_progress_test__"
)


_now = time.time()


with NARAD_API_IDEMPOTENCY_LOCK:

    NARAD_API_IDEMPOTENCY_RECORDS[
        _test_progress_key
    ] = {

        "fingerprint":
            "test",

        "status":
            "IN_PROGRESS",

        "created_at_unix":
            _now,

        "updated_at_unix":
            _now,

        "event":
            threading.Event(),

        "response":
            None,

        "error":
            None
    }


_token = (

    _NARAD_ACTIVE_IDEMPOTENCY_KEY.set(
        _test_progress_key
    )
)


try:

    _callback = (

        _narad_api_progress_callback_for_current_request()
    )


    assert callable(
        _callback
    )


    _callback(
        _payload
    )


finally:

    _NARAD_ACTIVE_IDEMPOTENCY_KEY.reset(
        _token
    )


with NARAD_API_IDEMPOTENCY_LOCK:

    _test_record = (

        NARAD_API_IDEMPOTENCY_RECORDS[
            _test_progress_key
        ]
    )


    assert (
        _test_record[
            "progress"
        ][
            "zz_bits"
        ]

        ==
        205
    )


    _public = (

        _narad_api_public_idempotency_record(

            _test_progress_key,

            _test_record
        )
    )


assert (
    _public[
        "progress"
    ][
        "xx_bits"
    ]

    ==
    17
)


print(
    "[PASS] Live telemetry propagates into GET establishment status."
)


# ======================================================================
# Cleanup fake record.
# ======================================================================

with NARAD_API_IDEMPOTENCY_LOCK:

    NARAD_API_IDEMPOTENCY_RECORDS.pop(

        _test_progress_key,

        None
    )


# ======================================================================
# TEST 6 — API routes occur exactly once
# ======================================================================

for (
    path,
    method
) in (

    (
        "/capabilities",
        "GET"
    ),

    (
        "/runtime/capacity",
        "GET"
    ),

    (
        "/session/establish",
        "POST"
    ),

    (
        "/establishment/{idempotency_key}",
        "GET"
    )
):

    matches = [

        route

        for route
        in app.routes

        if (

            getattr(
                route,
                "path",
                None
            )

            ==

            path

            and

            method
            in
            (
                getattr(
                    route,
                    "methods",
                    set()
                )

                or

                set()
            )
        )
    ]


    assert (
        len(
            matches
        )
        ==
        1
    ), (
        f"{method} {path} appears "
        f"{len(matches)} times."
    )


print(
    "[PASS] Worker/progress API routes are registered exactly once."
)


# ======================================================================
# TEST 7 — stack patch still points through previously hardened builder
# ======================================================================

assert callable(
    _NARAD_PRE_6_1F_BUILD_SELECTED_PATH_STACK
)


assert callable(
    _NARAD_PRE_6_1F_ESTABLISH_KEY
)


assert callable(
    _narad_run_qkd_until_actual_key
)


print(
    "[PASS] Existing physics/QBER hardening remains underneath this."
)


print("\n" + "=" * 100)
print("REGRESSION SUITE PASSED")
print("=" * 100)

In [ ]:
# @title Start Temporary Cloudflare Tunnel

import json
import queue
import re
import socket
import subprocess
import threading
import time
import urllib.error
import urllib.request


print("=" * 100)
print("NARAD-QC CLOUDFLARE QUICK TUNNEL")
print("=" * 100)


# ======================================================================
# 1. Confirm LOCAL API is genuinely ready first
# ======================================================================

_local_ready = (

    _narad_http_json(

        NARAD_API_LOCAL_URL
        +
        "/ready",

        timeout_s=3.0
    )
)


if (

    not _local_ready.get(
        "ready"
    )

    or

    _local_ready.get(
        "runtime_generation"
    )

    !=

    NARAD_API_RUNTIME_GENERATION
):

    raise RuntimeError(

        "Current local NARAD-QC API is not ready. "
        "Cloudflare tunnel will not be started."
    )


print(
    "[PASS] Local NARAD-QC runtime is ready."
)


# ======================================================================
# 2. Shut down previous tunnel created by notebook
# ======================================================================

_previous_cloudflare = globals().get(

    "cloudflare_process"
)


if (

    _previous_cloudflare is not None

    and

    _previous_cloudflare.poll()
    is None
):

    print(
        "[INFO] Terminating previous Cloudflare tunnel..."
    )


    _previous_cloudflare.terminate()


    try:

        _previous_cloudflare.wait(
            timeout=5.0
        )


    except subprocess.TimeoutExpired:

        _previous_cloudflare.kill()

        _previous_cloudflare.wait(
            timeout=5.0
        )


# ======================================================================
# 3. Start fresh Quick Tunnel
# ======================================================================

cloudflare_process = (

    subprocess.Popen(

        [
            "cloudflared",
            "tunnel",
            "--url",
            NARAD_API_LOCAL_URL
        ],

        stdout=subprocess.PIPE,

        stderr=subprocess.STDOUT,

        text=True,

        bufsize=1
    )
)


# ======================================================================
# 4. Read Cloudflare output asynchronously
#
# Avoid blocking readline() defeating timeout logic.
# ======================================================================

_cloudflare_output_queue = (
    queue.Queue()
)


def _narad_cloudflare_reader():

    try:

        for line in cloudflare_process.stdout:

            _cloudflare_output_queue.put(
                line
            )

    finally:

        _cloudflare_output_queue.put(
            None
        )


_cloudflare_reader_thread = (

    threading.Thread(

        target=(
            _narad_cloudflare_reader
        ),

        name=(
            "NARAD-QC-CLOUDFLARE-READER"
        ),

        daemon=True
    )
)


_cloudflare_reader_thread.start()


# ======================================================================
# 5. Detect generated trycloudflare.com URL
# ======================================================================

NARAD_PUBLIC_API_URL = None


_url_pattern = re.compile(

    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
)


_tunnel_creation_deadline = (

    time.monotonic()
    +
    45.0
)


while (

    time.monotonic()

    <

    _tunnel_creation_deadline
):

    # Tunnel process crashed
    if (

        cloudflare_process.poll()

        is not None
    ):

        break


    remaining = (

        _tunnel_creation_deadline
        -
        time.monotonic()
    )


    try:

        line = (

            _cloudflare_output_queue.get(

                timeout=min(

                    0.5,

                    max(
                        remaining,
                        0.01
                    )
                )
            )
        )


    except queue.Empty:

        continue


    if line is None:

        break


    print(
        line.rstrip()
    )


    match = (

        _url_pattern.search(
            line
        )
    )


    if match:

        NARAD_PUBLIC_API_URL = (

            match.group(0)
            .rstrip("/")
        )

        break


if NARAD_PUBLIC_API_URL is None:

    return_code = (
        cloudflare_process.poll()
    )


    if return_code is None:

        cloudflare_process.terminate()


    raise RuntimeError(

        "Cloudflare Quick Tunnel URL was not detected "
        "within 45 seconds. "
        f"cloudflared return code={return_code}"
    )


print(
    "\n[PASS] Cloudflare Quick Tunnel created:"
)

print(
    NARAD_PUBLIC_API_URL
)


# ======================================================================
# 6. PUBLIC PROPAGATION / READINESS CHECK
#
# IMPORTANT:
#
# A newly generated *.trycloudflare.com hostname may temporarily fail
# DNS resolution.
#
# Example:
#
#     [Errno -2] Name or service not known
#
# That is NOT immediately considered tunnel failure.
# ======================================================================

_public_ready_url = (

    NARAD_PUBLIC_API_URL

    +

    "/ready"
)


_public_propagation_deadline = (

    time.monotonic()

    +

    120.0
)


_public_ready_payload = None

_public_last_error = None

_attempt = 0

_last_status_print = 0.0


while (

    time.monotonic()

    <

    _public_propagation_deadline
):

    _attempt += 1


    # ==============================================================
    # If cloudflared itself exited, waiting for DNS is pointless.
    # ==============================================================

    return_code = (
        cloudflare_process.poll()
    )


    if return_code is not None:

        raise RuntimeError(

            "Cloudflare tunnel process exited before the "
            "public endpoint became reachable. "
            f"Return code={return_code}"
        )


    try:

        _public_ready_payload = (

            _narad_http_json(

                _public_ready_url,

                timeout_s=5.0
            )
        )


        if (

            _public_ready_payload.get(
                "ready"
            )

            is True

            and

            _public_ready_payload.get(
                "runtime_generation"
            )

            ==

            NARAD_API_RUNTIME_GENERATION
        ):

            break


        # ----------------------------------------------------------
        # Host resolved, but result was not our current runtime.
        # ----------------------------------------------------------

        _public_last_error = (

            RuntimeError(

                "Public endpoint responded but readiness/runtime "
                "generation did not yet match."
            )
        )


    except urllib.error.HTTPError as error:

        # ----------------------------------------------------------
        # DNS resolved and HTTP exists, but Cloudflare/backend may
        # still be propagating.
        # ----------------------------------------------------------

        _public_last_error = error


    except urllib.error.URLError as error:

        # ----------------------------------------------------------
        # Includes temporary:
        #
        #   Name or service not known
        #   Temporary failure in name resolution
        #   connection refused/reset
        #
        # These are expected for a short period after Quick Tunnel
        # creation.
        # ----------------------------------------------------------

        _public_last_error = error


    except socket.gaierror as error:

        _public_last_error = error


    except Exception as error:

        _public_last_error = error


    now = time.monotonic()


    # ==============================================================
    # Don't spam notebook output.
    # Print roughly every 10 seconds.
    # ==============================================================

    if (

        now

        -

        _last_status_print

        >=

        10.0
    ):

        remaining_s = max(

            0.0,

            _public_propagation_deadline

            -

            now
        )


        print(

            f"[WAIT] Public tunnel propagation... "
            f"attempt={_attempt}, "
            f"remaining≈{remaining_s:.0f}s"
        )


        if _public_last_error is not None:

            print(

                "       Last result:",

                repr(
                    _public_last_error
                )
            )


        _last_status_print = now


    time.sleep(
        1.0
    )


# ======================================================================
# 7. Final validation
# ======================================================================

_public_verified = bool(

    _public_ready_payload is not None

    and

    _public_ready_payload.get(
        "ready"
    )

    is True

    and

    _public_ready_payload.get(
        "runtime_generation"
    )

    ==

    NARAD_API_RUNTIME_GENERATION
)


if not _public_verified:

    # ==============================================================
    # Tunnel really failed our bounded verification window.
    # Only NOW terminate it.
    # ==============================================================

    if (

        cloudflare_process.poll()

        is None
    ):

        cloudflare_process.terminate()


    raise RuntimeError(

        "Cloudflare Quick Tunnel was created, but its public "
        "/ready endpoint did not become reachable within "
        "120 seconds. "
        f"Last probe error: {_public_last_error!r}"
    )


# ======================================================================
# 8. Success
# ======================================================================

print("\n" + "=" * 100)

print(
    "[PASS] CLOUDFLARE TUNNEL VERIFIED END-TO-END"
)

print("=" * 100)


print(
    "\nNARAD-QC PUBLIC API:"
)

print(
    NARAD_PUBLIC_API_URL
)


print(
    "\nPublic readiness:"
)

print(
    _public_ready_url
)


print(
    "\nSwagger UI:"
)

print(

    NARAD_PUBLIC_API_URL

    +

    "/docs"
)


print(
    "\nRuntime generation:"
)

print(
    NARAD_API_RUNTIME_GENERATION
)


print(
    "\nPublic readiness payload:"
)

print(

    json.dumps(

        _public_ready_payload,

        indent=2
    )
)


print("\n" + "=" * 100)

In [ ]:
# @title GUI bridge
# Paste this cell into Colab AFTER running the notebook through Phase 6.
# Upload the updated gui_bridge.py to your Drive folder or /content/gui_bridge.py.
from pathlib import Path
import importlib.util
import threading
import time
import urllib.request
import json
import uvicorn

_bridge_path = next((p for p in [Path('/content/drive/MyDrive/NARAD_QC/gui_bridge.py'), Path('/content/gui_bridge.py')] if p.exists()), Path('/content/gui_bridge.py'))
if not _bridge_path.exists():
    raise FileNotFoundError('Upload gui/backend/gui_bridge.py to the Colab Files panel first.')
_previous_gui = globals().get('NARAD_GUI_SERVER')
_previous_gui_thread = globals().get('NARAD_GUI_THREAD')
_previous_gui_app = globals().get('NARAD_GUI_APP')
def _gui_jobs_active():
    return _previous_gui_app is not None and any(j['status'] in ('QUEUED','RUNNING') for j in getattr(_previous_gui_app.state,'gui_jobs',{}).values())
if _gui_jobs_active():
    raise RuntimeError('Wait for current GUI jobs to finish before replacing the bridge.')
if _previous_gui_thread is not None and _previous_gui_thread.is_alive():
    _previous_gui.should_exit = True
    _previous_gui_thread.join(timeout=5)
    if _previous_gui_thread.is_alive():
        raise RuntimeError('Previous GUI server did not stop cleanly. Restart the runtime before installing.')
if _gui_jobs_active():
    raise RuntimeError('A job started while the server was stopping. Let it finish before rerunning this cell.')
if _previous_gui_app is not None:
    _previous_gui_app.state.gui_pool.shutdown(wait=True)
_spec = importlib.util.spec_from_file_location('narad_gui_bridge', _bridge_path)
_bridge = importlib.util.module_from_spec(_spec)
# Required by Pydantic to resolve postponed type annotations.
import sys
sys.modules[_spec.name] = _bridge
_spec.loader.exec_module(_bridge)
NARAD_GUI_APP = _bridge.install_gui_bridge(globals())
NARAD_GUI_SERVER = uvicorn.Server(uvicorn.Config(NARAD_GUI_APP, host='127.0.0.1', port=8002, log_level='warning'))
NARAD_GUI_THREAD = threading.Thread(target=NARAD_GUI_SERVER.run, name='NARAD-GUI', daemon=True)
NARAD_GUI_THREAD.start()
for _ in range(50):
    try:
        _request = urllib.request.Request('http://127.0.0.1:8002/gui/health', headers={'X-NARAD-API-Key': NARAD_API_ACCESS_KEY})
        with urllib.request.urlopen(_request, timeout=1) as _response:
            assert json.load(_response)['bridge'] == 'NARAD-QC-GUI'
        break
    except Exception:
        time.sleep(.2)
else:
    raise RuntimeError('GUI bridge failed to start on port 8002.')
print('GUI bridge ready on http://127.0.0.1:8002')
print('Bridge version:', _bridge.VERSION)
print('Use your existing NARAD_API_ACCESS_KEY in the React connection dialog; do not save it in source.')
# Expose this NEW port, not the legacy port 8001:
# !cloudflared tunnel --url http://127.0.0.1:8002
# Copy its https://….trycloudflare.com URL into the GUI connection dialog.

In [ ]:
print(NARAD_API_ACCESS_KEY)

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8002